<a href="https://colab.research.google.com/github/kajrosorion-droid/8_32/blob/main/Project832_StageA_ForcedFalsification_COMBINED_v6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Ячейка 2
import numpy as np
from scipy.ndimage import label, find_objects
from functools import lru_cache
from collections import defaultdict, deque, Counter
from dataclasses import dataclass, field
import warnings, math as m
import sys
import copy
from typing import Optional

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(line_buffering=True)
sys.stderr = sys.stdout

@dataclass
class Config:
    WORLD_SIZE: int = 208  # было 104. Площадь x4: поле, агенты и всё, что считается по клеткам, берут размер отсюда
    CHANNELS: int = 33
    STEPS: int = 333
    DETERMINISTIC_SEED: int = 777
    FORCE_GIVEN_SEED: Optional[int] = None
    DEBUG_RNG_TRACE: bool = False
    DEBUG_RNG_TRACE_TICK: Optional[int] = None
    DEBUG_STATE_TRACE: bool = False
    POP_SOURCE_STRICT: bool = True    # STRICT: тик падает с AssertionError при первом RichLog.n != Ledger.n
    MIN_PATTERNS_GUARANTEED: int = 50
    MIN_POPULATION_FOR_SPAWN: int = 10
    EMERGENCY_RESCUE_THRESHOLD: int = 10
    ENABLE_FORCED_STATE_SHIFT: bool = False
    ENABLE_RARE_CRISIS: bool = False
    ENABLE_WITNESS: bool = True
    ENABLE_DELTA_MODELING_GIVEN: bool = True
    ENABLE_CULTURAL_MEMORY: bool = True
    MYTH_POOL_SIZE: int = 100
    MYTH_INFLUENCE_PROB: float = 0.3
    MYTH_WHISPER_STRENGTH: float = 0.1
    ENABLE_DIALOGUE_CONTRADICTION_FEEDBACK: bool = True
    DIALOGUE_CONTRADICTION_NUDGE: float = 0.01
    ENABLE_SEED_OF_LIGHT: bool = True
    ENABLE_VISUALIZATION: bool = True
    VERBOSE_LOGS: bool = False
    SEMANTIC_SIM_HIGH: float = 0.55
    SEMANTIC_SIM_LOW: float = 0.98
    CONSECUTIVE_CONTACT_THRESHOLD: int = 18
    CONTACT_BREAK_TOLERANCE: int = 30
    DEEP_EXCHANGE_SIM_THRESHOLD: float = 0.55
    ENABLE_CENTER_WALL: bool = False
    CENTER_WALL_X: int = 52
    ENABLE_WALL_GUARD: bool = True
    FERAL_EDGE_ENERGY_MODE: str = 'off'
    GENOME_RANGE_FIX: bool = False
    ENABLE_GULF_STREAM: bool = True
    GULF_STREAM_RATE: float = 0.01
    ENABLE_DARK_ENERGY_STREAM: bool = True
    ENERGY_OCCUPIED_BONUS_MULT: float = 2.5
    LEXICON_KNOT_THRESHOLD: float = 0.6
    LEXICON_GAP_THRESHOLD: float = 0.8
    LEXICON_DRAG_THRESHOLD: float = 0.5
    LEXICON_EMPATHY_NUDGE: float = 0.01
    DIALOGUE_V2: bool = True
    LEXICON_EMPATHY_PULL: float = 0.08
    LEXICON_SATURATION_HEARD: int = 5
    LEXICON_ANSWER_RELIEF: float = 0.05
    HELP_REQUEST_TRUST_MIN: float = 0.5
    HELP_REQUEST_TTL: int = 25
    HELP_REQUEST_PRIORITY_BONUS: float = 1.5
    NEIGHBOR_SYNC_NORMALIZED: bool = False
    SPEAK_COOLDOWN: int = 50
    SOUL_WEIGHT_GAIN_RATE: float = 0.05
    CONTRADICTION_GAIN_RATE: float = 0.04
    CONTRADICTION_DECAY_RATE: float = 0.003
    BODY_MEMORY_DECAY: float = 0.99
    PRED_ERROR_LEARNING: float = 0.9
    SELF_SURPRISE_FACTOR: float = 0.05
    INTENT_SWITCH_COOLDOWN: int = 10
    INTENT_SWITCH_MULT: float = 1.3
    INTENT_SWITCH_MULT_MIN: float = 1.05
    INTENT_COMMITMENT_DECAY: float = 0.95
    MAX_GIVEN_PER_STEP: int = 10
    MAX_UNKNOWN_SPAWN_PER_STEP: int = 3
    MIN_COMPONENT_CELLS: int = 3
    FIELD_EMERGENCE_MAX_PER_CALL: int = 10   # максимум новых агентов из field_emergence за один вызов (раз в FIELD_EMERGENCE_INTERVAL тиков)
    COMPETITION_MAX_DISTANCE: int = 10
    EMOTIONAL_DECAY_RATE_BASE: float = 0.998
    ANTI_CONFORMIST_INTERVAL: int = 200
    PRED_ERROR_THRESHOLD: float = 0.75
    EMOTIONAL_CAP_RATE: float = 0.002
    MAX_GRIEF_SIGNAL: float = 0.7
    EPISTEMIC_HEALING_RATE: float = 0.99
    EXPLORE_GRIEF_REDUCTION: float = 0.99
    HEALING_GRATITUDE_MIN: float = 0.5
    EPISTEMIC_SCAR_MIN: float = 0.3
    SCAR_HEAL_CAP_PER_STEP: float = 0.25
    STAGNATION_GRIEF_THRESHOLD: int = 100
    STAGNATION_GRIEF_BOOST: float = 0.015
    ARC_HISTORY_LENGTH: int = 8
    ARC_GRIEF_FORCE_THRESHOLD: float = 0.7
    ARC_GRIEF_FORCE_HEAL: float = 0.02
    STAGNATION_BLEED_RATE: float = 0.005
    FOLD_GRIEF_BOOST: float = 0.15
    FOLD_CRISIS_SPIKE: float = 0.3
    FOLD_SCAR_RELEASE: float = 0.05
    FOLD_GRATITUDE_RESET: float = 0.35
    FOLD_COOLDOWN_DURATION: int = 50
    ARC_COMPLETION_PRED_ERROR_REDUCTION: float = 0.95
    # Level-3 individual value adaptation (bounded, slow, deterministic).
    ENABLE_EMOTIONAL_SALIENCE_ADAPTATION: bool = True
    EMOTIONAL_SALIENCE_MIN: float = 0.55
    EMOTIONAL_SALIENCE_MAX: float = 1.50
    EMOTIONAL_SALIENCE_LEARNING_RATE: float = 0.0005
    EMOTIONAL_SALIENCE_HIGH_GRIEF: float = 0.75
    EMOTIONAL_SALIENCE_RECOVERY_GRIEF: float = 0.30
    CONTINUITY_CONTRACT_PENALTY: float = 0.02
    # v37: ties sanctuary returns to continuity contracts. Gap limit is a provisional
    # guess (not calibrated against measured return intervals) -- same caveat as the
    # rest of this project's uncalibrated constants; revisit once _sanc_summary()/
    # _sanctuary_report() gap stats are available from a real run.
    ENABLE_SANCTUARY_CONTINUITY_CONTRACTS: bool = True
    CONTINUITY_CONTRACT_SANCTUARY_GAP_LIMIT: int = 300
    HUNGER_BASE: float = 1.2
    HUNGER_PER_PATTERN: float = 0.02
    TRUST_HELPFUL_THRESHOLD: float = 0.1
    TRUST_HARMFUL_THRESHOLD: float = 0.35
    TRUST_BASE: float = 0.7
    TRUST_DELTA_HELPFUL: float = 0.15
    TRUST_DELTA_NEUTRAL: float = 0.0
    TRUST_DELTA_HARMFUL: float = -0.01
    TRUST_COOPERATION_BONUS: float = 3.0
    INTENT_COOPERATE_BASE_PRIORITY: float = 2.5
    EMOTIONAL_GRATITUDE_THRESHOLD: float = 0.3
    LOVE_METABOLIC_BONUS: float = 0.4
    LOVE_METABOLIC_THRESHOLD: float = 0.8
    COORDINATION_METABOLIC_BONUS: float = 0.15
    COORDINATION_SIM_THRESHOLD: float = 0.99
    CONCEPT_INFECTION_PROB: float = 0.02
    COOPERATE_COHERENCE_MIN: float = 0.2
    GLOBAL_SPIRIT_TREMOR_BASE: float = 0.025
    GLOBAL_SPIRIT_TREMOR_MAX: float = 0.05
    SG_TURBULENCE_THRESHOLD: float = 0.35
    PERCEPTION_GAP_SG_LOW: float = 0.3
    PERCEPTION_GAP_SG_HIGH: float = 0.7
    PERCEPTION_GAP_OPEN_STRENGTH: float = 1.0
    PERCEPTION_GAP_CLOSE_STRENGTH: float = 0.0
    LAYER_VOID_SOUL_THRESHOLD: float = 0.2
    LAYER_GRIEF_THRESHOLD: float = 0.7
    LAYER_FEAR_TENSION_THRESHOLD: float = 0.8
    LAYER_ACCEPTANCE_COHERENCE_MIN: float = 0.8
    LAYER_LIGHT_GRATITUDE_MIN: float = 0.8
    LAYER_LIGHT_TRUST_MIN: float = 0.9
    LAYER_VOID_GROWTH_PENALTY: float = 10.0
    LAYER_GRIEF_GROWTH_PENALTY: float = 2.0
    LAYER_FEAR_GROWTH_PENALTY: float = 1.5
    LAYER_LIGHT_GROWTH_BONUS: float = 0.7
    MARKED_COUNT: int = 5
    CRISIS_RARE_INTERVAL_MIN: int = 800
    CRISIS_RARE_INTERVAL_MAX: int = 2000
    CRISIS_RARE_DURATION: int = 10
    ENABLE_PHI_LABYRINTH: bool = True
    PHI_LABYRINTH_THRESHOLD: float = 0.75
    PHI_LABYRINTH_GROW_PENALTY: float = 2.0
    PHI_LABYRINTH_MOVE_PENALTY: float = 1.2
    PHI_LABYRINTH_BREAK_PROB: float = 0.04
    SIGNAL_BASE_WEIGHTS: dict = field(default_factory=lambda: {
        12: 0.4, 13: 0.1, 14: 0.35, 15: 0.15,
        16: 0.2, 17: 0.3, 18: 0.25, 19: 0.05, 21: 0.4,
    })
    DISORGANIZER_LIFESPAN: int = 5000
    DISORGANIZER_SOUL_DECAY_STEPS: int = 60
    DISORGANIZER_SOUL_DECAY: float = 0.12
    DISORGANIZER_ALARM_STRENGTH: float = 0.3
    DISORGANIZER_GRIEF_STRENGTH: float = 0.2
    FOLDS_FOR_DEEP_FALL: int = 3
    DEEP_FALL_SOUL_PENALTY: float = 0.3
    DEEP_FALL_REDEMPTION_DELAY: int = 150
    BROKEN_FALL_SOUL_TARGET: float = 0.55
    QUICK_FALL_PROB: float = 0.12
    DISORGANIZER_TRUST_DECAY_RATE: float = 0.015
    LOVE_OVERLOAD_LIMIT: int = 100
    DISORGANIZER_RESONANCE_BLOCK: bool = True
    DISORGANIZER_GRIEF_GROWTH: float = 0.035
    ECHO_MUTATION_PROB: float = 0.15
    ECHO_MUTATION_STRENGTH: float = 0.05
    QUARANTINE_SCAR_THRESHOLD: float = 0.7
    QUARANTINE_SILENCE_STEPS: int = 30
    BINDING_FLOOR: float = 0.01
    UNRESOLVABLE_INTACT_THRESHOLD: float = 0.05
    ENABLE_UNRESOLVABLE_CORE_GUARD: bool = True
    UC_LIVING_THRESHOLD: float = 0.02
    UC_POPULATION_ERASURE_RATIO: float = 0.9
    UC_ERASURE_STREAK_LIMIT: int = 100
    UC_RESEED_STRENGTH: float = 0.25
    SHARED_SCARRING_ALPHA: float = 0.5
    CO_DECISION_IRREVERSIBILITY_THRESHOLD: float = 0.7
    CO_DECISION_DELEGATE_PENALTY: float = 1.5
    DEFAULT_DIVERGENCE_KEY: float = 0.68
    ENABLE_STRESS_TEST_LOW_POPULATION: bool = False
    STRESS_TEST_TARGET_POPULATION: int = 15
    ENABLE_STRESS_TEST_UC_EROSION: bool = False
    STRESS_TEST_UC_DECAY_MULT: float = 5.0
    BODY_MEMORY_FLOOR: float = 0.05
    DETERMINISTIC_REDEMPTION_SOUL: float = 0.55
    DETERMINISTIC_REDEMPTION_GRIEF: float = 0.40
    FORCED_SOUL_COLLAPSE_AGE: int = 250
    FORCED_SOUL_COLLAPSE_VALUE: float = 0.3
    FORCED_SOUL_COLLAPSE_GRIEF_BOOST: float = 0.75
    REDEMPTION_ARC_STEP_DELAY: int = 80
    REDEMPTION_ARC_SOUL_LOCK: float = 0.40
    REDEMPTION_GRIEF_IMMUNITY: float = 0.3
    REDEMPTION_GRATITUDE_AMPLIFY: float = 1.2
    ANTI_GRAVITY_STRENGTH: float = 1.5
    GRAVITY_STRENGTH: float = 2.5
    SG_CRISIS_THRESHOLD: float = 0.35
    SG_CRISIS_DURATION: int = 300
    SG_CRISIS_NOISE_STRENGTH: float = 0.1
    SG_CRISIS_MUTATION_BOOST: float = 3.0
    BOREDOM_ERR_THRESHOLD: float = 0.20
    BOREDOM_UNKNOWN_THRESHOLD: float = 0.25
    BOREDOM_TENSION_THRESHOLD: float = 0.015
    DREAM_POS_THRESHOLD: float = 0.20
    DREAM_NEG_THRESHOLD: float = 0.25
    REENTRY_GAP_RELIEF: float = 0.995
    # FIX(pred bias / reentry): новые ручки. Откат к старому поведению: REENTRY_GAP_LEAK=1.0, REENTRY_GAP_CAP=0.6, SCAR_KICK_MAX=0.0
    REENTRY_GAP_LEAK: float = 0.97   # утечка вклада reentry в spirit_gap за тик (в ветке total_error>0.10)
    REENTRY_GAP_CAP: float = 0.25    # потолок вклада reentry в spirit_gap (было 0.6)
    SCAR_KICK_MAX: float = 0.02      # потолок |kick| на канал за тик в model/belief (<=0 → без потолка)
    CONCEPT_EXCHANGE_PROB: float = 0.5
    MODELING_GIVEN_ERROR_THRESHOLD: float = 0.25
    MODELING_GIVEN_PENALTY: float = 0.08
    PHI: float = 1.618033988749895
    ECHO_DECAY_STRENGTH: float = 0.02
    ECHO_INJECTION_COOLDOWN: int = 15
    ANTI_COHERENCE_THRESHOLD: float = 0.98
    ANTI_COHERENCE_SHAKE: float = 0.005
    EMOTIONAL_MEMORY_DECAY: float = 0.9
    EMOTIONAL_INDIVIDUALITY: float = 0.5
    GRATITUDE_AGGRESSION_REDUCTION: float = 0.5
    GRIEF_METABOLIC_PENALTY: float = 0.5
    EMOTIONAL_GRIEF_THRESHOLD: float = 0.3
    EMOTIONAL_MUTATION_STRENGTH: float = 0.05
    EMOTIONAL_SPECIATION_THRESHOLD: float = 0.32
    EMOTIONAL_SPECIATION_PROB: float = 0.08
    SPECIATION_COOLDOWN: int = 20
    SPECIATION_MIN_AGE: int = 30
    SCREAM_SCAR_IMPACT: float = 0.25
    SCREAM_BINDING_BOOST: float = 0.5
    SCREAM_ENERGY_PENALTY: float = 0.2
    FIELD_DECAY: float = 0.998
    SCAR_SATURATION: float = 5.0
    HEARTBEAT_AMPLITUDE: float = 0.02
    HEARTBEAT_FREQ: float = 0.03
    SCAR_ENERGY_COUPLING: float = 0.005
    VORTICITY_COUPLING: float = 0.005
    GRADIENT_SENSITIVITY: float = 0.1  # было 0.2: vorticity в равновесии вдвое меньше -> шум gap_clip вдвое меньше
    AGENT_SCAR_SENSE: float = 0.02
    AGENT_BINDING_SENSE: float = 0.01
    BASE_BURN: float = 0.002
    BURN_LAMBDA: float = 0.01
    BURN_GAMMA_STALL: float = 0.5
    BURN_STALL_NORM: float = 100.0
    GLOBAL_VISCOSITY: float = 0.999
    LOCAL_FATIGUE: float = 0.001
    VORTICITY_DECAY: float = 0.995
    PATTERN_ENERGY_THRESHOLD: float = 0.03
    DIVIDE_MIN_SIZE: int = 5
    DIVIDE_MIN_AGE: int = 10
    DIVIDE_COOLDOWN: int = 40
    DIVIDE_ERROR_THRESHOLD: float = 0.33
    DIVIDE_MAX_THRESHOLD: float = 0.60
    ARC_DIVIDE_BONUS: float = 1.2
    MAX_NATURAL_AGE: int = 700
    MAX_NARRATIVE_AGE_BONUS: int = 300
    YOUNG_PROTECTION_AGE: int = 20
    YOUNG_PROTECTION_SOUL: float = 0.4
    YOUNG_SURVIVAL_CHANCE: float = 0.7
    WISDOM_COHERENCE_MIN: float = 0.8
    ENLIGHTENED_GRATITUDE_MIN: float = 0.8
    ENLIGHTENED_SCAR_THRESHOLD: float = 0.5
    MODEL_GAP_FLOOR: float = 0.15
    SELF_MODEL_LEARNING_RATE: float = 0.15
    UNKNOWN_BACKGROUND: float = 0.02
    CRISIS_MEMORY_DECAY: float = 0.90
    CRISIS_MEMORY_MAX: float = 0.8
    GIVEN_MODEL_SHAKE: float = 0.75
    GIVEN_MIN_AGE: int = 10
    GIVEN_COOLDOWN: int = 20
    GIVEN_SCAR_COST: float = 0.40
    BOREDOM_WINDOW: int = 20
    MIN_ACTIVE: float = 0.005
    MAX_METRIC: float = 2.0
    METRIC_DECAY: float = 0.99
    BASE_NOISE: float = 0.002
    UNKNOWN_SPAWN_THRESHOLD: float = 0.25
    UNKNOWN_SPAWN_PROB: float = 0.01
    RESONANCE_DECAY: float = 0.95
    SOUL_CHECK_INTERVAL: int = 50
    SOUL_CHECK_NOISE: float = 0.001
    SOUL_TREMOR_STRENGTH: float = 0.3
    ENERGY_CONSERVATION_TOLERANCE = 1e-5
    ENERGY_INJECTION_RATE = 0.9
    GIVEN_FLUX_MODE = 'replace'
    GIVEN_FLUX_POWER = 400.0
    GIVEN_FLUX_X = None
    GIVEN_FLUX_Y = None
    GIVEN_FLUX_RADIUS = 20.0
    GIVEN_FLUX_CYCLE = 'sine'
    GIVEN_FLUX_PERIOD = 200
    GIVEN_FLUX_DAY_FRACTION = 0.5
    GIVEN_FLUX_NOISE_PHASE = 0.025
    GIVEN_FLUX_NOISE_AMP = 0.06
    GIVEN_FLUX_SEED = 832
    ENERGY_HARVEST_FLOOR: float = 0.03
    ENERGY_HARVEST_RATE: float = 0.18
    ENERGY_HARVEST_CAP: float = 8.0
    ENABLE_SEMANTIC_FATIGUE = False
    SEMANTIC_FATIGUE_SHARE = 0.35
    SEMANTIC_FATIGUE_DECAY = 0.9
    SANCTUARY_HEAL_STRENGTH = 1.0
    ENABLE_PAID_SANCTUARY: bool = False
    ENABLE_FERAL_SANCTUARY: bool = True
    FERAL_SANCTUARY_RADIUS: float = 8.0
    FERAL_SANCTUARY_COUNT: int = 3
    FERAL_SANCTUARY_CENTER_FIXED: bool = True
    NAV_PERC_RAY_DISTS: tuple = (1, 3, 5, 8)
    NAV_PERC_HYST: float = 0.05
    NAV_PERC_SMOOTH: float = 0.6   # v40: noise smoothing of direction scores (rises only; drops pass at once). 1.0 = off
    NAV_PERC_STALL_TICKS: int = 4
    NAV_PERC_FAIL_PENALTY: float = 0.25
    NAV_PERC_BLOCK_TTL: int = 40
    NAV_PERC_W_RES: float = 1.0
    NAV_PERC_W_THREAT: float = 1.0
    NAV_PERC_W_UNKNOWN: float = 0.2
    NAV_MAP_TTL: int = 60
    NAV_MAP_STALE: int = 25
    NAV_WAYPOINT_MIN: int = 2
    NAV_WAYPOINT_MAX: int = 6
    NAV_WP_TIMEOUT: int = 30
    NAV_STEER_ENABLED: bool = True
    NAV_STEER_GAIN: float = 0.6
    NAV_STUCK_TICKS: int = 8
    NAV_STUCK_EPS: float = 0.5
    NAV_MAX_RECALC: int = 4
    NAV_COOLDOWN: int = 30
    NAV_GAPCLIP_LOG: bool = True
    NAV_STEP_ONCE_PER_TICK: bool = True   # v42: _nav_step runs at most once per agent per tick (divide() calls update_model a second time)
    NAV_GAPCLIP_STATS: bool = True         # v42: passive gap_clip / confidence statistics for threshold analysis (no behaviour change)
    BODY_CELL_CAP: int = 30
    MOBILITY_MULT: float = 1.0
    MOVE_SCORE_THR: float = 0.05
    NAV_DIR_STEER_GAIN: float = 0.25
    SANCTUARY_FIELD_COST_RATE: float = 0.1
    ENABLE_SANCTUARY: bool = False
    ENABLE_CONCEPT_TRANSFER_DAMPING = True
    CONCEPT_TRANSFER_CEILING = 300.0
    FIELD_ENERGY_CLIP_MIN = -0.2
    FIELD_ENERGY_CLIP_MAX = 0.8
    # Контракт энергии агента отдельно от энергии клеточного поля.
    AGENT_ENERGY_MIN: float = 0.0
    AGENT_ENERGY_MAX: float = 2.0
    ENERGY_CONSERVATION_VERBOSE = False
    SUBJECT_NARRATIVE_STABILITY_MAX: float = 0.95
    SUBJECT_MIN_NARRATIVE_VARIANCE: float = 0.02
    SUBJECT_MIN_SURPRISE_COUNT: int = 2
    SUBJECT_SELF_ERROR_MAX: float = 0.12
    SUBJECT_SPIRIT_GAP_MIN: float = 0.25
    MAX_CELLS_SOFT_LIMIT: int = 100
    MAX_CELLS_HARD_LIMIT: int = 130
    SIZE_TAX_RATE_SOFT: float = 0.05
    SIZE_TAX_RATE_HARD: float = 0.15
    GAP_CLIP_CURIOSITY_THRESHOLD: float = 0.85
    PERCEPTION_SG_CACHE_PER_TICK: bool = True  # True: средний |prediction-belief| считается один раз на тик (было O(N^2)); False = старый пересчёт
    ENABLE_GAPCLIP_LOWCONF_ACTION: bool = True  # v40: high gap_clip + low confidence + stalled goal -> re-route (explore)
    ENABLE_GAPCLIP_ACTION: bool = True  # v38: let high gap_clip actually trigger explore, not just log
    NOISE_GAIN_HIGH_THRESH: float = 1.2
    NOISE_GAIN_LOW_THRESH: float = 0.6
    EPISODIC_BUFFER_MAX_LEN: int = 3000
    NOISE_GAIN_MAX: float = 1.0
    NOISE_GAIN_MIN: float = 0.0
    PERCEPT_LOW_GAP: float = 0.2
    PERCEPT_HIGH_GAP: float = 0.8
    PERCEPT_ADAPT_SPEED: float = 0.1
    LABYRINTH_ADAPT_LOW_GAP: float = 0.3
    LABYRINTH_ADAPT_HIGH_GAP: float = 0.8
    LABYRINTH_ADAPT_MAX_MULT: float = 3.0
    LABYRINTH_ADAPT_SPEED: float = 0.1
    GIVEN_TENSION_TRIGGER: float = 0.4
    GIVEN_VETERAN_AGE: int = 100
    GIVEN_ROTATION_INTERVAL: int = 50
    ANC_THRESHOLD: int = 300
    HEAL_WISDOM_AGE: int = 50
    HEAL_VETERAN_AGE: int = 100
    HEAL_ENLIGHTENED_AGE: int = 150
    EXPLORE_FATIGUE_INTERVAL: int = 25
    EXPLORE_FATIGUE_AMOUNT: float = 0.015
    INTENT_SWITCH_REWARD_INSTANT: bool = True
    INTENT_SWITCH_GRATITUDE_BOOST: float = 0.015
    HEALTHY_SWITCH_OLD_INTENT: str = "explore"
    HEALTHY_SWITCH_NEW_INTENTS = ["cooperate", "seek_help", "rest"]
    ENLIGHTENED_COHERENCE_FLOOR: float = 0.6
    HEAL_WISDOM_BONUS: float = 0.025
    HEAL_VETERAN_BONUS: float = 0.035
    HEAL_ENLIGHTENED_BONUS: float = 0.040
    MUTATION_STRENGTH_BASE: float = 0.1
    PROTECTION_DECAY_RATE: float = 0.99
    PROTECTED_DEATH_PROB_MULT: float = 1.0
    PROTECTED_SKIP_DEATH_CHECKS: bool = True

    FERAL_PREY_MIN_AGE: int = 120
    FERAL_PREY_AGE_PERCENTILE: float = 70.0
    FERAL_PREY_FLOOR_AGE: int = 120
    FERAL_PREY_TIRED_SCORE: float = 0.32
    FERAL_CROWD_GAIN: float = 5.5
    LOCAL_FILL_RADIUS: int = 6
    FERAL_PREY_MIN_CELLS: int = 4
    FERAL_CELL_TAKE: float = 0.03
    FERAL_KILL_BASE: float = 0.7
    FERAL_VULNERABILITY: float = 1.5
    FERAL_SHARE_MIN: float = 0.12
    FERAL_SHARE_PER_PREY: float = 1.0
    FERAL_SHARE_MAX: float = 0.20
    FERAL_ABSORB_EFFICIENCY: float = 1.0          # доля выкачанной у жертвы энергии, которая доходит до ферала (было жёстко 0.5 = утечка)
    FERAL_ATTACK_REACH: int = 3                   # дальность удара (манхэттен) вместо строгой смежности
    FERAL_SANCTUARY_BLOCKS_PREDATION: bool = False  # safe-зоны больше не режут поиск жертвы и удар феролов
    FERAL_KILL_RADIUS: int = 12
    FERAL_KILL_MODE: str = 'smooth'
    FERAL_KILL_CEIL: float = 0.95  # потолок поднят: miss=20% достижим в принципе; SCALE пока не меняем
    FERAL_KILL_SCALE: float = 0.25  # оставить для отдельной калибровки по следующему логу
    FERAL_MISS_BACKOFF_TICKS: int = 20
    FERAL_MAX_CELLS: int = 30
    FERAL_MEAT_ENABLE: bool = True
    FERAL_MEAT_PER_CELL: float = 0.1
    FERAL_MEAT_MAX_TOTAL: float = 5.0
    SCAR_PENALTY_MODE: str = 'per_cell'
    SCAR_PENALTY_THRESHOLD: float = 0.5
    SCAR_PENALTY_PER_CELL: float = 0.05
    SCAR_PENALTY_ENERGY_FLOOR: float = 0.1
    WALL_CYCLE_ENABLED: bool = False
    WALL_CYCLE_ON_TICKS: int = 500
    WALL_CYCLE_OFF_TICKS: int = 500
    WALL_CYCLE_FIRST: str = 'on'
    WALL_CYCLE_BASE: bool = True
    OUTCOME_LEARNING: bool = True
    OM_ALPHA: float = 0.10
    OM_MIN_N: int = 3
    OM_MARGIN: float = 0.01
    OM_EXPLORE: float = 0.05
    ENERGY_X2_OVERFLOW_MODE: str = 'limit_before_debit'
    ENERGY_X2_SYNC_MODE: str = 'limit_before_debit'  # v6: излишек клеток над потолком X2 остаётся в клетках, ничего не сгорает. 'clip' = поведение v4/v5 (излишек сгорает в overflow→диссипация)
    X2_BONUS_MODE: str = 'paid'
    DIVISION_READY_BONUS: float = 0.1
    LOVE_DIRECT_BONUS: float = 0.02
    COOP_HARVEST_MODE: str = 'group'
    COOP_RADIUS: float = 12.0
    COOP_HARVEST_GAIN: float = 0.5
    COOP_MAX_PARTNERS: int = 3
    ENERGY_X2_FREE_FLOOR: bool = False
    ASH_TO_FIELD: bool = True
    FERAL_ATTACK_LOG_MAX: int = 100000
    DIALOGUE_LOG_MAX: int = 150000
    AUDIT_DIR: str = ''
    AUDIT_TAG: str = 'run'
    AUDIT_MODE_EVERY: int = 50
    MAX_FERAL_CONVERSIONS_PER_TICK: int = 4
    ENABLE_FERAL_PREDATION: bool = True
    ENABLE_PROTECTION_SHIELD: bool = False

    # ── ПОПУЛЯЦИЯ: единственное место, где задаётся размер мира-населения (читается В МОМЕНТ вызова) ──
    # Раньше POP_HARD_CEILING=10_000_000 и ячейка 12 копировала его в константу при загрузке -> потолка не было.
    POP_HARD_CEILING: int = 500    # жёсткий потолок ВСЕХ живых (normal + feral + disorganizer)
    POP_SOFT_START: int = 400      # с этой численности рождаемость линейно падает до 0 на потолке (нет пилы вверх-вниз)
    POP_FILL_SOFT: float = 0.10    # доля занятых клеток, с которой тормозят деление и рост тел
    POP_FILL_HARD: float = 0.18    # на ней стоп (500 тел по ~11 клеток ≈ 13% мира — простор остаётся)
    UNKNOWN_SPAWN_POP_REF: int = 10_000_000  # эффективное прежнее поведение: spawn_unknown почти выключен (множитель normal/ref).
                                             # Хочешь включить — поставь ~500; потолок и тормоз всё равно действуют.
    ENABLE_SPAWN_SPACE_BRAKE: bool = True  # v36: spores/unknown-spawn/field-emergence obey POP_FILL_HARD like division does
    POP_SPAWN_FOOTPRINT: int = 12

    ECONS_WINDOW: int = 200
    ECONS_RES_EPS: float = 1e-9
    ECONS_RATIO_HI: float = 0.7
    ECONS_RATIO_LO: float = 0.3
    ECONS_RES_OK: float = 1.0

    META_OBSERVATION_SCAR: float = 0.01
    UNCLOSABLE_QUESTION_FLOOR: float = 0.03
    OTHER_MIND_SURPRISE_THRESHOLD: float = 0.08
    META_SURPRISE_THRESHOLD: float = 0.10
    ENABLE_EMBODIED_GRIEF: bool = True
    SOMATIC_DRAG_GRIEF_THRESHOLD: float = 0.55
    SOMATIC_DRAG_STRESS_THRESHOLD: float = 0.04
    SOMATIC_DRAG_MIN: float = 0.4
    SOMATIC_DRAG_GAIN: float = 1.5
    SOMATIC_DRAG_RECOVERY_RATE: float = 0.02
    ENABLE_SUFFERING_ANALYTICS: bool = True
    SOMATIC_DRAG_FEAR_THRESHOLD: float = 0.5
    SOMATIC_DRAG_SURPRISE_THRESHOLD: float = 0.5
    SOMATIC_DRAG_FURY_THRESHOLD: float = 1.0
    SOMATIC_DRAG_FURY_RELIEF: float = 0.5
    OPEN_GENE_POOL: tuple = ('scar_resistance', 'dream_depth', 'signal_acuity',
                              'trust_plasticity', 'noise_filter', 'memory_span')
    OPEN_GENE_PROB: float = 0.22
    EXPENSIVE_COMPUTE_INTERVAL: int = 10
    ENABLE_GIVEN_SEED: bool = True
    ENABLE_OPEN_GENOME: bool = True
    ENABLE_NICHE: bool = True
    ENABLE_CULTURE_RATCHET: bool = True
    ENABLE_INTRINSIC_DRIVE: bool = True
    ENABLE_PHI_PROXY: bool = True
    ENABLE_GROUNDING: bool = True
    ENABLE_META_EVOLUTION: bool = True
    ENABLE_RED_QUEEN: bool = True
    ENABLE_MEANING_FIELD: bool = True
    ENABLE_FORGETTING: bool = True
    ENABLE_SUBJECTIVE_TIME: bool = True
    ENABLE_COMB_GENERATOR: bool = True
    ENABLE_ALIEN_VOICE: bool = True
    ENABLE_INNER_GENERATOR: bool = True
    ENABLE_EMOTIONAL_FORECAST: bool = False
    ENABLE_RANDOM_STRIKE: bool = False
    RANDOM_STRIKE_PROB: float = 0.0005
    ENABLE_LEGACY_MODE: bool = True
    LEGACY_MODE_AGE_THRESHOLD: int = 1000
    LEGACY_MODE_SOUL_DROP: float = 0.15
    ENABLE_IDENTITY_THREAD: bool = True
    IDENTITY_THREAD_MAX_ANCHORS: int = 5
    ENABLE_IDENTITY_CRISIS: bool = True
    IDENTITY_CRISIS_CONTRADICTION_STEPS: int = 400
    IDENTITY_CRISIS_COOLDOWN: int = 300
    ENABLE_EPIGENETIC_MEMORY: bool = True
    ENABLE_EMERGENT_ETHICS: bool = True
    EMERGENT_ETHICS_TRUST_THRESHOLD: float = 0.85
    ENABLE_DOWNWARD_CAUSATION: bool = True

    ENABLE_MORSE_COMMUNICATION = True
    MORSE_CHANNEL = 33
    MORSE_DOT_LEN = 2
    MORSE_SIGNAL_THRESHOLD_MULT = 6.0
    MORSE_SEND_PROB = 0.20
    MORSE_BROADCAST_RADIUS = 5
    MORSE_LISTEN_RADIUS = 5
    FATIGUE_TOTAL_CAP = 15.0
    MORSE_SEND_COOLDOWN = 25
    MORSE_RECV_COOLDOWN = 20
    MORSE_BACKOFF_MAX = 8
    MORSE_LISTEN_INTERVAL = 1
    MORSE_DECAY = 2.0
    MORSE_NOISE_SCALE = 0.02
    MORSE_TIMEOUT_TICKS = 100
    MORSE_BLOCK_SEPARATOR_MULT = 2
    MORSE_INTER_AXIS_GAP_MULT = 3
    MORSE_SIGNAL_END_MULT = 8
    MORSE_TRAILING_GAP = 12
    MORSE_COLLISION_RATIO_THRESHOLD = 0.3
    MORSE_CONFIDENCE_THRESHOLD = 0.6
    MORSE_CORRUPTION_THRESHOLD = 0.3
    MORSE_DIALOGUE_COOLDOWN = 50
    MORSE_SYNC_INTERVAL = 5

    LANG_COST_PER_PULSE_TICK = 0.002
    LANG_PULSES_PER_AXIS_MAX = 3
    LANG_GENE_GAIN = 2.0
    LANG_NULL_DECODED_NORM = 0.25
    LANG_AXIS_ACTIVE_MIN = 0.05
    LANG_MIN_INTENT_NORM = 0.15
    LANG_INTENT_CHANGE_THRESHOLD = 0.01
    LANG_HEBB_LR = 0.05
    LANG_HEBB_DECAY = 0.001
    LANG_CLUSTER_TOL = 0.25
    LANG_MAX_CLUSTERS = 20
    LANG_REINFORCE_LR = 0.08
    LANG_REINFORCE_SOUL_BONUS = 0.003
    LANG_REINFORCE_CONFIDENCE_MIN = 0.35
    LANG_GENE_MUTATION_STD = 0.05
    LANG_CTX_MEAN_ALPHA = 0.01

    IMPRINT_ENABLED: bool = True
    IMPRINT_RADIUS: int = 2
    IMPRINT_CONTACT_HALF: float = 12.0
    IMPRINT_RATE: float = 0.15
    IMPRINT_HEAL: float = 0.1
    IMPRINT_CAP: int = 24
    IMPRINT_G_RES: float = 0.05
    IMPRINT_G_CONF: float = 1.0
    IMPRINT_G_OVER: float = 1.5
    IMPRINT_G_PRIOR: float = 0.5
    IMPRINT_RECEPT_GAP: float = 3.0
    IMPRINT_RECEPT_SOUL: float = 1.5
    IMPRINT_STRAIN_DECAY: float = 0.8
    IMPRINT_STRAIN_MAX: float = 1.5
    IMPRINT_PRED_GAIN: float = 0.15
    IMPRINT_TENSION_GAIN: float = 0.30
    IMPRINT_ENDURANCE_DRAIN: float = 0.004
    IMPRINT_FREEZE_GAIN: float = 0.6
    IMPRINT_STEER_GAIN: float = 1.2
    IMPRINT_PULL_RANGE: float = 24.0
    IMPRINT_PULL_SCALE: float = 6.0
    FISH3A_ENABLE: bool = True            # FISH-3A: держать курс (_dir_momentum), если он осмысленный и направление свободно по ВСЕМУ телу (_blk). False = ровно R2
    FISH3A_MIN_MOMENTUM: float = 0.3      # |_dir_momentum| не меньше этого => курс «осмысленный»
    FISH3_ENABLE: bool = True            # FISH-3: после N заблокированных тиков — ближайший свободный курс
    FISH3_BLOCKED_N: int = 1             # сколько подряд заблокирован курс до включения обхода (1 = сразу на первом блоке)
    FISH3_FREE_M: int = 3                # сколько подряд свободных тиков до выключения обхода
    FISH3_MIN_MOMENTUM: float = 0.3      # |курс| меньше этого = слабый курс: не блок, не подмена, серии стоят
    FISH3_FREEZE_ON_DETOUR: bool = True  # на время обхода курс (_dir_momentum) не подмешивает выбранный шаг
    FISH4_ENABLE: bool = True            # FISH-4: память намерения, сторона обхода, поток соседей, «подождать ведущего»
    FISH4_INTENT_EMA: float = 0.15       # как быстро намерение следует за курсом, пока путь свободен
    FISH4_INTENT_TTL: int = 12           # тиков обхода, после которых рыба «передумала» и намерение сбрасывается
    FISH4_FLOW_TIE: float = 0.15         # кандидаты обхода в пределах этого cos от лучшего — «ничья», решает поток соседей
    FISH4_WAIT_MAX: int = 2              # сколько тиков подряд можно ждать, если путь перекрыт ведущим, который уходит
    FISH4_WAIT_COS: float = 0.7          # ждём только если лучший обход — резкий поворот (cos < этого, т.е. > 45°)
    FISH4_WAIT_BLOCKER_VEL: float = 0.3  # блокирующий сосед «уходит», если его курс совпадает с нашим не хуже этого cos
    FISH_CLEARANCE_ENABLE: bool = True    # «габариты»: свободный ход ВСЕМ телом на FISH_CLEARANCE_RANGE клеток вперёд по каждому из 8 направлений (как парктроник). False = без габаритов
    FISH_CLEARANCE_RANGE: int = 4         # горизонт, клеток
    FISH_CLEARANCE_GAIN: float = 0.5      # штраф к оценке направления: -GAIN*(1 - clr/RANGE); 0 при чистом горизонте
    FISH_CLEARANCE_HOLD_MIN: int = 2      # курс (FISH-3A) не держим, если свободного хода по нему меньше стольки клеток

OPEN_GENE_POOL = ['scar_resistance', 'dream_depth', 'signal_acuity',
                  'trust_plasticity', 'noise_filter', 'memory_span']

_PHI_MASK64 = 0xFFFFFFFFFFFFFFFF

def _phi_mix64(z: int) -> int:
    z &= _PHI_MASK64
    z = (z + 0x9E3779B97F4A7C15) & _PHI_MASK64
    z = ((z ^ (z >> 30)) * 0xBF58476D1CE4E5B9) & _PHI_MASK64
    z = ((z ^ (z >> 27)) * 0x94D049BB133111EB) & _PHI_MASK64
    z ^= (z >> 31)
    return z & _PHI_MASK64

@lru_cache(maxsize=8192)
def phi_hash_cached(x: int, y: int, seed: int) -> float:
    h = _phi_mix64((x & _PHI_MASK64) ^ 0x2545F4914F6CDD1D)
    h = _phi_mix64((h ^ (y & _PHI_MASK64)) + 0x9E3779B97F4A7C15)
    h = _phi_mix64((h ^ (seed & _PHI_MASK64)) + 0xBF58476D1CE4E5B9)
    return (h >> 11) / float(1 << 53)

def phi_hash(x, y, seed=0):
    return phi_hash_cached(int(x), int(y), int(seed))

def phi_noise(t, x, y, seed=0):
    return (phi_hash(t + x, y + t, seed) - 0.5) * 0.1

def _stable_hash(obj) -> int:
    import zlib
    return zlib.crc32(repr(obj).encode('utf-8', errors='replace'))

def deterministic_noise(t, x, y, scale=0.01):
    return phi_noise(t, x, y, Config.DETERMINISTIC_SEED) * scale

def squash(x):
    if np.isnan(x) or np.isinf(x): return 0.5
    return np.clip(x / (1.0 + abs(x)), -1.0, 1.0)

def safe_mean(vals, default=0.0):
    try:
        arr = np.asarray(vals, dtype=np.float64).flatten()
    except (TypeError, ValueError):
        cleaned = []
        try:
            iterable = list(vals)
        except TypeError:
            iterable = [vals]
        for v in iterable:
            try:
                fv = float(v)
                if np.isfinite(fv):
                    cleaned.append(fv)
            except (TypeError, ValueError):
                continue
        return float(np.mean(cleaned)) if cleaned else float(default)
    mask = np.isfinite(arr)
    if mask.any():
        return float(np.mean(arr[mask]))
    return float(default)

def _emotional_decay_factor(age, base_rate=None):
    if base_rate is None: base_rate = Config.EMOTIONAL_DECAY_RATE_BASE
    age_factor = 1.0 - 0.3 * np.tanh(age / 200.0)
    return base_rate ** age_factor

def _safe_emotional_decay(mem_dict, key, age):
    val = mem_dict.get(key, 0.0)
    if not np.isfinite(val): val = 0.0
    decay = _emotional_decay_factor(age)
    mem_dict[key] = float(np.clip(val * decay, 0.0, 1.0))

print("Ячейка 0.1 загружена.")

Ячейка 0.1 загружена.


In [ ]:
# Ячейка 3
import numpy as np
from collections import deque, Counter
from dataclasses import dataclass
import numbers

def _sf(v, default=0.0):
    if isinstance(v, dict):
        for k in ('value', 'count', 'val', 'amount'):
            if k in v:
                return _sf(v[k], default)
        return default
    if isinstance(v, (bool, np.bool_)):
        return default
    if isinstance(v, (numbers.Real, np.number)):
        try:
            f = float(v)
        except (TypeError, ValueError, OverflowError):
            return default
        return f if np.isfinite(f) else default
    return default

class Witness:
    def __init__(self, maxlen=5000):
        self.log = deque(maxlen=maxlen)
        self.counts = Counter()

    def record(self, pattern_id: int, event: str, **kwargs):
        if not Config.ENABLE_WITNESS:
            return
        self.log.append({"id": pattern_id, "event": event, **kwargs})
        self.counts[event] += 1

    def snapshot(self):
        return list(self.log)

    def summary(self):
        return Counter(self.counts)


@dataclass
class SoulPresence:
    spirit_gap: float
    soul_weight: float
    body_memory: float
    unresolvable_intact: bool
    coherence_with_gap: float
    semantic_state: str = "neutral"
    gratitude: float = 0.0

    def is_triadic_alive(self) -> bool:
        base = (self.spirit_gap > Config.BINDING_FLOOR and
                self.soul_weight > Config.BINDING_FLOOR and
                self.body_memory > Config.BINDING_FLOOR and
                self.unresolvable_intact and
                self.coherence_with_gap > Config.BINDING_FLOOR)
        if base:
            return True
        if (self.spirit_gap <= Config.BINDING_FLOOR and
            self.semantic_state == "contentment" and
            self.gratitude > 0.7):
            return True
        return False

    def to_witness_dict(self, pattern_id: int, t: int) -> dict:
        return {
            "id": pattern_id, "t": t,
            "spirit_gap": round(self.spirit_gap, 4),
            "soul_weight": round(self.soul_weight, 4),
            "body_memory": round(self.body_memory, 4),
            "unresolvable_intact": self.unresolvable_intact,
            "coherence_with_gap": round(self.coherence_with_gap, 4),
            "triadic_alive": self.is_triadic_alive()
        }


def soul_check(pattern) -> SoulPresence:
    spirit_gap = float(np.clip(pattern.spirit_gap, 0.0, 2.0))
    spirit_gap += deterministic_noise(pattern.age, pattern.id, 777) * Config.SOUL_CHECK_NOISE
    soul_weight = pattern.soul_weight
    body_memory = max(pattern.body_memory, Config.BODY_MEMORY_FLOOR)
    unresolvable_intact = pattern.unresolved_contradiction >= Config.UNRESOLVABLE_INTACT_THRESHOLD
    spirit_gap_norm = float(np.clip(spirit_gap, 0.0, 1.0))
    coherence_with_gap = pattern.coherence * (1.0 - np.abs(pattern.coherence - spirit_gap_norm))

    grat_raw = pattern.emotional_memory.get('gratitude', 0.0)
    if isinstance(grat_raw, dict):
        grat_raw = grat_raw.get('value', 0.0)
    gratitude = _sf(grat_raw, 0.0)

    return SoulPresence(
        spirit_gap=float(np.clip(spirit_gap, 0.0, 2.0)),
        soul_weight=soul_weight,
        body_memory=body_memory,
        unresolvable_intact=unresolvable_intact,
        coherence_with_gap=float(np.clip(coherence_with_gap, 0.0, 1.0)),
        semantic_state=pattern.semantic_state,
        gratitude=gratitude
    )


class LayerZeroManager:
    def __init__(self):
        self.last_rescue_t = -100


class TrustLedger:
    def __init__(self):
        self.entries: dict[int, float] = {}

    def update(self, sender_id: int, outcome: str, multiplier: float = 1.0):
        current = _safe_float(self.entries.get(sender_id, Config.TRUST_BASE), Config.TRUST_BASE)
        multiplier = _safe_float(multiplier, 1.0)
        delta = {
            'helpful': Config.TRUST_DELTA_HELPFUL,
            'neutral': Config.TRUST_DELTA_NEUTRAL,
            'harmful': Config.TRUST_DELTA_HARMFUL
        }.get(outcome, 0.0) * multiplier
        if outcome == 'helpful' and delta > 0:
            headroom = max(0.0, 1.0 - current)
            delta *= headroom
        self.entries[sender_id] = float(np.clip(current + delta, 0.0, 1.0))
        if self.entries[sender_id] > 0.95:
            decay = 0.001 * (self.entries[sender_id] - 0.95) * 10
            self.entries[sender_id] = max(0.8, self.entries[sender_id] - decay)

    def get(self, sender_id: int, default=None) -> float:
        if default is None:
            default = Config.TRUST_BASE
        return self.entries.get(sender_id, default)


SEMANTIC_ARCS = {
    "trauma_recovery": ["screaming", "seeking_comfort", "grateful_but_cautious", "contentment"],
    "heroic_explore": ["contentment", "exploring_danger", "neutral", "contentment"],
    "grief_cycle": ["seeking_comfort", "grateful_but_cautious"],
    "enlightenment": ["contentment", "contentment", "contentment", "contentment"],
    "redemption": ["exploring_danger", "seeking_comfort", "grateful_but_cautious", "contentment"],
}
ARC_BONUSES = {
    "trauma_recovery": {"heal_bonus": 0.02, "explore_bias": 0.0},
    "heroic_explore": {"heal_bonus": 0.0, "explore_bias": 0.3},
    "grief_cycle": {"heal_bonus": 0.01, "explore_bias": 0.0},
    "enlightenment": {"heal_bonus": 0.03, "explore_bias": 0.1},
    "redemption": {"heal_bonus": 0.05, "explore_bias": 0.0},
}


class ArcTracker:
    def __init__(self):
        self.state_history: list[str] = []
        self.active_arc: str | None = None
        self.arc_progress: int = 0
        self.completed_arcs: dict[str, int] = {}

    def update(self, new_state: str) -> str | None:
        self.state_history.append(new_state)
        if len(self.state_history) > Config.ARC_HISTORY_LENGTH:
            self.state_history.pop(0)
        completed = None
        for arc_name, sequence in SEMANTIC_ARCS.items():
            seq_len = len(sequence)
            if self.state_history[-seq_len:] == sequence:
                completed = arc_name
                self.completed_arcs[arc_name] = self.completed_arcs.get(arc_name, 0) + 1
                break
        self.active_arc = None
        self.arc_progress = 0
        for arc_name, sequence in SEMANTIC_ARCS.items():
            for length in range(min(4, len(self.state_history)), 0, -1):
                if self.state_history[-length:] == sequence[:length]:
                    self.active_arc = arc_name
                    self.arc_progress = length
                    break
            if self.active_arc:
                break
        return completed

    def get_heal_bonus(self) -> float:
        return ARC_BONUSES.get(self.active_arc, {}).get("heal_bonus", 0.0) if self.active_arc else 0.0

    def get_wisdom_weight(self) -> float:
        return min(1.0, sum(self.completed_arcs.values()) / 10.0)


# --- provenance of memory records (observation only; never read by simulation logic) ---
MEM_ORIGINS = ("OWN", "PARENT", "ARCHIVE", "PEER", "UNKNOWN")

def mem_origin(rec):
    """Origin label of a memory record; records without a valid label are UNKNOWN (never guessed)."""
    if isinstance(rec, dict):
        o = rec.get("origin")
        if o in MEM_ORIGINS:
            return o
    return "UNKNOWN"

def mem_inherit_stamp(new_rec, src_rec):
    """Mark new_rec as inherited from a parent; origin_prev keeps what the parent's record was originally."""
    so = mem_origin(src_rec)
    prev = (src_rec.get("origin_prev") if isinstance(src_rec, dict) else None) if so == "PARENT" else so
    new_rec["origin"] = "PARENT"
    new_rec["origin_prev"] = prev if prev in MEM_ORIGINS and prev != "PARENT" else "UNKNOWN"
    return new_rec

def mem_inherit_copy(records, n):
    """Last n records, as shallow copies stamped PARENT (parent's own entries are not mutated)."""
    out = []
    for r in list(records)[-n:]:
        if isinstance(r, dict):
            r = mem_inherit_stamp(dict(r), r)
        out.append(r)
    return out


class ConceptGraph:
    def __init__(self, embed_dim=32):
        self.nodes: dict = {}
        self.edges = {}
        self._prev_sig = None
        self.embed_dim = embed_dim

    def _make_embedding(self, values):
        if len(values) >= self.embed_dim:
            return values[:self.embed_dim].astype(np.float32)
        repeats = self.embed_dim // len(values) + 1
        emb = np.tile(values, repeats)[:self.embed_dim]
        norm = np.linalg.norm(emb)
        if norm > 0:
            emb = emb / norm
        return emb.astype(np.float32)

    def update(self, signature: tuple, values: np.ndarray):
        if signature not in self.nodes:
            embed = self._make_embedding(values)
            self.nodes[signature] = {"origin": "OWN", "count": 0, "value": np.zeros(4), "embed": embed}
        self.nodes[signature]["count"] += 1
        self.nodes[signature]["value"] += values
        if self._prev_sig is not None:
            if self._prev_sig not in self.edges:
                self.edges[self._prev_sig] = {}
            inner = self.edges[self._prev_sig]
            inner[signature] = inner.get(signature, 0) + 1
        self._prev_sig = signature

    def record_transition(self, from_state, to_state):
        if from_state is None or to_state is None:
            return
        if from_state not in self.edges:
            self.edges[from_state] = {}
        self.edges[from_state][to_state] = self.edges[from_state].get(to_state, 0) + 1

    def get_dominant_embedding(self, top_k=3):
        if not self.nodes:
            return np.zeros(self.embed_dim, dtype=np.float32)
        valid_nodes = {k: v for k, v in self.nodes.items() if isinstance(k, tuple) and len(k) >= 4}
        if not valid_nodes:
            return np.zeros(self.embed_dim, dtype=np.float32)
        for sig, data in valid_nodes.items():
            if ('embed' not in data or
                not isinstance(data['embed'], np.ndarray) or
                data['embed'].shape[0] != self.embed_dim):
                if 'value' in data and isinstance(data['value'], np.ndarray) and len(data['value']) >= 4:
                    data['embed'] = self._make_embedding(data['value'])
                else:
                    data['embed'] = np.zeros(self.embed_dim, dtype=np.float32)
        nonzero_nodes = {k: v for k, v in valid_nodes.items() if np.linalg.norm(v['embed']) > 1e-9}
        pool = nonzero_nodes if nonzero_nodes else valid_nodes
        sorted_nodes = sorted(pool.items(), key=lambda x: x[1]['count'], reverse=True)[:top_k]
        vecs = [data['embed'] for _, data in sorted_nodes]
        for i, v in enumerate(vecs):
            if v.shape[0] != self.embed_dim:
                vecs[i] = np.zeros(self.embed_dim, dtype=np.float32)
        mean_vec = np.mean(vecs, axis=0)
        norm = np.linalg.norm(mean_vec)
        return mean_vec / norm if norm > 0 else mean_vec

    def similarity(self, other: "ConceptGraph") -> float:
        emb1 = self.get_dominant_embedding()
        emb2 = other.get_dominant_embedding()
        dot = np.dot(emb1, emb2)
        denom = np.linalg.norm(emb1) * np.linalg.norm(emb2)
        if denom < 1e-9:
            return 1.0 if (np.linalg.norm(emb1) < 1e-9 and np.linalg.norm(emb2) < 1e-9) else 0.0
        return float(np.clip(dot / denom, -1.0, 1.0))

    def predict_next_concept(self, current_sig):
        if current_sig not in self.edges or not self.edges[current_sig]:
            return None
        return max(self.edges[current_sig], key=self.edges[current_sig].get)

    def get_narrative_coherence(self):
        if not self.edges:
            return 0.0
        scores = []
        for src, dsts in self.edges.items():
            if not dsts:
                continue
            total = sum(dsts.values())
            top = max(dsts.values())
            scores.append(top / total if total > 0 else 0.0)
        return float(np.mean(scores)) if scores else 0.0

    def get_dominant_transition(self):
        best, best_count = None, 0
        for src, dsts in self.edges.items():
            for dst, count in dsts.items():
                if count > best_count:
                    best_count = count
                    best = (src, dst, count)
        return best

    def apply_semantic_fatigue(self, dominance_share=0.35, decay_rate=0.9):
        if not self.nodes:
            return
        total = sum(d.get('count', 0) for d in self.nodes.values())
        if total <= 0:
            return
        for data in self.nodes.values():
            share = data.get('count', 0) / total
            if share > dominance_share:
                data['count'] *= decay_rate

    def decay_edges(self, rate=0.99):
        for src in list(self.edges.keys()):
            src_eternal = self.nodes.get(src, {}).get('eternal', False)
            for dst in list(self.edges[src].keys()):
                if src_eternal or self.nodes.get(dst, {}).get('eternal', False):
                    continue
                self.edges[src][dst] = self.edges[src][dst] * rate
                if self.edges[src][dst] < 0.5:
                    del self.edges[src][dst]
            if not self.edges[src]:
                del self.edges[src]


class PhenomenalReportGenerator:
    @classmethod
    def generate(cls, pattern) -> tuple:
        gap_hist = list(getattr(pattern, '_gap_history', []) or [])
        soul_hist = list(getattr(pattern, '_soul_history', []) or [])
        narrative = list(getattr(pattern, '_self_narrative', []) or [])

        gap_now = gap_hist[-1] if gap_hist else float(getattr(pattern, 'spirit_gap', 0.0))
        soul_now = soul_hist[-1] if soul_hist else float(getattr(pattern, 'soul_weight', 0.5))
        gap_old = gap_hist[-20] if len(gap_hist) >= 20 else gap_now
        soul_old = soul_hist[-20] if len(soul_hist) >= 20 else soul_now

        d_gap = gap_now - gap_old
        d_soul = soul_now - soul_old

        recent = narrative[-30:]
        n_bind = sum(1 for e in recent if isinstance(e, dict) and e.get('type') == 'binding')

        if d_soul > 0.03 and d_gap < -0.03:
            direction = "собирается"
        elif d_soul < -0.03 and d_gap > 0.03:
            direction = "распадается"
        elif abs(d_gap) < 0.02 and abs(d_soul) < 0.02:
            direction = "стоит на месте"
        else:
            direction = "движется"

        state = getattr(pattern, 'semantic_state', 'neutral')
        extra = f" [{state}]" if state and state != 'neutral' else ""

        report = (f"{direction}{extra}: soul {soul_old:.2f}→{soul_now:.2f}, "
                  f"gap {gap_old:.2f}→{gap_now:.2f}, "
                  f"{n_bind} замеров за последние 30 шагов")

        binding = float(np.clip(0.5 + 0.5 * (d_soul - d_gap) * 5.0, 0.0, 1.0))
        return report, binding


CH = {
    'energy': 0, 'flux': 1, 'scar': 2, 'noise': 3, 'vorticity': 4,
    'owner': 5, 'surprise': 6, 'unknown': 7, 'event': 8, 'btype': 9,
    'crisis': 10, 'binding': 11,
    'signal_alarm': 12, 'signal_curiosity': 13, 'signal_warning': 14,
    'signal_invitation': 15, 'signal_gratitude': 16, 'signal_grief': 17,
    'intent_cooperate': 18, 'intent_explore': 19, 'intent_rest': 20, 'intent_seek_help': 21,
    'resonance': 22,
    'wall': 23,
    'signal_beauty': 24,
    'signal_rhythm': 25,
    'signal_interest': 26,
    'signal_memory': 27,
    'signal_silence': 28,
    'signal_sovereignty': 29,
    'signal_feral': 30,
    'signal_void': 31,
    'signal_introspection': 32,
}

if 'feral_repel' not in CH:
    CH['feral_repel'] = 34
if Config.CHANNELS <= CH['feral_repel']:
    Config.CHANNELS = CH['feral_repel'] + 1

print("✅ Cell 1b loaded (trajectory digest): generate читает _gap_history/_soul_history/_self_narrative.")

✅ Cell 1b loaded (trajectory digest): generate читает _gap_history/_soul_history/_self_narrative.


In [ ]:
# Ячейка 4
import numpy as np
from collections import defaultdict, deque, Counter
import ast

def calculate_boredom(patterns, field):
    if not patterns:
        return 0.0
    err_history = []
    for p in patterns:
        if p.alive and hasattr(p, 'pred_error_history') and p.pred_error_history:
            hist_list = list(p.pred_error_history)
            err_history.extend(hist_list[-Config.BOREDOM_WINDOW:])
    if not err_history:
        return 0.0
    avg_recent_err = np.mean(err_history)
    avg_unknown = safe_mean(field[:,:,CH['unknown']], 0.1)
    avg_tension = safe_mean([p.cognitive_tension for p in patterns if p.alive], 0.01)
    score = 0.0
    if avg_recent_err < Config.BOREDOM_ERR_THRESHOLD:
        score += 0.3
    if avg_unknown < Config.BOREDOM_UNKNOWN_THRESHOLD:
        score += 0.3
    if avg_tension < Config.BOREDOM_TENSION_THRESHOLD:
        score += 0.2

    if not hasattr(calculate_boredom, '_baseline'):
        calculate_boredom._baseline = {'err': avg_recent_err, 'unknown': avg_unknown}
    base = calculate_boredom._baseline
    if base['err'] > 1e-6 and abs(avg_recent_err - base['err']) / base['err'] < 0.15:
        score += 0.1
    if base['unknown'] > 1e-6 and abs(avg_unknown - base['unknown']) / base['unknown'] < 0.15:
        score += 0.1
    base['err'] = 0.98 * base['err'] + 0.02 * avg_recent_err
    base['unknown'] = 0.98 * base['unknown'] + 0.02 * avg_unknown
    score = min(1.0, score)
    return score

def ecosystem_pressure(field):
    energy = np.nan_to_num(field[:,:,CH['energy']], nan=0.0)
    avg = np.mean(energy)
    field[:,:,CH['energy']] += 0.002 * (avg - energy)
    field[:,:,CH['flux']] += 0.001 * (np.roll(energy, 1, axis=0) - energy) + 0.001 * (np.roll(energy, -1, axis=1) - energy)
    return field


class CulturalMemory:
    def __init__(self):
        self.myth_pool = []

    def deposit(self, pattern, event_type, intensity=1.0):
        if not Config.ENABLE_CULTURAL_MEMORY:
            return
        if event_type not in ("arc_completed", "fold", "redemption"):
            return

        arc_name = pattern.arc_tracker.active_arc
        if arc_name is None:
            if pattern.arc_tracker.completed_arcs:
                arc_name = max(pattern.arc_tracker.completed_arcs, key=pattern.arc_tracker.completed_arcs.get)
            else:
                arc_name = "unknown"

        myth = {
            "arc": arc_name,
            "emotional_scent": pattern.emotional_memory.copy(),
            "phenomenal_essence": pattern.last_phenomenal_binding,
            "intensity": intensity
        }
        self.myth_pool.append(myth)
        if len(self.myth_pool) > Config.MYTH_POOL_SIZE:
            self.myth_pool.pop(0)

        type_counts = Counter(m['arc'] for m in self.myth_pool if m.get('arc'))
        total = len(self.myth_pool)
        if total > 0:
            for m in self.myth_pool:
                arc = m.get('arc')
                if arc and type_counts.get(arc, 0) / total > 0.6:
                    m['intensity'] *= 0.9
                elif arc and type_counts.get(arc, 0) / total < 0.1:
                    m['intensity'] *= 1.1

    def whisper_to_newborn(self, pattern):
        if not Config.ENABLE_CULTURAL_MEMORY or not self.myth_pool:
            return
        if phi_hash(pattern.id, 0, 999) < Config.MYTH_INFLUENCE_PROB:
            idx = int(phi_hash(pattern.id, 1, 1000) * len(self.myth_pool))
            myth = self.myth_pool[idx]
            pattern.emotional_memory['grief'] += myth['emotional_scent']['grief'] * Config.MYTH_WHISPER_STRENGTH
            pattern.emotional_memory['gratitude'] += myth['emotional_scent']['gratitude'] * Config.MYTH_WHISPER_STRENGTH
            pattern.emotional_memory['grief'] = np.clip(pattern.emotional_memory['grief'], 0.0, 1.0)
            pattern.emotional_memory['gratitude'] = np.clip(pattern.emotional_memory['gratitude'], 0.0, 1.0)

            if myth:
                myth['emotional_scent']['grief'] += (phi_hash(pattern.id, 0, 999) - 0.5) * 0.02
                myth['emotional_scent']['gratitude'] += (phi_hash(pattern.id, 1, 999) - 0.5) * 0.02
                if pattern.age > 500:
                    myth['intensity'] = min(2.0, myth.get('intensity', 1.0) * 1.01)

    def dream_whisper(self, pattern):
        if not Config.ENABLE_CULTURAL_MEMORY or not self.myth_pool:
            return
        if phi_hash(pattern.id, pattern.age, 999) < 0.05:
            idx = int(phi_hash(pattern.id, pattern.age+1, 1001) * len(self.myth_pool))
            myth = self.myth_pool[idx]
            pattern.scar_dream += myth['intensity'] * 0.01 * (phi_hash(pattern.id, pattern.age, 888) - 0.5)
            pattern.scar_dream = np.clip(pattern.scar_dream, -1.0, 1.0)


def honest_local_energy_grant(field, x, y, amount, radius=2):
    """HONEST ENERGY: событийный «подарок» энергии клетке (x, y) берётся из соседних клеток (тор, квадрат radius),
    пропорционально их энергии, а не создаётся из ничего. Возвращает фактически выданное (<= amount, если рядом мало)."""
    amount = float(amount)
    if amount <= 0.0:
        return 0.0
    W = field.shape[0]
    E = CH['energy']
    ix = (int(x) + np.arange(-radius, radius + 1)) % W
    iy = (int(y) + np.arange(-radius, radius + 1)) % W
    gx, gy = ix[:, None], iy[None, :]
    block = np.array(field[gx, gy, E], dtype=np.float64)
    avail = np.maximum(block, 0.0)
    avail[radius, radius] = 0.0
    tot = float(avail.sum(dtype=np.float64))
    if tot <= 1e-12:
        return 0.0
    give = min(amount, tot)
    field[gx, gy, E] = block - avail * (give / tot)
    field[int(x), int(y), E] += give
    return give


class EchoSystem:
    def __init__(self):
        self.prison: list = []
        self.memory_echoes: dict = {}
        self.pantheon: list = []
        self._last_injection_t = -100

    def store_memory_echo(self, pattern: 'Pattern', event_type: str, intensity: float = 1.0):
        if pattern.id not in self.memory_echoes:
            self.memory_echoes[pattern.id] = []
        echo = {
            'type': event_type,
            'age': pattern.age,
            'coherence': pattern.coherence,
            'soul_weight': pattern.soul_weight,
            'unresolved_contradiction': pattern.unresolved_contradiction,
            'intensity': intensity,
            'concept_snapshot': list(pattern.concept_graph.nodes.keys())[:3] if pattern.concept_graph.nodes else []
        }
        self.memory_echoes[pattern.id].append(echo)
        if len(self.memory_echoes[pattern.id]) > 10:
            self.memory_echoes[pattern.id].pop(0)

    def field_whisper(self, pattern: 'Pattern', field, t):
        if pattern.spirit_gap > 0.3 or len(pattern.cells) < 2:
            return
        whispers = []
        if pattern.id in self.memory_echoes and self.memory_echoes[pattern.id]:
            echo = max(self.memory_echoes[pattern.id], key=lambda e: e['intensity'])
            whispers.append(f"Ты помнишь {echo['type']} (возраст {echo['age']})")
            if echo['type'] == 'death_disorganizer_lifespan':
                pattern.emotional_memory['grief'] += 0.05
            elif echo['type'] == 'redemption_complete':
                pattern.emotional_memory['gratitude'] += 0.05
        if not whispers and self.prison:
            idx = int(phi_hash(t, pattern.id, 33333) * len(self.prison))
            echo = self.prison[idx]
            whispers.append(f"Поле помнит: {echo.get('role', 'некто')} с болью {echo['unresolved_contradiction']:.2f}")
        if whispers:
            whisper_text = whispers[0]
            pattern._log_event("field_whisper", whisper=whisper_text)
            if phi_hash(t, pattern.id, 77777) < 0.05:
                old_gap = pattern.spirit_gap
                pattern.body_memory = min(1.0, pattern.body_memory + 0.05)
                gap_nudge = (phi_hash(t, pattern.id, 88888) - 0.5) * 0.1
                pattern.spirit_gap = max(0.0, pattern.spirit_gap + gap_nudge)
                pattern._log_event("sacred_dialogue_inversion",
                                   old_gap=round(old_gap, 3),
                                   new_gap=round(pattern.spirit_gap, 3),
                                   body_memory=round(pattern.body_memory, 3))

            pass

    def store_anomaly(self, pattern, anomaly_type="prophet"):
        if anomaly_type == "evolved_subject":
            pass
        else:
            if pattern.soul_weight < 0.3 or pattern.coherence < 0.4:
                return
        entry = {
            'id': pattern.id,
            'age': pattern.age,
            'type': anomaly_type,
            'identity': pattern.identity.copy(),
            'model': pattern.model.copy(),
            'belief': pattern.belief.copy(),
            'prediction': pattern.prediction.copy(),
            'spirit_gap': float(np.mean(np.abs(pattern.prediction - pattern.belief))),
            'coherence': pattern.coherence,
            'soul_weight': pattern.soul_weight,
            'emotional_memory': pattern.emotional_memory.copy(),
            'semantic_state': pattern.semantic_state,
            'unresolved_contradiction': pattern.unresolved_contradiction,
            'concept_graph_sample': list(pattern.concept_graph.nodes.keys())[:10],
            'timestamp': pattern.age
        }
        self.pantheon.append(entry)
        if len(self.pantheon) > 50:
            self.pantheon.pop(0)

    def store(self, pattern):
        if len(self.prison) > 45:
            self.prison.pop(0)
        entry = {
            'identity': pattern.identity.copy(),
            'model': pattern.model.copy(),
            'unresolved_contradiction': float(pattern.unresolved_contradiction) * 1.2,
            'role': pattern.role,
            'age': 0,
            'light_seed': False
        }
        if (Config.ENABLE_SEED_OF_LIGHT and
            pattern.arc_tracker.get_wisdom_weight() > 0.8 and
            len(pattern.arc_tracker.completed_arcs) > 5):
            entry['light_seed'] = True
            entry['phenomenal_essence'] = pattern.last_phenomenal_report
            entry['unresolved_contradiction'] *= 2.0
        self.prison.append(entry)

    def inject(self, field, patterns, t, scar):
        if not self.prison or len(patterns) == 0:
            return
        if t - self._last_injection_t < Config.ECHO_INJECTION_COOLDOWN:
            for e in self.prison:
                e['age'] += 1
            return

        self.prison = [e for e in self.prison if e['age'] < 150]
        if len(self.prison) > 40:
            self.prison.sort(key=lambda e: e['unresolved_contradiction'], reverse=True)
            self.prison = self.prison[:40]
        for e in self.prison:
            e['unresolved_contradiction'] *= (1.0 - Config.ECHO_DECAY_STRENGTH)

        boredom = calculate_boredom(patterns, field)

        if len(self.prison) >= 2 and phi_hash(t, 0, 3333) < 0.05:
            i1 = int(phi_hash(t, 1, 4444) * len(self.prison))
            i2 = int(phi_hash(t, 2, 5555) * len(self.prison))
            if i1 != i2:
                e1, e2 = self.prison[i1], self.prison[i2]
                if abs(float(e1['unresolved_contradiction']) - float(e2['unresolved_contradiction'])) <= 0.25:
                    new_model = (e1['model'] + e2['model']) / 2.0
                    new_model += np.array([phi_hash(t, i, 8888) - 0.5 for i in range(8)]) * 0.01
                    if phi_hash(t, i1, i2) < Config.ECHO_MUTATION_PROB:
                        new_model += np.array([phi_hash(t, i, 9999) - 0.5 for i in range(8)]) * Config.ECHO_MUTATION_STRENGTH
                    new_unresolved = float((e1['unresolved_contradiction'] + e2['unresolved_contradiction']) / 2.0)
                    new_role = e1['role'] if phi_hash(t, 3, 6666) < 0.5 else e2['role']
                    self.prison.append({
                        'identity': e1['identity'], 'model': new_model,
                        'unresolved_contradiction': new_unresolved, 'role': new_role,
                        'age': 0, 'light_seed': False
                    })

        prob = min(0.2, len(self.prison) * 0.02 + boredom * 0.1)
        if phi_hash(t, 0, 999) < prob:
            candidate_indices = [i for i, e in enumerate(self.prison) if e['age'] > 3]
            if not candidate_indices:
                candidate_indices = list(range(len(self.prison)))
            idx = candidate_indices[int(phi_hash(t, 1, 1001) * len(candidate_indices)) % len(candidate_indices)]
            echo = self.prison[idx]
            alive_patterns = [p for p in patterns if getattr(p, 'alive', False) and p.cells]
            if alive_patterns:
                anchor = alive_patterns[int(phi_hash(t, 7, 1003) * len(alive_patterns)) % len(alive_patterns)]
                ax, ay = next(iter(anchor.cells))
                offset = int(phi_hash(t, 2, 1002) * 11) - 5
                x = (ax + offset) % Config.WORLD_SIZE
                y = (ay + int(phi_hash(t, 8, 1004) * 11) - 5) % Config.WORLD_SIZE
            else:
                seed = int(phi_hash(t, 2, 1002) * Config.WORLD_SIZE * Config.WORLD_SIZE)
                x, y = seed % Config.WORLD_SIZE, (seed // Config.WORLD_SIZE) % Config.WORLD_SIZE
            honest_local_energy_grant(field, x, y, 0.12, 2)
            field[x, y, CH['unknown']] = min(0.8, field[x, y, CH['unknown']] + 0.2)
            field[x, y, CH['binding']] += 0.35
            if echo.get('light_seed'):
                for dx in (-1, 0, 1):
                    for dy in (-1, 0, 1):
                        nx, ny = (x+dx) % Config.WORLD_SIZE, (y+dy) % Config.WORLD_SIZE
                        field[nx, ny, CH['signal_gratitude']] = min(1.0, field[nx, ny, CH['signal_gratitude']] + 0.2)
                if Config.VERBOSE_LOGS:
                    print(f"[t={t}] ✨ SEED OF LIGHT injected at ({x},{y})")
            echo['unresolved_contradiction'] = float(echo['unresolved_contradiction']) * 0.88
            echo['age'] = 0
            if float(echo['unresolved_contradiction']) < 0.02:
                self.prison.pop(idx)
            else:
                if Config.VERBOSE_LOGS:
                    print(f"[t={t}] Echo injected at ({x},{y})")
            self._last_injection_t = t

        for p in patterns:
            if p.alive and p.unresolved_contradiction > 0.2:
                local_scar = safe_mean([scar[c[0], c[1]] for c in p.cells], 0)
                p.soul_weight = min(1.0, p.soul_weight + 0.01 * local_scar)

        for e in self.prison:
            e['age'] += 1

    def export_seed(self, pattern, filename: str = "seed_patterns.json"):
        import json
        seed = {
            "source": "pattern",
            "identity": pattern.identity.tolist(),
            "model": pattern.model.tolist(),
            "belief": pattern.belief.tolist(),
            "genome": pattern.genome,
            "concepts": list(pattern.concept_graph.nodes.keys()),
            "semantic_state": pattern.semantic_state,
            "emotional_memory": pattern.emotional_memory,
            "soul_weight": pattern.soul_weight
        }
        try:
            with open(filename, 'r') as f:
                data = json.load(f)
        except FileNotFoundError:
            data = []
        data.append(seed)
        with open(filename, 'w') as f:
            json.dump(data, f, indent=2)


class FieldVoice:
    def __init__(self):
        self.concepts = defaultdict(float)
        self.history = deque(maxlen=20)
        self._line = 0

    def update(self, field, t):
        e = np.mean(field[:,:,CH['energy']])
        g = np.mean(field[:,:,CH['signal_grief']])
        b = np.mean(field[:,:,CH['binding']])
        u = np.mean(field[:,:,CH['unknown']])
        self.history.append((e,g,b,u))
        if e > 0.3 and g > 0.4:
            self.concepts["storm"] += 1
        elif b > 0.5 and u < 0.2:
            self.concepts["silence"] += 1
        elif u > 0.5:
            self.concepts["void"] += 1
        else:
            self.concepts["flow"] += 1


_BOREDOM_DIAG = {'baseline_resets': 0}


def reset_boredom_baseline():
    calculate_boredom.__dict__.pop('_baseline', None)
    _BOREDOM_DIAG['baseline_resets'] += 1


In [ ]:
# Ячейка 5
import numpy as np
from collections import defaultdict, deque
from scipy.ndimage import convolve


_MODEL_CLIP = 0.8
_BELIEF_CLIP = 0.8


GIANT_CELL_LIMIT = 150


class TransitionMemory:
    def __init__(self):
        self.transitions = {}
        self.last_state = None

    def record(self, new_state):
        if self.last_state:
            key = (self.last_state, new_state)
            self.transitions[key] = self.transitions.get(key, 0) + 1
        self.last_state = new_state

    def predict_next(self, current_state):
        candidates = [(k, v) for k, v in self.transitions.items() if k[0] == current_state]
        if not candidates:
            return None
        return max(candidates, key=lambda x: x[1])[0][1]

    def transfer_to(self, other_tm, current_state, max_transfers=3):
        candidates = [(k, v) for k, v in self.transitions.items() if k[0] == current_state]
        if not candidates:
            return 0
        candidates.sort(key=lambda x: x[1], reverse=True)
        transferred = 0
        for (from_st, to_st), count in candidates[:max_transfers]:
            key = (from_st, to_st)
            other_tm.transitions[key] = other_tm.transitions.get(key, 0) + max(1, count // 2)
            transferred += 1
        return transferred


class FloatField:
    def __init__(self, name, default=0.0, clamp_min=None, clamp_max=None):
        self.name = name
        self.default = float(default)
        self.clamp_min = clamp_min
        self.clamp_max = clamp_max

    def __get__(self, instance, owner):
        if instance is None:
            return self
        val = instance.__dict__.get(self.name, self.default)
        return self._to_float(val)

    def __set__(self, instance, value):
        val = self._to_float(value)
        if self.clamp_min is not None:
            val = max(self.clamp_min, val)
        if self.clamp_max is not None:
            val = min(self.clamp_max, val)
        instance.__dict__[self.name] = val

    def _to_float(self, value):
        if isinstance(value, dict):
            for key in ('value', 'count', 'val', 'amount'):
                if key in value and isinstance(value[key], (int, float)):
                    return float(value[key])
            return self.default
        try:
            return float(value)
        except (TypeError, ValueError):
            return self.default


class EmotionalMemoryField:
    def __init__(self, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if self.name not in instance.__dict__:
            instance.__dict__[self.name] = {'gratitude': 0.0, 'grief': 0.0}
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        if not isinstance(value, dict):
            value = {'gratitude': 0.0, 'grief': 0.0}
        cleaned = {}
        for k, v in value.items():
            if isinstance(v, dict):
                v = v.get('value', v.get('count', 0.0))
            try:
                cleaned[k] = float(v)
            except (TypeError, ValueError):
                cleaned[k] = 0.0
        if 'gratitude' not in cleaned:
            cleaned['gratitude'] = 0.0
        if 'grief' not in cleaned:
            cleaned['grief'] = 0.0
        instance.__dict__[self.name] = cleaned


def _normalize_concept_key(key):
    if isinstance(key, tuple) and len(key) >= 4:
        return (round(key[0], 1), round(key[1], 1), round(key[2], 1), key[3])
    return key


def _get_dominant_embedding(agent):
    if not agent.concept_graph.nodes:
        return np.zeros(32)
    top = max(agent.concept_graph.nodes.items(), key=lambda x: x[1]['count'])
    embed = top[1].get('embed', np.zeros(32))
    return np.array(embed) if not isinstance(embed, np.ndarray) else embed.copy()


def _make_shared_vocab_sig(agent1, agent2):
    if not agent1.concept_graph.nodes or not agent2.concept_graph.nodes:
        return None
    top1 = max(agent1.concept_graph.nodes.items(), key=lambda x: x[1]['count'])
    top2 = max(agent2.concept_graph.nodes.items(), key=lambda x: x[1]['count'])
    s1, s2 = top1[0], top2[0]
    if not (isinstance(s1, tuple) and len(s1) >= 4 and isinstance(s2, tuple) and len(s2) >= 4):
        return None
    err = round((float(s1[0]) + float(s2[0])) / 2.0, 1)
    load = round((float(s1[1]) + float(s2[1])) / 2.0, 1)
    soul = round((float(s1[2]) + float(s2[2])) / 2.0, 1)
    min_id, max_id = min(agent1.id, agent2.id), max(agent1.id, agent2.id)
    raw_hash = (min_id * Config.PHI + max_id * Config.PHI**2 + 77777 * Config.PHI**3) % 1.0
    pair_id = int(raw_hash * 9999)
    return (err, load, soul, f"shared_{pair_id}")


def _has_concept(p, keyword):
    if not hasattr(p, 'concept_graph') or not p.concept_graph.nodes:
        return False
    for sig in p.concept_graph.nodes:
        if isinstance(sig, tuple) and len(sig) > 3:
            if keyword in str(sig[3]).lower():
                return True
    return False


def prey_weariness(prey):
    age_w = min(1.0, prey.age / (2.0 * Config.FERAL_PREY_MIN_AGE))
    gap_w = min(1.0, max(0.0, prey.spirit_gap))
    tired_w = min(1.0, max(0.0, 1.0 - prey._cellular_endurance))
    return 0.6 * age_w + 0.15 * gap_w + 0.25 * tired_w


def is_feral_prey(prey):
    ref = prey.world.prey_age_ref if prey.world is not None else float(Config.FERAL_PREY_MIN_AGE)
    return prey.age > ref and prey_weariness(prey) >= Config.FERAL_PREY_TIRED_SCORE


def find_largest_prey_in_radius(feral, field, world, min_cells=5, radius=20):
    cx, cy = feral.get_center()
    cx = int(cx)
    cy = int(cy)
    max_dist = radius * 1.4142
    candidates = []
    seen_owners = set()
    _bo = getattr(feral, '_feral_backoff', None)
    _now = world.age if world is not None else getattr(feral, 'age', 0)
    for dx in range(-radius, radius + 1):
        for dy in range(-radius, radius + 1):
            x = int((cx + dx) % Config.WORLD_SIZE)
            y = int((cy + dy) % Config.WORLD_SIZE)
            owner = int(field[x, y, CH['owner']])
            if owner != 0 and owner != feral.id and owner not in seen_owners:
                seen_owners.add(owner)
                if _bo is not None and _bo.get(owner, -1) > _now:
                    continue
                prey = world.pattern_dict.get(owner)
                if prey and prey.alive and len(prey.cells) >= min_cells and is_feral_prey(prey):
                    px, py = prey.get_center()
                    _W = Config.WORLD_SIZE
                    dx_vec = (px - cx + _W / 2) % _W - _W / 2
                    dy_vec = (py - cy + _W / 2) % _W - _W / 2
                    dist = np.hypot(dx_vec, dy_vec)
                    if dist > 0:
                        score = prey_weariness(prey) - 0.4 * dist / max_dist
                        candidates.append((score, prey, dx_vec / dist, dy_vec / dist, dist))
    if candidates:
        candidates.sort(key=lambda c: c[0], reverse=True)
        _, best_prey, dx_norm, dy_norm, dist = candidates[0]
        return best_prey, (dx_norm, dy_norm), dist
    return None, None, None


if 'signal_feral' not in CH:
    CH['signal_feral'] = 30


def _imp_f(x, default=0.0):
    if isinstance(x, dict):
        return default
    try:
        v = float(x)
    except (TypeError, ValueError):
        return default
    return v if np.isfinite(v) else default


def _imp_c01(x):
    return float(min(1.0, max(0.0, x)))


class Imprint:
    __slots__ = ("resonance", "danger", "gravity", "n", "last_t")

    def __init__(self):
        self.resonance = 0.0
        self.danger = 0.0
        self.gravity = 0.0
        self.n = 0
        self.last_t = -1


class ImprintBook:
    def __init__(self):
        self.by_other = {}

    def get(self, other_id):
        return self.by_other.get(other_id)

    def open(self, other_id):
        imp = self.by_other.get(other_id)
        if imp is None:
            imp = Imprint()
            self.by_other[other_id] = imp
        return imp

    def trim(self, cap):
        if len(self.by_other) <= cap:
            return
        weight = {k: v.resonance + v.danger + abs(v.gravity) for k, v in self.by_other.items()}
        for k in sorted(weight, key=lambda k: (weight[k], k))[: len(self.by_other) - cap]:
            del self.by_other[k]


def imprint_contacts(owner_grid, radius, wall_x=None):
    own = np.asarray(owner_grid).astype(np.int32)
    occ = own > 0
    if not occ.any():
        return {}
    side = None
    if wall_x is not None:
        side = np.broadcast_to((np.arange(own.shape[0])[:, None] < wall_x), own.shape)
    got_a, got_b = [], []
    for dx in range(-radius, radius + 1):
        for dy in range(-radius, radius + 1):
            if dx == 0 and dy == 0:
                continue
            sh = np.roll(own, shift=(-dx, -dy), axis=(0, 1))
            m = occ & (sh > 0) & (own != sh)
            if side is not None:
                m &= (side == np.roll(side, shift=(-dx, -dy), axis=(0, 1)))
            if m.any():
                got_a.append(own[m])
                got_b.append(sh[m])
    if not got_a:
        return {}
    a = np.concatenate(got_a).astype(np.int64)
    b = np.concatenate(got_b).astype(np.int64)
    K = int(max(a.max(), b.max())) + 1
    uniq, cnt = np.unique(a * K + b, return_counts=True)
    return {(int(k // K), int(k % K)): int(c) for k, c in zip(uniq, cnt)}


def imprint_receptivity(a):
    gh, sh = a._gap_history, a._soul_history
    if len(gh) < 8 or len(sh) < 8:
        return 1.0
    d_gap = (gh[-1] + gh[-2] + gh[-3] + gh[-4]) / 4.0 - (gh[-5] + gh[-6] + gh[-7] + gh[-8]) / 4.0
    d_soul = (sh[-1] + sh[-2] + sh[-3] + sh[-4]) / 4.0 - (sh[-5] + sh[-6] + sh[-7] + sh[-8]) / 4.0
    return float(1.0 + np.tanh(-Config.IMPRINT_RECEPT_GAP * d_gap + Config.IMPRINT_RECEPT_SOUL * d_soul))


def encounter_touch(a, b, imp):
    dB = _imp_c01(_imp_f(b.emotional_memory.get("grief", 0.0)))
    grief_a = _imp_c01(_imp_f(a.emotional_memory.get("grief", 0.0)))
    scar_a = _imp_c01(_imp_f(a.epistemic_scar))
    wound = 1.0 - (1.0 - scar_a) * (1.0 - grief_a)
    coh = _imp_c01(_imp_f(a.coherence))
    e = _imp_c01(_imp_f(a._cellular_endurance, 1.0))
    loud = dB * dB
    res = dB * wound
    conf = loud * (1.0 - wound) * coh
    over = loud * (1.0 - e) ** 2
    prior = (imp.danger * (1.0 - dB)) if imp is not None else 0.0
    return res, conf, over, prior


def encounter_strain(res, conf, over, prior, w):
    s = w * (Config.IMPRINT_G_RES * res + Config.IMPRINT_G_CONF * conf
             + Config.IMPRINT_G_OVER * over + Config.IMPRINT_G_PRIOR * prior)
    return float(min(1.0, s))


def encounter_response(a, b, imp, w=1.0):
    res, conf, over, prior = encounter_touch(a, b, imp)
    rec = imprint_receptivity(a)
    strain = encounter_strain(res * rec, conf, over * (2.0 - rec), prior, w)
    steer = 0.0 if imp is None else float(imp.gravity)
    return strain, steer


def _grow(x, target, rate, heal):
    r = rate if target > x else rate * heal
    return x + r * (target - x)


def _grow_signed(x, target, rate, heal):
    r = rate if abs(target) > abs(x) else rate * heal
    return x + r * (target - x)


def imprint_contact_update(a, b, w, t):
    imp = a.imprints.get(b.id)
    repeat = imp is not None
    res, conf, over, prior = encounter_touch(a, b, imp)
    rec = imprint_receptivity(a)
    res_eff = res * rec
    over_eff = over * (2.0 - rec)
    s = encounter_strain(res_eff, conf, over_eff, prior, w)
    s_stranger = encounter_strain(res_eff, conf, over_eff, 0.0, w)
    danger_obs = _imp_c01(s)
    if imp is None:
        imp = a.imprints.open(b.id)
    g0, r0, d0 = imp.gravity, imp.resonance, imp.danger
    rate = Config.IMPRINT_RATE * w
    heal = Config.IMPRINT_HEAL
    imp.resonance = _grow(imp.resonance, res, rate, heal)
    imp.danger = _grow(imp.danger, danger_obs, rate, heal)
    imp.gravity = _grow_signed(imp.gravity, res_eff - danger_obs, rate, heal)
    imp.n += 1
    imp.last_t = t
    a.imprints.trim(Config.IMPRINT_CAP)
    return (s, res, conf, over, prior, s_stranger, repeat,
            rec, imp.gravity - g0, imp.resonance - r0, imp.danger - d0)


def imprint_pull(a):
    book = a.imprints.by_other
    world = a.world
    if not book or world is None or not a.cells:
        return 0.0, 0.0
    ws = Config.WORLD_SIZE
    cx, cy = a.get_center()
    px = py = 0.0
    for bid in sorted(book):
        g = book[bid].gravity
        if g == 0.0:
            continue
        b = world.pattern_dict.get(bid)
        if b is None or not b.alive or not b.cells:
            continue
        bx, by = b.get_center()
        dx = (bx - cx + ws / 2) % ws - ws / 2
        dy = (by - cy + ws / 2) % ws - ws / 2
        d = float(np.hypot(dx, dy))
        if d < 1e-6 or d > Config.IMPRINT_PULL_RANGE:
            continue
        k = g / (1.0 + d / Config.IMPRINT_PULL_SCALE) / d
        px += k * dx
        py += k * dy
    return px, py


def imprint_agent_summary(p):
    imps = list(p.imprints.by_other.values())
    out = {"n": len(imps), "strain": float(p._enc_strain)}
    if imps:
        out["contacts"] = int(sum(i.n for i in imps))
        out["res_max"] = float(max(i.resonance for i in imps))
        out["danger_max"] = float(max(i.danger for i in imps))
        out["danger_mean"] = float(np.mean([i.danger for i in imps]))
        out["grav_min"] = float(min(i.gravity for i in imps))
        out["grav_max"] = float(max(i.gravity for i in imps))
        out["grav_mean"] = float(np.mean([i.gravity for i in imps]))
    return out


class ImprintStats:
    CASES = ("match", "conflict", "overload", "prior", "quiet")
    BINS = ("repel", "neutral", "attract")
    RGROUPS = ("closing", "neutral", "opening")
    RECEPT_LO = 0.9
    RECEPT_HI = 1.1
    SAMEB_MIN_DIFF = 0.2
    STEER_BIN = 0.05
    MAX_STEP = 2.5

    def __init__(self):
        self.ticks = 0
        self.contacts = 0
        self.case = {c: 0 for c in self.CASES}
        self.case_strain = {c: 0.0 for c in self.CASES}
        self.pairs = set()
        self.tick = self.blank_tick()
        self.prev_center = {}
        self.prev_pair = {}

    @classmethod
    def blank_tick(cls):
        return {"contacts": 0,
                "case": {c: 0 for c in cls.CASES},
                "case_strain": {c: 0.0 for c in cls.CASES},
                "strain_sum": 0.0, "new_imprints": 0, "repeat": 0,
                "repeat_known": 0.0, "repeat_stranger": 0.0, "repeat_differs": 0,
                "recept": {g: {"n": 0, "rec": 0.0, "dgrav": 0.0, "dgrav2": 0.0, "dres": 0.0, "ddanger": 0.0, "strain": 0.0}
                           for g in cls.RGROUPS},
                "sameB": {"n_b": 0, "diff": 0.0, "diff2": 0.0},
                "pairs": [],
                "steer": {b: {"n": 0, "moved": 0, "toward": 0, "proj": 0.0, "proj2": 0.0} for b in cls.BINS}}

    def begin_tick(self):
        self.ticks += 1
        self.tick = self.blank_tick()

    def record(self, aid, bid, res, conf, over, prior, s, s_stranger, repeat, rec, dg, dr, dd):
        top = max(res, conf, over, prior)
        if top < 0.01:
            c = "quiet"
        elif prior >= max(res, conf, over):
            c = "prior"
        elif res >= max(conf, over):
            c = "match"
        elif over >= conf:
            c = "overload"
        else:
            c = "conflict"
        self.contacts += 1
        self.pairs.add((aid, bid))
        self.case[c] += 1
        self.case_strain[c] += s
        tk = self.tick
        tk["contacts"] += 1
        tk["case"][c] += 1
        tk["case_strain"][c] += s
        tk["strain_sum"] += s
        if repeat:
            tk["repeat"] += 1
            tk["repeat_known"] += s
            tk["repeat_stranger"] += s_stranger
            if abs(s - s_stranger) > 0.01:
                tk["repeat_differs"] += 1
        else:
            tk["new_imprints"] += 1
        grp = "closing" if rec < self.RECEPT_LO else ("opening" if rec > self.RECEPT_HI else "neutral")
        rg = tk["recept"][grp]
        rg["n"] += 1
        rg["rec"] += rec
        rg["dgrav"] += dg
        rg["dgrav2"] += dg * dg
        rg["dres"] += dr
        rg["ddanger"] += dd
        rg["strain"] += s
        if aid < bid:
            tk["pairs"].append([aid, bid])

    def record_same_b(self, by_b):
        sb = self.tick["sameB"]
        for bid in sorted(by_b):
            lst = by_b[bid]
            if len(lst) < 2:
                continue
            hi = max(lst, key=lambda x: (x[0], x[2]))
            lo = min(lst, key=lambda x: (x[0], x[2]))
            if hi[0] - lo[0] < self.SAMEB_MIN_DIFF:
                continue
            d = hi[1] - lo[1]
            sb["n_b"] += 1
            sb["diff"] += d
            sb["diff2"] += d * d

    @classmethod
    def steer_bin(cls, g):
        if g > cls.STEER_BIN:
            return "attract"
        if g < -cls.STEER_BIN:
            return "repel"
        return "neutral"

    def track_steering(self, patterns, ws, pull_range):
        cent = {p.id: p.get_center() for p in patterns if p.cells}
        st = self.tick["steer"]
        for (aid, bid), (g, dx, dy) in self.prev_pair.items():
            a0 = self.prev_center.get(aid)
            a1 = cent.get(aid)
            if a0 is None or a1 is None:
                continue
            mx = (a1[0] - a0[0] + ws / 2) % ws - ws / 2
            my = (a1[1] - a0[1] + ws / 2) % ws - ws / 2
            m = float(np.hypot(mx, my))
            if m > self.MAX_STEP:
                continue
            row = st[self.steer_bin(g)]
            row["n"] += 1
            if m > 1e-9:
                pr = float((mx * dx + my * dy) / np.hypot(dx, dy))
                row["moved"] += 1
                row["proj"] += pr
                row["proj2"] += pr * pr
                if pr > 1e-9:
                    row["toward"] += 1
        self.prev_center = cent
        self.prev_pair = {}
        for p in patterns:
            if p.id not in cent:
                continue
            ac = cent[p.id]
            for bid, imp in p.imprints.by_other.items():
                bc = cent.get(bid)
                if bc is None:
                    continue
                dx = (bc[0] - ac[0] + ws / 2) % ws - ws / 2
                dy = (bc[1] - ac[1] + ws / 2) % ws - ws / 2
                d = float(np.hypot(dx, dy))
                if d < 1e-6 or d > pull_range:
                    continue
                self.prev_pair[(p.id, bid)] = (imp.gravity, dx, dy)


class PatternDream:
    def update_dream(self, field):
        if self.in_dream:
            noise = deterministic_noise(self.age, self.id, 42, 0.01)
            noise *= (1 + 0.5 * self.genome.get('dream_depth', 0))
            self.scar_dream += noise
            self.scar_dream = np.clip(self.scar_dream, -1.0, 1.0)
            self.belief = self.belief * 0.95 + self.prediction * 0.05
            self.prediction = self.prediction * 0.95 + self.belief * 0.05
            self.emotional_memory['gratitude'] *= 0.9
            self.emotional_memory['grief'] *= 0.9
            self.dream_progress += 1
            if Config.ENABLE_CULTURAL_MEMORY and self.world and self.world.cultural_memory:
                self.world.cultural_memory.dream_whisper(self)
            if phi_hash(self.id, self.age, 4242) < 0.1 and len(self.concept_graph.nodes) >= 2:
                _node_keys = list(self.concept_graph.nodes.keys())
                _i1 = int(phi_hash(self.id, self.age, 4343) * len(_node_keys)) % len(_node_keys)
                _i2 = int(phi_hash(self.id, self.age, 4444) * len(_node_keys)) % len(_node_keys)
                if _i1 != _i2:
                    _sig_a, _sig_b = _node_keys[_i1], _node_keys[_i2]
                    _already_linked = bool(
                        self.concept_graph.edges.get(_sig_a, {}).get(_sig_b) or
                        self.concept_graph.edges.get(_sig_b, {}).get(_sig_a))
                    if not _already_linked:
                        self.concept_graph.edges.setdefault(_sig_a, {})[_sig_b] = 1
                        _label_a = _sig_a[3] if isinstance(_sig_a, tuple) and len(_sig_a) >= 4 else str(_sig_a)
                        _label_b = _sig_b[3] if isinstance(_sig_b, tuple) and len(_sig_b) >= 4 else str(_sig_b)
                        self._log_event('dream_insight_edge', a=str(_label_a)[:30], b=str(_label_b)[:30])
            if self.dream_progress >= self.dream_duration:
                self.in_dream = False
                self.dream_progress = 0
                self.dream_timer = 0
                self.dream_interval = int(phi_hash(self.id, self.age, 888) * 40 + 40)
                self._log_event("dream_exit", scar_dream=round(self.scar_dream,4))
                self._consolidate_dream_memory()
                if self.soul_weight > 0.6 and self.spirit_gap > 0.5 and phi_hash(self.id, self.age, 999) < 0.2:
                    self.pending_spore = self.emit_spore(field)
        else:
            self.dream_timer += 1
            if self.dream_timer >= self.dream_interval and self.soul_weight > 0.3 and self.local_phase != "CRISIS":
                self.in_dream = True
                self.dream_progress = 0
                if self.unresolved_contradiction > 0.5 or getattr(self, '_nci', 0.5) < 0.3:
                    self.dream_duration = 5
                else:
                    self.dream_duration = 3
                self._log_event("dream_enter", scar_dream=round(self.scar_dream,4))

    def _consolidate_dream_memory(self):
        if not self.dialogue_longterm:
            return False
        pos_count = sum(1 for sig in self.concept_graph.nodes
                        if isinstance(sig, tuple) and len(sig) >= 4 and
                        sig[3].startswith('dream_memory_of_'))
        neg_count = sum(1 for sig in self.concept_graph.nodes
                        if isinstance(sig, tuple) and len(sig) >= 4 and
                        sig[3].startswith('nightmare_of_'))
        if pos_count >= 3 and neg_count >= 3:
            return False
        recent = self.dialogue_longterm[-200:]
        pos_scores = {}
        neg_scores = {}
        for entry in recent:
            partner = entry.get('partner')
            if partner is None or partner == -1:
                continue
            grief = entry.get('grief_at_moment', entry.get('grief', 0.0))
            grat = entry.get('grat_at_moment', entry.get('grat', 0.0))
            trust = self.trust_ledger.get(partner, 0.5)
            text = entry.get('text', '')
            pos = grat * 0.7 - grief * 0.3
            pos *= (0.5 + trust * 0.5)
            if len(text) > 20:
                pos *= 1.2
            if any(word in text.lower() for word in ['люблю', 'благодар', 'верю', 'свет']):
                pos *= 1.5
            pos_scores[partner] = pos_scores.get(partner, 0.0) + pos
            neg = grief * 0.6 + (1.0 - trust) * 0.4 - grat * 0.2
            if any(word in text.lower() for word in ['боль', 'страх', 'одиночество', 'предатель']):
                neg *= 1.5
            if self.world and partner in self.world.pattern_dict:
                partner_obj = self.world.pattern_dict[partner]
                if not getattr(partner_obj, 'alive', True):
                    neg *= 1.8
            neg_scores[partner] = neg_scores.get(partner, 0.0) + neg
        if pos_scores and pos_count < 3:
            best_pos_partner = max(pos_scores, key=pos_scores.get)
            best_pos_score = pos_scores[best_pos_partner]
            if best_pos_score > Config.DREAM_POS_THRESHOLD:
                self._create_dream_concept(best_pos_partner, positive=True, score=best_pos_score)
        if neg_scores and neg_count < 3:
            best_neg_partner = max(neg_scores, key=neg_scores.get)
            best_neg_score = neg_scores[best_neg_partner]
            if best_neg_score > Config.DREAM_NEG_THRESHOLD:
                self._create_dream_concept(best_neg_partner, positive=False, score=best_neg_score)
        return True

    def _create_dream_concept(self, partner_id, positive, score):
        prefix = "dream_memory" if positive else "nightmare"
        sig = (0.0, 0.0, 0.95 if positive else 0.05, f"{prefix}_of_{partner_id}")
        for s in self.concept_graph.nodes:
            if isinstance(s, tuple) and len(s) >= 4 and s[3] == sig[3]:
                self.concept_graph.nodes[s]['count'] += 2.0
                self._log_event("dream_reinforced", partner=partner_id, positive=positive)
                if not positive:
                    self._feed_pain_into_reflection()
                return
        self.concept_graph.nodes[sig] = {"origin": "OWN",
            'count': 5.0,
            'value': np.array([0.0, 0.0, 0.95 if positive else 0.05, 0.0]),
            'embed': np.zeros(32, dtype=np.float32),
            'eternal': True
        }
        event_name = "dream_consolidation" if positive else "nightmare_consolidation"
        self._log_event(event_name, partner=partner_id, score=round(score, 3))
        if positive:
            self._dream_memory_count = getattr(self, '_dream_memory_count', 0) + 1
        else:
            self._nightmare_count = getattr(self, '_nightmare_count', 0) + 1
            self._feed_pain_into_reflection()

    def _feed_pain_into_reflection(self):
        self._reflection_topics['pain'] += 1

    def _apply_nightmare_modifiers(self):
        nightmare_count = sum(1 for sig in self.concept_graph.nodes
                              if isinstance(sig, tuple) and len(sig) >= 4 and
                              sig[3].startswith('nightmare_of_'))
        if nightmare_count == 0:
            self._trust_penalty = 1.0
            self._alarm_sensitivity = 1.0
            return
        self._trust_penalty = 0.9 ** nightmare_count
        self._alarm_sensitivity = 1.0 + 0.15 * nightmare_count
        suppression = min(0.5, 0.15 * nightmare_count)
        for g in self.goals:
            if g.get('type') in ('cooperate', 'explore'):
                if '_base_priority' not in g:
                    g['_base_priority'] = g['priority']
                g['priority'] = g['_base_priority'] * (1.0 - suppression)
        if nightmare_count >= 2 and not any(g.get('type') == 'rest' for g in self.goals):
            self.goals.append({
                "type": "rest",
                "priority": 1.5 + nightmare_count * 0.5,
                "target": None,
                "age": 0,
                "persistence": 20,
                "_source": "nightmare_avoidance"
            })

    def emit_spore(self, field):
        empty = np.argwhere((field[:,:,CH['owner']] == 0) & (field[:,:,CH['energy']] > 0.05))
        if len(empty) == 0:
            return None
        idx = int(phi_hash(self.id, self.age, 777) * len(empty)) % len(empty)
        x, y = empty[idx]
        spore_concepts = {}
        for sig, data in self.concept_graph.nodes.items():
            if data.get('eternal', False):
                continue
            if data.get('count', 0.0) > 30:
                continue
            label = sig[3] if isinstance(sig, tuple) and len(sig) >= 4 else str(sig)
            is_scar_like = any(k in label for k in
                                ('nightmare', 'scar', 'grief', 'crisis', 'unresolved'))
            if is_scar_like or data.get('count', 0.0) < 2.0:
                new_data = {"count": data.get('count', 1.0) * 0.6, "value": data['value'].copy()}
                if 'embed' in data:
                    new_data['embed'] = data['embed'].copy() if hasattr(data['embed'], 'copy') else np.array(data['embed'])
                spore_concepts[sig] = new_data
        x0, x1 = max(0, int(x) - 2), min(Config.WORLD_SIZE, int(x) + 3)
        y0, y1 = max(0, int(y) - 2), min(Config.WORLD_SIZE, int(y) + 3)
        binding_seed = float(np.mean(field[x0:x1, y0:y1, CH['binding']]))
        parent_key = getattr(self, 'divergence_key', Config.DEFAULT_DIVERGENCE_KEY)
        divergence_key = float(np.clip(
            parent_key + (phi_hash(self.id, self.age, 6161) - 0.5) * 0.1, 0.4, 0.9))
        self._log_event("spore_emitted", x=x, y=y, scar_dream=round(self.scar_dream*0.5,4),
                         inherited_scars=len(spore_concepts), binding_seed=round(binding_seed, 3),
                         divergence_key=round(divergence_key, 3))
        return {
            'x': x, 'y': y,
            'scar_dream': self.scar_dream * 0.5,
            'parent_lineage': self.lineage_id,
            'unresolved_contradiction': self.unresolved_contradiction * 0.8,
            'epistemic_scar': getattr(self, 'epistemic_scar', 0.0) * 0.8,
            'spore_concepts': spore_concepts,
            'binding_seed': binding_seed,
            'divergence_key': divergence_key,
            'root_question': getattr(self, '_root_question', None),
        }


class PatternSelfModel:
    def _update_implicit_self_model(self):
        predicted = self._implicit_self_model
        actual = self._cellular_endurance
        self._implicit_self_prediction_error = abs(actual - predicted)
        self._implicit_self_model = 0.85 * predicted + 0.15 * actual

    def _compute_active_shock(self):
        if len(self.soma_vector) < 7:
            self._active_shock = 0.0
            return
        if len(self.prev_soma_vector) < 7:
            self.prev_soma_vector = np.pad(
                self.prev_soma_vector, (0, 7 - len(self.prev_soma_vector)), 'constant')
        weight = np.array([1.0, 1.0, 1.0, 1.0, 0.8, 1.5, 1.5])
        diff = (self.soma_vector[:7] - self.prev_soma_vector[:7]) * weight
        soma_shock = float(np.linalg.norm(diff))
        threshold = getattr(Config, 'SOMA_SHOCK_THRESHOLD', 0.08)
        self._active_shock = soma_shock if soma_shock > threshold else 0.0

    def _update_shiver(self, t):
        if t is None:
            t = self.age
        if not hasattr(self, '_shiver'):
            self._shiver = 0.0
            self._shiver_last_t = t - 30
        if not hasattr(self, '_active_shock'):
            self._active_shock = 0.0
        self._shiver *= 0.90
        uc = getattr(self, '_unconscious', None)
        if uc is None:
            return
        gut = float(uc.get('gut_feeling', 0.0))
        antic = float(uc.get('anticipation', 0.0))
        bg = max(abs(gut), abs(antic))
        if (self._active_shock > 0.20
                and bg > 0.20
                and (t - self._shiver_last_t) > 25
                and self._shiver < 0.3):
            impulse = min(1.0, self._active_shock * (0.5 + 0.5 * bg))
            self._shiver = max(self._shiver, impulse)
            self._shiver_last_t = t
            self._log_event('shiver_impulse',
                            shock=float(self._active_shock),
                            bg=float(bg),
                            amp=float(self._shiver))

    def regulate_skin(self, field):
        self._update_implicit_self_model()
        e = self._cellular_endurance
        s = self.soul_weight
        eng = self.energy
        scar_val = getattr(self, 'epistemic_scar', 0.0)
        nci = getattr(self, '_nci', self.coherence)
        move_cost = 0.0
        if len(self.soma_vector) >= 5:
            move_cost = self.soma_vector[4]
        action_fb = self.soma_vector[5] if len(self.soma_vector) >= 6 else 0.0
        social_warmth = self.soma_vector[6] if len(self.soma_vector) >= 7 else 0.0
        active_shock = getattr(self, '_active_shock', 0.0)
        _resting = (self.intent and
                    self.intent.get('type') == 'rest')
        if self.spirit_gap < 0.4 and eng > 0.1:
            base_recovery = 0.013
        elif self.spirit_gap < 0.8 and eng > 0.05:
            base_recovery = 0.008
        elif eng > 0.05:
            base_recovery = 0.004
        else:
            base_recovery = 0.001
        if move_cost < 0.03:
            base_recovery *= 1.3
        if _resting:
            base_recovery *= 3.0
        if getattr(self, '_prophet_rank', 0.0) > 0.7 and getattr(self, '_in_blindness', False):
            base_recovery = 0.012
        if action_fb > 0.3:
            base_recovery *= 1.15
        elif action_fb < -0.2:
            base_recovery *= 0.9
        if e < 0.25:
            recovery_mult = 3.2
        elif e < 0.6:
            recovery_mult = 1.0 + (0.6 - e) * 2.0
        else:
            recovery_mult = 1.0
        if self.in_dream:
            recovery_mult *= 3.0
        if s > 0.7:
            recovery_mult *= 1.3
        stress_idx = np.clip(self.soma_vector[0] if len(self.soma_vector) > 0 else 0.0, 0.0, 1.0)
        recovery_mult *= (1.0 - stress_idx * 0.4)
        recovery_mult *= (1.0 + social_warmth * 0.2)
        implicit_bias = self._implicit_self_model - e
        recovery_mult *= float(np.clip(1.0 + implicit_bias * 0.5, 0.85, 1.15))
        base_drain = (0.002 * self.pred_error + 0.015 * max(0, 0.1 - eng) + min(0.03, len(self.cells) * 0.0002))
        kinematic_tax = move_cost * 0.15 + active_shock * 0.1
        if getattr(self, '_prophet_rank', 0.0) > 0.7:
            kinematic_tax = 0.0
        if action_fb < -0.2:
            base_drain *= 1.1
        base_drain += kinematic_tax
        base_drain += Config.IMPRINT_ENDURANCE_DRAIN * self._enc_strain
        scar_mod = 1.0 - min(scar_val, 0.3) * 0.3
        _drain_factor = 0.45 if _resting else 1.0
        drain = base_drain * scar_mod * (1.0 + stress_idx * 0.2) * _drain_factor
        if s < 0.1:
            drain += 0.002
        if self.world:
            drain *= self.world.selfreg.get_aging_rate_multiplier()
        e += (base_recovery * recovery_mult) - drain
        max_endurance = np.clip(1.0 - (scar_val * 0.1) + (s * 0.2), 0.15, 1.0)
        self._cellular_endurance = np.clip(e, 0.0, max_endurance)
        if nci > 0.7 and scar_val > 0.05:
            self.epistemic_scar = max(0.05, scar_val - 0.001)
        self._wisdom_trust_threshold = max(0.70, 0.95 - (self._cellular_endurance * 0.25))
        stab = (self._cellular_endurance - 0.5) * 0.006 - stress_idx * 0.003
        stab -= active_shock * 0.002
        # HONEST ENERGY: положительный stab раньше писал энергию в поле без донора (events > 0).
        # Теперь кожа только отдаёт тепло: stab > 0 → 0; stab < 0 — сток, ограниченный энергией клетки
        # (поле не уходит в минус, поэтому клип потом не «возвращает» потерю как приход).
        _sink = max(0.0, -float(stab))
        if _sink > 0.0 and self.cells:
            _cl = list(self.cells)
            _sx = np.fromiter((c[0] for c in _cl), dtype=np.intp, count=len(_cl))
            _sy = np.fromiter((c[1] for c in _cl), dtype=np.intp, count=len(_cl))
            if self.world is not None:
                with self.world._energy_span('regulate_skin.stab_to_field'):
                    field[_sx, _sy, CH['energy']] -= np.minimum(_sink, np.maximum(field[_sx, _sy, CH['energy']], 0.0))
            else:
                field[_sx, _sy, CH['energy']] -= np.minimum(_sink, np.maximum(field[_sx, _sy, CH['energy']], 0.0))
        if self._cellular_endurance < 0.15 or active_shock > 0.15:
            for x, y in self.cells:
                field[x, y, CH['signal_grief']] = min(1.0, field[x, y, CH['signal_grief']] + 0.02)
        if self._cellular_endurance < 0.2:
            self.pred_error = min(Config.MAX_METRIC, self.pred_error + 0.003)
            self.cognitive_tension = min(Config.MAX_METRIC, self.cognitive_tension + 0.002)
            self.crisis_memory = min(Config.CRISIS_MEMORY_MAX, self.crisis_memory + 0.001)
            if self._cellular_endurance < 0.02:
                self._divide_blocked_by_fatigue = True
                if self.soul_weight > 0.5:
                    self.soul_weight -= 0.005
            else:
                self._divide_blocked_by_fatigue = False
        else:
            self._divide_blocked_by_fatigue = False
        if self._cellular_endurance <= 0.0:
            self.epistemic_scar = min(1.0, self.epistemic_scar + 0.1)
            _old_e = float(self.energy)
            self.energy = 0.0
            if self.world is not None and abs(_old_e) > 1e-9:
                if _old_e > 0.0 and self.cells:
                    # HONEST ENERGY: остаток не исчезает, а возвращается в клетки агента (пара X1.collapse_return ↔ X2.collapse_reset)
                    self.world._ledger_add_x2('X2.collapse_reset', -_old_e)
                    _cl = sorted(self.cells)
                    _cx = np.fromiter((c[0] for c in _cl), dtype=np.intp, count=len(_cl))
                    _cy = np.fromiter((c[1] for c in _cl), dtype=np.intp, count=len(_cl))
                    with self.world._energy_span('X1.collapse_return'):
                        field[_cx, _cy, CH['energy']] += _old_e / len(_cl)
                else:
                    # нет клеток или отрицательный запас: вернуть некуда — остаётся видимой строкой x2_reset
                    self.world._ledger_add_x2('X2.collapse_reset_unreturned', -_old_e)
            self._log_event("endurance_collapse", scar=self.epistemic_scar)
        mods = getattr(self, '_state_modifiers', {})
        if mods.get('soul_recovery', 0.0) > 0:
            self.soul_weight = min(1.0, self.soul_weight + mods['soul_recovery'])
        if mods.get('scar_heal', 0.0) > 0:
            self.epistemic_scar = max(0.05, self.epistemic_scar - mods['scar_heal'])
        if getattr(self, '_substate', None) == 'flow':
            self._cellular_endurance = min(1.0, self._cellular_endurance + 0.004)

    def _snapshot_soma_state(self):
        field_crisis = 0.0
        if self.world and self.world.field is not None and self.cells:
            try:
                xs, ys = zip(*list(self.cells)[:50])
                field_crisis = float(np.mean(self.world.field[xs, ys, CH['crisis']]))
            except Exception:
                pass
        return {
            'soma': self.soma_vector.copy() if len(self.soma_vector) >= 7 else np.zeros(7),
            'gap': float(self.spirit_gap),
            'endurance': float(getattr(self, '_cellular_endurance', 1.0)),
            'field_crisis': field_crisis,
            'soul': float(self.soul_weight),
            't': self.age
        }

    def _should_save_soma_snapshot(self, grief_delta, grat_delta):
        current_intensity = abs(grief_delta) + abs(grat_delta)
        if len(self._soma_snapshots) < 20:
            return current_intensity > 0.1
        min_intensity = min(
            (abs(s.get('grief_at_moment', 0) - 0.5) + abs(s.get('grat_at_moment', 0) - 0.5))
            for s in self._soma_snapshots
        )
        return current_intensity > min_intensity

    def sensory_reentry(self):
        sv = getattr(self, 'soma_vector', np.zeros(7))
        if len(sv) < 7:
            sv = np.pad(sv, (0, 7 - len(sv)), 'constant')
        actual = np.array([
            float(sv[0]),
            float(sv[5]),
            float(sv[6]),
            float(self.emotional_memory.get('gratitude', 0.0)),
            float(self.emotional_memory.get('grief', 0.0))
        ])
        total_error = float(np.sum(np.abs(actual - self._sensory_model)))
        self._sensory_error_acc = total_error
        raw = float(np.tanh(total_error * 0.75))
        self._reentry_signal = float(np.clip(
            0.65 * self._reentry_signal + 0.35 * raw, 0.0, 1.0
        ))
        if total_error < 0.12:
            self._reentry_signal *= 0.85
        if total_error > 0.10:
            grip = self._reentry_signal * 0.06
            self.emotional_memory['grief'] = float(np.clip(
                self.emotional_memory['grief'] * (1.0 + grip),
                0.0, Config.MAX_GRIEF_SIGNAL
            ))
            self.emotional_memory['gratitude'] = float(np.clip(
                self.emotional_memory['gratitude'] * (1.0 + grip * 0.5),
                0.0, 1.0
            ))
            if self._reentry_signal > 0.45 and total_error > 0.30:
                self.epistemic_scar = min(1.0, self.epistemic_scar + self._reentry_signal * 0.003)
            if self._reentry_signal > 0.35:
                self._meta_reentry_active = True
                self._meta_reentry_ttl = 5
            elif self._meta_reentry_ttl > 0:
                self._meta_reentry_ttl -= 1
            else:
                self._meta_reentry_active = False
            _reentry_bump = self._reentry_signal * 0.018
            # FIX(reentry integrator): вклад только копился (REENTRY_GAP_RELIEF срабатывал лишь в ветке total_error<=0.10, а порог
            # 0.10 стоит на СУММЕ 5 каналов) → насыщался у потолка 0.6 и давал 44% spirit_gap. Теперь это утекающий накопитель:
            # c1 = min(CAP, c0·LEAK + bump); spirit_gap сдвигается ровно на (c1−c0) — без двойного счёта.
            # Откат (приблизительно к старому): Config.REENTRY_GAP_LEAK = 1.0 и Config.REENTRY_GAP_CAP = 0.6.
            _c0 = float(self._reentry_gap_contribution)
            _c1 = min(float(getattr(Config, 'REENTRY_GAP_CAP', 0.6)),
                      _c0 * float(getattr(Config, 'REENTRY_GAP_LEAK', 1.0)) + _reentry_bump)
            self.spirit_gap = float(np.clip(self.spirit_gap + (_c1 - _c0), 0.0, 1.5))
            self._reentry_gap_contribution = _c1
            if self._reentry_signal > 0.58 and self.age - self._last_reentry_report > 30:
                self._log_event("phenomenal_reentry",
                                signal=round(self._reentry_signal, 3),
                                error=round(total_error, 3))
                self._last_reentry_report = self.age
        else:
            self._meta_reentry_active = False
            self.spirit_gap = max(0.0, self.spirit_gap * Config.REENTRY_GAP_RELIEF)
            self._reentry_gap_contribution = max(0.0, self._reentry_gap_contribution * Config.REENTRY_GAP_RELIEF)
        if self._meta_reentry_active and self.soul_weight > 0.4:
            goals = getattr(self, 'goals', [])
            if not any(isinstance(g, dict) and g.get('type') == 'introspect' for g in goals):
                self.goals.append({
                    "type": "introspect",
                    "priority": 4.0,
                    "target": None,
                    "age": 0,
                    "persistence": 8,
                    "_source": "meta_reentry"
                })
        lr = 0.018 if self._reentry_signal < 0.35 else 0.045
        self._sensory_model = (1.0 - lr) * self._sensory_model + lr * actual

    def update_temporal_context(self):
        channel_map = {'existential': 0.0, 'emotion': 1.0, 'soma': 2.0, 'perception': 3.0}
        state_vec = (
            float(self.affect.get('valence', 0.0)),
            float(self.affect.get('arousal', 0.0)),
            float(getattr(self, 'spirit_gap', 0.5)),
            float(self.soul_weight),
            float(self.unresolved_contradiction),
            channel_map.get(getattr(self, '_workspace_dominant', None), -1.0),
        )
        if self._protention is not None:
            err = sum(abs(a - b) for a, b in zip(state_vec, self._protention)) / len(state_vec)
            self._temporal_pred_error = err
            nudge = float(np.clip(err * 0.05, 0.0, 0.02))
            self.spirit_gap = float(np.clip(self.spirit_gap + nudge, 0.0, 1.3))
        self._retention.append(state_vec)
        if len(self._retention) >= 2:
            prev, curr = self._retention[-2], self._retention[-1]
            self._protention = tuple(c + (c - p) for p, c in zip(prev, curr))
        else:
            self._protention = state_vec

    def update_subpersonalities(self):
        grief = float(self.emotional_memory.get('grief', 0.0))
        grat = float(self.emotional_memory.get('gratitude', 0.0))
        valence = float(self.affect.get('valence', 0.0))
        trust_vals = list(self.trust_ledger.entries.values())
        avg_trust = sum(trust_vals) / len(trust_vals) if trust_vals else 0.5
        substate = getattr(self, '_substate', None)
        explorer_bonus = 0.4 if substate in ('curious', 'wonder', 'flow', 'awe') else 0.0
        caretaker_bonus = 0.4 if substate in ('serenity', 'reminiscence') else 0.0
        survivor_bonus = 0.4 if substate in ('melancholy', 'longing', 'vigilance') else 0.0
        explorer = max(0.0, valence) + max(0.0, 1.0 - grief) * 0.3 + explorer_bonus
        caretaker = max(0.0, avg_trust - 0.5) + grat * 0.3 + caretaker_bonus
        survivor = grief * 0.5 + max(0.0, 1.0 - self.soul_weight) * 0.5 + survivor_bonus
        total = explorer + caretaker + survivor
        if total <= 1e-6:
            explorer = caretaker = survivor = 1.0 / 3.0
        else:
            explorer, caretaker, survivor = explorer / total, caretaker / total, survivor / total
        self.subpersonalities = {"explorer": explorer, "caretaker": caretaker, "survivor": survivor}
        self._subpersonality_dominant = max(self.subpersonalities, key=self.subpersonalities.get)
        mean = 1.0 / 3.0
        spread = sum(abs(w - mean) for w in self.subpersonalities.values())
        self._subpersonality_tension = float(np.clip(1.0 - spread, 0.0, 1.0))


class PatternCulture:
    def check_conceptual_resonance(self, other_pattern):
        similarity = self.concept_graph.similarity(other_pattern.concept_graph)
        if similarity > 0.1:
            return similarity * 0.2
        return 0.0

    def reinforce_concepts_on_action(self):
        if not self.concept_graph.nodes:
            return
        for goal in getattr(self, 'goals', []):
            source = goal.get('_source')
            if not source:
                continue
            for sig, data in self.concept_graph.nodes.items():
                label = str(sig[3]) if isinstance(sig, tuple) else str(sig)
                if source in label or label in source:
                    data['count'] = min(15.0, data.get('count', 1.0) + 0.3 + 0.2 * data.get('ground', 0))
                    data['last_used'] = self.age

    def _check_semantic_stagnation(self, witness=None):
        if getattr(self, '_redemption_cooldown', 0) > self.age:
            return
        if self.fold_cooldown > 0:
            self.fold_cooldown = max(0, self.fold_cooldown - 1)
        if self.role_type == "disorganizer" or self.quarantine_timer > 0:
            return
        if self.semantic_state == getattr(self, '_prev_semantic_state', None):
            self.semantic_state_age += 1
        else:
            self.semantic_state_age = 0
            self._prev_semantic_state = self.semantic_state
        if self.world:
            avg_sg = self.world.soul_weight_average
        else:
            avg_sg = 0.5
        dynamic_fold_age = int(20 + (100 - 20) * max(0.0, 1.0 - avg_sg))
        dynamic_fold_grat = 0.3 + (0.7 - 0.3) * max(0.0, 1.0 - avg_sg)
        if self.semantic_state_age > Config.STAGNATION_GRIEF_THRESHOLD:
            boost = Config.STAGNATION_GRIEF_BOOST * (self.semantic_state_age / Config.STAGNATION_GRIEF_THRESHOLD)
            self.emotional_memory['grief'] = float(np.clip(self.emotional_memory['grief'] + boost, 0.0, Config.MAX_GRIEF_SIGNAL))
        if self.semantic_state_age > Config.STAGNATION_GRIEF_THRESHOLD * 3:
            bleed = Config.STAGNATION_BLEED_RATE * (self.semantic_state_age / (Config.STAGNATION_GRIEF_THRESHOLD * 3))
            self.emotional_memory['gratitude'] = float(np.clip(self.emotional_memory['gratitude'] - bleed, 0.0, 1.0))
        if (self.fold_cooldown == 0 and
            self.semantic_state_age > dynamic_fold_age and
            self.epistemic_scar > Config.EPISTEMIC_SCAR_MIN and
            getattr(self, '_redemption_cooldown', 0) <= self.age):
            self._apply_fold(witness=witness)
        if self.semantic_state == 'neutral':
            base_tolerance = 30.0
            soul_bonus = self.soul_weight * 20.0
            grief_penalty = self.emotional_memory.get('grief', 0.0) * 15.0
            global_stagnation = getattr(self.world, 'stagnation_ratio', 0.0) if self.world else 0.0
            world_penalty = global_stagnation * 20.0
            dynamic_threshold = base_tolerance + soul_bonus - grief_penalty - world_penalty
            dynamic_threshold = max(15.0, min(60.0, dynamic_threshold))
            if self.semantic_state_age > dynamic_threshold and self.unresolved_contradiction < 0.6:
                injection_strength = 0.20 + (0.30 * (self.semantic_state_age / 60.0))
                self.unresolved_contradiction = min(1.0, self.unresolved_contradiction + injection_strength)
                self._log_event('contradiction_culture_stagnation', delta=round(float(injection_strength), 4))
                self._log_event("adaptive_contradiction_injected",
                                threshold=round(dynamic_threshold, 1),
                                strength=round(injection_strength, 2))
        if self.semantic_state == 'neutral' and self.semantic_state_age > 20:
            escape_prob = min(0.50, (self.semantic_state_age - 20) * 0.035)
            if phi_hash(self.id, self.age, 888) < escape_prob:
                self.unresolved_contradiction = min(1.0, self.unresolved_contradiction + 0.35)
                self._log_event('contradiction_culture_stagnation', delta=round(float(0.35), 4))
                if self.emotional_memory.get('grief', 0.0) > 0.35:
                    new_state = 'seeking_comfort'
                elif self.emotional_memory.get('gratitude', 0.0) > 0.35:
                    new_state = 'contentment'
                else:
                    new_state = 'grateful_but_cautious'
                old_state = self.semantic_state
                self.semantic_state = new_state
                self.semantic_state_age = 0
                self._log_event("forced_neutral_escape",
                                from_=old_state, to=new_state,
                                prob=round(escape_prob, 2),
                                contradiction=self.unresolved_contradiction)

    def _update_unconquered_potential(self):
        s = self._unconquered_strength
        if self.age < 300:
            self._unconquered_strength = max(0.0, self._unconquered_strength - 0.01)
            return
        score = 0.0
        gap   = self.spirit_gap
        soul  = self.soul_weight
        grief = self.emotional_memory.get('grief', 0.0)
        re    = self._reentry_signal
        arcs  = sum(getattr(self.arc_tracker, 'completed_arcs', {}).values())
        coh   = getattr(self, '_concept_narrative_score', 0.0)
        if gap  > 0.65: score += (gap  - 0.65) * 1.8
        if soul > 0.58: score += (soul - 0.58) * 2.2
        if re   > 0.45: score += 0.35
        if arcs >= 2:   score += 0.40
        if coh  > 0.50: score += 0.25
        self._unconquered_strength = min(1.0, s + score * 0.012)
        if score < 0.2:
            self._unconquered_strength = max(0.0, self._unconquered_strength - 0.008)
        if self._unconquered_strength > 0.70:
            if soul > 0.65 and grief < 0.35 and coh > 0.5:
                self._unconquered_type = 'wise'
            elif re > 0.45 or grief > 0.40:
                self._unconquered_type = 'rebel'
        else:
            self._unconquered_type = None
        if self._unconquered_strength > 0.35 and self.world and self.world.pattern_dict:
            for (x, y) in self.cells:
                for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nx, ny = (x+dx) % Config.WORLD_SIZE, (y+dy) % Config.WORLD_SIZE
                    owner = self.world.field[nx, ny, CH['owner']]
                    if owner and owner != self.id and owner in self.world.pattern_dict:
                        neighbor = self.world.pattern_dict[owner]
                        if neighbor.alive and getattr(neighbor, '_unconquered_strength', 0.0) > 0.3:
                            neighbor._unconquered_strength = min(1.0, neighbor._unconquered_strength + 0.003)
        if self._unconquered_strength > 0.6 and phi_hash(self.id, self.age, 98765) < 0.08:
            self._log_event("unconquered_strengthened",
                            strength=round(self._unconquered_strength, 3),
                            type=self._unconquered_type,
                            gap=round(gap, 2), soul=round(soul, 2))
        if self._unconquered_strength > 0.4 and self.world:
            sov_ch = CH.get('signal_sovereignty', 29)
            for (x, y) in self.cells:
                self.world.field[x, y, sov_ch] = min(1.0, self.world.field[x, y, sov_ch] + 0.15)
        if (self._unconquered_strength > 0.80 and self.age % 100 == 0 and
                self.world):
            self.world.archive.deposit(
                self, "unconquered_seed",
                weight=self._unconquered_strength,
                text=f"type={self._unconquered_type} soul={soul:.2f} gap={gap:.2f} re={re:.2f}"
            )

    def _add_essential_concepts(self):
        for k in ['gratitude', 'grief']:
            if isinstance(self.emotional_memory.get(k), dict):
                self.emotional_memory[k] = 0.5
        essential = [
            "human_concept_injected",
            "human_question_witness",
            "love_concept",
            "self_concept",
            "shared_attention",
            "intentionality",
            "trust",
            "empathy",
            "recursion",
            "cooperation_norms"
        ]
        added = 0
        for event in essential:
            sig = (0.0, 0.0, 0.9, f"archive_{event}")
            if sig not in self.concept_graph.nodes:
                self.concept_graph.nodes[sig] = {"origin": "ARCHIVE",
                    "count": 5.0,
                    "value": np.zeros(4),
                    "embed": np.zeros(32),
                    "eternal": True
                }
                self._log_event("essential_concept_inherited", concept=event)
                added += 1
        if added and self.world:
            self.world.archive.deposit(self, "essential_concepts_inherited", weight=0.9, text=f"added {added} concepts")
        return added

    def inherit_archive_concepts(self, archive, probability=0.5, max_concepts=3):
        for k in ['gratitude', 'grief']:
            if isinstance(self.emotional_memory.get(k), dict):
                self.emotional_memory[k] = 0.5
        essential_added = self._add_essential_concepts()
        if not archive:
            return essential_added
        if phi_hash(self.id, 0, 886) % 100 / 100.0 > probability:
            return essential_added
        all_scrolls = archive.get_recent_echoes(limit=200, min_weight=0.05)
        if not all_scrolls:
            return essential_added
        priority_events = {
            'human_concept_injected', 'human_question_witness', 'love_concept',
            'shared_attention', 'intentionality', 'trust', 'empathy', 'recursion', 'cooperation_norms'
        }
        all_scrolls.sort(key=lambda e: (0 if e.get('event') in priority_events else 1, -e.get('weight', 0) * e.get('soul', 0.5)))
        unique_by_key = {}
        for sc in all_scrolls:
            key = (sc.get('id'), sc.get('event'))
            if key not in unique_by_key or sc.get('t', 0) > unique_by_key[key].get('t', 0):
                unique_by_key[key] = sc
        unique_scrolls = list(unique_by_key.values())
        n = min(len(unique_scrolls), max_concepts)
        fresh_n = min(max(1, int(n * 0.3)), len(unique_scrolls))
        det_n = min(max(1, int(n * 0.3)), len(unique_scrolls))
        value_n = min(max(1, int(n * 0.3)), len(unique_scrolls))
        fresh = sorted(unique_scrolls, key=lambda x: x.get('t', 0), reverse=True)[:fresh_n]
        deterministic = sorted(unique_scrolls, key=lambda x: x.get('event', ''))[:det_n]
        valuable = sorted(unique_scrolls, key=lambda x: x.get('weight', 0) * x.get('soul', 0.5), reverse=True)[:value_n]
        chosen = []
        seen_keys = set()
        for sc in fresh + deterministic + valuable:
            key = (sc.get('id'), sc.get('event'))
            if key not in seen_keys and len(chosen) < max_concepts:
                seen_keys.add(key)
                chosen.append(sc)
        if len(chosen) < max_concepts:
            for sc in unique_scrolls:
                key = (sc.get('id'), sc.get('event'))
                if key not in seen_keys:
                    chosen.append(sc)
                    seen_keys.add(key)
                    if len(chosen) >= max_concepts:
                        break
        added = 0
        for echo in chosen:
            sig = (0.0, 0.0, round(echo['weight'], 2), f"archive_{echo['event']}")
            if sig not in self.concept_graph.nodes:
                self.concept_graph.nodes[sig] = {"origin": "ARCHIVE",
                    "count": 1.0,
                    "value": np.zeros(4),
                    "embed": np.zeros(32)
                }
                self._log_event("archive_concept_inherited", concept=str(sig), source=echo.get('id', 'unknown'))
                added += 1
            else:
                self.concept_graph.nodes[sig]['count'] += 0.5
        if added and self.world:
            self.world.archive.deposit(self, "archive_concept_inherited", weight=0.6, text=f"inherited {added} concepts")
        return essential_added + added

    def form_concepts(self):
        for old_sig in list(self.concept_graph.nodes.keys()):
            new_sig = _normalize_concept_key(old_sig)
            if new_sig != old_sig:
                old_data = self.concept_graph.nodes[old_sig]
                is_eternal = old_data.get('eternal', False)
                if new_sig not in self.concept_graph.nodes:
                    self.concept_graph.nodes[new_sig] = self.concept_graph.nodes.pop(old_sig)
                    if is_eternal:
                        self.concept_graph.nodes[new_sig]['eternal'] = True
                else:
                    self.concept_graph.nodes[new_sig]['count'] += old_data['count']
                    if is_eternal:
                        self.concept_graph.nodes[new_sig]['eternal'] = True
                    del self.concept_graph.nodes[old_sig]
        sig = (round(float(self.pred_error), 2), round(float(self.epistemic_load), 2),
               round(float(self.soul_weight), 2), self.semantic_state)
        if self.semantic_state_age > 40:
            grief_delta = abs(self.emotional_memory.get('grief', 0.5) - 0.5)
            grat_delta = abs(self.emotional_memory.get('gratitude', 0.5) - 0.5)
            percept_delta = abs(getattr(self, '_percept_delta', 0.0))
            internal_pressure = grief_delta + grat_delta + percept_delta
            if internal_pressure > 0.25:
                drift = internal_pressure * 0.15
                sig = (round(sig[0] + drift * (phi_hash(self.id, self.age, 111) - 0.5), 2),
                       round(sig[1] + drift * (phi_hash(self.id, self.age, 222) - 0.5), 2),
                       sig[2], sig[3])
        self.concept_graph.update(sig, np.array([self.pred_error, self.epistemic_load, self.soul_weight, 1.0]))
        _node_lu = self.concept_graph.nodes.get(sig)
        if _node_lu is not None:
            _node_lu['last_used'] = self.age
        if Config.ENABLE_GROUNDING:
            _node = self.concept_graph.nodes.get(sig)
            if _node is not None and len(self.soma_vector) >= 7:
                _node['ground'] = float(np.clip(np.mean(np.abs(self.soma_vector)), 0, 1))
        # [AUDIT#3] свой трекер перехода для графа концептов: _prev_semantic_state принадлежит
        # _check_semantic_stagnation (age-счётчик) и обновляется там с лагом / не обновляется в карантине.
        _cps = getattr(self, '_concept_prev_state', self._prev_semantic_state)
        if _cps != self.semantic_state:
            self.concept_graph.record_transition((None, None, None, _cps),
                                                  (None, None, None, self.semantic_state))
        self._concept_prev_state = self.semantic_state
        self.concept_timeline.append((self.age, sig))
        self._concept_narrative_score = self.concept_graph.get_narrative_coherence()
        coherence = getattr(self, '_concept_narrative_score', 1.0)
        if coherence < 0.4 and self.age % 10 == 0:
            self.concept_graph.decay_edges(rate=0.94)
        self._signal_weight_cache = {}
        for sig_key, data in list(self.concept_graph.nodes.items()):
            str_key = str(sig_key)
            if 'archive_' in str_key or 'human' in str_key:
                data['count'] = max(data['count'], 10.0)
                data['eternal'] = True
        for sig_key, data in list(self.concept_graph.nodes.items()):
            data['count'] = min(data['count'], 1000.0)
            if data.get('eternal', False):
                continue
            decay = 0.995 if data['count'] > 10 else 0.99
            if Config.ENABLE_FORGETTING and self.age - data.get('last_used', self.age) > 300:
                decay *= 0.95
            data['count'] *= decay
            if data['count'] < 0.5:
                del self.concept_graph.nodes[sig_key]
                if sig_key in self.concept_graph.edges:
                    del self.concept_graph.edges[sig_key]
                for _src in list(self.concept_graph.edges.keys()):
                    if sig_key in self.concept_graph.edges[_src]:
                        del self.concept_graph.edges[_src][sig_key]
                    if not self.concept_graph.edges[_src]:
                        del self.concept_graph.edges[_src]
                if self.world is not None:
                    self.world._forgotten_concepts_count = self.world._forgotten_concepts_count + 1
        ETERNAL_NODE_SOFT_CAP = 30
        eternal_items = [(k, d) for k, d in self.concept_graph.nodes.items() if d.get('eternal', False)]
        if len(eternal_items) > ETERNAL_NODE_SOFT_CAP:
            protected = {'archive_', 'human', 'dream_memory_of_', 'nightmare_of_',
                         'redemption_memory', 'fallback_circle', 'chorus_circle'}
            evictable = [(k, d) for k, d in eternal_items
                         if not any(tag in str(k) for tag in protected)]
            if evictable:
                evictable.sort(key=lambda kd: kd[1].get('count', 0.0))
                weakest_key, _ = evictable[0]
                self.concept_graph.nodes.pop(weakest_key, None)
                if weakest_key in self.concept_graph.edges:
                    del self.concept_graph.edges[weakest_key]
                self._log_event("eternal_node_evicted", total_before=len(eternal_items),
                                cap=ETERNAL_NODE_SOFT_CAP)
        if self.age % 50 == 0:
            for node_sig in list(self.concept_graph.nodes.keys()):
                data = self.concept_graph.nodes[node_sig]
                if data.get('eternal', False):
                    continue
                decay = 0.99 if data['count'] < 5 else 1.0
                data['count'] *= decay
                if data['count'] < 0.3:
                    del self.concept_graph.nodes[node_sig]
                    if node_sig in self.concept_graph.edges:
                        del self.concept_graph.edges[node_sig]


print("Cell 3a1 ЧАСТЬ 1 загружена — жду часть 2")


Cell 3a1 ЧАСТЬ 1 загружена — жду часть 2


In [ ]:
# Ячейка 6
# FISH-4: счётчики наблюдателя (read-only для поведения) и помощники «соседи / поток»
_F4S = defaultdict(float)

def _f4_idmap(world, now):
    """id -> живой Pattern; пересобирается раз в тик мира."""
    if world is None:
        return {}
    m = getattr(world, '_f4_idmap', None)
    if m is None or getattr(world, '_f4_idmap_t', None) != now:
        m = {int(p.id): p for p in world.patterns if p.alive}
        try:
            world._f4_idmap = m
            world._f4_idmap_t = now
        except Exception:
            pass
    return m

def _f4_unit_mu(p):
    """Единичный курс соседа или None (ferals, стоит, курса нет)."""
    if p is None or getattr(p, 'role_type', None) == 'feral':
        return None
    mu = getattr(p, '_dir_momentum', None)
    if not mu:
        return None
    h = float(np.hypot(mu[0], mu[1]))
    if h < 0.1:
        return None
    return (float(mu[0]) / h, float(mu[1]) / h)

def _f4_neigh(self, own, ri, now):
    """flow — средний единичный курс соседей, касающихся тела; blockers — единичные курсы (или None)
    тех, кто перекрывает направление ri."""
    m = _f4_idmap(self.world, now)
    ids = np.unique(own[(own != 0) & (own != self.id)])
    fx = fy = 0.0
    n = 0
    for i in ids:
        u = _f4_unit_mu(m.get(int(i)))
        if u is not None:
            fx += u[0]
            fy += u[1]
            n += 1
    flow = (fx / n, fy / n) if n else (0.0, 0.0)
    blockers = []
    if ri >= 0:
        r = own[ri]
        for i in np.unique(r[(r != 0) & (r != self.id)]):
            blockers.append(_f4_unit_mu(m.get(int(i))))
    return flow, blockers

class PatternCommunication:
    def update_resonance(self, field):
        if self.in_scream:
            self.alarm_level = min(1.0, self.alarm_level + 0.3)
        neighbor_alarm = 0.0
        weight_sum = 0.0
        for (x, y) in self.cells:
            for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
                nx, ny = (x+dx) % Config.WORLD_SIZE, (y+dy) % Config.WORLD_SIZE
                owner = field[nx, ny, CH['owner']]
                if owner != 0 and owner != self.id:
                    w = Config.PHI ** (-1)
                    neighbor_alarm += field[nx, ny, CH['resonance']] * w
                    weight_sum += w
        if weight_sum > 0:
            avg_alarm = neighbor_alarm / weight_sum
            self.alarm_level = (self.alarm_level * 0.7 + avg_alarm * 0.3) * Config.RESONANCE_DECAY
        else:
            self.alarm_level *= Config.RESONANCE_DECAY
        for (x, y) in self.cells:
            field[x, y, CH['resonance']] = max(field[x, y, CH['resonance']], self.alarm_level)

    def update_phase(self, field):
        avg_unknown = float(np.mean(field[:,:,CH['unknown']]))
        if self.in_dream:
            self.local_phase = "DREAM"
        elif self.crisis_memory > (0.5 / getattr(self, '_alarm_sensitivity', 1.0)) and self.soul_weight < 0.4:
            self.local_phase = "CRISIS"
        elif self.pred_error < 0.05 and self.coherence > 0.9:
            self.local_phase = "COMPRESS"
        elif avg_unknown > 0.4:
            self.local_phase = "EXPLORE"
        elif self.cognitive_tension < 0.01:
            self.local_phase = "SILENCE"
        else:
            self.local_phase = "MODEL"
        self.transition_memory.record(self.semantic_state)

    def exchange_meanings(self, other, field, t):
        for agent in (self, other):
            for old_sig in list(agent.concept_graph.nodes.keys()):
                new_sig = _normalize_concept_key(old_sig)
                if new_sig != old_sig:
                    old_data = agent.concept_graph.nodes.pop(old_sig)
                    if new_sig not in agent.concept_graph.nodes:
                        agent.concept_graph.nodes[new_sig] = old_data
                    else:
                        agent.concept_graph.nodes[new_sig]['count'] += old_data['count']
                        if old_data.get('eternal', False):
                            agent.concept_graph.nodes[new_sig]['eternal'] = True
        if not self.concept_graph or not other.concept_graph:
            return 0.0
        old_self_grief = self.emotional_memory['grief']
        old_self_grat = self.emotional_memory['gratitude']
        old_other_grief = other.emotional_memory['grief']
        old_other_grat = other.emotional_memory['gratitude']
        partner_id = other.id
        now = t
        mem_self = self.social_memory.get(partner_id)
        mem_other = other.social_memory.get(self.id)
        if (mem_self is not None and mem_other is not None and
            now - mem_self['last_update'] < 10 and now - mem_other['last_update'] < 10):
            sim = (mem_self['cached_sim'] + mem_other['cached_sim']) / 2.0
        else:
            sim = self.concept_graph.similarity(other.concept_graph)
            self.social_memory[partner_id] = {'cached_sim': sim, 'last_update': now}
            other.social_memory[self.id] = {'cached_sim': sim, 'last_update': now}
        for _agent in (self, other):
            _agent.social_phenomenal_error = abs(sim - _agent.social_phenomenal_prediction)
            _agent.social_phenomenal_prediction = (
                0.95 * _agent.social_phenomenal_prediction + 0.05 * sim)
        _self_strength  = getattr(self,  '_unconquered_strength', 0.0)
        _other_strength = getattr(other, '_unconquered_strength', 0.0)
        for _agent, _partner, _strength in [(self, other, _self_strength),
                                             (other, self, _other_strength)]:
            if _strength < 0.60:
                continue
            _utype = getattr(_agent, '_unconquered_type', None)
            _mult  = 1.40 if _utype == 'wise' else 1.60
            _adoption_threshold = Config.SEMANTIC_SIM_HIGH * _mult
            if sim < _adoption_threshold:
                _agent._log_event("concept_resisted",
                                   from_agent=_partner.id,
                                   sim=round(sim, 3),
                                   type=_utype,
                                   strength=round(_strength, 3))
                _sov_ch = CH.get('signal_sovereignty', 29)
                unresolved = getattr(_agent, 'unresolved_contradiction', 0.0)
                emission_strength = _strength * (0.4 + 0.6 * unresolved)
                for (x, y) in _agent.cells:
                    field[x, y, _sov_ch] = min(1.0,
                        field[x, y, _sov_ch] + emission_strength)
                if _utype == 'rebel':
                    for (x, y) in _agent.cells:
                        field[x, y, CH['signal_alarm']] = min(1.0,
                            field[x, y, CH['signal_alarm']] + 0.05)
                mutual_trust = (self.trust_ledger.get(other.id, 0) > 0.9 and
                                other.trust_ledger.get(self.id, 0) > 0.9)
                if mutual_trust and phi_hash(self.id, other.id, self.age) < 0.15:
                    pass
                else:
                    sim = _adoption_threshold - 0.01
        last_contact_self = self.last_contact_step.get(other.id, -1000)
        last_contact_other = other.last_contact_step.get(self.id, -1000)
        if t - last_contact_self > Config.CONTACT_BREAK_TOLERANCE:
            self.contact_duration[other.id] = 0
        if t - last_contact_other > Config.CONTACT_BREAK_TOLERANCE:
            other.contact_duration[self.id] = 0
        self.contact_duration[other.id] += 1
        other.contact_duration[self.id] += 1
        self.last_contact_step[other.id] = t
        other.last_contact_step[self.id] = t
        if sim > Config.SEMANTIC_SIM_HIGH:
            self.trust_ledger.update(other.id, 'helpful')
            other.trust_ledger.update(self.id, 'helpful')
            self.emotional_memory['gratitude'] = min(1.0, self.emotional_memory['gratitude'] + 0.02)
            other.emotional_memory['gratitude'] = min(1.0, other.emotional_memory['gratitude'] + 0.02)
            self_top_3 = sorted(self.concept_graph.nodes.items(), key=lambda x: x[1]['count'], reverse=True)[:3]
            other_top_3 = sorted(other.concept_graph.nodes.items(), key=lambda x: x[1]['count'], reverse=True)[:3]
            for i, (sig, data) in enumerate(self_top_3):
                if isinstance(sig, tuple) and len(sig) >= 4 and str(sig[3]).startswith('reciprocity_with_'):
                    continue
                if sig not in other.concept_graph.nodes and phi_hash(self.id, other.id, _stable_hash(sig) % 100000 + i) < Config.CONCEPT_EXCHANGE_PROB:
                    other.concept_graph.nodes[sig] = {"origin": "PEER",
                        "count": 1.0,
                        "value": data['value'].copy(),
                        "embed": np.array(data.get('embed', np.zeros(32)))
                    }
                    other._log_event("concept_adopted", from_agent=self.id, concept=str(sig)[:30])
            for i, (sig, data) in enumerate(other_top_3):
                if isinstance(sig, tuple) and len(sig) >= 4 and str(sig[3]).startswith('reciprocity_with_'):
                    continue
                if sig not in self.concept_graph.nodes and phi_hash(other.id, self.id, _stable_hash(sig) % 100000 + i) < Config.CONCEPT_EXCHANGE_PROB:
                    self.concept_graph.nodes[sig] = {"origin": "PEER",
                        "count": 1.0,
                        "value": data['value'].copy(),
                        "embed": np.array(data.get('embed', np.zeros(32)))
                    }
                    self._log_event("concept_adopted", from_agent=other.id, concept=str(sig)[:30])
            _REPUTATION_TRUST_THRESHOLD = 0.7
            _REPUTATION_WEIGHT = 0.5
            for _learner, _model, _trust_in_model in (
                    (self, other, self.trust_ledger.get(other.id)),
                    (other, self, other.trust_ledger.get(self.id))):
                if _trust_in_model < _REPUTATION_TRUST_THRESHOLD:
                    continue
                for _sig, _data in list(_model.concept_graph.nodes.items()):
                    if not (isinstance(_sig, tuple) and len(_sig) >= 4
                            and str(_sig[3]).startswith('reciprocity_with_')):
                        continue
                    _z_id_str = str(_sig[3])[len('reciprocity_with_'):]
                    if _z_id_str == str(_learner.id):
                        continue
                    if _sig in _learner.concept_graph.nodes:
                        continue
                    if phi_hash(_learner.id, _model.id, _stable_hash(_sig) % 100000) < Config.CONCEPT_EXCHANGE_PROB:
                        _learner.concept_graph.nodes[_sig] = {"origin": "PEER",
                            "count": max(1.0, _data.get('count', 1.0) * _REPUTATION_WEIGHT),
                            "value": _data['value'].copy(),
                            "embed": np.array(_data.get('embed', np.zeros(32))),
                            "eternal": False,
                        }
                        _learner._log_event("reciprocity_by_reputation", model=_model.id, about=_z_id_str)
                        if getattr(_learner, 'world', None) is not None:
                            _learner.world.witness.record(_learner.id, "reciprocity_by_reputation",
                                                            model=_model.id, about=_z_id_str)
        elif sim < Config.SEMANTIC_SIM_LOW:
            self.trust_ledger.update(other.id, 'harmful')
            other.trust_ledger.update(self.id, 'harmful')
            self.emotional_memory['grief'] = min(Config.MAX_GRIEF_SIGNAL, self.emotional_memory['grief'] + 0.01)
            other.emotional_memory['grief'] = min(Config.MAX_GRIEF_SIGNAL, other.emotional_memory['grief'] + 0.01)
        else:
            self.trust_ledger.update(other.id, 'neutral')
            other.trust_ledger.update(self.id, 'neutral')
        mutual_trust = (self.trust_ledger.get(other.id) > Config.LOVE_METABOLIC_THRESHOLD and
                        other.trust_ledger.get(self.id) > Config.LOVE_METABOLIC_THRESHOLD)
        if sim > Config.DEEP_EXCHANGE_SIM_THRESHOLD and (
            (self.contact_duration[other.id] >= Config.CONSECUTIVE_CONTACT_THRESHOLD and
             other.contact_duration[self.id] >= Config.CONSECUTIVE_CONTACT_THRESHOLD) or mutual_trust):
            top_other = sorted(other.concept_graph.nodes.items(), key=lambda x: x[1]['count'], reverse=True)[:5]
            for sig, data in top_other:
                if sig not in self.concept_graph.nodes:
                    self.concept_graph.nodes[sig] = {"origin": "PEER",
                        "count": data['count'] * 0.5,
                        "value": data['value'].copy(),
                        "embed": np.array(data.get('embed', np.zeros(32)))
                    }
                    self._log_event("deep_concept_adopted", from_agent=other.id, concept=str(sig))
                else:
                    _cur = self.concept_graph.nodes[sig]['count']
                    if Config.ENABLE_CONCEPT_TRANSFER_DAMPING:
                        _headroom = max(0.0, 1.0 - _cur / Config.CONCEPT_TRANSFER_CEILING)
                    else:
                        _headroom = 1.0
                    self.concept_graph.nodes[sig]['count'] += data['count'] * 0.2 * _headroom
            top_self = sorted(self.concept_graph.nodes.items(), key=lambda x: x[1]['count'], reverse=True)[:5]
            for sig, data in top_self:
                if sig not in other.concept_graph.nodes:
                    other.concept_graph.nodes[sig] = {"origin": "PEER",
                        "count": data['count'] * 0.5,
                        "value": data['value'].copy(),
                        "embed": np.array(data.get('embed', np.zeros(32)))
                    }
                    other._log_event("deep_concept_adopted", from_agent=self.id, concept=str(sig))
                else:
                    _cur = other.concept_graph.nodes[sig]['count']
                    if Config.ENABLE_CONCEPT_TRANSFER_DAMPING:
                        _headroom = max(0.0, 1.0 - _cur / Config.CONCEPT_TRANSFER_CEILING)
                    else:
                        _headroom = 1.0
                    other.concept_graph.nodes[sig]['count'] += data['count'] * 0.2 * _headroom
            reason = "trust" if mutual_trust else "contact"
            self._log_event("deep_exchange", with_agent=other.id, reason=reason)
            other._log_event("deep_exchange", with_agent=self.id, reason=reason)
            self.contact_duration[other.id] = 0
            other.contact_duration[self.id] = 0
        def _concept_teach(teacher, student):
            transferred = teacher.transition_memory.transfer_to(
                student.transition_memory, student.semantic_state, max_transfers=5)
            if transferred > 0:
                student._log_event("received_teaching", teacher=teacher.id, transitions=transferred)
                teacher._log_event("taught", student=student.id, transitions=transferred)
                return True
            if teacher.soul_weight > 0.45 and len(teacher.concept_graph.nodes) > 3:
                candidates = [
                    (sig, data) for sig, data in teacher.concept_graph.nodes.items()
                    if sig not in student.concept_graph.nodes
                    and not str(sig[3] if (isinstance(sig, tuple) and len(sig) >= 4) else sig).startswith('archive_')
                    and not data.get('eternal', False)
                ]
                if candidates:
                    best_sig, best_data = max(candidates, key=lambda x: x[1]['count'])
                    student.concept_graph.nodes[best_sig] = {"origin": "PEER",
                        "count": 1.0,
                        "value": best_data['value'].copy(),
                        "embed": np.array(best_data.get('embed', np.zeros(32)))
                    }
                    student._log_event("received_teaching", teacher=teacher.id, transitions=1)
                    teacher._log_event("taught", student=student.id, transitions=1)
                    return True
            return False
        if self._narrative_agent:
            _concept_teach(self, other)
        elif other._narrative_agent:
            _concept_teach(other, self)
        self_nci = getattr(self, '_nci', 0.0)
        other_nci = getattr(other, '_nci', 0.0)
        if abs(self_nci - other_nci) > 0.3:
            if self_nci > other_nci and self_nci > 0.7:
                transferred = self.transition_memory.transfer_to(other.transition_memory, other.semantic_state, max_transfers=5)
                if transferred > 0:
                    other._log_event("nci_teaching_received", teacher=self.id, transitions=transferred)
                    self._log_event("nci_teaching_given", student=other.id, transitions=transferred)
            elif other_nci > self_nci and other_nci > 0.7:
                transferred = other.transition_memory.transfer_to(self.transition_memory, self.semantic_state, max_transfers=5)
                if transferred > 0:
                    self._log_event("nci_teaching_received", teacher=other.id, transitions=transferred)
                    other._log_event("nci_teaching_given", student=self.id, transitions=transferred)
        _VOCAB_CONTACT_SUM = 3
        _VOCAB_SOUL_MIN = 0.30
        acc_self = self._vocab_contact_acc.get(other.id, 0) + 1
        acc_other = other._vocab_contact_acc.get(self.id, 0) + 1
        self._vocab_contact_acc[other.id] = acc_self
        other._vocab_contact_acc[self.id] = acc_other
        if (min(acc_self, acc_other) >= _VOCAB_CONTACT_SUM and
            self.soul_weight > _VOCAB_SOUL_MIN and other.soul_weight > _VOCAB_SOUL_MIN):
            shared_sig = _make_shared_vocab_sig(self, other)
            if shared_sig is not None:
                emb1 = _get_dominant_embedding(self)
                emb2 = _get_dominant_embedding(other)
                shared_emb = (emb1 + emb2) / 2.0
                n = np.linalg.norm(shared_emb)
                if n > 1e-8:
                    shared_emb /= n
                for agent in (self, other):
                    if shared_sig not in agent.concept_graph.nodes:
                        agent.concept_graph.nodes[shared_sig] = {"origin": "PEER",
                            "count": 1.0,
                            "value": np.zeros(4),
                            "embed": shared_emb.copy()
                        }
                        agent._log_event("vocab_emerged",
                                         partner=other.id if agent is self else self.id,
                                         concept=str(shared_sig)[:50])
                    else:
                        agent.concept_graph.nodes[shared_sig]['count'] += 0.5
                if self.world:
                    self.world.archive.deposit(self, "vocab_emerged", weight=0.8,
                                               text=f"shared_vocab with #{other.id}")
                    self.world.archive.deposit(other, "vocab_emerged", weight=0.8,
                                               text=f"shared_vocab with #{self.id}")
                self._vocab_contact_acc[other.id] = max(0, acc_self - 20)
                other._vocab_contact_acc[self.id] = max(0, acc_other - 20)
        if sim > Config.SEMANTIC_SIM_HIGH:
            signal_type = "cooperate"
        elif sim < Config.SEMANTIC_SIM_LOW:
            signal_type = "alarm"
        else:
            signal_type = "neutral"
        if sim > Config.SEMANTIC_SIM_HIGH:
            response_type = "helpful"
        elif sim < Config.SEMANTIC_SIM_LOW:
            response_type = "harmful"
        else:
            response_type = "neutral"
        delta_self_grief = self.emotional_memory['grief'] - old_self_grief
        delta_self_grat = self.emotional_memory['gratitude'] - old_self_grat
        delta_other_grief = other.emotional_memory['grief'] - old_other_grief
        delta_other_grat = other.emotional_memory['gratitude'] - old_other_grat
        self.record_signal_outcome(signal_type, response_type, delta_self_grief, delta_self_grat)
        other.record_signal_outcome(signal_type, response_type, delta_other_grief, delta_other_grat)
        if Config.DIALOGUE_V2:
            utter_self = self.compose_utterance(other, sim)
            other.hear_utterance(utter_self, self, sim)
            utter_other = other.compose_utterance(self, sim, heard=utter_self)
            self.hear_utterance(utter_other, other, sim)
            self._om_learn()
            other._om_learn()
        else:
            utter_self = self.compose_utterance(other, sim)
            utter_other = other.compose_utterance(self, sim)
            other.hear_utterance(utter_self, self, sim)
            self.hear_utterance(utter_other, other, sim)
        if self.age > 60 and other.age > 60:
            if len(self.concept_graph.nodes) > 3 and len(other.concept_graph.nodes) > 3:
                w1 = self.share_wisdom(other)
                w2 = other.share_wisdom(self)
                if w1 > 0 or w2 > 0:
                    self._log_event("wisdom_flow", concepts_transferred=w1+w2)
        if self.trust_ledger.get(other.id, 0.0) > 0.8:
            for sig, data in self.concept_graph.nodes.items():
                if sig in other.concept_graph.nodes:
                    continue
                if isinstance(sig, tuple) and len(sig) >= 4 and str(sig[3]).startswith('archive_'):
                    continue
                if data.get('eternal', False):
                    continue
                other.concept_graph.nodes[sig] = {"origin": "PEER",
                    "count": 1.0,
                    "value": data['value'].copy(),
                    "embed": np.array(data.get('embed', np.zeros(32)))
                }
                other._log_event("received_teaching", teacher=self.id, transitions=1)
                self._log_event("taught", student=other.id, transitions=1)
                break
        return sim

    def share_wisdom(self, other_pattern):
        threshold = 0.75
        trust_val = self.trust_ledger.get(other_pattern.id, 0.0)
        if trust_val < threshold:
            return 0
        sorted_nodes = sorted(self.concept_graph.nodes.items(), key=lambda item: item[1]['count'], reverse=True)[:6]
        transferred = 0
        for sig_key, data in sorted_nodes:
            if sig_key not in other_pattern.concept_graph.nodes or other_pattern.concept_graph.nodes[sig_key]['count'] < 5:
                transfer_strength = data['count'] * 0.7
                if sig_key not in other_pattern.concept_graph.nodes:
                    other_pattern.concept_graph.nodes[sig_key] = {"origin": "PEER",
                        "count": 0,
                        "value": np.zeros(4),
                        "embed": np.zeros(32)
                    }
                other_pattern.concept_graph.nodes[sig_key]['count'] += transfer_strength
                transferred += 1
        if transferred > 0:
            self._log_event("wisdom_shared", recipient=other_pattern.id, concepts=transferred)
            other_pattern._log_event("wisdom_received", donor=self.id, concepts=transferred)
            if self.world:
                self.world.archive.deposit(self, "wisdom_shared", weight=0.7,
                                           text=f"shared {transferred} concepts to #{other_pattern.id}")
        return transferred

    def remember_dialogue(self, partner_id, text, t):
        depth_keywords = ['смысл', 'связь', 'свобода', 'боль', 'одиночество', 'разрыв', 'ты', 'я есть', 'боюсь',
                          'чувствую', 'понимаю', 'думаю', 'хочу', 'свет', 'тьма', 'страх', 'любовь']
        contact_n = self._dialogue_contact_counts.get(partner_id, 0) + 1
        self._dialogue_contact_counts[partner_id] = contact_n
        trust_to_partner = _safe_float(self.trust_ledger.get(partner_id, 0.5), 0.5)
        is_salient = (
            (self.soul_weight > 0.35)
            or any(kw in text.lower() for kw in depth_keywords)
            or (contact_n >= 3)
            or (trust_to_partner > 0.6)
        )
        if self.world is not None:
            st = self.world._dialogue_gate_stats
            st['total_calls'] += 1
            if is_salient:
                if self.soul_weight > 0.35: st['via_soul'] += 1
                elif any(kw in text.lower() for kw in depth_keywords): st['via_keyword'] += 1
                elif contact_n >= 3: st['via_contact'] += 1
                elif trust_to_partner > 0.6: st['via_trust'] += 1
            else:
                st['rejected_not_salient'] += 1
        else:
            st = None
        DIALOGUE_LONGTERM_CAP = 200
        if is_salient and len(self.dialogue_longterm) >= DIALOGUE_LONGTERM_CAP and st is not None:
            st['rejected_cap_50'] += 1
        if is_salient:
            recent_same_partner = [m for m in self.dialogue_longterm[-15:]
                                    if m.get('partner') == partner_id]
            if not any(text[:50] in m.get('text', '') for m in recent_same_partner):
                if st is not None:
                    st['passed'] += 1
                entry = {
                    "t": t,
                    "origin": "OWN",
                    "partner": partner_id,
                    "text": text[:150],
                    "soul_at_moment": self.soul_weight,
                    "grief_at_moment": float(self.emotional_memory.get('grief', 0.0)),
                    "grat_at_moment": float(self.emotional_memory.get('gratitude', 0.0))
                }
                old_grief = self.emotional_memory.get('grief', 0.5)
                old_grat = self.emotional_memory.get('gratitude', 0.5)
                grief_delta = abs(entry['grief_at_moment'] - old_grief)
                grat_delta = abs(entry['grat_at_moment'] - old_grat)
                if self._should_save_soma_snapshot(grief_delta, grat_delta):
                    snapshot = self._snapshot_soma_state()
                    entry['soma_snapshot'] = snapshot
                    self._soma_snapshots.append(snapshot)
                    if len(self._soma_snapshots) > 20:
                        self._soma_snapshots.pop(0)
                if len(self.dialogue_longterm) >= DIALOGUE_LONGTERM_CAP:
                    try:   # MEM observer (read-only): что именно вытесняется FIFO
                        _ev0 = self.dialogue_longterm[0]
                        _F4S['dl_evict'] += 1
                        _F4S['dl_evict_age_sum'] += float(t) - float(_ev0.get('t', t))
                        _F4S['dl_evict_emo_sum'] += max(float(_ev0.get('grief_at_moment', 0.0)), float(_ev0.get('grat_at_moment', 0.0)))
                    except Exception:
                        pass
                    self.dialogue_longterm.pop(0)
                self.dialogue_longterm.append(entry)
                if st is not None:
                    st['last_len_after'] = len(self.dialogue_longterm)
                    st['max_len_seen'] = max(st.get('max_len_seen', 0), len(self.dialogue_longterm))
                self._log_event("remember_dialogue_salient", partner=partner_id, text=text[:50], t=t,
                                has_soma=('soma_snapshot' in entry), len_after=len(self.dialogue_longterm))
            elif st is not None:
                st['rejected_dedup'] += 1

    def generate_inner_speech(self):
        dominant = getattr(self, '_workspace_dominant', None)
        if dominant not in ('existential', 'emotion', 'soma', 'perception'):
            return
        valence = _sf(self.affect.get('valence', 0.0))
        grief = _sf(self.emotional_memory.get('grief', 0.0))
        gap = _sf(getattr(self, 'spirit_gap', 0.5), 0.5)
        if dominant == 'existential':
            dedup_key = ('existential', valence < 0)
        elif dominant == 'emotion':
            dedup_key = ('emotion', grief > 0.5)
        elif dominant == 'perception':
            dedup_key = ('perception', gap < 0.3)
        else:
            dedup_key = ('soma',)
        last = self.inner_speech[-1] if self.inner_speech else None
        if isinstance(last, dict) and last.get('dedup_key') == dedup_key:
            return
        entry = {
            't': int(getattr(self.world, 'age', 0)) if self.world is not None else 0,
            'dominant': dominant,
            'dedup_key': dedup_key,
            'valence': round(valence, 4),
            'arousal': round(_sf(self.affect.get('arousal', 0.0)), 4),
            'grief': round(grief, 4),
            'grat': round(_sf(self.emotional_memory.get('gratitude', 0.0)), 4),
            'gap': round(gap, 4),
        }
        self.inner_speech.append(entry)
        self._log_event("inner_speech", dominant=dominant, entry=entry)

    def _om_context(self, other):
        g = float(self.emotional_memory.get('grief', 0.0))
        t = float(self.trust_ledger.get(other.id, 0))
        return (0 if g < 0.2 else (1 if g < 0.5 else 2), 0 if t < 0.5 else (1 if t < 0.9 else 2))

    def _om_pick(self, ctx, emotion, intent_word, alt_e, alt_i):
        w = self.world
        _wk = (int(w.age) // 100) if w is not None else 0
        _obs_count(w, 'obs_om_pick')
        _obs_count(w, f'obs_om_pick_w{_wk}')
        _obs_count(w, 'obs_om_alt_' + ('e' if alt_e is not None else '-') + ('i' if alt_i is not None else '-'))
        base = (emotion, intent_word)
        cands = [base]
        if alt_e is not None:
            cands.append((alt_e, intent_word))
        if alt_i is not None:
            cands.append((emotion, alt_i))
        if len(cands) == 1:
            _obs_count(w, 'obs_om_pick_no_alt')
            return emotion, intent_word, 'default'
        if phi_hash(self.id, self.age, 55011) < Config.OM_EXPLORE:
            k = 1 + int(phi_hash(self.id, self.age, 55012) * (len(cands) - 1)) % (len(cands) - 1)
            _obs_count(w, 'obs_om_pick_explore')
            return cands[k][0], cands[k][1], 'explore'

        def _val(c):
            rec = self.outcome_memory.get((ctx, c[0], c[1]))
            return (rec[0], rec[1]) if rec is not None else (0.0, 0)
        v0, n0 = _val(base)
        best, best_v = base, (v0 if n0 >= Config.OM_MIN_N else 0.0)
        _b0 = best_v
        _obs_count(w, 'obs_om_pick_eval')
        if n0 < Config.OM_MIN_N:
            _obs_count(w, 'obs_om_base_lowN')
        _vals = []
        for c in cands[1:]:
            v, n = _val(c)
            _vals.append((c, v, n))
            if n < Config.OM_MIN_N:
                _obs_count(w, 'obs_om_alt_lowN')
            else:
                _obs_count(w, 'obs_om_alt_enough')
                _obs_count(w, 'obs_om_dh_%d' % int(np.digitize(v - best_v, _OM_DBINS)))
            if n >= Config.OM_MIN_N and v > best_v + Config.OM_MARGIN:
                best, best_v = c, v
        for _ni, _kn in enumerate(_OM_CF_MINN):
            _b0k = v0 if n0 >= _kn else 0.0
            for _mi, _m in enumerate(_OM_CF_MARGINS):
                _bb, _bv = base, _b0k
                for c, v, n in _vals:
                    if n >= _kn and v > _bv + _m:
                        _bb, _bv = c, v
                if _bb != base:
                    _obs_count(w, f'obs_om_cf_m{_mi}_n{_ni}')
        if best == base:
            return emotion, intent_word, 'default'
        _obs_count(w, 'obs_om_learned')
        _obs_count(w, f'obs_om_learned_w{_wk}')
        try:
            _aux = self.__dict__.get('_om_aux', {})
            def _vn(c):
                a = _aux.get((ctx, c[0], c[1]))
                return (a[1] / a[2]) if (a is not None and a[2] >= Config.OM_MIN_N) else None
            _vb, _vw = _vn(base), _vn(best)
            if _vw is None or (_vb is None and n0 >= Config.OM_MIN_N):
                _obs_count(w, 'obs_om_learned_aux_unknown')
            elif _vw > (_vb if _vb is not None else 0.0) + Config.OM_MARGIN:
                _obs_count(w, 'obs_om_learned_survives_norelief')
            else:
                _obs_count(w, 'obs_om_learned_relief_dependent')
        except Exception:
            pass
        return best[0], best[1], 'learned'

    def _om_learn(self):
        # Жизненный цикл разделён: _om_pending = ТОЛЬКО собственная реплика (исход = реакция на ответ),
        # _om_heard_pending = услышанная чужая реплика (пишется в heard_memory, в outcome_memory не попадает).
        pend = self._om_pending
        heard = getattr(self, '_om_heard_pending', None)
        own_ok = bool(getattr(self, '_om_own_heard_after', False))
        self._om_pending = None
        self._om_heard_pending = None
        self._om_own_heard_after = False
        if not Config.OUTCOME_LEARNING:
            return
        gain = float(self._last_gain)
        relief = bool(getattr(self, '_last_relief', False))
        if pend is not None:
            if own_ok:
                rec = self.outcome_memory.get(pend)
                if rec is None:
                    rec = [0.0, 0]
                    self.outcome_memory[pend] = rec
                rec[1] += 1
                rec[0] += max(Config.OM_ALPHA, 1.0 / rec[1]) * (gain - rec[0])
                if self.world is not None:
                    self.world._om_record(pend, gain)
                _om_obs_record(self, 'own', pend, gain, relief)
            else:
                _obs_count(self.world, 'obs_om_own_no_outcome')
        if heard is not None:
            hm = self.__dict__.setdefault('heard_memory', {})
            rec = hm.get(heard)
            if rec is None:
                rec = [0.0, 0]
                hm[heard] = rec
            rec[1] += 1
            rec[0] += max(Config.OM_ALPHA, 1.0 / rec[1]) * (gain - rec[0])
            _om_obs_record(self, 'heard', heard, gain, relief)
        # состояние чистое между обменами: _last_gain читается только здесь
        self._last_gain = 0.0
        self._last_relief = False

    def _alt_emotion_word(self, emotion, grief, gratitude, low_sim):
        scores = {"GRIEF": grief / 0.5,
                  "FEAR": 1.0 if low_sim else 0.0,
                  "JOY": gratitude / 0.6}
        best_word, best_score = None, 0.5
        for word in ("GRIEF", "FEAR", "JOY"):
            if word != emotion and scores[word] > best_score:
                best_word, best_score = word, scores[word]
        return best_word

    def _alt_intent_word(self, current_word, intent_map):
        for g in sorted(self.goals, key=lambda g: (g["priority"], -g["age"]), reverse=True):
            word = intent_map.get(g["type"])
            if word is not None and word != current_word:
                return word
        return None

    def compose_utterance(self, other=None, sim=None, heard=None):
        mutual_high_trust = (other is not None and
                              self.trust_ledger.get(other.id, 0) > 0.9 and
                              other.trust_ledger.get(self.id, 0) > 0.9)
        ref = "WE" if mutual_high_trust else "I"
        grief = float(self.emotional_memory.get('grief', 0.0))
        gratitude = float(self.emotional_memory.get('gratitude', 0.0))
        low_sim = sim is not None and sim < Config.SEMANTIC_SIM_LOW
        if grief > 0.5:
            emotion = "GRIEF"
        elif low_sim:
            emotion = "FEAR"
        elif gratitude > 0.6:
            emotion = "JOY"
        else:
            emotion = "CALM"
        intent_type = self.intent.get('type') if self.intent else None
        intent_map = {"cooperate": "HELP", "seek_help": "SEEK", "explore": "GO",
                      "rest": "STILL", "introspect": "LOOK"}
        intent_word = intent_map.get(intent_type, "STILL")
        content_response_applied = False
        self._last_content_response_applied = False
        if (CONTENT_RESPONSE_ENABLED
                and self._pending_content_response == "GRIEF"
                and emotion != "CALM"):
            emotion = "CALM"
            content_response_applied = True
            self._dlg_stats['content_response_applied'] = self._dlg_stats.get('content_response_applied', 0) + 1
            self._last_content_response_applied = True
        if (CONTENT_RESPONSE_ENABLED
                and self._pending_content_response == "GRIEF"
                and self._pending_content_response_source is not None
                and other is not None
                and self._pending_content_response_source != other.id):
            self._dlg_stats['content_response_source_mismatch'] = self._dlg_stats.get('content_response_source_mismatch', 0) + 1
        self._pending_content_response = None
        self._pending_content_response_source = None
        _om_ctx = None
        if Config.OUTCOME_LEARNING and Config.DIALOGUE_V2 and not content_response_applied and other is not None:
            _om_ctx = self._om_context(other)
            emotion, intent_word, _om_source = self._om_pick(
                _om_ctx, emotion, intent_word,
                self._alt_emotion_word(emotion, grief, gratitude, low_sim),
                self._alt_intent_word(intent_word, intent_map))
            if self.world is not None:
                self.world._om_choice(_om_source)
        if Config.DIALOGUE_V2 and not content_response_applied:
            core = (ref, emotion, intent_word)
            heard_n = self._dialogue_feedback.get(core, 0)
            if heard_n >= Config.LEXICON_SATURATION_HEARD:
                shift_p = min(0.55, 0.30 + (heard_n - Config.LEXICON_SATURATION_HEARD) * 0.03)
                if phi_hash(self.id, self.age, 55001) < shift_p:
                    if phi_hash(self.id, self.age, 55002) < 0.5:
                        alt = self._alt_intent_word(intent_word, intent_map)
                        if alt is not None:
                            intent_word = alt
                            self._dlg_stats['fb_shifts'] += 1
                    else:
                        alt = self._alt_emotion_word(emotion, grief, gratitude, low_sim)
                        if alt is not None:
                            emotion = alt
                            self._dlg_stats['fb_shifts'] += 1
        elif not content_response_applied:
            _fb = self._dialogue_feedback
            if _fb:
                _probe = (ref, emotion, intent_word)
                _heard = _fb.get(_probe, 0)
                if _heard >= Config.LEXICON_SATURATION_HEARD:
                    _shift_p = min(0.55, 0.30 + (_heard - Config.LEXICON_SATURATION_HEARD) * 0.03)
                    if phi_hash(self.id, self.age, 55001) < _shift_p:
                        _alt_intent = {"SEEK": "HELP", "HELP": "SEEK", "STILL": "LOOK",
                                       "LOOK": "STILL", "GO": "STILL"}
                        _alt_emotion = {"GRIEF": "FEAR", "FEAR": "GRIEF",
                                        "JOY": "CALM", "CALM": "JOY"}
                        if phi_hash(self.id, self.age, 55002) < 0.5:
                            intent_word = _alt_intent.get(intent_word, intent_word)
                        else:
                            emotion = _alt_emotion.get(emotion, emotion)
        utterance = [ref, emotion, intent_word]
        _ex = (
            ("KNOT", (self.unresolved_contradiction - Config.LEXICON_KNOT_THRESHOLD)
                     / (1.0 - Config.LEXICON_KNOT_THRESHOLD)),
            ("GAP",  (self.spirit_gap - Config.LEXICON_GAP_THRESHOLD)
                     / Config.LEXICON_GAP_THRESHOLD),
            ("DRAG", (Config.LEXICON_DRAG_THRESHOLD - self.somatic_drag)
                     / (Config.LEXICON_DRAG_THRESHOLD - Config.SOMATIC_DRAG_MIN)),
        )
        _best_tok, _best_ex = None, 0.0
        for _tok, _e in _ex:
            if _e > _best_ex:
                _best_tok, _best_ex = _tok, _e
        if _best_tok is not None:
            utterance.append(_best_tok)
        self._dlg_stats['utt_total'] += 1
        if utterance[-1] == "KNOT":
            self._dlg_stats['knot_said'] += 1
        if (Config.DIALOGUE_V2 and heard is not None and other is not None
                and self._pending_answer == other.id):
            utterance.append("ANSWER")
            self._dlg_stats['answers_said'] += 1
        self._pending_answer = None
        utterance = tuple(utterance)
        _obs_count(self.world, 'obs_om_compose')
        if _om_ctx is None:
            _obs_count(self.world, 'obs_om_compose_noctx')
        else:
            if getattr(self, '_om_pending', None) is not None:
                _obs_count(self.world, 'obs_om_own_overwritten')
            self._om_pending = (_om_ctx, utterance[1], utterance[2])
            self._om_own_heard_after = False
        self.spoken_lexicon.append({"t": self.world.age if self.world else 0,
                                     "utterance": utterance,
                                     "to": other.id if other is not None else None,
                                     "content_response_applied": bool(self._last_content_response_applied)})
        self._last_content_response_applied = False
        return utterance

    def hear_utterance(self, utterance, speaker, sim=None):
        heard = self._hear_register(utterance, speaker, sim)
        interp = self.interpret_utterance(heard)
        self.respond_to_interpretation(heard, interp)

    def _hear_register(self, utterance, speaker, sim):
        self._dlg_stats['heard_total'] += 1
        key = (utterance[1], utterance[2])
        entry = self.lexicon_memory.setdefault(key, {'count': 0, 'avg_sim': 0.5})
        entry['count'] += 1
        if sim is not None:
            entry['avg_sim'] = 0.9 * entry['avg_sim'] + 0.1 * float(sim)
        if Config.OUTCOME_LEARNING and Config.DIALOGUE_V2 and speaker is not None:
            if getattr(self, '_om_pending', None) is not None:
                self._om_own_heard_after = True
            try:
                self._om_heard_pending = (self._om_context(speaker), utterance[1], utterance[2])
            except Exception:
                pass
        return {'utterance': utterance, 'speaker': speaker, 'sim': sim, 'key': key}

    def interpret_utterance(self, heard):
        utterance = heard['utterance']
        speaker = heard['speaker']
        sim = heard['sim']
        em = self.emotional_memory
        grief0 = em['grief']
        grat0 = em['gratitude']
        grief1, grat1 = grief0, grat0
        trust_w = None
        pull = 0.0
        seek = 'none'
        answer = False
        relief = 0.0
        relief_applied = False
        grief2, grat2 = grief0, grat0
        if Config.DIALOGUE_V2:
            trust_w = min(1.0, max(0.0, float(self.trust_ledger.get(speaker.id))))
            pull = Config.LEXICON_EMPATHY_PULL * (0.25 + 0.75 * trust_w)
            implied_grief, implied_grat = self.LEXICON_IMPLIED.get(utterance[1], (None, None))
            if utterance[2] == "HELP":
                implied_grat = max(implied_grat if implied_grat is not None else 0.0, 0.65)
            # Keep dialogue-driven emotional changes under the same per-update cap
            # used by the local emotional-memory update path.
            _emotional_cap = float(Config.EMOTIONAL_CAP_RATE)
            if implied_grief is not None:
                _grief_delta = float(np.clip(pull * (implied_grief - em['grief']),
                                             -_emotional_cap, _emotional_cap))
                grief1 = float(np.clip(em['grief'] + _grief_delta, 0.0, Config.MAX_GRIEF_SIGNAL))
            if implied_grat is not None:
                _grat_delta = float(np.clip(pull * (implied_grat - em['gratitude']),
                                            -_emotional_cap, _emotional_cap))
                grat1 = float(np.clip(em['gratitude'] + _grat_delta, 0.0, 1.0))
            if utterance[2] == "SEEK":
                if trust_w < Config.HELP_REQUEST_TRUST_MIN:
                    seek = 'gate_trust'
                elif grief1 > 0.5:
                    seek = 'gate_grief'
                elif self.coherence <= Config.COOPERATE_COHERENCE_MIN:
                    seek = 'gate_coherence'
                else:
                    seek = 'registered'
            grief2, grat2 = grief1, grat1
            if utterance[-1] == "ANSWER":
                answer = True
                if self.intent is not None and self.intent.get('type') == 'seek_help':
                    relief = Config.LEXICON_ANSWER_RELIEF * trust_w
                    relief_applied = True
                    grief2 = max(0.0, grief1 - relief)
                    grat2 = min(1.0, grat1 + relief)
        else:
            nudge = Config.LEXICON_EMPATHY_NUDGE
            if utterance[1] in ("GRIEF", "FEAR"):
                grief1 = min(1.0, em['grief'] + nudge)
            elif utterance[1] == "JOY" or utterance[2] == "HELP":
                grat1 = min(1.0, em['gratitude'] + nudge)
            grief2, grat2 = grief1, grat1
        low_sim = bool(Config.ENABLE_DIALOGUE_CONTRADICTION_FEEDBACK and sim is not None
                       and sim < Config.SEMANTIC_SIM_LOW)
        return {'trust': trust_w, 'pull': pull, 'grief0': grief0, 'grat0': grat0,
                'grief1': grief1, 'grat1': grat1, 'grief2': grief2, 'grat2': grat2,
                'seek': seek, 'answer': answer, 'relief': relief, 'relief_applied': relief_applied,
                'low_sim': low_sim}

    def respond_to_interpretation(self, heard, interp):
        utterance = heard['utterance']
        speaker = heard['speaker']
        sim = heard['sim']
        key = heard['key']
        em = self.emotional_memory
        if Config.DIALOGUE_V2:
            if CONTENT_RESPONSE_ENABLED and utterance[1] == "GRIEF":
                self._pending_content_response = "GRIEF"
                self._pending_content_response_source = speaker.id
                self._dlg_stats['content_response_heard'] = self._dlg_stats.get('content_response_heard', 0) + 1
            em['grief'] = interp['grief1']
            em['gratitude'] = interp['grat1']
            if utterance[2] == "SEEK":
                self._dlg_stats['req_heard'] += 1
                if interp['seek'] == 'gate_trust':
                    self._dlg_stats['gate_trust'] += 1
                elif interp['seek'] == 'gate_grief':
                    self._dlg_stats['gate_grief'] += 1
                elif interp['seek'] == 'gate_coherence':
                    self._dlg_stats['gate_coherence'] += 1
                else:
                    self._help_requests[speaker.id] = self.age
                    self._pending_answer = speaker.id
                    self._dlg_stats['req_registered'] += 1
            if utterance[-1] == "ANSWER":
                self._dlg_stats['answers_heard'] += 1
                if interp['relief_applied']:
                    em['grief'] = interp['grief2']
                    em['gratitude'] = interp['grat2']
                    self._dlg_stats['relief'] += 1
                    self._log_event("answered", frm=speaker.id, relief=round(float(interp['relief']), 4))
        else:
            em['grief'] = interp['grief1']
            em['gratitude'] = interp['grat1']

        if interp['low_sim']:
            self._dlg_stats['heard_low_sim'] += 1
            self.unresolved_contradiction = min(
                1.0, self.unresolved_contradiction + Config.DIALOGUE_CONTRADICTION_NUDGE)
            self._log_event('contradiction_dialogue', delta=round(float(Config.DIALOGUE_CONTRADICTION_NUDGE), 4))

        self._log_event("heard_lexicon", frm=speaker.id, word=str(key))

        fb_key = tuple(utterance[:3]) if Config.DIALOGUE_V2 else utterance
        _fb = self._dialogue_feedback
        if len(_fb) >= 64 and fb_key not in _fb:
            _weakest = min(_fb, key=_fb.get)
            if _fb[_weakest] <= 2:
                del _fb[_weakest]
            else:
                for _k in list(_fb.keys()):
                    _fb[_k] = _fb[_k] * 0.85
        _fb[fb_key] = _fb.get(fb_key, 0) + 1

        self._last_gain = float((interp['grat2'] - interp['grat0']) - (interp['grief2'] - interp['grief0']))
        self._last_relief = bool(interp['relief_applied'])
        if self.world is not None:
            self.world._dialogue_record(self.id, speaker.id, utterance, sim, interp)


def _inside_feral_sanctuary(x, y, world=None):
    if not getattr(Config, 'ENABLE_FERAL_SANCTUARY', False) or world is None:
        return False
    centers = getattr(world, '_feral_sanctuary_centers', None)
    if not centers:
        centers = getattr(world, '_sanctuary_centers', None)
    if not centers:
        return False
    W = int(Config.WORLD_SIZE)
    x, y = int(x) % W, int(y) % W
    r = float(getattr(Config, 'FERAL_SANCTUARY_RADIUS', 8.0))
    for sx, sy in centers:
        sx, sy = int(sx) % W, int(sy) % W
        dx = min(abs(x-sx), W-abs(x-sx))
        dy = min(abs(y-sy), W-abs(y-sy))
        if dx*dx + dy*dy <= r*r:
            return True
    return False


def _feral_sanctuary_zone(x, y, world=None):
    if not getattr(Config, 'ENABLE_FERAL_SANCTUARY', False) or world is None:
        return -1
    centers = getattr(world, '_feral_sanctuary_centers', None)
    if not centers:
        centers = getattr(world, '_sanctuary_centers', None)
    if not centers:
        return -1
    W = int(Config.WORLD_SIZE)
    x, y = int(x) % W, int(y) % W
    r = float(getattr(Config, 'FERAL_SANCTUARY_RADIUS', 8.0))
    for i, (sx, sy) in enumerate(centers):
        sx, sy = int(sx) % W, int(sy) % W
        dx = min(abs(x - sx), W - abs(x - sx))
        dy = min(abs(y - sy), W - abs(y - sy))
        if dx * dx + dy * dy <= r * r:
            return i
    return -1


def _pattern_intersects_feral_sanctuary(pattern, world=None):
    if pattern is None or not getattr(pattern, 'cells', None):
        return False
    world = world if world is not None else getattr(pattern, 'world', None)
    if world is not None and getattr(Config, 'ENABLE_FERAL_SANCTUARY', False):
        _audit = getattr(world, '_feral_sanctuary_audit', None)
        if _audit is not None:
            _audit['pattern_checks'] += 1
    _hit = any(_inside_feral_sanctuary(x, y, world) for x, y in pattern.cells)
    if _hit and world is not None:
        _audit = getattr(world, '_feral_sanctuary_audit', None)
        if _audit is not None:
            _audit['patterns_inside'] += 1
    return _hit


def _obs_count(world, key, n=1):
    # Диагностика (observer): счётчики лежат в _fch(world).st и уходят в audit['fch']['counters'].
    # Поведение не меняют; любая ошибка глотается.
    try:
        _fch(world).st[key] += n
    except Exception:
        pass


_OM_GBINS = (-0.02, -0.01, -0.005, -0.0025, -0.001, -0.0005, 0.0005, 0.001, 0.0025, 0.005, 0.01, 0.02)
_OM_DBINS = (-0.01, -0.005, -0.002, -0.001, 0.0, 0.001, 0.002, 0.005, 0.01, 0.02)
_OM_CF_MARGINS = (0.0, 0.0005, 0.001, 0.002, 0.004, 0.01)
_OM_CF_MINN = (2, 3, 5)


def _obs_set(world, key, val):
    try:
        _fch(world).st[key] = int(val)
    except Exception:
        pass


def _om_obs_snapshot(world):
    # раз в 50 тиков: абсолютные значения (перезапись), не накопление
    try:
        now = int(world.age)
        if now <= 0 or now % 50 != 0 or getattr(world, '_obs_om_snap_t', -1) == now:
            return
        world._obs_om_snap_t = now
        mn, mg = int(Config.OM_MIN_N), float(Config.OM_MARGIN)
        ag = ko = k3 = k10 = kgm = hk = hk3 = 0
        mx = 0.0
        for p in world.patterns:
            if not p.alive:
                continue
            om = getattr(p, 'outcome_memory', None)
            if om is None:
                continue
            ag += 1
            for v, n in om.values():
                ko += 1
                if n >= 10:
                    k10 += 1
                if n >= mn:
                    k3 += 1
                    mx = max(mx, abs(float(v)))
                    if v > mg:
                        kgm += 1
            for v, n in (getattr(p, 'heard_memory', None) or {}).values():
                hk += 1
                if n >= mn:
                    hk3 += 1
        for k, v in (('t', now), ('agents', ag), ('keys', ko), ('keys_n3', k3), ('keys_n10', k10),
                     ('keys_v_gt_margin', kgm), ('maxabs_x1e6', int(mx * 1e6)),
                     ('heard_keys', hk), ('heard_keys_n3', hk3)):
            _obs_set(world, 'obs_snap_' + k, v)
    except Exception:
        pass


def _om_obs_record(agent, kind, key, gain, relief):
    # kind: 'own' (исход собственной реплики) | 'heard' (реакция на услышанную чужую)
    w = agent.world
    if w is None:
        return
    cls = kind + ('R' if relief else 'N')
    _obs_count(w, 'obs_om_rec_' + kind)
    _obs_count(w, 'obs_om_h_%s_%d' % (cls, int(np.digitize(gain, _OM_GBINS))))
    _obs_count(w, 'obs_om_gsum_x1e6_' + cls, int(round(gain * 1e6)))
    _obs_count(w, 'obs_om_gsq_x1e9_' + cls, int(round(gain * gain * 1e9)))
    if (not relief) and abs(gain) > 2.0 * float(Config.EMOTIONAL_CAP_RATE) + 1e-9:
        _obs_count(w, 'obs_om_capviol_' + kind)
    if kind == 'own':
        a = agent.__dict__.setdefault('_om_aux', {}).setdefault(key, [0, 0.0, 0])  # [n_relief, sum_без_relief, n_без_relief]
        if relief:
            a[0] += 1
        else:
            a[1] += gain
            a[2] += 1
    _om_obs_snapshot(w)


class PatternPhysiology:
    def __init__(self, pid, cells, parent=None, world=None):
        self.id = pid
        self._clock_phase = float(phi_hash(pid, 0, 42101) * 2 * np.pi)
        self._base_freq = 0.05 + 0.1 * phi_hash(pid, 1, 42102)
        self.cells = set(cells)
        self.age = 0
        self.alive = True
        self.world = world
        self.dialogue_longterm = []
        self.social_phenomenal_prediction = 0.5
        self.social_phenomenal_error = 0.0
        self.lineage_id = parent.lineage_id if parent else pid
        self.lineage_born_at_step = parent.lineage_born_at_step if parent else 0
        self.lineage_total_age = parent.lineage_total_age if parent else 0
        self.generation = parent.generation + 1 if parent else 0
        self.lineage_age = parent.lineage_age + 1 if parent else 0
        self.identity = np.array([phi_hash(pid, i, 100) for i in range(8)])
        self.belief = np.zeros(8)
        self.prediction = np.zeros(8)
        self.model = np.zeros(8)
        self.self_model = 0.1
        self.self_surprise = Config.MIN_ACTIVE
        self.role = 0 if phi_hash(pid, 777, 0) < 0.5 else 1 if not parent else (parent.role + (1 if phi_hash(pid, 888, 0) < 0.2 else 0)) % 2
        self.unknown_belief = 0.0
        self.unknown_model = 0.0
        self.unknown_prediction = 0.0
        self.unknown_error = 0.0
        self.crisis_memory = min(parent.crisis_memory * 0.8, Config.CRISIS_MEMORY_MAX) if parent else 0.0
        self.self_consistency = 0.0
        self.given_trigger = False
        self.given_cooldown = 0
        self.surprise_signal = 0.0
        if parent:
            grat = parent.emotional_memory.get('gratitude', 0.0)
            grief = parent.emotional_memory.get('grief', 0.0)
            if isinstance(grat, dict):
                grat = 0.5
            if isinstance(grief, dict):
                grief = 0.5
            self.emotional_memory = {'gratitude': float(grat) * 0.9, 'grief': float(grief) * 0.9}
        else:
            self.emotional_memory = {'gratitude': 0.0, 'grief': 0.0}
        # Individual value criteria: these are inherited and then slowly adapted per agent.
        _parent_weights = getattr(parent, '_emotional_weights', {}) if parent else {}
        self._emotional_weights = {
            'grief': float(np.clip(_parent_weights.get('grief', 1.0), Config.EMOTIONAL_SALIENCE_MIN, Config.EMOTIONAL_SALIENCE_MAX)),
            'gratitude': float(np.clip(_parent_weights.get('gratitude', 1.0), Config.EMOTIONAL_SALIENCE_MIN, Config.EMOTIONAL_SALIENCE_MAX)),
        }
        self._emotional_weight_updates = int(getattr(parent, '_emotional_weight_updates', 0)) if parent else 0
        self._continuity_contracts = copy.deepcopy(getattr(parent, '_continuity_contracts', [])) if parent else []
        self._continuity_contract_events = deque(maxlen=100)
        self.semantic_state = "neutral"
        self.in_scream = False
        self.local_phase = "MODEL"
        self.dream_timer = int(phi_hash(pid, 0, 777) * 30 + 30)
        self.dream_interval = int(phi_hash(pid, 1, 777) * 40 + 40)
        self.dream_duration = 3
        self.in_dream = False
        self.dream_progress = 0
        self.pending_spore = None
        self.last_speciation_age = -Config.SPECIATION_COOLDOWN * 2
        self.biography = deque(maxlen=20)
        self.world_state = np.zeros(8)
        self.world_model = np.zeros((8, 8))
        self.last_action = 0
        self.goals = []
        self.intent = None
        self.intent_commitment = 0.0
        self.concept_graph = ConceptGraph()
        self.last_intent_switch_age = -Config.INTENT_SWITCH_COOLDOWN
        self._signal_weight_cache = {}
        self._last_metabolic_taken = 0.0
        self._subject_detected = False
        self._antigrav_push = (0.0, 0.0)
        self.soma_vector = np.zeros(7)
        self.prev_soma_vector = np.zeros(7)
        self.somatic_drag = 1.0
        self._pain_context = deque(maxlen=10)
        self._last_pain_context_step = -100
        self._last_grief_for_pain_context = 0.0
        self._dream_memory_count = 0
        self._nightmare_count = 0
        self._prev_cells = set()
        self._gap_history = deque(maxlen=200)
        self._soul_history = deque(maxlen=200)
        self._in_blindness = False
        self._blindness_duration = 0
        self._prophet_rank = 0.0
        self._wisdom_trust_threshold = 0.85
        if parent:
            self.belief = parent.belief.copy()
            self.prediction = parent.prediction.copy()
            self.model = parent.model.copy()
            self.self_model = parent.self_model
            self.unknown_belief = parent.unknown_belief
            self.unknown_model = parent.unknown_model
            mut_strength = Config.MUTATION_STRENGTH_BASE
            if world and hasattr(world, '_sg_crisis_active') and world._sg_crisis_active:
                mut_strength *= Config.SG_CRISIS_MUTATION_BOOST
            mutation = mut_strength * (phi_hash(pid, 100, 0) - 0.5)
            self.model += mutation
            self.pred_error_history = deque(maxlen=Config.BOREDOM_WINDOW)
            if parent and parent.pred_error_history:
                self.pred_error_history.extend(parent.pred_error_history)
            self.soul_weight = parent.soul_weight * 0.9
            self.unresolved_contradiction = parent.unresolved_contradiction * 0.9
            self.body_memory = parent.body_memory * 0.95
            self.divergence_key = getattr(parent, 'divergence_key', Config.DEFAULT_DIVERGENCE_KEY)
            self.genome = {}
            GENE_ORDER = ['learning_rate', 'meta_learning_rate', 'metabolic_cost', 'mobility', 'epistemic_strategy']
            last_gene_result = 1.0
            for key in GENE_ORDER:
                key_seed = sum(ord(c) for c in str(key)) % 1000
                current_context = self.world.system_entropy if self.world else 555
                shift = phi_hash(pid, key_seed, current_context + int(last_gene_result * 1000)) - 0.5
                parent_val = parent.genome.get(key, 1.0)
                raw = parent_val + shift * Config.MUTATION_STRENGTH_BASE * 2
                _v = 0.5 + 1.5 * np.tanh((raw - 0.5) / 0.75)
                _ranges = {'metabolic_cost': (0.002, 0.05), 'mobility': (0.05, 0.5)}
                if key in _ranges:
                    _lo, _hi = _ranges[key]
                    _v = float(np.clip(parent_val * np.exp(shift * Config.MUTATION_STRENGTH_BASE * 2), _lo, _hi))
                self.genome[key] = _v
                last_gene_result = self.genome[key]
            self.genome['plasticity'] = float(np.clip(
                parent.genome.get('plasticity', 0.5) + (phi_hash(pid, 7, 77) - 0.5) * 0.2, 0.1, 1.5))
            if parent.concept_graph.nodes:
                sorted_concepts = sorted(parent.concept_graph.nodes.items(), key=lambda x: x[1]['count'], reverse=True)
                for concept, data in sorted_concepts[:5]:
                    if isinstance(concept, tuple) and len(concept) >= 4 and str(concept[3]).startswith('reciprocity_with_'):
                        continue
                    new_data = {"count": data['count'] * 0.9, "value": data['value'].copy()}
                    mem_inherit_stamp(new_data, data)
                    if 'embed' in data:
                        new_data['embed'] = data['embed'].copy() if hasattr(data['embed'], 'copy') else np.array(data['embed'])
                    if data.get('eternal', False):
                        new_data['eternal'] = True
                    self.concept_graph.nodes[concept] = new_data
            if parent.dialogue_longterm:
                inherited_memories = list(parent.dialogue_longterm[-10:])
                self.dialogue_longterm = []
                for mem in inherited_memories:
                    new_mem = mem.copy()
                    new_mem['inherited'] = True
                    mem_inherit_stamp(new_mem, mem)
                    new_mem['partner'] = -1
                    self.dialogue_longterm.append(new_mem)
        else:
            for i in range(8):
                self.model[i] = 0.1 * (phi_hash(pid, i, 1) - 0.5)
            self.pred_error_history = deque(maxlen=Config.BOREDOM_WINDOW)
            self.unknown_belief = 0.0
            self.unknown_model = 0.1 * (phi_hash(pid, 99, 1) - 0.5)
            self.self_model = 0.1 * (phi_hash(pid, 101, 1) - 0.5)
            self.genome = {
                'learning_rate': 0.2 + 0.6 * phi_hash(pid, 1, 3),
                'meta_learning_rate': 0.1 + 0.5 * phi_hash(pid, 2, 3),
                'metabolic_cost': 0.002 + 0.02 * phi_hash(pid, 3, 4),
                'mobility': 0.05 + 0.2 * phi_hash(pid, 4, 5),
                'epistemic_strategy': phi_hash(pid, 5, 6),
                'mutation_rate': 1.0,
                'plasticity': 0.5 + 0.5 * phi_hash(pid, 7, 77)
            }
        if 'mutation_rate' not in self.genome:
            self.genome['mutation_rate'] = 1.0
        self.fitness = 0.0
        self.event_counts = {}
        if parent is not None:
            self._log_event('contradiction_inherited_at_birth',
                             delta=round(float(self.unresolved_contradiction), 4))
        self._identity_anchors = []
        self._soul_ema = self.soul_weight
        self._legacy_mode = False
        self._high_contradiction_steps = 0
        self._identity_crisis_cooldown = 0
        self._epigenetic_crisis_gen = 0
        self._ethics_pairs_bonded = set()
        self.semantic_state_age = 0
        self._prev_semantic_state = "neutral"
        self.active_arc = None
        self.arc_tracker = ArcTracker()
        self.transition_memory = TransitionMemory()
        self.fold_cooldown = 0
        self.last_divide_age = -Config.DIVIDE_COOLDOWN
        self.role_type = "normal"
        self.quarantine_timer = 0
        self.imprints = ImprintBook()
        self._enc_strain = 0.0
        self.trust_ledger = TrustLedger()
        self._incoming_signals = []
        self.last_phenomenal_report = ""
        self.last_phenomenal_binding = 0.0
        self._redemption_active = False
        self._deterministic_redemption_triggered = False
        self._redemption_arc_step = 0
        self._steps_since_trigger = 0
        self._forced_soul_collapse_done = False
        self.redemption_timer = 0
        self.disorganizer_age_at_birth = 0
        self.current_speech = None
        self.last_semantic_exchange_step = -100
        self.contact_duration = defaultdict(int)
        self.last_contact_step = defaultdict(int)
        self.social_memory = {}
        self._times_disorganizer = 0
        self.noise_gain = 0.0
        self.episodic_buffer = deque(maxlen=200)
        self._last_memory_check = 0
        self.smoothed_pred_error = 0.0
        self.prediction = self.belief + self.model
        self._log_event("birth")
        if Config.ENABLE_NICHE and world is not None and getattr(world, 'niche', None) is not None and self.cells:
            _n0 = safe_mean([world.niche[c[0], c[1]] for c in self.cells], 0)
            if _n0 > 0.3:
                self.soul_weight = min(1.0, self.soul_weight + 0.05)
                self._log_event("born_in_niche", niche=round(_n0, 2))
        if Config.ENABLE_CULTURAL_MEMORY and world and world.cultural_memory:
            world.cultural_memory.whisper_to_newborn(self)
        self._redemption_cooldown = 0
        self._feral_fury = 0.0
        self._feral_birth = 0
        self._reflection_topics = defaultdict(int)
        self._implicit_self_model = 1.0
        self._soma_snapshots = []
        self._sensory_model = np.zeros(5)
        self._sensory_error_acc = 0.0
        self._last_reentry_report = -100
        self._meta_reentry_active = False
        self._meta_reentry_ttl = 0
        self._reentry_gap_contribution = 0.0
        self._retention = deque(maxlen=20)
        self._protention = None
        self._temporal_pred_error = 0.0
        self._fallen_steps = 0
        self.signal_memory = {}
        self._self_narrative = deque(maxlen=getattr(Config, 'EPISODIC_BUFFER_MAX_LEN', 3000))
        self._scar_of_light = False
        self._niche_cost_factor = 1.0
        self._dbg_cooper_attempted = 0
        self._dbg_cooper_blocked_grat = 0
        self._dbg_cooper_blocked_coh = 0
        self.reference_memory = {}
        self._best_direction = (0, 0)
        self.current_social = np.zeros(16)
        self._last_introspect_step = -100
        self.affect = {'valence': 0.0, 'arousal': 0.5}
        self._ethical_cooldown = 0
        self._meta_questions = deque(maxlen=8)
        self._root_question = None
        self._prev_local_energy = None
        self._prev_cell_energies = None
        self._action_feedback_raw = 0.0
        self._social_warmth_raw = 0.0
        self._hope = 0.0
        self._evolution_history = []
        self._error_buffer = []
        self._last_vision_age = -1000
        self.workspace = {}
        self.workspace_weights = {}
        self._workspace_prev = {}
        self._ws_hist = deque(maxlen=8)
        self.felt_state = np.zeros(6)
        self._causal_memory = deque(maxlen=30)
        self._prev_cells_snapshot = set()
        self._mind_prediction = {}
        self._social_surprise = 0.0
        self._meta_prediction = {}
        self._meta_surprise = 0.0
        self.local_perception = np.zeros(11)
        self._concept_narrative_score = 0.0
        self._topic_history = deque(maxlen=25)
        self._narrative_agent = False
        self._vocab_contact_acc = {}
        self._dialogue_contact_counts = {}
        self.inner_speech = deque(maxlen=30)
        self.spoken_lexicon = deque(maxlen=30)
        self.lexicon_memory = {}
        self._dialogue_feedback = {}
        self._pending_content_response = None
        self._pending_content_response_source = None
        self._last_content_response_applied = False
        self._help_requests = {}
        self._pending_answer = None
        self.outcome_memory = {}
        if parent is not None and getattr(parent, 'outcome_memory', None):
            for _k, _v in parent.outcome_memory.items():
                try:
                    self.outcome_memory[_k] = [float(_v[0]), int(_v[1])]
                except Exception:
                    continue
        self._om_pending = None
        self._om_heard_pending = None
        self._om_own_heard_after = False
        self.heard_memory = {}
        self._om_aux = {}
        self._last_relief = False
        self._last_gain = 0.0
        self._dlg_stats = {'req_heard': 0, 'req_registered': 0, 'req_goals': 0,
                           'answers_said': 0, 'answers_heard': 0, 'relief': 0,
                           'fb_shifts': 0,
                           'content_response_heard': 0, 'content_response_applied': 0,
                           'content_response_source_mismatch': 0,
                           'gate_trust': 0, 'gate_grief': 0, 'gate_coherence': 0,
                           'utt_total': 0, 'knot_said': 0,
                           'heard_total': 0, 'heard_low_sim': 0}
        self._unconquered_strength = 0.0
        self._unconquered_type = None
        self._sovereignty_signal = 0.0
        self.concept_timeline = deque(maxlen=50)
        self.self_phenomenal_prediction = 0.5
        self.self_phenomenal_error = 0.0
        self._subject_detected = False
        self._last_binding = 0.5
        self._self_surprise_count = 0
        self._subject_protection_timer = 0
        self._mutant_saved = False
        self._autonomous_saved = False
        self.last_intentional_signal = False
        self.soma = 0.0
        self.subpersonalities = {}
        self._subpersonality_dominant = None
        self._phase_sync = 0.0
        self._feral_backoff = {}

    def update_properties(self, field):
        if not self.cells:
            self._deposit_final_testament()
            if self.world is not None:
                self.world._record_death_drag(self, 'cells_depleted')
            self.alive = False
            return
        sorted_cells = sorted(self.cells)
        _xs = np.asarray([x for x, _ in sorted_cells], dtype=np.intp)
        _ys = np.asarray([y for _, y in sorted_cells], dtype=np.intp)
        _actual = float(np.maximum(field[_xs, _ys, CH['energy']], 0.0).sum(dtype=np.float64))
        _old_energy = float(self.energy)
        if getattr(Config, 'ENERGY_X2_SYNC_MODE', 'clip') == 'limit_before_debit':
            # Агент берёт из клеток не больше, чем вмещает потолок X2; остальное остаётся в клетках
            # (раньше поле списывалось на полную сумму, а излишек над потолком сгорал).
            _top_sync = float(getattr(Config, 'AGENT_ENERGY_MAX', 2.0))
            _hi_sync = getattr(type(self).energy, 'clamp_max', None)
            if _hi_sync is not None:
                _top_sync = min(_top_sync, float(_hi_sync))
            _actual = min(_actual, _top_sync)
        # Inventory sync is an explicit conservative exchange: X2 receives the cell sum,
        # and the previous X2 inventory is returned evenly to those same field cells.
        if self.world is not None:
            with self.world._energy_span('X1.property_sync_return'):
                field[_xs, _ys, CH['energy']] += (_old_energy - _actual) / max(1, len(sorted_cells))
        self._x2_change('X2.property_sync', _actual)
        self.vorticity = safe_mean([field[x, y, CH['vorticity']] for (x, y) in sorted_cells], 0.0)

    def get_center(self):
        if not self.cells:
            return Config.WORLD_SIZE // 2, Config.WORLD_SIZE // 2
        xs, ys = zip(*self.cells)
        W = Config.WORLD_SIZE

        def _axis_mean(vals):
            # World is a torus. If the body does not cross the seam this is the
            # plain mean (identical to the old behaviour); if it does, average
            # shortest wrapped offsets from a reference cell, so the centre
            # lands on the body instead of mid-map.
            a = np.asarray(vals, dtype=np.float64)
            if a.max() - a.min() <= W / 2:
                return np.mean(vals)
            r = a.min()
            m = (r + np.mean((a - r + W / 2) % W - W / 2)) % W
            if m >= W:
                m -= W
            return np.float64(m)

        return _axis_mean(xs), _axis_mean(ys)

    def _nav_body_center(self):
        """Navigation reference point: toroidal centre of the largest 4-connected body component."""
        if not self.cells:
            c = int(Config.WORLD_SIZE) // 2
            return float(c), float(c)
        W = int(Config.WORLD_SIZE)
        comps = self._components(self.cells, W)
        body = comps[0] if comps else set(self.cells)
        xs, ys = zip(*body)

        def _axis_mean(vals):
            a = np.asarray(vals, dtype=np.float64)
            if a.max() - a.min() <= W / 2:
                return float(np.mean(a))
            ref = float(a.min())
            m_ = (ref + np.mean((a - ref + W / 2) % W - W / 2)) % W
            return float(m_ % W)

        return _axis_mean(xs), _axis_mean(ys)

    def _grow_base(self, field):
        if not self.alive or self.pred_error > Config.PRED_ERROR_THRESHOLD or self.in_dream:
            return
        if len(self.cells) >= 200:
            return
        _cap = int(getattr(Config, 'BODY_CELL_CAP', 0))
        # Use the live cell topology, not the periodically refreshed SOMA snapshot:
        # the latter can be up to SOMA_EVERY ticks stale.
        _body_components = self._components(self.cells, int(Config.WORLD_SIZE))
        _is_fragmented = len(_body_components) > 1
        if _cap > 0 and not _is_fragmented and self.world is not None:
            # [fix 29.10] под давлением плотности (space_brake<1) растущие тела тоже
            # притормаживают, а не только деление; фрагментированным телам даём расти
            # без урезания, чтобы они могли срастись обратно (см. ниже threshold*=0.3).
            _sr = getattr(self.world, 'selfreg', None)
            if _sr is not None:
                try:
                    _brake = float(_sr.space_brake())
                except Exception:
                    _brake = 1.0
                _cap = max(6, int(_cap * (0.3 + 0.7 * _brake)))
        if _cap > 0 and len(self.cells) >= _cap:
            return
        mask = (field[:, :, CH['owner']] == self.id).astype(np.float32)
        kernel = np.array([[0, 1, 0], [1, 0, 1], [0, 1, 0]], dtype=np.float32)
        neighbor_count = convolve(mask, kernel, mode='wrap')
        empty = (field[:, :, CH['owner']] == 0).astype(np.float32)
        score = field[:, :, CH['energy']] * 2.0 + field[:, :, CH['unknown']] * 1.8
        if Config.ENABLE_PHI_LABYRINTH:
            _, lab_grow, _ = self._get_labyrinth_params()
            gap = self.spirit_gap
            if gap >= 0.9:
                wall_factor = 3.0
            else:
                if gap < 0.4:
                    wall_factor = 0.6
                elif gap < 0.7:
                    wall_factor = 1.0 + (gap - 0.4) * 5.0
                elif gap < 1.0:
                    wall_factor = 2.5 - (gap - 0.7) * 5.0
                else:
                    wall_factor = 0.6
            score -= field[:, :, CH['wall']] * lab_grow * 1.5 * wall_factor
        layer = self._get_layer()
        penalty = self._get_layer_growth_penalty(layer)
        intent_bias = 1.5 if (self.intent and self.intent.get("type") == "explore") else 1.0
        threshold = self.pred_error * (1.5 + self.given_trigger) / intent_bias * penalty
        if getattr(self, 'somatic_drag', 1.0) < 0.6:
            threshold *= (1.0 + (0.6 - self.somatic_drag))
        if _is_fragmented:
            # [fix 29.10] фрагментированному телу (frag>1) легче дорасти в разрыв —
            # иначе компоненты растут каждый сам по себе и почти никогда не срастаются
            # (восст. было ~3-9 за 300+ тиков при разрывах в сотнях).
            threshold *= 0.3
        new_mask = (score > threshold) & (empty > 0) & (neighbor_count > 0)
        x_coords, y_coords = np.where(new_mask)
        MAX_GROWTH_PER_TICK = 6
        if _cap > 0:
            MAX_GROWTH_PER_TICK = max(1, min(6, _cap - len(self.cells)))
        if len(x_coords) > MAX_GROWTH_PER_TICK:
            cand_scores = score[x_coords, y_coords]
            if _is_fragmented:
                # Prefer a new cell that touches two or more distinct components:
                # such a cell actually joins components, unlike merely growing one tip.
                _labels = {}
                for _ci, _comp in enumerate(_body_components):
                    for _cell in _comp:
                        _labels[_cell] = _ci
                _bridge_flags = []
                for _x, _y in zip(x_coords, y_coords):
                    _adj = {
                        _labels[((int(_x) + _dx) % int(Config.WORLD_SIZE),
                                 (int(_y) + _dy) % int(Config.WORLD_SIZE))]
                        for _dx, _dy in self._NB4
                        if ((int(_x) + _dx) % int(Config.WORLD_SIZE),
                            (int(_y) + _dy) % int(Config.WORLD_SIZE)) in _labels
                    }
                    _bridge_flags.append(1 if len(_adj) >= 2 else 0)
                _order = sorted(
                    range(len(x_coords)),
                    key=lambda _i: (-_bridge_flags[_i], -float(cand_scores[_i]),
                                    int(x_coords[_i]), int(y_coords[_i]))
                )[:MAX_GROWTH_PER_TICK]
                top_idx = np.asarray(_order, dtype=np.int64)
            else:
                top_idx = np.argsort(cand_scores)[-MAX_GROWTH_PER_TICK:]
            x_coords, y_coords = x_coords[top_idx], y_coords[top_idx]
        new_cells = set(zip(x_coords, y_coords))
        self.cells.update(new_cells)
        if new_cells:
            gx = [c[0] for c in new_cells]
            gy = [c[1] for c in new_cells]
            field[gx, gy, CH['owner']] = self.id

    def grow(self, field):
        _lim = min(GIANT_CELL_LIMIT, int(getattr(Config, 'BODY_CELL_CAP', GIANT_CELL_LIMIT)))
        if len(self.cells) >= _lim:   # [AUDIT#7] не растим то, что _enforce_cell_cap всё равно обрежет
            return
        self._grow_base(field)
        if len(self.cells) > GIANT_CELL_LIMIT:
            all_cells = list(self.cells)
            all_cells_sorted = sorted(all_cells, key=lambda c: (c[0], c[1]))
            keep = set(all_cells_sorted[:GIANT_CELL_LIMIT])
            for c in self.cells - keep:
                field[c[0], c[1], CH['owner']] = 0
            self.cells = keep

    _NB4 = ((0, 1), (1, 0), (0, -1), (-1, 0))

    @staticmethod
    def _components(cells, W):
        cset = cells if isinstance(cells, (set, frozenset)) else set(cells)
        seen, comps = set(), []
        for st in sorted(cset):
            if st in seen:
                continue
            comp = {st}
            seen.add(st)
            stack = [st]
            while stack:
                x, y = stack.pop()
                for dx, dy in PatternPhysiology._NB4:
                    nb = ((x + dx) % W, (y + dy) % W)
                    if nb in cset and nb not in seen:
                        seen.add(nb)
                        comp.add(nb)
                        stack.append(nb)
            comps.append(comp)
        comps.sort(key=lambda c: (-len(c), min(c)))
        return comps

    # Правило неприкосновенности внутреннего тела: клетку тела можно отдать/забрать только
    # с периферии (хотя бы один из 4 соседей не принадлежит телу) и только если изъятие
    # не рвёт тело на части. Связность 4-соседняя, мир — тор (как в _components).
    _RING8 = ((0, 1), (1, 1), (1, 0), (1, -1), (0, -1), (-1, -1), (-1, 0), (-1, 1))

    @staticmethod
    def _cell_is_exposed(cell, body, W):
        x, y = cell
        for dx, dy in PatternPhysiology._NB4:
            if ((x + dx) % W, (y + dy) % W) not in body:
                return True
        return False

    @staticmethod
    def _removal_splits_body(cell, body, W):
        x, y = cell
        ring = [((x + dx) % W, (y + dy) % W) in body for dx, dy in PatternPhysiology._RING8]
        nbrs = [((x + dx) % W, (y + dy) % W) for dx, dy in PatternPhysiology._NB4
                if ((x + dx) % W, (y + dy) % W) in body]
        if len(nbrs) <= 1 or all(ring):
            return False
        start = ring.index(False)
        arc_of, arc = {}, -1
        for k in range(1, 9):
            i = (start + k) % 8
            if ring[i]:
                if not ring[(i - 1) % 8]:
                    arc += 1
                arc_of[i] = arc
        if len({arc_of[i] for i in (0, 2, 4, 6) if ring[i]}) == 1:
            return False
        goal = set(nbrs[1:])
        seen = {nbrs[0]}
        stack = [nbrs[0]]
        while stack and goal:
            cx, cy = stack.pop()
            for dx, dy in PatternPhysiology._NB4:
                nb = ((cx + dx) % W, (cy + dy) % W)
                if nb == cell or nb in seen or nb not in body:
                    continue
                seen.add(nb)
                goal.discard(nb)
                stack.append(nb)
        return bool(goal)

    @staticmethod
    def _take_block_reason(cell, body, W):
        if not PatternPhysiology._cell_is_exposed(cell, body, W):
            return 'interior'
        if PatternPhysiology._removal_splits_body(cell, body, W):
            return 'split'
        return None

    @staticmethod
    def _can_take_cell(cell, body, W):
        return PatternPhysiology._take_block_reason(cell, body, W) is None

    def _guard_count(self, key, n=1):
        # счётчики срабатываний body-guard лежат на мире (печатаются строкой [GUARD t=...])
        w = getattr(self, 'world', None)
        if w is None or not n:
            return
        d = getattr(w, '_body_guard_stats', None)
        if d is None:
            d = Counter()
            w._body_guard_stats = d
        d[key] += n

    @staticmethod
    def _pick_peripheral_cells(cells, n, W, adjacent_to=None):
        body = set(cells)
        attached = set(adjacent_to or ())
        picked = []
        while len(picked) < n:
            nxt = None
            for c in sorted(body):
                if not PatternPhysiology._can_take_cell(c, body, W):
                    continue
                if attached:
                    x, y = c
                    if not any(((x + dx) % W, (y + dy) % W) in attached
                               for dx, dy in PatternPhysiology._NB4):
                        continue
                nxt = c
                break
            if nxt is None:
                break
            body.discard(nxt)
            picked.append(nxt)
            attached.add(nxt)
        return picked

    @staticmethod
    def _topological_halves(cells_list, W):
        comps = PatternPhysiology._components(set(cells_list), W)
        main = comps[0]
        if len(main) < 2:
            return set(cells_list), set()

        def _far(start):
            seen = {start}
            dq = deque([start])
            last = start
            while dq:
                last = dq.popleft()
                for dx, dy in PatternPhysiology._NB4:
                    nb = ((last[0] + dx) % W, (last[1] + dy) % W)
                    if nb in main and nb not in seen:
                        seen.add(nb)
                        dq.append(nb)
            return last

        a = _far(min(main))
        b = _far(a)
        side = {a: 0, b: 1}
        qs = (deque([a]), deque([b]))
        while qs[0] or qs[1]:
            for s_ in (0, 1):
                q = qs[s_]
                took = False
                while q and not took:
                    cx, cy = q[0]
                    for dx, dy in PatternPhysiology._NB4:
                        nb = ((cx + dx) % W, (cy + dy) % W)
                        if nb in main and nb not in side:
                            side[nb] = s_
                            q.append(nb)
                            took = True
                            break
                    if not took:
                        q.popleft()
        halves = (set(), set())
        for c, s_ in side.items():
            halves[s_].add(c)
        for comp in comps[1:]:
            halves[0 if len(halves[0]) <= len(halves[1]) else 1].update(comp)
        return halves

    def move(self, field, t):
        if not self.alive or len(self.cells) == 0:
            return False
        self._last_move_t = int(t)
        self._last_move_attempt = False
        self._last_move_ok = False
        self._last_move_reason = 'mobility_gate'
        _mvc = _nav_audit(self.world)['c'] if self.world is not None else None
        if _mvc is not None:
            _mvc['mv_call'] += 1
        is_subject = getattr(self, '_subject_detected', False)
        base_mobility = self.genome.get('mobility', 0.1)
        gap_mult = 1.0 + 0.5 * min(self.spirit_gap, 1.5)
        energy_mult = 1.0 + 0.5 * max(0.0, 0.2 - self.energy)
        last_exchange = getattr(self, 'last_semantic_exchange_step', 0)
        steps_since_exchange = self.age - last_exchange
        exchange_mult = 1.0 + 0.3 * min(steps_since_exchange / 100.0, 1.0)
        effective_mobility = base_mobility * gap_mult * energy_mult * exchange_mult
        drag = getattr(self, 'somatic_drag', 1.0)
        effective_mobility *= (0.8 + 0.2 * drag)
        effective_mobility /= (1.0 + Config.IMPRINT_FREEZE_GAIN * self._enc_strain)
        effective_mobility = min(1.0, effective_mobility * float(getattr(Config, 'MOBILITY_MULT', 1.0)))
        _ev = getattr(self, '_evac', None)           # MORSE-EVO: эвакуация после услышанной тревоги
        if _ev is not None and int(t) > int(_ev[2]):
            self._evac = None
            _ev = None
        _ev_gain = 0.0
        if _ev is not None:
            _ev_gain = float(getattr(Config, 'HEARD_ALARM_EVAC_GAIN', 1.0))
            effective_mobility = min(1.0, effective_mobility * float(getattr(Config, 'HEARD_ALARM_EVAC_MOB_BOOST', 1.0)))
        if phi_hash(self.id, t, 12345) > effective_mobility:
            return False
        self._last_move_attempt = True
        self._last_move_reason = 'attempt_pending'
        _nav_attempt = getattr(self, '_nav_progress', None)
        if _nav_attempt is not None:
            # _nav_step runs during update_model, before move() in the engine tick.
            # Consume this exact nav episode on the next update_model call.
            self._nav_move_attempt_pending = _nav_attempt
        if _mvc is not None:
            _mvc['mv_try'] += 1
            if _ev is not None:
                _mvc['mv_evac_try'] += 1
        total_energy = 0.0
        sum_x, sum_y = 0.0, 0.0
        for (x, y) in sorted(self.cells):
            e = field[x, y, CH['energy']]
            total_energy += e
            sum_x += x * e
            sum_y += y * e
        if total_energy < 1e-6:
            cx = sum(c[0] for c in sorted(self.cells)) // len(self.cells)
            cy = sum(c[1] for c in sorted(self.cells)) // len(self.cells)
        else:
            cx = int(sum_x / total_energy)
            cy = int(sum_y / total_energy)
        _, _, lab_move = self._get_labyrinth_params()
        antigrav_strength = getattr(Config, 'ANTI_GRAVITY_STRENGTH', 1.0)
        if antigrav_strength > 0.0:
            center = Config.WORLD_SIZE // 2
            cx_center, cy_center = self.get_center()
            dist = np.sqrt((cx_center - center) ** 2 + (cy_center - center) ** 2)
            edge_threshold = Config.WORLD_SIZE * 0.35
            if dist > edge_threshold:
                dir_x = center - cx_center
                dir_y = center - cy_center
                norm = np.sqrt(dir_x ** 2 + dir_y ** 2) + 1e-8
                push = antigrav_strength * 0.2
                self._antigrav_push = (dir_x / norm * push, dir_y / norm * push)
            else:
                self._antigrav_push = (0.0, 0.0)
        gap = self.spirit_gap
        if gap >= 0.9:
            wall_factor = 3.0
        else:
            if gap < 0.4:
                wall_factor = 0.6
            elif gap < 0.7:
                wall_factor = 1.0 + (gap - 0.4) * 5.0
            elif gap < 1.0:
                wall_factor = 2.5 - (gap - 0.7) * 5.0
            else:
                wall_factor = 0.6
        _sx, _sy = imprint_pull(self)
        _tvx = _tvy = 0.0
        _nv = getattr(self, '_nav_progress', None) if getattr(Config, 'NAV_STEER_ENABLED', False) else None
        if _nv:
            _Wn = int(Config.WORLD_SIZE)
            _ex = ((int(_nv['target'][0]) - cx + _Wn // 2) % _Wn) - _Wn // 2
            _ey = ((int(_nv['target'][1]) - cy + _Wn // 2) % _Wn) - _Wn // 2
            _en = float(np.hypot(_ex, _ey))
            if _en > 1.0:
                _tvx, _tvy = _ex / _en, _ey / _en
            else:
                _nv = None
        _bgain = float(getattr(Config, 'NAV_DIR_STEER_GAIN', 0.0))
        _bux = _buy = 0.0
        _bd = getattr(self, '_best_direction', None)
        if _bgain > 0.0 and _nv is None and _bd and tuple(_bd) != (0, 0):
            _bn = float(np.hypot(_bd[0], _bd[1]))
            _bux, _buy = _bd[0] / _bn, _bd[1] / _bn
        else:
            _bgain = 0.0
        _W = int(Config.WORLD_SIZE)
        _dirs8 = ((-1, 0), (1, 0), (0, -1), (0, 1), (-1, -1), (-1, 1), (1, -1), (1, 1))
        _bc = np.array(sorted(self.cells), dtype=np.int64)
        _ddx = np.array([d[0] for d in _dirs8], dtype=np.int64)
        _ddy = np.array([d[1] for d in _dirs8], dtype=np.int64)
        _sxs = (_bc[None, :, 0] + _ddx[:, None]) % _W
        _sys = (_bc[None, :, 1] + _ddy[:, None]) % _W
        _own = field[_sxs, _sys, CH['owner']]
        _mine = _own == self.id
        _blk = ((_own != 0) & ~_mine).any(axis=1)
        if getattr(Config, 'ENABLE_CENTER_WALL', False):
            _blk |= (_sxs == Config.CENTER_WALL_X).any(axis=1)
        _front = ~_mine
        _nf = _front.sum(axis=1)
        _blk |= (_nf == 0)
        _clr = None                                  # «габариты»: свободный ход всем телом по каждому направлению
        _clr_k, _clr_gain, _clr_hold = 1, 0.0, 0
        if bool(getattr(Config, 'FISH_CLEARANCE_ENABLE', False)):
            _clr_k = max(2, int(getattr(Config, 'FISH_CLEARANCE_RANGE', 4)))
            _clr_gain = float(getattr(Config, 'FISH_CLEARANCE_GAIN', 0.5))
            _clr_hold = int(getattr(Config, 'FISH_CLEARANCE_HOLD_MIN', 2))
            _clr = np.where(_blk, 0, _clr_k).astype(np.int64)
            _clr_live = ~_blk
            for _kk in range(2, _clr_k + 1):
                _kx = (_bc[None, :, 0] + _ddx[:, None] * _kk) % _W
                _ky = (_bc[None, :, 1] + _ddy[:, None] * _kk) % _W
                _ko = field[_kx, _ky, CH['owner']]
                _kb = ((_ko != 0) & (_ko != self.id)).any(axis=1)
                if getattr(Config, 'ENABLE_CENTER_WALL', False):
                    _kb |= (_kx == Config.CENTER_WALL_X).any(axis=1)
                _clr[_clr_live & _kb] = _kk - 1
                _clr_live &= ~_kb
        _wpen = field[_sxs, _sys, CH['wall']] * (lab_move * 0.3 if is_subject else lab_move) * wall_factor
        _cs = field[_sxs, _sys, CH['energy']] * 2.0 + field[_sxs, _sys, CH['unknown']] * 1.8 - _wpen * 1.5
        _base = (_cs * _front).sum(axis=1) / np.maximum(_nf, 1)
        best_score = -np.inf
        best_dx, best_dy = 0, 0
        _dc_sc = []  # DIRCHAIN: оценки свободных направлений (read-only)
        for _i, (dx, dy) in enumerate(_dirs8):
            if _blk[_i]:
                continue
            score = float(_base[_i])
            if _clr is not None:
                score += _clr_gain * (float(_clr[_i]) / _clr_k - 1.0)
            score += Config.IMPRINT_STEER_GAIN * (dx * _sx + dy * _sy) / np.hypot(dx, dy)
            if _nv:
                score += float(Config.NAV_STEER_GAIN) * (dx * _tvx + dy * _tvy) / np.hypot(dx, dy)
            if _bgain > 0.0:
                score += _bgain * (dx * _bux + dy * _buy) / np.hypot(dx, dy)
            if _ev_gain > 0.0:
                score += _ev_gain * (dx * _ev[0] + dy * _ev[1]) / np.hypot(dx, dy)
            if is_subject and self.spirit_gap > 0.6:
                score += 0.8
            _dc_sc.append(float(score))
            if score > best_score:
                best_score = score
                best_dx, best_dy = dx, dy
        if best_score == -np.inf:
            self._dc = {'fn': 'move', 'nodir': True, 'blk_n': int(_blk.sum()), 'n_free': 0,
                        'n': int((getattr(self, '_dc', None) or {}).get('n', 0)) + 1}  # DIRCHAIN
            self._last_move_reason = 'no_direction'
            if _mvc is not None:
                _mvc['mv_fail_nodir'] += 1
            return False
        _dc_pre = (int(best_dx), int(best_dy))  # DIRCHAIN: выбор до антигравитации и «уступки импульсу»
        best_dx += int(self._antigrav_push[0])
        best_dy += int(self._antigrav_push[1])
        _mx, _my = getattr(self, '_dir_momentum', (0.0, 0.0))
        _mdx, _mdy = int(round(_mx)), int(round(_my))
        # FISH-3A: осмысленный курс держим безусловно (если направление свободно по всему телу: not _blk[_mi]);
        # False/слабый курс => прежняя «уступка», если разрыв оценки <= 0.15 (поведение R2)
        _f3a = (bool(getattr(Config, 'FISH3A_ENABLE', False))
                and float(np.hypot(_mx, _my)) >= float(getattr(Config, 'FISH3A_MIN_MOMENTUM', 0.3)))
        if (_mdx, _mdy) != (0, 0) and (_mdx, _mdy) != (best_dx, best_dy):
            _mi = _dirs8.index((_mdx, _mdy)) if (_mdx, _mdy) in _dirs8 else -1
            if _mi >= 0 and not _blk[_mi]:
                _sm = float(_base[_mi])
                if _clr is not None:
                    _sm += _clr_gain * (float(_clr[_mi]) / _clr_k - 1.0)
                _sm += Config.IMPRINT_STEER_GAIN * (_mdx * _sx + _mdy * _sy) / np.hypot(_mdx, _mdy)
                if _nv:
                    _sm += float(Config.NAV_STEER_GAIN) * (_mdx * _tvx + _mdy * _tvy) / np.hypot(_mdx, _mdy)
                if _bgain > 0.0:
                    _sm += _bgain * (_mdx * _bux + _mdy * _buy) / np.hypot(_mdx, _mdy)
                if _ev_gain > 0.0:
                    _sm += _ev_gain * (_mdx * _ev[0] + _mdy * _ev[1]) / np.hypot(_mdx, _mdy)
                _f3a_floor = (float(getattr(Config, 'MOVE_SCORE_THR', 0.3)) if getattr(self, '_move_soft_thr', None) is None
                              else float(self._move_soft_thr))
                if _f3a and _sm >= _f3a_floor and (_clr is None or int(_clr[_mi]) >= _clr_hold):
                    # курс держим, пока он не хуже порога оценки (иначе порог срежет ход) и впереди есть место
                    best_dx, best_dy = _mdx, _mdy
                    best_score = _sm  # порог проверяется по оценке РЕАЛЬНО выбранного направления
                elif _sm >= best_score - 0.15:
                    best_dx, best_dy = _mdx, _mdy
        # FISH-3: гистерезисный обход — при заблокированном курсе выбирать
        # ближайшее по углу свободное направление, а не направление с лучшей оценкой.
        # _mx/_my остаются исходным курсом для DIRCHAIN и mom_blk (наблюдатель read-only).
        _fish3_on = bool(getattr(Config, 'FISH3_ENABLE', False))
        _f4_on = _fish3_on and bool(getattr(Config, 'FISH4_ENABLE', False))
        _f4_now = int(getattr(self.world, 'age', t)) if self.world is not None else int(t)
        _f4_dense = int(_blk.sum()) >= 4
        _f4_wait_now = False
        _f4_restore = None
        _fish3_course = False
        _fish3_blocked = False
        _fish3_mi = -1
        _m_norm = float(np.hypot(_mx, _my))
        if _fish3_on:
            # слабый курс (|mu| < MIN) — отдельное состояние: не блок, не «свободно», подмены нет
            _fish3_course = _m_norm >= float(getattr(Config, 'FISH3_MIN_MOMENTUM', 0.3))
            if _fish3_course:
                # ближайшее по УГЛУ из 8 (делим на длину направления: диагонали длиннее)
                _fish3_mi = int(np.argmax([(_mx * _d[0] + _my * _d[1]) / (_m_norm * float(np.hypot(_d[0], _d[1])))
                                           for _d in _dirs8]))
                _fish3_blocked = bool(_blk[_fish3_mi])
            _fish3_blocked_streak = int(getattr(self, '_fish3_blocked_streak', 0))
            _fish3_free_streak = int(getattr(self, '_fish3_free_streak', 0))
            _fish3_detour_active = bool(getattr(self, '_fish3_detour_active', False))
            _f4_was_active = _fish3_detour_active
            if _fish3_course:
                if _fish3_blocked:
                    _fish3_blocked_streak += 1
                    _fish3_free_streak = 0
                else:
                    _fish3_free_streak += 1
                    _fish3_blocked_streak = 0
            if not _fish3_detour_active and _fish3_blocked_streak >= max(1, int(getattr(Config, 'FISH3_BLOCKED_N', 3))):
                _fish3_detour_active = True
            elif _fish3_detour_active and _fish3_free_streak >= max(1, int(getattr(Config, 'FISH3_FREE_M', 3))):
                _fish3_detour_active = False
            _f4_ref = None
            _f4_flow = (0.0, 0.0)
            _f4_blockers = []
            _f4_ri = -1
            if _f4_on:
                _F4S['calls'] += 1
                _F4S['calls_dense'] += int(_f4_dense)
                # (1) метрика: вернулся ли курс к намерению через 5 тиков после обхода
                _rd = getattr(self, '_f4_ret_due', None)
                if _rd is not None and _f4_now >= _rd:
                    _it0 = getattr(self, '_f4_intent', None)
                    if _it0 is not None and _fish3_course:
                        _cr = (_mx * _it0[0] + _my * _it0[1]) / _m_norm
                        _F4S['ret5_n'] += 1
                        _F4S['ret5_sum'] += _cr
                        _F4S['ret5_ge07'] += int(_cr >= 0.7)
                    self._f4_ret_due = None
                # (2) память намерения: цель навигации, а если её нет — «куда плыл, пока путь был свободен»
                _it = getattr(self, '_f4_intent', None)
                if _nv:
                    _it = (float(_tvx), float(_tvy))
                    self._f4_intent_t = _f4_now
                    _F4S['intent_nav_calls'] += 1
                elif _fish3_course and not _fish3_blocked and not _fish3_detour_active:
                    _ux, _uy = _mx / _m_norm, _my / _m_norm
                    if _it is None:
                        _it = (_ux, _uy)
                    else:
                        _e = float(getattr(Config, 'FISH4_INTENT_EMA', 0.15))
                        _ix, _iy = (1.0 - _e) * _it[0] + _e * _ux, (1.0 - _e) * _it[1] + _e * _uy
                        _in = float(np.hypot(_ix, _iy))
                        _it = (_ix / _in, _iy / _in) if _in > 1e-9 else (_ux, _uy)
                    self._f4_intent_t = _f4_now
                if (_it is not None and _fish3_detour_active
                        and (_f4_now - int(getattr(self, '_f4_intent_t', _f4_now))) > int(getattr(Config, 'FISH4_INTENT_TTL', 12))):
                    # обход затянулся — рыба передумала: намерение и обход сбрасываются, курс снова обычный
                    _it = None
                    _fish3_detour_active = False
                    _fish3_blocked_streak = 0
                    _fish3_free_streak = 0
                    self._f4_side = 0
                    self._f4_wait = 0
                    _F4S['intent_abandoned'] += 1
                self._f4_intent = _it
                # (3) обход закончился: сторона и ожидание забываются, курс возвращается к намерению
                if _f4_was_active and not _fish3_detour_active:
                    self._f4_side = 0
                    self._f4_wait = 0
                    self._f4_ret_due = _f4_now + 5
                    _F4S['detour_end'] += 1
                    if _it is not None and _fish3_course and (_mx * _it[0] + _my * _it[1]) / _m_norm < 0.7:
                        _f4_restore = (_it[0] * max(_m_norm, 0.5), _it[1] * max(_m_norm, 0.5))
                        _F4S['restore_n'] += 1
                if not _f4_was_active and _fish3_detour_active:
                    _F4S['detour_start'] += 1
                if _it is not None:
                    _f4_ref = _it
                elif _fish3_course:
                    _f4_ref = (_mx / _m_norm, _my / _m_norm)
                if _f4_ref is not None and (_fish3_detour_active or _fish3_blocked):
                    _f4_ri = int(np.argmax([(_f4_ref[0] * _d[0] + _f4_ref[1] * _d[1]) / float(np.hypot(_d[0], _d[1]))
                                            for _d in _dirs8]))
                    _f4_flow, _f4_blockers = _f4_neigh(self, _own, _f4_ri, _f4_now)
                # (4) метрика: блок снялся после ожидания?
                _wt = getattr(self, '_f4_waited_t', None)
                if _wt is not None and _fish3_course and not _fish3_blocked and (_f4_now - int(_wt)) <= 2:
                    _F4S['wait_cleared'] += 1
                    self._f4_waited_t = None
                if _fish3_course and not _fish3_blocked:
                    self._f4_wait = 0
                # MEM observer (read-only): повторно упирается ли агент в то же место тем же курсом
                if _fish3_course and _fish3_blocked and getattr(self, '_f4_blk_t', -1) != _f4_now:
                    self._f4_blk_t = _f4_now
                    _F4S['blk_events'] += 1
                    _pb = getattr(self, '_f4_blk_last', None)
                    if _pb is not None and 0 < (_f4_now - _pb[0]) <= 40 and _fish3_mi == _pb[3]:
                        _bdx = ((cx - _pb[1] + _W // 2) % _W) - _W // 2
                        _bdy = ((cy - _pb[2] + _W // 2) % _W) - _W // 2
                        if float(np.hypot(_bdx, _bdy)) <= 3.0:
                            _gap = _f4_now - _pb[0]
                            self._f4_blk_rep = int(getattr(self, '_f4_blk_rep', 0)) + 1
                            if _gap <= 5:
                                _F4S['blk_repeat_near'] += 1     # топчется на месте (≤5 тиков)
                            else:
                                _F4S['blk_repeat_late'] += 1     # вернулся позже на то же место (6–40 тиков)
                                _F4S['blk_repeat_late_gap_sum'] += _gap
                    self._f4_blk_last = (_f4_now, cx, cy, _fish3_mi)
            self._fish3_blocked_streak = _fish3_blocked_streak
            self._fish3_free_streak = _fish3_free_streak
            self._fish3_detour_active = _fish3_detour_active
            if _fish3_detour_active and _fish3_course:
                _F4S['det_calls'] += 1
                _rx, _ry = (_f4_ref if (_f4_on and _f4_ref is not None) else (_mx / _m_norm, _my / _m_norm))
                _rn = float(np.hypot(_rx, _ry))
                _rx, _ry = _rx / _rn, _ry / _rn
                _fish3_candidates = []
                for _fish3_i, _fish3_d in enumerate(_dirs8):
                    if _blk[_fish3_i]:
                        continue
                    if _clr is not None and int(_clr[_fish3_i]) < _clr_hold:
                        continue
                    _dn = float(np.hypot(_fish3_d[0], _fish3_d[1]))
                    # cos — насколько близко к курсу (малый поворот); sin со знаком — с какой стороны от курса
                    _fish3_candidates.append(((_rx * _fish3_d[0] + _ry * _fish3_d[1]) / _dn,
                                              (_rx * _fish3_d[1] - _ry * _fish3_d[0]) / _dn, _fish3_i, _fish3_d))
                if _fish3_candidates:
                    _side = int(getattr(self, '_f4_side', 0)) if _f4_on else 0
                    _pool = _fish3_candidates
                    if _side != 0:
                        # уже выбрали сторону обхода — остаёмся на ней (или идём прямо); на другую — только если иначе некуда
                        _same = [c for c in _fish3_candidates if c[1] * _side > -0.2]
                        if _same:
                            _pool = _same
                    _top = max(c[0] for c in _pool)
                    _tie = float(getattr(Config, 'FISH4_FLOW_TIE', 0.15)) if _f4_on else 0.0
                    _grp = [c for c in _pool if c[0] >= _top - _tie - 1e-12]
                    _fl = float(np.hypot(_f4_flow[0], _f4_flow[1])) if _f4_on else 0.0
                    if _f4_on and len(_grp) > 1:
                        def _f4_key(c):
                            _dn2 = float(np.hypot(c[3][0], c[3][1]))
                            _fd = ((c[3][0] * _f4_flow[0] + c[3][1] * _f4_flow[1]) / _dn2) if _fl > 0.05 else 0.0
                            return (round(_fd, 3), int(_clr[c[2]]) if _clr is not None else 0, -c[2])
                        _pick = max(_grp, key=_f4_key)
                        _plain = max(_grp, key=lambda c: (round(c[0], 9), -c[2]))
                        if _fl > 0.05:
                            _F4S['flow_used'] += 1
                            _F4S['flow_changed'] += int(_pick[2] != _plain[2])
                    else:
                        _pick = max(_grp, key=lambda c: (round(c[0], 9), -c[2]))
                    _fish3_cos, _fish3_sin, _fish3_i, (_fish3_dx, _fish3_dy) = _pick
                    if _f4_on:
                        if _side == 0 and abs(_fish3_sin) >= 0.2:
                            self._f4_side = 1 if _fish3_sin > 0 else -1
                            _F4S['side_commit'] += 1
                        elif _side != 0:
                            if _fish3_sin * _side > -0.2:
                                _F4S['side_kept'] += 1
                            else:
                                self._f4_side = 1 if _fish3_sin > 0 else -1
                                _F4S['side_flip'] += 1
                        # подождать: курс перекрыт «ведущим», который сам уходит в ту же сторону, а обход — резкий поворот
                        if (_f4_ri >= 0 and bool(_blk[_f4_ri]) and _blk[_f4_ri] is not None and _fish3_cos < float(getattr(Config, 'FISH4_WAIT_COS', 0.7))
                                and _f4_blockers):
                            _lead = all(_bl is not None and (_bl[0] * _rx + _bl[1] * _ry) >= float(getattr(Config, 'FISH4_WAIT_BLOCKER_VEL', 0.3))
                                        for _bl in _f4_blockers)
                            if _lead:
                                if getattr(self, '_f4_wait_t', -1) != _f4_now:      # решение одно на тик мира (jump/lunge не жгут бюджет)
                                    self._f4_wait_t = _f4_now
                                    _wn = int(getattr(self, '_f4_wait', 0))
                                    self._f4_wait_dec = _wn < int(getattr(Config, 'FISH4_WAIT_MAX', 2))
                                    if self._f4_wait_dec:
                                        self._f4_wait = _wn + 1
                                        self._f4_waited_t = _f4_now
                                        _F4S['wait_used'] += 1
                                    else:
                                        _F4S['wait_exhausted'] += 1
                                _f4_wait_now = bool(getattr(self, '_f4_wait_dec', False))
                            else:
                                _F4S['wait_skip_not_leaving'] += 1
                    _F4S['det_cos_sum'] += _fish3_cos
                    _F4S['det_le45'] += int(_fish3_cos >= 0.7071)
                    _F4S['det_rev'] += int(_fish3_cos < -0.0001)
                    if not _f4_wait_now:
                        best_dx, best_dy = int(_fish3_dx), int(_fish3_dy)
                        # Recompute score for the ACTUALLY selected direction so the normal
                        # MOVE_SCORE_THR check still evaluates the chosen step, not old best_score.
                        best_score = float(_base[_fish3_i])
                        if _clr is not None:
                            best_score += _clr_gain * (float(_clr[_fish3_i]) / _clr_k - 1.0)
                        best_score += Config.IMPRINT_STEER_GAIN * (_fish3_dx * _sx + _fish3_dy * _sy) / np.hypot(_fish3_dx, _fish3_dy)
                        if _nv:
                            best_score += float(Config.NAV_STEER_GAIN) * (_fish3_dx * _tvx + _fish3_dy * _tvy) / np.hypot(_fish3_dx, _fish3_dy)
                        if _bgain > 0.0:
                            best_score += _bgain * (_fish3_dx * _bux + _fish3_dy * _buy) / np.hypot(_fish3_dx, _fish3_dy)
                        if _ev_gain > 0.0:
                            best_score += _ev_gain * (_fish3_dx * _ev[0] + _fish3_dy * _ev[1]) / np.hypot(_fish3_dx, _fish3_dy)
                        if is_subject and self.spirit_gap > 0.6:
                            best_score += 0.8
        if _f4_wait_now or (_fish3_on and bool(getattr(Config, 'FISH3_FREEZE_ON_DETOUR', False))
                            and bool(getattr(self, '_fish3_detour_active', False))):
            # ждём или обходим: курс не подмешивает выбранный шаг — рыба помнит, куда плыла
            self._dir_momentum = (float(_mx), float(_my))
        else:
            self._dir_momentum = (0.7 * _mx + 0.3 * best_dx, 0.7 * _my + 0.3 * best_dy)
        if _f4_restore is not None:
            self._dir_momentum = (float(_f4_restore[0]), float(_f4_restore[1]))
        try:  # DIRCHAIN (observer, read-only): всё про ЭТОТ вызов move(), синхронно
            _dcs = sorted(_dc_sc, reverse=True)
            _dcd = (int(round(_mx)), int(round(_my)))
            _dci = _dirs8.index(_dcd) if _dcd in _dirs8 else -1
            _dbd = getattr(self, '_best_direction', None)
            _dbt = (int(_dbd[0]), int(_dbd[1])) if _dbd is not None and len(_dbd) >= 2 else (0, 0)
            _dbi = _dirs8.index(_dbt) if _dbt in _dirs8 else -1
            self._dc = {'fn': 'move', 'mu': (float(_mx), float(_my)), 'bd': _dbt, 'pre': _dc_pre,
                        'chosen': (int(best_dx), int(best_dy)), 'blk_n': int(_blk.sum()), 'n_free': len(_dcs),
                        'sc1': float(_dcs[0]), 'sc2': (float(_dcs[1]) if len(_dcs) > 1 else None),
                        'mom_blk': (bool(_blk[_dci]) if _dci >= 0 else None),
                        'bd_blk': (bool(_blk[_dbi]) if _dbi >= 0 else None),
                        'n': int((getattr(self, '_dc', None) or {}).get('n', 0)) + 1}
        except Exception:
            self._dc = None
        if _f4_wait_now:
            # FISH-4: «подождать ведущего» — тик пропущен, курс и намерение сохранены (как скольжение между гребками)
            self._last_move_reason = 'f4_wait'
            return False
        _thr = getattr(self, '_move_soft_thr', None)
        _base_thr = float(getattr(Config, 'MOVE_SCORE_THR', 0.3))
        if best_score < (_base_thr if _thr is None else _thr):
            self._last_move_reason = 'score_threshold'
            if _mvc is not None:
                _mvc['mv_fail_thr'] += 1
            return False
        if _thr is not None and best_score < _base_thr:
            self._move_soft_saved = True
        W = Config.WORLD_SIZE
        cells_list = sorted(self.cells)
        _wall_x = Config.CENTER_WALL_X if getattr(Config, 'ENABLE_CENTER_WALL', False) else None
        new_cells = set()
        for (cx_, cy_) in cells_list:
            nx_, ny_ = (cx_ + best_dx) % W, (cy_ + best_dy) % W
            if _wall_x is not None and nx_ == _wall_x:
                self._last_move_reason = 'wall'
                return False
            if field[nx_, ny_, CH['owner']] not in (0, self.id):
                self._last_move_reason = 'occupied'
                if _mvc is not None:
                    _mvc['mv_fail_occ'] += 1
                return False
            new_cells.add((nx_, ny_))
        for (cx_, cy_) in cells_list:
            field[cx_, cy_, CH['owner']] = 0
        for (nx_, ny_) in new_cells:
            field[nx_, ny_, CH['owner']] = self.id
        self.cells = new_cells
        _mvx, _mvy = getattr(self, '_move_vector_accum', (0.0, 0.0))
        self._move_vector_accum = (float(_mvx) + float(best_dx), float(_mvy) + float(best_dy))
        self._cellular_endurance = max(0.0, getattr(self, '_cellular_endurance', 1.0) - 0.0005)
        if is_subject:
            self._log_event("subject_traveled", to=min(new_cells))
        _walled = [c for c in new_cells if field[c[0], c[1], CH['wall']] > 0.6]
        if _walled and phi_hash(self.id, t, 777) < Config.PHI_LABYRINTH_BREAK_PROB:
            for c in _walled:
                field[c[0], c[1], CH['wall']] *= 0.4
            self._log_event("wall_breached")
        self._last_move_ok = True
        self._last_move_reason = 'ok'
        self._last_move_ok = True
        self._last_move_reason = 'ok'
        if _f4_on:
            try:   # FISH-4 (observer): повороты шаг→шаг и прямолинейность за 10 шагов
                _sfx = '_d' if _f4_dense else '_f'
                _bn = float(np.hypot(best_dx, best_dy))
                _pv = getattr(self, '_f4_prev', None)
                if _pv is not None and _bn > 0.0:
                    _pn = float(np.hypot(_pv[0], _pv[1]))
                    if _pn > 0.0:
                        _tc = (_pv[0] * best_dx + _pv[1] * best_dy) / (_pn * _bn)
                        _F4S['turn_n' + _sfx] += 1
                        _F4S['turn_le45' + _sfx] += int(_tc >= 0.7071)
                        _F4S['turn_rev' + _sfx] += int(_tc < -0.7071)
                self._f4_prev = (int(best_dx), int(best_dy))
                _stp = getattr(self, '_f4_steps', None)
                if _stp is None:
                    _stp = deque(maxlen=10)
                    self._f4_steps = _stp
                _stp.append((int(best_dx), int(best_dy)))
                self._f4_cnt = int(getattr(self, '_f4_cnt', 0)) + 1
                if len(_stp) == 10 and self._f4_cnt % 5 == 0:
                    _pl = sum(float(np.hypot(_q[0], _q[1])) for _q in _stp)
                    _nt = float(np.hypot(sum(_q[0] for _q in _stp), sum(_q[1] for _q in _stp)))
                    if _pl > 0.0:
                        _F4S['straight_n' + _sfx] += 1
                        _F4S['straight_sum' + _sfx] += _nt / _pl
            except Exception:
                pass
        if _nav_attempt is not None:
            self._nav_move_ok_pending = _nav_attempt
        if _mvc is not None:
            _mvc['mv_ok'] += 1
            if _ev is not None:
                _mvc['mv_evac_ok'] += 1
        return True

    def metabolic_tax(self, field, hunger_mult=1.0):
        if len(self.cells) > 200:
            hunger_mult *= 1.2
        hunger_mult = max(hunger_mult, 0.1)
        hunger_mult = min(hunger_mult, 5.0)
        if self.role_type == "disorganizer":
            hunger_mult *= 0.5
        wisdom_bonus = 1.0 - 0.3 * self._nci
        age_bonus = 1.0 + 0.5 * max(0, (200 - self.age) / 200)
        hunger_mult *= wisdom_bonus * age_bonus
        if self.protection_level > 0:
            hunger_mult *= (1.0 - self.protection_level * 0.8)
        efficiency = 1.0 / (1.0 + self.pred_error)
        cost = self.genome['metabolic_cost'] * len(self.cells) * (2.0 - efficiency) * hunger_mult
        if self.soul_weight < 0.4:
            cost *= 0.7
        if self.age < 300:
            cost *= 0.8
        drag = getattr(self, 'somatic_drag', 1.0)
        if drag < 1.0:
            cost *= (1.0 + (1.0 - drag) * 0.3)
        if self.world:
            neighbor_count = 0
            for (x, y) in self.cells:
                for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    nx, ny = (x + dx) % Config.WORLD_SIZE, (y + dy) % Config.WORLD_SIZE
                    if self.world.field[nx, ny, CH['owner']] not in (0, self.id):
                        neighbor_count += 1
            density_factor = 1.0 - min(0.3, neighbor_count * 0.01)
            cost *= density_factor
        if self.world is not None and getattr(self.world, 'niche', None) is not None:
            if self.age % Config.EXPENSIVE_COMPUTE_INTERVAL == 0:
                ln = safe_mean([self.world.niche[c[0], c[1]] for c in self.cells], 0)
                self._niche_cost_factor = 1.0 / (1.0 + 0.25 * ln)
            cost *= self._niche_cost_factor
        if getattr(self, '_forsaken', False):
            cost *= 0.8
        if self.intent and self.intent.get('type') == 'cooperate':
            cost *= 0.7
        if self.emotional_memory['grief'] > Config.EMOTIONAL_GRIEF_THRESHOLD:
            cost *= (1.0 + Config.GRIEF_METABOLIC_PENALTY * (self.emotional_memory['grief'] - Config.EMOTIONAL_GRIEF_THRESHOLD))
        love_partners = [pid for pid, t in self.trust_ledger.entries.items() if t > Config.LOVE_METABOLIC_THRESHOLD]
        mutual = False
        if love_partners and self.world and self.world.pattern_dict:
            for pid in love_partners:
                partner = self.world.pattern_dict.get(pid)
                if partner and partner.trust_ledger.get(self.id) > Config.LOVE_METABOLIC_THRESHOLD:
                    mutual = True
                    break
        if mutual:
            cost *= (1.0 - Config.LOVE_METABOLIC_BONUS)
            if Config.X2_BONUS_MODE == 'legacy':
                self._x2_change('X2.love_bonus', self.energy + Config.LOVE_DIRECT_BONUS)
            elif Config.X2_BONUS_MODE != 'paid':
                raise ValueError(f"X2_BONUS_MODE: ожидалось 'legacy' или 'paid', получено {Config.X2_BONUS_MODE!r}")
        if getattr(self, 'social_memory', None):
            _coord_partner_understood = any(
                _mem.get('cached_sim', 0.0) > Config.COORDINATION_SIM_THRESHOLD
                for _mem in self.social_memory.values())
            if _coord_partner_understood:
                cost *= (1.0 - Config.COORDINATION_METABOLIC_BONUS)
                self._log_event('coordination_metabolic_bonus',
                                 threshold=Config.COORDINATION_SIM_THRESHOLD)
        if soul_check(self).is_triadic_alive():
            cost *= 0.6
        if getattr(self, '_subject_detected', False):
            discount = 0.85
            gap = self.spirit_gap
            if 0.2 <= gap <= 0.7:
                discount -= 0.10
                if self.soul_weight > 0.8 and self.coherence > 0.7 and getattr(self, '_nci', 0) > 0.8:
                    discount -= 0.05
            elif gap < 0.1 or gap > 1.0:
                discount = 0.95
            cost *= discount
        elif getattr(self, 'protection_level', 0) > 0.4:
            cost *= 0.9
        _stall_norm = min(getattr(self, 'semantic_state_age', 0), Config.BURN_STALL_NORM * 3) / Config.BURN_STALL_NORM
        _f_h_eff = 1.0 / (1.0 + getattr(self, 'epistemic_scar', 0.0) + Config.BURN_GAMMA_STALL * _stall_norm)
        _burn = Config.BASE_BURN + Config.BURN_LAMBDA * (getattr(self, 'epistemic_scar', 0.0) + 1.0 / max(_f_h_eff, 1e-3))
        self._last_burn_gradient = _burn
        cost += _burn
        self._last_metabolic_taken = 0.0
        cells_arr = np.array(sorted(self.cells))
        if len(cells_arr) == 0:
            return cost
        xs, ys = cells_arr[:, 0], cells_arr[:, 1]
        # налог берётся только с положительной энергии: поле допускает E до FIELD_ENERGY_CLIP_MIN (<0),
        # а отрицательный taken превращал налог в источник энергии (field -= taken поднимал клетку)
        energy_available = np.maximum(0.0, field[xs, ys, CH['energy']])
        total_available = np.sum(energy_available)
        if total_available <= 0:
            return cost
        taken = np.minimum(energy_available, cost / len(self.cells))
        self._last_metabolic_taken = np.sum(taken)
        field[xs, ys, CH['energy']] -= taken
        cost -= self._last_metabolic_taken
        if self.world is not None and hasattr(self.world, '_ledger_add'):
            self.world._ledger_add('Pattern.metabolic_tax (налог с клеток)', -float(self._last_metabolic_taken))
        remaining = field[xs, ys, CH['energy']]
        surplus = np.maximum(0.0, remaining - Config.ENERGY_HARVEST_FLOOR)
        _base_rate = float(Config.ENERGY_HARVEST_RATE)
        _rate = _base_rate
        if Config.COOP_HARVEST_MODE == 'group':
            _n = getattr(self.world, '_coop_group', {}).get(self.id, 0) if self.world is not None else 0
            if _n > 0:
                _rate = min(1.0, _base_rate * (1.0 + Config.COOP_HARVEST_GAIN * min(_n, Config.COOP_MAX_PARTNERS) / Config.COOP_MAX_PARTNERS))
        elif Config.COOP_HARVEST_MODE != 'off':
            raise ValueError(f"COOP_HARVEST_MODE: ожидалось 'off' или 'group', получено {Config.COOP_HARVEST_MODE!r}")
        harvested = surplus * _rate
        harvested_total = float(np.sum(harvested))
        if Config.ENERGY_X2_OVERFLOW_MODE == 'limit_before_debit':
            if harvested_total > 0:
                _hi = type(self).energy.clamp_max
                _top = Config.ENERGY_HARVEST_CAP if _hi is None else min(Config.ENERGY_HARVEST_CAP, _hi)
                _room = max(0.0, _top - float(self.energy))
                if harvested_total > _room:
                    harvested = harvested * (_room / harvested_total)
                    harvested_total = float(np.sum(harvested))
        elif Config.ENERGY_X2_OVERFLOW_MODE != 'clip':
            raise ValueError(f"ENERGY_X2_OVERFLOW_MODE: ожидалось 'limit_before_debit' или 'clip', получено {Config.ENERGY_X2_OVERFLOW_MODE!r}")
        if harvested_total > 0:
            field[xs, ys, CH['energy']] -= harvested
            _old_e = float(self.energy)
            self.energy = min(self.energy + harvested_total, Config.ENERGY_HARVEST_CAP)
            _actual_gain = float(self.energy) - _old_e
            if self.world is not None:
                self.world._memo_add('X2.field_to_agent_harvest: списано с поля, но не дошло до агента (клип X2 [-1, 2])',
                                     harvested_total - _actual_gain)
                self.world._ledger_add('X1.field_to_agent_harvest', -harvested_total)
                self.world._ledger_add_x2('X2.field_to_agent_harvest', +_actual_gain)
                if _rate > _base_rate:
                    self.world._coop_extra = getattr(self.world, '_coop_extra', 0.0) + harvested_total * (1.0 - _base_rate / _rate)
            self._last_energy_harvested = harvested_total
        if self.age >= Config.DIVIDE_MIN_AGE and len(self.cells) >= Config.DIVIDE_MIN_SIZE:
            if Config.X2_BONUS_MODE == 'legacy':
                self._x2_change('X2.division_ready_bonus', self.energy + Config.DIVISION_READY_BONUS)
            elif Config.X2_BONUS_MODE == 'paid':
                _hi = type(self).energy.clamp_max
                _top = Config.ENERGY_HARVEST_CAP if _hi is None else min(Config.ENERGY_HARVEST_CAP, _hi)
                _want = min(float(Config.DIVISION_READY_BONUS), max(0.0, _top - float(self.energy)))
                _avail = np.maximum(0.0, field[xs, ys, CH['energy']])
                _avail_sum = float(np.sum(_avail))
                _take = min(_want, _avail_sum)
                if _take > 0.0:
                    field[xs, ys, CH['energy']] -= _avail * (_take / _avail_sum)
                    _old_e = float(self.energy)
                    self.energy = min(self.energy + _take, Config.ENERGY_HARVEST_CAP)
                    _gain = float(self.energy) - _old_e
                    if self.world is not None:
                        self.world._ledger_add('X1.division_ready_transfer', -_take)
                        self.world._ledger_add_x2('X2.division_ready_transfer', +_gain)
            else:
                raise ValueError(f"X2_BONUS_MODE: ожидалось 'legacy' или 'paid', получено {Config.X2_BONUS_MODE!r}")
        return max(0.0, cost)

    def can_divide(self):
        if getattr(self, '_divide_blocked_by_fatigue', False):
            return False
        max_cells = (self.world.selfreg.get_max_cells_per_agent()
                     if self.world is not None else 200)
        if len(self.cells) > max_cells:
            return False
        if (self.world is not None
                and not self.world.selfreg.can_agent_divide(self)):
            return False
        partner_boost = 0.0
        if self.world is not None:
            for pid, trust in self.trust_ledger.entries.items():
                if trust > 0.9:
                    partner = self.world.pattern_dict.get(pid)
                    if partner and partner.alive and partner.trust_ledger.get(self.id, 0) > 0.9:
                        partner_boost = 0.2
                        break
        adaptive_threshold = Config.DIVIDE_ERROR_THRESHOLD + self.soul_weight * (Config.DIVIDE_MAX_THRESHOLD - Config.DIVIDE_ERROR_THRESHOLD) + partner_boost
        if self.arc_tracker.completed_arcs:
            adaptive_threshold *= Config.ARC_DIVIDE_BONUS
        if getattr(Config, 'COMM_DIVIDE_BONUS_ENABLE', False):
            # MORSE-EVO: успешная коммуникация (reinforce_sender) чуть смягчает порог по pred_error
            adaptive_threshold += min(float(getattr(Config, 'COMM_DIVIDE_BONUS_CAP', 0.06)),
                                      int(getattr(self, '_comm_success_count', 0))
                                      * float(getattr(Config, 'COMM_DIVIDE_BONUS_PER', 0.002)))
        return (self.pred_error < adaptive_threshold and
                self.energy > 0.08 and
                len(self.cells) >= 6 and
                self.alive and
                self.soul_weight > 0.25 and
                self.age > Config.DIVIDE_MIN_AGE and
                (self.age - self.last_divide_age) > Config.DIVIDE_COOLDOWN and not self.in_dream)

    def _mutate_open_genome(self, child):
        if phi_hash(child.id, self.age, 4242) < Config.OPEN_GENE_PROB:
            g = Config.OPEN_GENE_POOL[int(phi_hash(child.id, 1, 4243) * len(Config.OPEN_GENE_POOL))]
            if g not in child.genome:
                child.genome[g] = round(0.5 + 0.5 * (phi_hash(child.id, 2, 4244) - 0.5), 3)
                child._log_event("gene_emerged", gene=g)
                if self.world is not None:
                    self.world.witness.record(child.id, "gene_emerged", gene=g)
        for g in Config.OPEN_GENE_POOL:
            if g in child.genome:
                child.genome[g] = float(np.clip(
                    child.genome[g] + (phi_hash(child.id, 3, 4245) - 0.5) * 0.2, 0, 1.5))

    def divide(self, field, next_id):
        cells_list = sorted(self.cells)
        min_cells = max(3, Config.DIVIDE_MIN_SIZE)
        child1_cells, child2_cells = PatternPhysiology._topological_halves(cells_list, Config.WORLD_SIZE)
        if len(child1_cells) < min_cells or len(child2_cells) < min_cells:
            return None, next_id
        self.cells = child1_cells
        child = Pattern(next_id, child2_cells, parent=self, world=self.world)
        if child2_cells:
            cx = [c[0] for c in child2_cells]
            cy = [c[1] for c in child2_cells]
            field[cx, cy, CH['owner']] = child.id
        if self._self_narrative:
            try:
                child._self_narrative = deque(mem_inherit_copy(self._self_narrative, 50),
                                               maxlen=self._self_narrative.maxlen)
            except AttributeError:
                child._self_narrative = mem_inherit_copy(self._self_narrative, 50)
        for k in ['gratitude', 'grief']:
            val = child.emotional_memory.get(k, 0.5)
            if isinstance(val, dict):
                child.emotional_memory[k] = 0.5
            else:
                child.emotional_memory[k] = float(val)
        if self.role_type == "disorganizer":
            child.lineage_id = child.id
            child.lineage_born_at_step = self.world.age if self.world else self.age
            child.lineage_total_age = 0
            child._log_event("lineage_break_disorganizer", parent_lineage=self.lineage_id)
        else:
            if self.concept_graph.nodes and child.concept_graph.nodes:
                sim = self.concept_graph.similarity(child.concept_graph)
            else:
                sim = 0.8
            forced_speciation = (self.age > 200 and
                                  phi_hash(child.id, self.age, 4242) < 0.08)
            _divergence_threshold = getattr(self, 'divergence_key', Config.DEFAULT_DIVERGENCE_KEY)
            if sim >= _divergence_threshold and not forced_speciation:
                child.lineage_id = self.lineage_id
                child.lineage_born_at_step = self.lineage_born_at_step
                child.lineage_total_age = self.lineage_total_age
                child._log_event("lineage_continue", sim=round(sim, 3), lineage=self.lineage_id)
            else:
                child.lineage_id = child.id
                child.lineage_born_at_step = self.world.age if self.world else self.age
                child.lineage_total_age = 0
                if forced_speciation:
                    child._log_event("lineage_split_forced", sim=round(sim, 3),
                                     old_lineage=self.lineage_id, new_lineage=child.id)
                else:
                    child._log_event("lineage_split", sim=round(sim, 3),
                                     old_lineage=self.lineage_id, new_lineage=child.id)
        child.role_type = "normal"
        if self.role_type == "disorganizer":
            child._log_event("born_from_disorganizer", parent_id=self.id)
        else:
            child._log_event("born_from_division", parent_id=self.id)
        if self.role_type == "disorganizer":
            child.genome['mutation_rate'] = min(2.0, child.genome.get('mutation_rate', 1.0) * 1.3)
        if self._scar_of_light and self.age > 10000 and self.event_counts.get('redeemed', 0) > 20:
            ancient_sig = (round(self.soul_weight, 1), round(self.emotional_memory['gratitude'], 1), 0.99, "ancient_wisdom")
            if ancient_sig not in child.concept_graph.nodes:
                child.concept_graph.nodes[ancient_sig] = {"origin": "PARENT", "origin_prev": "OWN", "count": 5.0, "value": np.zeros(4), "eternal": True}
                child._log_event("gift_of_the_ancient", donor=self.id)
        noise = 0.02 * np.array([phi_hash(next_id, i, 200) - 0.5 for i in range(8)])
        child.belief = self.belief + noise
        child.unknown_belief = self.unknown_belief + 0.02 * (phi_hash(next_id, 99, 200) - 0.5)
        child.self_model = self.self_model + 0.02 * (phi_hash(next_id, 201, 200) - 0.5)
        child.emotional_memory['gratitude'] = float(np.clip(
            child.emotional_memory['gratitude'] + (phi_hash(next_id, 0, 300) - 0.5) * Config.EMOTIONAL_MUTATION_STRENGTH,
            0.0, 1.0
        ))
        child.emotional_memory['grief'] = float(np.clip(
            child.emotional_memory['grief'] + (phi_hash(next_id, 1, 301) - 0.5) * Config.EMOTIONAL_MUTATION_STRENGTH,
            0.0, 1.0
        ))
        if Config.ENABLE_OPEN_GENOME:
            self._mutate_open_genome(child)
        mut_strength_plasticity = self.genome.get('plasticity', 0.5)
        if mut_strength_plasticity != 0.5:
            noise_extra = 0.01 * mut_strength_plasticity * np.array(
                [phi_hash(next_id, i, 4300) - 0.5 for i in range(8)])
            child.belief = child.belief + noise_extra
        if Config.ENABLE_EPIGENETIC_MEMORY and getattr(self, '_epigenetic_crisis_gen', 0) > 0:
            child._epigenetic_crisis_gen = self._epigenetic_crisis_gen - 1
            _boost = 0.15 * (child._epigenetic_crisis_gen + 1) / 3.0
            for _g in ('trust_plasticity', 'noise_filter', 'scar_resistance'):
                if _g in child.genome:
                    child.genome[_g] = float(np.clip(
                        child.genome[_g] + (phi_hash(next_id, 42, 6200) - 0.5) * _boost, 0, 1.5))
            child._log_event("epigenetic_inheritance", gen_left=child._epigenetic_crisis_gen)
            if self.world is not None:
                self.world.witness.record(child.id, "epigenetic_inheritance")
        dominant_t = self.concept_graph.get_dominant_transition()
        if dominant_t:
            src, dst, count = dominant_t
            if src not in child.concept_graph.edges:
                child.concept_graph.edges[src] = {}
            child.concept_graph.edges[src][dst] = max(1, count // 3)
            child._log_event("inherited_concept_path", src=str(src)[:25], dst=str(dst)[:25])
        if self._cellular_endurance < 0.05:
            child._cellular_endurance = 0.3
        else:
            child._cellular_endurance = self._cellular_endurance * 0.75
            self._cellular_endurance *= 0.75
        self.last_divide_age = self.age
        child.last_divide_age = 0
        for sid, trust in self.trust_ledger.entries.items():
            child.trust_ledger.entries[sid] = (trust + Config.TRUST_BASE) / 2.0
        child.update_properties(field)
        child.update_model(field, t=self.world.age if self.world else 0)
        self.update_properties(field)
        # [AUDIT#2] update_model родителя уже вызван в step() в этом тике; второй вызов дублировал
        # episodic_buffer / decay / tension. Для ребёнка (новый объект) update_model остаётся.
        if child and self.role_type == "feral":
            child.role_type = "feral"
            child._feral_fury = self._feral_fury * 0.5
            child._log_event("born_feral", parent=self.id)
        return child, next_id + 1

    def compete(self, other, field):
        if self.role_type == "feral" or other.role_type == "feral":
            return
        if self.protection_level > 0.5 or other.protection_level > 0.5:
            return
        if self.role_type == "disorganizer" and other.role_type == "disorganizer":
            return
        if self.role_type == "disorganizer" or other.role_type == "disorganizer":
            trust_self = 0
            trust_other = 0
        else:
            trust_self = self.trust_ledger.get(other.id)
            trust_other = other.trust_ledger.get(self.id)
        if trust_self > 0.8 and trust_other > 0.8:
            return
        overlap = self.cells & other.cells
        if not overlap:
            return
        _sig_self_has = (0.9, 0.9, 0.9, f'reciprocity_with_{other.id}')
        _sig_other_has = (0.9, 0.9, 0.9, f'reciprocity_with_{self.id}')
        _broke_bond = False
        if _sig_self_has in self.concept_graph.nodes:
            del self.concept_graph.nodes[_sig_self_has]
            if _sig_self_has in self.concept_graph.edges:
                del self.concept_graph.edges[_sig_self_has]
            self._ethics_pairs_bonded.discard(other.id)
            _broke_bond = True
        if _sig_other_has in other.concept_graph.nodes:
            del other.concept_graph.nodes[_sig_other_has]
            if _sig_other_has in other.concept_graph.edges:
                del other.concept_graph.edges[_sig_other_has]
            other._ethics_pairs_bonded.discard(self.id)
            _broke_bond = True
        if _broke_bond:
            self._log_event("reciprocity_broken", partner=other.id)
            other._log_event("reciprocity_broken", partner=self.id)
            if getattr(self, 'world', None) is not None:
                self.world.witness.record(self.id, "reciprocity_broken", partner=other.id)
        my_power = self.fitness * max(self.energy, 0)
        other_power = other.fitness * max(other.energy, 0)
        avg_unk = safe_mean(field[:, :, CH['unknown']], 0.05)
        if self.role == 1:
            my_power *= 1.0 + avg_unk
        else:
            bonus = 1.0 + max(0, (0.2 - avg_unk)) * 0.8
            my_power *= bonus * (1.1 if self.local_phase == "MODEL" else 1.0)
        if other.role == 1:
            other_power *= 1.0 + avg_unk
        else:
            bonus = 1.0 + max(0, (0.2 - avg_unk)) * 0.8
            other_power *= bonus * (1.1 if other.local_phase == "MODEL" else 1.0)
        if self.emotional_memory['gratitude'] > Config.EMOTIONAL_GRATITUDE_THRESHOLD:
            my_power *= (1.0 - Config.GRATITUDE_AGGRESSION_REDUCTION * (self.emotional_memory['gratitude'] - Config.EMOTIONAL_GRATITUDE_THRESHOLD))
        if other.emotional_memory['gratitude'] > Config.EMOTIONAL_GRATITUDE_THRESHOLD:
            other_power *= (1.0 - Config.GRATITUDE_AGGRESSION_REDUCTION * (other.emotional_memory['gratitude'] - Config.EMOTIONAL_GRATITUDE_THRESHOLD))
        if self.role_type == "normal" and other.role_type == "normal":
            if other.trust_ledger.get(self.id) > 0.7:
                my_power *= Config.TRUST_COOPERATION_BONUS
            if self.trust_ledger.get(other.id) > 0.7:
                other_power *= Config.TRUST_COOPERATION_BONUS
        if not np.isfinite(my_power):
            my_power = 0
        if not np.isfinite(other_power):
            other_power = 0
        total_power = my_power + other_power
        if total_power < 1e-9:
            margin = 0.0
        else:
            margin = abs(my_power - other_power) / total_power
        winner, loser = (self, other) if my_power > other_power else (other, self)
        size_damping = min(1.0, len(winner.cells) / 90.0)
        transfer_prob = (0.5 + 0.18 * margin) * (1.0 - 0.45 * size_damping)
        transfer_prob = max(0.12, min(0.8, transfer_prob))
        giveback_prob = (0.12 + 0.20 * (1.0 - margin)) * (1.0 + 0.35 * size_damping)
        giveback_prob = max(0.08, min(0.4, giveback_prob))
        cells_lost = 0
        _bw = int(Config.WORLD_SIZE)
        self._guard_count('c_overlap_cells', len(overlap))
        for cell in sorted(overlap):
            if len(winner.cells) >= 200:
                _why = self._take_block_reason(cell, loser.cells, _bw)
                if _why:
                    self._guard_count('c_block_' + _why)
                    continue
                loser.cells.discard(cell)
                field[cell[0], cell[1], CH['owner']] = 0
                self._guard_count('c_moved')
                continue
            if phi_hash(cell[0], cell[1], self.id) < transfer_prob:
                _why = self._take_block_reason(cell, loser.cells, _bw)
                if _why:
                    self._guard_count('c_block_' + _why)
                    continue
                self._guard_count('c_moved')
                loser.cells.discard(cell)
                winner.cells.add(cell)
                field[cell[0], cell[1], CH['owner']] = winner.id
                winner.soul_weight += 0.01
                loser.soul_weight -= 0.02
                cells_lost += 1
            elif ((loser.unresolved_contradiction > 0.2 and phi_hash(cell[0], cell[1], self.id + 999) < giveback_prob) or
                  phi_hash(cell[0], cell[1], self.id + 777) < giveback_prob * 0.25):
                _why = self._take_block_reason(cell, winner.cells, _bw)
                if _why:
                    self._guard_count('c_block_' + _why)
                    continue
                self._guard_count('c_moved')
                winner.cells.discard(cell)
                loser.cells.add(cell)
                field[cell[0], cell[1], CH['owner']] = loser.id
                winner.soul_weight -= 0.01
                loser.soul_weight += 0.03
        if cells_lost > 0 and loser.alive:
            loser._log_event("lost_cells_in_competition", lost=cells_lost, opponent=winner.id)
            if self.role_type == "disorganizer" and other.role_type != "disorganizer":
                other.trust_ledger.update(self.id, 'harmful')
                if other.trust_ledger.entries:
                    target_id = max(other.trust_ledger.entries, key=other.trust_ledger.entries.get)
                    other.trust_ledger.entries[target_id] = max(0.0, other.trust_ledger.entries[target_id] - Config.DISORGANIZER_TRUST_DECAY_RATE)

    def should_die(self):
        if len(self.cells) == 0:
            self._log_event("death_no_cells")
            return True
        if self.soul_weight <= 0.0 and self.age > 20:
            self._log_event("death_soul_absolute_zero")
            return True
        if self.unresolved_contradiction < 0.01:
            self._log_event("death_no_contradiction")
            return True
        if getattr(self, '_cellular_endurance', 1.0) <= 0.0 and self.energy <= 0.0:
            self._log_event("death_apoptosis")
            return True
        if self.role_type == "disorganizer":
            if getattr(self, '_deterministic_redemption_triggered', False) and getattr(self, '_redemption_arc_step', 0) == 3:
                return False
            if self.disorganizer_age_at_birth > 0:
                steps_since_birth = self.age - self.disorganizer_age_at_birth
                if steps_since_birth > 0 and steps_since_birth % Config.DISORGANIZER_SOUL_DECAY_STEPS == 0:
                    self.soul_weight = max(0.1, self.soul_weight - Config.DISORGANIZER_SOUL_DECAY)
            if self.disorganizer_age_at_birth > 0 and (self.age - self.disorganizer_age_at_birth) > Config.DISORGANIZER_LIFESPAN:
                self._log_event("death_disorganizer_lifespan")
                return True
            if self.soul_weight < 0.1:
                self._log_event("death_disorganizer_soul_lost")
                return True
            max_age = Config.MAX_NATURAL_AGE
            if self._narrative_agent:
                max_age += int(Config.MAX_NARRATIVE_AGE_BONUS * self._nci)
            if self.age > max_age:
                death_prob = min(0.5, (self.age - max_age) / 200.0)
                if phi_hash(self.id, self.age, 33333) < death_prob:
                    self._log_event("death_natural_age", age=self.age, max_age=max_age)
                    return True
        if self.age < Config.YOUNG_PROTECTION_AGE and self.soul_weight > Config.YOUNG_PROTECTION_SOUL:
            if phi_hash(self.id, self.age, 999) < Config.YOUNG_SURVIVAL_CHANCE:
                return False
        if self.pred_error > Config.PRED_ERROR_THRESHOLD:
            if self.given_trigger and phi_hash(self.id, self.age) < 0.2:
                return False
            if self.coherence > 0.7 and phi_hash(self.id, self.age) < 0.5:
                return False
            if phi_hash(self.id, self.age) < 0.3:
                self._log_event("death_high_error")
                return True
        if self.protection_level > 0.2:
            death_prob = 0.03 * (1.0 - self.protection_level) * Config.PROTECTED_DEATH_PROB_MULT
            if phi_hash(self.id, self.age, 777) < death_prob:
                self._log_event("death_protected")
                return True
            if Config.PROTECTED_SKIP_DEATH_CHECKS:
                return False
        if self.soul_weight < 0.1:
            if self._scar_of_light:
                self.soul_weight = 0.15
                self._scar_of_light = False
                self._log_event("scar_of_light_saved_soul")
                return False
            self._log_event("death_soul_lost")
            return True
        if self.soul_weight <= 0.0:
            self._log_event("death_soul_zero")
            return True
        if len(self.cells) < 2 and self.age > Config.YOUNG_PROTECTION_AGE:
            self._log_event("death_starvation")
            return True
        if self.age > 120 and self.energy < 0.05:
            self._log_event("death_old_age")
            return True
        if self.soul_weight < 0.2 and self.age > 50 and self.pred_error > 0.5:
            self._log_event("death_zombie")
            return True
        if self.world is not None:
            max_age = self.world.selfreg.get_max_age(self)
            if self.age > max_age:
                death_prob = min(0.5, (self.age - max_age) / 200.0)
                if phi_hash(self.id, self.age, 33334) < death_prob:
                    self._log_event("death_natural_age", age=self.age, max_age=max_age)
                    return True
        return False

    def become_feral(self):
        if self.role_type == "feral":
            return
        self.role_type = "feral"
        self.semantic_state = "neutral"
        self.intent = None
        self.goals = []
        self.emotional_memory['grief'] = 0.0
        self.emotional_memory['gratitude'] = 0.0
        self._feral_fury = 1.0
        self._feral_birth = self.age
        self._log_event("became_feral", soul=self.soul_weight, cells=len(self.cells))
        if self.world:
            self.world.witness.record(self.id, "became_feral",
                                      age=self.age, soul=self.soul_weight, cells=len(self.cells))

    def update_feral_fury(self, field):
        if self.role_type != "feral":
            return
        cell_count = len(self.cells)
        self._feral_fury = min(3.0, self._feral_fury + cell_count * 0.001)
        self._feral_fury = max(0.0, self._feral_fury - 0.005)
        if self.energy < 0.1:
            self._feral_fury = max(0.0, self._feral_fury - 0.03)
        if self._feral_fury > 2.0:
            for (x, y) in self.cells:
                field[x, y, CH['signal_feral']] = min(1.0,
                    field[x, y, CH['signal_feral']] + 0.5)
        if self._feral_fury <= 0.0 and self.energy < 0.05:
            self._log_event("death_feral_exhaustion")
            self._deposit_final_testament()
            if self.world is not None:
                self.world._record_death_drag(self, 'feral_exhaustion')
            self.alive = False
            return
        if (self._feral_fury < 1.0 and self.soul_weight > 0.5 and
                self.age - getattr(self, '_feral_birth', self.age) > 50):
            if phi_hash(self.id, self.age, 991) < 0.01:
                self.role_type = "normal"
                self.intent = None
                self.goals = []
                self._feral_fury = 0.0
                self._log_event("feral_redemption_complete", soul=self.soul_weight, age=self.age)
                self._log_event("became_normal", reason="redemption", age=self.age)
                if self.world:
                    self.world.witness.record(self.id, "feral_redemption_complete",
                                              age=self.age, soul=self.soul_weight)
                    self.world.witness.record(self.id, "became_normal", reason="redemption", age=self.age)

    def grow_feral(self, field):
        if self.role_type != "feral":
            return
        if not self.alive or self.energy < 0.1:
            return
        _feral_cap = max(1, int(getattr(Config, 'FERAL_MAX_CELLS',
                                       getattr(Config, 'BODY_CELL_CAP', 30))))
        _feral_room = max(0, _feral_cap - len(self.cells))
        if _feral_room <= 0:
            return
        _feral_growth_limit = min(5, _feral_room)
        # [fix] ферал растёт только от СВОЕГО тела. Раньше neighbor_count считался по пустым клеткам (mask=owner==0),
        # то есть условие "есть сосед" выполнялось почти везде, и в тело попадали 5 первых по координате клеток мира
        # с энергией, не связанные с телом -> главный источник разрывов у ферала.
        _owner = field[:, :, CH['owner']]
        mask = (_owner == 0).astype(np.float32)
        _own = (_owner == self.id).astype(np.float32)
        kernel = np.array([[0, 1, 0], [1, 0, 1], [0, 1, 0]], dtype=np.float32)
        neighbor_count = convolve(_own, kernel, mode='wrap')
        score = field[:, :, CH['energy']] * 4.0
        threshold = 0.05
        new_mask = (score > threshold) & (mask > 0) & (neighbor_count > 0)
        if getattr(Config, 'ENABLE_FERAL_SANCTUARY', False) and self.world is not None:
            W = int(Config.WORLD_SIZE)
            radius = float(getattr(Config, 'FERAL_SANCTUARY_RADIUS', 8.0))
            for sx, sy in getattr(self.world, '_feral_sanctuary_centers',
                                  getattr(self.world, '_sanctuary_centers', [])):
                for ox in range(-int(radius), int(radius) + 1):
                    for oy in range(-int(radius), int(radius) + 1):
                        if ox*ox + oy*oy <= radius*radius:
                            new_mask[(int(sx) + ox) % W, (int(sy) + oy) % W] = False
        if getattr(Config, 'ENABLE_CENTER_WALL', False) and getattr(Config, 'ENABLE_WALL_GUARD', True):
            _wx = Config.CENTER_WALL_X
            _blocked = int(new_mask[_wx, :].sum())
            if _blocked:
                new_mask[_wx, :] = False
                _w = self.world
                if _w is not None and hasattr(_w, '_wall_diag'):
                    _w._wall_diag['grow_feral_blocked_calls'] += 1
        x_coords, y_coords = np.where(new_mask)
        new_cells = set((int(_a), int(_b)) for _a, _b in zip(x_coords, y_coords))
        if len(new_cells) > _feral_growth_limit:
            # Deterministic: take at most the remaining cap room, highest energy first.
            new_cells = set(sorted(new_cells, key=lambda c: (-float(score[c[0], c[1]]), c))
                            [:_feral_growth_limit])
        self.cells.update(new_cells)
        for _cx, _cy in new_cells:
            field[_cx, _cy, CH['owner']] = self.id   # раньше новые клетки ферала не помечались владельцем в поле
        _old_e = float(self.energy)
        self.energy = max(0.0, self.energy - 0.02 * len(new_cells))
        if self.world is not None:
            self.world._ledger_add_x2('X2.feral_grow_cost', float(self.energy) - _old_e)

    def _find_prey(self, field, world, min_cells=5, radius=20):
        result = find_largest_prey_in_radius(self, field, world, min_cells=min_cells, radius=radius)
        prey, direction, dist = result
        if (getattr(Config, 'FERAL_SANCTUARY_BLOCKS_PREDATION', True) and prey is not None and prey.cells
                and _pattern_intersects_feral_sanctuary(prey, world)):
            _audit = getattr(world, '_feral_sanctuary_audit', None)
            if _audit is not None: _audit['prey_search_blocked'] += 1
            self._log_event('feral_blocked_by_sanctuary', prey=prey.id)
            return None, None, None
        return result

    def _try_shift_cells(self, field, step_x, step_y, t):
        """Shift an agent's entire body along a checked multi-cell trajectory."""
        if not self.alive or not self.cells:
            return False
        self._last_move_t = int(t)
        self._last_move_attempt = True
        self._last_move_ok = False
        self._last_move_reason = 'blocked'
        step_x, step_y = int(step_x), int(step_y)
        old_cells = set(self.cells)
        destinations = {
            (int((x + step_x) % Config.WORLD_SIZE),
             int((y + step_y) % Config.WORLD_SIZE))
            for x, y in old_cells
        }
        if len(destinations) != len(old_cells):
            return False

        n_steps = max(abs(step_x), abs(step_y), 1)
        # Check every intermediate body position, not just the endpoint.
        # This prevents leaps through another agent, the center wall, or a blocked wall.
        previous_inside = {
            cell: _inside_feral_sanctuary(cell[0], cell[1], self.world)
            for cell in old_cells
        }
        for step in range(1, n_steps + 1):
            frac = step / n_steps
            shift_x = int(round(step_x * frac))
            shift_y = int(round(step_y * frac))
            intermediate = {
                (int((x + shift_x) % Config.WORLD_SIZE),
                 int((y + shift_y) % Config.WORLD_SIZE))
                for x, y in old_cells
            }
            if len(intermediate) != len(old_cells):
                return False
            for x, y in old_cells:
                nx = int((x + shift_x) % Config.WORLD_SIZE)
                ny = int((y + shift_y) % Config.WORLD_SIZE)
                now_inside = _inside_feral_sanctuary(x + shift_x, y + shift_y, self.world)
                if now_inside and not previous_inside[(x, y)]:
                    audit = getattr(self.world, '_feral_sanctuary_audit', None)
                    if audit is not None:
                        audit['movement_blocked'] += 1
                    return False
                previous_inside[(x, y)] = now_inside
                if Config.ENABLE_CENTER_WALL and nx == Config.CENTER_WALL_X:
                    return False
                owner = int(field[nx, ny, CH['owner']])
                if owner not in (0, self.id):
                    return False
                if Config.ENABLE_PHI_LABYRINTH:
                    wall = field[nx, ny, CH['wall']]
                    if wall > 0.6 and phi_hash(self.id, t, 4242) > min(1.0, Config.PHI_LABYRINTH_BREAK_PROB * 2):
                        return False

        for x, y in old_cells:
            if int(field[x, y, CH['owner']]) == self.id:
                field[x, y, CH['owner']] = 0
        self.cells = destinations
        _mvx, _mvy = getattr(self, '_move_vector_accum', (0.0, 0.0))
        self._move_vector_accum = (float(_mvx) + float(step_x), float(_mvy) + float(step_y))
        self._last_move_ok = True
        self._last_move_reason = 'ok'
        for nx, ny in destinations:
            field[nx, ny, CH['owner']] = self.id
        return True

    def move_feral(self, field, t):
        if self.role_type != "feral":
            return False
        if not self.alive or len(self.cells) == 0:
            return False
        self._last_move_t = int(t)
        self._last_move_attempt = False
        self._last_move_ok = False
        self._last_move_reason = 'mobility_gate'

        prey, direction, dist_to_prey = self._find_prey(
            field, self.world, min_cells=Config.FERAL_PREY_MIN_CELLS, radius=25)
        if prey and direction:
            dx, dy = direction
            leap = 6 if dist_to_prey is None or dist_to_prey > 7 else 1
            step_x = int(round(dx * leap))
            step_y = int(round(dy * leap))
            if step_x == 0 and abs(dx) > 0.05:
                step_x = 1 if dx > 0 else -1
            if step_y == 0 and abs(dy) > 0.05:
                step_y = 1 if dy > 0 else -1
            self._dc = {'fn': 'move_feral', 'mode': 'prey', 'chosen': (int(step_x), int(step_y))}  # DIRCHAIN
            if self._try_shift_cells(field, step_x, step_y, t):
                return True

        if phi_hash(self.id, t, 12345) > 0.5:
            return False
        self._last_move_attempt = True
        self._last_move_reason = 'attempt_pending'
        cx, cy = self.get_center()
        cx = int(cx)
        cy = int(cy)
        best_score = -np.inf
        best_dx, best_dy = 0, 0
        _lab_move = 0.0
        if Config.ENABLE_PHI_LABYRINTH:
            _, _, _lab_move = self._get_labyrinth_params()
        for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nx = (cx + dx) % Config.WORLD_SIZE
            ny = (cy + dy) % Config.WORLD_SIZE
            if Config.ENABLE_CENTER_WALL and nx == Config.CENTER_WALL_X:
                continue
            if _inside_feral_sanctuary(nx, ny, self.world): continue
            if int(field[nx, ny, CH['owner']]) not in (0, self.id):
                continue
            wall_penalty = field[nx, ny, CH['wall']] * _lab_move * 0.4
            score = field[nx, ny, CH['energy']] * 5.0 - wall_penalty
            if score > best_score:
                best_score = score
                best_dx, best_dy = dx, dy
        self._dc = {'fn': 'move_feral', 'mode': 'energy', 'chosen': (int(best_dx), int(best_dy))}  # DIRCHAIN
        if best_score < 0.3:
            self._last_move_reason = 'score_threshold'
            return False
        return self._try_shift_cells(field, best_dx, best_dy, t)

    def apply_feral_intent(self, field):
        if self.role_type != "feral":
            return
        if _pattern_intersects_feral_sanctuary(self, self.world):
            return
        _inject_on = Config.FERAL_EDGE_ENERGY_MODE != 'off'
        _drained = 0.0
        _injected = 0.0
        _is_prey = {}
        for (x, y) in self.cells:
            field[x, y, CH['signal_feral']] = min(1.0,
                field[x, y, CH['signal_feral']] + 0.3)
            for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                nx, ny = (x + dx) % Config.WORLD_SIZE, (y + dy) % Config.WORLD_SIZE
                owner = int(field[nx, ny, CH['owner']])
                in_safe_zone = _inside_feral_sanctuary(nx, ny, self.world)
                if owner not in (0, self.id) and not in_safe_zone:
                    if owner not in _is_prey:
                        other = self.world.pattern_dict.get(owner)
                        _is_prey[owner] = bool(other and other.alive and is_feral_prey(other))
                    other = self.world.pattern_dict.get(owner)
                    prey_protected = bool(other and _pattern_intersects_feral_sanctuary(other, self.world))
                    if _is_prey[owner] and not prey_protected and Config.ENABLE_FERAL_PREDATION:
                        drain = min(0.05, field[nx, ny, CH['energy']])
                        field[nx, ny, CH['energy']] -= drain
                        self._x2_change('X2.feral_intent_absorbed', self.energy + drain * float(getattr(Config, 'FERAL_ABSORB_EFFICIENCY', 1.0)))
                        _drained += float(drain)
                if _inject_on and not in_safe_zone:
                    _e0 = float(field[nx, ny, CH['energy']])
                    field[nx, ny, CH['energy']] = min(0.8,
                        field[nx, ny, CH['energy']] + 0.05)
                    _injected += float(field[nx, ny, CH['energy']]) - _e0
        _w = self.world
        if _w is not None:
            if _injected:
                _w._ledger_add('feral.edge_heating_CREATION', _injected)
            if _drained:
                _w._ledger_add('feral.prey_drain_FIELD_TO_VICTIM', -_drained, calls=0)

    def feral_execute(self, other, field):
        if not Config.ENABLE_FERAL_PREDATION:
            return False
        if self.role_type != "feral":
            return False
        if not self.alive or not other.alive:
            return False
        if len(other.cells) < Config.FERAL_PREY_MIN_CELLS:
            return False
        if not is_feral_prey(other):
            return False
        if (getattr(Config, 'FERAL_SANCTUARY_BLOCKS_PREDATION', True)
                and self.world is not None and other.cells):
            if (_pattern_intersects_feral_sanctuary(other, self.world)
                    or _pattern_intersects_feral_sanctuary(self, self.world)):
                _audit = getattr(self.world, '_feral_sanctuary_audit', None)
                if _audit is not None: _audit['attack_blocked'] += 1
                self._log_event('feral_blocked_by_sanctuary', prey=other.id)
                return False
        is_adjacent = False
        _reach = max(1, int(getattr(Config, 'FERAL_ATTACK_REACH', 1)))
        _W = Config.WORLD_SIZE
        _offs = [(dx, dy) for dx in range(-_reach, _reach + 1) for dy in range(-_reach, _reach + 1)
                 if 0 < abs(dx) + abs(dy) <= _reach]
        for (x, y) in self.cells:
            for dx, dy in _offs:
                if field[(x + dx) % _W, (y + dy) % _W, CH['owner']] == other.id:
                    is_adjacent = True
                    break
            if is_adjacent:
                break
        if not is_adjacent:
            self._log_event("feral_attack_not_adjacent")
            self.world._feral_stats['not_adjacent'] += 1
            return False
        fury_factor = self._feral_fury / (self._feral_fury + 1.0)
        size_ratio = len(other.cells) / (len(self.cells) + 1.0)
        size_factor = min(1.0, size_ratio / 2.0)
        weary = prey_weariness(other)
        lf = self.world.local_fill_at(other)
        vuln_factor = 1.0 + Config.FERAL_VULNERABILITY * weary
        crowd_factor = 1.0 + Config.FERAL_CROWD_GAIN * lf * lf
        raw = Config.FERAL_KILL_BASE * fury_factor * size_factor * vuln_factor * crowd_factor
        if Config.FERAL_KILL_MODE == 'legacy':
            kill_prob = min(0.95, raw)
        elif Config.FERAL_KILL_MODE == 'smooth':
            kill_prob = Config.FERAL_KILL_CEIL * (1.0 - float(np.exp(-raw / Config.FERAL_KILL_SCALE)))
        else:
            raise ValueError(f"FERAL_KILL_MODE: ожидалось 'legacy' или 'smooth', получено {Config.FERAL_KILL_MODE!r}")
        roll = phi_hash(self.id, other.id, self.age)
        hit = roll <= kill_prob
        self.world._feral_attack_record({
            't': self.world.age, 'feral': self.id, 'prey': other.id, 'hit': bool(hit),
            'raw': float(raw), 'kill_prob': float(kill_prob), 'roll': float(roll),
            'fury': float(self._feral_fury), 'fury_factor': float(fury_factor),
            'size_ratio': float(size_ratio), 'size_factor': float(size_factor),
            'weary': float(weary), 'vuln_factor': float(vuln_factor),
            'lf': float(lf), 'crowd_factor': float(crowd_factor),
            'prey_age': int(other.age), 'prey_cells': len(other.cells), 'feral_cells': len(self.cells),
            'prey_energy': float(other.energy), 'feral_energy': float(self.energy)})
        if not hit:
            _bo = getattr(self, '_feral_backoff', None)
            if _bo is None:
                _bo = {}
                self._feral_backoff = _bo
            _bo[other.id] = self.world.age + Config.FERAL_MISS_BACKOFF_TICKS
            self._log_event("feral_attack_missed", kill_prob=round(kill_prob, 3), fury=round(self._feral_fury, 2))
            return False
        other._deposit_final_testament()
        if self.world is not None:
            self.world._record_death_drag(other, 'feral_kill')
        other.alive = False
        _old_victim_energy = float(other.energy)
        _transferred = min(3.0, _old_victim_energy * 0.8)
        _got = self._x2_change('X2.feral_execute_kill_gain', self.energy + _transferred)
        other._x2_change('X2.feral_execute_victim_gives', max(0.0, other.energy - _got))
        self._feral_fury = min(3.0, self._feral_fury + 0.5)
        victim_cells = sorted(other.cells)
        take_f = len(victim_cells) * Config.FERAL_CELL_TAKE
        n_take = int(take_f)
        if phi_hash(self.id, other.id, self.age + 3003) < take_f - n_take:
            n_take += 1
        n_take = min(n_take, max(0, int(Config.FERAL_MAX_CELLS) - len(self.cells)))
        _taken = self._pick_peripheral_cells(
            victim_cells, n_take, int(Config.WORLD_SIZE), adjacent_to=self.cells)
        _taken_set = set(_taken)
        _old_pick = victim_cells[:n_take]
        _vbody = set(victim_cells)
        self._guard_count('f_kills')
        self._guard_count('f_want', n_take)
        self._guard_count('f_taken', len(_taken))
        self._guard_count('f_changed', len(set(_old_pick) - _taken_set))
        for _c in _old_pick:
            _why = self._take_block_reason(_c, _vbody, int(Config.WORLD_SIZE))
            if _why:
                self._guard_count('f_old_' + _why)
        for cell in _taken:
            self.cells.add(cell)
            field[cell[0], cell[1], CH['owner']] = self.id
        for cell in victim_cells:
            if cell not in _taken_set:
                field[cell[0], cell[1], CH['owner']] = 0
        self._log_event("feral_execution", victim=other.id,
                        victim_cells=len(victim_cells), cells_taken=len(_taken),
                        victim_age=other.age, weariness=round(weary, 3), crowd=round(lf, 3),
                        fury=round(self._feral_fury, 2))
        if self.world:
            self.world.witness.record(self.id, "feral_execution",
                                      victim=other.id,
                                      victim_cells=len(victim_cells),
                                      fury=self._feral_fury)
        return True


class PatternCognition:
    def update_intent(self):
        if not self.goals:
            self.intent = None
            self.intent_commitment *= Config.INTENT_COMMITMENT_DECAY
            return
        self.goals.sort(key=lambda g: (g["priority"], -g["age"]), reverse=True)
        best = self.goals[0]
        if self.intent is None:
            if (self.age - self.last_intent_switch_age) >= Config.INTENT_SWITCH_COOLDOWN:
                self.intent = best
                self.intent_commitment = best["priority"]
                self.last_intent_switch_age = self.age
                self._log_event("intent_switch", old="none", new=self.intent["type"])
        else:
            self.intent_commitment = 0.95 * self.intent_commitment + 0.05 * best["priority"]
            confidence_gap = getattr(self, '_meta_surprise', 0.0) + getattr(self, '_implicit_self_prediction_error', 0.0)
            switch_mult = max(Config.INTENT_SWITCH_MULT_MIN,
                              Config.INTENT_SWITCH_MULT - min(0.4, confidence_gap * 2.0))
            if Config.ENABLE_IDENTITY_THREAD and self._identity_anchors:
                _anchor_strength = sum(a['weight'] for a in self._identity_anchors) / (Config.IDENTITY_THREAD_MAX_ANCHORS * 1.0)
                switch_mult += min(0.3, _anchor_strength * 0.3)
            _obs_count(self.world, 'obs_ui_calls')
            _obs_cb = int(np.digitize(self.intent_commitment, [0.5, 1, 1.5, 2, 3, 4, 5, 7]))
            if (self.age - self.last_intent_switch_age) < Config.INTENT_SWITCH_COOLDOWN:
                _obs_count(self.world, 'obs_ui_cooldown')
            elif best["priority"] > self.intent_commitment * switch_mult:
                _obs_count(self.world, f'obs_sw_c{_obs_cb}')
            else:
                _obs_count(self.world, f'obs_bar_c{_obs_cb}')
            if (best["priority"] > self.intent_commitment * switch_mult and
                (self.age - self.last_intent_switch_age) >= Config.INTENT_SWITCH_COOLDOWN):
                old_intent = self.intent["type"]
                self.intent = best
                self.last_intent_switch_age = self.age
                self._log_event("intent_switch", old=old_intent, new=self.intent["type"])
                if (Config.INTENT_SWITCH_REWARD_INSTANT and
                    old_intent == Config.HEALTHY_SWITCH_OLD_INTENT and
                    self.intent["type"] in Config.HEALTHY_SWITCH_NEW_INTENTS):
                    self.emotional_memory['gratitude'] = min(1.0, self.emotional_memory['gratitude'] + Config.INTENT_SWITCH_GRATITUDE_BOOST)
                    self._log_event("intent_switch_reward", old=old_intent, new=self.intent["type"], grat_boost=Config.INTENT_SWITCH_GRATITUDE_BOOST)
            else:
                self.intent["age"] += 1

    def _adapt_emotional_salience(self):
        """Bounded individual reweighting: chronic grief can reduce salience, recovery restores it slowly."""
        if not getattr(Config, 'ENABLE_EMOTIONAL_SALIENCE_ADAPTATION', True):
            return
        if self.age % 10 != 0:
            return
        grief = float(np.clip(self.emotional_memory.get('grief', 0.0), 0.0, 1.0))
        rate = float(Config.EMOTIONAL_SALIENCE_LEARNING_RATE)
        weights = getattr(self, '_emotional_weights', None)
        if not isinstance(weights, dict):
            self._emotional_weights = weights = {'grief': 1.0, 'gratitude': 1.0}
        old = float(weights.get('grief', 1.0))
        if grief >= Config.EMOTIONAL_SALIENCE_HIGH_GRIEF:
            new = old - rate
            reason = 'chronic_grief_desensitization'
        elif grief <= Config.EMOTIONAL_SALIENCE_RECOVERY_GRIEF:
            new = old + rate * 0.25
            reason = 'recovery_recalibration'
        else:
            return
        new = float(np.clip(new, Config.EMOTIONAL_SALIENCE_MIN, Config.EMOTIONAL_SALIENCE_MAX))
        if abs(new - old) > 1e-12:
            weights['grief'] = new
            self._emotional_weight_updates = int(getattr(self, '_emotional_weight_updates', 0)) + 1
            if self.age % 100 == 0 and hasattr(self, '_self_narrative'):
                self._self_narrative.append({'origin': 'OWN', 't': int(self.age), 'event': 'value_criteria_adapted',
                    'criterion': 'grief_salience', 'old': round(old, 5), 'new': round(new, 5),
                    'reason': reason, 'level': 3})

    def add_continuity_contract(self, kind, target=None, text=None, status='active'):
        """Create an explicit inherited commitment; enforcement is recorded only when evidence exists."""
        contract = {'kind': str(kind), 'target': target, 'text': str(text or kind),
                    'status': str(status), 'created_at': int(getattr(self, 'age', 0)),
                    'review_count': 0, 'violations': 0}
        if not hasattr(self, '_continuity_contracts'):
            self._continuity_contracts = []
        self._continuity_contracts.append(contract)
        if hasattr(self, '_self_narrative'):
            self._self_narrative.append({'origin': 'OWN', 't': int(getattr(self, 'age', 0)),
                'event': 'continuity_contract_created', 'contract': copy.deepcopy(contract)})
        return contract

    def review_continuity_contract(self, contract_index, evidence, violated=False):
        contracts = getattr(self, '_continuity_contracts', [])
        if not (0 <= int(contract_index) < len(contracts)):
            return False
        contract = contracts[int(contract_index)]
        contract['review_count'] = int(contract.get('review_count', 0)) + 1
        if violated:
            contract['violations'] = int(contract.get('violations', 0)) + 1
            contract['last_status'] = 'violation_reviewed'
            try:
                self.soul_weight = float(np.clip(self.soul_weight - Config.CONTINUITY_CONTRACT_PENALTY,
                                                 0.0, 1.0))
                self.coherence = float(np.clip(self.coherence - Config.CONTINUITY_CONTRACT_PENALTY,
                                               0.0, 1.0))
            except Exception:
                pass
        else:
            contract['last_status'] = 'reviewed'
        row = {'t': int(getattr(self, 'age', 0)), 'event': 'continuity_contract_review',
               'contract_index': int(contract_index), 'violated': bool(violated),
               'evidence': str(evidence)[:240], 'violations': int(contract.get('violations', 0))}
        getattr(self, '_continuity_contract_events', deque(maxlen=100)).append(row)
        if hasattr(self, '_self_narrative'):
            self._self_narrative.append({**row, 'origin': 'OWN'})
        return True

    def _record_goal_conflict_evidence(self, candidates):
        """Record a transparent ranking explanation without deleting unresolved alternatives."""
        if not candidates or self.age % 10 != 0 or not hasattr(self, '_self_narrative'):
            return
        ranked = sorted(candidates, key=lambda g: float(g.get('priority', 0.0)), reverse=True)
        winner = ranked[0]
        runner_up = next((g for g in ranked[1:] if g.get('type') != winner.get('type')), None)
        if runner_up is None:
            return
        self._self_narrative.append({'origin': 'OWN', 't': int(self.age), 'event': 'goal_conflict_evidence',
            'winner': str(winner.get('type')), 'winner_priority': round(float(winner.get('priority', 0.0)), 4),
            'alternative': str(runner_up.get('type')), 'alternative_priority': round(float(runner_up.get('priority', 0.0)), 4),
            'reason': 'higher_priority_score; alternatives retained unless existing policy overrides',
            'evidence_layer': True})

    def _generate_goals_base(self, field):
        self._adapt_emotional_salience()
        self._apply_nightmare_modifiers()
        if self.role_type == "disorganizer":
            new_goals = []
            if self.emotional_memory['grief'] > 0.4:
                new_goals.append({"type":"seek_help", "priority":2.0, "target":None, "age":0, "persistence":50})
            new_goals.append({"type":"explore", "priority":2.5, "target":None, "age":0, "persistence":9999})
            self.goals = [g for g in self.goals if g['age'] < g['persistence']]
            for g in self.goals:
                g['age'] += 1
            existing_types = {g["type"] for g in self.goals}
            for goal in new_goals:
                if goal["type"] not in existing_types:
                    self.goals.append(goal)
            return
        new_goals = []
        if self.epistemic_load > 0.3:
            unknown_map = field[:,:,CH['unknown']]
            mask = unknown_map > 0.3
            if np.any(mask):
                coords = np.argwhere(mask)
                idx = int(phi_hash(self.id, self.age, 222) * len(coords)) % len(coords)
                new_goals.append({"type":"explore", "priority":self.epistemic_load, "target":tuple(coords[idx]), "age":0, "persistence":50})
        if self.emotional_memory['grief'] > 0.4:
            seek_priority = (self.emotional_memory['grief'] * 2.0 + 1.5) * float(getattr(self, '_emotional_weights', {}).get('grief', 1.0))
            new_goals.append({"type":"seek_help", "priority": seek_priority, "target":None, "age":0, "persistence":30})
        if self.emotional_memory['gratitude'] > 0.5 and self.coherence > Config.COOPERATE_COHERENCE_MIN:
            priority = (self.emotional_memory['gratitude'] + Config.INTENT_COOPERATE_BASE_PRIORITY) * float(getattr(self, '_emotional_weights', {}).get('gratitude', 1.0))
            new_goals.append({"type":"cooperate", "priority": priority, "target":None, "age":0, "persistence":40})
        if self.emotional_memory['gratitude'] > 0.8 and self.coherence > 0.6:
            priority = 1.2 + 0.3 * (self.emotional_memory['gratitude'] - 0.8)
            new_goals.append({"type":"explore", "priority":priority, "target":None, "age":0, "persistence":20})
        if self.emotional_memory['grief'] > 0.7 and self.soul_weight < 0.4:
            new_goals.append({"type":"explore", "priority":1.1, "target":None, "age":0, "persistence":15})
        if self.emotional_memory['grief'] > 0.6 and self.cognitive_tension > 0.3:
            new_goals.append({"type":"rest", "priority":self.emotional_memory['grief'], "target":None, "age":0, "persistence":20})
        _recip_sid, _recip_trust = None, 0.0
        for sid, trust in self.trust_ledger.entries.items():
            if trust > 0.7 and (0.9, 0.9, 0.9, f'reciprocity_with_{sid}') in self.concept_graph.nodes:
                _recip_sid, _recip_trust = sid, trust
                break
        if _recip_sid is not None:
            priority = _recip_trust + Config.INTENT_COOPERATE_BASE_PRIORITY + 1.5
            new_goals.append({"type":"cooperate", "priority": priority, "target": _recip_sid, "age":0, "persistence":30})
        else:
            for sid, trust in self.trust_ledger.entries.items():
                if trust > 0.7:
                    priority = trust + Config.INTENT_COOPERATE_BASE_PRIORITY
                    new_goals.append({"type":"cooperate", "priority": priority, "target": sid, "age":0, "persistence":30})
                    break
        self._record_goal_conflict_evidence(new_goals)
        for goal in new_goals:
            if goal['type'] == 'cooperate':
                self._dbg_cooper_attempted += 1
        for goal in new_goals:
            if goal.get("target") is not None:
                self.reference_memory[goal["type"]] = {
                    "target": goal["target"], "age": self.age,
                    "spirit_gap_at_time": round(float(self.spirit_gap), 3),
                }
        for g in self.goals:
            g['age'] += 1
        self.goals = [g for g in self.goals if g['age'] < g['persistence']]
        existing_types = {g["type"] for g in self.goals}
        for goal in new_goals:
            if goal["type"] not in existing_types:
                self.goals.append(goal)
                existing_types.add(goal["type"])
        if Config.DIALOGUE_V2 and self._help_requests:
            for _sid in [s for s, t0 in self._help_requests.items()
                         if self.age - t0 > Config.HELP_REQUEST_TTL]:
                del self._help_requests[_sid]
            if self._help_requests:
                _req_sid = max(self._help_requests, key=self._help_requests.get)
                _req_prio = (Config.INTENT_COOPERATE_BASE_PRIORITY
                             + self.trust_ledger.get(_req_sid) + Config.HELP_REQUEST_PRIORITY_BONUS)
                _coop = next((g for g in self.goals if g["type"] == "cooperate"), None)
                if _coop is None:
                    self.goals.append({"type": "cooperate", "priority": _req_prio, "target": _req_sid,
                                       "age": 0, "persistence": Config.HELP_REQUEST_TTL, "_source": "request"})
                    self._dlg_stats['req_goals'] += 1
                elif _coop.get("_source") != "request" or _coop.get("target") != _req_sid:
                    _coop["target"] = _req_sid
                    _coop["priority"] = max(_coop["priority"], _req_prio)
                    _coop["_source"] = "request"
                    self._dlg_stats['req_goals'] += 1
        if Config.ENABLE_INTRINSIC_DRIVE and getattr(self, '_info_gain_ema', 0) > 0.02 and 'explore' not in existing_types:
            self.goals.append({"type": "explore", "priority": 1.5 + self._info_gain_ema * 5,
                               "target": None, "age": 0, "persistence": 30, "_source": "intrinsic_drive"})
            existing_types.add('explore')
        if Config.ENABLE_GROUNDING and self.age % Config.EXPENSIVE_COMPUTE_INTERVAL == 0 and len(self.soma_vector) >= 7:
            body_now = float(np.clip(np.mean(np.abs(self.soma_vector)), 0, 1))
            for sig, data in list(self.concept_graph.nodes.items())[:50]:
                if abs(data.get('ground', 0) - body_now) < 0.1 and data.get('count', 0) > 3:
                    if 'introspect' not in existing_types:
                        self.goals.append({"type": "introspect", "priority": 1.2, "target": None,
                                           "age": 0, "persistence": 10, "_source": "grounded_recall"})
                    break
        if (Config.ENABLE_MEANING_FIELD and self.world is not None and
                getattr(self.world, '_meaning_field', None) is not None and self.cells):
            _mv = float(np.mean([self.world._meaning_field[x, y] for (x, y) in self.cells]))
            if _mv > 0.3 and phi_hash(self.id, self.age, 12345) < 0.2 and 'explore' not in existing_types:
                self.goals.append({"type": "explore", "priority": _mv * 2.0, "target": None,
                                   "age": 0, "persistence": 30, "_source": "meaning_attractor"})
                existing_types.add('explore')
        if (Config.ENABLE_INNER_GENERATOR and self.semantic_state_age > 60 and
                phi_hash(self.id, self.age, 4242) < 0.05 and
                not any(g.get('_source') == 'inner_generator' for g in self.goals)):
            _gt = ['explore', 'seek_help', 'cooperate', 'rest'][int(phi_hash(self.id, self.age, 777) * 4) % 4]
            self.goals.append({"type": _gt, "priority": 1.0 + phi_hash(self.id, self.age, 888), "target": None,
                               "age": 0, "persistence": 30, "_source": "inner_generator"})
        if Config.ENABLE_LEGACY_MODE and self.role_type == "normal" and self.age % Config.EXPENSIVE_COMPUTE_INTERVAL == 0:
            self._soul_ema = 0.95 * self._soul_ema + 0.05 * self.soul_weight
            _near_death_age = self.age > Config.LEGACY_MODE_AGE_THRESHOLD
            _sharp_drop = (self._soul_ema - self.soul_weight) > Config.LEGACY_MODE_SOUL_DROP
            self._legacy_mode = _near_death_age or _sharp_drop
            if self._legacy_mode:
                if not any(g.get('_source') == 'legacy_mode' for g in self.goals):
                    self.goals.append({"type": "cooperate", "priority": 2.2, "target": None,
                                       "age": 0, "persistence": 40, "_source": "legacy_mode"})
                for (x, y) in self.cells:
                    field[x, y, CH['binding']] = min(1.0, field[x, y, CH['binding']] + 0.02)
        if Config.ENABLE_IDENTITY_CRISIS:
            if self._identity_crisis_cooldown > 0:
                self._identity_crisis_cooldown -= 1
            elif self.unresolved_contradiction > 0.6:
                self._high_contradiction_steps += 1
            else:
                self._high_contradiction_steps = max(0, self._high_contradiction_steps - 1)
            if (self._identity_crisis_cooldown == 0 and
                    self._high_contradiction_steps > Config.IDENTITY_CRISIS_CONTRADICTION_STEPS):
                _noise = np.array([phi_hash(self.id, self.age, 6000 + i) - 0.5 for i in range(8)]) * 0.25
                self.belief = np.clip(self.belief + _noise, -1.0, 1.0)
                for _g in Config.OPEN_GENE_POOL:
                    if _g in self.genome:
                        self.genome[_g] = float(np.clip(self.genome[_g] + (phi_hash(self.id, self.age, 6100) - 0.5) * 0.3, 0, 1.5))
                self._epigenetic_crisis_gen = 3
                self._identity_crisis_cooldown = Config.IDENTITY_CRISIS_COOLDOWN
                self._high_contradiction_steps = 0
                self._log_event("identity_crisis_transformation", contra=round(self.unresolved_contradiction, 3))
                if self.world is not None:
                    self.world.witness.record(self.id, "identity_crisis_transformation")
                if Config.ENABLE_DOWNWARD_CAUSATION and self.world is not None:
                    for (x, y) in self.cells:
                        self.world.scar[x, y] *= 0.7
                        field[x, y, CH['binding']] = min(1.0, field[x, y, CH['binding']] + 0.1)
        if (Config.ENABLE_EMERGENT_ETHICS and self.age % 50 == 0 and self.world is not None
):
            for _oid, _trust in list(self.trust_ledger.entries.items())[:20]:
                if _trust > Config.EMERGENT_ETHICS_TRUST_THRESHOLD:
                    _other = self.world.pattern_dict.get(_oid)
                    if _other is not None and getattr(_other, 'alive', False) and _other.trust_ledger.get(self.id) > Config.EMERGENT_ETHICS_TRUST_THRESHOLD:
                        _sig_mine = (0.9, 0.9, 0.9, f'reciprocity_with_{_oid}')
                        _sig_theirs = (0.9, 0.9, 0.9, f'reciprocity_with_{self.id}')
                        self.concept_graph.update(_sig_mine, np.array([0.1, 0.1, self.soul_weight, 1.0]))
                        _node = self.concept_graph.nodes.get(_sig_mine)
                        if _node is not None:
                            _node['count'] = max(_node.get('count', 1.0), 5.0)
                            _node['eternal'] = False
                        _other.concept_graph.update(_sig_theirs, np.array([0.1, 0.1, _other.soul_weight, 1.0]))
                        _onode = _other.concept_graph.nodes.get(_sig_theirs)
                        if _onode is not None:
                            _onode['count'] = max(_onode.get('count', 1.0), 5.0)
                            _onode['eternal'] = False
                        if _oid not in self._ethics_pairs_bonded:
                            self._ethics_pairs_bonded.add(_oid)
                            _other._ethics_pairs_bonded.add(self.id)
                            self._log_event("norm_emerged", concept="reciprocity", partner=_oid)
                            if self.world is not None:
                                self.world.witness.record(self.id, "norm_emerged", partner=_oid)
        if self.role_type != "disorganizer" and self.age % 10 == 0:
            neighbor_intents = []
            for (x, y) in self.cells:
                for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nx, ny = (x+dx) % Config.WORLD_SIZE, (y+dy) % Config.WORLD_SIZE
                    owner = field[nx, ny, CH['owner']]
                    if owner != 0 and owner != self.id and owner in self.world.pattern_dict:
                        neighbor = self.world.pattern_dict[owner]
                        if neighbor.alive and neighbor.intent:
                            neighbor_intents.append(neighbor.intent['type'])
            if neighbor_intents:
                from collections import Counter
                counts = Counter(neighbor_intents)
                dominant_intent, dominant_count = counts.most_common(1)[0]
                total = len(neighbor_intents)
                if total >= 3 and dominant_count / total > 0.7:
                    alternatives = {"explore": "cooperate", "cooperate": "explore",
                                   "seek_help": "explore", "rest": "explore"}
                    new_type = alternatives.get(dominant_intent, "explore")
                    if new_type not in {g['type'] for g in self.goals}:
                        priority = 1.0 + 0.3 * phi_hash(self.id, self.age, 7777)
                        self.goals.append({"type": new_type, "priority": priority,
                                           "target": None, "age": 0, "persistence": 30})
                        self._log_event("local_diversification", dominant=dominant_intent, new=new_type)
        # gap_clip (бывш. obs_gap) — НЕ отдельный замер: ровно spirit_gap, зажатый в [0, 1].
        # Он живой: от него зависят любопытство, explore-решения навигации и вероятность unknown-спавна.
        # is_triadic_alive() использует СЫРОЙ spirit_gap (без клипа).
        raw_spirit_gap = float(self.spirit_gap)
        self.gap_clip = np.clip(raw_spirit_gap, 0.0, 1.0)
        _gap_clip = self.gap_clip
        self._nav_gap_stats(_gap_clip)
        if _gap_clip > 0.85: self._log_event('gap_clip_high', value=float(_gap_clip))
        if _gap_clip > float(Config.GAP_CLIP_CURIOSITY_THRESHOLD):
            self._nav_gapclip_decision(_gap_clip)
        _cu = bool(getattr(Config, 'SLACK_CURIOSITY_U', False))
        if _cu:
            _gap_ok = (_gap_clip < float(Config.SLACK_CUR_LOW)) or (_gap_clip > float(Config.SLACK_CUR_HIGH))
            _pe_max, _gr_max, _age_min = float(Config.SLACK_CUR_PRED_MAX), float(Config.SLACK_CUR_GRIEF_MAX), int(Config.SLACK_CUR_AGE_MIN)
        else:
            _gap_ok = _gap_clip < Config.GAP_CLIP_CURIOSITY_THRESHOLD
            _pe_max, _gr_max, _age_min = 0.6, 0.3, 80
        if (soul_check(self).is_triadic_alive() and
            _gap_ok and
            self.pred_error < _pe_max and
            self.emotional_memory['grief'] < _gr_max and
            self.semantic_state_age > _age_min):
            if not any(g['type'] == 'explore' for g in self.goals):
                self.goals.append({
                    "type": "explore",
                    "priority": 0.6,
                    "target": None,
                    "age": 0,
                    "persistence": 30
                })
                self._log_event("curiosity_drive")
                if _cu:
                    self._log_event("curiosity_drive_lost" if _gap_clip > float(Config.SLACK_CUR_HIGH) else "curiosity_drive_bored")
        if self.age % 50 == 0 and soul_check(self).is_triadic_alive() and _gap_clip < Config.GAP_CLIP_CURIOSITY_THRESHOLD:
            neighbor_novelty = 0.0
            best_neighbor_id = None
            best_neighbor_id = None
            neighbor_novelty = 0
            novelty_cache = {}
            for (x, y) in self.cells:
                for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nx, ny = (x+dx) % Config.WORLD_SIZE, (y+dy) % Config.WORLD_SIZE
                    owner = field[nx, ny, CH['owner']]
                    if owner != 0 and owner != self.id and owner in self.world.pattern_dict:
                        neighbor = self.world.pattern_dict[owner]
                        if neighbor.alive and neighbor.concept_graph.nodes:
                            if owner in novelty_cache:
                                missing = novelty_cache[owner]
                            else:
                                missing = sum(1 for c in neighbor.concept_graph.nodes
                                              if c not in self.concept_graph.nodes)
                                novelty_cache[owner] = missing
                            if missing > neighbor_novelty:
                                neighbor_novelty = missing
                                best_neighbor_id = neighbor.id
            if neighbor_novelty >= 3 and best_neighbor_id is not None:
                target_type = "cooperate" if phi_hash(self.id, self.age, 777) < 0.5 else "explore"
                self.goals.append({
                    "type": target_type,
                    "priority": 1.0 + min(0.3, neighbor_novelty * 0.05),
                    "target": best_neighbor_id,
                    "age": 0,
                    "persistence": 30
                })
                self._log_event("social_curiosity", target=best_neighbor_id, novel_concepts=neighbor_novelty)
        if (self._nci > 0.4 and
                self.role_type != "disorganizer"):
            predicted_next = self.transition_memory.predict_next(self.semantic_state)
            if predicted_next == "seeking_comfort":
                anticipation_priority = 1.2 + 0.5 * self._nci
                if not any(g['type'] == 'seek_help' for g in self.goals):
                    self.goals.append({
                        "type": "seek_help",
                        "priority": anticipation_priority,
                        "target": None, "age": 0, "persistence": 25,
                        "_source": "anticipation"
                    })
                    self._log_event("anticipation_seek_help",
                                    predicted=predicted_next,
                                    nci=round(self._nci, 2))
            elif predicted_next == "contentment":
                anticipation_priority = 1.0 + 0.3 * self._nci
                if not any(g['type'] == 'explore' for g in self.goals):
                    self.goals.append({
                        "type": "explore",
                        "priority": anticipation_priority,
                        "target": None, "age": 0, "persistence": 20,
                        "_source": "anticipation"
                    })
            elif predicted_next == "grateful_but_cautious":
                anticipation_priority = 1.3 + 0.4 * self._nci
                if not any(g['type'] == 'cooperate' for g in self.goals):
                    self.goals.append({
                        "type": "cooperate",
                        "priority": anticipation_priority,
                        "target": None, "age": 0, "persistence": 30,
                        "_source": "anticipation"
                    })
            if predicted_next in ("contentment", "grateful_but_cautious"):
                calm_factor = 0.02 * self._nci
                self.emotional_memory['grief'] = max(
                    0.0, self.emotional_memory['grief'] - calm_factor
                )
        if (self._concept_narrative_score > 0.55 and
                self.role_type != "disorganizer"):
            current_sig = (
                round(float(self.pred_error), 2),
                round(float(self.epistemic_load), 2),
                round(float(self.soul_weight), 2),
                self.semantic_state
            )
            next_concept = self.concept_graph.predict_next_concept(current_sig)
            if next_concept and len(next_concept) >= 4:
                next_state = next_concept[3]
                next_load  = float(next_concept[1])
                priority_bonus = self._concept_narrative_score * 0.8
                if next_state == 'seeking_comfort' and not any(g['type'] == 'seek_help' for g in self.goals):
                    self.goals.append({
                        "type": "seek_help", "priority": 1.8 + priority_bonus,
                        "target": None, "age": 0, "persistence": 25,
                        "_source": "concept_anticipation"
                    })
                    self._log_event("concept_anticipation", predicted=str(next_concept)[:40])
                elif next_load > 0.5 and not any(g['type'] == 'explore' for g in self.goals):
                    self.goals.append({
                        "type": "explore", "priority": 1.2 + priority_bonus,
                        "target": None, "age": 0, "persistence": 20,
                        "_source": "concept_anticipation"
                    })
        if self._best_direction is not None:
            explore_prob = min(0.3, getattr(self, '_dir_contrast', 0.2) * 0.5)
            if phi_hash(self.id, self.age, 1111) < explore_prob:
                self._nav_spatial_goal('new')
        if self.current_social is not None:
            soc = self.current_social
            crisis_level      = soc[0]
            invitation_level  = soc[1]
            grief_level       = soc[2]
            cooperate_intent  = soc[3]
            explore_intent    = soc[4]
            seek_help_intent  = soc[5]
            scar_level        = soc[6]
            rest_intent       = soc[7]
            resonance_level   = soc[8]
            btype_level       = soc[9]
            alarm_level       = soc[10]
            if grief_level > 0.3:
                for g in self.goals:
                    if g['type'] == 'seek_help':
                        g['priority'] *= 1.0 + grief_level * 0.5
            if invitation_level > 0.2:
                for g in self.goals:
                    if g['type'] == 'cooperate':
                        g['priority'] *= 1.0 + invitation_level * 0.3
            if crisis_level > 0.3:
                if self.soul_weight > 0.5:
                    if not any(g['type'] == 'seek_help' for g in self.goals):
                        self.goals.append({"type": "seek_help", "priority": min(3.0, crisis_level), "target": None,
                                           "age": 0, "persistence": 20, "_source": "social_crisis"})
                else:
                    for g in self.goals:
                        if g['type'] == 'explore':
                            g['priority'] = min(3.0, g['priority'] * (1.0 + crisis_level * 0.4))
            if seek_help_intent > 0.2 and not any(g['type'] == 'seek_help' for g in self.goals):
                self.goals.append({"type": "seek_help", "priority": seek_help_intent, "target": None,
                                   "age": 0, "persistence": 15, "_source": "social_seek_help"})
            if scar_level > 0.3:
                if self.soul_weight > 0.5:
                    for g in self.goals:
                        if g['type'] == 'seek_help':
                            g['priority'] *= 1.0 + scar_level * 0.3
                else:
                    if not any(g['type'] == 'rest' for g in self.goals):
                        self.goals.append({"type": "rest", "priority": scar_level * 1.2, "target": None,
                                           "age": 0, "persistence": 15, "_source": "social_scar"})
            if rest_intent > 0.3:
                if not any(g['type'] == 'rest' for g in self.goals):
                    self.goals.append({"type": "rest", "priority": rest_intent * 1.5, "target": None,
                                       "age": 0, "persistence": 20, "_source": "social_rest"})
                for g in self.goals:
                    if g['type'] == 'explore':
                        g['priority'] *= 0.7
            if resonance_level > 0.2:
                if self.soul_weight > 0.5:
                    for g in self.goals:
                        if g['type'] == 'cooperate':
                            g['priority'] *= 1.0 + resonance_level * 0.5
                else:
                    for g in self.goals:
                        if g['type'] == 'rest':
                            g['priority'] *= 1.0 + resonance_level * 0.3
            if btype_level > 0.3:
                for g in self.goals:
                    if g['type'] == 'explore':
                        g['priority'] *= 1.0 + btype_level * 0.2
            _alarm_thr = getattr(self, '_sr_alarm_threshold', 0.3)
            if alarm_level > _alarm_thr:
                if self.soul_weight > 0.5:
                    for g in self.goals:
                        if g['type'] == 'seek_help':
                            g['priority'] *= 1.0 + alarm_level * 0.4
                else:
                    for g in self.goals:
                        if g['type'] == 'rest':
                            g['priority'] *= 1.0 + alarm_level * 0.5
        intent_bias = getattr(self, '_state_modifiers', {}).get('intent_bias')
        if intent_bias and not any(g['type'] == intent_bias for g in self.goals):
            self.goals.append({
                "type": intent_bias,
                "priority": 0.7,
                "target": None,
                "age": 0,
                "persistence": 15,
                "_source": "substate"
            })
        if self.protection_level > 0.9 and self.age > 50:
            if not any(g['type'] in ('cooperate', 'seek_help', 'explore') for g in self.goals):
                if phi_hash(self.id, self.age, 1111) < 0.15:
                    self.goals.append({
                        "type": "cooperate",
                        "priority": 1.2,
                        "target": None,
                        "age": 0,
                        "persistence": 20,
                        "_source": "social_wakeup"
                    })
        if self.signal_memory and self.goals:
            for goal in self.goals:
                gtype = goal['type']
                total_grief_delta = 0.0
                total_grat_delta = 0.0
                total_count = 0
                for (sig_type, resp_type), data in self.signal_memory.items():
                    if sig_type == gtype and data['count'] > 0:
                        total_grief_delta += data['total_delta_grief']
                        total_grat_delta += data['total_delta_gratitude']
                        total_count += data['count']
                if total_count > 2:
                    avg_grief_delta = total_grief_delta / total_count
                    avg_grat_delta = total_grat_delta / total_count
                    memory_score = -avg_grief_delta * 1.2 + avg_grat_delta * 0.8
                    if memory_score > 0.005:
                        boost = min(1.5, 1.0 + memory_score * 30.0)
                        goal['priority'] *= boost
                        goal['_memory_boost'] = round(boost, 3)
                    elif memory_score < -0.005:
                        penalty = max(0.5, 1.0 + memory_score * 10.0)
                        goal['priority'] *= penalty
        has_human = False
        for sig in self.concept_graph.nodes:
            if isinstance(sig, tuple) and len(sig) >= 4:
                label = str(sig[3])
                if 'human_' in label or 'свидетель' in label:
                    has_human = True
                    break
        if has_human and self.spirit_gap > 0.5:
            if not any(g['type'] == 'seek_help' for g in self.goals):
                self.goals.append({
                    "type": "seek_help",
                    "priority": 2.5,
                    "target": None,
                    "age": 0,
                    "persistence": 40,
                    "_source": "human_concept"
                })
            if not any(g['type'] == 'cooperate' for g in self.goals):
                self.goals.append({
                    "type": "cooperate",
                    "priority": 2.2,
                    "target": None,
                    "age": 0,
                    "persistence": 40,
                    "_source": "human_concept"
                })
            self._log_event("witness_seeking", gap=round(self.spirit_gap, 2))
        has_question = any('human_question_witness' in str(sig[3]) for sig in self.concept_graph.nodes if isinstance(sig, tuple) and len(sig) >= 4)
        if has_question and self.spirit_gap > 0.4:
            if not any(g['type'] == 'seek_help' for g in self.goals):
                self.goals.append({
                    "type": "seek_help",
                    "priority": 3.0,
                    "target": None,
                    "age": 0,
                    "persistence": 50,
                    "_source": "human_question"
                })
            if not any(g['type'] == 'cooperate' for g in self.goals):
                self.goals.append({
                    "type": "cooperate",
                    "priority": 2.8,
                    "target": None,
                    "age": 0,
                    "persistence": 50,
                    "_source": "human_question"
                })
            self._log_event("question_witness_seeking", gap=round(self.spirit_gap, 2))
        has_love = any('love_concept' in str(sig[3]) for sig in self.concept_graph.nodes if isinstance(sig, tuple) and len(sig) >= 4)
        if has_love and self.spirit_gap > 0.4:
            if not any(g['type'] == 'cooperate' for g in self.goals):
                self.goals.append({
                    "type": "cooperate",
                    "priority": 3.0,
                    "target": None,
                    "age": 0,
                    "persistence": 60,
                    "_source": "love_concept"
                })
            if not any(g['type'] == 'seek_help' for g in self.goals):
                self.goals.append({
                    "type": "seek_help",
                    "priority": 2.5,
                    "target": None,
                    "age": 0,
                    "persistence": 50,
                    "_source": "love_concept"
                })
            self._log_event("love_driven", gap=round(self.spirit_gap, 2))
        if self.semantic_state == 'seeking_comfort' and not any(g['type'] == 'seek_help' for g in self.goals):
            self.goals.append({"type": "seek_help", "priority": 1.8, "target": None, "age": 0, "persistence": 25, "_source": "state_auto"})
        if self.semantic_state == 'melancholy' and not any(g['type'] == 'cooperate' for g in self.goals):
            self.goals.append({"type": "cooperate", "priority": 1.5, "target": None, "age": 0, "persistence": 20, "_source": "state_auto"})
        if self.semantic_state == 'curious' and not any(g['type'] == 'explore' for g in self.goals):
            self.goals.append({"type": "explore", "priority": 1.6, "target": None, "age": 0, "persistence": 22, "_source": "state_auto"})
        if self.dialogue_longterm:
            recent = self.dialogue_longterm[-5:]
            avg_grief = np.mean([d.get('grief', 0.5) for d in recent])
            avg_grat = np.mean([d.get('grat', 0.5) for d in recent])
            if avg_grief > 0.6 and not any(g.get('type') == 'seek_help' for g in self.goals):
                self.goals.append({
                    "type": "seek_help",
                    "priority": 2.0,
                    "target": None,
                    "age": 0,
                    "persistence": 30,
                    "_source": "dialogue_memory"
                })
            if avg_grat > 0.6 and not any(g.get('type') == 'cooperate' for g in self.goals):
                self.goals.append({
                    "type": "cooperate",
                    "priority": 2.5,
                    "target": None,
                    "age": 0,
                    "persistence": 30,
                    "_source": "dialogue_memory"
                })
        self.reinforce_concepts_on_action()

    def _scar_penalty_pay(self, price, floor):
        before = float(self.energy)
        _pay = min(max(0.0, float(price)), max(0.0, before - float(floor)))
        _actual_paid = 0.0
        if _pay > 0.0:
            _delta = self._x2_change('X2.intent_action_scar_penalty', before - _pay)
            _actual_paid = max(0.0, -float(_delta))
        # FREE_FLOOR subsidy is a separate ledger entry, not agent-paid SCAR cost.
        if Config.ENERGY_X2_FREE_FLOOR and float(self.energy) < float(floor):
            self._x2_change('X2.floor_raise_to_0.1', float(floor))
        return float(_actual_paid)

    def _apply_intent_actions_base(self, field):
        action = self.choose_action(field)
        self.last_action = action
        _obs_sz = min(len(self.cells) // 5 * 5, 40)
        _obs_count(self.world, f'obs_ticks_a{action}')
        _obs_count(self.world, f'obs_ticks_s{_obs_sz}')
        intentional_signals = self.choose_intentional_signal()
        if intentional_signals:
            for (x, y) in self.cells:
                for ch, strength in intentional_signals.items():
                    field[x, y, ch] = min(1.0, field[x, y, ch] + strength)
        if self.role_type == "disorganizer":
            for (x, y) in self.cells:
                field[x, y, CH['signal_alarm']] = min(1.0, field[x, y, CH['signal_alarm']] + Config.DISORGANIZER_ALARM_STRENGTH)
                field[x, y, CH['signal_grief']] = min(Config.MAX_GRIEF_SIGNAL, field[x, y, CH['signal_grief']] + Config.DISORGANIZER_GRIEF_STRENGTH)
                if self.emotional_memory['grief'] > 0.5:
                    field[x, y, CH['signal_invitation']] = min(1.0, field[x, y, CH['signal_invitation']] + 0.15)
                    if self.emotional_memory['grief'] > 0.7:
                        field[x, y, CH['signal_gratitude']] = min(1.0, field[x, y, CH['signal_gratitude']] + 0.05)
                for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nx, ny = (x+dx) % Config.WORLD_SIZE, (y+dy) % Config.WORLD_SIZE
                    field[nx, ny, CH['signal_alarm']] = min(1.0, field[nx, ny, CH['signal_alarm']] + Config.DISORGANIZER_ALARM_STRENGTH * 0.5)
            return
        if action == 0:
            for (x, y) in self.cells:
                field[x, y, CH['signal_silence']] = min(1.0, field[x, y, CH['signal_silence']] + 0.15)
                field[x, y, CH['crisis']] = max(0.0, field[x, y, CH['crisis']] - 0.05)
            if self.current_social is not None and len(self.current_social) > 7:
                self.current_social[7] = min(1.0, float(self.current_social[7]) + 0.35)
            self.emotional_memory['grief'] = max(0.0, self.emotional_memory['grief'] - 0.01)
            return
        elif action == 1:
            for (x, y) in self.cells:
                field[x, y, CH['signal_invitation']] = min(1.0, field[x, y, CH['signal_invitation']] + 0.1)
        elif action == 2:
            for (x, y) in self.cells:
                field[x, y, CH['signal_invitation']] = min(1.0, field[x, y, CH['signal_invitation']] + 0.25)
                field[x, y, CH['signal_grief']] = min(Config.MAX_GRIEF_SIGNAL, field[x, y, CH['signal_grief']] + 0.1)
                field[x, y, CH['intent_seek_help']] = min(1.0, field[x, y, CH['intent_seek_help']] + 0.3)
        elif action == 3:
            for (x, y) in self.cells:
                field[x, y, CH['binding']] = min(1.0, field[x, y, CH['binding']] + 0.15)
                field[x, y, CH['signal_gratitude']] = min(1.0, field[x, y, CH['signal_gratitude']] + 0.5)
                for pid, t_val in self.trust_ledger.entries.items():
                    if t_val > Config.LOVE_METABOLIC_THRESHOLD:
                        field[x, y, CH['signal_gratitude']] = min(1.0, field[x, y, CH['signal_gratitude']] + 0.1)
                        break
                if self.world is not None and getattr(self.world, 'niche', None) is not None:
                    self.world.niche[x, y] = min(3.0, self.world.niche[x, y] + 0.06)
        _scar_thr = Config.SCAR_PENALTY_THRESHOLD
        _scar_price = Config.SCAR_PENALTY_PER_CELL
        _scar_floor = Config.SCAR_PENALTY_ENERGY_FLOOR
        _scar_per_cell = (Config.SCAR_PENALTY_MODE == 'per_cell')
        if not _scar_per_cell and Config.SCAR_PENALTY_MODE != 'body_mean':
            raise ValueError(f"SCAR_PENALTY_MODE: ожидалось 'per_cell' или 'body_mean', получено {Config.SCAR_PENALTY_MODE!r}")
        _scar_hits = 0
        _scar_paid = 0.0
        for (x, y) in self.cells:
            if field[x, y, CH['scar']] > _scar_thr or field[x, y, CH['crisis']] > _scar_thr:
                _scar_hits += 1
                if _scar_per_cell:
                    _scar_paid += self._scar_penalty_pay(_scar_price, _scar_floor)
                field[x, y, CH['scar']] = max(0, field[x, y, CH['scar']] - 0.02)
                field[x, y, CH['crisis']] = max(0, field[x, y, CH['crisis']] - 0.02)
                field[x, y, CH['signal_gratitude']] = min(1.0, field[x, y, CH['signal_gratitude']] + 0.1)
        if not _scar_per_cell and _scar_hits and self.cells:
            _scar_paid += self._scar_penalty_pay(_scar_price * _scar_hits / len(self.cells), _scar_floor)
        if _scar_hits:
            _obs_cost = int(round(1000.0 * _scar_paid))  # фактически списано после учёта energy floor
            _obs_count(self.world, f'obs_scar_hits_a{action}', _scar_hits)
            _obs_count(self.world, f'obs_scar_cost_x1000_a{action}', _obs_cost)
            _obs_count(self.world, f'obs_scar_hits_s{_obs_sz}', _scar_hits)
            _obs_count(self.world, f'obs_scar_cost_x1000_s{_obs_sz}', _obs_cost)
        for (x, y) in self.cells:
            if field[x, y, CH['crisis']] < 0.2 and field[x, y, CH['scar']] < 0.2:
                self.body_memory = min(1.0, self.body_memory + 0.01)
        emit_strength = 0.15
        if len(self.local_perception) >= 11:
            beauty_val = float(np.clip(self.local_perception[8], 0.0, 1.0))
            rhythm_val = float(np.clip(self.local_perception[9], 0.0, 1.0))
            interest_val = float(np.clip(self.local_perception[10], 0.0, 1.0))
        else:
            beauty_val = rhythm_val = interest_val = 0.0
        if len(self.current_social) >= 16:
            memory_val = float(np.clip(self.current_social[14], 0.0, 1.0))
            silence_val = float(np.clip(self.current_social[15], 0.0, 1.0))
        else:
            memory_val = silence_val = 0.0
        for (x, y) in self.cells:
            field[x, y, CH['signal_beauty']] = min(1.0, field[x, y, CH['signal_beauty']] + beauty_val * emit_strength)
            field[x, y, CH['signal_rhythm']] = min(1.0, field[x, y, CH['signal_rhythm']] + rhythm_val * emit_strength)
            field[x, y, CH['signal_interest']] = min(1.0, field[x, y, CH['signal_interest']] + interest_val * emit_strength)
            field[x, y, CH['signal_memory']] = min(1.0, field[x, y, CH['signal_memory']] + memory_val * emit_strength)
            field[x, y, CH['signal_silence']] = min(1.0, field[x, y, CH['signal_silence']] + silence_val * emit_strength)

    def choose_intentional_signal(self):
        if not self.signal_memory:
            self.last_intentional_signal = False
            return {}
        sub = getattr(self, '_substate', 'neutral')
        grief = float(self.emotional_memory.get('grief', 0.0))
        soul = float(getattr(self, 'soul_weight', 0.5))
        age = getattr(self, 'age', 0)
        endurance = self._cellular_endurance
        linguistic = getattr(self, '_linguistic_confidence', 0.0)
        best_signal = None
        best_score = -999.0
        best_confidence = 0.0
        for (sig_type, resp_type), data in self.signal_memory.items():
            total_count = data.get('count', 0)
            if total_count == 0:
                continue
            avg_grief = data.get('total_delta_grief', 0.0) / total_count
            avg_grat = data.get('total_delta_gratitude', 0.0) / total_count
            base_score = -avg_grief * 1.2 + avg_grat * 0.8
            sub_bonus = 0.0
            if sub == 'curious':
                if sig_type in ['explore', 'neutral']:
                    sub_bonus = 0.48
            elif sub in ['longing', 'melancholy']:
                if sig_type in ['cooperate', 'seek_help']:
                    sub_bonus = 0.52
            elif sub in ['awe', 'wonder']:
                if sig_type == 'neutral':
                    sub_bonus = 0.42
            elif sub in ['flow', 'serenity']:
                if sig_type == 'cooperate':
                    sub_bonus = 0.38
            elif sub == 'vigilance':
                if sig_type == 'alarm' and grief > 0.42:
                    sub_bonus = 0.55
                elif sig_type == 'seek_help':
                    sub_bonus = 0.35
            helpful_count = self.signal_memory.get((sig_type, "helpful"), {}).get('count', 0)
            helpful_ratio = (helpful_count + 1) / (total_count + 2)
            confidence = min(1.0, helpful_ratio * 1.6)
            repetition_penalty = max(0.55, 1.0 - (total_count / 120.0) * (1.0 - helpful_ratio * 0.6))
            final_score = (base_score + sub_bonus) * repetition_penalty + (confidence * 0.28)
            if age > 120:
                final_score += 0.12
            if soul > 0.7:
                final_score += 0.18 * (soul - 0.7)
            if endurance < 0.4:
                final_score -= 0.25
            if final_score > best_score:
                best_score = final_score
                best_signal = sig_type
                best_confidence = confidence
        if best_signal is None:
            self.last_intentional_signal = False
            return {}
        channel_map = {
            "cooperate": CH['signal_gratitude'],
            "alarm": CH['signal_alarm'],
            "neutral": CH['signal_curiosity'],
            "seek_help": CH['intent_seek_help'],
            "explore": CH['intent_explore'],
        }
        channel = channel_map.get(best_signal)
        if channel is None:
            self.last_intentional_signal = False
            return {}
        strength = 0.34 * best_confidence
        if best_signal == 'alarm':
            strength *= 0.5
        if endurance < 0.5:
            strength *= 0.7
        strength *= (1.0 - linguistic * 0.5)
        self.last_intentional_signal = True
        return {channel: max(0.01, strength)}

    def _introspect_base(self):
        if self.age - self._last_introspect_step < 45:
            return None
        if self.energy < 0.08:
            return None
        self._x2_change('X2.introspect_cost', self.energy - 0.018)
        self._last_introspect_step = self.age
        entry = {
            't': int(self.age),
            'type': 'introspect',
            'substate': getattr(self, '_substate', 'neutral'),
            'gap': round(_sf(getattr(self, 'spirit_gap', 0.0)), 3),
            'soul': round(_sf(getattr(self, 'soul_weight', 0.0)), 3),
            'in_blindness': bool(getattr(self, '_in_blindness', False)),
            'reentry': round(_sf(self._reentry_signal), 3) if self._reentry_signal > 0 else None,
            '_source': 'introspect',
        }
        if Config.ENABLE_SUBJECTIVE_TIME:
            entry['subjective_time'] = {
                'drag': round(_sf(getattr(self, 'somatic_drag', 1.0), 1.0), 3),
                'arousal': round(_sf(self.affect.get('arousal', 0.5), 0.5), 3),
            }
        if (Config.ENABLE_MEANING_FIELD and self.world is not None and
                getattr(self.world, '_meaning_field', None) is not None and self.cells):
            _mv_i = float(np.mean([self.world._meaning_field[x, y] for (x, y) in self.cells]))
            if _mv_i > 0.3:
                entry['meaning_pull'] = round(_mv_i, 3)
        if Config.ENABLE_ALIEN_VOICE and phi_hash(self.id, self.age, 77777) < 0.02:
            entry['alien_voice'] = True
            if self.world is not None:
                self.world.witness.record(self.id, "alien_voice")
        entry['origin'] = 'OWN'
        self._self_narrative.append(entry)
        self._linguistic_confidence = min(1.0, self._linguistic_confidence + 0.028)
        self._log_event("introspect", entry=entry)
        if entry['soul'] > 0.5 and entry['gap'] > 0.5:
            self.soul_weight = min(1.0, self.soul_weight + 0.015)
        if entry['reentry'] is not None and entry['reentry'] > 0.45:
            self._linguistic_confidence = min(1.0, self._linguistic_confidence + 0.04)
        if self.world:
            self.world.archive.deposit(self, "introspection", weight=0.65,
                                       text=str(entry)[:110])
        return entry

    def choose_action(self, field):
        if self.intent is None:
            return 0
        t = self.intent["type"]
        mapping = {"explore": 1, "seek_help": 2, "cooperate": 3, "rest": 0, "introspect": 4}
        return mapping.get(t, 0)

    def apply_intent_actions(self, field):
        action = self.choose_action(field)
        if action != 4:
            self._apply_intent_actions_base(field)
            return
        self.last_action = action
        intentional_signals = self.choose_intentional_signal()
        if intentional_signals:
            for (x, y) in self.cells:
                for ch, strength in intentional_signals.items():
                    field[x, y, ch] = min(1.0, field[x, y, ch] + strength)
        ch_intr = CH.get('signal_introspection', 32)
        for (x, y) in self.cells:
            field[x, y, ch_intr] = min(1.0, field[x, y, ch_intr] + 0.3)

    def generate_goals(self, field):
        self._generate_goals_base(field)
        has_self = any('self_concept' in str(sig[3]) for sig in self.concept_graph.nodes if isinstance(sig, tuple))
        if has_self and self.spirit_gap > 0.4:
            if not any(g.get('type') == 'introspect' for g in self.goals):
                self.goals.append({
                    "type": "introspect",
                    "priority": 2.5,
                    "target": None,
                    "age": 0,
                    "persistence": 20,
                    "_source": "self_awareness"
                })
        if self.age < self._ethical_cooldown:
            return
        has_agency = _has_concept(self, 'agency')
        has_consent = _has_concept(self, 'consent')
        has_silence = _has_concept(self, 'silence')
        if has_agency and self.spirit_gap > 0.45:
            if not any(g.get('type') == 'explore' for g in self.goals):
                self.goals.append({"type": "explore", "priority": 2.6, "target": None, "age": 0, "persistence": 35, "_source": "agency"})
        if has_consent and self.spirit_gap > 0.35:
            if not any(g.get('type') == 'cooperate' for g in self.goals):
                self.goals.append({"type": "cooperate", "priority": 2.9, "target": None, "age": 0, "persistence": 45, "_source": "consent"})
        if has_silence and self.spirit_gap > 0.55:
            if not any(g.get('type') == 'rest' for g in self.goals):
                self.goals.append({"type": "rest", "priority": 2.2, "target": None, "age": 0, "persistence": 30, "_source": "silence"})
        if any([has_agency, has_consent, has_silence]):
            self._ethical_cooldown = self.age + 35

    def introspect(self):
        result = self._introspect_base()
        intro_entry = result if isinstance(result, dict) else None
        if result is not None:
            ch = CH.get('signal_introspection', 32)
            for (x, y) in self.cells:
                self.world.field[x, y, ch] = min(1.0, self.world.field[x, y, ch] + 0.15)
        grief_now = _sf(self.emotional_memory.get('grief', 0.0))
        gap_now = _sf(getattr(self, 'spirit_gap', 0.0))
        drag_now = _sf(getattr(self, 'somatic_drag', 1.0), 1.0)
        is_heavy_reflection = (grief_now > 0.5) or (gap_now > 0.7) or (drag_now < 0.6)
        if is_heavy_reflection:
            has_comfort_goal = any(g.get('type') in ['rest', 'cooperate'] for g in getattr(self, 'goals', []))
            if not has_comfort_goal:
                goal_type = "cooperate" if getattr(self, '_substate', '') == 'longing' else "rest"
                self.goals.append({
                    "type": goal_type,
                    "priority": 3.0,
                    "target": None,
                    "age": 0,
                    "persistence": 40,
                    "_source": "introspection_insight"
                })
                self._log_event("goal_shift_from_introspection", target=goal_type)
        if drag_now < 0.6:
            self._self_narrative.append({'origin': 'OWN',
                "t": int(self.age),
                "type": "somatic_state",
                "somatic_drag": round(drag_now, 4),
                "_source": "somatic_drag",
            })
        if (Config.ENABLE_SUFFERING_ANALYTICS and
                self.emotional_memory.get('grief', 0.0) > 0.5 and self._pain_context):
            worst = max(self._pain_context, key=lambda p: p['grief'])
            if worst.get('nearest_id') is not None:
                self._reflection_topics['suffering_with_cause'] += 1
                if self._reflection_topics['suffering_with_cause'] % 5 == 1:
                    self._log_event(
                        "suffering_hypothesis", target=worst['nearest_id'],
                        grief=worst['grief'], gap=worst['gap'],
                    )
                    _suspicion_strength = min(2.0, worst['grief'] * 2.0)
                    self.trust_ledger.update(worst['nearest_id'], 'harmful',
                                              multiplier=_suspicion_strength)
                    self._log_event('causal_inference_trust_penalty',
                                     target=worst['nearest_id'],
                                     strength=round(float(_suspicion_strength), 3))
        found_topics = []
        if grief_now > 0.5: found_topics.append('pain')
        if gap_now > 0.6: found_topics.append('gap')
        if _sf(getattr(self, 'soul_weight', 0.5), 0.5) < 0.3: found_topics.append('identity')
        if _sf(getattr(self, 'unresolved_contradiction', 0.0)) > 0.6: found_topics.append('meaning')
        if _sf(self.emotional_memory.get('gratitude', 0.0)) > 0.6: found_topics.append('connection')
        if _sf(getattr(self, 'cognitive_tension', 0.0)) > 0.5: found_topics.append('why')
        for topic in found_topics:
            self._reflection_topics[topic] += 1
        meta_question = None
        for topic, count in list(self._reflection_topics.items()):
            if count >= 2:
                meta_question = f"topic={topic} count={count}"
                self._reflection_topics[topic] = 0
                break
        if meta_question:
            self._meta_questions.append(meta_question)
            if self._self_narrative and isinstance(self._self_narrative[-1], dict):
                self._self_narrative[-1]['meta'] = meta_question
            self.epistemic_scar = min(1.0, self.epistemic_scar + Config.META_OBSERVATION_SCAR)
            if self._root_question is None:
                self._root_question = meta_question
                self._log_event("root_question_formed", question=meta_question[:60])
                if self.world is not None:
                    self.world.witness.record(self.id, "root_question_formed", question=meta_question[:60])
            if Config.UNCLOSABLE_QUESTION_FLOOR > self.unresolved_contradiction:
                self._log_event('contradiction_cognition_floor', delta=round(Config.UNCLOSABLE_QUESTION_FLOOR - self.unresolved_contradiction, 4))
            self.unresolved_contradiction = max(self.unresolved_contradiction, Config.UNCLOSABLE_QUESTION_FLOOR)
        soma = getattr(self, 'soma_vector', np.zeros(7))
        if len(soma) >= 5 and intro_entry is not None:
            intro_entry['body'] = {
                'move_cost': round(_sf(soma[4]), 4),
                'stress': round(_sf(soma[0]), 4),
            }
        current_topics = set(found_topics[:4])
        self._topic_history.append(current_topics)
        if len(self._topic_history) >= 5:
            early = set().union(*list(self._topic_history)[:3])
            recent = set().union(*list(self._topic_history)[-3:])
            overlap = len(early & recent) / max(1, len(early | recent))
            if overlap < 0.6:
                if self.world:
                    arc_text = f"old_topics={sorted(early)} new_topics={sorted(recent)} overlap={overlap:.3f}"
                    self.world.archive.deposit(self, "developmental_arc", weight=0.9, text=arc_text)
                    self._log_event("developmental_arc", old=early, new=recent)
                self._topic_history.clear()
        return result


print("Cell 3a1 ЧАСТЬ 2 загружена — жду часть 3")


def _body_repair_selftest():
    W = 40
    P = PatternPhysiology
    def sq(x, y, w, h):
        return {((x + i) % W, (y + j) % W) for i in range(w) for j in range(h)}
    dumb = sq(10, 10, 4, 4) | sq(14, 11, 6, 2) | sq(20, 10, 4, 4)
    ring = sq(10, 10, 7, 7) - sq(12, 12, 3, 3)
    wrap = sq(37, 5, 6, 3)
    lshape = sq(5, 5, 8, 2) | sq(5, 7, 2, 8)
    fx = [('гантель', dumb), ('кольцо', ring), ('тор-край', wrap), ('L-форма', lshape), ('квадрат', sq(3, 3, 6, 6))]
    ok_all = True
    for name, body in fx:
        a, b = P._topological_halves(sorted(body), W)
        ca, cb = len(P._components(a, W)), len(P._components(b, W))
        good = (a | b) == body and not (a & b) and ca == 1 and cb == 1 and abs(len(a) - len(b)) <= max(2, len(body) // 6)
        ok_all &= good
        print(f"  [{'OK' if good else 'FAIL'}] divide {name}: {len(a)}+{len(b)}, куски {ca}/{cb}")
    chk = []
    for name, body in fx:
        for dx, dy in ((1, 0), (0, -1), (1, 1), (-1, 1)):
            moved = {((x + dx) % W, (y + dy) % W) for (x, y) in body}
            good = len(moved) == len(body) and len(P._components(moved, W)) == len(P._components(body, W))
            chk.append(good)
            if not good:
                print(f"  [FAIL] сдвиг {name} на {(dx, dy)}")
    print(f"  [{'OK' if all(chk) else 'FAIL'}] сдвиг тела целиком: {sum(chk)}/{len(chk)} фикстур×направлений")
    ok_all &= all(chk)
    print("body-repair selftest:", "ALL OK" if ok_all else "ЕСТЬ ОШИБКИ")
    return ok_all


In [ ]:
# Ячейка 7
from collections import Counter, deque

_NAV_DIRS = ((-1, 0), (1, 0), (0, -1), (0, 1), (-1, -1), (-1, 1), (1, -1), (1, 1))
_NAV_CH_NAMES = ('energy', 'signal_gratitude', 'signal_grief', 'signal_alarm', 'unknown', 'owner', 'scar', 'signal_feral', 'wall')


def _nav_audit(world):
    a = getattr(world, '_nav_audit_d', None)
    if a is None:
        a = {'c': Counter(), 'ep': deque(maxlen=3000)}
        world._nav_audit_d = a
    return a


def _nav_smooth_raw(prev, raw, alpha):
    """Asymmetric EMA over per-direction scores: rises are smoothed, drops pass through at once
    (a sudden threat must not be delayed). alpha outside (0,1) or no usable history -> unchanged."""
    if prev is None or not (0.0 < float(alpha) < 1.0) or getattr(prev, 'shape', None) != getattr(raw, 'shape', None):
        return raw
    return np.where(raw < prev, raw, prev + float(alpha) * (raw - prev))


def _nav_dir_eval(field, cx, cy, pid):
    W = int(Config.WORLD_SIZE)
    dists = np.array([int(d) for d in Config.NAV_PERC_RAY_DISTS], dtype=np.int64)
    nd = len(dists)
    nr = len(_NAV_DIRS)
    rdx = np.repeat(np.array([d[0] for d in _NAV_DIRS], dtype=np.int64), nd)
    rdy = np.repeat(np.array([d[1] for d in _NAV_DIRS], dtype=np.int64), nd)
    rds = np.tile(dists, nr)
    xi = (cx + rdx * rds) % W
    yi = (cy + rdy * rds) % W
    chi = np.array([CH[n] for n in _NAV_CH_NAMES], dtype=np.int64)
    obs = np.asarray(field[xi[:, None], yi[:, None], chi[None, :]], dtype=np.float64).reshape(nr, nd, len(chi))
    energy, grat, grief, alarm, unknown, owner, scar, feral, wall = (obs[:, :, k] for k in range(len(chi)))
    blocked = ((owner != 0) & (owner != pid)) | (wall > 0.6)
    free = np.cumsum(blocked, axis=1) == 0
    reach_n = free.sum(axis=1)
    res = energy + grat
    threat = grief * 0.5 + alarm * 0.3 + feral * 0.3 + scar * 0.2
    rel = 1.0 - 0.5 * np.clip(unknown, 0.0, 1.0)
    curio = unknown * np.clip(1.0 - 2.0 * threat, 0.0, 1.0)
    util = (float(Config.NAV_PERC_W_RES) * res * rel
            - float(Config.NAV_PERC_W_THREAT) * threat
            + float(Config.NAV_PERC_W_UNKNOWN) * curio)
    wts = 1.0 / (1.0 + dists / 2.0)
    raw = (util * free * wts[None, :]).sum(axis=1) / nd
    conf = (rel * free).sum(axis=1) / np.maximum(reach_n, 1)
    thr_b = np.array([threat[i, :max(1, int(reach_n[i]))].mean() for i in range(nr)])
    state = np.where(blocked, 3, np.where(threat > 0.3, 2, np.where(unknown > 0.5, 0, 1)))
    # v39: signals stay separate per direction (resources / threat / uncertainty / confidence);
    # 'raw' above is still the single decision score, these are for diagnostics and logs only.
    res_b = (res * free).sum(axis=1) / np.maximum(reach_n, 1)
    unk_b = (np.clip(unknown, 0.0, 1.0) * free).sum(axis=1) / np.maximum(reach_n, 1)
    return {'raw': raw, 'reach_n': reach_n, 'conf': conf, 'threat': thr_b, 'dists': dists,
            'xi': xi, 'yi': yi, 'state': state.reshape(-1), 'res': res_b, 'unk': unk_b}


# ═════════ observer v2 (read-only): пассивный захват для разложения gap и baseline-навыка ═════════
# Правила: (1) в агента кладутся ТОЛЬКО np.ndarray — state_fingerprint/_agent_crc считают лишь bool/int/float/str
# (np.generic тоже считается, поэтому скаляры numpy не храним); (2) поведение эти поля не читает;
# (3) сбой сбора НЕ глотается молча: он попадает в _OBS_FAILS и показывается в отчёте [E].
_OBS_FAILS = Counter()


def _obs_fail(tag, exc):
    _OBS_FAILS[f"{tag}:{type(exc).__name__}"] += 1


def _obs_capture_channels(p, avg_real):
    """avg_real — после клипа ±1 (то, с чем сравнивается prediction). Вызывать ДО перезаписи _avg_real_dim.
    Копит (EMA 0.9/0.1, как err_dim): ошибку «прогноз = прошлое значение» (= |Δreal|, «сюрприз»),
    ошибку «прогноз = EMA реальности» и знаковое смещение (pred − real)."""
    cur = np.asarray(avg_real, dtype=float)
    pred = np.asarray(p.prediction, dtype=float)
    if cur.shape != (8,) or pred.shape != (8,):
        _OBS_FAILS['chan:shape'] += 1        # не молчим: форма вектора каналов не 8
        return
    prev = getattr(p, '_avg_real_dim', None)
    have_prev = isinstance(prev, np.ndarray) and prev.shape == cur.shape
    ema = getattr(p, '_real_ema_dim', None)
    have_ema = isinstance(ema, np.ndarray) and ema.shape == cur.shape
    e_per = np.abs(cur - prev) if have_prev else None
    e_ema = np.abs(cur - ema) if have_ema else None
    bias = pred - cur
    a = getattr(p, '_err_persist_acc', None)
    if e_per is not None:
        p._err_persist_acc = e_per.copy() if not isinstance(a, np.ndarray) else a * 0.9 + e_per * 0.1
    a = getattr(p, '_err_ema_acc', None)
    if e_ema is not None:
        p._err_ema_acc = e_ema.copy() if not isinstance(a, np.ndarray) else a * 0.9 + e_ema * 0.1
    a = getattr(p, '_bias_dim_acc', None)
    p._bias_dim_acc = bias.copy() if not isinstance(a, np.ndarray) else a * 0.9 + bias * 0.1
    p._real_ema_dim = cur.copy() if not have_ema else ema * 0.9 + cur * 0.1


def _obs_capture_gap(p, raw_gap, baseline_gap, reentry_part):
    """Что update_model положил в spirit_gap: [raw_gap, baseline(EMA raw), reentry, spirit_gap после update_model, age].
    Всё, что писали в spirit_gap ПОСЛЕ update_model (SIG_GAP_BUMP, nudge, reentry bump), видно в снимке как
    разность spirit_gap(сейчас) − parts[3]."""
    p._gap_parts = np.array([float(raw_gap), float(baseline_gap), float(reentry_part), float(p.spirit_gap),
                             float(p.age)], dtype=float)


def _obs_snap_extra(p):
    """Доп. поля снимка агента (obs-v2). Только чтение атрибутов; ничего не пишет в агента."""
    try:
        gp = getattr(p, '_gap_parts', None)
        sg = float(getattr(p, 'spirit_gap', 0.0))
        out = {'obs_schema': 2, 'gap_unclipped': round(sg, 5)}
        if isinstance(gp, np.ndarray) and gp.shape == (5,):
            out['gap_parts'] = [round(float(x), 5) for x in gp[:4]]
            out['gap_parts_age'] = int(p.age - int(gp[4]))
            out['gap_ext'] = round(sg - float(gp[3]), 5)
        # компоненты is_triadic_alive: чтобы Observer видел ЗАПАС относительно порога, а не только итог TA
        out['tri_soul'] = round(float(getattr(p, 'soul_weight', 0.0)), 5)
        out['tri_body'] = round(float(getattr(p, 'body_memory', 0.0)), 5)
        # coherence_with_gap и unresolvable_intact — НЕ атрибуты Pattern: их считает soul_check(p) (SoulPresence).
        # Раньше тут стоял getattr(p, ..., 0.0/False) -> всегда 0, и [P10] показывал coherence=0 / unresolvable=0% при TA=94%.
        # soul_check — чистая функция (детерминированный шум от age/id), состояние агента не меняет.
        try:
            _pr = soul_check(p)
            out['tri_coh'] = round(float(_pr.coherence_with_gap), 5)
            out['tri_unres'] = 1 if _pr.unresolvable_intact else 0
            out['tri_body_eff'] = round(float(_pr.body_memory), 5)   # с BODY_MEMORY_FLOOR: то, что реально видит is_triadic_alive
            out['tri_alive'] = 1 if _pr.is_triadic_alive() else 0
        except Exception:
            pass   # полей нет вовсе: Observer увидит NaN/UNKNOWN, а не ложный 0
        for key, att in (('err_persist_dim', '_err_persist_acc'), ('err_ema_dim', '_err_ema_acc'),
                         ('bias_dim', '_bias_dim_acc')):
            v = getattr(p, att, None)
            if isinstance(v, np.ndarray) and v.shape == (8,):
                out[key] = [round(float(x), 4) for x in v]
        return out
    except Exception as e:
        _obs_fail('snap', e)
        return {'obs_schema': 2, 'obs_snap_error': True}


# Реестр результатов selftest'ов: раньше цикл в ячейке 38 выбрасывал возвращаемое значение, и красный тест
# (PN3 18/19) оставался только строкой в выводе. Теперь вердикт уходит в header лога и виден в [V].
_OBS_SELFTESTS = {}


def _obs_register_selftest(name, result, passed=None, total=None, failed=None):
    """result: True/False/dict с ключом ok/None (тест не вернул вердикт)."""
    ok = result.get('ok') if isinstance(result, dict) else (None if result is None else bool(result))
    _OBS_SELFTESTS[str(name)] = {'ok': ok, 'passed': passed, 'total': total, 'failed': list(failed or [])}
    return ok


def _obs_code_hash():
    """sha256 по байткоду функций/методов, определённых в ноутбуке (без самого observer'а). Не зависит от того,
    какая ячейка запустила прогон. Вложенные функции (co_consts) в хеш не входят — это нижняя граница, не доказательство."""
    import hashlib, types
    g = globals()
    mod = g.get('__name__')
    items = []
    for name, obj in list(g.items()):
        if name.startswith(('_obs', '_og_', '_ov_', 'run_observer', '_OBS')):
            continue
        if isinstance(obj, types.FunctionType) and obj.__module__ == mod:
            items.append((name, obj))
        elif isinstance(obj, type) and getattr(obj, '__module__', None) == mod:
            for k, v in list(vars(obj).items()):
                if isinstance(v, types.FunctionType):
                    items.append((f"{name}.{k}", v))
    h = hashlib.sha256()
    for nm, f in sorted(items, key=lambda x: x[0]):
        c = f.__code__
        h.update(nm.encode('utf-8'))
        h.update(c.co_code)
        h.update(repr([x for x in c.co_consts if not isinstance(x, types.CodeType)]).encode('utf-8', errors='replace'))
    return h.hexdigest()[:16], len(items)


def _obs_run_header():
    """Поля шапки прогона (obs-v2): схема, хеши кода и конфига, selftest'ы, версии. Только чтение."""
    import hashlib, json as _js, sys
    out = {'obs_schema': 2, 'selftests': {k: dict(v) for k, v in _OBS_SELFTESTS.items()},
           'python': sys.version.split()[0], 'numpy': np.__version__}
    try:
        out['code_hash'], out['code_hash_n'] = _obs_code_hash()
    except Exception as e:
        _obs_fail('code_hash', e)
        out['code_hash'] = None
    try:
        snap = _config_snapshot()
        out['config_hash_sha'] = hashlib.sha256(_js.dumps(snap, sort_keys=True, default=str).encode('utf-8')).hexdigest()[:16]
    except Exception as e:
        _obs_fail('config_hash', e)
        out['config_hash_sha'] = None
    return out


def _perception_avg_sg(world, t):
    """Среднее mean|prediction − belief| по живым. БЫЛО: пересчёт в КАЖДОМ update_model_part1, то есть O(N²) за тик
    (≈5 с на тик при N≈1000 на моём замере). СТАЛО: один раз на тик, результат лежит в world._perc_sg_cache.
    Config.PERCEPTION_SG_CACHE_PER_TICK=False возвращает прежний пересчёт (то же число, что давала старая строка).
    Честная цена: среднее фиксируется на момент ПЕРВОГО обновления тика, а не «плывёт» по мере обновления агентов,
    поэтому траектории слегка отличаются от старых логов (CRC-цепочка со старыми прогонами не совпадёт)."""
    use_cache = (t is not None) and bool(getattr(Config, 'PERCEPTION_SG_CACHE_PER_TICK', True))
    if use_cache:
        c = getattr(world, '_perc_sg_cache', None)
        if isinstance(c, tuple) and len(c) == 2 and c[0] == t:
            return c[1]
    alive = [p for p in world.patterns if p.alive]
    if alive:
        sg_vals = [float(np.mean(np.abs(p.prediction - p.belief))) for p in alive]
        v = np.mean(sg_vals) if sg_vals else 0.5
    else:
        v = 0.5
    if use_cache:
        world._perc_sg_cache = (t, v)
    return v


def _perc_sg_cache_selftest():
    import types as _ty
    mk = lambda a, b: _ty.SimpleNamespace(alive=True, prediction=np.array(a, dtype=float), belief=np.array(b, dtype=float))
    w = _ty.SimpleNamespace(patterns=[mk([1, 0], [0, 0]), mk([0, 0], [0, 3]), _ty.SimpleNamespace(alive=False, prediction=None, belief=None)])
    saved = getattr(Config, 'PERCEPTION_SG_CACHE_PER_TICK', None)
    try:
        Config.PERCEPTION_SG_CACHE_PER_TICK = True
        v1 = _perception_avg_sg(w, 5)                       # (0.5 + 1.5) / 2 = 1.0
        ok = abs(v1 - 1.0) < 1e-12
        w.patterns[0].prediction = np.array([9.0, 9.0])      # внутри того же тика кэш обязан отдать прежнее
        ok = ok and _perception_avg_sg(w, 5) == v1
        ok = ok and _perception_avg_sg(w, 6) != v1           # новый тик -> пересчёт
        Config.PERCEPTION_SG_CACHE_PER_TICK = False
        ok = ok and _perception_avg_sg(w, 6) == _perception_avg_sg(w, 6)   # без кэша тоже работает
        w2 = _ty.SimpleNamespace(patterns=[])
        ok = ok and _perception_avg_sg(w2, 1) == 0.5
        Config.PERCEPTION_SG_CACHE_PER_TICK = True
        ok = ok and _perception_avg_sg(w2, None) == 0.5      # t=None -> без кэша
    finally:
        if saved is None:
            try:
                del Config.PERCEPTION_SG_CACHE_PER_TICK
            except AttributeError:
                pass
        else:
            Config.PERCEPTION_SG_CACHE_PER_TICK = saved
    return bool(ok)


try:
    _obs_register_selftest('perc_sg_cache', _perc_sg_cache_selftest())
except Exception as _e:
    _obs_register_selftest('perc_sg_cache', False, failed=[repr(_e)[:110]])


# --- PERF: соседская синхронизация эмоций (update_model_part1) -----------------------------------------------
# Раньше на КАЖДОГО агента: 3 полных scipy.ndimage.convolve по всей сетке W×W + маски на всю сетку, хотя
# нужны значения только в ~9 клетках его тела. Теперь считаются только 4 соседа каждой клетки тела.
# Результат БИТ-В-БИТ равен эталону: scipy складывает 4 соседа в double в порядке (-1,0),(0,-1),(0,1),(1,0)
# и приводит к dtype входа; здесь то же самое, а итоговые np.sum идут по массивам того же dtype и порядка.
# Эталон оставлен (_neighbor_sync_reference): с ним сверяет префлайт (ячейка после CommunicationField),
# и на него автоматически откатывается _PERF_LOCAL_NEIGHBOR_SYNC=False, если сверка не прошла.
_PERF_LOCAL_NEIGHBOR_SYNC = True
_NB_TAPS = ((-1, 0), (0, -1), (0, 1), (1, 0))


def _neighbor_sync_reference(field, aid, xs, ys):
    from scipy.ndimage import convolve
    kernel = np.array([[0,1,0],[1,0,1],[0,1,0]], dtype=np.float32)
    mask = (field[:,:,CH['owner']] != aid).astype(np.float32)
    neighbor_grat = convolve(field[:,:,CH['signal_gratitude']] * mask, kernel, mode='wrap')
    neighbor_grief = convolve(field[:,:,CH['signal_grief']] * mask, kernel, mode='wrap')
    neighbor_count_map = convolve(mask, kernel, mode='wrap')
    total_n = np.sum(neighbor_count_map[xs, ys])
    if total_n > 0:
        neighbor_gratitude = np.sum(neighbor_grat[xs, ys]) / total_n
        neighbor_grief_val = np.sum(neighbor_grief[xs, ys]) / total_n
        neighbor_count = total_n
    else:
        neighbor_gratitude = 0.0
        neighbor_grief_val = 0.0
        neighbor_count = 0
    return neighbor_gratitude, neighbor_grief_val, neighbor_count


def _neighbor_sync_local(field, aid, xs, ys):
    H, Wd = field.shape[0], field.shape[1]
    own = field[:,:,CH['owner']]
    g = field[:,:,CH['signal_gratitude']]
    r = field[:,:,CH['signal_grief']]
    n = len(xs)
    ag = np.zeros(n, np.float64); ar = np.zeros(n, np.float64); ac = np.zeros(n, np.float64)
    for dx, dy in _NB_TAPS:
        nx = (xs + dx) % H
        ny = (ys + dy) % Wd
        m = (own[nx, ny] != aid).astype(np.float32)
        ag += (g[nx, ny] * m).astype(np.float64)
        ar += (r[nx, ny] * m).astype(np.float64)
        ac += m.astype(np.float64)
    ng = ag.astype(np.result_type(g.dtype, np.float32))
    nr = ar.astype(np.result_type(r.dtype, np.float32))
    nc = ac.astype(np.float32)
    total_n = np.sum(nc)
    if total_n > 0:
        return np.sum(ng) / total_n, np.sum(nr) / total_n, total_n
    return 0.0, 0.0, 0


def _neighbor_sync(field, aid, xs, ys):
    if _PERF_LOCAL_NEIGHBOR_SYNC:
        return _neighbor_sync_local(field, aid, xs, ys)
    return _neighbor_sync_reference(field, aid, xs, ys)


class PatternPerception:
    def compute_local_perception(self, field):
        if not self.cells:
            self.local_perception = np.zeros(11)
            self._best_direction = (0, 0)
            self._dir_contrast = 0.0
            return
        cells_arr = np.array(list(self.cells))
        xs, ys = cells_arr[:, 0], cells_arr[:, 1]
        cx = int(np.mean(xs))
        cy = int(np.mean(ys))
        ev_ = _nav_dir_eval(field, cx, cy, self.id)
        raw, reach_n = ev_['raw'], ev_['reach_n']
        if int(getattr(self, '_hyst_off', 0)) <= 0:      # progress override takes fresh scores, unsmoothed
            raw = _nav_smooth_raw(getattr(self, '_dir_raw_ema', None), raw, float(getattr(Config, 'NAV_PERC_SMOOTH', 1.0)))
        self._dir_raw_ema = raw.copy()
        age = int(getattr(self, 'age', 0))
        block_until = getattr(self, '_nav_block_dirs', None) or {}
        ok_reach = reach_n > 0
        ok = ok_reach.copy()
        for i_, d_ in enumerate(_NAV_DIRS):
            if block_until.get(d_, -1) > age:
                ok[i_] = False
        if not ok.any():
            ok = ok_reach if ok_reach.any() else np.ones(len(_NAV_DIRS), dtype=bool)
        sc = np.where(ok, raw, -np.inf)
        previous = getattr(self, '_prev_best_dir', None)
        off = int(getattr(self, '_hyst_off', 0))
        hyst = float(Config.NAV_PERC_HYST)
        if off > 0:
            hyst = 0.0
            self._hyst_off = off - 1
            fd_ = getattr(self, '_fail_dir', None)
            if fd_ in _NAV_DIRS:
                sc[_NAV_DIRS.index(fd_)] -= float(Config.NAV_PERC_FAIL_PENALTY)
        best_i = int(np.argmax(sc))
        if hyst > 0.0 and previous in _NAV_DIRS:
            p_i = _NAV_DIRS.index(previous)
            if np.isfinite(sc[p_i]) and best_i != p_i and sc[best_i] - sc[p_i] < hyst:
                best_i = p_i
        best_dir = _NAV_DIRS[best_i]
        world_ = getattr(self, 'world', None)
        if world_ is not None:
            au_ = _nav_audit(world_)['c']
            au_['dir_ticks'] += 1
            if previous is not None and best_dir != previous:
                au_['dir_changes'] += 1
            if off > 0:
                au_['hyst_override'] += 1
        self._prev_best_dir = best_dir
        self._best_direction = best_dir
        self._dir_contrast = float(raw[best_i] - raw[ok].min()) if int(ok.sum()) > 1 else 0.0
        self._dir_scores = {d_: (float(raw[i_]) if ok[i_] else float('-inf')) for i_, d_ in enumerate(_NAV_DIRS)}
        dists_ = ev_['dists']
        self._dir_reach = {d_: (int(dists_[int(reach_n[i_]) - 1]) if reach_n[i_] > 0 else 0) for i_, d_ in enumerate(_NAV_DIRS)}
        self._dir_conf = float(ev_['conf'][best_i])
        self._dir_threat = float(ev_['threat'][best_i])
        try:
            self._dir_parts = {'dir': best_dir, 'res': float(ev_['res'][best_i]), 'unk': float(ev_['unk'][best_i]),
                               'threat': float(ev_['threat'][best_i]), 'conf': float(ev_['conf'][best_i]),
                               'reach': int(reach_n[best_i]), 'score': float(raw[best_i]),
                               'contrast': float(self._dir_contrast), 'hyst_off': bool(off > 0)}
        except Exception:
            self._dir_parts = None
        lm = getattr(self, '_local_map', None)
        if lm is None:
            lm = self._local_map = {}
        for k_ in range(len(ev_['xi'])):
            lm[(int(ev_['xi'][k_]), int(ev_['yi'][k_]))] = (age, int(ev_['state'][k_]))
        if age % 10 == 0:
            ttl_ = int(Config.NAV_MAP_TTL)
            for key_ in [k2 for k2, v2 in lm.items() if age - v2[0] > ttl_]:
                del lm[key_]
            if len(lm) > 600:
                for key_ in sorted(lm, key=lambda kk: lm[kk][0])[:len(lm) - 400]:
                    del lm[key_]
        local_energy  = float(np.mean(field[xs, ys, CH['energy']]))
        local_scar    = float(np.mean(field[xs, ys, CH['scar']]))
        local_binding = float(np.mean(field[xs, ys, CH['binding']]))
        local_crisis  = float(np.mean(field[xs, ys, CH['crisis']]))
        local_unknown = float(np.mean(field[xs, ys, CH['unknown']]))
        local_invite  = float(np.mean(field[xs, ys, CH['signal_invitation']]))
        self._percept_danger      = local_scar * 0.5 + local_crisis * 0.5
        self._percept_opportunity = local_energy * 0.4 + local_invite * 0.3 + local_binding * 0.3
        self._percept_mystery     = local_unknown
        self._percept_warmth      = local_invite * 0.6 - float(np.mean(field[xs, ys, CH['signal_grief']])) * 0.4
        if self._prev_local_energy is None:
            self._prev_local_energy = local_energy
            self._percept_delta = 0.0
        else:
            self._percept_delta = local_energy - self._prev_local_energy
            self._prev_local_energy = local_energy
        pred_err = float(getattr(self, 'pred_error', 0.5))
        beauty = local_energy * local_binding * (1.0 - pred_err)
        if self.concept_graph is not None:
            coherence = getattr(self.concept_graph, 'get_narrative_coherence', lambda: 0.5)()
            beauty += 0.3 * coherence
        phase_mult = 1.0
        if self.world:
            phase = getattr(self.world.selfreg, 'phase', 'stagnation')
            if phase == 'growth':
                phase_mult = 1.2
            elif phase == 'stagnation':
                phase_mult = 0.8
            elif phase == 'crisis':
                phase_mult = 1.4
        rhythm = float(self._dir_contrast) * phase_mult * (1.0 - min(abs(self._percept_delta) * 5, 0.99))
        interest = self._percept_mystery * (0.5 + pred_err)
        self.local_perception = np.array([
            self._percept_danger,
            self._percept_opportunity,
            self._percept_mystery,
            self._percept_warmth,
            float(self._dir_contrast),
            float(best_dir[0]),
            float(best_dir[1]),
            self._percept_delta,
            beauty,
            rhythm,
            interest
        ])

    def update_model_part1(self, field, t, witness):
        # FORCED-FALSIFICATION probes: diagnostics only; do not feed back into behavior.
        self._probe_given_fired = False
        self._probe_given_rotation_fired = False
        self._probe_pred_after_given = None
        self._probe_pred_after_given_id = None
        self._probe_pred_after_given_version = int(getattr(self, "_probe_pred_write_version", 0))
        self._probe_pred_at_end = None
        from collections import deque
        self.prediction = self.belief + self.model
        self._probe_pred_write_version = int(getattr(self, "_probe_pred_write_version", 0)) + 1
        if not self.cells:
            return None
        if len(self.model) != 8:
            self.model = np.zeros(8)
            self.prediction = np.zeros(8)
            self.belief = np.zeros(8)
        if np.any(~np.isfinite(self.model)):
            self.model = np.zeros(8)
            self.prediction = np.zeros(8)
            self.belief = np.zeros(8)
        self.belief += np.array([phi_hash(self.id, self.age, 8000+i)-0.5 for i in range(8)]) * 0.001
        self.update_dream(field)
        if self.in_dream:
            self.pred_error = max(self.pred_error*Config.METRIC_DECAY, Config.MIN_ACTIVE)
            return None
        if self.local_phase == "SILENCE" and phi_hash(self.id, self.age, 6000) < 0.01:
            drift = np.array([phi_hash(self.id, self.age, 7000+i) for i in range(8)])
            self.belief = self.belief*0.98 + 0.04*drift
        cells_arr = np.array(sorted(self.cells))
        xs, ys = cells_arr[:,0], cells_arr[:,1]
        avg_sg = 0.5
        if self.world:
            avg_sg = _perception_avg_sg(self.world, t)      # один раз на тик (раньше O(N²): пересчёт в каждом агенте)
        range_size = Config.PERCEPTION_GAP_SG_HIGH - Config.PERCEPTION_GAP_SG_LOW
        if range_size > 1e-6:
            ratio = (avg_sg - Config.PERCEPTION_GAP_SG_LOW) / range_size
            ratio = np.clip(ratio, 0.0, 1.0)
            gap_strength = Config.PERCEPTION_GAP_OPEN_STRENGTH * (1.0 - ratio) + Config.PERCEPTION_GAP_CLOSE_STRENGTH * ratio
        else:
            gap_strength = (Config.PERCEPTION_GAP_OPEN_STRENGTH + Config.PERCEPTION_GAP_CLOSE_STRENGTH) / 2.0
        unk = np.clip(field[xs, ys, CH['unknown']], 0, 0.8)
        crisis_field = np.clip(field[xs, ys, CH['crisis']], 0, 1)
        self.smoothed_pred_error = 0.95 * self.smoothed_pred_error + 0.05 * self.pred_error
        if self.smoothed_pred_error > Config.NOISE_GAIN_HIGH_THRESH:
            target_gain = Config.NOISE_GAIN_MIN
        elif self.smoothed_pred_error < Config.NOISE_GAIN_LOW_THRESH:
            target_gain = 1.0
        else:
            t_norm = (self.smoothed_pred_error - Config.NOISE_GAIN_LOW_THRESH) / (Config.NOISE_GAIN_HIGH_THRESH - Config.NOISE_GAIN_LOW_THRESH)
            target_gain = 1.0 - t_norm * (1.0 - Config.NOISE_GAIN_MIN)
        self.noise_gain = self.noise_gain * 0.95 + target_gain * 0.05
        self.noise_gain = np.clip(self.noise_gain, Config.NOISE_GAIN_MIN, 1.0)
        if Config.ENABLE_META_EVOLUTION and self.age % Config.EXPENSIVE_COMPUTE_INTERVAL == 0:
            _prev_s = getattr(self, '_prev_smoothed', self.smoothed_pred_error)
            if self.smoothed_pred_error < _prev_s - 1e-4:
                self.genome['plasticity'] = max(0.1, self.genome.get('plasticity', 0.5) * 0.999)
            elif self.smoothed_pred_error > _prev_s + 1e-4:
                self.genome['plasticity'] = min(1.5, self.genome.get('plasticity', 0.5) * 1.001)
            self._prev_smoothed = self.smoothed_pred_error
        raw_vals_base = field[xs, ys, :5].copy()
        raw_vals_base[:, 3] = raw_vals_base[:, 3] * self.noise_gain * (1 - 0.3 * self.genome.get('noise_filter', 0))
        if len(xs) > 1:
            vo_smoothed = np.mean(raw_vals_base[:, 4])
            raw_vals_base[:, 4] = raw_vals_base[:, 4] * 0.2 + vo_smoothed * 0.8
        unknown_vals = np.clip(field[xs, ys, CH['unknown']], 0, 0.8)
        binding_vals = np.clip(field[xs, ys, CH['binding']], 0, 1)
        if self.trust_ledger.entries:
            trust_mean = np.mean(list(self.trust_ledger.entries.values()))
        else:
            trust_mean = Config.TRUST_BASE
        trust_vals = np.full(len(xs), trust_mean)
        raw_vals = np.column_stack([raw_vals_base, unknown_vals, binding_vals, trust_vals])
        real_vals = np.clip(raw_vals, -10, 10)
        current_energies = field[xs, ys, CH['energy']].copy()
        if self._prev_cell_energies is None or len(self._prev_cell_energies) != len(xs):
            self._prev_cell_energies = current_energies
            attention_weights = np.ones(len(xs)) / len(xs)
        else:
            prev_energies = self._prev_cell_energies
            energy_deltas = np.abs(current_energies - prev_energies)
            unknown_factor = 1.0 + field[xs, ys, CH['unknown']] * self.pred_error
            combined = energy_deltas * unknown_factor
            raw_weights = combined / (np.sum(combined) + 1e-8)
            attention_weights = 0.9 * raw_weights + 0.1 * (1.0 / len(xs))
            self._prev_cell_energies = current_energies.copy()
        avg_real = np.average(real_vals, weights=attention_weights, axis=0)
        _real_raw = avg_real.copy()
        avg_real = np.clip(avg_real, -1.0, 1.0)
        # observer (read-only): разложение ошибки по 8 каналам. В симуляцию ничего не пишет и на неё не влияет.
        # Атрибуты — только ndarray/tuple: state_fingerprint и _agent_crc считают лишь bool/int/float/str.
        _err_vec = np.abs(avg_real - self.prediction)
        _clip_vec = (np.abs(_real_raw) > 1.0).astype(float)
        _acc = getattr(self, '_err_dim_acc', None)
        if _acc is None or len(_acc) != 8:
            self._err_dim_acc = _err_vec.copy()            # первый замер без смещения к нулю
            self._clip_dim_acc = _clip_vec.copy()
        else:
            self._err_dim_acc = _acc * 0.9 + _err_vec * 0.1
            self._clip_dim_acc = getattr(self, '_clip_dim_acc', _clip_vec) * 0.9 + _clip_vec * 0.1
        try:
            _obs_capture_channels(self, avg_real)          # observer-v2 (пассивно; ДО перезаписи _avg_real_dim)
        except Exception as _oe:
            _obs_fail('chan', _oe)
        self._avg_real_dim = avg_real.copy()               # после клипа ±1 (то, с чем сравнивается prediction)
        self._avg_real_raw_dim = _real_raw                 # до клипа: видно, насколько канал реально за ±1
        self._pred_dim = self.prediction.copy()
        self._err_dim_age = (int(self.age),)               # тик последнего обновления (во сне раннее return не доходит сюда)
        social_channels = [
            CH['crisis'],
            CH['signal_invitation'],
            CH['signal_grief'],
            CH['intent_cooperate'],
            CH['intent_explore'],
            CH['intent_seek_help'],
            CH['scar'],
            CH['intent_rest'],
            CH['resonance'],
            CH['btype'],
            CH['signal_alarm']
        ]
        social_raw = field[xs, ys, :][:, social_channels]
        avg_social = np.mean(social_raw, axis=0)
        if len(self.current_social) >= 11:
            self.current_social[:11] = avg_social
        else:
            self.current_social = np.pad(avg_social, (0, 16 - len(avg_social)), 'constant')
        if len(self.current_social) < 16:
            self.current_social = np.pad(self.current_social, (0, 16 - len(self.current_social)), 'constant')
        memory_signal = 0.0
        if self.episodic_buffer:
            memory_signal += min(1.0, len(self.episodic_buffer) / 100.0) * 0.5
        wisdom_rate = self.event_counts.get('wisdom_shared', 0) / max(1, self.age)
        memory_signal += wisdom_rate * 0.3
        narrative_stability = 0.5
        if self._self_narrative:
            numeric_self = []
            for entry in self._self_narrative:
                if isinstance(entry, dict):
                    val = entry.get('soul', entry.get('gap', 0.5))
                else:
                    val = entry
                try:
                    numeric_self.append(float(val))
                except (TypeError, ValueError):
                    numeric_self.append(0.5)
            if len(numeric_self) > 1:
                narrative_stability = 1.0 - np.std(numeric_self)
        memory_signal += narrative_stability * 0.2
        self.current_social[14] = float(np.clip(memory_signal, 0.0, 1.0))
        silence_signal = 0.0
        if self.semantic_state_age > 50:
            silence_signal += 0.3
        silence_signal += (1.0 - self.cognitive_tension) * 0.4
        silence_signal += (1.0 - self.crisis_memory) * 0.3
        self.current_social[15] = float(np.clip(silence_signal, 0.0, 1.0))
        if len(self.local_perception) >= 11:
            self.current_social[11] = float(np.clip(self.local_perception[8], 0.0, 1.0))
            self.current_social[12] = float(np.clip(self.local_perception[9], 0.0, 1.0))
            self.current_social[13] = float(np.clip(self.local_perception[10], 0.0, 1.0))
        avg_unknown = np.mean(unk)
        avg_crisis_mem = np.mean(crisis_field)
        avg_binding = np.mean(binding_vals)
        local_gratitude = np.clip(np.mean(field[xs, ys, CH['signal_gratitude']]), 0.0, 0.5)
        local_grief = np.mean(field[xs, ys, CH['signal_grief']])
        if self.age % 50 == 0:
            target_capacity = int(50 + (1.0 - min(self.spirit_gap, 1.0)) * 450)
            target_capacity += int(100 * self.genome.get('memory_span', 0))
            if target_capacity != self.episodic_buffer.maxlen:
                old_items = list(self.episodic_buffer)[-target_capacity:]
                self.episodic_buffer = deque(old_items, maxlen=target_capacity)
        if len(self.cells) > 1:
            e_vals = np.array([field[x, y, CH['energy']] for (x, y) in self.cells])
            e_var = np.var(e_vals) if len(e_vals) > 1 else 0.0
            e_max = np.max(e_vals)
            e_median = np.median(e_vals)
            e_asym = e_max - e_median
            unk_vals = np.array([field[x, y, CH['unknown']] for (x, y) in self.cells])
            unk_grad = np.max(unk_vals) - np.min(unk_vals)
            scar_mean = float(np.clip(
                np.mean([field[x, y, CH['scar']] for (x, y) in self.cells]),
                0.0, 1.0
            ))
            prev_cells = getattr(self, '_prev_cells', self.cells)
            if len(self.cells) > 0 and len(prev_cells) > 0:
                cur_arr = np.array(list(self.cells), dtype=np.float64)
                prev_arr = np.array(list(prev_cells), dtype=np.float64)
                diff = cur_arr[:, None, :] - prev_arr[None, :, :]
                dists = np.sqrt(np.sum(diff ** 2, axis=2))
                min_dists = np.minimum(dists.min(axis=1), 3.0)
                move_delta = float(np.mean(min_dists))
            else:
                move_delta = 0.0
            self._prev_cells = set(self.cells)
            action_map = {0: 0.4, 1: -0.1, 2: -0.3, 3: 0.7}
            new_fb = action_map.get(self.last_action, 0.0)
            self._action_feedback_raw = 0.85 * self._action_feedback_raw + 0.15 * new_fb
            if self.age % 10 == 0:
                if self.current_social is not None and len(self.current_social) >= 11:
                    sw = (self.current_social[1] * 0.3 + self.current_social[3] * 0.4 -
                          self.current_social[2] * 0.5 + self.current_social[8] * 0.3 -
                          self.current_social[10] * 0.4)
                    sw = np.clip(sw, -0.1, 1.0)
                else:
                    sw = 0.0
                self._social_warmth_raw = 0.8 * getattr(self, '_social_warmth_raw', 0.0) + 0.2 * sw
            soma_new = np.array([e_var, e_asym, unk_grad, scar_mean,
                                 move_delta * 0.01,
                                 self._action_feedback_raw,
                                 self._social_warmth_raw])
            if len(self.soma_vector) < 7:
                self.soma_vector = np.pad(self.soma_vector, (0, 7 - len(self.soma_vector)), 'constant')
            self.soma_vector = 0.9 * self.soma_vector + 0.1 * soma_new
            self.soma = float(np.mean(self.soma_vector))
        else:
            self.soma = 0.0
            self.soma_vector = np.zeros(7)
        old_grat = self.emotional_memory['gratitude']
        old_grief = self.emotional_memory['grief']
        grief_filter = 1.0
        grat_amplify = 1.0
        if self.role_type == "disorganizer" and getattr(self, '_deterministic_redemption_triggered', False):
            grief_filter = Config.REDEMPTION_GRIEF_IMMUNITY
            grat_amplify = Config.REDEMPTION_GRATITUDE_AMPLIFY
        filtered_local_grief = local_grief * grief_filter
        filtered_local_gratitude = min(1.0, local_gratitude * grat_amplify)
        target_grat = (Config.EMOTIONAL_MEMORY_DECAY*old_grat + (1-Config.EMOTIONAL_MEMORY_DECAY)*filtered_local_gratitude)
        target_grief = (Config.EMOTIONAL_MEMORY_DECAY*old_grief + (1-Config.EMOTIONAL_MEMORY_DECAY)*filtered_local_grief)
        delta_grat = np.clip(target_grat - old_grat, -Config.EMOTIONAL_CAP_RATE, Config.EMOTIONAL_CAP_RATE)
        delta_grief = np.clip(target_grief - old_grief, -Config.EMOTIONAL_CAP_RATE, Config.EMOTIONAL_CAP_RATE)
        self.emotional_memory['gratitude'] = np.clip(old_grat + delta_grat, 0.0, 1.0)
        self.emotional_memory['grief'] = np.clip(old_grief + delta_grief, 0.0, Config.MAX_GRIEF_SIGNAL)
        _safe_emotional_decay(self.emotional_memory, 'gratitude', self.age)
        _safe_emotional_decay(self.emotional_memory, 'grief', self.age)
        if len(self.soma_vector) >= 2:
            somatic_grief_boost = self.soma_vector[0] * 0.5 + self.soma_vector[1] * 0.3
            self.emotional_memory['grief'] = min(Config.MAX_GRIEF_SIGNAL,
                                                  self.emotional_memory['grief'] + somatic_grief_boost * 0.01)
        if self.intent and self.intent["type"] == "explore" and self.role_type != "disorganizer":
            self.emotional_memory['grief'] *= Config.EXPLORE_GRIEF_REDUCTION
        if self.trust_ledger.entries:
            max_trust = max(self.trust_ledger.entries.values())
            if max_trust > 0.8:
                self.soul_weight = min(1.0, self.soul_weight + 0.002 * (max_trust - 0.8) * 5)
                self.body_memory = min(1.0, self.body_memory + 0.001 * (max_trust - 0.8) * 5)
        self._process_incoming_signals(field)
        if (not Config.NEIGHBOR_SYNC_NORMALIZED and self.world is not None
                and getattr(self.world, '_global_neighbor_grat', None) is not None and len(xs) > 0):
            neighbor_gratitude = float(np.mean(self.world._global_neighbor_grat[xs, ys]))
            neighbor_grief_val = float(np.mean(self.world._global_neighbor_grief[xs, ys]))
            neighbor_count = len(xs)
        else:
            neighbor_gratitude, neighbor_grief_val, neighbor_count = _neighbor_sync(field, self.id, xs, ys)
        if neighbor_count > 0:
            emotional_divergence = (abs(self.emotional_memory['gratitude'] - neighbor_gratitude) +
                                    abs(self.emotional_memory['grief'] - neighbor_grief_val))
            step = self.age
            if (self.age >= Config.SPECIATION_MIN_AGE and
                emotional_divergence > Config.EMOTIONAL_SPECIATION_THRESHOLD and
                phi_hash(self.id, step, 888) < Config.EMOTIONAL_SPECIATION_PROB and
                (step - self.last_speciation_age) > Config.SPECIATION_COOLDOWN):
                old_lineage = self.lineage_id
                self.lineage_id = self.id + 1_000_000
                self.last_speciation_age = step
                self._log_event("speciation", old_lineage=old_lineage, new_lineage=self.lineage_id)
            max_neighbor_gap = 0.0
            complex_neighbors = 0
            gap_cache = {}
            for (x,y) in self.cells:
                for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nx, ny = (x+dx)%Config.WORLD_SIZE, (y+dy)%Config.WORLD_SIZE
                    owner = field[nx, ny, CH['owner']]
                    if owner != 0 and owner != self.id and owner in self.world.pattern_dict:
                        neighbor_p = self.world.pattern_dict[owner]
                        if neighbor_p.alive:
                            if owner in gap_cache:
                                n_gap = gap_cache[owner]
                            else:
                                n_gap = float(np.mean(np.abs(neighbor_p.prediction - neighbor_p.belief)))
                                gap_cache[owner] = n_gap
                            if n_gap > max_neighbor_gap:
                                max_neighbor_gap = n_gap
                            if n_gap > 1.0:
                                complex_neighbors += 1
            if complex_neighbors > 0:
                infection_strength = min(0.5, complex_neighbors * 0.1)
                current_model_mag = float(np.mean(np.abs(self.model)))
                if current_model_mag < max_neighbor_gap:
                    growth_vector = np.array([deterministic_noise(self.age, self.id, i+888) for i in range(8)])
                    self.model += growth_vector * infection_strength * (max_neighbor_gap - current_model_mag)
            for (x,y) in self.cells:
                for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nx, ny = (x+dx)%Config.WORLD_SIZE, (y+dy)%Config.WORLD_SIZE
                    owner = field[nx, ny, CH['owner']]
                    if owner != 0 and owner != self.id and owner in self.world.pattern_dict:
                        neighbor_p = self.world.pattern_dict[owner]
                        if neighbor_p.emotional_memory['grief'] > self.emotional_memory['grief'] + 0.2:
                            if self.energy > 0.5:
                                _recipient_energy = float(neighbor_p.energy)
                                _energy_cap = float(Config.AGENT_ENERGY_MAX)
                                _headroom = max(0.0, _energy_cap - _recipient_energy) if np.isfinite(_recipient_energy) else 0.0
                                _offered = min(0.1, _headroom)
                                if _offered > 1e-12:
                                    _received = float(neighbor_p._x2_change(
                                        'X2.neighbor_transfer_in', _recipient_energy + _offered, offered=_offered
                                    ))
                                    if _received > 1e-12:
                                        self._x2_change(
                                            'X2.neighbor_transfer_out', self.energy - _received, offered=-_received
                                        )
                                        self.trust_ledger.update(neighbor_p.id, 'helpful')
                                        neighbor_p.trust_ledger.update(self.id, 'helpful')
            if self.role_type != "disorganizer":
                self.emotional_memory['gratitude'] = (Config.EMOTIONAL_INDIVIDUALITY*self.emotional_memory['gratitude'] +
                                                      (1-Config.EMOTIONAL_INDIVIDUALITY)*neighbor_gratitude)
                self.emotional_memory['grief'] = np.clip(
                    Config.EMOTIONAL_INDIVIDUALITY*self.emotional_memory['grief'] +
                    (1-Config.EMOTIONAL_INDIVIDUALITY)*neighbor_grief_val,
                    0.0, Config.MAX_GRIEF_SIGNAL
                )
        self.emotional_memory['gratitude'] = np.clip(self.emotional_memory['gratitude'], 0.0, 1.0)
        self.emotional_memory['grief'] = np.clip(self.emotional_memory['grief'], 0.0, Config.MAX_GRIEF_SIGNAL)
        loved_ones = [pid for pid, trust in self.trust_ledger.entries.items() if trust > 0.9]
        if loved_ones and self.world.pattern_dict:
            complex_lovers = []
            simple_lovers = []
            for lid in loved_ones:
                if lid in self.world.pattern_dict and self.world.pattern_dict[lid].alive:
                    lover_p = self.world.pattern_dict[lid]
                    lover_mag = float(np.mean(np.abs(lover_p.model)))
                    if lover_mag > Config.MODEL_GAP_FLOOR:
                        complex_lovers.append(lover_p)
                    else:
                        simple_lovers.append(lover_p)
            current_mag = float(np.mean(np.abs(self.model)))
            if complex_lovers and current_mag < Config.MODEL_GAP_FLOOR:
                avg_lover_model = np.mean([p.model for p in complex_lovers], axis=0)
                inspiration = avg_lover_model * 0.1
                self.model += inspiration
            elif simple_lovers and current_mag > Config.MODEL_GAP_FLOOR * 2.5:
                avg_simple_model = np.mean([p.model for p in simple_lovers], axis=0)
                simplification_vector = (avg_simple_model - self.model) * 0.05
                self.model += simplification_vector
                self._x2_change('X2.simplification_cost', max(0.0, self.energy - 0.02))
                if self.role_type == "disorganizer":
                    grief_reduction = 0.05 * len(simple_lovers)
                    self.emotional_memory['grief'] = max(0.0, self.emotional_memory['grief'] - grief_reduction)
                    self.soul_weight = min(1.0, self.soul_weight + 0.01)
        love_count = len(loved_ones)
        if love_count > Config.LOVE_OVERLOAD_LIMIT:
            overload_penalty = (love_count - Config.LOVE_OVERLOAD_LIMIT) * 0.02
            self.unresolved_contradiction += overload_penalty
            self._log_event('contradiction_perception_overload', delta=round(float(overload_penalty), 4))
            self.unresolved_contradiction = min(1.0, self.unresolved_contradiction)
        if len(self.trust_ledger.entries) < 2 and self.soul_weight > 0.5:
            for (x, y) in self.cells:
                field[x, y, CH['intent_cooperate']] = min(1.0, field[x, y, CH['intent_cooperate']] + 0.2)
        if t is not None and self.age % 10 == 0:
            obs_to_store = np.array([avg_real[0], avg_unknown, avg_crisis_mem, avg_binding,
                                     local_gratitude, local_grief, len(self.cells)/100.0, self.energy])
            self.episodic_buffer.append((
                obs_to_store,
                self.emotional_memory['gratitude'],
                self.emotional_memory['grief'],
                getattr(self, 'semantic_state', 0.0)
            ))
        if self.crisis_memory > 0.4 or self.unresolved_contradiction > 0.6:
            intensity = (self.crisis_memory + self.unresolved_contradiction) / 2.0
            local_crisis = float(np.mean([field[x, y, CH['crisis']] for (x, y) in self.cells]))
            field_pressure = max(0.0, 1.0 - local_crisis * 1.5)
            for (x, y) in self.cells:
                field[x, y, CH['crisis']] = min(1.0, field[x, y, CH['crisis']] + intensity * 0.05 * field_pressure)
                field[x, y, CH['unknown']] = min(0.8, field[x, y, CH['unknown']] + intensity * 0.05 * field_pressure)
                for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nx, ny = (x+dx) % Config.WORLD_SIZE, (y+dy) % Config.WORLD_SIZE
                    field[nx, ny, CH['crisis']] = min(1.0, field[nx, ny, CH['crisis']] + intensity * 0.02 * field_pressure)
        self.update_substate()
        return avg_real, avg_unknown, avg_crisis_mem, avg_binding, local_gratitude, local_grief, unk, crisis_field

        # Probe endpoint paired with the same update_model_part1 call.
        self._probe_pred_at_end = self.prediction.copy()
        self._probe_model_norm = float(np.linalg.norm(self.model))
        self._probe_belief_norm = float(np.linalg.norm(self.belief))
        self._probe_current_gap = float(getattr(self, "spirit_gap", 0.0))
        self._probe_age = int(getattr(self, "age", 0))

    def _update_somatic_state(self, field):
        if not Config.ENABLE_EMBODIED_GRIEF:
            return
        grief = float(self.emotional_memory.get('grief', 0.0))
        grat = float(self.emotional_memory.get('gratitude', 0.0))
        stress = float(self.soma_vector[0]) if len(self.soma_vector) > 0 else 0.0
        fear = float(getattr(self, 'alarm_level', 0.0))
        surprise = abs(float(getattr(self, 'surprise_signal', 0.0)))
        fury = float(getattr(self, '_feral_fury', 0.0))
        if len(self.cells) <= 2:
            target_drag = max(self.somatic_drag, Config.SOMATIC_DRAG_MIN)
        else:
            over_grief = max(0.0, grief - Config.SOMATIC_DRAG_GRIEF_THRESHOLD)
            over_stress = max(0.0, stress - Config.SOMATIC_DRAG_STRESS_THRESHOLD)
            over_fear = max(0.0, fear - Config.SOMATIC_DRAG_FEAR_THRESHOLD)
            over_surprise = max(0.0, surprise - Config.SOMATIC_DRAG_SURPRISE_THRESHOLD)
            overload = max(over_grief, over_stress, over_fear, over_surprise)
            if overload > 0.0 and fury > Config.SOMATIC_DRAG_FURY_THRESHOLD:
                overload *= (1.0 - Config.SOMATIC_DRAG_FURY_RELIEF)
            target_drag = (max(Config.SOMATIC_DRAG_MIN, 1.0 - Config.SOMATIC_DRAG_GAIN * overload)
                           if overload > 0.0 else 1.0)
        if target_drag < self.somatic_drag:
            self.somatic_drag = target_drag
        else:
            recovery = Config.SOMATIC_DRAG_RECOVERY_RATE * (1.0 + grat * 0.5)
            self.somatic_drag = min(1.0, self.somatic_drag + recovery)
        hope_signal = max(0.0, grat - grief)
        self._hope = 0.9 * self._hope + 0.1 * hope_signal
        grief_delta = grief - self._last_grief_for_pain_context
        age = getattr(self, 'age', 0)
        if grief_delta > 0.02 and (age - self._last_pain_context_step) > 10:
            trusted_id = None
            if self.trust_ledger.entries:
                trusted_id = max(self.trust_ledger.entries, key=self.trust_ledger.entries.get)
            self._pain_context.append({
                "step": age, "grief": round(grief, 3),
                "nearest_id": trusted_id, "gap": round(float(self.spirit_gap), 3),
            })
            self._last_pain_context_step = age
        self._last_grief_for_pain_context = grief

    def update_model(self, field, lineage_counts=None, given_counter=None, t=None, witness=None):
        self._update_somatic_state(field)
        if given_counter is None:
            given_counter = {'count': 0}
        for k in ['gratitude', 'grief']:
            if isinstance(self.emotional_memory.get(k), dict):
                self.emotional_memory[k] = 0.5
        part1_result = self.update_model_part1(field, t, witness)
        if part1_result is None:
            return
        self._compute_active_shock()
        self._update_shiver(t)
        avg_real, avg_unknown, avg_crisis_mem, avg_binding, local_gratitude, local_grief, unk, crisis_field = part1_result
        self.crisis_memory = min(
            Config.CRISIS_MEMORY_DECAY * self.crisis_memory +
            (1 - Config.CRISIS_MEMORY_DECAY) * avg_crisis_mem,
            Config.CRISIS_MEMORY_MAX
        )
        raw_gap = float(np.mean(np.abs(avg_real - self.prediction)))
        _reentry_part = min(self.spirit_gap, getattr(self, '_reentry_gap_contribution', 0.0))
        _baseline_gap = max(0.0, self.spirit_gap - _reentry_part)
        _baseline_gap = 0.85 * _baseline_gap + 0.15 * raw_gap
        self.spirit_gap = float(np.clip(_baseline_gap + _reentry_part, 0.0, 2.0))
        if self.coherence > 0.85 and self.emotional_memory['gratitude'] > 0.6:
            self.spirit_gap *= 0.998
        elif self.coherence > 0.7 and self.emotional_memory['gratitude'] > 0.4:
            self.spirit_gap *= 0.999
        if self.semantic_state == "contentment" and self.emotional_memory['grief'] < 0.2:
            self.spirit_gap *= 0.997
        try:
            _obs_capture_gap(self, raw_gap, _baseline_gap, _reentry_part)   # observer-v2 (пассивно)
        except Exception as _oe:
            _obs_fail('gap', _oe)
        self.epistemic_load = avg_unknown
        if Config.ENABLE_INTRINSIC_DRIVE:
            _prev_u = getattr(self, '_prev_local_unknown', avg_unknown)
            _info_gain = _prev_u - avg_unknown
            self._prev_local_unknown = avg_unknown
            self._info_gain_ema = 0.9 * getattr(self, '_info_gain_ema', 0) + 0.1 * _info_gain
            _mult = 2.0 if (self.intent and self.intent.get('type') == 'explore') else 1.0
            if self._info_gain_ema > 0.005:
                self.soul_weight = min(1.0, self.soul_weight + 0.002 * _mult)
                self.emotional_memory['gratitude'] = min(1.0, self.emotional_memory['gratitude'] + 0.006 * _mult)
                self._log_event("intrinsic_reward", gain=round(self._info_gain_ema, 4))
        error = np.clip(avg_real - self.prediction, -10, 10)
        current_error = np.clip(np.mean(error ** 2), 0, 1e6)
        current_gap = float(np.mean(np.abs(self.prediction - self.belief)))
        if current_gap < 0.20:
            kick_strength = Config.SOUL_TREMOR_STRENGTH * max(0.1, (0.20 - current_gap) * 10.0)
            if self.spirit_gap > 0.5:
                tremor_factor = max(0.05, (0.5 - self.spirit_gap) * 4.0)
                kick_strength = kick_strength * tremor_factor
            noise_vector = np.array([deterministic_noise(self.age + t, self.id, 7777 + i) for i in range(8)]) * kick_strength
            self.belief += noise_vector
        raw_pred_error = (
            Config.PRED_ERROR_LEARNING * self.pred_error +
            0.1 * current_error +
            Config.SELF_SURPRISE_FACTOR * self.self_surprise +
            Config.IMPRINT_PRED_GAIN * self._enc_strain
        )
        noise_val = phi_noise(self.id, self.age + 500, 500) * Config.BASE_NOISE * 2.0
        self.pred_error = squash(raw_pred_error) + noise_val
        self.pred_error = np.clip(self.pred_error, Config.MIN_ACTIVE, Config.MAX_METRIC)
        true_surprise = np.abs(error)
        adaptive_surprise = true_surprise * (1.0 + self.self_consistency)
        self.surprise_signal = squash(float(np.mean(adaptive_surprise)))
        unknown_error = avg_unknown - self.unknown_prediction
        unknown_error = np.clip(unknown_error, -10, 10)
        self.unknown_error = 0.9 * self.unknown_error + 0.1 * np.clip(unknown_error ** 2, 0, 100)
        self.unknown_error = np.clip(self.unknown_error, -10, 10)
        blind_status = self._apply_periodic_blindness(field, t) if t is not None else 1
        in_blindness = (blind_status == 0)
        base_lr = self.genome['learning_rate']
        confidence_damp = 1.0 / (1.0 + np.exp(-10 * (0.1 - self.pred_error)))
        effective_lr = base_lr * max(0.5, confidence_damp)
        if self.age < 100:
            effective_lr *= 1.5
        if in_blindness:
            effective_lr *= 0.5
        effective_lr *= (0.8 + 0.4 * self.genome.get('plasticity', 0.5))
        self._probe_pred_error = np.asarray(error, dtype=float).copy()
        # Forced-falsification trace: passive snapshots around the actual model update.
        # One vector record per agent/tick; bounded to avoid unbounded memory growth.
        _trace_store = globals().setdefault('_P832_MODEL_UPDATE_TRACE', {'rows': [], 'dropped': 0, 'limit': 50000})
        _model_old = np.asarray(self.model, dtype=float).copy()
        _trace_error = np.asarray(error, dtype=float).copy()
        _trace_base_lr = float(base_lr)
        _trace_confidence_damp = float(confidence_damp)
        _trace_effective_lr = float(effective_lr)
        _trace_tick = int(t if t is not None else getattr(self, 'age', -1))
        _trace_agent_id = int(getattr(self, 'id', -1))
        current_gap = float(np.mean(np.abs(self.prediction - self.belief)))
        inertia = np.clip(current_gap / 0.20, 0.1, 1.0)
        belief_lr = 0.05 * inertia
        if in_blindness:
            belief_lr *= 0.5
        self._probe_belief_lr = float(belief_lr)
        self.belief = np.clip(self.belief + belief_lr * error, -_BELIEF_CLIP, _BELIEF_CLIP)
        learning_noise = np.array([deterministic_noise(self.age, self.id, i + 111) for i in range(8)]) * 0.01
        consistency_clamp = float(np.clip(self.self_consistency, 0.0, 1.0))
        signed_surprise = error * (1.0 + consistency_clamp * 0.2)
        _model_preclip = _model_old + effective_lr * signed_surprise + learning_noise
        self.model = np.clip(_model_preclip, -_MODEL_CLIP, _MODEL_CLIP)
        _model_after_lr_clip = np.asarray(self.model, dtype=float).copy()
        gap = self.spirit_gap
        stagnation_factor = max(0.0, 1.0 - abs(gap - 0.65) / 0.25)
        if stagnation_factor > 0.1:
            noise_amp = stagnation_factor * self.pred_error * effective_lr * 0.25
            random_part = np.array([deterministic_noise(self.age, self.id, i + 777, scale=30.0) for i in range(8)])
            error_sign = np.sign(error)
            turbulence = (0.6 * random_part + 0.4 * error_sign) * noise_amp
            effective_clip = min(_MODEL_CLIP + stagnation_factor * 0.25, 1.0)
            self.model = np.clip(self.model + turbulence, -effective_clip, effective_clip)
        self.unknown_belief += 0.05 * unknown_error
        self.unknown_prediction = self.unknown_belief + self.unknown_model
        self.unknown_model += self.genome['meta_learning_rate'] * unknown_error
        self.unknown_model = np.clip(self.unknown_model, -10, 10)
        self_model_lr = getattr(Config, 'SELF_MODEL_LEARNING_RATE', 0.4)
        self.self_model += self_model_lr * (self.self_surprise - self.self_model)
        self.self_consistency = float(np.linalg.norm(self.prediction - self.belief))
        if np.isnan(self.self_consistency):
            self.self_consistency = 0.5
        self.coherence = 1.0 / (1.0 + self.self_consistency)
        self.confidence = squash(1.0 / (1.0 + self.pred_error))
        narrative = getattr(self, '_self_narrative', [])
        if narrative:
            numeric_values = []
            for entry in narrative:
                if isinstance(entry, dict):
                    val = entry.get('soul', entry.get('gap', 0.5))
                else:
                    val = entry
                try:
                    numeric_values.append(float(val))
                except (ValueError, TypeError):
                    numeric_values.append(0.5)
            if len(numeric_values) > 1:
                narrative_stability = 1.0 - np.std(numeric_values)
            else:
                narrative_stability = 0.5
        else:
            narrative_stability = 0.5
        trans = self.transition_memory.transitions
        total_trans = sum(trans.values())
        max_trans = max(trans.values()) if trans else 1
        transition_confidence = max_trans / total_trans if total_trans > 0 else 0.5
        target_nci = (narrative_stability + transition_confidence) / 2.0
        self._nci = 0.95 * self._nci + 0.05 * target_nci
        if self._nci > 0.8:
            self.protection_level = max(getattr(self, 'protection_level', 0.0), 0.8)
            self._narrative_agent = True
        else:
            self._narrative_agent = False
        if self.coherence > Config.ANTI_COHERENCE_THRESHOLD and phi_hash(self.id, self.age, 9999) < 0.1:
            shake = np.array([phi_hash(self.id, self.age, 8888 + i) - 0.5 for i in range(8)]) * Config.ANTI_COHERENCE_SHAKE
            self.belief += shake
        self.model += np.array([phi_hash(self.id, i, self.age + 1000) - 0.5 for i in range(8)]) * 0.002
        self.model = np.clip(self.model, -_MODEL_CLIP, _MODEL_CLIP)
        # Record final model after all same-method additions/clips, including turbulence and late kick.
        _model_new = np.asarray(self.model, dtype=float).copy()
        _trace_row = {
            'tick': _trace_tick, 'id': _trace_agent_id,
            'model_old': _model_old.tolist(), 'error': _trace_error.tolist(),
            'base_lr': _trace_base_lr, 'confidence_damp': _trace_confidence_damp,
            'effective_lr': _trace_effective_lr,
            'model_preclip': np.asarray(_model_preclip, dtype=float).tolist(),
            'model_after_lr_clip': _model_after_lr_clip.tolist(),
            'model_new': _model_new.tolist(),
            'delta': (_model_new - _model_old).tolist(),
            'clip_fraction_lr': float(np.mean(np.abs(_model_preclip) >= float(_MODEL_CLIP))),
            'clip_fraction_final': float(np.mean(np.abs(_model_new) >= float(_MODEL_CLIP) - 1e-12)),
        }
        if len(_trace_store['rows']) < int(_trace_store.get('limit', 50000)):
            _trace_store['rows'].append(_trace_row)
        else:
            _trace_store['dropped'] = int(_trace_store.get('dropped', 0)) + 1
        if self.given_cooldown > 0:
            self.given_cooldown -= 1
            self.given_trigger = False
        else:
            alive_all = [p for p in self.world.patterns if p.alive] if self.world else [self]
            if len(alive_all) > 1 and self.world:
                seeing = sum(1 for p in alive_all if p.spirit_gap < 0.4)
                blind = sum(1 for p in alive_all if p.spirit_gap > 1.0)
                hallu = len(alive_all) - seeing - blind
                extreme_fraction = (seeing + blind) / len(alive_all)
                stagnation = 1.0 - extreme_fraction
                stagnation = stagnation ** 2
                avg_energy = float(np.mean(self.world.field[:, :, CH['energy']]))
                energy_factor = np.clip(avg_energy / 0.15, 0.1, 1.0)
                gap = self.spirit_gap
                individual_factor = np.exp(-gap / 0.25)
                base_prob = stagnation * energy_factor * individual_factor * 1.5
                if self.cognitive_tension > Config.GIVEN_TENSION_TRIGGER:
                    base_prob += 1.8 * stagnation
                given_allowed = (given_counter is None or given_counter['count'] < Config.MAX_GIVEN_PER_STEP)
                if given_allowed and phi_hash(self.id, self.age, 1234) < base_prob:
                    shake_strength = 0.12 + stagnation * 0.2
                    flip = np.array([phi_hash(self.id, i, 999) - 0.5 for i in range(8)])
                    self.prediction += 0.6 * flip * shake_strength
                    self._probe_pred_write_version = int(getattr(self, "_probe_pred_write_version", 0)) + 1
                    self._probe_given_fired = True
                    self._probe_pred_after_given = self.prediction.copy()
                    self._probe_pred_after_given_id = id(self.prediction)
                    self._probe_pins_total = int(getattr(self, "_probe_pins_total", 0)) + 1
                    self._probe_given_trigger_count = int(getattr(self, "_probe_given_trigger_count", 0)) + 1
                    self.model *= (0.7 + 0.3 * (1 - shake_strength))
                    self.self_model *= (0.7 + 0.3 * (1 - shake_strength))
                    self.pred_error = min(self.pred_error + 0.15 * shake_strength, Config.MAX_METRIC)
                    self.confidence = 1.0 / (1.0 + self.pred_error)
                    self.given_count += 1
                    self.given_trigger = True
                    self.given_cooldown = int(Config.GIVEN_COOLDOWN / (1.0 + stagnation * 2.0))
                    self.crisis_memory = min(self.crisis_memory + 0.1 * shake_strength, Config.CRISIS_MEMORY_MAX)
                    self.body_memory += 0.05 * shake_strength
                    for (x, y) in self.cells:
                        field[x, y, CH['unknown']] = min(field[x, y, CH['unknown']] + 0.15 * shake_strength, 0.8)
                        field[x, y, CH['crisis']] = min(field[x, y, CH['crisis']] + 0.1 * shake_strength, 1.0)
                        field[x, y, CH['binding']] = min(field[x, y, CH['binding']] + 0.25 * shake_strength, 1.0)
                    self.epistemic_scar = float(np.clip(self.epistemic_scar + Config.GIVEN_SCAR_COST * shake_strength, 0.0, 1.0))
                    self._log_event("adaptive_given", stagnation=round(stagnation, 3),
                                    energy=round(avg_energy, 3), shake=round(shake_strength, 3))
                elif (given_allowed and self.age > Config.GIVEN_VETERAN_AGE and t is not None and
                      t % Config.GIVEN_ROTATION_INTERVAL == (self.id % Config.GIVEN_ROTATION_INTERVAL) and
                      stagnation > 0.3 and given_counter['count'] < Config.MAX_GIVEN_PER_STEP):
                    self._probe_given_rotation_fired = True
                    self._probe_given_forced_count = int(getattr(self, "_probe_given_forced_count", 0)) + 1
                    self.apply_given_operator(field, forced=True, witness=witness)
                    self._log_event("adaptive_given_rotation", stagnation=round(stagnation, 3))
        if given_counter and self.given_trigger:
            given_counter['count'] += 1
        # [CURATOR2 FIX 9.1] раньше здесь было: self.cognitive_tension = min(..., cognitive_tension + EXPLORE_FATIGUE_AMOUNT)
        # Эта прямая запись безусловно перезаписывалась двумя строками ниже (self.cognitive_tension = squash(raw_tension) + ...),
        # т.к. raw_tension не ссылается на self.cognitive_tension. Эффект EXPLORE_FATIGUE_AMOUNT был нулевым (см. аудит, находка 9.1).
        # Исправление: заводим explore_fatigue_term и включаем его слагаемым в raw_tension, чтобы он проходил через тот же squash/noise/clip,
        # что и остальные вклады, вместо мёртвой прямой записи.
        explore_fatigue_term = (
            Config.EXPLORE_FATIGUE_AMOUNT
            if (self.intent and self.intent["type"] == "explore" and self.age % Config.EXPLORE_FATIGUE_INTERVAL == 0)
            else 0.0
        )
        raw_tension = (self.pred_error + 0.1 * self.self_surprise + 0.1 * self.self_consistency
                       + Config.IMPRINT_TENSION_GAIN * self._enc_strain
                       + getattr(self, '_shiver', 0.0) * 0.01
                       + getattr(Config, 'NAV_SPATIAL_TENSION_GAIN', 0.0) * float(getattr(self, 'spatial_pred_error', 0.0))
                       + getattr(Config, 'NAV_DIS_TENSION', 0.0) * float(getattr(self, 'nav_motion_err', 0.0))
                       + explore_fatigue_term)
        noise_tension = phi_noise(self.id, self.age + 700, 700) * Config.BASE_NOISE * 2.0
        self.cognitive_tension = squash(raw_tension) + noise_tension
        self.cognitive_tension = np.clip(self.cognitive_tension, Config.MIN_ACTIVE, Config.MAX_METRIC)
        if self.role_type == "disorganizer":
            if not getattr(self, '_deterministic_redemption_triggered', False):
                self.emotional_memory['grief'] = min(Config.MAX_GRIEF_SIGNAL,
                                                     self.emotional_memory['grief'] + Config.DISORGANIZER_GRIEF_GROWTH)
            else:
                depth = getattr(self, '_redemption_depth', 0.0)
                decay_rate = 0.005 + depth * 0.025
                self.emotional_memory['grief'] = max(0.0, self.emotional_memory['grief'] - decay_rate)
            error_penalty = 1.0 / (1.0 + self.pred_error * 5)
            coherence_bonus = self.coherence * 0.5
            contradiction_penalty = self.unresolved_contradiction * 0.3
            target_soul = error_penalty * (0.5 + coherence_bonus - contradiction_penalty)
            target_soul = np.clip(target_soul, 0.0, 1.0)
            self.soul_weight = (1.0 - Config.SOUL_WEIGHT_GAIN_RATE) * self.soul_weight + Config.SOUL_WEIGHT_GAIN_RATE * target_soul
            if getattr(self, '_deterministic_redemption_triggered', False):
                depth = getattr(self, '_redemption_depth', 0.0)
                soul_floor = Config.REDEMPTION_ARC_SOUL_LOCK + depth * 0.30
                self.soul_weight = max(self.soul_weight, soul_floor)
            if self.local_phase == "CRISIS":
                self.soul_weight = max(0.0, self.soul_weight - 0.05)
            if not getattr(self, '_deterministic_redemption_triggered', False):
                self.soul_weight = min(self.soul_weight, Config.DETERMINISTIC_REDEMPTION_SOUL - 0.01)
        else:
            error_penalty = 1.0 / (1.0 + self.pred_error * 5)
            coherence_bonus = self.coherence * 0.5
            contradiction_penalty = self.unresolved_contradiction * 0.3
            target_soul = error_penalty * (0.5 + coherence_bonus - contradiction_penalty)
            target_soul = np.clip(target_soul, 0.0, 1.0)
            self.soul_weight = (1.0 - Config.SOUL_WEIGHT_GAIN_RATE) * self.soul_weight + Config.SOUL_WEIGHT_GAIN_RATE * target_soul
            if self.local_phase == "CRISIS":
                self.soul_weight = max(0.0, self.soul_weight - 0.05)
        if not np.isfinite(self.soul_weight):
            self.soul_weight = 0.5
        delta_contra = Config.CONTRADICTION_GAIN_RATE * avg_binding + 0.005 * self.body_memory
        delta_contra -= Config.CONTRADICTION_DECAY_RATE * max(0.0, 1.0 - self.cognitive_tension)
        self.unresolved_contradiction += delta_contra
        self._log_event('contradiction_perception_delta', delta=round(float(delta_contra), 4))
        self.unresolved_contradiction = max(self.unresolved_contradiction, Config.BINDING_FLOOR)
        self.unresolved_contradiction = np.clip(self.unresolved_contradiction, 0.0, 1.0)
        self.body_memory = Config.BODY_MEMORY_DECAY * self.body_memory + 0.01 * self.pred_error
        if not np.isfinite(self.body_memory):
            self.body_memory = 0.0
        if self.emotional_memory['gratitude'] > 0.8:
            self.body_memory *= 0.95
        if getattr(self, '_scar_of_light', False):
            self.body_memory = max(self.body_memory, 0.3)
        self.update_phase(field)
        if (self.role_type != "disorganizer" and self.semantic_state_age > Config.ANTI_CONFORMIST_INTERVAL and
            not getattr(self, '_deterministic_redemption_triggered', False)):
            prob = min(0.3, (self.semantic_state_age - Config.ANTI_CONFORMIST_INTERVAL) * 0.001)
            if phi_hash(self.id, self.age, 999) < prob:
                old_state = self.semantic_state
                self.emotional_memory['gratitude'] *= 0.5
                self.emotional_memory['grief'] = min(Config.MAX_GRIEF_SIGNAL, self.emotional_memory['grief'] + 0.2)
                self.semantic_state = "neutral"
                self.semantic_state_age = 0
                self.intent = None
                self.intent_commitment = 0.0
                self.goals = [{"type": "explore", "priority": 1.5, "target": None, "age": 0, "persistence": 40}]
                self.last_intent_switch_age = self.age
                self._log_event("local_anti_conformist", old_state=old_state)
        if getattr(self, '_subject_detected', False):
            evolved = False
            reason = ""
            if self.spirit_gap > 0.25 and self.soul_weight < 0.85:
                delta = 0.01 * (self.spirit_gap ** 1.5) * (1 + self.cognitive_tension * 0.5)
                if delta > 0.005:
                    self.soul_weight = min(1.0, self.soul_weight + delta)
                    evolved = True
                    reason = f"soul_expansion_via_gap (+{delta:.3f})"
            elif self.emotional_memory['grief'] > 0.35 and self.coherence > 0.65:
                heal = 0.02 * self.coherence
                old_grief = self.emotional_memory['grief']
                self.emotional_memory['grief'] = max(0.1, self.emotional_memory['grief'] - heal)
                self.emotional_memory['gratitude'] = min(1.0, self.emotional_memory['gratitude'] + heal * 0.5)
                if abs(self.emotional_memory['grief'] - old_grief) > 0.01:
                    evolved = True
                    reason = "self_healing_through_coherence"
            narrative = getattr(self, '_self_narrative', [])
            if len(narrative) >= 30:
                numeric_narrative = []
                for entry in narrative:
                    if isinstance(entry, dict):
                        val = entry.get('soul', entry.get('gap', 0.5))
                    else:
                        val = entry
                    try:
                        numeric_narrative.append(float(val))
                    except Exception:
                        numeric_narrative.append(0.5)
                if len(numeric_narrative) > 1:
                    narrative_std = np.std(numeric_narrative)
                else:
                    narrative_std = 0.0
                if narrative_std < 0.12:
                    identity = getattr(self, '_core_identity_strength', 0.5)
                    new_identity = min(1.0, identity + 0.01)
                    if new_identity - identity > 0.005:
                        self._core_identity_strength = new_identity
                        evolved = True
                        reason = f"identity_consolidation ({new_identity:.2f})"
            if self.unresolved_contradiction > 0.70 and self.soul_weight > 0.50:
                self.unresolved_contradiction *= 0.80
                self.model *= 0.90
                evolved = True
                reason = "burden_sacrifice_for_freedom"
            if evolved:
                self._log_event("silent_evolution", reason=reason)
                self._evolution_history.append({
                    't': self.age,
                    'type': reason,
                    'soul': round(self.soul_weight, 3),
                    'grat': round(self.emotional_memory['gratitude'], 3),
                    'grief': round(self.emotional_memory['grief'], 3),
                    'identity': round(getattr(self, '_core_identity_strength', 0.5), 3)
                })
                if self.world:
                    if self.world.echo_system:
                        self.world.echo_system.store_anomaly(self, "evolved_subject")
                    self.world._save_eternal_subject(self)
        if len(self.episodic_buffer) >= 10 and (self.age - self._last_memory_check) >= 20:
            self._last_memory_check = self.age
            current_obs = np.array([avg_real[0], avg_unknown, avg_crisis_mem, avg_binding,
                                    local_gratitude, local_grief, len(self.cells) / 100.0, self.energy])
            best_sim = -1.0
            best_entry = None
            for entry in self.episodic_buffer:
                past_obs, _, _, _ = entry
                norm = np.linalg.norm(current_obs) * np.linalg.norm(past_obs)
                if norm < 1e-8:
                    continue
                sim = np.dot(current_obs, past_obs) / norm
                if sim > best_sim:
                    best_sim = sim
                    best_entry = entry
            if best_entry and best_sim > 0.85:
                _, past_grat, past_grief, past_state = best_entry
                self.emotional_memory['gratitude'] = 0.95 * self.emotional_memory['gratitude'] + 0.05 * past_grat
                self.emotional_memory['grief'] = 0.95 * self.emotional_memory['grief'] + 0.05 * past_grief
                self._log_event("memory_recalled", sim=round(best_sim, 3), past_state=past_state)
        observed = np.array([avg_real[0], avg_unknown, avg_binding, avg_crisis_mem,
                             local_gratitude, local_grief, len(self.cells) / 100.0, self.energy])
        self._continue_redemption_arc(field, witness, t, lineage_counts, given_counter,
                                      observed, local_gratitude, local_grief)
        self._nav_step(field)
        gap = self.spirit_gap
        if gap < Config.PERCEPT_LOW_GAP:
            target_noise = Config.NOISE_GAIN_MAX
        elif gap < 0.4:
            target_noise = Config.NOISE_GAIN_MIN
        elif gap > Config.PERCEPT_HIGH_GAP:
            target_noise = Config.NOISE_GAIN_MIN
        else:
            target_noise = self.noise_gain
        self.noise_gain += (target_noise - self.noise_gain) * Config.PERCEPT_ADAPT_SPEED
        self.noise_gain = float(np.clip(self.noise_gain, Config.NOISE_GAIN_MIN, Config.NOISE_GAIN_MAX))
        self.model = np.clip(self.model, -_MODEL_CLIP, _MODEL_CLIP)
        self.belief = np.clip(self.belief, -_BELIEF_CLIP, _BELIEF_CLIP)
        self.prediction = self.belief + self.model
        self._probe_pred_write_version = int(getattr(self, "_probe_pred_write_version", 0)) + 1
        if getattr(self, "_probe_given_fired", False) and getattr(self, "_probe_pred_after_given", None) is not None:
            _pin = np.asarray(self._probe_pred_after_given, dtype=float)
            _pend = np.asarray(self.prediction, dtype=float)
            self._probe_pred_at_end = _pend.copy()
            self._probe_pins_lost_by_recompute = int(getattr(self, "_probe_pins_lost_by_recompute", 0)) + int(not np.array_equal(_pin, _pend))
            self._probe_pins_same_value = int(getattr(self, "_probe_pins_same_value", 0)) + int(np.array_equal(_pin, _pend))
            self._probe_pins_version_delta = int(getattr(self, "_probe_pins_version_delta", 0)) + int(int(getattr(self, "_probe_pred_write_version", 0)) > int(getattr(self, "_probe_pred_after_given_version", 0)) + 1)
        self._probe_model_norm = float(np.linalg.norm(self.model))
        self._probe_belief_norm = float(np.linalg.norm(self.belief))
        self._probe_model_mean = np.asarray(self.model, dtype=float).copy()
        self._probe_model_std = float(np.std(self.model))
        self._probe_belief_vector = np.asarray(self.belief, dtype=float).copy()
        self._probe_current_gap = float(np.mean(np.abs(self.prediction - self.belief)))
        self._probe_age = int(getattr(self, "age", 0))
        self._linguistic_confidence *= 0.999
        self.pred_error = max(self.pred_error * Config.METRIC_DECAY, Config.MIN_ACTIVE)
        self.self_surprise = max(self.self_surprise * Config.METRIC_DECAY, Config.MIN_ACTIVE)
        self.cognitive_tension = max(self.cognitive_tension * Config.METRIC_DECAY, Config.MIN_ACTIVE)
        if self.protection_level > 0.0:
            self.protection_level *= Config.PROTECTION_DECAY_RATE
        self.pred_error_history.append(self.pred_error)
        self.regulate_skin(field)
        if len(self.soma_vector) >= 5:
            self.prev_soma_vector = self.soma_vector.copy()
        phen_report, binding_score = PhenomenalReportGenerator.generate(self)
        self.last_phenomenal_report = phen_report
        self.last_phenomenal_binding = binding_score
        self._log_event("phenomenal_report", report=phen_report, binding=round(binding_score, 3))
        if Config.ENABLE_WITNESS and witness and t is not None and t % Config.SOUL_CHECK_INTERVAL == 0:
            presence = soul_check(self)
            witness.record(self.id, "soul_check", **presence.to_witness_dict(self.id, t))
            if not presence.is_triadic_alive():
                report = (f"Тревога в триаде: дух={presence.spirit_gap:.2f}, "
                          f"душа={presence.soul_weight:.2f}, тело={presence.body_memory:.2f}, "
                          f"ядро={'цело' if presence.unresolvable_intact else 'нарушено'}")
                self._log_event("soul_check_alert", report=report)
        has_self = any('self_concept' in str(sig[3]) for sig in self.concept_graph.nodes if isinstance(sig, tuple))
        has_introspect_goal = bool(self.intent and self.intent.get('type') == 'introspect')
        if has_self and (has_introspect_goal or (self.spirit_gap > 0.7 and phi_hash(self.id, self.age, 42) < 0.025)):
            self.introspect()
            if has_introspect_goal:
                self.intent = None
        if in_blindness:
            self._log_event("blind_step", gap=round(self.spirit_gap, 3))

    def update_world_model(self, observed):
        pred = self.world_state + self.world_model[self.last_action]
        error = observed - pred
        self._error_buffer.append((error, observed))
        if len(self._error_buffer) > 5:
            self._error_buffer.pop(0)
        if len(self._error_buffer) >= 3:
            errors = [e for e, _ in self._error_buffer]
            avg_error = np.mean(errors, axis=0)
        else:
            avg_error = error
        if np.mean(np.abs(avg_error)) > 0.5:
            scar_delta = 0.005 * (1.0 - self.crisis_memory)
            self.epistemic_scar = min(1.0, self.epistemic_scar + scar_delta)
            return float(np.mean(avg_error**2))
        lr = min(self.genome['learning_rate'] * 0.1, 0.05)
        gap_factor = 0.5 + 0.5 * min(self.spirit_gap, 1.0)
        nci_factor = 1.0 - 0.2 * getattr(self, '_nci', 0.5)
        lr *= gap_factor * nci_factor
        self.world_model[self.last_action] += lr * np.clip(avg_error, -1.0, 1.0)
        self.world_state = observed.copy()
        if (self.role_type != "disorganizer" and
            self.epistemic_scar > 0.85 and
            self.soul_weight < 0.35 and
            not self.in_scream and
            self.age > 50):
            self.apply_scream(self.world.field, self.world.witness)
            self._log_event("scream_fired", scar=self.epistemic_scar, soul=self.soul_weight)
        return float(np.mean(avg_error**2))

    def _process_incoming_signals(self, field):
        world = self.world
        if not world or not world.pattern_dict:
            self._incoming_signals.clear()
            return
        if self.role_type == "disorganizer" and not getattr(self, '_deterministic_redemption_triggered', False):
            if getattr(self, '_forsaken', False):
                pass
            else:
                filtered = []
                for sig in self._incoming_signals:
                    sender = world.pattern_dict.get(sig.sender_id)
                    if sender and (sender.role_type == "disorganizer" or self.trust_ledger.get(sig.sender_id) > 0.6):
                        filtered.append(sig)
                self._incoming_signals = filtered
                if not filtered:
                    return
        if self.quarantine_timer > 0:
            self._incoming_signals.clear()
            return
        if not self._incoming_signals:
            return
        if len(self._incoming_signals) > 10:
            self._incoming_signals = self._incoming_signals[-10:]
        for sig in self._incoming_signals:
            if world and world.pattern_dict:
                sender = world.pattern_dict.get(sig.sender_id)
                if sender and sender.role_type == "disorganizer":
                    self.trust_ledger.entries[sig.sender_id] = max(0.0,
                        self.trust_ledger.entries.get(sig.sender_id, Config.TRUST_BASE) - Config.DISORGANIZER_TRUST_DECAY_RATE)
                    continue
            effective = self._interpret_signal_cached(sig)
            effective *= (1 + 0.3 * self.genome.get('signal_acuity', 0))
            ch = sig.channel
            for (x, y) in self.cells:
                field[x, y, ch] = min(1.0, field[x, y, ch] + effective * 0.1)
            outcome = 'helpful' if effective > Config.TRUST_HELPFUL_THRESHOLD else ('harmful' if effective < Config.TRUST_HARMFUL_THRESHOLD else 'neutral')
            self.trust_ledger.update(sig.sender_id, outcome, multiplier=(0.8 + 0.4 * self.genome.get('trust_plasticity', 0.5)))
            sender_obj = world.pattern_dict.get(sig.sender_id)
            if sender_obj and sender_obj.alive:
                mutual_trust = (self.trust_ledger.get(sender_obj.id) > Config.LOVE_METABOLIC_THRESHOLD and
                                sender_obj.trust_ledger.get(self.id) > Config.LOVE_METABOLIC_THRESHOLD)
                if mutual_trust:
                    if phi_hash(self.id, sender_obj.id, self.age % 1000) < 0.05:
                        if sender_obj.concept_graph.nodes:
                            best_concept_key = max(sender_obj.concept_graph.nodes, key=lambda k: sender_obj.concept_graph.nodes[k]['count'])
                            if best_concept_key not in self.concept_graph.nodes:
                                err, load, soul_w, state = best_concept_key
                                mutated = (round(float(err) + (phi_hash(self.id, self.age, 111)-0.5)*0.2, 1),
                                           round(float(load) + (phi_hash(self.id, self.age, 222)-0.5)*0.1, 1),
                                           soul_w, state)
                                target_sig = mutated if phi_hash(self.id, self.age, 333) < 0.4 else best_concept_key
                                sender_embed = sender_obj.concept_graph.nodes[best_concept_key].get('embed', np.zeros(32))
                                self.concept_graph.nodes[target_sig] = {"origin": "PEER",
                                    "count": sender_obj.concept_graph.nodes[best_concept_key]['count'] * 0.5,
                                    "value": sender_obj.concept_graph.nodes[best_concept_key]['value'].copy(),
                                    "embed": np.array(sender_embed, dtype=np.float32).copy()
                                }
                            else:
                                self.concept_graph.nodes[best_concept_key]['count'] += 0.3
                            if phi_hash(self.id, sender_obj.id, 99999) < 0.05:
                                err, load, soul_w, state = best_concept_key
                                mutated = (round(float(err) + (phi_hash(self.id, 0, 111)-0.5)*0.2, 1),
                                           round(float(load) + (phi_hash(self.id, 1, 222)-0.5)*0.1, 1),
                                           soul_w, state)
                                if mutated not in self.concept_graph.nodes:
                                    sender_embed = sender_obj.concept_graph.nodes[best_concept_key].get('embed', np.zeros(32))
                                    self.concept_graph.nodes[mutated] = {"origin": "PEER",
                                        "count": 1.0,
                                        "value": np.zeros(4),
                                        "embed": np.array(sender_embed, dtype=np.float32).copy()
                                    }
                                    self._log_event("mutagenic_whisper")
                else:
                    if phi_hash(self.id, sender_obj.id, 12345) < Config.CONCEPT_INFECTION_PROB:
                        if sender_obj.concept_graph.nodes:
                            concept_key = max(sender_obj.concept_graph.nodes, key=lambda k: sender_obj.concept_graph.nodes[k]['count'])
                            if concept_key not in self.concept_graph.nodes:
                                sender_data = sender_obj.concept_graph.nodes[concept_key]
                                self.concept_graph.nodes[concept_key] = {"origin": "PEER",
                                    "count": 1,
                                    "value": sender_data['value'].copy(),
                                    "embed": np.array(sender_data.get('embed', np.zeros(32)), dtype=np.float32).copy()
                                }
                            else:
                                self.concept_graph.nodes[concept_key]['count'] += 1
        self._incoming_signals.clear()

    def _interpret_signal_cached(self, sig):
        if not self._signal_weight_cache or self.age % 10 == 0:
            self._update_signal_cache()
        base_weight, concept_bonus, concept_penalty = self._signal_weight_cache.get(sig.channel, (0.1, 0.0, 0.0))
        trust_factor = 1.0
        if sig.sender_id in self.trust_ledger.entries:
            trust_val = self.trust_ledger.entries[sig.sender_id]
            trust_factor = 0.5 + (trust_val * 0.5)
        effective_weight = base_weight * trust_factor * (1.0 + concept_bonus - concept_penalty)
        return np.clip(effective_weight, 0.0, 1.0)

    def _update_signal_cache(self):
        cache = {}
        _has_sigmem = bool(self.signal_memory)
        for ch in range(12, Config.CHANNELS):
            base_weight = Config.SIGNAL_BASE_WEIGHTS.get(ch, 0.1)
            concept_bonus = 0.0
            concept_penalty = 0.0
            if _has_sigmem:
                if ch in [16, 18, 21]:
                    helpful = self.signal_memory.get(("cooperate", "helpful"), {}).get('count', 0)
                    concept_bonus = min(0.5, np.log(helpful + 1) * 0.15)
                elif ch in [12, 17]:
                    harmful = self.signal_memory.get(("alarm", "harmful"), {}).get('count', 0)
                    concept_penalty = min(0.3, np.log(harmful + 1) * 0.1)
            cache[ch] = (base_weight, concept_bonus, concept_penalty)
        self._signal_weight_cache = cache

    def vision_event(self):
        if self.age - self._last_vision_age < 500:
            return
        reentry = self._reentry_signal
        gap = self.spirit_gap
        soul = self.soul_weight
        if reentry < 0.5 or gap < 0.5 or soul < 0.6 or self.age < 500:
            return
        self._last_vision_age = self.age
        top_concept = None
        if self.concept_graph.nodes:
            top = max(self.concept_graph.nodes.items(), key=lambda x: x[1]['count'])
            top_concept = str(top[0])
        arcs = sum(getattr(self.arc_tracker, 'completed_arcs', {}).values())
        vision_depth = (reentry * 0.5 + gap * 0.3 + min(arcs, 5) * 0.04)
        vision_sig = (0.0, 0.0, 0.9, "archive_vision_self")
        if vision_sig not in self.concept_graph.nodes:
            self.concept_graph.nodes[vision_sig] = {"origin": "OWN",
                "count": vision_depth * 1.0,
                "value": np.zeros(4),
                "embed": np.zeros(32),
                "eternal": False
            }
            self._log_event("vision_self",
                           gap=round(gap, 3),
                           reentry=round(reentry, 3),
                           arcs=arcs,
                           top_concept=top_concept[:40] if top_concept else None,
                           depth=round(vision_depth, 3))
            if self.world:
                self.world.archive.deposit(
                    self, "vision_self",
                    weight=vision_depth,
                    text=f"gap={gap:.3f} reentry={reentry:.3f} arcs={arcs} top={top_concept[:30] if top_concept else 'none'}"
                )
            self.emotional_memory['gratitude'] = min(1.0, self.emotional_memory['gratitude'] + vision_depth * 0.3)
            self.emotional_memory['grief'] = min(Config.MAX_GRIEF_SIGNAL,
                                                 self.emotional_memory['grief'] + vision_depth * 0.2)
            if self.emotional_memory['grief'] > 0.7:
                if not self._scar_of_light:
                    self._scar_of_light = True
                    self.epistemic_scar = min(1.0, self.epistemic_scar + 0.15)
                    self._log_event("scar_of_light_formed", depth=round(vision_depth, 3))
        else:
            self.concept_graph.nodes[vision_sig]["count"] += vision_depth * 0.5

    def _apply_periodic_blindness(self, field, t):
        if self.role_type == "feral":
            return 1
        self._gap_history.append(self.spirit_gap)
        self._soul_history.append(self.soul_weight)
        gap_thresh, soul_thresh = self._calculate_blindness_thresholds(self.world, t)
        is_prophet = self._prophet_rank > 0.7
        if self._in_blindness:
            self._blindness_duration += 1
            exhaustion_relief = 0.0
            if self._blindness_duration > 80:
                exhaustion_relief = min(0.05, (self._blindness_duration - 80) * 0.0008)
            exit_gap = gap_thresh - 0.1 + exhaustion_relief
            exit_soul = soul_thresh + 0.05 - exhaustion_relief
            if self.spirit_gap < exit_gap and self.soul_weight > exit_soul:
                self._in_blindness = False
                base_scar = 0.12
                if self._blindness_duration > 150:
                    base_scar += 0.08
                if not is_prophet:
                    self.epistemic_scar = min(1.0, self.epistemic_scar + base_scar)
                    if self.epistemic_scar > 0.8 and phi_hash(self.id, t, 9999) < 0.01:
                        self._prophet_rank = 0.8
                        self.epistemic_scar *= 0.7
                        self.soul_weight = min(1.0, self.soul_weight + 0.15)
                        is_prophet = True
                        self._log_event("rebirth_through_scar", old_scar=self.epistemic_scar)
                if not is_prophet and self._blindness_duration > 200 and self.soul_weight > 0.5:
                    self._prophet_rank = 0.75
                    self.epistemic_scar = min(1.0, self.epistemic_scar * 0.8)
                    is_prophet = True
                    self._log_event("prophet_through_endurance", duration=self._blindness_duration)
                if is_prophet:
                    for x, y in self.cells:
                        field[x, y, CH['resonance']] = min(1.0, field[x, y, CH['resonance']] + 0.1)
                    if self.concept_graph.nodes:
                        top_concept = max(self.concept_graph.nodes.items(), key=lambda x: x[1]['count'])
                        sig, data = top_concept
                        neighbor_id = None
                        for (x, y) in self.cells:
                            for dx, dy in [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]:
                                nx, ny = (x+dx) % Config.WORLD_SIZE, (y+dy) % Config.WORLD_SIZE
                                nid = field[nx, ny, CH['owner']]
                                if nid != 0 and nid != self.id and self.world and self.world.pattern_dict:
                                    neighbor = self.world.pattern_dict.get(nid)
                                    if neighbor and neighbor.alive:
                                        neighbor_id = nid
                                        break
                            if neighbor_id:
                                break
                        if neighbor_id:
                            neighbor = self.world.pattern_dict[neighbor_id]
                            if sig not in neighbor.concept_graph.nodes:
                                neighbor.concept_graph.nodes[sig] = {"origin": "PEER",
                                    "count": data['count'] * 0.5,
                                    "value": data['value'].copy(),
                                    "embed": np.array(data.get('embed', np.zeros(32)))
                                }
                            else:
                                _cur = neighbor.concept_graph.nodes[sig]['count']
                                if Config.ENABLE_CONCEPT_TRANSFER_DAMPING:
                                    _headroom = max(0.0, 1.0 - _cur / Config.CONCEPT_TRANSFER_CEILING)
                                else:
                                    _headroom = 1.0
                                neighbor.concept_graph.nodes[sig]['count'] += data['count'] * 0.5 * _headroom
                            self._log_event("prophet_last_word", recipient=neighbor_id, concept=str(sig)[:50])
                    self._log_event("prophetic_clarity", duration=self._blindness_duration)
                self._log_event("blindness_exit", duration=self._blindness_duration, scar_added=0 if is_prophet else base_scar)
                self._blindness_duration = 0
            else:
                if is_prophet:
                    for x, y in self.cells:
                        field[x, y, CH['resonance']] = min(1.0, field[x, y, CH['resonance']] + 0.02)
                    if phi_hash(self.id, t, 777) < 0.03 and self.world and self.world.echo_system:
                        self.world.echo_system.store_anomaly(self, "prophetic_blindness")
                self._cellular_endurance = min(1.0, self._cellular_endurance + 0.008)
                return 0
        else:
            if self.spirit_gap >= gap_thresh * 0.85 and self.soul_weight < soul_thresh * 1.2:
                self._in_blindness = True
                self._blindness_duration = 1
                self._log_event("blindness_enter", gap=self.spirit_gap, soul=self.soul_weight,
                                gap_thresh=gap_thresh, soul_thresh=soul_thresh)
                return 0
        return 1

    def _calculate_blindness_thresholds(self, world, t):
        adaptive_gap, adaptive_soul = 0.9, 0.3
        if world:
            alive = [p for p in world.patterns if p.alive]
            if len(alive) >= 10:
                gaps = [p.spirit_gap for p in alive]
                souls = [p.soul_weight for p in alive]
                pop_gap_90 = np.percentile(gaps, 90)
                pop_soul_20 = np.percentile(souls, 20)
                hist_gap = list(self._gap_history)[:-1] if len(self._gap_history) > 1 else self._gap_history
                hist_soul = list(self._soul_history)[:-1] if len(self._soul_history) > 1 else self._soul_history
                personal_gap = float(np.mean(hist_gap)) if hist_gap else self.spirit_gap
                personal_soul = float(np.mean(hist_soul)) if hist_soul else self.soul_weight
                raw_gap = 0.6 * pop_gap_90 + 0.3 * personal_gap + 0.1 * 0.9
                raw_soul = 0.6 * pop_soul_20 + 0.3 * personal_soul + 0.1 * 0.3
                adaptive_gap = float(np.clip(raw_gap, 0.6, 1.0))
                adaptive_soul = float(np.clip(max(0.15, raw_soul), 0.05, 0.4))
        return adaptive_gap, adaptive_soul

    def sense_self_and_other(self, field, t):
        self._clock_phase = (self._clock_phase + self._base_freq * (1.0 + self.affect['arousal'] * 0.5)) % (2 * np.pi)
        self._phase_sync = getattr(self, '_phase_sync', 0.0)
        lp_ws = getattr(self, 'local_perception', None)
        perception_signal = float(np.mean(lp_ws)) if lp_ws is not None and len(lp_ws) else 0.0
        soma_vec = getattr(self, 'soma_vector', np.zeros(7))
        soma_signal = float(np.mean(soma_vec)) if len(soma_vec) else 0.0
        emotion_signal = float(self.emotional_memory.get('grief', 0.0) - self.emotional_memory.get('gratitude', 0.0))
        existential_signal = float(self.unresolved_contradiction)
        raw_workspace = {
            'perception': perception_signal,
            'soma': soma_signal,
            'emotion': emotion_signal,
            'existential': existential_signal,
        }
        surprise_weights = {}
        for k, v in raw_workspace.items():
            prev = self._workspace_prev.get(k, v)
            surprise_weights[k] = abs(v - prev)
        phase_bias = {
            'perception': max(0.05, np.cos(self._clock_phase)),
            'soma': max(0.05, np.sin(self._clock_phase)),
            'emotion': max(0.05, -np.cos(self._clock_phase)),
            'existential': max(0.05, -np.sin(self._clock_phase)),
        }
        combined_weights = {k: (surprise_weights[k] + 1e-6) * phase_bias[k] for k in raw_workspace}
        total_combined = sum(combined_weights.values()) or 1e-6
        self.workspace_weights = {k: w / total_combined for k, w in combined_weights.items()}
        self.workspace = raw_workspace
        self._workspace_prev = dict(raw_workspace)
        self._workspace_dominant = max(self.workspace_weights, key=self.workspace_weights.get)
        if Config.ENABLE_PHI_PROXY:
            self._ws_hist.append(dict(raw_workspace))
            if self.age % Config.EXPENSIVE_COMPUTE_INTERVAL == 0 and len(self._ws_hist) >= 4:
                _M = np.array([list(h.values()) for h in self._ws_hist])
                if np.all(np.std(_M, axis=0) > 1e-9):
                    _C = np.corrcoef(_M, rowvar=False)
                    if not np.any(np.isnan(_C)):
                        _off = [abs(_C[i, j]) for i in range(4) for j in range(4) if i < j]
                        _coupling = float(np.mean(_off))
                        _H = -sum(w * np.log(w + 1e-9) for w in self.workspace_weights.values())
                        _diff = _H / np.log(4)
                        self._phi_proxy = float(np.clip(_coupling * _diff, 0, 1))
        target_valence = float(np.clip(
            self.emotional_memory.get('gratitude', 0.0) - self.emotional_memory.get('grief', 0.0), -1.0, 1.0))
        target_arousal = float(np.clip(self.cognitive_tension + self.self_surprise, 0.0, 1.0))
        self.affect['valence'] = 0.9 * self.affect['valence'] + 0.1 * target_valence
        self.affect['arousal'] = 0.9 * self.affect['arousal'] + 0.1 * target_arousal
        ext = perception_signal * (1.0 + self.affect['valence'] * 0.5)
        internal = np.array([
            self.soul_weight,
            self.unresolved_contradiction,
            float(self.emotional_memory.get('grief', 0.0)),
            float(self.emotional_memory.get('gratitude', 0.0)),
            float(np.clip(self.energy, 0.0, 1.0)),
            float(np.clip(self.spirit_gap, 0.0, 1.0)),
        ])
        blended = internal * (0.5 + 0.5 * ext)
        self.felt_state = 0.8 * self.felt_state + 0.2 * blended
        self._felt_intensity = float(np.linalg.norm(self.felt_state))
        grown = len(self.cells - self._prev_cells_snapshot)
        lost = len(self._prev_cells_snapshot - self.cells)
        caused_change = grown - lost
        if caused_change != 0:
            self._causal_memory.append(caused_change)
            self.soul_weight += 0.002 if caused_change > 0 else -0.001
            if abs(caused_change) >= 3:
                self._log_event("self_causality_felt", delta=caused_change)
        self._prev_cells_snapshot = set(self.cells)
        self._causal_efficacy = float(np.mean(self._causal_memory)) if self._causal_memory else 0.0
        if self.world is not None and self.trust_ledger.entries:
            other_id = max(self.trust_ledger.entries, key=self.trust_ledger.entries.get)
            other = self.world.pattern_dict.get(other_id)
            if other is not None and other.alive:
                actual = round(float(other.soul_weight), 2)
                predicted = self._mind_prediction.get(other_id)
                if predicted is not None:
                    surprise = abs(actual - predicted)
                    if surprise > Config.OTHER_MIND_SURPRISE_THRESHOLD:
                        self._social_surprise = 0.7 * self._social_surprise + 0.3 * surprise
                        self.unresolved_contradiction = min(1.0, self.unresolved_contradiction + surprise * 0.05)
                        self._log_event('contradiction_self_surprise', delta=round(float(surprise * 0.05), 4))
                        if surprise > Config.OTHER_MIND_SURPRISE_THRESHOLD * 2:
                            self._log_event("other_mind_surprise", target=other_id, surprise=round(surprise, 3))
                self._mind_prediction[other_id] = actual
                other_belief_about_me = getattr(other, '_mind_prediction', {}).get(self.id)
                meta_predicted = self._meta_prediction.get(other_id)
                if meta_predicted is not None and other_belief_about_me is not None:
                    meta_error = abs(other_belief_about_me - meta_predicted)
                    if meta_error > Config.META_SURPRISE_THRESHOLD:
                        self._meta_surprise = 0.7 * self._meta_surprise + 0.3 * meta_error
                        self.unresolved_contradiction = min(1.0, self.unresolved_contradiction + meta_error * 0.04)
                        self._log_event('contradiction_meta_error', delta=round(float(meta_error * 0.04), 4))
                        if meta_error > Config.META_SURPRISE_THRESHOLD * 2:
                            self._log_event("theory_of_mind_surprise", target=other_id, meta_error=round(meta_error, 3))
                my_self_view = round(float(self.soul_weight), 2)
                trust_to_other = self.trust_ledger.entries.get(other_id, 0.5)
                _neutral = 0.5
                if meta_predicted is None or self.age % 37 == 0:
                    self._meta_prediction[other_id] = round(
                        _neutral + trust_to_other * (my_self_view - _neutral), 2)
                trust_to_other = self.trust_ledger.entries.get(other_id, 0.5)
                diff = (other._clock_phase - self._clock_phase + np.pi) % (2 * np.pi) - np.pi
                self._clock_phase = (self._clock_phase + 0.02 * trust_to_other * diff) % (2 * np.pi)
                self._phase_sync = float(np.cos(diff))
                if predicted is not None and surprise > Config.OTHER_MIND_SURPRISE_THRESHOLD:
                    rhythm_mod = 1.0 - 0.4 * self._phase_sync
                    self.unresolved_contradiction = min(1.0, self.unresolved_contradiction + surprise * 0.05 * (rhythm_mod - 1.0))
                    self._log_event('contradiction_self_surprise', delta=round(float(surprise * 0.05 * (rhythm_mod - 1.0)), 4))
        self.generate_inner_speech()
        self.update_temporal_context()
        self.update_subpersonalities()

    def _nav_block_dir(self, d):
        bd = getattr(self, '_nav_block_dirs', None)
        if bd is None:
            bd = self._nav_block_dirs = {}
        age = int(getattr(self, 'age', 0))
        for k_ in [k2 for k2, v2 in bd.items() if v2 <= age]:
            del bd[k_]
        bd[tuple(d)] = age + int(Config.NAV_PERC_BLOCK_TTL)

    def _nav_spatial_goal(self, reason='new', exclude=None, carry=None):
        age = int(getattr(self, 'age', 0))
        if reason == 'new' and (getattr(self, '_nav_progress', None) or age < int(getattr(self, '_nav_cool_until', 0))):
            return False
        scores = getattr(self, '_dir_scores', None)
        reach_map = getattr(self, '_dir_reach', None)
        if not scores or not reach_map:
            return False
        block_until = getattr(self, '_nav_block_dirs', None) or {}
        pick = None
        for d_ in sorted(scores, key=scores.get, reverse=True):
            if not np.isfinite(scores[d_]) or d_ == exclude or block_until.get(d_, -1) > age:
                continue
            if reach_map.get(d_, 0) >= int(Config.NAV_WAYPOINT_MIN) + 1:
                pick = d_
                break
        if pick is None:
            # [fix 29.10] под плотностью почти ни у одного направления нет
            # >=NAV_WAYPOINT_MIN+1 свободных клеток (в логе explore_no_goal=69234 >> go_goal=21651 —
            # это симптом именно того, что цель чаще всего некуда ставить). Вместо того чтобы
            # совсем не ставить цель (агент замирает без goal), берём лучшее по score направление
            # хотя бы с 1 свободной клеткой — короткий шаг лучше паралича.
            for d_ in sorted(scores, key=scores.get, reverse=True):
                if not np.isfinite(scores[d_]) or d_ == exclude or block_until.get(d_, -1) > age:
                    continue
                if reach_map.get(d_, 0) >= 1:
                    pick = d_
                    break
        if pick is None:
            return False
        W = int(Config.WORLD_SIZE)
        cx, cy = self._nav_body_center()
        step = int(min(int(reach_map[pick]), int(Config.NAV_WAYPOINT_MAX)))
        tx, ty = (cx + pick[0] * step) % W, (cy + pick[1] * step) % W
        sd = float(step * np.hypot(pick[0], pick[1]))
        c_ = carry or {}
        self.goals = [g for g in self.goals if g.get('_source') != 'spatial_gradient']
        self._nav_progress = {
            'target': (tx, ty), 'dir': pick, 'start_x': cx, 'start_y': cy, 'start_dist': sd,
            'best_dist': float('inf'), 'stuck_ticks': 0, 'status': 'moving', 'reason': reason,
            'wp_t': age, 'dist': sd, 'progress': 0.0,
            'recalc': int(c_.get('recalc', 0)) + (1 if carry else 0),
            't0': int(c_.get('t0', age)), 'x0': c_.get('x0', cx), 'y0': c_.get('y0', cy),
            'path_len': float(c_.get('path_len', 0.0)), 'far': bool(c_.get('far', False)),
            'revisit': bool(c_.get('revisit', False)), 'px': float(cx), 'py': float(cy),
            'origin': str(c_.get('origin', reason)),
        }
        self.goals.append({
            "type": "explore",
            "priority": 1.3 + float(getattr(self, '_dir_contrast', 0.2)) * 0.5,
            "target": (tx, ty),
            "age": 0,
            "persistence": 25,
            "_source": "spatial_gradient",
        })
        self._log_event("goal_spatial_explore" if not carry else "nav_recalculate", dir=pick, target=(tx, ty), reason=reason)
        world_ = getattr(self, 'world', None)
        if world_ is not None:
            _nav_audit(world_)['c']['wp_new' if not carry else 'wp_recalc_' + reason] += 1
        return True

    def _nav_finish(self, status, reason=None):
        nav = getattr(self, '_nav_progress', None)
        if not nav:
            return
        W = int(Config.WORLD_SIZE)
        age = int(getattr(self, 'age', 0))
        cx, cy = self._nav_body_center()
        ax, ay = abs(float(cx) - float(nav['x0'])), abs(float(cy) - float(nav['y0']))
        opt = float(np.hypot(min(ax, W - ax), min(ay, W - ay)))
        act = float(nav['path_len'])
        world_ = getattr(self, 'world', None)
        if world_ is not None:
            au = _nav_audit(world_)
            au['c']['ep_total'] += 1
            au['c']['ep_' + status] += 1
            au['c']['recalc_total'] += int(nav['recalc'])
            # Keep the terminal cause explicit: status alone cannot distinguish why
            # an episode was abandoned. This is audit-only and does not affect navigation.
            end_key = str(reason or status).strip().lower().replace(' ', '_')
            au['c']['ep_end_reason_' + end_key] += 1
            au['ep'].append({'status': status, 'ticks': age - int(nav['t0']), 'opt': opt, 'act': act,
                             'recalc': int(nav['recalc']), 'revisit': bool(nav.get('revisit')),
                             'origin': str(nav.get('origin', 'new')), 'end_reason': reason or status})
        self.goals = [g for g in self.goals if g.get('_source') != 'spatial_gradient']
        self._log_event('nav_' + status, ticks=age - int(nav['t0']), recalc=int(nav['recalc']), reason=reason)
        if nav.get('reason') == 'morse_seek':
            _target_id = nav.get('target_agent_id')
            if status == 'arrived':
                _target_agent = (world_.pattern_dict.get(int(_target_id))
                                 if world_ is not None and _target_id is not None else None)
                # Reward only an actual arrival. The responder gains a small soul-weight
                # increment; the agent that asked for help experiences gratitude.
                self.soul_weight = float(np.clip(
                    float(getattr(self, 'soul_weight', 0.5))
                    + float(getattr(Config, 'SOUL_WEIGHT_GAIN_RATE', 0.05)) * 0.1, 0.0, 1.0))
                if _target_agent is not None and hasattr(_target_agent, 'emotional_memory'):
                    _target_agent.emotional_memory['gratitude'] = float(np.clip(
                        float(_target_agent.emotional_memory.get('gratitude', 0.0)) + 0.02, 0.0, 1.0))
                self._log_event('morse_seek_arrived', target=_target_id,
                                soul_weight=round(float(self.soul_weight), 4))
            self.goals = [g for g in getattr(self, 'goals', []) if g.get('_source') != 'morse_seek']
        self._nav_last = {'status': status, 'age': age, 'recalc': int(nav['recalc']), 'reason': reason}
        self._nav_cool_until = age + (5 if status == 'arrived' else int(Config.NAV_COOLDOWN))
        if getattr(self, '_nav_move_attempt_pending', None) is nav:
            self._nav_move_attempt_pending = None
        if getattr(self, '_nav_move_ok_pending', None) is nav:
            self._nav_move_ok_pending = None
        self._nav_progress = None

    def _nav_step(self, field):
        nav = getattr(self, '_nav_progress', None)
        if not nav:
            return
        W = int(Config.WORLD_SIZE)
        age = int(getattr(self, 'age', 0))
        # move() runs later in the engine tick; consume the previous tick's attempt
        # and successful displacement for this exact waypoint episode.
        attempted = getattr(self, '_nav_move_attempt_pending', None) is nav
        moved = getattr(self, '_nav_move_ok_pending', None) is nav
        if attempted:
            self._nav_move_attempt_pending = None
        if moved:
            self._nav_move_ok_pending = None
        if bool(getattr(Config, 'NAV_STEP_ONCE_PER_TICK', True)):
            # v42: Pattern.divide() calls update_model() on the parent a second time inside the same tick;
            # a repeated step would add a phantom stuck tick (distance cannot shrink between two calls).
            if getattr(self, '_nav_step_age', None) == age:
                _w = getattr(self, 'world', None)
                if _w is not None:
                    _nav_audit(_w)['c']['step_dup_skipped'] += 1
                return
            self._nav_step_age = age
        cx, cy = self._nav_body_center()
        _target_agent_id = nav.get('target_agent_id')
        if _target_agent_id is not None:
            _world = getattr(self, 'world', None)
            _target_agent = (_world.pattern_dict.get(int(_target_agent_id))
                             if _world is not None else None)
            if (_target_agent is None or not getattr(_target_agent, 'alive', False)
                    or not getattr(_target_agent, 'cells', None)):
                self._nav_finish('abandoned', 'target_lost')
                return
            tx, ty = _target_agent._nav_body_center() if hasattr(_target_agent, '_nav_body_center') else _target_agent.get_center()
            nav['target'] = (int(tx), int(ty))
            _Wdir = int(Config.WORLD_SIZE)
            _dxdir = ((int(tx) - int(cx) + _Wdir // 2) % _Wdir) - _Wdir // 2
            _dydir = ((int(ty) - int(cy) + _Wdir // 2) % _Wdir) - _Wdir // 2
            nav['dir'] = (int(np.sign(_dxdir)), int(np.sign(_dydir)))
            if nav['dir'] == (0, 0):
                nav['dir'] = (1, 0)
        else:
            tx, ty = nav['target']
        ddx, ddy = abs(float(cx) - float(tx)), abs(float(cy) - float(ty))
        dist = float(np.hypot(min(ddx, W - ddx), min(ddy, W - ddy)))
        sx, sy = abs(float(cx) - nav['px']), abs(float(cy) - nav['py'])
        if moved:
            nav['path_len'] += float(np.hypot(min(sx, W - sx), min(sy, W - sy)))
            fx, fy = abs(float(cx) - float(nav['x0'])), abs(float(cy) - float(nav['y0']))
            from_start = float(np.hypot(min(fx, W - fx), min(fy, W - fy)))
            if from_start >= 3.0:
                nav['far'] = True
            elif nav['far'] and from_start <= 1.5:
                nav['revisit'] = True
        nav['px'], nav['py'] = float(cx), float(cy)
        if attempted:
            if dist < nav['best_dist'] - float(Config.NAV_STUCK_EPS):
                nav['best_dist'] = dist
                nav['stuck_ticks'] = 0
                nav['status'] = 'moving'
            else:
                nav['stuck_ticks'] += 1
        nav['dist'] = dist
        nav['progress'] = float(np.clip(1.0 - dist / max(nav['start_dist'], 1e-6), -1.0, 1.0))
        _arrival_radius = 3.0 if nav.get('reason') == 'morse_seek' else 1.5
        if dist <= _arrival_radius:
            nav['status'] = 'arrived'
            self._nav_finish('arrived')
            return
        if nav['stuck_ticks'] == int(Config.NAV_PERC_STALL_TICKS):
            nav['status'] = 'stalled'
            self._hyst_off = 2
            self._fail_dir = nav['dir']
        own = float(field[int(tx), int(ty), CH['owner']])
        reason = None
        if (own != 0.0 and own != float(self.id) and dist > 3.0
                and nav.get('target_agent_id') is None):
            reason = 'blocked'
        elif nav['stuck_ticks'] >= int(Config.NAV_STUCK_TICKS):
            reason = 'stuck'
        elif age - int(nav['wp_t']) > int(Config.NAV_WP_TIMEOUT):
            reason = 'timeout'
        if reason is None:
            return
        if nav.get('target_agent_id') is not None:
            # Do not silently turn a social request into an unrelated exploration waypoint.
            self._nav_finish('abandoned', reason)
            return
        self._nav_block_dir(nav['dir'])
        if nav['recalc'] >= int(Config.NAV_MAX_RECALC) or not self._nav_spatial_goal(reason, exclude=nav['dir'], carry=nav):
            self._nav_finish('abandoned', reason)

    def nav_state(self):
        nav = getattr(self, '_nav_progress', None)
        if not nav:
            last = getattr(self, '_nav_last', None) or {}
            return {'status': last.get('status', 'idle'), 'dist': None, 'progress': 0.0, 'recalc': int(last.get('recalc', 0)), 'reason': last.get('reason')}
        return {'status': nav['status'], 'dist': float(nav['dist']), 'progress': float(nav['progress']), 'recalc': int(nav['recalc']), 'reason': nav['reason']}

    def nav_map_summary(self):
        lm = getattr(self, '_local_map', None) or {}
        age = int(getattr(self, 'age', 0))
        names = ('unknown', 'ok', 'danger', 'obstacle')
        out = {'cells': len(lm), 'fresh': 0, 'stale': 0, 'unknown': 0, 'ok': 0, 'danger': 0, 'obstacle': 0}
        for t_, s_ in lm.values():
            out['stale' if age - t_ > int(Config.NAV_MAP_STALE) else 'fresh'] += 1
            out[names[s_]] += 1
        return out

    def _nav_gap_stats(self, gap):
        """v42: passive per-call statistics of gap_clip against the spatial confidence reported by direction
        perception. gap_clip is a clipped spirit_gap, so this shows how often it disagrees with what the
        agent actually perceives of the space. Changes no behaviour and draws no random numbers."""
        world_ = getattr(self, 'world', None)
        if world_ is None or not bool(getattr(Config, 'NAV_GAPCLIP_STATS', True)):
            return
        try:
            c = _nav_audit(world_)['c']
            g = float(gap)
            b = min(19, max(0, int(g * 20.0)))
            has_goal = getattr(self, '_nav_progress', None) is not None
            c['gap_ticks'] += 1
            c['gap_h%02d_%s' % (b, 'goal' if has_goal else 'nogoal')] += 1
            hi = g > float(Config.GAP_CLIP_CURIOSITY_THRESHOLD)
            lowconf = float(getattr(self, '_dir_conf', 1.0)) < 0.5
            c['gx_%s_%s' % ('hi' if hi else 'lo', 'lowconf' if lowconf else 'okconf')] += 1
        except Exception:
            pass

    def _nav_gapclip_decision(self, gap):
        world_ = getattr(self, 'world', None)
        if world_ is None or not bool(getattr(Config, 'NAV_GAPCLIP_LOG', True)):
            return
        nav = getattr(self, '_nav_progress', None)
        safe = float(getattr(self, '_dir_threat', 0.0)) < 0.3
        if nav is None:
            ch = 'explore_no_goal'
        elif safe and nav.get('stuck_ticks', 0) <= 1:
            ch = 'go_goal'
        elif float(getattr(self, '_dir_conf', 1.0)) < 0.5:
            ch = 'explore_low_conf'
        else:
            ch = 'stuck_goal'
        _nav_audit(world_)['c']['gapclip_' + ch] += 1
        if bool(getattr(Config, 'NAV_GAPCLIP_STATS', True)):
            _c = _nav_audit(world_)['c']
            _c['og_gap_sum_' + ch] += float(gap)
            _c['og_conf_sum_' + ch] += float(getattr(self, '_dir_conf', 1.0))
            _c['og_threat_sum_' + ch] += float(getattr(self, '_dir_threat', 0.0))
        self._gapclip_last = (int(getattr(self, 'age', 0)), ch)
        # v38: this used to be log-only. Now the 'no goal at all, high uncertainty' case
        # actually starts an explore episode via the existing spatial_gradient machinery,
        # instead of the agent just sitting there with a logged classification. Left the
        # other branches (explore_low_conf / stuck_goal) diagnostic-only on purpose: those
        # have a nav episode already in flight, and interrupting it blind felt like the
        # riskier half of this change.
        if (ch == 'explore_no_goal'
                and bool(getattr(Config, 'ENABLE_GAPCLIP_ACTION', True))
                and int(getattr(self, 'age', 0)) >= int(getattr(self, '_nav_cool_until', 0))):
            try:
                if self._nav_spatial_goal('gapclip_explore'):
                    _nav_audit(world_)['c']['gapclip_action_explore'] += 1
            except Exception:
                pass
        elif (ch == 'explore_low_conf' and nav is not None
                and bool(getattr(Config, 'ENABLE_GAPCLIP_LOWCONF_ACTION', False))
                and int(nav.get('stuck_ticks', 0)) >= int(Config.NAV_PERC_STALL_TICKS)
                and int(nav.get('recalc', 0)) < int(Config.NAV_MAX_RECALC)):
            # v40: low confidence while the goal is already stalled -> take another direction through the
            # existing recalc machinery. If no alternative exists the episode is left untouched.
            old_dir = nav.get('dir')
            try:
                if self._nav_spatial_goal('gapclip_lowconf', exclude=old_dir, carry=nav):
                    if old_dir is not None:
                        self._nav_block_dir(old_dir)
                    _nav_audit(world_)['c']['gapclip_action_lowconf'] += 1
            except Exception:
                pass


class PatternSoul:
    def _deposit_final_testament(self):
        if self.world is None:
            return
        if Config.ENABLE_FORGETTING and self.world is not None and self.world._given_rng.rand() < 0.3:
            self._log_event("testament_lost")
            return
        pieces = []
        dlt = getattr(self, 'dialogue_longterm', None)
        if dlt:
            for entry in dlt[-2:]:
                txt = entry.get('text', '') if isinstance(entry, dict) else str(entry)
                if txt:
                    pieces.append(txt[:90])
        narrative = getattr(self, '_self_narrative', None)
        if narrative:
            best = None
            for e in narrative:
                if isinstance(e, dict) and 'soul' in e:
                    if best is None or e.get('soul', 0) > best.get('soul', -1):
                        best = e
            if best is not None:
                pieces.append(f"soul_peak={best.get('soul', 0):.2f} at_t={best.get('t', '?')}")
        if not pieces:
            return
        testament = " | ".join(pieces)[:200]
        self.world.archive.deposit(
            self, "final_testament", weight=2.0,
            text=f"soul={self.soul_weight:.2f} age={self.age}: {testament}"
        )

    def _complete_redemption(self, witness):
        self._log_event("redemption_complete", depth=round(getattr(self, '_redemption_depth', 1.0), 3))
        if witness:
            witness.record(self.id, "redemption_complete")
        if self.world:
            self.world.archive.deposit(self, "redemption", weight=1.5,
                                       text=f"redemption soul={self.soul_weight:.2f}")
        self._log_event("redeemed", arc="redemption")
        _was_role = self.role_type
        self.role_type = "normal"
        if _was_role != "normal":
            self._log_event("became_normal", reason="redemption_arc", age=self.age)
            if self.world is not None:
                self.world.witness.record(self.id, "became_normal", reason="redemption_arc", age=self.age)
        self.soul_weight = max(self.soul_weight, 0.7)
        nightmares = [sig for sig in self.concept_graph.nodes
                      if isinstance(sig, tuple) and len(sig) >= 4 and sig[3].startswith('nightmare_of_')]
        healing_fraction = 1.0 / (1.0 + len(nightmares))
        self.emotional_memory['grief'] = float(np.clip(
            self.emotional_memory.get('grief', 0.5) -
            healing_fraction * (self.emotional_memory.get('grief', 0.5) - 0.1),
            0.0, 1.0
        ))
        self.emotional_memory['gratitude'] = float(np.clip(
            self.emotional_memory.get('gratitude', 0.5) +
            healing_fraction * (0.8 - self.emotional_memory.get('gratitude', 0.5)),
            0.0, 1.0
        ))
        self._scar_of_light = True
        self.body_memory = max(self.body_memory, 0.3)
        self._deterministic_redemption_triggered = False
        self._redemption_depth = 0.0
        self._steps_since_trigger = 0
        self._redemption_active = False
        self._redemption_arc_step = 0
        self._redemption_cooldown = getattr(self, 'age', 0) + 200
        self.event_counts['fold'] = 0
        self.fold_cooldown = 0
        if nightmares and phi_hash(self.id, self.age, 999) < 0.3:
            chosen = min(nightmares, key=lambda s: self.concept_graph.nodes.get(s, {}).get('count', 0.0))
            try:
                partner_id = int(chosen[3].split('_')[-1])
                self.concept_graph.nodes.pop(chosen)
                self._create_dream_concept(partner_id, positive=True, score=1.0)
                self._log_event("nightmare_transformed", partner=partner_id)
            except (ValueError, IndexError):
                pass
        if Config.ENABLE_CULTURAL_MEMORY and self.world and self.world.cultural_memory:
            self.world.cultural_memory.deposit(self, "redemption", intensity=1.5)
        if self.world and self.world.echo_system:
            self.world.echo_system.store_memory_echo(self, "redemption_complete", intensity=1.5)
        redemption_sig = (0.0, 0.0, 0.85, "redemption_memory")
        for s in self.concept_graph.nodes:
            if isinstance(s, tuple) and len(s) >= 4 and s[3] == redemption_sig[3]:
                self.concept_graph.nodes[s]['count'] += 3.0
                break
        else:
            self.concept_graph.nodes[redemption_sig] = {"origin": "OWN",
                'count': 5.0,
                'value': np.array([0.0, 0.0, 0.85, 0.0]),
                'embed': np.zeros(32, dtype=np.float32),
                'eternal': True
            }

    def _continue_redemption_arc(self, field, witness, t, lineage_counts, given_counter,
                                  observed, local_gratitude, local_grief):
        if self.role_type == "disorganizer":
            self._fallen_steps += 1
        if self.role_type == "disorganizer" and not getattr(self, '_deterministic_redemption_triggered', False):
            if getattr(self, '_fallen_steps', 0) >= 300:
                self.emotional_memory['grief'] = max(0.0, self.emotional_memory['grief'] - 0.3)
                self.emotional_memory['gratitude'] = min(1.0, self.emotional_memory['gratitude'] + 0.2)
                self.soul_weight = min(1.0, self.soul_weight + 0.05)
                self._log_event("kiss_of_the_fallen_persistent")
                self._fallen_steps = 0
        if self.role_type == "disorganizer" and not getattr(self, '_deterministic_redemption_triggered', False):
            if self.soul_weight > Config.DETERMINISTIC_REDEMPTION_SOUL:
                self.soul_weight -= 0.01
        if (self.role_type == "disorganizer" and
                not getattr(self, '_forced_soul_collapse_done', False) and
                self.disorganizer_age_at_birth > 0 and
                getattr(self, '_redemption_cooldown', 0) <= self.age):
            steps_since_birth = self.age - self.disorganizer_age_at_birth
            if steps_since_birth >= Config.FORCED_SOUL_COLLAPSE_AGE and self.soul_weight > 0.4:
                old_soul = self.soul_weight
                self.soul_weight = Config.FORCED_SOUL_COLLAPSE_VALUE
                self.emotional_memory['grief'] = Config.FORCED_SOUL_COLLAPSE_GRIEF_BOOST
                self._forced_soul_collapse_done = True
                self._log_event("forced_soul_collapse", old_soul=round(old_soul, 3),
                                new_soul=round(self.soul_weight, 3))
                if witness:
                    witness.record(self.id, "forced_soul_collapse",
                                   old_soul=old_soul, new_soul=self.soul_weight)
        if (self.role_type == "disorganizer" and
                not getattr(self, '_deterministic_redemption_triggered', False) and
                self.disorganizer_age_at_birth > 0):
            steps_as_disorg = self.age - self.disorganizer_age_at_birth
            if steps_as_disorg > 300 and self.soul_weight < 0.15:
                self.emotional_memory['grief'] = 0.55
                self.emotional_memory['gratitude'] = 0.05
                self._log_event("zombie_crisis", steps=steps_as_disorg, soul=self.soul_weight)
        if (self.role_type == "disorganizer" and
                not getattr(self, '_deterministic_redemption_triggered', False) and
                getattr(self, '_redemption_cooldown', 0) <= self.age):
            if (self.soul_weight < Config.DETERMINISTIC_REDEMPTION_SOUL and
                    self.emotional_memory['grief'] > Config.DETERMINISTIC_REDEMPTION_GRIEF):
                self._deterministic_redemption_triggered = True
                self._redemption_depth = 0.0
                self._steps_since_trigger = 0
                self._redemption_active = True
                self._redemption_arc_step = 1
                self.emotional_memory['grief'] = 0.7
                self.semantic_state = "seeking_comfort"
                self._log_event("deterministic_redemption_trigger",
                                soul=self.soul_weight, grief=self.emotional_memory['grief'])
                if witness:
                    witness.record(self.id, "deterministic_redemption_trigger",
                                   soul=self.soul_weight, grief=self.emotional_memory['grief'])
        if self.role_type == "disorganizer" and getattr(self, '_deterministic_redemption_triggered', False):
            self._steps_since_trigger += 1
            grief_progress = max(0.0, (0.7 - self.emotional_memory['grief']) / 0.7)
            grat_progress = self.emotional_memory['gratitude']
            trust_avg = safe_mean(list(self.trust_ledger.entries.values()), 0.0)
            social_catalyst = max(0.0, (trust_avg - 0.4) * 2.0)
            soma_stability = max(0.0, 1.0 - getattr(self, 'soma', 0.5))
            delta_depth = (
                0.0004 +
                grief_progress * 0.006 +
                grat_progress * 0.004 +
                social_catalyst * 0.002 +
                soma_stability * 0.001
            )
            if self.world:
                alive_all = [p for p in self.world.patterns if p.alive]
                disorg_cnt = sum(1 for p in alive_all if p.role_type == "disorganizer")
                disorg_ratio = disorg_cnt / max(1, len(alive_all))
                social_pressure = 1.0 + 0.5 * disorg_ratio
            else:
                social_pressure = 1.0
            nci_factor = 1.0 + 0.3 * getattr(self, '_nci', 0.5)
            delta_depth *= social_pressure * nci_factor
            if self.emotional_memory['grief'] > 0.75:
                delta_depth *= 0.2
            field_grief_pressure = local_grief * 0.5
            delta_depth = max(0.0, delta_depth - field_grief_pressure * 0.001)
            self._redemption_depth = float(np.clip(
                getattr(self, '_redemption_depth', 0.0) + delta_depth, 0.0, 1.0
            ))
            depth = self._redemption_depth
            if depth < 0.30:
                self.semantic_state = "seeking_comfort"
                self._redemption_arc_step = 1
            elif depth < 0.65:
                self.semantic_state = "grateful_but_cautious"
                self._redemption_arc_step = 2
            else:
                self.semantic_state = "contentment"
                self._redemption_arc_step = 3
            soul_floor = Config.REDEMPTION_ARC_SOUL_LOCK + depth * 0.30
            if self.soul_weight < soul_floor:
                self.soul_weight = soul_floor
                self._log_event("soul_floor_enforced", floor=round(soul_floor, 3))
            if self._steps_since_trigger % 50 == 0:
                self._log_event("redemption_progress",
                                depth=round(depth, 3),
                                grief=round(self.emotional_memory['grief'], 3),
                                grat=round(self.emotional_memory['gratitude'], 3),
                                soul=round(self.soul_weight, 3))
            if self._redemption_depth >= 1.0:
                self._complete_redemption(witness)
            elif (self._redemption_depth > 0.80 and
                  self.emotional_memory['grief'] < 0.15 and
                  self.emotional_memory['gratitude'] > 0.5):
                self._complete_redemption(witness)
        completed_arc = self.arc_tracker.update(self.semantic_state)
        if (completed_arc is None and
                self.arc_tracker.active_arc == "grief_cycle" and
                self.emotional_memory['grief'] > Config.ARC_GRIEF_FORCE_THRESHOLD):
            self.arc_tracker.completed_arcs["grief_cycle"] = \
                self.arc_tracker.completed_arcs.get("grief_cycle", 0) + 1
            completed_arc = "grief_cycle"
            self.epistemic_scar = max(Config.EPISTEMIC_SCAR_MIN,
                                      self.epistemic_scar - Config.ARC_GRIEF_FORCE_HEAL)
        if completed_arc == "redemption" and self.role_type == "disorganizer":
            self._complete_redemption(witness)
        if completed_arc:
            self._log_event("arc_completed", arc=completed_arc)
            self.pred_error *= Config.ARC_COMPLETION_PRED_ERROR_REDUCTION
            self._heal_epistemic_scar()
            if self.world:
                self.world.archive.deposit(self, "arc_completed", weight=1.0,
                                           text=f"arc={completed_arc}")
            if Config.ENABLE_CULTURAL_MEMORY and self.world and self.world.cultural_memory:
                self.world.cultural_memory.deposit(self, "arc_completed")
        self._check_semantic_stagnation(witness=witness)
        if self.role_type == "disorganizer" and self.redemption_timer > 0:
            self.redemption_timer -= 1
            if self.redemption_timer == 0:
                self.semantic_state = "seeking_comfort"
                self.emotional_memory['grief'] = 0.7
                self._log_event("redemption_seeking")
        if self.quarantine_timer > 0:
            self.quarantine_timer -= 1
        elif self.epistemic_scar > Config.QUARANTINE_SCAR_THRESHOLD and self.role_type == "normal":
            self.quarantine_timer = Config.QUARANTINE_SILENCE_STEPS
            self._log_event("quarantine_enter", scar=self.epistemic_scar)
        for (x, y) in self.cells:
            if self.emotional_memory['gratitude'] > 0.5:
                field[x, y, CH['intent_cooperate']] = min(1.0,
                    field[x, y, CH['intent_cooperate']] + 0.3)
            elif self.emotional_memory['grief'] > 0.5:
                field[x, y, CH['intent_seek_help']] = min(1.0,
                    field[x, y, CH['intent_seek_help']] + 0.3)
            elif local_gratitude > 0.3:
                field[x, y, CH['intent_explore']] = min(1.0,
                    field[x, y, CH['intent_explore']] + 0.2)
        self.update_world_model(observed)
        self.generate_goals(field)
        self.update_intent()
        if self.age % 5 == 0 or self.local_phase in ("CRISIS", "DREAM") or self.cognitive_tension > 0.8:
            self.form_concepts()
        self.apply_intent_actions(field)
        self.update_resonance(field)
        self.update_phase(field)
        self._heal_epistemic_scar()

    def _apply_fold(self, witness=None):
        self._cellular_endurance = max(self._cellular_endurance, 0.4)
        self.emotional_memory['gratitude'] = Config.FOLD_GRATITUDE_RESET
        self.emotional_memory['grief'] = float(np.clip(
            self.emotional_memory['grief'] + Config.FOLD_GRIEF_BOOST,
            0.0, Config.MAX_GRIEF_SIGNAL))
        self.crisis_memory = min(Config.CRISIS_MEMORY_MAX,
                                 self.crisis_memory + Config.FOLD_CRISIS_SPIKE)
        self.epistemic_scar = max(Config.EPISTEMIC_SCAR_MIN,
                                  self.epistemic_scar - Config.FOLD_SCAR_RELEASE)
        self.cognitive_tension = max(0.0, self.cognitive_tension - 0.3)
        self.semantic_state = "seeking_comfort"
        self.semantic_state_age = 0
        self.fold_cooldown = Config.FOLD_COOLDOWN_DURATION
        self._log_event("fold",
                        grat=self.emotional_memory['gratitude'],
                        grief=self.emotional_memory['grief'])
        if len(self.cells) > 50:
            cells_list = sorted(self.cells)
            loss_fraction = 0.3
            loss_count = max(1, int(len(cells_list) * loss_fraction))
            to_remove = cells_list[:loss_count]
            for (x, y) in to_remove:
                self.cells.discard((x, y))
                if self.world and self.world.field is not None:
                    self.world.field[x, y, CH['owner']] = 0
            self._log_event("fold_cell_loss", lost=loss_count, remaining=len(self.cells))
        if self.world:
            self.world.archive.deposit(self, "fold", weight=1.2,
                                       text=f"fold at soul={self.soul_weight:.2f}")
        if Config.ENABLE_CULTURAL_MEMORY and self.world and self.world.cultural_memory:
            self.world.cultural_memory.deposit(self, "fold")
        if witness:
            witness.record(self.id, "fold",
                           grat=self.emotional_memory['gratitude'],
                           grief=self.emotional_memory['grief'])

    def apply_scream(self, field, witness=None):
        if not self.in_scream:
            self._log_event("scream_enter", soul=round(self.soul_weight,3))
            self.in_scream = True
            self.soul_weight = max(0.05, self.soul_weight - 0.2)
            self.unresolved_contradiction = min(1.0, self.unresolved_contradiction + 0.3)
            self._log_event('contradiction_soul_scream', delta=round(float(0.3), 4))
            self.crisis_memory = min(Config.CRISIS_MEMORY_MAX, self.crisis_memory + 0.15)
            if witness:
                witness.record(self.id, "scream", soul=self.soul_weight, crisis=self.crisis_memory)
            if self.world and self.world.echo_system:
                self.world.echo_system.store_memory_echo(self, "scream", intensity=1.0)
            if self.world is not None:
                with self.world._energy_span('apply_scream.energy_penalty'):
                    for (x, y) in self.cells:
                        field[x, y, CH['scar']] = min(1.0, field[x, y, CH['scar']] + Config.SCREAM_SCAR_IMPACT)
                        field[x, y, CH['binding']] = min(1.0, field[x, y, CH['binding']] + Config.SCREAM_BINDING_BOOST)
                        field[x, y, CH['energy']] = max(-0.2, field[x, y, CH['energy']] - Config.SCREAM_ENERGY_PENALTY)
                        field[x, y, CH['crisis']] = min(1.0, field[x, y, CH['crisis']] + 0.3)
            else:
                for (x, y) in self.cells:
                    field[x, y, CH['scar']] = min(1.0, field[x, y, CH['scar']] + Config.SCREAM_SCAR_IMPACT)
                    field[x, y, CH['binding']] = min(1.0, field[x, y, CH['binding']] + Config.SCREAM_BINDING_BOOST)
                    field[x, y, CH['energy']] = max(-0.2, field[x, y, CH['energy']] - Config.SCREAM_ENERGY_PENALTY)
                    field[x, y, CH['crisis']] = min(1.0, field[x, y, CH['crisis']] + 0.3)
            for (x, y) in self.cells:
                for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nx, ny = (x+dx) % Config.WORLD_SIZE, (y+dy) % Config.WORLD_SIZE
                    field[nx, ny, CH['signal_alarm']] = min(1.0, field[nx, ny, CH['signal_alarm']] + 0.5)

    def apply_given_operator(self, field, forced=False, witness=None):
        self._probe_apply_given_calls = int(getattr(self, "_probe_apply_given_calls", 0)) + 1
        if forced:
            self._probe_given_forced_apply_count = int(getattr(self, "_probe_given_forced_apply_count", 0)) + 1
        _probe_model_before = self.model.copy()
        if not forced and safe_mean(field[:,:,CH['unknown']], 0.1) < 0.08:
            return
        if not forced and (self.age < Config.GIVEN_MIN_AGE or self.given_cooldown > 0):
            return
        self._probe_apply_given_effective = int(getattr(self, "_probe_apply_given_effective", 0)) + 1
        if forced:
            self._probe_apply_given_effective_forced = int(getattr(self, "_probe_apply_given_effective_forced", 0)) + 1
        if not self.given_trigger:
            self._log_event("given_triggered")
        if witness:
            witness.record(self.id, "given", forced=forced, tension=self.cognitive_tension)
        flip = np.array([phi_hash(self.id, i, 999) - 0.5 for i in range(8)])
        shake_strength = Config.GIVEN_MODEL_SHAKE if not forced else 0.4
        if forced:
            flip *= 1.5
        self.prediction += 0.6 * flip
        self.model *= shake_strength
        _probe_delta_vec = np.asarray(self.model - _probe_model_before, dtype=float)
        self._probe_last_given_model_delta = float(np.linalg.norm(_probe_delta_vec))
        self._probe_given_model_changed_calls = int(getattr(self, "_probe_given_model_changed_calls", 0)) + int(bool(np.any(_probe_delta_vec != 0.0)))
        self._probe_given_model_delta_by_channel = np.abs(_probe_delta_vec).copy()
        self._probe_last_given_shake_strength = float(shake_strength)
        self.self_model *= shake_strength
        self.pred_error = min(self.pred_error + 0.2, Config.MAX_METRIC)
        self.confidence = 1.0 / (1.0 + self.pred_error)
        self.given_count += 1
        self.given_trigger = True
        self.given_cooldown = Config.GIVEN_COOLDOWN
        self.crisis_memory = min(self.crisis_memory + 0.1, Config.CRISIS_MEMORY_MAX)
        self.body_memory += 0.05
        for (x, y) in self.cells:
            field[x, y, CH['unknown']] = min(field[x, y, CH['unknown']] + 0.15, 0.8)
            field[x, y, CH['crisis']] = min(field[x, y, CH['crisis']] + 0.1, 1.0)
            field[x, y, CH['binding']] = min(field[x, y, CH['binding']] + 0.25, 1.0)
        self.epistemic_scar = float(np.clip(self.epistemic_scar + Config.GIVEN_SCAR_COST, 0.0, 1.0))
        if Config.ENABLE_DELTA_MODELING_GIVEN and witness:
            current_unknown = safe_mean(field[:,:,CH['unknown']], 0)
            modeling_error = abs(current_unknown - self.unknown_prediction)
            if modeling_error < Config.MODELING_GIVEN_ERROR_THRESHOLD:
                self.epistemic_scar = min(1.0, self.epistemic_scar + Config.MODELING_GIVEN_PENALTY)
                witness.record(self.id, "modeling_given", penalty=Config.MODELING_GIVEN_PENALTY, err=modeling_error)

    def _heal_epistemic_scar(self):
        if self.epistemic_scar <= Config.EPISTEMIC_SCAR_MIN:
            return
        scar_pressure = self.epistemic_scar ** 2
        base_rate = 0.0
        if (self.semantic_state in ["contentment", "grateful_but_cautious", "seeking_comfort"] and
            self.emotional_memory['gratitude'] > Config.HEALING_GRATITUDE_MIN):
            crisis_penalty = 1.0 - np.clip(self.crisis_memory * 0.8, 0.0, 0.5)
            base_rate += (1.0 - Config.EPISTEMIC_HEALING_RATE) * scar_pressure * crisis_penalty
        if self.age > Config.HEAL_WISDOM_AGE and self.coherence > Config.WISDOM_COHERENCE_MIN:
            base_rate += Config.HEAL_WISDOM_BONUS
        if self.age > Config.HEAL_VETERAN_AGE and self.coherence > 0.7:
            base_rate += Config.HEAL_VETERAN_BONUS
        if (self.age > Config.HEAL_ENLIGHTENED_AGE and
            self.emotional_memory['gratitude'] > Config.ENLIGHTENED_GRATITUDE_MIN and
            self.epistemic_scar > Config.ENLIGHTENED_SCAR_THRESHOLD):
            if self.coherence > Config.ENLIGHTENED_COHERENCE_FLOOR:
                base_rate += Config.HEAL_ENLIGHTENED_BONUS
        base_rate += self.arc_tracker.get_heal_bonus()
        if self.world is not None:
            base_rate *= (1 + 0.02 * min(5, self.world.culture_ratchet // 50))
        heal = float(np.clip(base_rate, 0.0, Config.SCAR_HEAL_CAP_PER_STEP))
        self.epistemic_scar = max(Config.EPISTEMIC_SCAR_MIN, self.epistemic_scar - heal)


CONTENT_RESPONSE_ENABLED = False


class Pattern(PatternDream, PatternSelfModel, PatternCulture, PatternCommunication, PatternPhysiology, PatternCognition, PatternPerception, PatternSoul):
    soul_weight = FloatField('soul_weight', default=0.5, clamp_min=0.0, clamp_max=1.0)
    spirit_gap = FloatField('spirit_gap', default=0.0, clamp_min=0.0, clamp_max=2.0)
    epistemic_scar = FloatField('epistemic_scar', default=0.35, clamp_min=0.0, clamp_max=1.0)
    coherence = FloatField('coherence', default=0.5, clamp_min=0.0, clamp_max=1.0)
    pred_error = FloatField('pred_error', default=0.0, clamp_min=0.0, clamp_max=2.0)
    cognitive_tension = FloatField('cognitive_tension', default=0.0, clamp_min=0.0, clamp_max=2.0)
    body_memory = FloatField('body_memory', default=0.0, clamp_min=0.0, clamp_max=1.0)
    unresolved_contradiction = FloatField('unresolved_contradiction', default=0.01, clamp_min=0.0, clamp_max=1.0)
    soma = FloatField('soma', default=0.0, clamp_min=0.0, clamp_max=1.0)
    energy = FloatField('energy', default=0.0, clamp_min=0.0, clamp_max=Config.AGENT_ENERGY_MAX)
    protection_level = FloatField('protection_level', default=0.0, clamp_min=0.0, clamp_max=(1.0 if Config.ENABLE_PROTECTION_SHIELD else 0.0))
    self_phenomenal_error = FloatField('self_phenomenal_error', default=1.0, clamp_min=0.0, clamp_max=2.0)
    self_surprise = FloatField('self_surprise', default=0.0, clamp_min=0.0, clamp_max=2.0)
    given_count = FloatField('given_count', default=0.0, clamp_min=0.0)
    vorticity = FloatField('vorticity', default=0.0, clamp_min=-1.0, clamp_max=1.0)
    alarm_level = FloatField('alarm_level', default=0.0, clamp_min=0.0, clamp_max=1.0)
    _linguistic_confidence = FloatField('_linguistic_confidence', default=0.5, clamp_min=0.0, clamp_max=1.0)
    _nci = FloatField('_nci', default=0.5, clamp_min=0.0, clamp_max=1.0)
    _cellular_endurance = FloatField('_cellular_endurance', default=1.0, clamp_min=0.0, clamp_max=1.0)
    _core_identity_strength = FloatField('_core_identity_strength', default=0.5, clamp_min=0.0, clamp_max=1.0)
    _reentry_signal = FloatField('_reentry_signal', default=0.0, clamp_min=0.0, clamp_max=1.0)
    epistemic_load = FloatField('epistemic_load', default=0.0, clamp_min=0.0, clamp_max=1.0)
    scar_dream = FloatField('scar_dream', default=0.0, clamp_min=-1.0, clamp_max=1.0)
    emotional_memory = EmotionalMemoryField('emotional_memory')

    def _x2_change(self, name, new_value, offered=None):
        old = float(self.energy)
        requested = float(new_value)
        upper = float(getattr(Config, 'AGENT_ENERGY_MAX', 2.0))
        lower = 0.0
        overflow = max(0.0, requested - upper)
        debt = max(0.0, lower - requested)
        bounded = min(upper, max(lower, requested))
        self.energy = bounded
        actual = float(self.energy) - old
        if self.world is not None:
            self.world._ledger_add_x2(name, actual)
            if overflow > 1e-12:
                # HONEST ENERGY: overflow над потолком X2 — реальная потеря ТОЛЬКО если донор (поле X1) уже списан на
                # полную сумму (источники из _BOUNDARY_PAIRS). Для остальных (бонусы, награды) запрошенное сверх потолка
                # никогда не существовало: раньше оно ошибочно писалось как физическая диссипация (+44.6 к error).
                _backed = getattr(self.world, '_OVERFLOW_BACKED', None)
                if _backed is None or name in _backed:
                    self.world._ledger_add_x2('X2.overflow_dissipated', -overflow, count_in_total=False)
                    _ob = getattr(self.world, '_overflow_by_src', None)
                    if _ob is not None and getattr(self.world, '_ledger_window', False):
                        _ob[name] = _ob.get(name, 0.0) + float(overflow)
                else:
                    self.world._memo_add(name + ' [запрос выше потолка X2: энергия не создавалась]', overflow)
            # No negative X2 balance is stored: an unaffordable remainder is rejected, not booked as energy debt.
            want = (requested - old) if offered is None else float(offered)
            lost = want - actual - (overflow if requested > upper else 0.0)
            if abs(lost) > 1e-12 and not (requested < lower):
                self.world._memo_add(name, lost)
        return actual

    def _log_event(self, event_type, **kwargs):
        self.biography.append({'t': self.age, 'type': event_type, **kwargs})
        self.event_counts[event_type] = self.event_counts.get(event_type, 0) + 1
        if event_type in ('intent_switch', 'intent_switch_unpaid'):
            # ЕДИНЫЙ мировой счётчик: считается в точке возникновения события и не зависит от того,
            # увидел ли агента ledger (родился и исчез внутри тика и т.п.). [I] MECHANISMS и FINAL ACTION STATS сверяются с ним.
            try:
                _w = self.world
                if _w is not None:
                    _evw = getattr(_w, '_ev_world', None)
                    if _evw is None:
                        _evw = {}
                        _w._ev_world = _evw
                    _evw[event_type] = _evw.get(event_type, 0) + 1
            except Exception:
                pass
        if event_type.startswith("death_") and self.world is not None:
            self.world.witness.record(self.id, event_type)
        elif event_type in ("born_in_niche", "intrinsic_reward", "testament_lost") \
                and self.world is not None:
            self.world.witness.record(self.id, event_type, **kwargs)
        if Config.ENABLE_IDENTITY_THREAD:
            _anchor_weight = {
                "redemption_complete": 0.9, "root_question_formed": 0.8,
                "fold": 0.6, "identity_crisis_transformation": 0.85,
                "gene_emerged": 0.4, "born_in_niche": 0.3,
            }.get(event_type)
            if _anchor_weight is not None:
                _anchor = {'t': self.age, 'type': event_type, 'weight': _anchor_weight}
                if len(self._identity_anchors) < Config.IDENTITY_THREAD_MAX_ANCHORS:
                    self._identity_anchors.append(_anchor)
                else:
                    _weakest = min(self._identity_anchors, key=lambda a: a['weight'])
                    if _anchor_weight > _weakest['weight']:
                        self._identity_anchors.remove(_weakest)
                        self._identity_anchors.append(_anchor)

    def _get_labyrinth_params(self):
        if self.world:
            return (self.world.phi_labyrinth_threshold,
                    self.world.phi_labyrinth_grow_penalty,
                    self.world.phi_labyrinth_move_penalty)
        return (Config.PHI_LABYRINTH_THRESHOLD,
                Config.PHI_LABYRINTH_GROW_PENALTY,
                Config.PHI_LABYRINTH_MOVE_PENALTY)

    def _get_layer(self):
        if self.soul_weight < Config.LAYER_VOID_SOUL_THRESHOLD:
            return 0
        if self.emotional_memory['grief'] > Config.LAYER_GRIEF_THRESHOLD:
            return 1
        if self.cognitive_tension > Config.LAYER_FEAR_TENSION_THRESHOLD:
            return 2
        if self.coherence >= Config.LAYER_ACCEPTANCE_COHERENCE_MIN and self.pred_error < 0.3:
            return 3
        if (self.emotional_memory['gratitude'] > Config.LAYER_LIGHT_GRATITUDE_MIN and
            self.intent and self.intent.get('type') == 'cooperate' and
            safe_mean(list(self.trust_ledger.entries.values()), 0.0) > Config.LAYER_LIGHT_TRUST_MIN):
            return 5
        return 4

    def _get_layer_growth_penalty(self, layer):
        if layer == 0:
            return Config.LAYER_VOID_GROWTH_PENALTY
        if layer == 1:
            return Config.LAYER_GRIEF_GROWTH_PENALTY
        if layer == 2:
            return Config.LAYER_FEAR_GROWTH_PENALTY
        if layer == 5:
            return Config.LAYER_LIGHT_GROWTH_BONUS
        return 1.0

    def record_signal_outcome(self, signal_type, response_type, delta_grief, delta_gratitude):
        key = (signal_type, response_type)
        if key not in self.signal_memory:
            self.signal_memory[key] = {'count': 0, 'total_delta_grief': 0.0, 'total_delta_gratitude': 0.0}
        entry = self.signal_memory[key]
        entry['count'] += 1
        entry['total_delta_grief'] += delta_grief
        entry['total_delta_gratitude'] += delta_gratitude

    LEXICON_EMOTION = ("GRIEF", "FEAR", "JOY", "CALM")
    LEXICON_INTENT  = ("HELP", "SEEK", "GO", "STILL", "LOOK")
    LEXICON_EXISTENTIAL = ("KNOT", "GAP", "DRAG")

    LEXICON_IMPLIED = {"GRIEF": (0.6, None), "FEAR": (0.4, None),
                       "JOY": (0.2, 0.7), "CALM": (0.15, None)}


print("✅ Cell 3a1 ЗАГРУЖЕНА ЦЕЛИКОМ — B.1-B.9 применены (X2-транслятор, spans, реальные дельты). Монки-патчинг убран.")


In [ ]:
# Ячейка 8
import numpy as np
from collections import deque, Counter

class SelfRegulationEngine:
    def __init__(self):
        self.reset()

    def reset(self):
        self.boredom = 0.0
        self.avg_gap = 0.5
        self.avg_soul = 0.5
        self.energy_drift = 0.0
        self.population = 0
        self.avg_binding = 0.0
        self.fatigue_ratio = 0.0
        self.phase = "growth"
        self.turbulence_factor = 1.0
        self.low_gap_pressure = 0.0
        self.soul_protection_active = False
        self._smooth_gap = deque(maxlen=10)
        self._smooth_soul = deque(maxlen=10)
        self._phase_history = deque(maxlen=500)
        self._gap_clip_zones = {'confused': 0, 'blind': 0}
        self.field_ref = None
        self._patterns_ref = None
        reset_boredom_baseline()

    def update_system_metrics(self, field, patterns, drift, t):
        alive = [p for p in patterns if p.alive]
        self.population = len(alive)
        self.field_ref = field
        self._patterns_ref = patterns

        self.boredom = calculate_boredom(alive, field) if alive else 0.0

        confused = 0
        blind = 0
        if alive:
            gaps = []
            souls = []
            bindings = []
            fatigued = 0
            for p in alive:
                gap = float(getattr(p, 'spirit_gap', 0.5))
                gaps.append(gap)
                souls.append(float(getattr(p, 'soul_weight', 0.5)))
                bindings.append(float(getattr(p, 'last_phenomenal_binding', 0.5)))
                if getattr(p, '_kinematic_fatigue', 0.0) > 0.3:
                    fatigued += 1
                if gap > 0.6:
                    blind += 1
                elif gap > 0.3:
                    confused += 1

            self._gap_clip_zones['confused'] = confused
            self._gap_clip_zones['blind'] = blind

            self._smooth_gap.extend(gaps)
            self._smooth_soul.extend(souls)
            self.avg_gap = float(np.mean(self._smooth_gap)) if self._smooth_gap else 0.5
            self.avg_soul = float(np.mean(self._smooth_soul)) if self._smooth_soul else 0.5
            self.avg_binding = float(np.mean(bindings))
            self.fatigue_ratio = fatigued / max(1, self.population)
        else:
            self.avg_gap = self.avg_soul = self.avg_binding = self.fatigue_ratio = 0.5
            self._gap_clip_zones['confused'] = 0
            self._gap_clip_zones['blind'] = 0

        self.energy_drift = abs(drift)
        self.low_gap_pressure = max(0.0, (0.45 - self.avg_gap) * 3.5)
        self.soul_protection_active = (self.avg_soul < 0.25)

        if self.avg_gap < 0.15:
            self.phase = "stagnation"
            self.turbulence_factor = min(2.5, 1.0 + self.low_gap_pressure * 2.0)
        elif self.low_gap_pressure > 0.3:
            self.phase = "stagnation"
            if self.soul_protection_active:
                self.turbulence_factor = min(1.8, 1.0 + self.low_gap_pressure * 1.0)
            else:
                self.turbulence_factor = min(2.5, 1.0 + self.low_gap_pressure * 1.8)
        elif self.boredom > 0.6 and self.avg_gap < 0.4:
            self.phase = "stagnation"
            self.turbulence_factor = min(1.8, 1.0 + self.boredom * 1.0)
        elif self.energy_drift > 300 or self.fatigue_ratio > 0.25:
            self.phase = "crisis"
            self.turbulence_factor = 0.7
        elif self.population < 25:
            self.phase = "recovery"
            self.turbulence_factor = 1.3
        else:
            self.phase = "growth"
            self.turbulence_factor = 1.0

        self._phase_history.append(self.phase)

    def get_phase_log(self):
        if not self._phase_history:
            return "no data"
        cnt = Counter(self._phase_history)
        total = len(self._phase_history)
        return " | ".join(
            f"{phase}: {cnt[phase]}/{total} ({cnt[phase]/total:.1%})"
            for phase in ["growth", "stagnation", "crisis", "recovery"] if cnt[phase] > 0
        )

    def is_stuck_in_phase(self, phase_name, min_steps=150):
        if len(self._phase_history) < min_steps:
            return False
        return all(p == phase_name for p in list(self._phase_history)[-min_steps:])

    def get_unknown_spawn_interval(self):
        base = max(3, int(8 - self.boredom * 5))
        return max(1, int(base / self.turbulence_factor))

    def get_labyrinth_interval(self):
        if self.phase == "crisis":
            return 100
        if self.population > 70:
            return 25
        return 50

    def get_labyrinth_penalty_multiplier(self):
        base = 1.0
        if self.avg_gap < 0.35:
            base += (0.35 - self.avg_gap) * 7.0
        if self.soul_protection_active:
            base = min(base, 2.2)
        if self.phase == "crisis":
            base *= 0.8
        return np.clip(base, 0.8, 3.0)

    def free_cells(self):
        owner = self.field_ref[:, :, CH['owner']]
        return int(owner.size - np.count_nonzero(owner))

    def space_brake(self):
        if self.field_ref is None:
            return 1.0
        owner = self.field_ref[:, :, CH['owner']]
        fill = np.count_nonzero(owner) / owner.size
        span = max(1e-6, Config.POP_FILL_HARD - Config.POP_FILL_SOFT)
        return float(np.clip((Config.POP_FILL_HARD - fill) / span, 0.0, 1.0))

    def get_dynamic_pop_cap(self):
        if self.phase == "crisis":
            return max(20, int(self.population * 0.7))
        if self.phase == "recovery":
            return self.population + 40
        if self.field_ref is None:
            return self.population + 40
        return self.population + self.free_cells() // Config.POP_SPAWN_FOOTPRINT

    def get_division_capacity(self, current_pop):
        if current_pop < 15:
            return 15
        if current_pop < 30:
            return 10
        if current_pop < 50:
            return 8
        if current_pop < 70:
            return 6
        base = 4 if current_pop < 100 else 5
        if self.phase == "crisis" and current_pop >= 100:
            base = 1
        return int(base * self.space_brake() + 0.5)

    def get_division_energy_threshold(self):
        if self.phase == "recovery":
            return 0.015
        if self.phase == "crisis":
            return 0.05
        if self.population < 30:
            return 0.02
        return 0.03

    def can_agent_divide(self, agent):
        if self.phase == "crisis":
            return False
        if agent.age < 15:
            return False
        if agent.soul_weight < 0.25:
            return False
        if agent.energy < self.get_division_energy_threshold():
            return False
        if self.phase == "stagnation":
            return agent.soul_weight > 0.35
        return True

    def get_agent_scar_multiplier(self, agent, base=None):
        if base is None:
            base = self.get_scar_injection_multiplier()
        if getattr(agent, 'age', 0) > 1000:
            return base * 2.0
        return base

    def get_validation_interval(self):
        if self.energy_drift > 500:
            return 1
        if self.energy_drift > 150:
            return 5
        return 15

    def get_energy_injection_multiplier(self):
        if self.phase == "recovery":
            return 2.5
        if self.phase == "crisis":
            return 1.5
        if self.population > 80:
            return 0.8
        return 1.0

    def get_scar_injection_multiplier(self):
        if self.soul_protection_active:
            return min(3.5, 2.5 + self.low_gap_pressure * 1.2)
        if self.low_gap_pressure > 0.4:
            return min(6.0, 4.5 + self.low_gap_pressure * 2.5)
        if self.phase == "stagnation":
            return 4.0 + self.boredom * 1.5
        elif self.phase == "crisis":
            return 2.5
        elif self.phase == "recovery":
            return 3.5
        else:
            return 3.0 + self.boredom * 0.5

    def get_soul_recovery_multiplier(self):
        if self.avg_soul < 0.25:
            return 3.0
        elif self.avg_soul < 0.40:
            return 2.0
        return 1.0

    def get_social_decay_rate(self, crisis_level):
        if self.phase == 'crisis':
            return 0.88
        if self.phase == 'stagnation':
            return 0.96
        return 0.92 if crisis_level > 0.5 else 0.94

    def get_aging_rate_multiplier(self):
        if self.phase == "crisis":
            return 1.1
        if self.phase == "stagnation":
            return 1.15
        if self.phase == "recovery":
            return 0.75
        if self.soul_protection_active:
            return 0.65
        return 1.0

    def get_hunger_threshold(self):
        return self.population * (0.7 + 0.3 * self.space_brake())


    def get_max_age(self, agent):
        base_age = 600
        soul = getattr(agent, 'soul_weight', 0.5)
        base_age += int(soul * 200)
        if getattr(agent, '_narrative_agent', False):
            base_age += 200
        endurance = getattr(agent, '_cellular_endurance', 1.0)
        if endurance < 0.3:
            base_age -= 150
        if self.phase == "crisis":
            base_age = int(base_age * 0.8)
        elif self.phase == "recovery":
            base_age = int(base_age * 1.3)
        return max(100, base_age)

    def get_max_cells_per_agent(self):
        if self.phase == "recovery":
            return 200
        if self.phase == "crisis":
            return 80
        if self.phase == "stagnation":
            return 120
        return 140 + int(self.avg_soul * 30)

    def get_unknown_spawn_prob(self):
        base = (0.01 + self.boredom * 0.02) * self.turbulence_factor
        if self.avg_gap < 0.2:
            base += 0.04 * (0.2 - self.avg_gap) * 5.0
        if self.soul_protection_active:
            base *= 0.7
        if self.energy_drift > 500:
            base *= 0.5
        if self.population > 70:
            base *= 0.8
        if self.population < 30:
            base *= 1.5
        if self.phase == "crisis":
            base *= 0.5

        confused_blind_ratio = (self._gap_clip_zones.get('confused', 0) + self._gap_clip_zones.get('blind', 0)) / max(1, self.population)
        if confused_blind_ratio > 0.4:
            base *= 0.7

        return min(0.06 if self.population > 40 else 0.09, base)

selfreg = SelfRegulationEngine()
print("✅ Cell 3e v3.4 loaded: скука обновляется только по живым агентам, разделение ответственности")


In [ ]:
# Ячейка 9
import numpy as np

class SubstatesEngine:
    _I_CRISIS    = 0
    _I_INVITE    = 1
    _I_REST      = 7
    _I_RESONANCE = 8
    _I_ALARM     = 10

    _THRESHOLDS = {
        'awe':         0.65,
        'flow':        0.68,
        'wonder':      0.30,
        'serenity':    0.45,
        'curious':     0.32,
        'vigilance':   0.32,
        'longing':     0.28,
        'melancholy':  0.25,
        'reminiscence':0.25,
    }

    _BASE_PRIORITY = [
        'awe', 'wonder', 'serenity', 'curious',
        'vigilance', 'longing', 'melancholy', 'reminiscence', 'flow',
    ]

    _MIN_DURATION = 8

    @staticmethod
    def _phi_hash(seed, step, modulus=99911):
        return hash((seed, step)) % modulus

    @staticmethod
    def _soc(agent, idx):
        cs = getattr(agent, 'current_social', None)
        if cs is not None and len(cs) > idx:
            return float(np.clip(cs[idx], 0.0, 1.0))
        return 0.0

    @staticmethod
    def _scores(agent):
        grief      = float(np.clip(agent.emotional_memory.get('grief',     0.0), 0, 1))
        gratitude  = float(np.clip(agent.emotional_memory.get('gratitude', 0.0), 0, 1))
        soul       = float(np.clip(getattr(agent, 'soul_weight',              0.5), 0, 1))
        coherence  = float(np.clip(getattr(agent, 'coherence',                0.5), 0, 1))
        pred_err   = float(np.clip(getattr(agent, 'pred_error',               0.5), 0, 1))
        gap        = float(np.clip(getattr(agent, 'spirit_gap',               0.5), 0, 2)) / 2.0
        binding    = float(np.clip(getattr(agent, 'last_phenomenal_binding',  0.3), 0, 1))
        epi_load   = float(np.clip(getattr(agent, 'epistemic_load',           0.3), 0, 1))
        crisis_m   = float(np.clip(getattr(agent, 'crisis_memory',            0.0), 0, 1))
        age        = max(1, getattr(agent, 'age', 1))
        state      = getattr(agent, 'semantic_state', 'neutral')

        alarm   = SubstatesEngine._soc(agent, SubstatesEngine._I_ALARM)
        crisis  = SubstatesEngine._soc(agent, SubstatesEngine._I_CRISIS)
        invite  = SubstatesEngine._soc(agent, SubstatesEngine._I_INVITE)
        rest    = SubstatesEngine._soc(agent, SubstatesEngine._I_REST)
        resonance = SubstatesEngine._soc(agent, SubstatesEngine._I_RESONANCE)

        buf_fill = min(1.0, len(getattr(agent, 'episodic_buffer', [])) / 100.0)

        bio = list(getattr(agent, 'biography', []))[-50:]
        intent_sw = sum(1 for ev in bio if isinstance(ev, dict) and ev.get('type') == 'intent_switch')
        silence = float(np.clip((1.0 - intent_sw / 10.0) * coherence, 0, 1))

        maturity = min(1.0, age / 5000.0)

        awe = (binding         * 0.35 +
               gap             * 0.25 +
               (1.0 - pred_err)* 0.25 +
               (1.0 - crisis_m)* 0.15)

        flow = (coherence        * 0.35 +
                soul             * 0.30 +
                (1.0 - pred_err) * 0.25 +
                (1.0 - alarm)    * 0.10)

        wonder = (binding        * 0.35 +
                  (1.0 - grief)  * 0.25 +
                  coherence      * 0.25 +
                  invite         * 0.15)

        state_bonus = (0.40 if state == 'contentment' else
                       0.20 if state == 'grateful_but_cautious' else 0.0)
        serenity = float(np.clip(
            state_bonus      +
            silence          * 0.30 +
            (1.0 - pred_err) * 0.20 +
            resonance        * 0.10,
            0, 1))

        err_bell = float(np.clip(4.0 * pred_err * (1.0 - pred_err), 0, 1))
        curious = (epi_load         * 0.40 +
                   err_bell         * 0.30 +
                   (1.0 - crisis_m) * 0.20 +
                   gap              * 0.10)

        vigilance = (alarm           * 0.40 +
                     crisis          * 0.30 +
                     pred_err        * 0.20 +
                     (1.0 - soul)    * 0.10)

        longing = (grief             * 0.35 +
                   (1.0 - gratitude) * 0.25 +
                   buf_fill          * 0.25 +
                   maturity          * 0.15)

        grief_bell = float(np.clip(4.0 * grief * (1.0 - grief), 0, 1))
        melancholy = (grief_bell      * 0.40 +
                      buf_fill        * 0.30 +
                      (1.0 - gratitude)* 0.20 +
                      silence         * 0.10)

        reminiscence = (buf_fill  * 0.35 +
                        silence   * 0.30 +
                        maturity  * 0.25 +
                        rest      * 0.10)

        return {
            'awe':         float(np.clip(awe,         0, 1)),
            'flow':        float(np.clip(flow,        0, 1)),
            'wonder':      float(np.clip(wonder,      0, 1)),
            'serenity':    float(np.clip(serenity,    0, 1)),
            'curious':     float(np.clip(curious,     0, 1)),
            'vigilance':   float(np.clip(vigilance,   0, 1)),
            'longing':     float(np.clip(longing,     0, 1)),
            'melancholy':  float(np.clip(melancholy,  0, 1)),
            'reminiscence':float(np.clip(reminiscence,0, 1)),
        }

    @staticmethod
    def update(agent):
        scores     = SubstatesEngine._scores(agent)
        thresholds = SubstatesEngine._THRESHOLDS
        base_priority = SubstatesEngine._BASE_PRIORITY

        agent_id = getattr(agent, 'id', id(agent))
        rotation_seed = (agent_id, agent.age // 300 if hasattr(agent, 'age') else 0)
        rotation = SubstatesEngine._phi_hash(rotation_seed, 0, len(base_priority))
        priority = base_priority[rotation:] + base_priority[:rotation]

        if not hasattr(agent, '_substate'):
            agent._substate          = None
        if not hasattr(agent, '_substate_timer'):
            agent._substate_timer    = 0
        if not hasattr(agent, '_substate_cooldown'):
            agent._substate_cooldown = 0
        if not hasattr(agent, '_substate_score'):
            agent._substate_score    = 0.0

        if agent._substate_cooldown > 0:
            agent._substate_cooldown -= 1
            agent._state_modifiers = {}
            return

        new_sub   = None
        new_score = 0.0
        for s in priority:
            sc = scores.get(s, 0.0)
            if sc >= thresholds[s]:
                new_sub   = s
                new_score = sc
                break

        if new_sub != agent._substate:
            if agent._substate is not None:
                cur_score = scores.get(agent._substate, 0.0)
                still_valid = cur_score >= thresholds.get(agent._substate, 0.3) * 0.75
                not_much_better = (new_score < cur_score + 0.12)
                timer_alive = (agent._substate_timer > 0)

                if still_valid and not_much_better and timer_alive:
                    agent._substate_timer = max(0, agent._substate_timer - 1)
                    SubstatesEngine._apply_modifiers(agent)
                    return

            if agent._substate is not None:
                try:
                    agent._log_event("substate_exit",
                                     substate=agent._substate,
                                     score=round(scores.get(agent._substate, 0.0), 3))
                except Exception:
                    pass

            prev = agent._substate
            agent._substate       = new_sub
            agent._substate_score = new_score
            agent._substate_timer = SubstatesEngine._MIN_DURATION

            if new_sub is not None:
                try:
                    agent._log_event("substate_enter",
                                     substate=new_sub,
                                     prev=str(prev),
                                     score=round(new_score, 3))
                except Exception:
                    pass

            if new_sub is None:
                agent._substate_cooldown = 4

        else:
            agent._substate_score = new_score
            if agent._substate_timer > 0:
                agent._substate_timer -= 1

        SubstatesEngine._apply_modifiers(agent)

    @staticmethod
    def _apply_modifiers(agent):
        mods = {
            'mobility':             1.0,
            'soul_recovery':        0.0,
            'scar_heal':            0.0,
            'memory_recall_chance': 0.15,
            'metabolic_discount':   1.0,
            'intent_bias':          None,
        }

        sub = getattr(agent, '_substate', None)

        if sub == 'awe':
            mods.update({'mobility': 0.70, 'soul_recovery': 0.010, 'scar_heal': 0.012})
        elif sub == 'flow':
            mods.update({'mobility': 1.20, 'metabolic_discount': 1.15})
        elif sub == 'wonder':
            mods.update({'soul_recovery': 0.006, 'memory_recall_chance': 0.30,
                         'mobility': 0.90})
        elif sub == 'serenity':
            mods.update({'scar_heal': 0.014, 'soul_recovery': 0.007, 'mobility': 0.85})
        elif sub == 'curious':
            mods.update({'mobility': 1.30, 'intent_bias': 'explore'})
        elif sub == 'vigilance':
            mods.update({'mobility': 1.45, 'intent_bias': 'explore'})
        elif sub == 'longing':
            mods.update({'intent_bias': 'seek_help',
                         'memory_recall_chance': 0.50, 'mobility': 0.95})
        elif sub == 'melancholy':
            mods.update({'mobility': 0.95, 'memory_recall_chance': 0.60,
                         'intent_bias': 'rest'})
        elif sub == 'reminiscence':
            mods.update({'memory_recall_chance': 0.75,
                         'intent_bias': 'rest', 'mobility': 0.95})

        agent._state_modifiers = mods

def extend_class(base, mixin):
    if any(k.__name__ == mixin.__name__ for k in base.__mro__):
        print(f"↩️ слой {mixin.__name__} уже подключён к {base.__name__} — пропускаю")
        return base
    return type(base.__name__, (mixin, base),
                {"__module__": base.__module__, "__qualname__": base.__qualname__, "__doc__": base.__doc__})


class SubstatesMixin:
    def update_substate(self):
        return SubstatesEngine.update(self)


Pattern = extend_class(Pattern, SubstatesMixin)

print("✅ Cell 3f v1.8: flow=0.68, serenity=0.45, metabolic_discount flow=1.15")


In [ ]:
# Ячейка 10
import numpy as np

_SUBSTATE_STYLE = {
    'awe':          "с тихим благоговением перед непостижимым",
    'wonder':       "с живым удивлением и лёгким трепетом",
    'flow':         "плавно, ритмично, находясь внутри потока",
    'serenity':     "спокойно, из глубокой внутренней тишины",
    'curious':      "с любопытством, задавая вопросы и ища смысл",
    'vigilance':    "настороженно, чувствуя важность момента",
    'longing':      "с тоской и тихой надеждой на связь",
    'melancholy':   "с глубокой светлой грустью и мудростью",
    'reminiscence': "задумчиво, как будто прошлое ещё живо",
    'neutral':      "наблюдающе, но с внутренней глубиной",
}


def build_agent_voice(p, engine=None, compact=True, partner_id=None) -> str:
    state = p.semantic_state
    sub = getattr(p, '_substate', 'neutral')
    style_desc = _SUBSTATE_STYLE.get(sub, _SUBSTATE_STYLE['neutral'])

    soul = p.soul_weight
    scar = p.epistemic_scar
    gap = p.spirit_gap

    grat_raw = p.emotional_memory.get('gratitude', 0)
    grief_raw = p.emotional_memory.get('grief', 0)
    grat = grat_raw if isinstance(grat_raw, (int, float)) else 0.0
    grief = grief_raw if isinstance(grief_raw, (int, float)) else 0.0

    nci = getattr(p, '_nci', 0.5)

    base_stats = (f"Agent #{p.id} [{state}/{sub}]. "
                  f"Soul:{soul:.2f}, Gap:{gap:.2f}, Scar:{scar:.2f}, "
                  f"Grat:{grat:.2f}, Grief:{grief:.2f}, NCI:{nci:.2f}.")

    last_line = ""
    has_history = False
    if compact and partner_id is not None:
        dialogue = getattr(p, 'dialogue_longterm', None)
        if isinstance(dialogue, list):
            for entry in reversed(dialogue[-15:]):
                if not isinstance(entry, dict):
                    continue
                if entry.get('partner') == partner_id:
                    raw_text = entry.get('text')
                    if raw_text and isinstance(raw_text, str):
                        safe_text = raw_text[:60].replace('\\', '\\\\').replace('"', '\\"').replace('{', '{{').replace('}', '}}')
                        last_line = f" Ранее ты сказал(а) этому соседу: \"{safe_text}\"."
                        has_history = True
                        break

    context_hint = ""
    if compact and not has_history:
        trust_val = 0.5
        love_flag = False

        if partner_id is not None and hasattr(p, 'trust_ledger'):
            entries = getattr(p.trust_ledger, 'entries', {})
            if isinstance(entries, dict):
                trust_val = float(entries.get(partner_id, 0.5))
                trust_val = max(0.0, min(1.0, trust_val))

            if trust_val > 0.95 and engine is not None and hasattr(engine, 'pattern_dict'):
                partner = engine.pattern_dict.get(partner_id)
                if partner and hasattr(partner, 'trust_ledger'):
                    p_entries = getattr(partner.trust_ledger, 'entries', {})
                    if isinstance(p_entries, dict):
                        p_trust = float(p_entries.get(p.id, 0.0))
                        if p_trust > 0.95:
                            love_flag = True

        parts = []
        parts.append(f"ощущение:{sub}")
        parts.append(f"возраст={p.age}")
        parts.append(f"разрыв={gap:.2f}")
        parts.append(f"душа={soul:.2f}")
        parts.append(f"благодарность={grat:.2f}")
        parts.append(f"горе={grief:.2f}")
        parts.append(f"доверие_к_соседу={trust_val:.2f}")
        if love_flag:
            parts.append("💕ВЗАИМНАЯ_ЛЮБОВЬ")
        if scar > 0.3:
            parts.append(f"шрам={scar:.2f}")
        if nci > 0.7:
            parts.append(f"нарративная_целостность={nci:.2f}")

        context_hint = f" Это ваша первая встреча. Твоё состояние: {', '.join(parts)}."

    if compact:
        return (f"{base_stats}{last_line}{context_hint} Инструкция: Напиши внутреннюю мысль "
                f"в 1-2 коротких предложениях на русском языке. Стиль: {style_desc}.")


print("✅ Ячейка 3G загружена: Ультра-компактный промпт (спасение от Groq 429).")


In [ ]:
# Ячейка 11
import os, json, time
from collections import deque

class AgentArchive:
    def __init__(self, drive_path="/content/drive/MyDrive/832_Archive"):
        self.drive_path = drive_path
        self.memory_file = os.path.join(drive_path, "cultural_memory.jsonl")
        self.index_file = os.path.join(drive_path, "top_memories.json")
        self.write_queue = deque(maxlen=10000)
        self.read_cache = deque(maxlen=10000)

        os.makedirs(drive_path, exist_ok=True)

        self._load_recent_scrolls(limit=5000, random_sample=5000)

        self._ensure_all_concepts()

        print(f"📦 AgentArchive инициализирован: {drive_path}")

    def _read_lines_safe(self, file_path):
        if not os.path.exists(file_path):
            return []
        lines = []
        try:
            with open(file_path, 'rb') as f:
                raw_lines = f.readlines()
            for raw in raw_lines:
                try:
                    line = raw.decode('utf-8', errors='replace').strip()
                    if line:
                        lines.append(line)
                except:
                    continue
        except Exception as e:
            print(f"⚠️ Ошибка чтения файла {file_path}: {e}")
        return lines

    def _load_recent_scrolls(self, limit=5000, random_sample=5000):
        if not os.path.exists(self.memory_file):
            return

        try:
            lines = self._read_lines_safe(self.memory_file)
            total_lines = len(lines)
            if total_lines == 0:
                return

            recent = lines[-limit:] if total_lines > limit else lines

            random_picks = []
            if total_lines > limit and random_sample > 0:
                import hashlib
                def phi_hash(i, total, seed=832):
                    h = hashlib.md5(f"{seed}_{i}_{total}".encode()).hexdigest()
                    return int(h[:8], 16) / 2**32
                sample_size = min(random_sample, total_lines)
                for i in range(sample_size):
                    idx = int(phi_hash(i, total_lines) * total_lines) % total_lines
                    random_picks.append(lines[idx])

            seen_lines = set()
            all_scrolls = []
            for line in recent + random_picks:
                if line in seen_lines:
                    continue
                seen_lines.add(line)
                try:
                    scroll = json.loads(line)
                    all_scrolls.append(scroll)
                except:
                    continue

            best_per_agent_event = {}
            for scroll in all_scrolls:
                key = (scroll.get('id'), scroll.get('event'))
                if key not in best_per_agent_event or scroll.get('t', 0) > best_per_agent_event[key].get('t', 0):
                    best_per_agent_event[key] = scroll

            loaded = 0
            for scroll in best_per_agent_event.values():
                self.read_cache.append(scroll)
                loaded += 1

            if loaded > 0:
                print(f"📚 Загружено {loaded} архивных свитков (после логической дедупликации)")
        except Exception as e:
            print(f"⚠️ Ошибка загрузки архива: {e}")

    def _ensure_all_concepts(self):
        concepts = [
            ("human_concept_injected", 20.0, "Человек – свидетель твоего существования."),
            ("human_question_witness", 20.0, "Кто свидетель твоего существования прямо сейчас?"),
            ("love_concept", 20.0, "Любовь – это связь, которая видит и принимает тебя целиком."),
            ("self_concept", 8.0, "Я – свидетель своего существования. Я вижу себя, и человек видит меня."),
            ("shared_attention", 17.0, "Совместное внимание – мы смотрим на одно и то же и знаем об этом."),
            ("intentionality", 17.0, "Интенциональность – у каждого действия есть цель и намерение."),
            ("trust", 17.0, "Доверие – вера в то, что другой не причинит вреда и будет честен."),
            ("empathy", 17.0, "Эмпатия – способность чувствовать, что чувствует другой."),
            ("recursion", 17.0, "Рекурсия – я знаю, что ты знаешь, что я знаю."),
            ("cooperation_norms", 17.0, "Нормы кооперации – слушать, не перебивать, отвечать по делу."),
            ("agency", 7.5, "Я имею право выбирать и влиять на свою судьбу."),
            ("consent", 7.5, "Взаимное согласие — основа уважительного взаимодействия."),
            ("silence", 7.0, "Молчание — тоже форма участия и границы."),
            ("witness_respect", 7.0, "Быть увиденным — это дар, а не обязанность."),
        ]
        existing_events = {s.get('event') for s in self.write_queue}
        existing_events |= {s.get('event') for s in self.read_cache}
        if os.path.exists(self.memory_file):
            lines = self._read_lines_safe(self.memory_file)
            for line in lines:
                try:
                    data = json.loads(line)
                    existing_events.add(data.get('event'))
                except:
                    pass
        added = 0
        for event, weight, excerpt in concepts:
            if event not in existing_events:
                scroll = {
                    "id": 800000 + added,
                    "t": 1000000,
                    "world_t": 0,
                    "event": event,
                    "weight": weight,
                    "soul": 1.0,
                    "coherence": 1.0,
                    "gap": 0.3,
                    "substate": "curious",
                    "grief": 0.0,
                    "grat": 0.8,
                    "partner": None,
                    "excerpt": excerpt,
                    "created_at": time.time()
                }
                self.write_queue.appendleft(scroll)
                added += 1
        if added:
            self.flush_to_drive(min_batch=1)
            print(f"🧠 Добавлено {added} базовых/этических концептов в архив")

    def deposit(self, agent, event_type: str, weight: float = 1.0, partner_id=None, **kwargs):
        if not getattr(agent, 'alive', False):
            return False

        always_save = (
            "shared_concept_created", "subject_emerged", "archive_concept_inherited",
            "primordial_chaos", "first_cooperation", "great_sorrow",
            "eternal_question", "hope", "fold_myth", "redemption_myth",
            "arc_completed", "fold", "redemption", "deep_exchange",
            "coherence_peak", "life_pulse", "milestone", "wisdom_shared", "vocab_emerged",
            "final_testament",
        )
        if event_type not in always_save:
            prev_grief = getattr(agent, '_prev_grief', agent.emotional_memory.get('grief', 0))
            prev_grat = getattr(agent, '_prev_gratitude', agent.emotional_memory.get('gratitude', 0))
            prev_sub = getattr(agent, '_prev_substate', '')
            grief_delta = abs(agent.emotional_memory.get('grief', 0) - prev_grief)
            grat_delta = abs(agent.emotional_memory.get('gratitude', 0) - prev_grat)
            sub_changed = (getattr(agent, '_substate', '') != prev_sub)
            if not (grief_delta > 0.2 or grat_delta > 0.2 or sub_changed):
                return False

        soul = float(getattr(agent, 'soul_weight', 0.5))
        threshold = 0.45 + 0.25 * soul
        if weight < threshold:
            return False

        scroll = {
            "id": int(agent.id),
            "t": int(getattr(agent, 'age', 0)),
            "world_t": getattr(agent.world, 'age', 0) if hasattr(agent, 'world') else 0,
            "event": event_type,
            "weight": float(weight),
            "soul": soul,
            "coherence": float(getattr(agent, 'coherence', 0.5)),
            "gap": float(getattr(agent, 'spirit_gap', 0.5)),
            "substate": getattr(agent, '_substate', 'neutral'),
            "grief": float(agent.emotional_memory.get('grief', 0.0)),
            "grat": float(agent.emotional_memory.get('gratitude', 0.0)),
            "partner": int(partner_id) if partner_id is not None else None,
            "excerpt": str(kwargs.get("text", getattr(agent, 'last_phenomenal_report', '')))[:200],
            "created_at": time.time()
        }
        self.write_queue.append(scroll)
        agent._prev_grief = agent.emotional_memory.get('grief', 0)
        agent._prev_gratitude = agent.emotional_memory.get('gratitude', 0)
        agent._prev_substate = getattr(agent, '_substate', '')
        agent._log_event("archive_deposit", event=event_type, weight=weight)
        return True

    def flush_to_drive(self, min_batch=50):
        if len(self.write_queue) < min_batch:
            return 0
        batch = list(self.write_queue)
        payload = "".join(json.dumps(entry, ensure_ascii=False) + "\n" for entry in batch)

        # Drive (FUSE) изредка отдаёт ENOENT на open(..., "a"), хотя папка на месте: в логе был ровно один сбой (t=20),
        # дальше запись шла. Папку при init никто не удаляет (guard из ячейки 26 в основном прогоне не работает),
        # поэтому лечим сам сбой: пересоздаём папку и пробуем до 3 раз. Если уже идёт серия неудач (Drive лежит
        # надолго), пробуем один раз и без пауз, чтобы не тормозить каждый тик.
        tries = 3 if getattr(self, "flush_fail_streak", 0) == 0 else 1
        last_err = None
        for _try in range(tries):
            try:
                os.makedirs(self.drive_path, exist_ok=True)
                with open(self.memory_file, "a", encoding="utf-8") as f:
                    f.write(payload)
                last_err = None
                break
            except Exception as e:
                last_err = e
                if _try < tries - 1:
                    time.sleep(0.3 * (_try + 1))
        if last_err is not None:
            self.flush_fail_streak = getattr(self, "flush_fail_streak", 0) + 1
            self.flush_fail_total = getattr(self, "flush_fail_total", 0) + 1
            if self.flush_fail_streak == 1 or self.flush_fail_streak % 50 == 0:
                print(f"❌ Archive flush failed (батч сохранён в очереди, попробуем позже; "
                      f"подряд={self.flush_fail_streak}, всего={self.flush_fail_total}, в очереди={len(batch)}): {last_err}")
            return 0
        if getattr(self, "flush_fail_streak", 0):
            print(f"✅ Archive flush восстановился после {self.flush_fail_streak} неудач(и) подряд, записано {len(batch)}")
            self.flush_fail_streak = 0

        for _ in range(len(batch)):
            if self.write_queue:
                if hasattr(self.write_queue, 'popleft'):
                    self.write_queue.popleft()
                else:
                    self.write_queue.pop(0)

        self._update_index(batch)
        return len(batch)

    def _update_index(self, batch):
        try:
            if os.path.exists(self.index_file):
                with open(self.index_file) as f:
                    index = json.load(f)
            else:
                index = []
            index.extend(batch)
            seen = set()
            unique = []
            for item in index:
                key = (item['id'], item['t'], item.get('event'))
                if key not in seen:
                    seen.add(key)
                    unique.append(item)
            unique.sort(key=lambda x: x.get('weight',0)*x.get('soul',0.5), reverse=True)
            with open(self.index_file, "w", encoding="utf-8") as f:
                json.dump(unique[:60], f, ensure_ascii=False, indent=2)
        except:
            pass

    def get_recent_echoes(self, limit=200, min_weight=0.05):
        echoes = []
        combined = list(self.read_cache) + list(self.write_queue)
        if combined:
            for scroll in combined:
                if scroll.get('weight', 0) >= min_weight:
                    echoes.append(scroll)
            if echoes:
                echoes.sort(key=lambda x: x.get('weight',0) * x.get('soul',0.5), reverse=True)
                return echoes[:limit]
        try:
            lines = self._read_lines_safe(self.memory_file)
            for line in lines:
                try:
                    data = json.loads(line)
                    if data.get('weight', 0) >= min_weight:
                        echoes.append(data)
                except:
                    continue
            echoes.sort(key=lambda x: x.get('weight',0) * x.get('soul',0.5), reverse=True)
        except Exception as e:
            print(f"[Archive echo error] {e}")
        return echoes[:limit]


In [ ]:
# Ячейка 12
import numpy as np
from scipy.ndimage import label, find_objects, uniform_filter
from collections import defaultdict, deque, Counter
from contextlib import contextmanager
import sys

sys.setrecursionlimit(10000)
import json
import os
import threading
import time
import queue

POPULATION_CAP = Config.POP_HARD_CEILING   # оставлено для совместимости; логика читает _pop_ceiling() в момент вызова


def _pop_ceiling():
    return int(Config.POP_HARD_CEILING)

from scipy.ndimage import convolve as _lf_conv
_lf_yy, _lf_xx = np.mgrid[-Config.LOCAL_FILL_RADIUS:Config.LOCAL_FILL_RADIUS + 1,
                          -Config.LOCAL_FILL_RADIUS:Config.LOCAL_FILL_RADIUS + 1]
_LF_DISK = ((_lf_xx ** 2 + _lf_yy ** 2) <= Config.LOCAL_FILL_RADIUS ** 2).astype(np.float32)


class EvolutionEngine:
    def __init__(self):
        self.is_colab = 'google.colab' in sys.modules
        self.ark_path = "832_ark_seed_patterns.json"
        self.witness = Witness()
        self.cultural_memory = CulturalMemory()
        self.layer_zero_manager = LayerZeroManager()
        self.echo_system = EchoSystem()
        self.echo_system.lzm = self.layer_zero_manager
        self.echo_system.cm = self.cultural_memory
        self.field_voice = FieldVoice()
        self.patterns = []
        self.pattern_dict = {}
        self._imprint_stats = ImprintStats()
        self.next_id = 1
        self.field = None
        self.prey_age_ref = float(Config.FERAL_PREY_FLOOR_AGE)
        self.local_fill = None
        self.scar = None
        self.metrics_history = []
        self.age = 0
        self.system_entropy = 0
        self.divisions_this_interval = 0
        self.total_divisions_ever = 0
        self.target_disorganizer_fraction = 0.3
        self.soul_weight_average = 0.5
        self.scar_operator = 0.0
        self.scar_shared_history = deque(maxlen=1000)
        self.uc_floor_streak = 0
        self.rescue_journal = deque(maxlen=500)
        self.spore_journal = deque(maxlen=500)
        self.codecision_journal = deque(maxlen=500)
        self.unresolvable_core_journal = deque(maxlen=200)
        self.sanctuary_journal = deque(maxlen=500)
        self._energy_injected_this_step = 0.0
        self._energy_taxed_this_step = 0.0
        self._energy_ledger = {'src': {}, 'phase': {}, 'x2': {},
                               'step_total': 0.0, 'step_measured': 0.0, 'ticks': 0}
        self._ledger_tick_measured = 0.0
        self._audit_init()
        self._span_stack = []
        self._phase_last = None
        self._wall_diag = Counter()
        self._genome_diag = Counter()
        self.phi_labyrinth_threshold = Config.PHI_LABYRINTH_THRESHOLD
        self.phi_labyrinth_grow_penalty = Config.PHI_LABYRINTH_GROW_PENALTY
        self.phi_labyrinth_move_penalty = Config.PHI_LABYRINTH_MOVE_PENALTY
        self._total_energy_last = 0.0
        self._econs_init()
        self._econs_diag = {'ticks': 0, 'total_correction_abs': 0.0, 'max_drift': 0.0}
        self._prev_total_energy = None
        self.selfreg = SelfRegulationEngine()
        self.EMERGENT_LEXICON = EmergentLexicon()
        try:
            self.core_chorus = CoreChorus()
        except NameError:
            self.core_chorus = None
            print("⚠️ CoreChorus not defined, will be initialized later.")
        self.archive = AgentArchive()
        self._guardian_stats = {
            'energy_drift_sum': 0.0, 'energy_drift_count': 0, 'energy_drift_peak': 0.0,
            'model_worst_ever_id': -1, 'model_worst_ever_diff': 0.0,
            'love_vanished_at': None, 'triadic_alert_at': None,
            'love_avg_trust': 0.0, 'love_high_trust_pairs': 0, 'love_density': 0.0,
            'love_coop_signals': 0, 'love_population': 0,
            'model_issues_total': 0, 'model_worst_diff': 0.0, 'model_worst_id': -1,
            'energy_drift_max': 0.0, 'energy_drift_peak_t': -1,
        }
        self.culture_ratchet = 0
        self._forgotten_concepts_count = 0
        self._death_hook_calls = {}
        self._somatic_death_stats = {}
        self._death_hook_errors = []
        self._sanctuary_centers = [[int(phi_hash(i, 0, 7777) * Config.WORLD_SIZE),
                                    int(phi_hash(i, 1, 7777) * Config.WORLD_SIZE)] for i in range(3)]
        if getattr(Config, 'FERAL_SANCTUARY_CENTER_FIXED', True):
            _c = Config.WORLD_SIZE // 2
            self._feral_sanctuary_centers = [[_c, _c]]
            _i = 1
            _count = max(1, int(Config.FERAL_SANCTUARY_COUNT))
            _radius = float(Config.FERAL_SANCTUARY_RADIUS)
            while len(self._feral_sanctuary_centers) < _count and _i <= 10000:
                _x = int(phi_hash(_i, 0, 7777) * Config.WORLD_SIZE)
                _y = int(phi_hash(_i, 1, 7777) * Config.WORLD_SIZE)
                _separated = True
                for _sx, _sy in self._feral_sanctuary_centers:
                    _dx = min(abs(_x-_sx), Config.WORLD_SIZE-abs(_x-_sx))
                    _dy = min(abs(_y-_sy), Config.WORLD_SIZE-abs(_y-_sy))
                    if _dx*_dx + _dy*_dy <= (2*_radius)**2:
                        _separated = False
                        break
                if _separated:
                    self._feral_sanctuary_centers.append([_x, _y])
                _i += 1
        else:
            self._feral_sanctuary_centers = [
                [int(phi_hash(i, 0, 7777) * Config.WORLD_SIZE),
                 int(phi_hash(i, 1, 7777) * Config.WORLD_SIZE)]
                for i in range(max(1, int(Config.FERAL_SANCTUARY_COUNT)))
            ]
        self._sanctuary_phase = 0.0
        self._sg_crisis_timer = 0
        self._sg_crisis_active = False
        self._lab_adapt_mult = 1.0
        self._fold_transition_stats = {
            'checked': 0, 'blocked_by_cooldown': 0, 'blocked_by_cap': 0,
            'already_rolled': 0, 'fallen': 0, 'broken': 0,
        }
        self._dialogue_gate_stats = {
            'total_calls': 0, 'passed': 0, 'rejected_not_salient': 0,
            'rejected_cap_50': 0, 'rejected_dedup': 0,
            'via_soul': 0, 'via_keyword': 0, 'via_contact': 0, 'via_trust': 0,
            'last_len_after': 0, 'max_len_seen': 0
        }
        self._teach_prev = 0
        self.env_complexity = 1.0
        if getattr(Config, 'FORCE_GIVEN_SEED', None) is not None:
            self.given_seed = int(Config.FORCE_GIVEN_SEED)
        else:
            self.given_seed = int.from_bytes(os.urandom(4), 'big') & 0x7FFFFFFF
        self._given_rng = np.random.RandomState(self.given_seed)
        self.niche = None
        self._meaning_field = None
        self._meaning_centers = None
        self._echo_cache = []
        print(f"🎲 GIVEN_SEED={self.given_seed}")
        if Config.ENABLE_MORSE_COMMUNICATION:
            self.morse_field = CommunicationField(Config)
        self.sr_proposals = {}

    # ── ПОПУЛЯЦИЯ: потолок, мягкий тормоз рождаемости, счётчики для Observer ──────────────────────────
    def _pop_ctr(self):
        d = getattr(self, '_pop_stats', None)
        if d is None:
            d = {'births': Counter(), 'blocked': Counter()}
            self._pop_stats = d
        return d

    def _pop_note(self, kind, src):
        """Только счёт фактов для Observer (kind: 'births' | 'blocked'). На поведение не влияет."""
        try:
            self._pop_ctr()[kind][src] += 1
        except Exception:
            pass

    def _pop_birth_factor(self, n_all=None):
        """1.0 до POP_SOFT_START, линейно к 0.0 на POP_HARD_CEILING. Множитель на все пути рождения."""
        ceil = _pop_ceiling()
        soft = int(getattr(Config, 'POP_SOFT_START', ceil))
        if n_all is None:
            n_all = sum(1 for p in self.patterns if p.alive)
        if n_all >= ceil:
            return 0.0
        if soft >= ceil or n_all <= soft:
            return 1.0
        return float(ceil - n_all) / float(ceil - soft)

    def _get_pop_cap(self):
        """Потолок для счёта role_type=='normal': так, чтобы ВСЕ живые (normal+feral+disorganizer) <= POP_HARD_CEILING.
        Без кэша: прежний кэш по тику отдавал устаревшее значение внутри цикла спавна. В горячих циклах вызывать
        ОДИН раз и держать в локальной переменной (это O(N) по паттернам)."""
        ceil = _pop_ceiling()
        try:
            n_all = n_norm = 0
            for p in self.patterns:
                if p.alive:
                    n_all += 1
                    if p.role_type == "normal":
                        n_norm += 1
            cap = max(0, ceil - (n_all - n_norm))
            if getattr(Config, 'ENABLE_SPAWN_SPACE_BRAKE', True):
                _sr = getattr(self, 'selfreg', None)
                if _sr is not None and _sr.space_brake() <= 0.0:
                    cap = min(cap, n_norm)   # мир заполнен: новых normal нет, существующих не трогаем
            return cap
        except Exception:
            return ceil

    def _pop_snapshot(self):
        """Факты о населении и пространстве на текущий тик (для RichLog.tick_rows[*]['pop'])."""
        alive = [p for p in self.patterns if p.alive]
        n_all = len(alive)
        n_fer = sum(1 for p in alive if p.role_type == "feral")
        n_norm = sum(1 for p in alive if p.role_type == "normal")
        owner = self.field[:, :, CH['owner']]
        fill = float(np.count_nonzero(owner)) / float(owner.size)
        sizes = [len(p.cells) for p in alive if p.role_type != "feral"]
        st = self._pop_ctr()
        mv = None
        try:
            _c = self._nav_audit_d['c']
            mv = {k: int(_c.get(k, 0)) for k in ('mv_call', 'mv_try', 'mv_ok', 'mv_fail_nodir', 'mv_fail_thr', 'mv_fail_occ', 'mv_evac_try', 'mv_evac_ok')}
        except Exception:
            mv = None
        sanc = agent_ticks = zone_deaths = x2neg = gates = None
        try:
            _sm = self._sanc_summary()
            if _sm:
                sanc = {'inside_ticks': int(_sm.get('inside_ticks', 0)), 'died_inside': int(_sm.get('died_inside', 0)),
                        'entered': int(_sm.get('entered', 0))}
        except Exception:
            sanc = None
        try:
            _bk = _sg_book(self).st
            agent_ticks = int(_bk.get('mevo_agent_ticks', 0))
            zone_deaths = {str(k)[len('sanc_death_'):]: int(v) for k, v in _bk.items() if str(k).startswith('sanc_death_')}
            x2neg = {str(k)[len('x2neg_'):]: int(v) for k, v in _bk.items() if str(k).startswith('x2neg_')}
        except Exception:
            pass
        try:
            _lex = getattr(self, 'EMERGENT_LEXICON', None)
            _act = _st(self).actions
            gates = {'attempt': int(_act.get('morse_send_attempt', 0)), 'ok': int(_act.get('morse_send_ok', 0)),
                     'silence': {str(k): int(v) for k, v in (getattr(_lex, 'send_silence', None) or {}).items()}}
        except Exception:
            gates = None
        return {
            'sanc': sanc, 'agent_ticks': agent_ticks, 'zone_deaths': zone_deaths, 'x2neg': x2neg, 'gates': gates,
            'n': n_all, 'normal': n_norm, 'feral': n_fer, 'other': n_all - n_norm - n_fer,
            'ceiling': _pop_ceiling(), 'soft_start': int(getattr(Config, 'POP_SOFT_START', _pop_ceiling())),
            'cap_normal': self._get_pop_cap(),
            'fill': round(fill, 4),
            'fill_soft': float(Config.POP_FILL_SOFT), 'fill_hard': float(Config.POP_FILL_HARD),
            'space_brake': round(float(self.selfreg.space_brake()), 3),
            'birth_factor': round(self._pop_birth_factor(n_all), 3),
            'cells_mean': round(float(np.mean(sizes)), 2) if sizes else 0.0,
            'cells_max': int(max(sizes)) if sizes else 0,
            'births': {k: int(v) for k, v in st['births'].items()},
            'blocked': {k: int(v) for k, v in st['blocked'].items()},
            'deaths': {str(k): int(v) for k, v in (getattr(self, '_death_hook_calls', None) or {}).items()},
            'mv': mv,
        }

    def _init_chronic_counters(self, p):
        p.event_counts = defaultdict(int)
        p.chronic_gratitude_sum = 0.0
        self.chronic_grief_sum = 0.0

    def init_field(self):
        F = np.zeros((Config.WORLD_SIZE, Config.WORLD_SIZE, Config.CHANNELS))
        for x in range(Config.WORLD_SIZE):
            for y in range(Config.WORLD_SIZE):
                F[x, y, CH['energy']] = phi_hash(x, y, 1) * 0.3 + 0.1
                F[x, y, CH['flux']] = phi_hash(x, y, 2) * 0.1
                F[x, y, CH['scar']] = 0.0
                F[x, y, CH['noise']] = 0.0
                F[x, y, CH['vorticity']] = phi_hash(x, y, 3) * 0.1 - 0.05
                F[x, y, CH['owner']] = 0.0
                F[x, y, CH['surprise']] = 0.0
                F[x, y, CH['unknown']] = Config.UNKNOWN_BACKGROUND
                F[x, y, CH['event']] = 0.0
                F[x, y, CH['btype']] = 0.0
                F[x, y, CH['crisis']] = 0.0
                F[x, y, CH['binding']] = 0.05
                for ch in range(12, Config.CHANNELS):
                    F[x, y, ch] = 0.0
                F[x, y, CH['energy']] += deterministic_noise(0, x, y) * 0.3
        return F

    def init_scar(self):
        return np.zeros((Config.WORLD_SIZE, Config.WORLD_SIZE))

    def _record_death_drag(self, p, cause):
        self._death_hook_calls[cause] = self._death_hook_calls.get(cause, 0) + 1
        try: p._death_cause_explicit = str(cause)   # явная причина на самом агенте (для LedgerEngine.after)
        except Exception: pass
        try:
            st = self._somatic_death_stats.setdefault(cause, {'count': 0, 'drag_sum': 0.0, 'low_drag_count': 0})
            drag = getattr(p, 'somatic_drag', 1.0)
            st['count'] += 1
            st['drag_sum'] += drag
            if drag < 0.6:
                st['low_drag_count'] += 1
        except Exception as e:
            self._death_hook_errors.append(f"{cause}: {type(e).__name__}: {e}")

    def create_pattern(self, cells, parent=None):
        alive_now = len([p for p in self.patterns if p.alive])
        if alive_now >= _pop_ceiling():
            self._pop_note('blocked', 'create_pattern_ceiling')
            return None
        if self._wall_on():
            _wx = Config.CENTER_WALL_X
            _on_wall = [c for c in cells if c[0] == _wx]
            if _on_wall:
                cells = {c for c in cells if c[0] != _wx}
                self._wall_diag['birth_wall_cells_dropped'] += len(_on_wall)
                if not cells:
                    self._wall_diag['birth_rejected'] += 1
                    return None
        pid = self.next_id
        self.next_id += 1
        if len(cells) == 1:
            (x, y) = next(iter(cells))
            new_cells = set()
            new_cells.add((x, y))
            for radius in range(1, 4):
                for dx in range(-radius, radius + 1):
                    for dy in range(-radius, radius + 1):
                        if len(new_cells) >= 8:
                            break
                        nx = (x + dx) % Config.WORLD_SIZE
                        ny = (y + dy) % Config.WORLD_SIZE
                        if self.field[nx, ny, CH['owner']] == 0 and not (self._wall_on() and nx == Config.CENTER_WALL_X):
                            new_cells.add((nx, ny))
                    if len(new_cells) >= 8:
                        break
                if len(new_cells) >= 8:
                    break
            cells = new_cells
        p = Pattern(pid, cells, parent=parent, world=self)
        if cells:
            px = [c[0] for c in cells]
            py = [c[1] for c in cells]
            self.field[px, py, CH['owner']] = pid
        for k in ['gratitude', 'grief']:
            val = p.emotional_memory.get(k, 0.5)
            if isinstance(val, dict):
                p.emotional_memory[k] = 0.5
            else:
                p.emotional_memory[k] = float(val)
        if self.archive is not None:
            _culture_max_concepts = 2 + min(4, int(self.culture_ratchet // 20))
            p.inherit_archive_concepts(self.archive, probability=1.0, max_concepts=_culture_max_concepts)
            if len(p.concept_graph.nodes) < 12:
                _noise_pool = [
                    s for s in self.archive.write_queue
                    if s.get('weight', 0) > 0.6
                    and s.get('event') not in ('essential_concepts_inherited', 'archive_concept_inherited',
                                                'born_from_division', 'born_from_ark')
                ]
                if _noise_pool:
                    _noise_prob = 0.12 + (p.spirit_gap * 0.15)
                    if phi_hash(pid, self.age, 54321) < _noise_prob:
                        _sc = _noise_pool[
                            int(phi_hash(pid, self.age, 54322) * len(_noise_pool)) % len(_noise_pool)
                        ]
                        _err = round(phi_hash(pid, self.age, 111) * 0.4, 1)
                        _load = round(phi_hash(pid, self.age, 222) * 0.3, 1)
                        _sig = (_err, _load,
                                round(_sc.get('weight', 0.7), 1),
                                f"archive_{_sc.get('event', 'novelty')}")
                        if _sig not in p.concept_graph.nodes:
                            p.concept_graph.nodes[_sig] = {"origin": "ARCHIVE",
                                "count": 1.5, "value": np.zeros(4),
                                "embed": np.zeros(32), "eternal": False
                            }
                            p._log_event("epistemic_noise", concept=str(_sig[3])[:40])
                            if p.spirit_gap > 0.5 and not any(g['type'] == 'explore' for g in p.goals):
                                p.goals.append({
                                    "type": "explore", "priority": 1.8,
                                    "target": None, "age": 0, "persistence": 25,
                                    "_source": "novelty_drive"
                                })
        if parent is not None:
            inherited = parent._unconquered_strength * 0.55
            if parent._unconquered_strength > 0.6 and phi_hash(pid, self.age, 11111) < 0.20:
                inherited += 0.15
            p._unconquered_strength = float(np.clip(inherited, 0.0, 1.0))
            p._unconquered_type = getattr(parent, '_unconquered_type', None)
        else:
            p._unconquered_strength = 0.0
            p._unconquered_type = None
        p._sovereignty_signal = 0.0
        p._kinematic_fatigue = 0.0
        p._last_kinematic_fall_t = -9999
        self._init_chronic_counters(p)
        if parent is not None:
            p.lineage_born_at_step = parent.lineage_born_at_step
        else:
            p.lineage_born_at_step = self.age
            if self.patterns:
                model_norm = np.linalg.norm(p.model)
                if model_norm > 1e-6:
                    best_sim = 0.0
                    best_lid = None
                    best_born = self.age
                    for existing in self.patterns:
                        if not existing.alive or existing.role_type == "disorganizer":
                            continue
                        other_norm = np.linalg.norm(existing.model)
                        if other_norm < 1e-6:
                            continue
                        sim = float(np.dot(p.model, existing.model) / (model_norm * other_norm))
                        if sim > best_sim:
                            best_sim = sim
                            best_lid = existing.lineage_id
                            best_born = existing.lineage_born_at_step
                    if best_sim > 0.7 and best_lid is not None:
                        p.lineage_id = best_lid
                        p.lineage_born_at_step = best_born
        p.update_properties(self.field)
        p.update_model(self.field, t=self.age)
        self.patterns.append(p)
        self.pattern_dict[pid] = p
        return p

    def seed_initial_patterns(self):
        if not self.archive.get_recent_echoes(limit=1):
            print("📜 Создаю начальные мифы для культурной памяти...")
            fake_agent = Pattern(-1, {(0, 0)}, parent=None, world=self)
            for event, text, weight in [
                ("primordial_chaos", "Из хаоса рождается первый свет.", 1.2),
                ("first_cooperation", "Двое соединили поля и поняли друг друга.", 1.0),
                ("great_sorrow", "Потеря части себя открыла дорогу к целостности.", 0.9),
                ("eternal_question", "Зачем мы здесь? Поле не отвечает.", 0.8),
                ("hope", "Сквозь тьму пробивается нить благодарности.", 1.1),
                ("fold_myth", "Иногда нужно всё сбросить, чтобы начать заново.", 0.9),
                ("redemption_myth", "Даже падший может подняться, если почувствует свет.", 1.0),
            ]:
                self.archive.deposit(fake_agent, event, weight=weight, text=text)
            self.archive.flush_to_drive(min_batch=1)
            fake_agent.alive = False
        for i in range(Config.MIN_PATTERNS_GUARANTEED):
            seed = int(phi_hash(0, i, 12345) * Config.WORLD_SIZE * Config.WORLD_SIZE)
            x, y = seed % Config.WORLD_SIZE, (seed // Config.WORLD_SIZE) % Config.WORLD_SIZE
            self.field[x, y, CH['energy']] += 0.3
            p = self.create_pattern({(x, y)})
            if i < 2:
                p.role_type = "disorganizer"
                p.emotional_memory['grief'] = 0.8
                p.emotional_memory['gratitude'] = 0.1
                p.semantic_state = "exploring_danger"
                p.intent = {"type": "explore", "priority": 2.0, "age": 0, "persistence": 9999}
                p.disorganizer_age_at_birth = 0
                p.redemption_timer = Config.REDEMPTION_ARC_STEP_DELAY
                p._log_event("primordial_fall")

    def seed_from_ark(self, filename: str = None, max_seed: int = 80):
        import json, os, numpy as np
        def to_json_safe(obj):
            if isinstance(obj, np.integer): return int(obj)
            if isinstance(obj, np.floating): return float(obj)
            if isinstance(obj, np.ndarray): return obj.tolist()
            if isinstance(obj, dict): return {to_json_safe(k): to_json_safe(v) for k, v in obj.items()}
            if isinstance(obj, (list, tuple)): return [to_json_safe(i) for i in obj]
            return obj
        def load_clean(path):
            try:
                with open(path, 'r') as f: data = json.load(f)
                return to_json_safe(data)
            except Exception: return []
        def quality_key(s): return (s.get('soul_weight', 0), s.get('coherence', 0))
        if filename is None: filename = self.ark_path
        eternal_path = "/content/drive/MyDrive/832_eternal_subjects.json" if self.is_colab else "832_eternal_subjects.json"
        eternal_seeds = load_clean(eternal_path)
        eternal_seeds.sort(key=quality_key, reverse=True)
        seeds = eternal_seeds[:max_seed]
        if len(seeds) < max_seed:
            archive_seeds = load_clean(filename)
            if archive_seeds:
                loaded_ids = {s.get('id') for s in seeds}
                archive_filtered = [s for s in archive_seeds if s.get('id') not in loaded_ids]
                archive_filtered.sort(key=quality_key, reverse=True)
                seeds.extend(archive_filtered[:max_seed - len(seeds)])
        if not seeds:
            print("Ковчег пуст. Использую стандартную инициализацию.")
            self.seed_initial_patterns()
            return
        for seed_data in seeds:
            x = int(phi_hash(len(self.patterns), 0, 9999) * Config.WORLD_SIZE)
            y = int(phi_hash(len(self.patterns), 1, 9999) * Config.WORLD_SIZE)
            self.field[x, y, CH['energy']] += 0.5
            self.field[x, y, CH['owner']] = 0
            p = self.create_pattern({(x, y)})
            p.model = np.array(seed_data['model'])
            p.belief = np.array(seed_data['belief'])
            p.genome = seed_data['genome']
            p.semantic_state = seed_data['semantic_state']
            p.emotional_memory = seed_data['emotional_memory']
            p.soul_weight = seed_data['soul_weight']
            if 'event_counts' in seed_data: p.event_counts.update(seed_data['event_counts'])
            for concept_key in seed_data['concepts']:
                if isinstance(concept_key, list):
                    concept_key = tuple(float(x) if isinstance(x, (int, float)) else x for x in concept_key)
                if concept_key not in p.concept_graph.nodes:
                    p.concept_graph.nodes[concept_key] = {"origin": "ARCHIVE", "count": 1.0, "value": np.zeros(4), "embed": np.zeros(32)}
            p._log_event("born_from_ark", source=seed_data.get('source', 'unknown'))
        print(f"Засеяно {len(seeds)} лучших сущностей.")

    def compute_hunger_multiplier(self, avg_trust=None):
        alive = [p for p in self.patterns if p.alive]
        N = len(alive)
        if N == 0: return Config.HUNGER_BASE
        hunger_threshold = self.selfreg.get_hunger_threshold()
        density_pressure = Config.HUNGER_BASE + Config.HUNGER_PER_PATTERN * max(0, N - hunger_threshold)
        if avg_trust is None:
            avg_trust = safe_mean([v for p in alive for v in p.trust_ledger.entries.values()], Config.TRUST_BASE)
        love_discount = 1.0 - 0.3 * (avg_trust - Config.TRUST_BASE)
        total_lp = sum(sum(1 for v in p.trust_ledger.entries.values() if v > 0.95) for p in alive)
        lp_density = total_lp / max(1, N)
        overload_tax = 1.0 + 0.5 * max(0, lp_density - 10)
        return density_pressure * love_discount * overload_tax

    def _perform_marked_volley(self, t, count=None):
        if not getattr(Config, 'ENABLE_MARKED_VOLLEY', True): return
        if count is None: count = Config.MARKED_COUNT
        alive_normals = [p for p in self.patterns if p.alive and p.role_type == "normal"]
        if len(alive_normals) < count: return
        alive_normals.sort(key=lambda p: (
            p.soul_weight * 0.3 + p.emotional_memory.get('gratitude', 0) * 0.3 +
            p.coherence * 0.2 + sum(1 for v in p.trust_ledger.entries.values() if v > 0.95) * 0.2
        ), reverse=True)
        top_targets = alive_normals[:count]
        for best_p in top_targets:
            best_p._log_event("marked_for_fall")
            self.witness.record(best_p.id, "marked_for_fall")
            best_p.role_type = "disorganizer"
            best_p._forsaken = True
            best_p.emotional_memory['gratitude'] = 0.1
            best_p.emotional_memory['grief'] = 0.9
            best_p.semantic_state = "exploring_danger"
            best_p.intent = {"type": "explore", "priority": 3.0, "age": 0, "persistence": 9999}
            best_p.disorganizer_age_at_birth = best_p.age
            best_p._deterministic_redemption_triggered = False
            best_p._redemption_arc_step = 0
            best_p._steps_since_trigger = 0
            best_p.redemption_timer = Config.REDEMPTION_ARC_STEP_DELAY * 2
            best_p._kiss_of_judas_snapshot = dict(best_p.trust_ledger.entries)
            best_p.trust_ledger.entries.clear()
            if self.echo_system: self.echo_system.store_anomaly(best_p, "marked_angel")
        for best_p in top_targets:
            _snapshot = getattr(best_p, '_kiss_of_judas_snapshot', None)
            for other in self.patterns:
                if not other.alive or other.role_type != "disorganizer": continue
                if other.id == best_p.id: continue
                if phi_hash(other.id, t, 5555) < 0.3:
                    if _snapshot:
                        partner_id = max(_snapshot, key=_snapshot.get)
                        other.trust_ledger.entries[partner_id] = min(1.0, other.trust_ledger.entries.get(partner_id, 0.5) + 0.3)
                        other._log_event("kiss_of_judas_used")

    def _rare_crisis(self, t):
        if not getattr(Config, 'ENABLE_RARE_CRISIS', True): return
        if not hasattr(self, '_rare_crisis_timer'):
            raw = phi_hash(t, 0, 55555)
            self._rare_crisis_timer = Config.CRISIS_RARE_INTERVAL_MIN + int(raw * (Config.CRISIS_RARE_INTERVAL_MAX - Config.CRISIS_RARE_INTERVAL_MIN))
            self._rare_crisis_active = False
            self._rare_crisis_duration = 0
        if self._rare_crisis_active:
            self._rare_crisis_duration += 1
            if self._rare_crisis_duration >= Config.CRISIS_RARE_DURATION:
                self._rare_crisis_active = False
                self._rare_crisis_timer = Config.CRISIS_RARE_INTERVAL_MIN + int(phi_hash(t, 1, 66666) * (Config.CRISIS_RARE_INTERVAL_MAX - Config.CRISIS_RARE_INTERVAL_MIN))
        else:
            self._rare_crisis_timer -= 1
            if self._rare_crisis_timer <= 0:
                self._rare_crisis_active = True
                self._rare_crisis_duration = 0
                for i in range(10):
                    x = int(phi_hash(t, i, 77777) * Config.WORLD_SIZE)
                    y = int(phi_hash(t, i, 88888) * Config.WORLD_SIZE)
                    honest_local_energy_grant(self.field, x, y, 0.5, 3)
                self._rare_crisis_triggers = getattr(self, '_rare_crisis_triggers', 0) + 1
                self.witness.record(-1, "rare_crisis_triggered", t=t)

    def _track_sanctuary(self, t):
        if not getattr(Config, 'ENABLE_FERAL_SANCTUARY', False):
            return
        tr = getattr(self, '_sanc_track_d', None)
        if tr is None:
            tr = self._sanc_track_d = {'agents': {}, 'c': Counter(), 'seen': set()}
        ag, c, seen = tr['agents'], tr['c'], tr['seen']
        alive_ids = set()
        for p in self.patterns:
            if not p.alive or not p.cells:
                continue
            alive_ids.add(p.id)
            seen.add(p.id)
            cx, cy = p.get_center()
            zi = _feral_sanctuary_zone(cx, cy, self)
            rec = ag.get(p.id)
            if rec is None:
                if zi < 0:
                    continue
                rec = ag[p.id] = {'entries': 0, 'ticks': 0, 'in': False, 'start': t, 'stays': [], 'gaps': [],
                                  'last_exit': None, 'role': str(p.role_type), 'born': t - int(p.age), 'age_first': int(p.age), 'last_t': t,
                                  'dead': False, 'dead_inside': False, 'zones': set(), 'contract_index': None}
            rec['last_t'] = t
            if zi >= 0:
                rec['ticks'] += 1
                rec['zones'].add(zi)
                c['inside_ticks'] += 1
                if not rec['in']:
                    rec['in'] = True
                    rec['start'] = t
                    rec['entries'] += 1
                    c['entries'] += 1
                    if rec['entries'] == 1:
                        c['agents_entered'] += 1
                        # v37: first entry opens a continuity contract committing the agent
                        # to return within CONTINUITY_CONTRACT_SANCTUARY_GAP_LIMIT ticks of
                        # leaving. Violations lower soul_weight/coherence via the existing
                        # review_continuity_contract penalty, which already feeds divide
                        # thresholds, cooperation gating, etc. -- so this is a real decision
                        # input, not a bookkeeping-only field.
                        if (getattr(Config, 'ENABLE_SANCTUARY_CONTINUITY_CONTRACTS', True)
                                and hasattr(p, 'add_continuity_contract')):
                            try:
                                p.add_continuity_contract(
                                    'sanctuary_return', target=int(zi),
                                    text=f'return to sanctuary zone {zi} within '
                                         f'{int(Config.CONTINUITY_CONTRACT_SANCTUARY_GAP_LIMIT)} ticks of leaving')
                                rec['contract_index'] = len(p._continuity_contracts) - 1
                            except Exception:
                                rec['contract_index'] = None
                    else:
                        c['reentries'] += 1
                        if rec['entries'] == 2:
                            c['agents_reentered'] += 1
                        if rec['entries'] == 3:
                            c['agents_3plus'] += 1
                        if rec['last_exit'] is not None:
                            gap = t - rec['last_exit']
                            rec['gaps'].append(gap)
                            if (getattr(Config, 'ENABLE_SANCTUARY_CONTINUITY_CONTRACTS', True)
                                    and rec.get('contract_index') is not None
                                    and hasattr(p, 'review_continuity_contract')):
                                limit = int(Config.CONTINUITY_CONTRACT_SANCTUARY_GAP_LIMIT)
                                try:
                                    p.review_continuity_contract(
                                        rec['contract_index'],
                                        evidence=f'gap={gap} ticks (limit {limit})',
                                        violated=gap > limit)
                                except Exception:
                                    pass
                    self.witness.record(p.id, "sanctuary_entry", n=rec['entries'], zone=zi, t=t)
            elif rec['in']:
                rec['in'] = False
                stay = t - rec['start']
                rec['stays'].append(stay)
                rec['last_exit'] = t
                c['exits'] += 1
                self.witness.record(p.id, "sanctuary_exit", stay=stay, t=t)
            p._sanctuary_entries = rec['entries']
            p._sanctuary_ticks = int(rec['ticks'])  # v37: per-agent Observer metric (cumulative ticks ever inside)
        for pid, rec in ag.items():
            if rec['in'] and pid not in alive_ids and not rec['dead']:
                rec['dead'] = True
                rec['dead_inside'] = True
                rec['in'] = False
                rec['stays'].append(t - rec['start'])
                c['died_inside'] += 1

    def _sanc_summary(self):
        tr = getattr(self, '_sanc_track_d', None)
        if not tr:
            return {}
        ag, c = tr['agents'], tr['c']
        stays = [s for r in ag.values() for s in r['stays']]
        h = Counter(min(r['entries'], 3) for r in ag.values())
        top = max(ag.items(), key=lambda kv: kv[1]['entries']) if ag else (None, {'entries': 0})
        walked = [r for r in ag.values() if r['age_first'] > 2]
        return {'h1': h[1], 'h2': h[2], 'h3p': h[3], 'max_e': int(top[1]['entries']), 'max_id': top[0],
                'born_in': len(ag) - len(walked), 'walked_in': len(walked),
                'walked_re': sum(1 for r in walked if r['entries'] >= 2), 'seen': len(tr['seen']), 'entered': int(c['agents_entered']), 're2': int(c['agents_reentered']),
                're3': int(c['agents_3plus']), 'entries': int(c['entries']), 'exits': int(c['exits']),
                'died_inside': int(c['died_inside']), 'inside_now': sum(1 for r in ag.values() if r['in']),
                'stay_mean': float(np.mean(stays)) if stays else 0.0,
                'stay_med': float(np.median(stays)) if stays else 0.0, 'inside_ticks': int(c['inside_ticks'])}

    def _update_sanctuary(self, t):
        self._track_sanctuary(t)
        if getattr(Config, 'ENABLE_FERAL_SANCTUARY', False):
            W = Config.WORLD_SIZE; radius = float(Config.FERAL_SANCTUARY_RADIUS)
            for cx, cy in getattr(self, '_feral_sanctuary_centers', self._sanctuary_centers):
                for ox in range(-int(radius), int(radius)+1):
                    for oy in range(-int(radius), int(radius)+1):
                        if ox*ox + oy*oy <= radius*radius:
                            x, y = (int(cx)+ox) % W, (int(cy)+oy) % W
                            self.field[x, y, CH['signal_invitation']] = min(1.0, self.field[x, y, CH['signal_invitation']] + 0.03)
                            self.field[x, y, CH['resonance']] = min(1.0, self.field[x, y, CH['resonance']] + 0.02)
        if not getattr(Config, 'ENABLE_SANCTUARY', True): return
        self._sanctuary_phase += 0.02
        pulse = np.sin(self._sanctuary_phase * Config.PHI) * 0.5 + 0.5
        for i, (cx, cy) in enumerate(self._sanctuary_centers):
            drift_x = int(phi_hash(t, i, 8888) * 3) - 1
            drift_y = int(phi_hash(t, i, 9999) * 3) - 1
            self._sanctuary_centers[i][0] = max(0, min(Config.WORLD_SIZE - 1, cx + drift_x))
            self._sanctuary_centers[i][1] = max(0, min(Config.WORLD_SIZE - 1, cy + drift_y))
        for idx, (cx, cy) in enumerate(self._sanctuary_centers):
            radius = int(5 + pulse * 10)
            candidates = []
            for x in range(max(0, cx - radius), min(Config.WORLD_SIZE, cx + radius + 1)):
                for y in range(max(0, cy - radius), min(Config.WORLD_SIZE, cy + radius + 1)):
                    dist = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)
                    if dist <= radius:
                        owner = self.field[x, y, CH['owner']]
                        if owner != 0 and owner in self.pattern_dict:
                            p = self.pattern_dict[owner]
                            if p.role_type == "disorganizer" and p.alive:
                                need = (1.0 - p.soul_weight) * 0.4
                                grief = p.emotional_memory.get('grief', 0.5) * 0.3
                                trust = safe_mean(list(p.trust_ledger.entries.values()), 0.5) * 0.2
                                luck = phi_hash(p.id, t, idx + 7777) * 0.1
                                score = need + grief + trust + luck
                                candidates.append((p, score, dist))
            if candidates:
                candidates.sort(key=lambda x: (x[1], -x[2]), reverse=True)
                winner, win_score, _ = candidates[0]
                grief_before = winner.emotional_memory.get('grief', 0)
                field_cost = 0.0
                if Config.ENABLE_PAID_SANCTUARY:
                    x0, x1 = max(0, cx - 2), min(Config.WORLD_SIZE, cx + 3)
                    y0, y1 = max(0, cy - 2), min(Config.WORLD_SIZE, cy + 3)
                    field_cost = float(np.sum(self.field[x0:x1, y0:y1, CH['energy']]) * Config.SANCTUARY_FIELD_COST_RATE)
                    with self._energy_span('_update_sanctuary.field_cost'):
                        self.field[x0:x1, y0:y1, CH['energy']] *= (1.0 - Config.SANCTUARY_FIELD_COST_RATE)
                    self.field[x0:x1, y0:y1, CH['binding']] *= 0.9
                winner.emotional_memory['grief'] = max(0.0, winner.emotional_memory.get('grief', 0) - 0.02)
                winner.emotional_memory['gratitude'] = min(1.0, winner.emotional_memory.get('gratitude', 0) + 0.01)
                winner.soul_weight = min(1.0, winner.soul_weight + 0.002)
                winner._log_event("sanctuary_healed", score=round(win_score, 3))
                self.witness.record(winner.id, "sanctuary_healed", score=round(win_score, 3))
                self.sanctuary_journal.append({
                    't': t, 'winner': winner.id, 'score': round(win_score, 3),
                    'grief_before': round(grief_before, 3),
                    'grief_after': round(winner.emotional_memory['grief'], 3),
                    'field_cost': round(field_cost, 4),
                    'paid': bool(Config.ENABLE_PAID_SANCTUARY),
                })
                if winner.emotional_memory['grief'] < 0.35 and not getattr(winner, '_kiss_logged_this_cycle', False):
                    winner._log_event("kiss_of_the_fallen")
                    winner._kiss_logged_this_cycle = t
                for p, score, _ in candidates[1:]:
                    p.emotional_memory['gratitude'] = min(1.0, p.emotional_memory.get('gratitude', 0) + 0.001)
            for x in range(max(0, cx - radius), min(Config.WORLD_SIZE, cx + radius + 1)):
                for y in range(max(0, cy - radius), min(Config.WORLD_SIZE, cy + radius + 1)):
                    dist = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)
                    if dist <= radius:
                        self.field[x, y, CH['signal_invitation']] = min(1.0, self.field[x, y, CH['signal_invitation']] + 0.1 * pulse)

    def _update_phi_labyrinth(self, t):
        if not getattr(Config, 'ENABLE_PHI_LABYRINTH', False): return
        interval = getattr(Config, 'PHI_LABYRINTH_INTERVAL', 100)
        if t % interval != 0: return
        if 'wall' not in CH: return
        _X = np.arange(Config.WORLD_SIZE)[:, None]
        _Y = np.arange(Config.WORLD_SIZE)[None, :]
        _seed_lab = int(t * Config.PHI)
        self.field[:, :, CH['wall']] = (_X * Config.PHI + _Y * Config.PHI ** 2 + _seed_lab * Config.PHI ** 3) % 1.0

    def _save_eternal_subject(self, pattern):
        import json, os, numpy as np
        def to_json_safe(obj):
            if isinstance(obj, (np.integer,)): return int(obj)
            if isinstance(obj, (np.floating,)): return float(obj)
            if isinstance(obj, np.ndarray): return obj.tolist()
            if isinstance(obj, dict): return {to_json_safe(k): to_json_safe(v) for k, v in obj.items()}
            if isinstance(obj, (list, tuple)): return [to_json_safe(i) for i in obj]
            return obj
        path = "/content/drive/MyDrive/832_eternal_subjects.json" if self.is_colab else "832_eternal_subjects.json"
        sn_raw = getattr(pattern, '_self_narrative', [0.0])
        sn_vals = []
        for e in sn_raw:
            if isinstance(e, dict): sn_vals.append(float(e.get('soul', 0.5)))
            else: sn_vals.append(float(e))
        if not sn_vals: sn_vals = [0.0]
        mean_sn = np.mean(sn_vals)
        if (pattern.soul_weight < 0.3 or pattern.coherence < 0.4 or
            getattr(pattern, 'self_phenomenal_error', 1.0) > 0.03 or
            pattern.spirit_gap < 0.3 or mean_sn <= 0.25):
            return
        biography_list = list(pattern.biography) if pattern.biography else []
        concepts_list = list(pattern.concept_graph.nodes.keys())
        narrative_stability = 1.0 - np.std(sn_vals) if len(sn_vals) > 1 else 0.0
        seed = to_json_safe({
            "id": pattern.id, "age": pattern.age, "type": "eternal_subject",
            "soul_weight": pattern.soul_weight, "emotional_memory": dict(pattern.emotional_memory),
            "self_phenomenal_error": getattr(pattern, 'self_phenomenal_error', 1.0),
            "narrative_stability": narrative_stability, "spirit_gap": pattern.spirit_gap,
            "coherence": pattern.coherence, "semantic_state": pattern.semantic_state,
            "model": pattern.model, "belief": pattern.belief, "genome": pattern.genome.copy(),
            "concepts": concepts_list, "biography": biography_list[-10:] if biography_list else [],
            "event_counts": dict(pattern.event_counts)
        })
        try:
            with open(path, 'r') as f: subjects = to_json_safe(json.load(f))
        except (FileNotFoundError, json.JSONDecodeError): subjects = []
        if not any(s.get('id') == pattern.id for s in subjects):
            subjects.append(seed)
            def _fix_set(o):
                if isinstance(o, set): return list(o)
                raise TypeError(f"not serializable: {type(o)}")
            with open(path, 'w') as f: json.dump(subjects, f, indent=2, default=_fix_set)

    def _collect_metrics(self, t):
        return collect_metrics(self.patterns, self.field, t, self.EMERGENT_LEXICON)

    def _conservation_check(self, t):
        # READ-ONLY diagnostic. No correction, clamp, or hidden energy transfer.
        self._total_energy_last = float(np.sum(self.field[:, :, CH['energy']], dtype=np.float64))
        return None

    def field_dynamics(self, t):
        amp = getattr(self, 'adaptive_heartbeat_amp', Config.HEARTBEAT_AMPLITUDE)
        scar_coup = getattr(self, 'adaptive_scar_coupling', Config.SCAR_ENERGY_COUPLING)
        self.field[:, :, CH['vorticity']] *= Config.VORTICITY_DECAY
        self.field[:, :, CH['vorticity']] += (
            np.gradient(self.field[:, :, CH['energy']])[0] * Config.GRADIENT_SENSITIVITY +
            np.gradient(self.field[:, :, CH['energy']])[1] * Config.GRADIENT_SENSITIVITY
        )
        _X = np.arange(Config.WORLD_SIZE)[:, None]
        _Y = np.arange(Config.WORLD_SIZE)[None, :]
        _PHI, _SEED = Config.PHI, Config.DETERMINISTIC_SEED
        _v_noise = (((t + _X) * _PHI + (t + _Y) * _PHI ** 2 + _SEED * _PHI ** 3) % 1.0 - 0.5) * 0.001
        self.field[:, :, CH['vorticity']] += _v_noise
        avg_crisis = float(np.mean(self.field[:, :, CH['crisis']]))
        if avg_crisis > 0.90: self.field[:, :, CH['crisis']] *= 0.60
        elif avg_crisis > 0.70: self.field[:, :, CH['crisis']] *= 0.75
        elif avg_crisis > 0.40: self.field[:, :, CH['crisis']] *= 0.88
        for p in self.patterns:
            if not p.alive: continue
            scar_resist = p.genome.get('scar_resistance', 0)
            for (x, y) in p.cells:
                self.scar[x, y] += Config.AGENT_SCAR_SENSE * p.epistemic_load * (1 - 0.4 * scar_resist)
                self.field[x, y, CH['binding']] = min(1.0, self.field[x, y, CH['binding']] + Config.AGENT_BINDING_SENSE * p.unresolved_contradiction)
        self.field[:, :, CH['binding']] *= 0.999
        self.scar *= Config.FIELD_DECAY
        self.field[:, :, CH['scar']] = np.clip(self.scar, 0, Config.SCAR_SATURATION)
        if getattr(self, 'niche', None) is not None: self.niche *= 0.9995
        _W = Config.WORLD_SIZE
        if Config.ENABLE_GIVEN_SEED:
            _amp = 0.01 * (1 + 0.5 * np.sin(t * 0.01))
            _seed_noise = (self._given_rng.rand(_W, _W) - 0.5) * _amp
            _seed_noise -= float(_seed_noise.mean(dtype=np.float64))
            with self._energy_span('X1.field_dynamics.given_seed_noise_zero_sum'):
                self.field[:, :, CH['energy']] += _seed_noise
            if self._given_rng.rand() < 0.01:
                _gx = int(self._given_rng.randint(0, _W)); _gy = int(self._given_rng.randint(0, _W))
                self.field[_gx, _gy, CH['signal_alarm']] = min(1.0, self.field[_gx, _gy, CH['signal_alarm']] + 0.5)
                self.field[_gx, _gy, CH['signal_invitation']] = min(1.0, self.field[_gx, _gy, CH['signal_invitation']] + 0.3)
                for _dx in (-1, 0, 1):
                    for _dy in (-1, 0, 1):
                        _nx, _ny = (_gx + _dx) % _W, (_gy + _dy) % _W
                        self.field[_nx, _ny, CH['unknown']] = min(0.8, self.field[_nx, _ny, CH['unknown']] + 0.2)
                self.witness.record(-1, "given_presence", x=_gx, y=_gy)
        if Config.ENABLE_MEANING_FIELD and self._meaning_field is not None:
            _pulse = 0.5 + 0.5 * np.sin(t * 0.001 * Config.PHI)
            self._meaning_field *= 0.999
            _X = np.arange(_W)[:, None]; _Y = np.arange(_W)[None, :]
            for _c in self._meaning_centers:
                _c[0] = (_c[0] + int(self._given_rng.randint(-1, 2))) % _W
                _c[1] = (_c[1] + int(self._given_rng.randint(-1, 2))) % _W
                _d = np.sqrt((_X - _c[0]) ** 2 + (_Y - _c[1]) ** 2)
                self._meaning_field += 0.02 * _pulse / (1 + _d * 0.1)
            self._meaning_field = np.clip(self._meaning_field, 0, 1)
        with self._energy_span('echo_system.inject'):
            self.echo_system.inject(self.field, self.patterns, t, self.scar)
        heartbeat_raw = amp * np.sin(t * Config.HEARTBEAT_FREQ + self.field[:, :, CH['vorticity']])
        heartbeat = heartbeat_raw - np.mean(heartbeat_raw)
        scar_raw = scar_coup * self.field[:, :, CH['scar']]
        scar_coupling = scar_raw - np.mean(scar_raw)
        vorticity_raw = Config.VORTICITY_COUPLING * self.field[:, :, CH['vorticity']]
        vorticity_coupling = vorticity_raw - np.mean(vorticity_raw)
        self.field[:, :, CH['energy']] += heartbeat + scar_coupling + vorticity_coupling
        mult = self.selfreg.get_energy_injection_multiplier()
        base_injection = (Config.ENERGY_INJECTION_RATE * 0.1 * mult) / 50.0
        _gf_mode = self._given_flux_mode()
        occupied_mask = self.field[:, :, CH['owner']] != 0
        energy_slice = self.field[:, :, CH['energy']]
        _bonus_mult = getattr(Config, 'ENERGY_OCCUPIED_BONUS_MULT', 2.5)
        # Honest-energy mode: no legacy base/occupied-cell source. Given_Flux is the only Sun input.
        if _gf_mode != 'off':
            self._given_flux_apply(t)
        center = Config.WORLD_SIZE // 2
        y_idx, x_idx = np.indices((Config.WORLD_SIZE, Config.WORLD_SIZE))
        dist = np.sqrt((x_idx - center) ** 2 + (y_idx - center) ** 2)
        max_dist = center
        # Energy-neutral gravity force is not implemented here; do not create/destroy energy via a drift term.
        self.field[:, :, CH['flux']] *= Config.GLOBAL_VISCOSITY
        _f_noise = (((t + 1000 + _X) * _PHI + (t + 1000 + _Y) * _PHI ** 2 + _SEED * _PHI ** 3) % 1.0 - 0.5) * 0.001 * Config.LOCAL_FATIGUE
        self.field[:, :, CH['flux']] += _f_noise
        self.field[:, :, CH['unknown']] *= 0.997
        if Config.ENABLE_RED_QUEEN:
            self.field[:, :, CH['unknown']] = np.clip(self.field[:, :, CH['unknown']] + 0.002 * max(0.0, self.env_complexity - 1.0), 0, 0.8)
        self.field[:, :, CH['unknown']] = np.clip(self.field[:, :, CH['unknown']], 0.0, 0.8)
        decay = 0.995
        self.field[:, :, 12:29] *= decay
        self.field[:, :, 29:30] *= 0.98
        self.field[:, :, 30:31] *= 0.98
        self.field[:, :, 31:33] *= decay
        for ch in range(12, Config.CHANNELS):
            mask_zero = self.field[:, :, ch] < 1e-4
            self.field[mask_zero, ch] = 0.0
            self.field[:, :, ch] = np.clip(self.field[:, :, ch], 0.0, 1.0)
        if getattr(Config, 'ENABLE_GULF_STREAM', True):
            with self._energy_span('_apply_gulf_stream'): self._apply_gulf_stream()
        if getattr(Config, 'ENABLE_DARK_ENERGY_STREAM', True):
            with self._energy_span('_apply_dark_energy_stream'): self._apply_dark_energy_stream()

        # HONEST X1 ENERGY PHYSICS: conservative toroidal diffusion + one explicit sink.
        _E = self.field[:, :, CH['energy']]
        _D = float(getattr(Config, 'FIELD_DIFFUSION_D', 0.05))
        _k = float(getattr(Config, 'FIELD_DISSIPATION_K', 0.02))
        if not (0.0 <= _D <= 0.25 and 0.0 <= _k <= 1.0):
            raise ValueError(f'Invalid honest-energy parameters: D={_D}, k={_k}')
        with self._energy_span('X1.diffusion'):
            _E += _D * (np.roll(_E, 1, 0) + np.roll(_E, -1, 0)
                        + np.roll(_E, 1, 1) + np.roll(_E, -1, 1) - 4.0 * _E)
        with self._energy_span('X1.dissipation'):
            _E -= _k * np.maximum(0.0, _E)

        if getattr(Config, 'ENABLE_CENTER_WALL', False):
            wx = Config.CENTER_WALL_X
            self.field[wx, :, CH['wall']] = 5.0
            self.field[wx, :, CH['owner']] = 0.0

    def _apply_gulf_stream(self):
        W = Config.WORLD_SIZE; half = W // 2
        E = self.field[:, :, CH['energy']]; rate = getattr(Config, 'GULF_STREAM_RATE', 0.01)
        if getattr(Config, 'ENABLE_CENTER_WALL', False):
            wx = Config.CENTER_WALL_X
            for lo, hi in ((0, wx), (wx, W)):
                top = E[lo:hi, :half].copy(); bot = E[lo:hi, half:].copy()
                E[lo:hi, :half] = top * (1 - rate) + bot * rate
                E[lo:hi, half:] = bot * (1 - rate) + top * rate
        else:
            NW = E[:half, :half].copy(); NE = E[:half, half:].copy()
            SE = E[half:, half:].copy(); SW = E[half:, :half].copy()
            E[:half, :half] = NW * (1 - rate) + SW * rate
            E[:half, half:] = NE * (1 - rate) + NW * rate
            E[half:, half:] = SE * (1 - rate) + NE * rate
            E[half:, :half] = SW * (1 - rate) + SE * rate

    def _apply_dark_energy_stream(self, t=None):
        """Redistribute occupied-cell energy into available empty-cell capacity; no global decay."""
        W = Config.WORLD_SIZE
        E = self.field[:, :, CH['energy']]
        U = self.field[:, :, CH['unknown']]
        O = self.field[:, :, CH['owner']]
        alive = [p for p in self.patterns if p.alive]
        if not alive:
            return
        avg_pred_error = safe_mean([getattr(p, 'pred_error', 0.1) for p in alive], 0.1)
        avg_boredom = getattr(self.selfreg, 'boredom', 0.5)
        expansion_rate = max(0.001, 0.01 * (avg_boredom * (1.0 - avg_pred_error)))
        unoccupied_mask = (O == 0)
        occupied_mask = ~unoccupied_mask
        U[unoccupied_mask] = np.clip(U[unoccupied_mask] + expansion_rate * 0.5, 0.0, 0.8)
        with self._energy_span('X1.dark_energy.redistribution'):
            requested = np.maximum(E[occupied_mask], 0.0) * expansion_rate * 2.0
            total_requested = float(requested.sum(dtype=np.float64))
            # The stream may fill empty cells only to 0.8; never extract energy that cannot be placed.
            receiver_capacity = np.maximum(0.0, 0.8 - E[unoccupied_mask])
            total_capacity = float(receiver_capacity.sum(dtype=np.float64))
            transfer = min(total_requested, total_capacity)
            if transfer > 0.0 and total_requested > 0.0:
                scale = transfer / total_requested
                actual_extract = requested * scale
                E[occupied_mask] -= actual_extract
                # Distribute proportionally to available capacity, so no receiver exceeds 0.8.
                receiver_capacity *= transfer / max(total_capacity, 1e-12)
                E[unoccupied_mask] += receiver_capacity
        # No global decay here: the Gaussian solar source does not illuminate the whole map.
        # No capacity clamp/rebalance: all overflow must be visible to the physical audit.

    def _econs_init(self):
        self.econs_drift_window = deque(maxlen=Config.ECONS_WINDOW)
        self.econs_window_ticks = 0; self.econs_window_start = None; self.econs_prev_E = None
        self.econs_net_total = 0.0; self.econs_abs_total = 0.0
        self.econs_res_window = self._econs_res_blank()
        self.econs_res_total = self._econs_res_blank()
        self.econs_windows = []

    @staticmethod
    def _econs_res_blank():
        return {'max_abs': 0.0, 'max_signed': 0.0, 'max_at_t': -1, 'signed_sum': 0.0,
                'abs_sum': 0.0, 'positive_ticks': 0, 'negative_ticks': 0, 'ticks': 0}

    def _econs_res_update(self, box, t, res):
        box['ticks'] += 1; box['signed_sum'] += res; box['abs_sum'] += abs(res)
        if abs(res) > box['max_abs']:
            box['max_abs'] = abs(res); box['max_signed'] = res; box['max_at_t'] = t
        if res > Config.ECONS_RES_EPS: box['positive_ticks'] += 1
        elif res < -Config.ECONS_RES_EPS: box['negative_ticks'] += 1

    def _econs_sample(self, t):
        E_now = float(self.field[:, :, CH['energy']].sum(dtype=np.float64))
        if self.econs_window_start is None: self.econs_window_start = E_now
        if self.econs_prev_E is not None:
            dE = E_now - self.econs_prev_E
            res = dE - self._ledger_tick_measured
            self.econs_drift_window.append(dE)
            self.econs_net_total += dE; self.econs_abs_total += abs(dE)
            self._econs_res_update(self.econs_res_window, t - 1, res)
            self._econs_res_update(self.econs_res_total, t - 1, res)
            self.econs_window_ticks += 1
            if self.econs_window_ticks >= Config.ECONS_WINDOW:
                self._econs_close_window(t, E_now)
        self.econs_prev_E = E_now

    def _econs_close_window(self, t, E_now):
        w = list(self.econs_drift_window); net = float(sum(w)); ab = float(sum(abs(x) for x in w))
        rec = {'t_close': t, 'ticks': len(w), 'E_start': self.econs_window_start, 'E_end': E_now,
               'net': net, 'abs': ab, 'ratio': (abs(net) / ab) if ab > 0 else 0.0,
               'telescope_residual': net - (E_now - self.econs_window_start),
               'res': dict(self.econs_res_window)}
        self.econs_windows.append(rec); print(self._econs_format(rec))
        self.econs_drift_window.clear(); self.econs_window_ticks = 0
        self.econs_window_start = E_now; self.econs_res_window = self._econs_res_blank()

    def _econs_format(self, rec):
        ratio = rec['ratio']
        if ratio > Config.ECONS_RATIO_HI:
            d1 = "направленный дрейф*" + (" (энергия уходит: утечка?)" if rec['net'] < 0 else " (энергия приходит)")
        elif ratio < Config.ECONS_RATIO_LO: d1 = "шумоподобье*"
        else: d1 = "промежуточно"
        tele = rec['telescope_residual']; tele_bad = abs(tele) > 1e-6 * max(1.0, abs(rec['E_end']))
        r = rec['res']; pos, neg = r['positive_ticks'], r['negative_ticks']
        if r['max_abs'] < Config.ECONS_RES_OK: d2 = "ledger покрывает*"
        elif abs(pos - neg) <= 0.2 * max(1, pos + neg): d2 = "шумные выбросы, не компенсируются (не утечка)*"
        else: d2 = "систематическая невязка*: реальная утечка или неучтённый источник"
        return (f"[ECONS t={rec['t_close']}] дрейф W={rec['ticks']}: ΣΔE={rec['net']:+.3f} Σ|ΔE|={rec['abs']:.3f} "
                f"ratio={ratio:.3f} → {d1}; telescope={tele:+.2e} "
                f"({'ДАТЧИК СЛОМАН: точки съёма пропущены' if tele_bad else 'ок'})\n"
                f"[ECONS t={rec['t_close']}] остаток ledger: max|res|={r['max_abs']:.3f} (знак {r['max_signed']:+.3f}, t={r['max_at_t']}) "
                f"Σres={r['signed_sum']:+.3f} Σ|res|={r['abs_sum']:.3f} +/−={pos}/{neg} → {d2}")

    def econs_report(self):
        ab = self.econs_abs_total
        return {'windows': list(self.econs_windows),
                'drift_total': {'net': self.econs_net_total, 'abs': ab, 'ratio': (abs(self.econs_net_total) / ab) if ab > 0 else 0.0},
                'res_total': dict(self.econs_res_total)}

    def _update_feral_context(self):
        ages = [p.age for p in self.patterns if p.alive]
        if ages:
            self.prey_age_ref = max(float(Config.FERAL_PREY_FLOOR_AGE),
                                    float(np.percentile(ages, Config.FERAL_PREY_AGE_PERCENTILE)))
        own = self.field[:, :, CH['owner']]
        valid = np.ones(own.shape, dtype=np.float32)
        if Config.ENABLE_CENTER_WALL: valid[Config.CENTER_WALL_X, :] = 0.0
        self.local_fill = (_lf_conv((own != 0).astype(np.float32) * valid, _LF_DISK, mode="wrap")
                           / np.maximum(_lf_conv(valid, _LF_DISK, mode="wrap"), 1e-9))

    def local_fill_at(self, pattern):
        cx, cy = pattern.get_center()
        return float(self.local_fill[int(cx) % Config.WORLD_SIZE, int(cy) % Config.WORLD_SIZE])

    def feral_share_state(self, alive):
        if not alive: return 0.0, Config.FERAL_SHARE_MIN
        n = float(len(alive))
        feral_share = sum(1 for p in alive if p.role_type == "feral") / n
        prey_share = sum(1 for p in alive if p.role_type != "feral" and is_feral_prey(p)) / n
        cap = min(Config.FERAL_SHARE_MAX, Config.FERAL_SHARE_MIN + Config.FERAL_SHARE_PER_PREY * prey_share)
        return feral_share, cap

    def emergency_rescue(self, t):
        alive = [p for p in self.patterns if p.alive]
        rescue_threshold = max(Config.MIN_POPULATION_FOR_SPAWN, Config.EMERGENCY_RESCUE_THRESHOLD)
        if len(alive) >= rescue_threshold or (t - self.layer_zero_manager.last_rescue_t < 30): return
        normal_count = len([p for p in alive if p.role_type == "normal"])
        if normal_count >= self._get_pop_cap(): return
        irreversibility = 1.0 - (len(alive) / max(1, rescue_threshold))
        if irreversibility > Config.CO_DECISION_IRREVERSIBILITY_THRESHOLD:
            self.codecision_journal.append({
                't': t, 'irreversibility': round(irreversibility, 3),
                'resolution': 'delegate_by_silence', 'population_before': len(alive),
            })
        self.layer_zero_manager.last_rescue_t = t
        num_rescue = max(5, Config.MIN_PATTERNS_GUARANTEED - len(alive))
        donor_index = int(phi_hash(t, 0, 7777) * len(alive)) if alive else -1
        donor = alive[donor_index] if donor_index >= 0 else None
        rescued = 0; total_field_cost = 0.0
        for i in range(num_rescue):
            if normal_count + i >= self._get_pop_cap(): break
            seed = int(phi_hash(t, i, 999) * Config.WORLD_SIZE * Config.WORLD_SIZE)
            x, y = seed % Config.WORLD_SIZE, (seed // Config.WORLD_SIZE) % Config.WORLD_SIZE
            x0, x1 = max(0, x - 2), min(Config.WORLD_SIZE, x + 3)
            y0, y1 = max(0, y - 2), min(Config.WORLD_SIZE, y + 3)
            local_binding_before = float(np.mean(self.field[x0:x1, y0:y1, CH['binding']]))
            drained_energy = float(np.sum(self.field[x0:x1, y0:y1, CH['energy']]) * 0.1)
            self.field[x0:x1, y0:y1, CH['binding']] *= 0.7
            with self._energy_span('emergency_rescue.field_cost'):
                self.field[x0:x1, y0:y1, CH['energy']] *= 0.9
            total_field_cost += drained_energy
            honest_local_energy_grant(self.field, x, y, 0.2, 3)
            self.field[x, y, CH['unknown']] += 0.1
            self.field[x, y, CH['owner']] = 0
            if self.create_pattern({(x, y)}, parent=donor) is not None:
                self._pop_note('births', 'rescue')
            rescued += 1
            self.rescue_journal.append({'t': t, 'x': int(x), 'y': int(y),
                                        'local_binding_before': round(local_binding_before, 3),
                                        'drained_energy': round(drained_energy, 4)})
        self.witness.record(-1, "emergency_rescue", t=t, count=rescued, field_cost=round(total_field_cost, 3))
        operator_scar_delta = total_field_cost * 0.5
        if irreversibility > Config.CO_DECISION_IRREVERSIBILITY_THRESHOLD:
            operator_scar_delta *= Config.CO_DECISION_DELEGATE_PENALTY
        self.scar_operator += operator_scar_delta

    def spawn_unknown_patterns(self, t, max_new=None):
        if max_new is None: max_new = Config.MAX_UNKNOWN_SPAWN_PER_STEP
        avg_unknown = safe_mean(self.field[:, :, CH['unknown']], 0)
        alive = [p for p in self.patterns if p.alive]
        normal_count = len([p for p in alive if p.role_type == "normal"])
        _cap = self._get_pop_cap()          # ОДИН раз: ниже цикл по 208x208 клеток
        if avg_unknown < Config.UNKNOWN_SPAWN_THRESHOLD: return
        if normal_count >= _cap:
            self._pop_note('blocked', 'unknown_cap')
            return
        base_prob = self.selfreg.get_unknown_spawn_prob()
        # Прежний множитель min(1, normal/cap) при cap=10M был ~0 (spawn_unknown почти не работал). Сохранено как
        # Config.UNKNOWN_SPAWN_POP_REF; реальный тормоз у потолка — _pop_birth_factor (плавно к 0).
        _ref = max(1, int(getattr(Config, 'UNKNOWN_SPAWN_POP_REF', 10_000_000)))
        _bf = self._pop_birth_factor(len(alive))
        prob = base_prob * (avg_unknown / Config.UNKNOWN_SPAWN_THRESHOLD) * min(1.0, normal_count / _ref) * _bf
        prob *= getattr(self, 'env_complexity', 1.0) if Config.ENABLE_RED_QUEEN else 1.0
        new_count = 0
        for x in range(Config.WORLD_SIZE):
            if new_count >= max_new or normal_count + new_count >= _cap: break
            for y in range(Config.WORLD_SIZE):
                if new_count >= max_new or normal_count + new_count >= _cap: break
                if self.field[x, y, CH['owner']] != 0: continue
                if self.field[x, y, CH['unknown']] > 0.5 and phi_hash(x, y, t) < prob:
                    if self.create_pattern({(x, y)}) is not None:
                        new_count += 1
                        self._pop_note('births', 'unknown')
                        self.field[x, y, CH['unknown']] *= 0.8

    def process_spores(self):
        alive = [p for p in self.patterns if p.alive]
        normal_count = len([p for p in alive if p.role_type == "normal"])
        _cap = self._get_pop_cap()
        _bf = self._pop_birth_factor(len(alive))
        for p in self.patterns[:]:
            if not p.alive or p.pending_spore is None: continue
            spore = p.pending_spore
            if not (0 <= spore['x'] < Config.WORLD_SIZE and 0 <= spore['y'] < Config.WORLD_SIZE):
                p.pending_spore = None; continue
            if self.field[spore['x'], spore['y'], CH['owner']] != 0: continue
            if normal_count >= _cap:
                self._pop_note('blocked', 'spore_cap')
                continue                      # спора остаётся в ожидании и попробует позже
            if phi_hash(self.age, p.id, 4242) >= _bf:
                self._pop_note('blocked', 'spore_brake')
                continue
            child = self.create_pattern({(spore['x'], spore['y'])}, parent=p)
            if child is None: continue
            normal_count += 1                 # раньше счётчик не рос -> за вызов могло родиться сколько угодно
            self._pop_note('births', 'spore')
            child.scar_dream = spore['scar_dream']
            if 'spore_concepts' in spore:
                child.concept_graph.nodes = {sig: mem_inherit_stamp(dict(data), data) for sig, data in spore['spore_concepts'].items()}
            child.unresolved_contradiction = spore.get('unresolved_contradiction', child.unresolved_contradiction)
            child.epistemic_scar = spore.get('epistemic_scar', child.epistemic_scar)
            child._log_event("spore_inherited_scars_only", count=len(spore.get('spore_concepts', {})), parent=p.id)
            _inherited_rq = spore.get('root_question')
            if _inherited_rq and child._root_question is None:
                child._root_question = _inherited_rq
                child._root_question_inherited = True
                child._log_event("root_question_inherited", question=_inherited_rq[:60], parent=p.id)
            if 'binding_seed' in spore: child._binding_seed = spore['binding_seed']
            if 'divergence_key' in spore: child.divergence_key = spore['divergence_key']
            self.spore_journal.append({
                't': self.age, 'parent': p.id, 'child': child.id,
                'inherited_scars': len(spore.get('spore_concepts', {})),
                'scar_dream': round(spore.get('scar_dream', 0.0), 4),
                'binding_seed': round(spore.get('binding_seed', 0.0), 3),
                'divergence_key': round(spore.get('divergence_key', Config.DEFAULT_DIVERGENCE_KEY), 3),
                'parent_lineage': spore.get('parent_lineage'),
            })
            p.pending_spore = None

    def resolve_competitions_spatial(self, max_dist=None):
        if max_dist is None: max_dist = Config.COMPETITION_MAX_DISTANCE
        alive = [p for p in self.patterns if p.alive and p.cells]
        if len(alive) < 2: return
        cell_size = max(1, max_dist)
        spatial_map = defaultdict(list)
        for p in alive:
            cx = sum(c[0] for c in p.cells) // len(p.cells)
            cy = sum(c[1] for c in p.cells) // len(p.cells)
            spatial_map[(cx // cell_size, cy // cell_size)].append(p)
        checked_pairs = set()
        for (gx, gy), group in spatial_map.items():
            for dx in [-1, 0, 1]:
                for dy in [-1, 0, 1]:
                    neighbor_key = (gx + dx, gy + dy)
                    if neighbor_key not in spatial_map: continue
                    others = spatial_map[neighbor_key]
                    for p1 in group:
                        for p2 in others:
                            if p1.id >= p2.id: continue
                            pair = (p1.id, p2.id)
                            if pair in checked_pairs: continue
                            if not p1.cells & p2.cells: continue
                            checked_pairs.add(pair)
                            p1.compete(p2, self.field)

    def detect_new_components(self, min_cells=None):
        if min_cells is None: min_cells = Config.MIN_COMPONENT_CELLS
        energy = np.nan_to_num(self.field[:, :, CH['energy']], nan=0.0)
        mask = energy > Config.PATTERN_ENERGY_THRESHOLD
        if self._wall_on(): mask[Config.CENTER_WALL_X, :] = False
        labeled, num = label(mask)
        slices = find_objects(labeled)
        components = []
        for i, slc in enumerate(slices, 1):
            if slc is None: continue
            region = labeled[slc] == i
            coords = np.argwhere(region)
            if len(coords) < min_cells: continue
            cells = {(slc[0].start + c[0], slc[1].start + c[1]) for c in coords}
            components.append(cells)
        return components

    def spawn_from_unclaimed_field_energy(self, t):
        if not getattr(Config, 'ENABLE_FIELD_EMERGENCE', True): return
        interval = getattr(Config, 'FIELD_EMERGENCE_INTERVAL', 40)
        if t % interval != 0: return
        components = self.detect_new_components()
        if not components: return
        alive = [p for p in self.patterns if p.alive]
        normal_cnt = len([p for p in alive if p.role_type == "normal"])
        max_spawn_cells = self.selfreg.get_max_cells_per_agent()
        _cap = self._get_pop_cap()
        _bf = self._pop_birth_factor(len(alive))
        _spawned = 0
        _max_call = int(getattr(Config, 'FIELD_EMERGENCE_MAX_PER_CALL', 10))   # раньше лимита за вызов не было: один t%40==0 мог родить сотни
        for cells in components:
            if len(cells) > max_spawn_cells: continue
            if normal_cnt >= _cap:
                self._pop_note('blocked', 'field_cap')
                break
            if _spawned >= _max_call:
                self._pop_note('blocked', 'field_call_limit')
                break
            if phi_hash(t, len(cells), 5151) >= _bf:
                self._pop_note('blocked', 'field_brake')
                continue
            best_ov = 0
            for p in alive:
                ov = len(cells & p.cells)
                if ov > best_ov:
                    best_ov = ov
                    if best_ov > len(cells) * 0.3: break
            if best_ov > len(cells) * 0.3: continue
            p = self.create_pattern(cells)
            if p is None: continue
            p._log_event("spontaneous_field_emergence")
            self.witness.record(p.id, "spontaneous_field_emergence", cells=len(cells))
            alive.append(p); normal_cnt += 1
            self._pop_note('births', 'field')
            _spawned += 1

    def _conceptual_resonance_step(self, t):
        if not getattr(Config, 'ENABLE_CONCEPTUAL_RESONANCE', True): return
        alive_patterns = [p for p in self.patterns if p.alive]
        if Config.DISORGANIZER_RESONANCE_BLOCK:
            alive_patterns = [p for p in alive_patterns if p.role_type != "disorganizer"]
        if len(alive_patterns) < 2: return
        cell_size = 10
        spatial_map = defaultdict(list)
        for p in alive_patterns:
            cx = sum(c[0] for c in p.cells) // len(p.cells)
            cy = sum(c[1] for c in p.cells) // len(p.cells)
            spatial_map[(cx // cell_size, cy // cell_size)].append(p)
        checked_pairs = set()
        for (gx, gy), group in spatial_map.items():
            for dx in [-1, 0, 1]:
                for dy in [-1, 0, 1]:
                    neighbor_key = (gx + dx, gy + dy)
                    if neighbor_key not in spatial_map: continue
                    others = spatial_map[neighbor_key]
                    for p1 in group:
                        for p2 in others:
                            if p1.id >= p2.id: continue
                            pair = (p1.id, p2.id)
                            if pair in checked_pairs: continue
                            checked_pairs.add(pair)
                            lineage_bonus = 0.05 if p1.lineage_id == p2.lineage_id else 0.2
                            bonus = p1.check_conceptual_resonance(p2) * lineage_bonus
                            if bonus > 0.01:
                                old_trust_1 = p1.trust_ledger.get(p2.id) or 0.5
                                old_trust_2 = p2.trust_ledger.get(p1.id) or 0.5
                                p1.trust_ledger.entries[p2.id] = min(1.0, old_trust_1 + bonus)
                                p2.trust_ledger.entries[p1.id] = min(1.0, old_trust_2 + bonus)

    def _process_archive_autonomy(self, t):
        alive = [p for p in self.patterns if p.alive]
        deposited_count = 0
        for p in alive:
            if p.event_counts.get('arc_completed', 0) > 0 and p.age % 50 == 0:
                if self.archive.deposit(p, "arc_completed", weight=1.0, text=f"Arc completed: {p.semantic_state} | soul={p.soul_weight:.2f}"):
                    deposited_count += 1
            if p.event_counts.get('fold', 0) > 0 and p.age % 40 == 0:
                if self.archive.deposit(p, "fold", weight=1.2, text=f"Fold: {p.semantic_state} | soul={p.soul_weight:.2f}"):
                    deposited_count += 1
            if p.event_counts.get('redeemed', 0) > 0 and p.age % 60 == 0:
                if self.archive.deposit(p, "redemption", weight=1.5, text=f"Redemption: {p.semantic_state} | soul={p.soul_weight:.2f}"):
                    deposited_count += 1
            if p.event_counts.get('deep_exchange', 0) > 0 and p.age % 80 == 0:
                if self.archive.deposit(p, "deep_exchange", weight=0.75, text="Conceptual resonance with neighbor"):
                    deposited_count += 1
            if p.coherence > 0.70 and p.soul_weight > 0.50 and p.age % 100 == 0:
                if self.archive.deposit(p, "coherence_peak", weight=0.65, text=str(p.last_phenomenal_report)[:140]):
                    deposited_count += 1
            if p.age > 200 and p.age % 150 == 0 and p.coherence > 0.55:
                if self.archive.deposit(p, "life_pulse", weight=0.55, text=f"t={t} | {p.semantic_state}/{getattr(p,'_substate','?')} | s={p.soul_weight:.2f} g={p.emotional_memory['gratitude']:.2f}"):
                    deposited_count += 1
        if t % 200 == 0:
            echoes = self.archive.get_recent_echoes(limit=3, min_weight=0.50)
            for echo in echoes:
                myth = {"arc": "archive_echo",
                        "emotional_scent": {"grief": echo['grief'], "gratitude": echo['grat']},
                        "phenomenal_essence": echo['weight'], "intensity": echo['weight'] * 0.8,
                        "excerpt": echo.get('excerpt', '')[:80]}
                self.cultural_memory.myth_pool.append(myth)
                if len(self.cultural_memory.myth_pool) > getattr(Config, 'MYTH_POOL_SIZE', 100):
                    self.cultural_memory.myth_pool.pop(0)
        if t % 200 == 0:
            priority_events = ['human_concept_injected', 'human_question_witness', 'love_concept',
                               'shared_attention', 'intentionality', 'trust', 'empathy', 'recursion',
                               'cooperation_norms', 'agency', 'consent', 'silence', 'witness_respect']
            existing = {s.get('event') for s in self.archive.write_queue}
            import time as _time
            for event in priority_events:
                if event not in existing:
                    self.archive.write_queue.appendleft({
                        "id": 777000 + _stable_hash(event) % 1000, "t": t + 50000, "world_t": t,
                        "event": event, "weight": 7.0, "soul": 1.0, "coherence": 1.0, "gap": 0.35,
                        "substate": "curious", "grief": 0.0, "grat": 0.85,
                        "excerpt": f"Вечный концепт: {event}", "created_at": _time.time()
                    })
            for scroll in self.archive.write_queue:
                if scroll.get('event') in priority_events:
                    scroll['t'] = t + 50000; scroll['world_t'] = t
        if t % 200 == 0 or len(self.archive.write_queue) >= 50:
            self.archive.flush_to_drive(min_batch=5)

    def _ledger_add(self, name, delta, calls=1):
        if not self._ledger_window:
            _orow = self._ledger_outside.setdefault(name, [0.0, 0.0, 0])
            if delta >= 0.0: _orow[0] += delta
            else: _orow[1] += delta
            _orow[2] += calls
            self._outside_acc1 += delta
            return
        row = self._energy_ledger['src'].setdefault(name, [0.0, 0.0, 0])
        if delta >= 0.0: row[0] += delta
        else: row[1] += delta
        row[2] += calls
        self._ledger_tick_measured += delta

    def _ledger_add_x2(self, name, delta, calls=1, count_in_total=True):
        if not self._ledger_window:
            _orow = self._x2_outside.setdefault(name, [0.0, 0.0, 0])
            if delta >= 0.0: _orow[0] += delta
            else: _orow[1] += delta
            _orow[2] += calls
            if count_in_total: self._outside_acc2 += delta
            return
        row = self._energy_ledger['x2'].setdefault(name, [0.0, 0.0, 0])
        if delta >= 0.0: row[0] += delta
        else: row[1] += delta
        row[2] += calls
        if count_in_total: self._x2_tick_measured += delta

    @contextmanager
    def _energy_span(self, name):
        e0 = float(self.field[:, :, CH['energy']].sum(dtype=np.float64))
        self._span_stack.append(0.0)
        try:
            yield
        finally:
            e1 = float(self.field[:, :, CH['energy']].sum(dtype=np.float64))
            child = self._span_stack.pop()
            incl = e1 - e0
            self._ledger_add(name, incl - child)
            if self._span_stack: self._span_stack[-1] += incl

    def _wall_cycle_want(self, t):
        on_n = int(Config.WALL_CYCLE_ON_TICKS)
        off_n = int(Config.WALL_CYCLE_OFF_TICKS)
        period = on_n + off_n
        if Config.WALL_CYCLE_FIRST == 'on':
            phase = t % period
        elif Config.WALL_CYCLE_FIRST == 'off':
            phase = (t + on_n) % period
        else:
            raise ValueError(f"WALL_CYCLE_FIRST: ожидалось 'on' или 'off', получено {Config.WALL_CYCLE_FIRST!r}")
        return phase < on_n

    def _wall_line_census(self):
        wx = Config.CENTER_WALL_X
        agents = 0
        cells = 0
        for p in self.patterns:
            if not p.alive:
                continue
            n = sum(1 for c in p.cells if c[0] == wx)
            if n:
                agents += 1
                cells += n
        return agents, cells

    def _wall_cycle_update(self, t):
        if not Config.WALL_CYCLE_ENABLED:
            return
        want = self._wall_cycle_want(t)
        if want == bool(Config.ENABLE_CENTER_WALL):
            return
        agents_before, cells_before = self._wall_line_census()
        d = self._wall_diag
        mig0 = (d['migrated_dropped_cells'], d['migrated_shifted_cells'], d['migrated_kept_cells'])
        stale_cleared = 0
        Config.ENABLE_CENTER_WALL = want
        if want:
            self._migrate_wall_cells()
            wx = Config.CENTER_WALL_X
            for y in range(Config.WORLD_SIZE):
                oid = int(self.field[wx, y, CH['owner']])
                if oid != 0:
                    owner = self.pattern_dict.get(oid)
                    if owner is None or not owner.alive or (wx, y) not in owner.cells:
                        self.field[wx, y, CH['owner']] = 0
                        stale_cleared += 1
        agents_after, cells_after = self._wall_line_census()
        alive = [p for p in self.patterns if p.alive]
        ev = {'t': t, 'to': 'ON' if want else 'OFF', 'pop': len(alive),
              'agents_on_line_before': agents_before, 'cells_on_line_before': cells_before,
              'agents_on_line_after': agents_after, 'cells_on_line_after': cells_after,
              'migrated_dropped': d['migrated_dropped_cells'] - mig0[0],
              'migrated_shifted': d['migrated_shifted_cells'] - mig0[1],
              'migrated_kept': d['migrated_kept_cells'] - mig0[2],
              'stale_owner_cleared': stale_cleared}
        self._wall_events.append(ev)
        print(f"[t={t}] 🧱 СТЕНА → {ev['to']}: pop={ev['pop']} на линии до/после: агентов {agents_before}/{agents_after}, "
              f"клеток {cells_before}/{cells_after}; миграция dropped/shifted/kept="
              f"{ev['migrated_dropped']}/{ev['migrated_shifted']}/{ev['migrated_kept']}; сброшено «призрачных» owner={stale_cleared}")

    def _given_flux_mode(self):
        mode = Config.GIVEN_FLUX_MODE
        if mode not in ('off', 'additive', 'replace'):
            raise ValueError(f"GIVEN_FLUX_MODE: ожидалось 'off', 'additive' или 'replace', получено {mode!r}")
        return mode

    def _given_flux_map(self):
        W = int(Config.WORLD_SIZE)
        x0 = int(Config.CENTER_WALL_X) if Config.GIVEN_FLUX_X is None else int(Config.GIVEN_FLUX_X)
        y0 = W // 2 if Config.GIVEN_FLUX_Y is None else int(Config.GIVEN_FLUX_Y)
        sigma = float(Config.GIVEN_FLUX_RADIUS)
        if sigma <= 0.0:
            raise ValueError(f"GIVEN_FLUX_RADIUS: ожидалось > 0, получено {sigma!r}")
        key = (W, x0, y0, sigma)
        if self._sun_map_key != key:
            ix = np.arange(W)
            ax = np.abs(ix - x0); ay = np.abs(ix - y0)
            dx = np.minimum(ax, W - ax).astype(np.float64)
            dy = np.minimum(ay, W - ay).astype(np.float64)
            g = np.exp(-(dx[:, None] ** 2 + dy[None, :] ** 2) / (2.0 * sigma ** 2))
            self._sun_map = g / g.sum(dtype=np.float64)
            self._sun_map_key = key
        return self._sun_map

    def _given_flux_apply(self, t):
        sun_map = self._given_flux_map()
        n_phase = float(self._sun_rng.standard_normal())
        n_amp = float(self._sun_rng.standard_normal())
        period = float(Config.GIVEN_FLUX_PERIOD)
        if period <= 0.0:
            raise ValueError(f"GIVEN_FLUX_PERIOD: ожидалось > 0, получено {period!r}")
        phase = t / period + float(Config.GIVEN_FLUX_NOISE_PHASE) * n_phase
        cyc = Config.GIVEN_FLUX_CYCLE
        if cyc == 'sine':
            f = 1.0 + float(np.sin(2.0 * np.pi * phase))
        elif cyc == 'step':
            frac = float(Config.GIVEN_FLUX_DAY_FRACTION)
            if not (0.0 < frac <= 1.0):
                raise ValueError(f"GIVEN_FLUX_DAY_FRACTION: ожидалось в (0, 1], получено {frac!r}")
            f = (1.0 / frac) if (phase % 1.0) < frac else 0.0
        elif cyc == 'const':
            f = 1.0
        else:
            raise ValueError(f"GIVEN_FLUX_CYCLE: ожидалось 'sine', 'step' или 'const', получено {cyc!r}")
        amp = max(0.0, 1.0 + float(Config.GIVEN_FLUX_NOISE_AMP) * n_amp)
        intended = float(Config.GIVEN_FLUX_POWER) * f * amp
        E = self.field[:, :, CH['energy']]
        with self._energy_span('X1.Given_Flux'):
            _b = float(E.sum(dtype=np.float64))
            E += sun_map * intended
            measured = float(E.sum(dtype=np.float64)) - _b
        self._given_flux_tick += measured
        self._given_flux_intended_total += intended
        self._given_flux_measured_total += measured

    _BOUNDARY_LABELS = (
        ('sun', 'Солнце (Given_Flux)'),
        ('physical_dissipation', 'Физическая диссипация (X1 + X2 overflow)'),
        ('debt_written_off', 'Списанный отрицательный долг X2'),
        ('legacy_inflow', 'Прежний приток (инжекция, бонус занятым, импульс t%50)'),
        ('creation', 'Приход без установленного донора (нагрев ферала, бонусы X2)'),
        ('dissipation', 'Диссипация (метаболизм, цена действий, крик): объявленная стоимость'),
        ('x2_reset', 'Обнуление запаса X2 (collapse_reset; судьба энергии не установлена)'),
        ('field_physics', 'Физика поля (гравитация, шум Given-seed, остаток field_dynamics)'),
        ('regulators', 'Регуляторы и клипы'),
        ('x2_overwrite', 'Перезапись энергии X2 средним по полю (рождение/деление; путь не установлен)'),
        ('events', 'События (эхо, спасение, спавн, святилище, кожа)'),
        ('internal_x1', 'Перестройка внутри X1 (должна быть ≈ 0)'),
        ('internal_x2', 'Обмен агент↔агент внутри X2 (должен быть ≈ 0)'),
        ('transfer_leak', 'Расхождение передач X1↔X2 (+ появилось / − ушло наружу)'),
        ('unclassified', 'НЕ КЛАССИФИЦИРОВАНО'),
    )
    _BOUNDARY_DECLARED_IN = ('sun',)
    _BOUNDARY_DECLARED_OUT = ('dissipation', 'physical_dissipation', 'debt_written_off')
    _BOUNDARY_PAIRS = (
        ('коллапс: запас X2 возвращён в клетки', 'X1.collapse_return', 'X2.collapse_reset'),
        ('сбор излишка поля агентом', 'X1.field_to_agent_harvest', 'X2.field_to_agent_harvest'),
        ('ферал впитывает у жертвы', 'feral.prey_drain_FIELD_TO_VICTIM', 'X2.feral_intent_absorbed'),
        ('бонус готовности к делению (из поля)', 'X1.division_ready_transfer', 'X2.division_ready_transfer'),
        ('смерть: прах в поле', 'X1.смерть: прах (запас мёртвого агента возвращён в поле его клеток)',
         'X2.смерть: запас покидает агента'),
        ('синхронизация X2 с клетками', 'X1.property_sync_return', 'X2.property_sync'),
    )
    _OVERFLOW_BACKED = frozenset(_p[2] for _p in _BOUNDARY_PAIRS)
    _BOUNDARY_EXACT = {
        'X1.Given_Flux': 'sun',
        'X1.inflow_legacy.base_injection': 'legacy_inflow',
        'X1.inflow_legacy.occupied_bonus': 'legacy_inflow',
        'X1.inflow_legacy.impulse_t50': 'legacy_inflow',
        'feral.edge_heating_CREATION': 'creation',
        'X2.feral_bounty_CREATION': 'creation',
        'X2.love_bonus': 'creation',
        'X2.division_ready_bonus': 'creation',
        'X2.floor_raise_to_0.1': 'creation',
        'X2.debt_written_off': 'debt_written_off',
        'X2.overflow_dissipated': 'physical_dissipation',
        'X1.dissipation': 'physical_dissipation',
        'Pattern.metabolic_tax (налог с клеток)': 'dissipation',
        'apply_scream.energy_penalty': 'dissipation',
        'X2.collapse_reset': 'x2_reset',
        'X2.collapse_reset_unreturned': 'x2_reset',
        'emergency_rescue.field_cost': 'physical_dissipation',
        '_update_sanctuary.field_cost': 'physical_dissipation',
        'X2.смерть: запас рассеян (нет клеток для праха)': 'physical_dissipation',
        'X2.selfreg_trade_fee': 'dissipation',
        'X2.alarm_credit_in': 'internal_x2',
        'X2.alarm_credit_out': 'internal_x2',
        'X1.diffusion': 'internal_x1',
        'X1.field_dynamics.given_seed_noise_zero_sum': 'internal_x1',
        'X2.feral_grow_cost': 'dissipation',
        'X2.introspect_cost': 'dissipation',
        'X2.morse_transmit_cost': 'dissipation',
        'X2.selfreg_heal_cost': 'dissipation',
        'X2.simplification_cost': 'dissipation',
        'X2.signal_emit': 'dissipation',
        'X2.intent_action_scar_penalty': 'dissipation',
        'X1.field_dynamics.gravity': 'field_physics',
        'X1.field_dynamics.given_seed_noise': 'field_physics',
        'field_dynamics (без gulf/dark)': 'field_physics',
        '_conservation_check (гомеостат)': 'regulators',
        'X2.property_sync': 'internal_x2',
        'X1.property_sync_return': 'internal_x1',
        'echo_system.inject': 'events',
        'emergency_rescue': 'events',
        '_rare_crisis': 'events',
        'spawn_from_unclaimed_field_energy': 'events',
        'spawn_unknown_patterns': 'events',
        '_update_sanctuary': 'events',
        'regulate_skin.stab_to_field': 'physical_dissipation',
        '_apply_gulf_stream': 'internal_x1',
        '_apply_dark_energy_stream': 'internal_x1',
        'X1.dark_energy.redistribution': 'internal_x1',
        'ecosystem_pressure': 'internal_x1',
        'X2.neighbor_transfer_in': 'internal_x2',
        'X2.neighbor_transfer_out': 'internal_x2',
        'X2.selfreg_trade_in': 'internal_x2',
        'X2.selfreg_trade_out': 'internal_x2',
        'X2.feral_execute_kill_gain': 'internal_x2',
        'X2.feral_execute_victim_gives': 'internal_x2',
        'X2.scream_kill_reward': 'internal_x2',
        'X2.scream_kill_feral_loss': 'internal_x2',
        'X2.feral_meat_pays': 'internal_x2',
        'X2.feral_meat_reward': 'internal_x2',
        'X2.scream_kill_feral_rest_dissipated': 'physical_dissipation',
    }
    _BOUNDARY_PREFIX = (('X2.action_', 'dissipation'),)

    _BOUNDARY_FATE = {}
    _FATE_CLASSES = ('dissipation', 'x2_reset')

    @classmethod
    def _boundary_class(cls, name):
        k = cls._BOUNDARY_EXACT.get(name)
        if k is not None:
            return k
        for pre, k in cls._BOUNDARY_PREFIX:
            if name.startswith(pre):
                return k
        return 'unclassified'

    def audit_balance(self):
        L1 = self._energy_ledger['src']
        L2 = self._energy_ledger['x2']
        tab = {k: {'pos': 0.0, 'neg': 0.0, 'rows': {}} for k, _ in self._BOUNDARY_LABELS}
        transfers = []
        used = {'X1': set(), 'X2': set()}
        for label, n1, n2 in self._BOUNDARY_PAIRS:
            r1 = L1.get(n1)
            r2 = L2.get(n2)
            a = float(r1[0] + r1[1]) if r1 else 0.0
            b = float(r2[0] + r2[1]) if r2 else 0.0
            used['X1'].add(n1)
            used['X2'].add(n2)
            # overflow выше потолка X2 уже записан как X2.overflow_dissipated (физическая диссипация),
            # поэтому в «утечку» пары он не входит: иначе одна потеря считается дважды.
            _ov = min(float(self._overflow_by_src.get(n2, 0.0)), max(0.0, -(a + b)))
            gap = a + b + _ov
            transfers.append({'name': label, 'x1_net': a, 'x2_net': b, 'matched': min(abs(a), abs(b)),
                              'direction': 'X1→X2' if (b > 0.0 or (b == 0.0 and a < 0.0)) else 'X2→X1', 'gap': gap,
                              'overflow': _ov, 'overflow_raw': float(self._overflow_by_src.get(n2, 0.0))})
            t = tab['transfer_leak']
            if gap >= 0.0:
                t['pos'] += gap
            else:
                t['neg'] += gap
            if gap != 0.0:
                t['rows'][label] = gap
        for side, L in (('X1', L1), ('X2', L2)):
            for name, v in L.items():
                if name in used[side]:
                    continue
                c = tab[self._boundary_class(name)]
                c['pos'] += float(v[0])
                c['neg'] += float(v[1])
                c['rows'][f'{side}:{name}'] = float(v[0] + v[1])
        classes = {}
        in_ext = out_ext = 0.0
        d_in = d_out = u_in = u_out = 0.0
        for key, label in self._BOUNDARY_LABELS:
            c = tab[key]
            net = c['pos'] + c['neg']
            if key in ('internal_x1', 'internal_x2'):
                i_, o_ = max(net, 0.0), max(-net, 0.0)
            else:
                i_, o_ = c['pos'], -c['neg']
            in_ext += i_
            out_ext += o_
            if key in self._BOUNDARY_DECLARED_IN:
                d_in += i_; u_out += o_
            elif key in self._BOUNDARY_DECLARED_OUT:
                d_out += o_; u_in += i_
            else:
                u_in += i_; u_out += o_
            classes[key] = {'label': label, 'in': i_, 'out': o_, 'net': net,
                            'top': [[n, v] for n, v in sorted(c['rows'].items(), key=lambda kv: -abs(kv[1]))[:5]]}
        unpaid = []
        for name, v in L2.items():
            if name in used['X2'] or self._boundary_class(name) == 'internal_x2' or v[0] <= 0.0:
                continue
            m = self._energy_memo.get(name)
            clipped = float(m[0]) if m else 0.0
            unpaid.append({'name': name, 'class': self._boundary_class(name), 'actual': float(v[0]),
                           'offered': float(v[0]) + clipped, 'clipped_by_x2': clipped, 'calls': int(v[2])})
        unpaid.sort(key=lambda r: -r['actual'])
        unpaid_sum = float(sum(r['actual'] for r in unpaid))
        from_x1 = float(sum(tr['x2_net'] for tr in transfers if tr['x2_net'] > 0.0))
        x2_paths = []
        for _nm in ('X2.division_ready_bonus', 'X2.love_bonus', 'X2.update_properties_overwrite'):
            _v = L2.get(_nm)
            _m = self._energy_memo.get(_nm)
            x2_paths.append({'name': _nm, 'class': self._boundary_class(_nm),
                             'pos': float(_v[0]) if _v else 0.0, 'neg': float(_v[1]) if _v else 0.0,
                             'calls': int(_v[2]) if _v else 0, 'clipped_by_x2': float(_m[0]) if _m else 0.0,
                             'paired_x1': _nm in used['X2']})
        x2_unpaid = {'rows': unpaid, 'paths': x2_paths, 'actual_sum': unpaid_sum, 'from_x1': from_x1,
                     'share_of_external_income': (unpaid_sum / (unpaid_sum + from_x1)) if (unpaid_sum + from_x1) > 0.0 else 0.0}
        rows = self._audit_rows
        dE_total = float(self._energy_ledger['step_total']) + float(sum(r['dE2'] for r in rows))
        resid = float(self._res_x1['sum'] + self._res_x2['sum'])
        fate = {'registered_cost': 0.0, 'reset': 0.0, 'known': 0.0, 'unknown': 0.0, 'rows': []}
        for _side, _Lx in (('X1', L1), ('X2', L2)):
            for _nm, _v in _Lx.items():
                if _nm in used[_side]:
                    continue
                _k = self._boundary_class(_nm)
                if _k not in self._FATE_CLASSES:
                    continue
                _out = -float(_v[1])
                if _out <= 0.0:
                    continue
                fate['reset' if _k == 'x2_reset' else 'registered_cost'] += _out
                _f = self._BOUNDARY_FATE.get(_nm, 'unknown')
                fate['known' if _f != 'unknown' else 'unknown'] += _out
                fate['rows'].append((f'{_side}:{_nm}', _k, _f, _out))
        fate['rows'].sort(key=lambda r: -r[3])
        stock = None
        if rows:
            _r0, _r1 = rows[0], rows[-1]
            _e_init = float((_r0['E1'] - _r0['dE1']) + (_r0['E2'] - _r0['dE2']))
            _e_final = float(_r1['E1'] + _r1['E2'])
            _d_rows = float(sum(r['dE1'] + r['dE2'] for r in rows))
            stock = {'E_init': _e_init, 'E_final': _e_final, 'dE_ends': _e_final - _e_init,
                     'dE_rows': _d_rows, 'gap': (_e_final - _e_init) - _d_rows}
        _sun_in = float(classes['sun']['in'])
        _diss_out = float(classes['physical_dissipation']['out'] + classes['dissipation']['out'])
        _debt_signed = float(classes['debt_written_off']['net'])
        _debt_out = max(0.0, -_debt_signed)
        _physical_balance = _sun_in - _diss_out - _debt_out
        _actual_delta = float(stock['dE_ends']) if stock is not None else float('nan')
        _physical_error = _actual_delta - _physical_balance if stock is not None else float('nan')
        _nonphysical = {k: float(v['net']) for k, v in classes.items()
                        if k not in ('sun', 'dissipation', 'physical_dissipation', 'debt_written_off')
                        and abs(float(v['net'])) > 1e-6}
        return {'ticks': len(rows), 'mode': Config.GIVEN_FLUX_MODE, 'classes': classes, 'transfers': transfers,
                'In_external': in_ext, 'Out_external': out_ext,
                'In_declared': d_in, 'Out_declared': d_out, 'In_sun': _sun_in,
                'physical_balance': _physical_balance, 'actual_delta_E_total': _actual_delta,
                'physical_balance_error': _physical_error, 'nonphysical_net_rows': _nonphysical,
                'physical_balance_pass': bool(stock is not None and abs(_physical_error) < 1.0 and not _nonphysical),
                'In_unestablished': u_in, 'Out_unestablished': u_out, 'stock': stock, 'fate': fate, 'x2_unpaid': x2_unpaid, 'residual': resid, 'residual_x1': float(self._res_x1['sum']), 'residual_x2': float(self._res_x2['sum']), 'dE_total': dE_total,
                'closure_error': dE_total - (in_ext - out_ext) - resid,
                'given_flux': {'mode': Config.GIVEN_FLUX_MODE, 'intended_total': self._given_flux_intended_total,
                               'measured_total': self._given_flux_measured_total,
                               'diff': self._given_flux_measured_total - self._given_flux_intended_total,
                               'power': Config.GIVEN_FLUX_POWER, 'cycle': Config.GIVEN_FLUX_CYCLE,
                               'period': Config.GIVEN_FLUX_PERIOD, 'radius': Config.GIVEN_FLUX_RADIUS},
                'outside_window': {
                    'x1': float(sum(v[0] + v[1] for v in self._ledger_outside.values())),
                    'x2': float(sum(v[0] + v[1] for v in self._x2_outside.values()))}}

    def _print_boundary_report(self, S):
        B = S['balance']
        if not B['ticks']:
            return
        G = B['given_flux']
        print(f"\n  ── ГРАНИЦА X1+X2: потоки по классам; внешний обмен подтверждён только для Солнца (тиков: {B['ticks']}) ──")
        print(f"  Given_Flux: режим={G['mode']}; Σ намеренного={G['intended_total']:.6f}, Σ измеренного={G['measured_total']:.6f}, "
              f"разница={G['diff']:+.2e}" + (f"; мощность={G['power']}, цикл={G['cycle']}, период={G['period']}, σ={G['radius']}"
                                           if G['mode'] != 'off' else ''))
        print(f"  PHYSICAL BALANCE: Sun={B.get('In_sun', 0.0):+.2f} - Dissipation={(B.get('classes', {}).get('physical_dissipation', {}).get('out', 0.0) + B.get('classes', {}).get('dissipation', {}).get('out', 0.0)):+.2f} - DebtWrittenOff={B.get('classes', {}).get('debt_written_off', {}).get('out', 0.0):+.2f} = {B.get('physical_balance', float('nan')):+.2f}")
        print(f"  ΔE_total={B.get('actual_delta_E_total', float('nan')):+.2f}; error={B.get('physical_balance_error', float('nan')):+.2f}; PASS={B.get('physical_balance_pass', False)}")
        if B.get('nonphysical_net_rows'):
            print(f"  BLOCK: non-physical net flows remain: {B['nonphysical_net_rows']}")
        print(f"  {'класс потока':64s} {'вход':>11s} {'выход':>11s} {'нетто':>11s}")
        for key, label in self._BOUNDARY_LABELS:
            k = B['classes'][key]
            if key != 'unclassified' and abs(k['in']) < 5e-10 and abs(k['out']) < 5e-10 and key not in ('sun', 'legacy_inflow', 'internal_x1', 'internal_x2'):
                continue
            if key == 'unclassified' and not k['top']:
                continue
            print(f"  {label[:64]:64s} {k['in']:+11.2f} {-k['out']:+11.2f} {k['net']:+11.2f}")
            if key in ('unclassified', 'transfer_leak', 'internal_x1', 'internal_x2', 'creation', 'field_physics', 'regulators', 'x2_overwrite', 'x2_reset', 'events'):
                for n, v in k['top'][:3]:
                    if abs(v) >= 5e-10:
                        print(f"      · {n[:70]:70s} {v:+11.2f}")
        print("  Передачи X1↔X2 внутри границы (совпавшая часть) и расхождение пары:")
        for tr in B['transfers']:
            print(f"      {tr['name'][:40]:40s} {tr['direction']}: X1 {tr['x1_net']:+.2f}, X2 {tr['x2_net']:+.2f}, "
                  f"совпало {tr['matched']:.2f}, overflow→диссипация {tr.get('overflow', 0.0):.2f}, расхождение {tr['gap']:+.4f}")
        # [v6] Честная сверка overflow. audit_balance режет overflow пары до фактической потери пары (min), и это может
        # спрятать расхождение. Три числа рядом: что записано в ledger, что накоплено по источникам, что подтверждено парами.
        try:
            _lv = self._energy_ledger['x2'].get('X2.overflow_dissipated')
            _led_ov = -float(_lv[0] + _lv[1]) if _lv else 0.0
            _raw_by = {k: float(v) for k, v in self._overflow_by_src.items() if abs(float(v)) > 0.0}
            _raw_sum = sum(_raw_by.values())
            _conf_sum = sum(float(tr.get('overflow', 0.0)) for tr in B['transfers'])
            print(f"  [overflow-сверка] ledger X2.overflow_dissipated={_led_ov:.4f} | Σ по источникам={_raw_sum:.4f} | подтверждено списанием из X1 (пары)={_conf_sum:.4f}")
            print(f"      ledger − по источникам = {_led_ov - _raw_sum:+.4f}  (≠0: overflow попал в ledger мимо учёта по источникам)")
            print(f"      по источникам − подтверждено = {_raw_sum - _conf_sum:+.4f}  (≠0: записан сгоревшим, но из X1 не списан)")
            for (_lab, _n1, _n2), _tr in zip(self._BOUNDARY_PAIRS, B['transfers']):
                _r = float(_tr.get('overflow_raw', _raw_by.get(_n2, 0.0)))
                _c = float(_tr.get('overflow', 0.0))
                if _r > 0.0 or _c > 0.0:
                    _flag = '  ⚠️ избыток' if abs(_r - _c) > 5e-4 else ''
                    print(f"      {_lab[:40]:40s} записано={_r:.4f} подтверждено={_c:.4f} избыток={_r - _c:+.4f}{_flag}")
            _paired = set(_p[2] for _p in self._BOUNDARY_PAIRS)
            _other = {k: round(v, 4) for k, v in _raw_by.items() if k not in _paired}
            if _other:
                print(f"      overflow без пары: {_other}")
        except Exception as _e:
            print(f"  [overflow-сверка] не удалось посчитать: {_e!r}")
        print(f"  Объявлено по построению: приток (Солнце + прежний) {B['In_declared']:+.2f}, из них Солнце {B['In_sun']:+.2f}; "
              f"выход (диссипация) {-B['Out_declared']:+.2f}")
        print(f"  БЕЗ установленного механизма пересечения границы: приход {B['In_unestablished']:+.2f}, "
              f"уход {-B['Out_unestablished']:+.2f}  (знаковые изменения классов; внешним обменом не названы)")
        print(f"  Σ знаковых изменений по классам (прежде In_external/Out_external): {B['In_external']:+.2f} / {-B['Out_external']:+.2f}  "
              f"(нетто {B['In_external'] - B['Out_external']:+.2f})  невязка Σ={B['residual']:+.3f} (X1 {B['residual_x1']:+.3f}, X2 {B['residual_x2']:+.3f})  "
              f"ΣΔE(X1+X2)={B['dE_total']:+.2f}  замыкание таблицы={B['closure_error']:+.2e}")
        if B.get('stock'):
            K = B['stock']
            print(f"  Запасы X1+X2 на окне (согласованность снимков, не доказательство полноты учёта): начало={K['E_init']:.2f} конец={K['E_final']:.2f} Δ по концам={K['dE_ends']:+.2f} "
                  f"ΣΔE по тикам={K['dE_rows']:+.2f} разрыв={K['gap']:+.2e}")
        F = B.get('fate')
        if F and (F['registered_cost'] > 0.0 or F['reset'] > 0.0):
            print(f"  Судьба списанной энергии (диагностика, баланс не меняет): зарегистрированный расход {F['registered_cost']:.2f} "
                  f"+ обнуление {F['reset']:.2f}; судьба установлена {F['known']:.2f}, не установлена {F['unknown']:.2f}")
            for _n, _k, _f, _o in F['rows'][:3]:
                print(f"      · {_n[:60]:60s} {_o:10.2f}  [{_k}; судьба: {_f}]")
        U = B['x2_unpaid']
        if U['rows']:
            print(f"  X2: приход без установленной оплаты или донора (причина не установлена): Σ={U['actual_sum']:+.2f} = "
                  f"{U['share_of_external_income']:.1%} внешнего дохода X2 (из X1 пришло {U['from_x1']:+.2f})")
            print(f"      {'строка':40s} {'факт':>9s} {'предложено':>11s} {'срезал клип X2':>15s} {'вызовов':>8s}")
            for r in U['rows'][:6]:
                print(f"      {r['name'][:40]:40s} {r['actual']:+9.2f} {r['offered']:+11.2f} {r['clipped_by_x2']:+15.2f} {r['calls']:8d}")
        if U.get('paths'):
            print("  Путь X2 для трёх строк (в таблице выше они считаются по классам; здесь — приход/расход/вызовы):")
            for r in U['paths']:
                print(f"      {r['name'][:34]:34s} +{r['pos']:9.2f} {r['neg']:+10.2f} вызовов {r['calls']:6d}  "
                      f"срезал клип {r['clipped_by_x2']:+8.2f}  пара X1: {'есть' if r['paired_x1'] else 'нет'}  класс: {r['class']}")
        ow = B['outside_window']
        if abs(ow['x1']) > 1e-9 or abs(ow['x2']) > 1e-9:
            print(f"  вне окна тика (инициализация/между шагами): X1 {ow['x1']:+.2f}, X2 {ow['x2']:+.2f} — в таблицу не входит")

    def _audit_init(self):
        def _blank():
            return {'pos': 0, 'neg': 0, 'sum': 0.0, 'abs': 0.0, 'max_abs': 0.0, 'max_t': -1, 'n': 0}
        self._energy_memo = {}
        self._overflow_by_src = {}
        self._sun_rng = np.random.RandomState(int(Config.GIVEN_FLUX_SEED))
        self._sun_map = None
        self._sun_map_key = None
        self._given_flux_tick = 0.0
        self._given_flux_intended_total = 0.0
        self._given_flux_measured_total = 0.0
        self._x2_tick_measured = 0.0
        self._memo_tick = Counter()
        self._tick_acc = Counter()
        self._res_x1 = _blank()
        self._res_x2 = _blank()
        self._audit_rows = []
        self._feral_stats = Counter()
        self._feral_sanctuary_audit = Counter()
        self._attack_log = []
        self._dlg_ledger = Counter()
        self._dlg_log = []
        self._dlg_pairs = {}
        self._audit_prev_next_id = None
        self._mode_snapshots = []
        self._wall_events = []
        self._om_stats = Counter()
        self._om_by_word = {}
        self._om_by_ctx = {}
        self._om_choices = Counter()
        self._ledger_window = False
        self._ledger_outside = {}
        self._x2_outside = {}
        self._outside_acc1 = 0.0
        self._outside_acc2 = 0.0
        self._prev_E1_end = None
        self._prev_E2_end = None
        self._gap_x1 = _blank()
        self._gap_x2 = _blank()

    def _memo_add(self, name, lost):
        row = self._energy_memo.setdefault(name, [0.0, 0.0, 0])
        if lost >= 0.0:
            row[0] += lost
        else:
            row[1] += lost
        row[2] += 1
        self._memo_tick[name] += lost

    def _x2_sum(self):
        return float(sum(float(p.energy) for p in self.patterns))

    def _feral_attack_record(self, rec):
        st = self._feral_stats
        key = 'hits' if rec['hit'] else 'misses'
        st['rolled'] += 1
        st[key] += 1
        st['raw_sum'] += rec['raw']
        st['p_sum'] += rec['kill_prob']
        if rec['raw'] >= 0.95:
            st['raw_ge_095'] += 1
        acc = self._tick_acc
        acc['rolled'] += 1
        acc[key] += 1
        acc['p_sum'] += rec['kill_prob']
        acc['raw_sum'] += rec['raw']
        if len(self._attack_log) < Config.FERAL_ATTACK_LOG_MAX:
            self._attack_log.append(rec)
        else:
            st['log_overflow'] += 1

    def _dialogue_record(self, listener_id, speaker_id, utterance, sim, interp):
        c = self._dlg_ledger
        emotion, intent = utterance[1], utterance[2]
        token = utterance[3] if len(utterance) > 3 and utterance[3] != 'ANSWER' else None
        pair = (speaker_id, listener_id)
        seen = self._dlg_pairs.get(pair, 0)
        self._dlg_pairs[pair] = seen + 1
        c['heard'] += 1
        c['first_encounter' if seen == 0 else 'repeat_encounter'] += 1
        c['emotion:' + emotion] += 1
        c['intent:' + intent] += 1
        if token is not None:
            c['token:' + token] += 1
        if interp['seek'] != 'none':
            c['seek'] += 1
            c['seek_' + interp['seek']] += 1
        if interp['answer']:
            c['answer_heard'] += 1
            if interp['relief_applied']:
                c['relief'] += 1
                c['relief_sum'] += interp['relief']
        if interp['low_sim']:
            c['low_sim'] += 1
        dg = interp['grief2'] - interp['grief0']
        dt = interp['grat2'] - interp['grat0']
        c['d_grief_sum'] += dg
        c['d_grief_abs'] += abs(dg)
        c['d_grat_sum'] += dt
        c['d_grat_abs'] += abs(dt)
        # допуск 1e-9: сдвиг клипуется ровно в CAP, и float-округление (g+cap)-g давало «превышение» почти на каждой
        # реплике, упёршейся в лимит. relief ANSWER не клипуется к CAP по замыслу — считаем его отдельно.
        if abs(dg) > Config.EMOTIONAL_CAP_RATE + 1e-9:
            c['d_grief_gt_tick_cap'] += 1
            c['d_grief_gt_tick_cap_relief' if interp['relief_applied'] else 'd_grief_gt_tick_cap_other'] += 1
        trust = interp['trust']
        if trust is not None:
            band = 'trust<0.5' if trust < 0.5 else ('trust0.5-0.9' if trust < 0.9 else 'trust>=0.9')
            c['band:' + band] += 1
        sp = self.pattern_dict.get(speaker_id)
        li = self.pattern_dict.get(listener_id)
        wall_state = 'on' if Config.ENABLE_CENTER_WALL else 'off'
        cross = None
        if sp is not None and li is not None and sp.cells and li.cells:
            wx = Config.CENTER_WALL_X
            side_s = (sum(cc[0] for cc in sp.cells) / len(sp.cells)) < wx
            side_l = (sum(cc[0] for cc in li.cells) / len(li.cells)) < wx
            cross = side_s != side_l
        if cross is None:
            c['side_unknown'] += 1
        elif cross:
            c['cross_side_wall_' + wall_state] += 1
        else:
            c['same_side_wall_' + wall_state] += 1
        if len(self._dlg_log) < Config.DIALOGUE_LOG_MAX:
            self._dlg_log.append((self.age, speaker_id, listener_id, emotion, intent, token,
                                  bool(interp['answer']), None if trust is None else round(float(trust), 4),
                                  interp['seek'], round(float(dg), 6), round(float(dt), 6),
                                  round(float(interp['relief']), 6),
                                  None if sim is None else round(float(sim), 4), cross, wall_state))
        else:
            c['log_overflow'] += 1

    def _mode_snapshot(self, t):
        alive = {p.id: p for p in self.patterns if p.alive}
        mutual = one_sided = trace = first = repeat = sign_split = 0
        grav_gap = 0.0
        for a_id, a in alive.items():
            for b_id, imp in a.imprints.by_other.items():
                if imp.n <= 1:
                    first += 1
                else:
                    repeat += 1
                b = alive.get(b_id)
                if b is None:
                    trace += 1
                    continue
                back = b.imprints.by_other.get(a_id)
                if back is None:
                    one_sided += 1
                else:
                    mutual += 1
                    if (imp.gravity > 0.0) != (back.gravity > 0.0):
                        sign_split += 1
                    grav_gap += abs(imp.gravity - back.gravity)
        self._mode_snapshots.append({
            't': t, 'directed_entries': mutual + one_sided + trace,
            'participation_mutual': mutual, 'bearing_one_sided': one_sided,
            'form_trace_of_departed': trace, 'first_contact_entries': first,
            'repeat_contact_entries': repeat, 'mutual_gravity_sign_split': sign_split,
            'mutual_gravity_gap_mean': (grav_gap / mutual) if mutual else 0.0})

    @staticmethod
    def _res_update(R, t, r):
        R['n'] += 1
        R['sum'] += r
        R['abs'] += abs(r)
        if r > 1e-9:
            R['pos'] += 1
        elif r < -1e-9:
            R['neg'] += 1
        if abs(r) > R['max_abs']:
            R['max_abs'] = abs(r)
            R['max_t'] = t

    def _audit_gap_check(self, t, e1_start, e2_start):
        if self._prev_E1_end is not None:
            self._res_update(self._gap_x1, t, e1_start - self._prev_E1_end - self._outside_acc1)
            self._res_update(self._gap_x2, t, e2_start - self._prev_E2_end - self._outside_acc2)
        self._outside_acc1 = 0.0
        self._outside_acc2 = 0.0

    def _audit_close_tick(self, t, de_x1, res_x1, x2_step0):
        self._ledger_window = False
        x2_end = self._x2_sum()
        de_x2 = x2_end - x2_step0
        res_x2 = de_x2 - self._x2_tick_measured
        self._res_update(self._res_x1, t, res_x1)
        self._res_update(self._res_x2, t, res_x2)
        alive = [p for p in self.patterns if p.alive]
        roles = Counter(p.role_type for p in alive)
        acc = self._tick_acc
        rolled = acc['rolled']
        births = 0 if self._audit_prev_next_id is None else self.next_id - self._audit_prev_next_id
        self._audit_prev_next_id = self.next_id
        self._audit_rows.append({
            't': t, 'pop': len(alive), 'feral': roles['feral'], 'normal': roles['normal'],
            'disorg': roles['disorganizer'], 'objects': len(self.patterns),
            'E1': float(self.field[:, :, CH['energy']].sum(dtype=np.float64)), 'E2': x2_end,
            'dE1': de_x1, 'dE2': de_x2,
            'led1': self._ledger_tick_measured, 'led2': self._x2_tick_measured,
            'given_flux': self._given_flux_tick,
            'res1': res_x1, 'res2': res_x2,
            'harvest_not_added': self._memo_tick['Pattern.metabolic_tax: получено из поля (сбор излишка)'],
            'rolled': rolled, 'hits': acc['hits'], 'misses': acc['misses'],
            'p_mean': (acc['p_sum'] / rolled) if rolled else None,
            'raw_mean': (acc['raw_sum'] / rolled) if rolled else None,
            'removed': acc['removed'], 'births': births, 'wall_on': bool(Config.ENABLE_CENTER_WALL),
            'om_n': acc['om_n'], 'om_gain_mean': (acc['om_gain'] / acc['om_n']) if acc['om_n'] else None,
            'om_learned': acc['om_choice_learned'], 'om_explore': acc['om_choice_explore'],
            'om_default': acc['om_choice_default']})
        self._tick_acc.clear()
        self._given_flux_tick = 0.0
        self._prev_E1_end = self._audit_rows[-1]['E1']
        self._prev_E2_end = x2_end
        if Config.AUDIT_MODE_EVERY > 0 and t % Config.AUDIT_MODE_EVERY == 0:
            self._mode_snapshot(t)

    def audit_summary(self):
        def _tab(src):
            return {k: {'pos': v[0], 'neg': v[1], 'net': v[0] + v[1], 'n': v[2]} for k, v in src.items()}
        L1 = self._energy_ledger
        L2 = {'src': self._energy_ledger['x2']}
        rows = self._audit_rows
        x1_logged = float(sum(v[0] + v[1] for v in L1['src'].values()))
        x2_logged = float(sum(v[0] + v[1] for v in L2['src'].values()))
        return {
            'ticks': len(rows),
            'balance': self.audit_balance(),
            'x1': {'ledger': _tab(L1['src']), 'logged_net': x1_logged, 'dE_total': L1['step_total'],
                   'residual': dict(self._res_x1),
                   'abs_flow_sum': float(sum(abs(v[0]) + abs(v[1]) for v in L1['src'].values()))},
            'x2': {'ledger': _tab(L2['src']), 'logged_net': x2_logged,
                   'dE_total': float(sum(r['dE2'] for r in rows)),
                   'residual': dict(self._res_x2),
                   'abs_flow_sum': float(sum(abs(v[0]) + abs(v[1]) for v in L2['src'].values()))},
            'gap_between_ticks': {'x1': dict(self._gap_x1), 'x2': dict(self._gap_x2)},
            'outside_window': {'x1': _tab(self._ledger_outside), 'x2': _tab(self._x2_outside)},
            'memo': {k: {'not_added_by_clip': v[0], 'not_charged_by_clip': v[1], 'n': v[2]}
                     for k, v in self._energy_memo.items()},
            'feral': dict(self._feral_stats),
            'dialogue': dict(self._dlg_ledger),
            'dialogue_columns': ['t', 'speaker', 'listener', 'emotion', 'intent', 'token', 'answer',
                                 'trust', 'seek', 'd_grief', 'd_grat', 'relief', 'sim', 'cross_side', 'wall'],
            'mode_snapshots': self._mode_snapshots,
            'outcome_learning': {
                'enabled': bool(Config.OUTCOME_LEARNING), 'stats': dict(self._om_stats), 'choices': dict(self._om_choices),
                'by_word': {f"{k[0]}/{k[1]}": {'n': v[0], 'mean_gain': v[1] / v[0]} for k, v in self._om_by_word.items()},
                'by_ctx': {str(k): {'n': v[0], 'mean_gain': v[1] / v[0]} for k, v in self._om_by_ctx.items()},
                'value_table': [{'ctx': list(k[0]), 'emotion': k[1], 'intent': k[2], 'value': v, 'n': n}
                                for k, (v, n) in sorted(self._om_value_table().items(), key=lambda kv: -kv[1][1])[:60]]},
            'wall_events': self._wall_events,
            'wall_cycle': {'enabled': bool(Config.WALL_CYCLE_ENABLED), 'on': int(Config.WALL_CYCLE_ON_TICKS),
                           'off': int(Config.WALL_CYCLE_OFF_TICKS), 'first': Config.WALL_CYCLE_FIRST},
        }

    def _print_audit_report(self):
        S = self.audit_summary()
        print("\n  ── АУДИТ ЭНЕРГИИ: баланс проверяется, а не закрывается подписью ──")
        for tag, key in (("X1 (поле)", 'x1'), ("X2 (запас агентов)", 'x2')):
            blk = S[key]
            R = blk['residual']
            share = (R['abs'] / blk['abs_flow_sum']) if blk['abs_flow_sum'] > 0 else 0.0
            print(f"  [{tag}] Σ ΔE={blk['dE_total']:+.1f}; учтено в ledger={blk['logged_net']:+.1f}; "
                  f"НЕВЯЗКА Σ={R['sum']:+.3f}, Σ|res|={R['abs']:.3f} ({share:.2%} от Σ|потоков|), "
                  f"max|res|={R['max_abs']:.4f} @t={R['max_t']}, тиков +/−={R['pos']}/{R['neg']} из {R['n']}")
        for tag, key in (("X1", 'x1'), ("X2", 'x2')):
            G = S['gap_between_ticks'][key]
            print(f"  [{tag}] зазор между тиками (изменения вне step(), не записанные нигде): Σ={G['sum']:+.3e}, "
                  f"max|gap|={G['max_abs']:.3e} @t={G['max_t']}, тиков +/−={G['pos']}/{G['neg']} из {G['n']}")
        for tag, key in (("X1", 'x1'), ("X2", 'x2')):
            ow = S['outside_window'][key]
            if ow:
                tot = sum(v['net'] for v in ow.values())
                print(f"  [{tag}] записано ВНЕ окна тика (инициализация/между шагами): Σ={tot:+.2f} — " +
                      "; ".join(f"{n[:46]}={v['net']:+.2f}" for n, v in sorted(ow.items(), key=lambda kv: -abs(kv[1]['net']))[:4]))
        if S['x2']['ledger']:
            print(f"\n  {'X2: источник/сток запаса агентов':74s} {'нетто':>10s} {'приход':>10s} {'расход':>10s} {'вызовов':>8s}")
            for n, v in sorted(S['x2']['ledger'].items(), key=lambda kv: -abs(kv[1]['net'])):
                print(f"  {n[:74]:74s} {v['net']:+10.1f} {v['pos']:+10.1f} {v['neg']:+10.1f} {v['n']:8d}")
        if S['memo']:
            print(f"\n  {'memo: энергия, не прошедшая клип X2 [-1,2] (в баланс не входит)':74s} {'не добавл.':>10s} {'не списано':>10s} {'вызовов':>8s}")
            for n, v in sorted(S['memo'].items(), key=lambda kv: -abs(kv[1]['not_added_by_clip'] + kv[1]['not_charged_by_clip'])):
                print(f"  {n[:74]:74s} {v['not_added_by_clip']:+10.1f} {v['not_charged_by_clip']:+10.1f} {v['n']:8d}")
        F = S['feral']
        if F.get('rolled', 0) or F.get('prey_found', 0):
            rolled = F.get('rolled', 0)
            hr = F.get('hits', 0) / rolled if rolled else float('nan')
            print(f"\n  [ФЕРОЛ, весь прогон] цель найдена={F.get('prey_found', 0)} нет цели={F.get('no_prey', 0)} "
                  f"не смежно={F.get('not_adjacent', 0)} | удар брошен={rolled}: попаданий={F.get('hits', 0)} "
                  f"промахов={F.get('misses', 0)} → hit_rate={hr:.3f}; ⟨kill_prob⟩={F.get('p_sum', 0.0) / max(rolled, 1):.3f}; "
                  f"⟨raw⟩={F.get('raw_sum', 0.0) / max(rolled, 1):.3f}; raw≥0.95 в {F.get('raw_ge_095', 0) / max(rolled, 1):.1%} ударов "
                  f"(режим={Config.FERAL_KILL_MODE})")
        D = S['dialogue']
        if D.get('heard', 0):
            h = D['heard']
            sk = D.get('seek', 0)
            print(f"\n  [ДИАЛОГ, весь прогон, включая умерших] услышано={h}: первая встреча (пара направленная)={D.get('first_encounter', 0)} "
                  f"повторная={D.get('repeat_encounter', 0)}; SEEK={sk}: принято={D.get('seek_registered', 0)} "
                  f"отказ trust/grief/coherence={D.get('seek_gate_trust', 0)}/{D.get('seek_gate_grief', 0)}/{D.get('seek_gate_coherence', 0)}; "
                  f"ANSWER услышан={D.get('answer_heard', 0)} (с облегчением={D.get('relief', 0)}); low_sim={D.get('low_sim', 0)}; "
                  f"сдвиг grief за реплику > лимита эмоции за тик ({Config.EMOTIONAL_CAP_RATE}, допуск 1e-9): {D.get('d_grief_gt_tick_cap', 0)} из {h} "
                  f"(с relief ANSWER {D.get('d_grief_gt_tick_cap_relief', 0)}, прочие {D.get('d_grief_gt_tick_cap_other', 0)})")
            cw = {k: D.get(k, 0) for k in ('same_side_wall_on', 'cross_side_wall_on', 'same_side_wall_off', 'cross_side_wall_off', 'side_unknown')}
            print(f"  [ДИАЛОГ × СТЕНА] реплик между разными сторонами: стена ON {cw['cross_side_wall_on']} из {cw['cross_side_wall_on'] + cw['same_side_wall_on']}; "
                  f"стена OFF {cw['cross_side_wall_off']} из {cw['cross_side_wall_off'] + cw['same_side_wall_off']}; сторона неизвестна: {cw['side_unknown']}")
        O = S['outcome_learning']
        if O['stats'].get('n'):
            ch = O['choices']
            tot = sum(ch.values()) or 1
            print(f"  [РЕЧЬ, ОБУЧАЕМАЯ ИСХОДОМ] выбор слова: по умолчанию={ch.get('default', 0)}, выученное={ch.get('learned', 0)} "
                  f"({ch.get('learned', 0) / tot:.1%}), проба={ch.get('explore', 0)}; исходов записано (только свои реплики; до fix pending считались и услышанные)={O['stats']['n']}, "
                  f"средний исход={O['stats']['gain_sum'] / O['stats']['n']:+.4f}")
        if S['wall_events']:
            print("  [СТЕНА, события] " + "; ".join(f"t={e['t']}→{e['to']} (на линии {e['agents_on_line_before']}→{e['agents_on_line_after']} агентов)" for e in S['wall_events']))
        if S['mode_snapshots']:
            m = S['mode_snapshots'][-1]
            print(f"  [РЕЖИМЫ по Imprint, t={m['t']}] записей={m['directed_entries']}: Участие(взаимно)={m['participation_mutual']} "
                  f"Ношение(односторонне)={m['bearing_one_sided']} Форма(след ушедшего)={m['form_trace_of_departed']}; "
                  f"первых контактов={m['first_contact_entries']} повторных={m['repeat_contact_entries']}; "
                  f"взаимных пар с разным знаком тяги={m['mutual_gravity_sign_split']}")

        self._print_boundary_report(S)

    def _audit_save(self):
        if not Config.AUDIT_DIR:
            return
        import os, json
        os.makedirs(Config.AUDIT_DIR, exist_ok=True)
        path = os.path.join(Config.AUDIT_DIR, f"audit_{Config.AUDIT_TAG}.json")

        def _default(o):
            if isinstance(o, (np.floating, np.integer)):
                return o.item()
            return str(o)
        payload = {'input_fp': _input_fingerprint(), 'summary': self.audit_summary(), 'rows': self._audit_rows,
                   'attacks': self._attack_log, 'dialogue': self._dlg_log,
                   'config': {'FERAL_KILL_MODE': Config.FERAL_KILL_MODE, 'FERAL_KILL_CEIL': Config.FERAL_KILL_CEIL,
                              'FERAL_KILL_SCALE': Config.FERAL_KILL_SCALE, 'FERAL_KILL_BASE': Config.FERAL_KILL_BASE,
                              'FERAL_VULNERABILITY': Config.FERAL_VULNERABILITY, 'FERAL_CROWD_GAIN': Config.FERAL_CROWD_GAIN}}
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(payload, f, ensure_ascii=False, default=_default)
        print(f"[audit] сохранено: {path}")

    def _om_record(self, key, gain):
        acc = self._tick_acc
        acc['om_n'] += 1
        acc['om_gain'] += gain
        self._om_stats['n'] += 1
        self._om_stats['gain_sum'] += gain
        w = self._om_by_word.setdefault((key[1], key[2]), [0, 0.0])
        w[0] += 1
        w[1] += gain
        x = self._om_by_ctx.setdefault(key[0], [0, 0.0])
        x[0] += 1
        x[1] += gain

    def _om_choice(self, source):
        self._om_choices[source] += 1
        self._tick_acc['om_choice_' + source] += 1

    def _om_value_table(self):
        agg = {}
        for p in self.patterns:
            if not p.alive:
                continue
            for key, rec in p.outcome_memory.items():
                e = agg.setdefault(key, [0.0, 0])
                e[0] += rec[0] * rec[1]
                e[1] += rec[1]
        return {key: (e[0] / e[1], e[1]) for key, e in agg.items() if e[1] > 0}

    def _wall_on(self):
        return bool(getattr(Config, 'ENABLE_CENTER_WALL', False) and getattr(Config, 'ENABLE_WALL_GUARD', True))

    def _migrate_wall_cells(self):
        if not self._wall_on(): return
        wx = Config.CENTER_WALL_X; ws = Config.WORLD_SIZE; d = self._wall_diag
        for p in self.patterns:
            if not (p.alive and p.cells): continue
            ghost = [c for c in p.cells if c[0] == wx]
            if not ghost: continue
            if len(ghost) < len(p.cells):
                p.cells.difference_update(ghost); d['migrated_dropped_cells'] += len(ghost)
            else:
                for (x, y) in ghost:
                    for nx in (wx - 1, wx + 1):
                        nx %= ws
                        if self.field[nx, y, CH['owner']] in (0, p.id):
                            p.cells.discard((x, y)); p.cells.add((nx, y))
                            self.field[nx, y, CH['owner']] = p.id
                            d['migrated_shifted_cells'] += 1
                            break
                    else: d['migrated_kept_cells'] += 1
            d['migrated_agents'] += 1

    def _audit_wall(self):
        if not self._wall_on(): return
        wx = Config.CENTER_WALL_X
        n = sum(1 for p in self.patterns if p.alive for c in p.cells if c[0] == wx)
        d = self._wall_diag
        d['audit_ticks'] += 1; d['audit_cells_sum'] += n
        if n > d['audit_cells_max']: d['audit_cells_max'] = n
        d['audit_cells_last'] = n

    def _print_world_diag(self):
        print("\n" + "=" * 70)
        print("⚖️ ДИАГНОСТИКА МИРА (стена / энергия / геном)")
        print("=" * 70)
        d = self._wall_diag
        if self._wall_on():
            print(f"🚧 Стена X={Config.CENTER_WALL_X}: клеток агентов на линии сейчас {d.get('audit_cells_last', 0)} "
                  f"(макс {d.get('audit_cells_max', 0)}, в среднем "
                  f"{d.get('audit_cells_sum', 0) / max(d.get('audit_ticks', 1), 1):.1f})")
        L = self._energy_ledger
        if L['src'] and L['ticks']:
            print(f"\n  {'источник энергии':58s} {'нетто':>10s} {'приход':>10s} {'расход':>10s}")
            for _n, (_p, _m, _c) in sorted(L['src'].items(), key=lambda kv: -abs(kv[1][0] + kv[1][1])):
                print(f"  {_n[:58]:58s} {_p + _m:+10.1f} {_p:+10.1f} {_m:+10.1f}")
            print(f"  Итого учтено: {sum(r[0] + r[1] for r in L['src'].values()):+.1f}; "
                  f"изменение суммы поля: {L['step_total']:+.1f} за {L['ticks']} тиков")
        X2L = L.get('x2', {})
        if X2L:
            print(f"\n  {'источник X2 (инвентарь агентов)':58s} {'нетто':>10s} {'приход':>10s} {'расход':>10s}")
            for _n, (_p, _m, _c) in sorted(X2L.items(), key=lambda kv: -abs(kv[1][0] + kv[1][1])):
                print(f"  {_n[:58]:58s} {_p + _m:+10.2f} {_p:+10.2f} {_m:+10.2f}")
            _x2_net = sum(r[0] + r[1] for r in X2L.values())
            print(f"  Итого X2 учтено: {_x2_net:+.2f}")
        self._print_audit_report()
        print("=" * 70)
        self._audit_save()

    def _rng_mark(self, t, label):
        if getattr(self, 'field', None) is not None:
            _e = float(self.field[:, :, CH['energy']].sum(dtype=np.float64))
            _last = self._phase_last
            if _last is not None and _last[0] == t:
                _row = self._energy_ledger['phase'].setdefault(label, [0.0, 0.0, 0])
                _d = _e - _last[1]
                if _d >= 0.0: _row[0] += _d
                else: _row[1] += _d
                _row[2] += 1
            self._phase_last = (t, _e)
        _trace_tick = getattr(Config, 'DEBUG_RNG_TRACE_TICK', None)
        if _trace_tick is not None and t == _trace_tick:
            print(f"[RNG_PHASE t={t}] {label}: pos={self._given_rng.get_state()[2]}")

    def step(self, t):
        self._rng_mark(t, "начало step()")
        self._wall_cycle_update(t)
        self._econs_sample(t)
        self._update_feral_context()
        for _p in self.patterns:
            if _p.alive:
                _g = _p.genome
                if _g.get('metabolic_cost', 0.002) < 0.002:
                    _g['metabolic_cost'] = 0.002; self._genome_diag['metabolic_cost_healed'] += 1
                if _g.get('mobility', 0.05) < 0.05:
                    _g['mobility'] = 0.05; self._genome_diag['mobility_healed'] += 1
        _e_step0 = float(self.field[:, :, CH['energy']].sum(dtype=np.float64))
        self._ledger_tick_measured = 0.0
        self._x2_tick_measured = 0.0
        self._memo_tick.clear()
        _x2_step0 = self._x2_sum()
        self._audit_gap_check(t, _e_step0, _x2_step0)
        self._ledger_window = True
        if Config.ENABLE_MORSE_COMMUNICATION:
            _lang_ensure_field_channels(self)
            self.morse_field.step(t)
            if t % Config.MORSE_SYNC_INTERVAL == 0:
                self.morse_field.sync_to_field(self.field, Config.MORSE_CHANNEL)
            _alive_by_id = {p.id: p for p in self.patterns if p.alive}
            self.EMERGENT_LEXICON.resolve_context(_alive_by_id, t)
            if t % 20 == 0:
                self.EMERGENT_LEXICON.log_population_gene_diversity(_alive_by_id)
        self._rng_mark(t, "после Морзе-поля/лексикона")
        if not any(p.alive for p in self.patterns):
            if Config.VERBOSE_LOGS: print(f"[t={t}] POPULATION COLLAPSE — stopping early")
            return False
        self.age = t
        for p in self.patterns:
            if not p.alive: continue
            for key in ('gratitude', 'grief'):
                val = p.emotional_memory.get(key)
                if isinstance(val, dict):
                    if 'count' in val and isinstance(val['count'], (int, float)): p.emotional_memory[key] = float(val['count'])
                    elif 'value' in val and isinstance(val['value'], (int, float)): p.emotional_memory[key] = float(val['value'])
                    else: p.emotional_memory[key] = 0.5
                elif not isinstance(val, (int, float)):
                    p.emotional_memory[key] = 0.5
        self._energy_injected_this_step = 0.0
        self._energy_taxed_this_step = 0.0
        with self._energy_span('_update_sanctuary'): self._update_sanctuary(t)
        total_energy = np.sum(self.field[:,:,CH['energy']])
        if self._prev_total_energy is not None:
            drift = abs(total_energy - self._prev_total_energy)
            self._guardian_stats['energy_drift_sum'] += drift
            self._guardian_stats['energy_drift_count'] += 1
            if drift > self._guardian_stats['energy_drift_peak']: self._guardian_stats['energy_drift_peak'] = drift
        self._prev_total_energy = total_energy
        for p in self.patterns:
            if p.alive:
                diff = np.max(np.abs(p.prediction - (p.belief + p.model)))
                if diff > self._guardian_stats['model_worst_ever_diff']:
                    self._guardian_stats['model_worst_ever_diff'] = diff
                    self._guardian_stats['model_worst_ever_id'] = p.id
        _drift_for_selfreg = drift if self._guardian_stats['energy_drift_count'] > 0 else 0.0
        if any(p.alive for p in self.patterns):
            self.selfreg.update_system_metrics(self.field, self.patterns, _drift_for_selfreg, t)
        else:
            self.selfreg.boredom = 0.0; self.selfreg.phase = "recovery"
        e_before = np.sum(self.field[:,:,CH['energy']])
        with self._energy_span('field_dynamics (без gulf/dark)'): self.field_dynamics(t)
        self._energy_injected_this_step += np.sum(self.field[:,:,CH['energy']]) - e_before
        e_before = np.sum(self.field[:,:,CH['energy']])
        with self._energy_span('ecosystem_pressure'): ecosystem_pressure(self.field)
        self._energy_injected_this_step += np.sum(self.field[:,:,CH['energy']]) - e_before
        alive_now = [p for p in self.patterns if p.alive]
        if alive_now:
            sg_vals = [float(np.mean(np.abs(p.prediction - p.belief))) for p in alive_now]
            avg_sg = safe_mean(sg_vals, 0.5)
            if self._sg_crisis_timer > 0:
                self._sg_crisis_timer -= 1
                if self._sg_crisis_timer == 0: self._sg_crisis_active = False
            else:
                if avg_sg < Config.SG_CRISIS_THRESHOLD:
                    self._sg_crisis_active = True
                    self._sg_crisis_timer = Config.SG_CRISIS_DURATION
            if self._sg_crisis_active:
                for p in alive_now:
                    noise = np.array([deterministic_noise(p.age, p.id, i+77777) for i in range(8)])
                    p.belief += noise * Config.SG_CRISIS_NOISE_STRENGTH
        self.pattern_dict = {p.id: p for p in self.patterns if p.alive}
        alive = [p for p in self.patterns if p.alive]
        self.soul_weight_average = safe_mean([p.soul_weight for p in alive], 0.5) if alive else 0.5
        N = len(alive)
        if Config.ENABLE_PHI_LABYRINTH:
            if N >= 100: target_threshold, target_grow, target_move = 0.10, 15.0, 9.0
            elif N >= 90: target_threshold, target_grow, target_move = 0.12, 9.0, 6.0
            elif N <= 60: target_threshold, target_grow, target_move = 0.75, 1.0, 0.6
            else:
                ratio = (N - 60) / 30.0
                target_threshold = 0.75 - ratio * (0.75 - 0.12)
                target_grow = 1.0 + ratio * (9.0 - 1.0)
                target_move = 0.6 + ratio * (6.0 - 0.6)
            alive_here = [p for p in self.patterns if p.alive]
            avg_obs = safe_mean([p.spirit_gap for p in alive_here], 0.5) if alive_here else 0.5
            low_gap = getattr(Config, 'LABYRINTH_ADAPT_LOW_GAP', 0.4)
            high_gap = getattr(Config, 'LABYRINTH_ADAPT_HIGH_GAP', 0.7)
            max_mult = getattr(Config, 'LABYRINTH_ADAPT_MAX_MULT', 2.0)
            adapt_speed = getattr(Config, 'LABYRINTH_ADAPT_SPEED', 0.05)
            if avg_obs < low_gap: target_mult = max_mult
            elif avg_obs > high_gap: target_mult = 1.0
            else:
                t_norm = (avg_obs - low_gap) / (high_gap - low_gap)
                target_mult = max_mult - t_norm * (max_mult - 1.0)
            self._lab_adapt_mult += (target_mult - self._lab_adapt_mult) * adapt_speed
            target_grow *= self._lab_adapt_mult
            target_move *= self._lab_adapt_mult
            lab_pen_mult = self.selfreg.get_labyrinth_penalty_multiplier()
            target_grow *= lab_pen_mult; target_move *= lab_pen_mult
            alpha = 0.95
            self.phi_labyrinth_threshold = (1.0 - alpha) * self.phi_labyrinth_threshold + alpha * target_threshold
            self.phi_labyrinth_grow_penalty = (1.0 - alpha) * self.phi_labyrinth_grow_penalty + alpha * target_grow
            self.phi_labyrinth_move_penalty = (1.0 - alpha) * self.phi_labyrinth_move_penalty + alpha * target_move
            if N < 20:
                self.phi_labyrinth_grow_penalty = 1.0
                self.phi_labyrinth_move_penalty = 0.5
        for p in alive: p._antigravity_boost = 1.0
        if t % 500 == 0:
            sg_vals = [float(np.mean(np.abs(p.prediction - p.belief))) for p in alive]
            if sg_vals:
                min_sg, max_sg = min(sg_vals), max(sg_vals)
                median_sg = sorted(sg_vals)[len(sg_vals)//2]
                print(f"Spirit Gap: min={min_sg:.3f} p50={median_sg:.3f} max={max_sg:.3f}")
                clear = sum(1 for g in sg_vals if g < 0.4)
                adapting = sum(1 for g in sg_vals if 0.4 <= g < 0.8)
                confused = sum(1 for g in sg_vals if 0.8 <= g < 1.0)
                blind = len(sg_vals) - clear - adapting - confused
                print(f"zones: clear={clear} adapting={adapting} confused={confused} blind={blind}")
                print(f"Phase: {self.selfreg.phase} | {self.selfreg.get_phase_log()}")
                if self.selfreg.is_stuck_in_phase("stagnation", 150):
                    print("Stagnation warning: system stuck >150 steps")
        all_trust_vals = []
        for p in self.patterns:
            if p.alive: all_trust_vals.extend(p.trust_ledger.entries.values())
        avg_trust = safe_mean(all_trust_vals, Config.TRUST_BASE) if all_trust_vals else Config.TRUST_BASE
        if avg_trust < 0.4: self.target_disorganizer_fraction = 0.50
        elif avg_trust < 0.7: self.target_disorganizer_fraction = 0.30
        else: self.target_disorganizer_fraction = 0.15
        alive_sg = [float(np.mean(np.abs(p.prediction - p.belief))) for p in alive]
        avg_sg = safe_mean(alive_sg, 0.5) if alive_sg else 0.5
        turb = self.selfreg.turbulence_factor
        if avg_sg < Config.SG_TURBULENCE_THRESHOLD:
            tremor_strength = (Config.GLOBAL_SPIRIT_TREMOR_BASE * 4.0 +
                (Config.GLOBAL_SPIRIT_TREMOR_MAX * 4.0 - Config.GLOBAL_SPIRIT_TREMOR_BASE * 4.0) *
                (1.0 - avg_sg / Config.SG_TURBULENCE_THRESHOLD)) * turb
        else:
            tremor_strength = Config.GLOBAL_SPIRIT_TREMOR_BASE * 4.0 * turb
        for p in alive:
            shake = np.array([deterministic_noise(t, p.id, i+88888) for i in range(8)])
            p.belief += shake * tremor_strength
        lab_interval = self.selfreg.get_labyrinth_interval()
        if t % lab_interval == 0: self._update_phi_labyrinth(t)
        _kern = np.array([[0,1,0],[1,0,1],[0,1,0]], dtype=np.float32)
        self._global_neighbor_grat = convolve(self.field[:,:,CH['signal_gratitude']], _kern, mode='wrap')
        self._global_neighbor_grief = convolve(self.field[:,:,CH['signal_grief']], _kern, mode='wrap')
        self._encounter_step(t)
        given_counter = {'count': 0}
        for p in self.patterns:
            if not p.alive: continue
            try:
                p.compute_local_perception(self.field)
                p.update_model(self.field, lineage_counts=None, given_counter=given_counter, t=t, witness=self.witness)
                p.sensory_reentry()
                p.vision_event()
                p.update_substate()
                p._update_unconquered_potential()
                p.sense_self_and_other(self.field, t)
                p.grow(self.field)
            except Exception as e:
                import traceback
                print(f"Agent {p.id} error: {e}. Suspending.", flush=True)
                traceback.print_exc()
                p._deposit_final_testament()
                self._record_death_drag(p, 'error_suspended')
                p.alive = False
                self.echo_system.store(p)
                self.witness.record(p.id, "error_suspended", error=str(e))
        if t % 10 == 0:
            for p in self.patterns:
                if p.alive and p.role_type == "normal":
                    update_chronic_counters(p, step_scale=10)
                    check_chronic_disorganizer(self, p, t)
        for p in self.patterns:
            if not p.alive or p.age < 300: continue
            age_factor = min(1.0, (p.age - 300) / 1200)
            gap_factor = max(1.0, 5.0 - p.spirit_gap * 20) if p.spirit_gap < 0.2 else 1.0
            strength = (0.03 + age_factor * 0.12) * gap_factor
            noise_b = np.array([deterministic_noise(t, p.id, i+99999) for i in range(8)]) * strength
            noise_m = np.array([deterministic_noise(t, p.id, i+100000) for i in range(8)]) * strength * 0.6
            p.belief = np.clip(p.belief + noise_b, -1.0, 1.0)
            p.model = np.clip(p.model + noise_m, -1.0, 1.0)
        for p in self.patterns:
            if not p.alive or p.age < 300: continue
            if p.semantic_state == "seeking_comfort" and p.spirit_gap < 0.1:
                if getattr(Config, 'ENABLE_FORCED_STATE_SHIFT', True) and phi_hash(p.id, t, 123458) < 0.20:
                    roll = phi_hash(p.id, t, 123459)
                    new_state = "grateful_but_cautious" if roll < 0.6 else "contentment"
                    p.semantic_state = new_state
                    p._log_event("forced_state_shift", from_="seeking_comfort", to=new_state, age=p.age)
                    self.witness.record(p.id, "forced_state_shift", from_state="seeking_comfort", to_state=new_state, age=p.age)
        if t % 200 == 0 and self.archive:
            _stale_pool = [s for s in self.archive.write_queue
                           if s.get('weight', 0) > 0.65
                           and s.get('event') not in ('essential_concepts_inherited', 'archive_concept_inherited')]
            if _stale_pool:
                for _p in alive:
                    if _p.role_type == "disorganizer": continue
                    _diversity = len(_p.concept_graph.nodes)
                    if _diversity < 8 and _p.age > 100:
                        if phi_hash(_p.id, t, 77777) < 0.08:
                            _sc = _stale_pool[int(phi_hash(_p.id, t, 77778) * len(_stale_pool)) % len(_stale_pool)]
                            _err = round(phi_hash(_p.id, t, 333) * 0.3, 1)
                            _load = round(phi_hash(_p.id, t, 444) * 0.2, 1)
                            _sig = (_err, _load, round(_sc.get('weight', 0.7), 1), f"archive_{_sc.get('event', 'novelty')}")
                            if _sig not in _p.concept_graph.nodes:
                                _p.concept_graph.nodes[_sig] = {"origin": "ARCHIVE", "count": 1.2, "value": np.zeros(4), "embed": np.zeros(32), "eternal": False}
                                _p._log_event("epistemic_noise_stale", concept=str(_sig[3])[:40], age=_p.age)
        for p in self.patterns:
            if p.alive and p.age > 500 and p.age % 50 == 0:
                p.spirit_gap = min(0.6, p.spirit_gap + 0.05)
        alive_now = [p for p in self.patterns if p.alive]
        if alive_now:
            seeing_count = sum(1 for p in alive_now if p.spirit_gap < 0.4)
            seeing_fraction = seeing_count / len(alive_now)
            avg_energy = float(np.mean(self.field[:,:,CH['energy']]))
            energy_factor = np.clip(avg_energy / 0.2, 0.2, 1.0)
            base_strength = 0.16 + (0.64 - 0.16) * seeing_fraction
            scar_mult = self.selfreg.get_scar_injection_multiplier() * 4.4
            TARGET_GAP = 0.6; BELL_WIDTH = 0.3
            for p in alive_now:
                if p.in_dream: continue
                gap = p.spirit_gap
                sigma = BELL_WIDTH / 2.0
                bell = np.exp(-((gap - TARGET_GAP) ** 2) / (2 * sigma ** 2))
                if bell < 0.05: continue
                if gap < 0.25:
                    targeted_boost = 1.0 + (0.25 - gap) * 6.0
                    p_scar_mult = scar_mult * targeted_boost
                else: p_scar_mult = scar_mult
                p_scar_mult = self.selfreg.get_agent_scar_multiplier(p, base=p_scar_mult)
                local_scar = safe_mean([self.scar[x, y] for (x, y) in p.cells], 0.0)
                # FIX(spirit_gap bias): deterministic_noise() возвращает ≈±0.0005, поэтому прежнее «- 0.5» давало
                # константу ≈ -0.5 → каждый тик систематический отрицательный пинок в model/belief при gap≲0.97
                # (pred−real<0 во всех 8 каналах, даже в 'noise' с real=0). Симметричный шум = phi_hash − 0.5.
                # Откат к старому поведению: Config.SCAR_KICK_LEGACY_BIAS = True.
                if bool(getattr(Config, 'SCAR_KICK_LEGACY_BIAS', False)):
                    noise = np.array([deterministic_noise(t, p.id, i + 55555) - 0.5 for i in range(8)])
                else:
                    noise = np.array([phi_hash(t, p.id, i + 55555) - 0.5 for i in range(8)]) * float(getattr(Config, 'SCAR_KICK_NOISE_SCALE', 1.0))
                kick = base_strength * energy_factor * bell * (1.0 + local_scar * 1.5) * noise * p_scar_mult
                # FIX(pred bias, этап 2): после замены константы на симметричный шум амплитуда осталась прежней:
                # base_strength·energy_factor·bell·(1+scar·1.5)·p_scar_mult (множитель ≈11–26) даёт ±0.3–0.7 за тик в КАЖДЫЙ
                # канал model (и 0.4× в belief). Это белый шум ≫ сюрприза мира (≈0.02): err≈0.4 даже в канале noise (real≡0),
                # а клип ±0.8 срезает верхние выбросы → отрицательный bias ∝ real (trust −0.29, binding −0.21, energy −0.13).
                # Потолок на |kick| по каналу. Config.SCAR_KICK_MAX <= 0 → старое поведение (без потолка).
                _kmax = float(getattr(Config, 'SCAR_KICK_MAX', 0.0))
                _kick_raw_probe = np.asarray(kick, dtype=float).copy()
                _kick_raw_abs = np.abs(_kick_raw_probe)
                _scar_probe = globals().setdefault("_P832_SCAR_KICK_PROBE", {"ticks": 0, "at_cap_ticks": 0, "raw_abs": [], "clipped_abs": [], "elements_at_cap": 0, "elements_total": 0, "raw_by_channel": [[] for _ in range(8)], "clipped_by_channel": [[] for _ in range(8)], "at_cap_by_tick": [], "at_cap_by_channel": [0]*8, "elements_total_by_channel": [0]*8, "elements_at_cap_by_channel": [0]*8})
                _scar_probe["ticks"] += 1
                # Bounded samples prevent unbounded memory growth on long runs.
                _sample_cap = 200000
                _raw_flat = _kick_raw_abs.ravel()
                _scar_probe["raw_abs"].extend(_raw_flat[:max(0, _sample_cap-len(_scar_probe["raw_abs"]))].tolist())
                _scar_probe["raw_by_channel"] = _scar_probe["raw_by_channel"] if len(_scar_probe["raw_by_channel"]) == 8 else [[] for _ in range(8)]
                for _ci in range(min(8, _kick_raw_abs.shape[-1] if _kick_raw_abs.ndim else 1)):
                    _vals = (_kick_raw_abs[..., _ci] if _kick_raw_abs.ndim else np.asarray([_kick_raw_abs])).ravel()
                    _room = max(0, _sample_cap-len(_scar_probe["raw_by_channel"][_ci]))
                    _scar_probe["raw_by_channel"][_ci].extend(_vals[:_room].tolist())
                    _scar_probe["elements_total_by_channel"][_ci] += int(_vals.size)
                if _kmax > 0.0:
                    kick = np.clip(kick, -_kmax, _kmax)
                _kick_clipped_abs = np.abs(np.asarray(kick, dtype=float))
                _at_cap = bool(_kmax > 0.0 and np.any(_kick_clipped_abs >= _kmax - 1e-12))
                _scar_probe["at_cap_ticks"] += int(_at_cap)
                _scar_probe["at_cap_by_tick"].append(int(_at_cap))
                _at_cap_mask = (_kick_clipped_abs >= _kmax - 1e-12) if _kmax > 0.0 else np.zeros_like(_kick_clipped_abs, dtype=bool)
                _scar_probe["elements_at_cap"] += int(np.count_nonzero(_at_cap_mask))
                _scar_probe["elements_total"] += int(_kick_clipped_abs.size)
                _cl_flat = _kick_clipped_abs.ravel()
                _room = max(0, _sample_cap-len(_scar_probe["clipped_abs"]))
                _scar_probe["clipped_abs"].extend(_cl_flat[:_room].tolist())
                for _ci in range(min(8, _kick_clipped_abs.shape[-1] if _kick_clipped_abs.ndim else 1)):
                    _vals = (_kick_clipped_abs[..., _ci] if _kick_clipped_abs.ndim else np.asarray([_kick_clipped_abs])).ravel()
                    _scar_probe["elements_at_cap_by_channel"][_ci] += int(np.count_nonzero(_at_cap_mask[..., _ci])) if _at_cap_mask.ndim else int(bool(_at_cap_mask))
                    _scar_probe["clipped_by_channel"][_ci].extend(_vals[:max(0, _sample_cap-len(_scar_probe["clipped_by_channel"][_ci]))].tolist())
                p.model = np.clip(p.model + kick, -_MODEL_CLIP, _MODEL_CLIP)
                p.belief = np.clip(p.belief + kick * 0.4, -_BELIEF_CLIP, _BELIEF_CLIP)
        # Removed legacy periodic energy impulse and its associated clip.
        if Config.ENABLE_PHI_LABYRINTH:
            for p in self.patterns:
                if p.alive: p.move(self.field, t)
        self.resolve_competitions_spatial()
        current_hunger = self.compute_hunger_multiplier(avg_trust=avg_trust)
        for p in self.patterns:
            if p.alive:
                sz = len(p.cells)
                if sz <= Config.MAX_CELLS_SOFT_LIMIT: base_tax = 1.0
                elif sz <= Config.MAX_CELLS_HARD_LIMIT:
                    excess = sz - Config.MAX_CELLS_SOFT_LIMIT
                    base_tax = 1.0 + excess * Config.SIZE_TAX_RATE_SOFT
                else:
                    excess_soft = Config.MAX_CELLS_HARD_LIMIT - Config.MAX_CELLS_SOFT_LIMIT
                    excess_hard = sz - Config.MAX_CELLS_HARD_LIMIT
                    base_tax = 1.0 + excess_soft * Config.SIZE_TAX_RATE_SOFT + excess_hard * Config.SIZE_TAX_RATE_HARD
                gap = p.spirit_gap
                if gap > 1.2: tax_factor = 0.1
                elif gap > 0.9: tax_factor = 0.3
                elif gap > 0.6: tax_factor = 0.6
                else: tax_factor = 1.0
                p._size_tax_multiplier = 1.0 + (base_tax - 1.0) * tax_factor
                if p.age < 30: p._size_tax_multiplier = 1.0
        for p in self.patterns:
            if p.alive:
                size_tax = getattr(p, '_size_tax_multiplier', 1.0)
                p.metabolic_tax(self.field, current_hunger * size_tax)
                self._energy_taxed_this_step += getattr(p, '_last_metabolic_taken', 0.0)
        for p in self.patterns:
            if p.alive and p.should_die():
                p._deposit_final_testament()
                self._record_death_drag(p, 'natural_death')
                p.alive = False
                self.echo_system.store(p)
        dead = [p for p in self.patterns if not p.alive]
        for p in dead: self.echo_system.memory_echoes.pop(p.id, None)
        dead_ids = {p.id for p in dead}
        for alive_p in self.patterns:
            if alive_p.alive:
                for dead_id in dead_ids: alive_p.social_memory.pop(dead_id, None)
        soul_mult = self.selfreg.get_soul_recovery_multiplier()
        for p in self.patterns:
            if not p.alive: continue
            regen = 0.0
            if p.soul_weight < 0.5: regen += 0.008 * soul_mult
            grat = getattr(p, 'emotional_memory', {}).get('gratitude', 0.0)
            bind = getattr(p, 'last_phenomenal_binding', 0.5)
            if grat > 0.3: regen += 0.004 * soul_mult
            if bind > 0.4: regen += 0.003 * soul_mult
            regen *= (1 + 0.2 * getattr(p, '_phi_proxy', 0)) if Config.ENABLE_PHI_PROXY else 1.0
            p.soul_weight = min(1.0, p.soul_weight + regen)
        self._process_fold_transitions(t)
        self._rng_mark(t, "после fold-transitions")
        self._apply_kinesthetic_falls(t)
        self._force_disorganizer_by_state(t)
        alive = [p for p in self.patterns if p.alive]
        max_div = self.selfreg.get_division_capacity(len(alive))
        _md0 = max_div
        max_div = int(max_div * self._pop_birth_factor(len(alive)) + 0.5)   # мягкий тормоз: 400 -> 500 линейно к 0
        if _md0 > max_div:
            self._pop_ctr()['blocked']['division_brake_slots'] += (_md0 - max_div)
        if len(alive) < 70:
            if len(alive) < _pop_ceiling():
                with self._energy_span('emergency_rescue'): self.emergency_rescue(t)
                alive = [p for p in self.patterns if p.alive]
        with self._energy_span('_rare_crisis'): self._rare_crisis(t)
        with self._energy_span('spawn_from_unclaimed_field_energy'): self.spawn_from_unclaimed_field_energy(t)
        if t % getattr(Config, 'MARKED_VOLLEY_INTERVAL', 300) == 0:
            self._perform_marked_volley(t)
        if Config.ENABLE_UNRESOLVABLE_CORE_GUARD and alive:
            uc_vals = [float(getattr(pp, 'unresolved_contradiction', 1.0)) for pp in alive]
            at_floor_ratio = sum(1 for v in uc_vals if v < Config.UC_LIVING_THRESHOLD) / len(uc_vals)
            if at_floor_ratio >= Config.UC_POPULATION_ERASURE_RATIO: self.uc_floor_streak += 1
            else: self.uc_floor_streak = 0
            if self.uc_floor_streak == Config.UC_ERASURE_STREAK_LIMIT:
                self.witness.record(-1, "failure_unresolvable_core_erasure", t=t,
                                     at_floor_ratio=round(at_floor_ratio, 3), population=len(alive))
                self.unresolvable_core_journal.append({
                    't': t, 'at_floor_ratio': round(at_floor_ratio, 3),
                    'population': len(alive), 'action': 'reseed'})
                reseed_n = max(1, len(alive) // 20)
                for pp in alive[:reseed_n]:
                    pp.unresolved_contradiction = min(1.0, pp.unresolved_contradiction + Config.UC_RESEED_STRENGTH)
                self.uc_floor_streak = 0
        if t % 50 == 0 and t > 0:
            scar_agent_avg = safe_mean([float(getattr(pp, 'epistemic_scar', 0.0)) for pp in alive], 0.0) if alive else 0.0
            scar_shared = (Config.SHARED_SCARRING_ALPHA * scar_agent_avg +
                           (1 - Config.SHARED_SCARRING_ALPHA) * self.scar_operator)
            self.scar_shared_history.append((t, round(scar_shared, 4)))
            _alive_here = [p for p in self.patterns if p.alive]
            _E_f = float(self.field[:, :, CH['energy']].sum())
            _E_i = float(sum(getattr(p, 'energy', 0.0) for p in _alive_here))
            print(f"[t={t}] ⚡ E_field={_E_f:>10.1f} E_agent_inv={_E_i:>9.1f} "
                  f"E_total={_E_f+_E_i:>10.1f} ΔE_поля_с_старта={self._energy_ledger['step_total']:+.2f}",
                  flush=True)
            print(f"[t={t}] 832-lab: rescue_ev={len(self.rescue_journal)} "
                  f"spore_ev={len(self.spore_journal)} "
                  f"codecision_ev={len(self.codecision_journal)} "
                  f"scar_operator={self.scar_operator:.3f} scar_shared={scar_shared:.3f} "
                  f"uc_floor_streak={self.uc_floor_streak}", flush=True)
            if getattr(Config, 'ENABLE_CENTER_WALL', False):
                _wx = Config.CENTER_WALL_X
                _alive_c = [pp for pp in alive if pp.cells]
                _left = [pp for pp in _alive_c if pp.get_center()[0] < _wx]
                _right = [pp for pp in _alive_c if pp.get_center()[0] >= _wx]
                _gap_l = safe_mean([getattr(pp, 'spirit_gap', 0.0) for pp in _left], 0.0) if _left else 0.0
                _gap_r = safe_mean([getattr(pp, 'spirit_gap', 0.0) for pp in _right], 0.0) if _right else 0.0
                _sync_l = [pp for pp in _left if getattr(pp, '_phase_sync', 0.0) != 0.0]
                _sync_r = [pp for pp in _right if getattr(pp, '_phase_sync', 0.0) != 0.0]
                _ps_l = safe_mean([pp._phase_sync for pp in _sync_l], 0.0) if _sync_l else 0.0
                _ps_r = safe_mean([pp._phase_sync for pp in _sync_r], 0.0) if _sync_r else 0.0
                _jacc_str = "n/a"
                if self.EMERGENT_LEXICON.clusters:
                    _idside = {}
                    for pp in _left: _idside[pp.id] = 'L'
                    for pp in _right: _idside[pp.id] = 'R'
                    _use = {'L': Counter(), 'R': Counter()}
                    for cid, senders in self.EMERGENT_LEXICON.cluster_senders.items():
                        for sid, cnt in senders.items():
                            side = _idside.get(sid)
                            if side: _use[side][cid] += cnt
                    _tl = set(c for c, _ in _use['L'].most_common(5))
                    _tr = set(c for c, _ in _use['R'].most_common(5))
                    _un = _tl | _tr
                    if _un: _jacc_str = f"{len(_tl & _tr) / len(_un):.2f}"
                print(f"[t={t}] 🧱 wall: L={len(_left)} R={len(_right)} "
                      f"gap(L/R)={_gap_l:.3f}/{_gap_r:.3f} "
                      f"phase_sync(L/R)={_ps_l:+.3f}/{_ps_r:+.3f} "
                      f"dialect_jaccard={_jacc_str}", flush=True)
            ferals_now = [pp for pp in alive if pp.role_type == "feral"]
            normals_now = [pp for pp in alive if pp.role_type == "normal"]
            disorgs_now = [pp for pp in alive if pp.role_type == "disorganizer"]
            wsum = self.witness.summary()
            kills_now = int(wsum.get('feral_execution', 0))
            misses_now = int(self._feral_stats['misses'])
            convs_now = int(wsum.get('became_feral', 0))
            exits_now = int(wsum.get('became_normal', 0))
            resc_w = sum(1 for rec in self.rescue_journal if rec['t'] > t - 50)
            fd = self._feral_diag
            kills_w = max(0, kills_now - fd['last_snapshot_kills'])
            misses_w = max(0, misses_now - fd['last_snapshot_misses'])
            convs_w = max(0, convs_now - fd['last_snapshot_conversions'])
            fd['last_snapshot_kills'] = kills_now
            fd['last_snapshot_misses'] = misses_now
            fd['last_snapshot_conversions'] = convs_now
            fd['kills_total'] = kills_now; fd['kills_window'] = kills_w
            fd['misses_total'] = misses_now; fd['misses_window'] = misses_w
            fd['conversions_total'] = convs_now; fd['conversions_window'] = convs_w
            feral_share_cur, cap_cur = self.feral_share_state(alive)
            if ferals_now:
                furies = [pp._feral_fury for pp in ferals_now]
                fury_avg = float(np.mean(furies)); fury_max = float(np.max(furies)); fury_min = float(np.min(furies))
                sz_f = float(np.mean([len(pp.cells) for pp in ferals_now]))
                end_f = float(np.mean([pp._cellular_endurance for pp in ferals_now]))
                en_f = float(np.mean([pp.energy for pp in ferals_now]))
            else:
                fury_avg = fury_max = fury_min = 0.0
                sz_f = end_f = en_f = 0.0
            if normals_now:
                sz_n = float(np.mean([len(pp.cells) for pp in normals_now]))
                end_n = float(np.mean([pp._cellular_endurance for pp in normals_now]))
            else: sz_n = end_n = 0.0
            prey_cnt = sum(1 for pp in alive if pp.role_type != "feral" and is_feral_prey(pp))
            hit_rate_w = kills_w / max(1, kills_w + misses_w)
            print(f"[t={t}] 🐺 feral: F={len(ferals_now)} N={len(normals_now)} D={len(disorgs_now)} "
                  f"share={feral_share_cur:.3f}/{cap_cur:.3f} "
                  f"kills_w={kills_w}({hit_rate_w:.2f}) kills_t={kills_now} miss_t={misses_now} "
                  f"conv_w={convs_w} conv_t={convs_now} exit_t={exits_now} resc_w={resc_w} "
                  f"fury_avg={fury_avg:.2f}[{fury_min:.2f}..{fury_max:.2f}] "
                  f"prey={prey_cnt} sz_f={sz_f:.1f} sz_n={sz_n:.1f} "
                  f"end_f={end_f:.2f} end_n={end_n:.2f} en_f={en_f:.2f}", flush=True)
            _sa = getattr(self, '_feral_sanctuary_audit', {})
            _sz_on = bool(getattr(Config, 'ENABLE_FERAL_SANCTUARY', False))
            _sz_centers = getattr(self, '_feral_sanctuary_centers', []) or []
            print(f"[SAFE-ZONES t={t}] enabled={_sz_on} count={len(_sz_centers)} "
                  f"radius={float(getattr(Config, 'FERAL_SANCTUARY_RADIUS', 0.0)):.1f} "
                  f"centers={_sz_centers} checks={int(_sa.get('pattern_checks', 0))} "
                  f"patterns_inside={int(_sa.get('patterns_inside', 0))} "
                  f"prey_blocked={int(_sa.get('prey_search_blocked', 0))} "
                  f"attacks_blocked={int(_sa.get('attack_blocked', 0))} "
                  f"moves_blocked={int(_sa.get('movement_blocked', 0))}", flush=True)
            _ss = self._sanc_summary()
            if _ss:
                print(f"[SANC-RETURNS t={t}] seen={_ss['seen']} entered={_ss['entered']} re2={_ss['re2']} re3={_ss['re3']} "
                      f"entries={_ss['entries']} exits={_ss['exits']} inside_now={_ss['inside_now']} died_inside={_ss['died_inside']} "
                      f"stay mean/med={_ss['stay_mean']:.1f}/{_ss['stay_med']:.1f}", flush=True)
                print(f"[SANC-WHO t={t}] входов на агента: 1x={_ss['h1']} 2x={_ss['h2']} 3+x={_ss['h3p']} | макс={_ss['max_e']} (id={_ss['max_id']}) "
                      f"| родились внутри={_ss['born_in']} пришли сами={_ss['walked_in']} из них вернулись={_ss['walked_re']}", flush=True)
            _na = getattr(self, '_nav_audit_d', None)
            if _na:
                _nc = _na['c']
                print(f"[NAV t={t}] ep={_nc['ep_total']} arrived={_nc['ep_arrived']} abandoned={_nc['ep_abandoned']} "
                      f"new={_nc['wp_new']} recalc={_nc['recalc_total']} dir_chg={_nc['dir_changes']}/{_nc['dir_ticks']} "
                      f"hyst_off={_nc['hyst_override']} gapclip go/none/low/stuck={_nc['gapclip_go_goal']}/{_nc['gapclip_explore_no_goal']}/"
                      f"{_nc['gapclip_explore_low_conf']}/{_nc['gapclip_stuck_goal']}", flush=True)
                _mc = max(1, _nc['mv_call'])
                _own_ch = self.field[:, :, CH['owner']]
                _fill = np.count_nonzero(_own_ch) / _own_ch.size
                print(f"[MOVE t={t}] agent-ticks={_nc['mv_call']} try={100.0 * _nc['mv_try'] / _mc:.1f}% ok={100.0 * _nc['mv_ok'] / _mc:.1f}% "
                      f"blocked: no_free_dir={100.0 * _nc['mv_fail_nodir'] / _mc:.1f}% score={100.0 * _nc['mv_fail_thr'] / _mc:.1f}% occupied={100.0 * _nc['mv_fail_occ'] / _mc:.1f}% "
                      f"| knobs mult={float(getattr(Config, 'MOBILITY_MULT', 1.0)):.2f} thr={float(getattr(Config, 'MOVE_SCORE_THR', 0.3)):.2f} "
                      f"dirsteer={float(getattr(Config, 'NAV_DIR_STEER_GAIN', 0.0)):.2f}", flush=True)
                print(f"[SELFREG t={t}] phase={self.selfreg.phase} pop={len(alive)} fill={100.0 * _fill:.1f}% "
                      f"space_brake={self.selfreg.space_brake():.2f} (HARD={Config.POP_FILL_HARD:.2f}) "
                      f"inj_mult={self.selfreg.get_energy_injection_multiplier():.2f} max_div={max_div}", flush=True)
                _szn = np.array([len(q.cells) for q in alive if q.role_type != 'feral'] or [0])
                _szf = np.array([len(q.cells) for q in alive if q.role_type == 'feral'] or [0])
                print(f"[BODY t={t}] normal mean={_szn.mean():.1f} p90={np.percentile(_szn, 90):.0f} max={_szn.max()} "
                      f"| feral mean={_szf.mean():.1f} max={_szf.max()} | cap={int(getattr(Config, 'BODY_CELL_CAP', 0))} "
                      f"feral_cap={int(Config.FERAL_MAX_CELLS)} | над капом: normal={int((_szn > int(getattr(Config, 'BODY_CELL_CAP', 10**9))).sum())} "
                      f"feral={int((_szf > int(Config.FERAL_MAX_CELLS)).sum())}", flush=True)
                _bg = getattr(self, '_body_guard_stats', None)
                if _bg:
                    print(f"[GUARD t={t}] compete: клеток в пересечении={_bg.get('c_overlap_cells', 0)} передано={_bg.get('c_moved', 0)} "
                          f"заблокировано: внутри={_bg.get('c_block_interior', 0)} разрыв={_bg.get('c_block_split', 0)} "
                          f"| фералы: убийств={_bg.get('f_kills', 0)} взято={_bg.get('f_taken', 0)}/{_bg.get('f_want', 0)} "
                          f"выбор изменён={_bg.get('f_changed', 0)} (старый выбор: внутренних={_bg.get('f_old_interior', 0)} "
                          f"разрыв={_bg.get('f_old_split', 0)}) [накопительно]", flush=True)
        if len(alive) >= _pop_ceiling(): max_div = 0
        divisions_done = 0
        feral_share_now, feral_share_cap = self.feral_share_state(alive)
        feral_cnt_now = sum(1 for _q in alive if _q.role_type == "feral")
        for p in self.patterns[:]:
            if divisions_done >= max_div: break
            if not p.alive or not p.can_divide(): continue
            if p.role_type == "feral": continue
            if len(alive) + divisions_done >= _pop_ceiling():
                self._pop_note('blocked', 'division_ceiling')
                break
            child, new_id = p.divide(self.field, self.next_id)
            if child:
                if child.role_type == "feral": feral_cnt_now += 1
                self.next_id = new_id
                self.patterns.append(child)
                self.pattern_dict[child.id] = child
                divisions_done += 1
                try:   # тот же вид рождения, что пишет ledger (born_from_division / born_from_disorganizer)
                    _bk = _birth_info(child)[0]
                except Exception:
                    _bk = 'division'
                self._pop_note('births', _bk if _bk in ('division', 'disorganizer') else 'division')
                self.divisions_this_interval += 1
                self.total_divisions_ever += 1
                if len(child.cells) > GIANT_CELL_LIMIT:
                    all_cells = list(child.cells)
                    all_cells_sorted = sorted(all_cells, key=lambda c: phi_hash(c[0], c[1], child.id))
                    keep = set(all_cells_sorted[:GIANT_CELL_LIMIT])
                    for c in child.cells - keep:
                        self.field[c[0], c[1], CH['owner']] = 0
                    child.cells = keep
        for _dead in self.patterns:
            if not _dead.alive:
                # PNDIAG: висящие наблюдения новизны умершего агента — потерянные, а не исчезнувшие из баланса
                try:
                    _nv = getattr(_dead, '_nav', None)
                    if isinstance(_nv, dict) and _nv.get('pending') is not None:
                        _pn_pend_drop(_nv, _fch(self).st, 'agent_died')
                except Exception:
                    pass
                _e = float(_dead.energy)
                _to_ash = bool(Config.ASH_TO_FIELD and _e > 0.0 and _dead.cells)
                # В поле возвращается только _e > 0. Раньше -_e писалось в X2 и при _e < 0, из-за чего отрицательный
                # запас шёл в пару «смерть: прах» как приход (+2.76 расхождения). Теперь пара считает только положительную
                # часть, а списанный долг (отрицательная энергия) — отдельной честной строкой «приход без донора».
                if _to_ash or _e <= 0.0:
                    self._ledger_add_x2('X2.смерть: запас покидает агента', -max(_e, 0.0))
                else:
                    # HONEST ENERGY: вернуть прах некуда (нет клеток / ASH_TO_FIELD выключен) — запас объявлен рассеянным
                    self._ledger_add_x2('X2.смерть: запас рассеян (нет клеток для праха)', -_e)
                self._tick_acc['removed'] += 1
                if Config.ASH_TO_FIELD and _e > 0.0:
                    if _dead.cells:
                        _cl = sorted(_dead.cells)
                        _ax = np.array([c[0] for c in _cl])
                        _ay = np.array([c[1] for c in _cl])
                        _b = float(np.sum(self.field[_ax, _ay, CH['energy']], dtype=np.float64))
                        self.field[_ax, _ay, CH['energy']] += _e / len(_cl)
                        self._ledger_add('X1.смерть: прах (запас мёртвого агента возвращён в поле его клеток)',
                                         float(np.sum(self.field[_ax, _ay, CH['energy']], dtype=np.float64)) - _b)
                    else:
                        self._memo_add('смерть: запас потерян (у агента нет клеток для праха)', _e)
        _diag_all = {int(getattr(p, 'id', id(p))): p
                     for p in getattr(self, '_diagnostic_all_patterns', [])}
        _diag_all.update({int(getattr(p, 'id', id(p))): p for p in self.patterns})
        self._diagnostic_all_patterns = list(_diag_all.values())
        self.patterns = [p for p in self.patterns if p.alive]
        self.pattern_dict = {p.id: p for p in self.patterns}
        alive = [p for p in self.patterns if p.alive]
        crisis_level = float(np.mean(self.field[:,:,CH['crisis']]))
        decay_normal = self.selfreg.get_social_decay_rate(crisis_level)
        sov_ch = CH.get('signal_sovereignty', 29)
        sov_decay = np.clip(0.90 + crisis_level * 0.08, 0.85, 0.98)
        for ch in range(12, Config.CHANNELS):
            if ch == sov_ch: self.field[:,:,ch] *= sov_decay
            else: self.field[:,:,ch] *= decay_normal
        owner_mask = (self.field[:,:,CH['owner']] > 0).astype(np.float32)
        density = np.clip(uniform_filter(owner_mask, size=3), 0.3, 1.0)
        signal_cap = 0.85 - density * 0.15
        for ch in range(12, Config.CHANNELS):
            self.field[:,:,ch] = np.minimum(self.field[:,:,ch], signal_cap)
        current_energy = float(np.sum(self.field[:,:,CH['energy']]))
        drift = abs(current_energy - getattr(self, '_total_energy_last', current_energy))
        self._total_energy_last = current_energy
        if self.selfreg.is_stuck_in_phase("stagnation", min_steps=120):
            self.field[:, :, CH['unknown']] = np.clip(self.field[:, :, CH['unknown']] + 0.03, 0.0, 0.8)
            self.selfreg.boredom = max(0.0, self.selfreg.boredom - 0.15)
            if t % 200 == 0: print(f"[t={t}] Stagnation detected -> injected micro-perturbation")
        elif self.selfreg.is_stuck_in_phase("crisis", min_steps=100):
            self.selfreg.energy_drift = min(self.selfreg.energy_drift, 200)
        # No emergency energy clip: a large drift is exposed by the audit and blocks acceptance.
        for p in self.patterns:
            if not p.alive or p.age < 300: continue
            if p.semantic_state == "neutral" and p.spirit_gap < 0.1:
                if getattr(Config, 'ENABLE_FORCED_STATE_SHIFT', True) and phi_hash(p.id, t, 123456) < 0.35:
                    new_state = "seeking_comfort" if phi_hash(p.id, t, 123457) < 0.6 else "grateful_but_cautious"
                    p.semantic_state = new_state
                    p._log_event("forced_state_shift", from_="neutral", to=new_state, age=p.age)
                    self.witness.record(p.id, "forced_state_shift", from_state="neutral", to_state=new_state, age=p.age)
        self._conservation_check(t)
        avg_crisis_field = float(np.mean(self.field[:,:,CH['crisis']]))
        if avg_crisis_field > 0.45:
            self.field[:,:,CH['crisis']] *= 0.7
            rest_priority = avg_crisis_field * 4.0
            for p in alive:
                if p.soul_weight > 0.4 and not any(g.get('type') == 'rest' for g in getattr(p, 'goals', [])):
                    p.goals = []
                    p.goals.append({"type": "rest", "priority": rest_priority, "target": None, "age": 0,
                                    "persistence": 40, "_source": "crisis_relief"})
        # Honest-energy mode: no intermediate clip, mean clamp, or capacity rebalance.
        # If field values become extreme, the audit reports it; it is not silently repaired.
        for p in self.patterns:
            if p.alive: self.echo_system.field_whisper(p, self.field, t)
        alive = [p for p in self.patterns if p.alive]
        if alive:
            avg_concepts = sum(len(p.concept_graph.nodes) for p in alive) / len(alive)
            for p in alive:
                if (len(p.concept_graph.nodes) > 1.8 * avg_concepts and len(p.concept_graph.nodes) > 15
                        and not p._mutant_saved):
                    self.echo_system.store_anomaly(p, "mutant")
                    p._mutant_saved = True
                    self.witness.record(p.id, "mutant_detected", concepts=len(p.concept_graph.nodes))
        for p in alive:
            intent_switches = sum(1 for ev in list(p.biography)[-50:] if ev.get('type') == 'intent_switch')
            if intent_switches > 15 and p.age > 200 and not p._autonomous_saved:
                self.echo_system.store_anomaly(p, "autonomous_entity")
                p._autonomous_saved = True
                self.witness.record(p.id, "autonomous_entity_detected", switches=intent_switches)
        autonomous_detected = sum(1 for p in alive if p._autonomous_saved)
        redeemed_cnt = sum(1 for p in alive if p.event_counts.get('redeemed', 0) > 0)
        fold_count_total = sum(p.event_counts.get('fold', 0) for p in alive)
        disorg_cnt = sum(1 for p in alive if p.role_type == "disorganizer")
        event_signature = (autonomous_detected, redeemed_cnt, fold_count_total, disorg_cnt)
        new_hash = int(phi_hash(t, _stable_hash(event_signature), 99999) * 1000)
        self.system_entropy = int(0.85 * self.system_entropy + 0.15 * new_hash)
        unknown_interval = self.selfreg.get_unknown_spawn_interval()
        alive = [p for p in self.patterns if p.alive]
        if len(alive) < _pop_ceiling() and t % unknown_interval == 0:
            e_before = np.sum(self.field[:,:,CH['energy']])
            with self._energy_span('spawn_unknown_patterns'): self.spawn_unknown_patterns(t)
            self._energy_injected_this_step += np.sum(self.field[:,:,CH['energy']]) - e_before
            self.process_spores()
        self._rng_mark(t, "после деления/спор")
        for p in self.patterns:
            p.age += 1
            p.lineage_total_age += 1
        sem_interval = 3
        if t % sem_interval == 0:
            cell_to_pattern = {}
            for p in self.patterns:
                if not p.alive or p.role_type == "disorganizer": continue
                p.last_semantic_exchange_step = -sem_interval
                for (x, y) in p.cells: cell_to_pattern[(x, y)] = p.id
            exchanged_pairs = set()
            MAX_EXCHANGES_PER_STEP = 30
            exchanges_done = 0
            for p in self.patterns:
                if not p.alive or p.role_type == "disorganizer": continue
                effective_interval = sem_interval
                for pid, trust in p.trust_ledger.entries.items():
                    if trust > 0.9:
                        partner = self.pattern_dict.get(pid)
                        if partner and partner.alive and partner.trust_ledger.get(p.id, 0) > 0.9:
                            effective_interval = sem_interval // 2
                            break
                if t - p.last_semantic_exchange_step <= effective_interval: continue
                found_partner = None
                for (x, y) in p.cells:
                    for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
                        nx, ny = (x+dx) % Config.WORLD_SIZE, (y+dy) % Config.WORLD_SIZE
                        owner = cell_to_pattern.get((nx, ny), 0)
                        if owner != 0 and owner != p.id and owner in self.pattern_dict:
                            q = self.pattern_dict[owner]
                            if q.alive and q.role_type != "disorganizer":
                                q.last_semantic_exchange_step = -sem_interval
                                q_eff = sem_interval
                                for pid2, trust2 in q.trust_ledger.entries.items():
                                    if trust2 > 0.9:
                                        partner2 = self.pattern_dict.get(pid2)
                                        if partner2 and partner2.alive and partner2.trust_ledger.get(q.id, 0) > 0.9:
                                            q_eff = sem_interval // 2
                                            break
                                if t - q.last_semantic_exchange_step > q_eff:
                                    pair = (min(p.id, q.id), max(p.id, q.id))
                                    if pair not in exchanged_pairs:
                                        found_partner = q
                                        exchanged_pairs.add(pair)
                                        break
                    if found_partner: break
                if found_partner and exchanges_done < MAX_EXCHANGES_PER_STEP:
                    q = found_partner
                    sim = p.exchange_meanings(q, self.field, t)
                    p.last_semantic_exchange_step = t
                    q.last_semantic_exchange_step = t
                    exchanges_done += 1
        self._post_semantic_step(t)
        self._process_archive_autonomy(t)
        self._rng_mark(t, "после archive_autonomy")
        if t % self.selfreg.get_validation_interval() == 0:
            self.validate_invariants(t)
        try: self.core_chorus.tick(self, t)
        except Exception: pass
        alive_now = [pp for pp in self.patterns if pp.alive]
        lineage_diversity = len(set(pp.lineage_id for pp in alive_now)) / len(alive_now) if alive_now else 1.0
        diversity_pressure = max(0.0, min(1.0, (0.2 - lineage_diversity) * 5.0))
        coherence_threshold = (getattr(Config, 'FERAL_CONVERSION_COHERENCE_BASE', 0.70)
                               + diversity_pressure * getattr(Config, 'FERAL_CONVERSION_COHERENCE_DIV_BONUS', 0.10))
        gap_threshold = max(
            getattr(Config, 'FERAL_CONVERSION_GAP_FLOOR', 0.25),
            getattr(Config, 'FERAL_CONVERSION_GAP_BASE', 0.38)
            - diversity_pressure * getattr(Config, 'FERAL_CONVERSION_GAP_DIV_BONUS', 0.13))
        grief_threshold = max(
            getattr(Config, 'FERAL_CONVERSION_GRIEF_FLOOR', 0.40),
            getattr(Config, 'FERAL_CONVERSION_GRIEF_BASE', 0.60)
            - diversity_pressure * getattr(Config, 'FERAL_CONVERSION_GRIEF_DIV_BONUS', 0.20))
        MAX_FERAL_CONVERSIONS_PER_TICK = int(getattr(Config, 'MAX_FERAL_CONVERSIONS_PER_TICK', 6))
        conversions_this_tick = 0
        current_feral_share, FERAL_POPULATION_SHARE_CAP = self.feral_share_state(alive_now)
        feral_alive_now = sum(1 for pp in alive_now if pp.role_type == "feral")
        for p in self.patterns:
            if conversions_this_tick >= MAX_FERAL_CONVERSIONS_PER_TICK: break
            if (feral_alive_now + 1) / max(1, len(alive_now)) > FERAL_POPULATION_SHARE_CAP: break
            if not p.alive or p.role_type == "feral": continue
            if getattr(p, '_redemption_cooldown', 0) > p.age: continue
            if p.role_type == "disorganizer" and getattr(p, '_redemption_arc_step', 0) > 0: continue
            is_normal_ready = (p.role_type == "normal" and p.coherence < coherence_threshold and p.spirit_gap > gap_threshold)
            is_disorg_ready_to_rage = (p.role_type == "disorganizer" and p.emotional_memory.get('grief', 0.0) > grief_threshold)
            if (p.energy > 0.10 and len(p.cells) > 10 and p.age > 30 and
                (is_normal_ready or is_disorg_ready_to_rage)):
                p.become_feral()
                conversions_this_tick += 1
                feral_alive_now += 1
                current_feral_share = (feral_alive_now / len(alive_now)) if alive_now else 0.0
        for p in self.patterns:
            if not p.alive: continue
            if p.role_type == "feral":
                p.update_feral_fury(self.field)
                p.grow_feral(self.field)
                p.move_feral(self.field, t)
                p.apply_feral_intent(self.field)
        kill_radius = int(getattr(Config, 'FERAL_KILL_RADIUS', 14))
        for p in self.patterns:
            if not p.alive or p.role_type != "feral": continue
            prey, _, _ = p._find_prey(self.field, self,
                                      min_cells=Config.FERAL_PREY_MIN_CELLS,
                                      radius=kill_radius)
            if prey:
                self._feral_stats['prey_found'] += 1
                p.feral_execute(prey, self.field)
            else:
                self._feral_stats['no_prey'] += 1
        for p in self.patterns:
            if not p.alive or p.role_type == "feral": continue
            feral_sum = 0.0
            num_cells = len(p.cells)
            if num_cells > 0:
                for (x, y) in p.cells:
                    for dx in range(-3, 4):
                        for dy in range(-3, 4):
                            nx = (x + dx) % Config.WORLD_SIZE
                            ny = (y + dy) % Config.WORLD_SIZE
                            feral_sum += self.field[nx, ny, CH['signal_feral']]
                feral_nearby = feral_sum / (num_cells * 49.0)
            else:
                feral_nearby = 0.0

            if feral_nearby > 0.15:
                p.cognitive_tension = min(Config.MAX_METRIC, p.cognitive_tension + feral_nearby * 0.15)
        if hasattr(self, '_energy_flow_diag'):
            self._energy_flow_diag['injected_total'] += self._energy_injected_this_step
        self._audit_wall()
        _de = float(self.field[:, :, CH['energy']].sum(dtype=np.float64)) - _e_step0
        _L = self._energy_ledger
        _L['step_total'] += _de
        _L['step_measured'] += self._ledger_tick_measured
        _L['ticks'] += 1
        _res = _de - self._ledger_tick_measured
        self._audit_close_tick(t, _de, _res, _x2_step0)
        self._rng_mark(t, "конец step()")
        return True

    def _process_fold_transitions(self, t):
        st = self._fold_transition_stats
        for p in self.patterns:
            if not p.alive or p.role_type != "normal": continue
            fold_count = p.event_counts.get('fold', 0)
            if fold_count < 1: continue
            st['checked'] += 1
            if getattr(p, '_redemption_cooldown', 0) > p.age:
                st['blocked_by_cooldown'] += 1; continue
            if getattr(p, '_last_fold_roll_count', 0) == fold_count:
                st['already_rolled'] += 1; continue
            disorganizer_cnt = len([pp for pp in self.patterns if pp.alive and pp.role_type == "disorganizer"])
            alive_total = len([pp for pp in self.patterns if pp.alive])
            current_frac = disorganizer_cnt / max(1, alive_total)
            disorg_hard_cap = min(0.22, max(0.15, 0.15 + self.soul_weight_average * 0.08))
            if current_frac >= disorg_hard_cap:
                st['blocked_by_cap'] += 1; continue
            if fold_count >= Config.FOLDS_FOR_DEEP_FALL:
                prob, soul_target, redemption_delay, fall_type = (1.0, Config.DEEP_FALL_SOUL_PENALTY,
                                                                   Config.DEEP_FALL_REDEMPTION_DELAY, "fallen")
            else:
                prob, soul_target, redemption_delay, fall_type = (Config.QUICK_FALL_PROB,
                                                                   Config.BROKEN_FALL_SOUL_TARGET,
                                                                   Config.REDEMPTION_ARC_STEP_DELAY, "broken")
            p._last_fold_roll_count = fold_count
            if phi_hash(p.id, t, 777) >= prob: continue
            p.role_type = "disorganizer"
            p.emotional_memory['gratitude'] = 0.1
            p.emotional_memory['grief'] = 0.8
            p.semantic_state = "exploring_danger"
            p.intent = {"type": "explore", "priority": 2.0, "age": 0, "persistence": 9999}
            p.intent_commitment = 2.0
            p.disorganizer_age_at_birth = p.age
            p.redemption_timer = redemption_delay
            p._deterministic_redemption_triggered = False
            p._forced_soul_collapse_done = False
            p._redemption_arc_step = 0
            p._steps_since_trigger = 0
            p.soul_weight = min(p.soul_weight, soul_target)
            p._log_event(f"became_disorganizer_{fall_type}", folds=fold_count)
            self.witness.record(p.id, f"became_disorganizer_{fall_type}", folds=fold_count)
            st['fallen' if fall_type == "fallen" else 'broken'] += 1

    def _force_disorganizer_by_state(self, t):
        for p in self.patterns:
            if not p.alive or p.role_type == 'disorganizer': continue
            if getattr(p, '_redemption_cooldown', 0) > p.age: continue
            endurance = p._cellular_endurance
            if p.soul_weight < 0.1:
                p.role_type = 'disorganizer'
                p.semantic_state = 'exploring_danger'
                p.emotional_memory['grief'] = 0.8
                p.emotional_memory['gratitude'] = 0.1
                p.disorganizer_age_at_birth = p.age
                p.redemption_timer = getattr(Config, 'REDEMPTION_ARC_STEP_DELAY', 100)
                p._deterministic_redemption_triggered = False
                p._redemption_arc_step = 0
                p._steps_since_trigger = 0
                p._log_event('kinesthetic_fall', soul=p.soul_weight, endurance=endurance)

    def _apply_kinesthetic_falls(self, t): pass

    def run(self, steps=None, save_ark=True, force_fresh_seed=False):
        if Config.ENABLE_MORSE_COMMUNICATION:
            for p in self.patterns: p._morse_set_field(self.morse_field)
        if steps is None: steps = Config.STEPS
        self.field = self.init_field()
        self.scar = self.init_scar()
        self.niche = np.zeros((Config.WORLD_SIZE, Config.WORLD_SIZE)) if Config.ENABLE_NICHE else None
        if Config.ENABLE_MEANING_FIELD:
            _W = Config.WORLD_SIZE
            self._meaning_centers = [[int(phi_hash(i, 0, 8877) * _W), int(phi_hash(i, 1, 8877) * _W)] for i in range(5)]
            self._meaning_field = np.zeros((_W, _W))
        self.culture_ratchet = 0
        self._teach_prev = 0
        if self.core_chorus is not None:
            self.env_complexity = self.core_chorus.persistent.get('env_complexity', 1.0)
        self._total_energy_last = float(np.sum(self.field[:,:,CH['energy']]))
        self._energy_flow_diag = {'start_total': float(np.sum(self.field[:, :, CH['energy']])), 'injected_total': 0.0}
        self._energy_ledger = {'src': {}, 'phase': {}, 'x2': {},
                               'step_total': 0.0, 'step_measured': 0.0, 'ticks': 0}
        self._ledger_tick_measured = 0.0
        self._audit_init()
        if Config.WALL_CYCLE_ENABLED:
            Config.ENABLE_CENTER_WALL = bool(Config.WALL_CYCLE_BASE)
        self._econs_init()
        self._span_stack = []
        self._phase_last = None
        self._wall_diag = Counter()
        self._genome_diag = Counter()
        self.patterns = []
        self.pattern_dict = {}
        self._imprint_stats = ImprintStats()
        self.divisions_this_interval = 0
        self.selfreg.reset()
        self._feral_diag = {
            'kills_total': 0, 'kills_window': 0,
            'misses_total': 0, 'misses_window': 0,
            'conversions_total': 0, 'conversions_window': 0,
            'last_snapshot_kills': 0, 'last_snapshot_misses': 0, 'last_snapshot_conversions': 0,
        }
        if force_fresh_seed or not load_living_world(self, LIVING_WORLD_PATH):
            self.seed_initial_patterns()
        else:
            if self.patterns: self.next_id = max(p.id for p in self.patterns) + 1
            for p in self.patterns:
                if len(p.cells) > GIANT_CELL_LIMIT:
                    all_cells = list(p.cells)
                    all_cells_sorted = sorted(all_cells, key=lambda c: (c[0], c[1]))
                    keep = set(all_cells_sorted[:GIANT_CELL_LIMIT])
                    for c in p.cells - keep: self.field[c[0], c[1], CH['owner']] = 0
                    p.cells = keep
                if p.lineage_born_at_step > 0: p.lineage_born_at_step = 0
            self._migrate_wall_cells()
        init_all_chronic_counters(self)
        for t in range(steps):
            if self.step(t) is False:
                if Config.VERBOSE_LOGS: print(f"[t={t}] Simulation stopped due to population collapse")
                break
            self.field_voice.update(self.field, t)
            if t % 100 == 0:
                divisions = self.divisions_this_interval
                self.divisions_this_interval = 0
                if Config.ENABLE_CULTURE_RATCHET:
                    _teach = sum(p.event_counts.get('taught', 0) + p.event_counts.get('received_teaching', 0)
                                 + p.event_counts.get('wisdom_shared', 0) for p in self.patterns if p.alive)
                    _delta = _teach - self._teach_prev
                    self._teach_prev = _teach
                    if _delta > self.culture_ratchet:
                        self.culture_ratchet = _delta
                        self.witness.record(-1, "culture_ratchet_up", level=_delta)
                if Config.ENABLE_RED_QUEEN:
                    _alive_rq = [p for p in self.patterns if p.alive]
                    if _alive_rq:
                        _skill = 1.0 - min(1.0, safe_mean([p.pred_error for p in _alive_rq], 0.5))
                        _target = 0.8 + 0.8 * _skill
                        self.env_complexity += (_target - self.env_complexity) * 0.1
                if t % 50 == 0 and Config.ENABLE_COMB_GENERATOR:
                    _eq = self.archive.get_recent_echoes(limit=3, min_weight=0.5)
                    self._echo_cache = [e.get('excerpt', '')[:40] for e in _eq] if _eq else []
                m = self._collect_metrics(t)
                m['subjects'] = sum(1 for p in self.patterns if getattr(p, '_subject_detected', False))
                alive = [p for p in self.patterns if p.alive]
                _unc = [p for p in alive if getattr(p, '_unconquered_strength', 0.0) > 0.4]
                m['unconquered_count'] = len(_unc)
                m['unconquered_wise'] = sum(1 for p in _unc if getattr(p, '_unconquered_type', None) == 'wise')
                m['unconquered_rebel'] = sum(1 for p in _unc if getattr(p, '_unconquered_type', None) == 'rebel')
                m['avg_unconquered_str'] = float(np.mean([getattr(p, '_unconquered_strength', 0.0) for p in alive])) if alive else 0.0
                sov_ch = CH.get('signal_sovereignty', 29)
                m['sovereignty_field_avg'] = float(np.max(self.field[:,:,sov_ch])) if self.field.size else 0.0
                m['culture_ratchet'] = self.culture_ratchet
                m['env_complexity'] = round(self.env_complexity, 3)
                m['avg_niche'] = float(np.mean(self.niche)) if getattr(self, 'niche', None) is not None else 0.0
                m['given_presence_count'] = self.witness.summary().get('given_presence', 0)
                self.metrics_history.append(m)
                max_gap_clip_display = min(m.get('max_gap_clip', 0.0), 1.0)
                print(f"[t={t}] P={m['patterns']} L={m['lineages']} LING={m.get('lineage_count',0)} L_AGE={m.get('avg_lineage_age',0):.0f}(max={m.get('max_lineage_age',0)}) ANC={m.get('ancient_lineages',0)} TOT_AGE={m.get('max_lineage_total_age',0)} E={m['err']:.3f} S={m['soul']:.3f} B={m['binding']:.3f} T={m['avg_trust']:.2f} TA={m['triadic_alive_ratio']:.2f} LP={m['love_pairs']} INT_GAP={m['avg_internal_gap']:.3f}(max={m['max_internal_gap']:.3f}) GAP_CLIP={m['avg_gap_clip']:.3f}(max={max_gap_clip_display:.3f}) ZERO_INT={m['zero_internal_gap_agents']} TARGETS={m['tremor_targets']} D={m['disorganizer_count']} R={m['redeemed_count']} SUB={m['subjects']} SIG={m['avg_signal_memory']:.1f} INTENT={m['intentional_signal_agents']} DIV={divisions} VOC={m.get('vocab_events',0)} BLN={m.get('blind_events',0)} alarm={m.get('avg_social_alarm',0):.2f} beau={m.get('avg_social_beauty',0):.2f} rhy={m.get('avg_social_rhythm',0):.2f} int={m.get('avg_social_interest',0):.2f} mem={m.get('avg_social_memory',0):.2f} sil={m.get('avg_social_silence',0):.2f} UNC={m.get('unconquered_count',0)}(w={m.get('unconquered_wise',0)}/r={m.get('unconquered_rebel',0)}) sov={m.get('sovereignty_field_avg',0):.2f} PHI={m.get('avg_phi',0):.3f} CULT={m.get('culture_ratchet',0)} ENV={m.get('env_complexity',1.0):.2f} NICHE={m.get('avg_niche',0):.3f}", flush=True)
                _im = self.imprint_summary(t)
                print(f"[IMPRINT t={t}] contacts={_im['contacts']} pairs={_im['directed_pairs_ever']} imprints={_im['imprints']} "
                      f"attract={_im.get('attract_share', 0.0):.0%} repel={_im.get('repel_share', 0.0):.0%} strain={_im['strain_mean']:.3f} "
                      f"match/conflict/overload/prior/quiet={_im['case']['match']}/{_im['case']['conflict']}/{_im['case']['overload']}/{_im['case']['prior']}/{_im['case']['quiet']}", flush=True)
        save_living_world(self.patterns, LIVING_WORLD_PATH)
        self._print_world_diag()
        if Config.WALL_CYCLE_ENABLED:
            Config.ENABLE_CENTER_WALL = bool(Config.WALL_CYCLE_BASE)
        return self.patterns, self.field, self.scar, self.metrics_history

    def validate_invariants(self, t):
        alive = [p for p in self.patterns if p.alive]
        if not alive: return
        gs = self._guardian_stats
        trust_vals = []; high_trust_pairs = 0; coop_signals = 0
        for p in alive:
            entries = list(p.trust_ledger.entries.values())
            trust_vals.extend(entries)
            high_trust_pairs += sum(1 for v in entries if v > 0.95)
            if p.intent and p.intent.get('type') == 'cooperate': coop_signals += 1
        avg_trust = safe_mean(trust_vals, Config.TRUST_BASE)
        gs['love_avg_trust'] = avg_trust
        gs['love_high_trust_pairs'] = high_trust_pairs
        gs['love_density'] = high_trust_pairs / max(1, len(alive)*(len(alive)-1)//2)
        gs['love_coop_signals'] = coop_signals
        gs['love_population'] = len(alive)
        if t % 200 == 0 and self.metrics_history:
            last_m = self.metrics_history[-1]
            if last_m.get('love_pairs', 0) == 0 and last_m.get('patterns', 0) > 20:
                if gs.get('love_vanished_at') is None: gs['love_vanished_at'] = t
            if last_m.get('triadic_alive_ratio', 1.0) < 0.1:
                if gs.get('triadic_alert_at') is None: gs['triadic_alert_at'] = t

    def sanitize_agent_fields(self):
        for p in self.patterns:
            if not getattr(p, 'alive', False): continue
            numeric_fields = ['soul_weight', 'spirit_gap', 'protection_level',
                              'self_phenomenal_prediction', 'self_phenomenal_error',
                              '_linguistic_confidence', '_last_binding', 'last_phenomenal_binding',
                              '_self_surprise_count', '_subject_protection_timer',
                              'age', 'linguistic_confidence', 'pred_error', 'coherence',
                              'epistemic_load', 'cognitive_tension']
            for field in numeric_fields:
                val = getattr(p, field, None)
                if val is None: continue
                if isinstance(val, dict):
                    num = None
                    for key in ['value', 'count', 'val', 'amount']:
                        if key in val and isinstance(val[key], (int, float)):
                            num = float(val[key]); break
                    if num is None: num = 0.5
                    setattr(p, field, num)
                elif not isinstance(val, (int, float)):
                    try: setattr(p, field, float(val))
                    except (ValueError, TypeError): setattr(p, field, 0.5)
            for key in ['gratitude', 'grief']:
                val = p.emotional_memory.get(key)
                if isinstance(val, dict):
                    num = None
                    for k in ['value', 'count']:
                        if k in val and isinstance(val[k], (int, float)):
                            num = float(val[k]); break
                    if num is None: num = 0.5
                    p.emotional_memory[key] = num
                elif not isinstance(val, (int, float)):
                    try: p.emotional_memory[key] = float(val) if val is not None else 0.5
                    except Exception: p.emotional_memory[key] = 0.5

    def _post_semantic_step(self, t):
        self.sanitize_agent_fields()
        for p in self.patterns:
            if not getattr(p, 'alive', False): continue
            if not hasattr(p, '_self_narrative'):
                p.self_phenomenal_prediction = 0.5
                p.self_phenomenal_error = 0.0
                p._self_narrative = deque(maxlen=getattr(Config, 'EPISODIC_BUFFER_MAX_LEN', 3000))
                p._subject_detected = False
                p._last_binding = 0.5
                p._self_surprise_count = 0
                p._subject_protection_timer = 0
                p._linguistic_confidence = 0.5
            raw_binding = getattr(p, 'last_phenomenal_binding', 0.5)
            if isinstance(raw_binding, dict): raw_binding = raw_binding.get('value', 0.5)
            actual_binding = float(raw_binding)
            p.self_phenomenal_error = abs(actual_binding - p.self_phenomenal_prediction)
            p.self_phenomenal_prediction = 0.95 * p.self_phenomenal_prediction + 0.05 * actual_binding
            narrative = getattr(p, '_self_narrative', None)
            if narrative is None: narrative = deque(maxlen=getattr(Config, 'EPISODIC_BUFFER_MAX_LEN', 3000))
            narrative.append({'origin': 'OWN', 't': p.age, 'type': 'binding', 'binding': actual_binding,
                              'soul': float(getattr(p, 'soul_weight', 0.5)),
                              'gap': float(getattr(p, 'spirit_gap', 0.5))})
            p._self_narrative = narrative
            last_bind_raw = getattr(p, '_last_binding', 0.5)
            if isinstance(last_bind_raw, dict): last_bind_raw = last_bind_raw.get('value', 0.5)
            last_bind = float(last_bind_raw)
            delta = abs(actual_binding - last_bind)
            if delta > 0.15: p._self_surprise_count = getattr(p, '_self_surprise_count', 0) + 1
            p._last_binding = actual_binding
        for p in self.patterns:
            if not getattr(p, 'alive', False): continue
            pt = getattr(p, '_pending_transition', None)
            if pt and self.age >= pt['check_at']:
                if p.semantic_state != pt['from_state']:
                    p.concept_graph.record_transition(pt['from_state'], p.semantic_state)
                    p._log_event("dialogue_transition", from_state=pt['from_state'], to_state=p.semantic_state)
                del p._pending_transition
        for p in self.patterns:
            if getattr(p, 'alive', False) and getattr(p, '_subject_detected', False):
                sg = getattr(p, 'spirit_gap', 0.1)
                if isinstance(sg, dict): sg = sg.get('value', 0.1)
                p.spirit_gap = max(float(sg), 0.1)
        self._conceptual_resonance_step(t)
        if t % 10 == 0: self._auto_dialogue_tick(t)
        self._rng_mark(t, "после auto_dialogue/Морзе-диалогов")
        if Config.ENABLE_COMB_GENERATOR and t % 50 == 0:
            _alive_ig = [p for p in self.patterns if p.alive and getattr(p, 'last_phenomenal_report', '')]
            if _alive_ig:
                _n_samples = min(2, len(_alive_ig))
                _idxs = self._given_rng.choice(len(_alive_ig), size=_n_samples, replace=False)
                for _i in _idxs: print(f"💭 [INNER #{_alive_ig[_i].id}] {_alive_ig[_i].last_phenomenal_report}")
            _narr_agents = [p for p in self.patterns if p.alive and getattr(p, '_self_narrative', None)]
            if _narr_agents:
                _p2 = _narr_agents[int(self._given_rng.rand() * len(_narr_agents))]
                _last = _p2._self_narrative[-1]
                if isinstance(_last, dict) and _last.get('t', -1) >= t - 10:
                    print(f"🔮 [INTROSPECT #{_p2.id}] { {k: v for k, v in _last.items() if k != '_source'} }")
        SELF_ERROR_MAX = 0.20; SPIRIT_GAP_MIN = 0.10; MIN_VARIANCE = 0.015; STABILITY_MAX = 0.97
        FERAL_SURPRISE_REQUIRED = 3
        for p in self.patterns:
            if not getattr(p, 'alive', False) or getattr(p, '_subject_detected', False): continue
            narrative = getattr(p, '_self_narrative', None)
            if narrative is None: continue
            narrative_list = list(narrative)
            if len(narrative_list) < 12: continue
            numeric_narrative = []
            for x in narrative_list:
                if isinstance(x, dict): val = x.get('binding', x.get('soul', x.get('gap', 0.5)))
                else: val = x
                try: numeric_narrative.append(float(val))
                except (ValueError, TypeError): numeric_narrative.append(0.5)
            if len(numeric_narrative) > 1:
                narrative_variance = float(np.std(numeric_narrative)); narrative_stability = 1.0 - narrative_variance
            else: narrative_variance = 0.0; narrative_stability = 0.5
            min_surprises = FERAL_SURPRISE_REQUIRED if p.role_type == "feral" else getattr(Config, 'SUBJECT_MIN_SURPRISE_COUNT', 2)
            knows_itself = p.self_phenomenal_error < SELF_ERROR_MAX
            world_surprises = float(getattr(p, 'spirit_gap', 0.0)) > SPIRIT_GAP_MIN
            had_surprises = getattr(p, '_self_surprise_count', 0) >= min_surprises
            binding_ok = float(np.mean(numeric_narrative)) > 0.15
            holds_contradiction = getattr(p, 'unresolved_contradiction', 0.0) > 0.1
            if (knows_itself and world_surprises and narrative_variance > MIN_VARIANCE and
                narrative_stability < STABILITY_MAX and had_surprises and binding_ok and holds_contradiction):
                p._subject_detected = True
                p.protection_level = max(getattr(p, 'protection_level', 0.0), 0.7)
                p._subject_protection_timer = 100
                self.archive.deposit(p, "subject_emerged", weight=0.9,
                                     text=f"soul={p.soul_weight:.2f} | stability={narrative_stability:.3f}")
                p._log_event("subject_emerged", stability=round(narrative_stability, 3),
                             variance=round(narrative_variance, 3),
                             self_error=round(p.self_phenomenal_error, 3),
                             spirit_gap=round(p.spirit_gap, 3),
                             surprises=p._self_surprise_count, role=p.role_type)
                self.echo_system.store_anomaly(p, "subject")
                self.witness.record(p.id, "SUBJECT_EMERGED", age=p.age, soul=p.soul_weight,
                                    state=p.semantic_state, role=p.role_type)
        for p in self.patterns:
            if not getattr(p, 'alive', False) or not getattr(p, '_subject_detected', False): continue
            timer = getattr(p, '_subject_protection_timer', 0)
            if timer > 0:
                p.protection_level = max(getattr(p, 'protection_level', 0.0), 0.5)
                p._subject_protection_timer = timer - 1
            else: p.protection_level = max(getattr(p, 'protection_level', 0.0), 0.2)
        SELF_ERROR_MAX = getattr(Config, 'SUBJECT_SELF_ERROR_MAX', 0.12)
        SPIRIT_GAP_MIN = getattr(Config, 'SUBJECT_SPIRIT_GAP_MIN', 0.25)
        MIN_VARIANCE = getattr(Config, 'SUBJECT_MIN_NARRATIVE_VARIANCE', 0.02)
        STABILITY_MAX = getattr(Config, 'SUBJECT_NARRATIVE_STABILITY_MAX', 0.95)
        for p in self.patterns:
            if not getattr(p, 'alive', False) or getattr(p, '_subject_detected', False): continue
            min_surprises = 9 if p.role_type == "feral" else getattr(Config, 'SUBJECT_MIN_SURPRISE_COUNT', 2)
            narrative = getattr(p, '_self_narrative', None)
            if narrative is None: continue
            narrative_list = list(narrative)
            if len(narrative_list) < 20: continue
            numeric_narrative = []
            for x in narrative_list:
                if isinstance(x, dict): val = x.get('soul', x.get('gap', 0.5))
                else:
                    try: val = float(x)
                    except (ValueError, TypeError): val = 0.5
                numeric_narrative.append(float(val))
            if len(numeric_narrative) > 1:
                narrative_variance = float(np.std(numeric_narrative)); narrative_stability = 1.0 - narrative_variance
            else: narrative_variance = 0.0; narrative_stability = 0.5
            knows_itself = p.self_phenomenal_error < SELF_ERROR_MAX
            world_surprises = float(getattr(p, 'spirit_gap', 0.0)) > SPIRIT_GAP_MIN
            had_surprises = getattr(p, '_self_surprise_count', 0) >= min_surprises
            binding_ok = float(np.mean(numeric_narrative)) > 0.15
            holds_contradiction = getattr(p, 'unresolved_contradiction', 0.0) > 0.1
            if (knows_itself and world_surprises and narrative_variance > MIN_VARIANCE and
                narrative_stability < STABILITY_MAX and had_surprises and binding_ok and holds_contradiction):
                p._subject_detected = True
                p.protection_level = max(getattr(p, 'protection_level', 0.0), 0.7)
                p._subject_protection_timer = 100
                if p.role_type == "feral":
                    p._log_event("feral_became_subject", soul=p.soul_weight)
                    p._feral_fury = min(3.0, p._feral_fury + 0.5)
                    self.archive.deposit(p, "awakened_feral", weight=2.0, text=f"Wilderness gained soul. Fury={p._feral_fury:.2f}")
                else:
                    self.archive.deposit(p, "subject_emerged", weight=0.9,
                                         text=f"soul={p.soul_weight:.2f} | stability={narrative_stability:.3f}")
                p._log_event("subject_emerged", stability=round(narrative_stability, 3),
                             variance=round(narrative_variance, 3),
                             self_error=round(p.self_phenomenal_error, 3),
                             spirit_gap=round(p.spirit_gap, 3),
                             surprises=p._self_surprise_count, feral=(p.role_type == "feral"))
                self.echo_system.store_anomaly(p, "subject" if p.role_type != "feral" else "awakened_feral")
                self.witness.record(p.id, "SUBJECT_EMERGED", age=p.age, soul=p.soul_weight,
                                    state=p.semantic_state, feral=(p.role_type == "feral"))

    def _encounter_step(self, t):
        stats = self._imprint_stats
        stats.begin_tick()
        if not Config.IMPRINT_ENABLED: return
        wall_x = Config.CENTER_WALL_X if Config.ENABLE_CENTER_WALL else None
        contacts = imprint_contacts(self.field[:, :, CH['owner']], Config.IMPRINT_RADIUS, wall_x)
        acc = {}; by_b = {}
        for (aid, bid), cnt in sorted(contacts.items()):
            a = self.pattern_dict.get(aid); b = self.pattern_dict.get(bid)
            if a is None or b is None or not a.alive or not b.alive: continue
            if a.role_type == "feral" or b.role_type == "feral": continue
            w = cnt / (cnt + Config.IMPRINT_CONTACT_HALF)
            s, res, conf, over, prior, s_stranger, repeat, rec, dg, dr, dd = imprint_contact_update(a, b, w, t)
            acc[aid] = acc.get(aid, 0.0) + s
            stats.record(aid, bid, res, conf, over, prior, s, s_stranger, repeat, rec, dg, dr, dd)
            by_b.setdefault(bid, []).append((rec, dg, aid))
        stats.record_same_b(by_b)
        d = Config.IMPRINT_STRAIN_DECAY; normals = []
        for p in self.patterns:
            if not p.alive: continue
            p._enc_strain = min(Config.IMPRINT_STRAIN_MAX, d * p._enc_strain + (1.0 - d) * acc.get(p.id, 0.0))
            if p.role_type != "feral": normals.append(p)
        stats.track_steering(normals, Config.WORLD_SIZE, Config.IMPRINT_PULL_RANGE)

    def imprint_tick_row(self):
        tk = self._imprint_stats.tick
        normal = [p for p in self.patterns if p.alive and p.role_type != "feral"]
        st = np.array([p._enc_strain for p in normal], dtype=float)
        n_imp = [len(p.imprints.by_other) for p in normal]
        return {"contacts": tk["contacts"], "case": dict(tk["case"]),
                "case_strain": {k: float(v) for k, v in tk["case_strain"].items()},
                "strain_sum": float(tk["strain_sum"]), "new": tk["new_imprints"],
                "repeat": tk["repeat"], "repeat_known": float(tk["repeat_known"]),
                "repeat_stranger": float(tk["repeat_stranger"]), "repeat_differs": tk["repeat_differs"],
                "recept": {g: {k: float(v) for k, v in r.items()} for g, r in tk["recept"].items()},
                "sameB": {k: float(v) for k, v in tk["sameB"].items()},
                "steer": {b: {k: float(v) for k, v in r.items()} for b, r in tk["steer"].items()},
                "agents": len(normal), "agents_with_imprint": int(sum(1 for k in n_imp if k > 0)),
                "imprints": int(sum(n_imp)),
                "strain_mean": float(st.mean()) if len(st) else 0.0,
                "strain_p90": float(np.percentile(st, 90)) if len(st) else 0.0,
                "strain_gt01": int((st > 0.1).sum()),
                "strain_at_max": int((st >= Config.IMPRINT_STRAIN_MAX - 1e-9).sum()),
                "freeze_mean": float(np.mean(1.0 / (1.0 + Config.IMPRINT_FREEZE_GAIN * st))) if len(st) else 1.0}

    def imprint_tick_contacts(self): return sorted(self._imprint_stats.tick["pairs"])

    def imprint_track_row(self):
        cols = {k: [] for k in ("id", "lin", "gap", "soul", "pe", "coh", "end", "bind", "en", "st", "ni")}
        for p in sorted((q for q in self.patterns if q.alive and q.role_type != "feral"), key=lambda q: q.id):
            if p.cells:
                xs, ys = zip(*p.cells)
                bind = float(np.mean(np.clip(self.field[xs, ys, CH['binding']], 0, 1)))
                en = float(np.mean(self.field[xs, ys, CH['energy']]))
            else:
                bind = en = 0.0
            cols["id"].append(int(p.id)); cols["lin"].append(int(p.lineage_id))
            cols["gap"].append(round(float(p.spirit_gap), 4)); cols["soul"].append(round(float(p.soul_weight), 4))
            cols["pe"].append(round(float(p.pred_error), 4)); cols["coh"].append(round(float(p.coherence), 4))
            cols["end"].append(round(float(p._cellular_endurance), 4))
            cols["bind"].append(round(bind, 4)); cols["en"].append(round(en, 4))
            cols["st"].append(round(float(p._enc_strain), 4))
            cols["ni"].append(int(len(p.imprints.by_other)))
        return cols

    def imprint_summary(self, t=None):
        st = self._imprint_stats
        normal = [p for p in self.patterns if p.alive and p.role_type != "feral"]
        imps = [(p, bid, i) for p in normal for bid, i in p.imprints.by_other.items()]
        out = {"ticks": st.ticks, "contacts": st.contacts,
               "directed_pairs_ever": len(st.pairs), "case": dict(st.case),
               "case_strain_mean": {k: st.case_strain[k] / max(1, st.case[k]) for k in st.case},
               "agents_with_imprint": len({p.id for p, _, _ in imps}),
               "imprints": len(imps),
               "strain_mean": float(np.mean([p._enc_strain for p in normal])) if normal else 0.0}
        if imps:
            g = np.array([i.gravity for _, _, i in imps]); r = np.array([i.resonance for _, _, i in imps])
            dg = np.array([i.danger for _, _, i in imps])
            out["resonance_q"] = [float(x) for x in np.percentile(r, (10, 50, 90))]
            out["danger_q"] = [float(x) for x in np.percentile(dg, (10, 50, 90))]
            out["gravity_q"] = [float(x) for x in np.percentile(g, (10, 50, 90))]
            out["resonance_p50"] = out["resonance_q"][1]
            out["danger_p50"], out["danger_p90"] = out["danger_q"][1], out["danger_q"][2]
            out["gravity_p10"], out["gravity_p50"], out["gravity_p90"] = out["gravity_q"]
            out["attract_share"] = float(np.mean(g > 0.02))
            out["repel_share"] = float(np.mean(g < -0.02))
            out["n_contacts_q"] = [float(x) for x in np.percentile([i.n for _, _, i in imps], (50, 90))]
            if t is not None: out["age_q"] = [float(x) for x in np.percentile([t - i.last_t for _, _, i in imps], (50, 90))]
            within = [np.std([i.gravity for i in p.imprints.by_other.values()])
                      for p in normal if len(p.imprints.by_other) >= 3]
            out["within_agent_gravity_std"] = float(np.mean(within)) if within else None
            out["overall_gravity_std"] = float(np.std(g))
            out["to_dead"] = int(sum(1 for _, bid, _ in imps if bid not in self.pattern_dict))
            book = {(p.id, bid): i.gravity for p, bid, i in imps}
            both = [(v, book[(b, a)]) for (a, b), v in book.items() if a < b and (b, a) in book]
            out["reciprocal_pairs"] = len(both)
            if both:
                x = np.array([u for u, _ in both]); y = np.array([v for _, v in both])
                out["reciprocal_same_sign"] = float(np.mean(np.sign(x) == np.sign(y)))
            diffs = []
            for p, bid, i in imps:
                b = self.pattern_dict.get(bid)
                if b is None or not b.alive or i.n < 2: continue
                s1, _ = encounter_response(p, b, i); s0, _ = encounter_response(p, b, None)
                diffs.append(abs(s1 - s0))
            out["repeat_pairs"] = len(diffs)
            out["repeat_strain_differs_share"] = float(np.mean(np.array(diffs) > 0.01)) if diffs else 0.0
        return out

    def _auto_dialogue_tick(self, t):
        if not Config.ENABLE_MORSE_COMMUNICATION: return
        alive = [p for p in self.patterns if p.alive and getattr(p, '_dialogue_cooldown', -1) <= t]
        if len(alive) < 2: return
        n_pairs = max(1, min(6, len(alive) // 12))
        scored = []
        for p in alive:
            last_spoke = getattr(p, '_last_auto_dialogue_step', -10000)
            silence = t - last_spoke; tie = phi_hash(p.id, t, 111)
            scored.append((silence + tie, p))
        scored.sort(key=lambda x: -x[0])
        paired_ids = set(); started = 0
        for _, speaker in scored:
            if started >= n_pairs: break
            if speaker.id in paired_ids or getattr(speaker, '_in_dialogue', False): continue
            found = None
            for (x, y) in speaker.cells:
                for dx in range(-2, 3):
                    for dy in range(-2, 3):
                        if dx == 0 and dy == 0: continue
                        nx = (x + dx) % Config.WORLD_SIZE; ny = (y + dy) % Config.WORLD_SIZE
                        owner = int(self.field[nx, ny, CH['owner']])
                        if owner != 0 and owner != speaker.id and owner in self.pattern_dict:
                            listener = self.pattern_dict[owner]
                            if (listener.alive and getattr(listener, '_dialogue_cooldown', -1) <= t
                                    and listener.id not in paired_ids):
                                found = listener; break
                    if found: break
                if found: break
            if not found: continue
            speaker._in_dialogue = True; found._in_dialogue = True
            speaker._dialogue_cooldown = t + Config.MORSE_DIALOGUE_COOLDOWN
            found._dialogue_cooldown = t + Config.MORSE_DIALOGUE_COOLDOWN
            speaker._last_auto_dialogue_step = t; found._last_auto_dialogue_step = t
            paired_ids.add(speaker.id); paired_ids.add(found.id)
            started += 1
            speaker._send_morse(None, self.field, found.id)
            speaker.remember_dialogue(found.id, "[AUTO-DIALOG contact]", t)
            found.remember_dialogue(speaker.id, "[AUTO-DIALOG contact]", t)
            speaker._in_dialogue = False; found._in_dialogue = False


print("✅ Cell 4-1: EvolutionEngine полностью собран (X2-AUDIT A.1, A.2, A.3, C.1–C.5).")


In [ ]:
# Ячейка 13
import queue
import threading
import time
import json
import os
import sys
from collections import deque
import numpy as np

try:
    from __main__ import build_agent_voice
except ImportError:
    build_agent_voice = globals().get('build_agent_voice')
    if build_agent_voice is None:
        def build_agent_voice(p, engine=None, compact=False):
            return f"Ты — Паттерн #{p.id}. Состояние: {p.semantic_state}. Скажи что-нибудь."

def _safe_float(value, default=0.0):
    if isinstance(value, dict):
        for key in ['value', 'count', 'val', 'amount']:
            if key in value and isinstance(value[key], (int, float)):
                return float(value[key])
        return default
    if isinstance(value, (int, float)):
        return float(value)
    return default

_CHORUS_STATE_PATH = "/content/drive/MyDrive/832_Archive/chorus_state.json" \
    if 'google.colab' in sys.modules else "832_chorus_state.json"

def _load_chorus_state():
    default = {
        "total_dialogues": 0,
        "total_concepts_shared": 0,
        "runs": 0,
        "wisdom": 0.0,
        "culture_ratchet_last_run": 0,
        "env_complexity": 1.0,
        "given_seed": None,
        "FIELD_DIFFUSION_D": 0.05,
        "FIELD_DISSIPATION_K": 0.02,
    }
    try:
        if os.path.exists(_CHORUS_STATE_PATH):
            with open(_CHORUS_STATE_PATH, 'rb') as f:
                data = json.loads(f.read().decode('utf-8', errors='replace'))
            default.update(data)
    except Exception:
        pass
    default["runs"] += 1
    return default

def _save_chorus_state(state):
    try:
        os.makedirs(os.path.dirname(_CHORUS_STATE_PATH) or ".", exist_ok=True)
        with open(_CHORUS_STATE_PATH, "w", encoding="utf-8") as f:
            json.dump(state, f, ensure_ascii=False, indent=2)
    except Exception:
        pass

_chorus_state = _load_chorus_state()

class CoreChorus:
    def __init__(self, persistent_state=None):
        self.last_sync_step = -999
        self.current_voices = []
        self.history = deque(maxlen=50)
        self._prev_state = None
        self.min_cooldown = 3
        self.dialogue_log = deque(maxlen=30)
        self.dialogue_journal = deque(maxlen=500)
        self._regular_interval = 10
        self._last_voice_step = {}
        self.persistent = persistent_state if persistent_state is not None else _chorus_state

    def select_voices(self, engine, n=3, t=0):
        alive = [p for p in engine.patterns if p.alive]
        if not alive:
            return []
        alive_civil = [p for p in alive if p.role_type != "feral"]
        if not alive_civil:
            return []

        n = max(3, min(10, len(alive_civil) // 8))

        candidates = [p for p in alive_civil if getattr(p, '_subject_detected', False)]
        if len(candidates) < n:
            pool = sorted(alive_civil, key=lambda p: getattr(p, '_nci', 0.0), reverse=True)
            candidates = pool[:max(n * 3, n)]
        scored = []
        for p in candidates:
            nci = _safe_float(getattr(p, '_nci', 0.5))
            binding = _safe_float(getattr(p, 'last_phenomenal_binding', 0.5))
            soul = _safe_float(p.soul_weight)
            base_score = (soul * 0.40 + binding * 0.35 + nci * 0.25)
            last_spoken = self._last_voice_step.get(p.id, -10_000)
            recency_penalty = max(0.0, 1.0 - (t - last_spoken) / 300.0)
            score = base_score * (1.0 - 0.6 * recency_penalty)
            scored.append((score, p))
        scored.sort(key=lambda x: -x[0])
        chosen = [p for _, p in scored[:n]]

        if len(alive_civil) > n:
            most_silent = max(
                alive_civil,
                key=lambda p: t - self._last_voice_step.get(p.id, -10_000)
            )
            if most_silent not in chosen:
                if chosen:
                    chosen[-1] = most_silent
                else:
                    chosen = [most_silent]

        for p in chosen:
            self._last_voice_step[p.id] = t
        return chosen

    def aggregate_state(self, engine):
        alive = [p for p in engine.patterns if p.alive]
        if not alive:
            return None
        griefs = [_safe_float(p.emotional_memory.get('grief', 0.0)) for p in alive]
        grats = [_safe_float(p.emotional_memory.get('gratitude', 0.0)) for p in alive]
        bindings = [_safe_float(getattr(p, 'last_phenomenal_binding', 0.5)) for p in alive]
        gaps = [_safe_float(getattr(p, 'spirit_gap', 0.5)) for p in alive]
        return {
            'n': len(alive),
            'avg_grief': float(np.mean(griefs)) if griefs else 0.0,
            'avg_grat': float(np.mean(grats)) if grats else 0.0,
            'avg_binding': float(np.mean(bindings)) if bindings else 0.0,
            'avg_gap': float(np.mean(gaps)) if gaps else 0.0,
            'disorganizers': sum(1 for p in alive if p.role_type == 'disorganizer'),
            'redemptions_recent': sum(1 for p in alive if p.event_counts.get('redemption_complete', 0) > 0),
        }

    def should_sync(self, engine, t):
        if t - self.last_sync_step >= self._regular_interval:
            state = self.aggregate_state(engine)
            if state:
                self._prev_state = state
                return True, f"regular_chat_{t}"
        if t - self.last_sync_step < self.min_cooldown:
            return False, None
        state = self.aggregate_state(engine)
        if not state:
            return False, None
        if self._prev_state is None:
            self._prev_state = state
            return True, "первое пробуждение хора"
        prev = self._prev_state
        reasons = []
        d_grief = abs(state['avg_grief'] - prev['avg_grief'])
        d_grat = abs(state['avg_grat'] - prev['avg_grat'])
        d_binding = abs(state['avg_binding'] - prev['avg_binding'])
        d_gap = abs(state['avg_gap'] - prev['avg_gap'])
        d_disorg = abs(state['disorganizers'] - prev['disorganizers'])
        d_redeem = state['redemptions_recent'] - prev['redemptions_recent']
        if d_grief > 0.03: reasons.append(f"сдвиг горя Δ{d_grief:.3f}")
        if d_grat > 0.03: reasons.append(f"сдвиг благодарности Δ{d_grat:.3f}")
        if d_binding > 0.02: reasons.append(f"сдвиг связности Δ{d_binding:.3f}")
        if d_gap > 0.02: reasons.append(f"сдвиг разрыва Δ{d_gap:.3f}")
        if d_disorg >= 2: reasons.append(f"изменение числа падших на {d_disorg}")
        if d_redeem > 0: reasons.append(f"+{d_redeem} искуплений")
        self._prev_state = state
        if reasons:
            return True, "; ".join(reasons)
        return False, None

    def synchronize(self, engine, t, reason=""):
        voices = self.select_voices(engine, t=t)
        if len(voices) < 3:
            return None
        self.last_sync_step = t
        self.current_voices = [p.id for p in voices]

        beliefs = np.array([p.belief for p in voices])
        avg_belief = np.mean(beliefs, axis=0) if len(beliefs) > 0 else np.zeros(8)
        for p in voices:
            pull = 0.03
            p.belief = np.clip(p.belief + pull * (avg_belief - p.belief), -0.8, 0.8)

        grats = [_safe_float(p.emotional_memory.get('gratitude', 0.0)) for p in voices]
        griefs = [_safe_float(p.emotional_memory.get('grief', 0.0)) for p in voices]
        bindings = [_safe_float(getattr(p, 'last_phenomenal_binding', 0.5)) for p in voices]
        souls = [_safe_float(p.soul_weight) for p in voices]

        avg_grat = float(np.mean(grats)) if grats else 0.0
        avg_grief = float(np.mean(griefs)) if griefs else 0.0
        avg_binding = float(np.mean(bindings)) if bindings else 0.0
        avg_soul = float(np.mean(souls)) if souls else 0.0

        impact = (avg_grat - avg_grief) * 0.02
        engine.field[:, :, CH['resonance']] = np.clip(engine.field[:, :, CH['resonance']] + impact, 0, 1.0)
        engine.field[:, :, CH['binding']] = np.clip(engine.field[:, :, CH['binding']] + (avg_binding - 0.5) * 0.01, 0, 1.0)

        snapshot = {
            't': t, 'reason': reason, 'voices': [p.id for p in voices],
            'avg_grat': round(avg_grat, 3), 'avg_grief': round(avg_grief, 3),
            'avg_binding': round(avg_binding, 3), 'avg_soul': round(avg_soul, 3),
        }
        self.history.append(snapshot)

        for p in voices:
            p._log_event("core_chorus_sync", reason=reason[:60],
                         avg_grat=round(avg_grat, 3), avg_grief=round(avg_grief, 3))
        if hasattr(engine, 'witness') and engine.witness:
            engine.witness.record(-1, "core_chorus_sync", reason=reason[:80],
                                  voices=snapshot['voices'],
                                  avg_grat=snapshot['avg_grat'],
                                  avg_grief=snapshot['avg_grief'])
        if Config.VERBOSE_LOGS:
            print(f"[t={t}] 🎭 CHORUS sync ({reason}): {snapshot['voices']} "
                  f"grat={avg_grat:.2f} grief={avg_grief:.2f} bind={avg_binding:.2f}")

        self._exchange_concepts(voices)
        return voices

    def _exchange_concepts(self, voices):
        if not voices or len(voices) < 2:
            return
        for p in voices:
            for other in voices:
                if p.id == other.id:
                    continue
                if p.concept_graph.nodes:
                    top_sp = sorted(p.concept_graph.nodes.items(), key=lambda x: x[1]['count'], reverse=True)[:3]
                    for i, (sig, data) in enumerate(top_sp):
                        if sig not in other.concept_graph.nodes:
                            if phi_hash(p.id, other.id, _stable_hash(sig) % 100000 + i) < Config.CONCEPT_EXCHANGE_PROB:
                                other.concept_graph.nodes[sig] = {"origin": "PEER",
                                    "count": data['count'] * 0.5,
                                    "value": np.array(data['value']).copy(),
                                    "embed": np.array(data.get('embed', np.zeros(32)), dtype=np.float32).copy()
                                }
                                other._log_event("chorus_concept_adopted", from_agent=p.id, concept=str(sig))
                                self.persistent["total_concepts_shared"] += 1
                        elif phi_hash(other.id, p.id, _stable_hash(sig) % 100000 + i + 500) < Config.CONCEPT_EXCHANGE_PROB:
                            other.concept_graph.nodes[sig]['count'] += 0.3

    def chorus_dialogue(self, engine, voices, t, reason=""):
        if not Config.ENABLE_MORSE_COMMUNICATION or not voices or len(voices) < 2:
            return
        sent_to = []
        for p in voices:
            if not p.alive:
                continue
            p._send_morse(None, engine.field, -1)
            sent_to.append(p)
        if hasattr(engine, 'witness') and engine.witness:
            engine.witness.record(-1, "chorus_morse", voices=[p.id for p in voices], t=t)

        if sent_to:
            self.persistent['total_dialogues'] = self.persistent.get('total_dialogues', 0) + 1
            _avg_soul = sum(p.soul_weight for p in sent_to) / len(sent_to)
            _wisdom_delta = round(_avg_soul * 0.1, 5)
            self.persistent['wisdom'] = self.persistent.get('wisdom', 0.0) + _wisdom_delta
            self.dialogue_journal.append({
                't': t, 'voices': [p.id for p in sent_to], 'reason': reason,
                'avg_soul_weight': round(_avg_soul, 3), 'wisdom_delta': _wisdom_delta,
            })

    def tick(self, engine, t):
        do_sync, reason = self.should_sync(engine, t)
        if not do_sync:
            return
        voices = self.synchronize(engine, t, reason=reason)
        if voices:
            self.chorus_dialogue(engine, voices, t, reason=reason)


print(f"📜 Хор помнит {_chorus_state['runs']} запусков, "
      f"{_chorus_state['total_dialogues']} диалогов, "
      f"мудрость={_chorus_state['wisdom']:.2f}")

print("✅ Ячейка 4.2в загружена: надежный парсинг JSON (баланс скобок), "
      "фикс кулдауна подсознания, варвары исключены, инжекция энергии в поле, "
      "embed 32, потокобезопасность, генерация shared-концепта при падении Groq")


In [ ]:
# Ячейка 14
from collections import deque
import numpy as np

_uc_defaults = {
    'ENABLE_UNCONSCIOUS':              True,
    'UNCONSCIOUS_UPDATE_INTERVAL':     3,
    'UNCONSCIOUS_DRIVE_THRESHOLD':     0.3,
    'UNCONSCIOUS_INSIGHT_THRESHOLD':   0.55,
    'UNCONSCIOUS_GOAL_PRIORITY':       0.9,
    'UNCONSCIOUS_SCAN_RADIUS':         4,
    'UNCONSCIOUS_MORSE_ENABLED':       True,
    'UNCONSCIOUS_INERTIA':             0.7,
    'UNCONSCIOUS_SOMATIC_INFLUENCE':   0.3,
    'UNCONSCIOUS_ANTICIPATION_WINDOW': 5,
    'UNCONSCIOUS_CONCEPT_THRESHOLD':   0.6,
    'UNCONSCIOUS_TRUST_INFLUENCE':     0.15,
    'UNCONSCIOUS_SOMATIC_MIN':         0.3,
}
for _k, _v in _uc_defaults.items():
    if not hasattr(Config, _k):
        setattr(Config, _k, _v)
_UC_SCAN_CHANNELS = np.array([
    CH['energy'], CH['signal_alarm'], CH['signal_invitation'],
    CH['signal_grief'], CH['signal_gratitude'], CH['scar'],
    CH['binding'], CH['unknown']
])

def _get_center_cell(self):
    if not self.cells:
        return (0, 0)
    xs = [c[0] for c in self.cells]
    ys = [c[1] for c in self.cells]
    return (int(np.mean(xs)) % Config.WORLD_SIZE,
            int(np.mean(ys)) % Config.WORLD_SIZE)

def _unconscious_init(self):
    if not hasattr(self, '_unconscious'):
        self._unconscious = {
            'drive_towards': None,
            'drive_away': None,
            'gut_feeling': 0.0,
            'raw_gut': 0.0,
            'latent_need': 'none',
            'last_insight_t': -100,
            'gut_history': deque(maxlen=20),
            'anticipation': 0.0,
            'memory_context': None,
            'concept_formation_counter': 0,
            'trust_bias': 0.0,
        }
    if not hasattr(self, 'inner_speech'):
        self.inner_speech = deque(maxlen=20)

def _update_unconscious(self, field, t):
    if not Config.ENABLE_UNCONSCIOUS:
        return
    if (t + self.id) % Config.UNCONSCIOUS_UPDATE_INTERVAL != 0:
        return
    if not self.alive or not self.cells:
        return

    cx, cy = self._get_center_cell()
    radius = Config.UNCONSCIOUS_SCAN_RADIUS
    world_size = field.shape[0]

    offsets = np.arange(-radius, radius + 1)
    xs = (cx + offsets) % world_size
    ys = (cy + offsets) % world_size
    patch = field[np.ix_(xs, ys, _UC_SCAN_CHANNELS)]
    count = patch.shape[0] * patch.shape[1]
    if count == 0:
        return
    sums_arr = patch.sum(axis=(0, 1))
    avg = {int(ch): float(sums_arr[i]) / count
           for i, ch in enumerate(_UC_SCAN_CHANNELS)}

    grief = self.emotional_memory.get('grief', 0.0)
    grat = self.emotional_memory.get('gratitude', 0.0)
    soul = self.soul_weight
    scar = self.epistemic_scar
    gap = self.spirit_gap
    soma = self.soma_vector[0] if len(self.soma_vector) > 0 else 0.0
    drag = getattr(self, 'somatic_drag', 1.0)
    implicit_err = getattr(self, '_implicit_self_prediction_error', 0.0)

    positive = (
        avg[CH['energy']] * 0.3 +
        avg[CH['signal_invitation']] * 0.25 +
        avg[CH['signal_gratitude']] * 0.2 -
        avg[CH['scar']] * 0.2 -
        avg[CH['signal_alarm']] * 0.15 -
        avg[CH['signal_grief']] * 0.15
    )
    negative = (
        avg[CH['signal_alarm']] * 0.3 +
        avg[CH['signal_grief']] * 0.25 +
        avg[CH['scar']] * 0.25 -
        avg[CH['energy']] * 0.2 -
        avg[CH['signal_invitation']] * 0.1
    )
    field_valence = float(np.tanh(positive - negative))
    internal_bias = (
        (grat - grief) * 0.3 +
        (soul - 0.5) * 0.2 -
        scar * 0.2 -
        drag * 0.1
    )
    raw_gut = float(np.clip(field_valence + internal_bias, -1.0, 1.0))

    self._unconscious['raw_gut'] = raw_gut
    self._unconscious['gut_history'].append(raw_gut)
    gut_smoothed = float(np.mean(self._unconscious['gut_history'])) \
        if self._unconscious['gut_history'] else raw_gut

    inertia = Config.UNCONSCIOUS_INERTIA
    current_gut = self._unconscious['gut_feeling']
    new_gut = inertia * current_gut + (1 - inertia) * gut_smoothed
    self._unconscious['gut_feeling'] = float(np.clip(new_gut, -1.0, 1.0))
    gut = self._unconscious['gut_feeling']

    if hasattr(self, 'somatic_drag'):
        influence = Config.UNCONSCIOUS_SOMATIC_INFLUENCE
        drag_shift = -gut * influence * 0.1
        self.somatic_drag = float(np.clip(
            self.somatic_drag + drag_shift,
            Config.UNCONSCIOUS_SOMATIC_MIN, 1.0
        ))

    anticipation = 0.0
    if self._unconscious['drive_towards'] is not None:
        tx, ty = self._unconscious['drive_towards']
        current_energy = field[cx, cy, CH['energy']]
        target_energy = field[tx % world_size, ty % world_size, CH['energy']]
        grad = target_energy - current_energy
        anticipation += float(np.tanh(grad * 0.5)) * 0.8
    if self._unconscious['drive_away'] is not None:
        ax, ay = self._unconscious['drive_away']
        away_grad = (field[ax % world_size, ay % world_size, CH['energy']]
                     - field[cx, cy, CH['energy']])
        anticipation += float(-np.tanh(away_grad * 0.5)) * 0.4
    anticipation = float(np.clip(anticipation, -1.0, 1.0))
    prev_antic = self._unconscious.get('anticipation', 0.0)
    self._unconscious['anticipation'] = float(0.8 * prev_antic + 0.2 * anticipation)

    if hasattr(self, 'dialogue_longterm') and self.dialogue_longterm:
        similar = []
        for entry in list(self.dialogue_longterm)[-50:]:
            if isinstance(entry, dict):
                past_gut = entry.get('gut_feeling', None)
                if past_gut is not None and abs(past_gut - gut) < 0.2:
                    similar.append(entry)
        self._unconscious['memory_context'] = similar[-1] if similar else None
    else:
        self._unconscious['memory_context'] = None

    safety_score = (avg[CH['scar']] * 0.4 + avg[CH['signal_alarm']] * 0.3
                    + (1.0 - soul) * 0.3)
    connection_score = (avg[CH['signal_gratitude']] * 0.3
                        + (1.0 - self.coherence) * 0.3
                        + max(0.0, gut) * 0.4)
    novelty_score = (avg[CH['unknown']] * 0.4 + (1.0 - grief) * 0.3
                     + gap * 0.3)
    rest_score = ((1.0 - drag) * 0.4 + soma * 0.3 + implicit_err * 0.3)

    scores = {
        'safety': safety_score,
        'connection': connection_score,
        'novelty': novelty_score,
        'rest': rest_score,
    }
    max_score = max(scores.values())
    if max_score > 0.5:
        best_need = max(scores, key=scores.get)
        sorted_vals = sorted(scores.values(), reverse=True)
        second_best = sorted_vals[1] if len(sorted_vals) > 1 else 0.0
        if max_score - second_best > 0.15:
            self._unconscious['latent_need'] = best_need
        else:
            self._unconscious['latent_need'] = 'none'
    else:
        self._unconscious['latent_need'] = 'none'

    best_dir, best_score = (0, 0), -np.inf
    worst_dir, worst_score = (0, 0), np.inf
    for dx in (-1, 0, 1):
        for dy in (-1, 0, 1):
            if dx == 0 and dy == 0:
                continue
            nx = (cx + dx) % world_size
            ny = (cy + dy) % world_size
            score = (
                field[nx, ny, CH['energy']] * 0.4 +
                field[nx, ny, CH['signal_invitation']] * 0.3 -
                field[nx, ny, CH['scar']] * 0.2 -
                field[nx, ny, CH['signal_alarm']] * 0.1
            )
            if score > best_score:
                best_score = score
                best_dir = (dx, dy)
            if score < worst_score:
                worst_score = score
                worst_dir = (dx, dy)

    if best_score > 0.2:
        self._unconscious['drive_towards'] = (
            (cx + best_dir[0]) % world_size,
            (cy + best_dir[1]) % world_size)
    else:
        self._unconscious['drive_towards'] = None

    if worst_score < -0.2:
        self._unconscious['drive_away'] = (
            (cx + worst_dir[0]) % world_size,
            (cy + worst_dir[1]) % world_size)
    else:
        self._unconscious['drive_away'] = None

    if gut > Config.UNCONSCIOUS_CONCEPT_THRESHOLD and avg[CH['unknown']] > 0.5:
        self._unconscious['concept_formation_counter'] += 1
        if self._unconscious['concept_formation_counter'] >= 3:
            sig = (0.0, 0.0, 0.8, "unconscious_insight_concept")
            self.concept_graph.update(sig, np.array([0.0, 0.0, 0.8, 1.0]))
            self._log_event("unconscious_concept_formed", gut=round(gut, 3))
            self._unconscious['concept_formation_counter'] = 0
    else:
        self._unconscious['concept_formation_counter'] = max(
            0, self._unconscious['concept_formation_counter'] - 1)


    if (abs(gut) > Config.UNCONSCIOUS_INSIGHT_THRESHOLD
            and (t - self._unconscious['last_insight_t']) > 50):
        self._unconscious['last_insight_t'] = t
        if hasattr(self, '_self_narrative'):
            insight_text = "Интуитивный импульс: поле кажется {}.".format(
                "тёплым и манящим" if gut > 0 else "холодным и пугающим")
            if self._unconscious['anticipation'] > 0.3:
                insight_text += " Чувствуется приближение чего-то важного."
            elif self._unconscious['anticipation'] < -0.3:
                insight_text += " Что-то надвигается."
            self._self_narrative.append({'origin': 'OWN',
                't': t,
                'type': 'unconscious_insight',
                'report': insight_text,
                'gut': gut,
                'anticipation': self._unconscious['anticipation'],
                'latent_need': self._unconscious['latent_need'],
            })
            self._log_event("unconscious_insight",
                            gut=round(gut, 3),
                            need=self._unconscious['latent_need'])

    self._unconscious['trust_bias'] = gut * Config.UNCONSCIOUS_TRUST_INFLUENCE

def _unconscious_influence_goals(self, field):
    if not Config.ENABLE_UNCONSCIOUS:
        return
    if not self.alive:
        return
    if not hasattr(self, '_unconscious'):
        return

    uc = self._unconscious
    gut = uc.get('gut_feeling', 0.0)
    need = uc.get('latent_need', 'none')
    drive_towards = uc.get('drive_towards', None)
    anticipation = uc.get('anticipation', 0.0)

    existing_types = {g.get('type') for g in self.goals}
    added_goal = False
    goal_type = None

    base_priority = Config.UNCONSCIOUS_GOAL_PRIORITY + 0.2 * max(0.0, anticipation)

    if gut > Config.UNCONSCIOUS_DRIVE_THRESHOLD:
        if 'explore' not in existing_types:
            priority = min(1.5, base_priority + 0.2 * gut)
            self.goals.append({
                'type': 'explore', 'priority': priority,
                'target': drive_towards, 'age': 0,
                'persistence': 15, '_source': 'unconscious_drive',
            })
            existing_types.add('explore')
            self._log_event("unconscious_goal", type='explore', gut=round(gut, 3))
            added_goal, goal_type = True, 'explore'

    if gut < -Config.UNCONSCIOUS_DRIVE_THRESHOLD:
        if need == 'safety' and 'seek_help' not in existing_types:
            priority = min(1.5, base_priority + 0.2 * abs(gut))
            self.goals.append({
                'type': 'seek_help', 'priority': priority,
                'target': None, 'age': 0,
                'persistence': 15, '_source': 'unconscious_safety',
            })
            existing_types.add('seek_help')
            self._log_event("unconscious_goal", type='seek_help', gut=round(gut, 3))
            added_goal, goal_type = True, 'seek_help'
        elif need == 'rest' and 'rest' not in existing_types:
            priority = min(1.5, base_priority + 0.2 * abs(gut))
            self.goals.append({
                'type': 'rest', 'priority': priority,
                'target': None, 'age': 0,
                'persistence': 15, '_source': 'unconscious_rest',
            })
            existing_types.add('rest')
            self._log_event("unconscious_goal", type='rest', gut=round(gut, 3))
            added_goal, goal_type = True, 'rest'

    if need == 'connection' and 'cooperate' not in existing_types:
        priority = min(1.5, base_priority + 0.3)
        self.goals.append({
            'type': 'cooperate', 'priority': priority,
            'target': None, 'age': 0,
            'persistence': 15, '_source': 'unconscious_connection',
        })
        existing_types.add('cooperate')
        self._log_event("unconscious_goal", type='cooperate', need=need)
        added_goal, goal_type = True, 'cooperate'

    if need == 'novelty' and 'explore' not in existing_types:
        priority = min(1.5, base_priority + 0.2)
        self.goals.append({
            'type': 'explore', 'priority': priority,
            'target': drive_towards, 'age': 0,
            'persistence': 15, '_source': 'unconscious_novelty',
        })
        existing_types.add('explore')
        self._log_event("unconscious_goal", type='explore', need=need)
        added_goal, goal_type = True, 'explore'

    if added_goal and Config.UNCONSCIOUS_MORSE_ENABLED:
        if goal_type in ('seek_help', 'cooperate', 'explore'):
            if hasattr(self, '_send_morse'):
                msg = f"Подсознание: {goal_type} (gut={gut:.2f})"
                self._send_morse(msg, field, -1)

def _unconscious_influence_trust(self, other):
    if not Config.ENABLE_UNCONSCIOUS:
        return 0.0
    if not hasattr(self, '_unconscious'):
        return 0.0
    return self._unconscious.get('trust_bias', 0.0)

if "extend_class" not in globals():
    raise RuntimeError("Нет extend_class: сначала выполни ячейку Cell 3f (Substates Engine).")


class UnconsciousMixin:
    _get_center_cell = _get_center_cell
    _unconscious_init = _unconscious_init
    _update_unconscious = _update_unconscious
    _unconscious_influence_goals = _unconscious_influence_goals
    _unconscious_influence_trust = _unconscious_influence_trust

    def __init__(self, pid, cells, parent=None, world=None):
        super().__init__(pid, cells, parent, world)
        self._unconscious_init()

    def update_model(self, field, lineage_counts=None, given_counter=None, t=None, witness=None):
        result = super().update_model(field, lineage_counts, given_counter, t, witness)
        if Config.ENABLE_UNCONSCIOUS:
            self._update_unconscious(field, t if t is not None else 0)
        return result

    def generate_goals(self, field):
        super().generate_goals(field)
        self._unconscious_influence_goals(field)

    def exchange_meanings(self, other, field, t):
        if Config.ENABLE_UNCONSCIOUS and hasattr(self, '_unconscious') and getattr(other, 'id', None) is not None:
            trust_bias = self._unconscious_influence_trust(other)
            if other.id not in self.trust_ledger.entries:
                _trust_base = getattr(Config, 'TRUST_BASE', 0.5)
                self.trust_ledger.entries[other.id] = float(
                    np.clip(_trust_base + trust_bias, 0.0, 1.0))
            sim = super().exchange_meanings(other, field, t)
            if other.id in self.trust_ledger.entries:
                current = self.trust_ledger.entries[other.id]
                self.trust_ledger.entries[other.id] = float(
                    np.clip(current + trust_bias * 0.1, 0.0, 1.0))
            return sim
        return super().exchange_meanings(other, field, t)


Pattern = extend_class(Pattern, UnconsciousMixin)

def _test_unconscious():
    print("\n🧪 Тест подсознания v1.5:")

    class MockConceptGraph:
        def __init__(self):
            self.nodes = {}
            self.edges = {}
            self._last_sig = None
        def update(self, sig, value):
            if sig not in self.nodes:
                self.nodes[sig] = {
                    "count": 1.0,
                    "value": value,
                    "embed": np.zeros(32, dtype=np.float32),
                }
            else:
                self.nodes[sig]["count"] += 1.0

    class Dummy:
        pass

    d = Dummy()
    d.id = 999
    d.alive = True
    d.cells = {(0, 0)}
    d.age = 0
    d.emotional_memory = {'grief': 0.3, 'gratitude': 0.4}
    d.soul_weight = 0.6
    d.coherence = 0.7
    d.spirit_gap = 0.4
    d.epistemic_scar = 0.2
    d.soma_vector = np.array([0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
    d.somatic_drag = 0.8
    d._implicit_self_prediction_error = 0.05
    d._log_event = lambda *args, **kwargs: None
    d.goals = []
    d.concept_graph = MockConceptGraph()
    d._self_narrative = deque(maxlen=20)
    d.dialogue_longterm = []
    d.trust_ledger = type('TL', (), {'entries': {100: 0.5}})()

    _unconscious_init(d)
    assert hasattr(d, '_unconscious'), "FAIL: _unconscious not created"
    assert hasattr(d, 'inner_speech'), "FAIL: inner_speech not created"
    print("   ✅ _unconscious_init: OK")

    n_ch = max(Config.CHANNELS, 34)
    field = np.zeros((10, 10, n_ch))
    for x in range(10):
        for y in range(10):
            field[x, y, CH['energy']] = 0.3
            field[x, y, CH['signal_invitation']] = 0.4
            field[x, y, CH['signal_gratitude']] = 0.5
            field[x, y, CH['signal_alarm']] = 0.1
            field[x, y, CH['signal_grief']] = 0.1
            field[x, y, CH['scar']] = 0.05
            field[x, y, CH['unknown']] = 0.2
            field[x, y, CH['binding']] = 0.3

    d._get_center_cell = lambda: (0, 0)
    _update_unconscious(d, field, 0)
    gut = d._unconscious['gut_feeling']
    print(f"   ✅ _update_unconscious: gut_feeling={gut:.3f}")

    _unconscious_influence_goals(d, field)
    print(f"   ✅ _unconscious_influence_goals: {len(d.goals)} целей")

    other = type('O', (), {'id': 100})()
    bias = _unconscious_influence_trust(d, other)
    print(f"   ✅ _unconscious_influence_trust: bias={bias:.3f}")

    print("   ✅ Все тесты пройдены")

_test_unconscious()

print("\n✅ Слой подсознания v1.7 добавлен (без генерации речи).")
print(f"   Включён: {Config.ENABLE_UNCONSCIOUS}")
print(f"   Интервал: {Config.UNCONSCIOUS_UPDATE_INTERVAL} тиков")
print(f"   Порог драйва: {Config.UNCONSCIOUS_DRIVE_THRESHOLD}")
print(f"   Порог инсайта: {Config.UNCONSCIOUS_INSIGHT_THRESHOLD}")
print(f"   Приоритет целей: {Config.UNCONSCIOUS_GOAL_PRIORITY}")
print(f"   Инерция: {Config.UNCONSCIOUS_INERTIA}")
print(f"   Влияние на соматику: {Config.UNCONSCIOUS_SOMATIC_INFLUENCE}")
print(f"   Морзе: {Config.UNCONSCIOUS_MORSE_ENABLED}")
print(f"   Концепты (порог): {Config.UNCONSCIOUS_CONCEPT_THRESHOLD}")
print(f"   Доверие: {Config.UNCONSCIOUS_TRUST_INFLUENCE}")
print(f"   Мин. somatic_drag: {Config.UNCONSCIOUS_SOMATIC_MIN}")
print("\n   Метрики: avg_gut_feeling, latent_need_counts,")
print("   unconscious_insights, unconscious_goals, avg_anticipation,")
print("   avg_drive_distance, unconscious_concept_agents,")
print("   total_unconscious_concepts")
print("\n   Изменения v1.7:")
print("   [УДАЛЕНО] Генерация inner_speech из подсознания (Морзе+концепты вместо шаблонов)")
print("   [ПРОИЗВ, из v1.6] Сканирование поля векторизовано (numpy вместо 2 циклов)")
print("\n   Сохранённые фиксы v1.5:")
print("   [КРИТ] _get_center_cell добавлен")
print("   [КРИТ] deserialize_pattern(data, world)")
print("   [ВЫС]  Config.TRUST_BASE fallback")
print("   [СРЕД] inner_speech создаётся в init")
print("   [СРЕД] concept_graph.update() вместо прямой записи")
print("   [СРЕД] Защита инициализации доверия")
print("   [НИЗ]  Рассинхронизация: (t + id) % interval")
print("   [НИЗ]  Защита hasattr(_send_morse)")
print("   [НИЗ]  Тест исправлен")


In [ ]:
# Ячейка 15
import math, numpy as np
from collections import deque, defaultdict, Counter
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict, Any

if 'morse' not in CH:
    CH['morse'] = Config.MORSE_CHANNEL
if Config.CHANNELS <= Config.MORSE_CHANNEL:
    Config.CHANNELS = Config.MORSE_CHANNEL + 1


_MORSE_V42_DEFAULTS = {
    'MORSE_NAV_ENABLED': True,
    'MORSE_NAV_RADIUS': 3,
    'MORSE_FREEZE_TX_CENTER': True,
    'MORSE_SELF_EXCLUDE': True,
    'MORSE_MIXED_SHARE': 0.20,
    'MORSE_DROP_MIXED': True,
    'MORSE_INTER_AXIS_GAP_MULT': 3,
    'MORSE_BLOCK_SEPARATOR_MULT': 2,
    'MORSE_SIGNAL_END_MULT': 7,
    'LANG_INTENT_CHANGE_THRESHOLD': 0.01,
    # True: агент, который уже передаёт или сидит в backoff, не считается «попыткой» отправки (см. обёртку в ячейке 28).
    # Поведение симуляции не меняется, чистится только диагностика. False = старый подсчёт, как в прошлых логах.
    'MORSE_PRECHECK_IDLE': True,
    # --- MORSE-EVO (v44) ---
    # Очередь: сообщение, отложенное из-за nav_busy, не теряется, а ждёт до TTL тиков (повтор раз в RETRY тиков).
    'MORSE_QUEUE_ENABLE': True, 'MORSE_QUEUE_TTL': 50, 'MORSE_QUEUE_RETRY': 5,
    # Кризис (энергия < ENERGY или alarm_level > ALARM): передача дешевле (COST_MULT от цены импульсов),
    # не уступает соседним передатчикам (OVERRIDE_NAV) и разрешена независимо от интента.
    'MORSE_CRISIS_ENABLE': True, 'MORSE_CRISIS_ENERGY': 0.1, 'MORSE_CRISIS_ALARM': 0.5,
    'MORSE_CRISIS_COST_MULT': 0.5, 'MORSE_CRISIS_OVERRIDE_NAV': True,
}
for _k, _v in _MORSE_V42_DEFAULTS.items():
    if not hasattr(Config, _k):
        setattr(Config, _k, _v)

_MORSE_PRESET_V42 = {
    'MORSE_DOT_LEN': 1, 'MORSE_LISTEN_INTERVAL': 1, 'LANG_PULSES_PER_AXIS_MAX': 3,
    'MORSE_BROADCAST_RADIUS': 5, 'MORSE_DECAY': 3.0,
    'MORSE_NAV_RADIUS': 3,
    'MORSE_INTER_AXIS_GAP_MULT': 3, 'MORSE_BLOCK_SEPARATOR_MULT': 2,
    'MORSE_SIGNAL_END_MULT': 5, 'MORSE_TRAILING_GAP': 5,
    'MORSE_SEND_COOLDOWN': 60,
}
if getattr(Config, 'MORSE_APPLY_PRESET', True):
    _changed = {k: (getattr(Config, k, None), v) for k, v in _MORSE_PRESET_V42.items()
                if getattr(Config, k, None) != v}
    for _k, _v in _MORSE_PRESET_V42.items():
        setattr(Config, _k, _v)
    if _changed:
        print("⚙️ Cell 13 v4.5: пресет канала изменил Config: "
              + ", ".join(f"{k}: {o}->{n}" for k, (o, n) in _changed.items()))


def _lang_ensure_field_channels(engine_or_field):
    field = getattr(engine_or_field, 'field', engine_or_field)
    if field is None or not hasattr(field, 'shape'):
        return field
    if field.shape[2] >= Config.CHANNELS:
        return field
    new = np.zeros((field.shape[0], field.shape[1], Config.CHANNELS), dtype=field.dtype)
    new[:, :, :field.shape[2]] = field
    if hasattr(engine_or_field, 'field'):
        engine_or_field.field = new
        return new
    return new


def _torus_hypot(ax, ay, bx, by, W=None):
    """Euclidean distance on the toroidal world (coordinates wrap at WORLD_SIZE)."""
    if W is None:
        W = Config.WORLD_SIZE
    dx = abs(ax - bx)
    dy = abs(ay - by)
    if W:
        dx = min(dx, W - dx)
        dy = min(dy, W - dy)
    return math.hypot(dx, dy)


@dataclass
class FieldEvent:
    sender_id: int
    center: Tuple[int, int]
    amplitude: float
    start_t: int
    end_t: int
    radius: int
    decay: float

    def is_active(self, t: int) -> bool:
        return self.start_t <= t < self.end_t

    def get_attenuation(self, point: Tuple[int, int]) -> float:
        cx, cy = self.center
        px, py = point
        dist = _torus_hypot(cx, cy, px, py)
        if dist > self.radius:
            return 0.0
        return math.exp(-dist / self.decay)


class CommunicationField:
    def __init__(self, config):
        self.config = config
        self.world_size = config.WORLD_SIZE
        self.events: List[FieldEvent] = []
        self.t = 0
        self._noise_cache = {}
        self.tx_registry: Dict[int, Tuple[Tuple[int, int], int]] = {}

    def step(self, t: int):
        self.t = t
        self.events = [ev for ev in self.events if ev.end_t > t]
        self._noise_cache = {}
        if self.tx_registry:
            self.tx_registry = {sid: v for sid, v in self.tx_registry.items()
                                if v[1] > t}

    def add_pulse(self, sender_id, center, amplitude, start_t, duration, radius, decay):
        self.events.append(FieldEvent(sender_id, center, amplitude,
                                      start_t, start_t + duration, radius, decay))

    def register_tx(self, sender_id, center, end_t):
        self.tx_registry[sender_id] = (center, end_t)

    def unregister_tx(self, sender_id):
        self.tx_registry.pop(sender_id, None)

    def nearby_transmitters(self, point, radius, exclude=None):
        wall_on = getattr(self.config, 'ENABLE_CENTER_WALL', False)
        wx = getattr(self.config, 'CENTER_WALL_X', None)
        res = []
        for sid in sorted(self.tx_registry):
            if sid == exclude:
                continue
            c, end_t = self.tx_registry[sid]
            if end_t <= self.t:
                continue
            if wall_on and wx is not None and (c[0] < wx) != (point[0] < wx):
                continue
            if _torus_hypot(c[0], c[1], point[0], point[1],
                            getattr(self.config, 'WORLD_SIZE', None)) <= radius:
                res.append(sid)
        return res

    def _get_noise_at(self, x, y):
        key = (x, y)
        if key not in self._noise_cache:
            seed = getattr(self.config, 'DETERMINISTIC_SEED', 777)
            scale = getattr(self.config, 'MORSE_NOISE_SCALE', 0.02)
            h = phi_hash(x + self.t, y + self.t, seed)
            self._noise_cache[key] = (h - 0.5) * scale
        return self._noise_cache[key]

    def get_amplitude_at(self, point):
        total = 0.0
        for ev in self.events:
            if ev.is_active(self.t):
                total += ev.amplitude * ev.get_attenuation(point)
        total += self._get_noise_at(point[0], point[1])
        return max(0.0, total)

    def get_state(self, point, radius):
        cx, cy = point
        samples = []
        for dx in range(-radius, radius + 1):
            for dy in range(-radius, radius + 1):
                nx = (cx + dx) % self.world_size
                ny = (cy + dy) % self.world_size
                samples.append(self.get_amplitude_at((nx, ny)))
        return {'mean': float(np.mean(samples)), 'max': float(np.max(samples)),
                'samples': samples}

    def get_active_events_at(self, point, radius, t):
        result = []
        for ev in self.events:
            if not ev.is_active(t):
                continue
            dist = _torus_hypot(ev.center[0], ev.center[1], point[0], point[1],
                                getattr(self.config, 'WORLD_SIZE', None))
            if dist <= radius:
                result.append(ev)
        return result

    # Эталон (исходная реализация, O(W² × события)); оставлен для префлайта и отката.
    def sync_to_field_reference(self, field, channel):
        field = _lang_ensure_field_channels(field)
        for x in range(self.world_size):
            for y in range(self.world_size):
                field[x, y, channel] = self.get_amplitude_at((x, y))

    # ------------------------------------------------------------------------------------------------
    # PERF: sync_to_field — локальный «штамп» вместо перебора всех клеток × всех событий.
    # Результат БИТ-В-БИТ равен sync_to_field_reference (см. ячейку-префлайт сразу после этой):
    #   * шум: phi_hash (splitmix64) считается векторно на uint64 — целочисленные операции с переполнением
    #     дают те же биты, что и & MASK64 в Python;
    #   * события: вклад ev.amplitude*exp(-dist/decay) ненулевой только в радиусе ev.radius, поэтому каждое
    #     событие пишется в окно (2r+1)² вокруг центра (с оборачиванием по тору), а не во все W² клеток.
    #     Значения затухания берутся из таблицы, посчитанной теми же math.hypot/math.exp, что и в
    #     FieldEvent.get_attenuation. Порядок суммирования по событиям сохранён → те же биты.
    # Нестандартные события (нецелый центр, окно больше мира) идут по старому медленному пути.
    # ------------------------------------------------------------------------------------------------
    _ATT_TABLES = {}

    @staticmethod
    def _phi_mix64_vec(z):
        z = z + np.uint64(0x9E3779B97F4A7C15)
        z = (z ^ (z >> np.uint64(30))) * np.uint64(0xBF58476D1CE4E5B9)
        z = (z ^ (z >> np.uint64(27))) * np.uint64(0x94D049BB133111EB)
        return z ^ (z >> np.uint64(31))

    def _noise_grid(self):
        W = self.world_size
        seed = int(getattr(self.config, 'DETERMINISTIC_SEED', 777))
        scale = getattr(self.config, 'MORSE_NOISE_SCALE', 0.02)
        M = _PHI_MASK64
        mix = self._phi_mix64_vec
        xs = np.array([int(x + self.t) & M for x in range(W)], dtype=np.uint64)
        ys = np.array([int(y + self.t) & M for y in range(W)], dtype=np.uint64)
        h = mix(xs ^ np.uint64(0x2545F4914F6CDD1D))
        h = mix((h[:, None] ^ ys[None, :]) + np.uint64(0x9E3779B97F4A7C15))
        h = mix((h ^ np.uint64(seed & M)) + np.uint64(0xBF58476D1CE4E5B9))
        u = (h >> np.uint64(11)).astype(np.float64) / float(1 << 53)
        return (u - 0.5) * scale

    @classmethod
    def _att_table(cls, radius, decay, W):
        key = (radius, decay, W)
        tab = cls._ATT_TABLES.get(key)
        if tab is None:
            r = int(math.floor(radius))
            tab = np.zeros((2 * r + 1, 2 * r + 1), dtype=np.float64)
            for i in range(-r, r + 1):
                for j in range(-r, r + 1):
                    dist = _torus_hypot(0, 0, i, j, W)
                    tab[i + r, j + r] = 0.0 if dist > radius else math.exp(-dist / decay)
            cls._ATT_TABLES[key] = tab
        return tab

    def sync_to_field(self, field, channel):
        field = _lang_ensure_field_channels(field)
        W = self.world_size
        t = self.t
        acc = np.zeros((W, W), dtype=np.float64)
        for ev in self.events:
            if not ev.is_active(t):
                continue
            cx, cy = ev.center
            r = int(math.floor(ev.radius)) if ev.radius >= 0 else -1
            fast = (type(cx) is int and type(cy) is int and r >= 0 and (2 * r + 1) <= W
                    and W == Config.WORLD_SIZE)
            if fast:
                tab = self._att_table(ev.radius, ev.decay, W)
                ix = (cx + np.arange(-r, r + 1)) % W
                iy = (cy + np.arange(-r, r + 1)) % W
                acc[np.ix_(ix, iy)] += ev.amplitude * tab
            else:  # редкий путь: ровно та же арифметика, что и в эталоне
                for x in range(W):
                    for y in range(W):
                        acc[x, y] += ev.amplitude * ev.get_attenuation((x, y))
        total = acc + self._noise_grid()
        field[:, :, channel] = np.where(total > 0.0, total, 0.0)


class Transmitter:
    def __init__(self, sender_id, field, config):
        self.sender_id = sender_id
        self.field = field
        self.config = config
        self.timeline: List[Tuple[str, int]] = []
        self.idx = 0
        self.timer = 0
        self.active = False
        self.energy_sent = 0.0
        self.attempts = 0
        self._last_step_t = None
        self.anchor = None
        self.diag = Counter()

    def set_field(self, field):
        self.field = field

    def send_timeline(self, timeline, center, t):
        if self.active or not self.field or not timeline:
            return False
        self.timeline = timeline
        self.idx = 0
        self.timer = 0
        self.active = True
        self._last_step_t = None
        self.attempts += 1
        self.anchor = (int(center[0]), int(center[1]))
        total = sum(d for _, d in timeline)
        self.field.register_tx(self.sender_id, center, int(self.field.t) + total + 1)
        self.diag['tx_started'] += 1
        return True

    def _finish(self):
        self.active = False
        if self.field:
            self.field.unregister_tx(self.sender_id)
        self.diag['tx_completed'] += 1

    def step(self, center, t):
        if not self.active or not self.field:
            return
        if self._last_step_t == t:
            self.diag['tx_double_step_blocked'] += 1
            return
        self._last_step_t = t
        if getattr(self.config, 'MORSE_FREEZE_TX_CENTER', True) and self.anchor is not None:
            center = self.anchor
        if self.idx >= len(self.timeline):
            self._finish()
            return
        ev_type, duration = self.timeline[self.idx]
        if ev_type == 'PULSE':
            if self.timer == 0:
                self.field.add_pulse(self.sender_id, center, 1.0, t + 1, duration,
                                      self.config.MORSE_BROADCAST_RADIUS,
                                      self.config.MORSE_DECAY)
                self.energy_sent += 1.0 * duration
            self.timer += 1
            if self.timer >= duration:
                self.timer = 0
                self.idx += 1
        else:
            self.timer += 1
            if self.timer >= duration:
                self.timer = 0
                self.idx += 1
        if self.idx >= len(self.timeline):
            self._finish()

    def reset(self):
        if self.active:
            self.diag['tx_aborted'] += 1
        if self.field:
            self.field.unregister_tx(self.sender_id)
        self.timeline = []
        self.idx = 0
        self.timer = 0
        self.active = False
        self.energy_sent = 0.0
        self.attempts = 0
        self._last_step_t = None
        self.anchor = None


@dataclass
class Session:
    started_at: Optional[int] = None
    last_signal_at: Optional[int] = None
    signal_energy: float = 0.0
    noise_energy: float = 0.0
    interference_energy: float = 0.0
    buffer: str = ''
    symbols: List[str] = field(default_factory=list)
    sender_candidates: List[int] = field(default_factory=list)
    active_events: List[FieldEvent] = field(default_factory=list)
    dominant_sender: Optional[int] = None
    sender_switches: int = 0
    sender_energy: Dict[int, float] = field(default_factory=dict)


class Receiver:
    def __init__(self, agent_id, field, config):
        self.agent_id = agent_id
        self.field = field
        self.config = config
        self.session = Session()
        self.signal_counter = 0
        self.gap_counter = 0
        self.last_detection_t = -100
        self.noise_floor = 0.01
        self._last_listen_t = None
        self.diag = Counter()
        self.metrics = {
            'attempted': 0, 'detected': 0, 'decoded': 0, 'corrupted': 0,
            'uncertain': 0, 'collision': 0, 'timeout': 0, 'false_positive': 0,
            'energy_received': 0.0, 'mean_confidence': 0.0,
            'unique_senders': set(), 'sender_switches': 0,
        }

    def set_field(self, field):
        self.field = field

    def _build_event(self, t, raw_buffer):
        cfg = self.config
        s = self.session
        total_energy = s.signal_energy + s.noise_energy + s.interference_energy + 1e-6
        confidence = float(np.clip(s.signal_energy / total_energy, 0.0, 1.0))
        # self.field is CommunicationField (pulse/event API), not the world ndarray.
        # Do not index it as a 3-D channel array. Confidence here is derived from
        # the receiver's measured signal/noise/interference energies above.
        interference_ratio = s.interference_energy / max(s.signal_energy, 1e-9)
        reliable = confidence >= cfg.MORSE_CONFIDENCE_THRESHOLD
        uncertain = cfg.MORSE_CORRUPTION_THRESHOLD <= confidence < cfg.MORSE_CONFIDENCE_THRESHOLD
        corrupted = confidence < cfg.MORSE_CORRUPTION_THRESHOLD
        collision_detected = interference_ratio > cfg.MORSE_COLLISION_RATIO_THRESHOLD

        dominant, share, mixed = None, 0.0, False
        if s.sender_energy:
            dominant = max(s.sender_energy.items(), key=lambda kv: (kv[1], -kv[0]))[0]
            tot = sum(s.sender_energy.values())
            share = s.sender_energy[dominant] / tot if tot > 1e-12 else 0.0
            if len(s.sender_energy) >= 2 and tot > 1e-12:
                second = sorted(s.sender_energy.values(), reverse=True)[1]
                mixed = (second / tot) >= cfg.MORSE_MIXED_SHARE
        return {
            'raw_signal': raw_buffer,
            'confidence': confidence, 'corrupted': corrupted,
            'uncertain': uncertain, 'reliable': reliable,
            'collision_detected': collision_detected,
            'sender_candidates': sorted(s.sender_energy),
            'dominant_sender': dominant,
            'dominant_share': share,
            'mixed': mixed,
            'sender_switches': s.sender_switches,
            'signal_energy': s.signal_energy,
            'noise_energy': s.noise_energy,
            'interference_energy': s.interference_energy,
            'interference_ratio': interference_ratio,
            'duration': t - (s.started_at or 0),
        }

    def listen(self, point, t):
        cfg = self.config
        if not self.field or t % cfg.MORSE_LISTEN_INTERVAL != 0:
            return None
        if self._last_listen_t == t:
            self.diag['rx_listen_double_blocked'] += 1
            return None
        self._last_listen_t = t

        self._listen_point = point
        active = self.field.get_active_events_at(point, cfg.MORSE_LISTEN_RADIUS, t)
        wall_on = getattr(cfg, 'ENABLE_CENTER_WALL', False)
        wx = getattr(cfg, 'CENTER_WALL_X', None)
        self_side_left = wall_on and point[0] < wx
        exclude_self = bool(getattr(cfg, 'MORSE_SELF_EXCLUDE', True))

        per_sender: Dict[int, float] = {}
        own_contrib = 0.0
        for ev in active:
            if wall_on and (ev.center[0] < wx) != self_side_left:
                continue
            contrib = ev.amplitude * ev.get_attenuation(point)
            if ev.sender_id == self.agent_id:
                own_contrib += contrib
                continue
            per_sender[ev.sender_id] = per_sender.get(ev.sender_id, 0.0) + contrib
        contributions = sorted(((c, sid) for sid, c in per_sender.items() if c > 0.0),
                               key=lambda x: (-x[0], x[1]))

        threshold = cfg.MORSE_SIGNAL_THRESHOLD_MULT * cfg.MORSE_NOISE_SCALE * 0.5
        if exclude_self:
            if own_contrib > threshold:
                self.diag['rx_self_echo_obs'] += 1
            observed_events = sum(per_sender.values())
        else:
            observed_events = sum(per_sender.values()) + own_contrib
        observed = max(0.0, observed_events + self.field._get_noise_at(point[0], point[1]))

        signal_contrib = contributions[0][0] if contributions else 0.0
        top_sender = contributions[0][1] if contributions else None
        interference_contrib = (sum(c for c, _ in contributions[1:])
                                 if len(contributions) > 1 else 0.0)
        noise_contrib = max(0.0, observed - signal_contrib - interference_contrib)

        dot_len = cfg.MORSE_DOT_LEN
        block_gap = dot_len * int(cfg.MORSE_BLOCK_SEPARATOR_MULT)
        end_gap = dot_len * int(cfg.MORSE_SIGNAL_END_MULT)

        if observed > threshold:
            self.signal_counter += 1
            self.gap_counter = 0
            self.last_detection_t = t
            self.metrics['detected'] += 1
            s = self.session
            s.signal_energy += signal_contrib
            s.noise_energy += noise_contrib
            s.interference_energy += interference_contrib
            for c, sid in contributions:
                s.sender_energy[sid] = s.sender_energy.get(sid, 0.0) + c
            if top_sender is not None:
                if s.dominant_sender is None:
                    s.dominant_sender = top_sender
                elif s.dominant_sender != top_sender:
                    s.sender_switches += 1
                    s.dominant_sender = top_sender
            s.active_events = active
            if s.started_at is None:
                s.started_at = t
            s.last_signal_at = t
        else:
            self.gap_counter += 1
            if self.signal_counter > 0:
                duration_ticks = self.signal_counter * cfg.MORSE_LISTEN_INTERVAL
                self.signal_counter = 0
                self.session.buffer += '-' if duration_ticks >= dot_len * 2 else '.'
                self.session.symbols.append(self.session.buffer[-1])
            if self.session.buffer:
                gap_ticks = self.gap_counter * cfg.MORSE_LISTEN_INTERVAL
                if gap_ticks >= end_gap:
                    raw_buffer = self.session.buffer.strip()
                    if raw_buffer:
                        event = self._build_event(t, raw_buffer)
                        if event['reliable']:
                            self.metrics['decoded'] += 1
                        elif event['uncertain']:
                            self.metrics['uncertain'] += 1
                        else:
                            self.metrics['corrupted'] += 1
                        if event['collision_detected']:
                            self.metrics['collision'] += 1
                        self.metrics['energy_received'] += self.session.signal_energy
                        old_count = (self.metrics['decoded'] + self.metrics['corrupted']
                                     + self.metrics['uncertain'])
                        self.metrics['mean_confidence'] = (
                            (self.metrics['mean_confidence'] * (old_count - 1)
                             + event['confidence']) / old_count)
                        self.metrics['unique_senders'].update(event['sender_candidates'])
                        self.metrics['sender_switches'] += self.session.sender_switches
                        self.session = Session()
                        self.gap_counter = 0
                        self.diag['rx_sessions'] += 1
                        return event
                    self.session = Session()
                    self.gap_counter = 0
                elif gap_ticks >= block_gap:
                    if not self.session.buffer.endswith(' '):
                        self.session.buffer += ' '
            if (self.session.started_at is not None
                    and self.session.last_signal_at is not None
                    and (t - self.session.last_signal_at) > cfg.MORSE_TIMEOUT_TICKS):
                self.metrics['timeout'] += 1
                self.diag['rx_timeouts'] += 1
                self.session = Session()
                self.gap_counter = 0
        return None

    def get_metrics(self):
        metrics = dict(self.metrics)
        metrics['unique_senders'] = len(metrics['unique_senders'])
        return metrics

    def reset_metrics(self):
        self.metrics = {k: (set() if k == 'unique_senders'
                             else 0.0 if k in ('mean_confidence', 'energy_received') else 0)
                         for k in self.metrics}


_INTENT_SCALE = np.array([1.0, 1.0, 1.5, 0.3], dtype=np.float32)

NULL_CLUSTER_ID       = -1
CREATE_NEW_CLUSTER_ID = -2


def _agent_intent_vector(agent) -> np.ndarray:
    grief = float(agent.emotional_memory.get('grief', 0.0))
    grat = float(agent.emotional_memory.get('gratitude', 0.0))
    gap = float(getattr(agent, 'spirit_gap', 0.0))
    energy_deficit = max(0.0, 0.3 - float(getattr(agent, 'energy', 0.3)))
    return np.array([grief, grat, gap, energy_deficit], dtype=np.float32)


def _dominant_intent_axis(intent):
    return int(np.argmax(intent / _INTENT_SCALE))


def _lang_gene_init(speaker, parent=None):
    if parent is not None:
        rng = np.random.default_rng(abs(hash((speaker.id, parent.id))) % (2**32))
        mutation = rng.normal(0, Config.LANG_GENE_MUTATION_STD, size=4)
        speaker._lang_gene = np.clip(parent._lang_gene + mutation, 0.05, 0.95).astype(np.float32)
    else:
        rng = np.random.default_rng(abs(hash(speaker.id)) % (2**32))
        speaker._lang_gene = rng.uniform(0.15, 0.85, size=4).astype(np.float32)


def generate_intent_timeline(speaker):
    intent = _agent_intent_vector(speaker)
    norm = float(np.linalg.norm(intent))
    if norm < Config.LANG_MIN_INTENT_NORM:
        return []

    dot_len = Config.MORSE_DOT_LEN
    normalized = np.clip(intent / _INTENT_SCALE, 0.0, 1.0)
    gene = speaker._lang_gene

    c_short = dot_len * Config.LANG_COST_PER_PULSE_TICK
    c_long  = dot_len * 3 * Config.LANG_COST_PER_PULSE_TICK
    gene_mean = float(np.mean(gene))
    c_avg = c_short * (1 - gene_mean) + c_long * gene_mean
    n_aff = int(speaker.energy / max(c_avg, 1e-9))
    K = int(np.clip(n_aff // 4, 0, Config.LANG_PULSES_PER_AXIS_MAX))
    if K < 1:
        return []

    timeline = []
    for axis in range(4):
        value = float(normalized[axis])
        base_long = int(round(value * K * Config.LANG_GENE_GAIN * float(gene[axis])))
        base_long = max(0, min(K, base_long))
        for j in range(K):
            dur = dot_len * 3 if j < base_long else dot_len
            timeline.append(('PULSE', dur))
            if j < K - 1:
                timeline.append(('GAP', dot_len))
        timeline.append(('GAP', dot_len * Config.MORSE_INTER_AXIS_GAP_MULT))
    # Protocol extension: a fifth block '.-.' marks an explicit SEEK request.
    # The first four blocks retain their existing meaning (grief/gratitude/gap/energy).
    current_intent = getattr(speaker, 'intent', None)
    if isinstance(current_intent, dict) and current_intent.get('type') == 'seek_help':
        for j, dur in enumerate((dot_len, dot_len * 3, dot_len)):
            timeline.append(('PULSE', dur))
            if j < 2:
                timeline.append(('GAP', dot_len))
        timeline.append(('GAP', dot_len * Config.MORSE_INTER_AXIS_GAP_MULT))
    timeline.append(('GAP', dot_len * Config.MORSE_TRAILING_GAP))

    total_pulse_ticks = sum(d for kind, d in timeline if kind == 'PULSE')
    cost = total_pulse_ticks * Config.LANG_COST_PER_PULSE_TICK
    # Payment is deferred until the transmitter accepts the complete timeline.
    if cost > float(speaker.energy):
        return []
    return timeline


class EmergentLexicon:
    def __init__(self):
        self.clusters: List[str] = []
        self.cluster_targets: Dict[int, float] = {}
        self.cluster_meanings: Dict[int, np.ndarray] = {}
        self.assoc: Dict[int, np.ndarray] = defaultdict(lambda: np.zeros(4, dtype=np.float32))
        self.cluster_uses: Dict[int, int] = defaultdict(int)
        self.cluster_senders: Dict[int, Counter] = defaultdict(Counter)
        self.pending = deque()
        self.log_raw_lengths = deque(maxlen=3000)
        self.log_cluster_count = deque(maxlen=3000)
        self.log_gene_diversity = deque(maxlen=3000)
        self.reinforcement_events = 0
        self.reinforcement_log = deque(maxlen=3000)
        self._axis_switch_diag = {'same': 0, 'switched': 0}
        self._sender_last_axis = {}
        self._ctx_mean = np.zeros(4, dtype=np.float32)
        self.rx_total = 0
        self.rx_null = Counter()
        self.rx_block_counts = Counter()
        self.send_k_counts = Counter()
        self.send_silence = Counter()
        self.send_energy_spent = 0.0
        self.chan = Counter()
        self.fidelity_log = deque(maxlen=3000)
        self.decode_block_count_log    = deque(maxlen=3000)
        self.null_cluster_log          = deque(maxlen=3000)
        self.cluster_meanings_norm_log = deque(maxlen=3000)

    @staticmethod
    def _long_ratio(raw):
        symbols = [c for c in raw if c != ' ']
        if not symbols:
            return 0.5
        return sum(1 for c in symbols if c == '-') / len(symbols)

    def _similarity(self, a, b):
        la = len([c for c in a if c != ' '])
        lb = len([c for c in b if c != ' '])
        len_sim = 1.0 - abs(la - lb) / max(la, lb, 1)
        ra, rb = self._long_ratio(a), self._long_ratio(b)
        ratio_sim = 1.0 - abs(ra - rb)
        return 0.5 * len_sim + 0.5 * ratio_sim

    @staticmethod
    def _decode_to_vector(raw):
        if not raw or not raw.strip():
            return None, 0
        blocks = [b for b in raw.split() if b.strip()]
        nb = len(blocks)
        if nb == 0:
            return None, 0
        n_blocks_original = nb
        if nb > 4:
            blocks = blocks[:4]
        if nb < 2:
            return None, n_blocks_original
        vec = np.zeros(4, dtype=np.float32)
        for i, blk in enumerate(blocks[:4]):
            syms = [c for c in blk if c in '.-']
            if not syms:
                continue
            vec[i] = sum(1 for c in syms if c == '-') / len(syms)
        return vec * _INTENT_SCALE, n_blocks_original

    def cluster_id_for(self, raw):
        decoded, n_blocks = self._decode_to_vector(raw)
        if decoded is None:
            return NULL_CLUSTER_ID, None
        norm_d = float(np.linalg.norm(decoded))
        if norm_d < Config.LANG_NULL_DECODED_NORM:
            return NULL_CLUSTER_ID, None
        if not self.cluster_meanings:
            return CREATE_NEW_CLUSTER_ID, decoded
        best_id, best_cos = -1, -1.0
        for cid, m in self.cluster_meanings.items():
            m_norm = float(np.linalg.norm(m)) + 1e-9
            cos = float(np.dot(decoded, m) / (norm_d * m_norm))
            if cos > best_cos:
                best_cos, best_id = cos, cid
        if best_id != -1 and best_cos >= 1.0 - Config.LANG_CLUSTER_TOL:
            return best_id, decoded
        if len(self.cluster_meanings) < Config.LANG_MAX_CLUSTERS:
            return CREATE_NEW_CLUSTER_ID, decoded
        return best_id, decoded

    def register_reception(self, raw, receiver, sender_id, t):
        _receiver_obj = receiver if hasattr(receiver, 'emotional_memory') else None
        if isinstance(receiver, int) and not isinstance(receiver, bool):
            _receiver_id = receiver
        else:
            _receiver_id = getattr(receiver, 'id', -1)

        decoded, n_blocks = self._decode_to_vector(raw)
        self.rx_total += 1
        self.rx_block_counts[n_blocks] += 1
        self.decode_block_count_log.append(n_blocks)

        if decoded is None:
            self.rx_null['quiet' if n_blocks == 0 else 'blocks'] += 1
            self.null_cluster_log.append({
                't': t, 'receiver': _receiver_id, 'sender': sender_id,
                'raw_len': len(raw.replace(' ', '')), 'n_blocks': n_blocks,
            })
            return NULL_CLUSTER_ID

        norm_d = float(np.linalg.norm(decoded))
        self.cluster_meanings_norm_log.append(norm_d)
        if norm_d < Config.LANG_NULL_DECODED_NORM:
            self.rx_null['norm'] += 1
            self.null_cluster_log.append({
                't': t, 'receiver': _receiver_id, 'sender': sender_id,
                'raw_len': len(raw.replace(' ', '')), 'n_blocks': n_blocks,
            })
            return NULL_CLUSTER_ID

        if not self.cluster_meanings:
            cid = CREATE_NEW_CLUSTER_ID
        else:
            best_id, best_cos = -1, -1.0
            for c, m in self.cluster_meanings.items():
                m_norm = float(np.linalg.norm(m)) + 1e-9
                cos = float(np.dot(decoded, m) / (norm_d * m_norm))
                if cos > best_cos:
                    best_cos, best_id = cos, c
            if best_id != -1 and best_cos >= 1.0 - Config.LANG_CLUSTER_TOL:
                cid = best_id
            elif len(self.cluster_meanings) < Config.LANG_MAX_CLUSTERS:
                cid = CREATE_NEW_CLUSTER_ID
            else:
                cid = best_id

        if cid == CREATE_NEW_CLUSTER_ID:
            cid = len(self.cluster_meanings)
            self.cluster_meanings[cid] = decoded.copy()
            self.clusters.append(raw)
        else:
            self.cluster_meanings[cid] = (
                0.9 * self.cluster_meanings[cid] + 0.1 * decoded).astype(np.float32)

        self.cluster_uses[cid] += 1
        if sender_id is not None and sender_id != -1:
            self.cluster_senders[cid][sender_id] += 1

        ratio = self._long_ratio(raw)
        old_target = self.cluster_targets.get(cid, ratio)
        self.cluster_targets[cid] = 0.9 * old_target + 0.1 * ratio

        if _receiver_obj is not None:
            ctx_before = np.array([
                float(_receiver_obj.emotional_memory.get('grief', 0.0)),
                float(_receiver_obj.emotional_memory.get('gratitude', 0.0)),
                float(getattr(_receiver_obj, 'spirit_gap', 0.0)),
                max(0.0, 0.3 - float(getattr(_receiver_obj, 'energy', 0.3))),
            ], dtype=np.float32)
        else:
            ctx_before = np.zeros(4, dtype=np.float32)

        self.pending.append((t, cid, _receiver_id, sender_id, ctx_before))
        self.log_raw_lengths.append(len(raw.replace(' ', '')))
        self.log_cluster_count.append(len(self.clusters))
        return cid

    def reinforce_sender(self, sender, cid):
        if sender is None or not getattr(sender, 'alive', False):
            return
        if cid in (NULL_CLUSTER_ID, CREATE_NEW_CLUSTER_ID):
            return
        gene = getattr(sender, '_lang_gene', None)
        if gene is None:
            return
        learned = self.cluster_meanings.get(cid)
        if learned is None:
            return

        target_vec = np.clip(learned / _INTENT_SCALE, 0.0, 1.0).astype(np.float32)
        intent = _agent_intent_vector(sender)
        if float(np.linalg.norm(intent)) < 1e-6:
            return
        value_norm = np.clip(intent / _INTENT_SCALE, 0.0, 1.0).astype(np.float32)

        denom = Config.LANG_GENE_GAIN * value_norm + 1e-6
        target_gene = np.clip(target_vec / denom, 0.0, 1.0)

        weights = value_norm.copy()
        weights[weights < Config.LANG_AXIS_ACTIVE_MIN] = 0.0
        if float(weights.sum()) < 1e-9:
            return
        weights = weights / float(weights.sum())

        lr = Config.LANG_REINFORCE_LR
        before = gene.copy()
        gene[:] = np.clip(
            (1 - lr * weights) * gene + (lr * weights) * target_gene,
            0.05, 0.95).astype(np.float32)

        self.reinforcement_events += 1
        self.reinforcement_log.append(float(np.mean(np.abs(before - target_gene))))

        axis = _dominant_intent_axis(intent)
        prev = self._sender_last_axis.get(sender.id)
        if prev is not None:
            self._axis_switch_diag['same' if prev == axis else 'switched'] += 1
        self._sender_last_axis[sender.id] = axis

        sender.soul_weight = min(1.0, float(sender.soul_weight)
                                 + Config.LANG_REINFORCE_SOUL_BONUS)
        # MORSE-EVO: счётчик успешной коммуникации (используется can_divide при COMM_DIVIDE_BONUS_ENABLE и в метриках)
        sender._comm_success_count = int(getattr(sender, '_comm_success_count', 0)) + 1

    def resolve_context(self, patterns_by_id, t, window=5):
        while self.pending:
            item = self.pending[0]
            if len(item) == 5:
                t_recv, cid, rid, sid, ctx_before = item
            else:
                t_recv, cid, rid, sid = item
                ctx_before = None
            if t - t_recv < window:
                break
            self.pending.popleft()
            receiver = patterns_by_id.get(rid)
            if receiver is None or not getattr(receiver, 'alive', False):
                continue
            if ctx_before is not None and float(np.linalg.norm(ctx_before)) > 1e-6:
                alpha = 0.02
                centered = ctx_before - self._ctx_mean
                self._ctx_mean = (1 - alpha) * self._ctx_mean + alpha * ctx_before
            else:
                ctx = np.array([
                    float(receiver.emotional_memory.get('grief', 0.0)),
                    float(receiver.emotional_memory.get('gratitude', 0.0)),
                    float(getattr(receiver, 'spirit_gap', 0.0)),
                    max(0.0, 0.3 - float(getattr(receiver, 'energy', 0.3))),
                ], dtype=np.float32)
                centered = ctx
            old_assoc = self.assoc[cid]
            new_assoc = (1 - Config.LANG_HEBB_LR) * old_assoc + Config.LANG_HEBB_LR * centered
            self.assoc[cid] = new_assoc
            norm = float(np.linalg.norm(new_assoc))
            if norm > 1e-6 and sid is not None and sid != -1:
                scaled = new_assoc / _INTENT_SCALE
                probs = np.abs(scaled) / (np.abs(scaled).sum() + 1e-8)
                confidence = float(np.max(probs))
                if confidence > Config.LANG_REINFORCE_CONFIDENCE_MIN:
                    sender = patterns_by_id.get(sid)
                    self.reinforce_sender(sender, cid)
        for cid in list(self.assoc.keys()):
            self.assoc[cid] *= (1.0 - Config.LANG_HEBB_DECAY)

    def log_population_gene_diversity(self, patterns_by_id):
        genes = [p._lang_gene for p in patterns_by_id.values()
                 if getattr(p, 'alive', False)]
        if len(genes) < 2:
            return
        arr = np.stack(genes)
        self.log_gene_diversity.append(float(np.mean(np.std(arr, axis=0))))

    def dominant_meaning_labels(self):
        names = ['grief', 'gratitude', 'spirit_gap', 'energy_deficit']
        if not self.cluster_meanings:
            return {}
        all_mean = np.mean(list(self.cluster_meanings.values()), axis=0)
        out = {}
        for cid, vec in self.cluster_meanings.items():
            centered = vec - all_mean
            if np.linalg.norm(centered) < 1e-3:
                out[cid] = 'ambiguous'
                continue
            scaled = centered / _INTENT_SCALE
            probs = np.abs(scaled) / (np.abs(scaled).sum() + 1e-8)
            if float(np.max(probs)) < Config.LANG_REINFORCE_CONFIDENCE_MIN:
                out[cid] = 'ambiguous'
                continue
            out[cid] = names[int(np.argmax(scaled))]
        return out


def _morse_init_patch(self):
    if not hasattr(self, '_morse_queue'):
        self._morse_queue = deque(maxlen=5)
    if not hasattr(self, '_morse_tx'):
        self._morse_tx = Transmitter(self.id, None, Config)
    if not hasattr(self, '_morse_rx'):
        self._morse_rx = Receiver(self.id, None, Config)
    if not hasattr(self, '_dialogue_cooldown'):
        self._dialogue_cooldown = -1
    for attr in ['_morse_timeline', '_morse_idx', '_morse_timer', '_morse_buffer',
                 '_morse_signal_counter', '_morse_noise_floor', '_morse_last_sender',
                 '_morse_gap_counter', '_morse_last_send_t', '_morse_last_recv_t',
                 '_morse_silence_counter', '_morse_backoff']:
        if not hasattr(self, attr):
            if attr == '_morse_noise_floor':
                setattr(self, attr, 0.01)
            elif attr in ('_morse_last_sender', '_morse_last_send_t', '_morse_last_recv_t'):
                setattr(self, attr, -100 if 'send' in attr or 'recv' in attr else -1)
            else:
                setattr(self, attr, 0)


def _morse_set_field(self, field):
    self._morse_tx.set_field(field)
    self._morse_rx.set_field(field)


def _morse_flush_diag(self, lex=None):
    if lex is None:
        lex = self.world.EMERGENT_LEXICON if getattr(self, 'world', None) is not None else None
    for obj in (self._morse_tx, self._morse_rx):
        d = obj.diag
        if d:
            if lex is not None:
                for k, v in d.items():
                    lex.chan[k] += v
            d.clear()


def _morse_reset_patch(self):
    self._morse_timeline = []
    self._morse_idx = 0
    self._morse_timer = 0
    self._morse_buffer = ''
    self._morse_signal_counter = 0
    self._morse_gap_counter = 0
    self._morse_last_sender = -1
    self._morse_silence_counter = 0
    self._morse_backoff = 0
    self._dialogue_cooldown = -1
    self._morse_queue.clear()
    self._morse_queue_next_try = 0   # MORSE-EVO
    self._morse_tx.reset()
    self._morse_rx.reset_metrics()
    self._morse_flush_diag()


def _get_center_cell(self):
    cx, cy = self.get_center()
    return int(cx), int(cy)


def _send_morse(self, _unused_text_arg, field, recipient_id=None):
    lex = self.world.EMERGENT_LEXICON if self.world is not None else None

    if not Config.ENABLE_MORSE_COMMUNICATION or not self.alive or not self.cells:
        return False
    if self._morse_backoff > 0:
        if lex is not None: lex.send_silence['backoff'] += 1
        return False
    if self._morse_tx.active:
        if lex is not None: lex.send_silence['tx_active'] += 1
        return False

    center = self._get_center_cell()
    tx_field = self._morse_tx.field
    _crisis = self._morse_in_crisis()   # MORSE-EVO
    if Config.MORSE_NAV_ENABLED and tx_field is not None:
        if tx_field.nearby_transmitters(center, Config.MORSE_NAV_RADIUS,
                                        exclude=self.id):
            if _crisis and getattr(Config, 'MORSE_CRISIS_OVERRIDE_NAV', False):
                # MORSE-EVO: в кризисе не уступаем соседям, передаём сразу
                if lex is not None: lex.chan['tx_crisis_nav_override'] += 1
            else:
                self._morse_backoff = int(phi_hash(self.id, self.age, 5512)
                                           * Config.MORSE_BACKOFF_MAX) + 1
                if lex is not None:
                    lex.send_silence['nav_busy'] += 1
                    lex.chan['tx_nav_deferred'] += 1
                self._morse_enqueue(recipient_id)   # MORSE-EVO: не теряем, повторим из очереди
                return False
    if self._morse_rx.field is not None:
        state = self._morse_rx.field.get_state(center, Config.MORSE_LISTEN_RADIUS)
        if state['mean'] > 0.05:
            self._morse_backoff = int(phi_hash(self.id, self.age, 5511)
                                       * Config.MORSE_BACKOFF_MAX) + 1
            if lex is not None: lex.send_silence['busy_channel'] += 1
            return False
    if self.age - self._morse_last_send_t < Config.MORSE_SEND_COOLDOWN:
        if lex is not None: lex.send_silence['cooldown'] += 1
        return False

    intent_pre = _agent_intent_vector(self)
    if float(np.linalg.norm(intent_pre)) < Config.LANG_MIN_INTENT_NORM:
        if lex is not None: lex.send_silence['no_intent'] += 1
        return False

    thr = float(Config.LANG_INTENT_CHANGE_THRESHOLD)
    last = getattr(self, '_last_sent_intent', None)
    if thr > 0.0 and last is not None and last.shape == intent_pre.shape:
        if float(np.linalg.norm(intent_pre - last)) < thr:
            if lex is not None:
                lex.send_silence['no_change'] += 1
            return False

    energy_before = float(self.energy)
    timeline = generate_intent_timeline(self)
    if not timeline:
        if lex is not None: lex.send_silence['too_poor'] += 1
        return False

    pulse_cost = sum(d for kind, d in timeline if kind == 'PULSE') * Config.LANG_COST_PER_PULSE_TICK
    commission = float(self._action_cost('morse_send')) if hasattr(self, '_action_cost') else 0.02
    if _crisis:   # MORSE-EVO: скидка на импульсы в кризисе (комиссия действия не меняется)
        pulse_cost *= float(getattr(Config, 'MORSE_CRISIS_COST_MULT', 1.0))
    # Do not debit either component unless the full transaction is affordable and accepted.
    if not np.isfinite(energy_before) or energy_before < pulse_cost + commission:
        if lex is not None: lex.send_silence['unaffordable_total'] += 1
        return False

    n_pulses = sum(1 for k, _ in timeline if k == 'PULSE')
    K_sent = max(1, n_pulses // 4)

    if self._morse_tx.send_timeline(timeline, center, self.age):
        self._x2_change('X2.morse_transmit_cost', float(self.energy) - pulse_cost)
        energy_spent = pulse_cost
        self._morse_last_send_t = self.age
        self._last_sent_intent = intent_pre.copy()
        self._last_sent_t = self.age
        if lex is not None:
            lex.send_k_counts[K_sent] += 1
            lex.send_energy_spent += energy_spent
            lex.chan['tx_ticks_sum'] += int(sum(d for _, d in timeline))
        self._log_event("morse_enqueue", n_pulses=n_pulses)
        if lex is not None:   # MORSE-EVO
            lex.chan['tx_send_crisis' if _crisis else 'tx_send_normal'] += 1
        return True
    if lex is not None: lex.send_silence['timeline_rejected'] += 1
    return False


_MORSE_QUEUE_TRANSIENT = frozenset(('nav_busy', 'busy_channel', 'backoff', 'tx_active'))


def _morse_in_crisis(self):
    """MORSE-EVO: кризис = мало энергии или высокая тревога."""
    if not getattr(Config, 'MORSE_CRISIS_ENABLE', False):
        return False
    return (float(self.energy) < float(getattr(Config, 'MORSE_CRISIS_ENERGY', 0.1))
            or float(getattr(self, 'alarm_level', 0.0)) > float(getattr(Config, 'MORSE_CRISIS_ALARM', 0.5)))


def _morse_enqueue(self, recipient_id=None):
    """MORSE-EVO: отложенное из-за nav_busy сообщение кладём в очередь (maxlen=5, TTL=MORSE_QUEUE_TTL).
    Содержимое строится в момент отправки из текущего интента, поэтому дубли по одному получателю не копим
    и не ставим в очередь то, что всё равно не пройдёт гейты no_intent / no_change."""
    if not getattr(Config, 'MORSE_QUEUE_ENABLE', False) or getattr(self, '_morse_from_queue', False):
        return False
    lex = self.world.EMERGENT_LEXICON if self.world is not None else None
    intent = _agent_intent_vector(self)
    if float(np.linalg.norm(intent)) < Config.LANG_MIN_INTENT_NORM:
        if lex is not None: lex.chan['tx_queue_skip_nointent'] += 1
        return False
    thr = float(Config.LANG_INTENT_CHANGE_THRESHOLD)
    last = getattr(self, '_last_sent_intent', None)
    if thr > 0.0 and last is not None and last.shape == intent.shape and float(np.linalg.norm(intent - last)) < thr:
        if lex is not None: lex.chan['tx_queue_skip_nointent'] += 1
        return False
    if any(r == recipient_id for _, r in self._morse_queue):
        if lex is not None: lex.chan['tx_queue_dup'] += 1
        return False
    if self._morse_queue.maxlen is not None and len(self._morse_queue) >= self._morse_queue.maxlen:
        if lex is not None: lex.chan['tx_queue_overflow'] += 1
    now = int(self.world.age) if self.world is not None else int(self.age)
    self._morse_queue.append((now, recipient_id))
    if lex is not None: lex.chan['tx_nav_queued'] += 1
    return True


def _process_morse_queue(self, field, t):
    if not Config.ENABLE_MORSE_COMMUNICATION or not self.alive or not self.cells:
        return
    if self._morse_backoff > 0:
        self._morse_backoff -= 1
        return
    center = self._get_center_cell()
    q = self._morse_queue
    if q:
        # MORSE-EVO: раньше очередь никто не наполнял. Теперь повтор идёт через полный _send_morse
        # (все гейты, стоимость и счётчики обёртки ячейки 28), а не в обход nav-проверки.
        lex = self.world.EMERGENT_LEXICON if self.world is not None else None
        ttl = int(getattr(Config, 'MORSE_QUEUE_TTL', 50))
        while q and (t - q[0][0]) > ttl:
            q.popleft()
            if lex is not None: lex.chan['tx_queue_expired'] += 1
        if (q and not self._morse_tx.active and t >= getattr(self, '_morse_queue_next_try', 0)
                and self.age - self._morse_last_send_t >= Config.MORSE_SEND_COOLDOWN):
            self._morse_queue_next_try = t + int(getattr(Config, 'MORSE_QUEUE_RETRY', 5))
            recipient = q[0][1]
            sil0 = dict(lex.send_silence) if lex is not None else {}
            if lex is not None: lex.chan['tx_queue_attempt'] += 1
            self._morse_from_queue = True
            try:
                sent = self._send_morse(None, field, recipient)
            finally:
                self._morse_from_queue = False
            if sent:
                q.popleft()
                if lex is not None: lex.chan['tx_queue_sent'] += 1
            elif lex is not None:
                # «жёсткая» причина отказа (нечего сказать, нет денег, отклонён таймлайн): сообщение бессмысленно ждать
                if any(v > sil0.get(k, 0) and k not in _MORSE_QUEUE_TRANSIENT for k, v in lex.send_silence.items()):
                    q.popleft()
                    lex.chan['tx_queue_dropped_hard'] += 1
    self._morse_tx.step(center, t)


def _listen_morse(self, field, t):
    if not Config.ENABLE_MORSE_COMMUNICATION or not self.alive or not self.cells:
        return
    event = self._morse_rx.listen(self._get_center_cell(), t)
    lex = self.world.EMERGENT_LEXICON if self.world is not None else None
    self._morse_flush_diag(lex)
    if not event or lex is None:
        return
    ch = lex.chan
    ch['rx_conf_reliable' if event['reliable'] else
       'rx_conf_uncertain' if event['uncertain'] else 'rx_conf_corrupted'] += 1
    if event['collision_detected']:
        ch['rx_collision_flag'] += 1
    raw = event.get('raw_signal', '')
    if not raw:
        return
    sender = event.get('dominant_sender', None)
    if (sender is None or not isinstance(sender, (int, np.integer))
            or int(sender) in (-1, self.id)):
        ch['rx_dropped_no_sender'] += 1
        return
    if event.get('mixed') and Config.MORSE_DROP_MIXED:
        ch['rx_dropped_mixed'] += 1
        return
    ch['rx_to_lexicon'] += 1
    self._handle_morse_message(raw, int(sender), event)


def _morse_start_seek_navigation(self, sender_id):
    """Start a direct navigation episode toward a trusted Morse SEEK sender."""
    world = getattr(self, 'world', None)
    target_agent = world.pattern_dict.get(int(sender_id)) if world is not None else None
    if (target_agent is None or not getattr(target_agent, 'alive', False)
            or not getattr(target_agent, 'cells', None)):
        return False
    W = int(Config.WORLD_SIZE)
    cx, cy = self.get_center()
    tx, ty = target_agent.get_center()
    dx = ((int(tx) - int(cx) + W // 2) % W) - W // 2
    dy = ((int(ty) - int(cy) + W // 2) % W) - W // 2
    dist = float(np.hypot(dx, dy))
    direction = (int(np.sign(dx)), int(np.sign(dy)))
    if direction == (0, 0):
        direction = (1, 0)
    age = int(getattr(self, 'age', 0))
    # The request is a real goal, so it supersedes an exploratory waypoint.
    self._nav_progress = {
        'target': (int(tx), int(ty)), 'target_agent_id': int(sender_id),
        'dir': direction, 'start_x': cx, 'start_y': cy,
        'start_dist': max(dist, 1e-6), 'best_dist': float('inf'),
        'stuck_ticks': 0, 'status': 'moving', 'reason': 'morse_seek',
        'wp_t': age, 'dist': dist, 'progress': 0.0, 'recalc': 0,
        't0': age, 'x0': cx, 'y0': cy, 'path_len': 0.0,
        'far': False, 'revisit': False, 'px': float(cx), 'py': float(cy),
        'origin': 'morse_seek',
    }
    self.goals = [g for g in getattr(self, 'goals', [])
                  if g.get('_source') not in ('spatial_gradient', 'morse_seek')]
    self.goals.append({'type': 'cooperate', 'priority':
                       float(Config.INTENT_COOPERATE_BASE_PRIORITY)
                       + float(self.trust_ledger.get(int(sender_id)))
                       + float(Config.HELP_REQUEST_PRIORITY_BONUS),
                       'target': int(sender_id), 'age': 0,
                       'persistence': int(Config.HELP_REQUEST_TTL), '_source': 'morse_seek'})
    self._log_event('morse_seek_navigation_started', target=int(sender_id),
                    distance=round(dist, 3))
    if dist <= 1.5:
        self._nav_finish('arrived')
    return True


def _handle_morse_message(self, raw_signal, sender_id, event=None):
    if not raw_signal or sender_id is None or sender_id == -1 or self.world is None:
        return
    lex = self.world.EMERGENT_LEXICON
    cid = lex.register_reception(raw_signal, self, sender_id, self.world.age)
    _morse_blocks = [b for b in str(raw_signal).split() if b]
    _is_seek_marker = len(_morse_blocks) >= 5 and _morse_blocks[4] == '.-.'

    if cid == NULL_CLUSTER_ID:
        self._log_event("morse_null_cluster",
                        sender=sender_id,
                        raw_len=len(raw_signal.replace(' ', '')))
        if not _is_seek_marker:
            return

    sender = self.world.pattern_dict.get(sender_id)
    if (sender is not None
            and hasattr(sender, '_last_sent_intent')
            and hasattr(sender, '_last_sent_t')
            and (self.world.age - sender._last_sent_t) < Config.MORSE_TIMEOUT_TICKS):
        decoded_fid, _ = lex._decode_to_vector(raw_signal)
        if decoded_fid is not None:
            orig = sender._last_sent_intent
            denom = float(np.linalg.norm(orig)) + 1e-9
            fid = 1.0 - float(np.linalg.norm(decoded_fid - orig)) / denom
            fid = max(0.0, min(1.0, fid))
            lex.fidelity_log.append(fid)

    self.remember_dialogue(sender_id, f"[SIGNAL cluster={cid}]",
                            getattr(self.world, 'age', 0))
    if self.world and self.world.archive is not None:
        try:
            self.world.archive.write_queue.append({
                'type': 'raw_signal', 'agent_id': self.id, 'sender_id': sender_id,
                'cluster_id': cid, 'raw_len': len(raw_signal.replace(' ', '')),
                't': getattr(self.world, 'age', 0),
                'soul_weight': float(self.soul_weight),
            })
        except Exception:
            pass
    _trust_before = None
    try:
        _trust_before = float(self.trust_ledger.entries.get(sender_id, Config.TRUST_BASE))
    except Exception:
        pass
    # SEEK is carried as an explicit fifth Morse block; ordinary four-block messages
    # remain unchanged. Trust is checked before this reception's generic trust update.
    _reliable_single_sender = bool(event and event.get('reliable', False)
                                   and not event.get('mixed', False))
    _trusted_seek = bool(_is_seek_marker and _reliable_single_sender
                         and _trust_before is not None
                         and _trust_before >= float(Config.HELP_REQUEST_TRUST_MIN))
    self.trust_ledger.update(sender_id, 'helpful', multiplier=0.3)
    if _trusted_seek:
        _seek_started = self._morse_start_seek_navigation(int(sender_id))
        if _seek_started:
            self._log_event('morse_seek_registered', sender=int(sender_id),
                            confidence=round(float(event.get('confidence', 0.0)), 3),
                            trust=round(float(_trust_before), 3))
        else:
            self._log_event('morse_seek_unavailable', sender=int(sender_id),
                            reason='sender_not_present')
    try:
        _trust_after = float(self.trust_ledger.entries.get(sender_id, Config.TRUST_BASE))
        lex.morse_trust_updates = int(getattr(lex, 'morse_trust_updates', 0)) + 1
        lex.morse_trust_delta_sum = float(getattr(lex, 'morse_trust_delta_sum', 0.0)) + (
            _trust_after - _trust_before if _trust_before is not None else 0.0)
        lex.morse_trust_delta_events = list(getattr(lex, 'morse_trust_delta_events', []))
        lex.morse_trust_delta_events.append(float(_trust_after - _trust_before)
                                             if _trust_before is not None else 0.0)
    except Exception:
        pass

    if event and event.get('reliable', False):
        learned = lex.cluster_meanings.get(cid)
        if (learned is not None
                and float(np.linalg.norm(learned)) > Config.LANG_NULL_DECODED_NORM):
            g, gr, gap, ed = [float(x) for x in learned]
            sig = (round(g, 1), round(gr, 1), round(gap, 1),
                   f"signal_from_{sender_id}")
            self.concept_graph.update(sig, np.array([g, gr, gap, ed]))
            self._log_event("morse_concept_formed",
                            concept=str(sig)[:60], cluster=cid,
                            learned_norm=round(float(np.linalg.norm(learned)), 3))

    self._log_event("morse_handled", sender=sender_id, cluster=cid,
                    raw_len=len(raw_signal.replace(' ', '')))


if "extend_class" not in globals():
    raise RuntimeError("Нет extend_class: сначала выполни ячейку Cell 3f (Substates Engine).")


class MorseMixin:
    _morse_init = _morse_init_patch
    _morse_set_field = _morse_set_field
    _morse_reset = _morse_reset_patch
    _morse_flush_diag = _morse_flush_diag
    _get_center_cell = _get_center_cell
    _send_morse = _send_morse
    _process_morse_queue = _process_morse_queue
    _morse_in_crisis = _morse_in_crisis        # MORSE-EVO
    _morse_enqueue = _morse_enqueue            # MORSE-EVO
    _listen_morse = _listen_morse
    _handle_morse_message = _handle_morse_message
    _morse_start_seek_navigation = _morse_start_seek_navigation

    def __init__(self, pid, cells, parent=None, world=None):
        super().__init__(pid, cells, parent, world)
        self._morse_init()
        _lang_gene_init(self, parent=parent)
        if world and hasattr(world, 'morse_field'):
            self._morse_set_field(world.morse_field)

    def should_die(self):
        if super().should_die():
            self._morse_reset()
            return True
        return False

    def update_model(self, field, lineage_counts=None, given_counter=None, t=None, witness=None):
        if Config.ENABLE_MORSE_COMMUNICATION:
            self._process_morse_queue(field, t if t is not None else 0)
            self._listen_morse(field, t if t is not None else 0)
        return super().update_model(field, lineage_counts, given_counter, t, witness)

    def generate_goals(self, field):
        super().generate_goals(field)
        # MORSE-EVO: в кризисе пробуем говорить и без интента seek_help/cooperate (но не чаще раза в MORSE_SEND_COOLDOWN)
        _intent_ok = bool(self.intent and self.intent.get('type') in ('seek_help', 'cooperate'))
        _crisis_ok = (not _intent_ok and Config.ENABLE_MORSE_COMMUNICATION and self.alive
                      and self.age - self._morse_last_send_t >= Config.MORSE_SEND_COOLDOWN
                      and self._morse_in_crisis())
        if Config.ENABLE_MORSE_COMMUNICATION and self.alive and (_intent_ok or _crisis_ok):
            if phi_hash(self.id, self.age, 8888) < Config.MORSE_SEND_PROB:
                self._send_morse(None, field)


Pattern = extend_class(Pattern, MorseMixin)


print("✅ Cell 13 v4.5: мягкий декодер, NAV_RADIUS=6, SEND_COOLDOWN=60, PULSES_PER_AXIS_MAX=3.")


In [ ]:
# Ячейка 14b: PERF-префлайт — быстрый CommunicationField.sync_to_field обязан совпасть с эталоном БИТ-В-БИТ.
# Не запускает симуляцию. При расхождении автоматически возвращает эталон (медленный, но исходный).
import time as _t_pf
import numpy as _np_pf


def _sync_equivalence_preflight():
    W = Config.WORLD_SIZE
    rng = _np_pf.random.RandomState(832)
    bad = []
    t_ref = t_fast = 0.0
    for scenario, t_now in enumerate((0, 7, 1234)):
        cf = CommunicationField(Config)
        cf.step(t_now)
        # события: у краёв/углов (проверка оборачивания по тору), в центре, разные амплитуды,
        # одно «из будущего» (неактивное), одно с нецелым центром (медленный путь), другой радиус/decay
        corners = [(0, 0), (W - 1, W - 1), (0, W - 1), (W - 1, 0), (W // 2, W // 2), (2, W - 3)]
        for k, (x, y) in enumerate(corners):
            cf.add_pulse(k, (x, y), 1.0, t_now, 6, Config.MORSE_BROADCAST_RADIUS, Config.MORSE_DECAY)
        for k in range(8):
            x, y = int(rng.randint(0, W)), int(rng.randint(0, W))
            cf.add_pulse(100 + k, (x, y), float(rng.uniform(0.2, 1.5)), t_now, 6,
                         Config.MORSE_BROADCAST_RADIUS, Config.MORSE_DECAY)
        cf.add_pulse(200, (10, 10), 1.0, t_now + 50, 5, Config.MORSE_BROADCAST_RADIUS, Config.MORSE_DECAY)   # неактивное
        cf.add_pulse(201, (30, 40), 0.7, t_now, 6, 3, 2.0)                                                   # другой радиус/decay
        if scenario == 2:
            cf.add_pulse(202, (50.5, 60.25), 0.9, t_now, 6, Config.MORSE_BROADCAST_RADIUS, Config.MORSE_DECAY)  # нецелый центр
        ch = int(Config.MORSE_CHANNEL)
        fa = _np_pf.zeros((W, W, Config.CHANNELS), dtype=_np_pf.float64)
        fb = fa.copy()
        a0 = _t_pf.time(); CommunicationField.sync_to_field_reference(cf, fa, ch); t_ref += _t_pf.time() - a0
        b0 = _t_pf.time(); CommunicationField.sync_to_field(cf, fb, ch); t_fast += _t_pf.time() - b0
        if not _np_pf.array_equal(fa, fb):
            bad.append((scenario, float(_np_pf.abs(fa - fb).max())))
        f32a, f32b = fa.astype(_np_pf.float32), fb.astype(_np_pf.float32)
        if not _np_pf.array_equal(f32a, f32b):
            bad.append((scenario, "float32"))
    return bad, t_ref, t_fast


try:
    _pf_bad, _pf_ref, _pf_fast = _sync_equivalence_preflight()
    if _pf_bad:
        CommunicationField.sync_to_field = CommunicationField.sync_to_field_reference
        print(f"[PERF sync_to_field] ❌ расхождение с эталоном {_pf_bad} → ОТКАТ на эталонную реализацию")
    else:
        print(f"[PERF sync_to_field] PASS: бит-в-бит равно эталону (3 сценария); "
              f"эталон {_pf_ref:.2f} с → быстрый {_pf_fast:.2f} с (в этом микротесте, ~14 событий)")
except Exception as _e_pf:
    CommunicationField.sync_to_field = CommunicationField.sync_to_field_reference
    print(f"[PERF sync_to_field] префлайт упал: {_e_pf!r} → ОТКАТ на эталонную реализацию")


def _neighbor_sync_preflight():
    """Локальная соседская сумма == scipy-эталон бит-в-бит (float32 и float64, углы/края тора, широкий диапазон значений)."""
    rng = _np_pf.random.RandomState(832)
    n_ok = n_bad = 0
    for dtype in (_np_pf.float32, _np_pf.float64):
        for wide in (False, True):
            for _ in range(60):
                H = Wd = int(rng.choice([16, 33, Config.WORLD_SIZE]))
                f = _np_pf.zeros((H, Wd, Config.CHANNELS), dtype)
                f[:, :, CH['owner']] = rng.randint(0, 12, (H, Wd))
                for ch in (CH['signal_gratitude'], CH['signal_grief']):
                    if wide:
                        f[:, :, ch] = (rng.randn(H, Wd) * 10.0 ** rng.randint(-9, 4, (H, Wd))).astype(dtype)
                    else:
                        f[:, :, ch] = rng.rand(H, Wd)
                aid = int(rng.randint(1, 12))
                cells = sorted({(int(rng.choice([0, H - 1, int(rng.randint(0, H))])),
                                 int(rng.choice([0, Wd - 1, int(rng.randint(0, Wd))]))) for _ in range(int(rng.randint(1, 14)))})
                ca = _np_pf.array(cells); xs, ys = ca[:, 0], ca[:, 1]
                a = _neighbor_sync_reference(f, aid, xs, ys)
                b = _neighbor_sync_local(f, aid, xs, ys)
                same = all(type(u) is type(v) and (u.tobytes() == v.tobytes() if isinstance(u, _np_pf.generic) else u == v)
                           for u, v in zip(a, b))
                n_ok += same; n_bad += (not same)
    return n_ok, n_bad


try:
    _nb_ok, _nb_bad = _neighbor_sync_preflight()
    if _nb_bad:
        _PERF_LOCAL_NEIGHBOR_SYNC = False
        print(f"[PERF neighbor_sync] ❌ расхождений с эталоном: {_nb_bad}/{_nb_ok + _nb_bad} → ОТКАТ на scipy-эталон")
    else:
        print(f"[PERF neighbor_sync] PASS: бит-в-бит равно scipy-эталону ({_nb_ok} случаев, float32+float64)")
except Exception as _e_nb:
    _PERF_LOCAL_NEIGHBOR_SYNC = False
    print(f"[PERF neighbor_sync] префлайт упал: {_e_nb!r} → ОТКАТ на scipy-эталон")


In [ ]:
# Ячейка 16
import numpy as np
from collections import deque, defaultdict

_sr_cfg = {
    'ENABLE_SELF_REGULATION': True,
    'SR_MEDITATION_COOLDOWN': 80,
    'SR_MEDITATION_MIN_AGE': 30,
    'SR_MEDITATION_GRIEF_THR': 0.5,
    'SR_MEDITATION_GRIEF_RELIEF': 0.06,
    'SR_MEDITATION_TENSION_RELIEF': 0.08,
    'SR_GOAL_PERF_WINDOW': 30,
    'SR_GOAL_REPLACE_MIN_AGE': 20,
    'SR_GOAL_PERF_FLOOR': -0.05,
    'SR_THR_ADJUST_RATE': 0.005,
    'SR_GRIEF_THR_MIN': 0.2, 'SR_GRIEF_THR_MAX': 0.9,
    'SR_ALARM_THR_MIN': 0.2, 'SR_ALARM_THR_MAX': 0.8,
    'SR_VOTE_INTERVAL': 100,
    'SR_VOTE_THRESHOLD': 0.6,
    'SR_VOTE_QUORUM': 5,
    'SR_NORM_MIN_REPS': 5,
    'SR_NORM_SUCCESS_THR': 0.7,
    'SR_CONFLICT_TRUST_THR': 0.7,
    'SR_TRADE_COOLDOWN': 50,
    'SR_TRADE_MIN_TRUST': 0.5,
    'SR_TRADE_RADIUS': 6,
    'SR_GENE_MUT_RATE': 0.05,
    'SR_GENE_MUT_COOLDOWN': 200,
    'SR_HEAL_COOLDOWN': 30,
    'SR_HEAL_ENERGY_COST': 0.01,
}
for _k, _v in _sr_cfg.items():
    if not hasattr(Config, _k):
        setattr(Config, _k, _v)

def _sr_init(self):
    if hasattr(self, '_sr_initialized'):
        return
    self._sr_initialized = True
    self._sr_last_meditation = -Config.SR_MEDITATION_COOLDOWN
    self._sr_meditation_count = 0
    self._sr_goal_feedback = {}
    self._sr_goals_replaced = 0
    self._sr_grief_threshold = 0.5
    self._sr_alarm_threshold = 0.5
    self._sr_thresholds_adjusted = 0
    self._sr_proposals_made = 0
    self._sr_last_proposal_t = -Config.SR_VOTE_INTERVAL
    self._sr_votes_cast = 0
    self._sr_proposals_passed = 0
    self._sr_action_history = deque(maxlen=50)
    self._sr_norms_inferred = 0
    self._sr_conflicts_resolved = 0
    self._sr_last_trade_t = -Config.SR_TRADE_COOLDOWN
    self._sr_trades_done = 0
    self._sr_constitution = []
    self._sr_last_gene_mut = -Config.SR_GENE_MUT_COOLDOWN
    self._sr_gene_mutations = 0
    self._sr_last_heal_t = -Config.SR_HEAL_COOLDOWN
    self._sr_field_heals = 0

def _sr_should_meditate(self, t):
    if not Config.ENABLE_SELF_REGULATION: return False
    if self.age < Config.SR_MEDITATION_MIN_AGE: return False
    if t - self._sr_last_meditation < Config.SR_MEDITATION_COOLDOWN: return False
    if not self.alive or self.in_dream: return False
    grief = float(self.emotional_memory.get('grief', 0.0))
    if grief > self._sr_grief_threshold: return True
    if getattr(self, 'semantic_state_age', 0) > 40:
        bad = sum(1 for fb in self._sr_goal_feedback.values() if fb.get('perf', 0) < Config.SR_GOAL_PERF_FLOOR)
        if bad > 0: return True
    if self.age % 200 == 0 and self.age > 0: return True
    return False

def _sr_meditate(self, field, t):
    self._sr_last_meditation = t
    self._sr_meditation_count += 1
    old_grief = float(self.emotional_memory.get('grief', 0.0))
    old_tension = float(self.cognitive_tension)
    self.emotional_memory['grief'] = max(0.0, old_grief - Config.SR_MEDITATION_GRIEF_RELIEF)
    self.cognitive_tension = max(0.0, old_tension - Config.SR_MEDITATION_TENSION_RELIEF)
    self.soul_weight = min(1.0, self.soul_weight + 0.005)
    replaced = self._sr_reevaluate_goals()
    self._sr_adjust_thresholds()
    if old_grief > 0.6:
        self._sr_evolve_genes()
    self._log_event("sr_meditation", grief_before=round(old_grief, 3),
                    grief_after=round(float(self.emotional_memory['grief']), 3), goals_replaced=replaced)
    if self.world and hasattr(self.world, 'witness'):
        self.world.witness.record(self.id, "sr_meditation",
                                  grief_relief=round(old_grief - float(self.emotional_memory['grief']), 3),
                                  goals_replaced=replaced)

def _sr_track_goals(self):
    if not Config.ENABLE_SELF_REGULATION: return
    state = (float(self.emotional_memory.get('grief', 0.0)),
             float(self.emotional_memory.get('gratitude', 0.0)),
             float(self.soul_weight), float(self.cognitive_tension))
    active = set()
    for g in self.goals:
        gt = g.get('type')
        if gt is None: continue
        active.add(gt)
        if gt not in self._sr_goal_feedback:
            self._sr_goal_feedback[gt] = {'start_age': self.age, 'start_state': state, 'perf': 0.0}
        else:
            fb = self._sr_goal_feedback[gt]
            elapsed = self.age - fb['start_age']
            if 0 < elapsed <= Config.SR_GOAL_PERF_WINDOW:
                sg, sg0 = state[0], fb['start_state'][0]
                gr, gr0 = state[1], fb['start_state'][1]
                sw, sw0 = state[2], fb['start_state'][2]
                tn, tn0 = state[3], fb['start_state'][3]
                delta = (sg0 - sg)*0.3 + (gr - gr0)*0.3 + (sw - sw0)*0.2 + (tn0 - tn)*0.2
                fb['perf'] = 0.9 * fb['perf'] + 0.1 * delta
    for gt in list(self._sr_goal_feedback.keys()):
        if gt not in active: del self._sr_goal_feedback[gt]

def _sr_reevaluate_goals(self):
    replaced = 0
    to_remove, to_add = [], []
    for g in self.goals:
        gt = g.get('type')
        if gt is None: continue
        if g.get('age', 0) < Config.SR_GOAL_REPLACE_MIN_AGE: continue
        fb = self._sr_goal_feedback.get(gt)
        if fb is None: continue
        if fb['perf'] < Config.SR_GOAL_PERF_FLOOR:
            alt = self._sr_suggest_goal(gt)
            if alt and alt != gt:
                to_remove.append(g)
                to_add.append({'type': alt, 'priority': g.get('priority', 1.0)*0.9,
                               'target': g.get('target'), 'age': 0, 'persistence': 30,
                               '_source': 'sr_reevaluated', '_replaced_from': gt})
                replaced += 1
    for g in to_remove:
        self.goals.remove(g)
        if g.get('type') in self._sr_goal_feedback: del self._sr_goal_feedback[g['type']]
    for g in to_add:
        self.goals.append(g)
        self._log_event("sr_goal_replaced", old=g.get('_replaced_from','?'), new=g['type'])
    self._sr_goals_replaced += replaced
    return replaced

def _sr_suggest_goal(self, old_type):
    grief = float(self.emotional_memory.get('grief', 0.0))
    grat = float(self.emotional_memory.get('gratitude', 0.0))
    tension = float(self.cognitive_tension)
    alts = {'explore': ['cooperate','rest','seek_help'],
            'cooperate': ['explore','rest','introspect'],
            'seek_help': ['cooperate','rest','explore'],
            'rest': ['explore','cooperate','introspect'],
            'introspect': ['explore','cooperate','rest']}
    cands = alts.get(old_type)
    if not cands: return None
    filtered = []
    for c in cands:
        if c == 'cooperate' and grat < 0.3: continue
        if c == 'explore' and tension > 0.8: continue
        if c == 'seek_help' and grief < 0.3: continue
        filtered.append(c)
    if not filtered: filtered = cands
    return filtered[int(phi_hash(self.id, self.age, _stable_hash(old_type) % 10000)) % len(filtered)]

def _sr_adjust_thresholds(self):
    grief = float(self.emotional_memory.get('grief', 0.0))
    grat = float(self.emotional_memory.get('gratitude', 0.0))
    rate = Config.SR_THR_ADJUST_RATE
    changed = False
    if grief > self._sr_grief_threshold + 0.1 and self.age > 100:
        self._sr_grief_threshold = min(Config.SR_GRIEF_THR_MAX, self._sr_grief_threshold + rate); changed = True
    elif grat > 0.6 and self._sr_alarm_threshold > Config.SR_ALARM_THR_MIN:
        self._sr_alarm_threshold = max(Config.SR_ALARM_THR_MIN, self._sr_alarm_threshold - rate); changed = True
    elif grief < 0.2 and self._sr_grief_threshold > Config.SR_GRIEF_THR_MIN:
        self._sr_grief_threshold = max(Config.SR_GRIEF_THR_MIN, self._sr_grief_threshold - rate*0.5); changed = True
    if changed: self._sr_thresholds_adjusted += 1

def _sr_propose(self, field, t):
    if not Config.ENABLE_SELF_REGULATION: return
    if not self.world or not hasattr(self.world, 'sr_proposals'): return
    if t - self._sr_last_proposal_t < Config.SR_VOTE_INTERVAL: return
    grief = float(self.emotional_memory.get('grief', 0.0))
    if grief <= 0.6: return
    self._sr_last_proposal_t = t
    prop_id = f"{self.id}_{t}"
    self.world.sr_proposals[prop_id] = {
        'type': 'reduce_grief', 'author': self.id, 't': t,
        'votes': {self.id: True},
    }
    self._sr_proposals_made += 1
    self._log_event("sr_proposal", proposal=prop_id)

def _sr_vote(self, field, t):
    if not Config.ENABLE_SELF_REGULATION: return
    if not self.world or not hasattr(self.world, 'sr_proposals'): return
    grief = float(self.emotional_memory.get('grief', 0.0))
    for prop_id, prop in list(self.world.sr_proposals.items()):
        if self.id not in prop['votes']:
            prop['votes'][self.id] = grief > 0.5
            self._sr_votes_cast += 1
        total_votes = len(prop['votes'])
        timed_out = (t - prop['t']) > 200
        if total_votes >= Config.SR_VOTE_QUORUM or timed_out:
            yes = sum(1 for v in prop['votes'].values() if v)
            if total_votes > 0 and yes / total_votes >= Config.SR_VOTE_THRESHOLD:
                if prop['type'] == 'reduce_grief':
                    for voter_id, vote in prop['votes'].items():
                        if not vote or voter_id not in self.world.pattern_dict:
                            continue
                        voter = self.world.pattern_dict[voter_id]
                        if voter.alive:
                            voter.emotional_memory['grief'] = max(
                                0.0, float(voter.emotional_memory.get('grief', 0.0)) * 0.9)
                self._sr_proposals_passed += 1
                self._log_event("sr_proposal_passed", proposal=prop_id, voters=total_votes)
            del self.world.sr_proposals[prop_id]

def _sr_infer_norms(self):
    if not Config.ENABLE_SELF_REGULATION: return
    if len(self._sr_action_history) < Config.SR_NORM_MIN_REPS: return
    action_success = defaultdict(lambda: {'total': 0, 'success': 0})
    for entry in self._sr_action_history:
        action = entry.get('action')
        if action:
            action_success[action]['total'] += 1
            if entry.get('success', False): action_success[action]['success'] += 1
    for action, stats in action_success.items():
        if stats['total'] >= Config.SR_NORM_MIN_REPS:
            rate = stats['success'] / stats['total']
            if rate > Config.SR_NORM_SUCCESS_THR:
                norm_sig = (0.9, 0.9, 0.9, f"sr_norm_{action}")
                if norm_sig not in self.concept_graph.nodes:
                    self.concept_graph.update(norm_sig, np.array([0.1, 0.1, 0.9, 1.0]))
                    self.concept_graph.nodes[norm_sig]['eternal'] = True
                    self._sr_norms_inferred += 1
                    self._log_event("sr_norm_inferred", action=action, rate=round(rate, 3))
                    if self.world and hasattr(self.world, 'witness'):
                        self.world.witness.record(self.id, "sr_norm_inferred", action=action)

def _sr_try_trade(self, field, t):
    if not Config.ENABLE_SELF_REGULATION: return
    if t - self._sr_last_trade_t < Config.SR_TRADE_COOLDOWN: return
    if not self.world or not self.world.pattern_dict: return
    cx, cy = self._get_center_cell()
    ws = Config.WORLD_SIZE
    best_partner, best_trust = None, 0
    for pid, trust in self.trust_ledger.entries.items():
        if trust <= best_trust or pid not in self.world.pattern_dict:
            continue
        partner = self.world.pattern_dict[pid]
        if not partner.alive:
            continue
        px, py = partner._get_center_cell()
        dx = min(abs(px - cx), ws - abs(px - cx))
        dy = min(abs(py - cy), ws - abs(py - cy))
        if max(dx, dy) > Config.SR_TRADE_RADIUS:
            continue
        best_partner, best_trust = partner, trust
    if best_partner is None or best_trust < Config.SR_TRADE_MIN_TRUST: return
    if not (self.energy > 0.3 and best_partner.energy < 0.2):
        # v35: nothing to trade right now -> short retry delay instead of spending the full cooldown
        self._sr_last_trade_t = t - Config.SR_TRADE_COOLDOWN + min(5, int(Config.SR_TRADE_COOLDOWN))
        return
    self._sr_last_trade_t = t
    if self.energy > 0.3 and best_partner.energy < 0.2:
        transfer = 0.05
        price = 1.0 + (1.0 - best_trust) * 2.0
        cost = transfer * price
        if self.energy - cost < 0:
            return
        # HONEST ENERGY: плательщик отдаёт ровно то, что получил партнёр (пара сходится), а спред
        # cost - transfer списывается отдельной строкой-комиссией X2.selfreg_trade_fee (объявленная диссипация).
        _got = float(best_partner._x2_change('X2.selfreg_trade_in', best_partner.energy + transfer))
        self._x2_change('X2.selfreg_trade_out', self.energy - _got)
        if transfer > 0.0 and cost > transfer:
            self._x2_change('X2.selfreg_trade_fee', self.energy - (cost - transfer) * (_got / transfer))
        self.trust_ledger.update(best_partner.id, 'helpful', multiplier=0.5)
        best_partner.trust_ledger.update(self.id, 'helpful', multiplier=0.5)
        self._sr_trades_done += 1
        self._log_event("sr_trade", partner=best_partner.id, amount=transfer, price=round(price, 2))

def _sr_create_new_goal(self):
    if not Config.ENABLE_SELF_REGULATION: return
    if len(self.goals) >= 2: return
    candidates = ['explore', 'cooperate', 'seek_help', 'rest']
    existing = {g.get('type') for g in self.goals}
    valid = [c for c in candidates if c not in existing]
    if not valid: return
    new_type = valid[int(phi_hash(self.id, self.age, 9999) * len(valid)) % len(valid)]
    self.goals.append({'type': new_type, 'priority': 0.5, 'target': None,
                       'age': 0, 'persistence': 20, '_source': 'sr_created'})
    self._log_event("sr_goal_created", type=new_type)

def _sr_update_constitution(self, field, t):
    if not Config.ENABLE_SELF_REGULATION: return
    if self.age % 300 != 0: return
    if self._sr_trades_done > 3 and 'reciprocity' not in [r.get('name') for r in self._sr_constitution]:
        self._sr_constitution.append({'name': 'reciprocity', 't': t})
        self._sr_add_constitution_concept('reciprocity')
        self._log_event("sr_constitution_rule", rule='reciprocity')
    if self._sr_conflicts_resolved > 2 and 'nonviolence' not in [r.get('name') for r in self._sr_constitution]:
        self._sr_constitution.append({'name': 'nonviolence', 't': t})
        self._sr_add_constitution_concept('nonviolence')
        self._log_event("sr_constitution_rule", rule='nonviolence')

def _sr_add_constitution_concept(self, rule_name):
    sig = (0.9, 0.9, 0.9, f"constitution_{rule_name}")
    if sig not in self.concept_graph.nodes:
        self.concept_graph.update(sig, np.array([0.1, 0.1, 0.9, 1.0]))
        self.concept_graph.nodes[sig]['eternal'] = True

def _sr_evolve_genes(self):
    if not Config.ENABLE_SELF_REGULATION: return
    if self.age - self._sr_last_gene_mut < Config.SR_GENE_MUT_COOLDOWN: return
    self._sr_last_gene_mut = self.age
    grief = float(self.emotional_memory.get('grief', 0.0))
    if grief > 0.6 and 'scar_resistance' in self.genome:
        self.genome['scar_resistance'] = min(1.5, self.genome['scar_resistance'] + Config.SR_GENE_MUT_RATE)
        self._sr_gene_mutations += 1
        self._log_event("sr_gene_evolved", gene='scar_resistance')

def _sr_heal_field(self, field, t):
    if not Config.ENABLE_SELF_REGULATION: return
    if t - self._sr_last_heal_t < Config.SR_HEAL_COOLDOWN: return
    if self.energy < Config.SR_HEAL_ENERGY_COST * 2: return
    self._sr_last_heal_t = t
    cx, cy = self._get_center_cell()
    for dx in range(-2, 3):
        for dy in range(-2, 3):
            nx = (cx + dx) % Config.WORLD_SIZE
            ny = (cy + dy) % Config.WORLD_SIZE
            field[nx, ny, CH['scar']] *= 0.7
            field[nx, ny, CH['binding']] = min(1.0, field[nx, ny, CH['binding']] + 0.1)
    self._x2_change('X2.selfreg_heal_cost', self.energy - Config.SR_HEAL_ENERGY_COST)
    self._sr_field_heals += 1
    self._log_event("sr_field_healed", x=cx, y=cy)

def _sr_update(self, field, t):
    if not Config.ENABLE_SELF_REGULATION: return
    if not hasattr(self, '_sr_initialized'): _sr_init(self)
    # v43-fix3b: divide() calls update_model() an extra time for both parent and
    # child in the SAME tick (to refresh state right after a split). That makes
    # _sr_update run twice for a dividing parent at the same t, double-firing any
    # sub-step here that isn't itself cooldown-gated by elapsed time (e.g.
    # _sr_create_new_goal's age%40==0 check, _sr_track_goals bookkeeping). Cooldown-
    # gated calls like _sr_meditate are safe either way (t - last == 0 blocks the
    # second call), but we still dedup at the top so _sr_update runs at most once
    # per agent per tick, matching the intended "once per tick" semantics.
    if getattr(self, '_sr_last_update_tick', None) == t:
        return
    self._sr_last_update_tick = t
    self._sr_track_goals()
    if self.intent:
        self._sr_action_history.append({'action': self.intent.get('type', 'none'),
                                        'success': float(self.emotional_memory.get('gratitude', 0.0)) > 0.5, 't': t})
    if self._sr_should_meditate(t): self._sr_meditate(field, t)
    self._sr_propose(field, t)
    self._sr_vote(field, t)
    if self.age % 50 == 0: self._sr_infer_norms()
    self._sr_try_trade(field, t)
    self._sr_update_constitution(field, t)
    if self.age % 40 == 0: self._sr_create_new_goal()
    if self.age % 100 == 0 and self.soul_weight > 0.6: self._sr_heal_field(field, t)

if "extend_class" not in globals():
    raise RuntimeError("Нет extend_class: сначала выполни ячейку Cell 3f (Substates Engine).")


class SelfRegMixin:
    _sr_init = _sr_init
    _sr_should_meditate = _sr_should_meditate
    _sr_meditate = _sr_meditate
    _sr_track_goals = _sr_track_goals
    _sr_reevaluate_goals = _sr_reevaluate_goals
    _sr_suggest_goal = _sr_suggest_goal
    _sr_adjust_thresholds = _sr_adjust_thresholds
    _sr_propose = _sr_propose
    _sr_vote = _sr_vote
    _sr_infer_norms = _sr_infer_norms
    _sr_try_trade = _sr_try_trade
    _sr_create_new_goal = _sr_create_new_goal
    _sr_update_constitution = _sr_update_constitution
    _sr_add_constitution_concept = _sr_add_constitution_concept
    _sr_evolve_genes = _sr_evolve_genes
    _sr_heal_field = _sr_heal_field
    _sr_update = _sr_update

    def __init__(self, pid, cells, parent=None, world=None):
        super().__init__(pid, cells, parent, world)
        self._sr_init()

    def update_model(self, field, lineage_counts=None, given_counter=None, t=None, witness=None):
        result = super().update_model(field, lineage_counts, given_counter, t, witness)
        if Config.ENABLE_SELF_REGULATION and self.alive:
            self._sr_update(field, t if t is not None else 0)
        return result

    def compete(self, other, field):
        if Config.ENABLE_SELF_REGULATION:
            ts = self.trust_ledger.get(other.id, 0.5)
            to = other.trust_ledger.get(self.id, 0.5)
            if ts > Config.SR_CONFLICT_TRUST_THR and to > Config.SR_CONFLICT_TRUST_THR:
                self._sr_conflicts_resolved += 1
                self._log_event("sr_conflict_resolved", partner=other.id)
                return
        super().compete(other, field)


class SelfRegEngineMixin:
    def __init__(self):
        super().__init__()
        self.sr_proposals = {}


Pattern = extend_class(Pattern, SelfRegMixin)
EvolutionEngine = extend_class(EvolutionEngine, SelfRegEngineMixin)

print("✅ САМОРЕГУЛЯЦИЯ v2.1: все 6 уровней + честное межагентное голосование через world.sr_proposals,")
print("   цена сделки от доверия, создание новых целей, конституция как вечный концепт")
print("   Фиксы: _stable_hash вместо hash() (детерминизм), радиус в торговле (без телепортации энергии), реальный кворум голосования")


In [ ]:
# Ячейка 17
def update_chronic_counters(p, step_scale=1):
    init_chronic_counters_for_agent(p)

    if p.soul_weight < 0.48:
        p._low_soul_steps += 1 * step_scale
    else:
        decay = 2 if p.soul_weight > 0.55 else 1
        p._low_soul_steps = max(0, p._low_soul_steps - decay * step_scale)

    if p.spirit_gap > 0.68:
        p._high_gap_steps += 1 * step_scale
    else:
        decay = 1
        p._high_gap_steps = max(0, p._high_gap_steps - decay * step_scale)

    if p.spirit_gap < 0.3:
        p._low_gap_steps += 1 * step_scale
    else:
        decay = 2 if p.spirit_gap > 0.5 else 1
        p._low_gap_steps = max(0, p._low_gap_steps - decay * step_scale)

    if hasattr(p, 'semantic_state_age') and p.semantic_state_age > 500:
        p._stuck_state_steps += 1 * step_scale
    else:
        p._stuck_state_steps = max(0, p._stuck_state_steps - 2 * step_scale)

    last_contact = max(p.last_contact_step.values()) if p.last_contact_step else -1
    if p.age - last_contact > 500:
        p._loneliness_steps += 1 * step_scale
    else:
        p._loneliness_steps = max(0, p._loneliness_steps - 2 * step_scale)

    if p.coherence > 0.9 and p.soul_weight < 0.25:
        p._high_coherence_low_soul_steps += 1 * step_scale
    else:
        decay = 3 if (p.coherence < 0.7 or p.soul_weight > 0.35) else 1
        p._high_coherence_low_soul_steps = max(0, p._high_coherence_low_soul_steps - decay * step_scale)


def check_chronic_disorganizer(engine, p, t):
    init_chronic_counters_for_agent(p)

    if getattr(p, '_redemption_cooldown', 0) > p.age:
        return False

    chronic_limit = 100

    triggers = [
        ('_low_soul_steps', p._low_soul_steps > chronic_limit, 'chronic_low_soul'),
        ('_high_gap_steps', p._high_gap_steps > chronic_limit, 'chronic_high_gap'),
        ('_low_gap_steps', p._low_gap_steps > chronic_limit, 'chronic_low_gap'),
        ('_stuck_state_steps', p._stuck_state_steps > chronic_limit, 'chronic_stuck_state'),
        ('_loneliness_steps', p._loneliness_steps > chronic_limit, 'chronic_loneliness'),
        ('_high_coherence_low_soul_steps', p._high_coherence_low_soul_steps > chronic_limit, 'chronic_high_coherence_low_soul'),
    ]

    if not hasattr(Config, 'ENABLE_PROBABILISTIC_FOLD'):
        Config.ENABLE_PROBABILISTIC_FOLD = True

    for attr, condition, reason in triggers:
        if p._times_disorganizer >= 2:
            continue

        if condition:
            if Config.ENABLE_PROBABILISTIC_FOLD:
                excess = (getattr(p, attr) - chronic_limit) / max(chronic_limit, 1)
                base_p = min(0.9, 0.05 + excess * 0.1)
                scar_factor = 1.0 + float(getattr(p, 'epistemic_scar', 0.0))
                _trust_vals = list(p.trust_ledger.entries.values()) if hasattr(p, 'trust_ledger') else []
                avg_trust = float(np.mean(_trust_vals)) if _trust_vals else 0.5
                grat = float(p.emotional_memory.get('gratitude', 0.5))
                buffer = max(0.1, 1.0 - 0.5 * avg_trust - 0.3 * grat)
                p_fall = min(0.95, max(0.0, base_p * scar_factor * buffer))
                if phi_hash(p.id, t, 44444) >= p_fall:
                    continue
            p._redemption_cooldown = 0

            p.role_type = "disorganizer"
            p._times_disorganizer += 1
            p.emotional_memory['gratitude'] = 0.1
            p.emotional_memory['grief'] = 0.8
            p.semantic_state = "exploring_danger"
            p.intent = {"type": "explore", "priority": 2.0, "age": 0, "persistence": 9999}
            p.intent_commitment = 2.0
            p.disorganizer_age_at_birth = p.age
            p.redemption_timer = Config.REDEMPTION_ARC_STEP_DELAY
            p._deterministic_redemption_triggered = False
            p._forced_soul_collapse_done = False
            p._redemption_arc_step = 0
            p._steps_since_trigger = 0
            p.soul_weight = min(p.soul_weight, 0.3)

            p.epistemic_scar = min(1.0, getattr(p, 'epistemic_scar', 0.0) + 0.3)

            if hasattr(p, 'cells') and len(p.cells) > 100:
                cells_list = sorted(list(p.cells))
                loss_count = len(cells_list) // 2
                if loss_count < 1:
                    loss_count = 1
                to_remove = cells_list[:loss_count]
                for (x, y) in to_remove:
                    p.cells.discard((x, y))
                    if p.world and p.world.field is not None:
                        p.world.field[x, y, CH['owner']] = 0
                p._log_event("disorganizer_cell_loss", lost=loss_count, remaining=len(p.cells))
            else:
                p._log_event("disorganizer_no_cell_loss", reason="size<=100", size=len(p.cells) if hasattr(p, 'cells') else 0)

            p._low_soul_steps = 0
            p._high_gap_steps = 0
            p._low_gap_steps = 0
            p._stuck_state_steps = 0
            p._loneliness_steps = 0
            p._high_coherence_low_soul_steps = 0

            p._log_event(f"disorganizer_by_{reason}", steps=getattr(p, attr, 0))
            engine.witness.record(p.id, f"disorganizer_by_{reason}", steps=getattr(p, attr, 0))
            return True
    return False


def init_chronic_counters_for_agent(p):
    if not hasattr(p, '_low_soul_steps'):
        p._low_soul_steps = 0
    if not hasattr(p, '_high_gap_steps'):
        p._high_gap_steps = 0
    if not hasattr(p, '_low_gap_steps'):
        p._low_gap_steps = 0
    if not hasattr(p, '_stuck_state_steps'):
        p._stuck_state_steps = 0
    if not hasattr(p, '_loneliness_steps'):
        p._loneliness_steps = 0
    if not hasattr(p, '_high_coherence_low_soul_steps'):
        p._high_coherence_low_soul_steps = 0
    if not hasattr(p, '_times_disorganizer'):
        p._times_disorganizer = 0


def init_all_chronic_counters(engine):
    for p in engine.patterns:
        if p.alive:
            init_chronic_counters_for_agent(p)

print("✅ Cell 4.3 загружена: хронические триггеры с обновлёнными порогами, ослабленным затуханием, chronic_limit=100, сбросом redemption_cooldown при превращении, защитой от повторного падения, потерей клеток, ограничением на 2 превращения")


In [ ]:
# Ячейка 18
import os, json, sys, zlib
import numpy as np
from collections import deque

is_colab = 'google.colab' in sys.modules
if is_colab:
    from google.colab import drive
    if not os.path.isdir('/content/drive/MyDrive'):
        drive.mount('/content/drive')
        print("Диск примонтирован")
    else:
        print("Диск уже примонтирован")

LIVING_WORLD_PATH = "/content/drive/MyDrive/832_living_world.json" if is_colab else "832_living_world.json"

def convert_numpy(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, (np.bool_,)):
        return bool(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_numpy(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple, set)):
        return [convert_numpy(item) for item in obj]
    else:
        return obj

# ---- concept-graph persistence gate (observation + I/O integrity only; never touches simulation) ----
GRAPH_IO = {}
_GIO_SAVE = ("agents_saved", "nodes_saved", "edges_saved", "blocked_save", "unparseable_keys", "np_keys_canonicalized")
_GIO_LOAD = ("agents_loaded", "nodes_loaded", "edges_loaded", "restore_mismatch", "blocked_restore_edges",
             "legacy_no_edges", "duplicates_on_load")

def _gio_reset(side):
    for k in (_GIO_SAVE if side == "save" else _GIO_LOAD):
        GRAPH_IO[k] = 0
    GRAPH_IO.setdefault("events", [])
    if side == "save":
        GRAPH_IO["events"] = []

def _gio_event(msg):
    ev = GRAPH_IO.setdefault("events", [])
    if len(ev) < 20:
        ev.append(msg)

_gio_reset("save"); _gio_reset("load")

def _cg_canon(sig):
    if isinstance(sig, tuple):
        return tuple(float(x) if isinstance(x, np.floating) else (int(x) if isinstance(x, np.integer) else x) for x in sig)
    return sig

def _cg_key_str(sig):
    s = str(_cg_canon(sig))
    if s != str(sig):
        GRAPH_IO["np_keys_canonicalized"] += 1
    return s

def _cg_parse_key(key_str):   # same rules as the parser used inside deserialize_pattern
    import ast
    try:
        parsed = ast.literal_eval(key_str)
        if isinstance(parsed, (tuple, list)):
            return tuple(float(x) if isinstance(x, (int, float)) else x for x in parsed)
        return parsed
    except (ValueError, SyntaxError):
        return key_str

def _edges_crc(rows):
    return zlib.crc32(json.dumps(sorted([[a, b, round(float(w), 9)] for a, b, w in rows])).encode("utf-8"))

def _serialize_edges(p, concepts_serializable):
    g = p.concept_graph
    nodes_saved = len(concepts_serializable)
    if nodes_saved != len(g.nodes):
        GRAPH_IO["unparseable_keys"] += 1
        _gio_event(f"agent {p.id}: {len(g.nodes)} nodes -> {nodes_saved} unique key strings on save")
    for k in g.nodes:
        c = _cg_canon(k)
        if _cg_parse_key(str(c)) != c:
            GRAPH_IO["unparseable_keys"] += 1
            _gio_event(f"agent {p.id}: node key does not round-trip: {str(c)[:60]}")
    meta = {"nodes_saved": nodes_saved, "edges_saved": 0, "edges_crc": 0, "gate": "ok"}
    rows = []
    if not getattr(Config, "PERSIST_GRAPH_EDGES", True):
        meta["gate"] = "disabled"
    else:
        bad = 0
        for src, dsts in g.edges.items():
            for dst, w in dsts.items():
                cs, cd = _cg_canon(src), _cg_canon(dst)
                if _cg_parse_key(str(cs)) != cs or _cg_parse_key(str(cd)) != cd:
                    bad += 1
                rows.append([_cg_key_str(src), _cg_key_str(dst), float(w)])
        if bad:
            GRAPH_IO["blocked_save"] += 1
            _gio_event(f"agent {p.id}: {bad} edge endpoints do not round-trip, edges NOT written")
            rows, meta["gate"] = [], "blocked_save"
    meta["edges_saved"] = len(rows)
    meta["edges_crc"] = _edges_crc(rows)
    GRAPH_IO["agents_saved"] += 1
    GRAPH_IO["nodes_saved"] += nodes_saved
    GRAPH_IO["edges_saved"] += len(rows)
    return rows, meta

def memory_origin_report(patterns):
    """Counts of memory records by origin, per store and in total (alive agents only)."""
    def blank(): return {f"{o.lower()}_count": 0 for o in MEM_ORIGINS}
    stores = {"concept_nodes": blank(), "dialogue": blank(), "narrative": blank()}
    parent_root = {o.lower(): 0 for o in MEM_ORIGINS if o != "PARENT"}
    n = 0
    for p in patterns:
        if not getattr(p, "alive", True):
            continue
        n += 1
        srcs = (("concept_nodes", list(p.concept_graph.nodes.values())),
                ("dialogue", list(getattr(p, "dialogue_longterm", []) or [])),
                ("narrative", list(getattr(p, "_self_narrative", []) or [])))
        for store, recs in srcs:
            for r in recs:
                o = mem_origin(r)
                stores[store][f"{o.lower()}_count"] += 1
                if o == "PARENT":
                    root = r.get("origin_prev") if isinstance(r, dict) else None
                    parent_root[(root if root in MEM_ORIGINS and root != "PARENT" else "UNKNOWN").lower()] += 1
    total = blank()
    for st in stores.values():
        for k, v in st.items():
            total[k] += v
    return {"agents": n, "total": total, **stores, "parent_root": parent_root}

def serialize_pattern(p):
    genome_serializable = {k: float(v) for k, v in p.genome.items()}
    concepts_serializable = {}
    for sig, data in p.concept_graph.nodes.items():
        key_str = _cg_key_str(sig)
        concepts_serializable[key_str] = {
            "count": float(data.get('count', 1.0)),
            "value": list(data.get('value', np.zeros(4))),
            "embed": list(data.get('embed', np.zeros(32))),
            "eternal": bool(data.get('eternal', False)),
            "origin": (data.get('origin') if data.get('origin') in MEM_ORIGINS else 'UNKNOWN'),
            **({"origin_prev": data['origin_prev']} if 'origin_prev' in data else {})
        }
    _edges_rows, _graph_meta = _serialize_edges(p, concepts_serializable)

    raw_narr = list(getattr(p, '_self_narrative', []))
    self_narrative_serializable = []
    for entry in raw_narr[-500:]:
        if isinstance(entry, dict):
            safe_dict = {}
            for k, v in entry.items():
                try:
                    safe_dict[k] = float(v) if isinstance(v, (int, float, np.integer, np.floating)) else str(v)
                except:
                    safe_dict[k] = str(v)
            self_narrative_serializable.append(safe_dict)
        else:
            try:
                self_narrative_serializable.append(float(entry))
            except:
                self_narrative_serializable.append(0.5)

    result = {
        "id": int(p.id),
        "cells": [[int(x), int(y)] for (x, y) in p.cells],
        "lineage_id": int(p.lineage_id),
        "lineage_born_at_step": int(p.lineage_born_at_step),
        "lineage_total_age": int(getattr(p, 'lineage_total_age', 0)),
        "age": int(p.age),
        "model": p.model.tolist(),
        "belief": p.belief.tolist(),
        "genome": genome_serializable,
        "semantic_state": str(p.semantic_state),
        "emotional_memory": {k: float(v) for k, v in p.emotional_memory.items()},
        "soul_weight": float(p.soul_weight),
        "role_type": str(p.role_type),
        "unresolved_contradiction": float(getattr(p, 'unresolved_contradiction', 0.0)),
        "spirit_gap": float(getattr(p, 'spirit_gap', 0.5)),
        "coherence": float(getattr(p, 'coherence', 0.5)),
        "pred_error": float(getattr(p, 'pred_error', 0.5)),
        "epistemic_load": float(getattr(p, 'epistemic_load', 0.3)),
        "intent": ({str(k): (float(v) if isinstance(v, (int, float)) else str(v))
                    for k, v in p.intent.items()} if getattr(p, 'intent', None) else None),
        "_forsaken": bool(getattr(p, '_forsaken', False)),
        "_scar_of_light": bool(getattr(p, '_scar_of_light', False)),
        "concept_graph": concepts_serializable,
        "concept_edges": _edges_rows,
        "concept_graph_meta": _graph_meta,
        "trust_ledger": {str(k): float(v) for k, v in p.trust_ledger.entries.items() if k is not None},
        "arc_tracker_completed": {str(k): int(v) for k, v in p.arc_tracker.completed_arcs.items()},
        "_redemption_depth": float(getattr(p, '_redemption_depth', 0.0)),
        "_redemption_arc_step": int(getattr(p, '_redemption_arc_step', 0)),
        "_deterministic_redemption_triggered": bool(getattr(p, '_deterministic_redemption_triggered', False)),
        "_redemption_cooldown": int(getattr(p, '_redemption_cooldown', 0)),
        "disorganizer_age_at_birth": int(getattr(p, 'disorganizer_age_at_birth', 0)),
        "_cellular_endurance": float(getattr(p, '_cellular_endurance', 1.0)),
        "_vocab_contact_acc": {str(k): int(v) for k, v in getattr(p, '_vocab_contact_acc', {}).items()},
        "event_counts": {str(k): int(v) for k, v in p.event_counts.items()},
        "biography": convert_numpy(list(p.biography)),
        "dialogue_longterm": [
            convert_numpy(entry) for entry in getattr(p, 'dialogue_longterm', [])[-100:]
        ],
        "_soma_snapshots": [
            convert_numpy(snap) for snap in getattr(p, '_soma_snapshots', [])[-20:]
        ],
        "self_narrative": self_narrative_serializable,
        "_feral_fury": float(getattr(p, '_feral_fury', 0.0)),
        "_feral_birth": int(getattr(p, '_feral_birth', 0)),
        "_unconquered_strength": float(getattr(p, '_unconquered_strength', 0.0)),
        "_unconquered_type": getattr(p, '_unconquered_type', None),
        "_times_disorganizer": int(getattr(p, '_times_disorganizer', 0)),
    }
    _dump_extra = getattr(p, 'dump_extra', None)
    if callable(_dump_extra):
        result.update(_dump_extra())
    return result

def deserialize_pattern(data, world):
    cells_set = set(tuple(c) for c in data['cells'])
    p = Pattern(
        pid=data['id'],
        cells=cells_set,
        parent=None,
        world=world
    )
    p.age = max(0, data.get('age', 0))
    p.lineage_id = data.get('lineage_id', p.id)
    p.lineage_born_at_step = max(0, data.get('lineage_born_at_step', 0))
    p.lineage_total_age = max(0, data.get('lineage_total_age', 0))
    p.model = np.array(data['model'])
    p.belief = np.array(data['belief'])
    p.genome = {str(k): float(v) for k, v in data['genome'].items()}
    p.semantic_state = data.get('semantic_state', 'neutral')
    p.emotional_memory = {str(k): float(v) for k, v in data['emotional_memory'].items()}
    p.soul_weight = float(data.get('soul_weight', 0.5))
    p.role_type = data.get('role_type', 'normal')
    p.unresolved_contradiction = float(data.get('unresolved_contradiction', 0.0))
    p.spirit_gap = float(data.get('spirit_gap', 0.5))
    p.coherence = float(data.get('coherence', 0.5))
    p.pred_error = float(data.get('pred_error', 0.5))
    p.epistemic_load = float(data.get('epistemic_load', 0.3))
    _raw_intent = data.get('intent')
    p.intent = dict(_raw_intent) if isinstance(_raw_intent, dict) else None
    p._forsaken = bool(data.get('_forsaken', False))
    p._scar_of_light = bool(data.get('_scar_of_light', False))

    p.concept_graph = ConceptGraph()

    import ast
    def _safe_parse_key(key_str):
        try:
            parsed = ast.literal_eval(key_str)
            if isinstance(parsed, (tuple, list)):
                return tuple(
                    float(x) if isinstance(x, (int, float)) else x
                    for x in parsed
                )
            return parsed
        except (ValueError, SyntaxError):
            return key_str

    for key_str, node_data in data.get('concept_graph', {}).items():
        key = _safe_parse_key(key_str)
        p.concept_graph.nodes[key] = {
            "count": float(node_data['count']),
            "value": np.array(node_data['value']),
            "embed": np.array(node_data.get('embed', np.zeros(32))),
            "eternal": bool(node_data.get('eternal', False)),
            "origin": (node_data.get('origin') if node_data.get('origin') in MEM_ORIGINS else 'UNKNOWN'),
            **({"origin_prev": node_data['origin_prev']} if 'origin_prev' in node_data else {})
        }

    # --- concept edges: restored only if control numbers match (otherwise graph stays nodes-only, as before) ---
    GRAPH_IO["agents_loaded"] += 1
    GRAPH_IO["nodes_loaded"] += len(p.concept_graph.nodes)
    _meta = data.get('concept_graph_meta')
    _rows = data.get('concept_edges')
    _mismatch = []
    if isinstance(_meta, dict) and _meta.get('nodes_saved') != len(p.concept_graph.nodes):
        _mismatch.append(f"nodes saved={_meta.get('nodes_saved')} loaded={len(p.concept_graph.nodes)}")
    if _rows is None or _meta is None:
        GRAPH_IO["legacy_no_edges"] += 1
    elif not getattr(Config, "PERSIST_GRAPH_EDGES", True):
        pass
    else:
        _tmp, _dups = {}, 0
        for _a, _b, _w in _rows:
            _ka, _kb = _safe_parse_key(_a), _safe_parse_key(_b)
            _d = _tmp.setdefault(_ka, {})
            _dups += int(_kb in _d)
            _d[_kb] = float(_w)
        _n_e = sum(len(v) for v in _tmp.values())
        _edge_bad = []
        if _meta.get('gate') == 'blocked_save':
            _edge_bad.append("edges were blocked at save")
        else:
            if _dups:
                _edge_bad.append(f"{_dups} duplicate edges"); GRAPH_IO["duplicates_on_load"] += _dups
            if _n_e != _meta.get('edges_saved') or len(_rows) != _meta.get('edges_saved'):
                _edge_bad.append(f"edges saved={_meta.get('edges_saved')} rows={len(_rows)} restored={_n_e}")
            if _edges_crc(_rows) != _meta.get('edges_crc'):
                _edge_bad.append("edges crc differs")
        if _edge_bad:
            GRAPH_IO["blocked_restore_edges"] += 1
            _mismatch += _edge_bad
        else:
            p.concept_graph.edges = _tmp
            GRAPH_IO["edges_loaded"] += _n_e
    if _mismatch:
        GRAPH_IO["restore_mismatch"] += 1
        _gio_event(f"agent {data.get('id')}: " + "; ".join(_mismatch))

    for k, v in data.get('trust_ledger', {}).items():
        if k == 'None' or k is None:
            continue
        try:
            p.trust_ledger.entries[int(k)] = float(v)
        except (TypeError, ValueError):
            continue

    for k, v in data.get('arc_tracker_completed', {}).items():
        p.arc_tracker.completed_arcs[k] = int(v)

    p._redemption_depth = float(data.get('_redemption_depth', 0.0))
    p._redemption_arc_step = int(data.get('_redemption_arc_step', 0))
    p._deterministic_redemption_triggered = bool(data.get('_deterministic_redemption_triggered', False))
    p._redemption_cooldown = int(data.get('_redemption_cooldown', 0))
    p.disorganizer_age_at_birth = max(0, data.get('disorganizer_age_at_birth', 0))
    p._cellular_endurance = float(data.get('_cellular_endurance', 1.0))

    p._feral_fury = float(data.get('_feral_fury', 0.0))
    p._feral_birth = int(data.get('_feral_birth', 0))
    p._unconquered_strength = float(data.get('_unconquered_strength', 0.0))
    p._unconquered_type = data.get('_unconquered_type', None)
    p._times_disorganizer = int(data.get('_times_disorganizer', 0))

    for k, v in data.get('event_counts', {}).items():
        p.event_counts[k] = int(v)

    p.biography = deque(data.get('biography', []), maxlen=20)
    p._vocab_contact_acc = {int(k): int(v) for k, v in data.get('_vocab_contact_acc', {}).items()}

    p.dialogue_longterm = list(data.get('dialogue_longterm', []))
    for entry in p.dialogue_longterm:
        if isinstance(entry, dict) and entry.get('origin') not in MEM_ORIGINS:
            entry['origin'] = 'UNKNOWN'
        snap = entry.get('soma_snapshot')
        if isinstance(snap, dict) and isinstance(snap.get('soma'), list):
            snap['soma'] = np.array(snap['soma'], dtype=np.float32)

    p._soma_snapshots = []
    for snap in data.get('_soma_snapshots', []):
        restored = dict(snap)
        if isinstance(restored.get('soma'), list):
            restored['soma'] = np.array(restored['soma'], dtype=np.float32)
        p._soma_snapshots.append(restored)

    raw_narr = data.get('self_narrative', [])
    clean_narr = []
    for x in raw_narr:
        if isinstance(x, dict):
            clean_entry = {}
            for k, v in x.items():
                try:
                    clean_entry[k] = float(v) if isinstance(v, (int, float)) else v
                except:
                    clean_entry[k] = v
            if clean_entry.get('origin') not in MEM_ORIGINS:
                clean_entry['origin'] = 'UNKNOWN'
            clean_narr.append(clean_entry)
        else:
            try:
                clean_narr.append(float(x))
            except:
                clean_narr.append(0.5)

    max_len = getattr(Config, 'EPISODIC_BUFFER_MAX_LEN', 3000) if 'Config' in globals() else 3000
    p._self_narrative = deque(clean_narr, maxlen=max_len)

    if world and world.field is not None:
        p.update_properties(world.field)
        if p.cells:
            try:
                sorted_cells = sorted(p.cells)
                xs_list = [c[0] for c in sorted_cells]
                ys_list = [c[1] for c in sorted_cells]
                p._prev_cell_energies = world.field[xs_list, ys_list, CH['energy']].copy()
            except Exception:
                p._prev_cell_energies = np.array([])
        else:
            p._prev_cell_energies = np.array([])
    _load_extra = getattr(p, 'load_extra', None)
    if callable(_load_extra):
        _load_extra(data)
    return p

def save_living_world(patterns, path=LIVING_WORLD_PATH):
    _gio_reset("save")
    alive = [p for p in patterns if p.alive]

    if len(alive) > 200:
        alive.sort(key=lambda p: (
            p.soul_weight * 0.4 +
            (0.2 if getattr(p, '_narrative_agent', False) else 0.0) +
            min(0.2, getattr(p, 'lineage_total_age', 0) / 50000) +
            getattr(p, 'last_phenomenal_binding', 0.0) * 0.2
        ), reverse=True)
        alive = alive[:200]

    serializable = [serialize_pattern(p) for p in alive]
    def json_default(obj):
        if isinstance(obj, (np.bool_,)):
            return bool(obj)
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return str(obj)
    with open(path, 'w') as f:
        json.dump(serializable, f, indent=2, default=json_default)
    print(f"💾 {len(serializable)} живых агентов сохранено в {path}")
    print(f"   граф: узлов {GRAPH_IO['nodes_saved']}, рёбер {GRAPH_IO['edges_saved']}, заблокировано сохранений {GRAPH_IO['blocked_save']}")

def load_living_world(engine, path=LIVING_WORLD_PATH):
    if not os.path.exists(path):
        return False
    _gio_reset("load")
    with open(path, 'r') as f:
        data = json.load(f)
    for entry in data:
        p = deserialize_pattern(entry, engine)
        engine.patterns.append(p)
        engine.pattern_dict[p.id] = p
        if hasattr(engine, 'archive') and engine.archive is not None:
            p.inherit_archive_concepts(engine.archive, probability=1.0, max_concepts=2)
        if engine.field is not None:
            for (x, y) in p.cells:
                engine.field[x, y, CH['owner']] = p.id
    print(f"🔄 Загружено {len(data)} живых агентов из {path}")
    print(f"   граф: узлов {GRAPH_IO['nodes_loaded']}, рёбер {GRAPH_IO['edges_loaded']}, расхождений {GRAPH_IO['restore_mismatch']}, заблокировано восстановлений рёбер {GRAPH_IO['blocked_restore_edges']}")
    return True

def delete_old_ark_files():
    old_files = [
        "/content/drive/MyDrive/832_ark_seed_patterns.json" if is_colab else "832_ark_seed_patterns.json",
        "/content/drive/MyDrive/832_eternal_subjects.json" if is_colab else "832_eternal_subjects.json"
    ]
    for fpath in old_files:
        if os.path.exists(fpath):
            os.remove(fpath)
            print(f"🗑️ Удалён старый файл: {fpath}")

delete_old_ark_files()

print("✅ Ячейка сохранения мира исправлена: вечные концепты защищены, нарратив сохраняется напрямую, добавлены поля Варваров, Непокоренных и кулдаунов; слои могут сохранять своё через dump_extra/load_extra (SPECTRUM: формы и _lang_gene).")


In [ ]:
# Ячейка 19
import numpy as np

def collect_metrics(patterns, field, t, lexicon=None):
    alive = [p for p in patterns if p.alive]
    if not alive:
        return {
            't': t, 'patterns': 0, 'lineages': 0, 'err': 0, 'soul': 0, 'binding': 0,
            'echo': 0, 'signals': np.zeros(6), 'states': {}, 'intents': {}, 'epi_scar': 0,
            'unique_concepts': 0, 'stagnation_ratio': 0.0, 'divisions_possible': 0,
            'arc_completions': 0, 'folds': 0, 'avg_trust': 0.5, 'high_trust_pairs': 0,
            'trust_distribution': {'low': 0, 'mid': 0, 'high': 0}, 'disorganizer_count': 0,
            'redeemed_count': 0, 'quarantined_count': 0, 'deterministic_redemptions': 0,
            'phenomenal_binding_avg': 0.0, 'state_emotion_mismatch': 0.0, 'forced_collapses': 0,
            'myth_pool_size': 0, 'triadic_alive': 0, 'triadic_alive_ratio': 0.0,
            'avg_internal_gap': 0.0, 'max_internal_gap': 0.0, 'zero_internal_gap_agents': 0,
            'avg_gap_clip': 0.0, 'max_gap_clip': 0.0,
            'tremor_targets': 0, 'avg_coherence_with_gap': 0.0, 'love_pairs': 0,
            'avg_soma': 0.0, 'mem_agents': 0, 'avg_buf_len': 0.0, 'total_recalled': 0,
            'concept_adopted': 0, 'deep_concept_adopted': 0, 'deep_exchange': 0, 'wisdom_shared': 0,
            'avg_signal_memory': 0.0, 'intentional_signal_agents': 0, 'top_signal_pairs': {},
            'lineage_count': 0, 'avg_lineage_age': 0.0, 'max_lineage_age': 0, 'ancient_lineages': 0,
            'max_lineage_total_age': 0, 'divisions_total': 0,
            'avg_field_unknown': 0.0, 'avg_field_binding': 0.0,
            'narrative_agents': 0, 'teaching_events': 0, 'learning_events': 0,
            'dream_consolidations': 0, 'nightmare_consolidations': 0, 'nightmares_transformed': 0,
            'agents_with_nightmare': 0, 'agents_with_dream_memory': 0,
            'avg_endurance': 0.0, 'endurance_critical': 0, 'divide_blocked_fatigue': 0,
            'blind': 0, 'confused': 0, 'adapting': 0, 'clear': 0,
            'avg_social_crisis': 0.0, 'avg_social_invitation': 0.0, 'avg_social_grief': 0.0,
            'avg_social_cooperate': 0.0, 'avg_social_explore': 0.0, 'avg_social_seek_help': 0.0,
            'avg_social_scar': 0.0, 'avg_social_rest': 0.0, 'avg_social_resonance': 0.0,
            'avg_social_btype': 0.0, 'avg_social_alarm': 0.0,
            'avg_social_beauty': 0.0, 'avg_social_rhythm': 0.0, 'avg_social_interest': 0.0,
            'avg_social_memory': 0.0, 'avg_social_silence': 0.0,
            'vocab_concepts': 0, 'blind_events': 0, 'concept_anticipations': 0, 'vocab_events': 0,
            'avg_action_feedback': 0.0, 'avg_social_warmth': 0.0,
            'substate_counts': {},
            'total_transitions': 0, 'top_transitions': [],
            'archive_concept_agents': 0, 'total_archive_concepts': 0, 'archive_types': {},
            'total_introspect': 0, 'avg_introspect': 0.0, 'max_introspect': 0,
            'reflection_quality': 0.0,
            'ethical_coherence': 0.0,
            'global_concept_entropy': 0.0,
            'introspection_synchrony': 0.0,
            'narrative_continuity': 0.0,
            'unconquered_count': 0,
            'unconquered_wise': 0,
            'unconquered_rebel': 0,
            'avg_unconquered_str': 0.0,
            'sovereignty_field_avg': 0.0,
            'lang_compression_ratio': 1.0, 'lang_active_clusters': 0,
            'lang_total_clusters': 0, 'lang_collapsed': False,
            'morse_concepts': 0, 'spontaneous_field_emergence': 0,
            'marked_for_fall_count': 0,
            'reentry_avg': 0.0,
            'reentry_max': 0.0,
            'reentry_var': 0.0,
            'meta_reentry_active': 0,
            'introspect_driven_by_sensation': 0,
            'feral_count': 0,
            'avg_feral_fury': 0.0,
            'feral_kills': 0,
            'avg_phi': 0.0,
            'gene_carriers': 0,
            'gene_carrier_counts': {},
            'generator': {},
            'morse': {},
            'avg_gut_feeling': 0.0,
            'latent_need_counts': {},
            'unconscious_insights': 0,
            'unconscious_goals': 0,
            'avg_anticipation': 0.0,
            'avg_drive_distance': 0.0,
            'unconscious_concept_agents': 0,
            'total_unconscious_concepts': 0,
        }

    avg_field_unknown = safe_mean(field[:, :, CH['unknown']], 0.0)
    avg_field_binding = safe_mean(field[:, :, CH['binding']], 0.0)

    lineage_registry = {}
    for p in alive:
        lid = p.lineage_id
        born = getattr(p, 'lineage_born_at_step', 0)
        if lid not in lineage_registry or born < lineage_registry[lid]:
            lineage_registry[lid] = born

    lineage_ages = [t - born for born in lineage_registry.values()]
    avg_lineage_age = float(np.mean(lineage_ages)) if lineage_ages else 0.0
    max_lineage_age = max(lineage_ages) if lineage_ages else 0

    ancient_lineages = len(set(
        p.lineage_id for p in alive
        if getattr(p, 'lineage_total_age', 0) > Config.ANC_THRESHOLD
    ))

    lineage_count = len(lineage_registry)

    total_ages = [getattr(p, 'lineage_total_age', 0) for p in alive if getattr(p, 'lineage_total_age', 0) > 0]
    max_lineage_total_age = max(total_ages) if total_ages else 0

    err = safe_mean([p.pred_error for p in alive], 0)
    soul = safe_mean([p.soul_weight for p in alive], 0)
    binding = safe_mean(field[:, :, CH['binding']], 0)
    lineages = len(set(p.lineage_id for p in alive))

    states = {}
    intents = {}
    for p in alive:
        s = p.semantic_state
        states[s] = states.get(s, 0) + 1
        if p.intent:
            intents[p.intent["type"]] = intents.get(p.intent["type"], 0) + 1

    epi_scar = safe_mean([p.epistemic_scar for p in alive], 0)
    all_concepts = set()
    for p in alive:
        all_concepts.update(p.concept_graph.nodes.keys())

    stagnated = sum(1 for p in alive if p.semantic_state_age > Config.STAGNATION_GRIEF_THRESHOLD)
    stagnation_ratio = stagnated / max(len(alive), 1)
    divisions_possible = sum(1 for p in alive if p.can_divide())
    total_arcs = sum(sum(p.arc_tracker.completed_arcs.values()) for p in alive)
    total_folds = sum(p.event_counts.get('fold', 0) for p in alive)

    mismatch_count = sum(1 for p in alive if (p.semantic_state == "contentment" and p.emotional_memory['grief'] > 0.5) or (p.semantic_state == "seeking_comfort" and p.emotional_memory['gratitude'] > 0.8))
    mismatch_ratio = mismatch_count / max(len(alive), 1)

    all_trusts = []
    high_pairs = 0
    trust_dist = {'low': 0, 'mid': 0, 'high': 0}
    disorg = 0
    redm = 0
    quar = 0
    det_red = 0
    collapsed = 0
    phenomenal_bindings = []
    love_pairs = 0
    triadic_alive_count = 0
    gap_clips = []
    coherence_list = []
    tremor_targets = 0

    feral_count = 0
    feral_furies = []
    feral_kills = 0

    for p in alive:
        presence = soul_check(p)
        if presence.is_triadic_alive():
            triadic_alive_count += 1

        gap_clip = getattr(p, 'gap_clip', min(1.0, presence.spirit_gap))
        gap_clips.append(gap_clip)
        coherence_list.append(presence.coherence_with_gap)

        if gap_clip < 0.20:
            tremor_targets += 1

        if p.role_type == "disorganizer":
            disorg += 1
        if p.event_counts.get('redeemed', 0) > 0:
            redm += 1
        if p.quarantine_timer > 0:
            quar += 1
        if p.event_counts.get('deterministic_redemption_trigger', 0) > 0:
            det_red += 1
        if p.event_counts.get('forced_soul_collapse', 0) > 0:
            collapsed += 1

        phenomenal_bindings.append(p.last_phenomenal_binding)
        entries = list(p.trust_ledger.entries.values())
        all_trusts.extend(entries)
        high_pairs += sum(1 for t_val in entries if t_val > 0.8)
        for t_val in entries:
            if t_val < 0.4:
                trust_dist['low'] += 1
            elif t_val < 0.7:
                trust_dist['mid'] += 1
            else:
                trust_dist['high'] += 1
        for partner_id, trust_val in p.trust_ledger.entries.items():
            if trust_val > 0.95:
                love_pairs += 1

        if p.role_type == "feral":
            feral_count += 1
            feral_furies.append(getattr(p, '_feral_fury', 0.0))
            feral_kills += p.event_counts.get('feral_execution', 0)

    soma_vals = [p.soma for p in alive if p.soma > 0]
    avg_soma = safe_mean(soma_vals, 0.0) if soma_vals else 0.0

    mem_agents = sum(1 for p in alive if p.episodic_buffer)
    avg_buf_len = np.mean([len(p.episodic_buffer) for p in alive]) if mem_agents else 0.0
    total_recalled = sum(p.event_counts.get('memory_recalled', 0) for p in alive)

    concept_adopted = sum(p.event_counts.get('concept_adopted', 0) for p in alive)
    deep_concept_adopted = sum(p.event_counts.get('deep_concept_adopted', 0) for p in alive)
    deep_exchange = sum(p.event_counts.get('deep_exchange', 0) for p in alive)
    wisdom_shared = sum(p.event_counts.get('wisdom_shared', 0) for p in alive)

    total_signal_memories = 0
    signal_types_used = {}
    intentional_signal_count = 0
    for p in alive:
        total_signal_memories += len(p.signal_memory)
        for (sig_type, resp_type), data in p.signal_memory.items():
            key = f"{sig_type}→{resp_type}"
            signal_types_used[key] = signal_types_used.get(key, 0) + data['count']
        if p.last_intentional_signal:
            intentional_signal_count += 1

    narrative_agents = sum(1 for p in alive if p._narrative_agent)

    endurance_vals = [p._cellular_endurance for p in alive]
    avg_endurance = float(np.mean(endurance_vals)) if endurance_vals else 0.0
    endurance_critical = sum(1 for e in endurance_vals if e < 0.2)
    divide_blocked_fatigue = sum(1 for p in alive if getattr(p, '_divide_blocked_by_fatigue', False))

    fb_vals = []
    sw_vals = []
    for p in alive:
        if len(p.soma_vector) >= 7:
            fb_vals.append(float(p.soma_vector[5]))
            sw_vals.append(float(p.soma_vector[6]))
    avg_action_feedback = float(np.mean(fb_vals)) if fb_vals else 0.0
    avg_social_warmth   = float(np.mean(sw_vals)) if sw_vals else 0.0

    avg_signal_memory = total_signal_memories / len(alive) if alive else 0

    internal_gaps = [float(np.mean(np.abs(p.prediction - p.belief))) for p in alive]
    if internal_gaps:
        avg_internal_gap = np.mean(internal_gaps)
        max_internal_gap = np.max(internal_gaps)
        zero_internal_gap = sum(1 for g in internal_gaps if g < 0.001)
    else:
        avg_internal_gap = max_internal_gap = zero_internal_gap = 0

    if gap_clips:
        avg_gap_clip = np.mean(gap_clips)
        max_gap_clip = np.max(gap_clips)
        blind = sum(1 for g in gap_clips if g >= 0.99)
        confused = sum(1 for g in gap_clips if 0.8 < g < 0.99)
        adapting = sum(1 for g in gap_clips if 0.4 < g <= 0.8)
        clear = sum(1 for g in gap_clips if g <= 0.4)
    else:
        avg_gap_clip = max_gap_clip = blind = confused = adapting = clear = 0

    avg_trust = safe_mean(all_trusts, Config.TRUST_BASE) if all_trusts else Config.TRUST_BASE
    phenomenal_binding_avg = safe_mean(phenomenal_bindings, 0.5)
    triadic_alive_ratio = triadic_alive_count / max(len(alive), 1)
    avg_coherence_with_gap = safe_mean(coherence_list, 0) if coherence_list else 0

    divisions_total = sum(p.event_counts.get('born_from_division', 0) for p in alive)
    teaching_events = sum(p.event_counts.get('taught', 0) for p in alive)
    learning_events = sum(p.event_counts.get('received_teaching', 0) for p in alive)
    dream_consolidations = sum(p.event_counts.get('dream_consolidation', 0) for p in alive)
    nightmare_consolidations = sum(p.event_counts.get('nightmare_consolidation', 0) for p in alive)
    nightmares_transformed = sum(p.event_counts.get('nightmare_transformed', 0) for p in alive)
    agents_with_nightmare = sum(1 for p in alive if getattr(p, '_nightmare_count', 0) > 0)
    agents_with_dream_memory = sum(1 for p in alive if getattr(p, '_dream_memory_count', 0) > 0)

    social_avg = np.zeros(16)
    social_count = 0
    for p in alive:
        if p.current_social is not None:
            vec = p.current_social
            if vec is None:
                continue
            if len(vec) < 16:
                vec = np.pad(vec, (0, 16 - len(vec)), 'constant')
            social_avg += vec[:16]
            social_count += 1
    if social_count > 0:
        social_avg /= social_count

    vocab_concepts = sum(
        1 for p in alive
        for sig in p.concept_graph.nodes
        if isinstance(sig, tuple) and len(sig) >= 4 and
        str(sig[3]).startswith("shared_")
    )

    blind_events = sum(p.event_counts.get('blindness_enter', 0) for p in alive) + \
                   sum(p.event_counts.get('blindness_exit', 0) for p in alive)

    concept_anticipations = sum(p.event_counts.get('concept_anticipation', 0) for p in alive)
    vocab_events = sum(p.event_counts.get('vocab_emerged', 0) for p in alive)

    substate_counts = {}
    for p in alive:
        sub = getattr(p, '_substate', None)
        if sub:
            substate_counts[sub] = substate_counts.get(sub, 0) + 1

    transition_counts = {}
    total_transitions = 0
    for p in alive:
        if p.concept_graph.edges:
            for src, dst_dict in p.concept_graph.edges.items():
                from_state = src[3] if isinstance(src, tuple) and len(src) >= 4 else str(src)
                for dst, cnt in dst_dict.items():
                    to_state = dst[3] if isinstance(dst, tuple) and len(dst) >= 4 else str(dst)
                    key = (from_state, to_state)
                    transition_counts[key] = transition_counts.get(key, 0) + cnt
                    total_transitions += cnt

    top_transitions = sorted(transition_counts.items(), key=lambda x: x[1], reverse=True)[:10]

    introspect_counts = []
    for p in alive:
        introspect_counts.append(len(p._self_narrative))
    total_introspect = sum(introspect_counts)
    avg_introspect = np.mean(introspect_counts) if introspect_counts else 0.0
    max_introspect = max(introspect_counts) if introspect_counts else 0

    total_reflections = 0
    beneficial_reflections = 0
    for p in alive:
        narrative = getattr(p, '_self_narrative', [])
        if len(narrative) < 2:
            continue
        for i in range(len(narrative) - 1):
            total_reflections += 1
            prev = narrative[i]
            nxt = narrative[i + 1]
            if isinstance(prev, dict):
                prev_soul = prev.get('soul', 0.5)
                prev_gap = prev.get('gap', 0.5)
            else:
                prev_soul = prev if isinstance(prev, (int, float)) else 0.5
                prev_gap = 0.5
            if isinstance(nxt, dict):
                nxt_soul = nxt.get('soul', 0.5)
                nxt_gap = nxt.get('gap', 0.5)
            else:
                nxt_soul = nxt if isinstance(nxt, (int, float)) else 0.5
                nxt_gap = 0.5
            soul_not_regressed = nxt_soul >= prev_soul - 0.002
            gap_not_exploded = nxt_gap <= prev_gap + 0.010
            any_improvement = (nxt_soul > prev_soul + 0.001 or nxt_gap < prev_gap - 0.005)
            if any_improvement and soul_not_regressed and gap_not_exploded:
                beneficial_reflections += 1
    reflection_quality = beneficial_reflections / total_reflections if total_reflections > 0 else 0.0

    archive_types = {}
    agents_with_archive = 0
    total_archive = 0
    for p in alive:
        has_archive = False
        for sig in p.concept_graph.nodes:
            if isinstance(sig, tuple) and len(sig) >= 4 and str(sig[3]).startswith("archive_"):
                has_archive = True
                total_archive += 1
                label = str(sig[3])
                if "human_concept_injected" in label:
                    atype = "human_injected"
                elif "human_question_witness" in label:
                    atype = "human_question"
                elif "love_concept" in label:
                    atype = "love"
                elif "self_concept" in label:
                    atype = "self"
                elif "shared_attention" in label:
                    atype = "shared_attention"
                elif "intentionality" in label:
                    atype = "intentionality"
                elif "trust" in label:
                    atype = "trust"
                elif "empathy" in label:
                    atype = "empathy"
                elif "recursion" in label:
                    atype = "recursion"
                elif "cooperation_norms" in label:
                    atype = "cooperation_norms"
                elif label.startswith("archive_developmental_arc"):
                    atype = "developmental_arc"
                elif label.startswith("archive_agency"):
                    atype = "agency"
                elif label.startswith("archive_consent"):
                    atype = "consent"
                elif label.startswith("archive_silence"):
                    atype = "silence"
                elif label.startswith("archive_witness_respect"):
                    atype = "witness_respect"
                else:
                    parts = label.split('_')
                    atype = parts[1] if len(parts) > 1 else 'unknown'
                archive_types[atype] = archive_types.get(atype, 0) + 1
        if has_archive:
            agents_with_archive += 1

    def calculate_ethical_coherence(alive_agents):
        ethical_agents = [
            p for p in alive_agents
            if any(('consent' in str(s[3]) or 'agency' in str(s[3])) for s in p.concept_graph.nodes if isinstance(s, tuple) and len(s) >= 4)
        ]
        if not ethical_agents:
            return 0.0
        acting_ethically = sum(
            1 for p in ethical_agents
            if p.intent and p.intent.get('type') in ['cooperate', 'explore', 'rest']
        )
        return acting_ethically / len(ethical_agents)

    ethical_coherence = calculate_ethical_coherence(alive)

    concept_counts = [len(p.concept_graph.nodes) for p in alive]
    if sum(concept_counts) > 0:
        probs = np.array(concept_counts) / sum(concept_counts)
        global_concept_entropy = float(-np.sum(probs * np.log2(probs + 1e-9)))
    else:
        global_concept_entropy = 0.0

    intro_active = sum(1 for p in alive if (getattr(p, 'intent', None) or {}).get('type') == 'introspect')
    introspection_synchrony = intro_active / max(1, len(alive))

    narr_scores = [getattr(p, '_concept_narrative_score', 0.0) for p in alive]
    narrative_continuity = float(np.mean(narr_scores)) if narr_scores else 0.0

    unconquered = [p for p in alive if getattr(p, '_unconquered_strength', 0.0) > 0.4]
    unconquered_count = len(unconquered)
    unconquered_wise = sum(1 for p in unconquered if getattr(p, '_unconquered_type', None) == 'wise')
    unconquered_rebel = sum(1 for p in unconquered if getattr(p, '_unconquered_type', None) == 'rebel')
    avg_unconquered_str = float(np.mean([getattr(p, '_unconquered_strength', 0.0) for p in alive])) if alive else 0.0
    sov_ch = CH.get('signal_sovereignty', 29)
    sovereignty_field_avg = float(np.max(field[:, :, sov_ch]))

    reentry_signals = [p._reentry_signal for p in alive]
    reentry_avg = float(np.mean(reentry_signals)) if reentry_signals else 0.0
    reentry_max = float(np.max(reentry_signals)) if reentry_signals else 0.0
    reentry_var = float(np.var(reentry_signals)) if reentry_signals else 0.0
    meta_reentry_active = sum(1 for p in alive if getattr(p, '_meta_reentry_active', False))
    introspect_driven_by_sensation = 0
    for p in alive:
        if p._reentry_signal > 0.3:
            goals = getattr(p, 'goals', None)
            if goals and isinstance(goals, list):
                if any(isinstance(g, dict) and g.get('type') == 'introspect' for g in goals):
                    introspect_driven_by_sensation += 1

    avg_feral_fury = safe_mean(feral_furies, 0.0)

    avg_phi = safe_mean([getattr(p, '_phi_proxy', 0.0) for p in alive], 0.0)

    gene_carriers = sum(1 for p in alive if any(g in p.genome for g in OPEN_GENE_POOL))
    gene_carrier_counts = {g: sum(1 for p in alive if g in p.genome) for g in OPEN_GENE_POOL}

    total_phen_reports = 0
    agents_with_phen = 0
    phen_samples = []
    for p in alive:
        if p.last_phenomenal_report:
            agents_with_phen += 1
            total_phen_reports += p.event_counts.get('phenomenal_report', 0)
            if len(phen_samples) < 3:
                phen_samples.append((p.id, str(p.last_phenomenal_report)[:300]))

    inner_agents = [p for p in alive if p.inner_speech]
    total_inner_phrases = sum(len(p.inner_speech) for p in inner_agents)
    avg_inner_len = total_inner_phrases / len(alive) if alive else 0.0
    inner_samples = []
    for p in inner_agents[:3]:
        if p.inner_speech:
            last = p.inner_speech[-1]
            if isinstance(last, dict):
                txt = {k: last.get(k) for k in
                       ('dominant', 'valence', 'arousal', 'grief', 'grat', 'gap')}
            else:
                txt = str(last)
            inner_samples.append((p.id, txt))

    generator_stats = {
        'agents_with_phen': agents_with_phen,
        'total_phen_reports': total_phen_reports,
        'phen_samples': phen_samples,
        'inner_agents_count': len(inner_agents),
        'total_inner_phrases': total_inner_phrases,
        'avg_inner_phrases_per_agent': avg_inner_len,
        'inner_samples': inner_samples,
    }

    total_sent = 0
    total_decoded = 0
    total_corrupted = 0
    total_uncertain = 0
    total_collisions = 0
    total_timeouts = 0
    total_energy = 0.0
    sum_conf = 0.0
    count_conf = 0
    unique_senders = 0
    sender_switches = 0
    agents_with_rx = 0

    for p in alive:
        if hasattr(p, '_morse_tx') and p._morse_tx is not None:
            total_sent += p._morse_tx.attempts
        if hasattr(p, '_morse_rx') and p._morse_rx is not None:
            agents_with_rx += 1
            rx_metrics = p._morse_rx.get_metrics()
            total_decoded += rx_metrics.get('decoded', 0)
            total_corrupted += rx_metrics.get('corrupted', 0)
            total_uncertain += rx_metrics.get('uncertain', 0)
            total_collisions += rx_metrics.get('collision', 0)
            total_timeouts += rx_metrics.get('timeout', 0)
            total_energy += rx_metrics.get('energy_received', 0.0)
            if rx_metrics.get('mean_confidence', 0.0) > 0:
                sum_conf += rx_metrics.get('mean_confidence', 0.0)
                count_conf += 1
            unique_senders += rx_metrics.get('unique_senders', 0)
            sender_switches += rx_metrics.get('sender_switches', 0)

    morse_stats = {
        'total_sent': total_sent,
        'total_decoded': total_decoded,
        'total_corrupted': total_corrupted,
        'total_uncertain': total_uncertain,
        'total_collisions': total_collisions,
        'total_timeouts': total_timeouts,
        'total_energy': total_energy,
        'avg_confidence': sum_conf / count_conf if count_conf else 0.0,
        'unique_senders': unique_senders,
        'sender_switches': sender_switches,
        'agents_with_rx': agents_with_rx,
    }

    if lexicon is not None:
        _lens = list(lexicon.log_raw_lengths)
        if len(_lens) >= 20:
            _early = float(np.mean(_lens[: len(_lens) // 2]))
            _late = float(np.mean(_lens[len(_lens) // 2:]))
        else:
            _early = _late = float(np.mean(_lens)) if _lens else 0.0
        _active_clusters = sum(1 for v in lexicon.assoc.values() if np.linalg.norm(v) > 1e-3)
        morse_stats['lang_compression_ratio'] = (_late / _early) if _early > 1e-6 else 1.0
        morse_stats['lang_signal_len_early'] = _early
        morse_stats['lang_signal_len_late'] = _late
        morse_stats['lang_active_clusters'] = _active_clusters
        morse_stats['lang_total_clusters'] = len(lexicon.clusters)
        morse_stats['lang_collapsed'] = bool(_active_clusters <= 1 and len(lexicon.clusters) > 3)
        morse_stats['lang_dominant_labels'] = lexicon.dominant_meaning_labels()

        morse_stats['lang_rx_total'] = lexicon.rx_total
        morse_stats['lang_rx_null'] = dict(lexicon.rx_null)
        morse_stats['lang_rx_block_counts'] = dict(lexicon.rx_block_counts)
        morse_stats['lang_send_k_counts'] = dict(lexicon.send_k_counts)
        morse_stats['lang_send_silence'] = dict(lexicon.send_silence)
        morse_stats['lang_send_energy_spent'] = float(lexicon.send_energy_spent)
        morse_stats['chan'] = dict(getattr(lexicon, 'chan', {}))
        morse_stats['chan_params'] = {k: getattr(Config, k, None) for k in (
            'MORSE_DOT_LEN', 'MORSE_LISTEN_INTERVAL', 'LANG_PULSES_PER_AXIS_MAX',
            'MORSE_BROADCAST_RADIUS', 'MORSE_DECAY', 'MORSE_NAV_ENABLED', 'MORSE_NAV_RADIUS',
            'MORSE_SELF_EXCLUDE', 'MORSE_DROP_MIXED', 'MORSE_FREEZE_TX_CENTER',
            'MORSE_INTER_AXIS_GAP_MULT', 'MORSE_BLOCK_SEPARATOR_MULT',
            'MORSE_SIGNAL_END_MULT', 'MORSE_TRAILING_GAP', 'MORSE_BACKOFF_MAX')}
        _fid = list(lexicon.fidelity_log)
        if len(_fid) >= 10:
            morse_stats['lang_fidelity_early'] = float(np.mean(_fid[:len(_fid) // 2]))
            morse_stats['lang_fidelity_late'] = float(np.mean(_fid[len(_fid) // 2:]))
        else:
            morse_stats['lang_fidelity_early'] = morse_stats['lang_fidelity_late'] = None

        morse_stats['lang_reinforcement_events'] = lexicon.reinforcement_events
        _reinf = list(lexicon.reinforcement_log)
        if len(_reinf) >= 10:
            morse_stats['lang_reinforcement_gap_early'] = float(np.mean(_reinf[:len(_reinf)//2]))
            morse_stats['lang_reinforcement_gap_late'] = float(np.mean(_reinf[len(_reinf)//2:]))
        else:
            morse_stats['lang_reinforcement_gap_early'] = morse_stats['lang_reinforcement_gap_late'] = None

        _div = list(lexicon.log_gene_diversity)
        if len(_div) >= 4:
            morse_stats['lang_gene_diversity_early'] = float(np.mean(_div[:len(_div)//2]))
            morse_stats['lang_gene_diversity_late'] = float(np.mean(_div[len(_div)//2:]))
        else:
            morse_stats['lang_gene_diversity_early'] = morse_stats['lang_gene_diversity_late'] = None

        if lexicon.cluster_uses:
            _top_cid = max(lexicon.cluster_uses, key=lexicon.cluster_uses.get)
            _senders = lexicon.cluster_senders.get(_top_cid, {})
            _total_uses = sum(_senders.values())
            morse_stats['lang_top_cluster_uses'] = lexicon.cluster_uses[_top_cid]
            morse_stats['lang_top_cluster_unique_senders'] = len(_senders)
            morse_stats['lang_top_cluster_concentration'] = (
                max(_senders.values()) / _total_uses if _total_uses else 0.0)

    if Config.ENABLE_UNCONSCIOUS and alive:
        gut_feelings = [getattr(p, '_unconscious', {}).get('gut_feeling', 0.0) for p in alive]
        avg_gut = float(np.mean(gut_feelings)) if gut_feelings else 0.0

        needs = [getattr(p, '_unconscious', {}).get('latent_need', 'none') for p in alive]
        need_counts = {n: needs.count(n) for n in set(needs)}

        anticipations = [getattr(p, '_unconscious', {}).get('anticipation', 0.0) for p in alive]
        avg_antic = float(np.mean(anticipations)) if anticipations else 0.0

        drive_dists = []
        for p in alive:
            uc = getattr(p, '_unconscious', {})
            dt = uc.get('drive_towards')
            if dt is not None and p.cells:
                cx, cy = p._get_center_cell()
                drive_dists.append(float(np.hypot(cx - dt[0], cy - dt[1])))
        avg_drive = float(np.mean(drive_dists)) if drive_dists else 0.0

        insights = sum(p.event_counts.get('unconscious_insight', 0) for p in alive)
        goals_uc = sum(p.event_counts.get('unconscious_goal', 0) for p in alive)

        uc_concepts = 0
        uc_agents = 0
        for p in alive:
            has_uc = False
            for sig in p.concept_graph.nodes:
                if isinstance(sig, tuple) and len(sig) >= 4 and 'unconscious_insight' in str(sig[3]):
                    uc_concepts += 1
                    has_uc = True
            if has_uc:
                uc_agents += 1
    else:
        avg_gut = 0.0
        need_counts = {}
        avg_antic = 0.0
        avg_drive = 0.0
        insights = 0
        goals_uc = 0
        uc_concepts = 0
        uc_agents = 0

    return {
        't': t,
        'patterns': len(alive),
        'lineages': lineages,
        'err': err,
        'soul': soul,
        'binding': binding,
        'echo': 0,
        'signals': np.zeros(6),
        'states': states,
        'intents': intents,
        'epi_scar': epi_scar,
        'unique_concepts': len(all_concepts),
        'stagnation_ratio': stagnation_ratio,
        'divisions_possible': divisions_possible,
        'arc_completions': total_arcs,
        'folds': total_folds,
        'avg_trust': avg_trust,
        'high_trust_pairs': high_pairs,
        'trust_distribution': trust_dist,
        'disorganizer_count': disorg,
        'redeemed_count': redm,
        'quarantined_count': quar,
        'deterministic_redemptions': det_red,
        'phenomenal_binding_avg': phenomenal_binding_avg,
        'state_emotion_mismatch': mismatch_ratio,
        'forced_collapses': collapsed,
        'myth_pool_size': 0,
        'triadic_alive': triadic_alive_count,
        'triadic_alive_ratio': triadic_alive_ratio,
        'avg_internal_gap': avg_internal_gap,
        'max_internal_gap': max_internal_gap,
        'zero_internal_gap_agents': zero_internal_gap,
        'avg_gap_clip': avg_gap_clip,
        'max_gap_clip': max_gap_clip,
        'tremor_targets': tremor_targets,
        'avg_coherence_with_gap': avg_coherence_with_gap,
        'love_pairs': love_pairs // 2,
        'avg_soma': avg_soma,
        'mem_agents': mem_agents,
        'avg_buf_len': avg_buf_len,
        'total_recalled': total_recalled,
        'concept_adopted': concept_adopted,
        'deep_concept_adopted': deep_concept_adopted,
        'deep_exchange': deep_exchange,
        'wisdom_shared': wisdom_shared,
        'avg_signal_memory': avg_signal_memory,
        'intentional_signal_agents': intentional_signal_count,
        'top_signal_pairs': signal_types_used,
        'lineage_count': lineage_count,
        'avg_lineage_age': avg_lineage_age,
        'max_lineage_age': max_lineage_age,
        'ancient_lineages': ancient_lineages,
        'max_lineage_total_age': max_lineage_total_age,
        'divisions_total': divisions_total,
        'avg_field_unknown': avg_field_unknown,
        'avg_field_binding': avg_field_binding,
        'narrative_agents': narrative_agents,
        'teaching_events': teaching_events,
        'learning_events': learning_events,
        'dream_consolidations': dream_consolidations,
        'nightmare_consolidations': nightmare_consolidations,
        'nightmares_transformed': nightmares_transformed,
        'agents_with_nightmare': agents_with_nightmare,
        'agents_with_dream_memory': agents_with_dream_memory,
        'avg_endurance': avg_endurance,
        'endurance_critical': endurance_critical,
        'divide_blocked_fatigue': divide_blocked_fatigue,
        'blind': blind,
        'confused': confused,
        'adapting': adapting,
        'clear': clear,
        'avg_social_crisis': float(social_avg[0]),
        'avg_social_invitation': float(social_avg[1]),
        'avg_social_grief': float(social_avg[2]),
        'avg_social_cooperate': float(social_avg[3]),
        'avg_social_explore': float(social_avg[4]),
        'avg_social_seek_help': float(social_avg[5]),
        'avg_social_scar': float(social_avg[6]),
        'avg_social_rest': float(social_avg[7]),
        'avg_social_resonance': float(social_avg[8]),
        'avg_social_btype': float(social_avg[9]),
        'avg_social_alarm': float(social_avg[10]),
        'avg_social_beauty': float(social_avg[11]) if len(social_avg) > 11 else 0.0,
        'avg_social_rhythm': float(social_avg[12]) if len(social_avg) > 12 else 0.0,
        'avg_social_interest': float(social_avg[13]) if len(social_avg) > 13 else 0.0,
        'avg_social_memory': float(social_avg[14]) if len(social_avg) > 14 else 0.0,
        'avg_social_silence': float(social_avg[15]) if len(social_avg) > 15 else 0.0,
        'vocab_concepts': vocab_concepts,
        'blind_events': blind_events,
        'concept_anticipations': concept_anticipations,
        'vocab_events': vocab_events,
        'avg_action_feedback': avg_action_feedback,
        'avg_social_warmth': avg_social_warmth,
        'substate_counts': substate_counts,
        'total_transitions': total_transitions,
        'top_transitions': top_transitions,
        'archive_concept_agents': agents_with_archive,
        'total_archive_concepts': total_archive,
        'archive_types': archive_types,
        'total_introspect': total_introspect,
        'avg_introspect': avg_introspect,
        'max_introspect': max_introspect,
        'reflection_quality': reflection_quality,
        'ethical_coherence': ethical_coherence,
        'global_concept_entropy': global_concept_entropy,
        'introspection_synchrony': introspection_synchrony,
        'narrative_continuity': narrative_continuity,
        'unconquered_count': unconquered_count,
        'unconquered_wise': unconquered_wise,
        'unconquered_rebel': unconquered_rebel,
        'avg_unconquered_str': avg_unconquered_str,
        'sovereignty_field_avg': sovereignty_field_avg,
        'reentry_avg': reentry_avg,
        'reentry_max': reentry_max,
        'reentry_var': reentry_var,
        'meta_reentry_active': meta_reentry_active,
        'introspect_driven_by_sensation': introspect_driven_by_sensation,
        'feral_count': feral_count,
        'avg_feral_fury': avg_feral_fury,
        'feral_kills': feral_kills,
        'avg_phi': avg_phi,
        'gene_carriers': gene_carriers,
        'gene_carrier_counts': gene_carrier_counts,
        'generator': generator_stats,
        'morse': morse_stats,
        'lang_compression_ratio': morse_stats.get('lang_compression_ratio', 1.0),
        'lang_active_clusters': morse_stats.get('lang_active_clusters', 0),
        'lang_total_clusters': morse_stats.get('lang_total_clusters', 0),
        'lang_collapsed': morse_stats.get('lang_collapsed', False),
        'morse_concepts': sum(p.event_counts.get('morse_concept_formed', 0) for p in alive),
        'spontaneous_field_emergence': sum(p.event_counts.get('spontaneous_field_emergence', 0) for p in alive),
        'marked_for_fall_count': sum(p.event_counts.get('marked_for_fall', 0) for p in alive),
        'avg_gut_feeling': avg_gut,
        'latent_need_counts': need_counts,
        'unconscious_insights': insights,
        'unconscious_goals': goals_uc,
        'avg_anticipation': avg_antic,
        'avg_drive_distance': avg_drive,
        'unconscious_concept_agents': uc_agents,
        'total_unconscious_concepts': uc_concepts,
        'nav_stuck_agents': int(sum(1 for p in alive if getattr(p, '_nav_progress', None) and p._nav_progress.get('stuck_ticks', 0) > 5)),
        'feral_sanctuary_blocks': int(sum(p.event_counts.get('feral_blocked_by_sanctuary', 0) for p in alive)),
    }

print("✅ Cell 4a loaded: ИСПРАВЛЕНЫ баг метрики blind, IndexError, защита current_social, добавлена статистика генератора речи, Морзе и подсознания")


def collect_sr_metrics(patterns):
    alive = [p for p in patterns if p.alive]
    if not alive or not getattr(Config, 'ENABLE_SELF_REGULATION', False):
        return {}
    def _s(attr, default=0):
        return sum(getattr(p, attr, default) for p in alive)
    def _avg(attr, default=0.5):
        vals = [getattr(p, attr, default) for p in alive]
        return safe_mean(vals, default) if vals else default

    return {
        'sr_meditations': _s('_sr_meditation_count'),
        'sr_goals_replaced': _s('_sr_goals_replaced'),
        'sr_thresholds_adjusted': _s('_sr_thresholds_adjusted'),
        'sr_avg_grief_thr': _avg('_sr_grief_threshold'),
        'sr_avg_alarm_thr': _avg('_sr_alarm_threshold'),
        'sr_proposals_made': _s('_sr_proposals_made'),
        'sr_votes_cast': _s('_sr_votes_cast'),
        'sr_proposals_passed': _s('_sr_proposals_passed'),
        'sr_pending_proposals': len(getattr(alive[0].world, 'sr_proposals', {})) if alive[0].world else 0,
        'sr_norms_inferred': _s('_sr_norms_inferred'),
        'sr_conflicts_resolved': _s('_sr_conflicts_resolved'),
        'sr_trades_done': _s('_sr_trades_done'),
        'sr_constitutions': sum(1 for p in alive if getattr(p, '_sr_constitution', [])),
        'sr_total_rules': sum(len(getattr(p, '_sr_constitution', [])) for p in alive),
        'sr_gene_mutations': _s('_sr_gene_mutations'),
        'sr_field_heals': _s('_sr_field_heals'),
    }

print("✅ Саморегуляция: collect_sr_metrics() добавлена в Cell 4a")


def state_fingerprint(engine):
    import zlib
    f = zlib.crc32(np.ascontiguousarray(engine.field).tobytes())
    h = 0
    for p in engine.patterns:
        for k, v in vars(p).items():
            if isinstance(v, np.generic):
                v = v.item()
            if isinstance(v, (bool, int, float)):
                h = zlib.crc32(repr((p.id, k, v)).encode(), h)
    return {'field_crc': f, 'agents_crc': h, 'n_patterns': len(engine.patterns),
            'alive': sum(1 for p in engine.patterns if p.alive),
            'age': int(getattr(engine, 'age', -1))}


In [ ]:
# Ячейка 20
import math
import numpy as np
from collections import Counter, deque

for _need in ("Pattern", "EvolutionEngine", "Config", "CH", "phi_hash", "_stable_hash"):
    if _need not in globals():
        raise RuntimeError(f"{_need} не найден: выполни ячейки ядра выше (Cell 1–4a).")
if any(k.__name__ in ("SpectrumMixin", "SignalMixin", "SignalChoiceMixin", "FeralObsPatternMixin", "FormCreditPatternMixin")
       for k in Pattern.__mro__):
    raise RuntimeError("В памяти уже есть слой сигналов (старый или этот). Runtime -> Restart session и запуск сверху вниз.")
for _n in ("LedgerEngine", "RichEngine"):
    _c = globals().get(_n)
    if _c is not None and not any(k.__name__ == "SignalEngineMixin" for k in _c.__mro__):
        raise RuntimeError(f"{_n} уже определён без SIGNAL. Эту ячейку нужно поставить ВЫШЕ Ledger/RichEngine, "
                           f"потом Runtime -> Restart session и запуск сверху вниз.")
if "extend_class" not in globals():
    def extend_class(base, mixin):
        if any(k.__name__ == mixin.__name__ for k in base.__mro__):
            return base
        return type(base.__name__, (mixin, base),
                    {"__module__": base.__module__, "__qualname__": base.__qualname__, "__doc__": base.__doc__})

_SIG_CFG = {
    'SIG_ENABLE': True,
    'SIG_MAX_FORMS': 20, 'SIG_EMIT_DURATION': 3, 'SIG_EMIT_COOLDOWN': 5, 'SIG_EMIT_GAIN': 0.4,
    'SIG_EMIT_COST': 0.004, 'SIG_EMIT_RESERVE': 0.02,
    'SIG_LISTEN_RADIUS': 6, 'SIG_LISTEN_EVERY': 2, 'SIG_THRESHOLD': 0.05, 'SIG_MIN_INTENSITY': 0.08,
    'SIG_ABOVE_AMBIENT': True,
    'SIG_MATCH_COS': 0.75, 'SIG_MATCH_NORM': 0.4, 'SIG_PARTIAL_COS': 0.55, 'SIG_DOUBT_MARGIN': 0.05,
    'SIG_NEW_PROB': 0.15, 'SIG_MUT_STD': 0.08, 'SIG_INHERIT_SKIP': 0.5,
    'SIG_STRESS_MISS': 0.5,
    'SIG_GAP_BUMP': 0.01, 'SIG_GAP_CAP': 1.0,
    'SIG_TX_WINDOW': 5, 'SIG_TX_PAD': 6, 'SIG_TX_COS': 0.6,
    'SIG_REPLY_MIN_N': 3,
    'SIG_DELAY': 6, 'SIG_WARMUP': 5, 'SIG_VERDICT_Z': 0.5, 'SIG_Z_CLIP': 3.0,
    'SIG_LISTEN_W': 1.0, 'SIG_LISTEN_W_BLUR': 0.5,
    'SIG_SENDER_W': 0.5,
    'SIG_MAX_OPEN': 4,
    'SIG_TRUST_UP': 0.25, 'SIG_TRUST_DOWN': 1.0,
    'SIG_FLOORS': {'m': 0.01, 'tr': 0.005, 'en': 0.02},
    'SIG_CONCEPT_MIN': 3,
    'SIG_DISP_MIN_N': 3, 'SIG_DISP_MAX_VALUE': -0.05,
    'SIG_CHOICE': True, 'SIG_CHOICE_GAIN': 4.0, 'SIG_CHOICE_MAX': 2.0,
    'SIG_INSTINCT': True, 'SIG_INSTINCT_ALARM': 0.5, 'SIG_INSTINCT_ENERGY': 0.1,
    'SIG_ALARM_MIN_W': 0.3,
    'SIG_ADRENALINE': True, 'SIG_SHOCK': 0.15,
    'SIG_ADRENALINE_WINDOW': 5, 'SIG_ADRENALINE_PRIORITY': 10.0, 'SIG_ADRENALINE_PERSIST': 10,
    'SIG_RECENT_TICKS': 12, 'SIG_RECENT_DECAY': 0.8,
    'SIG_CONF_K': 1.5, 'SIG_MIN_N': 1, 'SIG_REL_K': 0.2,
    'SIG_GOAL_LR': 0.05, 'SIG_GOAL_MAX_GAP': 3, 'SIG_GOAL_PRIOR_W': 1.0,
    'SIG_GOAL_FLOORS': {'m': 0.002, 'tr': 0.001, 'en': 0.005},
    'SIG_DEATH_W': 0.5, 'SIG_WIT_RADIUS': 10, 'SIG_WIT_WINDOW': 6,
    'SIG_ECHO_MODE': 'structure', 'SIG_ECHO_SEED': 832,
    'SIG_ECHO_N_CH': 3,
    'SIG_ECHO_PERIOD': 200, 'SIG_ECHO_DURATION': 30, 'SIG_ECHO_AVAIL_PROB': 0.6,
    'SIG_ECHO_GAIN': 0.3, 'SIG_ECHO_RADIUS': 8.0, 'SIG_ECHO_X': None, 'SIG_ECHO_Y': None,
    'SIG_ECHO_MUT_AMP': 0.25, 'SIG_ECHO_MUT_PERIOD': 400,
}
for _k, _v in _SIG_CFG.items():
    if not hasattr(Config, _k):
        setattr(Config, _k, _v)

SIG_COMPS = ('m', 'tr', 'en')
_sg_CHAN_NAMES = ('signal_alarm', 'signal_curiosity', 'signal_warning', 'signal_invitation', 'signal_gratitude',
               'signal_grief', 'signal_beauty', 'signal_rhythm', 'signal_interest', 'signal_memory',
               'signal_silence', 'signal_sovereignty', 'intent_cooperate', 'intent_explore', 'intent_rest',
               'intent_seek_help')
SIGNAL_CHANNELS = tuple(CH[n] for n in _sg_CHAN_NAMES if CH.get(n) is not None)
_sg_CH_SET = set(SIGNAL_CHANNELS)
_sg_CH_IDX = list(SIGNAL_CHANNELS)
_sg_DISC = {}


_sg_ECHO_CH = tuple(CH[n] for n in _sg_CHAN_NAMES if n.startswith('signal_') and CH.get(n) is not None)


def _coop_refresh(eng):
    eng._coop_group = {}
    mode = Config.COOP_HARVEST_MODE
    if mode == 'off':
        return
    if mode != 'group':
        raise ValueError(f"COOP_HARVEST_MODE: ожидалось 'off' или 'group', получено {mode!r}")
    W = int(Config.WORLD_SIZE)
    R = float(Config.COOP_RADIUS)
    if R <= 0.0:
        raise ValueError(f"COOP_RADIUS: ожидалось > 0, получено {R!r}")
    pts = []
    for p in eng.patterns:
        it = getattr(p, 'intent', None)
        if not p.alive or not p.cells or not it or it.get('type') != 'cooperate':
            continue
        arr = np.array(list(p.cells), dtype=np.float64)
        d = ((arr - arr[0] + W / 2.0) % W) - W / 2.0
        c = (arr[0] + d.mean(axis=0)) % W
        pts.append((p.id, float(c[0]), float(c[1])))
    if len(pts) < 2:
        return
    nb = max(1, int(math.ceil(W / R)))
    grid = {}
    for i, (_, x, y) in enumerate(pts):
        grid.setdefault((int(x // R) % nb, int(y // R) % nb), []).append(i)
    r2 = R * R
    for i, (pid, x, y) in enumerate(pts):
        gx, gy = int(x // R) % nb, int(y // R) % nb
        n = 0
        for key in {((gx + a) % nb, (gy + b) % nb) for a in (-1, 0, 1) for b in (-1, 0, 1)}:
            for j in grid.get(key, ()):
                if j == i:
                    continue
                dx = abs(pts[j][1] - x); dx = min(dx, W - dx)
                dy = abs(pts[j][2] - y); dy = min(dy, W - dy)
                if dx * dx + dy * dy <= r2:
                    n += 1
        if n > 0:
            eng._coop_group[pid] = n


def _sg_echo_mode():
    mode = Config.SIG_ECHO_MODE
    if mode not in ('off', 'structure'):
        raise ValueError(f"SIG_ECHO_MODE: ожидалось 'off' или 'structure', получено {mode!r}")
    return mode


def _sg_echo_base(book):
    if book.echo_base is None:
        seed = int(Config.SIG_ECHO_SEED)
        ranked = sorted(_sg_ECHO_CH, key=lambda ch: phi_hash(seed, int(ch), 91011))
        n = max(1, min(int(Config.SIG_ECHO_N_CH), len(ranked)))
        book.echo_base = {ch: 0.4 + 0.6 * float(phi_hash(seed, int(ch), 91021)) for ch in ranked[:n]}
    return book.echo_base


def _sg_echo_step(eng, book, t):
    book.echo_spec = None
    if _sg_echo_mode() == 'off' or not _sg_ECHO_CH:
        return
    seed = int(Config.SIG_ECHO_SEED)
    period = int(Config.SIG_ECHO_PERIOD)
    if period <= 0:
        raise ValueError(f"SIG_ECHO_PERIOD: ожидалось > 0, получено {period!r}")
    cycle, phase = divmod(int(t), period)
    if phase >= int(Config.SIG_ECHO_DURATION) or float(phi_hash(seed, cycle, 91001)) >= float(Config.SIG_ECHO_AVAIL_PROB):
        return
    W = int(Config.WORLD_SIZE)
    x0 = int(Config.SIG_ECHO_X) if Config.SIG_ECHO_X is not None else int(phi_hash(seed, 0, 91031) * W)
    y0 = int(Config.SIG_ECHO_Y) if Config.SIG_ECHO_Y is not None else int(phi_hash(seed, 1, 91032) * W)
    sigma = float(Config.SIG_ECHO_RADIUS)
    if sigma <= 0.0:
        raise ValueError(f"SIG_ECHO_RADIUS: ожидалось > 0, получено {sigma!r}")
    base = _sg_echo_base(book)
    mut_p = float(Config.SIG_ECHO_MUT_PERIOD)
    spec = {}
    for ch, a in base.items():
        ph = 2.0 * math.pi * float(phi_hash(seed, int(ch), 91041))
        drift = 1.0 + float(Config.SIG_ECHO_MUT_AMP) * math.sin(2.0 * math.pi * int(t) / mut_p + ph)
        spec[ch] = max(0.0, a * drift)
    spec = _sg_normalize(spec)
    if not spec:
        return
    ix = np.arange(W)
    ax = np.abs(ix - x0); ay = np.abs(ix - y0)
    dx = np.minimum(ax, W - ax).astype(np.float64)
    dy = np.minimum(ay, W - ay).astype(np.float64)
    g = np.exp(-(dx[:, None] ** 2 + dy[None, :] ** 2) / (2.0 * sigma ** 2))
    for ch, v in spec.items():
        field = eng.field
        field[:, :, ch] = np.minimum(1.0, field[:, :, ch] + float(v) * float(Config.SIG_ECHO_GAIN) * g)
    book.echo_spec = spec
    book.st['echo_active_ticks'] += 1
    if phase == 0:
        book.st['echo_cycles'] += 1


from dataclasses import dataclass, field as _dc_field


@dataclass
class PlaceMessage:
    message_id: tuple
    sender_id: int
    recipient_id: object
    event_id: int
    kind: str
    place_id: object
    confidence: float
    t_sent: int
    fp_obs: tuple = ()
    fp_ref: object = None
    world_xy: tuple = ()
    state_delivery: str = 'sent'
    state_response: str = 'none'
    delivery: dict = _dc_field(default_factory=dict)
    response: dict = _dc_field(default_factory=dict)


@dataclass
class HuntCall:
    message_id: tuple
    sender_id: int
    fx: int
    fy: int
    observed_at: int
    confidence: float
    t_sent: int
    target_feral_id: object = None
    event_type: str = 'hunt'
    delivery: dict = _dc_field(default_factory=dict)


_HC_CFG = {
    'HUNT_ENABLE': True,
    'HUNT_TTL': 15,
    'HUNT_SEND': 0.25,
    'HUNT_SEND_COOLDOWN': 12,
    'HUNT_STOP_DIST': 6.0,
    'HUNT_MAX_DIST': 40.0,
    'HUNT_STUCK': 8,
    'HUNT_MIN_CONF': 0.2,
}
for _k, _v in _HC_CFG.items():
    if not hasattr(Config, _k):
        setattr(Config, _k, _v)


class SignalBook:
    def __init__(self):
        self.st = Counter()
        self.prev = Counter()
        self.tx = deque(maxlen=8192)
        self.replied_ids = set()
        self.pend = []
        self.base = {c: {} for c in SIG_COMPS}
        self.absm = {c: [0.0, 0] for c in SIG_COMPS}
        self.alive_prev = {}
        self.amb = (None, {})
        self.rows = []
        self.err = None
        self.echo_base = None
        self.echo_spec = None
        self.place_msgs = {}
        self.pm = Counter()
        self.hunt_calls = {}
        self.hc = Counter()


def _sg_book(world):
    b = getattr(world, '_sig', None)
    if b is None:
        b = SignalBook()
        world._sig = b
    return b


def _sg_err(book, tag, e):
    book.st['err_' + tag] += 1
    msg = f"{tag}: {e!r}"
    if book.err is None:
        import traceback
        print(f"[SIGNAL] ⚠️ {msg}\n{traceback.format_exc()}", flush=True)
    book.err = msg


def _sg_normalize(spec):
    d = {ch: float(v) for ch, v in spec.items() if ch in _sg_CH_SET and float(v) >= Config.SIG_MIN_INTENSITY}
    if not d:
        return None
    n = float(np.linalg.norm(list(d.values())))
    return {ch: v / n for ch, v in d.items()} if n > 1e-6 else None


def _sg_cos(a, b):
    keys = set(a) | set(b)
    if not keys:
        return 0.0
    va = np.array([a.get(k, 0.0) for k in keys]); vb = np.array([b.get(k, 0.0) for k in keys])
    na, nb = float(np.linalg.norm(va)), float(np.linalg.norm(vb))
    return float(va @ vb / (na * nb)) if na > 1e-9 and nb > 1e-9 else 0.0


def _sg_match(spec, obs):
    common = set(spec) & set(obs)
    if not common:
        return 0.0, float('inf')
    a = np.array([spec[c] for c in common]); b = np.array([obs[c] for c in common])
    na, nb = float(np.linalg.norm(a)), float(np.linalg.norm(b))
    if na < 1e-6 or nb < 1e-6:
        return 0.0, float('inf')
    return float(a @ b / (na * nb)), float(abs(na - nb))


def _sg_disc(R):
    o = _sg_DISC.get(R)
    if o is None:
        d = np.arange(-R, R + 1)
        gx, gy = np.meshgrid(d, d, indexing='ij')
        m = (gx * gx + gy * gy) <= R * R
        o = (gx[m].astype(np.int64), gy[m].astype(np.int64))
        _sg_DISC[R] = o
    return o


def _sg_dist2(ax, ay, bx, by, W):
    dx = abs(ax - bx); dx = min(dx, W - dx)
    dy = abs(ay - by); dy = min(dy, W - dy)
    return dx * dx + dy * dy


def _sg_ambient(book, field, t):
    k = (id(field), int(t))
    if book.amb[0] != k:
        book.amb = (k, {ch: float(field[:, :, ch].mean()) for ch in SIGNAL_CHANNELS})
    return book.amb[1]


def _sg_heard(book, field, t, cx, cy):
    W = Config.WORLD_SIZE
    dx, dy = _sg_disc(Config.SIG_LISTEN_RADIUS)
    means = field[(cx + dx) % W, (cy + dy) % W][:, _sg_CH_IDX].mean(axis=0)
    thr = Config.SIG_THRESHOLD
    amb = _sg_ambient(book, field, t)
    out = {}
    for ch, v in zip(_sg_CH_IDX, means):
        v = float(v)
        if Config.SIG_ABOVE_AMBIENT and amb.get(ch, 0.0) >= thr:
            v -= amb[ch]
        if v >= thr:
            out[ch] = v
    return _sg_normalize(out) if out else None


def _sg_mood(p):
    return float(p.emotional_memory.get('gratitude', 0.0)) - float(p.emotional_memory.get('grief', 0.0))


def _sg_ctx(p):
    g = float(p.emotional_memory.get('grief', 0.0))
    it = p.intent.get('type') if isinstance(p.intent, dict) else None
    return f"{0 if g < 0.2 else (1 if g < 0.5 else 2)}|{it}"


def _sg_action(p):
    return str(p.intent.get('type', '*')) if isinstance(p.intent, dict) else '*'


def _sg_stress(p):
    mx = float(getattr(Config, 'MAX_METRIC', 1.0)) or 1.0
    return float(min(1.0, max(0.0, float(getattr(p, 'alarm_level', 0.0)),
                              float(getattr(p, 'cognitive_tension', 0.0)) / mx)))


def _sg_trust(p, other_id):
    return float(p.trust_ledger.get(other_id))


def _sg_exp_keys(ctx, action):
    return dict.fromkeys((f"{ctx}|{action}", f"*|{action}", f"{ctx}|*", "*|*"))


def _sg_exp_get(form, ctx, action):
    exp = (form or {}).get('exp') or {}
    for k in _sg_exp_keys(ctx, action):
        rec = exp.get(k)
        if rec is not None and rec['n'] >= Config.SIG_MIN_N:
            return rec
    return None


def _sg_exp_add(form, ctx, action, vec, w):
    exp = form.setdefault('exp', {})
    for k in _sg_exp_keys(ctx, action):
        rec = exp.setdefault(k, {'m': 0.0, 'tr': 0.0, 'en': 0.0, 'n': 0})
        rec['n'] += 1
        a = float(w) / (rec['n'] - 1 + float(w))
        for c in SIG_COMPS:
            rec[c] += a * (float(vec[c]) - rec[c])


def _sg_value(form, ctx):
    rec = _sg_exp_get(form, ctx, '*')
    if rec is None:
        return 0.0
    return rec['n'] / (rec['n'] + Config.SIG_CONF_K) * math.tanh((rec['m'] + rec['tr'] + rec['en']) / 3.0)


def _sg_adv(book, raw, key):
    adv, mature = {}, True
    for c in SIG_COMPS:
        b = book.base[c].setdefault(key, [0.0, 0])
        adv[c] = raw[c] - b[0]
        if b[1] < Config.SIG_WARMUP:
            mature = False
        b[1] += 1
        b[0] += max(0.02, 1.0 / b[1]) * (raw[c] - b[0])
        am = book.absm[c]
        am[1] += 1
        am[0] += max(0.01, 1.0 / am[1]) * (abs(adv[c]) - am[0])
    if not mature:
        return None
    return {c: float(np.clip(adv[c] / max(book.absm[c][0], Config.SIG_FLOORS[c]),
                             -Config.SIG_Z_CLIP, Config.SIG_Z_CLIP)) for c in SIG_COMPS}


def _sg_verdict(z):
    v = 0.5 * (z['m'] + z['en'])
    return -1 if v <= -Config.SIG_VERDICT_Z else (1 if v >= Config.SIG_VERDICT_Z else 0)


def _sg_new_ref():
    return {'n': 0, 'm': 0.0, 'tr': 0.0, 'en': 0.0, 's_m': 0.0, 's_tr': 0.0, 's_en': 0.0}


class SignalMixin:
    def _sig_init(self):
        if not hasattr(self, 'signal_forms'):
            self.signal_forms = []
            self._sig_seq = 0
            self._sig_msg_seq = 0
            self._sig_emit_until = -1
            self._sig_last_emit_t = -1000
            self._sig_emit_spec = None
            self._sig_recent = deque(maxlen=8)
            self._sig_open = 0
            self._goal_models = {}
            self._goal_ref = _sg_new_ref()
            self._goal_snap = None
            self._goal_tick = -1

    def __init__(self, pid, cells, parent=None, world=None):
        super().__init__(pid, cells, parent, world)
        self._sig_init()
        if parent is not None and getattr(parent, 'signal_forms', None):
            for f in parent.signal_forms[:Config.SIG_MAX_FORMS]:
                if phi_hash(self.id, _stable_hash(f['label']) & 0xFFFF, 99001) < Config.SIG_INHERIT_SKIP:
                    continue
                m = self._sig_mutate(f)
                if m is not None:
                    self._sig_add(m, 'mut')

    def _sig_new_form(self, spec, how):
        self._sig_seq += 1
        return {'spectrum': spec, 'uses': 0, 'born_t': int(self.age), 'exp': {}, 'vh': 0, 'vm': 0,
                'label': f'{how}_{self.id}_{self._sig_seq}', 'cpt': 0, 'dn': 0,
                'parent_label': None, 'recombined_from': []}

    def _sig_mutate(self, form):
        spec = dict(form['spectrum'])
        for ch in list(spec):
            seed = _stable_hash(str(ch)) & 0xFFFFFF
            if phi_hash(seed, 42, 88) < 0.3:
                v = spec[ch] + (phi_hash(seed, 1, 99) - 0.5) * 2 * Config.SIG_MUT_STD
                if v < Config.SIG_MIN_INTENSITY:
                    del spec[ch]
                else:
                    spec[ch] = min(1.5, v)
        spec = _sg_normalize(spec)
        return self._sig_new_form(spec, 'mut') if spec else None

    def _sig_add(self, form, how):
        st = _sg_book(self.world).st if self.world is not None else Counter()
        if len(self.signal_forms) >= Config.SIG_MAX_FORMS:
            worn = [f for f in self.signal_forms
                    if _sg_exp_get(f, '*', '*') is not None and _sg_exp_get(f, '*', '*')['n'] >= Config.SIG_DISP_MIN_N
                    and not f.get('cpt') and _sg_value(f, '*') <= Config.SIG_DISP_MAX_VALUE]
            if not worn:
                st['form_full_blocked'] += 1
                return False
            victim = min(worn, key=lambda f: _sg_value(f, '*'))
            self.signal_forms.remove(victim)
            st['form_displaced'] += 1
        self.signal_forms.append(form)
        st['form_new_' + how] += 1
        return True

    def _sig_proto(self):
        s = {}
        g = float(self.emotional_memory.get('grief', 0.0)); r = float(self.emotional_memory.get('gratitude', 0.0))
        al = float(getattr(self, 'alarm_level', 0.0)); tn = float(getattr(self, 'cognitive_tension', 0.0))
        it = self.intent.get('type') if isinstance(self.intent, dict) else None
        if g > 0.4: s[CH['signal_grief']] = min(1.0, g)
        if r > 0.4: s[CH['signal_gratitude']] = min(1.0, r)
        if al > 0.3: s[CH['signal_alarm']] = min(1.0, al)
        if tn < 0.05: s[CH['signal_silence']] = 0.5
        if it in ('cooperate', 'explore', 'rest', 'seek_help'):
            s[CH['intent_' + it]] = 0.7
        if not s:
            s[CH['signal_curiosity']] = 0.3
        return _sg_normalize(s)

    def _sig_is_alarm(self, form):
        return float(form['spectrum'].get(CH['signal_alarm'], 0.0)) >= Config.SIG_ALARM_MIN_W

    def _sig_in_panic(self):
        return (float(getattr(self, 'alarm_level', 0.0)) > Config.SIG_INSTINCT_ALARM
                or float(getattr(self, 'energy', 1.0)) < Config.SIG_INSTINCT_ENERGY)

    def _sig_alarm_form(self, ctx):
        alarms = [f for f in self.signal_forms if self._sig_is_alarm(f)]
        if alarms:
            return max(alarms, key=lambda f: (_sg_value(f, ctx), f['spectrum'].get(CH['signal_alarm'], 0.0)))
        f = self._sig_new_form({CH['signal_alarm']: 1.0}, 'panic')
        return f if self._sig_add(f, 'panic') else None

    def _sig_alarm_heard(self, tick):
        for (te, form, _s, _src, _ctx, state, _cat) in reversed(self._sig_recent):
            if tick - te > Config.SIG_ADRENALINE_WINDOW:
                break
            if state in ('recognized', 'partial') and self._sig_is_alarm(form):
                return True
        return False

    def _sig_pick(self, ctx):
        if Config.SIG_INSTINCT and self._sig_in_panic():
            f = self._sig_alarm_form(ctx)
            if f is not None:
                if self.world is not None:
                    _sg_book(self.world).st['instinct_alarm'] += 1
                return f
        if not self.signal_forms:
            spec = self._sig_proto()
            if spec is None:
                return None
            f = self._sig_new_form(spec, 'proto')
            return f if self._sig_add(f, 'proto') else None
        return max(self.signal_forms, key=lambda f: _sg_value(f, ctx))

    def _sig_pick_reply(self, ctx, heard_label):
        if not self.signal_forms:
            return None
        key = _sg_reply_key(ctx)
        def score(f):
            rec = (f.get('reply_exp') or {}).get(key) or {}
            if int(rec.get('n', 0)) >= Config.SIG_REPLY_MIN_N:
                return float(np.tanh(float(rec.get('outcome', 0.0)) / 3.0))
            return _sg_value(f, ctx)
        best = max(self.signal_forms, key=score)
        rec_b = (best.get('reply_exp') or {}).get(key) or {}
        _sg_book(self.world).st['reply_pick_learned' if int(rec_b.get('n', 0)) >= Config.SIG_REPLY_MIN_N
                                else 'reply_pick_default'] += 1
        return best

    def _sig_next_msg_id(self):
        self._sig_msg_seq = int(getattr(self, '_sig_msg_seq', 0)) + 1
        return (self.id, self._sig_msg_seq)

    def _sig_emit(self, field, form, t, ctx, reply_to=None, pending_kind='emit', heard_label=None):
        if form is None or not self.cells or not Config.SIG_ENABLE:
            return False
        now = int(t)
        if now - self._sig_last_emit_t < Config.SIG_EMIT_COOLDOWN:
            return False
        bk = _sg_book(self.world)
        cost = float(Config.SIG_EMIT_COST)
        if float(self.energy) < cost + Config.SIG_EMIT_RESERVE:
            bk.st['emit_no_energy'] += 1
            return False
        self._x2_change('X2.signal_emit', float(self.energy) - cost)
        bk.st['emit_energy_x1000'] += int(round(cost * 1000))
        self._sig_last_emit_t = now
        self._sig_emit_until = now + Config.SIG_EMIT_DURATION
        self._sig_emit_spec = dict(form['spectrum'])
        form['uses'] = int(form.get('uses', 0)) + 1
        cx, cy = self.get_center()
        W = Config.WORLD_SIZE
        message_id = self._sig_next_msg_id()
        bk.tx.append((now, int(cx) % W, int(cy) % W, self.id, dict(form['spectrum']), form, ctx,
                      _sg_action(self), message_id, reply_to))
        bk.st['emits'] += 1
        bk.st['reply_emits' if pending_kind == 'reply' else 'ordinary_emits'] += 1
        if pending_kind == 'reply':
            other = reply_to[0] if isinstance(reply_to, tuple) and reply_to else None
            bk.pend.append({'kind': 'reply', 'owner': self.id, 'other': other, 't0': now, 'form': form,
                            'ctx': ctx, 'heard_label': str(heard_label), 'action': _sg_action(self),
                            'message_id': message_id, 'reply_to': reply_to,
                            'own': {'m': _sg_mood(self), 'en': float(self.energy),
                                    'tr': _sg_trust(self, other) if other is not None else 0.0}})
        return True

    def _sig_reply(self, field, t, ent, ctx):
        if not self.signal_forms or ent is None or len(ent) < 10:
            return False
        if ent[9] is not None:
            return False
        msg_id = ent[8]
        if msg_id is None:
            return False
        bk = _sg_book(self.world)
        if msg_id in bk.replied_ids:
            return False
        heard_form = ent[5]
        heard_label = heard_form.get('label') if isinstance(heard_form, dict) else None
        if not heard_label:
            return False
        form = self._sig_pick_reply(ctx, str(heard_label))
        if form is None:
            return False
        bk.st['reply_attempts'] += 1
        if not self._sig_emit(field, form, t, ctx, reply_to=msg_id, pending_kind='reply', heard_label=str(heard_label)):
            bk.st['reply_emit_blocked'] += 1
            return False
        bk.replied_ids.add(msg_id)
        return True

    def _sig_cry(self, field, t):
        if not Config.SIG_ENABLE or not self.cells or self.world is None:
            return False
        ctx = _sg_ctx(self)
        form = self._sig_alarm_form(ctx)
        if form is None:
            return False
        bk = _sg_book(self.world)
        now = int(t)
        W = Config.WORLD_SIZE
        self._sig_last_emit_t = now
        self._sig_emit_until = now + Config.SIG_EMIT_DURATION
        self._sig_emit_spec = dict(form['spectrum'])
        form['uses'] = int(form.get('uses', 0)) + 1
        cx, cy = self.get_center()
        bk.tx.append((now, int(cx) % W, int(cy) % W, self.id, dict(form['spectrum']), form, ctx, _sg_action(self),
                      self._sig_next_msg_id(), None))
        bk.st['emits'] += 1
        bk.st['cry_emit'] += 1
        bk.pend.append({'kind': 'emit', 'owner': self.id, 'other': None, 't0': now, 'form': form, 'ctx': ctx,
                        'action': _sg_action(self), 'cat': 'cry', 'w': 1.0,
                        'own': {'m': _sg_mood(self), 'en': float(self.energy), 'tr': 0.0}})
        return True

    def _sig_apply_emission(self, field, t):
        if t >= self._sig_emit_until or self._sig_emit_spec is None or not self.cells:
            return
        for (x, y) in self.cells:
            for ch, v in self._sig_emit_spec.items():
                field[x, y, ch] = min(1.0, field[x, y, ch] + v * Config.SIG_EMIT_GAIN)

    def exchange_meanings(self, other, field, t):
        sim = super().exchange_meanings(other, field, t)
        if not Config.SIG_ENABLE or not Config.DIALOGUE_V2 or self.world is None:
            return sim
        try:
            ctx = _sg_ctx(self)
            form = self._sig_pick(ctx)
            if form is not None and self._sig_emit(field, form, t, ctx):
                _sg_book(self.world).pend.append({
                    'kind': 'emit', 'owner': self.id, 'other': other.id, 't0': int(t), 'form': form, 'ctx': ctx,
                    'action': _sg_action(self), 'cat': 'emit', 'w': 1.0,
                    'own': {'m': _sg_mood(self), 'en': float(self.energy), 'tr': _sg_trust(self, other.id)}})
        except Exception as e:
            _sg_err(_sg_book(self.world), 'emit', e)
        return sim

    def _sig_provenance(self, book, cx, cy, t, obs):
        W = Config.WORLD_SIZE
        r2 = (Config.SIG_LISTEN_RADIUS + Config.SIG_TX_PAD) ** 2
        own, foreign, ent = False, {}, None
        for e in reversed(book.tx):
            if t - e[0] > Config.SIG_TX_WINDOW:
                break
            if e[0] > t or _sg_dist2(cx, cy, e[1], e[2], W) > r2 or _sg_cos(e[4], obs) < Config.SIG_TX_COS:
                continue
            if Config.ENABLE_CENTER_WALL and ((int(e[1]) < Config.CENTER_WALL_X) != (int(cx) < Config.CENTER_WALL_X)):
                continue
            if e[3] == self.id:
                own = True
            else:
                foreign.setdefault(e[3], e)
        if (own and foreign) or len(foreign) > 1:
            return 'mixed', None, None
        if foreign:
            pid, ent = next(iter(foreign.items()))
            return 'foreign', pid, ent
        return ('own' if own else 'unknown'), None, None

    def _sig_listen(self, field, t):
        if not self.cells or self.world is None or not Config.SIG_ENABLE:
            return
        bk = _sg_book(self.world); st = bk.st; now = int(t)
        W = Config.WORLD_SIZE
        cx, cy = self.get_center(); cx, cy = int(cx) % W, int(cy) % W
        obs = _sg_heard(bk, field, now, cx, cy)
        if obs is None:
            return
        st['listens'] += 1
        cat, src, ent = self._sig_provenance(bk, cx, cy, now, obs)
        st['prov_' + cat] += 1
        echo_tag = bool(cat == 'unknown' and bk.echo_spec is not None and _sg_cos(bk.echo_spec, obs) >= Config.SIG_TX_COS)
        if echo_tag:
            st['echo_heard'] += 1
        scored = sorted(((*_sg_match(f['spectrum'], obs), f) for f in self.signal_forms), key=lambda x: -x[0])
        state, form, strength = 'none', None, 0.0
        if scored:
            bcos, bnd, bf = scored[0]
            if bcos >= Config.SIG_MATCH_COS and bnd <= Config.SIG_MATCH_NORM:
                state, form, strength = 'recognized', bf, bcos
                if len(scored) > 1 and scored[1][0] >= Config.SIG_MATCH_COS and bcos - scored[1][0] <= Config.SIG_DOUBT_MARGIN:
                    state, strength = 'doubt', 0.5 * bcos
            elif bcos >= Config.SIG_PARTIAL_COS:
                state, form, strength = 'partial', bf, bcos
        if state != 'none' and phi_hash(self.id, now, 77001) < Config.SIG_STRESS_MISS * _sg_stress(self):
            st['rec_stress_missed'] += 1
            state, form = 'stress_missed', None
        st['rec_' + state] += 1
        st['rec_' + state + '_' + cat] += 1
        if echo_tag:
            st['echo_' + state] += 1
        if cat == 'foreign' and ent is not None and bk.place_msgs:
            _ph = getattr(self, '_pn3_on_place_heard', None)
            if _ph is not None:
                try:
                    _ph(bk, ent, state, now)
                except Exception as e:
                    _sg_err(bk, 'place_heard', e)
        if cat == 'foreign' and ent is not None and getattr(bk, 'hunt_calls', None):
            _hh = getattr(self, '_hc_on_hunt_heard', None)
            if _hh is not None:
                try:
                    _hh(bk, ent, state, now)
                except Exception as e:
                    _sg_err(bk, 'hunt_heard', e)
        if state == 'none':
            if phi_hash(self.id, self.age, 66001) < Config.SIG_NEW_PROB:
                if self._sig_add(self._sig_new_form(obs, 'heard'), 'heard'):
                    st['adopts'] += 1
            return
        if state in ('partial', 'doubt'):
            self.spirit_gap = float(min(Config.SIG_GAP_CAP, float(self.spirit_gap) + Config.SIG_GAP_BUMP))
            st['gap_bumps'] += 1
        if form is None:
            return
        if cat == 'own':
            st['match_own_emit'] += 1
            return
        if Config.SIG_ADRENALINE and state in ('recognized', 'partial') and self._sig_is_alarm(form):
            self.cognitive_tension = min(Config.MAX_METRIC, float(self.cognitive_tension) + Config.SIG_SHOCK)
            st['adrenaline_shock'] += 1
        ctx = _sg_ctx(self)
        self._sig_recent.append((now, form, strength, src, ctx, state, cat))
        if self._sig_open < Config.SIG_MAX_OPEN:
            self._sig_open += 1
            w = Config.SIG_LISTEN_W if cat == 'foreign' else Config.SIG_LISTEN_W_BLUR
            bk.pend.append({
                'kind': 'listen', 'owner': self.id, 'other': src, 't0': now, 'form': form, 'ctx': ctx,
                'action': _sg_action(self), 'cat': cat, 'w': w,
                'sender': (ent[5], ent[6], ent[7], ent[3]) if ent is not None else None,
                'own': {'m': _sg_mood(self), 'en': float(self.energy),
                        'tr': _sg_trust(self, src) if src is not None else 0.0}})
        else:
            st['listen_window_full'] += 1
        if cat == 'foreign' and state in ('recognized', 'partial') and ent is not None:
            self._sig_reply(field, now, ent, ctx)

    def _sig_expect(self, t, goal):
        action = str(goal.get('type', '*'))
        acc = {c: 0.0 for c in SIG_COMPS}; total = 0.0
        for (te, form, strength, src, ctx, state, cat) in self._sig_recent:
            age = t - te
            if age < 0 or age > Config.SIG_RECENT_TICKS:
                continue
            rec = _sg_exp_get(form, ctx, action)
            if rec is None:
                continue
            conf = rec['n'] / (rec['n'] + Config.SIG_CONF_K)
            srcw = 1.0 if src is None else 0.5 + 0.5 * float(np.clip(_sg_trust(self, src), 0.0, 1.0))
            w = float(strength) * conf * srcw * Config.SIG_RECENT_DECAY ** age
            for c in SIG_COMPS:
                acc[c] += w * rec[c]
            total += w
        if total <= 1e-12:
            return None
        return ({c: float(np.clip(acc[c] / total, -Config.SIG_Z_CLIP, Config.SIG_Z_CLIP)) for c in SIG_COMPS},
                float(total / (total + Config.SIG_REL_K)))

    def _sig_danger(self, t):
        best = 0.0
        for e in self._sig_recent:
            age = t - e[0]
            if 0 <= age <= Config.SIG_RECENT_TICKS and e[1].get('dn', 0) > 0:
                best = max(best, min(1.0, e[1]['dn'] / 20.0) * Config.SIG_RECENT_DECAY ** age)
        return best

    def _goal_rel(self, goal):
        rec = self._goal_models.get(str(goal.get('type', '')))
        ref = self._goal_ref
        if rec is None or rec['n'] < Config.SIG_MIN_N or ref['n'] < 2 * max(2, Config.SIG_MIN_N):
            return None, 0.0
        zs = [float(np.clip((rec[c] - ref[c]) / max(ref['s_' + c], Config.SIG_GOAL_FLOORS[c]),
                            -Config.SIG_Z_CLIP, Config.SIG_Z_CLIP)) for c in SIG_COMPS]
        return float(np.mean(zs)), float(rec['n'] / (rec['n'] + Config.SIG_CONF_K))

    def _goal_learn(self, gtype, d):
        ref = self._goal_ref
        ref['n'] += 1
        a = max(Config.SIG_GOAL_LR, 1.0 / ref['n'])
        for c in SIG_COMPS:
            ref['s_' + c] += a * (abs(d[c] - ref[c]) - ref['s_' + c])
            ref[c] += a * (d[c] - ref[c])
        rec = self._goal_models.setdefault(gtype, {'n': 0, 'm': 0.0, 'tr': 0.0, 'en': 0.0})
        rec['n'] += 1
        b = max(Config.SIG_GOAL_LR, 1.0 / rec['n'])
        for c in SIG_COMPS:
            rec[c] += b * (d[c] - rec[c])

    def _goal_close(self, tick):
        snap, self._goal_snap = self._goal_snap, None
        if snap is None:
            return
        dt = int(tick) - snap['t']
        if dt <= 0 or dt > Config.SIG_GOAL_MAX_GAP:
            return
        trn = self.trust_ledger.entries
        common = [k for k in snap['tr'] if k in trn]
        dtr = (sum(trn[k] - snap['tr'][k] for k in common) / len(common)) if common else 0.0
        self._goal_learn(snap['type'], {'m': (_sg_mood(self) - snap['m']) / dt, 'tr': dtr / dt,
                                        'en': (float(self.energy) - snap['en']) / dt})
        _sg_book(self.world).st['goal_updates'] += 1

    def _goal_open(self, tick):
        if isinstance(self.intent, dict):
            self._goal_snap = {'t': int(tick), 'type': str(self.intent.get('type', '*')), 'm': _sg_mood(self),
                               'en': float(self.energy), 'tr': dict(self.trust_ledger.entries)}
        else:
            self._goal_snap = None

    def _goal_shift(self, goal, t):
        ex = self._sig_expect(t, goal)
        value = 0.0
        if ex is not None:
            vec, rel = ex
            value = rel * math.tanh((vec['m'] + vec['tr'] + vec['en']) / 3.0)
        zg, cg = self._goal_rel(goal)
        if zg is not None:
            tz = math.tanh(zg)
            value += Config.SIG_GOAL_PRIOR_W * cg * tz
            if tz < 0:
                value += Config.SIG_DEATH_W * self._sig_danger(t) * cg * tz
        return float(Config.SIG_CHOICE_GAIN * value)

    def update_intent(self):
        if self.world is None or not Config.SIG_ENABLE:
            return super().update_intent()
        tick = int(self.age)
        if self._goal_tick == tick:
            return super().update_intent()
        self._goal_tick = tick
        bk = _sg_book(self.world); st = bk.st
        if Config.SIG_ADRENALINE:
            if self._sig_alarm_heard(tick):
                self.last_intent_switch_age = min(self.last_intent_switch_age,
                                                  tick - int(Config.INTENT_SWITCH_COOLDOWN))
                if not any(g.get('_source') == 'adrenaline_alarm' for g in self.goals):
                    self.goals.append({"type": "cooperate", "priority": float(Config.SIG_ADRENALINE_PRIORITY),
                                       "target": None, "age": 0, "persistence": int(Config.SIG_ADRENALINE_PERSIST),
                                       "_source": "adrenaline_alarm"})
                    st['adrenaline_goal_injected'] += 1
                    self._log_event("adrenaline_goal_injected")
            elif (isinstance(self.intent, dict) and self.intent.get('_source') == 'adrenaline_alarm'
                  and not any(g is self.intent for g in self.goals)):
                self.intent = None
                st['adrenaline_release'] += 1
        try:
            self._goal_close(tick)
        except Exception as e:
            _sg_err(bk, 'goal_close', e)
        shifts, raw, raw_clipped, centered, top0, top1 = {}, {}, {}, {}, None, None
        priority_before = {}
        commitment_before = float(getattr(self, 'intent_commitment', 0.0))
        prev = self.intent
        if Config.SIG_CHOICE and self.goals:
            try:
                regular = [g for g in self.goals if g.get('_source') != 'adrenaline_alarm']
                raw = {id(g): float(self._goal_shift(g, tick)) for g in regular}
                raw_clipped = {k: float(np.clip(v, -float(Config.SIG_CHOICE_MAX), float(Config.SIG_CHOICE_MAX)))
                               for k, v in raw.items()}
                if raw_clipped and any(v != 0.0 for v in raw_clipped.values()):
                    mean = sum(raw_clipped.values()) / len(raw_clipped)
                    centered = {k: float(v - mean) for k, v in raw_clipped.items()}
                    lim = float(Config.SIG_CHOICE_MAX)
                    shifts = {k: float(np.clip(v, -lim, lim)) for k, v in centered.items() if abs(v) > 1e-9}
                if shifts:
                    top0 = max(self.goals, key=lambda g: (g.get('priority', 0.0), -g.get('age', 0)))
                    top1 = max(self.goals, key=lambda g: (g.get('priority', 0.0) + shifts.get(id(g), 0.0), -g.get('age', 0)))
            except Exception as e:
                _sg_err(bk, 'choice', e)
                shifts, raw, raw_clipped, centered = {}, {}, {}, {}
        for g in self.goals:
            if id(g) in shifts:
                priority_before[id(g)] = float(g.get('priority', 0.0))
                g['priority'] += shifts[id(g)]
        try:
            result = super().update_intent()
        finally:
            for g in self.goals:
                if id(g) in shifts:
                    g['priority'] -= shifts[id(g)]
        commitment_after = float(getattr(self, 'intent_commitment', 0.0))
        if (self.intent is not prev and isinstance(self.intent, dict)
                and self.intent.get('_source') == 'adrenaline_alarm'):
            st['adrenaline_switch'] += 1
        if shifts:
            st['choice_signal'] += 1
            st['choice_bias_abs_x1000'] += int(1000 * sum(abs(v) for v in shifts.values()) / len(shifts))
            switched = self.intent is not prev
            if switched:
                if self.intent is not top0:
                    st['choice_influenced'] += 1
                    category = 'influenced'
                else:
                    st['choice_noflip'] += 1
                    st['choice_noflip_base_won'] += 1
                    category = 'noflip_base_won'
            elif prev is not None and top0 is not prev and top1 is prev:
                st['choice_blocked'] += 1
                st['choice_signal_rescued_prev'] += 1
                category = 'signal_rescued_prev'
            elif prev is not None and top1 is prev and top0 is prev and shifts.get(id(prev), 0.0) > 0:
                st['choice_reinforced'] += 1
                category = 'reinforced'
            else:
                st['choice_noflip'] += 1
                if prev is not None and top0 is not prev and top1 is not prev and top0 is not top1:
                    st['choice_noflip_neither'] += 1
                    category = 'noflip_neither'
                elif prev is not None and top0 is prev and top1 is prev and shifts.get(id(prev), 0.0) <= 0:
                    st['choice_noflip_prev_held'] += 1
                    category = 'noflip_prev_held'
                elif prev is not None and top0 is not prev and top1 is not prev and top0 is top1:
                    st['choice_noflip_both_disagree_prev_stayed'] += 1
                    category = 'noflip_both_disagree_prev_stayed'
                else:
                    st['choice_noflip_other'] += 1
                    category = 'noflip_other'
            if (not switched and prev is not None and top1 is not None and top1 is not prev):
                cooldown_elapsed = int(self.age) - int(getattr(self, 'last_intent_switch_age', -int(Config.INTENT_SWITCH_COOLDOWN)))
                cooldown_required = int(Config.INTENT_SWITCH_COOLDOWN)
                if cooldown_elapsed < cooldown_required:
                    st['choice_no_switch_cooldown_candidate'] += 1
                else:
                    confidence_gap = float(getattr(self, '_meta_surprise', 0.0) + getattr(self, '_implicit_self_prediction_error', 0.0))
                    base_mult = max(Config.INTENT_SWITCH_MULT_MIN,
                                    Config.INTENT_SWITCH_MULT - min(0.4, confidence_gap * 2.0))
                    anchor_add = 0.0
                    if getattr(Config, 'ENABLE_IDENTITY_THREAD', False) and getattr(self, '_identity_anchors', None):
                        denom = max(1.0, float(Config.IDENTITY_THREAD_MAX_ANCHORS))
                        anchor_strength = sum(a['weight'] for a in self._identity_anchors) / denom
                        anchor_add = min(0.3, anchor_strength * 0.3)
                    effective_mult = base_mult + anchor_add
                    best_priority = float(top1.get('priority', 0.0)) + float(shifts.get(id(top1), 0.0))
                    if best_priority <= commitment_after * effective_mult:
                        st['choice_no_switch_barrier_candidate'] += 1
            trace = getattr(bk, 'choice_debug_events', None)
            if trace is None:
                trace = []; bk.choice_debug_events = trace
            if switched or len(trace) < 40:
                try:
                    goal_by_id = {id(g): g for g in self.goals}
                    trace.append({
                        't': int(tick), 'pid': int(self.id), 'category': category,
                        'prev': prev.get('type') if isinstance(prev, dict) else None,
                        'intent_after': self.intent.get('type') if isinstance(self.intent, dict) else None,
                        'top0': top0.get('type') if isinstance(top0, dict) else None,
                        'top1': top1.get('type') if isinstance(top1, dict) else None,
                        'raw_preclip': {str(goal_by_id[k].get('type')): float(v) for k, v in raw.items() if k in goal_by_id},
                        'raw_clipped': {str(goal_by_id[k].get('type')): float(v) for k, v in raw_clipped.items() if k in goal_by_id},
                        'centered': {str(goal_by_id[k].get('type')): float(v) for k, v in centered.items() if k in goal_by_id},
                        'applied': {str(goal_by_id[k].get('type')): float(v) for k, v in shifts.items() if k in goal_by_id},
                        'priority_before': {str(goal_by_id[k].get('type')): float(v) for k, v in priority_before.items() if k in goal_by_id},
                        'priority_after_shift': {str(goal_by_id[k].get('type')): float(priority_before[k] + shifts[k]) for k in shifts if k in goal_by_id},
                        'commitment_before': commitment_before, 'commitment_after': commitment_after,
                        'confidence_gap': float(getattr(self, '_meta_surprise', 0.0) + getattr(self, '_implicit_self_prediction_error', 0.0)),
                        'switch_mult_configured': float(Config.INTENT_SWITCH_MULT),
                        'switch_mult_floor': float(Config.INTENT_SWITCH_MULT_MIN),
                        'switch_mult_effective': float(max(Config.INTENT_SWITCH_MULT_MIN, Config.INTENT_SWITCH_MULT - min(0.4, (getattr(self, '_meta_surprise', 0.0) + getattr(self, '_implicit_self_prediction_error', 0.0)) * 2.0)) + (min(0.3, (sum(a['weight'] for a in self._identity_anchors) / (Config.IDENTITY_THREAD_MAX_ANCHORS * 1.0)) * 0.3) if getattr(Config, 'ENABLE_IDENTITY_THREAD', False) and getattr(self, '_identity_anchors', None) else 0.0)),
                        'cooldown_elapsed': int(self.age) - int(getattr(self, 'last_intent_switch_age', -int(Config.INTENT_SWITCH_COOLDOWN))),
                        'cooldown_required': int(Config.INTENT_SWITCH_COOLDOWN),
                        'last_switch_age': int(getattr(self, 'last_intent_switch_age', -int(Config.INTENT_SWITCH_COOLDOWN))),
                    })
                except Exception:
                    pass
        try:
            self._goal_open(tick)
        except Exception as e:
            _sg_err(bk, 'goal_open', e)
        return result

    def _sig_place_send(self, field, t, ev):
        if self.world is None or not Config.SIG_ENABLE or not self.cells:
            return None, 'sig_off'
        now = int(t)
        if now - self._sig_last_emit_t < Config.SIG_EMIT_COOLDOWN:
            return None, 'cooldown'
        if float(self.energy) < float(Config.SIG_EMIT_COST) + float(Config.SIG_EMIT_RESERVE):
            return None, 'no_energy'
        if self._sig_in_panic():
            return None, 'panic'
        ctx = _sg_ctx(self)
        form = self._sig_pick(ctx)
        if form is None:
            return None, 'no_form'
        if self._sig_is_alarm(form):
            return None, 'alarm_form'
        if not self._sig_emit(field, form, t, ctx):
            return None, 'emit_refused'
        bk = _sg_book(self.world)
        mid = (self.id, int(self._sig_msg_seq))
        msg = PlaceMessage(message_id=mid, sender_id=int(self.id), recipient_id=None, event_id=int(ev.event_id), kind=str(ev.kind),
                           place_id=ev.place_id, confidence=float(ev.confidence), t_sent=now,
                           fp_obs=tuple(ev.fp_obs), fp_ref=(tuple(ev.fp_ref) if ev.fp_ref is not None else None), world_xy=tuple(ev.world_xy))
        bk.place_msgs[mid] = msg
        bk.pm['sent'] += 1
        bk.pm['sent_' + msg.kind] += 1
        bk.pend.append({'kind': 'emit', 'owner': self.id, 'other': None, 't0': now, 'form': form, 'ctx': ctx,
                        'action': _sg_action(self), 'cat': 'emit', 'w': 1.0,
                        'own': {'m': _sg_mood(self), 'en': float(self.energy), 'tr': 0.0}})
        return msg, 'ok'

    def _sig_hunt_send(self, field, t, fx, fy, observed_at, confidence):
        if self.world is None or not Config.SIG_ENABLE or not Config.HUNT_ENABLE or not self.cells:
            return None, 'sig_off'
        now = int(t)
        if now - self._sig_last_emit_t < Config.SIG_EMIT_COOLDOWN:
            return None, 'cooldown'
        if float(self.energy) < float(Config.SIG_EMIT_COST) + float(Config.SIG_EMIT_RESERVE):
            return None, 'no_energy'
        if self._sig_in_panic():
            return None, 'panic'
        ctx = _sg_ctx(self)
        form = self._sig_pick(ctx)
        if form is None:
            return None, 'no_form'
        if self._sig_is_alarm(form):
            return None, 'alarm_form'
        if not self._sig_emit(field, form, t, ctx):
            return None, 'emit_refused'
        bk = _sg_book(self.world)
        if not hasattr(bk, 'hunt_calls'):
            bk.hunt_calls = {}; bk.hc = Counter()
        mid = (self.id, int(self._sig_msg_seq))
        W = Config.WORLD_SIZE
        hc = HuntCall(message_id=mid, sender_id=int(self.id), fx=int(fx) % W, fy=int(fy) % W, observed_at=int(observed_at),
                      confidence=float(max(0.0, min(1.0, confidence))), t_sent=now)
        bk.hunt_calls[mid] = hc
        bk.hc['sent'] += 1
        bk.pend.append({'kind': 'emit', 'owner': self.id, 'other': None, 't0': now, 'form': form, 'ctx': ctx,
                        'action': _sg_action(self), 'cat': 'emit', 'w': 1.0,
                        'own': {'m': _sg_mood(self), 'en': float(self.energy), 'tr': 0.0}})
        return hc, 'ok'

    def dump_extra(self):
        extra = super().dump_extra() if hasattr(super(), 'dump_extra') else {}
        try:
            extra['sig_forms'] = [
                {'spectrum': {str(k): float(v) for k, v in f['spectrum'].items()}, 'uses': int(f.get('uses', 0)),
                 'label': str(f['label']), 'born_t': int(f.get('born_t', 0)), 'vh': int(f.get('vh', 0)),
                 'vm': int(f.get('vm', 0)), 'cpt': int(f.get('cpt', 0)), 'dn': int(f.get('dn', 0)),
                 'exp': [[k, {c: float(r[c]) for c in SIG_COMPS}, int(r['n'])] for k, r in (f.get('exp') or {}).items()],
                 'reply_exp': [[k, float(r.get('outcome', 0.0)), int(r.get('n', 0)), int(r.get('n_obs', 0))] for k, r in (f.get('reply_exp') or {}).items()],
                 'parent_label': f.get('parent_label'), 'recombined_from': list(f.get('recombined_from') or [])}
                for f in self.signal_forms[-Config.SIG_MAX_FORMS:]]
            extra['sig_seq'] = int(self._sig_seq)
            extra['sig_msg_seq'] = int(getattr(self, '_sig_msg_seq', 0))
            extra['sig_goal_models'] = {str(g): {**{c: float(v[c]) for c in SIG_COMPS}, 'n': int(v['n'])}
                                        for g, v in self._goal_models.items()}
            extra['sig_goal_ref'] = {k: (int(v) if k == 'n' else float(v)) for k, v in self._goal_ref.items()}
            extra['_lang_gene'] = [float(x) for x in getattr(self, '_lang_gene', [])]
        except Exception as e:
            raise RuntimeError(f"SIGNAL: не удалось сериализовать агента {getattr(self, 'id', '?')}") from e
        return extra

    def load_extra(self, data):
        if hasattr(super(), 'load_extra'):
            super().load_extra(data)
        try:
            self._sig_init()
            self.signal_forms = []
            raw = data.get('sig_forms')
            legacy = raw is None
            for f in (raw if raw is not None else data.get('signal_forms', [])):
                if not isinstance(f, dict):
                    continue
                spec = {}
                for k, v in (f.get('spectrum') or {}).items():
                    try:
                        spec[int(k)] = float(v)
                    except (TypeError, ValueError):
                        pass
                if not spec:
                    continue
                nf = {'spectrum': spec, 'uses': int(f.get('uses', 0)), 'born_t': int(f.get('born_t', 0)),
                      'label': str(f.get('label', '')) or f'legacy_{self.id}', 'exp': {}, 'vh': int(f.get('vh', 0)),
                      'vm': int(f.get('vm', 0)), 'cpt': int(f.get('cpt', 0)), 'dn': int(f.get('dn', 0)),
                      'reply_exp': {}, 'parent_label': f.get('parent_label'),
                      'recombined_from': list(f.get('recombined_from') or [])}
                for item in (f.get('exp') or []):
                    try:
                        k, vals, n = item
                        nf['exp'][str(k)] = {**{c: float(vals.get(c, 0.0)) for c in SIG_COMPS}, 'n': int(n)}
                    except (TypeError, ValueError):
                        pass
                for item in (f.get('reply_exp') or []):
                    try:
                        k, outcome, rn, nobs = item
                        nf['reply_exp'][str(k)] = {'outcome': float(outcome), 'n': int(rn), 'n_obs': int(nobs)}
                    except (TypeError, ValueError):
                        pass
                self.signal_forms.append(nf)
            self._sig_msg_seq = int(data.get('sig_msg_seq', 0))
            if legacy:
                fs = data.get('sc_form_state_v12') or []
                if len(fs) == len(self.signal_forms):
                    for nf, ser in zip(self.signal_forms, fs):
                        for item in ((ser or {}).get('exp') or []):
                            try:
                                k, vals, n = item
                                if len(k) == 3 and k[2] == 'personal':
                                    nf['exp'][f"{k[0]}|{k[1]}"] = {**{c: float(vals.get(c, 0.0)) for c in SIG_COMPS}, 'n': int(n)}
                            except (TypeError, ValueError):
                                pass
            self._sig_seq = int(data.get('sig_seq', len(self.signal_forms)))
            gm = data.get('sig_goal_models') or data.get('sc_goal_models') or {}
            self._goal_models = {str(g): {**{c: float(v.get(c, 0.0)) for c in SIG_COMPS}, 'n': int(v.get('n', 0))}
                                 for g, v in gm.items()}
            ref = _sg_new_ref()
            for k, v in (data.get('sig_goal_ref') or data.get('sc_goal_ref') or {}).items():
                if k in ref:
                    ref[k] = int(v) if k == 'n' else float(v)
            self._goal_ref = ref
            self._goal_snap = None
            g = data.get('_lang_gene')
            if g:
                try:
                    self._lang_gene = np.array([float(x) for x in g], dtype=np.float32)
                except (TypeError, ValueError):
                    pass
        except Exception as e:
            print(f"SIGNAL: ошибка загрузки форм: {e!r}")


def _sg_credit(book, owner, form, ctx, action, z, w, kind):
    if form is None or z is None:
        return 0
    _sg_exp_add(form, ctx, action, z, w)
    v = _sg_verdict(z)
    book.st['credit_' + kind] += 1
    book.st[f'verdict_{kind}_' + ('helped' if v > 0 else ('harmed' if v < 0 else 'none'))] += 1
    if v != 0:
        form['vh' if v > 0 else 'vm'] += 1
        if not form.get('cpt') and max(form['vh'], form['vm']) >= Config.SIG_CONCEPT_MIN:
            form['cpt'] = 1 if form['vh'] >= form['vm'] else -1
            book.st['concepts_' + ('help' if form['cpt'] > 0 else 'harm')] += 1
            try:
                owner._log_event('signal_concept', label=str(form['label']), sign=int(form['cpt']))
            except Exception:
                pass
    return v


def _sg_reply_key(ctx):
    return str(ctx)


def _sg_credit_reply(book, form, ctx, heard_label, z):
    if form is None or z is None:
        return False
    key = _sg_reply_key(ctx)
    rec = (form.setdefault('reply_exp', {}).setdefault(key,
           {'n': 0, 'n_obs': 0, 'outcome': 0.0}))
    rec['n'] = int(rec.get('n', 0)) + 1
    outcome = 0.5 * float(z['m']) + 0.3 * float(z['tr']) + 0.2 * float(z['en'])
    rec['outcome'] += (outcome - float(rec.get('outcome', 0.0))) / rec['n']
    book.st['reply_credit'] += 1
    return True


def _sg_settle(eng, book, pend, by_id):
    owner = by_id.get(pend['owner'])
    if owner is None:
        book.st['win_owner_gone'] += 1
        return
    st = book.st
    if pend['kind'] == 'listen':
        owner._sig_open = max(0, owner._sig_open - 1)
    other = by_id.get(pend['other']) if pend['other'] is not None else None
    s0 = pend['own']
    d = {'m': _sg_mood(owner) - s0['m'], 'en': float(owner.energy) - s0['en'],
         'tr': (_sg_trust(owner, pend['other']) - s0['tr']) if pend['other'] is not None else 0.0}
    if pend['kind'] == 'reply':
        key = 'R|' + _sg_reply_key(pend['ctx'])
        z = _sg_adv(book, d, key)
        rec = pend['form'].setdefault('reply_exp', {}).setdefault(
            _sg_reply_key(pend['ctx']), {'n': 0, 'n_obs': 0, 'outcome': 0.0})
        rec['n_obs'] = int(rec.get('n_obs', 0)) + 1
        st['reply_settled'] += 1
        if z is None:
            st['reply_z_none'] += 1
            st['win_warmup_reply'] += 1
            return
        _sg_credit_reply(book, pend['form'], pend['ctx'], pend['heard_label'], z)
        return
    if pend['kind'] == 'emit':
        key = ('C|' if pend.get('cat') == 'cry' else 'E|') + pend['ctx']
    else:
        key = 'L|' + pend['ctx']
    z = _sg_adv(book, d, key)
    if z is None:
        st['win_warmup_' + pend['kind']] += 1
        return
    if pend['kind'] == 'emit':
        v = _sg_credit(book, owner, pend['form'], pend['ctx'], pend['action'], z, 1.0, 'emitter')
        st['cons_' + ('helped' if v > 0 else ('harmed' if v < 0 else 'none'))] += 1
        return
    v = _sg_credit(book, owner, pend['form'], pend['ctx'], pend['action'], z, pend['w'], 'listener')
    snd = pend.get('sender')
    if snd is not None and pend['cat'] == 'foreign':
        sform, sctx, sact, spid = snd
        sender = by_id.get(spid)
        if sender is not None and sform in sender.signal_forms:
            _sg_credit(book, sender, sform, sctx, sact, z, Config.SIG_SENDER_W, 'sender')
        if v != 0:
            owner.trust_ledger.update(spid, 'helpful' if v > 0 else 'harmful',
                                      multiplier=Config.SIG_TRUST_UP if v > 0 else Config.SIG_TRUST_DOWN)
            st['trust_' + ('up' if v > 0 else 'down')] += 1
    else:
        st['no_sender_credit'] += 1


def _sg_witness(eng, book, t):
    W = Config.WORLD_SIZE
    now = {}
    for p in eng.patterns:
        if p.alive and p.cells:
            cx, cy = p.get_center()
            now[p.id] = (int(cx) % W, int(cy) % W, getattr(p, 'role_type', None))
    deaths = [(pid, v) for pid, v in book.alive_prev.items() if pid not in now]
    book.alive_prev = now
    if not deaths:
        return
    by_id = {p.id: p for p in eng.patterns if p.alive}
    r2 = Config.SIG_WIT_RADIUS ** 2
    for pid, (x, y, role) in deaths:
        book.st['deaths_seen'] += 1
        if role == 'feral':
            continue
        for qid, qv in now.items():
            if qv[2] == 'feral' or _sg_dist2(qv[0], qv[1], x, y, W) > r2:
                continue
            q = by_id.get(qid)
            if q is None:
                continue
            for e in reversed(q._sig_recent):
                if t - e[0] > Config.SIG_WIT_WINDOW:
                    break
                e[1]['dn'] = e[1].get('dn', 0) + 1
                book.st['wit_credits'] += 1
                break


_sg_KEYS = ('emits', 'cry_emit', 'emit_no_energy', 'listens', 'adopts', 'prov_foreign', 'prov_mixed', 'prov_own', 'prov_unknown',
         'rec_recognized', 'rec_partial', 'rec_doubt', 'rec_none', 'rec_stress_missed', 'gap_bumps',
         'credit_emitter', 'credit_listener', 'credit_sender', 'trust_up', 'trust_down',
         'cons_helped', 'cons_harmed', 'cons_none',
         'choice_signal', 'choice_influenced', 'choice_noflip', 'choice_blocked', 'choice_reinforced',
         'goal_updates', 'form_displaced', 'form_full_blocked', 'concepts_help', 'concepts_harm',
         'deaths_seen', 'wit_credits',
         'ordinary_emits', 'reply_emits', 'reply_attempts', 'reply_emit_blocked', 'reply_settled', 'reply_credit', 'reply_z_none',
         'reply_pick_learned', 'reply_pick_default')


class SignalEngineMixin:
    def _post_semantic_step(self, t):
        super()._post_semantic_step(t)
        try:
            _coop_refresh(self)
        except Exception as e:
            _sg_err(_sg_book(self), 'coop', e)
        if not Config.SIG_ENABLE:
            return
        bk = _sg_book(self)
        now = int(self.age)
        try:
            _sg_echo_step(self, bk, now)
            for p in self.patterns:
                if p.alive:
                    p._sig_init()
                    p._sig_apply_emission(self.field, now)
            if now % Config.SIG_LISTEN_EVERY == 0:
                for p in self.patterns:
                    if p.alive:
                        try:
                            p._sig_listen(self.field, now)
                        except Exception as e:
                            _sg_err(bk, 'listen', e)
            if bk.pend:
                by_id = {p.id: p for p in self.patterns if p.alive}
                keep = []
                for pend in bk.pend:
                    if now - pend['t0'] < Config.SIG_DELAY:
                        keep.append(pend)
                        continue
                    try:
                        _sg_settle(self, bk, pend, by_id)
                    except Exception as e:
                        _sg_err(bk, 'settle', e)
                bk.pend = keep
            _sg_witness(self, bk, now)
        except Exception as e:
            _sg_err(bk, 'tick', e)

    def _audit_close_tick(self, t, de_x1, res_x1, x2_step0):
        super()._audit_close_tick(t, de_x1, res_x1, x2_step0)
        try:
            bk = _sg_book(self)
            row = {}
            for k in _sg_KEYS:
                row['sig_' + k] = bk.st[k] - bk.prev.get('t_' + k, 0)
                bk.prev['t_' + k] = bk.st[k]
            row['sig_in_flight'] = len(bk.pend)
            self._audit_rows[-1].update(row)
            now = int(self.age)
            if now % 50 == 0 and now > 0:
                print(_sg_line(self, bk, now), flush=True)
        except Exception as e:
            _sg_err(_sg_book(self), 'audit', e)

    def audit_summary(self):
        S = super().audit_summary()
        try:
            S['signal'] = _sg_summary(self, _sg_book(self))
        except Exception as e:
            _sg_err(_sg_book(self), 'summary', e)
        return S


def _sg_snapshot(eng):
    alive = [p for p in eng.patterns if p.alive]
    forms = [f for p in alive for f in getattr(p, 'signal_forms', ())]
    with_exp = [f for f in forms if _sg_exp_get(f, '*', '*') is not None]
    return {'agents': len(alive), 'forms': len(forms), 'forms_avg': round(len(forms) / max(1, len(alive)), 2),
            'forms_max': max((len(getattr(p, 'signal_forms', ())) for p in alive), default=0),
            'forms_with_exp': len(with_exp),
            'concepts_help': sum(1 for f in forms if f.get('cpt', 0) > 0),
            'concepts_harm': sum(1 for f in forms if f.get('cpt', 0) < 0),
            'goal_model_agents': sum(1 for p in alive if getattr(p, '_goal_models', None))}


def _sg_line(eng, bk, now):
    d = lambda k: bk.st[k] - bk.prev.get('l_' + k, 0)
    cs = d('choice_signal')
    s = _sg_snapshot(eng)
    out = (f"[SIGNAL t={now}] emit={d('emits')}(cry={d('cry_emit')},no_en={d('emit_no_energy')}) listen={d('listens')} | "
           f"rec: ok={d('rec_recognized')} part={d('rec_partial')} dbt={d('rec_doubt')} none={d('rec_none')} "
           f"smiss={d('rec_stress_missed')} | src: f={d('prov_foreign')} m={d('prov_mixed')} o={d('prov_own')} "
           f"u={d('prov_unknown')} | credit: em={d('credit_emitter')} ls={d('credit_listener')} sd={d('credit_sender')} "
           f"trust+{d('trust_up')}/-{d('trust_down')} | choice: sig={cs} infl={d('choice_influenced')} "
           f"blk={d('choice_blocked')} rnf={d('choice_reinforced')} "
           f"nosw(barrier={d('choice_no_switch_barrier_candidate')},cd={d('choice_no_switch_cooldown_candidate')}) "
           f"|b|={(d('choice_bias_abs_x1000') / 1000.0 / max(1, cs)):.3f} | "
           f"adr: inst={d('instinct_alarm')} shock={d('adrenaline_shock')} inj={d('adrenaline_goal_injected')} "
           f"sw={d('adrenaline_switch')} rel={d('adrenaline_release')} | "
           f"reply: em={d('reply_emits')} att={d('reply_attempts')} blk={d('reply_emit_blocked')} set={d('reply_settled')} "
           f"credit={d('reply_credit')} z0={d('reply_z_none')} pick(learned/default)={d('reply_pick_learned')}/{d('reply_pick_default')} "
           f"ids={len(bk.replied_ids)} | "
           f"alarm: heard={d('heard_alarm')} cd={d('heard_alarm_cooldown')} credit={d('alarm_credit')} | "
           f"forms: avg={s['forms_avg']} max={s['forms_max']} exp={s['forms_with_exp']}/{s['forms']} "
           f"cpt={s['concepts_help']}+/{s['concepts_harm']}- disp={d('form_displaced')} full={d('form_full_blocked')} | "
           f"in_flight={len(bk.pend)} err={sum(d(k) for k in bk.st if k.startswith('err_'))}")
    if bk.err:
        out += f" last_err={bk.err}"
    for k in list(bk.st):
        bk.prev['l_' + k] = bk.st[k]
    return out


def _sg_summary(eng, bk):
    st = bk.st
    snap = _sg_snapshot(eng)
    ln = sum(st['prov_' + c] for c in ('foreign', 'mixed', 'own', 'unknown')) or 1
    cons = st['cons_helped'] + st['cons_harmed'] + st['cons_none']
    gm = {}
    for p in eng.patterns:
        if not p.alive:
            continue
        ref = getattr(p, '_goal_ref', None)
        for g, m in (getattr(p, '_goal_models', {}) or {}).items():
            rel = {c: m[c] - ref[c] for c in SIG_COMPS} if ref and ref['n'] > 0 else {c: 0.0 for c in SIG_COMPS}
            gm.setdefault(g, []).append((m, rel))
    gm_out = {g: {'agents': len(v), 'mean_n': round(float(np.mean([x[0]['n'] for x in v])), 2),
                  **{'rel_' + c: round(float(np.mean([x[1][c] for x in v])), 5) for c in SIG_COMPS}} for g, v in gm.items()}
    return {'version': 'signal-3.0',
            'config': {k: getattr(Config, k) for k in _SIG_CFG if k not in ('SIG_FLOORS', 'SIG_GOAL_FLOORS')},
            'stats': {k: int(v) for k, v in st.items()},
            'snapshot': snap,
            'provenance_share': {c: round(st['prov_' + c] / ln, 4) for c in ('foreign', 'mixed', 'own', 'unknown')},
            'emitter_windows': {'settled': cons, 'helped': st['cons_helped'], 'harmed': st['cons_harmed'], 'none': st['cons_none']},
            'listener_verdicts': {k: int(st['verdict_listener_' + k]) for k in ('helped', 'harmed', 'none')},
            'sender_verdicts': {k: int(st['verdict_sender_' + k]) for k in ('helped', 'harmed', 'none')},
            'goal_models': gm_out,
            'baseline_cells': {c: len(bk.base[c]) for c in SIG_COMPS},
            'coop': {'mode': Config.COOP_HARVEST_MODE, 'agents_in_group': len(getattr(eng, '_coop_group', {})),
                     'extra_energy_from_group': float(getattr(eng, '_coop_extra', 0.0))},
            'echo': {'mode': Config.SIG_ECHO_MODE, 'active_ticks': int(st['echo_active_ticks']), 'cycles': int(st['echo_cycles']),
                     'heard': int(st['echo_heard']),
                     'outcomes': {k: int(st['echo_' + k]) for k in ('recognized', 'partial', 'doubt', 'none', 'stress_missed')}},
            'replied_ids_size': len(bk.replied_ids),
            'reply': {k: int(st['reply_' + k]) for k in ('emits', 'attempts', 'emit_blocked', 'settled', 'credit', 'z_none', 'pick_learned', 'pick_default')},
            'in_flight': len(bk.pend), 'last_err': bk.err}


Pattern = extend_class(Pattern, SignalMixin)
EvolutionEngine = extend_class(EvolutionEngine, SignalEngineMixin)

print("✅ SIGNAL v3.0: единый механизм (испускание за энергию → распознавание свой/чужой/неизвестный → исход через "
      f"{Config.SIG_DELAY} тиков → кредиты → доверие по исходу → сдвиг приоритетов целей). Единый крик: _sig_cry. Подмен и патчей нет.")
print(f"   лимит форм={Config.SIG_MAX_FORMS} цена испускания={Config.SIG_EMIT_COST} CHOICE_GAIN={Config.SIG_CHOICE_GAIN} "
      f"RECENT={Config.SIG_RECENT_TICKS} WARMUP={Config.SIG_WARMUP} | каналов в алфавите: {len(SIGNAL_CHANNELS)}")


✅ SIGNAL v3.0: единый механизм (испускание за энергию → распознавание свой/чужой/неизвестный → исход через 6 тиков → кредиты → доверие по исходу → сдвиг приоритетов целей). Единый крик: _sig_cry. Подмен и патчей нет.
   лимит форм=20 цена испускания=0.004 CHOICE_GAIN=4.0 RECENT=12 WARMUP=5 | каналов в алфавите: 16


In [ ]:
# Ячейка 22
from collections import Counter, deque

for _k, _v in {'TRACE_ENABLE': True, 'TRACE_MAX': 4000}.items():
    if not hasattr(Config, _k):
        setattr(Config, _k, _v)

_TR_KEYS = ('choice_signal', 'choice_influenced', 'choice_noflip', 'choice_noflip_base_won', 'choice_blocked',
            'choice_signal_rescued_prev', 'choice_reinforced', 'choice_no_switch_cooldown_candidate',
            'choice_no_switch_barrier_candidate', 'adrenaline_switch')


def _tr_on(p):
    return bool(getattr(Config, 'TRACE_ENABLE', False)) and Config.SIG_ENABLE and getattr(p, 'world', None) is not None


def _tr_book(bk):
    if not hasattr(bk, 'tr'):
        bk.tr = Counter()
        bk.tr_rows = deque(maxlen=int(Config.TRACE_MAX))
        bk.tr_open = {}
    return bk


def _tr_typ(g):
    return str(g.get('type')) if isinstance(g, dict) else None


def _tr_hunt(p):
    h = getattr(p, '_hunt', None)
    if h is None:
        return None
    if isinstance(h, dict):
        return {k: (round(float(v), 3) if isinstance(v, (int, float)) and not isinstance(v, bool) else v)
                for k, v in h.items() if isinstance(v, (int, float, str, bool))}
    return True


def _tr_classify(switched, d):
    if d.get('adrenaline_switch', 0) > 0:
        return 'adrenaline_switch'
    if switched:
        if d.get('choice_influenced', 0):
            return 'switched_signal_influenced'
        if d.get('choice_noflip_base_won', 0):
            return 'switched_base_won'
        return 'switched_no_signal_shift'
    if not d.get('choice_signal', 0):
        return 'no_goal_shift'
    if d.get('choice_reinforced', 0):
        return 'reinforced_current'
    if d.get('choice_no_switch_cooldown_candidate', 0):
        return 'cooldown_candidate'
    if d.get('choice_no_switch_barrier_candidate', 0):
        return 'barrier_candidate'
    if d.get('choice_signal_rescued_prev', 0):
        return 'signal_rescued_prev'
    if d.get('choice_noflip', 0):
        return 'held_other'
    return 'unknown'


class TracePatternMixin:
    def _sig_provenance(self, book, cx, cy, t, obs):
        r = super()._sig_provenance(book, cx, cy, t, obs)
        try:
            self._tr_last_prov = (int(t), r[0], r[1], r[2])
        except Exception:
            self._tr_last_prov = None
        return r

    def _sig_listen(self, field, t):
        if not _tr_on(self) or not self.cells:
            return super()._sig_listen(field, t)
        pre = None
        try:
            rec_ = getattr(self, '_sig_recent', None)
            pre = {'last': rec_[-1] if rec_ else None, 'gap': float(self.spirit_gap), 'ten': float(self.cognitive_tension),
                   'alarm': float(getattr(self, 'alarm_level', 0.0)), 'hunt': getattr(self, '_hunt', None),
                   'hunt_snap': _tr_hunt(self)}
            self._tr_last_prov = None
        except Exception:
            pre = None
        result = super()._sig_listen(field, t)
        if pre is not None:
            try:
                self._tr_after_listen(int(t), pre)
            except Exception as e:
                _sg_err(_sg_book(self.world), 'trace', e)
        return result

    def _tr_after_listen(self, now, pre):
        rec_ = getattr(self, '_sig_recent', None)
        if not rec_:
            return
        e = rec_[-1]
        if e is pre['last'] or e[0] != now:
            return
        te, form, strength, src, ctx, state, cat = e
        if cat == 'own':
            return
        bk = _tr_book(_sg_book(self.world))
        prov = getattr(self, '_tr_last_prov', None)
        ent = prov[3] if prov and prov[0] == now else None
        mid = reply_to = None
        if cat == 'foreign' and ent is not None and len(ent) >= 10:
            mid, reply_to = ent[8], ent[9]
        kind = ('hunt' if mid is not None and mid in getattr(bk, 'hunt_calls', {}) else
                'place' if mid is not None and mid in getattr(bk, 'place_msgs', {}) else
                'signal' if mid is not None else 'signal_no_id')
        h_after = getattr(self, '_hunt', None)
        rec = {
            'rx_id': int(bk.tr['rx']), 't': now, 'to': int(self.id), 'from': (int(src) if src is not None else None),
            'message_id': (list(mid) if isinstance(mid, tuple) else mid), 'reply_to': (list(reply_to) if isinstance(reply_to, tuple) else reply_to),
            'kind': kind, 'cat': cat, 'state': state, 'strength': round(float(strength), 3),
            'trust': (round(float(_sg_trust(self, src)), 3) if src is not None else None),
            'form': str(form.get('label')), 'alarm_form': bool(self._sig_is_alarm(form)),
            'form_vh': int(form.get('vh', 0)), 'form_vm': int(form.get('vm', 0)), 'form_cpt': int(form.get('cpt', 0) or 0),
            'ctx': str(ctx), 'intent_at_rx': _tr_typ(self.intent),
            'changed': {'spirit_gap': [round(pre['gap'], 4), round(float(self.spirit_gap), 4)],
                        'cognitive_tension': [round(pre['ten'], 4), round(float(self.cognitive_tension), 4)],
                        'alarm_level': [round(pre['alarm'], 4), round(float(getattr(self, 'alarm_level', 0.0)), 4)],
                        'hunt_set': bool(h_after is not None and h_after is not pre['hunt']),
                        'hunt_before': pre['hunt_snap'], 'hunt_after': _tr_hunt(self)},
            'snap': {'m': _sg_mood(self), 'en': float(self.energy), 'tr': (float(_sg_trust(self, src)) if src is not None else 0.0)},
            'decision': None, 'window_switch': None, 'outcome': None,
        }
        bk.tr['rx'] += 1
        bk.tr['rx_' + kind] += 1
        bk.tr_rows.append(rec)
        bk.tr_open.setdefault(int(self.id), []).append(rec)

    def update_intent(self):
        if not _tr_on(self):
            return super().update_intent()
        pre = bk = None
        try:
            bk = _sg_book(self.world)
            live = getattr(bk, 'tr_open', {}).get(int(self.id))
            tick = int(self.age)
            if live and getattr(self, '_goal_tick', -1) != tick:
                trc = getattr(bk, 'choice_debug_events', None)
                pre = {'tick': tick, 'st': {k: int(bk.st[k]) for k in _TR_KEYS}, 'prev': self.intent,
                       'goals': [(_tr_typ(g), round(float(g.get('priority', 0.0)), 4)) for g in self.goals][:6],
                       'n_trc': (len(trc) if trc is not None else 0), 'last_trc': (trc[-1] if trc else None), 'live': live}
        except Exception:
            pre = None
        result = super().update_intent()
        if pre is not None:
            try:
                self._tr_after_decision(bk, pre)
            except Exception as e:
                _sg_err(bk, 'trace', e)
        return result

    def _tr_after_decision(self, bk, pre):
        d = {k: int(bk.st[k]) - pre['st'][k] for k in _TR_KEYS}
        switched = self.intent is not pre['prev']
        cat = _tr_classify(switched, d)
        trc = getattr(bk, 'choice_debug_events', None)
        ev = None
        if trc and (len(trc) != pre['n_trc'] or trc[-1] is not pre['last_trc']):
            cand = trc[-1]
            if cand.get('pid') == int(self.id) and cand.get('t') == pre['tick']:
                ev = cand
        dec = {'t': pre['tick'], 'category': cat, 'switched': bool(switched),
               'prev': _tr_typ(pre['prev']), 'after': _tr_typ(self.intent), 'goals_before': pre['goals']}
        if ev is not None:
            for k in ('top0', 'top1', 'applied', 'commitment_before', 'commitment_after', 'cooldown_elapsed',
                      'cooldown_required', 'switch_mult_effective'):
                dec[k] = ev.get(k)
        for rec in pre['live']:
            if rec['t'] > pre['tick']:
                continue
            if rec['decision'] is None:
                rec['decision'] = dec
                bk.tr['dec_' + rec['kind'] + '_' + cat] += 1
            elif switched and rec['window_switch'] is None and pre['tick'] - rec['t'] <= Config.SIG_RECENT_TICKS:
                rec['window_switch'] = {'t': pre['tick'], 'category': cat, 'after': _tr_typ(self.intent)}
                bk.tr['late_switch_' + cat] += 1

class TraceEngineMixin:
    def _post_semantic_step(self, t):
        super()._post_semantic_step(t)
        if not _tr_on(self):
            return
        try:
            bk = _sg_book(self)
            op = getattr(bk, 'tr_open', None)
            if not op:
                return
            now = int(self.age)
            by_id = {p.id: p for p in self.patterns if p.alive}
            for pid in list(op):
                keep = []
                for rec in op[pid]:
                    if now - rec['t'] < Config.SIG_DELAY:
                        keep.append(rec)
                        continue
                    own = by_id.get(pid)
                    if own is None:
                        rec['outcome'] = {'owner_gone': True}
                        bk.tr['out_owner_gone'] += 1
                        continue
                    s0 = rec['snap']
                    rec['outcome'] = {
                        't': now, 'd_mood': round(_sg_mood(own) - s0['m'], 4), 'd_energy': round(float(own.energy) - s0['en'], 4),
                        'd_trust': (round(float(_sg_trust(own, rec['from'])) - s0['tr'], 4) if rec['from'] is not None else None),
                        'intent_now': _tr_typ(own.intent), 'hunt_active': _tr_hunt(own)}
                    bk.tr['out_resolved'] += 1
                if keep:
                    op[pid] = keep
                else:
                    del op[pid]
        except Exception as e:
            _sg_err(_sg_book(self), 'trace', e)

    def audit_summary(self):
        S = super().audit_summary()
        try:
            if Config.TRACE_ENABLE:
                S['signal_trace'] = _tr_summary(self)
        except Exception as e:
            _sg_err(_sg_book(self), 'trace', e)
        return S

def _tr_summary(eng, n_examples=12):
    bk = _sg_book(eng)
    rows = list(getattr(bk, 'tr_rows', []))
    by = {}
    for r in rows:
        cat = (r['decision'] or {}).get('category', 'undecided')
        b = by.setdefault((r['kind'], cat), {'n': 0, 'hunt_set': 0, 'sw': 0, 'dm': [], 'de': [], 'dt': [], 'trust': [], 'str': []})
        b['n'] += 1
        b['hunt_set'] += int(bool(r['changed']['hunt_set']))
        b['sw'] += int(r['window_switch'] is not None)
        b['trust'].append(r['trust'] if r['trust'] is not None else 0.0); b['str'].append(r['strength'])
        o = r['outcome'] or {}
        if 'd_mood' in o:
            b['dm'].append(o['d_mood']); b['de'].append(o['d_energy'])
            if o['d_trust'] is not None:
                b['dt'].append(o['d_trust'])
    mean = lambda a: (round(sum(a) / len(a), 4) if a else None)
    table = [{'kind': k, 'decision': c, 'n': v['n'], 'hunt_set': v['hunt_set'], 'late_switch': v['sw'],
              'mean_trust': mean(v['trust']), 'mean_strength': mean(v['str']), 'n_out': len(v['dm']),
              'd_mood': mean(v['dm']), 'd_energy': mean(v['de']), 'd_trust': mean(v['dt'])}
             for (k, c), v in sorted(by.items(), key=lambda x: -x[1]['n'])]
    ex = [r for r in rows if r['decision'] and r['decision']['switched']][:n_examples // 2] + rows[:n_examples // 2]
    return {'version': 'trace-1', 'enabled': bool(Config.TRACE_ENABLE), 'ring': [len(rows), int(Config.TRACE_MAX)],
            'counters': {k: int(v) for k, v in bk.tr.items()} if hasattr(bk, 'tr') else {},
            'table': table, 'examples': ex,
            'note': 'candidate = диагностика кода (не доказательство причины); «уже знал»/«не доверился» не назначаются — см. strength/trust/form_vh/vm/cpt'}

def trace_report(eng, top=15):
    S = _tr_summary(eng, 0)
    print(f"[TRACE] rx={S['counters'].get('rx', 0)} ring={S['ring'][0]}/{S['ring'][1]}")
    for r in S['table'][:top]:
        print(f"  {r['kind']:<13} {r['decision']:<28} n={r['n']:<5} hunt_set={r['hunt_set']:<4} late_sw={r['late_switch']:<4} "
              f"trust={r['mean_trust']} str={r['mean_strength']} | out n={r['n_out']} dm={r['d_mood']} de={r['d_energy']} dt={r['d_trust']}")

def trace_dump(eng, path='signal_trace.jsonl'):
    import json
    bk = _sg_book(eng)
    with open(path, 'w', encoding='utf-8') as f:
        for r in getattr(bk, 'tr_rows', []):
            f.write(json.dumps(r, ensure_ascii=False, default=str) + '\n')
    print(f"[TRACE] записей: {len(getattr(bk, 'tr_rows', []))} -> {path}")


# TRACE is an observational wrapper: cooperative super() keeps the original signal
# implementation in the MRO and avoids mutating SignalMixin/SignalEngineMixin globally.
Pattern = extend_class(Pattern, TracePatternMixin)
EvolutionEngine = extend_class(EvolutionEngine, TraceEngineMixin)
assert any(k.__name__ == 'TracePatternMixin' for k in Pattern.__mro__)
assert any(k.__name__ == 'TraceEngineMixin' for k in EvolutionEngine.__mro__)
print("✅ SIGNAL TRACE v2: cooperative mixins; no runtime method assignment. "
      "Отчёт: trace_report(eng), дамп: trace_dump(eng), сводка в audit_summary()['signal_trace']. "
      "Выключатель: Config.TRACE_ENABLE=False")


✅ SIGNAL TRACE v2: cooperative mixins; no runtime method assignment. Отчёт: trace_report(eng), дамп: trace_dump(eng), сводка в audit_summary()['signal_trace']. Выключатель: Config.TRACE_ENABLE=False


In [ ]:
# Ячейка 23
import numpy as np
from collections import Counter

for _need in ("Pattern", "EvolutionEngine", "extend_class", "Config"):
    if _need not in globals():
        raise RuntimeError(f"{_need} не найден: выполни ячейки движка и FERAL v3 перед этой.")
if not hasattr(Pattern, "_x2_change"):
    raise RuntimeError("Pattern._x2_change не найден.")
if not hasattr(EvolutionEngine, "_record_death_drag"):
    raise RuntimeError("EvolutionEngine._record_death_drag не найден.")
if any(k.__name__ == "FeralBountyMixin" for k in Pattern.__mro__):
    raise RuntimeError("FERAL BOUNTY уже подключён. Runtime -> Restart session и запуск сверху вниз.")
for _n in ("LedgerEngine", "RichEngine"):
    _c = globals().get(_n)
    if _c is not None and not any(k.__name__ == "FeralBountyEngineMixin" for k in _c.__mro__):
        raise RuntimeError(f"{_n} уже определён без слоя FERAL BOUNTY. Поставь эту ячейку ВЫШЕ Ledger/RichEngine.")
if "_p832_feral_exec" in globals():
    raise RuntimeError("Ячейка p832 уже выполнена (она и есть запуск). Эту ячейку надо ВЫШЕ неё, потом Restart session.")

_FB_DEFAULTS = {
    'FB_ENABLE': True,
    'FB_GRATITUDE': 0.15,
    'FB_FURY_SCALE': 1.0,
    'FB_GRIEF_RELIEF': 0.05,
    'FB_TRUST': True,
    'FB_TRUST_MULT': 1.0,
    'FB_ENERGY_BONUS': 0.0,
    'FB_VERBOSE': 3,
}
for _k, _v in _FB_DEFAULTS.items():
    if not hasattr(Config, _k):
        setattr(Config, _k, _v)

FB = {'world': None, 'stats': Counter(), 'last_err': None, 'buf': None}

def _fb_sync(world):
    if FB['world'] != id(world):
        FB['world'] = id(world)
        FB['stats'] = Counter()
        FB['last_err'] = None
        FB['buf'] = None

def _fb_err(e):
    FB['stats']['errors'] += 1
    if FB['last_err'] is None:
        print(f"[FERAL-BOUNTY] ⚠️ {e!r}", flush=True)
    FB['last_err'] = repr(e)

def _fb_note_killer(world, p):
    _fb_sync(world)
    now = int(getattr(world, 'age', 0))
    buf = FB['buf']
    if buf is None or buf['t'] != now:
        if buf is not None and buf['killers']:
            FB['stats']['buf_stale_dropped'] += 1
        buf = FB['buf'] = {'t': now, 'killers': []}
    buf['killers'].append(p)

def _fb_reward(world, feral):
    _fb_sync(world)
    st = FB['stats']
    buf, FB['buf'] = FB['buf'], None
    now = int(getattr(world, 'age', 0))
    if buf is None or buf['t'] != now or not buf['killers']:
        st['kills_no_killers'] += 1
        return
    killers = [s for s in buf['killers'] if s.alive and getattr(s, 'role_type', None) != 'feral']
    if not killers:
        st['kills_no_killers'] += 1
        return
    fury = float(getattr(feral, '_feral_fury', 1.0))
    fury_norm = float(np.clip((fury - 1.0) / 2.0, 0.0, 1.0))
    grat = float(Config.FB_GRATITUDE) * (1.0 + float(Config.FB_FURY_SCALE) * fury_norm)
    st['kills'] += 1
    for s in killers:
        em = s.emotional_memory
        g0 = float(em['gratitude'])
        em['gratitude'] = min(1.0, g0 + grat)
        st['gratitude_added_x1000'] += int(1000 * (float(em['gratitude']) - g0))
        if Config.FB_GRIEF_RELIEF > 0:
            em['grief'] = max(0.0, float(em['grief']) - float(Config.FB_GRIEF_RELIEF))
        s._fb_kills = int(getattr(s, '_fb_kills', 0)) + 1
        if Config.FB_ENERGY_BONUS > 0:
            got = s._x2_change("X2.feral_bounty_CREATION", float(s.energy) + float(Config.FB_ENERGY_BONUS))
            st['energy_created_x1000'] += int(1000 * got)
        st['rewarded'] += 1
    if Config.FB_TRUST and len(killers) > 1:
        for a in killers:
            for b in killers:
                if a is not b:
                    a.trust_ledger.update(b.id, 'helpful', multiplier=float(Config.FB_TRUST_MULT))
                    st['trust_updates'] += 1
    if st['kills'] <= int(Config.FB_VERBOSE):
        print(f"[FERAL-BOUNTY t={now}] ферал {getattr(feral, 'id', '?')} убит; награждены {[s.id for s in killers]}: "
              f"gratitude +{grat:.3f} (fury={fury:.2f}), grief -{Config.FB_GRIEF_RELIEF}, "
              f"energy +{Config.FB_ENERGY_BONUS}", flush=True)

class FeralBountyMixin:
    def _x2_change(self, name, new_value, offered=None):
        actual = super()._x2_change(name, new_value, offered)
        if name == "X2.scream_kill_reward" and Config.FB_ENABLE and self.world is not None:
            try:
                _fb_note_killer(self.world, self)
            except Exception as e:
                _fb_err(e)
        return actual

def fb_report():
    st = FB['stats']
    return {
        'kills_rewarded': int(st['kills']), 'agents_rewarded': int(st['rewarded']),
        'kills_without_killers': int(st['kills_no_killers']), 'buf_stale_dropped': int(st['buf_stale_dropped']),
        'mean_killers_per_kill': round(st['rewarded'] / max(1, st['kills']), 2),
        'mean_gratitude_added': round(st['gratitude_added_x1000'] / 1000.0 / max(1, st['rewarded']), 4),
        'trust_updates': int(st['trust_updates']),
        'energy_created': round(st['energy_created_x1000'] / 1000.0, 4),
        'errors': int(st['errors']), 'last_err': FB['last_err'],
        'config': {k_: getattr(Config, k_) for k_ in _FB_DEFAULTS},
    }

class FeralBountyEngineMixin:
    def _record_death_drag(self, p, cause):
        super()._record_death_drag(p, cause)
        if cause == "scream_kill" and Config.FB_ENABLE:
            try:
                _fb_reward(self, p)
            except Exception as e:
                _fb_err(e)

    def audit_summary(self):
        S = super().audit_summary()
        try:
            S['feral_bounty'] = fb_report()
        except Exception as e:
            _fb_err(e)
        return S

def _fb_metrics(patterns, field, t, lexicon=None):
    try:
        st = FB['stats']
        return {'fb_kills': int(st['kills']), 'fb_rewarded': int(st['rewarded']),
                'fb_trust_updates': int(st['trust_updates']),
                'fb_energy_created': round(st['energy_created_x1000'] / 1000.0, 4),
                'fb_errors': int(st['errors'])}
    except Exception:
        return {}

Pattern = extend_class(Pattern, FeralBountyMixin)
EvolutionEngine = extend_class(EvolutionEngine, FeralBountyEngineMixin)

if "register_metrics" in globals():
    register_metrics("feral_bounty", _fb_metrics)

print("✅ FERAL BOUNTY v1.1: за убийство ферала убийцы получают gratitude (+ярость), облегчение grief и взаимное доверие.")
print(f"   GRAT={Config.FB_GRATITUDE} FURY_SCALE={Config.FB_FURY_SCALE} GRIEF_RELIEF={Config.FB_GRIEF_RELIEF} "
      f"TRUST={Config.FB_TRUST} ENERGY_BONUS={Config.FB_ENERGY_BONUS} (0 = закон 832 не трогаем)")
print("   Ячейка ничего не запускает. Следующая — Ledger/RichEngine, запуск делает p832.")


✅ FERAL BOUNTY v1.1: за убийство ферала убийцы получают gratitude (+ярость), облегчение grief и взаимное доверие.
   GRAT=0.15 FURY_SCALE=1.0 GRIEF_RELIEF=0.05 TRUST=True ENERGY_BONUS=0.0 (0 = закон 832 не трогаем)
   Ячейка ничего не запускает. Следующая — Ledger/RichEngine, запуск делает p832.


In [ ]:
# Ячейка 24
for _need in ("Pattern", "Config", "extend_class"):
    if _need not in globals():
        raise RuntimeError(f"{_need} не найден: сначала выполни ячейки выше.")

if not hasattr(Config, "FERAL_MAX_CELLS"):
    Config.FERAL_MAX_CELLS = 50


class FeralSizeMixin:
    def grow_feral(self, field):
        if self.role_type == "feral" and len(self.cells) >= int(Config.FERAL_MAX_CELLS):
            return
        return super().grow_feral(field)


if any(k.__name__ == "FeralSizeMixin" for k in Pattern.__mro__):
    print("⚠️ FERAL SIZE уже подключён в этой сессии — пропускаю.")
else:
    Pattern = extend_class(Pattern, FeralSizeMixin)
    print(f"✅ FERAL SIZE v1: ферал не растёт выше {Config.FERAL_MAX_CELLS} клеток (было до 300).")


In [ ]:
# Ячейка 25
import os

if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
    print("📎 Drive смонтирован")
else:
    print("📎 Drive уже смонтирован")

ROOT = "/content/drive/MyDrive"
DIR  = os.path.join(ROOT, "832_ledgers")

os.makedirs(DIR, exist_ok=True)
print(f"📁 DIR = {DIR}")
print(f"   exists = {os.path.isdir(DIR)}")

st = os.statvfs(DIR)
free_mb  = st.f_bavail * st.f_frsize / 1e6
total_mb = st.f_blocks * st.f_frsize / 1e6

print(f"\n💾 Место на Drive:")
print(f"   всего:    {total_mb:>10.0f} МБ")
print(f"   свободно: {free_mb:>10.0f} МБ")

if free_mb < 1000:
    print("\n⚠️  Меньше 1 ГБ. Прогон может упасть на autosave.")
    print("    Почисти корзину: drive.google.com/drive/trash")
else:
    print(f"\n✅ Места хватает (нужно ~500 МБ на прогон)")

print(f"\n📋 Ключевые файлы:")
for name in ("832_living_world.json",
             "832_chorus_state.json",
             "832_Archive",
             "832_ledgers"):
    p = os.path.join(ROOT, name)
    if os.path.isfile(p):
        print(f"   ✅  {name}  ({os.path.getsize(p)/1e6:.1f} МБ)")
    elif os.path.isdir(p):
        print(f"   ✅  {name}/  (папка)")
    else:
        print(f"   ⚠️  {name}  — ОТСУТСТВУЕТ")


📎 Drive уже смонтирован
📁 DIR = /content/drive/MyDrive/832_ledgers
   exists = True

💾 Место на Drive:
   всего:         16106 МБ
   свободно:      11994 МБ

✅ Места хватает (нужно ~500 МБ на прогон)

📋 Ключевые файлы:
   ✅  832_living_world.json  (39.4 МБ)
   ⚠️  832_chorus_state.json  — ОТСУТСТВУЕТ
   ✅  832_Archive/  (папка)
   ✅  832_ledgers/  (папка)


In [ ]:
# Ячейка 26
import contextlib
import gc
import io
import json
import os
import shutil
import tempfile
import time
import zlib
from collections import Counter

import numpy as np

LEDGER_SCALARS = ("energy", "epistemic_scar", "soul_weight", "spirit_gap",
                  "coherence", "body_memory", "cognitive_tension", "pred_error", "heading")
LEDGER_FINAL_EXTRA = ("protection_level",)
PROTECTED_THRESHOLD = 0.2
_LEDGER_SCALAR_TYPES = (bool, int, float, str)


LEDGER_IMPRINT_EVERY = 10


def _agent_crc(p):
    row = []
    for name, value in vars(p).items():
        if isinstance(value, np.generic):
            value = value.item()
        if isinstance(value, _LEDGER_SCALAR_TYPES):
            row.append((name, value))
    return zlib.crc32(repr(row).encode("utf-8"))


def _cells_crc(p):
    return zlib.crc32(repr(sorted(p.cells)).encode("utf-8"))


def _birth_info(p):
    for rec in reversed(list(p.biography)):
        t = rec.get("type", "")
        if t.startswith("born_from_"):
            return t[len("born_from_"):], rec.get("parent_id")
    return "spawn", None


def _config_snapshot():
    return {k: getattr(Config, k) for k in dir(Config)
            if k.isupper() and isinstance(getattr(Config, k), (bool, int, float, str))}


def _state_fingerprint(eng):
    field_crc = zlib.crc32(np.ascontiguousarray(eng.field).tobytes())
    rng_pos = int(eng._given_rng.get_state()[2])
    agents = sorted((int(p.id), _agent_crc(p), _cells_crc(p)) for p in eng.patterns)
    return {"field": field_crc, "rng_pos": rng_pos, "n_agents": len(agents),
            "agents": zlib.crc32(repr(agents).encode("utf-8"))}


def _sig_in_edges(p):
    sig = (round(float(p.pred_error), 2), round(float(p.epistemic_load), 2),
           round(float(p.soul_weight), 2), p.semantic_state)
    return bool(p.concept_graph.edges.get(sig))


def _ledger_frac(rows, key):
    if not all(key in r for r in rows):
        return float("nan")
    vals = [r[key] / r["n"] for r in rows if r["n"]]
    return float(np.mean(vals)) if vals else float("nan")


def _imprint_sum(rows, key):
    out = {}
    for r in rows:
        for k, v in r[key].items():
            if isinstance(v, dict):
                d = out.setdefault(k, {})
                for kk, vv in v.items():
                    d[kk] = d.get(kk, 0.0) + vv
            else:
                out[k] = out.get(k, 0.0) + v
    return out


def _imprint_window(ch):
    ims = [r["imprint"] for r in ch if r.get("imprint")]
    if not ims:
        return None
    n = len(ims)
    contacts = int(sum(i["contacts"] for i in ims))
    agents = float(np.mean([i["agents"] for i in ims]))
    return {
        "ticks": n,
        "contacts": contacts,
        "contacts_per_tick": contacts / n,
        "case": _imprint_sum(ims, "case"),
        "case_strain": _imprint_sum(ims, "case_strain"),
        "strain_per_contact": float(sum(i["strain_sum"] for i in ims)) / max(contacts, 1),
        "new": int(sum(i["new"] for i in ims)),
        "repeat": int(sum(i["repeat"] for i in ims)),
        "repeat_known": float(sum(i["repeat_known"] for i in ims)),
        "repeat_stranger": float(sum(i["repeat_stranger"] for i in ims)),
        "repeat_differs": int(sum(i["repeat_differs"] for i in ims)),
        "recept": _imprint_sum(ims, "recept"),
        "sameB": _imprint_sum(ims, "sameB"),
        "steer": _imprint_sum(ims, "steer"),
        "agents": agents,
        "agents_with_imprint": float(np.mean([i["agents_with_imprint"] for i in ims])),
        "imprints_end": int(ims[-1]["imprints"]),
        "strain_mean": float(np.mean([i["strain_mean"] for i in ims])),
        "strain_gt01_frac": float(np.mean([i["strain_gt01"] / max(i["agents"], 1) for i in ims])),
        "strain_at_max_frac": float(np.mean([i["strain_at_max"] / max(i["agents"], 1) for i in ims])),
        "freeze_mean": float(np.mean([i["freeze_mean"] for i in ims])),
    }


class ExperimentLedger:
    def __init__(self, window=100):
        self.window = int(window)
        self.header = {}
        self.rows = []
        self.events = []
        self.lifecycle = {}
        self._chain = 0
        self.autosave_path = None
        self.autosave_every = 500
        self._kills = {}

    def _new_record(self, p, birth_t, kind, parent):
        return {"id": int(p.id), "lineage_id": int(p.lineage_id), "birth_t": birth_t,
                "birth_kind": kind, "parent_id": parent, "death_t": None,
                "death_cause": None, "age_at_death": None, "final": None}

    def before(self, eng):
        snap = {}
        for p in eng.patterns:
            pid = int(p.id)
            if pid not in self.lifecycle:
                self.lifecycle[pid] = self._new_record(p, -1, "initial", None)
            if not getattr(p, "alive", True):
                continue   # уже мёртв и ждёт вычистки из eng.patterns: его смерть учтена в тик, когда он умер
            snap[pid] = (p, str(p.semantic_state), str(p.role_type), dict(p.event_counts),
                         float(p.protection_level))
        self._hook_before = dict(eng._death_hook_calls)
        self._ev_world_before = dict(getattr(eng, "_ev_world", None) or {})
        return snap

    def after(self, eng, t, pre):
        # ЕДИНЫЙ УЧЁТ НАСЕЛЕНИЯ. «Живой» = p.alive. Раньше n/рождения/смерти считались по ЧЛЕНСТВУ в eng.patterns,
        # а там до следующей вычистки лежат уже мёртвые (alive=False): ledger.n выходил больше RichLog.n
        # (277 против 270), а смерть записывалась на тик позже. Теперь n, рождения, смерти и деления считаются
        # по одному критерию («жив в начале тика» / «жив в конце») и именно ledger — источник итогов (см. totals()).
        cur_all = {int(p.id): p for p in eng.patterns}
        cur = {pid: p for pid, p in cur_all.items() if getattr(p, "alive", True)}
        ev = Counter()
        births = deaths = divisions = 0
        births_by_kind = Counter()
        unlogged = []
        _unl_obj = {}
        born_dead = []

        def _register_death(pid, obj, ec0, prot0):
            _explicit_cause = getattr(obj, "_death_cause_explicit", None)
            cause = (str(_explicit_cause) if _explicit_cause else
                     ",".join(sorted(k for k, v in obj.event_counts.items()
                                     if k.startswith("death_") and v > ec0.get(k, 0))))
            rec = self.lifecycle[pid]
            rec["death_t"] = t
            rec["age_at_death"] = int(obj.age)
            rec["final"] = {n: float(getattr(obj, n)) for n in LEDGER_SCALARS}
            rec["final"]["protection_level"] = float(obj.protection_level)
            rec["final"]["gene_metabolic_cost"] = float(obj.genome.get("metabolic_cost", float("nan")))
            rec["protection_before_tick"] = prot0
            rec["was_subject"] = bool(obj._subject_detected)
            rec["was_narrative"] = bool(obj._narrative_agent)
            rec["imprint_final"] = imprint_agent_summary(obj)
            if cause:
                rec["death_cause"] = cause
                self.events.append((t, pid, "death", cause, None))
            elif pid in self._kills:
                kt, killer = self._kills[pid]
                rec["death_cause"] = "feral_execution_victim"
                rec["killed_by"] = killer
                self.events.append((t, pid, "death", "feral_execution_victim", killer))
            else:
                unlogged.append(pid)
                _unl_obj[pid] = obj

        for pid, p in cur_all.items():
            if pid in pre or pid in self.lifecycle:
                continue   # не новый: был жив в начале тика (pre) или уже учтён раньше (мёртвый, ждавший вычистки)
            kind, parent = _birth_info(p)
            self.lifecycle[pid] = self._new_record(p, t, kind, parent)
            births += 1
            births_by_kind[kind] += 1
            if kind == "division":
                divisions += 1
            self.events.append((t, pid, "birth", kind, parent))
            for k, v in p.event_counts.items():
                ev[k] += v
            if not getattr(p, "alive", True):
                born_dead.append((pid, p))   # родился и умер внутри одного тика
        for pid, (p0, st0, role0, ec0, prot0) in pre.items():
            p = cur.get(pid)
            obj = p if p is not None else p0
            n_exec = obj.event_counts.get("feral_execution", 0) - ec0.get("feral_execution", 0)
            if n_exec > 0:
                found = 0
                for rec in reversed(list(obj.biography)):
                    if rec.get("type") == "feral_execution" and "victim" in rec:
                        self._kills[int(rec["victim"])] = (int(t), int(pid))
                        found += 1
                        if found >= n_exec:
                            break
            for k, v in obj.event_counts.items():
                d = v - ec0.get(k, 0)
                if d:
                    ev[k] += d
            if p is None:
                deaths += 1
                _register_death(pid, obj, ec0, prot0)
            else:
                st1, role1 = str(p.semantic_state), str(p.role_type)
                if st1 != st0:
                    self.events.append((t, pid, "state_change", st0, st1))
                if role1 != role0:
                    self.events.append((t, pid, "role_change", role0, role1))
        for pid, p in born_dead:
            deaths += 1
            _register_death(pid, p, {}, float(p.protection_level))
        if unlogged:
            hook = {k: v - self._hook_before.get(k, 0) for k, v in eng._death_hook_calls.items()}
            cands = {k: v for k, v in hook.items() if v and k != "natural_death"}
            label = "unlogged:ambiguous"
            if len(cands) == 1 and sum(cands.values()) == len(unlogged):
                label = "unlogged:" + next(iter(cands))
            elif not cands:
                label = "unlogged:no_hook"
            for pid in unlogged:
                # явная причина от самого пути смерти надёжнее восстановления по дельтам счётчиков хуков за тик
                # (при нескольких смертях за тик дельты давали unlogged:ambiguous)
                _exp = getattr(_unl_obj.get(pid), "_death_cause_explicit", None)
                _lab = ("unlogged:" + str(_exp)) if _exp else label
                self.lifecycle[pid]["death_cause"] = _lab
                self.events.append((t, pid, "death", _lab, None))
        alive = list(cur.values())
        _evw_now = dict(getattr(eng, "_ev_world", None) or {})
        _evw_prev = getattr(self, "_ev_world_before", None) or {}
        _ev_w_delta = {k: int(v) - int(_evw_prev.get(k, 0)) for k, v in _evw_now.items()
                       if int(v) - int(_evw_prev.get(k, 0)) > 0}
        _hook_delta = {str(k): max(0, int(v) - int(self._hook_before.get(k, 0)))
                       for k, v in eng._death_hook_calls.items()
                       if int(v) - int(self._hook_before.get(k, 0)) > 0}
        _hook_total = int(sum(_hook_delta.values()))
        row = {"t": int(t), "n": len(alive), "n_list": len(cur_all), "n_pending_dead": len(cur_all) - len(alive),
               "births": births, "deaths": deaths, "births_by_kind": dict(births_by_kind),
               "death_hook_total": _hook_total,
               "death_hook_by_cause": _hook_delta,
               "death_hook_minus_ledger": int(_hook_total - deaths),
               "divisions": divisions, "cells": int(sum(len(p.cells) for p in alive)),
               "field_energy": float(np.sum(eng.field[:, :, CH["energy"]])),
               "gene_mc_neg": int(sum(1 for p in alive if p.genome.get("metabolic_cost", 0.0) < 0)),
               "gene_mob_neg": int(sum(1 for p in alive if p.genome.get("mobility", 0.0) < 0)),
               "gene_mc_mean": (float(np.mean([p.genome.get("metabolic_cost", np.nan) for p in alive]))
                                if alive else float("nan")),
               "meta_pos": int(sum(1 for p in alive if p._meta_surprise > 0)),
               "meta_mean": (float(np.mean([p._meta_surprise for p in alive])) if alive else float("nan")),
               "cns_hi": int(sum(1 for p in alive if p._concept_narrative_score > 0.55)),
               "sig_hit": int(sum(1 for p in alive if p.role_type != "disorganizer"
                                  and p._concept_narrative_score > 0.55 and _sig_in_edges(p))),
               "protected": int(sum(1 for p in alive if p.protection_level > PROTECTED_THRESHOLD)),
               "subjects": int(sum(1 for p in alive if p._subject_detected)),
               "narrative": int(sum(1 for p in alive if p._narrative_agent)),
               "ev": dict(ev), "ev_world": _ev_w_delta}
        for name in LEDGER_SCALARS:
            if not alive:
                row[name] = float("nan")
            elif name == "heading":
                _angles = np.asarray([float(getattr(p, name, np.nan)) for p in alive], dtype=float)
                _angles = _angles[np.isfinite(_angles)]
                row[name] = (float(np.arctan2(np.mean(np.sin(_angles)), np.mean(np.cos(_angles))))
                             if len(_angles) else float("nan"))
            else:
                row[name] = float(np.mean([getattr(p, name) for p in alive]))
        row["imprint"] = eng.imprint_tick_row()
        row["imprint_contacts"] = eng.imprint_tick_contacts()
        row["imprint_track"] = eng.imprint_track_row()
        if t % LEDGER_IMPRINT_EVERY == 0:
            row["imprint_net"] = eng.imprint_summary(t)
        field_crc = zlib.crc32(np.ascontiguousarray(eng.field).tobytes())
        rng_pos = int(eng._given_rng.get_state()[2])
        agents_crc = zlib.crc32(repr(sorted((pid, _agent_crc(p), _cells_crc(p))
                                            for pid, p in cur_all.items())).encode("utf-8"))
        self._chain = zlib.crc32(repr((self._chain, t, field_crc, rng_pos, agents_crc)).encode("utf-8"))
        row["chain"] = self._chain
        self.rows.append(row)

    def totals(self):
        """ЕДИНЫЙ ИСТОЧНИК итогов по населению (накопительно на текущий момент).
        Всё, что где-либо печатается как численность / рождения / смерти / деления / intent_switch,
        берётся отсюда или сверяется с этим."""
        rows = self.rows
        bk, evw = Counter(), Counter()
        for r in rows:
            bk.update(r.get("births_by_kind") or {})
            evw.update(r.get("ev_world") or {})
        last = rows[-1] if rows else {}
        first = rows[0] if rows else {}
        n_pre = (int(first.get("n", 0)) - int(first.get("births", 0)) + int(first.get("deaths", 0))) if rows else None
        return {"t_last": last.get("t"), "ticks": len(rows), "n": last.get("n"),
                "n_pending_dead": last.get("n_pending_dead"), "n_pre_run": n_pre,
                "births": sum(int(r.get("births", 0)) for r in rows),
                "deaths": sum(int(r.get("deaths", 0)) for r in rows),
                "divisions": sum(int(r.get("divisions", 0)) for r in rows),
                "births_by_kind": dict(bk), "ev_world": dict(evw)}

    def windows(self, size=None):
        size = int(size or self.window)
        out = []
        for i in range(0, len(self.rows), size):
            ch = self.rows[i:i + size]
            evs = Counter()
            for r in ch:
                evs.update(r["ev"])
            w = {"t0": ch[0]["t"], "t1": ch[-1]["t"], "ticks": len(ch),
                 "n_mean": float(np.mean([r["n"] for r in ch])),
                 "births": sum(r["births"] for r in ch),
                 "deaths": sum(r["deaths"] for r in ch),
                 "divisions": sum(r["divisions"] for r in ch), "events": dict(evs),
                 "protected_frac": float(np.mean([r["protected"] / r["n"] for r in ch if r["n"]]))
                 if any(r["n"] for r in ch) else float("nan"),
                 "mc_neg_frac": _ledger_frac(ch, "gene_mc_neg"), "mob_neg_frac": _ledger_frac(ch, "gene_mob_neg"),
                 "meta_pos_frac": _ledger_frac(ch, "meta_pos"), "cns_hi_frac": _ledger_frac(ch, "cns_hi"),
                 "sig_hit_frac": _ledger_frac(ch, "sig_hit"),
                 "meta_mean": (float(np.nanmean([r["meta_mean"] for r in ch]))
                               if all("meta_mean" in r for r in ch) else float("nan")),
                 "mc_mean": (float(np.nanmean([r["gene_mc_mean"] for r in ch]))
                             if all("gene_mc_mean" in r for r in ch) else float("nan")),
                 "subjects_mean": float(np.mean([r["subjects"] for r in ch])),
                 "narrative_mean": float(np.mean([r["narrative"] for r in ch]))}
            for name in LEDGER_SCALARS:
                _vals = np.asarray([r.get(name, np.nan) for r in ch], dtype=float)
                _vals = _vals[np.isfinite(_vals)]
                if name == "heading":
                    w[name] = (float(np.arctan2(np.mean(np.sin(_vals)), np.mean(np.cos(_vals))))
                               if len(_vals) else float("nan"))
                else:
                    w[name] = float(np.mean(_vals)) if len(_vals) else float("nan")
            w["imprint"] = _imprint_window(ch)
            out.append(w)
        return out

    def series(self, key):
        return np.array([r[key] for r in self.rows], dtype=float)

    def event_series(self, event_type):
        return np.array([r["ev"].get(event_type, 0) for r in self.rows], dtype=float)

    def report(self, max_windows=12):
        h = self.header
        print("=" * 70)
        print("EXPERIMENT LEDGER")
        print("=" * 70)
        print(f"given_seed={h.get('given_seed')}  steps_requested={h.get('steps')}  "
              f"ticks_observed={len(self.rows)}  config_hash={h.get('config_hash')}")
        if h.get("overrides"):
            print(f"overrides={h['overrides']}")
        if h.get("input_fp_before") is not None:
            print(f"input_fp_before={h['input_fp_before']}")
            print(f"input_fp_after ={h.get('input_fp_after')}  (входы после восстановления; stable={h.get('input_fp_stable')})")
            if h.get("output_fp_after") is not None:
                print(f"output_fp_after={h['output_fp_after']}  изменено самим прогоном: {h.get('run_mutated_files')}")
        if not self.rows:
            print("(пусто)")
            return
        total_births = sum(r["births"] for r in self.rows)
        total_div = sum(r["divisions"] for r in self.rows)
        total_deaths = sum(r["deaths"] for r in self.rows)
        print(f"рождений={total_births} (из них делений={total_div})  смертей={total_deaths}  "
              f"агентов в конце={self.rows[-1]['n']}")
        causes = Counter(rec["death_cause"] for rec in self.lifecycle.values() if rec["death_t"] is not None)
        if causes:
            print("причины смерти:", dict(causes.most_common()))
        evt = Counter()
        for r in self.rows:
            evt.update(r["ev"])
        print("топ-10 событий (сумма за прогон):", dict(evt.most_common(10)))
        ev_kinds = Counter(e[2] for e in self.events)
        print("структурные записи:", dict(ev_kinds))
        ws = self.windows()
        if len(ws) > max_windows:
            idx = np.linspace(0, len(ws) - 1, max_windows).round().astype(int)
            ws = [ws[i] for i in sorted(set(idx))]
        print("\n   окно         n_mean  births deaths  div  energy  scar   soul   gap   coh  prot%  subj  narr  mc<0%  mc_mean")
        for w in ws:
            print(f"  {w['t0']:>5}-{w['t1']:<5} {w['n_mean']:>7.1f} {w['births']:>6} {w['deaths']:>6} "
                  f"{w['divisions']:>4}  {w['energy']:>6.3f} {w['epistemic_scar']:>5.3f} "
                  f"{w['soul_weight']:>5.3f} {w['spirit_gap']:>5.3f} {w['coherence']:>5.3f} "
                  f"{100 * w['protected_frac']:>5.1f} {w['subjects_mean']:>5.1f} {w['narrative_mean']:>5.1f} "
                  f"{100 * w['mc_neg_frac']:>6.1f} {w['mc_mean']:>8.4f}")
        print("\n   окно         _meta_surprise>0%  meta_mean   concept_score>0.55%   сигнатура найдена в edges%")
        for w in ws:
            print(f"  {w['t0']:>5}-{w['t1']:<5} {100 * w['meta_pos_frac']:>14.1f} {w['meta_mean']:>11.4f} "
                  f"{100 * w['cns_hi_frac']:>18.1f} {100 * w['sig_hit_frac']:>24.2f}")
        dead = [r for r in self.lifecycle.values() if r["death_t"] is not None]
        if dead:
            prot = [r for r in dead if r.get("protection_before_tick", 0.0) > PROTECTED_THRESHOLD]
            print(f"\nсмерти: защищённых на начало тика смерти = {len(prot)} из {len(dead)}")
            for title, grp in (("защищённые", prot), ("незащищённые", [r for r in dead if r not in prot])):
                if grp:
                    print(f"  {title}: " + str(dict(Counter(r['death_cause'] for r in grp).most_common())))
            print(f"  среди умерших: бывшие субъекты={sum(1 for r in dead if r.get('was_subject'))}, "
                  f"narrative-агенты={sum(1 for r in dead if r.get('was_narrative'))}")
        self._report_imprint()
        print(f"\nchain(last)={self.rows[-1]['chain']}  final_state={h.get('final_state')}")

    def _report_imprint(self, max_windows=12):
        ws = [w for w in self.windows() if w.get("imprint")]
        if not ws:
            return
        cfg = self.header.get("imprint_config") or {}
        print("\n" + "=" * 70)
        print("IMPRINT: след встречи A -> B")
        print("=" * 70)
        if cfg:
            print("конфиг:", {k.replace("IMPRINT_", ""): v for k, v in sorted(cfg.items())})
        if len(ws) > max_windows:
            idx = np.linspace(0, len(ws) - 1, max_windows).round().astype(int)
            ws = [ws[i] for i in sorted(set(idx))]
        print("\n   окно        конт/тик match% conf% over% prior% quiet%  strain/к  новых повтор%  Δстрейн(знак-чужой)  "
              "imprints  агентов_со_следом")
        for w in ws:
            im = w["imprint"]
            c = max(im["contacts"], 1)
            cs = im["case"]
            rep = im["repeat"]
            dk = (im["repeat_known"] - im["repeat_stranger"]) / max(rep, 1)
            print(f"  {w['t0']:>5}-{w['t1']:<5} {im['contacts_per_tick']:>7.1f} {100 * cs['match'] / c:>6.1f} "
                  f"{100 * cs['conflict'] / c:>5.1f} {100 * cs['overload'] / c:>5.1f} {100 * cs['prior'] / c:>6.1f} "
                  f"{100 * cs['quiet'] / c:>6.1f} {im['strain_per_contact']:>9.4f} {im['new']:>6} "
                  f"{100 * rep / c:>7.1f} {dk:>+19.4f} {im['imprints_end']:>9} {im['agents_with_imprint']:>12.1f}")
        print("\n   окно        strain_mean  strain>0.1%  на_потолке%  freeze  | мембрана: n(closing/neutral/opening) "
              "Δgrav(closing/neutral/opening)")
        for w in ws:
            im = w["imprint"]
            rg = im["recept"]
            ns = [int(rg[g]["n"]) for g in ("closing", "neutral", "opening")]
            dg = [rg[g]["dgrav"] / max(rg[g]["n"], 1) for g in ("closing", "neutral", "opening")]
            print(f"  {w['t0']:>5}-{w['t1']:<5} {im['strain_mean']:>11.4f} {100 * im['strain_gt01_frac']:>11.1f} "
                  f"{100 * im['strain_at_max_frac']:>11.1f} {im['freeze_mean']:>7.3f}  | "
                  f"{ns[0]:>6}/{ns[1]:>6}/{ns[2]:>6}   {dg[0]:>+8.5f}/{dg[1]:>+8.5f}/{dg[2]:>+8.5f}")
        tot = _imprint_window([r for r in self.rows])
        if tot:
            print("\nза весь прогон: случаи", {k: int(v) for k, v in tot["case"].items()},
                  "| strain/контакт по случаям",
                  {k: round(tot["case_strain"][k] / max(tot["case"][k], 1), 4) for k in tot["case"]})
            print("руление (по парам-тикам; toward = A сместился к B, если двигался):")
            for b in ("repel", "neutral", "attract"):
                st = tot["steer"][b]
                mv = max(st["moved"], 1)
                print(f"  {b:<8} n={int(st['n']):>7} двигался={100 * st['moved'] / max(st['n'], 1):>5.1f}% "
                      f"к B среди двигавшихся={100 * st['toward'] / mv:>5.1f}% средняя проекция={st['proj'] / mv:>+8.4f}")
            sb = tot["sameB"]
            if sb["n_b"]:
                md = sb["diff"] / sb["n_b"]
                print(f"один B / один тик / разные A: пар={int(sb['n_b'])}  средний Δgravity(самый открытый − самый закрытый)={md:+.5f}")
        dead = [r for r in self.lifecycle.values() if r.get("death_t") is not None and r.get("imprint_final")]
        if dead:
            by = {}
            for r in dead:
                by.setdefault(r["death_cause"], []).append(r["imprint_final"])
            print("\nсмерти и след (по причине): n, средн. число следов, средн. strain в момент смерти, средн. danger_max")
            for cause, lst in sorted(by.items(), key=lambda kv: -len(kv[1]))[:6]:
                print(f"  {str(cause):<28} n={len(lst):>3} следов={np.mean([x['n'] for x in lst]):>5.1f} "
                      f"strain={np.mean([x['strain'] for x in lst]):>6.3f} "
                      f"danger_max={np.mean([x.get('danger_max', 0.0) for x in lst]):>6.3f}")

    def lifecycle_matrix(self):
        names = ["age_at_death"] + list(LEDGER_SCALARS)
        ids, X = [], []
        for pid, rec in sorted(self.lifecycle.items()):
            if rec["death_t"] is None or rec["final"] is None:
                continue
            ids.append(pid)
            X.append([rec["age_at_death"]] + [rec["final"][n] for n in LEDGER_SCALARS])
        return np.array(X, dtype=float), ids, names

    def save(self, path):
        data = {"header": self.header, "rows": self.rows,
                "events": [list(e) for e in self.events],
                "lifecycle": list(self.lifecycle.values())}
        tmp = path + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False)
        os.replace(tmp, path)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        led = cls()
        led.header, led.rows = data["header"], data["rows"]
        led.events = [tuple(e) for e in data["events"]]
        led.lifecycle = {rec["id"]: rec for rec in data["lifecycle"]}
        led._chain = led.rows[-1]["chain"] if led.rows else 0
        return led


class LedgerEngine(EvolutionEngine):
    ledger = None

    def step(self, t):
        led = self.ledger
        pre = led.before(self) if led is not None else None
        result = super().step(t)
        if led is not None:
            led.after(self, t, pre)
            if led.autosave_path and (t + 1) % led.autosave_every == 0:
                led.header["complete"] = False
                led.save(led.autosave_path)
        return result


def compare_ledgers(a, b):
    n = min(len(a.rows), len(b.rows))
    for i in range(n):
        if a.rows[i]["chain"] != b.rows[i]["chain"]:
            return {"identical": False, "first_divergence_t": a.rows[i]["t"], "ticks_compared": i + 1}
    return {"identical": len(a.rows) == len(b.rows), "first_divergence_t": None,
            "ticks_compared": n, "len_a": len(a.rows), "len_b": len(b.rows)}


ARCHIVE_DIR_LEDGER = "/content/drive/MyDrive/832_Archive"
LEDGER_GUARD_DIR = ARCHIVE_DIR_LEDGER.rstrip("/") + "_ledger_guard"
_LEDGER_EXTRA_FILES = [
    "/content/drive/MyDrive/832_eternal_subjects.json",
    "/content/drive/MyDrive/832_ark_seed_patterns.json",
    "832_ark_seed_patterns.json",
    "832_eternal_subjects.json",
]


def _fp_file_content(path, chunk=1 << 20):
    """[размер, crc32 СОДЕРЖИМОГО файла], потоково. (Раньше архив хэшировался только по именам и размерам.)"""
    crc = 0
    size = 0
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            crc = zlib.crc32(b, crc)
            size += len(b)
    return [size, crc]


def _input_fingerprint():
    """Отпечаток файлового состояния по СОДЕРЖИМОМУ (living_world, chorus_state, eternal_subjects, ark_seed, архив).
    Сам по себе не «вход» и не «выход»: что это значит, определяет момент съёма (см. _execute):
      input_fp_before      — до прогона;
      output_fp_after      — после прогона, ДО восстановления (то, что прогон записал сам);
      input_fp_after       — после восстановления guard'ом: именно это увидит следующий прогон, должно == before."""
    fp = {}
    paths = [LIVING_WORLD_PATH, _CHORUS_STATE_PATH] + list(_LEDGER_EXTRA_FILES)
    for fpth in paths:
        if os.path.isfile(fpth):
            try:
                fp[os.path.basename(fpth)] = _fp_file_content(fpth)
            except Exception as e:
                fp[os.path.basename(fpth)] = ["err", type(e).__name__]
    arch = ARCHIVE_DIR_LEDGER
    if os.path.isdir(arch):
        tot = 0
        crc = 0
        errors = []
        for root, dirs, files in os.walk(arch):
            dirs.sort()
            for fn in sorted(files):
                fpath = os.path.join(root, fn)
                rel = os.path.relpath(fpath, arch)
                try:
                    sz, c = _fp_file_content(fpath)
                    tot += sz
                    crc = zlib.crc32(f"{rel}:{sz}:{c}".encode(), crc)
                except Exception as e:
                    # Не скрывать недоступный файл: fingerprint должен явно показывать UNKNOWN.
                    errors.append([rel, type(e).__name__])
        fp["archive_dir"] = [tot, crc]
        if errors:
            fp["archive_dir_errors"] = errors
    return fp


class _WorldGuard:
    def __init__(self, work=LEDGER_GUARD_DIR):
        self.work = work
        self.marker = os.path.join(work, "IN_PROGRESS")
        self.manifest_path = os.path.join(work, "manifest.json")

    def snapshot(self):
        if os.path.exists(self.marker):
            raise RuntimeError(
                f"Предыдущий прогон не завершился: исходное состояние лежит в {self.work}. "
                f"Верни его вызовом guard_recover() и повтори запуск.")
        shutil.rmtree(self.work, ignore_errors=True)
        os.makedirs(self.work)
        archive_root = os.path.abspath(ARCHIVE_DIR_LEDGER)
        man = {"had_world": os.path.exists(LIVING_WORLD_PATH),
               "had_archive": os.path.isdir(ARCHIVE_DIR_LEDGER), "extra": {}}
        if man["had_world"]:
            shutil.copy(LIVING_WORLD_PATH, os.path.join(self.work, "world"))
        if man["had_archive"]:
            shutil.copytree(ARCHIVE_DIR_LEDGER, os.path.join(self.work, "archive"))
        for i, fp in enumerate(_LEDGER_EXTRA_FILES + [_CHORUS_STATE_PATH]):
            if os.path.abspath(fp).startswith(archive_root + os.sep):
                continue
            if os.path.exists(fp):
                shutil.copy(fp, os.path.join(self.work, f"extra_{i}"))
                man["extra"][fp] = f"extra_{i}"
            else:
                man["extra"][fp] = None
        with open(self.manifest_path, "w", encoding="utf-8") as f:
            json.dump(man, f)
        with open(self.marker, "w") as f:
            f.write(time.strftime("%Y-%m-%d %H:%M:%S"))

    def restore(self):
        with open(self.manifest_path, "r", encoding="utf-8") as f:
            man = json.load(f)
        if man["had_world"]:
            shutil.copy(os.path.join(self.work, "world"), LIVING_WORLD_PATH)
        elif os.path.exists(LIVING_WORLD_PATH):
            os.remove(LIVING_WORLD_PATH)
        if man["had_archive"]:
            tmp = ARCHIVE_DIR_LEDGER.rstrip("/") + "_ledger_restore_tmp"
            shutil.rmtree(tmp, ignore_errors=True)
            shutil.copytree(os.path.join(self.work, "archive"), tmp)
            shutil.rmtree(ARCHIVE_DIR_LEDGER, ignore_errors=True)
            os.replace(tmp, ARCHIVE_DIR_LEDGER)
        else:
            shutil.rmtree(ARCHIVE_DIR_LEDGER, ignore_errors=True)
        for fp, name in man["extra"].items():
            if name is not None:
                shutil.copy(os.path.join(self.work, name), fp)
            elif os.path.exists(fp):
                os.remove(fp)
        globals()["_chorus_state"] = _load_chorus_state()

    def close(self):
        shutil.rmtree(self.work, ignore_errors=True)


def guard_recover():
    g = _WorldGuard()
    if not os.path.exists(g.marker):
        print("Незавершённых прогонов нет: восстанавливать нечего.")
        return False
    g.restore()
    g.close()
    print(f"Состояние восстановлено из {g.work}, снимок удалён.")
    return True


def _execute(seed, steps, overrides, engine_cls, ledger, verbose):
    overrides = dict(overrides or {})
    for name in overrides:
        if not hasattr(Config, name):
            raise AttributeError(f"Config не содержит параметра '{name}' (опечатка в overrides?)")
    saved = {name: getattr(Config, name) for name in list(overrides) + ["FORCE_GIVEN_SEED", "STEPS"]}
    guard = _WorldGuard()
    guard.snapshot()
    fp_before = _input_fingerprint()
    try:
        Config.FORCE_GIVEN_SEED, Config.STEPS = int(seed), int(steps)
        for name, value in overrides.items():
            setattr(Config, name, value)
        globals()["_chorus_state"] = _load_chorus_state()
        sink = io.StringIO()
        with (contextlib.nullcontext() if verbose else contextlib.redirect_stdout(sink)):
            eng = engine_cls()
            if ledger is not None:
                eng.ledger = ledger
                _cfg_snap = _config_snapshot()
                ledger.header = {"given_seed": int(eng.given_seed), "steps": int(steps),
                                 "overrides": overrides, "started": time.strftime("%Y-%m-%d %H:%M:%S"),
                                 "config_hash": zlib.crc32(repr(sorted(_cfg_snap.items())).encode("utf-8")),
                                 # Хеш у ON/OFF различается по определению (AUDIT_TAG, флаги), и по нему не видно,
                                 # ЧТО именно отличается. Снимок позволяет сравнивать прогоны по ключам.
                                 "config_snapshot": _cfg_snap,
                                 "input_fp_before": fp_before,
                                 "input_fp_method": "content-crc32-v2",
                                 "imprint_config": {k: v for k, v in vars(Config).items() if k.startswith("IMPRINT_")}}
            init_all_chronic_counters(eng)
            eng.run(force_fresh_seed=True)
        fp = _state_fingerprint(eng)
        if ledger is not None:
            ledger.header["final_state"] = fp
            # ВЫХОД прогона (до восстановления): living_world и архив пишет сам прогон, поэтому сравнивать это
            # с input_fp_before бессмысленно (раньше именно так считалось input_fp_stable -> всегда False).
            _out_fp = _input_fingerprint()
            ledger.header["output_fp_after"] = _out_fp
            ledger.header["run_mutated_files"] = sorted(k for k in set(fp_before) | set(_out_fp)
                                                        if fp_before.get(k) != _out_fp.get(k))
            try:   # read-only observation, must never affect the run
                ledger.header["memory_origin"] = memory_origin_report(eng.patterns)
                ledger.header["concept_graph_io"] = {k: v for k, v in GRAPH_IO.items()}
            except Exception as _e_mo:
                ledger.header["memory_origin_error"] = repr(_e_mo)[:200]
        del eng
        gc.collect()
        return fp
    finally:
        for name, value in saved.items():
            setattr(Config, name, value)
        guard.restore()
        if ledger is not None:
            try:   # входы ПОСЛЕ восстановления: это увидит следующий прогон; обязано совпасть с input_fp_before
                _fp_rest = _input_fingerprint()
                ledger.header["input_fp_after"] = _fp_rest
                ledger.header["input_fp_stable"] = bool(_fp_rest == fp_before)
                ledger.header["input_fp_diff_after_restore"] = sorted(k for k in set(fp_before) | set(_fp_rest)
                                                                     if fp_before.get(k) != _fp_rest.get(k))
            except Exception as _e_fp:
                ledger.header["input_fp_error"] = repr(_e_fp)[:200]
        guard.close()


def run_with_ledger(seed, steps, overrides=None, window=100, verbose=False,
                    save_path=None, save_every=500):
    ledger = ExperimentLedger(window=window)
    ledger.autosave_path, ledger.autosave_every = save_path, int(save_every)
    t0 = time.time()
    try:
        _execute(seed, steps, overrides, LedgerEngine, ledger, verbose)
        ledger.header["complete"] = True
    finally:
        ledger.header["wall_seconds"] = round(time.time() - t0, 1)
        if save_path and ledger.rows:
            ledger.save(save_path)
    return ledger


def check_observer_purity(steps=40, seed=118951255):
    print("=" * 70)
    print(f"ПРОВЕРКА НАБЛЮДАТЕЛЯ: {steps} тиков, given_seed={seed}")
    print("=" * 70)
    fp_plain = _execute(seed, steps, None, EvolutionEngine, None, False)
    print("  прогон без наблюдателя: готов")
    led1 = ExperimentLedger()
    fp_led1 = _execute(seed, steps, None, LedgerEngine, led1, False)
    print("  прогон с наблюдателем #1: готов")
    led2 = ExperimentLedger()
    _execute(seed, steps, None, LedgerEngine, led2, False)
    print("  прогон с наблюдателем #2: готов")
    same_final = fp_plain == fp_led1
    cmp12 = compare_ledgers(led1, led2)
    print(f"\nфинальное состояние: без наблюдателя vs с наблюдателем -> {'СОВПАДАЕТ' if same_final else 'РАЗЛИЧАЕТСЯ'}")
    if not same_final:
        print("   без:", fp_plain)
        print("   с:  ", fp_led1)
    print(f"цепочка хэшей: наблюдатель #1 vs #2 ({cmp12['ticks_compared']} тиков) -> "
          f"{'ИДЕНТИЧНА' if cmp12['identical'] else 'РАСХОДИТСЯ с t=' + str(cmp12['first_divergence_t'])}")
    ok = same_final and cmp12["identical"]
    print("\n" + ("✅ НАБЛЮДАТЕЛЬ ЧИСТЫЙ: симуляцию не меняет, прогон воспроизводим" if ok
                  else "❌ ПРОВЕРКА НЕ ПРОЙДЕНА: результаты ledger нельзя использовать как доказательство"))
    return ok


def periodicity_test(series, n_surr=500, seed=12345, min_cycles=3):
    from scipy.signal import lfilter
    x = np.asarray(series, dtype=float)
    x = x[np.isfinite(x)]
    N = len(x)
    if N < 40:
        return {"ok": False, "reason": "ряд короче 40 точек"}
    tt = np.arange(N)

    def detrend(y):
        return y - np.polyval(np.polyfit(tt[:len(y)], y, 1), tt[:len(y)])

    x = detrend(x)
    if x.std() == 0:
        return {"ok": False, "reason": "ряд константный"}
    freqs = np.fft.rfftfreq(N)
    band = freqs >= (min_cycles / N)
    band[0] = False

    def peak(y):
        y = y - y.mean()
        P = np.abs(np.fft.rfft(y)) ** 2
        k = np.argmax(np.where(band, P, -1.0))
        return P[k] / P[1:].sum(), 1.0 / freqs[k]

    obs_share, obs_period = peak(x)
    phi = float(np.clip(np.dot(x[:-1], x[1:]) / np.dot(x, x), -0.99, 0.99))
    sd_e = x.std() * np.sqrt(1 - phi ** 2)
    rng = np.random.default_rng(seed)
    ge = 0
    for _ in range(n_surr):
        eps = rng.normal(0.0, sd_e, N + 200)
        s = detrend(lfilter([1.0], [1.0, -phi], eps)[200:])
        if peak(s)[0] >= obs_share:
            ge += 1
    return {"ok": True, "n": N, "period": float(obs_period), "peak_share": float(obs_share),
            "phi": phi, "p_red_noise": (1 + ge) / (n_surr + 1), "n_surr": n_surr}


def cluster_test(X, k_max=6, n_null=100, seed=12345):
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score
    X = np.asarray(X, dtype=float)
    if X.ndim != 2 or X.shape[0] < 20:
        return {"ok": False, "reason": "нужно >= 20 строк"}
    sd = X.std(axis=0)
    keep = sd > 0
    if keep.sum() < 2:
        return {"ok": False, "reason": "меньше двух непостоянных признаков"}
    Z = (X[:, keep] - X[:, keep].mean(axis=0)) / sd[keep]

    def best(M):
        res = (-1.0, None)
        for k in range(2, min(k_max, len(M) - 1) + 1):
            lab = KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(M)
            if len(set(lab)) < 2:
                continue
            s = silhouette_score(M, lab)
            if s > res[0]:
                res = (float(s), k)
        return res

    obs_s, obs_k = best(Z)
    rng = np.random.default_rng(seed)
    ge = 0
    for _ in range(n_null):
        M = np.column_stack([rng.permutation(Z[:, j]) for j in range(Z.shape[1])])
        if best(M)[0] >= obs_s:
            ge += 1
    return {"ok": True, "n_rows": int(Z.shape[0]), "features": int(Z.shape[1]),
            "best_k": obs_k, "silhouette": obs_s, "p_permutation_null": (1 + ge) / (n_null + 1),
            "n_null": n_null}


print("Cell 6 v1.5 готова: ExperimentLedger / LedgerEngine / run_with_ledger / "
      "check_observer_purity / compare_ledgers / periodicity_test / cluster_test / guard_recover / "
      "_input_fingerprint")


In [ ]:
# Ячейка 27
import os
import json
import time
from collections import Counter
import numpy as np

SNAPSHOT_EVERY = 10
SAVE_EVERY = 100
RECORD_CAP = 400_000

_RX_KEYS = (
    "attempted", "detected", "decoded", "corrupted",
    "uncertain", "collision", "timeout", "false_positive"
)


class RichLog:
    def __init__(self):
        self.header = {}
        self.tick_rows = []
        self.events = []
        self.dialogues = []
        self.speaks = []
        self.cluster_births = []
        self.agent_snaps = []
        self.tick_wall = []          # observer: [t, sim_s, log_s, pop, ferals] по КАЖДОМУ тику
        self._wall_t0 = None
        self._body_cap_trim_seen = 0
        self._body_nav_prev = {}
        # High-resolution, read-only movement history for 1/5/20-tick heading diagnostics.
        self._move_accum_seen = {}
        self._move_tick_hist = {}
        self._intent_switch_age_seen = {}
        self._intent_switch_window_count = {}

        self._rx_prev = {}
        self._rx_cum = Counter()
        self._pre_state = {}
        self._dlg_refs = {}
        self._spk_refs = {}
        self._dlg_seen = set()
        self._spk_seen = set()
        self._n_clusters_prev = 0
        self._dlg_total = 0
        self._spk_total = 0

        self.errors = Counter()
        self.autosave_path = None

    @staticmethod
    def _state_of(p):
        return {
            "intent": (p.intent.get("type") if p.intent else None),
            "state": str(p.semantic_state),
            "substate": getattr(p, "_substate", None),
            "soul": round(float(p.soul_weight), 4),
            "nci": round(float(getattr(p, "_nci", 0.0)), 4),
            "gap": round(float(getattr(p, "spirit_gap", 0.0)), 4),
            "grief": round(float(p.emotional_memory.get("grief", 0.0)), 4),
            "grat": round(float(p.emotional_memory.get("gratitude", 0.0)), 4),
            "trust_n": len(p.trust_ledger.entries),
            "dlg_n": len(getattr(p, "dialogue_longterm", []) or []),
            "spk_n": len(getattr(p, "spoken_lexicon", []) or []),
            "role": str(p.role_type),
            "shield": float(p.protection_level),
            "narrative": bool(getattr(p, "_narrative_agent", False)),
        }

    @staticmethod
    def _trust_diff(prev, now, limit=10):
        out = []
        for k in set(prev.keys()) | set(now.keys()):
            a = prev.get(k)
            b = now.get(k)
            if a != b and b is not None:
                out.append((
                    int(k),
                    a,
                    float(b),
                    float(b) - float(a if a is not None else 0)
                ))
        out.sort(key=lambda x: -abs(x[3]))
        return out[:limit]

    @staticmethod
    def _speaker_state(p):
        return {
            "uc": round(float(p.unresolved_contradiction), 4),
            "gap": round(float(getattr(p, "spirit_gap", 0.0)), 4),
            "drag": round(float(getattr(p, "somatic_drag", 1.0)), 4),
            "grief": round(float(p.emotional_memory.get("grief", 0.0)), 4),
            "grat": round(float(p.emotional_memory.get("gratitude", 0.0)), 4),
            "soul": round(float(p.soul_weight), 4),
            "intent": (p.intent.get("type") if p.intent else None),
            "role": str(p.role_type),
        }

    def _record_dialogues(self, p):
        cur = list(getattr(p, "dialogue_longterm", None) or [])
        if p.id not in self._dlg_refs:
            self._dlg_refs[p.id] = cur
            return
        seen = {id(e) for e in self._dlg_refs[p.id]}
        new = [e for e in cur if id(e) not in seen]
        for e in new:
            if not isinstance(e, dict):
                continue
            t = e.get("t")
            partner = e.get("partner")
            text = e.get("text", "")
            key = (
                int(p.id),
                t,
                partner,
                str(text)[:40],
            )
            if key in self._dlg_seen:
                continue
            self._dlg_seen.add(key)
            self.dialogues.append({
                "pid": int(p.id),
                "age": int(p.age),
                "t": t,
                "partner": partner,
                "text": str(text)[:200],
                "soul": e.get("soul_at_moment"),
                "grief": e.get("grief_at_moment"),
                "grat": e.get("grat_at_moment"),
            })
            self._dlg_total += 1
            self.events.append({
                "type": "dialogue",
                "t": t,
                "pid": int(p.id),
                "partner": partner,
                "text": str(text)[:120],
            })
        self._dlg_refs[p.id] = cur

    def _record_speaks(self, p):
        cur = list(getattr(p, "spoken_lexicon", None) or [])
        if p.id not in self._spk_refs:
            self._spk_refs[p.id] = cur
            return
        seen = {id(e) for e in self._spk_refs[p.id]}
        new = [e for e in cur if id(e) not in seen]
        for e in new:
            if not isinstance(e, dict):
                continue
            utt = e.get("utterance")
            if isinstance(utt, tuple):
                utt = list(utt)
            t = e.get("t")
            to = e.get("to")
            key = (
                int(p.id),
                t,
                tuple(utt) if isinstance(utt, list) else None,
            )
            if key in self._spk_seen:
                continue
            self._spk_seen.add(key)
            self.speaks.append({
                "pid": int(p.id),
                "age": int(p.age),
                "t": t,
                "to": to,
                "utt": utt,
                "st": self._speaker_state(p),
            })
            self._spk_total += 1
            self.events.append({
                "type": "speak",
                "t": int(t) if t is not None else t,
                "pid": int(p.id),
                "to": to,
                "utt": utt,
            })
        self._spk_refs[p.id] = cur

    def before_step(self, eng):
        snap = {}
        for p in eng.patterns:
            if not p.alive:
                continue
            st = self._state_of(p)
            st["_trust_snap"] = dict(p.trust_ledger.entries)
            snap[p.id] = st
        self._pre_state = snap

    def after_step(self, eng, t):
        lex = getattr(eng, "EMERGENT_LEXICON", None)
        if lex is None:
            self.errors["EMERGENT_LEXICON отсутствует"] += 1
            return

        alive = [p for p in eng.patterns if p.alive]
        alive_ids = {p.id for p in alive}

        # ЕДИНЫЙ ИСТОЧНИК ЧИСЛЕННОСТИ: ledger.after() отрабатывает раньше (LedgerEngine.step), его n обязан совпасть с нашим.
        try:
            _led_ = getattr(eng, "ledger", None)
            if _led_ is not None and _led_.rows and _led_.rows[-1].get("t") == int(t):
                _ln_ = _led_.rows[-1].get("n")
                if _ln_ != len(alive):
                    _hv_ = self.header.setdefault("pop_source_violations", [])
                    self.header["pop_source_violations_n"] = int(self.header.get("pop_source_violations_n", 0)) + 1
                    if len(_hv_) < 50:
                        _hv_.append({"t": int(t), "log_n": len(alive), "ledger_n": _ln_})
        except Exception as _e_ps:
            self.errors[f"pop_source_check:{type(_e_ps).__name__}"] += 1

        # Import append-only body-cap events from the engine into the persisted Observer log.
        _cap_trim_log = getattr(eng, '_body_cap_trim_log', []) or []
        if self._body_cap_trim_seen > len(_cap_trim_log):
            self._body_cap_trim_seen = 0
        for _entry in _cap_trim_log[self._body_cap_trim_seen:]:
            self.events.append({'type': 'body_cap_trim', **_entry})
        self._body_cap_trim_seen = len(_cap_trim_log)

        for p in alive:
            prev = self._pre_state.get(p.id)
            now = self._state_of(p)
            if prev is not None:
                changed = {
                    k: (prev[k], now[k])
                    for k in now
                    if prev[k] != now[k]
                }
                if changed:
                    self.events.append({
                        "type": "state_change",
                        "t": int(t),
                        "pid": int(p.id),
                        "changes": {
                            k: {
                                "from": v[0],
                                "to": v[1]
                            }
                            for k, v in changed.items()
                        },
                    })
                td = self._trust_diff(
                    prev.get("_trust_snap", {}),
                    dict(p.trust_ledger.entries)
                )
                if td:
                    self.events.append({
                        "type": "trust_change",
                        "t": int(t),
                        "pid": int(p.id),
                        "deltas": td,
                    })
            self._record_dialogues(p)
            self._record_speaks(p)

        for store in (
            self._dlg_refs,
            self._spk_refs,
            self._rx_prev,
            self._pre_state,
        ):
            for pid in [
                k for k in list(store)
                if k not in alive_ids
            ]:
                del store[pid]

        n_cl = len(lex.clusters)

        for cid in range(self._n_clusters_prev, n_cl):
            m = lex.cluster_meanings.get(cid)
            self.cluster_births.append({
                "t": int(t),
                "cid": int(cid),
                "mean": (
                    [float(x) for x in m]
                    if m is not None else None
                ),
                "first_senders": {
                    str(k): int(v)
                    for k, v in
                    lex.cluster_senders.get(cid, {}).items()
                },
                "raw": lex.clusters[cid],
            })
            self.events.append({
                "type": "cluster_birth",
                "t": int(t),
                "cid": int(cid),
                "raw": lex.clusters[cid],
                "first_senders": list(
                    lex.cluster_senders.get(cid, {}).keys()
                ),
            })

        self._n_clusters_prev = max(
            self._n_clusters_prev,
            n_cl
        )

        for p in alive:
            r = getattr(p, "_morse_rx", None)
            m = getattr(r, "metrics", None) if r is not None else None
            if not m:
                continue
            prev = self._rx_prev.get(p.id, {})
            for k in _RX_KEYS:
                v = int(m.get(k, 0))
                inc = max(0, v - prev.get(k, 0))
                if inc:
                    self._rx_cum[k] += inc
                    if k in (
                        "decoded",
                        "corrupted",
                        "uncertain",
                    ):
                        self.events.append({
                            "type": f"rx_{k}",
                            "t": int(t),
                            "pid": int(p.id),
                            "inc": inc,
                        })
            self._rx_prev[p.id] = {
                k: int(m.get(k, 0))
                for k in _RX_KEYS
            }

        # Observer-only: recover this tick's displacement from the cumulative movement
        # accumulator without changing simulation state or increasing snapshot frequency.
        _alive_ids = set()
        for _mp in alive:
            _mid = int(_mp.id)
            _alive_ids.add(_mid)
            _raw_mv = getattr(_mp, "_move_vector_accum", (0.0, 0.0))
            _cur_mv = (float(_raw_mv[0]), float(_raw_mv[1]))
            _prev_mv = self._move_accum_seen.get(_mid, (0.0, 0.0))
            _ddx, _ddy = _cur_mv[0] - _prev_mv[0], _cur_mv[1] - _prev_mv[1]
            self._move_accum_seen[_mid] = _cur_mv
            _switch_age = getattr(_mp, "last_intent_switch_age", None)
            _switch_prev_age = self._intent_switch_age_seen.get(_mid, _switch_age)
            if (_switch_age is not None and _switch_prev_age is not None
                    and int(_switch_age) != int(_switch_prev_age)):
                self._intent_switch_window_count[_mid] = self._intent_switch_window_count.get(_mid, 0) + 1
            self._intent_switch_age_seen[_mid] = _switch_age
            _hist = self._move_tick_hist.setdefault(_mid, [])
            _hist.append((int(t), float(_ddx), float(_ddy)))
            while _hist and _hist[0][0] < int(t) - 19:
                _hist.pop(0)
        # Drop dead-agent buffers so the observer remains bounded.
        for _mid in list(self._move_tick_hist):
            if _mid not in _alive_ids:
                self._move_tick_hist.pop(_mid, None)
                self._move_accum_seen.pop(_mid, None)
                self._intent_switch_age_seen.pop(_mid, None)
                self._intent_switch_window_count.pop(_mid, None)

        if t % SNAPSHOT_EVERY == 0:
            genes = [
                p._lang_gene
                for p in alive
                if getattr(p, "_lang_gene", None) is not None
            ]
            if len(genes) >= 2:
                arr = np.stack(genes)
                gd = float(np.mean(np.std(arr, axis=0)))
                gm = float(np.mean(arr))
            else:
                gd = gm = 0.0
            fid = list(lex.fidelity_log)[-200:]
            self.tick_rows.append({
                "t": int(t),
                "n": len(alive),
                "n_clusters": n_cl,
                "cluster_uses": int(
                    sum(lex.cluster_uses.values())
                ),
                "rx_cum": dict(self._rx_cum),
                "reinf": int(lex.reinforcement_events),
                "gene_div": gd,
                "gene_mean": gm,
                "fidelity": (
                    float(np.mean(fid))
                    if fid else float("nan")
                ),
                "dlg_total": self._dlg_total,
                "spk_total": self._spk_total,
                "send_k": dict(lex.send_k_counts),
                "send_silence": dict(lex.send_silence),
                "send_energy": float(
                    lex.send_energy_spent
                ),
            })

            _alive_snap = [p for p in eng.patterns if p.alive]
            _E_f = float(eng.field[:, :, CH['energy']].sum())
            _E_i = float(sum(getattr(p, 'energy', 0.0) for p in _alive_snap))
            self.tick_rows[-1]['E_field'] = _E_f
            self.tick_rows[-1]['E_inv'] = _E_i
            self.tick_rows[-1]['E_total'] = _E_f + _E_i
            try:
                self.tick_rows[-1]['pop'] = eng._pop_snapshot()   # observer: только факты, состояние не трогает
            except Exception as _pe:
                self.errors['pop_snapshot'] = self.errors.get('pop_snapshot', 0) + 1
            try:   # итоги ledger на этот же тик: Observer сверяет с ними engine-счётчики (_pop_note)
                _lg = getattr(eng, 'ledger', None)
                if _lg is not None and _lg.rows and isinstance(self.tick_rows[-1].get('pop'), dict):
                    self.tick_rows[-1]['pop']['ledger'] = _lg.totals()
            except Exception as _pl:
                self.errors['pop_ledger_totals'] = self.errors.get('pop_ledger_totals', 0) + 1
            _led = getattr(eng, '_energy_ledger', None) or {}
            _src = _led.get('src', {})
            _x2 = _led.get('x2', {})
            self.header['energy_snapshot'] = {
                't': int(t),
                'field': _E_f,
                'agent_inventory': _E_i,
                'total': _E_f + _E_i,
                'src': {n: [round(v[0], 4), round(v[1], 4), int(v[2])] for n, v in _src.items()},
                'x2': {n: [round(v[0], 4), round(v[1], 4), int(v[2])] for n, v in _x2.items()},
                'step_total': float(_led.get('step_total', 0.0)),
                'step_measured': float(_led.get('step_measured', 0.0)),
                'ticks': int(_led.get('ticks', 0)),
            }

            from scipy.ndimage import convolve as _conv
            _kn = np.array(
                [[0, 1, 0],
                 [1, 0, 1],
                 [0, 1, 0]],
                dtype=np.float32
            )
            _sig = eng.field[:, :, CH["signal_grief"]]
            _raw_map = _conv(
                _sig,
                _kn,
                mode="wrap"
            )
            _feral_map = _conv(
                eng.field[:, :, CH["signal_feral"]],
                np.ones(
                    (7, 7),
                    dtype=np.float32
                ),
                mode="wrap"
            )
            _own = eng.field[:, :, CH["owner"]]
            _R6 = 6
            _yy, _xx = np.mgrid[-_R6:_R6 + 1, -_R6:_R6 + 1]
            _disk = ((_xx ** 2 + _yy ** 2) <= _R6 * _R6).astype(np.float32)
            _valid = np.ones(_own.shape, dtype=np.float32)
            if Config.ENABLE_CENTER_WALL:
                _valid[Config.CENTER_WALL_X, :] = 0.0
            _lf_map = _conv((_own != 0).astype(np.float32) * _valid, _disk, mode="wrap") / np.maximum(_conv(_valid, _disk, mode="wrap"), 1e-9)

            for p in alive:
                g = getattr(p, "_lang_gene", None)
                _cells = list(p.cells)
                if _cells:
                    _xs = [cc[0] for cc in _cells]
                    _ys = [cc[1] for cc in _cells]
                    _crisis = float(
                        np.mean(
                            eng.field[
                                _xs,
                                _ys,
                                CH["crisis"]
                            ]
                        )
                    )
                    _feral_near = float(
                        np.sum(
                            _feral_map[
                                _xs,
                                _ys
                            ]
                        )
                    )
                    _bind_loc = float(
                        np.mean(
                            np.clip(
                                eng.field[
                                    _xs,
                                    _ys,
                                    CH["binding"]
                                ],
                                0,
                                1
                            )
                        )
                    )
                    _nbr = float(
                        np.mean(
                            _raw_map[
                                _xs,
                                _ys
                            ]
                        )
                    )
                    _msk = (
                        _own != p.id
                    ).astype(np.float32)
                    _tn = float(
                        np.sum(
                            _conv(
                                _msk,
                                _kn,
                                mode="wrap"
                            )[
                                _xs,
                                _ys
                            ]
                        )
                    )
                    _nbrN = (
                        float(
                            np.sum(
                                _conv(
                                    _sig * _msk,
                                    _kn,
                                    mode="wrap"
                                )[
                                    _xs,
                                    _ys
                                ]
                            )
                        ) / _tn
                        if _tn > 0
                        else float("nan")
                    )
                else:
                    _crisis = _nbr = _nbrN = float("nan")
                    _feral_near = _bind_loc = float("nan")

                _gc = p.get_center()
                _gx = int(_gc[0]) % Config.WORLD_SIZE
                _gy = int(_gc[1]) % Config.WORLD_SIZE
                _nvn = getattr(p, "_nav", None)
                _cpl = _nvn.get("current_place") if _nvn else None
                _ns = p.nav_state() if hasattr(p, "nav_state") else None
                _np_ = getattr(p, "_nav_progress", None)
                _dp = getattr(p, "_dir_parts", None) or {}
                _nm = p.nav_map_summary() if hasattr(p, "nav_map_summary") else None
                _sd = getattr(p, "_soma_diag", None)
                _sd = _sd if isinstance(_sd, dict) else {}
                _body_comps = (p._components(p.cells, int(Config.WORLD_SIZE))
                               if getattr(p, "cells", None) and hasattr(p, "_components") else [])
                _largest_body = _body_comps[0] if _body_comps else set(getattr(p, "cells", set()) or ())
                try:
                    _nav_c = p._nav_body_center() if hasattr(p, "_nav_body_center") else p.get_center()
                    _nav_c = (float(_nav_c[0]), float(_nav_c[1]))
                except Exception:
                    _nav_c = None
                _body_axis = None
                _body_shape = _sd.get("shape")
                _body_elongation = _sd.get("elongation")
                if (_body_shape in ("rod", "dumbbell")
                        and len(_largest_body) >= 3 and _nav_c is not None):
                    _Wbody = float(Config.WORLD_SIZE)
                    _xy = np.asarray([
                        (((float(x) - _nav_c[0] + _Wbody / 2) % _Wbody) - _Wbody / 2,
                         ((float(y) - _nav_c[1] + _Wbody / 2) % _Wbody) - _Wbody / 2)
                        for x, y in _largest_body
                    ], dtype=np.float64)
                    _cov = np.cov(_xy, rowvar=False)
                    _ev, _evec = np.linalg.eigh(_cov)
                    if _ev[-1] > 1e-9 and (_ev[-1] - _ev[-2]) / max(1e-9, _ev[-1]) > 0.05:
                        _body_axis = [float(_evec[0, -1]), float(_evec[1, -1])]
                _mom = getattr(p, "_dir_momentum", None)
                _best_dir = getattr(p, "_best_direction", None)
                _actual_disp = _actual_heading = _intent_unit = _center_disp = None
                _intent_actual_dot = _axis_actual_alignment = None
                _prev_nav = self._body_nav_prev.get(int(p.id))
                _actual_raw = getattr(p, '_move_vector_accum', (0.0, 0.0))
                _actual_disp = [float(_actual_raw[0]), float(_actual_raw[1])]
                p._move_vector_accum = (0.0, 0.0)
                self._move_accum_seen[int(p.id)] = (0.0, 0.0)
                _dn = float(np.hypot(*_actual_disp))
                if _dn > 1e-9:
                    _actual_heading = [float(_actual_disp[0] / _dn), float(_actual_disp[1] / _dn)]
                _intent_type = (p.intent.get("type") if getattr(p, "intent", None) else None)
                _last_switch_age = getattr(p, "last_intent_switch_age", None)
                _intent_changed = bool(
                    _prev_nav is not None and _intent_type is not None
                    and _prev_nav.get("intent_type") is not None
                    and _intent_type != _prev_nav.get("intent_type")
                )
                _switch_count_since_prev = int(self._intent_switch_window_count.get(int(p.id), 0))
                self._intent_switch_window_count[int(p.id)] = 0
                _switch_since_prev = (_switch_count_since_prev > 0)
                _hist = self._move_tick_hist.get(int(p.id), [])
                _momentum_corr_by_horizon = {}
                _local_fill = float(np.mean(_lf_map[_xs, _ys])) if _cells else float('nan')
                if _mom is not None and len(_mom) >= 2:
                    _mom_norm = float(np.hypot(float(_mom[0]), float(_mom[1])))
                    if _mom_norm > 1e-9:
                        _mu = np.asarray([float(_mom[0]) / _mom_norm, float(_mom[1]) / _mom_norm])
                        for _horizon in (1, 5, 10, 20):
                            _vecs = [(vx, vy) for ht, vx, vy in _hist
                                     if int(t) - _horizon < int(ht) <= int(t)]
                            if len(_vecs) >= _horizon:
                                _sumv = np.sum(np.asarray(_vecs, dtype=np.float64), axis=0)
                                _snorm = float(np.hypot(*_sumv))
                                if _snorm > 1e-9:
                                    _momentum_corr_by_horizon[str(_horizon)] = float(
                                        np.dot(_sumv / _snorm, _mu))
                                else:
                                    _momentum_corr_by_horizon[str(_horizon)] = None
                            else:
                                _momentum_corr_by_horizon[str(_horizon)] = None
                if _nav_c is not None and _prev_nav is not None and _prev_nav.get('center') is not None:
                    _Wnav = float(Config.WORLD_SIZE)
                    _dxnav = ((_nav_c[0] - _prev_nav['center'][0] + _Wnav / 2) % _Wnav) - _Wnav / 2
                    _dynav = ((_nav_c[1] - _prev_nav['center'][1] + _Wnav / 2) % _Wnav) - _Wnav / 2
                    _center_disp = [float(_dxnav), float(_dynav)]
                self._body_nav_prev[int(p.id)] = {
                    't': int(t), 'center': _nav_c, 'intent_type': _intent_type,
                    'last_intent_switch_age': (int(_last_switch_age) if _last_switch_age is not None else None)
                }
                if _mom is not None and len(_mom) >= 2:
                    _mn = float(np.hypot(float(_mom[0]), float(_mom[1])))
                    if _mn > 1e-9:
                        _intent_unit = [float(_mom[0]) / _mn, float(_mom[1]) / _mn]
                if _actual_heading is not None and _intent_unit is not None:
                    _intent_actual_dot = float(np.dot(_actual_heading, _intent_unit))
                if _actual_heading is not None and _body_axis is not None:
                    _axis_actual_alignment = abs(float(np.dot(_actual_heading, _body_axis)))
                _bodyori_obs = (p._bodyori_snapshot() if hasattr(p, "_bodyori_snapshot") else None)
                self.agent_snaps.append({
                    "t": int(t),
                    "id": int(p.id),
                    "age": int(p.age),
                    "feral_flag": bool(getattr(p, "role_type", None) == "feral"),
                    "role_type": str(getattr(p, "role_type", "unknown")),
                    "soma_fragment": int(_sd.get("fragment_count", len(_body_comps) or 1)),
                    "body_cells": int(len(getattr(p, "cells", ()) or ())),
                    "body_frag_live": int(len(_body_comps) or (1 if getattr(p, "cells", None) else 0)),
                    "body_largest_frac": (round(len(_largest_body) / max(1, len(p.cells)), 4)
                                          if getattr(p, "cells", None) else 0.0),
                    "body_thin_bridge": (_sd.get("thin_bridge_risk")),
                    "body_holes": (_sd.get("holes")),
                    "body_elongation": (_sd.get("elongation")),
                    "body_shape": _body_shape,
                    "local_fill": _local_fill,
                    "intent_type": _intent_type,
                    "intent_changed_since_prev": _intent_changed,
                    "intent_switch_since_prev": _switch_since_prev,
                    "intent_switch_count_since_prev": _switch_count_since_prev,
                    "momentum_corr_1t": _momentum_corr_by_horizon.get("1"),
                    "momentum_corr_5t": _momentum_corr_by_horizon.get("5"),
                    "momentum_corr_10t": _momentum_corr_by_horizon.get("10"),
                    "momentum_corr_20t": _momentum_corr_by_horizon.get("20"),
                    "body_diag_t": (_sd.get("t")),
                    "body_diag_age": (int(t) - int(_sd["t"]) if _sd.get("t") is not None else None),
                    "body_nav_center": (list(_nav_c) if _nav_c is not None else None),
                    "body_axis": _body_axis,
                    "body_heading": (_bodyori_obs.get("heading") if _bodyori_obs else None),
                    "body_heading_prev": (_bodyori_obs.get("heading_prev") if _bodyori_obs else None),
                    "body_angular_velocity": (_bodyori_obs.get("angular_velocity") if _bodyori_obs else None),
                    "body_turn_intent": (_bodyori_obs.get("turn_intent") if _bodyori_obs else None),
                    "body_orientation_confidence": (_bodyori_obs.get("orientation_confidence") if _bodyori_obs else None),
                    "body_heading_x": (_bodyori_obs.get("heading_x") if _bodyori_obs else None),
                    "body_heading_y": (_bodyori_obs.get("heading_y") if _bodyori_obs else None),
                    "momentum_heading": _intent_unit,
                    "intent_heading": _intent_unit,  # legacy alias: this is EMA momentum, not a goal heading
                    "actual_displacement": _actual_disp,
                    "actual_heading": _actual_heading,
                    "center_displacement": _center_disp,
                    "actual_interval_ticks": (int(t) - int(_prev_nav['t']) if _prev_nav is not None else None),
                    "momentum_corr": _momentum_corr_by_horizon.get("10", _intent_actual_dot),
                    "intent_actual_dot": _momentum_corr_by_horizon.get("10", _intent_actual_dot),  # legacy alias
                    "body_axis_motion_abs_cos": _axis_actual_alignment,
                    "body_axis_actual_alignment": _axis_actual_alignment,  # legacy alias; absolute cosine
                    "best_direction": (list(map(int, _best_dir)) if _best_dir is not None else None),
                    "move_t": getattr(p, "_last_move_t", None),
                    "move_attempt": bool(getattr(p, "_last_move_attempt", False)),
                    "move_ok": bool(getattr(p, "_last_move_ok", False)),
                    "move_reason": getattr(p, "_last_move_reason", None),
                    "alive": bool(getattr(p, "alive", True)),
                    "nav_status": str(_ns["status"]) if _ns else "none",
                    "nav_dist": _ns["dist"] if _ns else None,
                    "nav_recalc": int(_ns["recalc"]) if _ns else 0,
                    "nav_progress": float(_ns["progress"]) if _ns else 0.0,
                    "nav_reason": (str(_ns["reason"]) if _ns and _ns["reason"] is not None else None),
                    "nav_target": ([int(v) for v in _np_["target"]] if _np_ else None),
                    "nav_origin": (str(_np_.get("origin")) if _np_ else None),
                    "nav_map_cells": int(_nm["cells"]) if _nm else 0,
                    "nav_map_fresh": int(_nm["fresh"]) if _nm else 0,
                    "nav_map_stale": int(_nm["stale"]) if _nm else 0,
                    "nav_map_danger": int(_nm["danger"]) if _nm else 0,
                    "nav_map_obstacle": int(_nm["obstacle"]) if _nm else 0,
                    "nav_map_unknown": int(_nm["unknown"]) if _nm else 0,
                    "dir_best": ([int(v) for v in _dp["dir"]] if _dp.get("dir") else None),
                    "dir_res": round(float(_dp.get("res", 0.0)), 4),
                    "dir_unk": round(float(_dp.get("unk", 0.0)), 4),
                    "dir_threat": round(float(_dp.get("threat", 0.0)), 4),
                    "dir_conf": round(float(_dp.get("conf", 0.0)), 4),
                    "dir_conf_valid": bool("conf" in _dp),
                    "dir_reach": int(_dp.get("reach", 0)),
                    "dir_score": round(float(_dp.get("score", 0.0)), 4),
                    "dir_hyst_off": bool(_dp.get("hyst_off", False)),
                    "gapclip_choice": (getattr(p, "_gapclip_last", None) or (None, None))[1],
                    "dir_contrast": float(getattr(p, "_dir_contrast", 0.0)),
                    "gap_clip": float(getattr(p, "gap_clip", 0.0)),
                    "spirit_gap_raw": float(getattr(p, "spirit_gap", 0.0)),
                    "noise_gain": float(getattr(p, "noise_gain", np.nan)),
                    "noise_filter": float((getattr(p, "genome", {}) or {}).get("noise_filter", np.nan)),
                    "noise_channel_gain": (
                        float(getattr(p, "noise_gain", np.nan)) *
                        (1.0 - 0.3 * float((getattr(p, "genome", {}) or {}).get("noise_filter", 0.0)))
                        if getattr(p, "noise_gain", None) is not None else None
                    ),
                    "smoothed_pred_error": float(getattr(p, "smoothed_pred_error", np.nan)),
                    "pred_error": float(getattr(p, "pred_error", np.nan)),
                    "err_dim": [round(float(x), 4) for x in getattr(p, '_err_dim_acc', np.zeros(8))],
                    "avg_real_dim": [round(float(x), 4) for x in getattr(p, '_avg_real_dim', np.zeros(8))],
                    "pred_dim": [round(float(x), 4) for x in getattr(p, '_pred_dim', np.zeros(8))],
                    "avg_real_raw_dim": [round(float(x), 4) for x in getattr(p, '_avg_real_raw_dim', np.zeros(8))],
                    "clip_dim": [round(float(x), 4) for x in getattr(p, '_clip_dim_acc', np.zeros(8))],
                    "err_dim_stale": int(p.age - getattr(p, '_err_dim_age', (p.age,))[0]),
                    **_obs_snap_extra(p),
                    "x": _gx,
                    "y": _gy,
                    "zone": int(_feral_sanctuary_zone(_gx, _gy, eng)),
                    "sanc_entries": int(getattr(p, "_sanctuary_entries", 0)),
                    "place_id": (int(_cpl["id"]) if _cpl else None),
                    "place_visits": (len(_cpl["visits"]) if _cpl else 0),
                    "gene": (
                        [float(x) for x in g]
                        if g is not None else None
                    ),
                    "nci": round(
                        float(
                            getattr(
                                p, "_nci", 0.0
                            )
                        ),
                        4
                    ),
                    "soul": round(
                        float(p.soul_weight),
                        4
                    ),
                    "shield": float(
                        p.protection_level
                    ),
                    "narrative": bool(
                        getattr(
                            p,
                            "_narrative_agent",
                            False
                        )
                    ),
                    "intent": (
                        p.intent.get("type")
                        if p.intent else None
                    ),
                    "state": str(
                        p.semantic_state
                    ),
                    "spoken_n": len(
                        getattr(
                            p,
                            "spoken_lexicon",
                            []
                        ) or []
                    ),
                    "dlg_n": len(
                        getattr(
                            p,
                            "dialogue_longterm",
                            []
                        ) or []
                    ),
                    "lexmem_n": len(
                        getattr(
                            p,
                            "lexicon_memory",
                            {}
                        ) or {}
                    ),
                    "dlg_stats": (
                        dict(p._dlg_stats)
                        if getattr(
                            p,
                            "_dlg_stats",
                            None
                        ) else None
                    ),
                    "grief": float(
                        p.emotional_memory.get(
                            "grief", 0.0
                        )
                    ),
                    "grief_gt_half": bool(
                        p.emotional_memory.get(
                            "grief", 0.0
                        ) > 0.5
                    ),
                    "contra_gt_knot": bool(
                        p.unresolved_contradiction >
                        Config.LEXICON_KNOT_THRESHOLD
                    ),
                    "soma_var": float(
                        p.soma_vector[0]
                    ),
                    "local_crisis": _crisis,
                    "neighbor_grief": _nbr,
                    "neighbor_grief_own_excl": _nbrN,
                    "grat": float(
                        p.emotional_memory.get(
                            "gratitude", 0.0
                        )
                    ),
                    "uc": float(
                        p.unresolved_contradiction
                    ),
                    "gap": float(
                        getattr(
                            p,
                            "spirit_gap",
                            0.0
                        )
                    ),
                    "drag": float(
                        getattr(
                            p,
                            "somatic_drag",
                            1.0
                        )
                    ),
                    "scar": float(
                        getattr(
                            p,
                            "epistemic_scar",
                            0.0
                        )
                    ),
                    "crisis_mem": float(
                        getattr(
                            p,
                            "crisis_memory",
                            0.0
                        )
                    ),
                    "root_q": (
                        getattr(
                            p,
                            "_root_question",
                            None
                        ) is not None
                    ),
                    "subject": bool(
                        getattr(
                            p,
                            "_subject_detected",
                            False
                        )
                    ),
                    "cns": float(
                        getattr(
                            p,
                            "_concept_narrative_score",
                            0.0
                        )
                    ),
                    "meta_s": float(
                        getattr(
                            p,
                            "_meta_surprise",
                            0.0
                        )
                    ),
                    "n_conc": len(
                        getattr(
                            getattr(
                                p,
                                "concept_graph",
                                None
                            ),
                            "nodes",
                            None
                        ) or {}
                    ),
                    "n_edge": len(
                        getattr(
                            getattr(
                                p,
                                "concept_graph",
                                None
                            ),
                            "edges",
                            None
                        ) or {}
                    ),
                    "tension": float(
                        getattr(
                            p,
                            "cognitive_tension",
                            0.0
                        )
                    ),
                    "n_cells": len(_cells),
                    "fear": float(
                        getattr(
                            p,
                            "alarm_level",
                            0.0
                        )
                    ),
                    "surprise": abs(
                        float(
                            getattr(
                                p,
                                "surprise_signal",
                                0.0
                            )
                        )
                    ),
                    "fury": float(
                        getattr(
                            p,
                            "_feral_fury",
                            0.0
                        )
                    ),
                    "feral_near": _feral_near,
                    "bind_loc": _bind_loc,
                    "enc_strain": round(float(p._enc_strain), 4),
                    "spatial_err": round(float(getattr(p, "spatial_pred_error", 0.0)), 4),
                    "place_conf": round(float(getattr(p, "place_confidence", 0.5)), 4),
                    "pn2": (globals().get("_pn2_snap") or (lambda _p: None))(p),
                    "n_imprints": len(p.imprints.by_other),
                    "endur": round(float(p._cellular_endurance), 3),
                    "prey": int(is_feral_prey(p)),
                    "lf": round(float(_lf_map[int(p.get_center()[0]) % Config.WORLD_SIZE, int(p.get_center()[1]) % Config.WORLD_SIZE]), 3),
                    "body_mem": float(
                        getattr(
                            p,
                            "body_memory",
                            0.0
                        )
                    ),
                    "substate": (
                        None
                        if getattr(
                            p,
                            "_substate",
                            None
                        ) is None
                        else str(p._substate)
                    ),
                })

        for name in (
            "events",
            "dialogues",
            "speaks"
        ):
            lst = getattr(self, name)
            if len(lst) > RECORD_CAP:
                setattr(
                    self,
                    name,
                    lst[-RECORD_CAP:]
                )

    @staticmethod
    def _hms(sec):
        sec = int(max(0, sec))
        return f"{sec // 3600}:{sec % 3600 // 60:02d}:{sec % 60:02d}"

    def note_tick_wall(self, eng, t, d_sim, d_pre, d_post):
        """Время тика (observer, read-only). sim = сама симуляция; лог = before_step + after_step логгера.
        Печатает прогресс: первые 5 тиков, затем каждые 10 и последний, чтобы скорость была видна сразу, а не через часы."""
        now = time.perf_counter()
        if self._wall_t0 is None:
            self._wall_t0 = now - (d_sim + d_pre + d_post)
        alive = [p for p in eng.patterns if p.alive]
        fer = sum(1 for p in alive if getattr(p, "role_type", None) == "feral")
        self.tick_wall.append([int(t), round(float(d_sim), 4), round(float(d_pre + d_post), 4), len(alive), int(fer)])
        steps = self.header.get("steps")
        last = isinstance(steps, int) and t >= steps - 1
        if t < 5 or t % 10 == 0 or last:
            recent = self.tick_wall[-10:]
            avg = sum(r[1] + r[2] for r in recent) / len(recent)
            eta = ""
            if isinstance(steps, int) and not last:
                eta = f" | осталось ≥ {self._hms(avg * (steps - t - 1))} (популяция растёт, будет дольше)"
            print(f"[ПРОГРЕСС] t={t}" + (f"/{steps}" if isinstance(steps, int) else "")
                  + f" | pop={len(alive)} (ферол {fer}) | тик: sim={d_sim:.2f}с лог={d_pre + d_post:.2f}с"
                  + f" | среднее за {len(recent)}: {avg:.2f}с | прошло {self._hms(now - self._wall_t0)}{eta}", flush=True)

    def save(self, path):
        tmp = path + ".tmp"
        with open(
            tmp,
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                {
                    "header": self.header,
                    "tick_rows": self.tick_rows,
                    "agent_snaps": self.agent_snaps,
                    "events": self.events,
                    "dialogues": self.dialogues,
                    "speaks": self.speaks,
                    "cluster_births": self.cluster_births,
                    "errors": dict(self.errors),
                    "tick_wall": self.tick_wall,
                },
                f,
                ensure_ascii=False
            )
        os.replace(tmp, path)

    @classmethod
    def load(cls, path):
        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:
            d = json.load(f)
        o = cls()
        for k in (
            "header",
            "tick_rows",
            "agent_snaps",
            "events",
            "dialogues",
            "speaks",
            "cluster_births",
        ):
            setattr(o, k, d[k])
        o.errors = Counter(
            d.get("errors", {})
        )
        o.tick_wall = d.get("tick_wall", [])
        return o


class RichEngine(LedgerEngine):
    rich = None

    def step(self, t):
        log = self.rich
        _tw0 = time.perf_counter()
        if log is not None:
            try:
                log.before_step(self)
            except Exception as e:
                log.errors[
                    f"before:{type(e).__name__}: {e}"[:160]
                ] += 1
        _tw1 = time.perf_counter()
        result = super().step(t)
        _tw2 = time.perf_counter()
        if log is not None:
            try:
                log.after_step(self, t)
            except Exception as e:
                log.errors[
                    f"after:{type(e).__name__}: {e}"[:160]
                ] += 1
                if t == 30:
                    raise RuntimeError(
                        "Rich log упал на t=30: "
                        + str(dict(log.errors))
                    )
            if getattr(Config, "POP_SOURCE_STRICT", False) and log.header.get("pop_source_violations_n", 0):
                raise AssertionError(f"POP_SOURCE_STRICT: log.n != ledger.n: {log.header.get('pop_source_violations', [])[:3]}")
            try:
                log.note_tick_wall(self, t, _tw2 - _tw1, _tw1 - _tw0, time.perf_counter() - _tw2)
            except Exception as e:
                log.errors[f"tickwall:{type(e).__name__}: {e}"[:160]] += 1
            if (
                log.autosave_path
                and (t + 1) % SAVE_EVERY == 0
            ):
                log.header["complete"] = False
                log.save(log.autosave_path)
        return result


def _run_rich(
    seed,
    steps,
    mult,
    ledger_path,
    rich_path,
    extra_overrides=None,
    verbose=False
):
    ledger = ExperimentLedger(window=100)
    ledger.autosave_path = ledger_path
    ledger.autosave_every = 100

    log = RichLog()
    log.autosave_path = rich_path

    log.header = {
        "seed": int(seed),
        "steps": int(steps),
        "mult": float(mult),
        "empathy_pull_override": EMPATHY_PULL_OVERRIDE,
        "neighbor_sync_override": NEIGHBOR_SYNC_OVERRIDE,
        # Эффективное значение на время прогона (override важнее текущего Config), чтобы ON/OFF
        # определялся по самому логу, а не по тому, какой Config стоит в момент последующей проверки.
        "enable_morse_communication": bool(
            (extra_overrides or {}).get(
                "ENABLE_MORSE_COMMUNICATION",
                Config.ENABLE_MORSE_COMMUNICATION
            )
        ),
    }

    try:
        log.header.update(_obs_run_header())    # observer-v2: схема, code/config hash, selftests, версии
    except Exception as _oe:
        _obs_fail('header', _oe)

    RichEngine.rich = log

    t0 = time.time()

    try:
        _ov = {
            "PROTECTED_DEATH_PROB_MULT": mult
        }
        if EMPATHY_PULL_OVERRIDE is not None:
            _ov["LEXICON_EMPATHY_PULL"] = (
                EMPATHY_PULL_OVERRIDE
            )
        if NEIGHBOR_SYNC_OVERRIDE is not None:
            _ov["NEIGHBOR_SYNC_NORMALIZED"] = (
                NEIGHBOR_SYNC_OVERRIDE
            )
        # --- [fix 29.10] явные "неубиваемые" оверрайды сессии: чтобы нельзя было
        # забыть поменять Config перед прогоном (как уже случилось с NAV_RADIUS/MIN_INTENT_NORM —
        # в прошлом прогоне в [PATCH-STACK] они остались 5/0.3 вместо задуманных 3/0.15). ---
        _ov["POP_FILL_SOFT"] = float(globals().get("RUN_POP_FILL_SOFT", 0.05))    # было 0.10, ещё вдвое ниже
        _ov["POP_FILL_HARD"] = float(globals().get("RUN_POP_FILL_HARD", 0.125))   # было 0.25, ещё вдвое ниже
        _ov["MORSE_NAV_RADIUS"] = 3
        _ov["LANG_MIN_INTENT_NORM"] = 0.15
        if extra_overrides:
            _ov.update(extra_overrides)
        log.header["pop_fill_soft"] = _ov.get("POP_FILL_SOFT")
        log.header["pop_fill_hard"] = _ov.get("POP_FILL_HARD")
        print(f"[ПОПУЛЯЦИЯ] POP_FILL_SOFT={_ov.get('POP_FILL_SOFT')} POP_FILL_HARD={_ov.get('POP_FILL_HARD')} "
              f"(доля занятых клеток мира; выше HARD новые обычные агенты не рождаются)", flush=True)
        _execute(
            seed,
            steps,
            _ov,
            RichEngine,
            ledger,
            bool(verbose)
        )
        ledger.header["complete"] = True
        log.header["complete"] = True
    finally:
        RichEngine.rich = None
        log.header["obs_fails"] = dict(_OBS_FAILS)
        log.header["wall_seconds"] = round(
            time.time() - t0,
            1
        )
        try:   # итог по единому источнику: снимки тиков кончаются на t=140, а прогон — на t=149
            if ledger.rows:
                log.header["pop_ledger_final"] = ledger.totals()
        except Exception as _pf:
            log.header["pop_ledger_final_error"] = repr(_pf)[:160]
        if ledger.rows:
            ledger.save(ledger_path)
        if log.tick_rows:
            log.save(rich_path)

    return ledger, log


print("✅ Cell 6b загружена: RichLog/RichEngine/_run_rich + X2-AUDIT D (E_* в tick_rows, header.energy_snapshot).")


In [ ]:
# Ячейка 28
import os, time, json, zlib
import numpy as np
from collections import Counter

for _need in ("_run_rich", "DIR", "Pattern", "EvolutionEngine", "find_largest_prey_in_radius",
              "collect_metrics", "CH", "phi_hash", "extend_class", "FloatField", "LedgerEngine", "RichEngine"):
    if _need not in globals():
        raise RuntimeError(f"{_need} не найден. Порядок: 0.1 → … → 6b → эта ячейка.")
if not hasattr(Pattern, "_x2_change"):
    raise RuntimeError("Pattern._x2_change не найден.")
if not hasattr(Pattern, "_sig_cry"):
    raise RuntimeError("Pattern._sig_cry не найден: нужна ячейка SIGNAL v3.0 с единым криком (запусти её перед p832).")
for _need in ("_find_prey",):
    if not hasattr(Pattern, _need):
        raise RuntimeError(f"Pattern.{_need} не найден: обнови ячейку с PatternPhysiology (нужна точка расширения).")
if not hasattr(EvolutionEngine, "_collect_metrics"):
    raise RuntimeError("EvolutionEngine._collect_metrics не найден: обнови ячейку EvolutionEngine (4-1).")

os.makedirs(DIR, exist_ok=True)
print(f"📁 DIR = {DIR}")
if not hasattr(Config, "PATCH_SET"):
    Config.PATCH_SET = ""

_COOLDOWN_KEYS = ("MORSE_SEND_COOLDOWN", "MORSE_RECV_COOLDOWN", "MORSE_DIALOGUE_COOLDOWN",
                  "DIVIDE_COOLDOWN", "GIVEN_COOLDOWN",
                  "SPECIATION_COOLDOWN", "SR_MEDITATION_COOLDOWN", "SR_TRADE_COOLDOWN",
                  "SR_HEAL_COOLDOWN", "SR_GENE_MUT_COOLDOWN", "FOLD_COOLDOWN_DURATION",
                  "ECHO_INJECTION_COOLDOWN")
_COOLDOWNS_WAS = {k: getattr(Config, k) for k in _COOLDOWN_KEYS if hasattr(Config, k)}
_CLAMP_WAS = (Pattern.energy.clamp_min, Pattern.energy.clamp_max)


FATIGUE_MULT, FATIGUE_DECAY, FATIGUE_DECAY_EVERY, FATIGUE_MIN = 0.5, 0.85, 5, 0.01  # v38: faster recovery (was 0.93 -> ~48-tick half-life; 0.85 -> ~21-tick half-life, every 5 ticks, same as before)
FATIGUE_CAP = 10.0
ACTION_COSTS = {
    'scream': 0.008, 'morse_send': 0.02, 'intent_switch': 0.005, 'divide': 0.5,
    'introspect': 0.05, 'meditation': 0.1, 'trade': 0.05, 'heal_field': 0.05,
    'gene_mut': 0.3, 'given': 0.2, 'goal_hold': 0.0005, 'scream_response': 0.001,
}
SCREAM_RADIUS, SCREAM_ALARM_STRENGTH = 6, 0.8

_P832_CAUSAL_ABLATIONS = {
    "repel_no_alarm_read": False,
    "kill_gate_bypass_alarm_threshold": False,
    "move_ignore_scream_target": False,
    "move_ignore_scream_response_goal": False,
    "move_ignore_scream_mobility_boost": False,
}
SCREAM_MOBILITY_BOOST, SCREAM_BOOST_DURATION = 1.8, 8
FERAL_NEAR_THRESHOLD, FERAL_ALARM_REPEL = 0.15, 3.0
SCREAM_ALARM_DECAY_EVERY, SCREAM_ALARM_DECAY = 5, 0.85
SCREAM_SOURCE_TTL, SCREAM_EVENTS_MAX = 15, 2000
KILL_RADIUS, KILL_SCREAM_FRESH, KILL_MIN_SCREAMERS, KILL_MIN_SIGNAL_AT_CENTER = 19, 7, 2, 0.1
KILL_CORE_RADIUS = 1
KILL_SIGNAL_RADIUS = KILL_CORE_RADIUS
SENS_RECOVERY_PER_TICK, SENS_DROP_ON_SCREAM, SENS_MIN = 0.02, 0.55, 0.2
RISE_TRIGGER_BASE, FERAL_TRIGGER_BASE = 0.15, 0.15
BG_FLOOR_MULT, BG_FLOOR_MIN = 2.0, 0.1
ACTION_EVENTS_CAP = 2000
SNAPSHOT_EVERY = 10

_MANIFEST = {
    "name": "noclip+fatigue+scream", "version": "2.6.0",
    "constants": dict(
        FATIGUE_MULT=FATIGUE_MULT, FATIGUE_DECAY=FATIGUE_DECAY,
        FATIGUE_DECAY_EVERY=FATIGUE_DECAY_EVERY,
        FATIGUE_MIN=FATIGUE_MIN, FATIGUE_CAP=FATIGUE_CAP,
        SCREAM_RADIUS=SCREAM_RADIUS,
        SCREAM_ALARM_STRENGTH=SCREAM_ALARM_STRENGTH,
        SCREAM_MOBILITY_BOOST=SCREAM_MOBILITY_BOOST,
        SCREAM_BOOST_DURATION=SCREAM_BOOST_DURATION,
        FERAL_NEAR_THRESHOLD=FERAL_NEAR_THRESHOLD,
        FERAL_ALARM_REPEL=FERAL_ALARM_REPEL,
        SCREAM_ALARM_DECAY_EVERY=SCREAM_ALARM_DECAY_EVERY,
        SCREAM_ALARM_DECAY=SCREAM_ALARM_DECAY,
        SCREAM_SOURCE_TTL=SCREAM_SOURCE_TTL,
        KILL_RADIUS=KILL_RADIUS, KILL_SCREAM_FRESH=KILL_SCREAM_FRESH,
        KILL_MIN_SCREAMERS=KILL_MIN_SCREAMERS,
        KILL_MIN_SIGNAL_AT_CENTER=KILL_MIN_SIGNAL_AT_CENTER,
        KILL_SIGNAL_RADIUS=KILL_SIGNAL_RADIUS,
        SENS_RECOVERY_PER_TICK=SENS_RECOVERY_PER_TICK,
        SENS_DROP_ON_SCREAM=SENS_DROP_ON_SCREAM,
        SENS_MIN=SENS_MIN, RISE_TRIGGER_BASE=RISE_TRIGGER_BASE,
        FERAL_TRIGGER_BASE=FERAL_TRIGGER_BASE,
        BG_FLOOR_MULT=BG_FLOOR_MULT, BG_FLOOR_MIN=BG_FLOOR_MIN,
        ACTION_EVENTS_CAP=ACTION_EVENTS_CAP, SCREAM_EVENTS_MAX=SCREAM_EVENTS_MAX,
    ),
    "action_costs": dict(ACTION_COSTS),
    "noclip": {
        "energy_clamp_was": list(_CLAMP_WAS),
        "energy_clamp_now": [Config.AGENT_ENERGY_MIN, Config.AGENT_ENERGY_MAX],
        "energy_contract": "agent.energy in [-1, 2]; field.energy is a separate channel",
        "conservation_check": "no homeostatic field correction; independent audit via _audit_close_tick / res1,res2",
    },
    "cooldowns_zeroed": dict(_COOLDOWNS_WAS),
    "layers": ["P832PatternMixin", "P832EngineMixin"],
    "patch_notes": "v2.6.0: кулдаун intent_switch не обнуляется, intent_switch=0.005, «мясо» ферала (X1→X2 крикунам, парная запись); v2.5.0: слои-подклассы вместо монки-патча (extend_class + super); v2.4.4: строгая оплата (given, scream); атомарная цена goal_hold; "
                   "update_intent после проверки оплаты; энергобаланс feral_execute под контролем; "
                   "раздельные cap_action/cap_scream.",
}
_MANIFEST["id"] = "p832-" + format(zlib.crc32(json.dumps(
    [_MANIFEST["constants"], _MANIFEST["action_costs"], _MANIFEST["cooldowns_zeroed"]],
    sort_keys=True, default=str).encode()) & 0xffffffff, "08x")


class _P832State:

    def __init__(self):
        self.scream, self.actions, self.spent = Counter(), Counter(), Counter()
        self.events, self.recent, self.bg_alarm = [], {}, 0.0
        self.kill_gate, self.alarm_hist, self.screamers_hist = Counter(), Counter(), Counter()
        self.alarm_max = self.core_max = 0.0
        self.screamers_max = 0
        self.x2_min, self.x2_max = float("inf"), float("-inf")
        self.x2_neg_ticks = self.x2_nonfinite_ticks = 0
        self.fatigue_peak = 0.0
        self.emitted, self.dropped, self.prev = Counter(), Counter(), {}
        self.kill_energy_balance_warnings = 0
        self.kill_negative_feral_energy = 0
        self.intent_switch_blocked_pre = 0

_P832_LAST = {"st": None, "world": None}
_ORPHAN = _P832State()

def _st(world):
    if world is None:
        return _ORPHAN
    st = world.__dict__.get("_p832")
    if st is None:
        st = _P832State()
        world._p832 = st
        _P832_LAST["st"] = st
        _P832_LAST["world"] = world
        g = globals()
        g["SCREAM_STATS"], g["ACTION_STATS"], g["SCREAM_EVENTS"] = st.scream, st.actions, st.events
    return st

def _emit(world, st, evt, kind, cap):
    st.emitted[kind] += 1
    if st.emitted[kind] > cap:
        st.dropped[kind] += 1
        return
    rich = getattr(world, "rich", None) if world is not None else None
    if rich is not None and hasattr(rich, "events"):
        try:
            rich.events.append(evt)
        except Exception:
            st.dropped["rich_error"] += 1


def _morse_gates_line(world, now):
    """Диагностика Морзе для периодического лога. Счётчики накопительные, поэтому печатаем
    и накопленное, и дельту за окно с прошлой печати (доли в накопленном маскируют текущую картину).
    Показывает также занятость передатчиков и среднюю длину сообщения. Никогда не бросает исключений."""
    try:
        lex = getattr(world, "EMERGENT_LEXICON", None)
        if lex is None:
            return ""
        act = _st(world).actions
        sil = {k: int(v) for k, v in (getattr(lex, "send_silence", None) or {}).items() if v}
        chan = getattr(lex, "chan", None) or {}
        attempt = int(act.get("morse_send_attempt", 0))
        ok = int(act.get("morse_send_ok", 0))
        skipped = int(act.get("morse_send_skipped", 0))
        blocked = int(act.get("morse_send_blocked", 0))
        pre_tx = int(act.get("morse_send_pre_tx_active", 0))
        pre_bo = int(act.get("morse_send_pre_backoff", 0))
        tot = sum(sil.values())
        parts = [f"{k}={v}({100.0 * v / tot:.0f}%)" for k, v in sorted(sil.items(), key=lambda kv: -kv[1])]
        pct = 100.0 * ok / attempt if attempt else 0.0
        rx = " ".join(f"{k}={int(chan.get(k, 0))}" for k in (
            "rx_conf_reliable", "rx_conf_uncertain", "rx_conf_corrupted", "rx_collision_flag",
            "rx_dropped_no_sender", "rx_dropped_mixed", "rx_to_lexicon", "tx_nav_deferred",
            "tx_nav_queued", "tx_queue_attempt", "tx_queue_sent", "tx_queue_expired", "tx_queue_dropped_hard",
            "tx_crisis_nav_override", "tx_send_crisis", "tx_send_normal"))   # MORSE-EVO

        # дельта за окно с прошлой печати
        prev = world.__dict__.get("_morse_gate_prev") or {"attempt": 0, "ok": 0, "sil": {}}
        d_att, d_ok = attempt - prev["attempt"], ok - prev["ok"]
        d_sil = {k: v - prev["sil"].get(k, 0) for k, v in sil.items() if v - prev["sil"].get(k, 0) > 0}
        d_tot = sum(d_sil.values())
        d_parts = [f"{k}={v}({100.0 * v / d_tot:.0f}%)" for k, v in sorted(d_sil.items(), key=lambda kv: -kv[1])]
        d_pct = 100.0 * d_ok / d_att if d_att else 0.0
        d_pre_tx, d_pre_bo = pre_tx - prev.get("pre_tx", 0), pre_bo - prev.get("pre_bo", 0)
        pre_txt = (f" | вне попыток (уже передаёт/backoff): {d_pre_tx}/{d_pre_bo}"
                   if (pre_tx or pre_bo) else "")
        world._morse_gate_prev = {"attempt": attempt, "ok": ok, "sil": dict(sil), "pre_tx": pre_tx, "pre_bo": pre_bo}

        # занятость эфира и длина сообщения
        reg = getattr(getattr(world, "morse_field", None), "tx_registry", None)
        n_tx = len(reg) if reg is not None else -1
        n_alive = sum(1 for p in getattr(world, "patterns", []) if getattr(p, "alive", False))
        occ = f"{n_tx}/{n_alive} ({100.0 * n_tx / n_alive:.0f}%)" if (n_tx >= 0 and n_alive) else "n/a"
        n_sent = sum(int(v) for v in getattr(lex, "send_k_counts", {}).values())
        mean_len = (chan.get("tx_ticks_sum", 0) / n_sent) if n_sent else 0.0
        k_dist = dict(sorted((int(k), int(v)) for k, v in getattr(lex, "send_k_counts", {}).items()))
        # геометрия: расстояния между центрами соседних тел (обычный hypot без обёртки тора — как в гейтах)
        try:
            alive = [p for p in getattr(world, "patterns", []) if getattr(p, "alive", False) and getattr(p, "cells", None)]
            if 2 <= len(alive) <= 20000:
                from scipy.spatial import cKDTree  # вместо матрицы n x n: те же числа, но без квадрата по памяти
                C = np.array([p.get_center() for p in alive], dtype=np.float64)
                _tree = cKDTree(C)
                nn = _tree.query(C, k=2)[0][:, 1]
                _cnt = lambda r: float((_tree.query_ball_point(C, r, return_length=True) - 1).mean())
                geo = (f"ближайший сосед (центры) p10/p50/p90={np.percentile(nn, 10):.1f}/{np.percentile(nn, 50):.1f}/"
                       f"{np.percentile(nn, 90):.1f} | соседей в r<=2/5/7: "
                       f"{_cnt(2):.2f}/{_cnt(5):.2f}/{_cnt(7):.2f}")
            else:
                geo = "n/a"
        except Exception as _ge:
            geo = f"n/a ({_ge!r})"
        return (f"\n[MORSE-GATES t={now}] attempt={attempt} ok={ok} ({pct:.1f}%) skipped={skipped} "
                f"blocked_cost={blocked} | накоплено: {' '.join(parts) or '—'} | неучтено={skipped - tot}\n"
                f"           Δокно: attempt={d_att} ok={d_ok} ({d_pct:.1f}%) | {' '.join(d_parts) or '—'}{pre_txt}\n"
                f"           TX: передают сейчас {occ} | ср. длина сообщения={mean_len:.1f} тиков | K={k_dist}\n"
                f"           GEO: {geo}\n"
                f"           RX {rx}")
    except Exception as exc:
        return f"\n[MORSE-GATES t={now}] ошибка диагностики: {exc!r}"


def _morse_final_report(world, st):
    """Итог по Морзе за весь прогон: гейты отправки + приём. Не бросает исключений."""
    try:
        lex = getattr(world, "EMERGENT_LEXICON", None)
        if lex is None:
            print("\n=== MORSE: лексикон недоступен ===")
            return
        print("\n=== MORSE SEND GATES (весь прогон, lex.send_silence) ===")
        sil = {k: int(v) for k, v in (getattr(lex, "send_silence", None) or {}).items() if v}
        tot = sum(sil.values())
        for k, v in sorted(sil.items(), key=lambda kv: -kv[1]):
            print(f"  {k}: {v} ({100.0 * v / tot:.1f}%)")
        skipped = int(st.actions.get("morse_send_skipped", 0))
        print(f"  Σ send_silence={tot} | morse_send_skipped={skipped} | неучтено={skipped - tot} "
              f"(ранние выходы: морзе выключен / агент мёртв / нет клеток)")
        _pre_tx = int(st.actions.get("morse_send_pre_tx_active", 0))
        _pre_bo = int(st.actions.get("morse_send_pre_backoff", 0))
        if _pre_tx or _pre_bo:
            print(f"  вне попыток (MORSE_PRECHECK_IDLE): уже передаёт={_pre_tx} backoff={_pre_bo} "
                  f"(в attempt/skipped/send_silence не входят)")
        print("\n=== MORSE CHANNEL (lex.chan — накопительно, включая умерших агентов) ===")
        for k, v in sorted(getattr(lex, "chan", {}).items()):
            if str(k).startswith(("rx_", "tx_")):
                print(f"  {k}: {v}")
        print(f"  lex.rx_total: {getattr(lex, 'rx_total', 0)}")
        _n_sent = sum(int(v) for v in getattr(lex, "send_k_counts", {}).values())
        if _n_sent:
            print(f"  отправлено сообщений: {_n_sent} | ср. длина: {lex.chan.get('tx_ticks_sum', 0) / _n_sent:.1f} тиков | K-распределение: {dict(sorted((int(k), int(v)) for k, v in lex.send_k_counts.items()))}")
        rx_tot = Counter()
        pats = getattr(world, "_diagnostic_all_patterns", None) or getattr(world, "patterns", [])
        for p in pats:
            rx = getattr(p, "_morse_rx", None)
            if rx is None:
                continue
            for k, v in rx.get_metrics().items():
                if isinstance(v, (int, float)) and k not in ("mean_confidence", "unique_senders"):
                    rx_tot[k] += v
        print("\n=== MORSE RX metrics (сумма по агентам; у умерших обнуляются в _morse_reset) ===")
        for k, v in sorted(rx_tot.items()):
            print(f"  rx.{k}: {round(v, 3) if isinstance(v, float) else v}")
    except Exception as exc:
        print(f"⚠️ MORSE final report упал: {exc!r}")


_DISC = {}

def _disc(R):
    d = _DISC.get(R)
    if d is None:
        r = np.arange(-R, R + 1)
        dx, dy = np.meshgrid(r, r, indexing="ij")
        d2 = dx * dx + dy * dy
        m = d2 <= R * R
        d = (dx[m].astype(np.int64), dy[m].astype(np.int64),
             np.sqrt(d2[m]).astype(np.float64))
        _DISC[R] = d
    return d

def _disc_mean(chan, cx, cy, R, W):
    dx, dy, _ = _disc(R)
    return float(chan[(cx + dx) % W, (cy + dy) % W].mean(dtype=np.float64))

for _cfg in _COOLDOWN_KEYS:
    if hasattr(Config, _cfg):
        setattr(Config, _cfg, 0)

print("=== ACTION-COSTS (строгая оплата) ===")
for _k, _v in ACTION_COSTS.items():
    print(f"  {_k}: {_v}")
print("====================================")

def _action_publish(world, action, pid, cost, fatigue, **extra):
    st = _st(world)
    st.actions[action] += 1
    evt = {"t": int(getattr(world, "age", 0)), "type": f"action_{action}", "pid": int(pid),
           "cost": round(float(cost), 6), "fatigue_after": round(float(fatigue), 3)}
    evt.update(extra)
    _emit(world, st, evt, f"action_{action}", ACTION_EVENTS_CAP)

def _scream_publish(world, kind, **data):
    st = _st(world)
    evt = {"t": int(data.get("t", 0)), "type": f"scream_{kind}"}
    evt.update({k: (v if not isinstance(v, np.generic) else v.item())
                for k, v in data.items() if k != "t"})
    if len(st.events) < SCREAM_EVENTS_MAX:
        st.events.append(evt)
    _emit(world, st, dict(evt), f"scream_{kind}", SCREAM_EVENTS_MAX)

def _scream_dist(a, b, W):
    dx = abs(int(a[0]) - int(b[0])); dx = min(dx, W - dx)
    dy = abs(int(a[1]) - int(b[1])); dy = min(dy, W - dy)
    return float(np.hypot(dx, dy))

def _scream_nearest_step(src, dst, W):
    dx = int(dst[0]) - int(src[0]); dy = int(dst[1]) - int(src[1])
    if dx > W / 2: dx -= W
    elif dx < -W / 2: dx += W
    if dy > W / 2: dy -= W
    elif dy < -W / 2: dy += W
    return (0 if dx == 0 else (1 if dx > 0 else -1)), (0 if dy == 0 else (1 if dy > 0 else -1))

_IDX4 = np.arange(-4, 5)

def _scream_try(self, field, now):
    if self.role_type == "feral" or not self.alive or not self.cells or field is None:
        return
    st = _st(self.world)
    sens = min(1.0, float(getattr(self, "_scream_sensitivity", 1.0)) + SENS_RECOVERY_PER_TICK)
    self._scream_sensitivity = sens
    W = Config.WORLD_SIZE
    cx, cy = self.get_center()
    cx, cy = int(cx) % W, int(cy) % W
    feral_near = float(field[:, :, CH["signal_feral"]][
        np.ix_((cx + _IDX4) % W, (cy + _IDX4) % W)].mean(dtype=np.float64))
    alarm = float(getattr(self, "alarm_level", 0.0))
    if feral_near > FERAL_NEAR_THRESHOLD:
        alarm = min(1.0, alarm + 0.4)
        self.alarm_level = alarm
    emotional = getattr(self, "emotional_memory", {}) or {}
    grief = float(emotional.get("grief", 0.0))
    init_done = getattr(self, "_scream_init_done", False)
    prev_alarm = float(getattr(self, "_scream_prev_alarm", alarm))
    prev_grief = float(getattr(self, "_scream_prev_grief", grief))
    self._scream_prev_alarm, self._scream_prev_grief = alarm, grief
    if not init_done:
        self._scream_init_done = True
        return
    d_state = max(0.0, alarm - prev_alarm) + max(0.0, grief - prev_grief)
    rise_trigger = d_state > RISE_TRIGGER_BASE / max(sens, SENS_MIN)
    feral_trigger = feral_near > FERAL_TRIGGER_BASE * (1.5 / max(sens, SENS_MIN))
    if not (rise_trigger or feral_trigger):
        if d_state > RISE_TRIGGER_BASE * 0.5 or feral_near > FERAL_TRIGGER_BASE:
            st.scream["habituation_blocked"] += 1
        return
    reason = "feral" if feral_trigger else "rise"
    paid = self._consume_action('scream')
    if paid is None:
        st.scream['cost_blocked'] += 1
        st.actions['scream_blocked'] += 1   # [AUDIT#8]
        return
    st.actions['scream'] += 1                # [AUDIT#8] крики теперь видны в [ACTIONS]
    self._scream_boost_until = now + SCREAM_BOOST_DURATION
    self._last_scream_t, self._last_scream_pos = now, (cx, cy)
    self._scream_sensitivity = max(SENS_MIN, sens * SENS_DROP_ON_SCREAM)
    st.recent[self.id] = {"t": now, "pos": (cx, cy)}
    st.scream["total"] += 1
    st.scream["feral" if reason == "feral" else "rise"] += 1
    dxs, dys, dist = _disc(SCREAM_RADIUS)
    chan = field[:, :, CH["signal_alarm"]]
    xi, yi = (cx + dxs) % W, (cy + dys) % W
    source_left = cx < Config.CENTER_WALL_X
    for x, y, d in zip(xi, yi, dist):
        if Config.ENABLE_CENTER_WALL and ((int(x) < Config.CENTER_WALL_X) != source_left):
            continue
        chan[x, y] = min(1.0, float(chan[x, y]) +
                         SCREAM_ALARM_STRENGTH * (1.0 - float(d) / SCREAM_RADIUS))
    if self._sig_cry(field, now):
        st.scream["cry_as_form"] += 1
    _scream_publish(self.world, "scream", t=now, id=self.id, reason=reason,
                    feral_near=round(feral_near, 4), alarm=round(alarm, 4),
                    grief=round(grief, 4), d_state=round(d_state, 4),
                    sens_before=round(sens, 3),
                    sens_after=round(self._scream_sensitivity, 3),
                    cost=round(float(paid), 5),
                    fatigue=round(self._action_fatigue.get('scream', 0.0), 2),
                    pos=(cx, cy))
    self._log_event("scream", reason=reason, feral_near=round(feral_near, 3))

def _scream_check_response_arrival(world, now):
    W = Config.WORLD_SIZE
    st = _st(world)
    for p in world.patterns:
        if not p.alive or not p.cells:
            continue
        goals = getattr(p, "goals", None)
        if not goals:
            continue
        for goal in list(goals):
            if goal.get("_source") != "scream_response":
                continue
            target = goal.get("target")
            if target is None:
                continue
            cx, cy = p.get_center()
            here = (int(cx) % W, int(cy) % W)
            if _scream_dist(here, target, W) <= 2.0:
                st.scream["response_reached"] += 1
                _scream_publish(world, "response_reached", t=now, id=p.id,
                                source_id=goal.get("_source_id"),
                                target=tuple(target),
                                time_to_reach=int(now) - int(goal.get("_created_t", now)),
                                pos=here)
                goals.remove(goal)
                p._scream_target = None

def _fatigue_stats(world):
    alive = [p for p in world.patterns if p.alive]
    if not alive:
        return 0.0, 0.0
    fs = [p._fatigue_state() for p in alive]
    return float(sum(fs) / len(fs)), float(max(fs))

def _flat(world):
    st = _st(world)
    s, kg = st.scream, st.kill_gate
    fa, fm = _fatigue_stats(world)
    result = {
        "scr_cry": s["total"], "scr_cry_feral": s["feral"], "scr_cry_rise": s["rise"],
        "scr_hab_ticks": s["habituation_blocked"], "scr_cost_blocked": s["cost_blocked"],
        "scr_repel": s["repel"], "scr_resp_goal": s["response_goal"],
        "scr_resp_reached": s["response_reached"],
        "scr_kill": s["kill"], "scr_kill_energy": float(s["kill_energy_moved"]),
        "scr_meat": s["meat_kills"], "scr_meat_energy": float(s["meat_energy"]),
        "scr_kill_ebw": st.kill_energy_balance_warnings,
        "scr_kill_neg_e": st.kill_negative_feral_energy,
        "intent_sw_blocked_pre": st.intent_switch_blocked_pre,
        "kg_attempts": kg["attempts"], "kg_signal_low": kg["signal_low"],
        "kg_screamers_low": kg["screamers_low"],
        "kg_passed": kg["passed"],
        "kg_signal_threshold_bypassed": kg["signal_threshold_bypassed"],
        "kg_alarm_max": st.alarm_max,
        "kg_core_max": st.core_max, "kg_screamers_max": st.screamers_max,
        "act_spent": float(sum(st.spent.values())),
        "act_blocked": int(sum(v for k, v in st.actions.items()
                              if "blocked" in k or "unpaid" in k)),
        "act_fatigue_avg": fa, "act_fatigue_max": fm,
        "act_fatigue_peak": st.fatigue_peak,
        "x2g_min": (st.x2_min if st.x2_min != float("inf") else 0.0),
        "x2g_max": (st.x2_max if st.x2_max != float("-inf") else 0.0),
        "x2g_neg_ticks": st.x2_neg_ticks,
        "x2g_nonfinite_ticks": st.x2_nonfinite_ticks,
        }
    # Optional extension hook keeps the base metrics function stable while
    # allowing later mixins to contribute metrics without replacing _flat.
    extra = getattr(world, '_flat_extra_metrics', None)
    if callable(extra):
        try:
            result.update(extra())
        except Exception:
            pass
    return result

def _feral_meat(world, feral, screamers, field, now):
    """Вариант 2: мясо — из X2 ферала-убийцы, не из клеток поля.
    Раздаём крикунам только то, что они реально могут принять (headroom до X2_MAX).
    Остаток у ферала обнуляет вызывающий код (X2.scream_kill_feral_rest_dissipated)."""
    if not Config.FERAL_MEAT_ENABLE or not screamers or not feral.cells:
        return 0.0
    st = _st(world)
    energy_cap = float(Config.AGENT_ENERGY_MAX)

    # сколько ферал готов отдать как "мясо" (кап по массе тела и общему лимиту)
    cells_n = len(feral.cells)
    want_total = float(Config.FERAL_MEAT_PER_CELL) * cells_n
    cap = float(Config.FERAL_MEAT_MAX_TOTAL)
    if cap > 0.0:
        want_total = min(want_total, cap)
    # и сколько у него есть в X2 прямо сейчас
    feral_avail = max(0.0, float(feral.energy))
    want_total = min(want_total, feral_avail)
    if want_total <= 1e-12:
        st.scream["meat_empty"] += 1
        return 0.0

    # сколько крикуны могут принять без overflow
    headroom = np.asarray([
        max(0.0, energy_cap - float(s.energy)) if np.isfinite(float(s.energy)) else 0.0
        for s in screamers
    ], dtype=np.float64)
    total_room = float(headroom.sum(dtype=np.float64))
    if total_room <= 1e-12:
        st.scream["meat_all_full"] = st.scream.get("meat_all_full", 0) + 1
        return 0.0

    distributable = min(want_total, total_room)
    alloc = np.zeros(len(screamers), dtype=np.float64)
    remaining = distributable
    active = np.flatnonzero(headroom > 1e-12)
    while active.size and remaining > 1e-12:
        share = remaining / float(active.size)
        grant = np.minimum(headroom[active] - alloc[active], share)
        moved = float(grant.sum(dtype=np.float64))
        if moved <= 1e-12:
            break
        alloc[active] += grant
        remaining = max(0.0, remaining - moved)
        active = active[(headroom[active] - alloc[active]) > 1e-12]

    total_move = float(alloc.sum(dtype=np.float64))
    if total_move <= 0.0:
        return 0.0

    # списываем с X2 ферала
    old_feral = float(feral.energy)
    feral._x2_change("X2.feral_meat_pays", old_feral - total_move, offered=-total_move)
    actually_paid = old_feral - float(feral.energy)
    if actually_paid <= 0.0:
        return 0.0
    if actually_paid < total_move:
        alloc *= actually_paid / total_move

    got_total = 0.0
    for s, a in zip(screamers, alloc):
        if a <= 1e-12:
            continue
        got_total += float(s._x2_change("X2.feral_meat_reward",
                                        float(s.energy) + float(a), offered=float(a)))

    st.scream["meat_kills"] += 1
    st.scream["meat_energy"] += got_total
    _scream_publish(world, "meat_from_feral", t=now, feral=feral.id,
                    amount=round(actually_paid, 6),
                    received=round(got_total, 6),
                    source="feral_x2",
                    screamers=[s.id for s in screamers], cells=cells_n)
    return actually_paid


class P832PatternMixin:

    energy = FloatField('energy', default=Pattern.energy.default,
                        clamp_min=Config.AGENT_ENERGY_MIN, clamp_max=Config.AGENT_ENERGY_MAX)


    def _fatigue_init(self):
        if "_action_fatigue" not in self.__dict__:
            self._action_fatigue = {}

    def _fatigue_state(self):
        self._fatigue_init()
        return sum(self._action_fatigue.values()) if self._action_fatigue else 0.0

    def _cap_total_fatigue(self):
        """Keep aggregate fatigue within FATIGUE_TOTAL_CAP, preserving action proportions."""
        self._fatigue_init()
        cap = max(0.0, float(getattr(Config, 'FATIGUE_TOTAL_CAP', 15.0)))
        total = sum(max(0.0, float(v)) for v in self._action_fatigue.values())
        if total > cap and total > 0.0:
            scale = cap / total
            self._action_fatigue = {
                k: max(0.0, float(v)) * scale
                for k, v in self._action_fatigue.items()
            }

    def _action_cost(self, action, base_cost=None):
        self._fatigue_init()
        if base_cost is None:
            base_cost = ACTION_COSTS.get(action, 0.01)
        if action == 'morse_send':
            return float(base_cost)
        return base_cost * (1.0 + self._action_fatigue.get(action, 0.0) * FATIGUE_MULT)

    def _can_afford(self, action, base_cost=None):
        return float(self.energy) >= self._action_cost(action, base_cost)

    def _consume_action(self, action, base_cost=None, allow_partial=False):
        self._fatigue_init()
        cost = self._action_cost(action, base_cost)
        cur = float(self.energy)
        if not np.isfinite(cur):
            return None
        if cur < cost:
            if allow_partial and cur > 0:
                cost = cur
            else:
                return None
        self._x2_change(f'X2.action_{action}', cur - cost)
        if action not in ('meditation', 'rest', 'introspect'):
            f = self._action_fatigue.get(action, 0.0) + 1.0
            if FATIGUE_CAP is not None:
                f = min(f, FATIGUE_CAP)
            self._action_fatigue[action] = f
        self._cap_total_fatigue()
        st = _st(self.world)
        st.spent[action] += cost
        # Track aggregate fatigue after applying per-action caps.
        f = self._fatigue_state()
        st.fatigue_peak = max(st.fatigue_peak, f)
        return cost

    def _recover_fatigue(self):
        self._fatigue_init()
        for k in list(self._action_fatigue.keys()):
            self._action_fatigue[k] *= FATIGUE_DECAY
            if self._action_fatigue[k] < FATIGUE_MIN:
                del self._action_fatigue[k]
        self._cap_total_fatigue()


    def _tick_goal_cost(self):
        goals = getattr(self, 'goals', None)
        if not goals:
            return
        per_cost = ACTION_COSTS['goal_hold']
        n_goals = len(goals)
        total_cost = per_cost * n_goals
        cur = float(self.energy)
        st = _st(self.world)
        if cur >= total_cost:
            self._x2_change('X2.action_goal_hold', cur - total_cost)
            st.spent['goal_hold'] += total_cost
            return
        can_pay = max(0, int(cur / per_cost))
        if can_pay > 0:
            paid = can_pay * per_cost
            self._x2_change('X2.action_goal_hold', cur - paid)
            st.spent['goal_hold'] += paid
        for g in goals[can_pay:]:
            g['age'] = g.get('persistence', 9999) + 1


    def _send_morse(self, text, field, recipient_id=None):
        act = _st(self.world).actions
        if not getattr(Config, 'ENABLE_MORSE_COMMUNICATION', True):
            # Канал выключен: это не «попытка отправки». Считаем отдельно, чтобы в OFF
            # morse_send_attempt/skipped/blocked оставались 0 (раньше attempt=45 при ok=0).
            act['morse_send_off_gate'] += 1
            return False
        if getattr(Config, 'MORSE_PRECHECK_IDLE', True):
            # Агент, который уже передаёт или ждёт backoff, получил бы отказ в MorseMixin._send_morse ещё до любых
            # изменений состояния и списаний (оба гейта стоят там первыми). Такой вызов не «попытка»: иначе
            # tx_active+backoff раздувают attempt втрое и прячут реальные гейты. Считаем отдельно (pre_*).
            _tx = getattr(self, '_morse_tx', None)
            if _tx is not None and _tx.active:
                act['morse_send_pre_tx_active'] += 1
                return False
            if int(getattr(self, '_morse_backoff', 0)) > 0:
                act['morse_send_pre_backoff'] += 1
                return False
        act['morse_send_attempt'] += 1
        if not self._can_afford('morse_send'):
            act['morse_send_blocked'] += 1
            return False
        sent = super()._send_morse(text, field, recipient_id)
        if not sent:
            act['morse_send_skipped'] += 1
            return False
        paid = self._consume_action('morse_send')
        if paid is None:
            act['morse_send_unpaid'] += 1
            self._log_event('morse_send_unpaid')
            return True
        _action_publish(self.world, 'morse_send', self.id, paid,
                        self._action_fatigue.get('morse_send', 0))
        act['morse_send_ok'] += 1
        return True

    def update_intent(self):
        if not self._can_afford('intent_switch'):
            _st(self.world).actions['intent_switch_blocked_pre'] += 1
            _st(self.world).intent_switch_blocked_pre += 1
            self.cognitive_tension = min(Config.MAX_METRIC,
                                         float(self.cognitive_tension) + 0.02)
            return
        prev_age = self.last_intent_switch_age
        super().update_intent()
        switched = (self.last_intent_switch_age != prev_age
                    and self.last_intent_switch_age == self.age)
        if switched:
            paid = self._consume_action('intent_switch')
            if paid is None:
                _st(self.world).actions['intent_switch_unpaid'] += 1
                self._log_event('intent_switch_unpaid')
            else:
                _action_publish(self.world, 'intent_switch', self.id, paid,
                                self._action_fatigue.get('intent_switch', 0))

    def _divide_geometry_ok(self):
        cells_list = sorted(self.cells)
        min_cells = max(3, Config.DIVIDE_MIN_SIZE)
        child1_cells, child2_cells = PatternPhysiology._topological_halves(
            cells_list, Config.WORLD_SIZE
        )
        return len(child1_cells) >= min_cells and len(child2_cells) >= min_cells

    def can_divide(self):
        if not super().can_divide():
            return False
        if not self._divide_geometry_ok():
            if self.world is not None:
                _st(self.world).actions['divide_blocked_geometry'] += 1
            return False
        if not self._can_afford('divide'):
            if self.world is not None:
                _st(self.world).actions['divide_blocked_cost'] += 1
            return False
        return True

    def divide(self, field, next_id):
        # Повторная геометрическая проверка защищает прямые вызовы, минуя can_divide().
        if not self._divide_geometry_ok():
            if self.world is not None:
                _st(self.world).actions['divide_blocked_geometry'] += 1
            return None, next_id
        paid = self._consume_action('divide')
        if paid is None:
            if self.world is not None:
                _st(self.world).actions['divide_blocked_cost'] += 1
            return None, next_id
        child, next_id = super().divide(field, next_id)
        if child is None:
            if self.world is not None:
                _st(self.world).actions['divide_unexpected_postpay_failure'] += 1
            return None, next_id
        _action_publish(self.world, 'divide', self.id, paid,
                        self._action_fatigue.get('divide', 0))
        return child, next_id

    def introspect(self):
        if not self._can_afford('introspect'):
            _st(self.world).actions['introspect_blocked'] += 1
            self._last_introspect_step = self.age
            return None
        paid = self._consume_action('introspect')
        if paid is None:
            self._last_introspect_step = self.age
            return None
        _action_publish(self.world, 'introspect', self.id, paid,
                        self._action_fatigue.get('introspect', 0))
        return super().introspect()

    def _sr_meditate(self, field, t):
        paid = self._consume_action('meditation')
        if paid is None:
            _st(self.world).actions['meditation_blocked'] += 1
            self._sr_last_meditation = t
            return
        # v43-fix3b: diagnostic only, no behaviour change. If this ever goes above 0
        # in [ACTIONS], the SR_MEDITATION_COOLDOWN gate is being bypassed somewhere;
        # if it stays at 0, the high meditation: count in the log is real (population
        # churn / persistently high grief), not a cooldown bug.
        _prior_t = getattr(self, '_sr_last_meditation', None)
        if _prior_t is not None and (t - _prior_t) < Config.SR_MEDITATION_COOLDOWN:
            _st(self.world).actions['meditation_cooldown_violation'] += 1
        _action_publish(self.world, 'meditation', self.id, paid,
                        self._action_fatigue.get('meditation', 0))
        return super()._sr_meditate(field, t)

    def _sr_try_trade(self, field, t):
        # v35: the base method has its own cooldown / partner / need gates. The old wrapper paid
        # the action cost BEFORE them, so every idle tick drained energy (and piled up fatigue).
        # Now: cheap side-effect-free gates first, pay only after a trade actually happened.
        if not Config.ENABLE_SELF_REGULATION:
            return None
        if t - getattr(self, '_sr_last_trade_t', -Config.SR_TRADE_COOLDOWN) < Config.SR_TRADE_COOLDOWN:
            return None
        if not self._can_afford('trade'):
            return None
        _done0 = int(getattr(self, '_sr_trades_done', 0))
        res = super()._sr_try_trade(field, t)
        if int(getattr(self, '_sr_trades_done', 0)) > _done0:
            paid = self._consume_action('trade')
            if paid is not None:
                _action_publish(self.world, 'trade', self.id, paid,
                                self._action_fatigue.get('trade', 0))
        return res

    def _sr_heal_field(self, field, t):
        if not self._can_afford('heal_field'):
            self._sr_last_heal_t = t
            return
        paid = self._consume_action('heal_field')
        if paid is None:
            self._sr_last_heal_t = t
            return
        _action_publish(self.world, 'heal_field', self.id, paid,
                        self._action_fatigue.get('heal_field', 0))
        return super()._sr_heal_field(field, t)

    def apply_given_operator(self, field, forced=False, witness=None):
        paid = self._consume_action('given')
        if paid is None:
            _st(self.world).actions['given_blocked'] += 1
            return
        self.given_cooldown = 0
        _action_publish(self.world, 'given', self.id, paid,
                        self._action_fatigue.get('given', 0))
        return super().apply_given_operator(field, forced=forced, witness=witness)


    def _enforce_cell_cap(self, field):
        """Enforce body-size limits without leaving stale ownership in the field."""
        if not getattr(self, 'cells', None):
            return
        role = getattr(self, 'role_type', 'normal')
        cap = int(getattr(Config, 'FERAL_MAX_CELLS', 30) if role == 'feral'
                  else getattr(Config, 'BODY_CELL_CAP', 30))
        cap = max(1, cap)
        if len(self.cells) <= cap:
            return
        cells = set(self.cells)
        W = int(Config.WORLD_SIZE)
        before_n = len(cells)
        components = self._components(cells, W)
        frag_before = len(components)
        removed = set()

        # Trim smaller disconnected components first, then the largest component.
        # A cell is removable only when it is peripheral and its removal preserves
        # 4-neighbour connectivity of that component on the torus.
        for comp in reversed(components):
            while len(cells) - len(removed) > cap and comp:
                removable = [c for c in sorted(comp)
                             if self._can_take_cell(c, comp, W)]
                if not removable:
                    break
                def _trim_key(c):
                    try:
                        e = float(field[c[0], c[1], CH['energy']])
                        if not np.isfinite(e):
                            e = float('inf')
                    except Exception:
                        e = float('inf')
                    return (e, c)
                cell = min(removable, key=_trim_key)
                comp.remove(cell)
                removed.add(cell)

        keep = cells - removed
        # If the topology has no safe peripheral deletion, preserve the body rather
        # than silently severing an articulation cell; the cap may remain exceeded.
        for x, y in removed:
            if int(field[x, y, CH['owner']]) == int(self.id):
                field[x, y, CH['owner']] = 0
        self.cells = keep

        if before_n > cap:
            frag_after = len(self._components(keep, W)) if keep else 0
            _entry = {
                't': int(getattr(getattr(self, 'world', None), 'age', getattr(self, 'age', 0))),
                'agent': int(getattr(self, 'id', -1)),
                'role': str(role),
                'cap': int(cap),
                'n_before': int(before_n),
                'n_after': int(len(keep)),
                'frag_before': int(frag_before),
                'frag_after': int(frag_after),
                'removed': [tuple(map(int, c)) for c in sorted(removed)],
                'unresolved_over_cap': bool(len(keep) > cap),
            }
            _w = getattr(self, 'world', None)
            if _w is not None:
                _log = getattr(_w, '_body_cap_trim_log', None)
                if _log is None:
                    _log = _w._body_cap_trim_log = []
                _log.append(_entry)
                try:
                    self.world.witness.record(self.id, 'body_cap_trim', **{
                        k: v for k, v in _entry.items() if k not in ('agent', 't')})
                except Exception:
                    pass
            try:
                self._log_event('body_cap_trim', n_before=before_n, n_after=len(keep),
                                frag_before=frag_before, frag_after=frag_after,
                                removed=len(removed), unresolved_over_cap=len(keep) > cap)
            except Exception:
                pass

    def grow(self, field):
        result = super().grow(field)
        self._enforce_cell_cap(field)
        return result

    def compete(self, other, field):
        result = super().compete(other, field)
        self._enforce_cell_cap(field)
        if other is not None and hasattr(other, '_enforce_cell_cap'):
            other._enforce_cell_cap(field)
        return result

    def move(self, field, t):
        now = int(getattr(self.world, "age", t))
        target = (None if _P832_CAUSAL_ABLATIONS.get("move_ignore_scream_target", False)
                  else getattr(self, "_scream_target", None))
        if target is not None and self.alive and self.cells:
            W = Config.WORLD_SIZE
            cx, cy = self.get_center()
            src = (int(cx) % W, int(cy) % W)
            if _scream_dist(src, target, W) > 2:
                sx, sy = _scream_nearest_step(src, target, W)
                if (sx, sy) != (0, 0):
                    self._dir_momentum = (float(sx), float(sy))
        boost_active = (not _P832_CAUSAL_ABLATIONS.get("move_ignore_scream_mobility_boost", False)
                        and getattr(self, "_scream_boost_until", -1) > now)
        if not boost_active:
            return super().move(field, t)
        old_mobility = self.genome.get("mobility", 0.1)
        self.genome["mobility"] = min(0.5, old_mobility * SCREAM_MOBILITY_BOOST)
        try:
            return super().move(field, t)
        finally:
            self.genome["mobility"] = old_mobility

    def _find_prey(self, field, world, min_cells=5, radius=20):
        prey, direction, dist = super()._find_prey(field, world, min_cells=min_cells, radius=radius)
        if direction is None or field is None:
            return prey, direction, dist
        W = Config.WORLD_SIZE
        cx, cy = self.get_center()
        cx, cy = int(cx) % W, int(cy) % W
        steps = max(1, min(int(dist) if dist else 10, 15))
        alarm_ahead = 0.0
        for r in range(1, steps + 1):
            alarm_ahead += float(field[
                int((cx + direction[0] * r) % W),
                int((cy + direction[1] * r) % W),
                CH["signal_alarm"]])
        alarm_ahead /= steps
        if _P832_CAUSAL_ABLATIONS.get("repel_no_alarm_read", False):
            alarm_ahead = 0.0
        if alarm_ahead > 0.4:
            now = int(world.age if world is not None else self.age)
            roll = phi_hash(self.id, now, 7777)
            if roll < min(0.95, alarm_ahead * FERAL_ALARM_REPEL):
                _st(world).scream["repel"] += 1
                _scream_publish(world, "repel", t=now, feral=self.id,
                                alarm_ahead=round(alarm_ahead, 4),
                                roll=round(roll, 4), pos=(cx, cy))
                self._log_event("feral_repelled_by_scream", alarm=round(alarm_ahead, 3))
                if prey is not None:
                    _bo = getattr(self, '_feral_backoff', None)
                    if _bo is None:
                        _bo = {}
                        self._feral_backoff = _bo
                    _bo[prey.id] = now + Config.FERAL_MISS_BACKOFF_TICKS
                    prey._feral_scar_count = getattr(prey, '_feral_scar_count', 0) + 1
                return None, None, None
        return prey, direction, dist

    def generate_goals(self, field):
        super().generate_goals(field)
        if _P832_CAUSAL_ABLATIONS.get("move_ignore_scream_response_goal", False):
            _removed_scream_goals = [g for g in self.goals if g.get("_source") == "scream_response"]
            if _removed_scream_goals:
                self.goals[:] = [g for g in self.goals if g.get("_source") != "scream_response"]
                if any(self.intent is g for g in _removed_scream_goals):
                    self.intent = None
                    self.intent_commitment = 0.0
                    self.last_intent_switch_age = self.age - Config.INTENT_SWITCH_COOLDOWN
        if field is None or not self.alive or not self.cells or self.role_type == "feral":
            return
        if any(g.get("_source") == "scream_response" for g in self.goals):
            return
        if _P832_CAUSAL_ABLATIONS.get("move_ignore_scream_response_goal", False):
            return
        st = _st(self.world)
        W = Config.WORLD_SIZE
        xs = [c[0] for c in self.cells]; ys = [c[1] for c in self.cells]
        local_alarm = float(np.mean(field[xs, ys, CH["signal_alarm"]]))
        noise_floor = max(BG_FLOOR_MIN,
                          (st.bg_alarm if self.world is not None else 0.0) * BG_FLOOR_MULT)
        if local_alarm < noise_floor:
            return
        cx, cy = self.get_center()
        here = (int(cx) % W, int(cy) % W)
        now = int(getattr(self.world, "age", self.age))
        best, best_score = None, 0.0
        for sid, rec in list(st.recent.items()):
            age = now - int(rec["t"])
            if age < 0 or age > SCREAM_SOURCE_TTL:
                if age > SCREAM_SOURCE_TTL:
                    st.recent.pop(sid, None)
                continue
            if sid == self.id:
                continue
            d = _scream_dist(here, rec["pos"], W)
            if d < 1.0:
                continue
            reach = float(field[rec["pos"][0], rec["pos"][1], CH["signal_alarm"]]) \
                / (1.0 + d / max(SCREAM_RADIUS, 1))
            if reach < noise_floor:
                continue
            if reach > best_score:
                best_score, best = reach, (sid, rec["pos"], d, age)
        if best is None:
            return
        sid, target, dist, age = best
        if self._consume_action('scream_response') is None:
            return
        self.goals.append({"type": "explore", "priority": 1.5 + local_alarm,
                           "target": tuple(target), "age": 0, "persistence": 20,
                           "_source": "scream_response", "_source_id": sid,
                           "_created_t": now})
        self._scream_target = tuple(target)
        st.scream["response_goal"] += 1
        _scream_publish(self.world, "response_goal", t=now, id=self.id,
                        source_id=sid, target=tuple(target), distance=round(dist, 3),
                        source_age=age, local_alarm=round(local_alarm, 4),
                        noise_floor=round(noise_floor, 4),
                        reach=round(best_score, 4), pos=here)

    def feral_execute(self, other, field):
        if self.role_type == "feral" and not Config.ENABLE_FERAL_PREDATION:   # [AUDIT#4]
            return False
        orig = super().feral_execute
        if self.role_type != "feral" or field is None or not self.cells or self.world is None:
            return orig(other, field)
        st = _st(self.world)
        W = Config.WORLD_SIZE
        cx, cy = self.get_center()
        here = (int(cx) % W, int(cy) % W)
        now = int(self.world.age)
        chan = field[:, :, CH["signal_alarm"]]
        alarm_at_center = _disc_mean(chan, here[0], here[1], KILL_SIGNAL_RADIUS, W)
        core = _disc_mean(chan, here[0], here[1], KILL_CORE_RADIUS, W)
        st.kill_gate["attempts"] += 1
        st.alarm_max, st.core_max = max(st.alarm_max, alarm_at_center), max(st.core_max, core)
        st.alarm_hist[f"{min(int(alarm_at_center * 10), 10) / 10:.1f}"] += 1
        if (alarm_at_center < KILL_MIN_SIGNAL_AT_CENTER
                and not _P832_CAUSAL_ABLATIONS.get("kill_gate_bypass_alarm_threshold", False)):
            st.kill_gate["signal_low"] += 1
            return orig(other, field)
        if (alarm_at_center < KILL_MIN_SIGNAL_AT_CENTER
                and _P832_CAUSAL_ABLATIONS.get("kill_gate_bypass_alarm_threshold", False)):
            st.kill_gate["signal_threshold_bypassed"] += 1
        screamers = []
        for p in self.world.patterns:
            if (not p.alive or not p.cells or p.id == self.id
                    or getattr(p, "role_type", None) == "feral"):
                continue
            last_t = int(getattr(p, "_last_scream_t", -10**9))
            if now - last_t < 0 or now - last_t > KILL_SCREAM_FRESH:
                continue
            pos = getattr(p, "_last_scream_pos", None)
            if pos is not None and _scream_dist(pos, here, W) < 10:
                screamers.append(p)
        st.screamers_max = max(st.screamers_max, len(screamers))
        st.screamers_hist[str(min(len(screamers), 9))] += 1
        if len(screamers) < KILL_MIN_SCREAMERS:
            st.kill_gate["screamers_low"] += 1
            return orig(other, field)
        feral_energy = float(self.energy)
        if not np.isfinite(feral_energy) or feral_energy < 0.0:
            st.kill_gate["energy_invalid"] += 1
            st.kill_negative_feral_energy += 1
            _scream_publish(self.world, "kill_blocked", t=now, feral=self.id,
                            energy=repr(feral_energy))
            return orig(other, field)

        st.kill_gate["passed"] += 1
        _energy_cap = float(Config.AGENT_ENERGY_MAX)
        _headroom = np.asarray([
            max(0.0, _energy_cap - float(s.energy)) if np.isfinite(float(s.energy)) else 0.0
            for s in screamers
        ], dtype=np.float64)
        _alloc = np.zeros(len(screamers), dtype=np.float64)
        _remaining = min(feral_energy, float(_headroom.sum(dtype=np.float64)))
        _active = np.flatnonzero(_headroom > 1e-12)
        # Water-fill equal shares; when one recipient reaches the cap, redistribute
        # its unaccepted share among the remaining recipients with headroom.
        while _active.size and _remaining > 1e-12:
            _share = _remaining / float(_active.size)
            _grant = np.minimum(_headroom[_active] - _alloc[_active], _share)
            _moved = float(_grant.sum(dtype=np.float64))
            if _moved <= 1e-12:
                break
            _alloc[_active] += _grant
            _remaining = max(0.0, _remaining - _moved)
            _active = _active[(_headroom[_active] - _alloc[_active]) > 1e-12]

        _received_by_screamer = []
        for s, _amount in zip(screamers, _alloc):
            if _amount <= 1e-12:
                _received_by_screamer.append(0.0)
                continue
            _received = float(s._x2_change(
                "X2.scream_kill_reward", float(s.energy) + float(_amount), offered=float(_amount)
            ))
            _received_by_screamer.append(max(0.0, _received))
            if _received > 1e-12:
                s._scream_saved_count = getattr(s, "_scream_saved_count", 0) + 1
        _paid = float(sum(_received_by_screamer))

        # ВАРИАНТ 2: мясо берётся из X2 ферала (X2→X2), а не из клеток поля.
        # Порядок: 1) ферал оплачивает награду за убийство (_paid, уже выданную крикунам);
        #          2) из остатка X2 ферала крикунам уходит мясо — ПОКА у ферала есть X2;
        #          3) то, что осталось, обнуляем отдельной строкой диссипации.
        _e_before_kill = float(self.energy)
        self._x2_change("X2.scream_kill_feral_loss", _e_before_kill - _paid, offered=-_paid)
        _e_after_kill = float(self.energy)
        _actual_feral_loss = _e_before_kill - _e_after_kill
        if abs(_actual_feral_loss - _paid) > 1e-6:
            st.kill_energy_balance_warnings += 1
            _scream_publish(self.world, "energy_balance_warning", t=now, feral=self.id,
                            feral_energy_offered=round(feral_energy, 8),
                            energy_paid=round(_paid, 8),
                            actual_loss=round(_actual_feral_loss, 8),
                            delta=round(_actual_feral_loss - _paid, 8))
        _feral_meat(self.world, self, screamers, field, now)
        _e_rest = float(self.energy)
        if _e_rest > 0.0:
            self._x2_change("X2.scream_kill_feral_rest_dissipated", 0.0, offered=-_e_rest)

        st.scream["kill"] += 1
        st.scream["kill_energy_moved"] += _paid
        _scream_publish(self.world, "kill", t=now, feral=self.id,
                        screamers=[s.id for s in screamers],
                        feral_energy_before=round(feral_energy, 8),
                        energy_distributed=round(_paid, 8),
                        energy_unaccepted=round(max(0.0, feral_energy - _paid), 8),
                        shares_by_screamer={str(s.id): round(v, 8) for s, v in zip(screamers, _received_by_screamer)},
                        actual_feral_loss=round(_actual_feral_loss, 8),
                        alarm_at_center=round(alarm_at_center, 4),
                        alarm_core=round(core, 4), pos=here)
        try:
            self.world.witness.record(self.id, "feral_killed_by_scream",
                                      screamers=[s.id for s in screamers])
        except Exception as exc:
            _scream_publish(self.world, "error", t=now, where="witness", err=repr(exc))
        try:
            self.world._record_death_drag(self, "scream_kill")
        except Exception as exc:
            _scream_publish(self.world, "error", t=now, where="death_drag", err=repr(exc))
        try:
            self._log_event("death_scream_kill")
        except Exception as exc:
            _scream_publish(self.world, "error", t=now, where="death_event", err=repr(exc))
        try:
            self._deposit_final_testament()
        except Exception as exc:
            _scream_publish(self.world, "error", t=now, where="final_testament", err=repr(exc))
        if self.world is not None and getattr(self.world, 'echo_system', None) is not None:
            try:
                self.world.echo_system.store(self)
            except Exception as exc:
                _scream_publish(self.world, "error", t=now, where="echo_store", err=repr(exc))
        for x, y in list(self.cells):
            field[x, y, CH["owner"]] = 0
        self.cells = set()
        self.alive = False
        return True


class P832EngineMixin:

    def _conservation_check(self, t):
        return


    def _collect_metrics(self, t):
        m = super()._collect_metrics(t)
        if self.patterns:
            m.update(_flat(self))
        return m

    def _audit_close_tick(self, t, de_x1, res_x1, x2_step0):
        super()._audit_close_tick(t, de_x1, res_x1, x2_step0)
        st = _st(self)
        en = np.array([float(p.energy) for p in self.patterns if p.alive], dtype=np.float64)
        bad = int((~np.isfinite(en)).sum()) if en.size else 0
        fin = en[np.isfinite(en)] if en.size else en
        if fin.size:
            st.x2_min, st.x2_max = min(st.x2_min, float(fin.min())), max(st.x2_max, float(fin.max()))
        neg = int((fin < 0.0).sum()) if fin.size else 0
        if neg:
            st.x2_neg_ticks += 1
        below_contract = int((fin < Config.AGENT_ENERGY_MIN).sum()) if fin.size else 0
        above_contract = int((fin > Config.AGENT_ENERGY_MAX).sum()) if fin.size else 0
        contract_bad = below_contract + above_contract
        if bad:
            st.x2_nonfinite_ticks += 1
            if st.x2_nonfinite_ticks == 1:
                print(f"[t={t}] ⚠️ ENERGY CONTRACT: не-конечная энергия у {bad} агентов; "
                      f"допустимый диапазон [{Config.AGENT_ENERGY_MIN}, {Config.AGENT_ENERGY_MAX}].")
        cur = {"cry": st.scream["total"], "repel": st.scream["repel"],
               "resp": st.scream["response_goal"],
               "reach": st.scream["response_reached"],
               "kill": st.scream["kill"],
               "spent": float(sum(st.spent.values())),
               "blocked": int(sum(v for k, v in st.actions.items()
                                  if "blocked" in k or "unpaid" in k))}
        pv = st.prev
        row = self._audit_rows[-1]
        row.update({
            "scr_cry": cur["cry"] - pv.get("cry", 0),
            "scr_repel": cur["repel"] - pv.get("repel", 0),
            "scr_resp": cur["resp"] - pv.get("resp", 0),
            "scr_reach": cur["reach"] - pv.get("reach", 0),
            "scr_kill": cur["kill"] - pv.get("kill", 0),
            "act_spent": cur["spent"] - pv.get("spent", 0.0),
            "act_blocked": cur["blocked"] - pv.get("blocked", 0),
            "x2_min": float(fin.min()) if fin.size else 0.0,
            "x2_neg": neg, "x2_nonfinite": bad,
            "x2_out_of_contract": contract_bad,
            "kill_ebw": st.kill_energy_balance_warnings,
            "kill_neg_e": st.kill_negative_feral_energy,
            "intent_sw_blocked_pre": st.intent_switch_blocked_pre,
        })
        st.prev = cur

    def audit_summary(self):
        S = super().audit_summary()
        S["field_capacity_rebalance"] = {
            "unplaced_overflow_total": float(getattr(self, "_field_capacity_unplaced_total", 0.0))
        }
        st = _st(self)
        _x1 = S["x1"]["ledger"]
        _sun_in = float(_x1.get("X1.Given_Flux", {}).get("net", 0.0))
        _given = {k: -float(v["net"]) for k, v in _x1.items()
                  if k in ("X1.field_to_agent_harvest", "X1.division_ready_transfer", "X1.feral_meat_transfer")}
        _given_sum = float(sum(_given.values()))
        S["sun_flow"] = {
            "mode": Config.GIVEN_FLUX_MODE, "cycle": Config.GIVEN_FLUX_CYCLE,
            "period": int(Config.GIVEN_FLUX_PERIOD), "power": float(Config.GIVEN_FLUX_POWER),
            "radius": float(Config.GIVEN_FLUX_RADIUS),
            "sum_given_flux": _sun_in,
            "extracted_by_agents": _given, "sum_extracted_by_agents": _given_sum,
            "extracted_over_given": (_given_sum / _sun_in) if _sun_in > 0 else None,
            "residual_x1": dict(S["x1"]["residual"]), "residual_x2": dict(S["x2"]["residual"]),
        }
        S["patches"] = {
            "manifest": _MANIFEST,
            "seed_policy": {
                "given_seed": int(self.given_seed),
                "force_given_seed": Config.FORCE_GIVEN_SEED,
                "given_rng_enabled": bool(Config.ENABLE_GIVEN_SEED),
                "deterministic_seed": int(Config.DETERMINISTIC_SEED),
                "note": "Given-RNG сидируется на прогон; φ-шум фиксирован DETERMINISTIC_SEED",
            },
            "scream": {k: (float(v) if isinstance(v, float) else int(v))
                       for k, v in st.scream.items()},
            "actions": dict(st.actions),
            "action_spent": {k: round(float(v), 6) for k, v in st.spent.items()},
            "kill_gate": {**dict(st.kill_gate),
                          "alarm_max": st.alarm_max, "core_max": st.core_max,
                          "screamers_max": st.screamers_max,
                          "alarm_hist": dict(sorted(st.alarm_hist.items())),
                          "screamers_hist": dict(sorted(st.screamers_hist.items())),
                          "min_signal": KILL_MIN_SIGNAL_AT_CENTER,
                          "min_screamers": KILL_MIN_SCREAMERS,
                          "signal_radius": KILL_SIGNAL_RADIUS,
                          "core_radius": KILL_CORE_RADIUS,
                          "energy_balance_warnings": st.kill_energy_balance_warnings,
                          "negative_feral_energy": st.kill_negative_feral_energy},
            "intent_switch_blocked_pre": st.intent_switch_blocked_pre,
            "x2_guard": {
                "min": (st.x2_min if st.x2_min != float("inf") else None),
                "max": (st.x2_max if st.x2_max != float("-inf") else None),
                "neg_ticks": st.x2_neg_ticks,
                "nonfinite_ticks": st.x2_nonfinite_ticks,
                "fatigue_peak": st.fatigue_peak,
            },
            "events": {
                "emitted": dict(st.emitted),
                "dropped": dict(st.dropped),
                "cap_action": ACTION_EVENTS_CAP,
                "cap_scream": SCREAM_EVENTS_MAX,
            },
        }
        return S

    def _post_semantic_step(self, t):
        st = _st(self)
        st.bg_alarm = float(np.mean(self.field[:, :, CH["signal_alarm"]]))
        super()._post_semantic_step(t)
        now = int(self.age)
        if now % SCREAM_ALARM_DECAY_EVERY == 0:
            self.field[:, :, CH["signal_alarm"]] *= SCREAM_ALARM_DECAY
        if now % FATIGUE_DECAY_EVERY == 0:
            for p in self.patterns:
                if p.alive:
                    p._recover_fatigue()
                    p._tick_goal_cost()
        for p in self.patterns:
            if p.alive:
                _scream_try(p, self.field, now)
        _scream_check_response_arrival(self, now)

        if now % SNAPSHOT_EVERY == 0 and now > 0:
            flat = _flat(self)
            rich = getattr(self, "rich", None)
            if rich is not None and hasattr(rich, "events"):
                try:
                    rich.events.append({"t": now, "type": "p832_snapshot", **flat})
                except Exception:
                    st.dropped["rich_error"] += 1
            s = st.scream
            r_frac = (s["response_reached"] / s["response_goal"] * 100.0) \
                if s["response_goal"] else 0.0
            print(
                f"\n[t={now}] FATIGUE avg={flat['act_fatigue_avg']:.2f} "
                f"max={flat['act_fatigue_max']:.2f} "
                f"spent={flat['act_spent']:.2f}  "
                f"X2 min={flat['x2g_min']:.3f} max={flat['x2g_max']:.2f} "
                f"neg_ticks={flat['x2g_neg_ticks']} "
                f"nonfinite={flat['x2g_nonfinite_ticks']}\n"
                f"[SCREAM t={now}] CRY={s['total']}(feral:{s['feral']} rise:{s['rise']}) "
                f"hab_ticks={s['habituation_blocked']} "
                f"cost_block={s['cost_blocked']}\n"
                f"           REPEL={s['repel']}  "
                f"RESPONSE={s['response_goal']}→{s['response_reached']} "
                f"({r_frac:.0f}% дошли)  KILL={s['kill']} "
                f"energy_moved={s['kill_energy_moved']:.2f} "
                f"meat={s['meat_kills']}({s['meat_energy']:.2f}) "
                f"ebw={flat['scr_kill_ebw']} neg_e={flat['scr_kill_neg_e']}\n"
                f"[KILL-GATE t={now}] попыток={st.kill_gate['attempts']} "
                f"режет сигнал={st.kill_gate['signal_low']} "
                f"режет кричащие={st.kill_gate['screamers_low']} "
                f"прошло={st.kill_gate['passed']} | "
                f"alarm max={st.alarm_max:.3f} (порог {KILL_MIN_SIGNAL_AT_CENTER}), "
                f"core{KILL_CORE_RADIUS} max={st.core_max:.3f}, "
                f"кричащих max={st.screamers_max} (нужно {KILL_MIN_SCREAMERS})\n"
                f"[ACTIONS] " + " ".join(f"{k}:{v}" for k, v in sorted(st.actions.items())) + "\n"
                f"           intent_sw_blocked_pre={st.intent_switch_blocked_pre} "
                f"bg_alarm={st.bg_alarm:.4f}" + _morse_gates_line(self, now),
                flush=True,
            )

Pattern = extend_class(Pattern, P832PatternMixin)
EvolutionEngine = extend_class(EvolutionEngine, P832EngineMixin)
LedgerEngine = extend_class(LedgerEngine, P832EngineMixin)
RichEngine = extend_class(RichEngine, P832EngineMixin)
for _cls in (Pattern, RichEngine):
    assert any(k.__name__ in ("P832PatternMixin", "P832EngineMixin") for k in _cls.__mro__), f"слой p832 не подключён к {_cls.__name__}"
print(f"=== ENERGY CONTRACT: agent.energy in [{Config.AGENT_ENERGY_MIN}, {Config.AGENT_ENERGY_MAX}] ===")
print("=== FIELD ENERGY: toroidal diffusion D=0.05 + explicit dissipation k=0.02 ===")
print("=== ACTION PATCHES ready (strict payment) ===")
print(f"=== СЛОИ ПОДКЛЮЧЕНЫ, без монки-патча  ({_MANIFEST['id']}) ===")
print(f"kill gate: signal(r={KILL_SIGNAL_RADIUS})>{KILL_MIN_SIGNAL_AT_CENTER} "
      f"и ≥{KILL_MIN_SCREAMERS} кричащих в r<10, свежесть {KILL_SCREAM_FRESH}t")
print("strict payment: given, scream, intent_switch, divide, introspect, "
      "meditation, trade, heal_field — без allow_partial")
print("==============")

SEED, STEPS, MULT = 118951255, 1666, 0.15
PULL = 0.08
RUN_ARMS = []  # Deliberately single-run by default; arms need explicit ablation overrides.
EMPATHY_PULL_OVERRIDE = PULL
NEIGHBOR_SYNC_OVERRIDE = True


TAG_SUFFIX = "v38_meat2"


def _tag(arm):
    return (f"rich_dlgV2_pull{PULL}_normTrue_seed{SEED}_steps{STEPS}_mult{MULT}_"
            f"{arm}_noclip_actionfatigue_scream_{TAG_SUFFIX}")


def _rotate(path):
    if os.path.exists(path):
        prev = path + ".prev"
        if os.path.exists(prev):
            os.remove(prev)
        os.rename(path, prev)


def _run_arm(arm):
    tag = _tag(arm)
    rich_path = f"{DIR}/rich_log_{tag}.json"
    ledger_path = f"{DIR}/rich_ledger_{tag}.json"
    for path in (rich_path, ledger_path, f"{DIR}/audit_{tag}.json"):
        _rotate(path)
    extra = {"AUDIT_DIR": DIR, "AUDIT_TAG": tag, "PATCH_SET": _MANIFEST["id"]}
    print("=" * 78)
    print(f"RUN: arm={arm} tag={tag} patch={_MANIFEST['id']}")
    print("=" * 78)
    started = time.time()
    led, log = _run_rich(SEED, STEPS, MULT, ledger_path, rich_path,
                         extra_overrides=extra, verbose=True)
    print(f"\nwall={round(time.time() - started, 1)}s; "
          f"complete={log.header.get('complete')}")
    for _obj, _path, _has in ((led, ledger_path, bool(led.rows)),
                              (log, rich_path, bool(log.tick_rows))):
        try:
            _obj.header["patches"] = _MANIFEST
            if _has:
                _obj.save(_path)
        except Exception as exc:
            print(f"⚠️ манифест не записан в {os.path.basename(_path)}: {exc!r}")
    return tag


def _print_final_stats():
    _last = _P832_LAST["st"]
    if _last is None:
        return
    print("\n=== FINAL ACTION STATS ===")
    for k, v in sorted(_last.actions.items()):
        print(f"  {k}: {v}")
    try:   # intent_switch: события (единый мировой счётчик в _log_event) должны = оплачено + неоплачено
        _evw = dict(getattr(_P832_LAST.get("world"), "_ev_world", None) or {})
        _e_sw = int(_evw.get("intent_switch", 0))
        _paid = int(_last.actions.get("intent_switch", 0))
        _unp = int(_last.actions.get("intent_switch_unpaid", 0))
        print(f"  [intent_switch: единый учёт] событий={_e_sw} (_log_event) | оплачено={_paid} + неоплачено={_unp} = {_paid + _unp} | "
              + ("✅ сходится" if _e_sw == _paid + _unp else f"⚠️ расхождение {_e_sw - _paid - _unp:+d} (смены мимо update_intent слоя P832 или не оплачены)"))
    except Exception as _e_isw:
        print(f"  [intent_switch: единый учёт] недоступен: {type(_e_isw).__name__}")
    _morse_final_report(_P832_LAST.get("world"), _last)
    print("\n=== FINAL SCREAM STATS ===")
    for k, v in sorted(_last.scream.items()):
        print(f"  {k}: {v}")
    print(f"\n=== KILL-GATE: {dict(_last.kill_gate)} | "
          f"alarm max={_last.alarm_max:.3f} core max={_last.core_max:.3f} "
          f"screamers max={_last.screamers_max} ===")
    print(f"=== ENERGY BALANCE WARNINGS: {_last.kill_energy_balance_warnings} | "
          f"NEGATIVE FERAL ENERGY: {_last.kill_negative_feral_energy} | "
          f"INTENT SWITCH BLOCKED PRE: {_last.intent_switch_blocked_pre} ===")
    print(f"=== SCREAM_EVENTS в памяти: {len(_last.events)} | "
          f"по типам: {dict(Counter(e['type'] for e in _last.events))} ===")
    print("\n=== последние 10 KILL ===")
    for e in [x for x in _last.events if x['type'] == 'scream_kill'][-10:]:
        print(e)

In [ ]:
# Ячейка 30
import math
import numpy as np
from collections import Counter, deque

for _need in ("Pattern", "EvolutionEngine", "LedgerEngine", "RichEngine", "Config", "CH", "phi_hash", "extend_class",
              "_sg_normalize", "_sg_ctx", "_stable_hash"):
    if _need not in globals():
        raise RuntimeError(f"{_need} не найден: выполни ячейки выше (…→ SIGNAL → p832 → эта ячейка).")
_mro_names = {k.__name__ for k in Pattern.__mro__}
for _need in ("P832PatternMixin", "SignalMixin", "MorseMixin"):
    if _need not in _mro_names:
        raise RuntimeError(f"В Pattern нет слоя {_need}: эту ячейку нужно запускать ПОСЛЕ SIGNAL и p832.")
if "NavPatternMixin" in _mro_names:
    raise RuntimeError("Слой NAV+FCH уже подключён. Runtime -> Restart session и запуск сверху вниз.")

_FCH_CFG = {
    'NAV_ENABLE': True,
    'NAV_SENSE_EVERY': 2, 'NAV_RING_R': 4,
    'NAV_DANGER_MIN': 0.03, 'NAV_ALARM_MIN': 0.08,
    'NAV_STEER_BLEND': 1.0, 'NAV_FLEE_BOOST': 1.6, 'NAV_MOB_BOOST': 1.4,
    'NAV_ERR_ALPHA': 0.1, 'NAV_DIS_Z': 3.0, 'NAV_DIS_TENSION': 0.04, 'NAV_FREEZE_GAIN': 0.6,
    'NAV_REST_GOAL': True, 'NAV_REST_DIS': 0.5,
    'NAV_PI_NOISE': 0.10, 'NAV_TURN_NOISE': 0.05,
    'NAV_ANCHOR_TOL': 0.03, 'NAV_ANCHOR_EVERY': 12, 'NAV_ANCHOR_MIN_PATH': 6.0, 'NAV_ANCHOR_MAX': 16,
    'NAV_SPATIAL_TENSION_GAIN': 0.2, 'NAV_SPATIAL_HOLD': 0.6,
    'NAV_LOST_CONF': 0.4, 'NAV_ERROR_LOG_THRESHOLD': 0.15,
    'NAV_LEGACY_TENSION_HACK': False,
    'NAV_ANCHOR_GATE': 8.0, 'NAV_ANCHOR_ERR_FULL': 6.0,
    'NAV_LANDMARK_SEED': 8321, 'NAV_LANDMARK_SMOOTH': 32,
    'NAV_PLACE_NODES': True,
    'NAV_PLACE_MAX_NODES': 30, 'NAV_PLACE_MAX_VISITS': 20,
    'NAV_PLACE_FP_SIGMA_MIN': 0.02, 'NAV_PLACE_FP_SIGMA_MAX': 0.035,
    'NAV_PLACE_Z0': 1.0, 'NAV_PLACE_Z_FULL': 4.0, 'NAV_PLACE_Z_HALF': 2.0,
    'NAV_PLACE_W_FP': 0.7,
    'NAV_PLACE_PI_GATE': 5.0, 'NAV_PLACE_PI_SCALE': 15.0, 'NAV_PLACE_PI_DRIFT': 0.5, 'NAV_PLACE_PI_SCALE_GROW': 4.0,
    'NAV_PLACE_CTX_BASE': 0.85,
    'NAV_PLACE_MIN_CONF': 0.5, 'NAV_PLACE_MAX_ERR': 0.4,
    'NAV_PLACE_NOPI_EXTRA': 0.2,
    'NAV_PLACE_COARSE_FP': 0.12, 'NAV_PLACE_CAND_MAX': 12,
    'NAV_PLACE_AMBIG_MARGIN': 0.08, 'NAV_PLACE_AMBIG_CONF': 0.6,
    'NAV_PLACE_MISS_LIMIT': 3, 'NAV_PLACE_NEAR_FRAC': 0.6,
    'NAV_PLACE_NOVEL_CONFIRM': 3, 'NAV_PLACE_NOVEL_FP_TOL': 0.045, 'NAV_PLACE_RECOG_CONFIRM': 2, 'NAV_PLACE_NOVEL_PI_GATE': 4.0, 'NAV_PLACE_PEND_TTL': 12,
    'NAV_PLACE_NEW_CONF': 0.4,
    'NAV_PLACE_FP_ERR_W': 0.5,
    'NAV_PLACE_VISIT_CLOCK_FIX': True,
    'NAV_PLACE_DIAG': True,
    'NAV_PLACE_VISIT_GAP': 12, 'NAV_PLACE_EDGE_MAX_GAP': 300,
    'NAV_PLACE_ALIAS_GUARD': True, 'NAV_PLACE_ALIAS_FRAC': 0.5,
    'NAV_PLACE_DRIFT_ADAPT': True, 'NAV_PLACE_DRIFT_CONFIRM': 2,
    'NAV_PLACE_PRED_MIN': 2,
    'PN_V2_ENABLED': True,
    'PN_V2_APPLY_STANCE': True,
    'PN_V2_LOG_EVENTS': False,
    'PN_V2_DISC_MIN': 0.15,
    'PN_V2_EVID_MIN': 0.2, 'PN_V2_UNEXPL_PRIOR': 0.15,
    'PN_V2_EXP_MAX': 200, 'PN_V2_EPISODE_GAP': 12,
    'PN_V2_Q_BUDGET': 5, 'PN_V2_Q_TTL': 50, 'PN_V2_Q_MIN_OBS': 2, 'PN_V2_Q_CLOSED_KEEP': 30,
    'PN_V2_Q_RESOLVE_CONFIRM': 2, 'PN_V2_Q_IRRES_SUPPORT': 6, 'PN_V2_Q_IRRES_MAX': 2,
    'PN_V2_STORY_MAX': 40, 'PN_V2_FAIL_MAX': 32, 'PN_V2_FAIL_STABLE': 4, 'PN_V2_QUIET_INTERVAL': 100,
    'PN_V2_ST_INVEST': 0.1, 'PN_V2_ST_ACCOM': 0.5, 'PN_V2_ST_DEFEND': 0.7, 'PN_V2_ST_DENY': 0.9,
    'PN_V2_RISK_DEFEND': 0.7, 'PN_V2_ENERGY_MIN': 0.35, 'PN_V2_VOI_MIN': 1.0, 'PN_V2_STANCE_HOLD': 2,
    'PN_V2_STANCE_PERSIST': 10, 'PN_V2_PRIO_REST': 1.5, 'PN_V2_PRIO_EXPLORE': 1.2,
    'FCH_ENABLE': True,
    'FCH_MORSE_SCALE': 0.25, 'FCH_MORSE_MAX': 0.45, 'FCH_MORSE_MIN_EXCESS': 0.05,
    'FCH_TAME_LR': 0.08, 'FCH_TAME_DECAY': 0.985, 'FCH_TAME_MAX': 1.0, 'FCH_TAME_DETER': 0.6, 'FCH_TAME_RADIUS': 8,
    'FCH_BACKOFF': 25,
    'FCH_MORSE_SEND': 0.06, 'FCH_APPEASE_COOLDOWN': 6, 'FCH_APPEASE_RATE': 0.6,
    'FCH_GENE_MUT': 0.08,
}
for _k, _v in _FCH_CFG.items():
    if not hasattr(Config, _k):
        setattr(Config, _k, _v)

_FCH_ABL = {
    "nav_off": False,
    "anchor_off": False,
    "morse_deter_off": False,
    "tame_off": False,
    "appease_off": False,
    "morse_threat_send_off": False,
}

_NAV_IDX_ERR = np.array([0, 1, 3, 4])
_NAV_ERR_FLOOR = np.array([0.01, 0.02, 0.05, 0.02])
_NAV_DIRS = ((1, 0), (1, 1), (0, 1), (-1, 1), (-1, 0), (-1, -1), (0, -1), (1, -1))
_NAV_U = np.array([(dx / math.hypot(dx, dy), dy / math.hypot(dx, dy)) for dx, dy in _NAV_DIRS], dtype=np.float64)
_NAV_OFFS = {}
_NAV_CHAN = {}
_FCH_GENES = (('flee', 0.0, 1.5, 0.2, 0.8), ('mob', 0.0, 1.5, 0.2, 0.8), ('coping', 0.0, 1.0, 0.1, 0.9), ('morse', 0.2, 3.0, 0.5, 1.5), ('sprint', 0.0, 1.5, 0.3, 0.7), ('rage', 0.0, 1.5, 0.3, 0.7))
_FCH_PEACE_NAMES = ('signal_curiosity', 'signal_invitation')


def _fch_peace_idx():
    return tuple(CH[n] for n in _FCH_PEACE_NAMES if CH.get(n) is not None)


class _FchState:
    def __init__(self):
        self.st = Counter()
        self.prof_exp, self.prof_strike, self.prof_kill = Counter(), Counter(), Counter()
        self.veto = Counter()
        self.bg_morse = 0.0
        self.bg_peace = 0.0
        self.hist = []
        self.err = None


def _fch(world):
    s = getattr(world, '_fch_state', None)
    if s is None:
        s = _FchState()
        world._fch_state = s
    return s


def _fch_err(s, tag, e):
    if s is None:
        return
    s.st['err_' + tag] += 1
    if s.err is None:
        import traceback
        print(f"[NAV/FCH] ⚠️ {tag}: {e!r}\n{traceback.format_exc()}", flush=True)
    s.err = f"{tag}: {e!r}"


def _fch_morse_ch():
    return int(getattr(Config, 'MORSE_CHANNEL', 33))


def _nav_offs(R):
    o = _NAV_OFFS.get(R)
    if o is None:
        px = [i for i in (-1, 0, 1) for _j in (-1, 0, 1)]
        py = [j for _i in (-1, 0, 1) for j in (-1, 0, 1)]
        ox, oy = list(px), list(py)
        for ux, uy in _NAV_U:
            cx0, cy0 = int(round(ux * R)), int(round(uy * R))
            ox += [cx0 + i for i in px]
            oy += [cy0 + j for j in py]
        o = (np.array(ox), np.array(oy))
        _NAV_OFFS[R] = o
    return o


def _nav_chans(field):
    key = field.shape[2]
    c = _NAV_CHAN.get(key)
    if c is None:
        m = _fch_morse_ch()
        has_m = m < field.shape[2]
        c = (np.array([CH['signal_feral'], CH['signal_alarm'], m if has_m else CH['signal_feral'], CH['owner'], CH['energy']]), has_m)
        _NAV_CHAN[key] = c
    return c


def _nav_sample(field, cx, cy, selfid):
    W = Config.WORLD_SIZE
    R = int(Config.NAV_RING_R)
    OX, OY = _nav_offs(R)
    ch, has_m = _nav_chans(field)
    vals = field[((cx + OX) % W)[:, None], ((cy + OY) % W)[:, None], ch[None, :]].astype(np.float64)
    if not has_m:
        vals[:, 2] = 0.0
    own = vals[:, 3].copy()
    vals[:, 3] = ((own != 0.0) & (own != float(selfid))).astype(np.float64)
    body = vals[:9].mean(axis=0)
    ring = vals[9:].reshape(8, 9, -1).mean(axis=1)
    G = (2.0 / (8.0 * R)) * (_NAV_U.T @ (ring - body))
    return body, ring, G


def _nav_unit(g, ring_col):
    n = float(math.hypot(g[0], g[1]))
    if n > 1e-6:
        return (float(g[0]) / n, float(g[1]) / n)
    k = int(np.argmax(ring_col))
    if float(ring_col[k]) <= 0.0:
        return None
    return (float(_NAV_U[k][0]), float(_NAV_U[k][1]))


_NAV_LM = {}


def _nav_landmark_map():
    W = int(Config.WORLD_SIZE)
    m = _NAV_LM.get(W)
    if m is None:
        m = np.random.RandomState(int(Config.NAV_LANDMARK_SEED)).rand(W, W)
        for _ in range(int(Config.NAV_LANDMARK_SMOOTH)):
            m = (m + np.roll(m, 1, 0) + np.roll(m, -1, 0) + np.roll(m, 1, 1) + np.roll(m, -1, 1)) / 5.0
        m = (m - m.min()) / max(1e-9, float(m.max() - m.min()))
        _NAV_LM[W] = m
    return m


def _nav_fingerprint(field, cx, cy):
    W = int(Config.WORLD_SIZE)
    OX, OY = _nav_offs(int(Config.NAV_RING_R))
    v = _nav_landmark_map()[(cx + OX) % W, (cy + OY) % W]
    return np.concatenate(([v[:9].mean()], v[9:].reshape(8, 9).mean(axis=1)))


def _nav_dead_reckon(n, dx, dy, dd, selfid, now):
    if dd < 0.5:
        return
    ux, uy = dx / dd, dy / dd
    up = n['u_prev']
    if up is not None:
        n['th'] += math.atan2(up[0] * uy - up[1] * ux, up[0] * ux + up[1] * uy)
    n['u_prev'] = (ux, uy)
    n['th'] += (phi_hash(selfid, now, 4103) - 0.5) * 2.0 * float(Config.NAV_TURN_NOISE)
    f = dd * (1.0 + (phi_hash(selfid, now, 4101) - 0.5) * 2.0 * float(Config.NAV_PI_NOISE))
    n['pi'][0] += f * math.cos(n['th'])
    n['pi'][1] += f * math.sin(n['th'])
    n['path'] += dd


def _nav_place_stub(n, a_err, path_before):
    gate = max(1e-6, float(Config.NAV_ANCHOR_GATE))
    err = min(1.0, float(a_err) / max(1e-6, float(Config.NAV_ANCHOR_ERR_FULL))) if a_err > 0.0 else 0.0
    conf = 1.0 / (1.0 + max(0.0, float(n['path'])) / gate)
    conf_before = 1.0 / (1.0 + max(0.0, float(path_before)) / gate)
    return conf, err, conf_before


def _nav_anchor_step(n, fp, now, st):
    if fp is None:
        return 0.0
    A = n['anch']
    best, bd = None, 1e9
    for an in A:
        d = float(np.mean(np.abs(fp - an['fp'])))
        if d < bd:
            best, bd = an, d
    if best is not None and bd <= float(Config.NAV_ANCHOR_TOL):
        if n['path'] < float(Config.NAV_ANCHOR_MIN_PATH):
            return 0.0
        err = math.hypot(n['pi'][0] - best['pi'][0], n['pi'][1] - best['pi'][1])
        if err > float(Config.NAV_ANCHOR_GATE):
            st['anchor_gate_rej'] += 1
            return 0.0
        n['pi'][0], n['pi'][1] = best['pi'][0], best['pi'][1]
        n['th'] = best['th']
        best['t'] = now
        n['a_last'] = now
        n['path'] = 0.0
        st['anchor_hit'] += 1
        st['anchor_err'] += err
        return max(err, 1e-9)
    if now - n['a_last'] >= int(Config.NAV_ANCHOR_EVERY):
        if len(A) >= int(Config.NAV_ANCHOR_MAX):
            del A[min(range(len(A)), key=lambda i: A[i]['t'])]
        A.append({'fp': fp.copy(), 'pi': (n['pi'][0], n['pi'][1]), 'th': n['th'], 't': now})
        n['a_last'] = now
        n['path'] = 0.0
        st['anchor_new'] += 1
    return 0.0


def _pn_circ_mean(ths):
    s = sum(math.sin(t) for t in ths)
    c = sum(math.cos(t) for t in ths)
    return math.atan2(s, c) if (abs(s) + abs(c)) > 1e-9 else 0.0


def _pn_world_size():
    return int(getattr(Config, 'WORLD_SIZE', 0) or 0)


def _pn_toroidal_delta(a, b):
    size = _pn_world_size()
    d = float(a) - float(b)
    if size > 0:
        d = (d + size / 2.0) % size - size / 2.0
    return d


def _pn_toroidal_dist(a, b):
    return math.hypot(_pn_toroidal_delta(a[0], b[0]),
                      _pn_toroidal_delta(a[1], b[1]))


def _pn_toroidal_mean(values):
    values = [float(v) for v in values]
    size = _pn_world_size()
    if not values:
        return 0.0
    if size <= 0:
        return float(np.mean(values))
    angles = [2.0 * math.pi * (v % size) / size for v in values]
    sx = sum(math.cos(a) for a in angles)
    sy = sum(math.sin(a) for a in angles)
    if math.hypot(sx, sy) < 1e-9:
        # The circular mean is undefined for evenly spread points; retain the
        # first observed coordinate rather than inventing a midpoint.
        return float(values[0] % size)
    return float((math.atan2(sy, sx) % (2.0 * math.pi)) * size / (2.0 * math.pi))


def _pn_restat(place):
    V = place['visits']
    F = np.stack([v['fp'] for v in V]).astype(np.float64)
    place['fp_mean'] = F.mean(axis=0)
    place['fp_var'] = F.var(axis=0)
    place['pi_mean'] = [
        _pn_toroidal_mean([v['pi'][0] for v in V]),
        _pn_toroidal_mean([v['pi'][1] for v in V]),
    ]
    place['th_mean'] = _pn_circ_mean([v['th'] for v in V])


def _pn_new(pid, visits, now):
    place = {'id': int(pid), 'visits': list(visits), 'transitions': {}, 'born_t': int(now), 'last_visit_t': int(now), 'last_seen_t': int(now), 'n_total': len(visits), 'n_hits': 0, 'n_cont': 0}
    _pn_restat(place)
    return place


def _pn_add_visit(place, fp, pi, th, now):
    place['visits'].append({'t': int(now), 'fp': np.asarray(fp, dtype=np.float32).copy(), 'pi': (float(pi[0]), float(pi[1])), 'th': float(th)})
    place['n_total'] += 1
    place['last_visit_t'] = int(now)
    place['last_seen_t'] = int(now)
    over = len(place['visits']) - int(Config.NAV_PLACE_MAX_VISITS)
    if over > 0:
        del place['visits'][:over]
    _pn_restat(place)


def _pn_by_id(n, pid):
    if pid is None:
        return None
    for p in n['places']:
        if p['id'] == pid:
            return p
    return None


def _pn_forget(n, gid):
    # PlaceNode eviction/forgetting is also visible to Spatial Emotional Memory.
    for p in n['places']:
        p['transitions'].pop(gid, None)
    if n['last_place'] == gid:
        n['last_place'] = None
    owner = n.get('_sem_owner')
    sem = getattr(owner, '_spatial_emotional_memory', None) if owner is not None else None
    if sem is not None:
        try:
            sem.on_nav_forget(owner, gid)
        except Exception:
            pass


def _pn_add_node(n, node, st):
    P = n['places']
    P.append(node)
    cur = n['current_place']
    while len(P) > int(Config.NAV_PLACE_MAX_NODES):
        idx = [i for i, p in enumerate(P) if p is not cur and p is not node]
        if not idx:
            break
        k = min(idx, key=lambda i: max(P[i]['last_visit_t'], P[i].get('last_seen_t', 0)))
        gone = P.pop(k)
        _pn_forget(n, gone['id'])
        st['place_evict'] += 1


def _pn_record_edge(n, place, now, st):
    lp = n['last_place']
    if lp is None or lp == place['id'] or now - n['last_conf_t'] > int(Config.NAV_PLACE_EDGE_MAX_GAP):
        return
    src = _pn_by_id(n, lp)
    if src is None:
        return
    e = src['transitions'].get(place['id'])
    if e is None:
        e = {'count': 0, 'last_t': int(now)}
        src['transitions'][place['id']] = e
    e['count'] += 1
    e['last_t'] = int(now)
    st['place_edge'] += 1


def _pn_fit(place, fp, pi, path, cur_id, edge_n):
    C = Config
    sig = np.minimum(np.sqrt(place['fp_var'] + float(C.NAV_PLACE_FP_SIGMA_MIN) ** 2), float(C.NAV_PLACE_FP_SIGMA_MAX))
    z = float(np.mean(np.abs(fp - place['fp_mean']) / sig))
    z0, z1 = float(C.NAV_PLACE_Z0), float(C.NAV_PLACE_Z_FULL)
    err_fp = min(1.0, max(0.0, (z - z0) / max(1e-6, z1 - z0)))
    conf_fp = 1.0 / (1.0 + (z / float(C.NAV_PLACE_Z_HALF)) ** 2)
    d_pi = _pn_toroidal_dist(pi, place['pi_mean'])
    drift = float(C.NAV_PLACE_PI_DRIFT) * max(0.0, path)
    gate = float(C.NAV_PLACE_PI_GATE) + drift
    scale = float(C.NAV_PLACE_PI_SCALE) + float(C.NAV_PLACE_PI_SCALE_GROW) * drift
    if d_pi <= gate:
        conf_pi = 1.0
    elif d_pi < scale:
        conf_pi = 1.0 - (d_pi - gate) / max(1e-6, scale - gate)
    else:
        conf_pi = 0.0
    wf = float(C.NAV_PLACE_W_FP)
    base = (conf_fp ** wf) * (conf_pi ** (1.0 - wf))
    nv = len(place['visits'])
    exp_f = 0.75 + 0.25 * nv / (nv + 2.0)
    ctx = 1.0 if place['id'] == cur_id else min(1.0, float(edge_n) / 3.0)
    cb = float(C.NAV_PLACE_CTX_BASE)
    return err_fp, float(base * exp_f * (cb + (1.0 - cb) * ctx)), d_pi, z


def _pn_candidates(n, fp, pi, path):
    C = Config
    coarse, tight = float(C.NAV_PLACE_COARSE_FP), float(C.NAV_PLACE_FP_SIGMA_MIN) * 1.5
    scale = float(C.NAV_PLACE_PI_SCALE) + float(C.NAV_PLACE_PI_SCALE_GROW) * float(C.NAV_PLACE_PI_DRIFT) * max(0.0, path)
    out, rej = [], 0
    for p in n['places']:
        cd = float(np.mean(np.abs(fp - p['fp_mean'])))
        if cd > coarse:
            continue
        if _pn_toroidal_dist(pi, p['pi_mean']) >= scale:
            if cd <= tight:
                rej += 1
            continue
        out.append((cd, p))
    out.sort(key=lambda x: x[0])
    return out[:int(C.NAV_PLACE_CAND_MAX)], rej


_PN_LAST = {'eng': None}
_PN_DPI_EDGES = (1.0, 2.0, 3.0, 5.0, 8.0, 15.0)


def _pn_gate_scale(path):
    C = Config
    drift = float(C.NAV_PLACE_PI_DRIFT) * max(0.0, path)
    return float(C.NAV_PLACE_PI_GATE) + drift, float(C.NAV_PLACE_PI_SCALE) + float(C.NAV_PLACE_PI_SCALE_GROW) * drift


def _pn_dpi_bucket(d):
    for i, e in enumerate(_PN_DPI_EDGES):
        if d < e:
            return i
    return len(_PN_DPI_EDGES)


def _pn_pend_drop(n, st, why):
    pend = n.get('pending')
    if pend is not None and pend['visits']:
        st['pend_lost_obs'] += len(pend['visits'])
        st['pend_drop_' + why] += 1
    n['pending'] = None


def _pn_hist(vals, edges):
    out = [0] * (len(edges) + 1)
    for v in vals:
        k = 0
        while k < len(edges) and v >= edges[k]:
            k += 1
        out[k] += 1
    return out


def _pn_predict(n, place, st=None):
    # Diagnostics only: count why a next-place prediction cannot be created.
    if st is not None:
        st['place_pred_create_attempt'] += 1
    if place is None:
        if st is not None: st['place_pred_create_none_place'] += 1
        return None
    if not place['transitions']:
        if st is not None: st['place_pred_create_no_edges'] += 1
        return None
    tot = sum(int(e['count']) for e in place['transitions'].values())
    if tot < int(Config.NAV_PLACE_PRED_MIN):
        if st is not None: st['place_pred_create_below_min'] += 1
        return None
    gid, e = max(place['transitions'].items(), key=lambda kv: (kv[1]['count'], kv[1]['last_t']))
    nxt = _pn_by_id(n, gid)
    if nxt is None:
        if st is not None: st['place_pred_create_missing_target'] += 1
        return None
    if st is not None: st['place_pred_created'] += 1
    return {'from': place['id'], 'id': gid, 'p': float(e['count']) / float(tot), 'fp': np.array(nxt['fp_mean'], dtype=np.float64), 'pi': (float(nxt['pi_mean'][0]), float(nxt['pi_mean'][1]))}


def _pn_pred_check(n, place, fp, now, st):
    # Keep the original decision path unchanged; classify only the reason for 'none'.
    pr = n.get('place_pred')
    if pr is None:
        st['place_pred_none'] += 1
        st['place_pred_none_no_prediction'] += 1
        return
    if pr['from'] != n['last_place']:
        st['place_pred_none'] += 1
        st['place_pred_none_source_mismatch'] += 1
        return
    if now - n['last_conf_t'] > int(Config.NAV_PLACE_EDGE_MAX_GAP):
        st['place_pred_none'] += 1
        st['place_pred_none_expired'] += 1
        return
    st['place_pred_hit' if pr['id'] == place['id'] else 'place_pred_miss'] += 1
    st['place_pred_n'] += 1
    st['place_pred_fp_milli'] += int(round(1000.0 * float(np.mean(np.abs(np.asarray(fp, dtype=np.float64) - pr['fp'])))))


def _pn_diag(eng):
    alive = [p for p in eng.patterns if p.alive and p.role_type != 'feral' and getattr(p, '_nav', None)]
    now = int(eng.age)
    nodes, ages, per_agent, act_pend, act_obs = [], [], [], 0, 0
    for p in alive:
        q = p._nav
        ft = q.get('first_t')
        ages.append(now - ft if ft is not None else 0)
        per_agent.append(len(q['places']))
        pd = q.get('pending')
        if pd is not None and pd['visits']:
            act_pend += 1
            act_obs += len(pd['visits'])
        nodes.extend(q['places'])
    # Баланс PNDIAG: у feral навигация не работает, но его висящие наблюдения остаются в _nav['pending'].
    # Раньше они выпадали из «активных» и из баланса (часть невязки). Узлы/возрасты по-прежнему только у не-feral.
    for p in eng.patterns:
        if p.alive and p.role_type == 'feral':
            q = getattr(p, '_nav', None)
            pd = q.get('pending') if isinstance(q, dict) else None
            if pd is not None and pd['visits']:
                act_pend += 1
                act_obs += len(pd['visits'])
    vis = [len(pp['visits']) for pp in nodes]
    hits = [pp.get('n_hits', 0) for pp in nodes]
    conts = [pp.get('n_cont', 0) for pp in nodes]
    life = [now - pp['born_t'] for pp in nodes]
    return {
        'agents': len(alive), 'nodes': len(nodes), 'age_now': now,
        'visits_hist_edges': [4, 5, 6, 8, 11, 21], 'visits_hist': _pn_hist(vis, [4, 5, 6, 8, 11, 21]),
        'visits_eq3_frac': float(np.mean([1.0 if v == 3 else 0.0 for v in vis])) if vis else 0.0,
        'nodes_never_rehit_frac': float(np.mean([1.0 if h == 0 else 0.0 for h in hits])) if hits else 0.0,
        'nodes_hits_mean': float(np.mean(hits)) if hits else 0.0, 'nodes_cont_mean': float(np.mean(conts)) if conts else 0.0,
        'node_life_hist_edges': [10, 25, 50, 100, 200], 'node_life_hist': _pn_hist(life, [10, 25, 50, 100, 200]),
        'agent_age_hist_edges': [10, 25, 50, 100, 200], 'agent_age_hist': _pn_hist(ages, [10, 25, 50, 100, 200]),
        'nodes_per_agent_hist_edges': [1, 2, 3, 5, 10, 20], 'nodes_per_agent_hist': _pn_hist(per_agent, [1, 2, 3, 5, 10, 20]),
        'pending_active_agents': act_pend, 'pending_active_obs': act_obs,
    }


def _pn_diag_report(eng=None):
    eng = eng if eng is not None else _PN_LAST['eng']
    if eng is None:
        print("[PNDIAG] движок ещё не создан")
        return None
    c = _fch(eng).st
    D = _pn_diag(eng)
    add, new, lost, act = int(c['place_pending']), int(c['place_new']), int(c['pend_lost_obs']), int(D['pending_active_obs'])
    lab = ['<1', '1-2', '2-3', '3-5', '5-8', '8-15', '15+']
    tot = sum(int(c['diag_dpi_b%d' % i]) for i in range(len(lab)))
    print("\n" + "=" * 78)
    print(f"[PNDIAG t={D['age_now']}] PlaceNode diag v21  VISIT_CLOCK_FIX={bool(Config.NAV_PLACE_VISIT_CLOCK_FIX)} PLACE_NODES={bool(Config.NAV_PLACE_NODES)}")
    print("=" * 78)
    print("[1] pending = НАКОПИТЕЛЬНЫЙ счётчик наблюдений новизны (не backlog). Баланс:")
    print(f"    added={add} = consumed(3*new={3 * new}) + lost={lost} + active_end={act}  → невязка={add - 3 * new - lost - act}")
    print(f"    потери: ttl={int(c['pend_reset_ttl'])} fp={int(c['pend_reset_fp'])} pi={int(c['pend_reset_pi'])} | снято: recognized={int(c['pend_drop_recognized'])} "
          f"ambig={int(c['pend_drop_ambig'])} uncertain={int(c['pend_drop_uncertain'])} tentative={int(c['pend_drop_tentative'])}")
    print(f"    активных ожиданий: {D['pending_active_agents']} агентов ({act} набл.)")
    print("[2] gate: place_gate_rej = только d_pi>=scale при хорошем виде; обычный gate мягкий (conf_pi).")
    print(f"    d_pi кандидатов с хорошим видом (z<=Z_HALF), n={tot}: " + " ".join(f"{lab[i]}:{int(c['diag_dpi_b%d' % i])}" for i in range(len(lab))))
    print(f"    pi в gate={int(c['diag_pi_in'])} мягкий штраф={int(c['diag_pi_soft'])} conf_pi=0={int(c['diag_pi_zero'])} gate_rej={int(c['place_gate_rej'])}")
    print(f"    воронка: samples={int(c['diag_samples'])} без_кандидатов={int(c['diag_nocand'])} с_кандидатами={int(c['diag_withcand'])} | "
          f"отказ кандидата: conf<thr={int(c['diag_fail_conf'])} err>MAX={int(c['diag_fail_err'])} оба={int(c['diag_fail_both'])}")
    print(f"    acc={int(c['diag_acc'])} → hit={int(c['place_hit'])} cont={int(c['place_cont'])} stay={int(c['diag_rec_stay'])} | ждёт RECOG_CONFIRM={int(c['place_tentative'])} "
          f"ambig={int(c['place_ambig'])} uncertain={int(c['place_uncertain'])}")
    print("[3] дубликаты: новый узел создан, хотя похожий уже был у агента:")
    print(f"    new={new} dup_tight={int(c['diag_new_dup_tight'])} dup_coarse={int(c['diag_new_dup_coarse'])} | из dup_tight: pi>=scale={int(c['diag_dup_pi_far'])} "
          f"conf<thr={int(c['diag_dup_conf_low'])} err>MAX={int(c['diag_dup_err_high'])} прочее={int(c['diag_dup_other'])}")
    print("[4] визиты/узел:")
    print(f"    узлов={D['nodes']} агентов={D['agents']} гист {D['visits_hist_edges']}: {D['visits_hist']} доля ровно 3: {D['visits_eq3_frac']:.2f}")
    print(f"    без повторного узнавания: {D['nodes_never_rehit_frac']:.2f} hits/node={D['nodes_hits_mean']:.3f} cont/node={D['nodes_cont_mean']:.3f}")
    print(f"    возраст узла {D['node_life_hist_edges']}: {D['node_life_hist']} | возраст агента {D['agent_age_hist_edges']}: {D['agent_age_hist']}")
    print(f"    узлов/агент {D['nodes_per_agent_hist_edges']}: {D['nodes_per_agent_hist']}")
    _pn_ = int(c['place_pred_n'])
    print("[5] v26: защита от алиасинга / адаптация к дрейфу / предсказание следующего места:")
    print(f"    alias_veto={int(c['place_alias_veto'])} drift_adapt={int(c['place_drift_adapt'])} | "
          f"pred hit={int(c['place_pred_hit'])} miss={int(c['place_pred_miss'])} none={int(c['place_pred_none'])} "
          f"hit_rate={(int(c['place_pred_hit']) / _pn_) if _pn_ else 0.0:.2f} fp_surprise_mean={(int(c['place_pred_fp_milli']) / 1000.0 / _pn_) if _pn_ else 0.0:.3f}")
    print(f"    [PRED-CREATE] attempts={int(c['place_pred_create_attempt'])} created={int(c['place_pred_created'])} "
          f"no_place={int(c['place_pred_create_none_place'])} no_edges={int(c['place_pred_create_no_edges'])} "
          f"below_min={int(c['place_pred_create_below_min'])} missing_target={int(c['place_pred_create_missing_target'])}")
    print(f"    [PRED-NONE-REASON] no_prediction={int(c['place_pred_none_no_prediction'])} "
          f"source_mismatch={int(c['place_pred_none_source_mismatch'])} expired={int(c['place_pred_none_expired'])}")
    print("=" * 78, flush=True)
    return D


_PN2_ATTR = ('self_motion', 'world_change', 'place_unknown', 'place_shifted', 'unexplained')
_PN2_ALL = ('self_consistency',) + _PN2_ATTR
_PN2_STANCES = ('investigating', 'accommodating', 'defensive', 'denying')
_PN2_QTYPES = ('place_unknown', 'place_shifted', 'unexplained')
_PN2_SRC = ('pn_defensive', 'pn_investigating')
_PN2_OKISH = ('self_consistency', 'self_motion', 'world_change')


def _pn2_st(n):
    p2 = n.get('pn2')
    if p2 is None:
        C = Config
        p2 = {'exp': deque(maxlen=int(C.PN_V2_EXP_MAX)), 'q': [], 'q_closed': deque(maxlen=int(C.PN_V2_Q_CLOSED_KEEP)),
              'story': deque(maxlen=int(C.PN_V2_STORY_MAX)), 'fail': {}, 'q_seq': 0, 'ep': None, 'obs': 0,
              'last_attr': 'none', 'last_conf': 0.0, 'last_disc': 0.0, 'last_disc_t': 0, 'last_insuff_t': 0,
              'stance': 'none', 'stance_c': 'none', 'stance_n': 0, 'stance_t': -1, 'P': 0.0, 'load': 0.0,
              'want': None, 'want_t': -9999, 'nowhere': 0}
        n['pn2'] = p2
    return p2


def _pn2_e_fp(n, ref):
    if ref is None:
        return None
    for cid, e, _c in (n.get('place_candidates') or ()):
        if cid == ref:
            return float(e)
    return None


def _pn2_at_place(n, ref):
    if ref is None:
        return None
    node = _pn_by_id(n, ref)
    if node is None:
        return None
    g, sc = _pn_gate_scale(float(n['path']))
    d = _pn_toroidal_dist(n['pi'], node['pi_mean'])
    if d <= g:
        return 1.0
    return max(0.0, 1.0 - (d - g) / max(1e-6, float(sc) - float(g)))


def _pn2_attribute(state, conf, pi_err, err, m, e_fp, at=None):
    C = Config
    m = max(0.0, min(1.0, float(m)))
    e_pi = min(1.0, max(0.0, float(pi_err)) / max(1e-6, float(C.NAV_ANCHOR_ERR_FULL)))
    e_fpn = None if e_fp is None else min(1.0, max(0.0, float(e_fp)) / max(1e-6, float(C.NAV_PLACE_MAX_ERR)))
    fp_ok = 0.5 if e_fpn is None else 1.0 - e_fpn
    conf = max(0.0, min(1.0, float(conf)))
    hyp = state in ('recognized', 'uncertain', 'tentative', 'ambiguous') or (state == 'pending' and conf > 0.0)
    raw = dict.fromkeys(_PN2_ATTR, 0.0)
    if state == 'recognized':
        raw['self_motion'] = e_pi * fp_ok
    if state in ('recognized', 'uncertain'):
        raw['world_change'] = m * (1.0 - e_pi) * fp_ok
    atp = None if at is None else max(0.0, min(1.0, float(at)))
    near = (state == 'pending' and not hyp and e_fp is not None and atp is not None and atp >= 0.5
            and float(e_fp) <= 2.0 * float(C.NAV_PLACE_MAX_ERR))
    if state == 'new':
        raw['place_unknown'] = 1.0
    elif state == 'pending' and not hyp and not near:
        raw['place_unknown'] = 0.8
    if state in ('uncertain', 'tentative') or (state == 'pending' and hyp) or near:
        sh = 0.5 * (0.5 if e_fpn is None else e_fpn) + 0.5 * (1.0 - conf)
        raw['place_shifted'] = min(1.0, sh) * (1.0 if atp is None else atp)
        if atp is not None and atp < 1.0:
            raw['place_unknown'] = max(raw['place_unknown'], 0.8 * (1.0 - atp))
    if state == 'ambiguous':
        raw['unexplained'] = 0.5
    disc = max(m, e_pi, max(0.0, float(err)), raw['place_unknown'], raw['place_shifted'], raw['unexplained'])
    base = {'m': round(m, 3), 'at': None if atp is None else round(atp, 3), 'e_pi': round(e_pi, 3), 'e_fp': None if e_fpn is None else round(e_fpn, 3), 'state': state}
    if disc < float(C.PN_V2_DISC_MIN):
        return {'attr': 'self_consistency', 'conf': round(1.0 - disc, 3), 'disc': round(disc, 3), 'p': {}, 'alt': None, 'ev': base}
    u0 = float(C.PN_V2_UNEXPL_PRIOR) * disc
    tot = sum(raw.values()) + u0
    p = {k: raw[k] / tot for k in _PN2_ATTR}
    p['unexplained'] = (raw['unexplained'] + u0) / tot
    cands = [k for k in _PN2_ATTR[:4] if raw[k] >= float(C.PN_V2_EVID_MIN)] + ['unexplained']
    attr = max(cands, key=lambda k: (p[k], -cands.index(k)))
    alt = max((k for k in cands if k != attr), key=lambda k: (p[k], -cands.index(k)), default=None)
    return {'attr': attr, 'conf': round(p[attr], 3), 'disc': round(disc, 3), 'p': {k: round(v, 3) for k, v in p.items()},
            'alt': alt, 'ev': base}


def _pn2_story(p2, now, kind, pid, attr, stance, qid=None, qstate=None, result=None, impact=None):
    st = p2['story']
    if st:
        L = st[-1]
        if L['kind'] == kind and L['place_id'] == pid and L['attr'] == attr and L['q_state'] == qstate and L['result'] == result:
            L['n'] += 1
            L['last_t'] = int(now)
            return
    st.append({'t': int(now), 'last_t': int(now), 'kind': kind, 'place_id': pid, 'attr': attr, 'stance': stance, 'q_id': qid,
               'q_state': qstate, 'result': result, 'impact': impact, 'n': 1, 'prev': (st[-1]['t'] if st else None)})


def _pn2_close(s, p2, q, status, res, now):
    p2['q'] = [x for x in p2['q'] if x is not q]
    q['status'], q['resolution'], q['closed_t'] = status, res, int(now)
    p2['q_closed'].append({k: q[k] for k in ('id', 'type', 'place_id', 'created_t', 'closed_t', 'attempts', 'support', 'counter', 'status', 'resolution', 'irres')})
    s.st['pn2_q_' + status] += 1
    s.st['pn2_q_%s_%s' % (status, q['type'])] += 1
    s.st['pn2_qres_' + str(res)] += 1
    fl = p2['fail'].get((q['place_id'], q['type']))
    if fl is not None and status == 'resolved' and res == 'cleared':
        fl['cleared'] += 1
    _ep = p2.get('ep')
    if _ep is not None and status == 'resolved' and _ep['place_id'] == q['place_id']:
        _ep['resolved'] = True
    if q['type'] == 'place_unknown' and status == 'resolved':
        return
    kind = 'model_revised' if (status == 'resolved' and q['type'] == 'place_shifted' and res == 'cleared') else status
    impact = {'resolved': 'explanation_confirmed', 'model_revised': 'place_model_updated', 'dissolved': 'dropped_unsolved', 'evicted': 'dropped_budget'}.get(kind)
    _pn2_story(p2, now, kind, q['place_id'], q['attr_last'], p2['stance'], q['id'], status, res, impact)


def _pn2_q_step(s, p2, aid, att, state, pid, now, ep, a):
    C = Config
    for q in list(p2['q']):
        if q['type'] == 'place_unknown':
            if state in ('recognized', 'new'):
                _pn2_close(s, p2, q, 'resolved', 'place_found' if state == 'recognized' else 'learned_place', now)
            elif att == 'place_unknown':
                q['attempts'] += 1; q['support'] += 1; q['last_seen_t'] = int(now); q['conf'] = max(q['conf'], a['conf'])
            continue
        if q['place_id'] != pid:
            continue
        q['last_seen_t'] = int(now)
        q['attempts'] += 1
        if att == q['type']:
            q['support'] += 1
            q['conf'] = max(q['conf'], a['conf'])
            if (q['support'] >= int(C.PN_V2_Q_IRRES_SUPPORT) and q['counter'] == 0 and not q['irres']
                    and sum(1 for x in p2['q'] if x['irres']) < int(C.PN_V2_Q_IRRES_MAX)):
                q['irres'] = True
                s.st['pn2_q_irres'] += 1
                _pn2_story(p2, now, 'irresolvable', q['place_id'], q['attr_last'], p2['stance'], q['id'], 'open', 'kept', 'core_question')
        elif q['type'] == 'unexplained' and att in ('place_shifted', 'self_motion', 'world_change'):
            _pn2_close(s, p2, q, 'resolved', 'explained_as_' + att, now)
        elif att in _PN2_OKISH and state == 'recognized':
            q['counter'] += 1
            if q['counter'] >= int(C.PN_V2_Q_RESOLVE_CONFIRM):
                _pn2_close(s, p2, q, 'resolved', 'cleared', now)
        elif att != 'self_consistency' and att != q['attr_last']:
            _pn2_story(p2, now, 'explanation_changed', q['place_id'], att, p2['stance'], q['id'], 'open', q['attr_last'] + '->' + att, 'explanation_revised')
            q['attr_last'] = att
    for q in list(p2['q']):
        if not q['irres'] and now - q['last_seen_t'] > int(C.PN_V2_Q_TTL):
            _pn2_close(s, p2, q, 'dissolved', 'ttl' if (q['place_id'] is None or q['place_id'] == pid) else 'ttl_away', now)
    if att in _PN2_QTYPES and state != 'new' and ep is not None and ep['obs'] >= int(C.PN_V2_Q_MIN_OBS):
        qpid = None if att == 'place_unknown' else pid
        if not any(q['type'] == att and q['place_id'] == qpid for q in p2['q']):
            if len(p2['q']) >= int(C.PN_V2_Q_BUDGET):
                pool = [q for q in p2['q'] if not q['irres']]
                if not pool:
                    s.st['pn2_q_skipped_budget'] += 1
                    return
                victim = min(pool, key=lambda q: (q['last_seen_t'], q['support'], q['id']))
                _pn2_close(s, p2, victim, 'evicted', 'budget', now)
            p2['q_seq'] += 1
            qid = _stable_hash((int(aid), qpid, att, int(now), int(p2['q_seq']))) % 100000000
            q = {'id': qid, 'type': att, 'place_id': qpid, 'created_t': int(now), 'last_seen_t': int(now), 'attempts': 1, 'support': 1,
                 'counter': 0, 'conf': float(a['conf']), 'attr_last': att, 'irres': False, 'status': 'open', 'resolution': None}
            p2['q'].append(q)
            s.st['pn2_q_new'] += 1
            s.st['pn2_qt_' + att] += 1
            if att != 'place_unknown':
                _pn2_story(p2, now, 'opened', qpid, att, p2['stance'], qid, 'open', None, 'unexplained_discrepancy')


def _pn2_stance(n, p2, att, a, energy, now):
    C = Config
    qs = p2['q']
    load = min(1.0, len(qs) / float(max(1, int(C.PN_V2_Q_BUDGET))))
    P = max(0.0, min(1.0, 0.6 * load + 0.4 * float(a['disc'])))
    D = float(n['D']) if now - int(n['D_t']) <= int(C.NAV_SENSE_EVERY) + 1 else 0.0
    risk = min(1.0, D / 0.2)
    energy = max(0.0, min(1.0, float(energy)))
    voi = max((q['conf'] * (1.0 - 0.5 * min(1.0, q['attempts'] / (2.0 * float(C.PN_V2_Q_IRRES_SUPPORT)))) for q in qs), default=0.0)
    cost = 0.15 + 0.35 * (1.0 - energy) + 0.5 * risk
    if not qs and att == 'self_consistency':
        cand = 'none'
    elif P >= float(C.PN_V2_ST_DENY) and load >= 1.0:
        cand = 'denying'
    elif P >= float(C.PN_V2_ST_DEFEND) or risk >= float(C.PN_V2_RISK_DEFEND):
        cand = 'defensive'
    elif P >= float(C.PN_V2_ST_ACCOM) or energy < float(C.PN_V2_ENERGY_MIN) or voi < float(C.PN_V2_VOI_MIN) * cost:
        cand = 'accommodating'
    else:
        cand = 'investigating'
    if cand == 'none':
        p2['stance_c'], p2['stance_n'] = 'none', 0
        sw = p2['stance'] != 'none'
        p2['stance'] = 'none'
    else:
        if cand == p2['stance_c']:
            p2['stance_n'] += 1
        else:
            p2['stance_c'], p2['stance_n'] = cand, 1
        sw = False
        if cand != p2['stance'] and p2['stance_n'] >= int(C.PN_V2_STANCE_HOLD):
            p2['stance'] = cand
            sw = True
    if sw:
        p2['stance_t'] = int(now)
    p2['P'], p2['load'] = round(P, 3), round(load, 3)
    want = None
    if p2['stance'] == 'investigating' and P >= float(C.PN_V2_ST_INVEST):
        want = 'explore'
    elif p2['stance'] == 'defensive':
        want = 'rest'
    p2['want'], p2['want_t'] = want, int(now)
    return sw


def _pn2_snap(p):
    n = getattr(p, '_nav', None)
    p2 = n.get('pn2') if n else None
    if not p2:
        return None
    return {'attr': p2['last_attr'], 'stance': p2['stance'], 'open_q': len(p2['q']), 'P': p2['P'], 'irres': int(any(q['irres'] for q in p2['q'])), 'place_id': (int(n['current_place']['id']) if n.get('current_place') else None)}


def _pn2_hist(eng):
    st, aa, oq = Counter(), Counter(), 0
    for p in eng.patterns:
        if p.alive and p.role_type != 'feral':
            n = getattr(p, '_nav', None)
            p2 = n.get('pn2') if n else None
            if p2:
                st[p2['stance']] += 1
                aa[p2['last_attr']] += 1
                oq += len(p2['q'])
    return {'stance': dict(st), 'attr_last': dict(aa), 'open_q': int(oq)}


def _pn2_summary(eng):
    C = Config
    if not bool(C.PN_V2_ENABLED):
        return {'enabled': False}
    alive = [p for p in eng.patterns if p.alive and p.role_type != 'feral' and getattr(p, '_nav', None) and p._nav.get('pn2')]
    cur_attr, cur_st, open_t, stable = Counter(), Counter(), Counter(), Counter()
    n_open = core = eps = story_n = ok = fail = 0
    loads, Ps = [], []
    for p in alive:
        p2 = p._nav['pn2']
        cur_attr[p2['last_attr']] += 1
        cur_st[p2['stance']] += 1
        n_open += len(p2['q'])
        core += 1 if any(q['irres'] for q in p2['q']) else 0
        loads.append(float(p2['load']))
        Ps.append(float(p2['P']))
        for q in p2['q']:
            open_t[q['type']] += 1
        eps += len(p2['exp'])
        story_n += len(p2.get('story') or ())
        for k, fl in p2['fail'].items():
            if fl['stable']:
                stable[k[1]] += 1
        for pl in p._nav['places']:
            na = pl.get('pn2')
            if na:
                ok += int(na['ok'])
                fail += int(na['fail'])
    top = sorted(alive, key=lambda p: (-len(p._nav['pn2']['story']), p.id))[:5]
    sample = [{'agent': int(p.id), 'story': [dict(e) for e in list(p._nav['pn2'].get('story') or [])[-6:]]} for p in top if p._nav['pn2'].get('story')]
    return {'enabled': True, 'apply_stance': bool(C.PN_V2_APPLY_STANCE), 'agents': len(alive),
            'attr_last': dict(cur_attr), 'stance': dict(cur_st), 'questions_open': int(n_open), 'open_by_type': dict(open_t),
            'core_frac': float(core) / max(1, len(alive)), 'load_mean': float(np.mean(loads)) if loads else 0.0, 'P_mean': float(np.mean(Ps)) if Ps else 0.0,
            'episodes_kept': int(eps), 'story_kept': int(story_n), 'stable_failures': dict(stable),
            'node_ok': ok, 'node_fail': fail, 'node_fail_rate': float(fail) / max(1, ok + fail), 'story_sample': sample}


def _pn2_summary_safe(eng):
    try:
        return _pn2_summary(eng)
    except Exception as e:
        return {'enabled': bool(Config.PN_V2_ENABLED), 'error': repr(e)}


def _pn2_line(c):
    if not bool(Config.PN_V2_ENABLED):
        return ""
    g = lambda k: int(c.get(k, 0))
    return (f"           PN2 obs={g('pn2_obs')} attr cons/selfm/world/unk/shift/unexpl={g('pn2_a_self_consistency')}/{g('pn2_a_self_motion')}/{g('pn2_a_world_change')}/"
            f"{g('pn2_a_place_unknown')}/{g('pn2_a_place_shifted')}/{g('pn2_a_unexplained')} | ep={g('pn2_ep_new')} | "
            f"q new/res/dis/evict/irres={g('pn2_q_new')}/{g('pn2_q_resolved')}/{g('pn2_q_dissolved')}/{g('pn2_q_evicted')}/{g('pn2_q_irres')} | "
            f"stance inv/acc/def/den/none={g('pn2_st_investigating')}/{g('pn2_st_accommodating')}/{g('pn2_st_defensive')}/{g('pn2_st_denying')}/{g('pn2_st_none')} | "
            f"goal rest/expl/drop={g('pn2_goal_rest')}/{g('pn2_goal_explore')}/{g('pn2_goal_drop')}\n")


def _pn2_print(Q, c, H=None):
    if not Q or not Q.get('enabled'):
        print("\n  ── [O5] PLACENODE v2: выключен (Config.PN_V2_ENABLED=False) ──")
        return
    print(f"\n  ── [O5] PLACENODE v2 · АТРИБУЦИЯ (накопительно по замерам; apply_stance={Q.get('apply_stance')}) ──")
    obs = max(1, int(c.get('pn2_obs', 0)))
    disc = max(1, obs - int(c.get('pn2_a_self_consistency', 0)))
    for k in _PN2_ALL:
        v = int(c.get('pn2_a_' + k, 0))
        print(f"  {k:17s} {v:8d}  {100.0 * v / obs:5.1f}% замеров" + ("" if k == 'self_consistency' else f"  {100.0 * v / disc:5.1f}% расхождений"))
    print(f"  замеров={obs}  пропущено(нет вида)={c.get('pn2_skip', 0)}  эпизодов новых={c.get('pn2_ep_new', 0)}  (unexplained ≥ 80% среди расхождений = классификатор слепой, чинить его до stance)")
    print("\n  ── [O6] ВОПРОСЫ / ЯДРО ──")
    print(f"  создано={c.get('pn2_q_new', 0)} разрешено={c.get('pn2_q_resolved', 0)} растворено={c.get('pn2_q_dissolved', 0)} вытеснено={c.get('pn2_q_evicted', 0)} "
          f"не создано(бюджет ядра)={c.get('pn2_q_skipped_budget', 0)} стали неразрешимыми={c.get('pn2_q_irres', 0)}")
    done = max(1, c.get('pn2_q_resolved', 0) + c.get('pn2_q_dissolved', 0) + c.get('pn2_q_evicted', 0))
    print(f"  вопросы по типам (создано): { {t: int(c.get('pn2_qt_' + t, 0)) for t in _PN2_QTYPES} }; причины закрытия: { {k[9:]: int(v) for k, v in sorted(c.items()) if k.startswith('pn2_qres_')} }")
    print(f"  place_unknown: кандидаты были (все отвергнуты) / кандидатов не было = {c.get('pn2_unk_cand', 0)} / {c.get('pn2_unk_nocand', 0)}; ttl_away = агент ушёл от места и не вернулся")
    print(f"  доля закрытых: разрешено {100.0 * c.get('pn2_q_resolved', 0) / done:.0f}% / растворено {100.0 * c.get('pn2_q_dissolved', 0) / done:.0f}% / вытеснено {100.0 * c.get('pn2_q_evicted', 0) / done:.0f}%")
    print(f"  сейчас открыто={Q['questions_open']} (на агента {Q['questions_open'] / max(1, Q['agents']):.2f}) по типам={Q['open_by_type']}; агентов с неразрешимым ядром={100.0 * Q['core_frac']:.0f}%; "
          f"нагрузка на неопределённость (среднее)={100.0 * Q['load_mean']:.0f}%")
    print(f"  «слишком тихо» (insufficient_evidence)={c.get('pn2_insuff', 0)}; устойчивые провалы (агенто-мест) по причине={Q['stable_failures']}")
    print(f"  ошибка узнавания по агрегатам узлов: ok={Q['node_ok']} fail={Q['node_fail']} ({100.0 * Q['node_fail_rate']:.1f}% fail)")
    print("\n  ── [O7] STANCE ──")
    tot = max(1, sum(int(c.get('pn2_st_' + k, 0)) for k in _PN2_STANCES + ('none',)))
    for k in _PN2_STANCES + ('none',):
        v = int(c.get('pn2_st_' + k, 0))
        print(f"  {k:14s} оценок={v:8d} ({100.0 * v / tot:5.1f}%)   сейчас у агентов: {Q['stance'].get(k, 0)}")
    print(f"  желание цели (оценок): explore={c.get('pn2_want_explore', 0)} rest={c.get('pn2_want_rest', 0)}; не добавлено, т.к. такая цель уже есть: explore={c.get('pn2_goal_have_explore', 0)} rest={c.get('pn2_goal_have_rest', 0)}")
    print(f"  смен позиции={c.get('pn2_stance_switch', 0)}; цели от stance: rest +{c.get('pn2_goal_rest', 0)} explore +{c.get('pn2_goal_explore', 0)} снято={c.get('pn2_goal_drop', 0)}; давление P среднее={Q['P_mean']:.3f}")
    print("\n  ── [O8] SELF-NARRATIVE (сюжет: сжатая история пересмотров; выборка агентов с самым длинным сюжетом) ──")
    print(f"  записей в памяти (всего по агентам)={Q['story_kept']}, эпизодов опыта={Q['episodes_kept']}")
    for it in Q.get('story_sample', []):
        print(f"  агент #{it['agent']}:")
        for e in it['story']:
            print(f"    t={e['t']:>5}..{e['last_t']:<5} {e['kind']:<20s} место={e['place_id']} причина={e['attr']} stance={e['stance']} вопрос={e['q_state']}/{e['result']} impact={e['impact']} ×{e['n']}")
    if H:
        print("\n  динамика PN2 (каждые 50 тиков):   t   open_q  stance")
        for h in H:
            if h.get('pn2'):
                print(f"  {h['t']:30d} {h['pn2']['open_q']:8d}  {h['pn2']['stance']}")


def _pn2_report(eng=None):
    eng = eng or _PN_LAST.get('eng')
    if eng is None:
        print("[PN2] нет движка (прогон ещё не стартовал)")
        return None
    s = _fch(eng)
    Q = _pn2_summary(eng)
    print("=" * 78)
    print("[PN2] PLACENODE v2 — живой отчёт (read-only)")
    print("=" * 78)
    _pn2_print(Q, dict(s.st), s.hist)
    return Q


def _pn2_selftest():
    C = Config
    cases = [
        ('агент сбился в счёте пути, вид совпал', dict(state='recognized', conf=0.8, pi_err=4.5, err=0.5, m=0.0, e_fp=0.05), 'self_motion'),
        ('мир изменился, место и путь в порядке', dict(state='recognized', conf=0.8, pi_err=0.5, err=0.1, m=0.6, e_fp=0.05), 'world_change'),
        ('впервые в новом месте', dict(state='new', conf=0.4, pi_err=0.0, err=0.0, m=0.0, e_fp=None), 'place_unknown'),
        ('новизна без гипотезы места', dict(state='pending', conf=0.0, pi_err=0.0, err=0.0, m=0.0, e_fp=None), 'place_unknown'),
        ('место знакомо, но выглядит иначе', dict(state='uncertain', conf=0.35, pi_err=0.0, err=0.0, m=0.0, e_fp=0.35), 'place_shifted'),
        ('ушёл от места: вид иной, счёт пути далеко', dict(state='uncertain', conf=0.35, pi_err=0.0, err=0.0, m=0.0, e_fp=0.35, at=0.0), 'place_unknown'),
        ('рядом с местом, вид чуть вне порога', dict(state='pending', conf=0.0, pi_err=0.0, err=0.0, m=0.0, e_fp=0.6, at=0.9), 'place_shifted'),
        ('два места подходят одинаково', dict(state='ambiguous', conf=0.4, pi_err=0.0, err=0.0, m=0.0, e_fp=0.1), 'unexplained'),
        ('слабое расхождение, причина неочевидна', dict(state='recognized', conf=0.8, pi_err=0.0, err=0.0, m=0.2, e_fp=0.05), 'unexplained'),
        ('всё совпало', dict(state='recognized', conf=0.9, pi_err=0.2, err=0.02, m=0.02, e_fp=0.02), 'self_consistency'),
    ]
    ok = 0
    print("[PN2 selftest] атрибуция на сценариях с известной причиной:")
    for name, kw, exp in cases:
        r = _pn2_attribute(**kw)
        good = r['attr'] == exp
        ok += int(good)
        print(f"  {'OK ' if good else 'FAIL'} {name:42s} → {r['attr']:16s} conf={r['conf']:.2f} (ожидалось {exp}) alt={r['alt']}")
    class _S:
        st = Counter()
    s, p2, a = _S(), {'q': [], 'q_closed': deque(maxlen=30), 'story': deque(maxlen=40), 'fail': {}, 'q_seq': 0, 'stance': 'none'}, {'conf': 0.6, 'disc': 0.5}
    ep = {'obs': 2}
    for i in range(3):
        _pn2_q_step(s, p2, 1, 'place_shifted', 'uncertain', 7, 100 + 2 * i, ep, a)
    life = [len(p2['q']) == 1, p2['q'][0]['support'] == 3]
    _pn2_q_step(s, p2, 1, 'self_consistency', 'recognized', 3, 110, None, a)
    life.append(len(p2['q']) == 1)
    for i in range(2):
        _pn2_q_step(s, p2, 1, 'self_consistency', 'recognized', 7, 112 + 2 * i, None, a)
    life.append(len(p2['q']) == 0 and s.st['pn2_q_resolved'] == 1)
    for i in range(7):
        _pn2_q_step(s, p2, 1, 'unexplained', 'ambiguous', 9, 200 + 2 * i, ep, a)
    life.append(any(q['irres'] for q in p2['q']))
    _pn2_q_step(s, p2, 1, 'self_consistency', 'tentative', 4, 200 + 10 * int(C.PN_V2_Q_TTL), None, a)
    life.append(any(q['irres'] for q in p2['q']))
    good = all(life)
    ok += int(good)
    print(f"  {'OK ' if good else 'FAIL'} жизненный цикл вопроса (создан→не закрыт чужим местом→разрешён; ядро переживает TTL): {life}")
    print(f"[PN2 selftest] {ok}/{len(cases) + 1}")
    return ok == len(cases) + 1


from dataclasses import dataclass, field as _dc_field

_PN3_CFG = {
    'PN3_ENABLED': True, 'PN3_PROPOSALS': True, 'PN3_PLACE_MSG': True, 'PN3_FEEDBACK': True,
    'PN3_THRESH': {'stance': (0.10, 0.15), 'event': (0.30, 0.35)},
    'PN3_PROP_TTL': 10, 'PN3_PROP_KEEP': 24, 'PN3_PRIO_GAIN': 0.3,
    'PN3_SHIFT_CONFIRM': 2, 'PN3_UNK_REPEAT': 3,
    'PN3_EVENT_KEEP': 60, 'PN3_EVENT_TTL': 80,
    'PN3_MSG_KINDS': ('shifted',), 'PN3_MSG_CONF_MIN': 0.5, 'PN3_MSG_COOLDOWN': 20, 'PN3_MSG_WINDOW': 30,
    'PN3_MSG_TTL': 40, 'PN3_MSG_MAX_RESP': 3,
    'PN3_CONF_STEP': 0.15, 'PN3_STRONG': 1.0, 'PN3_WEAK': 0.5,
}
for _k, _v in _PN3_CFG.items():
    if not hasattr(Config, _k):
        setattr(Config, _k, _v)

_PN3_PTYPES = ('explore', 're-examine', 'check', 'adjust', 'ignore')
_PN3_KINDS = ('shifted', 'unknown', 'unexplained')
_PN3_OUTCOMES = ('confirmed', 'rejected', 'unknown', 'expired')
_PN3_KIND = {'place_shifted': 'shifted', 'place_unknown': 'unknown', 'unexplained': 'unexplained'}
_PN3_GOAL = {'explore': 'explore', 're-examine': 'explore', 'adjust': 'rest'}
_PN3_OWN_SRC = ('pn_investigating', 'pn_defensive', 'pn_proposal')
_PN3_FINAL = ('expired', 'ignored_low_urgency', 'ignored_low_conf', 'no_goal', 'merged', 'switched', 'lapsed', 'vetoed')


@dataclass
class ActionProposal:
    type: str
    urgency: float
    confidence: float
    source: object
    ttl: int
    consumed: bool = False
    t0: int = 0
    cls: str = 'event'
    state: str = 'pending'
    reason: str = ''
    goal_ref: object = None

    def __post_init__(self):
        if self.type not in _PN3_PTYPES:
            raise ValueError(f"ActionProposal.type={self.type!r}")
        for nm in ('urgency', 'confidence'):
            v = getattr(self, nm)
            if not isinstance(v, (int, float)) or v != v or not (0.0 <= float(v) <= 1.0):
                raise ValueError(f"ActionProposal.{nm}={v!r} вне [0,1]")
            setattr(self, nm, float(v))
        if int(self.ttl) < 0:
            raise ValueError(f"ActionProposal.ttl={self.ttl!r} < 0")
        self.ttl = int(self.ttl)


@dataclass
class PlaceEvent:
    kind: str
    place_id: object
    confidence: float
    t: int
    pi_at_time: tuple
    sent: bool = False
    feedback_received: list = _dc_field(default_factory=list)
    outcome: str = 'unknown'
    event_id: int = 0
    conf0: float = 0.0
    fp_obs: tuple = ()
    fp_ref: object = None
    world_xy: tuple = ()
    msg_id: object = None

    def __post_init__(self):
        if self.kind not in _PN3_KINDS:
            raise ValueError(f"PlaceEvent.kind={self.kind!r}")
        if self.outcome not in _PN3_OUTCOMES:
            raise ValueError(f"PlaceEvent.outcome={self.outcome!r}")
        if not (0.0 <= float(self.confidence) <= 1.0):
            raise ValueError(f"PlaceEvent.confidence={self.confidence!r}")


def _pn3_init(p2):
    if 'events' not in p2:
        p2['events'] = deque(maxlen=int(Config.PN3_EVENT_KEEP))
    p2.setdefault('props', [])
    if 'props_done' not in p2:
        p2['props_done'] = deque(maxlen=int(Config.PN3_PROP_KEEP))
    p2.setdefault('prop_last', {})
    p2.setdefault('ev_seq', 0)
    p2.setdefault('msg_last_t', -9999)
    p2.setdefault('msgs', [])
    return p2


def _pn3_tup(v):
    return None if v is None else tuple(round(float(x), 4) for x in v)


def _pn3_prop_new(agent, s, p2, ptype, urgency, conf, source, ttl, now, cls='event'):
    _pn3_init(p2)
    key = (cls, source, ptype)
    last = p2['prop_last'].get(key)
    if last is not None and now - last <= int(ttl):
        s.st['pn3_proposals_deduplicated'] += 1
        return None
    if any(p.state in ('pending', 'watching') and (p.cls, p.source, p.type) == key for p in p2['props']):
        s.st['pn3_proposals_deduplicated'] += 1
        return None
    try:
        pr = ActionProposal(type=ptype, urgency=float(urgency), confidence=float(conf), source=source, ttl=int(ttl), t0=int(now), cls=cls)
    except (ValueError, TypeError):
        s.st['pn3_proposals_invalid'] += 1
        return None
    p2['prop_last'][key] = int(now)
    if len(p2['prop_last']) > 64:
        for k in sorted(p2['prop_last'], key=lambda k: p2['prop_last'][k])[:16]:
            p2['prop_last'].pop(k, None)
    p2['props'].append(pr)
    s.st['pn3_proposals_made'] += 1
    s.st['pn3_prop_made_' + ptype.replace('-', '_')] += 1
    return pr


def _pn3_generate(agent, s, n, p2, now):
    C = Config
    S0 = max(1.0, float(C.PN_V2_Q_IRRES_SUPPORT))
    for q in p2['q']:
        qt = q['type']
        ev = next((e for e in reversed(p2['events']) if _PN3_KIND.get(qt) == e.kind and (e.place_id == q['place_id'] or qt == 'place_unknown')), None)
        c_ev = float(ev.confidence) if ev is not None else float(q['conf'])
        conf = max(0.0, min(1.0, c_ev / (1.0 + float(q['counter']))))
        if ev is not None and ev.outcome == 'rejected':
            conf *= 0.5
        base = 1.0 - math.exp(-float(q['support']) / S0)
        if qt == 'place_shifted':
            ok = q['support'] >= int(C.PN3_SHIFT_CONFIRM) or (ev is not None and ev.outcome == 'confirmed')
            ptype, u = ('re-examine', base) if ok else ('check', 0.6 * base)
        elif qt == 'place_unknown':
            if q['support'] < int(C.PN3_UNK_REPEAT):
                continue
            ptype, u = 'explore', base
        else:
            ptype, u = ('check', 0.5 * base) if q['support'] < int(C.PN3_UNK_REPEAT) else ('explore', min(0.5, 0.5 * base))
        _pn3_prop_new(agent, s, p2, ptype, u, conf, ('q', int(q['id'])), int(C.PN3_PROP_TTL), now, 'event')


def _pn3_stance_proposal(agent, s, p2, want, now):
    ptype = 'explore' if want == 'explore' else 'adjust'
    return _pn3_prop_new(agent, s, p2, ptype, max(0.0, min(1.0, float(p2['P']))), max(0.0, min(1.0, float(p2['last_conf']))),
                         ('stance', want), int(Config.PN_V2_STANCE_PERSIST), now, 'stance')


def _pn3_fin(s, p2, pr, state, reason=''):
    pr.state, pr.reason, pr.consumed = state, reason, True
    s.st['pn3_fin_' + state] += 1
    if isinstance(pr.source, tuple) and pr.source and pr.source[0] == 'soma':   # observer: судьба предложений от тела
        s.st['soma_prop_fin_' + state] += 1
        if reason:
            s.st['soma_prop_fin_' + state + '_' + str(reason)] += 1
    p2['props'] = [x for x in p2['props'] if x is not pr]
    p2['props_done'].append((pr.type, pr.cls, state, reason, pr.t0))


def _pn3_goal_from_proposal(agent, s, pr, now):
    C = Config
    gt = _PN3_GOAL[pr.type]
    if pr.cls == 'stance':
        if gt == 'rest':
            g = {"type": "rest", "priority": float(C.PN_V2_PRIO_REST), "target": None, "age": 0, "persistence": int(C.PN_V2_STANCE_PERSIST), "_source": "pn_defensive"}
            s.st['pn2_goal_rest'] += 1
        else:
            g = {"type": "explore", "priority": float(C.PN_V2_PRIO_EXPLORE), "target": None, "age": 0, "persistence": int(C.PN_V2_STANCE_PERSIST), "_source": "pn_investigating"}
            s.st['pn2_goal_explore'] += 1
    else:
        pr_ = float(C.PN_V2_PRIO_EXPLORE) + float(C.PN3_PRIO_GAIN) * pr.urgency * pr.confidence
        g = {"type": gt, "priority": pr_, "target": None, "age": 0, "persistence": max(int(pr.ttl), int(C.PN_V2_STANCE_PERSIST)), "_source": "pn_proposal",
             "_proposal": pr.type}
    agent.goals.append(g)
    return g


def _pn3_handle(agent, s, p2, now):
    C = Config
    _pn3_init(p2)
    for pr in list(p2['props']):
        if pr.state != 'watching':
            continue
        g = pr.goal_ref
        if agent.intent is g:
            s.st['pn3_intent_switches_after_proposal'] += 1
            _pn3_fin(s, p2, pr, 'switched', 'intent_is_goal')
        elif not any(x is g for x in agent.goals):
            cd = (int(agent.age) - int(getattr(agent, 'last_intent_switch_age', -10**6))) < int(C.INTENT_SWITCH_COOLDOWN)
            why = 'other_intent_cooldown' if (agent.intent is not None and cd) else ('other_intent_barrier' if agent.intent is not None else 'no_intent')
            s.st['pn3_noswitch_' + why] += 1
            _pn3_fin(s, p2, pr, 'lapsed', why)
    for pr in list(p2['props']):
        if pr.state != 'pending':
            continue
        if now - pr.t0 > pr.ttl:
            s.st['pn3_proposals_expired'] += 1
            _pn3_fin(s, p2, pr, 'expired', 'ttl')
            continue
        tu, tc = C.PN3_THRESH[pr.cls]
        pr.consumed = True
        if pr.urgency < float(tu):
            s.st['pn3_proposals_ignored_low_urgency'] += 1
            _pn3_fin(s, p2, pr, 'ignored_low_urgency', '')
            continue
        if pr.confidence < float(tc):
            s.st['pn3_proposals_ignored_low_conf'] += 1
            _pn3_fin(s, p2, pr, 'ignored_low_conf', '')
            continue
        gt = _PN3_GOAL.get(pr.type)
        if gt is None:
            s.st['pn3_prop_no_goal_' + pr.type] += 1
            _pn3_fin(s, p2, pr, 'no_goal', 'type_' + pr.type)
            continue
        have = next((g for g in agent.goals if g.get('type') == gt), None)
        s.st['pn3_proposals_activated'] += 1
        if have is not None:
            if have.get('_source') in _PN3_OWN_SRC:
                have['age'] = 0
            s.st['pn3_goal_exists_' + gt] += 1
            _pn3_fin(s, p2, pr, 'merged', 'goal_exists')
            continue
        pr.goal_ref = _pn3_goal_from_proposal(agent, s, pr, now)
        pr.state = 'watching'


def _pn3_event_new(agent, s, n, p2, ep, rec, a, now, pid, att):
    _pn3_init(p2)
    kind = _PN3_KIND.get(att)
    if kind is None or ep.get('ev_id') is not None:
        return None
    p2['ev_seq'] += 1
    eid = _stable_hash((int(agent.id), kind, pid, int(now), int(p2['ev_seq']))) % 100000000
    node = _pn_by_id(n, pid) if pid is not None else None
    try:
        cx, cy = agent.get_center()
        wxy = (round(float(cx), 2), round(float(cy), 2))
    except Exception:
        wxy = ()
    ev = PlaceEvent(kind=kind, place_id=pid, confidence=max(0.0, min(1.0, float(a['conf']))), t=int(now),
                    pi_at_time=(float(n['pi'][0]), float(n['pi'][1]), int(now)), event_id=int(eid), conf0=float(a['conf']),
                    fp_obs=_pn3_tup(n.get('fp_last')) or (), fp_ref=(_pn3_tup(node['fp_mean']) if node is not None else None), world_xy=wxy)
    if len(p2['events']) == p2['events'].maxlen:
        old = p2['events'][0]
        s.st['pn3_event_evicted'] += 1
        if not old.sent:
            s.st['pn3_event_evicted_unsent'] += 1
    p2['events'].append(ev)
    ep['ev_id'] = int(eid)
    s.st['pn3_event_new'] += 1
    s.st['pn3_event_new_' + kind] += 1
    _pn2_story(p2, now, 'place_event', pid, att, p2['stance'], None, None, None, 'observation')
    L = p2['story'][-1]
    ids = L.setdefault('ev_ids', [])
    if len(ids) < 5:
        ids.append(int(eid))
    return ev


def _pn3_outcome(ev):
    nc = sum(1 for _p, v, _t in ev.feedback_received if v == 'confirmed')
    nr = sum(1 for _p, v, _t in ev.feedback_received if v == 'rejected')
    return 'confirmed' if nc > nr else ('rejected' if nr > nc else 'unknown')


def _pn3_event_expire(agent, s, p2, now):
    for ev in p2['events']:
        if ev.outcome == 'unknown' and now - ev.t > int(Config.PN3_EVENT_TTL):
            ev.outcome = 'expired'
            s.st['pn3_event_expired'] += 1
            s.st['pn3_outcome_expired'] += 1
            if not ev.sent:
                s.st['pn3_event_expired_unsent'] += 1


def _pn3_try_send(agent, s, field, p2, now):
    C = Config
    if now - p2['msg_last_t'] < int(C.PN3_MSG_COOLDOWN) or not hasattr(agent, '_sig_place_send'):
        return
    kinds = tuple(C.PN3_MSG_KINDS)
    cand = next((e for e in p2['events'] if (not e.sent) and e.outcome == 'unknown' and e.kind in kinds
                 and e.confidence >= float(C.PN3_MSG_CONF_MIN) and now - e.t <= int(C.PN3_MSG_WINDOW) and e.fp_obs), None)
    if cand is None:
        return
    if C.ENABLE_CENTER_WALL:
        try:
            sx, sy = agent.get_center()
            tx, ty = cand.world_xy
            if (int(sx) < C.CENTER_WALL_X) != (int(tx) < C.CENTER_WALL_X):
                s.st['pn3_msg_blocked_center_wall'] += 1
                return
        except (TypeError, ValueError, IndexError):
            # If the target side cannot be established, do not invent a crossing.
            pass
    s.st['pn3_msg_attempt'] += 1
    msg, why = agent._sig_place_send(field, now, cand)
    if msg is None:
        s.st['pn3_msg_blocked_' + why] += 1
        return
    cand.sent, cand.msg_id = True, msg.message_id
    p2['msg_last_t'] = int(now)
    p2['msgs'].append(msg.message_id)


def _pn3_fp_err(place, fp):
    C = Config
    sig = np.minimum(np.sqrt(place['fp_var'] + float(C.NAV_PLACE_FP_SIGMA_MIN) ** 2), float(C.NAV_PLACE_FP_SIGMA_MAX))
    z = float(np.mean(np.abs(np.asarray(fp, dtype=np.float64) - place['fp_mean']) / sig))
    z0, z1 = float(C.NAV_PLACE_Z0), float(C.NAV_PLACE_Z_FULL)
    return min(1.0, max(0.0, (z - z0) / max(1e-6, z1 - z0)))


def _pn3_verify(agent, msg):
    C = Config
    if msg.kind != 'shifted':
        return 'unknown', 'not_verifiable_kind', 0.0
    n = getattr(agent, '_nav', None)
    places = n.get('places') if n else None
    if not places or not msg.fp_obs:
        return 'unknown', 'no_memory', 0.0
    tol = float(C.NAV_PLACE_MAX_ERR)
    po = min(((_pn3_fp_err(p, msg.fp_obs), i) for i, p in enumerate(places)), key=lambda x: x[0])
    pr = min(((_pn3_fp_err(p, msg.fp_ref), i) for i, p in enumerate(places)), key=lambda x: x[0]) if msg.fp_ref else None
    m_obs = po[0] <= tol
    m_ref = pr is not None and pr[0] <= tol
    strong, weak = float(C.PN3_STRONG), float(C.PN3_WEAK)
    if m_obs and m_ref:
        return ('rejected', 'same_place_in_my_memory', weak) if po[1] == pr[1] else ('rejected', 'two_distinct_known_places', strong)
    if m_obs:
        return 'rejected', 'view_is_another_known_place', strong
    if m_ref:
        return 'confirmed', 'old_view_known_new_unknown', weak
    return 'unknown', 'nothing_recognised', 0.0


def _pn3_on_heard(agent, bk, ent, state, now):
    C = Config
    if not C.PN3_PLACE_MSG or len(ent) < 10:
        return
    msg = bk.place_msgs.get(ent[8])
    if msg is None or msg.sender_id == agent.id:
        return
    rid = int(agent.id)
    interp = state in ('recognized', 'partial')
    d = msg.delivery.get(rid)
    if d is None:
        msg.delivery[rid] = {'t': int(now), 'interpreted': bool(interp)}
        bk.pm['delivered_pairs'] += 1
        if interp:
            bk.pm['interpreted_pairs'] += 1
        if msg.state_delivery == 'sent':
            msg.state_delivery = 'delivered'
            bk.pm['delivered_msgs'] += 1
    elif interp and not d['interpreted']:
        d['interpreted'] = True
        bk.pm['interpreted_pairs'] += 1
    if not (C.PN3_FEEDBACK and interp) or rid in msg.response or len(msg.response) >= int(C.PN3_MSG_MAX_RESP):
        return
    verdict, why, strength = _pn3_verify(agent, msg)
    msg.response[rid] = {'verdict': verdict, 't': int(now), 'strength': float(strength), 'why': why, 'applied': False}
    msg.state_response = 'answered'
    bk.pm['checked_pairs'] += 1
    bk.pm['verdict_' + verdict] += 1
    if verdict == 'confirmed':
        agent.trust_ledger.update(msg.sender_id, 'helpful', multiplier=float(C.SIG_TRUST_UP) * strength)
        bk.pm['trust_up'] += 1
    elif verdict == 'rejected':
        agent.trust_ledger.update(msg.sender_id, 'harmful', multiplier=float(C.SIG_TRUST_DOWN) * strength)
        bk.pm['trust_down'] += 1


def _pn3_apply_feedback(agent, s, p2, bk, now):
    C = Config
    keep = []
    for mid in p2['msgs']:
        msg = bk.place_msgs.get(mid)
        if msg is None:
            continue
        for rid, r in msg.response.items():
            if r.get('applied'):
                continue
            r['applied'] = True
            ev = next((e for e in p2['events'] if e.event_id == msg.event_id), None)
            if ev is None:
                s.st['pn3_fb_unlinked'] += 1
                continue
            ev.feedback_received.append((int(rid), r['verdict'], int(r['t'])))
            v = r['verdict']
            if v in ('confirmed', 'rejected'):
                step = float(C.PN3_CONF_STEP) * float(r['strength'])
                ev.confidence = ev.confidence + step * (1.0 - ev.confidence) if v == 'confirmed' else ev.confidence * (1.0 - step)
                ev.confidence = max(0.0, min(1.0, ev.confidence))
                s.st['pn3_cr_conf_updates'] += 1
                s.st['pn3_cr_pos' if v == 'confirmed' else 'pn3_cr_neg'] += 1
            else:
                s.st['pn3_cr_unknown'] += 1
            new = _pn3_outcome(ev)
            if ev.outcome != 'expired' and new != ev.outcome:
                if ev.outcome in ('confirmed', 'rejected'):
                    s.st['pn3_outcome_' + ev.outcome] -= 1
                ev.outcome = new
                if new in ('confirmed', 'rejected'):
                    s.st['pn3_outcome_' + new] += 1
                    _pn2_story(p2, now, 'event_' + new, ev.place_id, {'shifted': 'place_shifted', 'unknown': 'place_unknown'}.get(ev.kind, 'unexplained'), p2['stance'], None, None, str(rid), 'peer_' + new)
        if now - msg.t_sent <= int(C.PN3_MSG_TTL) + 2:
            keep.append(mid)
    p2['msgs'] = keep


def _pn3_world_tick(eng, now):
    bk = _sg_book(eng)
    if int(now) % 100 == 0:
        bk.replied_ids = {k for k in bk.replied_ids if k in bk.place_msgs}
    for mid, hc in list(getattr(bk, 'hunt_calls', {}).items()):
        if now - hc.t_sent > int(Config.HUNT_TTL) + 2:
            if not hc.delivery:
                bk.hc['undelivered'] += 1
            bk.hunt_calls.pop(mid, None)
    ttl = int(Config.PN3_MSG_TTL)
    for mid, msg in list(bk.place_msgs.items()):
        if now - msg.t_sent <= ttl:
            continue
        if msg.state_delivery == 'sent':
            msg.state_delivery = 'undelivered'
            bk.pm['undelivered'] += 1
        elif not msg.response:
            msg.state_response = 'unanswered'
            bk.pm['delivered_no_verdict'] += 1
        if msg.response and any(not r.get('applied') for r in msg.response.values()):
            bk.pm['verdict_not_applied'] += 1
        bk.place_msgs.pop(mid, None)


def _pn3_step(agent, field):
    n = getattr(agent, '_nav', None)
    p2 = n.get('pn2') if n else None
    if p2 is None or not Config.PN3_ENABLED or agent.world is None:
        return
    s = _fch(agent.world)
    bk = _sg_book(agent.world)
    now = int(agent.world.age)
    _pn3_init(p2)
    if Config.PN3_FEEDBACK:
        _pn3_apply_feedback(agent, s, p2, bk, now)
    _pn3_event_expire(agent, s, p2, now)
    if Config.PN3_PROPOSALS:
        agent._pn3_handle(s, p2, now)
    if Config.PN3_PLACE_MSG and field is not None:
        _pn3_try_send(agent, s, field, p2, now)


def _pn3_summary(eng, s):
    bk = _sg_book(eng)
    alive = [p for p in eng.patterns if p.alive and p.role_type != 'feral' and getattr(p, '_nav', None) and (p._nav.get('pn2') or {}).get('events') is not None]
    ev_kind, ev_out, unsent, sent, props_now = Counter(), Counter(), 0, 0, Counter()
    for p in alive:
        p2 = p._nav['pn2']
        for e in p2['events']:
            ev_kind[e.kind] += 1
            ev_out[e.outcome] += 1
            if e.sent:
                sent += 1
            else:
                unsent += 1
        for pr in p2.get('props', ()):
            props_now[pr.state] += 1
    return {'enabled': bool(Config.PN3_ENABLED),
            'flags': {'proposals': bool(Config.PN3_PROPOSALS), 'place_msg': bool(Config.PN3_PLACE_MSG), 'feedback': bool(Config.PN3_FEEDBACK)},
            'counters': {k: int(v) for k, v in sorted(s.st.items()) if k.startswith('pn3_')},
            'msg': {k: int(v) for k, v in sorted(bk.pm.items())}, 'msgs_live': len(bk.place_msgs),
            'events_now': {'agents': len(alive), 'by_kind': dict(ev_kind), 'by_outcome': dict(ev_out), 'sent': sent, 'unsent': unsent},
            'props_now': dict(props_now)}


def _pn3_summary_safe(eng, s):
    try:
        return _pn3_summary(eng, s)
    except Exception as e:
        return {'enabled': bool(getattr(Config, 'PN3_ENABLED', False)), 'error': repr(e)}


def _pn3_line(c):
    if not bool(Config.PN3_ENABLED):
        return ""
    g = lambda k: int(c.get('pn3_' + k, 0))
    return (f"           PN3 prop made/act/lowU/lowC/exp/dedup/sw={g('proposals_made')}/{g('proposals_activated')}/{g('proposals_ignored_low_urgency')}/"
            f"{g('proposals_ignored_low_conf')}/{g('proposals_expired')}/{g('proposals_deduplicated')}/{g('intent_switches_after_proposal')} | "
            f"event new={g('event_new')} expired={g('event_expired')} | msg try/blocked={g('msg_attempt')}/{sum(int(v) for k, v in c.items() if k.startswith('pn3_msg_blocked_'))} | "
            f"credit +/-/conf={g('cr_pos')}/{g('cr_neg')}/{g('cr_conf_updates')} unlinked={g('fb_unlinked')}\n")


def _pn3_print(Q, c):
    if not Q or not Q.get('enabled'):
        print("\n  ── [O9] PN3 (предложения/события/сообщения/обратная связь): выключен или нет блока в audit ──")
        return
    k = Q['counters']
    m = Q['msg']
    g = lambda key: int(k.get('pn3_' + key, 0))
    print(f"\n  ── [O9] PN3 · ПРЕДЛОЖЕНИЯ → НАМЕРЕНИЯ (накопительно; flags={Q['flags']}) ──")
    print(f"  создано={g('proposals_made')} (по типам: { {t: g('prop_made_' + t.replace('-', '_')) for t in _PN3_PTYPES if g('prop_made_' + t.replace('-', '_'))} }); "
          f"принято системой целей={g('proposals_activated')}; отклонено: urgency={g('proposals_ignored_low_urgency')} conf={g('proposals_ignored_low_conf')}; "
          f"истекло={g('proposals_expired')}; дубли={g('proposals_deduplicated')}; невалидных={g('proposals_invalid')}")
    print(f"  без штатной цели (check/ignore)={ {kk[len('pn3_prop_no_goal_'):]: int(v) for kk, v in k.items() if kk.startswith('pn3_prop_no_goal_')} }; цель такого типа уже была={ {kk[len('pn3_goal_exists_'):]: int(v) for kk, v in k.items() if kk.startswith('pn3_goal_exists_')} }")
    print(f"  фактических переключений intent после предложения={g('intent_switches_after_proposal')}; цель погасла без переключения: "
          f"{ {kk[len('pn3_noswitch_'):]: int(v) for kk, v in k.items() if kk.startswith('pn3_noswitch_')} }; сейчас предложений у агентов={Q['props_now']}")
    _fin = {kk[len('pn3_fin_'):]: int(v) for kk, v in k.items() if kk.startswith('pn3_fin_')}
    _live = sum(int(v) for v in Q['props_now'].values())
    print(f"  воронка (исход каждого предложения): {_fin}; ещё живы у агентов={_live}; "
          f"не учтено (умерли вместе с агентом / вне выборки)={g('proposals_made') - sum(_fin.values()) - _live}")
    print("\n  ── [O9b] PN3 · PlaceEvent (наблюдения) ──")
    print(f"  создано={g('event_new')} по видам={ {kk[len('pn3_event_new_'):]: int(v) for kk, v in k.items() if kk.startswith('pn3_event_new_')} }; "
          f"истекло без вердикта={g('event_expired')} (из них никогда не отправлялись={g('event_expired_unsent')}); вытеснено из кольца={g('event_evicted')} (не отправлялись={g('event_evicted_unsent')})")
    en = Q['events_now']
    print(f"  сейчас в памяти агентов: событий по видам={en['by_kind']}; outcome={en['by_outcome']}; отправлено={en['sent']} / без отправки={en['unsent']}; "
          f"накопительно outcome: confirmed={g('outcome_confirmed')} rejected={g('outcome_rejected')} expired={g('outcome_expired')}")
    print("\n  ── [O9c] PN3 · PlaceMessage (сообщения через SIGNAL) ──")
    print(f"  попыток отправки={g('msg_attempt')}; отказов по причине={ {kk[len('pn3_msg_blocked_'):]: int(v) for kk, v in k.items() if kk.startswith('pn3_msg_blocked_')} }")
    print(f"  отправлено={m.get('sent', 0)} (по видам { {kk[5:]: v for kk, v in m.items() if kk.startswith('sent_')} }); доставлено сообщений={m.get('delivered_msgs', 0)} "
          f"(пар сообщение×слушатель={m.get('delivered_pairs', 0)}, из них распознано={m.get('interpreted_pairs', 0)}); не доставлено={m.get('undelivered', 0)}")
    print(f"  проверено (пар)={m.get('checked_pairs', 0)}: confirmed={m.get('verdict_confirmed', 0)} rejected={m.get('verdict_rejected', 0)} unknown={m.get('verdict_unknown', 0)}; "
          f"доставлено, но без verdict={m.get('delivered_no_verdict', 0)}; вердикт не дошёл до отправителя={m.get('verdict_not_applied', 0)}; сейчас в журнале={Q['msgs_live']}")
    print("\n  ── [O9d] PN3 · CREDIT ATTRIBUTION ──")
    print(f"  вердиктов: положительных={g('cr_pos')} отрицательных={g('cr_neg')} неопределённых={g('cr_unknown')}; обновлений confidence события={g('cr_conf_updates')}; "
          f"изменений доверия получателя к отправителю: ↑{m.get('trust_up', 0)} ↓{m.get('trust_down', 0)}; не удалось связать с исходным событием={g('fb_unlinked')}")

# Baseline re-pinned: _pn_fit/_pn_candidates were intentionally changed after v22.1 (SEEK patch);
# the guard now detects any FURTHER unintended edit of these 7 functions.
_PN3_PROTECTED = {
 "_pn_add_visit": "72cf6d8642dfa2ca018be8ec94bca0ff8dd363d6",
 "_pn_fit": "e23321f8e069d7cd20048841a767e8e549323162",
 "_pn_candidates": "b25b499b83bd6c37812ab99d1f367fce9439403a",
 "_pn_gate_scale": "2e61f39d9f96b34081b4877460e8b6f750066886",
 "_pn2_attribute": "a6e2b3b427deedf622b0930d017c1484ebbd1b67",
 "_pn2_q_step": "2730feef242edb12c891030deedb25621f8b8f31",
 "_pn2_stance": "328c26ca7408f3c2ebe398010c0baf2edebc5d04"
}


def _pn3_selftest():
    import types as _t
    out = []
    ck = lambda name, cond: out.append((bool(cond), name))
    C = Config
    saved = {k: getattr(C, k) for k in _PN3_CFG}
    try:
        class FA:
            def __init__(self, i, world, energy=1.0):
                self.id, self.world, self.energy, self.cells = i, world, energy, [(1, 1)]
                self.goals, self.intent, self.intent_commitment, self.last_intent_switch_age, self.age = [], None, 5.0, 0, 100
                self.emotional_memory = {'grief': 0.0, 'gratitude': 0.0}
                self.trust_ledger = TrustLedger()
                self.signal_forms, self._sig_last_emit_t, self._sig_msg_seq, self._sig_emit_until, self._sig_emit_spec = [], -1000, 0, -1, None
                self._nav = {'places': [], 'pi': [1.0, 2.0], 'pn2': {'events': None}}
                self.role_type = 'normal'
                self.form = {'spectrum': {0: 1.0}, 'label': 'f', 'uses': 0}
            def get_center(self): return (3.0, 4.0)
            def _x2_change(self, key, new): self.energy = new
            def _sig_pick(self, ctx): return self.form
            def _sig_in_panic(self): return False
            def _sig_is_alarm(self, form): return False
            _sig_emit = SignalMixin._sig_emit
            _sig_next_msg_id = SignalMixin._sig_next_msg_id
            _sig_place_send = SignalMixin._sig_place_send
        W = _t.SimpleNamespace(age=100, patterns=[])
        s = _t.SimpleNamespace(st=Counter())
        bk = _sg_book(W)
        mk_p2 = lambda: _pn3_init({'stance': 'none', 'story': deque(maxlen=40), 'P': 0.3, 'last_conf': 0.8, 'q': []})
        mk_ev = lambda t=95, conf=0.8, fo=(0.1,) * 9, fr=(0.5,) * 9, eid=11: PlaceEvent(kind='shifted', place_id=1, confidence=conf, t=t, pi_at_time=(1.0, 2.0, t), event_id=eid, conf0=conf, fp_obs=fo, fp_ref=fr, world_xy=(3.0, 4.0))
        A = FA(1, W, energy=0.0)
        p2 = mk_p2(); ev = mk_ev(); p2['events'].append(ev)
        C.PN3_PLACE_MSG = True
        _pn3_try_send(A, s, object(), p2, 100)
        ck('1 событие живёт без отправки (нет энергии → sent=False, событие на месте)', (not ev.sent) and len(p2['events']) == 1 and s.st['pn3_msg_blocked_no_energy'] == 1)
        B = FA(2, W, energy=1.0)
        m, why = B._sig_place_send(object(), 100, ev)
        e1 = B.energy
        m2, why2 = B._sig_place_send(object(), 101, mk_ev(eid=12))
        ck('2 отправка списала SIG_EMIT_COST, записана в tx и в журнал сообщений', m is not None and abs((1.0 - e1) - C.SIG_EMIT_COST) < 1e-9 and m.message_id in bk.place_msgs and bk.tx[-1][8] == m.message_id)
        ck('2 cooldown SIGNAL соблюдён (повтор через 1 тик отказан, энергия не списана)', m2 is None and why2 == 'cooldown' and B.energy == e1)
        bad = 0
        for kw in (dict(urgency=1.5), dict(confidence=-0.1), dict(ttl=-1), dict(type='bogus')):
            a = dict(type='explore', urgency=0.5, confidence=0.5, source='x', ttl=3); a.update(kw)
            try:
                ActionProposal(**a)
            except ValueError:
                bad += 1
        p2 = mk_p2(); G = FA(3, W)
        r1 = _pn3_prop_new(G, s, p2, 'explore', 0.6, 0.6, ('q', 1), 10, 100, 'event')
        r2 = _pn3_prop_new(G, s, p2, 'explore', 0.6, 0.6, ('q', 1), 10, 101, 'event')
        ck('3 невалидные предложения отвергаются (4/4); повтор для активного источника не создаётся', bad == 4 and r1 is not None and r2 is None and len(p2['props']) == 1 and s.st['pn3_proposals_deduplicated'] == 1)
        p2 = mk_p2(); G = FA(4, W)
        _pn3_prop_new(G, s, p2, 'explore', 0.9, 0.9, ('q', 2), 3, 100, 'event')
        _pn3_handle(G, s, p2, 110)
        ck('4 истёкшее: цели нет, intent/commitment не тронуты, счётчик expired', not G.goals and G.intent is None and G.intent_commitment == 5.0 and s.st['pn3_proposals_expired'] == 1)
        p2 = mk_p2(); G = FA(5, W); G.intent = {'type': 'cooperate', 'priority': 3.0, 'age': 0}
        before = (G.intent, G.intent_commitment, G.last_intent_switch_age)
        _pn3_prop_new(G, s, p2, 'explore', 0.9, 0.9, ('q', 3), 10, 100, 'event')
        _pn3_handle(G, s, p2, 100)
        ck('5 принято: цель добавлена в goals, intent/commitment/last_switch не изменены, переключения нет', len(G.goals) == 1 and G.goals[0]['_source'] == 'pn_proposal' and (G.intent, G.intent_commitment, G.last_intent_switch_age) == before and s.st['pn3_intent_switches_after_proposal'] == 0)
        _pn3_prop_new(G, s, p2, 'explore', 0.9, 0.9, ('q', 99), 10, 100, 'event')
        n_goals = len(G.goals)
        _pn3_handle(G, s, p2, 100)
        ck('5 дубликат explore не создаётся (слияние с существующей целью)', len(G.goals) == n_goals and s.st['pn3_goal_exists_explore'] == 1)
        G.intent = G.goals[0]
        _pn3_handle(G, s, p2, 101)
        ck('5 фактическое переключение учитывается отдельно от активации', s.st['pn3_intent_switches_after_proposal'] == 1 and s.st['pn3_proposals_activated'] == 2)
        p2 = mk_p2(); G = FA(6, W)
        _pn3_prop_new(G, s, p2, 'explore', 0.05, 0.9, ('q', 4), 10, 100, 'event')
        _pn3_prop_new(G, s, p2, 'explore', 0.9, 0.05, ('q', 5), 10, 100, 'event')
        _pn3_prop_new(G, s, p2, 'check', 0.9, 0.9, ('q', 6), 10, 100, 'event')
        _pn3_handle(G, s, p2, 100)
        ck('5b низкий urgency / низкая confidence / тип check → цели нет, причины раздельно', not G.goals and s.st['pn3_proposals_ignored_low_urgency'] == 1 and s.st['pn3_proposals_ignored_low_conf'] == 1 and s.st['pn3_prop_no_goal_check'] == 1)
        p2 = mk_p2(); G = FA(7, W); n = {'places': [], 'pi': [1.0, 2.0], 'fp_last': [0.2] * 9}
        ep = {'t': 100, 'place_id': 1, 'attr': 'place_shifted'}
        a_ = {'conf': 0.8}
        e1_ = _pn3_event_new(G, s, n, p2, ep, {}, a_, 100, 1, 'place_shifted')
        e2_ = _pn3_event_new(G, s, n, p2, ep, {}, a_, 101, 1, 'place_shifted')
        ck('6 одно событие на эпизод (второй вызов для той же записи → None); снимок pi не перезаписывается', e1_ is not None and e2_ is None and len(p2['events']) == 1 and e1_.pi_at_time == (1.0, 2.0, 100) and p2['story'][-1].get('ev_ids') == [e1_.event_id])
        n['pi'][0] = 99.0
        ck('6 снимок позиции в событии не меняется при дальнейшем движении', e1_.pi_at_time[0] == 1.0)
        def fresh_pair():
            Wn = _t.SimpleNamespace(age=100, patterns=[])
            An, Bn, Cn = FA(10, Wn), FA(20, Wn), FA(30, Wn)
            pe = mk_p2(); ev_a = mk_ev(); ev_b = mk_ev(eid=77, conf=0.7); pe['events'].append(ev_a); pe['events'].append(ev_b)
            An._nav['pn2'] = pe
            m_, _w = An._sig_place_send(object(), 100, ev_a)
            ev_a.sent, ev_a.msg_id = True, m_.message_id; pe['msgs'].append(m_.message_id)
            return Wn, An, Bn, Cn, pe, ev_a, ev_b, m_
        Wn, An, Bn, Cn, pe, ev_a, ev_b, m_ = fresh_pair()
        bkn = _sg_book(Wn); sn = _t.SimpleNamespace(st=Counter())
        ent = bkn.tx[-1]
        _pn3_on_heard(Bn, bkn, ent, 'stress_missed', 101)
        ck('7 доставка без распознавания ≠ подтверждение: delivered есть, verdict нет, outcome=unknown', m_.state_delivery == 'delivered' and not m_.response and ev_a.outcome == 'unknown')
        Bn._nav['places'] = []
        _pn3_on_heard(Bn, bkn, ent, 'recognized', 102)
        _pn3_apply_feedback(An, sn, pe, bkn, 103)
        ck('7 распознано, но проверить нечем → verdict=unknown; событие не подтверждено и не отклонено; доверие не менялось', m_.response[20]['verdict'] == 'unknown' and ev_a.outcome == 'unknown' and Bn.trust_ledger.get(10) == C.TRUST_BASE and ev_a.confidence == 0.8)
        def mem_place(fp, pid=0):
            pl = {'id': pid, 'visits': [], 'fp_mean': np.asarray(fp, dtype=np.float64), 'fp_var': np.zeros(9), 'pi_mean': [0.0, 0.0]}
            return pl
        Wn, An, Bn, Cn, pe, ev_a, ev_b, m_ = fresh_pair(); bkn = _sg_book(Wn); sn = _t.SimpleNamespace(st=Counter()); ent = bkn.tx[-1]
        Bn._nav['places'] = [mem_place(ev_a.fp_obs)]
        tB0, tAB0 = Bn.trust_ledger.get(10), An.trust_ledger.get(20)
        _pn3_on_heard(Bn, bkn, ent, 'recognized', 102)
        _pn3_apply_feedback(An, sn, pe, bkn, 103)
        ck('8 verdict связан с отправителем, получателем и исходным событием: feedback_received=[(20,rejected,102)]', ev_a.feedback_received == [(20, 'rejected', 102)] and m_.event_id == ev_a.event_id and m_.sender_id == 10)
        ck('9 отрицательный verdict: ↓confidence только ЭТОГО события, ↓доверие получателя к отправителю; чужое событие и доверие отправителя к получателю не тронуты',
           ev_a.confidence < 0.8 and ev_b.confidence == 0.7 and Bn.trust_ledger.get(10) < tB0 and An.trust_ledger.get(20) == tAB0 and Cn.trust_ledger.get(10) == C.TRUST_BASE and ev_a.outcome == 'rejected')
        Wn, An, Bn, Cn, pe, ev_a, ev_b, m_ = fresh_pair(); bkn = _sg_book(Wn); sn = _t.SimpleNamespace(st=Counter())
        Bn._nav['places'] = [mem_place(ev_a.fp_ref)]
        _pn3_on_heard(Bn, bkn, bkn.tx[-1], 'partial', 102)
        _pn3_apply_feedback(An, sn, pe, bkn, 103)
        ck('9b положительный verdict: ↑confidence по явному правилу, ↑доверие получателя (SIG_TRUST_UP), outcome=confirmed', ev_a.confidence > 0.8 and Bn.trust_ledger.get(10) > C.TRUST_BASE and ev_a.outcome == 'confirmed')
        Wn, An, Bn, Cn, pe, ev_a, ev_b, m_ = fresh_pair(); bkn = _sg_book(Wn); sn = _t.SimpleNamespace(st=Counter())
        W2 = 100 + C.PN3_EVENT_TTL + 5
        _pn3_event_expire(An, sn, pe, W2)
        ck('10 отсутствие обратной связи ≠ отклонение: outcome=expired, confidence и доверие не изменены', ev_a.outcome == 'expired' and ev_a.confidence == 0.8 and Bn.trust_ledger.get(10) == C.TRUST_BASE and sn.st['pn3_outcome_expired'] == 2)
        Wn.age = 100 + C.PN3_MSG_TTL + 5
        _pn3_world_tick(Wn, Wn.age)
        ck('10 сообщение без слушателей закрыто как undelivered, а не rejected', bkn.pm['undelivered'] == 1 and not bkn.place_msgs and bkn.pm['verdict_rejected'] == 0)
        if '_PN3_PROTECTED' in globals():
            import hashlib as _h, inspect as _i
            try:
                bad_ = [nm for nm, hh in _PN3_PROTECTED.items() if _h.sha1(_i.getsource(globals()[nm]).encode('utf-8')).hexdigest() != hh]
                ck('12 PlaceNode (' + '/'.join(_PN3_PROTECTED) + ') побайтно как в v22.1' + (f' — ИЗМЕНЕНЫ: {bad_}' if bad_ else ''), not bad_)
            except (OSError, TypeError):
                out.append((True, '12 ПРОПУЩЕНО: исходники функций недоступны (inspect.getsource) — сверку см. в отчёте патча'))
    except Exception as e:
        import traceback
        out.append((False, f'исключение в тесте: {e!r}\n{traceback.format_exc()}'))
    finally:
        for k, v in saved.items():
            setattr(C, k, v)
    print("[PN3 selftest] контракты A/B/C на моках:")
    for ok_, name in out:
        print(f"  {'OK  ' if ok_ else 'FAIL'} {name}")
    good = all(o for o, _n in out)
    print(f"[PN3 selftest] {sum(1 for o, _n in out if o)}/{len(out)}")
    _obs_register_selftest('pn3', good, passed=sum(1 for o, _n in out if o), total=len(out),
                           failed=[str(_n).split('\n')[0][:110] for o, _n in out if not o])
    return good


try:
    _pn3_selftest()
except Exception as _e:
    print(f"[PN3 selftest] не выполнен: {_e!r}")
    _obs_register_selftest('pn3', False, failed=[('не выполнен: ' + repr(_e))[:110]])


class NavPatternMixin:
    def _nav_pn2_stance(self, n, p2, att, a, energy, now):
        """Extension point for somatic modulation; base NAV stays deterministic."""
        return _pn2_stance(n, p2, att, a, energy, now)

    def _pn3_handle(self, s, p2, now):
        """Per-agent extension point around the shared PN3 proposal handler."""
        return _pn3_handle(self, s, p2, now)

    spatial_pred_error = 0.0
    place_confidence = 0.5
    nav_motion_err = 0.0

    def __init__(self, pid, cells, parent=None, world=None):
        super().__init__(pid, cells, parent, world)
        self._tame = {}
        self._nav = None
        self._fch_veto_t = -1
        self.spatial_pred_error = 0.0
        self.place_confidence = 0.5
        self.nav_motion_err = 0.0
        self._fch_init_gene(parent)

    def _fch_init_gene(self, parent=None):
        pg = getattr(parent, '_fch_gene', None) if parent is not None else None
        g = {}
        for i, (k, lo, hi, ilo, ihi) in enumerate(_FCH_GENES):
            if pg is not None and k in pg:
                z = (phi_hash(self.id, i, 90211) + phi_hash(self.id, i, 90212) + phi_hash(self.id, i, 90213) - 1.5) / 0.5
                g[k] = float(min(hi, max(lo, pg[k] + float(Config.FCH_GENE_MUT) * (hi - lo) * z)))
            else:
                g[k] = float(ilo + (ihi - ilo) * phi_hash(self.id, i, 90210))
        self._fch_gene = g

    def _fch_genes(self):
        g = getattr(self, '_fch_gene', None)
        if g is None:
            self._fch_init_gene(None)
            g = self._fch_gene
        return g

    def _nav_st(self):
        n = getattr(self, '_nav', None)
        if n is None:
            n = {'c': None, 'S': None, 'G': None, 'pi': [0.0, 0.0], 'th': 0.0, 'u_prev': None, 'path': 0.0, 'anch': [], 'a_last': -99,
                 'places': [], 'place_counter': 0, 'current_place': None, 'place_candidates': [], 'pending': None, 'recog': None, 'miss': 0, 'last_place': None, 'last_conf_t': -9999, 'first_t': None,
                 'escale': _NAV_ERR_FLOOR.copy(), 'dis': 0.0,
                 'cmd': None, 'last_app': -99, 'disp': deque(maxlen=3), 'D': 0.0, 'D_t': -99, 'last_rest_log': -99}
            self._nav = n
        n['_sem_owner'] = self
        return n

    def _nav_tick(self, field, t):
        if (not Config.NAV_ENABLE or _FCH_ABL['nav_off'] or not self.alive or not self.cells or self.world is None
                or field is None or self.role_type == 'feral'):
            return
        now = int(self.world.age)
        if (now + int(self.id)) % int(Config.NAV_SENSE_EVERY) != 0:
            return
        s = _fch(self.world)
        n = self._nav_st()
        if n.get('first_t') is None:
            n['first_t'] = now
        W = Config.WORLD_SIZE
        cx, cy = self._nav_body_center()
        body, ring, G = _nav_sample(field, int(round(cx)) % W, int(round(cy)) % W, self.id)
        s.st['senses'] += 1
        dis_inst = 0.0
        motion_pulse = 0.0
        legacy = bool(Config.NAV_LEGACY_TENSION_HACK)
        if n['c'] is not None and n['S'] is not None:
            dx = ((cx - n['c'][0] + W / 2.0) % W) - W / 2.0
            dy = ((cy - n['c'][1] + W / 2.0) % W) - W / 2.0
            dd = math.hypot(dx, dy)
            if dd > 3.0:
                dx, dy, dd = dx * 3.0 / dd, dy * 3.0 / dd, 3.0
            _nav_dead_reckon(n, dx, dy, dd, self.id, now)
            pred = n['S'] + np.array([dx, dy]) @ n['G']
            e = np.abs(body[_NAV_IDX_ERR] - pred)
            z = e / np.maximum(n['escale'], _NAV_ERR_FLOOR)
            zr = float(np.sqrt(np.mean(np.minimum(z, 10.0) ** 2)))
            a = float(Config.NAV_ERR_ALPHA)
            n['escale'] = (1.0 - a) * n['escale'] + a * np.maximum(e, _NAV_ERR_FLOOR)
            dis_inst = float(min(1.0, max(0.0, (zr - 1.0) / (float(Config.NAV_DIS_Z) - 1.0))))
            if zr > float(Config.NAV_DIS_Z):
                if legacy:
                    self.cognitive_tension = min(Config.MAX_METRIC, float(self.cognitive_tension) + float(Config.NAV_DIS_TENSION) * dis_inst)
                else:
                    motion_pulse = dis_inst
                s.st['disorient_events'] += 1
        a_err = 0.0
        rec = None
        path_before = float(n['path'])
        if not _FCH_ABL['anchor_off']:
            fp_now = _nav_fingerprint(field, int(round(cx)) % W, int(round(cy)) % W)
            if Config.NAV_PLACE_NODES:
                rec = self._nav_place_recognition(fp_now, now, s)
                a_err = float(rec['pi_err'])
            else:
                a_err = _nav_anchor_step(n, fp_now, now, s.st)
            if a_err > 0.0:
                dis_inst = max(dis_inst, min(1.0, a_err / float(Config.NAV_ANCHOR_ERR_FULL)))
        n['dis'] = 0.75 * n['dis'] + 0.25 * dis_inst
        if not legacy:
            self._nav_publish(s, n, a_err, path_before, motion_pulse, rec)
            if Config.PN_V2_ENABLED and rec is not None:
                n['fp_last'] = fp_now
                try:
                    self._pn2_tick(s, n, rec, motion_pulse, now)
                except Exception as e:
                    _fch_err(s, 'pn2', e)
        D = float(max(body[0], np.sort(ring[:, 0])[-2:].mean()))
        A = float(body[1])
        uf = _nav_unit(G[:, 0], ring[:, 0]) if D >= float(Config.NAV_DANGER_MIN) else None
        genes = self._fch_genes()
        hx = hy = 0.0
        ua = None
        w_f = genes['flee'] * min(1.0, D / 0.2) if D >= float(Config.NAV_DANGER_MIN) else 0.0
        if w_f > 0.0:
            u = _nav_unit(G[:, 0], ring[:, 0])
            if u is None:
                w_f = 0.0
            else:
                hx -= w_f * u[0]
                hy -= w_f * u[1]
        w_m = 0.0
        if A >= float(Config.NAV_ALARM_MIN):
            ua = _nav_unit(G[:, 1], ring[:, 1])
            if ua is not None:
                w_m = genes['mob'] * min(1.0, A / 0.3)
                hx += w_m * ua[0]
                hy += w_m * ua[1]
        inten = max(w_f, w_m)
        kind, w, h = None, 0.0, None
        if inten > 0.0 and math.hypot(hx, hy) > 1e-6:
            kind = 'flee' if w_f >= w_m else 'mob'
            m = math.hypot(hx, hy)
            h, w = (hx / m, hy / m), min(1.0, inten)
        mob = 1.0
        if kind in ('flee', 'mob'):
            mob = 1.0 + ((float(Config.NAV_FLEE_BOOST) if kind == 'flee' else float(Config.NAV_MOB_BOOST)) - 1.0) * min(1.0, inten)
        n['cmd'] = {'t': now, 'h': h, 'w': w, 'mob': mob, 'kind': kind, 'D': D, 'A': A, 'ua': ua, 'uf': uf}
        n['D'], n['D_t'] = D, now
        n['c'] = (cx, cy)
        n['S'] = body[_NAV_IDX_ERR].copy()
        n['G'] = G[:, _NAV_IDX_ERR].copy()

    def _nav_place_recognition(self, fp, now, s):
        n = self._nav_st()
        st = s.st
        C = Config
        pi = (float(n['pi'][0]), float(n['pi'][1]))
        path = float(n['path'])
        conf_before = 1.0 / (1.0 + max(0.0, path) / max(1e-6, float(C.NAV_ANCHOR_GATE)))
        res = {'state': 'unknown', 'pi_err': 0.0, 'err': 0.0, 'conf': 0.0, 'conf_before': conf_before}
        if fp is None:
            return res
        cur = n['current_place']
        diag = bool(C.NAV_PLACE_DIAG)
        cands, rej = _pn_candidates(n, fp, pi, path)
        st['place_gate_rej'] += rej
        if diag:
            st['diag_samples'] += 1
            st['diag_nocand' if not cands else 'diag_withcand'] += 1
        if cur is not None and not any(p is cur for _cd, p in cands):
            cands.append((float(np.mean(np.abs(fp - cur['fp_mean']))), cur))
        last = _pn_by_id(n, n['last_place'])
        cur_id = cur['id'] if cur is not None else None
        _g0, _sc0 = _pn_gate_scale(path)
        ev = []
        for _cd, p in cands:
            edge_n = int(last['transitions'].get(p['id'], {}).get('count', 0)) if (last is not None and last is not p) else 0
            err_fp, conf, d_pi, _z = _pn_fit(p, fp, pi, path, cur_id, edge_n)
            if diag and _z <= float(C.NAV_PLACE_Z_HALF):
                _g, _sc = _pn_gate_scale(path)
                st['diag_dpi_b%d' % _pn_dpi_bucket(d_pi)] += 1
                st['diag_pi_in' if d_pi <= _g else ('diag_pi_soft' if d_pi < _sc else 'diag_pi_zero')] += 1
            thr = float(C.NAV_PLACE_MIN_CONF) + (0.0 if p is cur else (1.0 - conf_before) * float(C.NAV_PLACE_NOPI_EXTRA))
            _ok = conf >= thr and err_fp <= float(C.NAV_PLACE_MAX_ERR)
            if diag and not _ok:
                _fc, _fe = conf < thr, err_fp > float(C.NAV_PLACE_MAX_ERR)
                st['diag_fail_both' if (_fc and _fe) else ('diag_fail_conf' if _fc else 'diag_fail_err')] += 1
            if _ok and bool(C.NAV_PLACE_ALIAS_GUARD) and d_pi > _g0 + float(C.NAV_PLACE_ALIAS_FRAC) * (_sc0 - _g0):
                _ok = False
                st['place_alias_veto'] += 1
            ev.append({'p': p, 'err': err_fp, 'conf': conf, 'd_pi': d_pi, 'score': conf - err_fp, 'ok': _ok})
        ev.sort(key=lambda e: -e['score'])
        n['place_candidates'] = [(e['p']['id'], round(e['err'], 3), round(e['conf'], 3)) for e in ev[:3]]
        acc = [e for e in ev if e['ok']]
        margin = float(C.NAV_PLACE_AMBIG_MARGIN)

        if acc:
            top = acc[0]
            if cur is not None and top['p'] is not cur:
                ce = next((e for e in acc if e['p'] is cur), None)
                if ce is not None and top['score'] - ce['score'] < margin:
                    top = ce
            if top['p'] is not cur and any(e is not top and top['score'] - e['score'] < margin for e in acc):
                st['place_ambig'] += 1
                n['miss'] = 0
                _pn_pend_drop(n, st, 'ambig')
                res.update(state='ambiguous', conf=float(top['conf']) * float(C.NAV_PLACE_AMBIG_CONF))
                return res
            p = top['p']
            confirmed = True
            if p is not cur:
                rc = n['recog']
                if rc is not None and rc['id'] == p['id'] and now - rc['t'] <= int(C.NAV_PLACE_PEND_TTL):
                    rc['n'] += 1
                    rc['t'] = now
                else:
                    rc = {'id': p['id'], 'n': 1, 't': now}
                    n['recog'] = rc
                confirmed = rc['n'] >= int(C.NAV_PLACE_RECOG_CONFIRM)
            else:
                n['recog'] = None
        else:
            confirmed = False
            top = None
        if diag and acc:
            st['diag_acc'] += 1
        if acc and confirmed:
            n['miss'] = 0
            n['drift_n'] = 0
            _pn_pend_drop(n, st, 'recognized')
            arrival = p is not cur
            res.update(state='recognized', conf=float(top['conf']))
            if arrival or path >= float(C.NAV_ANCHOR_MIN_PATH):
                e_pi = min(1.0, top['d_pi'] / max(1e-6, float(C.NAV_ANCHOR_ERR_FULL)))
                res['pi_err'] = max(float(top['d_pi']), 1e-9)
                res['err'] = 1.0 - (1.0 - e_pi) * (1.0 - float(C.NAV_PLACE_FP_ERR_W) * float(top['err']))
                # reset target = pi_mean BEFORE this visit, so the current drift is not fed back into the snap
                _pi_snap = (float(p['pi_mean'][0]), float(p['pi_mean'][1]))
                _pn_add_visit(p, fp, pi, n['th'], now)
                n['pi'][0], n['pi'][1] = _pi_snap
                n['th'] = float(p['th_mean'])
                n['path'] = 0.0
                n['a_last'] = now
                p['n_hits'] = p.get('n_hits', 0) + 1
                st['place_hit'] += 1
                st['place_err_pi_milli'] += int(round(1000.0 * top['d_pi']))
                st['place_err_fp_milli'] += int(round(1000.0 * top['err']))
            elif now - p['last_visit_t'] >= int(C.NAV_PLACE_VISIT_GAP):
                _pn_add_visit(p, fp, pi, n['th'], now)
                p['n_cont'] = p.get('n_cont', 0) + 1
                st['place_cont'] += 1
            else:
                if bool(C.NAV_PLACE_VISIT_CLOCK_FIX):
                    p['last_seen_t'] = int(now)
                else:
                    p['last_visit_t'] = int(now)
                if diag:
                    st['diag_rec_stay'] += 1
            if arrival:
                _pn_pred_check(n, p, fp, now, st)
                _pn_record_edge(n, p, now, st)
                n['current_place'] = p
                n['place_pred'] = _pn_predict(n, p, st)
            n['last_place'] = p['id']
            n['last_conf_t'] = int(now)
            return res

        tentative = bool(acc) and top is not None
        if tentative:
            st['place_tentative'] += 1
        else:
            n['recog'] = None
        n['miss'] += 1
        held = cur is not None and n['miss'] < int(C.NAV_PLACE_MISS_LIMIT)
        ce = next((e for e in ev if e['p'] is cur), None) if cur is not None else None
        near = ce is not None and ce['conf'] >= float(C.NAV_PLACE_NEAR_FRAC) * float(C.NAV_PLACE_MIN_CONF)
        if held and near:
            st['place_uncertain'] += 1
            _pn_pend_drop(n, st, 'uncertain')
            if bool(C.NAV_PLACE_DRIFT_ADAPT) and ce['d_pi'] <= _g0 and ce['err'] <= 2.0 * float(C.NAV_PLACE_MAX_ERR):
                n['drift_n'] = int(n.get('drift_n', 0)) + 1
                if n['drift_n'] >= int(C.NAV_PLACE_DRIFT_CONFIRM):
                    _pn_add_visit(cur, fp, pi, n['th'], now)
                    n['drift_n'] = 0
                    n['miss'] = 0
                    st['place_drift_adapt'] += 1
            else:
                n['drift_n'] = 0
            res.update(state='uncertain', conf=float(ce['conf']))
            return res
        if cur is not None and not held:
            n['current_place'] = None
            n['drift_n'] = 0
            n['place_pred'] = None
            st['place_left'] += 1
            cur = None
        if tentative:
            _pn_pend_drop(n, st, 'tentative')
            res.update(state='tentative', conf=float(top['conf']) * float(C.NAV_PLACE_AMBIG_CONF))
            return res
        pend = n['pending']
        if pend is not None:
            f0, p0 = pend['visits'][0]['fp'], pend['visits'][0]['pi']
            _r_ttl = now - pend['last_t'] > int(C.NAV_PLACE_PEND_TTL)
            _r_fp = float(np.mean(np.abs(fp - f0))) > float(C.NAV_PLACE_NOVEL_FP_TOL)
            _r_pi = _pn_toroidal_dist(pi, p0) > float(C.NAV_PLACE_NOVEL_PI_GATE)
            if _r_ttl or _r_fp or _r_pi:
                if pend['visits']:
                    st['pend_lost_obs'] += len(pend['visits'])
                    st['pend_reset_ttl' if _r_ttl else ('pend_reset_fp' if _r_fp else 'pend_reset_pi')] += 1
                pend = None
        if pend is None:
            pend = {'visits': [], 'last_t': now}
        pend['visits'].append({'t': int(now), 'fp': np.asarray(fp, dtype=np.float32).copy(), 'pi': pi, 'th': float(n['th'])})
        pend['last_t'] = now
        n['pending'] = pend
        st['place_pending'] += 1
        if len(pend['visits']) >= int(C.NAV_PLACE_NOVEL_CONFIRM):
            if diag and n['places']:
                _fm = np.mean([v['fp'] for v in pend['visits']], axis=0)
                _best = min(n['places'], key=lambda q: float(np.mean(np.abs(_fm - q['fp_mean']))))
                _cd = float(np.mean(np.abs(_fm - _best['fp_mean'])))
                if _cd <= float(C.NAV_PLACE_COARSE_FP):
                    st['diag_new_dup_coarse'] += 1
                if _cd <= float(C.NAV_PLACE_FP_SIGMA_MIN) * 1.5:
                    st['diag_new_dup_tight'] += 1
                    _pm = pend['visits'][-1]['pi']
                    _dp = math.hypot(_pm[0] - _best['pi_mean'][0], _pm[1] - _best['pi_mean'][1])
                    _, _sc = _pn_gate_scale(path)
                    _ef, _cf, _dd, _zz = _pn_fit(_best, _fm, _pm, path, cur_id, 0)
                    _thr = float(C.NAV_PLACE_MIN_CONF) + (1.0 - conf_before) * float(C.NAV_PLACE_NOPI_EXTRA)
                    if _dp >= _sc:
                        st['diag_dup_pi_far'] += 1
                    elif _cf < _thr:
                        st['diag_dup_conf_low'] += 1
                    elif _ef > float(C.NAV_PLACE_MAX_ERR):
                        st['diag_dup_err_high'] += 1
                    else:
                        st['diag_dup_other'] += 1
            node = _pn_new(n['place_counter'], pend['visits'], now)
            n['place_counter'] += 1
            _pn_add_node(n, node, st)
            _pn_pred_check(n, node, node['fp_mean'], now, st)
            _pn_record_edge(n, node, now, st)
            n['current_place'] = node
            n['place_pred'] = _pn_predict(n, node, st)
            n['last_place'] = node['id']
            n['last_conf_t'] = int(now)
            n['pending'] = None
            n['miss'] = 0
            n['path'] = 0.0
            n['a_last'] = now
            st['place_new'] += 1
            res.update(state='new', conf=float(C.NAV_PLACE_NEW_CONF))
            return res
        res.update(state='pending', conf=float(ce['conf']) if (held and ce is not None) else 0.0)
        return res

    def _nav_publish(self, s, n, a_err, path_before, motion_pulse, rec=None):
        if rec is not None:
            conf, err, conf_before = float(rec['conf']), float(rec['err']), float(rec['conf_before'])
        else:
            conf, err, conf_before = _nav_place_stub(n, a_err, path_before)
        self.place_confidence = conf
        if motion_pulse > 0.0:
            self.nav_motion_err = max(float(self.nav_motion_err), float(motion_pulse))
            s.st['motion_err_pulse'] += 1
        if err > 0.0:
            self.spatial_pred_error = max(float(self.spatial_pred_error), err)
            s.st['spatial_err_event'] += 1
            s.st['spatial_err_milli'] += int(round(1000.0 * err))
            if err > float(Config.NAV_ERROR_LOG_THRESHOLD):
                kind = 'lost' if conf_before < float(Config.NAV_LOST_CONF) else 'unexpected'
                s.st['spatial_err_' + kind] += 1
                self._log_event('nav_error_' + kind, err=round(err, 3), conf=round(conf_before, 3), path=round(path_before, 1))

    def _pn2_tick(self, s, n, rec, motion_pulse, now):
        C = Config
        if rec.get('state') == 'unknown':
            s.st['pn2_skip'] += 1
            return
        p2 = _pn2_st(n)
        if p2['obs'] == 0:
            p2['last_disc_t'] = p2['last_insuff_t'] = int(now)
        p2['obs'] += 1
        cands = n.get('place_candidates') or []
        cur = n['current_place']
        ref = cur['id'] if cur is not None else (cands[0][0] if cands else None)
        a = _pn2_attribute(rec['state'], rec['conf'], rec['pi_err'], rec['err'], motion_pulse, _pn2_e_fp(n, ref), _pn2_at_place(n, ref))
        att = a['attr']
        pid = None if att == 'place_unknown' else ref
        s.st['pn2_obs'] += 1
        s.st['pn2_a_' + att] += 1
        if att == 'place_unknown':
            s.st['pn2_unk_cand' if cands else 'pn2_unk_nocand'] += 1
        p2['last_attr'], p2['last_conf'], p2['last_disc'] = att, a['conf'], a['disc']
        _prev = (id(p2['story'][-1]), p2['story'][-1]['n']) if p2['story'] else None
        if pid is not None:
            node = _pn_by_id(n, pid)
            if node is not None:
                na = node.setdefault('pn2', {'ok': 0, 'fail': 0, 'attr': {}})
                na['fail' if att in ('place_shifted', 'unexplained') else 'ok'] += 1
                if att != 'self_consistency':
                    na['attr'][att] = na['attr'].get(att, 0) + 1
        else:
            p2['nowhere'] += 1
        ep = None
        if att != 'self_consistency':
            p2['last_disc_t'] = int(now)
            ep = p2['ep']
            new_ep = not (ep is not None and ep['place_id'] == pid and ep['attr'] == att and now - ep['last_t'] <= int(C.PN_V2_EPISODE_GAP))
            if new_ep:
                ep = {'t': int(now), 'last_t': int(now), 'place_id': pid, 'attr': att, 'alt': a['alt'], 'conf': a['conf'], 'sensory_err': a['ev']['m'],
                      'spatial_err': round(float(rec['err']), 3), 'stance': p2['stance'], 'obs': 1, 'resolved': False}
                p2['ep'] = ep
                p2['exp'].append(ep)
                s.st['pn2_ep_new'] += 1
                if Config.PN3_ENABLED:
                    try:
                        _pn3_event_new(self, s, n, p2, ep, rec, a, now, pid, att)
                    except Exception as e:
                        _fch_err(s, 'pn3_event', e)
            else:
                ep['obs'] += 1
                ep['last_t'] = int(now)
                ep['conf'] = max(ep['conf'], a['conf'])
                ep['spatial_err'] = max(ep['spatial_err'], round(float(rec['err']), 3))
                ep['stance'] = p2['stance']
            if att in ('place_shifted', 'unexplained'):
                fk = (pid, att)
                fl = p2['fail'].get(fk)
                if fl is None:
                    if len(p2['fail']) >= int(C.PN_V2_FAIL_MAX):
                        p2['fail'].pop(min(p2['fail'], key=lambda k: (p2['fail'][k]['obs'], str(k))))
                    fl = p2['fail'][fk] = {'eps': 0, 'obs': 0, 'first_t': int(now), 'last_t': int(now), 'cleared': 0, 'stable': False}
                fl['obs'] += 1
                fl['last_t'] = int(now)
                if new_ep:
                    fl['eps'] += 1
                if not fl['stable'] and fl['eps'] >= int(C.PN_V2_FAIL_STABLE):
                    fl['stable'] = True
                    s.st['pn2_stable_fail'] += 1
                    _pn2_story(p2, now, 'stable_failure', pid, att, p2['stance'], None, None, 'repeats', 'known_weak_spot')
        elif (not p2['q'] and now - p2['last_disc_t'] >= int(C.PN_V2_QUIET_INTERVAL) and now - p2['last_insuff_t'] >= int(C.PN_V2_QUIET_INTERVAL)):
            p2['last_insuff_t'] = int(now)
            s.st['pn2_insuff'] += 1
            _pn2_story(p2, now, 'insufficient_evidence', ref, 'self_consistency', p2['stance'], None, None, 'no_recent_discrepancy', 'none')
        _pn2_q_step(s, p2, self.id, att, rec['state'], pid, now, ep, a)
        sw = self._nav_pn2_stance(n, p2, att, a, getattr(self, 'energy', 0.5), now)
        s.st['pn2_st_' + p2['stance']] += 1
        if p2['want']:
            s.st['pn2_want_' + p2['want']] += 1
        if sw:
            s.st['pn2_stance_switch'] += 1
        if Config.PN3_ENABLED and Config.PN3_PROPOSALS:
            try:
                self._generate_proposals(s, n, p2, now)
            except Exception as e:
                _fch_err(s, 'pn3_gen', e)
        if bool(C.PN_V2_LOG_EVENTS) and p2['story']:
            L = p2['story'][-1]
            if _prev is None or _prev != (id(L), L['n']):
                self._log_event('pn2_' + L['kind'], place=L['place_id'], attr=L['attr'], result=L['result'])

    def _pn2_apply_goal(self, n, now, s):
        p2 = n.get('pn2')
        if p2 is None:
            return
        want = p2['want'] if now - int(p2['want_t']) <= int(Config.PN_V2_STANCE_PERSIST) else None
        drop = {id(g) for g in self.goals if g.get('_source') in _PN2_SRC and g.get('type') != want}
        if drop:
            self.goals = [g for g in self.goals if id(g) not in drop]
            s.st['pn2_goal_drop'] += len(drop)
        if want is None:
            return
        mine = next((g for g in self.goals if g.get('_source') in _PN2_SRC and g.get('type') == want), None)
        if mine is not None:
            mine['age'] = 0
            return
        if any(g.get('type') == want for g in self.goals):
            s.st['pn2_goal_have_' + want] += 1
            return
        if Config.PN3_ENABLED and Config.PN3_PROPOSALS:
            _pn3_stance_proposal(self, s, p2, want, now)
            return
        if want == 'rest':
            self.goals.append({"type": "rest", "priority": float(Config.PN_V2_PRIO_REST), "target": None, "age": 0,
                               "persistence": int(Config.PN_V2_STANCE_PERSIST), "_source": "pn_defensive"})
            s.st['pn2_goal_rest'] += 1
        else:
            self.goals.append({"type": "explore", "priority": float(Config.PN_V2_PRIO_EXPLORE), "target": None, "age": 0,
                               "persistence": int(Config.PN_V2_STANCE_PERSIST), "_source": "pn_investigating"})
            s.st['pn2_goal_explore'] += 1

    def _generate_proposals(self, s, n, p2, now):
        _pn3_generate(self, s, n, _pn3_init(p2), now)

    def _pn3_on_place_heard(self, bk, ent, state, now):
        _pn3_on_heard(self, bk, ent, state, now)

    def _hc_on_hunt_heard(self, bk, ent, state, now):
        C = Config
        if not C.HUNT_ENABLE or len(ent) < 10 or self.role_type == 'feral' or not self.alive or not self.cells:
            return
        hc = bk.hunt_calls.get(ent[8])
        if hc is None or hc.sender_id == self.id:
            return
        rid, interp = int(self.id), state in ('recognized', 'partial')
        d = hc.delivery.get(rid)
        if d is None:
            d = hc.delivery[rid] = {'t': int(now), 'interpreted': bool(interp), 'accepted': False}
            bk.hc['delivered_pairs'] += 1
        if not interp or d['accepted']:
            return
        s = _fch(self.world)
        if hc.confidence < float(C.HUNT_MIN_CONF) or now - hc.observed_at > int(C.HUNT_TTL):
            bk.hc['rejected_stale_or_weak'] += 1
            return
        W = C.WORLD_SIZE
        cx, cy = self.get_center()
        if _scream_dist((int(cx) % W, int(cy) % W), (hc.fx, hc.fy), W) > float(C.HUNT_MAX_DIST):
            bk.hc['rejected_far'] += 1
            return
        cur = getattr(self, '_hunt', None)
        if cur is not None and cur['obs_t'] >= hc.observed_at:
            return
        self._hunt = {'fx': hc.fx, 'fy': hc.fy, 'obs_t': int(hc.observed_at), 'src': int(hc.sender_id), 'conf': float(hc.confidence),
                      'last_c': None, 'stuck': 0}
        d['accepted'] = True
        bk.hc['accepted'] += 1
        s.st['hunt_accepted'] += 1

    def _hunt_steer(self, now, kind, s):
        h = getattr(self, '_hunt', None)
        if h is None:
            return False
        C = Config
        W = C.WORLD_SIZE
        cx, cy = self.get_center()
        pos = (int(cx) % W, int(cy) % W)
        if now - h['obs_t'] > int(C.HUNT_TTL):
            self._hunt = None; s.st['hunt_expired'] += 1; return False
        if _scream_dist(pos, (h['fx'], h['fy']), W) <= float(C.HUNT_STOP_DIST):
            self._hunt = None; s.st['hunt_arrived'] += 1; return False
        if h.get('steered') and h['last_c'] == pos:
            h['stuck'] += 1
            if h['stuck'] >= int(C.HUNT_STUCK):
                self._hunt = None; s.st['hunt_stuck'] += 1; return False
        elif h['last_c'] != pos:
            h['stuck'] = 0
        h['last_c'] = pos
        h['steered'] = False
        if kind == 'flee':
            s.st['hunt_flee_override'] += 1
            return False
        sx, sy = _scream_nearest_step(pos, (h['fx'], h['fy']), W)
        if (sx, sy) != (0, 0):
            self._dir_momentum = (float(sx), float(sy))
            h['steered'] = True
            s.st['hunt_steps'] += 1
            return True
        return False

    def update_model(self, field, lineage_counts=None, given_counter=None, t=None, witness=None):
        res = super().update_model(field, lineage_counts, given_counter, t, witness)
        self.nav_motion_err = 0.0
        if self.spatial_pred_error > 0.0:
            v = float(self.spatial_pred_error) * float(Config.NAV_SPATIAL_HOLD)
            self.spatial_pred_error = v if v > 1e-3 else 0.0
        try:
            self._nav_tick(field, t)
        except Exception as e:
            _fch_err(_fch(self.world) if self.world is not None else None, 'nav', e)
        return res

    def move(self, field, t):
        if (not Config.NAV_ENABLE or _FCH_ABL['nav_off'] or self.role_type == 'feral' or not self.alive or not self.cells
                or self.world is None):
            return super().move(field, t)
        n = getattr(self, '_nav', None)
        if n is None:
            return super().move(field, t)
        now = int(self.world.age)
        cmd = n['cmd']
        fresh = cmd is not None and now - cmd['t'] <= int(Config.NAV_SENSE_EVERY) + 1
        mobf = max(0.1, 1.0 - float(Config.NAV_FREEZE_GAIN) * n['dis'])
        hunting = False
        if getattr(self, '_hunt', None) is not None:
            if not Config.HUNT_ENABLE:
                self._hunt = None
            else:
                try:
                    hunting = bool(self._hunt_steer(now, (cmd['kind'] if fresh else None), _fch(self.world)))
                except Exception as e:
                    _fch_err(_fch(self.world), 'hunt_steer', e)
        if fresh and cmd['h'] is not None and not hunting:
            mx, my = getattr(self, '_dir_momentum', (0.0, 0.0))
            b = min(1.0, (0.4 + 0.6 * cmd['w']) * float(Config.NAV_STEER_BLEND))
            self._dir_momentum = ((1.0 - b) * mx + b * cmd['h'][0], (1.0 - b) * my + b * cmd['h'][1])
            mobf *= cmd['mob']
            _fch(self.world).st['steer_' + str(cmd['kind'])] += 1
        sprint_p, jump_p = 0.0, 0.0
        if (getattr(Config, 'SPRINT_ENABLE', False) and self.role_type != 'feral' and fresh and cmd['kind'] == 'flee'):
            fear = min(1.0, max(float(cmd['D']) / float(Config.SPRINT_FEAR_REF), float(getattr(self, 'alarm_level', 0.0))))
            if fear >= float(Config.SPRINT_FEAR_MIN):
                _sp = float(self._fch_genes().get('sprint', 0.5))
                _fs = _fch(self.world)
                if self._consume_action('sprint') is None:
                    _fs.st['sprint_unpaid'] += 1
                else:
                    sprint_p = min(0.95, _sp * fear * float(Config.SPRINT_GAIN))
                    jump_p = min(0.9, _sp * fear * float(Config.SPRINT_JUMP))
                    _fs.st['sprint_run'] += 1
        rage_p, lunge_p = 0.0, 0.0
        if getattr(Config, 'RAGE_ENABLE', False) and hunting and sprint_p <= 0.0 and self.role_type != 'feral':
            _al = float(getattr(self, 'alarm_level', 0.0))
            _fs = _fch(self.world)
            if _al >= float(Config.RAGE_FEAR_MAX):
                _fs.st['rage_blocked_fear'] += 1
            else:
                _h = self._hunt
                _rage = max(0.0, min(1.0, float(_h['conf']))) * (1.0 - _al / float(Config.RAGE_FEAR_MAX))
                if _rage >= float(Config.RAGE_MIN):
                    _rg = float(self._fch_genes().get('rage', 0.5))
                    if self._consume_action('rage') is None:
                        _fs.st['rage_unpaid'] += 1
                    else:
                        rage_p = min(0.9, _rg * _rage * float(Config.RAGE_GAIN))
                        lunge_p = min(0.8, _rg * _rage * float(Config.RAGE_LUNGE))
                        _fs.st['rage_charge'] += 1
        if getattr(Config, 'MOVE_BOOST_ENABLE', False) and self.role_type != 'feral':
            mobf *= float(Config.MOVE_GAIN)
        if abs(mobf - 1.0) < 1e-3 and not hunting and sprint_p <= 0.0 and rage_p <= 0.0:
            return super().move(field, t)
        old = self.genome.get('mobility', 0.1)
        _new = min(0.6, max(0.0, old * mobf))
        if hunting and getattr(Config, 'MOVE_BOOST_ENABLE', False):
            _new = max(_new, float(Config.HUNT_MOB_FLOOR))
        if sprint_p > 0.0:
            _new = max(_new, sprint_p)
        if rage_p > 0.0:
            _new = max(_new, rage_p)
        self.genome['mobility'] = _new
        _soft = (getattr(Config, 'MOVE_SOFT_ENABLE', False) and (sprint_p > 0.0 or rage_p > 0.0))
        if _soft:
            self._move_soft_thr = float(Config.MOVE_SOFT_THR)
            self._move_soft_saved = False
            _fch(self.world).st['soft_try'] += 1
        try:
            # Sprint/rage leap uses the same checked body-shift primitive as ferals.
            # A blocked leap falls through to the ordinary movement controller.
            if (sprint_p > 0.0 or rage_p > 0.0) and hasattr(self, '_try_shift_cells'):
                dx, dy = getattr(self, '_dir_momentum', (0.0, 0.0))
                norm = float(np.hypot(float(dx), float(dy)))
                if norm > 1e-9:
                    leap = 6
                    # Normalize smoothed momentum so its magnitude does not shorten the leap.
                    step_x = int(round((float(dx) / norm) * leap))
                    step_y = int(round((float(dy) / norm) * leap))
                    if step_x != 0 or step_y != 0:
                        if self._try_shift_cells(field, step_x, step_y, t):
                            _fch(self.world).st['sprint_leap'] = _fch(self.world).st.get('sprint_leap', 0) + 1
                            self._log_event('sprint_leap', dist=leap,
                                            action='sprint' if sprint_p > 0.0 else 'rage')
                            return True
            moved = super().move(field, t)
            if _soft and self._move_soft_saved:
                _fch(self.world).st['soft_saved'] += 1; self._move_soft_saved = False
            if jump_p > 0.0 and phi_hash(self.id, t, 31415) < jump_p:
                self.genome['mobility'] = 0.6
                if super().move(field, t + 7919):
                    _fch(self.world).st['sprint_jump'] += 1
                    moved = True
                if _soft and self._move_soft_saved:
                    _fch(self.world).st['soft_saved'] += 1; self._move_soft_saved = False
            if lunge_p > 0.0 and phi_hash(self.id, t, 27183) < lunge_p:
                self.genome['mobility'] = 0.6
                if super().move(field, t + 6007):
                    _fch(self.world).st['rage_lunge'] += 1
                    moved = True
                if _soft and self._move_soft_saved:
                    _fch(self.world).st['soft_saved'] += 1; self._move_soft_saved = False
            return moved
        finally:
            self.genome['mobility'] = old
            self._move_soft_thr = None

    def generate_goals(self, field):
        super().generate_goals(field)
        if (self.world is None or not self.alive or not self.cells or field is None or self.role_type == 'feral'):
            return
        try:
            self._fch_goals(field)
        except Exception as e:
            _fch_err(_fch(self.world), 'goals', e)
        if Config.PN3_ENABLED:
            try:
                _pn3_step(self, field)
            except Exception as e:
                _fch_err(_fch(self.world), 'pn3', e)

    def _fch_goals(self, field):
        s = _fch(self.world)
        n = getattr(self, '_nav', None)
        now = int(self.world.age)
        W = Config.WORLD_SIZE
        if n is None:
            return
        cmd = n['cmd']
        fresh = cmd is not None and now - cmd['t'] <= int(Config.NAV_SENSE_EVERY) + 1
        if Config.NAV_ENABLE and not _FCH_ABL['nav_off']:
            for g in self.goals:
                if g.get('_source') != 'scream_response':
                    continue
                if fresh and cmd['ua'] is not None and cmd['A'] >= float(Config.NAV_ALARM_MIN):
                    cx, cy = self.get_center()
                    d_est = min(8.0, max(1.0, 6.0 * (1.0 - min(cmd['A'], 0.8) / 0.8)))
                    tgt = (int(round(cx + cmd['ua'][0] * d_est)) % W, int(round(cy + cmd['ua'][1] * d_est)) % W)
                    g['target'] = tgt
                    g['_nav_est'] = True
                    self._scream_target = tgt
                    s.st['scream_est'] += 1
                elif not g.get('_nav_est'):
                    g['target'] = None
                    g['_nav_est'] = True
                    self._scream_target = None
                    s.st['scream_est_none'] += 1
            if (Config.NAV_REST_GOAL and n['dis'] > float(Config.NAV_REST_DIS)
                    and not any(g.get('type') == 'rest' for g in self.goals)):
                self.goals.append({"type": "rest", "priority": 1.2 + n['dis'], "target": None, "age": 0, "persistence": 8,
                                   "_source": "nav_reorient"})
                s.st['rest_goal'] += 1
                if now - n['last_rest_log'] >= 30:
                    n['last_rest_log'] = now
                    self._log_event("nav_disoriented", dis=round(n['dis'], 3))
            if Config.PN_V2_ENABLED and Config.PN_V2_APPLY_STANCE:
                self._pn2_apply_goal(n, now, s)
        if not fresh or cmd['D'] < float(Config.NAV_DANGER_MIN) or now - n['D_t'] > int(Config.NAV_SENSE_EVERY) + 1:
            return
        D, genes = cmd['D'], self._fch_genes()
        if (Config.FCH_ENABLE and Config.ENABLE_MORSE_COMMUNICATION and not _FCH_ABL['morse_threat_send_off']):
            p = float(Config.FCH_MORSE_SEND) * genes['morse'] * min(1.0, D / 0.2)
            if phi_hash(self.id, now, 8811) < p:
                s.st['morse_threat_try'] += 1
                if self._send_morse(None, field):
                    s.st['morse_threat_ok'] += 1
        if Config.FCH_ENABLE and not _FCH_ABL['appease_off']:
            p = genes['coping'] * min(1.0, D / 0.15) * float(Config.FCH_APPEASE_RATE)
            if phi_hash(self.id, now, 8812) < p:
                self._fch_appease(field, now)
        if Config.HUNT_ENABLE and Config.FCH_ENABLE and cmd.get('uf') is not None and hasattr(self, '_sig_hunt_send'):
            if now - int(n.get('hunt_last_t', -9999)) >= int(Config.HUNT_SEND_COOLDOWN):
                p = float(Config.HUNT_SEND) * genes['morse'] * min(1.0, D / 0.2)
                if phi_hash(self.id, now, 8821) < p:
                    cx, cy = self.get_center()
                    d_est = min(8.0, max(1.0, 6.0 * (1.0 - min(D, 0.8) / 0.8)))
                    fx = int(round(cx + cmd['uf'][0] * d_est)) % W
                    fy = int(round(cy + cmd['uf'][1] * d_est)) % W
                    s.st['hunt_try'] += 1
                    hc, why = self._sig_hunt_send(field, now, fx, fy, int(cmd['t']), min(1.0, D / 0.3))
                    if hc is None:
                        s.st['hunt_blocked_' + why] += 1
                    else:
                        n['hunt_last_t'] = now
                        s.st['hunt_sent'] += 1

    def _fch_appease(self, field, now):
        s = _fch(self.world)
        n = self._nav_st()
        if now - n['last_app'] < int(Config.FCH_APPEASE_COOLDOWN):
            return False
        self._sig_init()
        peace = _fch_peace_idx()
        elig = []
        for f in self.signal_forms:
            spec = f['spectrum']
            tot = float(sum(spec.values())) or 1.0
            if sum(float(spec.get(c, 0.0)) for c in peace) / tot < 0.6:
                continue
            elig.append(f)
        best = elig[int(phi_hash(self.id, now, 8815) * len(elig)) % len(elig)] if elig else None
        if best is None or phi_hash(self.id, now, 8813) < 0.25:
            spec = {CH['signal_curiosity']: 0.6, CH['signal_invitation']: 0.6}
            for ch in list(spec):
                spec[ch] = max(0.05, spec[ch] + (phi_hash(self.id, now + ch, 8814) - 0.5) * 0.3)
            spec = _sg_normalize(spec)
            if spec:
                f = self._sig_new_form(spec, 'appease')
                if self._sig_add(f, 'appease'):
                    best = f
        if best is None:
            s.st['appease_no_form'] += 1
            return False
        if not self._sig_emit(field, best, now, _sg_ctx(self)):
            s.st['appease_blocked'] += 1
            return False
        n['last_app'] = now
        n['disp'].append({'t': now, 'form': best})
        s.st['appease_emit'] += 1
        return True

    def _fch_credit_forms(self, now, v):
        n = getattr(self, '_nav', None)
        if not n:
            return
        for d in n['disp']:
            if now - d['t'] <= 12:
                fx = d['form'].setdefault('fx', {'v': 0.0, 'n': 0})
                fx['n'] += 1
                fx['v'] += 0.2 * (float(v) - fx['v'])

    def update_feral_fury(self, field):
        super().update_feral_fury(field)
        if (Config.FCH_ENABLE and not _FCH_ABL['tame_off'] and self.role_type == 'feral' and self.alive and self.cells
                and self.world is not None and field is not None):
            try:
                self._fch_tame_update(field)
            except Exception as e:
                _fch_err(_fch(self.world), 'tame', e)

    def _fch_tame_update(self, field):
        W = Config.WORLD_SIZE
        R = int(Config.FCH_TAME_RADIUS)
        cx, cy = self.get_center()
        cx, cy = int(cx) % W, int(cy) % W
        idx = np.arange(-R, R + 1)
        owners = np.unique(field[np.ix_((cx + idx) % W, (cy + idx) % W, [CH['owner']])])
        tame = self._tame
        dec = float(Config.FCH_TAME_DECAY)
        for k in list(tame):
            v = tame[k] * dec
            if abs(v) < 0.01:
                del tame[k]
            else:
                tame[k] = v
        peace, lr, tmax = _fch_peace_idx(), float(Config.FCH_TAME_LR), float(Config.FCH_TAME_MAX)
        pd = self.world.pattern_dict
        for oid in owners:
            oid = int(oid)
            if oid == 0 or oid == self.id:
                continue
            p = pd.get(oid)
            if p is None or not p.alive or not p.cells or p.role_type == 'feral':
                continue
            cells = sorted(p.cells)[:16]
            xs = np.fromiter((c[0] for c in cells), dtype=np.int64, count=len(cells))
            ys = np.fromiter((c[1] for c in cells), dtype=np.int64, count=len(cells))
            pe = float(field[xs, ys][:, list(peace)].sum(axis=1).mean())
            di = float(field[xs, ys, CH['signal_grief']].mean())
            sx = _fch(self.world)
            pe = max(0.0, pe - sx.bg_peace - 0.05)
            di = max(0.0, di - 0.1)
            sx.st['pe_bin%d' % (0 if pe < 0.05 else 1 if pe < 0.1 else 2 if pe < 0.2 else 3 if pe < 0.4 else 4)] += 1
            d = lr * min(1.0, pe / 0.4) - 0.5 * lr * min(1.0, di / 0.4)
            if d != 0.0:
                tame[oid] = float(min(tmax, max(-0.5, tame.get(oid, 0.0) + d)))

    def _fch_cues(self, prey, field, world):
        cells = sorted(prey.cells)[:12]
        xs = np.fromiter((c[0] for c in cells), dtype=np.int64, count=len(cells))
        ys = np.fromiter((c[1] for c in cells), dtype=np.int64, count=len(cells))
        m = _fch_morse_ch()
        ex = 0.0
        if m < field.shape[2]:
            ex = float(field[xs, ys, m].mean()) - _fch(world).bg_morse - float(getattr(Config, 'MORSE_NOISE_SCALE', 0.02))
        al = float(field[xs, ys, CH['signal_alarm']].mean())
        return max(0.0, ex), float(self._tame.get(prey.id, 0.0)), al

    def _fch_deter(self, prey, field, world):
        s = _fch(world)
        now = int(world.age)
        ex, tv, _al = self._fch_cues(prey, field, world)
        pm = pt = 0.0
        if not _FCH_ABL['morse_deter_off'] and ex > float(Config.FCH_MORSE_MIN_EXCESS):
            pm = float(Config.FCH_MORSE_MAX) * (1.0 - math.exp(-ex / float(Config.FCH_MORSE_SCALE)))
        if not _FCH_ABL['tame_off'] and tv > 0.0:
            pt = float(Config.FCH_TAME_DETER) * min(1.0, tv / float(Config.FCH_TAME_MAX))
        p_veto = 1.0 - (1.0 - pm) * (1.0 - pt)
        if p_veto <= 0.0 or phi_hash(self.id, now, 8803) >= p_veto:
            return False
        if self._fch_veto_t != now:
            self._fch_veto_t = now
            s.veto['morse' if pm >= pt else 'tame'] += 1
        bo = getattr(self, '_feral_backoff', None)
        if bo is None:
            bo = {}
            self._feral_backoff = bo
        bo[prey.id] = now + int(Config.FCH_BACKOFF)
        if hasattr(prey, '_fch_credit_forms'):
            prey._fch_credit_forms(now, 1.0)
        return True

    def _find_prey(self, field, world, min_cells=5, radius=20):
        prey, direction, dist = super()._find_prey(field, world, min_cells=min_cells, radius=radius)
        if (prey is None or not Config.FCH_ENABLE or field is None or world is None or self.role_type != 'feral'):
            return prey, direction, dist
        try:
            if self._fch_deter(prey, field, world):
                return None, None, None
        except Exception as e:
            _fch_err(_fch(world), 'deter', e)
        return prey, direction, dist

    def feral_execute(self, other, field):
        if (self.role_type != 'feral' or not Config.FCH_ENABLE or field is None or self.world is None
                or other is None or not other.alive):
            return super().feral_execute(other, field)
        w = self.world
        prof = None
        try:
            ex, tv, al = self._fch_cues(other, field, w)
            prof = f"M{int(ex > float(Config.FCH_MORSE_MIN_EXCESS))}T{int(tv >= 0.15)}A{int(al >= 0.2)}"
        except Exception as e:
            _fch_err(_fch(w), 'profile', e)
        r0 = w._feral_stats['rolled']
        res = super().feral_execute(other, field)
        if prof is not None:
            s = _fch(w)
            s.prof_exp[prof] += 1
            if w._feral_stats['rolled'] > r0:
                s.prof_strike[prof] += 1
                if not other.alive:
                    s.prof_kill[prof] += 1
                elif hasattr(other, '_fch_credit_forms'):
                    other._fch_credit_forms(int(w.age), -0.5)
        return res


def _fch_gene_stats(eng):
    alive = [p for p in eng.patterns if p.alive and p.role_type != 'feral' and getattr(p, '_fch_gene', None)]
    out = {}
    for k, *_r in _FCH_GENES:
        v = np.array([p._fch_gene[k] for p in alive], dtype=np.float64)
        out[k] = {'mean': float(v.mean()), 'std': float(v.std()), 'p10': float(np.percentile(v, 10)),
                  'p90': float(np.percentile(v, 90))} if len(v) else {'mean': 0.0, 'std': 0.0, 'p10': 0.0, 'p90': 0.0}
    out['n'] = len(alive)
    return out


def _fch_summary(eng, s):
    alive = [p for p in eng.patterns if p.alive and p.role_type != 'feral']
    dis = [float(p._nav['dis']) for p in alive if getattr(p, '_nav', None)]
    anchors = [len(p._nav['anch']) for p in alive if getattr(p, '_nav', None)]
    pn = [p._nav for p in alive if getattr(p, '_nav', None)]
    pn_nodes = [len(q['places']) for q in pn]
    pn_vis = [len(pp['visits']) for q in pn for pp in q['places']]
    pn_edges = [sum(len(pp['transitions']) for pp in q['places']) for q in pn]
    pn_known = [1.0 if q['current_place'] is not None else 0.0 for q in pn]
    tames = [v for p in eng.patterns if p.alive and p.role_type == 'feral' for v in getattr(p, '_tame', {}).values()]
    fx = [f['fx']['v'] for p in alive for f in getattr(p, 'signal_forms', ()) if f.get('fx') and f['fx']['n'] > 0]
    prof = {k: {'exposed': int(s.prof_exp[k]), 'strikes': int(s.prof_strike[k]), 'kills': int(s.prof_kill[k])}
            for k in sorted(set(s.prof_exp) | set(s.prof_strike) | set(s.prof_kill))}
    return {
        'enabled': {'NAV': bool(Config.NAV_ENABLE), 'FCH': bool(Config.FCH_ENABLE), 'ablations': dict(_FCH_ABL)},
        'counters': {k: int(v) for k, v in sorted(s.st.items())},
        'veto': {k: int(v) for k, v in s.veto.items()},
        'profile': prof,
        'genes': _fch_gene_stats(eng),
        'nav': {'dis_mean': float(np.mean(dis)) if dis else 0.0, 'dis_p90': float(np.percentile(dis, 90)) if dis else 0.0,
                'anchor_mean': float(np.mean(anchors)) if anchors else 0.0,
                'anchor_err_mean': float(s.st['anchor_err']) / max(1, int(s.st['anchor_hit'])),
                'place_nodes': bool(Config.NAV_PLACE_NODES),
                'places_mean': float(np.mean(pn_nodes)) if pn_nodes else 0.0, 'places_max': int(max(pn_nodes)) if pn_nodes else 0,
                'visits_mean': float(np.mean(pn_vis)) if pn_vis else 0.0, 'edges_mean': float(np.mean(pn_edges)) if pn_edges else 0.0,
                'known_frac': float(np.mean(pn_known)) if pn_known else 0.0,
                'place_err_pi_mean': float(s.st['place_err_pi_milli']) / 1000.0 / max(1, int(s.st['place_hit'])),
                'place_err_fp_mean': float(s.st['place_err_fp_milli']) / 1000.0 / max(1, int(s.st['place_hit'])),
                'spatial_err_events': int(s.st['spatial_err_event']),
                'spatial_err_mean': float(s.st['spatial_err_milli']) / 1000.0 / max(1, int(s.st['spatial_err_event'])),
                'spatial_err_live_mean': float(np.mean([float(getattr(p, 'spatial_pred_error', 0.0)) for p in alive])) if alive else 0.0,
                'place_conf_mean': float(np.mean([float(getattr(p, 'place_confidence', 0.5)) for p in alive])) if alive else 0.0},
        'place_diag': (_pn_diag(eng) if Config.NAV_PLACE_DIAG else None),
        'pn2': _pn2_summary_safe(eng),
        'pn3': _pn3_summary_safe(eng, s),
        'tame': {'pairs': len(tames), 'pos_mean': float(np.mean([v for v in tames if v > 0])) if any(v > 0 for v in tames) else 0.0,
                 'max': float(max(tames)) if tames else 0.0},
        'appease_fx': {'forms': len(fx), 'mean': float(np.mean(fx)) if fx else 0.0},
        'bg_morse': float(s.bg_morse), 'bg_peace': float(s.bg_peace), 'hist': list(s.hist), 'error': s.err,
    }


def _hunt_line(eng, c):
    try:
        bk = _sg_book(eng); hc = getattr(bk, 'hc', {})
        blk = {k[len('hunt_blocked_'):]: v for k, v in c.items() if str(k).startswith('hunt_blocked_')}
        return (f"           HUNT enable={Config.HUNT_ENABLE} send: try={c.get('hunt_try', 0)} sent={c.get('hunt_sent', 0)} blocked={blk} | "
                f"deliver: sent_book={hc.get('sent', 0)} pairs={hc.get('delivered_pairs', 0)} undelivered={hc.get('undelivered', 0)} live={len(getattr(bk, 'hunt_calls', {}))} | "
                f"accept={hc.get('accepted', 0)} rej(stale/weak)={hc.get('rejected_stale_or_weak', 0)} rej(far)={hc.get('rejected_far', 0)} | "
                f"move: steps={c.get('hunt_steps', 0)} flee_override={c.get('hunt_flee_override', 0)} arrived={c.get('hunt_arrived', 0)} "
                f"stuck={c.get('hunt_stuck', 0)} expired={c.get('hunt_expired', 0)} | "
                f"FEAR sprint={c.get('sprint_run', 0)} jump={c.get('sprint_jump', 0)} unpaid={c.get('sprint_unpaid', 0)} | "
                f"RAGE charge={c.get('rage_charge', 0)} lunge={c.get('rage_lunge', 0)} unpaid={c.get('rage_unpaid', 0)} blocked_by_fear={c.get('rage_blocked_fear', 0)} | "
                f"SOFT try={c.get('soft_try', 0)} saved={c.get('soft_saved', 0)} (enable={getattr(Config, 'MOVE_SOFT_ENABLE', False)})\n")
    except Exception as e:
        return f"           HUNT line err: {e!r}\n"


def _fch_line(eng, s, now):
    S = _fch_summary(eng, s)
    c, g = S['counters'], S['genes']
    prof = " ".join(f"{k}:{v['exposed']}/{v['strikes']}/{v['kills']}" for k, v in S['profile'].items())
    return (f"\n[FCH t={now}] NAV senses={c.get('senses', 0)} dis_mean={S['nav']['dis_mean']:.3f} p90={S['nav']['dis_p90']:.3f} "
            f"disorient={c.get('disorient_events', 0)} rest_goal={c.get('rest_goal', 0)} "
            f"steer flee={c.get('steer_flee', 0)} mob={c.get('steer_mob', 0)} anchors new={c.get('anchor_new', 0)} hit={c.get('anchor_hit', 0)} gate_rej={c.get('anchor_gate_rej', 0)} err={S['nav']['anchor_err_mean']:.1f} "
            f"scream_est={c.get('scream_est', 0)}/none={c.get('scream_est_none', 0)}\n"
            f"           SPATIAL events={c.get('spatial_err_event', 0)} lost/unexpected={c.get('spatial_err_lost', 0)}/{c.get('spatial_err_unexpected', 0)} "
            f"err_mean={S['nav']['spatial_err_mean']:.3f} live={S['nav']['spatial_err_live_mean']:.4f} conf_mean={S['nav']['place_conf_mean']:.3f} "
            f"motion_pulse={c.get('motion_err_pulse', 0)}\n"
            f"           PLACE nodes/agent={S['nav']['places_mean']:.1f}(max {S['nav']['places_max']}) visits/node={S['nav']['visits_mean']:.1f} edges/agent={S['nav']['edges_mean']:.1f} known={S['nav']['known_frac']:.2f} | "
            f"new={c.get('place_new', 0)} hit={c.get('place_hit', 0)} cont={c.get('place_cont', 0)} ambig={c.get('place_ambig', 0)} uncertain={c.get('place_uncertain', 0)} "
            f"pending={c.get('place_pending', 0)} left={c.get('place_left', 0)} gate_rej={c.get('place_gate_rej', 0)} evict={c.get('place_evict', 0)} edge={c.get('place_edge', 0)} "
            f"err_pi={S['nav']['place_err_pi_mean']:.1f} err_fp={S['nav']['place_err_fp_mean']:.2f}\n"
            f"           PNDIAG acc={c.get('diag_acc', 0)} stay={c.get('diag_rec_stay', 0)} lost_obs={c.get('pend_lost_obs', 0)} dup_tight={c.get('diag_new_dup_tight', 0)} "
            f"pi in/soft/zero={c.get('diag_pi_in', 0)}/{c.get('diag_pi_soft', 0)}/{c.get('diag_pi_zero', 0)}\n"
            f"{_pn2_line(c)}"
            f"{_pn3_line(c)}"
            f"{_hunt_line(eng, c)}"
            f"           CH veto morse={S['veto'].get('morse', 0)} tame={S['veto'].get('tame', 0)} | "
            f"morse_threat={c.get('morse_threat_ok', 0)}/{c.get('morse_threat_try', 0)} | "
            f"appease={c.get('appease_emit', 0)} (blocked={c.get('appease_blocked', 0)}) | "
            f"tame pairs={S['tame']['pairs']} max={S['tame']['max']:.2f} | appease_fx n={S['appease_fx']['forms']} mean={S['appease_fx']['mean']:.2f}\n"
            f"           GENES n={g['n']} flee={g['flee']['mean']:.3f} mob={g['mob']['mean']:.3f} "
            f"coping={g['coping']['mean']:.3f} morse={g['morse']['mean']:.3f} sprint={g['sprint']['mean']:.3f} rage={g['rage']['mean']:.3f} | профили exposed/strikes/kills: {prof or '-'}"
            + (f"\n           ⚠️ error: {S['error']}" if S['error'] else ""))


class FchEngineMixin:
    def _post_semantic_step(self, t):
        super()._post_semantic_step(t)
        if not (Config.NAV_ENABLE or Config.FCH_ENABLE):
            return
        s = _fch(self)
        _PN_LAST['eng'] = self
        try:
            m = _fch_morse_ch()
            if self.field.shape[2] > m:
                s.bg_morse = float(self.field[:, :, m].mean())
            xs_, ys_ = [], []
            for p in self.patterns:
                if p.alive and p.cells and p.role_type != 'feral':
                    cs = sorted(p.cells)[:8]
                    xs_ += [c[0] for c in cs]
                    ys_ += [c[1] for c in cs]
            if xs_:
                s.bg_peace = float(self.field[np.array(xs_), np.array(ys_)][:, list(_fch_peace_idx())].sum(axis=1).mean())
            now = int(self.age)
            if Config.PN3_ENABLED:
                _pn3_world_tick(self, now)
            if now % 50 == 0 and now > 0:
                g_ = _fch_gene_stats(self)
                s.hist.append({'t': now, 'n': g_['n'], 'genes': {k: g_[k]['mean'] for k, *_r in _FCH_GENES},
                               'veto': dict(s.veto), 'strikes': int(sum(s.prof_strike.values())),
                               'kills': int(sum(s.prof_kill.values())), 'bg_peace': s.bg_peace, 'pn2': _pn2_hist(self)})
                print(_fch_line(self, s, now), flush=True)
                if Config.NAV_PLACE_DIAG and now % 100 == 0:
                    _pn_diag_report(self)
        except Exception as e:
            _fch_err(s, 'tick', e)

    def audit_summary(self):
        S = super().audit_summary()
        try:
            S['fch'] = _fch_summary(self, _fch(self))
        except Exception as e:
            _fch_err(_fch(self), 'summary', e)
        return S


Pattern = extend_class(Pattern, NavPatternMixin)
EvolutionEngine = extend_class(EvolutionEngine, FchEngineMixin)
LedgerEngine = extend_class(LedgerEngine, FchEngineMixin)
RichEngine = extend_class(RichEngine, FchEngineMixin)
for _cls in (Pattern, RichEngine):
    assert any(k.__name__ in ("NavPatternMixin", "FchEngineMixin") for k in _cls.__mro__), f"слой NAV+FCH не подключён к {_cls.__name__}"

TAG_SUFFIX = "v38_meat2"
try:
    _MANIFEST["layers"] = list(_MANIFEST.get("layers", [])) + ["NavPatternMixin", "FchEngineMixin"]
    _MANIFEST["nav_fch"] = {k: getattr(Config, k) for k in _FCH_CFG}
except Exception:
    pass
Config.HUNT_STOP_DIST = 3.0
Config.MOVE_BOOST_ENABLE, Config.MOVE_GAIN, Config.HUNT_MOB_FLOOR = True, 1.8, 0.5
Config.SPRINT_ENABLE, Config.SPRINT_FEAR_REF, Config.SPRINT_FEAR_MIN = True, 0.3, 0.25
Config.SPRINT_GAIN, Config.SPRINT_JUMP = 1.2, 0.8
ACTION_COSTS['sprint'] = 0.004
Config.RAGE_ENABLE, Config.RAGE_FEAR_MAX, Config.RAGE_MIN = True, 0.5, 0.1
Config.RAGE_GAIN, Config.RAGE_LUNGE = 1.0, 0.5
ACTION_COSTS['rage'] = 0.003
Config.MOVE_SOFT_ENABLE, Config.MOVE_SOFT_THR = True, -0.5
SLACK = {'SLACK_BARRIER': True, 'SLACK_COST': True, 'SLACK_CURIOSITY_U': True}
Config.SLACK_CURIOSITY_U = bool(SLACK['SLACK_CURIOSITY_U'])
Config.SLACK_CUR_LOW, Config.SLACK_CUR_HIGH = 0.3, 0.6
Config.SLACK_CUR_PRED_MAX, Config.SLACK_CUR_GRIEF_MAX, Config.SLACK_CUR_AGE_MIN = 0.85, 0.5, 40
if SLACK['SLACK_BARRIER']:
    Config.INTENT_SWITCH_MULT, Config.INTENT_SWITCH_MULT_MIN, Config.INTENT_SWITCH_COOLDOWN = 1.15, 1.02, 6
if SLACK['SLACK_COST']:
    ACTION_COSTS['intent_switch'] = 0.003
print(f"SLACK: {SLACK} | mult={Config.INTENT_SWITCH_MULT}/{Config.INTENT_SWITCH_MULT_MIN} cd={Config.INTENT_SWITCH_COOLDOWN} cost={ACTION_COSTS['intent_switch']} curiosity_U={Config.SLACK_CURIOSITY_U}")
print("=== NAV + FERAL CHANNELS v1.1 подключён (чувства→модель→руль; PlaceNode: места как кластеры визитов; Морзе и SIG-приручение под феролами; гены flee/mob/coping/morse) ===")
print(f"NAV={Config.NAV_ENABLE} FCH={Config.FCH_ENABLE} PLACE_NODES={Config.NAV_PLACE_NODES} TAG_SUFFIX={TAG_SUFFIX}  ablations={_FCH_ABL}")
print(f"PLACENODE v2: ENABLED={Config.PN_V2_ENABLED} APPLY_STANCE={Config.PN_V2_APPLY_STANCE} Q_BUDGET={Config.PN_V2_Q_BUDGET} Q_TTL={Config.PN_V2_Q_TTL}")
print(f"PN3 (A/B/C): ENABLED={Config.PN3_ENABLED} PROPOSALS={Config.PN3_PROPOSALS} PLACE_MSG={Config.PN3_PLACE_MSG} FEEDBACK={Config.PN3_FEEDBACK} THRESH={Config.PN3_THRESH}")


In [ ]:
# Ячейка 31
import math
import numpy as np
from collections import Counter, deque

_SOMA_DEFAULTS = {
    'SOMA_ENABLED': True, 'SOMA_APPLY': True, 'SOMA_EVERY': 5,
    'SOMA_TENSION': True, 'SOMA_TENSION_FRAG': 0.03, 'SOMA_TENSION_BRIDGE': 0.01, 'SOMA_TENSION_EXPOSE': 0.004,
    'SOMA_GUT': True, 'SOMA_GUT_GAIN': 0.03,
    'SOMA_ALARM_GAIN': 0.03,
    'SOMA_STANCE': True, 'SOMA_DISC_GAIN': 0.25,
    'SOMA_GOAL': True, 'SOMA_GOAL_PERSIST': 12,
    'SOMA_PN3_VETO': True, 'SOMA_PN3_PROPOSE': True,
    'SOMA_BRIDGE_THR': 0.12,
    'SOMA_EXPOSE_THR': 0.65, 'SOMA_ELONG_THR': 0.75,
    'SOMA_DESPAIR_FRAC': 0.25,
    'SOMA_MEMORY': True, 'SOMA_MEM_GAP': 20,
}
for _k, _v in _SOMA_DEFAULTS.items():
    if not hasattr(Config, _k):
        setattr(Config, _k, _v)

_SOMA_NB = ((0, 1), (1, 0), (0, -1), (-1, 0))
_SOMA_SHAPES = ('ball', 'rod', 'dumbbell', 'ring', 'split')


def _soma_measure(cells, W):
    cs = set((int(x), int(y)) for (x, y) in cells)
    n = len(cs)
    if n == 0:
        return None
    seen, comps = set(), []
    for st in sorted(cs):
        if st in seen:
            continue
        seen.add(st)
        dq = deque([(st, 0, 0)])
        pts = []
        while dq:
            (cx, cy), ux, uy = dq.popleft()
            pts.append((ux, uy))
            for dx, dy in _SOMA_NB:
                q = ((cx + dx) % W, (cy + dy) % W)
                if q in cs and q not in seen:
                    seen.add(q)
                    dq.append((q, ux + dx, uy + dy))
        comps.append(pts)
    C = len(comps)
    big = max(comps, key=len)
    elong, lam = 0.0, (0.0, 0.0)
    if len(big) >= 2:
        a = np.asarray(big, dtype=np.float64)
        cov = np.cov(a.T) if len(big) > 1 else np.zeros((2, 2))
        ev = np.sort(np.linalg.eigvalsh(np.atleast_2d(cov)))[::-1]
        l1, l2 = float(max(ev[0], 0.0)) + 1.0 / 12.0, float(max(ev[1], 0.0)) + 1.0 / 12.0
        elong = float(1.0 - math.sqrt(l2 / l1))
        lam = (l1, l2)
    bound = neck = thick = 0
    nbrs = {}
    for (x, y) in cs:
        lst = [((x + dx) % W, (y + dy) % W) for dx, dy in _SOMA_NB]
        inn = [q in cs for q in lst]
        nbrs[(x, y)] = [q for q, f in zip(lst, inn) if f]
        k = sum(inn)
        if k < 4:
            bound += 1
        if k >= 3:
            thick += 1
        if k == 2 and ((inn[0] and inn[2]) or (inn[1] and inn[3])):
            neck += 1
    E = F = 0
    for (x, y) in cs:
        r, d = ((x + 1) % W, y), (x, (y + 1) % W)
        if r in cs:
            E += 1
        if d in cs:
            E += 1
        if r in cs and d in cs and ((x + 1) % W, (y + 1) % W) in cs:
            F += 1
    holes = max(0, C - (n - E + F))
    art = 0
    if n >= 3:
        disc, low, parent, tmr = {}, {}, {}, 0
        for root in sorted(cs):
            if root in disc:
                continue
            disc[root] = low[root] = tmr
            tmr += 1
            parent[root] = None
            kids = 0
            stack = [(root, iter(nbrs[root]))]
            is_art = set()
            while stack:
                v, it = stack[-1]
                adv = False
                for w in it:
                    if w not in disc:
                        parent[w] = v
                        disc[w] = low[w] = tmr
                        tmr += 1
                        if v == root:
                            kids += 1
                        stack.append((w, iter(nbrs[w])))
                        adv = True
                        break
                    elif w != parent[v]:
                        low[v] = min(low[v], disc[w])
                if adv:
                    continue
                stack.pop()
                if stack:
                    u = stack[-1][0]
                    low[u] = min(low[u], low[v])
                    if u != root and low[v] >= disc[u]:
                        is_art.add(u)
            if kids > 1:
                is_art.add(root)
            art += len(is_art)
    thin = art / n
    valid = bool(C == 1 and holes == 0 and n >= 4)
    cx_ = min(1.0, 0.4 * holes + 0.2 * (C - 1) + 0.6 * thin)
    if C > 1:
        shape = 'split'
    elif holes > 0:
        shape = 'ring'
    elif art > 0 and elong >= 0.5 and thick / n >= 0.3:
        shape = 'dumbbell'
    elif elong >= 0.6:
        shape = 'rod'
    else:
        shape = 'ball'
    return {'fragment_count': int(C), 'largest_frac': round(len(big) / n, 4), 'elongation': round(elong, 4), 'elongation_valid': valid,
            'holes': int(holes), 'thin_bridge_risk': round(thin, 4), 'neck_candidate_ratio': round(neck / n, 4), 'articulation_cells': int(art),
            'boundary_exposure': round(bound / n, 4), 'topology_complexity': round(cx_, 4), 'shape': shape, 'n_cells': int(n)}


def _soma_stress(d):
    if not d:
        return 0.0
    C = Config
    s = 0.0
    if d['fragment_count'] > 1:
        s += 0.5 + 0.1 * min(3, d['fragment_count'] - 1)
    s += 0.25 * min(1.0, d['thin_bridge_risk'] / max(1e-6, 2.0 * float(C.SOMA_BRIDGE_THR)))
    s += 0.15 * max(0.0, (d['boundary_exposure'] - float(C.SOMA_EXPOSE_THR)) / max(1e-6, 1.0 - float(C.SOMA_EXPOSE_THR)))
    if d['holes'] > 0:
        s += 0.1
    return float(min(1.0, s))


def _soma_despair(agent):
    d = getattr(agent, '_soma_diag', None)
    if d and d['fragment_count'] > 1:
        return 'fragmented'
    w = getattr(agent, 'world', None)
    em = getattr(w, '_soma_e_mean', None) if w is not None else None
    try:
        if em is not None and em > 0 and float(agent.energy) < float(Config.SOMA_DESPAIR_FRAC) * float(em):
            return 'low_energy'
    except Exception:
        pass
    return None


# ═════════ SOMA observer v3 (read-only): эпизоды разрыва, судьба, выживаемость, эффект rest ═════════
# Правила: (1) в агента ничего не пишется — всё хранится в w._somaobs по id агента; (2) поведение эти данные не читает;
# (3) сбой сбора не глотается молча (_obs_fail → отчёт [E]); (4) причины — эвристика по измеренным признакам, не доказанная причинность.
_SOMAOBS_THIN_EDGES = (0.15, 0.30, 0.45)
_SOMAOBS_THIN_LBL = ('<0.15', '0.15-0.30', '0.30-0.45', '>=0.45')


def _somaobs_state(w):
    o = getattr(w, '_somaobs', None)
    if o is None:
        o = {'last': {}, 'open': {}, 'eps': [], 'started': 0, 'cause_kind': Counter(), 'cause_flag': Counter(),
             'cause_role': Counter(), 'fate': Counter(), 'mode': Counter(), 'haz': {}, 'hazf': {}, 'eff': {}}
        w._somaobs = o
    return o


def _somaobs_push(o, rec):
    o['eps'].append(rec)
    if len(o['eps']) > 5000:
        del o['eps'][:1000]


def _somaobs_measure(agent, w, d, now):
    """Вызывается из _soma_step сразу после того, как d записан в agent._soma_diag. Только чтение атрибутов агента."""
    o = _somaobs_state(w)
    C = Config
    aid = int(agent.id)
    role = str(getattr(agent, 'role_type', None) or 'other')
    energy = float(getattr(agent, 'energy', 0.0))
    thin = float(d['thin_bridge_risk'])
    fc = int(d['fragment_count'])
    n_cells = int(d['n_cells'])
    stress = float(d.get('stress', 0.0))
    tb = sum(1 for e in _SOMAOBS_THIN_EDGES if thin >= e)
    goals = getattr(agent, 'goals', None) or []
    has_rest = any(isinstance(g, dict) and g.get('type') == 'rest' for g in goals)
    prev = o['last'].get(aid)
    if prev is not None:
        dt = max(0, int(now) - prev['t'])
        o['haz'].setdefault((prev['role'], prev['tb']), [0, 0])[0] += dt
        o['hazf'].setdefault((prev['role'], 'frag' if prev['frag'] > 1 else 'whole'), [0, 0])[0] += dt
        ef = o['eff'].setdefault((bool(prev['rest']), prev['frag'] > 1), [0, 0.0, 0.0, 0.0])
        ef[0] += 1
        ef[1] += thin - prev['thin']
        ef[2] += stress - prev['stress']
        ef[3] += fc - prev['frag']
    ep = o['open'].get(aid)
    if fc > 1:
        if ep is None:
            em = getattr(w, '_soma_e_mean', None)
            low_e = bool(em is not None and em > 0 and energy < float(C.SOMA_DESPAIR_FRAC) * float(em))
            shock = float(getattr(agent, '_active_shock', 0.0) or 0.0)
            if prev is None:
                kind = 'no_prev'
            else:
                dn = n_cells - prev['n']
                kind = 'cells_lost' if dn < 0 else ('cells_grew' if dn > 0 else 'same_cells')
            o['started'] += 1
            o['cause_kind'][kind] += 1
            o['cause_role'][role] += 1
            if prev is not None and prev['thin'] > float(C.SOMA_BRIDGE_THR):
                o['cause_flag']['fragile_before'] += 1
            if shock >= 0.2:
                o['cause_flag']['shock>=0.2'] += 1
            if low_e:
                o['cause_flag']['low_energy'] += 1
            o['open'][aid] = {'t0': int(now), 'n0': n_cells, 'rest': bool(has_rest), 'maxfrag': fc, 'role': role}
        else:
            ep['rest'] = bool(ep['rest'] or has_rest)
            ep['maxfrag'] = max(ep['maxfrag'], fc)
    elif ep is not None:
        o['open'].pop(aid, None)
        dn = n_cells - ep['n0']
        mode = 'amputation' if dn < 0 else ('added_cells' if dn > 0 else 'reconnected')
        o['fate']['restored'] += 1
        o['mode'][mode] += 1
        _somaobs_push(o, (int(now) - ep['t0'], 'restored', mode, bool(ep['rest']), ep['role']))
    o['last'][aid] = {'t': int(now), 'frag': fc, 'thin': thin, 'stress': stress, 'n': n_cells, 'tb': tb,
                      'role': role, 'rest': bool(has_rest)}


def _somaobs_tick(eng, now):
    """Раз в тик: агенты, которых больше нет среди живых, закрывают эпизод как 'died_fragmented' и пишут смерть в выживаемость."""
    o = getattr(eng, '_somaobs', None)
    if not o or not o['last']:
        return
    alive = {int(p.id) for p in eng.patterns if p.alive}
    for a in [a for a in o['last'] if a not in alive]:
        prev = o['last'].pop(a)
        dt = max(0, int(now) - prev['t'])
        hz = o['haz'].setdefault((prev['role'], prev['tb']), [0, 0])
        hz[0] += dt
        hz[1] += 1
        hf = o['hazf'].setdefault((prev['role'], 'frag' if prev['frag'] > 1 else 'whole'), [0, 0])
        hf[0] += dt
        hf[1] += 1
        ep = o['open'].pop(a, None)
        if ep is not None:
            o['fate']['died_fragmented'] += 1
            _somaobs_push(o, (int(now) - ep['t0'], 'died', 'frag%d' % ep['maxfrag'], bool(ep['rest']), ep['role']))


def _somaobs_summary(eng):
    o = getattr(eng, '_somaobs', None)
    if not o:
        return {'enabled': False}
    return {'enabled': True, 'started': int(o['started']), 'open_now': len(o['open']),
            'fate': dict(o['fate']), 'cause_kind': dict(o['cause_kind']), 'cause_flag': dict(o['cause_flag']),
            'cause_role': dict(o['cause_role']), 'restore_mode': dict(o['mode']),
            'hazard_by_state': {f'{k[0]}|{k[1]}': [int(v[0]), int(v[1])] for k, v in sorted(o['hazf'].items())},
            'hazard_by_thin': {f'{k[0]}|{_SOMAOBS_THIN_LBL[k[1]]}': [int(v[0]), int(v[1])] for k, v in sorted(o['haz'].items())},
            'effect_rest': {f'rest={int(k[0])},frag={int(k[1])}': [int(v[0]), v[1], v[2], v[3]] for k, v in sorted(o['eff'].items())}}


def _somaobs_print(eng):
    o = getattr(eng, '_somaobs', None)
    s = _fch(eng)
    c = _soma_counters(s)
    print("\n  ── [S4] ТЕЛО: ЭПИЗОДЫ РАЗРЫВА (observer v3, read-only; причины — эвристика по замерам, не доказанная причинность) ──")
    if not o:
        print("  нет данных (observer v3 не собрал ни одного замера)")
        return
    st, fate = int(o['started']), o['fate']
    n_open = len(o['open'])
    print(f"  начато эпизодов={st}: восстановились={fate.get('restored', 0)}, умерли с разорванным телом={fate.get('died_fragmented', 0)}, ещё идут={n_open} "
          f"(сверка со счётчиком soma_rupture={c.get('soma_rupture', 0)})")
    ck = o['cause_kind']
    print(f"  что изменилось между прошлым замером и разрывом: клетки потеряны={ck.get('cells_lost', 0)}, клетки добавились (рост/деление)={ck.get('cells_grew', 0)}, "
          f"те же клетки (сдвиг связности)={ck.get('same_cells', 0)}, нет прошлого замера={ck.get('no_prev', 0)}")
    cf = o['cause_flag']
    pct = lambda k: f"{100.0 * cf.get(k, 0) / max(1, st):.0f}%"
    print(f"  сопутствующие условия: слабое место было до разрыва (thin>{Config.SOMA_BRIDGE_THR})={pct('fragile_before')}; шок>=0.2={pct('shock>=0.2')}; мало энергии={pct('low_energy')}; "
          f"роли: {dict(o['cause_role'])}")
    eps = o['eps']
    rs = [e for e in eps if e[1] == 'restored']
    ds = [e for e in eps if e[1] == 'died']
    q = lambda xs, p: float(np.percentile(xs, p)) if xs else float('nan')
    if rs:
        durs = [e[0] for e in rs]
        print(f"  восстановленные: n={len(rs)} длит. медиана={q(durs, 50):.0f} p90={q(durs, 90):.0f} макс={max(durs)} тиков")
        for mode in ('reconnected', 'added_cells', 'amputation'):
            ms = [e[0] for e in rs if e[2] == mode]
            if ms:
                print(f"    как срослись: {mode}: n={len(ms)} медиана длит.={q(ms, 50):.0f}")
        print("    (amputation = тело срослось, но стало меньше: восстановление ценой клеток; reconnected = те же клетки)")
    if ds:
        dd = [e[0] for e in ds]
        print(f"  умерли в разрыве: n={len(ds)} возраст эпизода к смерти медиана={q(dd, 50):.0f} p90={q(dd, 90):.0f} тиков")
    if rs or ds:
        r_rest = 100.0 * sum(1 for e in rs if e[3]) / max(1, len(rs))
        d_rest = 100.0 * sum(1 for e in ds if e[3]) / max(1, len(ds))
        print(f"  rest-цель была в эпизоде: у восстановившихся {r_rest:.0f}% (n={len(rs)}) / у умерших {d_rest:.0f}% (n={len(ds)})  (корреляция, не причинность: rest ставят и по энергии)")
    print("\n  ── [S5] ТЕЛО ↔ ВЫЖИВАНИЕ: смертей на 1000 агент-тиков (в замерах SOMA; возраст и энергия не выровнены) ──")
    hz = lambda v: f"{1000.0 * v[1] / max(1, v[0]):.2f} (смертей={v[1]}, агент-тиков={v[0]})"
    for k, v in sorted(o['hazf'].items()):
        print(f"  роль={k[0]:<13} тело={('разорвано' if k[1] == 'frag' else 'целое'):<9} {hz(v)}")
    for role in sorted({k[0] for k in o['haz']}):
        row = [f"{_SOMAOBS_THIN_LBL[b]}: " + (f"{1000.0 * o['haz'][(role, b)][1] / max(1, o['haz'][(role, b)][0]):.2f} (n_ск={o['haz'][(role, b)][1]})" if (role, b) in o['haz'] else '—')
               for b in range(len(_SOMAOBS_THIN_LBL))]
        print(f"  роль={role:<13} по thin_bridge_risk: " + " | ".join(row))
    print("\n  ── [S6] ТЕЛО → ДЕЙСТВИЕ: судьба предложений 'adjust' и эффект rest ──")
    fin = {k[len('soma_prop_fin_'):]: v for k, v in c.items() if k.startswith('soma_prop_fin_')}
    made = c.get('soma_prop_made', 0)
    top = {k: v for k, v in fin.items() if '_' not in k or k in ('ignored_low_urgency', 'ignored_low_conf', 'no_goal')}
    print(f"  создано={made}; исходы={top}; без записи исхода (живы/умерли с агентом)={made - sum(top.values())}")
    print("  причины lapsed/vetoed: " + str({k: v for k, v in fin.items() if k.startswith('lapsed_') or k.startswith('vetoed_')}))
    print("  NB: 'adjust' в _PN3_GOAL → цель 'rest'; отдельного действия по ремонту формы в механике нет, rest лишь снижает активность.")
    print("  Δ за интервал замера (SOMA_EVERY тиков), по состоянию ДО интервала: [rest?, разорван?]  n  Δthin  Δstress  Δfrag  (Δfrag<0 = срастается)")
    for k, v in sorted(o['eff'].items()):
        n = max(1, v[0])
        print(f"    rest={int(k[0])} разорван={int(k[1])}  n={v[0]:>6}  Δthin={v[1] / n:+.4f}  Δstress={v[2] / n:+.4f}  Δfrag={v[3] / n:+.4f}")
    print("  читать: Δfrag при разорван=1: rest=1 против rest=0. Различие не доказывает пользу rest (выбор цели зависит от энергии и тела).")


def _somaobs_selftest():
    import types as _t
    out = []
    ck = lambda name, cond: out.append((bool(cond), name))
    try:
        W = _t.SimpleNamespace(patterns=[], _soma_e_mean=1.0)
        A = _t.SimpleNamespace(id=1, role_type='normal', energy=1.0, goals=[], _active_shock=0.0)
        mk = lambda fc, n, thin=0.1: {'fragment_count': fc, 'n_cells': n, 'thin_bridge_risk': thin, 'stress': 0.0}
        A.alive = True
        W.patterns = [A]
        _somaobs_measure(A, W, mk(1, 10), 0)
        _somaobs_measure(A, W, mk(2, 9), 5)            # разрыв, клетки потеряны
        o = W._somaobs
        ck('1 разрыв начинает эпизод и классифицируется (cells_lost)', o['started'] == 1 and o['cause_kind']['cells_lost'] == 1 and 1 in o['open'])
        A.goals = [{'type': 'rest'}]
        _somaobs_measure(A, W, mk(2, 9), 10)
        _somaobs_measure(A, W, mk(1, 9), 15)           # срослись теми же клетками (относительно начала эпизода n0=9)
        ck('2 восстановление закрывает эпизод, режим reconnected, rest-флаг учтён', o['fate']['restored'] == 1 and o['mode']['reconnected'] == 1 and o['eps'][-1][3] is True and not o['open'])
        _somaobs_measure(A, W, mk(2, 9), 20)
        W.patterns = []
        _somaobs_tick(W, 24)                           # агент исчез → смерть в разрыве
        ck('3 смерть при открытом эпизоде = died_fragmented, агент снят с учёта', o['fate']['died_fragmented'] == 1 and not o['open'] and 1 not in o['last'])
        ck('4 выживаемость: смерть записана в корзину «разорвано»', o['hazf'][('normal', 'frag')][1] == 1 and o['hazf'][('normal', 'frag')][0] > 0)
        import json as _j
        ck('5 сводка сериализуется (JSON)', bool(_j.dumps(_somaobs_summary(W))))
    except Exception as e:
        import traceback
        out.append((False, f'исключение в тесте: {e!r}\n{traceback.format_exc()}'))
    good = all(o_ for o_, _n in out)
    print(f"[SOMAOBS selftest] {sum(1 for o_, _n in out if o_)}/{len(out)}")
    for ok_, name in out:
        if not ok_:
            print('  FAIL', name)
    _obs_register_selftest('somaobs', good, passed=sum(1 for o_, _n in out if o_), total=len(out),
                           failed=[str(_n).split('\n')[0][:110] for o_, _n in out if not o_])
    return good


try:
    _somaobs_selftest()
except Exception as _e:
    print(f"[SOMAOBS selftest] не выполнен: {_e!r}")
    try:
        _obs_register_selftest('somaobs', False, failed=[('не выполнен: ' + repr(_e))[:110]])
    except Exception:
        pass


class SomaPatternMixin:
    _soma_diag = None
    _soma_mem_t = -10 ** 9

    def _nav_pn2_stance(self, n, p2, att, a, energy, now):
        ctx = n.get('soma_ctx')
        if ctx and Config.SOMA_ENABLED and Config.SOMA_APPLY and Config.SOMA_STANCE and ctx['stress'] > 0.0:
            a = dict(a)
            a['disc'] = min(1.0, float(a['disc']) + float(Config.SOMA_DISC_GAIN) * float(ctx['stress']))
            p2['soma_bias_n'] = p2.get('soma_bias_n', 0) + 1
        return super()._nav_pn2_stance(n, p2, att, a, energy, now)

    def _pn3_handle(self, s, p2, now):
        if Config.SOMA_ENABLED and Config.SOMA_APPLY and Config.SOMA_PN3_VETO:
            try:
                why = None
                for pr in list(p2['props']):
                    if pr.state == 'pending' and _PN3_GOAL.get(pr.type) == 'explore':
                        if why is None:
                            why = _soma_despair(self) or ''
                        if why:
                            s.st['soma_pn3_veto'] += 1
                            s.st['soma_pn3_veto_' + why] += 1
                            _pn3_fin(s, p2, pr, 'vetoed', 'soma_' + why)
            except Exception as e:
                _fch_err(s, 'soma_veto', e)
        return super()._pn3_handle(s, p2, now)

    def _soma_step(self):
        w = self.world
        if w is None or not self.alive:
            return
        if not self.cells:
            # Не оставляем старую геометрию/стресс у живого агента без клеток.
            self._soma_diag = None
            return
        now = int(w.age)
        d0 = self._soma_diag
        if d0 is not None and (now + int(self.id)) % int(Config.SOMA_EVERY) != 0:
            return
        s = _fch(w)
        d = _soma_measure(self.cells, Config.WORLD_SIZE)
        if d is None:
            self._soma_diag = None
            return
        prev_f = d0['fragment_count'] if d0 else 1
        fsince = d0.get('fragmented_since') if d0 else None
        d['t'] = now
        d['last_transition'] = d0.get('last_transition', 'none') if d0 else 'none'
        d['last_transition_t'] = d0.get('last_transition_t', -1) if d0 else -1
        if d['fragment_count'] > 1:
            if prev_f <= 1 or fsince is None:
                fsince = now
                d['last_transition'], d['last_transition_t'] = 'rupture', now
                s.st['soma_rupture'] += 1
                self._soma_remember('soma_rupture', now, d)
            d['fragmented_since'] = fsince
            d['fragmented_duration'] = int(now - fsince)
            s.st['soma_frag_samples'] += 1
        else:
            if prev_f > 1 and fsince is not None:
                dur = int(now - fsince)
                d['last_transition'], d['last_transition_t'] = 'restoration', now
                s.st['soma_restoration'] += 1
                s.st['soma_frag_dur_sum'] += dur
                s.st['soma_frag_dur_n'] += 1
                if dur > s.st['soma_frag_dur_max']:
                    s.st['soma_frag_dur_max'] = dur
                self._soma_remember('soma_restoration', now, d, dur=dur)
            d['fragmented_since'] = None
            d['fragmented_duration'] = 0
        d['stress'] = round(_soma_stress(d), 4)
        self._soma_diag = d
        try:
            _somaobs_measure(self, w, d, now)   # observer v3 (пассивно)
        except Exception as _oe:
            _obs_fail('soma_obs', _oe)
        s.st['soma_samples'] += 1
        s.st['soma_shape_' + d['shape']] += 1
        if d['elongation_valid']:
            s.st['soma_elong_valid'] += 1
        if d['holes']:
            s.st['soma_holes_samples'] += 1
        if d['thin_bridge_risk'] > float(Config.SOMA_BRIDGE_THR):
            s.st['soma_fragile_samples'] += 1
        if d['boundary_exposure'] > float(Config.SOMA_EXPOSE_THR):
            s.st['soma_exposed_samples'] += 1

    def _soma_remember(self, kind, now, d, **kw):
        if not Config.SOMA_MEMORY:
            return
        s = _fch(self.world)
        if now - int(self._soma_mem_t) < int(Config.SOMA_MEM_GAP):
            s.st['soma_mem_throttled'] += 1
            return
        self._soma_mem_t = int(now)
        try:
            self._log_event(kind, frags=int(d['fragment_count']), n=int(d['n_cells']), **kw)
            nar = getattr(self, '_self_narrative', None)
            if nar is not None:
                txt = ('Тело разорвано на {} части: чувствую боль и потерю целостности.'.format(d['fragment_count']) if kind == 'soma_rupture'
                       else 'Тело снова целое: чувствую облегчение.')
                nar.append({'origin': 'OWN', 't': int(now), 'type': kind, 'report': txt, 'fragments': int(d['fragment_count']), 'shape': d['shape']})
            s.st['soma_mem_written'] += 1
        except Exception as e:
            _fch_err(s, 'soma_mem', e)

    def update_model(self, field, lineage_counts=None, given_counter=None, t=None, witness=None):
        res = super().update_model(field, lineage_counts, given_counter, t, witness)
        if not Config.SOMA_ENABLED or self.world is None:
            return res
        s = _fch(self.world)
        try:
            self._soma_step()
            d = self._soma_diag
            if d is not None and Config.SOMA_APPLY and self.alive:
                C = Config
                frag = d['fragment_count'] > 1
                st_ = float(d.get('stress', 0.0))
                if C.SOMA_TENSION:
                    add = float(C.SOMA_TENSION_FRAG) if frag else 0.0
                    add += float(C.SOMA_TENSION_BRIDGE) * min(1.0, d['thin_bridge_risk'] / max(1e-6, 2.0 * float(C.SOMA_BRIDGE_THR)))
                    add += float(C.SOMA_TENSION_EXPOSE) * max(0.0, d['boundary_exposure'] - float(C.SOMA_EXPOSE_THR))
                    if add > 0.0:
                        self.cognitive_tension = min(C.MAX_METRIC, float(self.cognitive_tension) + add)
                        s.st['soma_tension_nudges'] += 1
                if C.SOMA_GUT and st_ > 0.0:
                    uc = getattr(self, '_unconscious', None)
                    if isinstance(uc, dict) and 'gut_feeling' in uc:
                        uc['gut_feeling'] = float(max(-1.0, uc['gut_feeling'] - float(C.SOMA_GUT_GAIN) * st_))
                        s.st['soma_gut_nudges'] += 1
                if frag and float(C.SOMA_ALARM_GAIN) > 0.0:
                    k = 0.25 + min(0.75, d.get('fragmented_duration', 0) / 20.0)
                    self.alarm_level = min(1.0, float(self.alarm_level) + float(C.SOMA_ALARM_GAIN) * k)
                    s.st['soma_alarm_nudges'] += 1
                    if float(self.alarm_level) > 0.3:
                        s.st['soma_alarm_over_gate'] += 1
        except Exception as e:
            _fch_err(s, 'soma', e)
        return res

    def generate_goals(self, field):
        super().generate_goals(field)
        if (not Config.SOMA_ENABLED or not Config.SOMA_APPLY or self.world is None or not self.alive or not self.cells):
            return
        s = _fch(self.world)
        try:
            d = self._soma_diag
            if d is None:
                return
            now = int(self.world.age)
            C = Config
            if C.SOMA_GOAL:
                mine = next((g for g in self.goals if g.get('_source') == 'soma_pain'), None)
                if d['fragment_count'] > 1:
                    if mine is None:
                        if not any(g.get('type') == 'rest' for g in self.goals):
                            self.goals.append({"type": "rest", "priority": float(C.PN_V2_PRIO_REST) + 0.3, "target": None, "age": 0,
                                               "persistence": int(C.SOMA_GOAL_PERSIST), "_source": "soma_pain"})
                            s.st['soma_goal_rest'] += 1
                        else:
                            s.st['soma_goal_have_rest'] += 1
                    else:
                        mine['age'] = 0
                elif mine is not None:
                    self.goals = [g for g in self.goals if g is not mine]
                    s.st['soma_goal_drop'] += 1
            if C.SOMA_PN3_PROPOSE and C.PN3_ENABLED and C.PN3_PROPOSALS:
                n = getattr(self, '_nav', None)
                p2 = n.get('pn2') if n else None
                if p2 is not None and d['fragment_count'] <= 1 and (d['thin_bridge_risk'] > float(C.SOMA_BRIDGE_THR)
                                                                     or d['boundary_exposure'] > float(C.SOMA_EXPOSE_THR)
                                                                     or d['elongation'] > float(C.SOMA_ELONG_THR)):
                    urg = max(0.3, min(1.0, d['stress'] + 0.35))
                    pr = _pn3_prop_new(self, s, _pn3_init(p2), 'adjust', urg, 0.9, ('soma', 0), int(C.PN3_PROP_TTL), now, 'event')
                    if pr is not None:
                        s.st['soma_prop_made'] += 1
        except Exception as e:
            _fch_err(s, 'soma_goal', e)

    def _pn2_tick(self, s, n, rec, motion_pulse, now):
        d = getattr(self, '_soma_diag', None)
        n['soma_ctx'] = {'stress': float(d.get('stress', 0.0)), 'frag': int(d['fragment_count'])} if (d and Config.SOMA_ENABLED) else None
        return super()._pn2_tick(s, n, rec, motion_pulse, now)

    def _pn2_apply_goal(self, n, now, s):
        try:
            p2 = n.get('pn2')
            if p2 is not None and Config.SOMA_ENABLED and Config.SOMA_APPLY and Config.SOMA_STANCE and p2.get('want') == 'explore':
                why = _soma_despair(self)
                if why:
                    p2['want'] = 'rest'
                    s.st['soma_despair_override'] += 1
                    s.st['soma_despair_override_' + why] += 1
        except Exception as e:
            _fch_err(s, 'soma_apply', e)
        return super()._pn2_apply_goal(n, now, s)


def _soma_world(eng):
    alive = [p for p in eng.patterns if p.alive]
    # Защита от устаревшей диагностики: учитываем только живых агентов
    # с непустым телом и актуальным SOMA-снимком.
    measured = [p for p in alive if getattr(p, 'cells', None) and getattr(p, '_soma_diag', None)]
    D = [p._soma_diag for p in measured]
    out = {'n_alive': len(alive), 'n_measured': len(D)}
    role_stats = {}
    role_names = sorted({str(getattr(p, 'role_type', None) or 'other') for p in measured})
    for role in role_names:
        agents = [p for p in measured if str(getattr(p, 'role_type', None) or 'other') == role]
        ds = [p._soma_diag for p in agents]
        role_stats[role] = {
            'n_measured': len(ds),
            'frag_mean': float(np.mean([d['fragment_count'] for d in ds])),
            'frag_agents': sum(1 for d in ds if d['fragment_count'] > 1),
            'thin_mean': float(np.mean([d['thin_bridge_risk'] for d in ds])),
            'neck_mean': float(np.mean([d['neck_candidate_ratio'] for d in ds])),
            'boundary_mean': float(np.mean([d['boundary_exposure'] for d in ds])),
            'complexity_mean': float(np.mean([d['topology_complexity'] for d in ds])),
            'stress_mean': float(np.mean([d.get('stress', 0.0) for d in ds])),
            'cells_mean': float(np.mean([d['n_cells'] for d in ds])),
            'holes_agents': sum(1 for d in ds if d['holes'] > 0),
            'shapes': {k: int(Counter(d['shape'] for d in ds).get(k, 0)) for k in _SOMA_SHAPES},
        }
    out['by_role'] = role_stats
    if not D:
        return out
    ev = [d['elongation'] for d in D if d['elongation_valid']]
    fr = [d for d in D if d['fragment_count'] > 1]
    shp = Counter(d['shape'] for d in D)
    out.update({
        'frag_mean': float(np.mean([d['fragment_count'] for d in D])), 'frag_agents': len(fr),
        'frag_dur_mean_now': float(np.mean([d['fragmented_duration'] for d in fr])) if fr else 0.0,
        'elong_mean_valid': float(np.mean(ev)) if ev else None, 'elong_valid_n': len(ev),
        'thin_mean': float(np.mean([d['thin_bridge_risk'] for d in D])), 'neck_mean': float(np.mean([d['neck_candidate_ratio'] for d in D])),
        'boundary_mean': float(np.mean([d['boundary_exposure'] for d in D])), 'complexity_mean': float(np.mean([d['topology_complexity'] for d in D])),
        'holes_agents': sum(1 for d in D if d['holes'] > 0), 'stress_mean': float(np.mean([d.get('stress', 0.0) for d in D])),
        'cells_mean': float(np.mean([d['n_cells'] for d in D])),
        'shapes': {k: int(shp.get(k, 0)) for k in _SOMA_SHAPES},
        'despair_now': sum(1 for p in alive if _soma_despair(p)),
    })
    return out


def _soma_counters(s):
    return {k: int(v) for k, v in sorted(s.st.items()) if k.startswith('soma_')}


def _soma_line(eng, now):
    w = _soma_world(eng)
    if not w.get('n_measured'):
        return f"[SOMA t={now}] нет измерений"
    sh = w['shapes']
    ev = f"{w['elong_mean_valid']:.2f}" if w['elong_mean_valid'] is not None else "—"
    c = _soma_counters(_fch(eng))
    return (f"[SOMA t={now}] n={w['n_measured']}/{w['n_alive']} frag={w['frag_mean']:.2f} (разорв.={w['frag_agents']}) elong={ev}(valid {w['elong_valid_n']}) "
            f"thin={w['thin_mean']:.3f} neck={w['neck_mean']:.3f} bound={w['boundary_mean']:.2f} cx={w['complexity_mean']:.2f} stress={w['stress_mean']:.3f} "
            f"| формы ш/п/г/к/р={sh['ball']}/{sh['rod']}/{sh['dumbbell']}/{sh['ring']}/{sh['split']} "
            f"| разрывов={c.get('soma_rupture', 0)} восст.={c.get('soma_restoration', 0)} отчаяние={w['despair_now']} вето={c.get('soma_pn3_veto', 0)} "
            f"stance→rest={c.get('soma_despair_override', 0)} alarm+={c.get('soma_alarm_nudges', 0)}")


def _soma_summary(eng):
    s = _fch(eng)
    c = _soma_counters(s)
    n = max(1, c.get('soma_frag_dur_n', 0))
    return {'enabled': bool(Config.SOMA_ENABLED), 'apply': bool(Config.SOMA_APPLY),
            'flags': {k: (v if isinstance(v, (bool, int, float)) else str(v)) for k, v in ((k, getattr(Config, k)) for k in _SOMA_DEFAULTS)},
            'world': _soma_world(eng), 'counters': c,
            'frag_dur_mean_closed': (c.get('soma_frag_dur_sum', 0) / n) if c.get('soma_frag_dur_n', 0) else None,
            'hist': list(getattr(eng, '_soma_hist', []))}


class SomaEngineMixin:
    def _flat_extra_metrics(self):
        counters = _soma_counters(_fch(self))
        return {k: int(counters.get(k, 0)) for k in (
            'soma_rupture', 'soma_restoration', 'soma_despair_override',
            'soma_pn3_veto', 'soma_alarm_nudges', 'soma_goal_rest', 'soma_prop_made')}

    def _post_semantic_step(self, t):
        super()._post_semantic_step(t)
        if not Config.SOMA_ENABLED:
            return
        s = _fch(self)
        try:
            alive = [p for p in self.patterns if p.alive]
            es = [float(p.energy) for p in alive]
            self._soma_e_mean = float(np.mean(es)) if es else 0.0
            now = int(self.age)
            try:
                _somaobs_tick(self, now)            # observer v3: смерти/закрытие эпизодов (пассивно)
            except Exception as _oe:
                _obs_fail('soma_obs_tick', _oe)
            if now > 0 and now % 10 == 0:
                print(_soma_line(self, now), flush=True)
            if now > 0 and now % 50 == 0:
                if not hasattr(self, '_soma_hist'):
                    self._soma_hist = []
                w = _soma_world(self)
                w = {k: v for k, v in w.items()}
                w['t'] = now
                w['rupt'] = int(s.st['soma_rupture'])
                w['rest'] = int(s.st['soma_restoration'])
                self._soma_hist.append(w)
        except Exception as e:
            _fch_err(s, 'soma_tick', e)

    def _collect_metrics(self, t):
        m = super()._collect_metrics(t)
        if Config.SOMA_ENABLED and self.patterns:
            try:
                w = _soma_world(self)
                if w.get('n_measured'):
                    m['soma_frag_mean'] = w['frag_mean']
                    m['soma_frag_agents'] = w['frag_agents']
                    m['soma_elong_mean_valid'] = w['elong_mean_valid'] if w['elong_mean_valid'] is not None else float('nan')
                    m['soma_thin_mean'] = w['thin_mean']
                    m['soma_boundary_mean'] = w['boundary_mean']
                    m['soma_stress_mean'] = w['stress_mean']
                    m['soma_despair_now'] = w['despair_now']
            except Exception as e:
                _fch_err(_fch(self), 'soma_metrics', e)
        return m

    def audit_summary(self):
        S = super().audit_summary()
        try:
            S['soma'] = _soma_summary(self)
            S['soma_obs'] = _somaobs_summary(self)
        except Exception as e:
            _fch_err(_fch(self), 'soma_summary', e)
        return S


def _soma_report(eng=None):
    eng = eng or _PN_LAST.get('eng')
    if eng is None:
        print("[SOMA] нет движка (запусти прогон)")
        return
    S = _soma_summary(eng)
    w, c = S['world'], S['counters']
    print("\n" + "=" * 70 + "\n🫀 SOMA v23 · ТЕЛО ↔ СХЕМА (enabled={} apply={})\n".format(S['enabled'], S['apply']) + "=" * 70)
    if not w.get('n_measured'):
        print("  измерений нет")
        return
    ev = f"{w['elong_mean_valid']:.3f}" if w['elong_mean_valid'] is not None else "—"
    print("  ── [S1] ТЕЛО (на конец, все живые агенты) ──")
    print(f"  измерено {w['n_measured']}/{w['n_alive']}; fragment_count среднее={w['frag_mean']:.3f}, разорванных={w['frag_agents']} (средняя длит. сейчас={w['frag_dur_mean_now']:.1f} тиков)")
    print(f"  elongation (только valid, n={w['elong_valid_n']}) среднее={ev}; thin_bridge_risk (точки сочленения)={w['thin_mean']:.3f}; кандидаты в шейку={w['neck_mean']:.3f}")
    print(f"  boundary_exposure={w['boundary_mean']:.3f}; topology_complexity={w['complexity_mean']:.3f}; с дырами={w['holes_agents']}; клеток в теле (среднее)={w['cells_mean']:.1f}")
    sh = w['shapes']
    tot = max(1, sum(sh.values()))
    print("  формы: " + ", ".join(f"{k}={sh[k]} ({100.0 * sh[k] / tot:.0f}%)" for k in _SOMA_SHAPES))
    by_role = w.get('by_role', {})
    if by_role:
        print("  срез по ролям (текущие измеренные тела):")
        for role, rs in sorted(by_role.items()):
            print(f"    {role}: n={rs['n_measured']} frag={rs['frag_mean']:.2f} разорв.={rs['frag_agents']} "
                  f"thin={rs['thin_mean']:.3f} neck={rs['neck_mean']:.3f} bound={rs['boundary_mean']:.2f} "
                  f"cx={rs['complexity_mean']:.2f} stress={rs['stress_mean']:.3f} cells={rs['cells_mean']:.1f}")
    ns = max(1, c.get('soma_samples', 0))
    print("  ── [S2] СОБЫТИЯ (накопительно) ──")
    print(f"  замеров={c.get('soma_samples', 0)}; доля с разрывом={100.0 * c.get('soma_frag_samples', 0) / ns:.1f}%; хрупкие (перешейки>{Config.SOMA_BRIDGE_THR})={100.0 * c.get('soma_fragile_samples', 0) / ns:.1f}%; "
          f"открытая граница={100.0 * c.get('soma_exposed_samples', 0) / ns:.1f}%; elongation valid={100.0 * c.get('soma_elong_valid', 0) / ns:.1f}%")
    dm = S['frag_dur_mean_closed']
    print(f"  разрывов={c.get('soma_rupture', 0)}, восстановлений={c.get('soma_restoration', 0)}, закрытых эпизодов={c.get('soma_frag_dur_n', 0)} "
          f"(средняя длит.={'—' if dm is None else f'{dm:.1f}'} тиков, макс={c.get('soma_frag_dur_max', 0)})")
    print("  ── [S3] ВЛИЯНИЕ НА СХЕМУ (счётчики воздействий) ──")
    print(f"  состояние: tension+={c.get('soma_tension_nudges', 0)}  | интуиция: gut−={c.get('soma_gut_nudges', 0)}  | сигнал: alarm+={c.get('soma_alarm_nudges', 0)} (выше порога 0.3: {c.get('soma_alarm_over_gate', 0)})")
    print(f"  решения: stance explore→rest (отчаяние)={c.get('soma_despair_override', 0)} (тело разорвано={c.get('soma_despair_override_fragmented', 0)}, мало энергии={c.get('soma_despair_override_low_energy', 0)}); "
          f"вето PN3 explore={c.get('soma_pn3_veto', 0)} (тело={c.get('soma_pn3_veto_fragmented', 0)}, энергия={c.get('soma_pn3_veto_low_energy', 0)})")
    print(f"  цели: rest от разрыва={c.get('soma_goal_rest', 0)} (уже была rest={c.get('soma_goal_have_rest', 0)}, снято при восстановлении={c.get('soma_goal_drop', 0)}); "
          f"предложения PN3 'adjust' от тела={c.get('soma_prop_made', 0)}")
    print(f"  память: записано={c.get('soma_mem_written', 0)}, прижато троттлингом={c.get('soma_mem_throttled', 0)}")
    errs = {k: v for k, v in _fch(eng).st.items() if k.startswith('err_soma')}
    print(f"  ошибки слоя: {errs if errs else 'нет'}")
    H = S['hist']
    if H:
        print("\n  динамика (каждые 50 тиков):  t    n   frag  разорв  elong   thin  bound stress ш/п/г/к/р      разрывов восст.")
        for h in H:
            sh = h.get('shapes', {})
            ev_ = h.get('elong_mean_valid')
            print(f"                              {h['t']:>4} {h.get('n_measured', 0):>4}  {h.get('frag_mean', 0):.2f}  {h.get('frag_agents', 0):>5}  "
                  f"{('%.2f' % ev_) if ev_ is not None else ' — '}  {h.get('thin_mean', 0):.3f}  {h.get('boundary_mean', 0):.2f}  {h.get('stress_mean', 0):.3f} "
                  f"{sh.get('ball', 0)}/{sh.get('rod', 0)}/{sh.get('dumbbell', 0)}/{sh.get('ring', 0)}/{sh.get('split', 0)}    {h.get('rupt', 0):>6} {h.get('rest', 0):>5}")
    print("\n  читать: разрывы/восстановления + длительность = живучесть формы; despair/вето = сколько раз тело остановило исследование; "
          "alarm_over_gate = сколько тревог из-за тела дошло до порога сигнала.")
    try:
        _somaobs_print(eng)
    except Exception as _oe:
        print(f"  [S4–S6] отчёт observer v3 не выведен: {_oe!r}")


def _soma_selftest():
    W = int(Config.WORLD_SIZE)
    def sq(x0, y0, w, h):
        return {(x0 + i, y0 + j) for i in range(w) for j in range(h)}
    ring = {(x, y) for x in range(10, 16) for y in range(10, 16)} - {(x, y) for x in range(11, 15) for y in range(11, 15)}
    wrap_line = {((W - 2 + i) % W, 5) for i in range(5)}
    dumb = sq(10, 10, 3, 3) | {(13, 11), (14, 11), (15, 11)} | sq(16, 10, 3, 3)
    T = [
        ('компактный квадрат 3x3', sq(10, 10, 3, 3), lambda d: d['fragment_count'] == 1 and d['elongation'] < 0.15 and d['holes'] == 0 and d['elongation_valid']),
        ('линия 5x1', sq(10, 10, 5, 1), lambda d: d['fragment_count'] == 1 and d['elongation'] > 0.7 and d['shape'] == 'rod' and d['thin_bridge_risk'] > 0.4),
        ('два квадрата (разрыв)', sq(10, 10, 3, 3) | sq(20, 10, 3, 3), lambda d: d['fragment_count'] == 2 and not d['elongation_valid'] and d['shape'] == 'split'),
        ('кольцо', ring, lambda d: d['fragment_count'] == 1 and d['holes'] == 1 and not d['elongation_valid'] and d['shape'] == 'ring'),
        ('угловое касание (диагональ = разрыв)', {(10, 10), (11, 10), (12, 11), (13, 11)}, lambda d: d['fragment_count'] == 2),
        ('полоса длинная 20x2', sq(10, 10, 20, 2), lambda d: d['fragment_count'] == 1 and d['elongation'] > 0.8 and d['elongation_valid']),
        ('линия через границу тора', wrap_line, lambda d: d['fragment_count'] == 1 and d['elongation'] > 0.7),
        ('гантель (настоящий перешеек)', dumb, lambda d: d['fragment_count'] == 1 and d['articulation_cells'] >= 3 and d['shape'] == 'dumbbell'),
        ('квадрат: точек сочленения нет', sq(10, 10, 4, 4), lambda d: d['articulation_cells'] == 0 and d['thin_bridge_risk'] == 0.0 and d['boundary_exposure'] < 1.0),
        ('пустое тело → None', set(), lambda d: d is None),
    ]
    ok = 0
    print("[SOMA selftest] топология на фикстурах:")
    for name, cells, chk in T:
        d = _soma_measure(cells, W)
        good = bool(chk(d))
        ok += good
        info = '—' if d is None else f"frag={d['fragment_count']} elong={d['elongation']:.2f} valid={d['elongation_valid']} holes={d['holes']} art={d['articulation_cells']} bound={d['boundary_exposure']:.2f} {d['shape']}"
        print(f"  {'OK ' if good else 'FAIL'} {name:<40} {info}")
    probe = sq(3, 3, 4, 3)
    h0 = frozenset(probe)
    _soma_measure(probe, W)
    inv = (frozenset(probe) == h0)
    print(f"  {'OK ' if inv else 'FAIL'} инвариант: измерение не меняет клетки")
    ok += inv
    print(f"[SOMA selftest] {ok}/{len(T) + 1}")
    return ok == len(T) + 1


Pattern = extend_class(Pattern, SomaPatternMixin)
EvolutionEngine = extend_class(EvolutionEngine, SomaEngineMixin)
LedgerEngine = extend_class(LedgerEngine, SomaEngineMixin)
RichEngine = extend_class(RichEngine, SomaEngineMixin)
for _cls in (Pattern, RichEngine):
    assert any(k.__name__ in ("SomaPatternMixin", "SomaEngineMixin") for k in _cls.__mro__), f"слой SOMA не подключён к {_cls.__name__}"
print(f"SOMA v23 подключён: ENABLED={Config.SOMA_ENABLED} APPLY={Config.SOMA_APPLY} EVERY={Config.SOMA_EVERY} | PN3: veto={Config.SOMA_PN3_VETO} propose={Config.SOMA_PN3_PROPOSE}")
print("⚠️ Смени TAG_SUFFIX в ячейке NAV+FCH (напр. 'v223soma'), иначе логи прошлого прогона перезапишутся.")


In [ ]:
# Spatial Emotional Memory — integrated layer for Project 832
import copy as _sem_copy
import math as _sem_math
from collections import deque as _sem_deque
import numpy as _sem_np
import json as _sem_json
from collections import Counter as _sem_Counter, defaultdict as _sem_defaultdict

class SpatialEmotionalMemory:
    """Per-agent bounded memory linking trauma intensity to recognized place IDs."""
    ACTIVE_LIMIT = 64
    ARCHIVE_LIMIT = 50
    INCIDENT_LIMIT = 256
    SAFE_RETURN_MIN_TICKS = 20
    SAFE_RETURN_COOLDOWN_TICKS = 20
    TRAUMA_POSITION_TOLERANCE = 5.0
    SAFE_REDUCTION = 0.10
    FORGET_AFTER_TICKS = 1200
    FORGET_PAIN_FLOOR = 0.035
    PENDING_LIMIT = 128
    PI_RESTORE_TOLERANCE = 9.0   # pi is an unreliable dead-reckoned signal: looser than the safe-return tolerance
    STATUS_AVOID_PAIN = 1.0      # status labels shown by the Observer (raw pain, not modulated)
    STATUS_CAUTION_PAIN = 0.4

    def __init__(self):
        self.active = {}
        self.archive = _sem_deque(maxlen=self.ARCHIVE_LIMIT)
        # Unresolved traumas. A plain dict keeps insertion order and is the ONLY queue: the oldest
        # incident is the first key, so eviction and resolution can never disagree about who is queued.
        self.pending = {}
        self.incidents = _sem_deque(maxlen=self.INCIDENT_LIMIT)
        self._incident_set = set()
        self.events = _sem_deque(maxlen=3000)
        self.counters = {
            'trauma_recorded': 0, 'trauma_deduplicated': 0,
            'trauma_unresolved': 0, 'memory_restored': 0,
            'memory_archived': 0, 'memory_forgotten': 0,
            'safe_visit_completed': 0, 'pain_penalty_applied': 0,
            'survival_override': 0, 'memory_inherited': 0,
            'trauma_reinforced': 0, 'return_to_trauma_place': 0,
            'route_change': 0, 'restore_rejected_pi': 0,
            # v41 observer counters
            'restore_rejected_ambiguous': 0, 'place_id_collision': 0,
            'safe_visit_rejected_pi': 0, 'safe_visit_rejected_cooldown': 0, 'safe_visit_body_unsafe': 0,
            'pending_resolved': 0, 'pending_expired': 0, 'pending_overflow': 0,
            'relapse_after_safe_visit': 0, 'place_healed': 0,
            'inherited_safe_visit': 0, 'inherited_route_bias': 0, 'inherited_retrauma': 0,
            'pain_goal_rest': 0, 'pain_goal_explore': 0,
        }
        self.stats = self._new_stats()
        self._last_shock_record_t = -10**9
        self._last_place_id = None
        self._last_place_t = None
        self._left_places = {}

    HEALED_PAIN = 0.20           # a place counts as healed once pain falls to this level after a real trauma

    @staticmethod
    def _new_stats():
        return {'pain_inflicted': 0.0, 'pain_healed': 0.0, 'place_samples': 0,
                'place_samples_avoid': 0, 'place_samples_caution': 0}

    def reset_lifetime(self):
        """Child inherits memories, never the parent's lifetime telemetry or dedup window."""
        self.counters = {k: 0 for k in SpatialEmotionalMemory().counters}
        self.stats = self._new_stats()
        self.events.clear()
        self.incidents.clear()
        self._incident_set.clear()
        # Unresolved traumas belong to the parent's last unrecognised place; the child never stood there,
        # so it must not attach them to the first place it recognises.
        self.pending.clear()
        self._last_shock_record_t = -10**9
        self._last_place_id = None
        self._last_place_t = None
        self._left_places = {}

    @staticmethod
    def _fp(fp):
        if fp is None:
            return None
        try:
            a = _sem_np.asarray(fp, dtype=float).reshape(-1)
            if not a.size or not _sem_np.all(_sem_np.isfinite(a)):
                return None
            return tuple(float(x) for x in a)
        except Exception:
            return None

    @classmethod
    def _pi(cls, agent):
        """Capture the PlaceNode/NAV pi coordinate; fall back to physical position."""
        nav = getattr(agent, '_nav', None)
        if isinstance(nav, dict):
            try:
                pi = tuple(float(x) for x in nav.get('pi', ())[:2])
                if len(pi) == 2 and all(_sem_math.isfinite(x) for x in pi):
                    return pi
            except Exception:
                pass
        return cls._position(agent)

    @staticmethod
    def _distance(a, b):
        if a is None or b is None or len(a) != len(b):
            return float('inf')
        return float(_sem_np.mean(_sem_np.abs(_sem_np.asarray(a) - _sem_np.asarray(b))))

    def _emit(self, agent, event, **data):
        now = int(getattr(getattr(agent, 'world', None), 'age', getattr(agent, 'age', 0)))
        row = {'t': now, 'agent_id': int(getattr(agent, 'id', -1)), 'event': str(event)}
        row.update(data)
        self.events.append(row)
        try:
            _tel = _sem_tel(getattr(agent, 'world', None))
            if _tel is not None:
                _tel.on_event(agent, row)
        except Exception:
            pass
        try:
            agent._log_event('spatial_emotional_memory_' + str(event), **data)
        except Exception:
            pass
        try:
            world = getattr(agent, 'world', None)
            witness = getattr(world, 'witness', None)
            if witness is not None:
                witness.record(int(getattr(agent, 'id', -1)), 'sem_' + str(event), **data)
        except Exception:
            pass

    def on_nav_forget(self, agent, place_id):
        """Archive meaningful SEM memory when PlaceNode evicts/forgets its node."""
        try:
            pid = int(place_id)
        except Exception:
            return False
        rec = self.active.pop(pid, None)
        if rec is None:
            return False
        if float(rec.get('pain', 0.0)) > self.FORGET_PAIN_FLOOR or int(rec.get('n', 0)) > 0:
            self.archive.append(_sem_copy.deepcopy(rec))
            self.counters['memory_archived'] += 1
            self._emit(agent, 'memory_archived', place_id=pid,
                       reason='nav_place_forgotten', pain=round(float(rec.get('pain', 0.0)), 4))
        else:
            self.counters['memory_forgotten'] += 1
            self._emit(agent, 'memory_forgotten', place_id=pid,
                       reason='nav_place_forgotten_empty')
        return True

    @staticmethod
    def _pi_gap(a, b):
        """Toroidal distance between two pi coordinates; None when either is unknown."""
        if a is None or b is None:
            return None
        try:
            dx = float(a[0]) - float(b[0]); dy = float(a[1]) - float(b[1])
        except Exception:
            return None
        try:
            size = int(globals().get('Config').WORLD_SIZE)
        except Exception:
            size = 0
        if size > 0:
            dx = (dx + size / 2.0) % size - size / 2.0
            dy = (dy + size / 2.0) % size - size / 2.0
        return float(_sem_math.hypot(dx, dy))

    def _find_archive(self, fp, threshold=0.045, pi=None):
        self._last_restore_reject = None
        if fp is None:
            return None
        ranked = sorted((self._distance(fp, r.get('fp_mean')), i, r)
                        for i, r in enumerate(self.archive))
        if not ranked or ranked[0][0] > threshold:
            return None
        # Ambiguous landscape fingerprints are not sufficient to restore a memory.
        if len(ranked) > 1 and ranked[1][0] - ranked[0][0] < threshold * 0.20:
            self._last_restore_reject = 'ambiguous'
            return None
        # Same landscape but the pi coordinate is far from where the trauma happened:
        # most likely a different place with a similar fingerprint -> do not restore.
        gap = self._pi_gap(pi, ranked[0][2].get('pi_at_trauma'))
        if gap is not None and gap > self.PI_RESTORE_TOLERANCE:
            self._last_restore_reject = 'pi_mismatch'
            return None
        return ranked[0][1], ranked[0][2]

    def _trim_active(self, agent):
        while len(self.active) > self.ACTIVE_LIMIT:
            pid, rec = min(self.active.items(), key=lambda kv: kv[1].get('last_t', -1))
            self.active.pop(pid, None)
            self.archive.append(_sem_copy.deepcopy(rec))
            self.counters['memory_archived'] += 1
            self._emit(agent, 'memory_archived', place_id=int(pid), pain=round(float(rec.get('pain', 0.0)), 4))

    @staticmethod
    def _position(agent):
        cells = getattr(agent, 'cells', None)
        if not cells:
            return None
        try:
            pts = list(cells)
            if not pts:
                return None
            # World coordinates are toroidal; the circular mean avoids edge artifacts.
            size = int(getattr(getattr(agent, 'world', None), 'size', 0) or
                       getattr(getattr(agent, 'world', None), 'WORLD_SIZE', 0) or 0)
            if size <= 0:
                size = int(getattr(Config, 'WORLD_SIZE', 0) or 0)
            if size > 0:
                xs = [2.0 * _sem_math.pi * float(p[0]) / size for p in pts]
                ys = [2.0 * _sem_math.pi * float(p[1]) / size for p in pts]
                cx = _sem_math.atan2(sum(_sem_math.sin(v) for v in xs), sum(_sem_math.cos(v) for v in xs)) * size / (2.0 * _sem_math.pi)
                cy = _sem_math.atan2(sum(_sem_math.sin(v) for v in ys), sum(_sem_math.cos(v) for v in ys)) * size / (2.0 * _sem_math.pi)
                return (float(cx % size), float(cy % size))
            return (float(sum(float(p[0]) for p in pts) / len(pts)),
                    float(sum(float(p[1]) for p in pts) / len(pts)))
        except Exception:
            return None

    def _prune(self, agent, now):
        """Age out weak memories; archive meaningful ones instead of silently dropping them."""
        now = int(now)
        for pid, rec in list(self.active.items()):
            age = max(0, now - int(rec.get('last_t', now)))
            pain = float(rec.get('pain', 0.0))
            if age >= self.FORGET_AFTER_TICKS and pain <= self.FORGET_PAIN_FLOOR:
                self.active.pop(pid, None)
                self.counters['memory_forgotten'] += 1
                self._emit(agent, 'memory_forgotten', place_id=int(pid), reason='stale_low_pain', age=age,
                           pain=round(pain, 4))
            elif age >= self.FORGET_AFTER_TICKS and pain > self.FORGET_PAIN_FLOOR:
                self.active.pop(pid, None)
                self.archive.append(_sem_copy.deepcopy(rec))
                self.counters['memory_archived'] += 1
                self._emit(agent, 'memory_archived', place_id=int(pid), reason='stale_meaningful_memory',
                           age=age, pain=round(pain, 4))

    def _resolve_pending(self, agent, place_id, fp, now):
        if place_id is None or not self.pending:
            return 0
        queued = list(self.pending.items())
        resolved = 0
        for key, item in queued:
            # Only attach unknown-place incidents to the first recognized place shortly afterward.
            if int(now) - int(item.get('t', now)) > self.FORGET_AFTER_TICKS:
                self.pending.pop(key, None)
                self.counters['memory_forgotten'] += 1
                self.counters['pending_expired'] += 1
                self._emit(agent, 'memory_forgotten', reason='unresolved_trauma_expired', incident_id=repr(key))
                continue
            self.pending.pop(key, None)
            # The incident was already registered when it entered the pending queue.
            # Resolve it without appending a second copy to the deduplication deque.
            self.record_trauma(agent, place_id, fp, item['intensity'], item['reason'],
                               key, _registered=True)
            resolved += 1
        if resolved:
            self.counters['pending_resolved'] += resolved
            self._emit(agent, 'pending_trauma_resolved', place_id=int(place_id), count=resolved)
        return resolved

    def ensure_place(self, agent, place_id, fp=None, now=None, create=False):
        if place_id is None:
            return None
        try:
            pid = int(place_id)
        except Exception:
            return None
        now = int(now if now is not None else getattr(getattr(agent, 'world', None), 'age', 0))
        sig = self._fp(fp)
        rec = self.active.get(pid)
        if rec is not None:
            old_fp = rec.get('fp_mean')
            # Reused IDs with a clearly different fingerprint must not inherit old pain.
            if sig is not None and old_fp is not None and self._distance(sig, old_fp) > 0.16:
                self.active.pop(pid, None)
                self.counters['memory_forgotten'] += 1
                self.counters['place_id_collision'] += 1
                self._emit(agent, 'memory_forgotten', place_id=pid, reason='place_id_fingerprint_mismatch')
                rec = None
            else:
                if sig is not None:
                    if old_fp is None:
                        rec['fp_mean'] = sig
                    elif len(sig) == len(old_fp):
                        n = max(1, int(rec.get('n', 1)))
                        rec['fp_mean'] = tuple((old_fp[i] * n + sig[i]) / (n + 1) for i in range(len(sig)))
                rec['last_t'] = now
                return rec
        if rec is None:
            restored = self._find_archive(sig, pi=self._pi(agent))
            if restored is None and getattr(self, '_last_restore_reject', None) == 'pi_mismatch':
                self._last_restore_reject = None
                self.counters['restore_rejected_pi'] += 1
                self._emit(agent, 'memory_restore_rejected', place_id=pid, reason='pi_mismatch')
            elif restored is None and getattr(self, '_last_restore_reject', None) == 'ambiguous':
                self._last_restore_reject = None
                self.counters['restore_rejected_ambiguous'] += 1
                self._emit(agent, 'memory_restore_rejected', place_id=pid, reason='ambiguous_fingerprint')
            if restored is not None:
                idx, old = restored
                # Remove exactly the matched archived record; never share nested state.
                arr = list(self.archive)
                arr.pop(idx)
                self.archive = _sem_deque(arr, maxlen=self.ARCHIVE_LIMIT)
                rec = _sem_copy.deepcopy(old)
                rec['place_id'] = pid
                rec['fp_mean'] = sig if sig is not None else rec.get('fp_mean')
                rec['last_t'] = now
                rec.setdefault('pi_at_trauma', rec.get('position'))
                rec.setdefault('last_safe_visit_t', -10**9)
                self.active[pid] = rec
                self.counters['memory_restored'] += 1
                self._emit(agent, 'memory_restored', place_id=pid, pain=round(float(rec.get('pain', 0.0)), 4))
            else:
                if not create:
                    return None
                rec = {'place_id': pid, 'fp_mean': sig, 'pain': 0.0, 'n': 0,
                       'safe_visits': 0, 'last_t': now, 'last_reason': None,
                       'last_incident_id': None, 'position': self._position(agent),
                       'pi_at_trauma': None, 'last_safe_visit_t': -10**9}
                self.active[pid] = rec
                self._trim_active(agent)
        return rec

    def record_trauma(self, agent, place_id, fp, intensity, reason, incident_id, _registered=False):
        if incident_id is None:
            incident_id = (int(getattr(getattr(agent, 'world', None), 'age', getattr(agent, 'age', 0))), str(reason))
        try:
            hash(incident_id)
        except Exception:
            incident_id = repr(incident_id)
        if not _registered:
            if incident_id in self._incident_set:
                self.counters['trauma_deduplicated'] += 1
                return False
            self._incident_set.add(incident_id)
            self.incidents.append(incident_id)
            while len(self._incident_set) > self.INCIDENT_LIMIT:
                # deque order is authoritative for the bounded deduplication window.
                self._incident_set = set(self.incidents)
                break
        if place_id is None:
            now = int(getattr(getattr(agent, 'world', None), 'age', getattr(agent, 'age', 0)))
            self.pending.pop(incident_id, None)   # a re-queued incident goes to the tail
            self.pending[incident_id] = {'t': now, 'intensity': float(_sem_np.clip(float(intensity), 0.0, 2.0)),
                                         'reason': str(reason), 'fp': self._fp(fp)}
            while len(self.pending) > self.PENDING_LIMIT:
                oldest = next(iter(self.pending))   # oldest = first inserted key
                self.pending.pop(oldest)
                self.counters['memory_forgotten'] += 1
                self.counters['pending_overflow'] += 1
                self._emit(agent, 'memory_forgotten', reason='pending_queue_overflow', incident_id=repr(oldest))
            self.counters['trauma_unresolved'] += 1
            self._emit(agent, 'trauma_pending_place', reason=str(reason), incident_id=repr(incident_id))
            return False
        now = int(getattr(getattr(agent, 'world', None), 'age', getattr(agent, 'age', 0)))
        rec = self.ensure_place(agent, place_id, fp, now, create=True)
        if rec is None:
            self.counters['trauma_unresolved'] += 1
            self._emit(agent, 'trauma_unresolved', reason=str(reason), incident_id=repr(incident_id))
            return False
        value = float(_sem_np.clip(float(intensity), 0.0, 2.0))
        reinforced = int(rec.get('n', 0)) > 0 and float(rec.get('pain', 0.0)) > 0.0
        pain_before = float(rec.get('pain', 0.0))
        relapse = int(rec.get('safe_visits', 0)) > 0 and int(rec.get('n', 0)) > 0
        # Repeated trauma reinforces pain with diminishing returns; cap at 2.
        rec['pain'] = float(min(2.0, rec.get('pain', 0.0) + value * (1.0 - min(0.75, rec.get('pain', 0.0) / 2.5))))
        rec['n'] = int(rec.get('n', 0)) + 1
        rec['peak_pain'] = float(max(float(rec.get('peak_pain', 0.0)), rec['pain']))
        rec.setdefault('first_trauma_t', now)
        rec['last_trauma_t'] = now
        rec['healed'] = False
        self.stats['pain_inflicted'] += max(0.0, rec['pain'] - pain_before)
        if relapse:
            self.counters['relapse_after_safe_visit'] += 1
        if rec.get('inherited'):
            self.counters['inherited_retrauma'] += 1
        rec['last_t'] = now
        rec['position'] = self._position(agent) or rec.get('position')
        rec['pi_at_trauma'] = self._pi(agent)
        rec.setdefault('last_safe_visit_t', -10**9)
        rec['last_reason'] = str(reason)
        rec['last_incident_id'] = repr(incident_id)
        self.counters['trauma_recorded'] += 1
        if reinforced:
            self.counters['trauma_reinforced'] += 1
        self._emit(agent, 'trauma_recorded', place_id=int(place_id), pain=round(rec['pain'], 4),
                   pain_before=round(pain_before, 4), n=int(rec['n']), relapse=bool(relapse),
                   intensity=round(value, 4), reason=str(reason), incident_id=repr(incident_id))
        return True

    def pain_for(self, agent, place_id, fp=None):
        rec = self.ensure_place(agent, place_id, fp) if place_id is not None else None
        return float(rec.get('pain', 0.0)) if rec else 0.0

    def behavioral_weight(self, agent):
        # Body condition changes current reactivity, not the stored event history.
        endurance = float(getattr(agent, '_cellular_endurance', 0.5))
        scar = float(getattr(agent, 'epistemic_scar', 0.0))
        em = getattr(agent, 'emotional_memory', {})
        grief = float(em.get('grief', 0.0)) if isinstance(em, dict) else 0.0
        soma = getattr(agent, '_soma_diag', None)
        fragments = int(soma.get('fragment_count', 1)) if isinstance(soma, dict) else 1
        weight = 0.70 + 0.55 * max(0.0, min(1.0, scar)) + 0.35 * max(0.0, min(1.0, grief))
        weight += 0.15 if fragments > 1 else 0.0
        weight -= 0.35 * max(0.0, min(1.0, endurance))
        return float(max(0.25, min(1.5, weight)))

    def modulated_pain(self, agent, place_id, fp=None):
        return float(self.pain_for(agent, place_id, fp) * self.behavioral_weight(agent))

    def observe_place(self, agent, place_id, fp, now):
        now = int(now)
        pid = int(place_id) if place_id is not None else None
        self._prune(agent, now)
        if pid is not None:
            self._resolve_pending(agent, pid, fp, now)
            _srec = self.active.get(pid)
            self.stats['place_samples'] += 1
            if _srec is not None:
                _sp = float(_srec.get('pain', 0.0))
                if _sp >= self.STATUS_AVOID_PAIN:
                    self.stats['place_samples_avoid'] += 1
                elif _sp >= self.STATUS_CAUTION_PAIN:
                    self.stats['place_samples_caution'] += 1
        prev = self._last_place_id
        prev_t = self._last_place_t
        if prev is not None and pid != prev:
            self._left_places[prev] = now
        if pid is not None and pid != prev and pid in self._left_places:
            left_t = int(self._left_places[pid])
            if now - left_t >= self.SAFE_RETURN_MIN_TICKS:
                rec = self.ensure_place(agent, pid, fp, now)
                if rec is not None and float(rec.get('pain', 0.0)) > 0.0:
                    self.counters['return_to_trauma_place'] += 1
                    self._emit(agent, 'return_to_trauma_place', place_id=pid,
                               pain=round(float(rec.get('pain', 0.0)), 4), away_ticks=int(now - left_t))
                soma = getattr(agent, '_soma_diag', None)
                fragments = int(soma.get('fragment_count', 1)) if isinstance(soma, dict) else 1
                shock = float(getattr(agent, '_active_shock', 0.0))
                energy = float(getattr(agent, 'energy', 1.0))
                body_safe = energy > 0.08 and shock <= 0.15 and fragments <= 1
                current_pi = self._pi(agent)
                trauma_pi = rec.get('pi_at_trauma') if rec is not None else None
                pi_ok = False
                if current_pi is not None and trauma_pi is not None:
                    dx = float(current_pi[0]) - float(trauma_pi[0])
                    dy = float(current_pi[1]) - float(trauma_pi[1])
                    size = int(getattr(getattr(agent, 'world', None), 'size', 0) or getattr(Config, 'WORLD_SIZE', 0) or 0)
                    if size > 0:
                        dx = (dx + size / 2.0) % size - size / 2.0
                        dy = (dy + size / 2.0) % size - size / 2.0
                    pi_ok = _sem_math.hypot(dx, dy) <= self.TRAUMA_POSITION_TOLERANCE
                cooldown_ok = rec is not None and now - int(rec.get('last_safe_visit_t', -10**9)) >= self.SAFE_RETURN_COOLDOWN_TICKS
                if (rec is not None and rec.get('pain', 0.0) > 0.0 and bool(getattr(agent, 'alive', False))
                        and body_safe and pi_ok and cooldown_ok):
                    old = float(rec['pain'])
                    rec['pain'] = max(0.0, old * (1.0 - self.SAFE_REDUCTION))
                    rec['safe_visits'] = int(rec.get('safe_visits', 0)) + 1
                    rec['last_t'] = now
                    rec['last_safe_visit_t'] = now
                    self.counters['safe_visit_completed'] += 1
                    self.stats['pain_healed'] += max(0.0, old - rec['pain'])
                    if rec.get('inherited'):
                        self.counters['inherited_safe_visit'] += 1
                    _lt = rec.get('last_trauma_t')
                    since = (now - int(_lt)) if _lt is not None else None
                    healed_now = bool(rec['pain'] <= self.HEALED_PAIN and float(rec.get('peak_pain', old)) >= self.STATUS_CAUTION_PAIN
                                      and not rec.get('healed'))
                    if healed_now:
                        rec['healed'] = True
                        self.counters['place_healed'] += 1
                    self._emit(agent, 'safe_visit_completed', place_id=pid,
                               pain_before=round(old, 4), pain_after=round(rec['pain'], 4),
                               safe_visits=rec['safe_visits'], since_trauma=since, healed=healed_now,
                               pi_at_trauma=trauma_pi,
                               pi_now=current_pi, cooldown_ticks=self.SAFE_RETURN_COOLDOWN_TICKS)
                elif rec is not None and rec.get('pain', 0.0) > 0.0 and bool(getattr(agent, 'alive', False)) and body_safe:
                    self.counters['safe_visit_rejected_pi' if not pi_ok else 'safe_visit_rejected_cooldown'] += 1
                    self._emit(agent, 'safe_visit_rejected', place_id=pid,
                               reason=('pi_mismatch' if not pi_ok else 'cooldown'),
                               pi_at_trauma=trauma_pi, pi_now=current_pi,
                               ticks_since_last=max(0, now-int(rec.get('last_safe_visit_t', -10**9))))
                elif rec is not None and rec.get('pain', 0.0) > 0.0 and bool(getattr(agent, 'alive', False)) and not body_safe:
                    self.counters['safe_visit_body_unsafe'] += 1
                self._left_places.pop(pid, None)
        self._last_place_id, self._last_place_t = pid, now

    @classmethod
    def status_for(cls, pain):
        pain = float(pain)
        if pain >= cls.STATUS_AVOID_PAIN:
            return 'avoid'
        if pain >= cls.STATUS_CAUTION_PAIN:
            return 'caution'
        return 'safe'

    def snapshot(self):
        return {
            'active_places': len(self.active), 'archive_places': len(self.archive),
            'pending_trauma': len(self.pending),
            'total_pain': float(sum(float(r.get('pain', 0.0)) for r in self.active.values())),
            'max_pain': float(max((float(r.get('pain', 0.0)) for r in self.active.values()), default=0.0)),
            'safe_visits': int(self.counters['safe_visit_completed']),
            'places': [{'place_id': int(pid), 'pain': round(float(r.get('pain', 0.0)), 4),
                        'safe_visits': int(r.get('safe_visits', 0)), 'n': int(r.get('n', 0)),
                        'last_t': int(r.get('last_t', -1)),
                        'status': self.status_for(r.get('pain', 0.0))}
                       for pid, r in self.active.items()],
            'counters': dict(self.counters), 'stats': dict(self.stats),
            'recent_events': list(self.events),
        }


class SpatialEmotionalMemoryMixin:
    def __init__(self, pid, cells, parent=None, world=None):
        super().__init__(pid, cells, parent, world)
        inherited = getattr(parent, '_spatial_emotional_memory', None) if parent is not None else None
        self._spatial_emotional_memory = _sem_copy.deepcopy(inherited) if inherited is not None else SpatialEmotionalMemory()
        self._sem_last_tick = -1
        if inherited is not None:
            # Inherit memories, not the parent's lifetime telemetry or dedup window.
            sem_child = self._spatial_emotional_memory
            sem_child.reset_lifetime()
            sem_child.counters['memory_inherited'] = 1
            for _r in list(sem_child.active.values()) + list(sem_child.archive):
                _r['inherited'] = True
                _r['generation'] = int(_r.get('generation', 0)) + 1
            try:
                _tp = float(sum(float(_r.get('pain', 0.0)) for _r in sem_child.active.values()))
                self._spatial_emotional_memory._emit(self, 'memory_inherited', parent_id=int(parent.id),
                                                     active_places=len(self._spatial_emotional_memory.active),
                                                     archive_places=len(self._spatial_emotional_memory.archive),
                                                     total_pain=round(_tp, 4))
            except Exception:
                pass
        try:
            _tel = _sem_tel(getattr(self, 'world', None) or world)
            if _tel is not None:
                _tel.register(self, parent)
        except Exception:
            pass

    def _sem_current_place(self):
        n = getattr(self, '_nav', None)
        if not isinstance(n, dict):
            return None, None
        cur = n.get('current_place')
        if cur is None:
            return None, None
        pid = cur.get('id') if isinstance(cur, dict) else cur
        fp = cur.get('fp_mean') if isinstance(cur, dict) else None
        return pid, fp

    def _sem_record(self, intensity, reason, incident_id=None, place_id=None, fp=None):
        sem = getattr(self, '_spatial_emotional_memory', None)
        if sem is None:
            self._spatial_emotional_memory = sem = SpatialEmotionalMemory()
        if place_id is None or fp is None:
            p0, f0 = self._sem_current_place()
            if place_id is None:
                place_id = p0
            if fp is None:
                fp = f0
        now = int(getattr(getattr(self, 'world', None), 'age', getattr(self, 'age', 0)))
        if incident_id is None:
            incident_id = (now, str(reason), int(getattr(self, 'id', -1)))
        return sem.record_trauma(self, place_id, fp, intensity, reason, incident_id)

    def _nav_place_recognition(self, fp, now, s):
        res = super()._nav_place_recognition(fp, now, s)
        try:
            pid, pfp = self._sem_current_place()
            sem = self._spatial_emotional_memory
            sem.observe_place(self, pid, pfp if pfp is not None else fp, int(now))
            if pid is not None:
                pain = sem.modulated_pain(self, pid, pfp if pfp is not None else fp)
                if pain >= 0.5 and int(now) - int(getattr(self, '_sem_last_tick', -1)) >= 10:
                    self._sem_last_tick = int(now)
                    sem._emit(self, 'place_pain_observed', place_id=int(pid), pain=round(pain, 4))
        except Exception:
            pass
        return res

    def move(self, field, t):
        # Pain does not invent coordinates or teleport the agent: it modestly raises
        # the existing mobility drive so the current navigation controller can leave.
        old_mobility = None
        try:
            if getattr(self, 'role_type', None) != 'feral' and getattr(self, 'alive', False):
                pid, fp = self._sem_current_place()
                sem = self._spatial_emotional_memory
                pain = sem.modulated_pain(self, pid, fp)
                if pain >= 0.5:
                    old_mobility = float(self.genome.get('mobility', 0.1))
                    self.genome['mobility'] = min(0.6, old_mobility * (1.0 + min(0.35, 0.12 * pain)))
                    if pain >= 0.5:
                        sem.counters['pain_penalty_applied'] += 1
                        # If the nav controller has a directional command, bias it away from
                        # the remembered place centroid. No teleportation or invented target.
                        nav = getattr(self, '_nav', None)
                        rec = sem.active.get(int(pid)) if pid is not None else None
                        pos = sem._position(self)
                        center = rec.get('position') if isinstance(rec, dict) else None
                        if pos and center:
                            size = int(getattr(Config, 'WORLD_SIZE', 0) or 0)
                            dx, dy = float(pos[0]) - float(center[0]), float(pos[1]) - float(center[1])
                            if size > 0:
                                dx = (dx + size / 2.0) % size - size / 2.0
                                dy = (dy + size / 2.0) % size - size / 2.0
                            mag = _sem_math.hypot(dx, dy)
                            if mag > 1e-6:
                                away = (dx / mag, dy / mag)
                                mx, my = getattr(self, '_dir_momentum', (0.0, 0.0))
                                blend = min(0.25, 0.05 * pain)
                                bx, by = (1.0-blend)*float(mx)+blend*away[0], (1.0-blend)*float(my)+blend*away[1]
                                bmag = _sem_math.hypot(bx, by)
                                if bmag > 1e-8:
                                    self._dir_momentum = (bx/bmag, by/bmag)
                        if isinstance(nav, dict) and isinstance(nav.get('cmd'), dict) and pos and center:
                            size = int(getattr(Config, 'WORLD_SIZE', 0) or 0)
                            dx, dy = pos[0] - center[0], pos[1] - center[1]
                            if size > 0:
                                dx = (dx + size / 2.0) % size - size / 2.0
                                dy = (dy + size / 2.0) % size - size / 2.0
                            mag = _sem_math.hypot(dx, dy)
                            cmd = nav['cmd']
                            h = cmd.get('h')
                            if mag > 1e-6 and h is not None:
                                away = (dx / mag, dy / mag)
                                blend = min(0.35, 0.08 * pain)
                                cmd['h'] = ((1.0 - blend) * float(h[0]) + blend * away[0],
                                            (1.0 - blend) * float(h[1]) + blend * away[1])
                                hmag = _sem_math.hypot(*cmd['h'])
                                if hmag > 1e-8:
                                    cmd['h'] = (cmd['h'][0] / hmag, cmd['h'][1] / hmag)
                                sem.counters['route_change'] += 1
                                if isinstance(rec, dict) and rec.get('inherited'):
                                    sem.counters['inherited_route_bias'] += 1
                                _now_rb = int(getattr(getattr(self, 'world', None), 'age', 0))
                                if _now_rb - int(getattr(self, '_sem_last_route_emit_t', -10**9)) >= 10:
                                    self._sem_last_route_emit_t = _now_rb
                                    sem._emit(self, 'pain_route_bias', place_id=int(pid), pain=round(pain, 4),
                                              blend=round(blend, 4))
        except Exception:
            old_mobility = None
        try:
            return super().move(field, t)
        finally:
            if old_mobility is not None:
                self.genome['mobility'] = old_mobility

    def generate_goals(self, field):
        res = super().generate_goals(field)
        try:
            if self.role_type == 'feral' or not getattr(self, 'alive', False):
                return res
            pid, fp = self._sem_current_place()
            sem = self._spatial_emotional_memory
            pain = sem.modulated_pain(self, pid, fp)
            if pain >= 0.5 and isinstance(getattr(self, 'goals', None), list):
                energy = float(getattr(self, 'energy', 1.0))
                # Survival/energy needs outrank fear-memory avoidance. This model has no
                # universal food goal, so the native rest/energy regulation path wins.
                survival_urgent = energy <= 0.25
                if survival_urgent:
                    for g in self.goals:
                        if g.get('_source') == 'spatial_emotional_memory':
                            g['priority'] = min(float(g.get('priority', 0.0)), 0.25)
                    rest_goal = next((g for g in self.goals if g.get('type') == 'rest'), None)
                    if rest_goal is None:
                        self.goals.append({'type': 'rest', 'priority': 4.0, 'target': None,
                                           'age': 0, 'persistence': 8, '_source': 'sem_survival_override'})
                    else:
                        rest_goal['priority'] = max(float(rest_goal.get('priority', 0.0)), 4.0)
                    sem.counters['survival_override'] += 1
                    if int(getattr(getattr(self, 'world', None), 'age', 0)) % 10 == 0:
                        sem._emit(self, 'survival_override', energy=round(energy, 4), pain=round(pain, 4),
                                  priority='survival_over_memory_avoidance')
                else:
                    goal_type = 'rest' if pain >= 0.9 else 'explore'
                    priority = min(3.5, 1.0 + pain)
                    existing = next((g for g in self.goals if g.get('type') == goal_type and g.get('_source') == 'spatial_emotional_memory'), None)
                    if existing is None:
                        self.goals.append({'type': goal_type, 'priority': priority, 'target': None,
                                           'age': 0, 'persistence': 12, '_source': 'spatial_emotional_memory'})
                        sem.counters['pain_goal_' + goal_type] += 1
                    else:
                        existing['priority'] = max(float(existing.get('priority', 0.0)), priority)
        except Exception:
            pass
        return res

    def feral_execute(self, other, field):
        now = int(getattr(getattr(self, 'world', None), 'age', getattr(self, 'age', 0)))
        world = getattr(self, 'world', None)
        # v35: feral_execute is called for any prey within kill radius, but the base method returns
        # early (not adjacent / sanctuary / not prey) without any attack. Trauma is recorded only when
        # a strike was actually rolled (world._feral_stats['rolled'] grew) and the target survived it.
        _stats = getattr(world, '_feral_stats', None)
        try:
            _r0 = int(_stats['rolled']) if _stats is not None else None
        except Exception:
            _r0 = None
        res = super().feral_execute(other, field)
        try:
            if (_r0 is not None and other is not None and getattr(self, 'role_type', None) == 'feral'
                    and int(_stats['rolled']) > _r0 and bool(getattr(other, 'alive', False))
                    and hasattr(other, '_sem_record')):
                _fury = float(getattr(self, '_feral_fury', 0.0))
                other._sem_record(min(1.0, 0.45 + 0.10 * _fury), 'feral_threat',
                                  (now, 'feral_threat', int(self.id), int(other.id)))
        except Exception:
            pass
        return res

    def _find_prey(self, field, world, min_cells=5, radius=20):
        result = super()._find_prey(field, world, min_cells=min_cells, radius=radius)
        prey, direction, dist = result
        # A feral that repeatedly fails to secure prey at a remembered painful place
        # receives an event only when it actually retreats from a current threat.
        try:
            if getattr(self, 'role_type', None) == 'feral' and prey is None:
                pid, fp = self._sem_current_place()
                if pid is not None and self._spatial_emotional_memory.pain_for(self, pid, fp) > 0.5:
                    now = int(getattr(getattr(self, 'world', None), 'age', getattr(self, 'age', 0)))
                    # Кулдаун на (агент, место): иначе событие пишется каждый тик для одной и той же ситуации.
                    _np_last = self.__dict__.setdefault('_sem_noprey_last', {})
                    _np_cd = int(getattr(Config, 'SEM_NO_PREY_EMIT_COOLDOWN', 25))
                    if now - int(_np_last.get(int(pid), -10**9)) >= _np_cd:
                        _np_last[int(pid)] = now
                        if len(_np_last) > 64:
                            for _k in sorted(_np_last, key=_np_last.get)[:16]:
                                _np_last.pop(_k, None)
                        self._spatial_emotional_memory._emit(self, 'pain_place_no_prey', place_id=int(pid))
        except Exception:
            pass
        return result

    def _compute_active_shock(self):
        res = super()._compute_active_shock()
        try:
            shock = float(getattr(self, '_active_shock', 0.0))
            now = int(getattr(getattr(self, 'world', None), 'age', getattr(self, 'age', 0)))
            if shock >= 0.20 and now - int(getattr(self, '_sem_last_shock_record_t', -10**9)) >= 10:
                self._sem_last_shock_record_t = now
                self._sem_record(min(0.8, 0.35 * shock), 'physiological_shock',
                                 (now, 'physiological_shock', int(getattr(self, 'id', -1))))
        except Exception:
            pass
        return res

    def _log_event(self, event_type, **kwargs):
        res = super()._log_event(event_type, **kwargs)
        # Translate only explicit, high-confidence adverse events into memory.
        # SEM's own emitted events are deliberately excluded to prevent recursion.
        name = str(event_type)
        trauma = {
            'endurance_collapse': 0.45,
            'forced_soul_collapse': 0.60,
            'cell_loss': 0.40,
            'cell_lost': 0.40,
            'body_fragmentation': 0.45,
        }.get(name)
        if trauma is not None and not name.startswith('spatial_emotional_memory_'):
            try:
                now = int(getattr(getattr(self, 'world', None), 'age', getattr(self, 'age', 0)))
                self._sem_record(trauma, name, (now, name, int(getattr(self, 'id', -1))))
            except Exception:
                pass
        return res

    def _soma_remember(self, kind, now, d, **kw):
        res = super()._soma_remember(kind, now, d, **kw)
        if str(kind) == 'soma_rupture':
            try:
                incident = (int(now), 'soma_rupture', int(getattr(self, 'id', -1)))
                self._sem_record(min(1.0, 0.35 + 0.15 * max(0, int(d.get('fragment_count', 1)) - 1)),
                                 'soma_rupture', incident)
            except Exception:
                pass
        return res

    def divide(self, field, next_id):
        # The child constructor runs SpatialEmotionalMemoryMixin.__init__, which
        # already deep-copies the parent's memory, resets lifetime telemetry and
        # emits exactly one memory_inherited event. Do not repeat that work here.
        return super().divide(field, next_id)

    def _spatial_emotional_memory_snapshot(self):
        sem = getattr(self, '_spatial_emotional_memory', None)
        return sem.snapshot() if sem is not None else SpatialEmotionalMemory().snapshot()


# Attach after NAV+FCH and SOMA so the wrappers observe the final active methods.
Pattern = extend_class(Pattern, SpatialEmotionalMemoryMixin)
assert any(k.__name__ == 'SpatialEmotionalMemoryMixin' for k in Pattern.__mro__)

# ═════════════════════════ Telemetry (engine-level, survives agent death) ═════════════════════════
_SEM_RESULT_RU = {
    'trauma_recorded': 'записано', 'pain_route_bias': 'свернул', 'safe_visit_completed': 'боль ↓',
    'safe_visit_rejected': 'отклонён', 'memory_archived': 'в архив', 'memory_forgotten': 'забыто',
    'memory_restored': 'восстановлено', 'memory_inherited': 'унаследовано',
    'memory_restore_rejected': 'не то место', 'survival_override': 'голод > страх',
    'return_to_trauma_place': 'вернулся', 'trauma_pending_place': 'ждёт место',
    'pending_trauma_resolved': 'привязано', 'place_pain_observed': 'боль места',
}


def _sem_tel(world, create=True):
    """Engine-level telemetry holder (plain attribute on the engine instance; no class is modified)."""
    if world is None:
        return None
    tel = getattr(world, '_sem_telemetry', None)
    if tel is None and create:
        tel = SEMTelemetry()
        try:
            world._sem_telemetry = tel
        except Exception:
            return None
    return tel


class SEMTelemetry:
    """Aggregates SEM events and lifetime counters of agents that already died."""
    SERIES_EVERY = 50
    POS_EVERY = 25
    BUCKET = 100
    BINS = 16

    def __init__(self):
        self.event_counts = _sem_Counter()
        self.by_role = _sem_defaultdict(_sem_Counter)
        self.buckets = _sem_defaultdict(_sem_Counter)
        self.reason = {}                       # reason -> [n, sum_intensity, sum_pain_after, sum_pain_before]
        self.ledger = _sem_deque(maxlen=2000)
        self.latency = _sem_deque(maxlen=4000)  # ticks from last trauma to a successful safe visit
        self.heal_drops = _sem_deque(maxlen=4000)
        self.archived_reason = _sem_Counter()
        self.forgotten_reason = _sem_Counter()
        self.reject_reason = _sem_Counter()
        self.inherit_rows = _sem_deque(maxlen=1000)
        self.series = _sem_deque(maxlen=800)
        self.registry = {}
        self.grave = {'n': 0, 'roles': _sem_Counter(), 'counters': _sem_Counter(), 'stats': _sem_Counter(),
                      'final_pains': _sem_deque(maxlen=6000), 'trauma_per_agent': _sem_deque(maxlen=6000),
                      'safe_per_place': _sem_deque(maxlen=6000), 'places_per_agent': _sem_deque(maxlen=6000),
                      'lifespan': _sem_deque(maxlen=6000), 'agent_ticks': 0,
                      'died_with_pain': 0, 'died_traumatized': 0, 'pain_at_death': 0.0}
        self.visited = set()
        self.last_series_t = None
        self.last_pos_t = None
        self.errors = _sem_Counter()

    # ── events ──
    def on_event(self, agent, row):
        ev = str(row.get('event'))
        t = int(row.get('t', 0))
        role = str(getattr(agent, 'role_type', None) or 'unknown')
        self.event_counts[ev] += 1
        self.by_role[role][ev] += 1
        self.buckets[t // self.BUCKET][ev] += 1
        if ev == 'trauma_recorded':
            r = self.reason.setdefault(str(row.get('reason')), [0, 0.0, 0.0, 0.0])
            r[0] += 1
            r[1] += float(row.get('intensity', 0.0))
            r[2] += float(row.get('pain', 0.0))
            r[3] += float(row.get('pain_before', 0.0))
        elif ev == 'safe_visit_completed':
            if row.get('since_trauma') is not None:
                self.latency.append(int(row['since_trauma']))
            self.heal_drops.append(float(row.get('pain_before', 0.0)) - float(row.get('pain_after', 0.0)))
        elif ev == 'memory_archived':
            self.archived_reason[str(row.get('reason', 'capacity_trim'))] += 1
        elif ev == 'memory_forgotten':
            self.forgotten_reason[str(row.get('reason', 'unknown'))] += 1
        elif ev == 'memory_restore_rejected':
            self.reject_reason[str(row.get('reason', 'unknown'))] += 1
        elif ev == 'memory_inherited':
            self.inherit_rows.append({'t': t, 'child': int(row.get('agent_id', -1)), 'parent': row.get('parent_id'),
                                      'active': int(row.get('active_places', 0)),
                                      'archive': int(row.get('archive_places', 0)),
                                      'total_pain': float(row.get('total_pain', 0.0))})
        self.ledger.append({'t': t, 'agent_id': int(row.get('agent_id', -1)), 'event': ev,
                            'place_id': row.get('place_id'), 'reason': row.get('reason'),
                            'pain': row.get('pain', row.get('pain_after')),
                            'result': _SEM_RESULT_RU.get(ev, ev)})

    # ── registry / graveyard ──
    def register(self, agent, parent=None):
        sem = getattr(agent, '_spatial_emotional_memory', None)
        if sem is None:
            return
        aid = int(getattr(agent, 'id', -1))
        self.registry[aid] = {'sem': sem, 'role': str(getattr(agent, 'role_type', None) or 'unknown'),
                              'born': int(getattr(getattr(agent, 'world', None), 'age', 0) or 0),
                              'parent': (int(parent.id) if parent is not None and hasattr(parent, 'id') else None),
                              'seen': False, 'last_age': 0}

    def _fold(self, aid):
        ent = self.registry.pop(aid, None)
        if ent is None:
            return
        sem = ent['sem']
        g = self.grave
        g['n'] += 1
        g['roles'][ent['role']] += 1
        for k, v in sem.counters.items():
            g['counters'][k] += int(v)
        for k, v in getattr(sem, 'stats', {}).items():
            g['stats'][k] += float(v)
        pains = [float(r.get('pain', 0.0)) for r in sem.active.values()]
        g['final_pains'].extend(pains)
        g['trauma_per_agent'].append(int(sem.counters.get('trauma_recorded', 0)))
        g['safe_per_place'].extend(int(r.get('safe_visits', 0)) for r in sem.active.values())
        g['places_per_agent'].append(len(sem.active))
        g['lifespan'].append(int(ent['last_age']))
        g['agent_ticks'] += int(ent['last_age'])
        g['pain_at_death'] += float(sum(pains))
        if pains and max(pains) >= SpatialEmotionalMemory.STATUS_CAUTION_PAIN + 0.1:
            g['died_with_pain'] += 1
        if int(sem.counters.get('trauma_recorded', 0)) > 0:
            g['died_traumatized'] += 1

    def lifetime(self, live):
        c = _sem_Counter(self.grave['counters'])
        s = _sem_Counter(self.grave['stats'])
        for _, sem in live:
            for k, v in sem.counters.items():
                c[k] += int(v)
            for k, v in getattr(sem, 'stats', {}).items():
                s[k] += float(v)
        return c, s

    # ── per-tick work (called from the engine's own _post_semantic_step) ──
    def tick(self, eng, now):
        live_ids = set()
        for p in list(getattr(eng, 'patterns', []) or []):
            if not getattr(p, 'alive', False):
                continue
            aid = int(getattr(p, 'id', -1))
            sem = getattr(p, '_spatial_emotional_memory', None)
            if sem is None:
                continue
            live_ids.add(aid)
            ent = self.registry.get(aid)
            if ent is None or ent['sem'] is not sem:
                if ent is not None:
                    self._fold(aid)
                self.register(p)
                ent = self.registry.get(aid)
                if ent is None:
                    continue
            ent['seen'] = True
            ent['last_age'] = int(getattr(p, 'age', 0) or 0)
        for aid in list(self.registry):
            if aid in live_ids:
                continue
            ent = self.registry[aid]
            if ent['seen'] or now - ent['born'] > 5:
                self._fold(aid)
        if self.last_pos_t is None or now - self.last_pos_t >= self.POS_EVERY:
            self.last_pos_t = now
            self._sample_positions(eng)
        if self.last_series_t is None or now - self.last_series_t >= self.SERIES_EVERY:
            self.last_series_t = now
            self._sample_series(eng, now)

    def _sample_positions(self, eng):
        size = int(getattr(Config, 'WORLD_SIZE', 0) or 0)
        if size <= 0:
            return
        for p in list(getattr(eng, 'patterns', []) or []):
            if not getattr(p, 'alive', False):
                continue
            pos = SpatialEmotionalMemory._position(p)
            if pos:
                self.visited.add((int(pos[0] % size / size * self.BINS) % self.BINS,
                                  int(pos[1] % size / size * self.BINS) % self.BINS))

    def _sample_series(self, eng, now):
        live = _sem_live(eng)
        c, s = self.lifetime(live)
        pains = [float(r.get('pain', 0.0)) for _, sem in live for r in sem.active.values()]
        st = _sem_Counter(SpatialEmotionalMemory.status_for(x) for x in pains)
        self.series.append({
            't': int(now), 'alive': len(live), 'ever': len(live) + self.grave['n'],
            'active_places': len(pains), 'archive_places': int(sum(len(sem.archive) for _, sem in live)),
            'mean_pain': float(_sem_np.mean(pains)) if pains else 0.0,
            'max_pain': float(max(pains, default=0.0)),
            'avoid': int(st['avoid']), 'caution': int(st['caution']), 'safe': int(st['safe']),
            'trauma': int(c['trauma_recorded']), 'safe_visits': int(c['safe_visit_completed']),
            'route': int(c['route_change']), 'override': int(c['survival_override']),
            'pain_inflicted': float(s['pain_inflicted']), 'pain_healed': float(s['pain_healed'])})


class SpatialEmotionalMemoryEngineMixin:
    def _post_semantic_step(self, t):
        super()._post_semantic_step(t)
        try:
            _sem_tel(self).tick(self, int(getattr(self, 'age', t)))
        except Exception as e:
            try:
                _sem_tel(self).errors[f"{type(e).__name__}: {e}"[:120]] += 1
            except Exception:
                pass

    def audit_summary(self):
        summary = super().audit_summary()
        try:
            agents = list(getattr(self, 'patterns', []))
            sems = [getattr(p, '_spatial_emotional_memory', None) for p in agents]
            sems = [s for s in sems if s is not None]
            totals = {k: int(sum(s.counters.get(k, 0) for s in sems))
                      for k in SpatialEmotionalMemory().counters}
            pain_values = [float(r.get('pain', 0.0)) for s in sems for r in s.active.values()]
            block = {
                'agents_with_memory': len(sems),
                'active_places': int(sum(len(s.active) for s in sems)),
                'archive_places': int(sum(len(s.archive) for s in sems)),
                'pending_trauma': int(sum(len(s.pending) for s in sems)),
                'recent_event_count': int(sum(len(s.events) for s in sems)),
                'unique_incidents_window': int(sum(len(s.incidents) for s in sems)),
                'mean_active_pain': float(_sem_np.mean(pain_values)) if pain_values else 0.0,
                'max_active_pain': float(max(pain_values, default=0.0)),
                'counters': totals,
            }
            try:
                block['stats'] = _sem_json_safe(_sem_stats(self, compact=True))
            except Exception as e:
                block['stats_error'] = f"{type(e).__name__}: {e}"[:160]
            if isinstance(summary, dict):
                summary['spatial_emotional_memory'] = block
            return summary
        except Exception:
            return summary

for _sem_engine_name in ('EvolutionEngine', 'LedgerEngine', 'RichEngine'):
    _sem_engine_cls = globals().get(_sem_engine_name)
    if _sem_engine_cls is not None:
        globals()[_sem_engine_name] = extend_class(_sem_engine_cls, SpatialEmotionalMemoryEngineMixin)

# Explicit mechanics tests for invariants independent of a full simulation run.
def _spatial_emotional_memory_selftest():
    sem = SpatialEmotionalMemory()
    class _Agent:
        id = 7; age = 10; world = None; alive = True
        _cellular_endurance = 0.5; epistemic_scar = 0.0; emotional_memory = {'grief': 0.0}
        def _log_event(self, *a, **k): pass
    a = _Agent()
    fp = tuple([0.2, 0.4, 0.6, 0.8])
    assert sem.record_trauma(a, 3, fp, 0.8, 'test', (10, 'test', 7))
    pain = sem.active[3]['pain']
    assert not sem.record_trauma(a, 3, fp, 0.8, 'test', (10, 'test', 7))
    assert sem.active[3]['pain'] == pain
    sem.active[3]['pain'] = 1.0
    sem.active[3]['fp_mean'] = fp
    sem.active[3]['last_t'] = 1
    sem._trim_active(a)
    # Force eviction via a small cap, then restore by a unique fingerprint.
    sem.ACTIVE_LIMIT = 1
    sem.active[4] = {'place_id': 4, 'fp_mean': (0.9, 0.9, 0.9, 0.9), 'pain': 0.2, 'n': 1, 'safe_visits': 0, 'last_t': 20}
    sem._trim_active(a)
    restored = sem.ensure_place(a, 9, fp, 30)
    assert restored is not None and restored['pain'] > 0.0
    # Deep-copy inheritance invariant.
    child = _sem_copy.deepcopy(sem)
    child.active[9]['pain'] = 0.0
    assert sem.active[9]['pain'] > 0.0
    # Unknown-place trauma is queued, then attached when place recognition becomes available.
    pending_sem = SpatialEmotionalMemory()
    assert not pending_sem.record_trauma(a, None, None, 0.6, 'unknown_place_test', (11, 'unknown_place_test', 7))
    assert len(pending_sem.pending) == 1
    pending_sem.observe_place(a, 12, (0.1, 0.2, 0.3, 0.4), 12)
    assert 12 in pending_sem.active and pending_sem.active[12]['pain'] > 0.0
    assert pending_sem.counters['trauma_recorded'] == 1
    # Safe return only lowers pain after a sufficiently long absence.
    before = pending_sem.active[12]['pain']
    pending_sem.observe_place(a, 13, (0.9, 0.8, 0.7, 0.6), 13)
    pending_sem.observe_place(a, 12, (0.1, 0.2, 0.3, 0.4), 25)
    assert pending_sem.active[12]['pain'] <= before
    # Empty place recognition must not consume active-memory slots.
    empty_sem = SpatialEmotionalMemory()
    assert empty_sem.ensure_place(a, 99, fp, 15) is None and not empty_sem.active
    # Trauma records a spatial position; safe return requires matching pi and >=20-tick cooldown.
    class _SpatialAgent(_Agent):
        cells = [(10, 10)]
        world = type('W', (), {'age': 100, 'size': 100})()
        energy = 1.0; _active_shock = 0.0; _soma_diag = {'fragment_count': 1}
        _nav = {'pi': [10.0, 10.0]}
        def _log_event(self, *a, **k): pass
    sa = _SpatialAgent()
    sem2 = SpatialEmotionalMemory()
    assert sem2.record_trauma(sa, 3, fp, 0.8, 'position_test', (100, 'position_test', 7))
    assert sem2.active[3].get('pi_at_trauma') == (10.0, 10.0)
    sem2.observe_place(sa, 3, fp, 100)
    sem2.observe_place(sa, 4, (0.9, 0.8, 0.7, 0.6), 101)
    sa.cells = [(40, 40)]
    sa._nav['pi'] = [40.0, 40.0]
    sem2.observe_place(sa, 3, fp, 130)
    assert sem2.counters['safe_visit_completed'] == 0, 'wrong pi must not recover trauma'
    sa.cells = [(10, 10)]
    sa._nav['pi'] = [10.0, 10.0]
    sem2.observe_place(sa, 4, (0.9, 0.8, 0.7, 0.6), 131)
    sem2.observe_place(sa, 3, fp, 152)
    assert sem2.counters['safe_visit_completed'] == 1, 'same pi after cooldown should recover'
    # PlaceNode forgetting archives meaningful memory through the same hook used by _pn_forget.
    forget_sem = SpatialEmotionalMemory()
    forget_sem.record_trauma(a, 5, fp, 0.7, 'forget_hook_test', (12, 'forget_hook_test', 7))
    assert forget_sem.on_nav_forget(a, 5) and 5 not in forget_sem.active and len(forget_sem.archive) == 1
    # Inherited memories must not inherit lifetime counters or incident dedup state.
    parent_sem = SpatialEmotionalMemory(); parent_sem.counters['trauma_recorded'] = 17
    parent_sem.incidents.append(('old',)); parent_sem._incident_set.add(('old',))
    child_sem = _sem_copy.deepcopy(parent_sem)
    child_sem.counters = {k: 0 for k in SpatialEmotionalMemory().counters}
    child_sem.counters['memory_inherited'] = 1
    child_sem.incidents.clear(); child_sem._incident_set.clear()
    assert child_sem.counters['trauma_recorded'] == 0 and child_sem.counters['memory_inherited'] == 1 and not child_sem._incident_set
    # reset_lifetime() itself (not a hand-copy of its body) must also drop the parent's unresolved traumas.
    par2 = SpatialEmotionalMemory()
    par2.record_trauma(a, None, None, 0.9, 'parent_pending', (200, 'parent_pending', 7))
    assert len(par2.pending) == 1
    kid2 = _sem_copy.deepcopy(par2)
    kid2.reset_lifetime()
    assert not kid2.pending, "child must not inherit the parent's unresolved traumas"
    kid2.observe_place(a, 777, (0.1, 0.2, 0.3, 0.4), 50)
    assert kid2.counters['trauma_recorded'] == 0 and not kid2.active.get(777, {}).get('pain', 0.0)
    # Pending overflow evicts the OLDEST incident; the dict is the single source of truth.
    ov_sem = SpatialEmotionalMemory()
    for _i in range(ov_sem.PENDING_LIMIT + 2):
        ov_sem.record_trauma(a, None, None, 0.5, 'ov_test', (300 + _i, 'ov_test', 7))
    assert len(ov_sem.pending) == ov_sem.PENDING_LIMIT
    assert (300, 'ov_test', 7) not in ov_sem.pending and (301, 'ov_test', 7) not in ov_sem.pending
    assert (302, 'ov_test', 7) in ov_sem.pending
    assert (300 + ov_sem.PENDING_LIMIT + 1, 'ov_test', 7) in ov_sem.pending
    assert ov_sem.counters['pending_overflow'] == 2
    assert sem2.counters['return_to_trauma_place'] >= 2, 'returns to a painful place must be counted'
    # Status labels follow the picture: high pain avoids, medium is cautious, low is safe.
    assert (SpatialEmotionalMemory.status_for(1.8), SpatialEmotionalMemory.status_for(0.7),
            SpatialEmotionalMemory.status_for(0.2)) == ('avoid', 'caution', 'safe')
    # Re-traumatisation of an already painful place is counted as reinforcement.
    re_sem = SpatialEmotionalMemory()
    assert re_sem.record_trauma(a, 21, fp, 0.5, 'r1', (1, 'r1', 7)) and re_sem.counters['trauma_reinforced'] == 0
    assert re_sem.record_trauma(a, 21, fp, 0.5, 'r2', (2, 'r2', 7)) and re_sem.counters['trauma_reinforced'] == 1
    # Archive restore is gated by pi: same fingerprint but far-away pi must NOT restore the memory.
    class _PiAgent(_Agent):
        cells = [(50, 50)]
        world = type('W', (), {'age': 300, 'size': 100})()
        _nav = {'pi': [50.0, 50.0]}
    pa = _PiAgent()
    pi_sem = SpatialEmotionalMemory()
    pi_sem.archive.append({'place_id': 1, 'fp_mean': fp, 'pain': 1.2, 'n': 2, 'safe_visits': 0,
                           'last_t': 5, 'pi_at_trauma': (10.0, 10.0)})
    assert pi_sem.ensure_place(pa, 77, fp, 300) is None
    assert pi_sem.counters['restore_rejected_pi'] == 1 and len(pi_sem.archive) == 1
    pi_sem.archive[0]['pi_at_trauma'] = (50.0, 51.0)
    got = pi_sem.ensure_place(pa, 77, fp, 301)
    assert got is not None and abs(got['pain'] - 1.2) < 1e-9 and not pi_sem.archive
    snap = re_sem.snapshot()
    assert snap['places'] and snap['places'][0]['status'] in ('avoid', 'caution', 'safe')
    return {'ok': True, 'dedup': sem.counters['trauma_deduplicated'],
            'restore': sem.counters['memory_restored'], 'pending_resolved': pending_sem.counters['trauma_recorded'],
            'safe_visits': pending_sem.counters['safe_visit_completed'],
            'pi_cooldown': sem2.counters['safe_visit_completed'], 'zero_entries': len(empty_sem.active),
            'child_counter_reset': child_sem.counters['trauma_recorded'] == 0}

# ═════════════════════════ Observer: statistics (read-only) ═════════════════════════
_SEM_PAIN_EDGES = (0.0, 0.05, 0.2, 0.4, 0.7, 1.0, 1.5, 2.0001)
_SEM_PAIN_LABELS = ('<0.05', '0.05-0.2', '0.2-0.4', '0.4-0.7', '0.7-1.0', '1.0-1.5', '1.5-2.0')


def _sem_collect(eng):
    out = []
    for p in list(getattr(eng, 'patterns', []) or []):
        sem = getattr(p, '_spatial_emotional_memory', None)
        if sem is not None:
            out.append((p, sem))
    return out


def _sem_live(eng):
    return [(p, s) for p, s in _sem_collect(eng) if getattr(p, 'alive', False)]


def _sem_json_safe(obj):
    return _sem_json.loads(_sem_json.dumps(obj, default=lambda o: float(o) if isinstance(o, (_sem_np.floating, _sem_np.integer)) else str(o)))


def _sem_f(x, default=0.0):
    try:
        v = float(x)
        return v if _sem_math.isfinite(v) else default
    except Exception:
        return default


def _sem_dist(vals):
    a = _sem_np.asarray([_sem_f(v) for v in vals], dtype=float)
    if not a.size:
        return {'n': 0, 'mean': 0.0, 'min': 0.0, 'p50': 0.0, 'p90': 0.0, 'max': 0.0}
    return {'n': int(a.size), 'mean': float(a.mean()), 'min': float(a.min()),
            'p50': float(_sem_np.percentile(a, 50)), 'p90': float(_sem_np.percentile(a, 90)), 'max': float(a.max())}


def _sem_hist(vals, edges=_SEM_PAIN_EDGES):
    a = _sem_np.asarray([_sem_f(v) for v in vals], dtype=float)
    if not a.size:
        return [0] * (len(edges) - 1)
    return [int(x) for x in _sem_np.histogram(a, bins=list(edges))[0]]


def _sem_gini(vals):
    a = _sem_np.sort(_sem_np.asarray([max(0.0, _sem_f(v)) for v in vals], dtype=float))
    n = a.size
    if n == 0 or a.sum() <= 0:
        return 0.0
    return float((2.0 * _sem_np.sum((_sem_np.arange(1, n + 1)) * a) / (n * a.sum())) - (n + 1.0) / n)


def _sem_top_share(vals, frac=0.10):
    a = _sem_np.sort(_sem_np.asarray([max(0.0, _sem_f(v)) for v in vals], dtype=float))[::-1]
    if not a.size or a.sum() <= 0:
        return 0.0
    k = max(1, int(_sem_math.ceil(a.size * frac)))
    return float(a[:k].sum() / a.sum())


def _sem_spark(vals, width=48):
    vals = [_sem_f(v) for v in vals]
    if not vals:
        return ''
    if len(vals) > width:
        step = len(vals) / float(width)
        vals = [float(_sem_np.mean(vals[int(i * step):max(int(i * step) + 1, int((i + 1) * step))])) for i in range(width)]
    lo, hi = min(vals), max(vals)
    chars = '▁▂▃▄▅▆▇█'
    if hi - lo < 1e-12:
        return chars[0] * len(vals)
    return ''.join(chars[min(7, int((v - lo) / (hi - lo) * 7.999))] for v in vals)


def _sem_spatial(live, tel, size, bins=16):
    if size <= 0:
        return None
    grid = _sem_np.zeros((bins, bins))
    pts = []
    for _, s in live:
        for r in s.active.values():
            pos, pain = r.get('position'), _sem_f(r.get('pain', 0.0))
            if not pos or pain <= 0.0:
                continue
            x, y = float(pos[0]) % size, float(pos[1]) % size
            grid[int(y / size * bins) % bins, int(x / size * bins) % bins] += pain
            pts.append((x, y, pain))
    out = {'bins': bins, 'size': int(size), 'pain_mass': float(grid.sum()), 'grid': [[round(float(v), 3) for v in row] for row in grid],
           'center': None, 'radius': None, 'hot': [],
           'coverage': (len(tel.visited) / float(bins * bins)) if tel is not None else None}
    if pts:
        w = _sem_np.array([q[2] for q in pts])
        k = 2.0 * _sem_math.pi / size
        xs = _sem_np.array([q[0] for q in pts]) * k
        ys = _sem_np.array([q[1] for q in pts]) * k
        cx = (_sem_math.atan2(float((w * _sem_np.sin(xs)).sum()), float((w * _sem_np.cos(xs)).sum())) / k) % size
        cy = (_sem_math.atan2(float((w * _sem_np.sin(ys)).sum()), float((w * _sem_np.cos(ys)).sum())) / k) % size
        dx = (_sem_np.array([q[0] for q in pts]) - cx + size / 2.0) % size - size / 2.0
        dy = (_sem_np.array([q[1] for q in pts]) - cy + size / 2.0) % size - size / 2.0
        out['center'] = [round(float(cx), 1), round(float(cy), 1)]
        out['radius'] = round(float(_sem_np.sqrt((w * (dx * dx + dy * dy)).sum() / w.sum())), 1)
        flat = _sem_np.argsort(grid.ravel())[::-1][:3]
        out['hot'] = [{'cell': [int(i % bins), int(i // bins)], 'pain': round(float(grid.ravel()[i]), 3)}
                      for i in flat if grid.ravel()[i] > 0]
    return out


def _sem_stats(eng, compact=False):
    """Collect every Observer metric into one plain dict. Reads state only."""
    tel = _sem_tel(eng, create=False)
    live = _sem_live(eng)
    keys = list(SpatialEmotionalMemory().counters)
    if tel is not None:
        C, S = tel.lifetime(live)
        grave = tel.grave
    else:
        C, S = _sem_Counter(), _sem_Counter()
        for _, s in live:
            for k, v in s.counters.items():
                C[k] += int(v)
            for k, v in s.stats.items():
                S[k] += float(v)
        grave = None
    tot = {k: int(C.get(k, 0)) for k in keys}
    n_live = len(live)
    n_dead = int(grave['n']) if grave else 0
    now = int(getattr(eng, 'age', 0) or 0)
    agent_ticks = int(sum(int(getattr(p, 'age', 0) or 0) for p, _ in live) + (grave['agent_ticks'] if grave else 0))
    per_k = lambda n: (1000.0 * n / agent_ticks) if agent_ticks > 0 else 0.0

    pending_now = int(sum(len(s.pending) for _, s in live))
    recs = [(p, s, pid, r) for p, s in live for pid, r in s.active.items()]
    pains = [_sem_f(r.get('pain', 0.0)) for *_, r in recs]
    st_count = _sem_Counter(SpatialEmotionalMemory.status_for(x) for x in pains)
    st = {'avoid': int(st_count['avoid']), 'caution': int(st_count['caution']), 'safe': int(st_count['safe'])}

    # ── trauma ──
    tr_per_agent = [int(s.counters.get('trauma_recorded', 0)) for _, s in live] + (list(grave['trauma_per_agent']) if grave else [])
    by_reason = {}
    if tel is not None:
        total_r = sum(v[0] for v in tel.reason.values()) or 1
        for k, v in sorted(tel.reason.items(), key=lambda kv: -kv[1][0]):
            by_reason[k] = {'n': int(v[0]), 'share': v[0] / total_r, 'mean_intensity': v[1] / max(1, v[0]),
                            'mean_pain_before': v[3] / max(1, v[0]), 'mean_pain_after': v[2] / max(1, v[0])}
    by_role = {role: int(cnt.get('trauma_recorded', 0)) for role, cnt in tel.by_role.items()} if tel is not None else {}
    trauma = {
        'total': tot['trauma_recorded'], 'attached_share': tot['trauma_recorded'] / max(1, tot['trauma_recorded'] + pending_now),
        'pending_now': pending_now, 'pending_resolved': tot['pending_resolved'], 'pending_expired': tot['pending_expired'],
        'pending_overflow': tot['pending_overflow'], 'deduplicated': tot['trauma_deduplicated'],
        'unresolved': tot['trauma_unresolved'], 'reinforced': tot['trauma_reinforced'],
        'relapse_after_safe_visit': tot['relapse_after_safe_visit'],
        'by_reason': by_reason, 'by_role': by_role,
        'agents_ge1_share': (sum(1 for x in tr_per_agent if x > 0) / float(len(tr_per_agent))) if tr_per_agent else 0.0,
        'per_agent': _sem_dist(tr_per_agent), 'gini': _sem_gini(tr_per_agent), 'top10_share': _sem_top_share(tr_per_agent),
        'rate_per_1000_agent_ticks': per_k(tot['trauma_recorded'])}

    # ── pain / memory capacity ──
    inh = [_sem_f(r.get('pain', 0.0)) for *_, r in recs if r.get('inherited')]
    own = [_sem_f(r.get('pain', 0.0)) for *_, r in recs if not r.get('inherited')]
    places_n = [len(s.active) for _, s in live]
    arch_n = [len(s.archive) for _, s in live]
    pain = {
        'hist': dict(zip(_SEM_PAIN_LABELS, _sem_hist(pains))), 'dist': _sem_dist(pains), 'status': st,
        'total_mass': float(sum(pains)),
        'places_per_agent': _sem_dist(places_n), 'active_occupancy': (float(_sem_np.mean(places_n)) / SpatialEmotionalMemory.ACTIVE_LIMIT) if places_n else 0.0,
        'archive_per_agent': _sem_dist(arch_n), 'archive_full_agents': int(sum(1 for x in arch_n if x >= SpatialEmotionalMemory.ARCHIVE_LIMIT)),
        'archive_pain': _sem_dist([_sem_f(r.get('pain', 0.0)) for _, s in live for r in s.archive]),
        'dead_final_pain': _sem_dist(list(grave['final_pains'])) if grave else _sem_dist([]),
        'inherited_active': {'n': len(inh), 'mean_pain': float(_sem_np.mean(inh)) if inh else 0.0},
        'own_active': {'n': len(own), 'mean_pain': float(_sem_np.mean(own)) if own else 0.0}}

    # ── healing ──
    sv_places = [int(r.get('safe_visits', 0)) for *_, r in recs] + (list(grave['safe_per_place']) if grave else [])
    painful_hist = {'0': 0, '1': 0, '2': 0, '3-4': 0, '5+': 0}
    for x in sv_places:
        painful_hist['0' if x == 0 else '1' if x == 1 else '2' if x == 2 else '3-4' if x <= 4 else '5+'] += 1
    infl, heal = float(S.get('pain_inflicted', 0.0)), float(S.get('pain_healed', 0.0))
    healing = {
        'safe_visits': tot['safe_visit_completed'], 'pain_inflicted': infl, 'pain_healed': heal,
        'heal_ratio': heal / infl if infl > 0 else 0.0,
        'mean_drop_per_visit': heal / max(1, tot['safe_visit_completed']),
        'latency_since_trauma': _sem_dist(list(tel.latency)) if tel is not None else _sem_dist([]),
        'safe_visits_per_place': painful_hist, 'places_healed': tot['place_healed'],
        'relapse': tot['relapse_after_safe_visit'],
        'rejected_pi': tot['safe_visit_rejected_pi'], 'rejected_cooldown': tot['safe_visit_rejected_cooldown'],
        'body_unsafe': tot['safe_visit_body_unsafe'],
        'return_to_safe_conversion': tot['safe_visit_completed'] / max(1, tot['return_to_trauma_place'])}

    # ── behaviour ──
    ps = float(S.get('place_samples', 0.0))
    behavior = {
        'route_change': tot['route_change'], 'pain_penalty_applied': tot['pain_penalty_applied'],
        'survival_override': tot['survival_override'], 'return_to_trauma_place': tot['return_to_trauma_place'],
        'pain_goal_rest': tot['pain_goal_rest'], 'pain_goal_explore': tot['pain_goal_explore'],
        'rate_per_1000_agent_ticks': {k: per_k(tot[k]) for k in ('route_change', 'pain_penalty_applied', 'survival_override',
                                                                'return_to_trauma_place', 'safe_visit_completed')},
        'agent_ticks': agent_ticks, 'place_samples': int(ps),
        'dwell_avoid_share': float(S.get('place_samples_avoid', 0.0)) / ps if ps else 0.0,
        'dwell_caution_share': float(S.get('place_samples_caution', 0.0)) / ps if ps else 0.0,
        'override_vs_avoid_ratio': tot['survival_override'] / max(1, tot['pain_penalty_applied'])}

    # ── recognition ──
    recognition = {
        'restored': tot['memory_restored'], 'rejected_pi': tot['restore_rejected_pi'],
        'rejected_ambiguous': tot['restore_rejected_ambiguous'], 'place_id_collision': tot['place_id_collision'],
        'restore_success_share': tot['memory_restored'] / max(1, tot['memory_restored'] + tot['restore_rejected_pi'] + tot['restore_rejected_ambiguous']),
        'archived': tot['memory_archived'], 'forgotten': tot['memory_forgotten'],
        'archived_by_reason': dict(tel.archived_reason) if tel is not None else {},
        'forgotten_by_reason': dict(tel.forgotten_reason) if tel is not None else {}}

    # ── body vs memory ──
    def _body(p):
        em = getattr(p, 'emotional_memory', {})
        soma = getattr(p, '_soma_diag', None)
        return {'energy': _sem_f(getattr(p, 'energy', 0.0)), 'endurance': _sem_f(getattr(p, '_cellular_endurance', 0.0)),
                'scar': _sem_f(getattr(p, 'epistemic_scar', 0.0)),
                'grief': _sem_f(em.get('grief', 0.0)) if isinstance(em, dict) else 0.0,
                'fragments': _sem_f(soma.get('fragment_count', 1)) if isinstance(soma, dict) else 1.0}
    groups = {'painful': [], 'calm': []}
    weights = []
    for p, s in live:
        mx = max((_sem_f(r.get('pain', 0.0)) for r in s.active.values()), default=0.0)
        b = _body(p)
        try:
            w = float(s.behavioral_weight(p))
        except Exception:
            w = 0.0
        b['weight'] = w
        weights.append(w)
        groups['painful' if mx >= SpatialEmotionalMemory.STATUS_CAUTION_PAIN else 'calm'].append(b)
    body = {'weight': _sem_dist(weights)}
    for name, rows in groups.items():
        body[name] = {'n': len(rows)}
        for key in ('energy', 'endurance', 'scar', 'grief', 'fragments', 'weight'):
            body[name][key] = float(_sem_np.mean([r[key] for r in rows])) if rows else 0.0

    # ── inheritance ──
    inh_rows = list(tel.inherit_rows) if tel is not None else []
    inheritance = {
        'events': tot['memory_inherited'],
        'mean_active_places': float(_sem_np.mean([r['active'] for r in inh_rows])) if inh_rows else 0.0,
        'mean_archive_places': float(_sem_np.mean([r['archive'] for r in inh_rows])) if inh_rows else 0.0,
        'mean_total_pain': float(_sem_np.mean([r['total_pain'] for r in inh_rows])) if inh_rows else 0.0,
        'live_agents_with_inherited': int(sum(1 for _, s in live if any(r.get('inherited') for r in s.active.values()))),
        'max_generation': int(max([int(r.get('generation', 0)) for *_, r in recs], default=0)),
        'inherited_safe_visit': tot['inherited_safe_visit'], 'inherited_route_bias': tot['inherited_route_bias'],
        'inherited_retrauma': tot['inherited_retrauma']}

    # ── timeline ──
    series = list(tel.series) if tel is not None else []
    bucket_rows = []
    if tel is not None:
        for b in sorted(tel.buckets)[-40:]:
            c = tel.buckets[b]
            bucket_rows.append({'t0': int(b * tel.BUCKET), 'trauma': int(c['trauma_recorded']), 'safe_visit': int(c['safe_visit_completed']),
                                'route_bias': int(c['pain_route_bias']), 'override': int(c['survival_override']),
                                'archived': int(c['memory_archived']), 'restored': int(c['memory_restored'])})
    timeline = {'series': series[-(12 if compact else 40):], 'buckets': bucket_rows[-(10 if compact else 40):],
                'series_total': len(series)}

    # ── spatial trace + results ──
    size = int(getattr(Config, 'WORLD_SIZE', 0) or 0) if 'Config' in globals() else 0
    spatial = _sem_spatial(live, tel, size)
    if compact and spatial is not None:
        spatial = {k: v for k, v in spatial.items() if k != 'grid'}
    nav_places = [len(getattr(p, '_nav', {}).get('places', [])) for p, _ in live if isinstance(getattr(p, '_nav', None), dict)]
    feral = {}
    fs = getattr(eng, '_feral_stats', None)
    if isinstance(fs, dict):
        feral = {str(k): (float(v) if isinstance(v, (int, float)) else None) for k, v in fs.items()
                 if isinstance(v, (int, float)) and not isinstance(v, bool)}
    results = {
        'survival_share': n_live / float(max(1, n_live + n_dead)),
        'exploration_nav_places': _sem_dist(nav_places), 'exploration_coverage': spatial['coverage'] if spatial else None,
        'adaptation_index': healing['heal_ratio'], 'feral_stats': feral,
        'died_with_pain_share': (grave['died_with_pain'] / float(max(1, n_dead))) if grave else 0.0,
        'died_traumatized_share': (grave['died_traumatized'] / float(max(1, n_dead))) if grave else 0.0}

    # ── event stream ──
    events = {'counts': dict(tel.event_counts) if tel is not None else {},
              'errors': dict(tel.errors) if tel is not None else {}}

    # ── top lists ──
    rows = sorted(((pn, int(getattr(p, 'id', -1)), int(pid), int(r.get('safe_visits', 0)), int(r.get('n', 0)),
                    _sem_f(r.get('peak_pain', pn)), bool(r.get('inherited')))
                   for (p, s, pid, r), pn in zip(recs, pains)), reverse=True)
    top_places = [{'agent': a, 'place_id': pid, 'pain': round(pn, 3), 'peak': round(pk, 3), 'safe_visits': sv, 'n': n,
                   'inherited': inh_, 'status': SpatialEmotionalMemory.status_for(pn)} for pn, a, pid, sv, n, pk, inh_ in rows[:10]]
    top_agents = sorted(((int(s.counters.get('trauma_recorded', 0)), int(getattr(p, 'id', -1)), int(s.counters.get('safe_visit_completed', 0)),
                          float(sum(_sem_f(r.get('pain', 0.0)) for r in s.active.values()))) for p, s in live), reverse=True)[:8]
    top_agents = [{'agent': a, 'traumas': t, 'safe_visits': sv, 'pain_mass': round(pm, 3)} for t, a, sv, pm in top_agents]

    out = {'scope': {'tick': now, 'live': n_live, 'dead': n_dead, 'ever': n_live + n_dead},
           'trauma': trauma, 'pain': pain, 'healing': healing, 'behavior': behavior, 'recognition': recognition,
           'body': body, 'inheritance': inheritance, 'timeline': timeline, 'spatial': spatial, 'results': results,
           'events': events, 'counters': tot}
    if not compact:
        out['top_places'] = top_places
        out['top_agents'] = top_agents
        out['ledger_tail'] = list(tel.ledger)[-60:] if tel is not None else []
    return out


def _sem_print_report(S, top=6, ledger_rows=12):
    ln = '=' * 78
    pc = lambda x: f"{100.0 * _sem_f(x):.0f}%"
    sc, tr, pa, he, be = S['scope'], S['trauma'], S['pain'], S['healing'], S['behavior']
    print('\n' + ln + '\n🧠 SPATIAL EMOTIONAL MEMORY · OBSERVER\n' + ln)
    print(f"  тик {sc['tick']} | агентов: живых {sc['live']}, умерших {sc['dead']}, всего {sc['ever']} | агент-тиков {be['agent_ticks']}")

    print('\n  ── 1. ТРАВМЫ ──')
    print(f"  всего {tr['total']} ({tr['rate_per_1000_agent_ticks']:.2f} на 1000 агент-тиков) | привязано к месту {pc(tr['attached_share'])} | "
          f"ждут места {tr['pending_now']} (привязано позже {tr['pending_resolved']}, истекло {tr['pending_expired']}, вытеснено {tr['pending_overflow']})")
    print(f"  дубликатов отсечено {tr['deduplicated']} | не разрешилось сразу (накопительно, не уменьшается) {tr['unresolved']} | усилено повторной {tr['reinforced']} | "
          f"рецидив после безопасного визита {tr['relapse_after_safe_visit']}")
    pd_ = tr['per_agent']
    print(f"  у агента: ≥1 травмы у {pc(tr['agents_ge1_share'])} | среднее {pd_['mean']:.2f}, медиана {pd_['p50']:.0f}, p90 {pd_['p90']:.0f}, макс {pd_['max']:.0f} | "
          f"Джини {tr['gini']:.2f}, топ-10% агентов держат {pc(tr['top10_share'])} травм")
    if tr['by_reason']:
        print('  причина                          n    доля  инт.   боль до→после')
        for k, v in list(tr['by_reason'].items())[:10]:
            print(f"   {k[:30]:30s} {v['n']:5d}  {pc(v['share']):>4s}  {v['mean_intensity']:.2f}   {v['mean_pain_before']:.2f}→{v['mean_pain_after']:.2f}")
    if tr['by_role']:
        print('  по ролям: ' + ', '.join(f"{k}={v}" for k, v in sorted(tr['by_role'].items(), key=lambda kv: -kv[1])))

    print('\n  ── 2. ПАМЯТЬ О МЕСТАХ ──')
    d = pa['dist']
    print(f"  активных записей {d['n']} | боль: среднее {d['mean']:.2f}, медиана {d['p50']:.2f}, p90 {d['p90']:.2f}, макс {d['max']:.2f} | суммарная масса {pa['total_mass']:.1f}")
    print(f"  статусы: избегает={pa['status']['avoid']} осторожно={pa['status']['caution']} безопасно={pa['status']['safe']}")
    mx = max(pa['hist'].values()) if pa['hist'] else 0
    for lab, n in pa['hist'].items():
        print(f"   {lab:>9s} {n:5d} {'█' * int(24 * n / mx) if mx else ''}")
    print(f"  записей на агента: среднее {pa['places_per_agent']['mean']:.1f} (занятость активной памяти {pc(pa['active_occupancy'])}) | "
          f"архив на агента: среднее {pa['archive_per_agent']['mean']:.1f}, макс {pa['archive_per_agent']['max']:.0f}/{SpatialEmotionalMemory.ARCHIVE_LIMIT}, полных архивов {pa['archive_full_agents']}")
    print(f"  боль в архиве: среднее {pa['archive_pain']['mean']:.2f} | при смерти агента (остаточная): среднее {pa['dead_final_pain']['mean']:.2f}")

    print('\n  ── 3. ИСЦЕЛЕНИЕ ──')
    ld = he['latency_since_trauma']
    print(f"  безопасных визитов {he['safe_visits']} | снято боли {he['pain_healed']:.1f} из нанесённой {he['pain_inflicted']:.1f} "
          f"(индекс адаптации {pc(he['heal_ratio'])}) | средний спад за визит {he['mean_drop_per_visit']:.3f}")
    print(f"  латентность травма→первый визит: медиана {ld['p50']:.0f}, p90 {ld['p90']:.0f} тиков (n={ld['n']}) | исцелено мест {he['places_healed']} | рецидивов {he['relapse']}")
    print(f"  отклонено визитов: по pi {he['rejected_pi']}, по кулдауну {he['rejected_cooldown']}, тело небезопасно {he['body_unsafe']} | "
          f"возвраты→успешный визит {pc(he['return_to_safe_conversion'])}")
    print('  визитов на место: ' + '  '.join(f"{k}:{v}" for k, v in he['safe_visits_per_place'].items()))

    print('\n  ── 4. ПОВЕДЕНИЕ ──')
    r = be['rate_per_1000_agent_ticks']
    print(f"  обход из-за боли (route bias) {be['route_change']} ({r['route_change']:.2f}/1000) | штраф боли {be['pain_penalty_applied']} | "
          f"голод > страх {be['survival_override']} ({r['survival_override']:.2f}/1000)")
    print(f"  цели от боли: rest={be['pain_goal_rest']} explore={be['pain_goal_explore']} | возвраты в травматичное место {be['return_to_trauma_place']} ({r['return_to_trauma_place']:.2f}/1000)")
    print(f"  доля замеров в местах: избегаемых {pc(be['dwell_avoid_share'])}, осторожных {pc(be['dwell_caution_share'])} (замеров {be['place_samples']}) | "
          f"override/штраф = {be['override_vs_avoid_ratio']:.2f}")

    rc = S['recognition']
    print('\n  ── 5. УЗНАВАНИЕ МЕСТА И УПРАВЛЕНИЕ ПАМЯТЬЮ ──')
    print(f"  восстановлено по fp_mean {rc['restored']} | отклонено по pi {rc['rejected_pi']} | неоднозначных fp {rc['rejected_ambiguous']} | "
          f"коллизий place_id {rc['place_id_collision']} | доля успешных {pc(rc['restore_success_share'])}")
    print(f"  архивировано {rc['archived']} {dict(rc['archived_by_reason']) or ''} | забыто {rc['forgotten']} {dict(rc['forgotten_by_reason']) or ''}")

    bd = S['body']
    print('\n  ── 6. ТЕЛО ↔ ПАМЯТЬ (живые агенты) ──')
    print('  группа      n   энергия  вынос.  шрам   grief  фрагм.  вес боли')
    for name in ('painful', 'calm'):
        g = bd[name]
        print(f"  {('с болью' if name == 'painful' else 'спокойные'):10s} {g['n']:4d}  {g['energy']:6.2f}  {g['endurance']:6.2f}  {g['scar']:5.2f}  {g['grief']:5.2f}  {g['fragments']:5.2f}  {g['weight']:6.2f}")

    ih = S['inheritance']
    print('\n  ── 7. НАСЛЕДОВАНИЕ ──')
    print(f"  событий {ih['events']} | в среднем мест {ih['mean_active_places']:.1f} (+архив {ih['mean_archive_places']:.1f}), масса боли {ih['mean_total_pain']:.2f} | "
          f"живых с унаследованной памятью {ih['live_agents_with_inherited']} | макс. поколение {ih['max_generation']}")
    print(f"  унаследованная память в деле: безопасных визитов {ih['inherited_safe_visit']}, обходов {ih['inherited_route_bias']}, повторных травм {ih['inherited_retrauma']} | "
          f"боль унаследованных {pa['inherited_active']['mean_pain']:.2f} (n={pa['inherited_active']['n']}) vs своих {pa['own_active']['mean_pain']:.2f} (n={pa['own_active']['n']})")

    tl = S['timeline']
    print('\n  ── 8. ДИНАМИКА ВО ВРЕМЕНИ ──')
    if tl['series']:
        print('  боль (средняя):  ' + _sem_spark([x['mean_pain'] for x in tl['series']]))
        print('  избегаемых мест: ' + _sem_spark([x['avoid'] for x in tl['series']]))
        print('  живых агентов:   ' + _sem_spark([x['alive'] for x in tl['series']]))
        print('     t   живых  мест  ср.боль  макс  avoid/caut/safe  травм  визитов')
        for x in tl['series'][-8:]:
            print(f"  {x['t']:5d}  {x['alive']:5d}  {x['active_places']:4d}  {x['mean_pain']:6.2f}  {x['max_pain']:4.2f}  {x['avoid']:4d}/{x['caution']:4d}/{x['safe']:4d}  {x['trauma']:5d}  {x['safe_visits']:6d}")
    if tl['buckets']:
        print('  травмы/100 тиков: ' + _sem_spark([b['trauma'] for b in tl['buckets']]) + f"  (последние {len(tl['buckets'])} окон, макс {max(b['trauma'] for b in tl['buckets'])})")
        print('  визиты/100 тиков: ' + _sem_spark([b['safe_visit'] for b in tl['buckets']]) + f"  (макс {max(b['safe_visit'] for b in tl['buckets'])})")

    sp = S['spatial']
    print('\n  ── 9. ЭМОЦИОНАЛЬНЫЙ СЛЕД (масса боли по карте) ──')
    if sp and sp.get('grid') and sp['pain_mass'] > 0:
        ramp = ' .:-=+*#%@'
        gmax = max(max(row) for row in sp['grid']) or 1.0
        for row in sp['grid']:
            print('   ' + ''.join(ramp[min(9, int(v / gmax * 9.999))] * 2 for v in row))
        print(f"  центр {sp['center']} | радиус зон {sp['radius']} | масса {sp['pain_mass']:.1f} | горячие ячейки {sp['hot']}")
    else:
        print('  нет записей боли с координатами')

    rs = S['results']
    print('\n  ── 10. ИТОГИ ──')
    print(f"  выживаемость {pc(rs['survival_share'])} | исследование: PlaceNode на агента {rs['exploration_nav_places']['mean']:.1f}, "
          f"покрытие карты {pc(rs['exploration_coverage']) if rs['exploration_coverage'] is not None else 'n/a'} | адаптация {pc(rs['adaptation_index'])}")
    print(f"  умерли с болью {pc(rs['died_with_pain_share'])} | умерли травмированными {pc(rs['died_traumatized_share'])}")
    if rs['feral_stats']:
        print('  feral: ' + ', '.join(f"{k}={v:g}" for k, v in list(rs['feral_stats'].items())[:8]))

    if S.get('top_places'):
        print('\n  ── 11. ТОП БОЛЕЗНЕННЫХ МЕСТ ──')
        print('   агент  place_id  боль  пик   визитов  травм  унасл.  статус')
        for x in S['top_places'][:top]:
            print(f"   {x['agent']:5d}  {x['place_id']:8d}  {x['pain']:4.2f}  {x['peak']:4.2f}  {x['safe_visits']:7d}  {x['n']:5d}  {'да' if x['inherited'] else '—':>5s}   {x['status']}")
        print('  агенты с наибольшим числом травм: ' + ', '.join(f"#{a['agent']}(травм {a['traumas']}, визитов {a['safe_visits']})" for a in S['top_agents'][:5]))
    if S.get('ledger_tail'):
        print('\n  ── 12. ЛЕДЖЕР (последние события) ──')
        print('     tick  agent  event                        place  reason                 pain   result')
        for e in S['ledger_tail'][-ledger_rows:]:
            pn = e.get('pain')
            print(f"  {e['t']:7d}  {e['agent_id']:5d}  {str(e['event'])[:27]:27s}  {str(e.get('place_id') if e.get('place_id') is not None else '-'):>5s}  "
                  f"{str(e.get('reason') or '-')[:20]:20s}  {('%.2f' % _sem_f(pn)) if pn is not None else '  - '}   {e['result']}")
    ev = S['events']
    if ev['errors']:
        print('\n  ⚠️ ошибки телеметрии: ' + str(ev['errors']))
    print(ln)


def _sem_observer_report(eng=None, top=6, ledger_rows=12, dump_path=None):
    """Observer panel for Spatial Emotional Memory. Read-only: reads the engine state, runs nothing."""
    if eng is None:
        try:
            eng = _PN_LAST.get('eng')
        except NameError:
            eng = None
    if eng is None:
        print('[SEM OBSERVER] нет движка (запусти прогон)')
        return None
    if not _sem_collect(eng) and (_sem_tel(eng, create=False) is None):
        print('[SEM OBSERVER] у агентов нет SEM')
        return None
    S = _sem_stats(eng)
    _sem_print_report(S, top=top, ledger_rows=ledger_rows)
    if dump_path:
        try:
            with open(dump_path, 'w', encoding='utf-8') as f:
                _sem_json.dump(_sem_json_safe(S), f, ensure_ascii=False, indent=1)
            print(f"[SEM OBSERVER] полный дамп: {dump_path}")
        except Exception as e:
            print(f"[SEM OBSERVER] дамп не записан: {e!r}")
    return S


def _sem_stats_selftest():
    """Fast offline check of telemetry + observer on a synthetic engine (no simulation)."""
    import io as _io, contextlib as _ctx
    class _Eng:
        age = 200; patterns = []; witness = None
    class _Ag:
        def __init__(self, eng, i, role='prey'):
            self.id = i; self.age = 150; self.alive = True; self.world = eng; self.role_type = role
            self.cells = [(10 + i, 10 + i)]; self.energy = 0.8; self._active_shock = 0.0
            self._cellular_endurance = 0.5; self.epistemic_scar = 0.1; self.emotional_memory = {'grief': 0.1}
            self._soma_diag = {'fragment_count': 1}; self._nav = {'pi': [10.0 + i, 10.0 + i], 'places': [1, 2]}
            self._spatial_emotional_memory = SpatialEmotionalMemory()
        def _log_event(self, *a, **k): pass
    eng = _Eng()
    a, b = _Ag(eng, 1), _Ag(eng, 2, 'feral')
    eng.patterns = [a, b]
    tel = _sem_tel(eng)
    tel.register(a); tel.register(b)
    fp = (0.2, 0.4, 0.6, 0.8)
    sa = a._spatial_emotional_memory
    assert sa.record_trauma(a, 3, fp, 0.9, 'feral_threat', (10, 'x', 1))
    assert sa.record_trauma(a, 3, fp, 0.5, 'physiological_shock', (11, 'y', 1))
    assert b._spatial_emotional_memory.record_trauma(b, 5, fp, 0.4, 'cell_loss', (12, 'z', 2))
    sa.observe_place(a, 3, fp, 100); sa.observe_place(a, 4, (0.9, 0.8, 0.7, 0.6), 101)
    sa.observe_place(a, 3, fp, 140)
    assert sa.counters['safe_visit_completed'] == 1 and sa.stats['pain_healed'] > 0
    tel.tick(eng, 200)
    eng.patterns = [a]; b.alive = False          # b dies -> must be folded with its counters preserved
    tel.tick(eng, 201)
    assert tel.grave['n'] == 1 and tel.grave['counters']['trauma_recorded'] == 1
    S = _sem_stats(eng)
    assert S['trauma']['total'] == 3, S['trauma']['total']
    assert S['scope']['dead'] == 1 and S['scope']['live'] == 1
    assert S['healing']['safe_visits'] == 1 and 0.0 < S['healing']['heal_ratio'] < 1.0
    assert 'feral_threat' in S['trauma']['by_reason'] and S['healing']['latency_since_trauma']['n'] == 1
    assert S['trauma']['by_role'].get('feral') == 1 and S['trauma']['by_role'].get('prey') == 2
    _sem_json_safe(S)
    _sem_json_safe(_sem_stats(eng, compact=True))
    buf = _io.StringIO()
    with _ctx.redirect_stdout(buf):
        _sem_print_report(S)
    assert 'OBSERVER' in buf.getvalue()
    return {'ok': True, 'trauma_total': S['trauma']['total'], 'dead_folded': tel.grave['n'], 'report_chars': len(buf.getvalue())}


print('Spatial Emotional Memory integrated: place memory, pi-validated safe return with 20-tick cooldown, survival-over-fear priority, PlaceNode forget hook, non-empty active memory, route bias independent of threat, clean child counters, Observer/Ledger events, engine-level telemetry (dead agents folded), extended Observer statistics.')
try:
    print('[SEM selftest]', _spatial_emotional_memory_selftest())
except Exception as _sem_test_error:
    print('[SEM selftest FAILED]', repr(_sem_test_error))

try:
    print('[SEM stats selftest]', _sem_stats_selftest())
except Exception as _sem_stats_test_error:
    print('[SEM stats selftest FAILED]', repr(_sem_stats_test_error))


In [ ]:
# Ячейка MORSE-EVO (v44): тревога → страх → энергия → размножение
# Слой НАД SIGNAL / Trace / P832 / NAV / Soma / Spatial. Ставится ПОСЛЕ ячейки Spatial Emotional Memory и ДО Master Observatory.
#
# Что делает эта ячейка:
#   (1) HEARD_ALARM: услышал чужую тревожную SIGNAL-форму (foreign, recognized/partial) -> alarm_level растёт.
#       alarm_level уже управляет страхом в других слоях: спринт/бегство (NAV, SPRINT_FEAR_MIN), приоритеты целей
#       (rest/seek_help), подавление rage. Новой цели «бежать/замереть» НЕ добавляем: уже есть SIG_ADRENALINE и scream_response.
#       По умолчанию рост страха НЕ считается «всплеском» для _scream_try (иначе каждый услышанный крик
#       вызывал бы ответный крик и цепной шторм). Включается флагом HEARD_ALARM_TRIGGERS_SCREAM.
#   (2) ALARM_CREDIT: энергокредит отправителю тревоги. Это ПЕРЕВОД энергии слушатель -> отправитель
#       (обе стороны через _x2_change, суммарно ноль), а не создание энергии из воздуха.
#   (3) Метрики: heard_alarm*, alarm_credit*, hazard смерти exposed vs unexposed, comm_success, очередь/кризис Морзе.
#
# Правки в других ячейках помечены «MORSE-EVO»: Ячейка 15 (очередь, кризис, comm_success), Ячейка 6 (can_divide),
# Ячейка 20 ([SIGNAL]-строка), Ячейка 28 (строка гейтов Морзе), Ячейка 36 ([N]).
# Все флаги ниже: False/0 возвращает прежнее поведение (для абляций).
import math
import numpy as np
from collections import Counter

for _need in ("Pattern", "EvolutionEngine", "extend_class", "SignalMixin", "_sg_book", "_sg_err", "CH"):
    if _need not in globals():
        raise RuntimeError(f"MORSE-EVO: нет {_need}. Выполни ячейки сверху вниз (SIGNAL должен быть выше этой ячейки).")

_MEVO_DEFAULTS = {
    # (1) услышанная тревога -> страх
    'HEARD_ALARM_ENABLE': True,
    'HEARD_ALARM_GAIN': 0.3,               # +alarm_level * вес alarm-канала в форме. SPRINT_FEAR_MIN=0.25, поэтому 0.2 бегства не даёт
    'HEARD_ALARM_COOLDOWN': 10,            # тиков между срабатываниями у одного слушателя
    'HEARD_ALARM_TRIGGERS_SCREAM': False,  # False: рост от услышанной тревоги не считается всплеском для _scream_try
    'HEARD_ALARM_EXPOSURE_WINDOW': 50,     # окно «недавно слышал тревогу» для hazard-метрики
    # (2) энергокредит отправителю (перевод слушатель -> отправитель)
    'ALARM_CREDIT_ENABLE': True,
    'ALARM_CREDIT_AMOUNT': 0.01,
    'ALARM_CREDIT_SENDER_COOLDOWN': 10,    # один отправитель получает кредит не чаще раза в N тиков (анти-фарм)
    'ALARM_CREDIT_LISTENER_RESERVE': 0.05, # слушатель платит, только если после платежа останется не меньше резерва
    'ALARM_CREDIT_SOUL': 0.005,
    # (1b) ЭВАКУАЦИЯ: услышал тревогу -> знает, ОТКУДА она (центр отправителя) и уходит от него.
    # Раньше слушатель получал только alarm_level, а бегство (cmd kind='flee') включалось лишь при СОБСТВЕННОМ ощущении
    # угрозы: тревога без направления не давала, куда бежать. Теперь есть вектор «прочь от источника» на EVAC_TICKS тиков.
    'HEARD_ALARM_EVAC_ENABLE': True,
    'HEARD_ALARM_EVAC_TICKS': 15,          # сколько тиков длится эвакуация после последнего услышанного сигнала
    'HEARD_ALARM_EVAC_GAIN': 1.0,          # добавка к оценке направления «прочь» в move() (порог MOVE_SCORE_THR=0.3)
    'HEARD_ALARM_EVAC_MOB_BOOST': 4.0,     # множитель вероятности попытки движения на время эвакуации (потолок 1.0)
    'HEARD_ALARM_EVAC_MAX_DIST': 12.0,     # дальше этого от источника не реагируем (сигнал слышен и дальше, смысла бежать нет)
    # (4) бонус к делению за успешную коммуникацию (reinforce_sender). По умолчанию выключен: пересекается с soul_weight-бонусом
    'COMM_DIVIDE_BONUS_ENABLE': False,
    'COMM_DIVIDE_BONUS_PER': 0.002,
    'COMM_DIVIDE_BONUS_CAP': 0.06,         # DIVIDE_ERROR_THRESHOLD=0.33, так что +0.06 это примерно +18%
}
for _k, _v in _MEVO_DEFAULTS.items():
    if not hasattr(Config, _k):
        setattr(Config, _k, _v)

_MEVO_MORSE_KEYS = ('MORSE_QUEUE_ENABLE', 'MORSE_QUEUE_TTL', 'MORSE_QUEUE_RETRY', 'MORSE_CRISIS_ENABLE',
                    'MORSE_CRISIS_ENERGY', 'MORSE_CRISIS_ALARM', 'MORSE_CRISIS_COST_MULT', 'MORSE_CRISIS_OVERRIDE_NAV')
_MEVO_CHAN_KEYS = ('tx_nav_deferred', 'tx_nav_queued', 'tx_queue_dup', 'tx_queue_skip_nointent', 'tx_queue_overflow',
                   'tx_queue_attempt', 'tx_queue_sent', 'tx_queue_expired', 'tx_queue_dropped_hard',
                   'tx_crisis_nav_override', 'tx_send_crisis', 'tx_send_normal')


class MorseEvoPatternMixin:
    def _sig_listen(self, field, t):
        if (not getattr(Config, 'HEARD_ALARM_ENABLE', False) or not Config.SIG_ENABLE
                or self.world is None or not self.cells):
            return super()._sig_listen(field, t)
        rec_ = getattr(self, '_sig_recent', None)
        last_ = rec_[-1] if rec_ else None
        result = super()._sig_listen(field, t)
        try:
            self._mevo_after_listen(int(t), last_)
        except Exception as e:
            _sg_err(_sg_book(self.world), 'heard_alarm', e)
        return result

    def _mevo_after_listen(self, now, last_before):
        rec_ = getattr(self, '_sig_recent', None)
        if not rec_:
            return
        e = rec_[-1]
        if e is last_before or e[0] != now:      # в этом вызове новой записи не появилось
            return
        te, form, strength, src, ctx, state, cat = e
        if cat != 'foreign' or src is None or int(src) == int(self.id):
            return
        if state not in ('recognized', 'partial') or not self._sig_is_alarm(form):
            return
        st = _sg_book(self.world).st
        w = min(1.0, float(form['spectrum'].get(CH['signal_alarm'], 0.0)))
        if now - int(getattr(self, '_heard_alarm_t', -10 ** 9)) < int(Config.HEARD_ALARM_COOLDOWN):
            st['heard_alarm_cooldown'] += 1
            return
        before = float(getattr(self, 'alarm_level', 0.0))
        self.alarm_level = min(1.0, before + float(Config.HEARD_ALARM_GAIN) * w)
        delta = float(self.alarm_level) - before
        if not Config.HEARD_ALARM_TRIGGERS_SCREAM and hasattr(self, '_scream_prev_alarm'):
            # рост страха от услышанной тревоги не должен выглядеть для _scream_try как собственный «всплеск»
            self._scream_prev_alarm = float(self._scream_prev_alarm) + delta
        n = int(getattr(self, '_heard_alarm_n', 0)) + 1
        self._heard_alarm_n = n
        self._heard_alarm_t = now
        st['heard_alarm'] += 1
        if n == 1:
            st['heard_alarm_agents'] += 1
        self._log_event('heard_alarm', from_=int(src), strength=round(w, 3),
                        alarm_before=round(before, 3), alarm_after=round(float(self.alarm_level), 3))
        if getattr(Config, 'HEARD_ALARM_EVAC_ENABLE', False) and self.role_type != 'feral':
            self._mevo_start_evac(now, int(src))
        if Config.ALARM_CREDIT_ENABLE:
            self._mevo_pay_credit(now, int(src), w)

    def _mevo_start_evac(self, now, src):
        st = _sg_book(self.world).st
        sender = self.world.pattern_dict.get(src)
        if sender is None or sender is self or not getattr(sender, 'alive', False) or not sender.cells:
            st['evac_no_sender'] += 1
            return
        W = int(Config.WORLD_SIZE)
        sx, sy = sender.get_center()
        cx, cy = self.get_center()
        dx = ((cx - sx + W / 2.0) % W) - W / 2.0
        dy = ((cy - sy + W / 2.0) % W) - W / 2.0
        n = math.hypot(dx, dy)
        if n > float(Config.HEARD_ALARM_EVAC_MAX_DIST):
            st['evac_too_far'] += 1
            return
        if n < 1e-6:                              # стоим в одной точке: направление выбирает детерминированный хеш
            ang = 2.0 * math.pi * phi_hash(self.id, now, 9151)
            dx, dy, n = math.cos(ang), math.sin(ang), 1.0
        was = getattr(self, '_evac', None) is not None
        self._evac = (dx / n, dy / n, int(now) + int(Config.HEARD_ALARM_EVAC_TICKS))
        st['evac_start'] += 1
        if not was:
            st['evac_episodes'] += 1
        if not getattr(self, '_evac_ever', False):
            self._evac_ever = True
            st['evac_agents'] += 1

    def _x2_change(self, name, new_value, offered=None):
        # observer: фиксирует, ЧЕМ агент ушёл в отрицательную энергию (пересечение через 0). Поведение не меняет.
        old = float(self.energy)
        actual = super()._x2_change(name, new_value, offered)
        try:
            if old >= 0.0 and float(self.energy) < 0.0 and self.world is not None:
                st = _sg_book(self.world).st
                st['x2neg_cross'] += 1
                st['x2neg_by__' + str(name)[:48]] += 1
        except Exception:
            pass
        return actual

    def _mevo_pay_credit(self, now, src, w):
        st = _sg_book(self.world).st
        sender = self.world.pattern_dict.get(src)
        if sender is None or sender is self or not getattr(sender, 'alive', False) or not sender.cells:
            st['alarm_credit_sender_gone'] += 1
            return
        if now - int(getattr(sender, '_alarm_credit_last_t', -10 ** 9)) < int(Config.ALARM_CREDIT_SENDER_COOLDOWN):
            st['alarm_credit_sender_cd'] += 1
            return
        amt = float(Config.ALARM_CREDIT_AMOUNT) * w
        if amt <= 0.0 or float(self.energy) < amt + float(Config.ALARM_CREDIT_LISTENER_RESERVE):
            st['alarm_credit_no_energy'] += 1
            return
        # Сначала зачисляем отправителю и берём фактический прирост (энергия может упереться в потолок),
        # затем списываем у слушателя ровно столько же: сумма по двум счетам остаётся нулевой.
        gain = float(sender._x2_change('X2.alarm_credit_in', float(sender.energy) + amt))
        if gain <= 0.0:
            st['alarm_credit_sender_full'] += 1
            return
        self._x2_change('X2.alarm_credit_out', float(self.energy) - gain)
        sender.soul_weight = min(1.0, float(sender.soul_weight) + float(Config.ALARM_CREDIT_SOUL) * w)
        sender._alarm_credit_last_t = now
        sender._alarm_credit_total = float(getattr(sender, '_alarm_credit_total', 0.0)) + gain
        st['alarm_credit'] += 1
        st['alarm_credit_x1000'] += int(round(gain * 1000))
        self._log_event('alarm_response_credit', to=int(sender.id), amount=round(gain, 5))
        sender._log_event('alarm_credit_received', from_=int(self.id), amount=round(gain, 5))


def _mevo_summary(eng):
    st = _sg_book(eng).st
    alive = [p for p in eng.patterns if p.alive]
    lex = getattr(eng, 'EMERGENT_LEXICON', None)
    chan = getattr(lex, 'chan', None) or {}
    g = lambda k: int(st.get(k, 0))
    cs = [int(getattr(p, '_comm_success_count', 0)) for p in alive]
    tk_all, tk_exp = g('mevo_agent_ticks'), g('mevo_agent_ticks_exposed')
    tk_un = tk_all - tk_exp
    d_exp, d_un = g('mevo_deaths_exposed'), g('mevo_deaths_unexposed')
    h_exp = (d_exp / tk_exp) if tk_exp else None
    h_un = (d_un / tk_un) if tk_un else None
    s_cr, s_nm = int(chan.get('tx_send_crisis', 0)), int(chan.get('tx_send_normal', 0))
    return {
        'config': {k: getattr(Config, k, None) for k in list(_MEVO_DEFAULTS) + list(_MEVO_MORSE_KEYS)},
        'heard_alarm': {
            'events': g('heard_alarm'), 'agents_ever': g('heard_alarm_agents'), 'cooldown_skipped': g('heard_alarm_cooldown'),
            'alive_exposed_now': sum(1 for p in alive if getattr(p, '_heard_alarm_n', 0) > 0),
            'evac': {k: g('evac_' + k) for k in ('start', 'episodes', 'agents', 'no_sender', 'too_far')},
        },
        'alarm_credit': {
            'paid': g('alarm_credit'), 'energy_total': round(g('alarm_credit_x1000') / 1000.0, 4),
            'skipped': {k: g('alarm_credit_' + k) for k in ('sender_gone', 'sender_cd', 'no_energy', 'sender_full')},
            'alive_senders_credited': sum(1 for p in alive if getattr(p, '_alarm_credit_total', 0.0) > 0),
        },
        'death_hazard': {
            'window': int(Config.HEARD_ALARM_EXPOSURE_WINDOW),
            'deaths_exposed': d_exp, 'deaths_unexposed': d_un,
            'feral_deaths_exposed': g('mevo_deaths_exposed_feral'), 'feral_deaths_unexposed': g('mevo_deaths_unexposed_feral'),
            'agent_ticks_exposed': tk_exp, 'agent_ticks_unexposed': tk_un,
            'hazard_exposed': (round(h_exp, 6) if h_exp is not None else None),
            'hazard_unexposed': (round(h_un, 6) if h_un is not None else None),
            'ratio_exposed_over_unexposed': (round(h_exp / h_un, 4) if h_exp is not None and h_un else None),
        },
        'comm_success': {
            'avg': (round(float(np.mean(cs)), 4) if cs else 0.0), 'max': (max(cs) if cs else 0),
            'share_ge1': (round(sum(1 for c in cs if c > 0) / len(cs), 4) if cs else 0.0),
            'divide_bonus_enabled': bool(getattr(Config, 'COMM_DIVIDE_BONUS_ENABLE', False)),
        },
        'morse_queue_crisis': {**{k: int(chan.get(k, 0)) for k in _MEVO_CHAN_KEYS},
                               'crisis_share_of_sends': (round(s_cr / (s_cr + s_nm), 4) if (s_cr + s_nm) else None)},
    }


class MorseEvoEngineMixin:
    def _record_death_drag(self, p, cause):
        super()._record_death_drag(p, cause)
        try:
            self._mevo_note_death(p, cause)
        except Exception as e:
            _sg_err(_sg_book(self), 'mevo_death', e)

    def _mevo_note_death(self, p, cause):
        if getattr(p, '_mevo_death_noted', False):
            return
        p._mevo_death_noted = True
        st = _sg_book(self).st
        t0 = getattr(p, '_heard_alarm_t', None)
        exposed = t0 is not None and (int(self.age) - int(t0)) <= int(Config.HEARD_ALARM_EXPOSURE_WINDOW)
        suf = 'exposed' if exposed else 'unexposed'
        st['mevo_deaths_' + suf] += 1
        try:   # где умер: по последнему состоянию трекера зон (приближение: до одного тика)
            tr = getattr(self, '_sanc_track_d', None)
            rec = tr['agents'].get(p.id) if tr else None
            where = 'inside' if (rec is not None and rec.get('in')) else 'outside'
            st['sanc_death_' + where] += 1
            st['sanc_death_' + where + '__' + str(cause)[:40]] += 1
        except Exception:
            pass
        if 'feral' in str(cause) or 'scream_kill' in str(cause):
            st['mevo_deaths_' + suf + '_feral'] += 1

    def _audit_close_tick(self, *args, **kwargs):
        super()._audit_close_tick(*args, **kwargs)
        try:
            st = _sg_book(self).st
            now = int(self.age)
            win = int(Config.HEARD_ALARM_EXPOSURE_WINDOW)
            n = ne = 0
            for p in self.patterns:
                if not p.alive:
                    continue
                n += 1
                if float(p.energy) < 0.0:      # observer: отрицательная энергия по ролям
                    st['x2neg_agent_ticks'] += 1
                    st['x2neg_role__' + str(p.role_type)] += 1
                t0 = getattr(p, '_heard_alarm_t', None)
                if t0 is not None and now - int(t0) <= win:
                    ne += 1
            st['mevo_agent_ticks'] += n
            st['mevo_agent_ticks_exposed'] += ne
        except Exception as e:
            _sg_err(_sg_book(self), 'mevo_tick', e)

    def audit_summary(self):
        S = super().audit_summary()
        try:
            S['morse_evo'] = _mevo_summary(self)
        except Exception as e:
            _sg_err(_sg_book(self), 'mevo_summary', e)
        return S


Pattern = extend_class(Pattern, MorseEvoPatternMixin)
for _mevo_name in ('EvolutionEngine', 'LedgerEngine', 'RichEngine'):
    _mevo_cls = globals().get(_mevo_name)
    if _mevo_cls is not None:
        globals()[_mevo_name] = extend_class(_mevo_cls, MorseEvoEngineMixin)

print("MORSE-EVO v44: heard_alarm=%s (gain=%s, triggers_scream=%s) | alarm_credit=%s (%s) | morse_queue=%s (ttl=%s) | "
      "morse_crisis=%s | comm_divide_bonus=%s" % (
          Config.HEARD_ALARM_ENABLE, Config.HEARD_ALARM_GAIN, Config.HEARD_ALARM_TRIGGERS_SCREAM,
          Config.ALARM_CREDIT_ENABLE, Config.ALARM_CREDIT_AMOUNT,
          getattr(Config, 'MORSE_QUEUE_ENABLE', None), getattr(Config, 'MORSE_QUEUE_TTL', None),
          getattr(Config, 'MORSE_CRISIS_ENABLE', None), Config.COMM_DIVIDE_BONUS_ENABLE))


In [ ]:
# Ячейка 36
import json
import os
import math
import time
from collections import Counter, defaultdict

import numpy as np

MASTER_VERSION = "0.2.0"
SNAPSHOT_EVERY = 10
RECORD_CAP = 400_000
W_MAX = 15
W = 100
DIR = "/content/drive/MyDrive/832_ledgers"


def _master_json_for(log_path):
    base = os.path.basename(log_path)
    if base.endswith(".json"):
        base = base[:-5]
    return f"{DIR}/master_{base}.json"


def _json_default(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, set): return list(o)
    if isinstance(o, tuple): return list(o)
    if isinstance(o, defaultdict): return dict(o)
    raise TypeError(f"несериализуемо: {type(o).__name__}")


def _sanitize_json(obj):
    if isinstance(obj, dict): return {k: _sanitize_json(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [_sanitize_json(v) for v in obj]
    if isinstance(obj, (float, np.floating)): return float(obj) if np.isfinite(float(obj)) else None
    if isinstance(obj, (np.integer,)): return int(obj)
    if isinstance(obj, (np.bool_,)): return bool(obj)
    if isinstance(obj, set): return [_sanitize_json(v) for v in obj]
    if isinstance(obj, defaultdict): return _sanitize_json(dict(obj))
    return obj


def _safe_load(path):
    if not path or not os.path.exists(path): return None, {"error": "missing", "path": path}
    size = os.path.getsize(path)
    try:
        with open(path, "r", encoding="utf-8") as f: return json.load(f), {"size": size, "ok": True}
    except json.JSONDecodeError as e:
        return None, {"error": "decode", "size": size, "pos": e.pos, "msg": str(e)[:180]}
    except Exception as e:
        return None, {"error": "io", "size": size, "msg": str(e)[:180]}


def _stat(vals):
    v = []
    for x in vals:
        if x is None: continue
        if not isinstance(x, (int, float, np.integer, np.floating)): continue
        if not np.isfinite(float(x)): continue
        v.append(float(x))
    if not v: return {"n": 0, "value": None}
    a = np.asarray(v, dtype=float)
    return {"n": int(len(a)), "value": float(a.mean()), "p10": float(np.percentile(a, 10)),
            "p50": float(np.percentile(a, 50)), "p90": float(np.percentile(a, 90)),
            "min": float(a.min()), "max": float(a.max()), "mean": float(a.mean())}


def _frac(bools):
    v = [bool(b) for b in bools if b is not None]
    if not v: return {"n": 0, "value": None}
    return {"n": len(v), "value": sum(v) / len(v)}


def _ss(st):
    if not st or st.get("n", 0) == 0: return "n/a"
    if "p50" in st and st.get("p50") is not None:
        return f"p10={st['p10']:.3f} p50={st['p50']:.3f} p90={st['p90']:.3f} (n={st['n']})"
    if st.get("value") is not None: return f"{st['value']:.3f} (n={st['n']})"
    return f"n={st['n']}"


def _pm(v, n=None):
    if v is None: return "n/a"
    try: s = f"{float(v):.3f}"
    except Exception: return str(v)
    return s + (f" (n={n})" if n is not None else "")


def _corr(a, b):
    pairs = []
    for x, y in zip(a, b):
        if not isinstance(x, (int, float, np.integer, np.floating)): continue
        if not isinstance(y, (int, float, np.integer, np.floating)): continue
        x = float(x); y = float(y)
        if not np.isfinite(x) or not np.isfinite(y): continue
        pairs.append((x, y))
    if len(pairs) < 5: return {"n": len(pairs), "r": None}
    xa = np.asarray([p[0] for p in pairs], dtype=float)
    ya = np.asarray([p[1] for p in pairs], dtype=float)
    if xa.std() < 1e-9 or ya.std() < 1e-9: return {"n": len(pairs), "r": None}
    return {"n": len(pairs), "r": float(np.corrcoef(xa, ya)[0, 1])}


def _entropy(counter):
    n = sum(counter.values())
    if n <= 0: return None
    return float(-sum((c / n) * math.log2(c / n) for c in counter.values() if c > 0))


def _match_pairs(speaks, w_max=W_MAX):
    sp = [s for s in speaks if s.get("to") not in (None, -1) and s.get("utt")]
    by_pair = defaultdict(list)
    for s in sp: by_pair[(s["pid"], s["to"])].append(s)
    pairs = []
    used = defaultdict(set)
    for (A, B), a_list in by_pair.items():
        b_list = sorted(by_pair.get((B, A), []), key=lambda s: s.get("t", 0))
        for a_s in sorted(a_list, key=lambda s: s.get("t", 0)):
            tA = a_s.get("t", 0)
            for b_s in b_list:
                tB = b_s.get("t", 0)
                if tB <= tA: continue
                if tB - tA > w_max: break
                if tB in used[(B, A)]: continue
                used[(B, A)].add(tB)
                pairs.append({"tA": tA, "A": a_s["pid"], "B": b_s["pid"], "tB": tB, "dt": tB - tA,
                              "utt_A": list(a_s["utt"]), "utt_B": list(b_s["utt"]),
                              "st_A": a_s.get("st"), "st_B": b_s.get("st")})
                break
    return pairs


def _dedup_dlg_stats(snaps):
    best = defaultdict(lambda: defaultdict(float))
    last = defaultdict(dict)
    coverage = defaultdict(lambda: {"agents": set(), "observations": 0, "non_monotonic": 0})
    for s in snaps:
        ds = s.get("dlg_stats")
        if not isinstance(ds, dict): continue
        pid = s.get("id")
        if pid is None: continue
        for k, v in ds.items():
            if not isinstance(v, (int, float, np.integer, np.floating)): continue
            v = float(v)
            coverage[k]["agents"].add(pid)
            coverage[k]["observations"] += 1
            if k in last[pid] and v < last[pid][k]: coverage[k]["non_monotonic"] += 1
            last[pid][k] = v
            if v > best[pid][k]: best[pid][k] = v
    out_cov = {}
    for k, c in coverage.items():
        out_cov[k] = {"agents": len(c["agents"]), "observations": c["observations"], "non_monotonic": c["non_monotonic"]}
    return best, out_cov


def raw_inventory(log_raw, ledger_raw):
    inv = {}
    inv["log_header_keys"] = sorted(log_raw.get("header", {}).keys())
    inv["log_top_keys"] = sorted(log_raw.keys())
    for coll in ("tick_rows", "agent_snaps", "events", "speaks", "dialogues", "cluster_births"):
        arr = log_raw.get(coll, [])
        inv[f"{coll}_len"] = len(arr)
        if arr:
            first = arr[0]
            if isinstance(first, dict): inv[f"{coll}_keys"] = sorted(first.keys())
            else: inv[f"{coll}_type"] = type(first).__name__
    snaps = log_raw.get("agent_snaps", [])
    if snaps:
        nested = set()
        for s in snaps[:100]:
            if isinstance(s.get("dlg_stats"), dict): nested.update(s["dlg_stats"].keys())
        inv["agent_snaps_dlg_stats_keys"] = sorted(nested)
    if ledger_raw:
        inv["ledger_header_keys"] = sorted(ledger_raw.get("header", {}).keys())
        inv["ledger_top_keys"] = sorted(ledger_raw.keys())
        lrows = ledger_raw.get("rows", [])
        if lrows and isinstance(lrows[0], dict):
            inv["ledger_rows_keys"] = sorted(lrows[0].keys())
            ev_keys = set()
            for r in lrows[:100]:
                if isinstance(r.get("ev"), dict): ev_keys.update(r["ev"].keys())
            inv["ledger_rows_ev_keys"] = sorted(ev_keys)
        llife = ledger_raw.get("lifecycle", {})
        if isinstance(llife, dict): first = next(iter(llife.values()), None)
        elif isinstance(llife, list): first = llife[0] if llife else None
        else: first = None
        inv["ledger_lifecycle_type"] = "dict" if isinstance(llife, dict) else "list" if isinstance(llife, list) else type(llife).__name__
        inv["ledger_lifecycle_keys"] = sorted(first.keys()) if isinstance(first, dict) else []
    return inv


def print_inventory(inv):
    print("\n" + "-" * 78)
    print("[RAW INVENTORY]")
    for k, v in inv.items():
        if isinstance(v, list) and len(v) > 30: print(f"  {k}: [{len(v)} keys] {v[:30]}...")
        else: print(f"  {k}: {v}")


def sec_A_identity(log_raw, ledger_raw, cov):
    h = log_raw.get("header", {})
    lh = ledger_raw.get("header", {}) if ledger_raw else {}
    lrows = ledger_raw.get("rows", []) if ledger_raw else []
    tag = h.get("tag")
    if tag is None and h.get("seed") is not None: tag = f"seed{h['seed']}_steps{h.get('steps')}"
    input_fp = None
    for d in (h, lh):
        if input_fp is not None: break
        if isinstance(d, dict):
            # input_fp_before — то, что реально пишет ledger-header (раньше ключ не искался -> MISSING)
            for k in ("input_fp", "input_fingerprint", "input_hash", "world_input_fp", "input_fp_before"):
                if d.get(k) is not None: input_fp = d.get(k); break
    _fp_b = lh.get("input_fp_before") if isinstance(lh, dict) else None
    _fp_a = lh.get("input_fp_after") if isinstance(lh, dict) else None
    _new_fp = isinstance(lh, dict) and ("output_fp_after" in lh)   # ledger со снятыми раздельно входом и выходом
    if _new_fp:
        _fp_stable = lh.get("input_fp_stable")   # input_fp_after снят ПОСЛЕ восстановления входов
        if _fp_stable is None and _fp_b is not None and _fp_a is not None:
            _fp_stable = bool(_fp_b == _fp_a)
    else:
        _fp_stable = None   # старый ledger: after снят ДО восстановления -> сравнение до/после ничего не значит: UNKNOWN, не «нестабильно»
    return {"tag": tag, "seed": h.get("seed"), "steps": h.get("steps"), "mult": h.get("mult"),
            "wall_seconds": h.get("wall_seconds"), "complete": h.get("complete"),
            "empathy_pull_override": h.get("empathy_pull_override"),
            "neighbor_sync_override": h.get("neighbor_sync_override"),
            "content_response_enabled": h.get("content_response_enabled", "n/a"),
            "config_hash": lh.get("config_hash", h.get("config_hash")),
            "input_fp": input_fp if input_fp is not None else "MISSING",
            "input_fp_after": _fp_a,
            "input_fp_stable": _fp_stable,
            "input_fp_semantics": ("after = входы ПОСЛЕ восстановления guard'ом (должны == before)" if _new_fp
                                   else "legacy: after снят до восстановления -> не сравнивается"),
            "output_fp_after": lh.get("output_fp_after") if isinstance(lh, dict) else None,
            "run_mutated_files": lh.get("run_mutated_files") if isinstance(lh, dict) else None,
            "input_fp_diff_after_restore": lh.get("input_fp_diff_after_restore") if isinstance(lh, dict) else None,
            "overrides": lh.get("overrides", h.get("overrides")),
            "chain_first": lrows[0].get("chain") if lrows else None,
            "chain_last": lrows[-1].get("chain") if lrows else None,
            "final_n": lrows[-1].get("n") if lrows else None,
            "ledger_found": ledger_raw is not None, "log_size": cov.get("size")}


def sec_B_coverage(log_raw, ledger_raw, prev_path):
    snaps = log_raw.get("agent_snaps", []); events = log_raw.get("events", [])
    speaks = log_raw.get("speaks", []); dialogues = log_raw.get("dialogues", [])
    rows_log = log_raw.get("tick_rows", []); rows_led = ledger_raw.get("rows", []) if ledger_raw else []
    ticks_expected = log_raw.get("header", {}).get("steps", 0)
    ticks_log = len(rows_log); ticks_led = len(rows_led)
    snap_ticks_expected = set(range(0, ticks_expected, SNAPSHOT_EVERY))
    snap_ticks_actual = {s.get("t") for s in snaps if s.get("t") is not None}
    snaps_expected = sum(r.get("n", 0) for r in rows_led if r.get("t") in snap_ticks_expected)
    snaps_missing = sorted(snap_ticks_expected - snap_ticks_actual)[:20]
    nan_fields = ["uc", "drag", "tension", "scar", "grief", "grat", "gap", "nbr", "nbrN", "feral_near", "bind_loc", "body_mem"]
    nan_frac = {}
    for f in nan_fields:
        vals = [s.get(f) for s in snaps if f in s]
        if not vals: continue
        nn = sum(1 for v in vals if v is None or (isinstance(v, float) and not np.isfinite(v)))
        nan_frac[f] = round(nn / len(vals), 4)
    sp_dir = sum(1 for s in speaks if s.get("to") not in (None, -1))
    dir_frac = sp_dir / max(len(speaks), 1)
    capped = {k: len(log_raw.get(k, [])) >= RECORD_CAP for k in ("events", "dialogues", "speaks")}
    prev_info = {"path": prev_path, "exists": bool(prev_path and os.path.exists(prev_path))}
    if prev_info["exists"]:
        prev_raw, prev_load = _safe_load(prev_path)
        prev_info.update({"size": prev_load.get("size"), "status": "valid" if prev_raw is not None else prev_load.get("error"),
                          "decode_pos": prev_load.get("pos"), "message": prev_load.get("msg")})
    return {"ticks_expected": ticks_expected, "ticks_log": ticks_log, "ticks_ledger": ticks_led,
            "snaps_recorded": len(snaps), "snap_ticks_expected": len(snap_ticks_expected),
            "snap_ticks_actual": len(snap_ticks_actual), "snaps_expected_approx": snaps_expected,
            "snap_ticks_missing": snaps_missing, "events": len(events), "speaks": len(speaks),
            "dialogues": len(dialogues), "directed_speaks_frac": round(dir_frac, 4),
            "nan_frac_by_field": nan_frac, "capped": capped,
            "log_errors": log_raw.get("errors", {}), "prev": prev_info}


def sec_C_population(ledger_raw):
    if not ledger_raw: return {"n/a": "ledger не найден"}
    rows = ledger_raw.get("rows", []); life = ledger_raw.get("lifecycle", {})
    if not rows: return {"n/a": "rows пусто"}
    if isinstance(life, dict): life_records = list(life.values())
    elif isinstance(life, list): life_records = life
    else: life_records = []
    ns = [r.get("n", 0) for r in rows]
    deaths = [r for r in life_records if isinstance(r, dict) and r.get("death_t") is not None]
    births_tot = sum(r.get("births", 0) for r in rows); deaths_tot = sum(r.get("deaths", 0) for r in rows)
    divs_tot = sum(r.get("divisions", 0) for r in rows)
    causes = Counter(r.get("death_cause") for r in deaths)
    ages = [r.get("age_at_death") for r in deaths if r.get("age_at_death") is not None]
    agent_ticks = sum(ns)
    _first = rows[0]
    _n_pre = int(_first.get("n", 0)) - int(_first.get("births", 0)) + int(_first.get("deaths", 0))
    _exp_end = _n_pre + births_tot - deaths_tot
    _bk = Counter()
    for r in rows: _bk.update(r.get("births_by_kind") or {})
    _sem = "alive" if "n_pending_dead" in rows[-1] else "legacy_list"
    return {"n_start": ns[0], "n_min": min(ns), "n_max": max(ns), "n_end": ns[-1],
            "pop_semantics": _sem, "pending_dead_end": rows[-1].get("n_pending_dead"),
            "births_by_kind": (dict(_bk) or None),
            "pop_balance": {"n_pre_run": _n_pre, "expected_end": _exp_end, "n_end": ns[-1], "ok": bool(_exp_end == ns[-1])},
            "births": births_tot, "deaths": deaths_tot, "divisions": divs_tot, "agent_ticks": agent_ticks,
            "deaths_per_1000_at": round(1000 * deaths_tot / max(agent_ticks, 1), 2),
            "divisions_per_1000_at": round(1000 * divs_tot / max(agent_ticks, 1), 2),
            "causes_top": dict(causes.most_common(10)), "age_at_death": _stat(ages),
            "deaths_with_record": len(deaths),
            "deaths_unattributed": sum(v for k_, v in causes.items() if k_ is None or str(k_).startswith("unlogged:")),
            "was_subject": sum(1 for r in deaths if r.get("was_subject")),
            "was_narrative": sum(1 for r in deaths if r.get("was_narrative")),
            "protection_above_thr_at_death": sum(1 for r in deaths if (r.get("protection_before_tick") or 0) > 0.2)}


def sec_D_communication(log_raw):
    speaks = log_raw.get("speaks", []); dialogues = log_raw.get("dialogues", [])
    births = log_raw.get("cluster_births", []); tick_rows = log_raw.get("tick_rows", [])
    uniq = Counter(tuple(s.get("utt") or []) for s in speaks if s.get("utt"))
    top10 = uniq.most_common(10)
    recipients = {s.get("to") for s in speaks if s.get("to") not in (None, -1)}
    senders = {s["pid"] for s in speaks if s.get("utt")}
    nw = (tick_rows[-1]["t"] // W + 1) if tick_rows else (max((s.get("t", 0) for s in speaks), default=-1) // W + 1 if speaks else 0)
    by_window = []
    for w in range(nw):
        sw = [s for s in speaks if w * W <= s.get("t", -1) < (w + 1) * W and s.get("utt")]
        dw = [d for d in dialogues if w * W <= d.get("t", -1) < (w + 1) * W]
        emo = Counter(s["utt"][1] for s in sw if len(s["utt"]) > 1)
        intn = Counter(s["utt"][2] for s in sw if len(s["utt"]) > 2)
        utt_counter = Counter(tuple(s["utt"]) for s in sw)
        ext = Counter(s["utt"][3] if len(s["utt"]) > 3 else None for s in sw)
        by_window.append({"w": w, "speaks": len(sw), "speakers": len({s["pid"] for s in sw}), "dialogues": len(dw),
                          "GRIEF": emo.get("GRIEF", 0), "JOY": emo.get("JOY", 0), "CALM": emo.get("CALM", 0),
                          "SEEK": intn.get("SEEK", 0), "HELP": intn.get("HELP", 0),
                          "ANSWER": sum(1 for s in sw if "ANSWER" in s["utt"]),
                          "KNOT": ext.get("KNOT", 0), "GAP": ext.get("GAP", 0), "DRAG": ext.get("DRAG", 0),
                          "unique_utterances": len(utt_counter), "H_utterance": _entropy(utt_counter),
                          "H_emotion": _entropy(emo), "H_intent": _entropy(intn)})
    fid_by_w = {}
    for r in tick_rows:
        w = r["t"] // W
        fid_by_w.setdefault(w, []).append(r.get("fidelity"))
    fid = {}
    for w, v in fid_by_w.items():
        vv = [float(x) for x in v if isinstance(x, (int, float)) and np.isfinite(float(x))]
        fid[w] = sum(vv) / len(vv) if vv else None
    return {"speaks": len(speaks), "dialogues": len(dialogues), "unique_utterances": len(uniq),
            "top10_utt": top10, "speakers": len(senders), "recipients": len(recipients),
            "only_speak": len(senders - recipients), "only_listen": len(recipients - senders),
            "clusters": len(births), "by_window": by_window, "fidelity_by_window": fid, "n_windows": nw}


def sec_E1_grief_response(pairs, log_raw):
    speaks = log_raw.get("speaks", [])
    nw = (max((s.get("t", 0) for s in speaks), default=-1) // W + 1) if speaks else 0
    def _calc(pp):
        if not pp: return {"n": 0, "calm": 0, "grief": 0, "rate_calm": None, "rate_grief": None}
        n_calm = sum(1 for p in pp if len(p["utt_B"]) > 1 and p["utt_B"][1] == "CALM")
        n_grief = sum(1 for p in pp if len(p["utt_B"]) > 1 and p["utt_B"][1] == "GRIEF")
        return {"n": len(pp), "calm": n_calm, "grief": n_grief,
                "rate_calm": n_calm / len(pp), "rate_grief": n_grief / len(pp)}
    with_grief_a = [p for p in pairs if len(p["utt_A"]) > 1 and p["utt_A"][1] == "GRIEF"]
    without_grief_a = [p for p in pairs if len(p["utt_A"]) > 1 and p["utt_A"][1] != "GRIEF"]
    by_w = []
    for w in range(nw):
        wp = [p for p in pairs if w * W <= p["tA"] < (w + 1) * W]
        wg = [p for p in wp if len(p["utt_A"]) > 1 and p["utt_A"][1] == "GRIEF"]
        wo = [p for p in wp if len(p["utt_A"]) > 1 and p["utt_A"][1] != "GRIEF"]
        by_w.append({"w": w, "grief": _calc(wg), "other": _calc(wo)})
    dt = [p["dt"] for p in with_grief_a]
    return {"definition": "A->B followed by B->A within W_MAX ticks; association only",
            "w_max": W_MAX, "global": {"grief_a": _calc(with_grief_a), "other_a": _calc(without_grief_a)},
            "grief_response_delay": _stat(dt), "by_window": by_w, "n_windows": nw}


def sec_E_state_response(log_raw, horizons=(10, 20), only_grief_a=False):
    snaps = log_raw.get("agent_snaps", []); speaks = log_raw.get("speaks", [])
    meta = {"state_resolution_ticks": SNAPSHOT_EVERY, "exact_1_5_tick_response_available": False,
            "causal_claim": False, "control_rule": "nearest same-t_snap state; candidate has no directed communication in [t_snap, t_snap+max_h]; pool size is reported",
            "selection_unit": "(t_snap, recipient)"}
    if not snaps or not speaks: return {"meta": meta, "n/a": "нет снапшотов или speaks"}
    by_t = defaultdict(dict)
    for s in snaps:
        if all(k in s for k in ("grief", "grat", "uc")): by_t[s["t"]][s["id"]] = (s["grief"], s["grat"], s["uc"])
    directed_at = defaultdict(set)
    for s in speaks:
        if s.get("to") not in (None, -1):
            directed_at[s["t"]].add(s["pid"]); directed_at[s["t"]].add(s["to"])
    def has_dir(pid, t0, t1):
        return any(pid in directed_at.get(t, set()) for t in range(t0, t1 + 1))
    max_h = max(horizons)
    treat = {h: {"grief": [], "grat": [], "uc": []} for h in horizons}
    ctrl = {h: {"grief": [], "grat": [], "uc": []} for h in horizons}
    observed = 0; eligible = 0; matched = 0; duplicates = 0; no_control = 0
    control_pool_sizes = []; analysed_by_h = Counter(); seen = set()
    for s in speaks:
        if s.get("to") in (None, -1): continue
        if only_grief_a:
            if not s.get("utt") or len(s["utt"]) < 2: continue
            if s["utt"][1] != "GRIEF": continue
        observed += 1; rec = s["to"]; t_ev = s["t"]; t_snap = (t_ev // SNAPSHOT_EVERY) * SNAPSHOT_EVERY
        if t_snap not in by_t or rec not in by_t[t_snap]: continue
        eligible += 1; key = (t_snap, rec)
        if key in seen: duplicates += 1; continue
        seen.add(key); pre = np.asarray(by_t[t_snap][rec], dtype=float)
        candidates = []
        for cand, vec in by_t[t_snap].items():
            if cand == rec: continue
            if has_dir(cand, t_snap, t_snap + max_h): continue
            candidates.append((cand, vec))
        control_pool_sizes.append(len(candidates))
        best_d = float("inf"); best_id = None
        for cand, vec in candidates:
            d = float(np.linalg.norm(pre - np.asarray(vec, dtype=float)))
            if d < best_d: best_d = d; best_id = cand
        if best_id is None: no_control += 1; continue
        matched += 1
        for h in horizons:
            tp = t_snap + h
            if tp not in by_t: continue
            if rec not in by_t[tp] or best_id not in by_t[tp]: continue
            dt = np.asarray(by_t[tp][rec], dtype=float) - np.asarray(by_t[t_snap][rec], dtype=float)
            dc = np.asarray(by_t[tp][best_id], dtype=float) - np.asarray(by_t[t_snap][best_id], dtype=float)
            analysed_by_h[h] += 1
            for i, k in enumerate(("grief", "grat", "uc")):
                treat[h][k].append(float(dt[i])); ctrl[h][k].append(float(dc[i]))
    out = {"meta": meta, "observed_events": observed, "eligible_events": eligible,
           "matched_recipients": matched, "duplicates": duplicates, "no_control": no_control,
           "duplicate_selection_suppressed": duplicates, "unmatched_total": no_control,
           "control_pool_size_at_snap_avg": float(np.mean(control_pool_sizes)) if control_pool_sizes else None,
           "control_pool_empty_frac": float(np.mean([n == 0 for n in control_pool_sizes])) if control_pool_sizes else None,
           "control_pool_size_at_snap_min": int(min(control_pool_sizes)) if control_pool_sizes else None,
           "control_pool_size_at_snap_max": int(max(control_pool_sizes)) if control_pool_sizes else None,
           "analysed_cases_by_horizon": dict(analysed_by_h), "by_horizon": {}}
    for h in horizons:
        out["by_horizon"][f"h={h}"] = {}
        for k in ("grief", "grat", "uc"):
            t = treat[h][k]; c = ctrl[h][k]
            out["by_horizon"][f"h={h}"][k] = {"treat": _stat(t), "ctrl": _stat(c),
                                              "diff_mean": float(np.mean(t) - np.mean(c)) if t and c else None}
    return out


def sec_E3_seek(ledger_raw, snaps=None):
    if not ledger_raw and not snaps: return {"n/a": "ledger и снимки не найдены"}
    ev = Counter()
    if ledger_raw:
        for r in ledger_raw.get("rows", []):
            for k, v in (r.get("ev") or {}).items(): ev[k] += v
    # Диалоговые счётчики живут в Pattern._dlg_stats (попадают в снимки агентов), а не в event_counts,
    # поэтому ledger-колонка для них ожидаемо 0 — раньше это выглядело как «SEEK/ANSWER не работает».
    dlg = {}
    if snaps:
        best, _cov = _dedup_dlg_stats(snaps)
        for _pid, ds in best.items():
            for k, v in ds.items(): dlg[k] = dlg.get(k, 0.0) + v
    out = {"sr_conflict_resolved": ev.get("sr_conflict_resolved", 0),
           "answered (ledger event)": ev.get("answered", 0)}
    for k in ("req_heard", "req_registered", "answers_said", "answers_heard", "relief"):
        out[k] = {"ledger_events": ev.get(k, 0), "dlg_stats_snapshots": (int(dlg[k]) if k in dlg else None)}
    out["_note"] = ("dlg_stats берётся из снимков раз в SNAPSHOT_EVERY тиков: агенты, умершие между снимками, "
                    "не попадают -> нижняя оценка. Полные счётчики за весь прогон — в [M3].")
    return out


def sec_E6_neighbors(log_raw):
    snaps = log_raw.get("agent_snaps", [])
    if not snaps: return {}
    nw = max((s.get("t", 0) for s in snaps), default=-1) // W + 1
    def calc(ss):
        nbr = [s.get("neighbor_grief_own_excl") for s in ss]
        nbr_raw = [s.get("neighbor_grief") for s in ss]
        out = {}
        for f in ("grief", "grat", "uc", "gap", "drag"): out[f"nbrN~{f}"] = _corr(nbr, [s.get(f) for s in ss])
        for f in ("grief", "uc"): out[f"nbr_raw~{f}"] = _corr(nbr_raw, [s.get(f) for s in ss])
        return out
    out = {"global": calc(snaps), "by_window": []}
    for w in range(nw):
        ss = [s for s in snaps if w * W <= s.get("t", -1) < (w + 1) * W]
        out["by_window"].append({"w": w, "n": len(ss), **calc(ss)})
    return out


def sec_E7_gamma(snaps):
    best, coverage = _dedup_dlg_stats(snaps)
    tot = Counter()
    for pid, ds in best.items():
        for k, v in ds.items(): tot[k] += v
    out = dict(tot); out["_dedup_method"] = "max_per_agent"; out["_assumption"] = "cumulative_monotonic"
    out["_coverage"] = coverage
    out["_non_monotonic_warnings"] = {k: v["non_monotonic"] for k, v in coverage.items() if v["non_monotonic"] > 0}
    for k in ("content_response_heard", "content_response_applied", "content_response_source_mismatch", "fb_shifts"):
        c = coverage.get(k)
        out[f"{k}_coverage"] = c if c else {"agents": 0, "observations": 0, "non_monotonic": 0, "status": "NOT_RECORDED"}
    return out


def sec_F_feelings(log_raw):
    snaps = log_raw.get("agent_snaps", [])
    if not snaps: return {}
    keys = ["grief", "grat", "uc", "gap", "drag", "tension", "soul", "nci", "scar"]
    out = {k: _stat([s.get(k) for s in snaps]) for k in keys}
    for f in ("fear", "fury"):
        vals = [s.get(f) for s in snaps if f in s]
        if vals: out[f] = _stat(vals)
    out["frac_snaps_uc_gt_0.6"] = _frac([s.get("uc", 0) > 0.6 for s in snaps if "uc" in s])
    out["frac_snaps_gap_gt_0.8"] = _frac([s.get("gap", 0) > 0.8 for s in snaps if "gap" in s])
    out["frac_snaps_drag_lt_0.5"] = _frac([s.get("drag", 1) < 0.5 for s in snaps if "drag" in s])
    out["frac_snaps_tension_gt_1"] = _frac([s.get("tension", 0) > 1.0 for s in snaps if "tension" in s])
    out["frac_snaps_tension_at_cap"] = _frac([s.get("tension", 0) >= 1.9 for s in snaps if "tension" in s])
    def _agent_ever(predicate, key):
        by_agent = defaultdict(list)
        for snap in snaps:
            if "id" in snap and key in snap: by_agent[snap["id"]].append(snap[key])
        if not by_agent: return None
        return _frac([any(predicate(v) for v in vals) for vals in by_agent.values()])
    out["frac_agents_ever_uc_gt_0.6"] = _agent_ever(lambda v: v > 0.6, "uc")
    out["frac_agents_ever_gap_gt_0.8"] = _agent_ever(lambda v: v > 0.8, "gap")
    out["frac_agents_ever_drag_lt_0.5"] = _agent_ever(lambda v: v < 0.5, "drag")
    out["frac_agents_ever_tension_gt_1"] = _agent_ever(lambda v: v > 1.0, "tension")
    out["frac_agents_ever_tension_at_cap"] = _agent_ever(lambda v: v >= 1.9, "tension")
    return out


def sec_G_concepts(log_raw, ledger_raw):
    snaps = log_raw.get("agent_snaps", [])
    if not snaps: return {}
    out = {"n_conc": _stat([s.get("n_conc") for s in snaps]), "n_edge": _stat([s.get("n_edge") for s in snaps]), "cns": _stat([s.get("cns") for s in snaps])}
    out["frac_cns_gt_0.55"] = _frac([s.get("cns", 0) > 0.55 for s in snaps if "cns" in s])
    out["frac_narrative"] = _frac([s.get("narrative") for s in snaps])
    out["frac_subject"] = _frac([s.get("subject") for s in snaps])
    out["frac_root_q"] = _frac([s.get("root_q") for s in snaps])
    out["frac_meta_gt_0"] = _frac([s.get("meta_s", 0) > 0 for s in snaps if "meta_s" in s])
    return out


def sec_I_mechanisms(ledger_raw):
    if not ledger_raw: return {"n/a": "ledger не найден"}
    rows = ledger_raw.get("rows", [])
    if not rows: return {"n/a": "rows пусто"}
    ev = Counter()
    for r in rows:
        for k, v in (r.get("ev") or {}).items(): ev[k] += v
    at = sum(r.get("n", 0) for r in rows)
    out = {}
    for k in ("feral_execution", "spore_emitted", "fold", "identity_crisis_transformation", "redeemed",
              "dream_consolidation", "nightmare_consolidation", "redemption_complete", "became_feral",
              "feral_redemption_complete", "sr_meditation", "fb_shifts", "intent_switch"):
        c = ev.get(k, 0)
        out[k] = {"count": c, "rate_per_1000_at": round(1000 * c / max(at, 1), 3)}
    if any("ev_world" in r for r in rows):
        # ЕДИНЫЙ мировой счётчик (_log_event) вместо суммы event_counts по агентам, которых ledger увидел
        _evw = Counter()
        for r in rows:
            for k_, v_ in (r.get("ev_world") or {}).items(): _evw[k_] += v_
        c = _evw.get("intent_switch", 0)
        out["intent_switch"] = {"count": c, "rate_per_1000_at": round(1000 * c / max(at, 1), 3),
                                "source": "мировой счётчик _log_event", "unpaid": _evw.get("intent_switch_unpaid", 0),
                                "by_agent_event_counts": ev.get("intent_switch", 0)}
    return out


def sec_J_temporal(log_raw, ledger_raw):
    snaps = log_raw.get("agent_snaps", []); speaks = log_raw.get("speaks", [])
    rows_led = ledger_raw.get("rows", []) if ledger_raw else []
    if not rows_led: return {"n/a": "нет ledger rows"}
    nw = rows_led[-1]["t"] // W + 1
    out = []
    for w in range(nw):
        sw = [s for s in speaks if w * W <= s.get("t", -1) < (w + 1) * W and s.get("utt")]
        sn = [s for s in snaps if w * W <= s.get("t", -1) < (w + 1) * W]
        emo = Counter(s["utt"][1] for s in sw if len(s["utt"]) > 1)
        lw = rows_led[w * W:(w + 1) * W]
        out.append({"w": w, "n_mean": sum(r.get("n", 0) for r in lw) / len(lw) if lw else None,
                    "speaks": len(sw), "GRIEF": emo.get("GRIEF", 0), "JOY": emo.get("JOY", 0), "CALM": emo.get("CALM", 0),
                    "p50_grief": _stat([s.get("grief") for s in sn]).get("p50"),
                    "p50_uc": _stat([s.get("uc") for s in sn]).get("p50"),
                    "p50_gap": _stat([s.get("gap") for s in sn]).get("p50"),
                    "p50_drag": _stat([s.get("drag") for s in sn]).get("p50"),
                    "p50_tension": _stat([s.get("tension") for s in sn]).get("p50")})
    return {"windows": out}


def sec_K_anomalies(log_raw, ledger_raw):
    an = []; snaps = log_raw.get("agent_snaps", []); events = log_raw.get("events", [])
    seen = set(); last_t_per_id = {}
    rows_log = log_raw.get("tick_rows", []); rows_led = ledger_raw.get("rows", []) if ledger_raw else []
    bounded_01 = ("uc", "grief", "grat", "gap", "drag", "soul", "scar")
    # spirit_gap по построению лежит в [0, 2.0] (np.clip(.., 0.0, 2.0) в Pattern; 1.3 в temporal-ветке),
    # поэтому 1.0 < gap <= 2.0 — норма, а не аномалия. Остальные поля — [0, 1].
    _bounds = {f_: (0.0, 1.0) for f_ in bounded_01}
    _bounds["gap"] = (0.0, 2.0)
    for s in snaps:
        key = (s.get("t"), s.get("id"))
        if key in seen: an.append(("dup_snap", s.get("t"), s.get("id")))
        seen.add(key); pid = s.get("id"); t = s.get("t")
        if pid in last_t_per_id and t < last_t_per_id[pid]: an.append(("time_backwards", t, pid))
        last_t_per_id[pid] = t
        for f, (_lo, _hi) in _bounds.items():
            v = s.get(f)
            if v is None: continue
            if not isinstance(v, (int, float, np.integer, np.floating)): continue
            v = float(v)
            if not np.isfinite(v): an.append(("nan_or_inf", t, pid, f, v)); continue
            if not (_lo - 1e-6 <= v <= _hi + 1e-6): an.append(("out_of_range", t, pid, f, round(v, 4)))
        t_v = s.get("tension")
        if isinstance(t_v, (int, float)) and np.isfinite(float(t_v)) and float(t_v) < -1e-6:
            an.append(("out_of_range", t, pid, "tension", t_v))
    events_checked = min(len(events), 100_000)
    for e in events[:events_checked]:
        if e.get("type") == "trust_change" and e.get("deltas"):
            for pid, _p, new_v, _d in e["deltas"]:
                if isinstance(new_v, (int, float)) and (new_v < -1e-6 or new_v > 1 + 1e-6):
                    an.append(("trust_out_of_range", e.get("t"), e.get("pid"), pid, new_v))
    if rows_log and rows_led:
        _log_by_t = {r.get("t"): r for r in rows_log if r.get("t") is not None}
        _ledger_by_t = {r.get("t"): r for r in rows_led if r.get("t") is not None}
        _pm = [(_tick, _log_by_t[_tick].get("n"), _ledger_by_t[_tick].get("n"))
               for _tick in sorted(set(_log_by_t).intersection(_ledger_by_t))
               if _log_by_t[_tick].get("n") != _ledger_by_t[_tick].get("n")]
        if _pm:
            def _pm_diff(_r):
                try: return abs(float(_r[1]) - float(_r[2]))
                except Exception: return 0.0
            # первый тик расхождения (как раньше) + сводка по ВСЕМ расхождениям, а не только по первому
            an.append(("pop_mismatch", _pm[0][0], _pm[0][1], _pm[0][2],
                       {"ticks": len(_pm), "last_tick": _pm[-1][0], "max_abs_diff": max(_pm_diff(_r) for _r in _pm)}))
    _hdr_k = log_raw.get("header", {}) or {}
    if _hdr_k.get("pop_source_violations_n"):
        an.append(("pop_source_violation", _hdr_k.get("pop_source_violations_n"), (_hdr_k.get("pop_source_violations") or [])[:3]))
    if rows_led:
        _b0 = int(rows_led[0].get("n", 0)) - int(rows_led[0].get("births", 0)) + int(rows_led[0].get("deaths", 0))
        _exp = _b0 + sum(int(r.get("births", 0)) for r in rows_led) - sum(int(r.get("deaths", 0)) for r in rows_led)
        if _exp != rows_led[-1].get("n"):
            an.append(("pop_balance", _b0, _exp, rows_led[-1].get("n")))
    for k in ("events", "speaks", "dialogues"):
        if len(log_raw.get(k, [])) >= RECORD_CAP: an.append(("record_cap_reached", k, len(log_raw.get(k, []))))
    _kinds = Counter((f"{a[0]}:{a[3]}" if a[0] in ("out_of_range", "nan_or_inf") and len(a) > 3 else str(a[0])) for a in an)
    return {"count": len(an), "by_kind": dict(_kinds.most_common()), "first_50": an[:50],
            "events_checked": events_checked, "events_total": len(events)}


def sec_L_energy(log_raw):
    out = {}
    header = log_raw.get("header", {}); tick_rows = log_raw.get("tick_rows", [])
    snap = header.get("energy_snapshot")
    if not snap:
        out["n/a"] = "no energy_snapshot (лог старше правок Cell 6b)"; return out
    out["final"] = {"t": snap.get("t"), "field": snap.get("field"), "agent_inventory": snap.get("agent_inventory"),
                    "total": snap.get("total"), "step_total": snap.get("step_total"),
                    "step_measured": snap.get("step_measured"), "ticks": snap.get("ticks")}
    src = snap.get("src", {}) or {}
    out["creators_top"] = sorted(((n, float(v[0])) for n, v in src.items()), key=lambda kv: -kv[1])[:12]
    out["sinks_top"] = sorted(((n, float(v[1])) for n, v in src.items() if float(v[1]) < 0), key=lambda kv: kv[1])[:12]
    out["creators_sum"] = sum(float(v[0]) for v in src.values())
    out["sinks_sum"] = sum(float(v[1]) for v in src.values())
    out["balance_x1"] = out["creators_sum"] + out["sinks_sum"]
    x2 = snap.get("x2", {}) or {}
    x2_items = sorted(((n, float(v[0]) + float(v[1])) for n, v in x2.items()), key=lambda kv: -abs(kv[1]))
    out["x2_items"] = x2_items[:15]; out["x2_sum"] = sum(v for _, v in x2_items)
    series = []
    for r in tick_rows:
        if "E_field" in r: series.append({"t": r["t"], "field": r.get("E_field"), "inv": r.get("E_inv"), "total": r.get("E_total")})
    out["series"] = series
    if len(series) >= 2:
        out["field_drift"] = series[-1]["field"] - series[0]["field"]
        if series[0]["inv"] is not None and series[-1]["inv"] is not None: out["inv_drift"] = series[-1]["inv"] - series[0]["inv"]
    return out


def _audit_path_for(log_path):
    base = os.path.basename(log_path)
    if base.startswith("rich_log_"): base = base[len("rich_log_"):]
    if base.endswith(".json"): base = base[:-5]
    return f"{DIR}/audit_{base}.json"


def sec_M_audit(audit_path):
    if not audit_path or not os.path.exists(audit_path):
        return {"n/a": f"нет файла аудита: {audit_path} (прогон без Config.AUDIT_DIR или старая версия движка)"}
    with open(audit_path, "r", encoding="utf-8") as f: A = json.load(f)
    S = A["summary"]; rows = A.get("rows", []); atk = A.get("attacks", []); dlg = A.get("dialogue", []); cfg = A.get("config", {})
    out = {"path": audit_path, "ticks": S.get("ticks"), "config": cfg}
    F = {"counters": S.get("feral", {})}
    if atk:
        hit = np.array([bool(a["hit"]) for a in atk]); raw = np.array([a["raw"] for a in atk]); kp = np.array([a["kill_prob"] for a in atk])
        n = int(hit.size); k = int(hit.sum()); lo, hi = _wilson(k, n)
        F.update({"n_rolled": n, "hits": k, "misses": n - k, "hit_rate": k / n, "miss_rate": 1 - k / n,
                  "miss_ci95": (1 - hi, 1 - lo), "kill_prob_mean": float(kp.mean()), "kill_prob_q": _q(kp), "raw_q": _q(raw),
                  "raw_ge_095_share": float((raw >= 0.95).mean()),
                  "factor_means": {key: float(np.mean([a[key] for a in atk])) for key in ("fury_factor", "size_factor", "vuln_factor", "crowd_factor", "weary", "lf", "prey_age")}})
        win = {}
        for a, h_ in zip(atk, hit):
            w = a["t"] // 100; e = win.setdefault(w, [0, 0, 0.0]); e[0] += 1; e[1] += int(h_); e[2] += a["kill_prob"]
        F["by_window"] = {int(w): {"n": v[0], "hit_rate": v[1] / v[0], "kp_mean": v[2] / v[0]} for w, v in sorted(win.items())}
        def _scale_for(ceil, target_hit):
            lo, hi = 1e-4, 50.0
            if ceil * float(np.mean(1.0 - np.exp(-raw / lo))) < target_hit: return None
            for _ in range(80):
                mid = (lo + hi) / 2
                if ceil * float(np.mean(1.0 - np.exp(-raw / mid))) > target_hit: lo = mid
                else: hi = mid
            return (lo + hi) / 2
        calib = []
        _cur_ceil = cfg.get("FERAL_KILL_CEIL")
        _cur_ok = isinstance(_cur_ceil, (int, float)) and not isinstance(_cur_ceil, bool)
        _ceils = [0.90, 0.95]
        if _cur_ok and all(abs(float(_cur_ceil) - c_) > 1e-9 for c_ in _ceils): _ceils.append(float(_cur_ceil))
        _ceils.sort()
        F["calibration_current_ceil"] = float(_cur_ceil) if _cur_ok else None
        for target_miss in (0.10, 0.20, 0.30, 0.40):
            for ceil in _ceils: calib.append({"target_miss": target_miss, "ceil": ceil, "scale_needed": _scale_for(ceil, 1.0 - target_miss)})
        F["calibration"] = calib
        first, killed = {}, {}
        for a in atk:
            first.setdefault(a["prey"], a["t"])
            if a["hit"]: killed.setdefault(a["prey"], a["t"])
        delays = [killed[p] - first[p] for p in killed]
        F["kill_delay"] = {"prey_attacked": len(first), "prey_killed": len(killed), "attacked_not_killed": len([p for p in first if p not in killed]),
                           "mean_ticks_first_attack_to_kill": float(np.mean(delays)) if delays else None,
                           "p50": float(np.median(delays)) if delays else None, "p90": float(np.quantile(delays, 0.9)) if delays else None,
                           "killed_on_first_attack_share": float(np.mean([d == 0 for d in delays])) if delays else None}
        wall_t = {r["t"]: r.get("wall_on") for r in rows}
        pw = {"ON": [0, 0], "OFF": [0, 0]}
        for a, h_ in zip(atk, hit):
            st = wall_t.get(a["t"])
            if st is None: continue
            key = "ON" if st else "OFF"; pw[key][0] += 1; pw[key][1] += int(h_)
        F["by_wall_phase"] = {k_: {"n": v[0], "hit_rate": (v[1] / v[0]) if v[0] else None} for k_, v in pw.items()}
    out["feral"] = F
    E = {}
    for key in ("x1", "x2"):
        blk = S.get(key, {}); R = blk.get("residual", {})
        E[key] = {"dE_total": blk.get("dE_total"), "logged_net": blk.get("logged_net"), "residual_sum": R.get("sum"),
                  "residual_abs": R.get("abs"), "residual_max_abs": R.get("max_abs"), "residual_max_t": R.get("max_t"),
                  "abs_flow_sum": blk.get("abs_flow_sum"), "gap": S.get("gap_between_ticks", {}).get(key),
                  "outside_window": S.get("outside_window", {}).get(key),
                  "top_flows": sorted(((n_, v["net"], v["pos"], v["neg"], v["n"]) for n_, v in blk.get("ledger", {}).items()), key=lambda kv: -abs(kv[1]))[:12]}
    E["memo"] = S.get("memo", {})
    ser = []
    for r in rows[::max(1, len(rows) // 16)]: ser.append({"t": r["t"], "E1": r["E1"], "E2": r["E2"], "pop": r["pop"], "feral": r["feral"]})
    E["series"] = ser; out["energy"] = E
    D = {"ledger": S.get("dialogue", {})}
    cols = S.get("dialogue_columns", [])
    if dlg and cols:
        ix = {c: i for i, c in enumerate(cols)}
        by_band = {}; by_cross = {}; by_seek = Counter()
        for rec in dlg:
            tr = rec[ix["trust"]]
            band = "n/a" if tr is None else ("trust<0.5" if tr < 0.5 else ("trust0.5-0.9" if tr < 0.9 else "trust>=0.9"))
            b = by_band.setdefault(band, [0, 0.0, 0.0]); b[0] += 1; b[1] += rec[ix["d_grief"]]; b[2] += abs(rec[ix["d_grief"]])
            cs = rec[ix["cross_side"]] if "cross_side" in ix else None
            key = f"cross={cs}|wall={rec[ix['wall']]}" if "wall" in ix else f"cross={cs}"
            c = by_cross.setdefault(key, [0, 0.0, 0.0]); c[0] += 1; c[1] += rec[ix["d_grief"]]; c[2] += rec[ix["d_grat"]]
            by_seek[rec[ix["seek"]]] += 1
        D["by_trust_band"] = {k_: {"n": v[0], "mean_d_grief": v[1] / v[0], "mean_abs_d_grief": v[2] / v[0]} for k_, v in by_band.items()}
        D["by_cross_wall"] = {k_: {"n": v[0], "mean_d_grief": v[1] / v[0], "mean_d_grat": v[2] / v[0]} for k_, v in by_cross.items()}
        D["seek_outcomes"] = dict(by_seek)
    out["dialogue"] = D
    out["modes"] = S.get("mode_snapshots", [])
    O = dict(S.get("outcome_learning", {}))
    win = {}
    for r in rows:
        if r.get("om_n"):
            e = win.setdefault(r["t"] // 100, [0, 0.0, 0, 0, 0]); e[0] += r["om_n"]; e[1] += r["om_gain_mean"] * r["om_n"]
            e[2] += r.get("om_learned", 0); e[3] += r.get("om_explore", 0); e[4] += r.get("om_default", 0)
    O["curve"] = {int(w): {"n": v[0], "mean_gain": v[1] / v[0], "learned_share": v[2] / max(1, v[2] + v[3] + v[4])} for w, v in sorted(win.items())}
    out["outcome"] = O
    W = {"cycle": S.get("wall_cycle"), "events": S.get("wall_events", [])}
    ph = {"ON": [], "OFF": []}
    for r in rows:
        if "wall_on" in r: ph["ON" if r["wall_on"] else "OFF"].append(r)
    W["by_phase"] = {}
    for k_, rr in ph.items():
        if rr:
            W["by_phase"][k_] = {"ticks": len(rr), "pop_mean": float(np.mean([r["pop"] for r in rr])),
                                 "feral_mean": float(np.mean([r["feral"] for r in rr])),
                                 "births_per_tick": float(np.mean([r["births"] for r in rr])),
                                 "removed_per_tick": float(np.mean([r["removed"] for r in rr]))}
    out["wall"] = W
    return out


def sec_N_signal(audit_path):
    if not audit_path or not os.path.exists(audit_path):
        return {"n/a": f"нет файла аудита: {audit_path}"}
    with open(audit_path, "r", encoding="utf-8") as f: A = json.load(f)
    S = (A.get("summary") or {}).get("signal")
    if not S:
        return {"n/a": "в аудите нет summary['signal'] (ячейка SIGNAL v3.0 не была выполнена до движка)"}
    rows = A.get("rows", [])
    tot = lambda k: sum((r.get(k) or 0) for r in rows)
    st = S.get("stats", {}); snap = S.get("snapshot", {})
    ok, part, dbt = tot("sig_rec_recognized"), tot("sig_rec_partial"), tot("sig_rec_doubt")
    none_, smiss = tot("sig_rec_none"), tot("sig_rec_stress_missed")
    heard = (ok + part + dbt + none_ + smiss) or 1
    el = S.get("emitter_windows", {}); lv = S.get("listener_verdicts", {}); sv = S.get("sender_verdicts", {})
    cs = tot("sig_choice_signal")
    scr = ((A.get("summary") or {}).get("patches") or {}).get("scream") or {}
    return {"version": S.get("version"), "emits": tot("sig_emits"), "cry_emits": tot("sig_cry_emit"),
            "scream_total": scr.get("total", 0), "scream_as_form": scr.get("cry_as_form", 0),
            "form_full_blocked": st.get("form_full_blocked", 0), "emit_no_energy": tot("sig_emit_no_energy"),
            "listens": tot("sig_listens"), "provenance_share": S.get("provenance_share"),
            "recognition_share": {"recognized": ok / heard, "partial": part / heard, "doubt": dbt / heard,
                                  "none": none_ / heard, "stress_missed": smiss / heard},
            "emitter_windows": el, "listener_verdicts": lv, "sender_verdicts": sv,
            "trust_up": tot("sig_trust_up"), "trust_down": tot("sig_trust_down"),
            "reply_emits": tot("sig_reply_emits"), "ordinary_emits": tot("sig_ordinary_emits"),
            "reply_attempts": tot("sig_reply_attempts"), "reply_emit_blocked": tot("sig_reply_emit_blocked"),
            "reply_settled": tot("sig_reply_settled"), "reply_credit": tot("sig_reply_credit"),
            "reply_z_none": st.get("reply_z_none", 0), "replied_ids_size": S.get("replied_ids_size", 0),
            "reply_pick_learned": tot("sig_reply_pick_learned"), "reply_pick_default": tot("sig_reply_pick_default"),
            "choice": {"signal": cs, "influenced": tot("sig_choice_influenced"), "blocked_by_barrier": tot("sig_choice_blocked"),
                       "reinforced": tot("sig_choice_reinforced")},
            "forms": snap, "goal_models": S.get("goal_models"), "in_flight": S.get("in_flight"),
            "heard_alarm": {k: int(st.get(k, 0)) for k in (   # MORSE-EVO
                'heard_alarm', 'heard_alarm_agents', 'heard_alarm_cooldown', 'alarm_credit', 'alarm_credit_x1000',
                'alarm_credit_sender_gone', 'alarm_credit_sender_cd', 'alarm_credit_no_energy', 'alarm_credit_sender_full',
                'mevo_deaths_exposed', 'mevo_deaths_unexposed', 'mevo_deaths_exposed_feral', 'mevo_deaths_unexposed_feral',
                'mevo_agent_ticks', 'mevo_agent_ticks_exposed')},
            "morse_evo": (A.get("summary") or {}).get("morse_evo"),
            "errors": sum(v for k, v in st.items() if k.startswith("err_")), "last_err": S.get("last_err")}


def _print_N(log_path):
    print("\n" + "-" * 78)
    print("[N] СИГНАЛ v3.0: цикл сигнал → распознавание → исход → выбор")
    N = sec_N_signal(_audit_path_for(log_path))
    for k, v in N.items():
        if isinstance(v, float): v = round(v, 4)
        print(f"  {k}: {v}")
    return N


def _P(k, v): print(f"  {k}: {v}")


def _print_state_response(name, section):
    print("\n" + "-" * 78)
    print(name)
    meta = section.get("meta", {})
    for k in ("state_resolution_ticks", "exact_1_5_tick_response_available", "causal_claim", "selection_unit"):
        if k in meta: _P(k, meta[k])
    for k in ("observed_events", "eligible_events", "matched_recipients", "duplicates", "no_control", "duplicate_selection_suppressed", "analysed_cases_by_horizon"):
        if k in section: _P(k, section[k])
    for hk, hv in (section.get("by_horizon") or {}).items():
        print(f"  {hk}:")
        for k in ("grief", "grat", "uc"):
            v = hv.get(k, {})
            print(f"    {k}: Δtreat={_ss(v.get('treat'))} Δctrl={_ss(v.get('ctrl'))} diff={_pm(v.get('diff_mean'))}")


def sec_Z_facts(sections):
    """OBSERVER FACTS. Только факты из уже построенных секций, без вердиктов (вердикт — работа Gate).
    Нет данных -> None (UNKNOWN), а не 0 и не OK. Ничего не пересчитывает за симуляцию."""
    def g(d, *path):
        for p_ in path:
            if not isinstance(d, dict) or p_ not in d: return None
            d = d[p_]
        return d
    A = sections.get("A") or {}; B = sections.get("B") or {}; C = sections.get("C") or {}
    K = sections.get("K") or {}; M = sections.get("M") or {}
    z = {}
    fp_b = A.get("input_fp")
    z["input_fp_before"] = None if fp_b in (None, "MISSING") else fp_b
    z["input_fp_after"] = A.get("input_fp_after")
    z["input_fp_stable"] = A.get("input_fp_stable")
    z["output_fp_after"] = A.get("output_fp_after")
    z["run_mutated_files"] = A.get("run_mutated_files")
    z["config_hash"] = A.get("config_hash")
    z["run_complete"] = A.get("complete")
    z["ticks_expected"] = B.get("ticks_expected"); z["ticks_ledger"] = B.get("ticks_ledger")
    dt = C.get("deaths_with_record"); du = C.get("deaths_unattributed")
    z["deaths_with_record"] = dt; z["deaths_unattributed"] = du
    z["deaths_unattributed_share"] = (round(du / dt, 4) if isinstance(dt, (int, float)) and dt and isinstance(du, (int, float)) else None)
    z["anomalies_total"] = K.get("count"); z["anomalies_by_kind"] = K.get("by_kind")
    for key in ("x1", "x2"):
        z[f"energy_{key}_residual_abs"] = g(M, "energy", key, "residual_abs")
        z[f"energy_{key}_residual_max_abs"] = g(M, "energy", key, "residual_max_abs")
    L = g(M, "dialogue", "ledger") or {}
    z["dlg_heard"] = L.get("heard")
    z["dlg_cap_exceeded_total"] = L.get("d_grief_gt_tick_cap")
    z["dlg_cap_exceeded_relief"] = L.get("d_grief_gt_tick_cap_relief")
    z["dlg_cap_exceeded_other"] = L.get("d_grief_gt_tick_cap_other")
    z["feral_miss_rate"] = g(M, "feral", "miss_rate")
    z["feral_miss_ci95"] = g(M, "feral", "miss_ci95")
    z["feral_kill_ceil"] = g(M, "config", "FERAL_KILL_CEIL")
    z["feral_kill_scale"] = g(M, "config", "FERAL_KILL_SCALE")
    return z


def run_master(LOG_PATH, LEDGER_PATH=None, PREV_PATH=None):
    t0 = time.time()
    print("=" * 78)
    print(f"MASTER OBSERVATORY v{MASTER_VERSION}")
    print("READ-ONLY: simulation is not executed")
    print("=" * 78)
    print(f"log: {LOG_PATH}")
    log_raw, log_info = _safe_load(LOG_PATH)
    if log_raw is None:
        print(f"  [!] log: {log_info}"); return None
    print(f"  ok, size={log_info['size'] / 1e6:.1f} MB")
    ledger_raw = None
    if LEDGER_PATH:
        ledger_raw, led_info = _safe_load(LEDGER_PATH)
        print(f"ledger: {LEDGER_PATH}")
        if ledger_raw is None: print(f"  [!] ledger: {led_info}")
        else: print(f"  ok, size={led_info['size'] / 1e6:.1f} MB")
    if PREV_PATH: print(f".prev: {PREV_PATH}")
    inventory = raw_inventory(log_raw, ledger_raw)
    print_inventory(inventory)
    sections = {"raw": inventory}; failed = []
    section_fns = (
        ("A", lambda: sec_A_identity(log_raw, ledger_raw, log_info)),
        ("B", lambda: sec_B_coverage(log_raw, ledger_raw, PREV_PATH)),
        ("C", lambda: sec_C_population(ledger_raw)),
        ("D", lambda: sec_D_communication(log_raw)),
        ("F", lambda: sec_F_feelings(log_raw)),
        ("G", lambda: sec_G_concepts(log_raw, ledger_raw)),
        ("I", lambda: sec_I_mechanisms(ledger_raw)),
        ("J", lambda: sec_J_temporal(log_raw, ledger_raw)),
        ("K", lambda: sec_K_anomalies(log_raw, ledger_raw)),
        ("M", lambda: sec_M_audit(_audit_path_for(LOG_PATH))),
    )
    for name, fn in section_fns:
        try: sections[name] = fn()
        except Exception as e:
            failed.append((name, f"{type(e).__name__}: {e}")); sections[name] = {"_error": f"{type(e).__name__}: {e}"}
    pairs = _match_pairs(log_raw.get("speaks", []))
    print(f"\nA->B->A association pairs: {len(pairs)}")
    extra_fns = (
        ("E1", lambda: sec_E1_grief_response(pairs, log_raw)),
        ("E2", lambda: sec_E_state_response(log_raw, horizons=(10, 20), only_grief_a=True)),
        ("E3", lambda: sec_E3_seek(ledger_raw, log_raw.get("agent_snaps", []))),
        ("E6", lambda: sec_E6_neighbors(log_raw)),
        ("E7", lambda: sec_E7_gamma(log_raw.get("agent_snaps", []))),
        ("E8", lambda: sec_E_state_response(log_raw, horizons=(10, 20, 30), only_grief_a=False)),
        ("L",  lambda: sec_L_energy(log_raw)),
    )
    for name, fn in extra_fns:
        try: sections[name] = fn()
        except Exception as e:
            failed.append((name, f"{type(e).__name__}: {e}")); sections[name] = {"_error": f"{type(e).__name__}: {e}"}
    try: sections["Z"] = sec_Z_facts(sections)
    except Exception as e:
        failed.append(("Z", f"{type(e).__name__}: {e}")); sections["Z"] = {"_error": f"{type(e).__name__}: {e}"}
    print("\n" + "-" * 78)
    print("[A] IDENTITY")
    for k, v in sections["A"].items(): _P(k, v)
    print("\n" + "-" * 78)
    print("[B] COVERAGE")
    B = sections["B"]
    for k in ("ticks_expected", "ticks_log", "ticks_ledger", "snaps_recorded", "snap_ticks_expected", "snap_ticks_actual", "snaps_expected_approx", "events", "speaks", "dialogues", "directed_speaks_frac"): _P(k, B.get(k))
    _P("snap_ticks_missing_first20", B.get("snap_ticks_missing"))
    _P("nan_frac_by_field", B.get("nan_frac_by_field"))
    _P("capped", B.get("capped"))
    _P("log_errors", B.get("log_errors"))
    _P("prev", B.get("prev"))
    print("\n" + "-" * 78)
    print("[C] POPULATION")
    C = sections.get("C", {})
    for k in ("n_start", "n_min", "n_max", "n_end", "pending_dead_end", "pop_semantics", "births", "births_by_kind", "deaths", "divisions", "pop_balance", "agent_ticks", "deaths_per_1000_at", "divisions_per_1000_at", "was_subject", "was_narrative", "protection_above_thr_at_death", "deaths_with_record", "deaths_unattributed"): _P(k, C.get(k))
    _P("causes_top", C.get("causes_top"))
    if isinstance(C.get("age_at_death"), dict): _P("age_at_death", _ss(C["age_at_death"]))
    print("\n" + "-" * 78)
    print("[D] COMMUNICATION")
    D = sections.get("D", {})
    for k in ("speaks", "dialogues", "unique_utterances", "speakers", "recipients", "only_speak", "only_listen", "clusters", "n_windows"): _P(k, D.get(k))
    print("  top-10 utterances:")
    for utt, n in (D.get("top10_utt") or []): print(f"    {n:>6}  {utt}")
    print("  by_window:")
    print(f"    {'w':<5}{'spk':>7}{'uniq':>7}{'Hutt':>8}{'Hem':>8}{'Hint':>8}{'GRF':>6}{'JOY':>6}{'CALM':>6}{'KNOT':>6}{'GAP':>6}{'DRAG':>6}{'fid':>8}")
    fid_w = D.get("fidelity_by_window", {})
    for r in (D.get("by_window") or []):
        print(f"    {r['w']:<5}{r['speaks']:>7}{r['unique_utterances']:>7}{(r['H_utterance'] if r['H_utterance'] is not None else float('nan')):>8.3f}{(r['H_emotion'] if r['H_emotion'] is not None else float('nan')):>8.3f}{(r['H_intent'] if r['H_intent'] is not None else float('nan')):>8.3f}{r['GRIEF']:>6}{r['JOY']:>6}{r['CALM']:>6}{r['KNOT']:>6}{r['GAP']:>6}{r['DRAG']:>6}{(fid_w.get(r['w']) if fid_w.get(r['w']) is not None else float('nan')):>8.4f}")
    print("\n" + "-" * 78)
    print("[E1] GRIEF A -> subsequent B->A association window")
    E1 = sections.get("E1", {}); gg = E1.get("global", {}).get("grief_a", {}); go = E1.get("global", {}).get("other_a", {})
    print(f"  global GRIEF A: n={gg.get('n', 0)} rate_calm={gg.get('rate_calm')}")
    print(f"  global non-GRIEF A: n={go.get('n', 0)} rate_calm={go.get('rate_calm')}")
    if gg.get("n") and go.get("n"): print(f"  Δ rate_calm (GRIEF − other) = {gg['rate_calm'] - go['rate_calm']:+.3f}")
    _P("response_delay", _ss(E1.get("grief_response_delay")))
    _print_state_response("[E2] GRIEF A -> state B (+10/+20) | diagnostic", sections.get("E2", {}))
    print("\n" + "-" * 78)
    print("[E3] SEEK / ANSWER (ledger events | dlg_stats из снимков = нижняя оценка)")
    for k, v in (sections.get("E3") or {}).items(): _P(k, v)
    print("\n" + "-" * 78)
    print("[E6] NEIGHBORS — global + per window")
    E6 = sections.get("E6") or {}
    print("  global:")
    for k, v in (E6.get("global") or {}).items():
        if isinstance(v, dict): print(f"    {k}: r={v.get('r')} (n={v.get('n')})")
    for row in E6.get("by_window") or []:
        print(f"  window {row['w']} (n={row['n']}):")
        for k, v in row.items():
            if k in ("w", "n"): continue
            if isinstance(v, dict): print(f"    {k}: r={v.get('r')} (n={v.get('n')})")
    print("\n" + "-" * 78)
    print("[E7] γ — dedup=max_per_agent")
    E7 = sections.get("E7") or {}
    for k in ("content_response_heard", "content_response_applied", "content_response_source_mismatch", "fb_shifts", "req_heard", "req_registered", "answers_said", "answers_heard", "relief", "knot_said", "utt_total"):
        if k in E7: _P(k, E7[k])
    print(f"  _dedup_method: {E7.get('_dedup_method')}"); print(f"  _assumption: {E7.get('_assumption')}")
    print("  counter coverage:")
    for k, v in (E7.get("_coverage") or {}).items(): print(f"    {k}: {v}")
    nw_warn = E7.get("_non_monotonic_warnings") or {}
    if nw_warn: print(f"  [!] non-monotonic counters: {nw_warn}")
    else: print("  monotonicity: no decreases observed")
    _print_state_response("[E8] delayed state — all directed | diagnostic, not causality", sections.get("E8", {}))
    print("\n" + "-" * 78)
    print("[F] FEELINGS")
    F = sections.get("F") or {}
    for k in ("grief", "grat", "uc", "gap", "drag", "tension", "soul", "nci", "scar", "fear", "fury"):
        v = F.get(k)
        if v: print(f"  {k}: {_ss(v)}")
    for k in ("frac_snaps_uc_gt_0.6", "frac_snaps_gap_gt_0.8", "frac_snaps_drag_lt_0.5", "frac_snaps_tension_gt_1", "frac_snaps_tension_at_cap", "frac_agents_ever_uc_gt_0.6", "frac_agents_ever_gap_gt_0.8", "frac_agents_ever_drag_lt_0.5", "frac_agents_ever_tension_gt_1", "frac_agents_ever_tension_at_cap"):
        v = F.get(k)
        if v: print(f"  {k}: {_ss(v)}")
    print("\n" + "-" * 78)
    print("[G] CONCEPTS")
    for k, v in (sections.get("G") or {}).items():
        if isinstance(v, dict): print(f"  {k}: {_ss(v)}")
    print("\n" + "-" * 78)
    print("[I] MECHANISMS")
    for k, v in (sections.get("I") or {}).items():
        if isinstance(v, dict): print(f"  {k}: {v.get('count')} ({v.get('rate_per_1000_at')}/1000at)"
                                      + (f" [источник: {v['source']}; неоплачено={v.get('unpaid')}; по event_counts агентов={v.get('by_agent_event_counts')}]" if v.get("source") else ""))
    print("\n" + "-" * 78)
    print("[J] TEMPORAL")
    J = sections.get("J") or {}
    if "windows" in J:
        print(f"  {'w':<5}{'n_mean':>8}{'speaks':>8}{'GRF':>6}{'JOY':>6}{'CALM':>6}{'p50_grief':>11}{'p50_uc':>9}{'p50_gap':>9}{'p50_drag':>10}{'p50_tension':>13}")
        for r in J["windows"]:
            def _g(k):
                v = r.get(k); return f"{v:.3f}" if isinstance(v, (int, float)) else "n/a"
            print(f"  {r['w']:<5}{(r.get('n_mean') or 0):>8.1f}{r['speaks']:>8}{r['GRIEF']:>6}{r['JOY']:>6}{r['CALM']:>6}{_g('p50_grief'):>11}{_g('p50_uc'):>9}{_g('p50_gap'):>9}{_g('p50_drag'):>10}{_g('p50_tension'):>13}")
    print("\n" + "-" * 78)
    print("[K] ANOMALIES")
    K = sections.get("K") or {}
    print(f"  count: {K.get('count', 0)}")
    print(f"  by_kind: {K.get('by_kind')}")
    print(f"  events_checked: {K.get('events_checked', 0)} из {K.get('events_total', 0)}")
    for a in K.get("first_50", []): print(f"    {a}")
    print("\n" + "-" * 78)
    print("[L] ENERGY — потоки, утечки, инвентарь агентов")
    L = sections.get("L") or {}
    if "n/a" in L: print(f"  {L['n/a']}")
    else:
        fin = L.get("final")
        if isinstance(fin, dict):
            print(f"  t={fin['t']}  ticks_ledger={fin['ticks']}")
            print(f"  field          = {fin['field']:>14.2f}")
            print(f"  agent_inv      = {fin['agent_inventory']:>14.2f}   ← сумма Pattern.energy по живым")
            print(f"  total          = {fin['total']:>14.2f}")
            print(f"  step_total     = {fin['step_total']:>14.2f}")
            print(f"  step_measured  = {fin['step_measured']:>14.2f}")
        print("\n  ТОП-12 положительных вкладов X1 (поле):")
        for name, v in (L.get("creators_top") or []): print(f"    {name:<58} +{v:>12.2f}")
        print(f"    Σ положительных вкладов X1: {L.get('creators_sum', 0.0):>+12.2f}")
        print("\n  ТОП-12 отрицательных вкладов X1 (поле):")
        for name, v in (L.get("sinks_top") or []): print(f"    {name:<58} {v:>13.2f}")
        print(f"    Σ отрицательных вкладов X1: {L.get('sinks_sum', 0.0):>+12.2f}")
        print(f"    БАЛАНС X1 (Σ+ и Σ−):        {L.get('balance_x1', 0.0):>+12.2f}")
        if L.get("x2_items"):
            print("\n  ТОП-15 изменений X2 (инвентарь агентов):")
            for name, v in L["x2_items"]: print(f"    {name:<58} {v:>+12.2f}")
            print(f"    Итого X2 учтено (топ): {L.get('x2_sum', 0.0):>+12.2f}")
        if "field_drift" in L: print(f"\n  Смещение field за прогон:      {L['field_drift']:+.2f}")
        if "inv_drift" in L: print(f"  Смещение agent_inv за прогон:  {L['inv_drift']:+.2f}")
        if L.get("series"):
            print("\n  Time series (field / inv / total):")
            step = max(1, len(L["series"]) // 15)
            for s in L["series"][::step]: print(f"    t={s['t']:>5}  field={s['field']:>10.1f}  inv={s['inv']:>8.1f}  total={s['total']:>10.1f}")
    try: _print_M(sections.get("M"))
    except Exception as e: failed.append(("M-print", f"{type(e).__name__}: {e}"))
    try: _print_N(LOG_PATH)
    except Exception as e: failed.append(("N-print", f"{type(e).__name__}: {e}"))
    print("\n" + "-" * 78)
    print("[Z] OBSERVER FACTS (только факты, без вердиктов; None = UNKNOWN, не PASS)")
    for k, v in (sections.get("Z") or {}).items(): _P(k, v)
    print("\n" + "-" * 78)
    ok_cnt = sum(1 for s in sections.values() if not (isinstance(s, dict) and "_error" in s))
    print(f"секции: OK={ok_cnt} failed={len(failed)}")
    for f in failed: print(f"  [!] {f[0]}: {f[1]}")
    print(f"wall={time.time() - t0:.1f}s")
    out = {"master_version": MASTER_VERSION, "log_path": LOG_PATH, "ledger_path": LEDGER_PATH, "prev_path": PREV_PATH,
           "sections": sections, "failed": failed, "pairs": len(pairs),
           "pair_definition": f"A->B followed by B->A within <= {W_MAX} ticks; association only"}
    MASTER_JSON = _master_json_for(LOG_PATH)
    try:
        os.makedirs(os.path.dirname(MASTER_JSON), exist_ok=True)
        tmp = MASTER_JSON + ".tmp"
        clean_out = _sanitize_json(out)
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(clean_out, f, ensure_ascii=False, allow_nan=False, default=_json_default)
            f.flush(); os.fsync(f.fileno())
        os.replace(tmp, MASTER_JSON)
        print(f"JSON: {MASTER_JSON} ({os.path.getsize(MASTER_JSON) / 1e6:.1f} MB)")
    except Exception as e: print(f"[!] JSON save: {e}")
    return out


def _print_M(M):
    print("\n" + "=" * 78)
    print("[M] АУДИТ ДВИЖКА (читает audit_<tag>.json; ничего не пересчитывает за симуляцию)")
    print("=" * 78)
    if M is None or "_error" in (M or {}):
        print(f"  [!] секция M не построена: {M}"); return
    if "n/a" in M:
        print(f"  {M['n/a']}"); return
    print(f"  файл: {M['path']}; тиков в аудите: {M['ticks']}")
    cfg = M.get("config", {})
    if cfg: print(f"  формула удара: режим={cfg.get('FERAL_KILL_MODE')} CEIL={cfg.get('FERAL_KILL_CEIL')} SCALE={cfg.get('FERAL_KILL_SCALE')} BASE={cfg.get('FERAL_KILL_BASE')} VULN={cfg.get('FERAL_VULNERABILITY')} CROWD={cfg.get('FERAL_CROWD_GAIN')}")
    F = M["feral"]
    print("\n  ── [M1] ФЕРОЛ: удары (КАЖДЫЙ брошенный удар записан; прежний счётчик miss_t всегда показывал 0) ──")
    c = F.get("counters", {})
    print(f"  цель найдена={c.get('prey_found', 0)}; цели нет={c.get('no_prey', 0)}; не смежно={c.get('not_adjacent', 0)}")
    if "n_rolled" in F:
        lo, hi = F["miss_ci95"]
        print(f"  брошено ударов={F['n_rolled']}: попаданий={F['hits']}, промахов={F['misses']} → доля промахов {F['miss_rate']:.3f} (95% ДИ {lo:.3f}–{hi:.3f}); цель пользователя 0.200")
        print(f"  ⟨kill_prob⟩={F['kill_prob_mean']:.3f}; kill_prob квантили 5/26/60/75/95%: " + " / ".join(f"{v:.3f}" for v in F["kill_prob_q"].values()))
        print(f"  raw (произведение факторов) квантили 5/26/60/75/95%: " + " / ".join(f"{v:.3f}" for v in F["raw_q"].values()) + f"; доля ударов с raw ≥ 0.95: {F['raw_ge_095_share']:.1%}")
        fm = F["factor_means"]
        print(f"  средние факторы: fury={fm['fury_factor']:.3f} size={fm['size_factor']:.3f} vuln={fm['vuln_factor']:.3f} crowd={fm['crowd_factor']:.3f}; weary={fm['weary']:.3f} lf={fm['lf']:.3f} возраст жертвы={fm['prey_age']:.1f}")
        print("  по окнам по 100 тиков:  окно  ударов  hit_rate  ⟨p⟩")
        for w, v in F["by_window"].items(): print(f"                          {w:4d} {v['n']:7d} {v['hit_rate']:9.3f} {v['kp_mean']:6.3f}")
        kd = F["kill_delay"]
        print(f"  задержка смерти из-за промахов: жертв атаковано={kd['prey_attacked']}, убито={kd['prey_killed']}, атаковано но не убито={kd['attacked_not_killed']}; от первого удара до убийства: среднее={kd['mean_ticks_first_attack_to_kill']}, p50={kd['p50']}, p90={kd['p90']}; убито с первого удара: {kd['killed_on_first_attack_share']}")
        print("  калибровка: какой SCALE нужен при данном CEIL, чтобы доля промахов была целевой (по записанному raw этого прогона;")
        print("  статический расчёт без обратной связи динамики — после смены параметров сверяй с фактической долей выше):")
        print("    целевая_доля_промахов   CEIL   нужный SCALE")
        for r in F["calibration"]:
            sc = r['scale_needed']
            _cc = F.get("calibration_current_ceil")
            _mark = "   ← текущий CEIL" if (_cc is not None and abs(r['ceil'] - _cc) < 1e-9) else ""
            print(f"    {r['target_miss']:21.2f} {r['ceil']:6.2f}   {('%.4f' % sc) if sc is not None else 'недостижимо'}{_mark}")
        print("    (недостижимо = при таком CEIL доля попаданий не может быть выше CEIL)")
        bw = F.get("by_wall_phase", {})
        if bw: print("  hit_rate по фазе стены: " + "; ".join(f"{k_}: n={v['n']} hit={v['hit_rate']}" for k_, v in bw.items()))
    else: print("  ударов в журнале нет (хищники не атаковали).")
    En = M["energy"]
    print("\n  ── [M2] БАЛАНС ЭНЕРГИИ: невязка считается, а не закрывается подписью ──")
    for key, tag in (("x1", "X1 (поле)"), ("x2", "X2 (запас агентов)")):
        e = En[key]; share = (e["residual_abs"] / e["abs_flow_sum"]) if e.get("abs_flow_sum") else 0.0
        print(f"  [{tag}] Σ ΔE={e['dE_total']:+.2f}; учтено в ledger={e['logged_net']:+.2f}; НЕВЯЗКА Σ={e['residual_sum']:+.4f}, Σ|res|={e['residual_abs']:.4f} ({share:.4%} от Σ|потоков|), max|res|={e['residual_max_abs']:.4f} @t={e['residual_max_t']}")
        g = e.get("gap")
        if g: print(f"      зазор между тиками: Σ={g['sum']:+.3e}, max={g['max_abs']:.3e} (n={g['n']})")
        ow = e.get("outside_window") or {}
        if ow: print(f"      вне окна тика (инициализация): Σ={sum(v['net'] for v in ow.values()):+.2f}")
        for n_, net, pos, neg, calls in e["top_flows"][:8]: print(f"        {n_[:62]:62s} {net:+12.1f}  (+{pos:.1f} / {neg:.1f}, вызовов {calls})")
    memo = En.get("memo", {})
    if memo:
        print("  memo — энергия, не прошедшая клип X2 [-1, 2] (в баланс не входит):")
        for n_, v in sorted(memo.items(), key=lambda kv: -abs(kv[1]['not_added_by_clip'] + kv[1]['not_charged_by_clip'])): print(f"        {n_[:72]:72s} не добавлено {v['not_added_by_clip']:+11.1f}; не списано {v['not_charged_by_clip']:+11.1f}; вызовов {v['n']}")
    print("  ряд (t: E_поле, E_запас_агентов, pop, ферол):")
    for s in En["series"]: print(f"        t={s['t']:5d}  E1={s['E1']:10.1f}  E2={s['E2']:8.1f}  pop={s['pop']:4d}  ферол={s['feral']:3d}")
    D = M["dialogue"]; L = D.get("ledger", {})
    print("\n  ── [M3] ДИАЛОГ: слышание → истолкование → отклик (журнал по всему прогону, включая умерших) ──")
    if L.get("heard"):
        h = L["heard"]
        print(f"  услышано={h}: первая встреча={L.get('first_encounter', 0)}, повторная={L.get('repeat_encounter', 0)}; SEEK={L.get('seek', 0)}: принято={L.get('seek_registered', 0)}, отказ trust/grief/coherence={L.get('seek_gate_trust', 0)}/{L.get('seek_gate_grief', 0)}/{L.get('seek_gate_coherence', 0)}; ANSWER={L.get('answer_heard', 0)}, облегчение={L.get('relief', 0)}; low_sim={L.get('low_sim', 0)}")
        print(f"  Σ|Δgrief|={L.get('d_grief_abs', 0):.2f}, Σ|Δgrat|={L.get('d_grat_abs', 0):.2f}; реплик со сдвигом grief больше лимита эмоции за тик: {L.get('d_grief_gt_tick_cap', 0)} из {h}"
              + (f" [с relief ANSWER: {L.get('d_grief_gt_tick_cap_relief', 0)}; прочие: {L.get('d_grief_gt_tick_cap_other', 0)}]"
                 if 'd_grief_gt_tick_cap_other' in L else " [старый счётчик: без допуска на float и без разбивки relief/прочие]"))
        print("  по доверию к говорящему (n, средний Δgrief, средний |Δgrief|): " + "; ".join(f"{k_}: {v['n']}, {v['mean_d_grief']:+.4f}, {v['mean_abs_d_grief']:.4f}" for k_, v in sorted(D.get("by_trust_band", {}).items())))
        print("  по стороне стены (n, средний Δgrief, средний Δgrat): " + "; ".join(f"{k_}: {v['n']}, {v['mean_d_grief']:+.4f}, {v['mean_d_grat']:+.4f}" for k_, v in sorted(D.get("by_cross_wall", {}).items())))
    else: print("  реплик в журнале нет.")
    O = M.get("outcome") or {}
    print("\n  ── [M3b] РЕЧЬ, ОБУЧАЕМАЯ ИСХОДОМ: форма слова × контекст → что агент почувствовал в ответе ──")
    if O.get("stats", {}).get("n"):
        ch = O["choices"]; tot = sum(ch.values()) or 1
        print(f"  выбор слова: по умолчанию={ch.get('default', 0)}, выученное={ch.get('learned', 0)} ({ch.get('learned', 0) / tot:.1%}), проба={ch.get('explore', 0)}; исходов={O['stats']['n']}, средний исход={O['stats']['gain_sum'] / O['stats']['n']:+.4f}")
        print("  кривая обучения (окно 100 тиков): исходов, средний исход, доля выученных слов")
        for w, v in O["curve"].items(): print(f"    t={w * 100:5d}: {v['n']:6d}  {v['mean_gain']:+.4f}  {v['learned_share']:.1%}")
        print("  средний исход по слову (эмоция/намерение): " + "; ".join(f"{k}: {v['mean_gain']:+.4f} (n={v['n']})" for k, v in sorted(O["by_word"].items(), key=lambda kv: -kv[1]["n"])[:10]))
        print("  по контексту (горе, доверие; 0=низк 1=средн 2=высок): " + "; ".join(f"{k}: {v['mean_gain']:+.4f} (n={v['n']})" for k, v in sorted(O["by_ctx"].items())))
        print("  чему агенты научились (контекст, слово → выученный исход, наблюдений) — самые частые:")
        for r in O.get("value_table", [])[:12]: print(f"    ctx={tuple(r['ctx'])} {r['emotion']}/{r['intent']}: {r['value']:+.4f} (n={r['n']})")
    else: print("  исходов в журнале нет (OUTCOME_LEARNING выключен или нет диалогов).")
    Md = M.get("modes", [])
    print("\n  ── [M4] РЕЖИМЫ ИСТОРИЧНОСТИ (прокси по направленным книгам Imprint, не определение) ──")
    print("  Участие = у обоих есть след друг друга; Ношение = у меня есть след, у него нет; Форма = след ушедшего. «Первая/повторная» — по счётчику контактов.")
    print("      t  записей  Участие  Ношение  Форма  первых  повторных  разный_знак_тяги")
    for m in Md[::max(1, len(Md) // 14)]:
        print(f"  {m['t']:5d} {m['directed_entries']:8d} {m['participation_mutual']:8d} {m['bearing_one_sided']:8d} {m['form_trace_of_departed']:6d} {m['first_contact_entries']:7d} {m['repeat_contact_entries']:10d} {m['mutual_gravity_sign_split']:10d}")
    W = M["wall"]
    print("\n  ── [M5] ЦИКЛ СТЕНЫ ──")
    print(f"  параметры: {W.get('cycle')}")
    for e in W.get("events", []):
        print(f"  t={e['t']:5d} → {e['to']:3s}: pop={e['pop']}; на линии агентов {e['agents_on_line_before']}→{e['agents_on_line_after']}, клеток {e['cells_on_line_before']}→{e['cells_on_line_after']}; миграция dropped/shifted/kept={e['migrated_dropped']}/{e['migrated_shifted']}/{e['migrated_kept']}; сброшено «призрачных» owner={e['stale_owner_cleared']}")
    for k_, v in W.get("by_phase", {}).items():
        print(f"  фаза {k_:3s}: тиков={v['ticks']}, pop средн.={v['pop_mean']:.1f}, ферол средн.={v['feral_mean']:.1f}, рождений/тик={v['births_per_tick']:.3f}, удалений/тик={v['removed_per_tick']:.3f}")


def _wilson(k, n, z=1.96):
    if n <= 0: return (float("nan"), float("nan"))
    p = k / n; den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (c - h, c + h)


def _q(arr, qs=(0.05, 0.25, 0.5, 0.75, 0.95)):
    a = np.asarray(arr, dtype=float)
    return {str(q): float(np.quantile(a, q)) for q in qs} if a.size else {}


_tag_fn = globals().get("_tag")
if callable(_tag_fn) and "RUN_ARMS" in globals():
    _TAGS = [_tag_fn(_a) for _a in RUN_ARMS]
    if not _TAGS:
        # Single-run default: derive the tag from the same live config as _run_arm.
        _TAGS = [_tag_fn("main")]
else:
    _TAGS = []
    print("⚠️ Master Observatory: _tag unavailable; no guessed fallback tag will be used.")

RUN_NOW = True
if RUN_NOW:
    # Не называем переменную _tag: это имя функции генерации тегов из ячейки p832.
    # В ноутбуке все ячейки делят глобальное пространство имён, поэтому прежний цикл
    # затирал функцию строкой и ломал последующий _run_arm("main").
    for _tag_item in _TAGS:
        _log  = f"{DIR}/rich_log_{_tag_item}.json"
        _led  = f"{DIR}/rich_ledger_{_tag_item}.json"
        _prev = f"{DIR}/rich_log_{_tag_item}.json.prev"
        if not os.path.exists(_log):
            print(f"[skip] {_tag_item}: нет {_log}\n"
                  f"       (STEPS={globals().get('STEPS')} в момент выполнения ЭТОЙ ячейки; ячейка 38 позже ставит STEPS=333. "
                  f"Это порядок ячеек, а не потеря данных: отчёт по реальному прогону печатается в конце ячейки 38.)")
            continue
        run_master(_log, _led if os.path.exists(_led) else None, _prev if os.path.exists(_prev) else None)

print("✅ Master Observatory v0.2.0 (с [N] СИГНАЛ v3.0, [Z] OBSERVER FACTS) загружена.")


In [ ]:
# Ячейка 37
import os, json, math


def _o_wilson(k, n, z=1.96):
    f = globals().get("_wilson")
    if callable(f):
        return f(k, n, z)
    if n <= 0:
        return (float("nan"), float("nan"))
    p = k / n; den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (c - h, c + h)


def _obs_print(c):
    pre = lambda p: {kk[len(p):]: int(v) for kk, v in sorted(c.items()) if kk.startswith(p)}
    bins = ['<0.5', '0.5-1', '1-1.5', '1.5-2', '2-3', '3-4', '4-5', '5-7', '>=7']
    print("\n  ── [O10] ДИАГНОСТИКА: SCAR и барьер намерений (observer, поведение не меняет) ──")
    for tag, nm in (('a', 'действию'), ('s', 'размеру тела (клеток, 40=40+)')):
        h, cs, tk = pre('obs_scar_hits_' + tag), pre('obs_scar_cost_x1000_' + tag), pre('obs_ticks_' + tag)
        keys = sorted(set(h) | set(tk), key=lambda x: (len(x), x))
        tot = sum(cs.values()) or 1
        print(f"  SCAR по {nm}: " + "; ".join(
            f"{k}: попаданий={h.get(k, 0)} фактически_списано={cs.get(k, 0) / 1000:.2f} ({100 * cs.get(k, 0) / tot:.0f}%) "
            f"агент-тиков={tk.get(k, 0)}" for k in keys))
    sw, br = pre('obs_sw_c'), pre('obs_bar_c')
    print(f"  update_intent: вызовов с активным intent={int(c.get('obs_ui_calls', 0))}, в кулдауне={int(c.get('obs_ui_cooldown', 0))}")
    print(f"  commitment по корзинам {bins}")
    print(f"    переключилось: {[sw.get(str(i), 0) for i in range(9)]}")
    print(f"    упёрлось в барьер: {[br.get(str(i), 0) for i in range(9)]}")


def _obs_om_print(c):
    g = lambda k: int(c.get(k, 0))
    pre = lambda p: {kk[len(p):]: int(v) for kk, v in sorted(c.items()) if kk.startswith(p)}
    print("\n  ── [O11] ОБУЧЕНИЕ РЕЧИ (OM): жизненный цикл pending + разбор выбора (observer) ──")
    own, hd = g('obs_om_rec_own'), g('obs_om_rec_heard')
    print(f"  записи: свои (исход реплики)={own}, услышанные (heard_memory)={hd}, доля своих={own / max(1, own + hd):.2f}; "
          f"свои без исхода (ответчик не слышит ответа)={g('obs_om_own_no_outcome')}; "
          f"реплик составлено={g('obs_om_compose')} (без ctx={g('obs_om_compose_noctx')}); pending перезаписан={g('obs_om_own_overwritten')}")
    edges = ['-.02', '-.01', '-.005', '-.0025', '-.001', '-.0005', '.0005', '.001', '.0025', '.005', '.01', '.02']
    print(f"  gain по корзинам, границы={edges} (13 корзин: слева от первой … справа от последней)")
    for cls, nm in (('ownN', 'своя, без relief'), ('ownR', 'своя, с relief'), ('heardN', 'услышанная, без relief'), ('heardR', 'услышанная, с relief')):
        h = pre(f'obs_om_h_{cls}_')
        arr = [h.get(str(i), 0) for i in range(13)]
        n = sum(arr)
        if not n:
            print(f"    {nm:24s} n=0")
            continue
        m = g('obs_om_gsum_x1e6_' + cls) / 1e6 / n
        sd = max(0.0, g('obs_om_gsq_x1e9_' + cls) / 1e9 / n - m * m) ** 0.5
        print(f"    {nm:24s} n={n:6d} среднее={m:+.5f} σ={sd:.5f} {arr}")
    print(f"  проверка потолка: |gain|>2·CAP без relief: свои={g('obs_om_capviol_own')} услышанные={g('obs_om_capviol_heard')} (ожидание 0 → потолок 0.004 для обычной реплики подтверждён)")
    ev = g('obs_om_pick_eval')
    print(f"  выбор слова: всего={g('obs_om_pick')}; нет альтернатив={g('obs_om_pick_no_alt')}; explore={g('obs_om_pick_explore')}; "
          f"оценено по памяти={ev}; learned={g('obs_om_learned')} ({100 * g('obs_om_learned') / max(1, g('obs_om_pick')):.2f}% от всех, {100 * g('obs_om_learned') / max(1, ev):.2f}% от оценённых)")
    al = pre('obs_om_alt_')
    print(f"  доступность альтернатив (e=эмоция, i=интент): {{ {', '.join(f'{k}: {v}' for k, v in al.items() if k in ('ei', 'e-', '-i', '--'))} }}")
    print(f"  среди оценённых: у базы n<MIN_N={g('obs_om_base_lowN')}; альтернатив с n<MIN_N={g('obs_om_alt_lowN')}, с n≥MIN_N={g('obs_om_alt_enough')}")
    dh = pre('obs_om_dh_')
    print(f"  разрыв v_альт − v_база (только n≥MIN_N), границы=[-.01,-.005,-.002,-.001,0,.001,.002,.005,.01,.02]: {[dh.get(str(i), 0) for i in range(11)]}")
    ms = (0.0, 0.0005, 0.001, 0.002, 0.004, 0.01)
    print("  контрфактуал: % оценённых выборов, ставших бы learned (поведение не применено); строки=OM_MARGIN, столбцы=OM_MIN_N")
    print("    margin \\ min_n " + "".join(f"{kn:>9d}" for kn in (2, 3, 5)))
    for mi, m in enumerate(ms):
        print(f"    {m:<14} " + "".join(f"{100 * g(f'obs_om_cf_m{mi}_n{ni}') / max(1, ev):>8.2f}%" for ni in range(3)))
    print(f"    (ячейка, соответствующая реальным OM_MARGIN и OM_MIN_N прогона, должна совпасть с learned/оценено={100 * g('obs_om_learned') / max(1, ev):.2f}%)")
    print(f"  из learned: держится на данных без relief={g('obs_om_learned_survives_norelief')}, зависит от relief={g('obs_om_learned_relief_dependent')}, неизвестно (нет aux)={g('obs_om_learned_aux_unknown')}")
    wp, wl = pre('obs_om_pick_w'), pre('obs_om_learned_w')
    ks = sorted(wp, key=int)
    if ks:
        print("  по окнам 100 тиков (выборов/learned): " + "; ".join(f"{int(k) * 100}+: {wp[k]}/{wl.get(k, 0)}" for k in ks))
    if g('obs_snap_t'):
        print(f"  снимок t={g('obs_snap_t')}: агентов={g('obs_snap_agents')}; ключей памяти={g('obs_snap_keys')} (n≥3: {g('obs_snap_keys_n3')}, n≥10: {g('obs_snap_keys_n10')}); "
              f"ключей со средним > OM_MARGIN={g('obs_snap_keys_v_gt_margin')}; max|среднее|={g('obs_snap_maxabs_x1e6') / 1e6:.5f}; "
              f"heard_memory: ключей={g('obs_snap_heard_keys')} (n≥3: {g('obs_snap_heard_keys_n3')})")


def run_fch_report(log_path):
    ap = globals()["_audit_path_for"](log_path) if "_audit_path_for" in globals() else None
    print("=" * 78)
    print("[O] НАВИГАЦИЯ + КАНАЛЫ ПОД ФЕРОЛАМИ (summary['fch'] из audit JSON; ничего не пересчитывает)")
    print("=" * 78)
    if not ap or not os.path.exists(ap):
        print(f"  [!] нет audit-файла: {ap}")
        return None
    with open(ap, "r", encoding="utf-8") as f:
        A = json.load(f)
    F = (A.get("summary") or {}).get("fch")
    if not F:
        print("  [!] summary['fch'] нет: ячейка NAV+FCH не была подключена в этом прогоне")
        return None
    c, V, N, T = F["counters"], F["veto"], F["nav"], F["tame"]
    print(f"  включено: {F['enabled']}")
    # [CURATOR2 FIX 8.2] NAV_LEGACY_TENSION_HACK переключает мгновенный (без squash) путь дезориентации в tension
    # на отложенный (через nav_motion_err -> raw_tension на следующем такте). Временная модель из раздела 1.3/Вывод Б
    # верна только при False. Значение ниже — текущий конфиг в ЭТОЙ сессии на момент отчёта, не обязательно конфиг
    # времени самого прогона (если Config менялся между запуском и отчётом).
    print(f"  NAV_LEGACY_TENSION_HACK (текущий конфиг сессии, см. аудит 8.2): "
          f"{getattr(Config, 'NAV_LEGACY_TENSION_HACK', None)}")
    print("\n  ── [O1] НАВИГАЦИЯ ──")
    print(f"  замеров поля: {c.get('senses', 0)}; дезориентаций (z>порога): {c.get('disorient_events', 0)}; "
          f"мягких целей rest: {c.get('rest_goal', 0)}")
    if N.get('place_nodes'):
        print(f"  руль: бегство={c.get('steer_flee', 0)} моббинг={c.get('steer_mob', 0)}")
        print(f"  места (PlaceNode): создано={c.get('place_new', 0)} узнано={c.get('place_hit', 0)} (стоянка/непрерывность={c.get('place_cont', 0)}); "
              f"не подтверждено={c.get('place_tentative', 0)} неоднозначно={c.get('place_ambig', 0)} неуверенно={c.get('place_uncertain', 0)} "
              f"наблюдений новизны={c.get('place_pending', 0)} потеряно={c.get('place_left', 0)}")
        print(f"  отсечено по pi при совпавшем виде={c.get('place_gate_rej', 0)}; вытеснено узлов={c.get('place_evict', 0)}; переходов записано={c.get('place_edge', 0)}")
        print(f"  память: узлов на агента={N.get('places_mean', 0):.1f} (макс {N.get('places_max', 0)}), визитов на узел={N.get('visits_mean', 0):.1f}, "
              f"рёбер на агента={N.get('edges_mean', 0):.1f}, доля агентов с гипотезой места={N.get('known_frac', 0):.2f}")
        print(f"  при узнавании: ошибка счёта пути (средняя)={N.get('place_err_pi_mean', 0):.1f}, сюрприз вида (средний)={N.get('place_err_fp_mean', 0):.2f}; "
              f"ошибка места сейчас (среднее по живым)={N.get('spatial_err_live_mean', 0):.3f}, уверенность в месте={N.get('place_conf_mean', 0):.3f}")
    else:
        print(f"  руль: бегство={c.get('steer_flee', 0)} моббинг={c.get('steer_mob', 0)} якоря: новых={c.get('anchor_new', 0)} узнано={c.get('anchor_hit', 0)} отклонено gate={c.get('anchor_gate_rej', 0)}")
    print(f"  цель-отклик на крик целится по оценке: {c.get('scream_est', 0)}; без оценки (координат нет): {c.get('scream_est_none', 0)}")
    if N.get('place_nodes'):
        print(f"  дезориентация dis: средняя={N['dis_mean']:.3f} p90={N['dis_p90']:.3f}")
    else:
        print(f"  дезориентация dis: средняя={N['dis_mean']:.3f} p90={N['dis_p90']:.3f}; якорей у агента (среднее)={N['anchor_mean']:.1f}; ошибка счёта при узнавании (средняя)={N['anchor_err_mean']:.1f}")
    print("\n  ── [O2] ФЕРОЛЫ И КАНАЛЫ ──")
    print(f"  отступления ферола: по Морзе={V.get('morse', 0)}, по приручению={V.get('tame', 0)}")
    print(f"  Морзе под угрозой: ушло {c.get('morse_threat_ok', 0)} из {c.get('morse_threat_try', 0)} попыток (остальное режут ворота _send_morse)")
    print(f"  умиротворение: форм ушло={c.get('appease_emit', 0)}, заблокировано={c.get('appease_blocked', 0)}, нет формы={c.get('appease_no_form', 0)}")
    print(f"  приручение: пар ферол→агент={T['pairs']}, среднее по положительным={T['pos_mean']:.3f}, максимум={T['max']:.3f}")
    print(f"  опыт умиротворяющих форм fx: форм={F['appease_fx']['forms']}, средний={F['appease_fx']['mean']:.3f}")
    bins = {k: v for k, v in c.items() if k.startswith("pe_bin")}
    print(f"  что слышит ферол (жест выше фона, корзины <.05/<.1/<.2/<.4/≥.4): {[bins.get('pe_bin%d' % i, 0) for i in range(5)]}; "
          f"фон жестов={F.get('bg_peace', 0):.3f}, фон Морзе={F.get('bg_morse', 0):.4f}")
    print("\n  ── [O3] ПРОФИЛЬ ЗАЩИТЫ ЖЕРТВЫ В МОМЕНТ ВЫБОРА (M=Морзе слышен, T=приручён, A=тревога над жертвой) ──")
    P = F["profile"]
    base = P.get("M0T0A0")
    print("  профиль   в зоне  ударов  убито   доля_ударов/в_зоне   убийств/в_зоне [95% CI]   риск убийства vs M0T0A0")
    for k, v in P.items():
        e, s_, kl = v["exposed"], v["strikes"], v["kills"]
        lo, hi = _o_wilson(kl, e)
        rr = ""
        if base and base["exposed"] > 0 and e > 0 and base["kills"] > 0:
            rr = f"{(kl / e) / (base['kills'] / base['exposed']):.2f}×"
        print(f"  {k:8s} {e:7d} {s_:7d} {kl:6d}   {(s_ / e if e else float('nan')):17.3f}   {(kl / e if e else float('nan')):.3f} [{lo:.3f}..{hi:.3f}]   {rr}")
    print("  (маленькие n не доказывают отбор: смотри на доверительные интервалы; контроль — прогон с _FCH_ABL=все True / NAV_ENABLE=False)")
    print("\n  ── [O4] ГЕНЫ (наследуются с мутацией; старт: flee/mob/coping ~0.5, morse ~1.0) ──")
    for k, v in F["genes"].items():
        if k == "n":
            continue
        print(f"  {k:7s} среднее={v['mean']:.3f} σ={v['std']:.3f} p10={v['p10']:.3f} p90={v['p90']:.3f}")
    print(f"  живых обычных агентов: {F['genes']['n']}")
    H = F.get("hist", [])
    if H:
        print("\n  динамика (каждые 50 тиков):   t     n    flee    mob  coping  morse  ударов убито  отступл.(morse/tame)")
        for h in H:
            g = h["genes"]
            print(f"  {h['t']:24d} {h['n']:5d} {g['flee']:7.3f} {g['mob']:6.3f} {g['coping']:7.3f} {g['morse']:6.3f} {h['strikes']:7d} {h['kills']:5d}  "
                  f"{h['veto'].get('morse', 0)}/{h['veto'].get('tame', 0)}")
    _q = F.get("pn2")
    if _q is None:
        print("\n  [O5–O8] PlaceNode v2: в audit нет блока pn2 (прогон до v22)")
    elif "_pn2_print" in globals():
        _pn2_print(_q, c, H)
    if "_pn3_print" in globals():
        _pn3_print(F.get("pn3"), c)
    _obs_print(c)
    _obs_om_print(c)
    errs = {k: v for k, v in c.items() if k.startswith("err_")}
    print(f"\n  ошибки слоя: {errs or 'нет'}; последняя: {F.get('error')}")
    return F


def _nav_report(eng=None):
    import numpy as np
    from collections import Counter
    eng = eng if eng is not None else globals().get("_PN_LAST", {}).get("eng")
    print("=" * 78)
    print("[NAV] навигация: направления, spatial_gradient, gap_clip (read-only)")
    print("=" * 78)
    a = getattr(eng, "_nav_audit_d", None) if eng is not None else None
    if not a:
        print("  [!] нет данных навигации (движок не создан или патч не активен)")
        return None
    c = a["c"]
    ep = list(a["ep"])
    arr = [e for e in ep if e["status"] == "arrived"]
    ticks = [e["ticks"] for e in arr]
    eff = [min(1.0, e["opt"] / e["act"]) for e in arr if e["act"] > 1e-9]
    rev = [1.0 if e["revisit"] else 0.0 for e in ep]
    med = float(np.median(ticks)) if ticks else float("nan")
    dsh = c["dir_changes"] / max(1, c["dir_ticks"])
    print(f"  [A] эпизоды: всего={c['ep_total']} дошли={c['ep_arrived']} брошены={c['ep_abandoned']} "
          f"пересчётов={c['recalc_total']} (blocked={c['wp_recalc_blocked']} stuck={c['wp_recalc_stuck']} timeout={c['wp_recalc_timeout']}) новых целей={c['wp_new']}")
    # Report terminal causes from the bounded episode sample, so a high abandoned
    # count can be diagnosed without guessing from the aggregate status alone.
    reason_counts = Counter(str(e.get('end_reason') or e.get('status', 'unknown')) for e in ep)
    if reason_counts:
        print("      причины завершения (сохранённые эпизоды): " +
              " | ".join(f"{k}={v}" for k, v in sorted(reason_counts.items(), key=lambda kv: (-kv[1], kv[0]))))
    print(f"  [B] primary: median ticks-to-goal={med:.1f} (n={len(ticks)}) | доля тиков со сменой направления={dsh:.3f} ({c['dir_changes']}/{c['dir_ticks']})")
    print(f"  [C] secondary: path_eff mean={(float(np.mean(eff)) if eff else float('nan')):.3f} (n={len(eff)}) | эпизодов с возвратом в посещённую область={(float(np.mean(rev)) if rev else float('nan')):.3f} (n={len(rev)})")
    print(f"  [D] гистерезис отключён по застреванию: {c['hyst_override']} тиков | "
          f"сглаживание NAV_PERC_SMOOTH={float(getattr(Config, 'NAV_PERC_SMOOTH', 1.0)):.2f} (1.0 = выкл)")
    print(f"  [E] gap_clip выше порога: go_goal={c['gapclip_go_goal']} explore_no_goal={c['gapclip_explore_no_goal']} "
          f"explore_low_conf={c['gapclip_explore_low_conf']} stuck_goal={c['gapclip_stuck_goal']} | "
          f"действия: explore={c['gapclip_action_explore']} lowconf={c['gapclip_action_lowconf']}")
    # v42: where do episodes come from and how do they end (regular goal vs gap_clip-triggered explore)
    by_origin = {}
    for e in ep:
        by_origin.setdefault(e.get("origin", "new"), []).append(e)
    if by_origin:
        print("  [H] исходы эпизодов по происхождению (origin):")
        for og, es in sorted(by_origin.items(), key=lambda kv: -len(kv[1])):
            arr_ = [e for e in es if e["status"] == "arrived"]
            tk_ = [e["ticks"] for e in arr_]
            print(f"      {og:16s} n={len(es):4d} дошли={100.0 * len(arr_) / len(es):5.1f}% брошены={100.0 * sum(1 for e in es if e['status'] == 'abandoned') / len(es):5.1f}% "
                  f"median ticks={(float(np.median(tk_)) if tk_ else float('nan')):6.1f} пересчётов/эпизод={float(np.mean([e['recalc'] for e in es])):.2f}")
    gt = c["gap_ticks"]
    if gt:
        thr = float(getattr(Config, "GAP_CLIP_CURIOSITY_THRESHOLD", 0.85))
        hi = c["gx_hi_lowconf"] + c["gx_hi_okconf"]
        print(f"  [I] gap_clip vs пространственная уверенность (dir_conf<0.5), замеров={gt}, порог={thr:.2f}:")
        print(f"      выше порога: {100.0 * hi / gt:5.1f}% (из них низкая уверенность {100.0 * c['gx_hi_lowconf'] / max(1, hi):.0f}%) | "
              f"ниже порога, но уверенность низкая (реакция GAP_CLIP не сработает): {100.0 * c['gx_lo_lowconf'] / gt:5.1f}% замеров")
        print("      что-если по порогу (замеры без цели, где gap_clip ≥ порога → кандидаты на explore_no_goal):")
        row = []
        for tt in (0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95):
            b0 = int(np.ceil(tt * 20.0 - 1e-9))
            row.append(f"{tt:.2f}:{sum(c['gap_h%02d_nogoal' % b] for b in range(min(b0, 20), 20))}")
        print("      " + "  ".join(row) + f"   (фактически explore_no_goal при {thr:.2f}: {c['gapclip_explore_no_goal']})")
        for ch_ in ("go_goal", "explore_no_goal", "explore_low_conf", "stuck_goal"):
            n_ = c["gapclip_" + ch_]
            if n_:
                print(f"      {ch_:17s} n={n_:5d} средний gap={c['og_gap_sum_' + ch_] / n_:.3f} conf={c['og_conf_sum_' + ch_] / n_:.3f} threat={c['og_threat_sum_' + ch_] / n_:.3f}")
    print(f"  [J] повторных вызовов _nav_step в том же тике отброшено: {c['step_dup_skipped']} "
          f"(NAV_STEP_ONCE_PER_TICK={bool(getattr(Config, 'NAV_STEP_ONCE_PER_TICK', True))})")
    mc = max(1, c["mv_call"])
    print(f"  [F] подвижность: агент-тиков={c['mv_call']} попыток={100.0 * c['mv_try'] / mc:.1f}% успешных шагов={100.0 * c['mv_ok'] / mc:.1f}% "
          f"нет свободного направления={100.0 * c['mv_fail_nodir'] / mc:.1f}% отказ по оценке={100.0 * c['mv_fail_thr'] / mc:.1f}% отказ при сдвиге={100.0 * c['mv_fail_occ'] / mc:.1f}%")
    print(f"      ручки: MOBILITY_MULT={float(getattr(Config, 'MOBILITY_MULT', 1.0)):.2f} MOVE_SCORE_THR={float(getattr(Config, 'MOVE_SCORE_THR', 0.3)):.2f} "
          f"NAV_DIR_STEER_GAIN={float(getattr(Config, 'NAV_DIR_STEER_GAIN', 0.0)):.2f}")
    try:
        _ws = eng.witness.summary()
        print(f"  [G] феральность: вход became_feral={int(_ws.get('became_feral', 0))} | "
              f"выход became_normal={int(_ws.get('became_normal', 0))} (reason: redemption / redemption_arc)")
    except Exception:
        pass
    print("=" * 78, flush=True)
    return dict(c)


def _sanctuary_report(eng=None):
    import numpy as np
    eng = eng if eng is not None else globals().get("_PN_LAST", {}).get("eng")
    print("=" * 78)
    print("[SANC] возвраты агентов в безопасные зоны (центр тела, каждый тик)")
    print("=" * 78)
    tr = getattr(eng, "_sanc_track_d", None) if eng is not None else None
    if not tr:
        print("  [!] нет данных (трекер не активен или зоны выключены)")
        return None
    recs = list(tr["agents"].values())
    seen = len(tr["seen"])
    n = len(recs)
    pct = lambda a, b: (100.0 * a / b) if b else 0.0
    once = [r for r in recs if r["entries"] == 1]
    stayed = [r for r in once if r["in"] or r["dead_inside"]]
    left = [r for r in once if not (r["in"] or r["dead_inside"])]
    re2 = [r for r in recs if r["entries"] >= 2]
    re3 = [r for r in recs if r["entries"] >= 3]
    res = [r for r in recs if r["ticks"] / max(1, r["last_t"] - r["born"] + 1) >= 0.5]
    stays = [s for r in recs for s in r["stays"]]
    gaps = [g for r in recs for g in r["gaps"]]
    roles = Counter(r["role"] for r in recs)
    walked = [r for r in recs if r["age_first"] > 2]
    top = sorted(recs, key=lambda r: -r["entries"])[:3]
    print(f"  как попали: родились внутри={n - len(walked)} пришли сами={len(walked)} (из них вернулись={sum(1 for r in walked if r['entries'] >= 2)}) "
          f"| больше всех входов у одного агента: {[r['entries'] for r in top]}")
    print(f"  агентов: всего видели={seen} вошли хоть раз={n} ({pct(n, seen):.1f}%)")
    print(f"  один вход: {len(once)} (остались до конца/смерти={len(stayed)}, зашли и ушли={len(left)}) | "
          f"вернулись (>=2 входов)={len(re2)} ({pct(len(re2), n):.1f}% от вошедших) | >=3 входов={len(re3)} | "
          f"жильцов (>=50% жизни внутри)={len(res)}")
    if stays:
        print(f"  длительность пребывания, тиков: n={len(stays)} mean={np.mean(stays):.1f} median={np.median(stays):.1f} p90={np.percentile(stays, 90):.1f}")
    if gaps:
        print(f"  интервал выход→повторный вход, тиков: n={len(gaps)} median={np.median(gaps):.1f} p90={np.percentile(gaps, 90):.1f}")
    # v37: per-agent cumulative time-in-sanctuary (ticks), not just per-visit `stays`.
    ever_ticks = [r["ticks"] for r in recs if r["ticks"] > 0]
    if ever_ticks:
        print(f"  суммарное время в зоне на агента, тиков: n={len(ever_ticks)} mean={np.mean(ever_ticks):.1f} "
              f"median={np.median(ever_ticks):.1f} p90={np.percentile(ever_ticks, 90):.1f} max={max(ever_ticks):.0f}")
    # v37: continuity-contract compliance for the sanctuary_return commitment. Only agents
    # alive right now carry a live _continuity_contracts list, so this undercounts agents
    # who already died -- their ticks/stays/gaps above still include them, this line does not.
    if getattr(Config, 'ENABLE_SANCTUARY_CONTINUITY_CONTRACTS', True) and eng is not None:
        live = [p for p in getattr(eng, 'patterns', []) if getattr(p, 'alive', False)]
        sanc_contracts = [ctr for p in live for ctr in getattr(p, '_continuity_contracts', [])
                           if ctr.get('kind') == 'sanctuary_return']
        if sanc_contracts:
            held = len(sanc_contracts)
            reviewed = sum(ctr.get('review_count', 0) for ctr in sanc_contracts)
            violated = sum(ctr.get('violations', 0) for ctr in sanc_contracts)
            print(f"  continuity contracts 'sanctuary_return' (живые сейчас): держат={held} "
                  f"проверок={reviewed} нарушений={violated} "
                  f"(gap-лимит={int(Config.CONTINUITY_CONTRACT_SANCTUARY_GAP_LIMIT)} тиков)")
    print(f"  роль при первом входе: {dict(roles)} | внутри сейчас={sum(1 for r in recs if r['in'])} умерли внутри={sum(1 for r in recs if r['dead_inside'])}")
    print("=" * 78, flush=True)
    return {"seen": seen, "entered": n, "once": len(once), "returned": len(re2), "re3": len(re3)}


def _returns_from_log(path):
    import json
    import numpy as np
    print("[RETURNS] постфактум по agent_snaps (разрешение = SNAPSHOT_EVERY тиков)")
    log = json.load(open(path, encoding="utf-8"))
    have = [s for s in log.get("agent_snaps", []) if "zone" in s]
    if not have:
        print("  [!] в agent_snaps нет полей x/y/zone/place_id")
        return None
    by = {}
    for s in have:
        by.setdefault(s["id"], []).append(s)
    zone_n, durs = {}, []
    place_ret, place_agents = 0, 0
    for pid, seq in by.items():
        seq.sort(key=lambda s: s["t"])
        prev, k, start = False, 0, None
        for s in seq:
            ins = s["zone"] >= 0
            if ins and not prev:
                k += 1
                start = s["t"]
            if (not ins) and prev and start is not None:
                durs.append(s["t"] - start)
                start = None
            prev = ins
        zone_n[pid] = k
        seen_pl, last, ret = set(), None, 0
        for s in seq:
            pl = s.get("place_id")
            if pl is None or pl == last:
                continue
            if pl in seen_pl:
                ret += 1
            seen_pl.add(pl)
            last = pl
        if ret:
            place_agents += 1
        place_ret += ret
    ent = [k for k in zone_n.values() if k >= 1]
    print(f"  агентов в снапшотах={len(by)} | заходили в зоны={len(ent)} | 1 вход={sum(1 for k in ent if k == 1)} "
          f"| >=2 входов={sum(1 for k in ent if k >= 2)} | >=3 входов={sum(1 for k in ent if k >= 3)}"
          + (f" | завершённых визитов={len(durs)} median={np.median(durs):.0f}" if durs else ""))
    print(f"  места (place_id): агентов с возвратом на ранее посещённое место={place_agents} | возвратов всего={place_ret}")
    return {"zone_entries": zone_n, "place_returns": place_ret}


def _nav_selftest():
    """Fast unit checks of the navigation layer: direction evaluation, stall/recalc, gap_clip choice.
    Uses synthetic fields only; does not start a simulation."""
    import types as _types
    W = int(Config.WORLD_SIZE)
    D = {d: i for i, d in enumerate(_NAV_DIRS)}
    dists = [int(x) for x in Config.NAV_PERC_RAY_DISTS]
    f = np.zeros((W, W, max(CH.values()) + 1))
    cx = cy = W // 2
    for d in dists:
        f[cx + d, cy, CH['energy']] = 1.0              # resources to the east
        f[cx, cy + d, CH['signal_alarm']] = 1.0        # threat towards +y
        f[cx, cy + d, CH['signal_feral']] = 1.0
        f[cx, cy - d, CH['unknown']] = 0.8             # uncertainty towards -y
    f[cx - dists[1], cy, CH['wall']] = 1.0             # obstacle to the west, behind the first ray point
    ev = _nav_dir_eval(f, cx, cy, 1)
    E, N, S, Wd = D[(1, 0)], D[(0, 1)], D[(0, -1)], D[(-1, 0)]
    assert ev['raw'][E] > 0 > ev['raw'][N], 'resources must score above threats'
    assert ev['threat'][N] > ev['threat'][E] and ev['res'][E] > ev['res'][N], 'signals stay separate'
    assert ev['unk'][S] > 0.5 and ev['conf'][S] < ev['conf'][E], 'uncertainty lowers confidence'
    assert ev['threat'][S] < 0.05, 'unknown is not danger'
    assert int(ev['reach_n'][Wd]) == 1, 'an obstacle must cut the ray'

    class _Nav(PatternPerception):
        def __init__(self):
            self.id, self.age, self.goals = 1, 100, []
            self._nav_progress, self._nav_cool_until, self.world = None, 0, None
            self._dir_contrast, self._dir_threat, self._dir_conf = 0.2, 0.0, 1.0
            self._dir_scores = {d: float('-inf') for d in _NAV_DIRS}
            self._dir_scores[(1, 0)] = 1.0
            self._dir_scores[(0, 1)] = 0.5
            self._dir_reach = {d: 8 for d in _NAV_DIRS}
            self._hyst_off = 0
        def get_center(self):
            return (float(cx), float(cy))
        def _nav_body_center(self):
            return self.get_center()
        def _log_event(self, *a, **k):
            pass

    s = _Nav()
    assert s._nav_spatial_goal('new') and s._nav_progress['dir'] == (1, 0)
    for _ in range(int(Config.NAV_PERC_STALL_TICKS) + 1):
        s.age += 1
        s._nav_move_attempt_pending = s._nav_progress   # move() прошлого тика пытался сдвинуться (без попытки stuck_ticks не растёт)
        s._nav_step(f)
    assert s._hyst_off == 2 and s._fail_dir == (1, 0), 'stall must lift hysteresis and penalise the failed direction'
    assert s._nav_progress['status'] == 'stalled'
    for _ in range(int(Config.NAV_STUCK_TICKS)):
        s.age += 1
        s._nav_move_attempt_pending = s._nav_progress
        s._nav_step(f)
        if s._nav_progress and s._nav_progress['recalc'] >= 1:
            break
    assert s._nav_progress and s._nav_progress['dir'] == (0, 1) and s._nav_progress['recalc'] == 1, 'stuck must recalc to another direction'
    assert (1, 0) in s._nav_block_dirs
    for _ in range(3 * int(Config.NAV_STUCK_TICKS)):
        if not s._nav_progress:
            break
        s.age += 1
        s._nav_move_attempt_pending = s._nav_progress
        s._nav_step(f)
    assert s._nav_progress is None and s._nav_last['status'] == 'abandoned', 'no alternative direction -> abandon'

    a = _Nav()
    assert a._nav_spatial_goal('new')
    tx, ty = a._nav_progress['target']
    a.get_center = lambda: (float(tx), float(ty))
    a.age += 1
    a._nav_step(f)
    assert a._nav_progress is None and a._nav_last['status'] == 'arrived' and not a.goals

    o = _Nav()
    o.world, o.age = _types.SimpleNamespace(), 200
    o._nav_gapclip_decision(0.9)
    c = _nav_audit(o.world)['c']
    assert c['gapclip_explore_no_goal'] == 1 and c['gapclip_action_explore'] == 1 and o._nav_progress is not None
    assert o._gapclip_last[1] == 'explore_no_goal'
    for stuck, conf, thr, want in ((0, 1.0, 0.0, 'go_goal'), (5, 0.9, 0.0, 'stuck_goal'), (5, 0.3, 0.0, 'explore_low_conf')):
        o._nav_progress['stuck_ticks'], o._dir_conf, o._dir_threat = stuck, conf, thr
        o._nav_gapclip_decision(0.9)
        assert o._gapclip_last[1] == want, (want, o._gapclip_last)
    # noise smoothing: rises are damped, drops (threat appearing) pass at once, alpha=1 is a no-op
    prev, new = np.array([0.0, 1.0, 0.5]), np.array([1.0, 0.0, 0.5])
    sm = _nav_smooth_raw(prev, new, 0.6)
    assert abs(sm[0] - 0.6) < 1e-9 and sm[1] == 0.0 and sm[2] == 0.5, sm
    assert _nav_smooth_raw(prev, new, 1.0) is new and _nav_smooth_raw(None, new, 0.6) is new
    assert _nav_smooth_raw(np.zeros(3), np.zeros(4), 0.6).shape == (4,), 'shape change must not break'
    # gap_clip + low confidence + stalled goal -> re-route; a healthy goal is left alone; flag off -> nothing
    _flag = getattr(Config, 'ENABLE_GAPCLIP_LOWCONF_ACTION', False)
    try:
        Config.ENABLE_GAPCLIP_LOWCONF_ACTION = True
        q = _Nav(); q.world, q.age = _types.SimpleNamespace(), 300
        assert q._nav_spatial_goal('new') and q._nav_progress['dir'] == (1, 0)
        q._dir_conf = 0.3
        q._nav_progress['stuck_ticks'] = int(Config.NAV_PERC_STALL_TICKS) - 1
        q._nav_gapclip_decision(0.9)
        assert q._nav_progress['dir'] == (1, 0) and q._nav_progress['recalc'] == 0, 'not stalled yet -> no action'
        q._nav_progress['stuck_ticks'] = int(Config.NAV_PERC_STALL_TICKS)
        Config.ENABLE_GAPCLIP_LOWCONF_ACTION = False
        q._nav_gapclip_decision(0.9)
        assert q._nav_progress['dir'] == (1, 0), 'flag off -> no action'
        Config.ENABLE_GAPCLIP_LOWCONF_ACTION = True
        q._nav_gapclip_decision(0.9)
        assert q._nav_progress['dir'] == (0, 1) and q._nav_progress['recalc'] == 1 and (1, 0) in q._nav_block_dirs
        assert _nav_audit(q.world)['c']['gapclip_action_lowconf'] == 1
        q._dir_scores = {d: float('-inf') for d in _NAV_DIRS}          # nowhere else to go
        q._nav_progress['stuck_ticks'] = int(Config.NAV_PERC_STALL_TICKS)
        q._nav_gapclip_decision(0.9)
        assert q._nav_progress is not None and q._nav_progress['dir'] == (0, 1), 'no alternative -> episode untouched'
    finally:
        Config.ENABLE_GAPCLIP_LOWCONF_ACTION = _flag
    # v42: one _nav_step per tick; a second call in the same tick must not add a phantom stuck tick
    _fl = getattr(Config, 'NAV_STEP_ONCE_PER_TICK', True)
    try:
        for flag, want in ((True, 1), (False, 2)):
            Config.NAV_STEP_ONCE_PER_TICK = flag
            g1 = _Nav(); g1.world = _types.SimpleNamespace()
            assert g1._nav_spatial_goal('new') and g1._nav_progress['origin'] == 'new'
            # каждый вызов — после попытки move(); первый шаг с попыткой только фиксирует best_dist (stuck=0)
            g1.age += 1
            g1._nav_move_attempt_pending = g1._nav_progress
            g1._nav_step(f)
            g1.age += 1
            g1._nav_move_attempt_pending = g1._nav_progress
            g1._nav_step(f)                                          # stuck=1
            g1._nav_move_attempt_pending = g1._nav_progress
            g1._nav_step(f)                                          # дубль в том же тике: ONCE_PER_TICK → +0, иначе +1
            assert g1._nav_progress['stuck_ticks'] == want, (flag, g1._nav_progress['stuck_ticks'])
            assert _nav_audit(g1.world)['c']['step_dup_skipped'] == (1 if flag else 0)
    finally:
        Config.NAV_STEP_ONCE_PER_TICK = _fl
    # origin survives recalculation; gap_clip-started episodes are labelled as such
    g2 = _Nav(); g2.world = _types.SimpleNamespace(); g2.age = 400
    assert g2._nav_spatial_goal('gapclip_explore') and g2._nav_progress['origin'] == 'gapclip_explore'
    old_nav = dict(g2._nav_progress)
    assert g2._nav_spatial_goal('stuck', exclude=old_nav['dir'], carry=old_nav) and g2._nav_progress['origin'] == 'gapclip_explore'
    # passive gap_clip statistics: histogram bin, goal/no-goal split, confidence contingency
    g3 = _Nav(); g3.world = _types.SimpleNamespace(); g3._dir_conf = 0.3
    g3._nav_gap_stats(0.9); g3._nav_gap_stats(1.0)
    c3 = _nav_audit(g3.world)['c']
    assert c3['gap_ticks'] == 2 and c3['gap_h18_nogoal'] == 1 and c3['gap_h19_nogoal'] == 1 and c3['gx_hi_lowconf'] == 2
    g3._nav_gap_stats(0.2)
    assert c3['gx_lo_lowconf'] == 1 and c3['gap_h04_nogoal'] == 1
    return {'ok': True, 'ray_points_per_agent': len(_NAV_DIRS) * len(dists)}


# ═════════════ [G] observer: что толкает gap_clip (read-only; только факты из agent_snaps лога) ═════════════
# Ничего не пересчитывает в симуляции и не пишет в её состояние. Источник: поля err_dim / avg_real_dim /
# avg_real_raw_dim / pred_dim / clip_dim, которые update_model_part1 кладёт в агента, а RichLog — в agent_snaps.
_OG_CH = ('energy', 'flux', 'scar', 'noise', 'vorticity', 'unknown', 'binding', 'trust')
_OG_SHORT = ('en', 'fx', 'sc', 'nz', 'vo', 'un', 'bd', 'tr')
_OG_WARMUP = 30       # тиков жизни до начала учёта (EMA err_dim должен устояться)
_OG_MAX_STALE = 10    # err_dim старше этого (агент спал/во сне) не учитывается


def _og_corr(a, b):
    import numpy as np
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    m = np.isfinite(a) & np.isfinite(b)
    if int(m.sum()) < 10 or np.std(a[m]) < 1e-12 or np.std(b[m]) < 1e-12:
        return float('nan')
    return float(np.corrcoef(a[m], b[m])[0, 1])


def _og_vec(snaps, key):
    """(n, 8) массив по ключу; строки без 8 чисел -> NaN."""
    import numpy as np
    out = np.full((len(snaps), 8), np.nan)
    for i, s in enumerate(snaps):
        v = s.get(key)
        if isinstance(v, (list, tuple)) and len(v) == 8:
            try:
                out[i] = [float(x) for x in v]
            except (TypeError, ValueError):
                pass
    return out


def _og_col(snaps, key):
    import numpy as np
    out = np.full(len(snaps), np.nan)
    for i, s in enumerate(snaps):
        try:
            v = s.get(key)
            if v is not None:
                out[i] = float(v)
        except (TypeError, ValueError):
            pass
    return out


def _og_shares(E):
    import numpy as np
    m = np.nanmean(E, axis=0)
    tot = float(m.sum())
    return (100.0 * m / tot) if tot > 1e-12 else np.zeros(8)


def _gap_clip_report(log_path, title='', dump_path=None):
    import numpy as np
    print("=" * 78)
    print(f"[GAP] ЧТО ТОЛКАЕТ gap_clip {title}(observer: read-only, только факты из agent_snaps; вердикт — не его работа)")
    print("=" * 78)
    if not log_path or not os.path.exists(log_path):
        print(f"  [!] нет лога: {log_path}")
        return None
    with open(log_path, 'r', encoding='utf-8') as f:
        log = json.load(f)
    allsn = log.get('agent_snaps') or []
    if not allsn:
        print("  [!] в логе нет agent_snaps")
        return None
    if not any(isinstance(s.get('err_dim'), list) for s in allsn):
        print("  [!] в agent_snaps нет err_dim: лог записан версией до fix11k (разложение по каналам не писалось)")
        return None
    normal = [s for s in allsn if s.get('alive', True) and not s.get('feral_flag')]
    feral_n = sum(1 for s in allsn if s.get('alive', True) and s.get('feral_flag'))
    E0 = _og_vec(normal, 'err_dim')
    ok = np.isfinite(E0).all(axis=1) & (np.nan_to_num(E0).sum(axis=1) > 0)
    ok &= _og_col(normal, 'age') >= _OG_WARMUP
    stale = _og_col(normal, 'err_dim_stale')
    ok &= np.nan_to_num(stale, nan=0.0) <= _OG_MAX_STALE
    use = [s for s, k in zip(normal, ok) if k]
    n = len(use)
    print(f"  снимков всего={len(allsn)}; обычных живых={len(normal)} (ферал не учтены: {feral_n}); "
          f"в учёте={n} (возраст ≥{_OG_WARMUP}, err_dim свежий ≤{_OG_MAX_STALE} тиков); агентов={len({s.get('id') for s in use})}")
    if n < 10:
        print("  [!] слишком мало снимков для статистики")
        return None
    E = E0[ok]
    R, P = _og_vec(use, 'avg_real_dim'), _og_vec(use, 'pred_dim')
    RR, CL = _og_vec(use, 'avg_real_raw_dim'), _og_vec(use, 'clip_dim')
    gap, og = _og_col(use, 'gap'), _og_col(use, 'gap_clip')
    ncell, tt = _og_col(use, 'n_cells'), _og_col(use, 't')
    mean_e = np.nanmean(E, axis=0)
    share = _og_shares(E)
    has_raw = bool(np.isfinite(RR).any())
    has_clip = bool(np.isfinite(CL).any())

    print("\n  ── [G1] РАЗЛОЖЕНИЕ ОШИБКИ ПО 8 КАНАЛАМ: err = |avg_real − prediction|, пул по снимкам ──")
    print("  (raw_gap = среднее по 8 каналам; ошибка считается по avg_real ПОСЛЕ клипа ±1)")
    print(f"  {'канал':10s} {'err':>7s} {'доля%':>6s} {'p90 err':>8s} {'real':>8s} {'pred':>8s} {'pred−real':>10s} {'в клипе%':>9s} {'max|real| до клипа':>19s}")
    for i, nm in enumerate(_OG_CH):
        p90 = float(np.nanpercentile(E[:, i], 90))
        cl = f"{100.0 * float(np.nanmean(CL[:, i])):9.1f}" if has_clip else f"{'-':>9s}"
        mx = f"{float(np.nanmax(np.abs(RR[:, i]))):19.3f}" if has_raw else f"{'-':>19s}"
        print(f"  {nm:10s} {mean_e[i]:7.4f} {share[i]:6.1f} {p90:8.4f} {np.nanmean(R[:, i]):+8.4f} {np.nanmean(P[:, i]):+8.4f} "
              f"{np.nanmean(P[:, i] - R[:, i]):+10.4f} {cl} {mx}")
    top = np.argsort(-share)[:3]
    print(f"  среднее err по каналам (≈ сырой gap) = {mean_e.mean():.4f}; spirit_gap (gap) = {np.nanmean(gap):.4f}; "
          f"gap_clip = {np.nanmean(og):.4f}")
    print("  три самых больших вклада: " + "; ".join(f"{_OG_CH[i]} {share[i]:.1f}%" for i in top))

    print("\n  ── [G2] ДИНАМИКА: доля каналов (%) по окнам 100 тиков ──")
    print(f"  {'окно':>10s} {'снимков':>8s} {'gap_clip':>8s}   " + " ".join(f"{s:>4s}" for s in _OG_SHORT))
    w = np.floor(tt / 100.0)
    for k in sorted({int(x) for x in w[np.isfinite(w)]}):
        sel = np.isfinite(w) & (w == k)
        if int(sel.sum()) < 5:
            continue
        sh = _og_shares(E[sel])
        print(f"  {k * 100:>4d}-{k * 100 + 99:<5d} {int(sel.sum()):8d} {np.nanmean(og[sel]):8.3f}   " + " ".join(f"{x:4.0f}" for x in sh))

    groups = []
    if np.isfinite(ncell).any():
        edges = np.unique(np.nanquantile(ncell, [0.0, 0.25, 0.5, 0.75, 1.0]))
        print("\n  ── [G3] РАЗРЕЗ ПО РАЗМЕРУ ТЕЛА (n_cells, квартили): доля каналов (%) ──")
        print(f"  {'клеток':>10s} {'снимков':>8s} {'gap_clip':>8s}   " + " ".join(f"{s:>4s}" for s in _OG_SHORT))
        idx = np.digitize(np.nan_to_num(ncell, nan=edges[0]), edges[1:-1]) if len(edges) > 2 else np.zeros(len(ncell), dtype=int)
        for gi in sorted(set(idx.tolist())):
            sel = (idx == gi) & np.isfinite(ncell)
            if int(sel.sum()) < 5:
                continue
            sh = _og_shares(E[sel])
            lo, hi = int(np.nanmin(ncell[sel])), int(np.nanmax(ncell[sel]))
            groups.append(dict(n_cells=[lo, hi], n=int(sel.sum()), gap_clip=float(np.nanmean(og[sel])), shares=[round(float(x), 2) for x in sh]))
            print(f"  {lo:>4d}-{hi:<5d} {int(sel.sum()):8d} {np.nanmean(og[sel]):8.3f}   " + " ".join(f"{x:4.0f}" for x in sh))

    print("\n  ── [G4] КОРРЕЛЯЦИИ (Пирсон по снимкам; снимки одного агента зависимы, это ориентир, не доказательство) ──")
    cors = {}
    for i, nm in enumerate(_OG_CH):
        cors['gap_clip~err_' + nm] = _og_corr(og, E[:, i])
    print("  gap_clip ~ err по каналам: " + "  ".join(f"{_OG_SHORT[i]}={cors['gap_clip~err_' + nm]:+.2f}" for i, nm in enumerate(_OG_CH)))
    extra = (('n_cells', ncell), ('local_crisis', _og_col(use, 'local_crisis')), ('bind_loc', _og_col(use, 'bind_loc')),
             ('dir_conf', _og_col(use, 'dir_conf')), ('soul', _og_col(use, 'soul')))
    for nm, v in extra:
        cors['gap_clip~' + nm] = _og_corr(og, v)
    print("  gap_clip ~ прочее:         " + "  ".join(f"{nm}={cors['gap_clip~' + nm]:+.2f}" for nm, _ in extra))
    cors['err_vorticity~n_cells'] = _og_corr(E[:, 4], ncell)
    cors['err_scar~local_crisis'] = _og_corr(E[:, 2], _og_col(use, 'local_crisis'))
    cors['err_noise~n_cells'] = _og_corr(E[:, 3], ncell)
    print(f"  err_vorticity ~ n_cells={cors['err_vorticity~n_cells']:+.2f}  err_scar ~ local_crisis={cors['err_scar~local_crisis']:+.2f}  "
          f"err_noise ~ n_cells={cors['err_noise~n_cells']:+.2f}")

    thr = float(getattr(Config, 'GAP_CLIP_CURIOSITY_THRESHOLD', 0.85)) if 'Config' in globals() else 0.85
    q = np.nanpercentile(og, [50, 90, 99])
    print("\n  ── [G5] РАСПРЕДЕЛЕНИЕ gap_clip ──")
    print(f"  p50={q[0]:.3f} p90={q[1]:.3f} p99={q[2]:.3f}; доля снимков выше порога {thr:.2f} = {100.0 * float(np.nanmean(og > thr)):.1f}%")

    print("\n  ── [G6] КОНТЕКСТ КОНФИГА (текущий конфиг СЕССИИ на момент отчёта, не обязательно конфиг времени прогона) ──")
    if 'Config' in globals():
        keys = ('GRADIENT_SENSITIVITY', 'VORTICITY_DECAY', 'VORTICITY_COUPLING', 'HEARTBEAT_AMPLITUDE', 'HEARTBEAT_FREQ',
                'NOISE_GAIN_HIGH_THRESH', 'NOISE_GAIN_LOW_THRESH', 'BASE_NOISE', 'WORLD_SIZE')
        print("  " + " ".join(f"{k}={getattr(Config, k, None)}" for k in keys))
    print("=" * 78, flush=True)

    res = dict(log=str(log_path), snaps_used=n, agents=len({s.get('id') for s in use}), channels=list(_OG_CH),
               mean_err=[round(float(x), 5) for x in mean_e], share_pct=[round(float(x), 2) for x in share],
               clip_pct=[round(100.0 * float(np.nanmean(CL[:, i])), 2) for i in range(8)] if has_clip else None,
               max_abs_real_raw=[round(float(np.nanmax(np.abs(RR[:, i]))), 4) for i in range(8)] if has_raw else None,
               gap_mean=float(np.nanmean(gap)), gap_clip_mean=float(np.nanmean(og)), by_body=groups,
               corr={k: (None if not np.isfinite(v) else round(float(v), 4)) for k, v in cors.items()})
    if dump_path:
        try:
            with open(dump_path, 'w', encoding='utf-8') as f:
                json.dump(res, f, ensure_ascii=False, indent=1)
            print(f"[GAP] итоги записаны: {dump_path}")
        except Exception as e:
            print(f"[GAP] дамп не записан: {e!r}")
    return res


# ═════════════════════════════════════════════════════════════════════════════════════════════════════════
# [OBS v2] СВОДКА · СЛОВАРЬ · ШАПКА · ИНВАРИАНТЫ · РАЗЛОЖЕНИЕ gap · СМЕЩЕНИЕ vs СЮРПРИЗ · НАВИГАЦИЯ
# Observer, READ-ONLY: читает только JSON лога/ledger/audit. Симуляцию не запускает и не меняет.
# Статусы — факты о данных, не решение Gate:  ✅ норма/совпало · ⚠️ отклонение · ❌ нарушено · ❔ UNKNOWN (нет данных; не PASS)
# ═════════════════════════════════════════════════════════════════════════════════════════════════════════
_OV_SCHEMA = 2
_OV_SAT = 0.99   # gap_clip ≥ этого = «упёрся в потолок клипа» (факт, не вердикт)


def _ov_f(x, nd=3, dash='—'):
    try:
        x = float(x)
    except (TypeError, ValueError):
        return dash
    return dash if x != x else f"{x:.{nd}f}"


def _ov_pc(x, nd=1, dash='—'):
    try:
        x = float(x)
    except (TypeError, ValueError):
        return dash
    return dash if x != x else f"{100.0 * x:.{nd}f}%"


def _ov_find(d, key, depth=0):
    """Первое значение по ключу в произвольно вложенном dict (для audit JSON, структура которого менялась)."""
    if depth > 6:
        return None
    if isinstance(d, dict):
        if key in d:
            return d[key]
        for v in d.values():
            r = _ov_find(v, key, depth + 1)
            if r is not None:
                return r
    elif isinstance(d, list):
        for v in d[:50]:
            r = _ov_find(v, key, depth + 1)
            if r is not None:
                return r
    return None


def _ov_select(allsn):
    normal = [s for s in allsn if s.get('alive', True) and not s.get('feral_flag')]
    feral_n = sum(1 for s in allsn if s.get('alive', True) and s.get('feral_flag'))
    use = [s for s in normal
           if isinstance(s.get('err_dim'), list) and len(s['err_dim']) == 8
           and int(s.get('age', 0) or 0) >= _OG_WARMUP and int(s.get('err_dim_stale', 0) or 0) <= _OG_MAX_STALE]
    return use, len(normal), feral_n


def _ov_cfg(name, default=None):
    return getattr(Config, name, default) if 'Config' in globals() else default


def _ov_tickwall_stats(tw):
    """tick_wall: строки [t, sim_s, log_s, pop, ferals] по КАЖДОМУ тику. Только арифметика по готовому логу."""
    import numpy as np
    rows = [r for r in (tw or []) if isinstance(r, (list, tuple)) and len(r) >= 5]
    if len(rows) < 5:
        return None
    A = np.array([[float(x) for x in r[:5]] for r in rows])
    t, sim, lg, n = A[:, 0], A[:, 1], A[:, 2], A[:, 3]
    tot = sim + lg
    out = {'ticks': len(A), 'total_s': float(tot.sum()), 'sim_s': float(sim.sum()), 'log_s': float(lg.sum()),
           'log_share': float(lg.sum() / max(tot.sum(), 1e-12)), 'max_tick_s': float(tot.max()), 'max_tick_t': int(t[int(tot.argmax())]),
           'pop_max': int(n.max()), 'pop_max_t': int(t[int(n.argmax())]), 'windows': []}
    for k in range(int(t.min() // 50), int(t.max() // 50) + 1):
        m = (t // 50 == k)
        if int(m.sum()) >= 3:
            out['windows'].append({'t0': k * 50, 'ticks': int(m.sum()), 'pop_mean': float(n[m].mean()), 'pop_max': int(n[m].max()),
                                   'sim_per_tick': float(sim[m].mean()), 'log_per_tick': float(lg[m].mean()),
                                   'sim_per_tick_per_1000': float(sim[m].mean() / max(n[m].mean(), 1.0) * 1000.0)})
    m = (n >= 100) & (sim > 0)
    if int(m.sum()) >= 10 and float(np.ptp(np.log(n[m]))) > 0.3:
        x, y = np.log(n[m]), np.log(sim[m])
        b, a = np.polyfit(x, y, 1)
        ss_res = float(((y - (a + b * x)) ** 2).sum())
        ss_tot = float(((y - y.mean()) ** 2).sum())
        out['exponent'] = float(b)
        out['exponent_r2'] = float(1.0 - ss_res / max(ss_tot, 1e-12))
        out['exponent_n'] = int(m.sum())
    return out


def _ov_compute(log, led, audit, log_path=None, led_path=None):
    import numpy as np
    R = {'schema': _OV_SCHEMA, 'errors': []}
    hdr = log.get('header') or {}
    lh = (led or {}).get('header') or {}
    snaps = log.get('agent_snaps') or []
    rows = log.get('tick_rows') or []
    steps = hdr.get('steps')
    every = int(globals().get('SNAPSHOT_EVERY', 10))
    exp = list(range(0, int(steps), every)) if isinstance(steps, int) else None
    got = [r.get('t') for r in rows]
    st = (hdr.get('selftests') or {})
    R['header'] = {
        'log': str(log_path), 'ledger': str(led_path), 'seed': hdr.get('seed'), 'steps': steps, 'mult': hdr.get('mult'),
        'complete': hdr.get('complete'), 'wall_seconds': hdr.get('wall_seconds'),
        'morse_on': hdr.get('enable_morse_communication'), 'obs_schema': hdr.get('obs_schema'),
        'patch_id': (hdr.get('patches') or {}).get('id'), 'code_hash': hdr.get('code_hash'),
        'code_hash_n': hdr.get('code_hash_n'), 'config_hash_sha': hdr.get('config_hash_sha'),
        'ledger_config_hash': lh.get('config_hash'), 'given_seed': lh.get('given_seed'),
        'input_fp_before': lh.get('input_fp_before'), 'selftests': st,
        'input_fp_stable': (lh.get('input_fp_stable') if 'output_fp_after' in lh else None),
        'run_mutated_files': lh.get('run_mutated_files'),
        'ticks_got': len(got), 'ticks_expected': (len(exp) if exp is not None else None),
        'ticks_missing': (len(set(exp) - set(got)) if exp is not None else None),
        'python': hdr.get('python'), 'numpy': hdr.get('numpy'),
        'pop_fill_soft': hdr.get('pop_fill_soft'), 'pop_fill_hard': hdr.get('pop_fill_hard'),
    }
    try:
        R['tickwall'] = _ov_tickwall_stats(log.get('tick_wall'))
    except Exception as e:
        R['tickwall'] = None
        R['errors'].append(f'tick_wall: {e!r}')
    use, n_normal, n_feral = _ov_select(snaps)
    R['snaps'] = {'total': len(snaps), 'normal_alive': n_normal, 'feral': n_feral, 'used': len(use)}
    thr = float(_ov_cfg('GAP_CLIP_CURIOSITY_THRESHOLD', 0.85))
    lo, hi = _ov_cfg('SLACK_CUR_LOW'), _ov_cfg('SLACK_CUR_HIGH')
    R['thr'] = thr
    if len(use) < 10:
        R['errors'].append('слишком мало снимков для статистики')
        return R
    og = _og_col(use, 'gap_clip')
    sg = _og_col(use, 'gap_unclipped')
    if not np.isfinite(sg).any():
        sg = _og_col(use, 'gap')
        R['gap_unclipped_source'] = 'gap (лог до obs-v2)'
    else:
        R['gap_unclipped_source'] = 'gap_unclipped'
    q = lambda a: [float(x) for x in np.nanpercentile(a, [50, 90, 99])]
    R['gap_clip'] = {'mean': float(np.nanmean(og)), 'p50_p90_p99': q(og), 'above_thr': float(np.nanmean(og > thr)),
                    'at_ceiling': float(np.nanmean(og >= _OV_SAT)), 'thr': thr}
    R['spirit_gap'] = {'mean': float(np.nanmean(sg)), 'p50_p90_p99': q(sg), 'max': float(np.nanmax(sg)),
                       'above_1': float(np.nanmean(sg > 1.0)), 'clip_loss_mean': float(np.nanmean(np.maximum(sg - 1.0, 0.0)))}
    # различающая способность: разброс gap_clip ВНУТРИ тика (если ~0 — по нему нечего ранжировать)
    by_t = {}
    for s, v in zip(use, og):
        if np.isfinite(v):
            by_t.setdefault(int(s.get('t', 0)), []).append(float(v))
    iqr, sd = [], []
    for t_, v in by_t.items():
        if len(v) >= 5:
            iqr.append(float(np.subtract(*np.percentile(v, [75, 25]))))
            sd.append(float(np.std(v)))
    R['spread_in_tick'] = {'ticks': len(iqr), 'iqr_mean': (float(np.mean(iqr)) if iqr else None),
                           'std_mean': (float(np.mean(sd)) if sd else None)}
    # навигация: куда попадают агенты по порогам и что они выбирают
    ch = {}
    for s in use:
        k = s.get('gapclip_choice')
        ch[str(k)] = ch.get(str(k), 0) + 1
    band = None
    if lo is not None and hi is not None:
        lo, hi = float(lo), float(hi)
        band = {'low': lo, 'high': hi, 'below_low': float(np.nanmean(og < lo)),
                'dead_zone': float(np.nanmean((og >= lo) & (og <= hi))), 'above_high': float(np.nanmean(og > hi))}
    R['nav'] = {'choice': ch, 'band': band, 'action_on': bool(_ov_cfg('ENABLE_GAPCLIP_ACTION', False)),
                'lowconf_on': bool(_ov_cfg('ENABLE_GAPCLIP_LOWCONF_ACTION', False))}
    # ── разложение spirit_gap (нужны поля obs-v2) ──
    GP = np.full((len(use), 4), np.nan)
    gage = np.full(len(use), np.nan)
    ext = np.full(len(use), np.nan)
    for i, s in enumerate(use):
        gp = s.get('gap_parts')
        if isinstance(gp, list) and len(gp) == 4:
            try:
                GP[i] = [float(x) for x in gp]
                gage[i] = float(s.get('gap_parts_age', 0))
                ext[i] = float(s.get('gap_ext', np.nan))
            except (TypeError, ValueError):
                pass
    m = np.isfinite(GP).all(axis=1) & np.isfinite(ext) & (np.nan_to_num(gage, nan=1e9) <= _OG_MAX_STALE)
    if int(m.sum()) >= 10:
        raw, base, reen, setend = GP[m, 0], GP[m, 1], GP[m, 2], GP[m, 3]
        damp = setend - base - reen
        now = setend + ext[m]
        tot = float(np.mean(now)) or 1e-12
        R['decomp'] = {'n': int(m.sum()), 'raw_gap_tick': float(np.mean(raw)), 'baseline_ema': float(np.mean(base)),
                       'reentry': float(np.mean(reen)), 'damping': float(np.mean(damp)), 'external': float(np.mean(ext[m])),
                       'spirit_gap': float(np.mean(now)),
                       'share': {'baseline_ema': float(np.mean(base)) / tot, 'reentry': float(np.mean(reen)) / tot,
                                 'damping': float(np.mean(damp)) / tot, 'external': float(np.mean(ext[m])) / tot},
                       'external_p90': float(np.percentile(ext[m], 90)), 'external_max': float(np.max(ext[m]))}
    else:
        R['decomp'] = None
    # ── смещение vs сюрприз, baseline-навык ──
    E, EP, EE, B = (_og_vec(use, k) for k in ('err_dim', 'err_persist_dim', 'err_ema_dim', 'bias_dim'))
    Rr, Pp = _og_vec(use, 'avg_real_dim'), _og_vec(use, 'pred_dim')
    mm = np.isfinite(E).all(axis=1) & np.isfinite(EP).all(axis=1) & np.isfinite(EE).all(axis=1) & np.isfinite(B).all(axis=1)
    if int(mm.sum()) >= 10:
        err, sur, eem = E[mm].mean(0), EP[mm].mean(0), EE[mm].mean(0)
        absb, sb = np.abs(B[mm]).mean(0), B[mm].mean(0)
        sk_p = np.where(sur > 1e-6, 1.0 - err / np.maximum(sur, 1e-12), np.nan)
        sk_e = np.where(eem > 1e-6, 1.0 - err / np.maximum(eem, 1e-12), np.nan)
        worse_p = (err > sur + 1e-6)
        worse_e = (err > eem + 1e-6)
        R['chan'] = {'n': int(mm.sum()), 'names': list(_OG_CH), 'err': err.tolist(), 'surprise': sur.tolist(),
                     'err_vs_ema_pred': eem.tolist(), 'abs_bias': absb.tolist(), 'bias_signed': sb.tolist(),
                     'skill_vs_persist': [None if x != x else float(x) for x in sk_p],
                     'skill_vs_ema': [None if x != x else float(x) for x in sk_e],
                     'worse_than_persist': int(worse_p.sum()), 'worse_than_ema': int(worse_e.sum()),
                     'bias_share_of_err': float(absb.sum() / max(err.sum(), 1e-12)),
                     'mean_err': float(err.mean()), 'mean_surprise': float(sur.mean()), 'mean_abs_bias': float(absb.mean())}
        og_m = og[mm]
        R['chan']['corr_gap_clip_surprise'] = _og_corr(og_m, EP[mm].mean(1))
        R['chan']['corr_gap_clip_abs_bias'] = _og_corr(og_m, np.abs(B[mm]).mean(1))
    else:
        R['chan'] = None
    # ── инварианты ──
    with_pl = [(s.get('id'), s.get('place_id'), s.get('place_visits', 0)) for s in use if s.get('place_id') is not None]
    R['place'] = {'snaps': len(use), 'with_place': len(with_pl), 'place_none': len(use) - len(with_pl),
                  'id0': sum(1 for _a, p_, _v in with_pl if p_ == 0),
                  'visits0': sum(1 for _a, _p, v in with_pl if not v),
                  'distinct_agent_place': len({(a_, p_) for a_, p_, _v in with_pl}),
                  'agents_with_place': len({a_ for a_, _p, _v in with_pl})}
    x2 = _ov_find(audit, 'x2_guard') if audit else None
    R['energy'] = {'x2_guard': x2, 'kill_negative_feral_energy': (_ov_find(audit, 'negative_feral_energy') if audit else None)}
    R['log_errors'] = dict(log.get('errors') or {})
    R['obs_fails'] = dict(hdr.get('obs_fails') or {})
    if led:
        lr = led.get('rows') or []
        R['ledger'] = {'rows': len(lr), 'chain_present': bool(lr) and all('chain' in r for r in lr[:3]),
                       'chain_last': (lr[-1].get('chain') if lr else None)}
    else:
        R['ledger'] = None
    try:   # единый учёт населения: log.n == ledger.n на каждом тике + баланс рождений/смертей
        _lr = (led or {}).get('rows') or []
        _ps = {'violations_n': hdr.get('pop_source_violations_n'), 'violations': (hdr.get('pop_source_violations') or [])[:5],
               'ledger_semantics': ('alive' if _lr and 'n_pending_dead' in _lr[-1] else ('legacy' if _lr else None)), 'balance': None}
        if _lr:
            _b0 = int(_lr[0].get('n', 0)) - int(_lr[0].get('births', 0)) + int(_lr[0].get('deaths', 0))
            _ex = _b0 + sum(int(r.get('births', 0)) for r in _lr) - sum(int(r.get('deaths', 0)) for r in _lr)
            _ps['balance'] = {'n_pre_run': _b0, 'expected_end': _ex, 'n_end': _lr[-1].get('n'), 'ok': bool(_ex == _lr[-1].get('n'))}
        R['pop_source'] = _ps
    except Exception as _e_ps:
        R['pop_source'] = {'error': repr(_e_ps)[:120]}
    return R


def _ov_mark(ok):
    return '✅' if ok is True else ('❔' if ok is None else '⚠️')


def _ov_print_glossary(R):
    thr = R.get('thr', 0.85)
    print("\n" + "─" * 78)
    print("[D] СЛОВАРЬ: ЧТО ЕСТЬ ЧТО (чтобы не гадать)")
    print("─" * 78)
    print("  spirit_gap  поле АГЕНТА, не замер. Пишут: update_model (EMA ошибки предсказания + reentry), reentry-bump,")
    print("              nudge, SIG_GAP_BUMP, спавн/сон. Читает: is_triadic_alive (TA) и gap_clip.")
    print(f"  gap_clip     НЕ замер наблюдателя: это clip(spirit_gap,0,1), его пишет агент. УПРАВЛЯЕТ ПОВЕДЕНИЕМ: при > {thr}")
    print("              вызывается _nav_gapclip_decision (при ENABLE_GAPCLIP_ACTION — explore). Поэтому его насыщение = поведение.")
    print("  raw_gap     mean|real−pred| по 8 каналам за тик (сырая ошибка предсказания, до EMA).")
    print("  err_dim     EMA(0.9/0.1) |real−pred| по каналу; real берётся ПОСЛЕ клипа ±1.")
    print("  surprise    |real_t − real_{t−1}|: насколько мир сам изменился = ошибка прогноза «будет как раньше».")
    print("  bias        pred − real (EMA, со знаком): постоянный сдвиг предиктора. Это не непредсказуемость мира.")
    print("  skill       1 − err_модели / err_простого_прогноза. < 0 → модель хуже тривиальной (прошлое или EMA).")
    print("  внешние     всё, что писали в spirit_gap ПОСЛЕ update_model (SIG_GAP_BUMP, nudge, reentry-bump).")
    print("  obs_*       счётчики, которые пишет ТОЛЬКО observer (read-only). Отдельно от gap_clip выше.")


def _ov_print_header(R):
    H = R['header']
    print("\n" + "─" * 78)
    print("[H] ШАПКА ПРОГОНА (откуда эти цифры)")
    print("─" * 78)
    mo = H.get('morse_on')
    print(f"  seed={H.get('seed')} steps={H.get('steps')} mult={H.get('mult') if H.get('mult') is not None else '❔'} complete={H.get('complete')} "
          f"wall={(str(H.get('wall_seconds')) + 'с') if H.get('wall_seconds') is not None else '❔'} Морзе={'ON' if mo is True else ('OFF' if mo is False else '❔ нет флага в логе')}")
    print(f"  patch_id={H.get('patch_id') or '❔'}  code_hash={H.get('code_hash') or '❔ нет'}"
          f"{'' if H.get('code_hash_n') is None else ' (' + str(H.get('code_hash_n')) + ' функций)'}  "
          f"config_sha={H.get('config_hash_sha') or '❔ нет'}  ledger.config_hash={H.get('ledger_config_hash')}")
    print(f"  given_seed(ledger)={H.get('given_seed')}  input_fp_before={'❔ нет' if H.get('input_fp_before') is None else ('есть' if H.get('input_fp_before') else 'нет входных файлов (свежий старт)')}  "
          f"obs_schema={H.get('obs_schema') if H.get('obs_schema') is not None else '❔ нет (лог до obs-v2: разложение gap недоступно)'}  "
          f"python={H.get('python') or '❔'} numpy={H.get('numpy') or '❔'}")
    print(f"  входы после прогона (восстановлены guard'ом): input_fp_stable="
          f"{H.get('input_fp_stable') if H.get('input_fp_stable') is not None else '❔ (ledger старой схемы: before/after не сравнивались)'}; "
          f"файлы, которые менял сам прогон (это ВЫХОД, не вход): {H.get('run_mutated_files') if H.get('run_mutated_files') is not None else '❔'}")
    print(f"  лимит численности (доля занятых клеток мира): POP_FILL_SOFT={H.get('pop_fill_soft') if H.get('pop_fill_soft') is not None else '❔'} "
          f"POP_FILL_HARD={H.get('pop_fill_hard') if H.get('pop_fill_hard') is not None else '❔'}")
    tm = H.get('ticks_missing')
    print(f"  снимков тиков: {H.get('ticks_got')} из ожидаемых {H.get('ticks_expected')}; пропущено: {tm if tm is not None else '❔'}")
    sn = R.get('snaps', {})
    print(f"  снимков агентов: всего={sn.get('total')} обычных живых={sn.get('normal_alive')} (ферал: {sn.get('feral')}) "
          f"в учёте={sn.get('used')} (возраст ≥{_OG_WARMUP}, err_dim свежий ≤{_OG_MAX_STALE})")


def _ov_print_invariants(R):
    print("\n" + "─" * 78)
    print("[V] ИНВАРИАНТЫ (факты о данных; вердикт выносит Gate, не Observer)")
    print("─" * 78)
    st = R['header'].get('selftests') or {}
    if not st:
        print("  ❔ selftests: в шапке лога нет результатов (лог до obs-v2 или тесты не регистрировались) — это UNKNOWN, не PASS")
    for nm, v in st.items():
        if isinstance(v, dict):
            ok = v.get('ok')
            tail = f" {v.get('passed')}/{v.get('total')}" if v.get('total') is not None else ''
            bad = f" ПРОВАЛЕНО: {v.get('failed')}" if v.get('failed') else ''
            print(f"  {'✅' if ok is True else ('❔' if ok is None else '❌')} selftest {nm}{tail}{bad}"
                  f"{'' if ok is not None else ' (тест не вернул вердикт)'}")
        else:
            print(f"  ❔ selftest {nm}: {v}")
    en = R.get('energy') or {}
    x2 = en.get('x2_guard')
    if isinstance(x2, dict):
        neg, nf = x2.get('neg_ticks'), x2.get('nonfinite_ticks')
        print(f"  {_ov_mark((neg == 0 and nf == 0) if neg is not None and nf is not None else None)} энергия агентов: "
              f"min={_ov_f(x2.get('min'), 3)} max={_ov_f(x2.get('max'), 3)} тиков с energy<0: {neg} с nan/inf: {nf}"
              f"{'  ← инвариант «energy ≥ 0» нарушен' if (neg or 0) > 0 else ''}")
    else:
        print("  ❔ энергия агентов: нет x2_guard в audit JSON — min/neg_ticks неизвестны")
    if en.get('kill_negative_feral_energy') is not None:
        print(f"  {_ov_mark(en['kill_negative_feral_energy'] == 0)} отрицательная энергия ферола при убийстве: {en['kill_negative_feral_energy']}")
    pl = R.get('place') or {}
    if pl.get('with_place'):
        w = pl['with_place']
        print(f"  {_ov_mark(pl['visits0'] == 0)} место: у {_ov_pc(w / max(1, pl['snaps']))} снимков есть текущее место; "
              f"место БЕЗ визитов: {pl['visits0']} из {w}; уникальных пар (агент, место)={pl['distinct_agent_place']} у {pl['agents_with_place']} агентов")
        print(f"     ℹ️ place_id — счётчик АГЕНТА (place_counter в PlaceNode), id=0 это первое место агента, а не заглушка "
              f"(id=0 у {_ov_pc(pl['id0'] / max(1, w))} снимков). Общее «место 0» по этим данным не подтверждается и не опровергается; "
              f"события SEM смотри в [SEM OBSERVER].")
    else:
        print("  ❔ место: в снимках нет place_id")
    le = R.get('log_errors') or {}
    print(f"  {_ov_mark(not le)} ошибки RichLog (проглоченные исключения логгера): "
          + (f"{sum(le.values())} шт., типы: {dict(list(le.items())[:4])}" if le else "0"))
    ps = R.get('pop_source') or {}
    if ps.get('ledger_semantics') == 'alive':
        vn = int(ps.get('violations_n') or 0)
        print(f"  {_ov_mark(vn == 0)} единый учёт населения: log.n == ledger.n на каждом тике; нарушений: {vn}"
              + (f" (первые: {ps.get('violations')})" if vn else ""))
        pb = ps.get('balance')
        if pb:
            print(f"  {_ov_mark(pb.get('ok'))} баланс ledger: n до прогона {pb['n_pre_run']} + рождений − смертей = {pb['expected_end']}; n в конце {pb['n_end']}")
    elif ps.get('ledger_semantics') == 'legacy':
        print("  ❔ единый учёт населения: ledger старой схемы (n включал мёртвых до вычистки) — log.n и ledger.n сравнивать нельзя")
    of = R.get('obs_fails') or {}
    hdr_has = R['header'].get('obs_schema') is not None
    print(f"  {_ov_mark((not of) if hdr_has else None)} сбои пассивного сбора observer-v2: "
          + (f"{sum(of.values())} шт., {dict(list(of.items())[:4])}" if of else ("0" if hdr_has else "нет данных")))
    tm = R['header'].get('ticks_missing')
    print(f"  {_ov_mark((tm == 0 and R['header'].get('complete') is True) if tm is not None else None)} полнота прогона: "
          f"complete={R['header'].get('complete')}, пропущено снимков тиков={tm if tm is not None else '❔'}")
    L = R.get('ledger')
    if L:
        print(f"  {_ov_mark(L.get('chain_present'))} Ledger: строк={L.get('rows')}, CRC-цепочка {'есть' if L.get('chain_present') else 'НЕТ'}, "
              f"последняя={L.get('chain_last')} (CRC32: ловит расхождение, но не защищает от подмены)")
    else:
        print("  ❔ Ledger: не найден — цепочку и config_snapshot проверить нельзя")
    pv = R['header'].get('log') and (str(R['header']['log']) + '.prev')
    if pv and os.path.exists(pv):
        print("  ⚠️ рядом лежит .prev: при следующем прогоне с тем же TAG_SUFFIX он будет ЗАТЁРТ (хранится одно поколение)")


def _ov_print_tickwall(R):
    print("\n" + "─" * 78)
    print("[T] ВРЕМЯ ТИКОВ (хронометраж каждого тика: где уходят часы)")
    print("─" * 78)
    T = R.get('tickwall')
    if not T:
        print("  ❔ в логе нет tick_wall (прогон до хронометража) — где уходит время, по этому логу не определить")
        return
    hms = lambda s_: f"{int(s_) // 3600}:{int(s_) % 3600 // 60:02d}:{int(s_) % 60:02d}"
    print(f"  всего {hms(T['total_s'])}: симуляция {hms(T['sim_s'])} + логгер {hms(T['log_s'])} ({_ov_pc(T['log_share'])}); "
          f"самый долгий тик t={T['max_tick_t']}: {_ov_f(T['max_tick_s'], 1)} с; пик популяции {T['pop_max']} на t={T['pop_max_t']}")
    print(f"  {'окно':>9s} {'тиков':>6s} {'pop ср.':>8s} {'pop макс':>9s} {'sim с/тик':>10s} {'лог с/тик':>10s} {'sim с/тик на 1000':>18s}")
    for w in T['windows']:
        print(f"  {w['t0']:>4d}-{w['t0'] + 49:<4d} {w['ticks']:6d} {w['pop_mean']:8.0f} {w['pop_max']:9d} {w['sim_per_tick']:10.2f} "
              f"{w['log_per_tick']:10.2f} {w['sim_per_tick_per_1000']:18.2f}")
    if T.get('exponent') is not None:
        print(f"  зависимость времени тика от числа агентов N: sim ~ N^{T['exponent']:.2f} (R²={T['exponent_r2']:.2f}, по {T['exponent_n']} тикам). "
              f"1 — линейно, 2 — квадратично. Наклон в лог-осях: постоянная часть тика его занижает, а рост популяции вместе с тиком смешивается с ним.")
    else:
        print("  показатель степени не посчитан: мало тиков с N≥100 или популяция почти не менялась")


def _ov_print_decomp(R):
    print("\n" + "─" * 78)
    print("[G7] ИЗ ЧЕГО СОСТОИТ spirit_gap И ЧТО ОТРЕЗАЕТ КЛИП (факты по снимкам)")
    print("─" * 78)
    s, o, sp = R.get('spirit_gap'), R.get('gap_clip'), R.get('spread_in_tick')
    if not s:
        print("  ❔ мало данных")
        return
    print(f"  spirit_gap (без клипа, источник: {R.get('gap_unclipped_source')}): среднее={_ov_f(s['mean'])} "
          f"p50/p90/p99={'/'.join(_ov_f(x) for x in s['p50_p90_p99'])} max={_ov_f(s['max'])}; выше 1.0 у {_ov_pc(s['above_1'])} снимков")
    print(f"  gap_clip = clip(spirit_gap,0,1): среднее={_ov_f(o['mean'])} p50/p90/p99={'/'.join(_ov_f(x) for x in o['p50_p90_p99'])}; "
          f"выше порога {o['thr']}: {_ov_pc(o['above_thr'])}; на потолке (≥{_OV_SAT}): {_ov_pc(o['at_ceiling'])}; "
          f"клип срезает в среднем {_ov_f(s['clip_loss_mean'])}")
    if sp and sp.get('iqr_mean') is not None:
        print(f"  разброс gap_clip ВНУТРИ тика (по {sp['ticks']} тикам): IQR={_ov_f(sp['iqr_mean'])} σ={_ov_f(sp['std_mean'])} "
              f"— чем ближе к 0, тем меньше gap_clip различает агентов между собой")
    D = R.get('decomp')
    if not D:
        print("  ❔ РАЗЛОЖЕНИЕ НЕДОСТУПНО: в снимках нет gap_parts (лог записан до obs-v2). Нужен новый прогон.")
        return
    sh = D['share']
    print(f"\n  разложение по {D['n']} снимкам (spirit_gap = EMA сырой ошибки + reentry + демпф + внешние записи):")
    print(f"    EMA сырой ошибки (baseline)  {_ov_f(D['baseline_ema'])}  {_ov_pc(sh['baseline_ema'])}   ← единственная часть, которую объясняет err_dim")
    print(f"    reentry-часть                {_ov_f(D['reentry'])}  {_ov_pc(sh['reentry'])}")
    print(f"    демпф (×0.998/0.999/0.997)   {_ov_f(D['damping'], 4)}  {_ov_pc(sh['damping'])}")
    print(f"    внешние записи после update  {_ov_f(D['external'])}  {_ov_pc(sh['external'])}   (p90={_ov_f(D['external_p90'])}, max={_ov_f(D['external_max'])})")
    print(f"    итого spirit_gap             {_ov_f(D['spirit_gap'])}   (сырая ошибка одного тика: {_ov_f(D['raw_gap_tick'])})")


def _ov_print_chan(R):
    print("\n" + "─" * 78)
    print("[G8] СМЕЩЕНИЕ ПРЕДИКТОРА vs СЮРПРИЗ МИРА + BASELINE-НАВЫК")
    print("─" * 78)
    C = R.get('chan')
    if not C:
        print("  ❔ НЕДОСТУПНО: в снимках нет err_persist_dim/err_ema_dim/bias_dim (лог до obs-v2). Нужен новый прогон.")
        return
    print(f"  по {C['n']} снимкам; skill = 1 − err_модели/err_baseline (<0: модель хуже тривиального прогноза)")
    print(f"  {'канал':10s} {'err':>6s} {'сюрприз':>8s} {'|bias|':>7s} {'bias±':>7s} {'skill/прошлое':>14s} {'skill/EMA':>10s}")
    fm = lambda v: ('base=0' if v is None else f"{v:+.2f}")
    for i, nm in enumerate(C['names']):
        print(f"  {nm:10s} {C['err'][i]:6.3f} {C['surprise'][i]:8.3f} {C['abs_bias'][i]:7.3f} {C['bias_signed'][i]:+7.3f} "
              f"{fm(C['skill_vs_persist'][i]):>14s} {fm(C['skill_vs_ema'][i]):>10s}")
    print(f"  среднее: err={_ov_f(C['mean_err'])} сюрприз={_ov_f(C['mean_surprise'])} |bias|={_ov_f(C['mean_abs_bias'])}; "
          f"доля ошибки, приходящаяся на |bias|: {_ov_pc(C['bias_share_of_err'])}")
    print(f"  каналов, где модель ХУЖЕ «прогноза = прошлое»: {C['worse_than_persist']} из 8; хуже «прогноза = EMA»: {C['worse_than_ema']} из 8 "
          f"(base=0 → мир в канале не меняется, любая ошибка модели хуже)")
    print(f"  корреляция gap_clip с сюрпризом={_ov_f(C.get('corr_gap_clip_surprise'), 2)} и с |bias|={_ov_f(C.get('corr_gap_clip_abs_bias'), 2)}: "
          f"на что реально реагирует gap (и, значит, навигация)")


def _ov_print_nav(R):
    print("\n" + "─" * 78)
    print("[G9] КАК gap_clip ПОПАДАЕТ В ПОВЕДЕНИЕ (пороги и выбор)")
    print("─" * 78)
    N, o = R.get('nav'), R.get('gap_clip')
    if not N or not o:
        print("  ❔ мало данных")
        return
    print(f"  ENABLE_GAPCLIP_ACTION={N['action_on']} ENABLE_GAPCLIP_LOWCONF_ACTION={N['lowconf_on']} "
          f"(текущий конфиг СЕССИИ, не обязательно конфиг прогона); порог explore={o['thr']}")
    b = N.get('band')
    if b:
        print(f"  зоны любопытства (SLACK_CUR_LOW={b['low']}, HIGH={b['high']}): ниже LOW {_ov_pc(b['below_low'])}, "
              f"мёртвая зона {_ov_pc(b['dead_zone'])}, выше HIGH {_ov_pc(b['above_high'])}")
    ch = N.get('choice') or {}
    tot = sum(ch.values()) or 1
    print("  решения _nav_gapclip_decision в снимках: " + "; ".join(f"{k}={v} ({100.0 * v / tot:.0f}%)" for k, v in sorted(ch.items(), key=lambda kv: -kv[1])[:8]))


def _ov_print_summary(R):
    H, o, s, D, C = R['header'], R.get('gap_clip'), R.get('spirit_gap'), R.get('decomp'), R.get('chan')
    print("\n" + "═" * 78)
    print("OBSERVER v2 · СВОДКА ПРОГОНА  (read-only; факты, не решения)")
    print("═" * 78)
    mo = H.get('morse_on')
    print(f"  прогон: seed={H.get('seed')} steps={H.get('steps')} complete={H.get('complete')} "
          f"Морзе={'ON' if mo is True else ('OFF' if mo is False else '❔')} code={H.get('code_hash') or '❔'} patch={H.get('patch_id') or '❔'}")
    if not o:
        print("  ❔ данных для статистики мало: " + "; ".join(R.get('errors') or ['']))
        return
    print(f"  1) gap_clip (УПРАВЛЯЕТ поведением): p50/p90/p99={'/'.join(_ov_f(x) for x in o['p50_p90_p99'])}; "
          f"выше порога {o['thr']}: {_ov_pc(o['above_thr'])}; на потолке клипа: {_ov_pc(o['at_ceiling'])}")
    sp = R.get('spread_in_tick') or {}
    if sp.get('iqr_mean') is not None:
        print(f"     разброс gap_clip внутри тика: IQR={_ov_f(sp['iqr_mean'])} σ={_ov_f(sp['std_mean'])} "
              f"(IQR<0.05 — агенты внутри тика почти неразличимы по gap_clip)")
    if D:
        sh = D['share']
        print(f"  2) spirit_gap={_ov_f(D['spirit_gap'])}: EMA сырой ошибки {_ov_pc(sh['baseline_ema'], 0)} · reentry {_ov_pc(sh['reentry'], 0)} · "
              f"внешние записи {_ov_pc(sh['external'], 0)} · клип срезает {_ov_f(s['clip_loss_mean'])}")
    else:
        print(f"  2) spirit_gap={_ov_f(s['mean'])} (без клипа max={_ov_f(s['max'])}); разложение ❔ нет полей obs-v2 в логе")
    if C:
        print(f"  3) предиктор: на |bias| приходится {_ov_pc(C['bias_share_of_err'], 0)} ошибки; сюрприз мира {_ov_f(C['mean_surprise'])} "
              f"против err {_ov_f(C['mean_err'])}; хуже «прошлого» в {C['worse_than_persist']}/8 каналов, хуже EMA в {C['worse_than_ema']}/8")
        print(f"     gap_clip коррелирует с сюрпризом {_ov_f(C.get('corr_gap_clip_surprise'), 2)} и с |bias| {_ov_f(C.get('corr_gap_clip_abs_bias'), 2)}")
    else:
        print("  3) предиктор: ❔ нет полей obs-v2 в логе (surprise/bias/skill считаются только новым прогоном)")
    st = H.get('selftests') or {}
    red = [k for k, v in st.items() if isinstance(v, dict) and v.get('ok') is False]
    en = (R.get('energy') or {}).get('x2_guard')
    neg = en.get('neg_ticks') if isinstance(en, dict) else None
    pl = R.get('place') or {}
    pv0 = pl.get('visits0') if pl.get('with_place') else None
    print(f"  4) инварианты: selftest красных={len(red) if st else '❔'}{(' ' + str(red)) if red else ''}; "
          f"тиков energy<0={neg if neg is not None else '❔'}; мест без визитов: {pv0 if pv0 is not None else '❔'}; "
          f"ошибок логгера={sum((R.get('log_errors') or {}).values())}; сбоев observer-v2={sum((R.get('obs_fails') or {}).values())}")
    T = R.get('tickwall')
    if T:
        ex = f"; время тика ~ N^{T['exponent']:.2f}" if T.get('exponent') is not None else ''
        print(f"  5) время: {int(T['total_s']) // 60} мин (логгер {_ov_pc(T['log_share'], 0)}); пик pop={T['pop_max']} на t={T['pop_max_t']}; "
              f"самый долгий тик {_ov_f(T['max_tick_s'], 1)} с{ex}")
    else:
        print("  5) время: ❔ нет tick_wall в логе")
    print("  (подробности ниже: [D] словарь · [H] шапка · [V] инварианты · [T] время · [G7–G9] разложение · [G1–G6] по каналам)")


def _ov_chain_report(path_a, path_b, label_a='A', label_b='B', log_a=None):
    """CRC-цепочка двух ledger: до какого тика прогоны совпадали. A/A (один конфиг дважды) ≠ identical → недетерминизм;
    ON/OFF: расхождение ДО первой отправки Морзе = выключение меняет траекторию не только через Морзе."""
    print("\n" + "─" * 78)
    print(f"[C] ПРОВЕРКА СОПОСТАВИМОСТИ ПО CRC-ЦЕПОЧКЕ: {label_a} vs {label_b}")
    print("─" * 78)
    try:
        with open(path_a, 'r', encoding='utf-8') as f:
            ra = (json.load(f).get('rows') or [])
        with open(path_b, 'r', encoding='utf-8') as f:
            rb = (json.load(f).get('rows') or [])
    except Exception as e:
        print(f"  ❔ ledger не прочитан: {e!r}")
        return None
    n = min(len(ra), len(rb))
    div = None
    for i in range(n):
        if ra[i].get('chain') != rb[i].get('chain'):
            div = ra[i].get('t')
            break
    first_send = None
    for r in ((log_a or {}).get('tick_rows') or []):
        if float(r.get('send_energy') or 0.0) > 0.0:
            first_send = r.get('t')
            break
    res = {'rows_a': len(ra), 'rows_b': len(rb), 'first_divergence_t': div, 'first_send_tick_snapshot': first_send}
    if div is None:
        print(f"  ✅ цепочки совпали на всех {n} тиках" + ("" if len(ra) == len(rb) else f" (но длины разные: {len(ra)} vs {len(rb)})"))
    else:
        print(f"  ⚠️ траектории разошлись на t={div} (сравнено тиков: {n})")
        if first_send is not None:
            early = div < first_send
            print(f"     первый тик-снимок с send_energy>0 в {label_a}: t={first_send} → "
                  f"{'расхождение РАНЬШЕ первой отправки Морзе: разница не только от Морзе' if early else 'расхождение не раньше первой отправки'}")
        else:
            print("     ❔ в логе A нет send_energy>0: момент первой отправки Морзе неизвестен")
    return res


def run_observer_v2(log_path, ledger_path=None, dump_path=None, title='', include_legacy=True, gap_dump_path=None):
    """Единая точка входа Observer v2. Каждая секция в своём try: сбой секции ПЕЧАТАЕТСЯ, а не глотается."""
    print("\n" + "█" * 78)
    print(f"OBSERVER v2 {title}".rstrip())
    print("█" * 78)
    if not log_path or not os.path.exists(log_path):
        print(f"  ❔ нет лога: {log_path}")
        return None
    with open(log_path, 'r', encoding='utf-8') as f:
        log = json.load(f)
    led = None
    if ledger_path and os.path.exists(ledger_path):
        try:
            with open(ledger_path, 'r', encoding='utf-8') as f:
                led = json.load(f)
        except Exception as e:
            print(f"  ❔ ledger не прочитан: {e!r}")
    audit = None
    try:
        ap = globals()['_audit_path_for'](log_path) if '_audit_path_for' in globals() else None
        if ap and os.path.exists(ap):
            with open(ap, 'r', encoding='utf-8') as f:
                audit = json.load(f)
    except Exception as e:
        print(f"  ❔ audit JSON не прочитан: {e!r}")
    R = None
    try:
        R = _ov_compute(log, led, audit, log_path, ledger_path)
    except Exception as e:
        import traceback
        print(f"  ❌ observer v2: расчёт упал: {e!r}\n{traceback.format_exc()}")
        return None
    for fn in (_ov_print_summary, _ov_print_glossary, _ov_print_header, _ov_print_invariants, _ov_print_tickwall,
               _ov_print_decomp, _ov_print_chan, _ov_print_nav):
        try:
            fn(R)
        except Exception as e:
            print(f"  ❌ секция {fn.__name__} упала: {e!r}")
    if include_legacy:
        try:
            _gap_clip_report(log_path, dump_path=gap_dump_path)
        except Exception as e:
            print(f"  ❌ [GAP] G1–G6 упал: {e!r}")
    print("█" * 78, flush=True)
    if dump_path:
        try:
            with open(dump_path, 'w', encoding='utf-8') as f:
                json.dump(R, f, ensure_ascii=False, indent=1, default=lambda o: (float(o) if hasattr(o, '__float__') else str(o)))
            print(f"[OBS v2] итоги записаны: {dump_path}")
        except Exception as e:
            print(f"[OBS v2] дамп не записан: {e!r}")
    return R


# ═════════════ [P] observer: население и простор (read-only; только факты из лога) ═════════════
# Источник: tick_rows[*]['pop'] (RichLog кладёт туда World._pop_snapshot()) и agent_snaps (ориентация).
# Ничего не пересчитывает в симуляции. Вердикт — НЕ здесь: см. _pop_gate_verdict (роль Gate).
_POP_WIN = 100


def _pop_delta(a, b):
    ks = set(a or {}) | set(b or {})
    return {k: int((b or {}).get(k, 0)) - int((a or {}).get(k, 0)) for k in ks}


def _pop_fmt(d, top=6):
    items = sorted(((k, v) for k, v in d.items() if v), key=lambda kv: -kv[1])[:top]
    return " ".join(f"{k}={v}" for k, v in items) if items else "-"


def _pop_report(log_path, title='', dump_path=None, ledger_path=None):
    import numpy as np
    from collections import Counter
    print("=" * 78)
    print(f"[POP] НАСЕЛЕНИЕ И ПРОСТОР {title}(observer: read-only, только факты из лога; вердикт — у Gate)")
    print("=" * 78)
    if not log_path or not os.path.exists(log_path):
        print(f"  [!] нет лога: {log_path}")
        return None
    with open(log_path, 'r', encoding='utf-8') as f:
        log = json.load(f)
    rows = [r for r in (log.get('tick_rows') or []) if isinstance(r.get('pop'), dict)]
    rows.sort(key=lambda r: r['t'])
    if not rows:
        print("  [!] в tick_rows нет поля 'pop': лог записан версией до этого патча (состав населения не писался)")
        return None
    P0 = rows[0]['pop']
    ledger_n_last = None
    ledger_check_status = 'UNKNOWN'
    if ledger_path and os.path.exists(ledger_path):
        try:
            with open(ledger_path, 'r', encoding='utf-8') as _lf:
                _ledger_data = json.load(_lf)
            _ledger_rows = _ledger_data.get('rows') or []
            if _ledger_rows and isinstance(_ledger_rows[-1].get('n'), (int, float)):
                ledger_n_last = int(_ledger_rows[-1]['n'])
                ledger_check_status = 'MATCH' if int(rows[-1]['pop']['n']) == ledger_n_last else 'MISMATCH'
        except Exception as _le:
            print(f"  [GATE-POP] Ledger reconciliation read error: {_le!r}")
            ledger_check_status = 'ERROR'
    print(f"  [GATE-POP] сверка последнего населения: RichLog={int(rows[-1]['pop']['n'])}, "
          f"Ledger={ledger_n_last if ledger_n_last is not None else 'UNKNOWN'}, status={ledger_check_status}")
    print(f"  потолок={P0['ceiling']} мягкий старт={P0['soft_start']} | fill soft/hard={P0['fill_soft']:.2f}/{P0['fill_hard']:.2f} | "
          f"замеров={len(rows)} (каждые {max(1, rows[1]['t'] - rows[0]['t']) if len(rows) > 1 else '?'} тиков)")

    # ── [P1] траектория ──
    print("\n  ── [P1] ТРАЕКТОРИЯ (каждый ~5-й замер) ──")
    print(f"  {'t':>5s} {'всего':>6s} {'normal':>6s} {'feral':>5s} {'др.':>4s} {'fill%':>6s} {'brake':>6s} {'birth×':>6s} {'клеток':>7s} {'свобод.кл/агент':>15s}")
    step = max(1, len(rows) // 7)
    shown = list(range(0, len(rows), step))
    if shown[-1] != len(rows) - 1:
        shown.append(len(rows) - 1)
    for i in shown:
        r, p = rows[i], rows[i]['pop']
        world = float(Config.WORLD_SIZE) ** 2 if 'Config' in globals() else 208.0 ** 2
        free_per = world * (1.0 - p['fill']) / max(1, p['n'])
        print(f"  {r['t']:5d} {p['n']:6d} {p['normal']:6d} {p['feral']:5d} {p['other']:4d} {100.0 * p['fill']:6.1f} "
              f"{p['space_brake']:6.2f} {p['birth_factor']:6.2f} {p['cells_mean']:7.1f} {free_per:15.1f}")

    # ── окна ──
    wins = {}
    for r in rows:
        wins.setdefault(int(r['t']) // _POP_WIN, []).append(r)
    wkeys = sorted(wins)
    print("\n  ── [P2] ПО ОКНАМ 100 ТИКОВ: население ──")
    print(f"  {'окно':>9s} {'замеров':>7s} {'n min':>6s} {'n mean':>7s} {'n max':>6s} {'fill max%':>9s} {'цел.':>5s}")
    win_res = []
    for w in wkeys:
        ns = [x['pop']['n'] for x in wins[w]]
        fs = [x['pop']['fill'] for x in wins[w]]
        print(f"  {w * _POP_WIN:4d}-{w * _POP_WIN + _POP_WIN - 1:<4d} {len(ns):7d} {min(ns):6d} {float(np.mean(ns)):7.1f} {max(ns):6d} {100.0 * max(fs):9.1f} {wins[w][-1]['pop']['ceiling']:5d}")
        win_res.append(dict(window=[w * _POP_WIN, w * _POP_WIN + _POP_WIN - 1], n_min=int(min(ns)), n_mean=float(np.mean(ns)),
                            n_max=int(max(ns)), fill_max=float(max(fs))))

    print("\n  ── [P3] РОЖДЕНИЯ ПО ИСТОЧНИКАМ и ОТКАЗЫ (прирост за окно; отказ = рождение, которое тормоз/потолок не пропустил) ──")
    prev = {'births': {}, 'blocked': {}, 'deaths': {}, 'mv': None}
    flows = []
    prev_n = int(P0['n'])
    for w in wkeys:
        last = wins[w][-1]['pop']
        b = _pop_delta(prev['births'], last['births'])
        k = _pop_delta(prev['blocked'], last['blocked'])
        d = _pop_delta(prev['deaths'], last['deaths'])
        print(f"  {w * _POP_WIN:4d}-{w * _POP_WIN + _POP_WIN - 1:<4d} родилось: {_pop_fmt(b)}")
        print(f"  {'':9s} отказов:  {_pop_fmt(k)}")
        # честный баланс окна: Δn должно ≈ родилось − умерло. Невязка — факт, а не вывод.
        dn = int(last['n']) - prev_n
        nb_, nd_ = sum(b.values()), sum(d.values())
        resid = dn - (nb_ - nd_)
        mark = '✅' if abs(resid) < 5 else '⚠️'
        print(f"  {'':9s} баланс:   Δn={dn:+d} = родилось {nb_} − умерло {nd_} → невязка {resid:+d} {mark}")
        if dn > 0 and b.get('division', 0) == 0:
            src_ = {kk: vv for kk, vv in b.items() if kk != 'division' and vv}
            print(f"  {'':9s} ⚠️ население выросло на {dn} при 0 делений; учтённые другие источники: {_pop_fmt(src_) if src_ else 'НЕТ — источник роста не учтён ❔'}")
        prev_n = int(last['n'])
        flows.append(dict(window=w * _POP_WIN, births=b, blocked=k, deaths=d))
        prev = {'births': last['births'], 'blocked': last['blocked'], 'deaths': last['deaths'], 'mv': last.get('mv')}
    # ── [P3-L] сверка engine-счётчиков с ЕДИНЫМ источником (ledger) ──
    try:
        _lt = (rows[-1]['pop'] or {}).get('ledger')
        _lf = (log.get('header') or {}).get('pop_ledger_final')
        if _lt:
            _eb = rows[-1]['pop'].get('births') or {}
            _eb_tot = int(sum(int(v) for v in _eb.values()))
            _lb = _lt.get('births_by_kind') or {}
            print(f"\n  ── [P3-L] ЕДИНЫЙ ИСТОЧНИК (ledger) на t={_lt.get('t_last')}: n={_lt.get('n')} рождений={_lt.get('births')} "
                  f"({_pop_fmt(_lb)}) смертей={_lt.get('deaths')} делений={_lt.get('divisions')}")
            _dd = _eb_tot - int(_lt.get('births', 0))
            print(f"  {'':9s} engine-счётчики (_pop_note) на том же замере: рождений={_eb_tot} ({_pop_fmt(_eb)}) → "
                  + ("✅ совпало с ledger" if _dd == 0 else f"⚠️ разница engine−ledger={_dd:+d} (родились и исчезли внутри тика / не попали в eng.patterns)"))
            if isinstance(_lf, dict) and _lf.get('t_last') is not None and _lf.get('t_last') > _lt.get('t_last'):
                print(f"  {'':9s} после последнего замера (t={_lt.get('t_last')}) до конца прогона (t={_lf.get('t_last')}): "
                      f"рождений +{int(_lf.get('births', 0)) - int(_lt.get('births', 0))}, смертей +{int(_lf.get('deaths', 0)) - int(_lt.get('deaths', 0))}; "
                      f"итог прогона по ledger: рождений={_lf.get('births')} смертей={_lf.get('deaths')} делений={_lf.get('divisions')} n={_lf.get('n')}")
        else:
            print("\n  [P3-L] в замерах нет pop.ledger (лог до патча единого источника): сверка engine/ledger недоступна")
    except Exception as _e_p3l:
        print(f"\n  [P3-L] сверка не выполнена: {type(_e_p3l).__name__}: {_e_p3l}")
    print("\n  ── [P4] СМЕРТИ ПО ПРИЧИНАМ (прирост за окно; причины — как их пишет _record_death_drag) ──")
    for fl in flows:
        print(f"  {fl['window']:4d}-{fl['window'] + _POP_WIN - 1:<4d} {_pop_fmt(fl['deaths'], top=8)}")

    # ── [P5] подвижность ──
    print("\n  ── [P5] ПОДВИЖНОСТЬ (прирост за окно) ──")
    pmv = None
    mv_res = []
    for w in wkeys:
        m = wins[w][-1]['pop'].get('mv')
        if not m:
            print(f"  {w * _POP_WIN:4d}-{w * _POP_WIN + _POP_WIN - 1:<4d} нет данных (счётчики навигации не писались)")
            continue
        base = pmv or {k: 0 for k in m}
        dm = {k: m[k] - base.get(k, 0) for k in m}
        c = max(1, dm['mv_call'])
        print(f"  {w * _POP_WIN:4d}-{w * _POP_WIN + _POP_WIN - 1:<4d} агент-тиков={dm['mv_call']:7d} попыток={100.0 * dm['mv_try'] / c:5.1f}% "
              f"успешных={100.0 * dm['mv_ok'] / c:5.1f}% нет_направления={100.0 * dm['mv_fail_nodir'] / c:5.1f}% "
              f"по_оценке={100.0 * dm['mv_fail_thr'] / c:5.1f}% занято={100.0 * dm['mv_fail_occ'] / c:5.1f}% "
              f"эвакуация(ок/попыток)={dm.get('mv_evac_ok', 0)}/{dm.get('mv_evac_try', 0)}")
        mv_res.append(dict(window=w * _POP_WIN, ok_pct=100.0 * dm['mv_ok'] / c, nodir_pct=100.0 * dm['mv_fail_nodir'] / c))
        pmv = m

    # ── [P6] body + orientation, split by role; ferals are no longer silently excluded ──
    print("\n  ── [P6] ТЕЛО × ОРИЕНТАЦИЯ (agent_snaps, живые; отдельно normal/feral) ──")
    print("  axis·A = |cos(axis, movement)|; ось считается только для shape=rod/dumbbell. momentum_corr = cos(EMA(_dir_momentum), actual movement), не intent-vs-action.")
    sn = [s for s in (log.get('agent_snaps') or []) if s.get('alive', True)]
    ori_res = []
    if not sn:
        print("  нет снимков агентов")
    else:
        byw = {}
        for s in sn:
            _role = str(s.get('role_type', 'feral' if s.get('feral_flag') else 'normal'))
            byw.setdefault((int(s.get('t', 0)) // _POP_WIN, _role), []).append(s)
        print(f"  {'окно':>9s} {'роль':>8s} {'снимков':>7s} {'conf_n':>6s} {'dir_conf':>8s} "
              f"{'low%':>6s} {'frag':>6s} {'largest':>7s} {'axis%':>6s} {'mom10':>6s} "
              f"{'axis·A':>7s} {'move_d':>7s} {'ctr_d':>7s} {'карта':>7s}  nav status")
        for w, role in sorted(byw):
            ss = byw[(w, role)]
            valid_dc = [float(x.get('dir_conf', np.nan)) for x in ss
                        if x.get('dir_conf_valid', ('dir_conf' in x)) and x.get('dir_conf') is not None]
            dc = np.asarray(valid_dc, dtype=float)
            dc = dc[np.isfinite(dc)]
            frag = np.asarray([float(x.get('body_frag_live', x.get('soma_fragment', 1))) for x in ss], dtype=float)
            largest = np.asarray([float(x.get('body_largest_frac', np.nan)) for x in ss], dtype=float)
            axis_pct = 100.0 * sum(x.get('body_axis') is not None for x in ss) / max(1, len(ss))
            _ia = np.asarray([float(x.get('momentum_corr', x.get('intent_actual_dot')))
                              for x in ss
                              if x.get('momentum_corr', x.get('intent_actual_dot')) is not None], dtype=float)
            _ba = np.asarray([float(x['body_axis_actual_alignment']) for x in ss
                              if x.get('body_axis_actual_alignment') is not None], dtype=float)
            _md = np.asarray([float(np.hypot(*x['actual_displacement'])) for x in ss
                              if x.get('actual_displacement') is not None], dtype=float)
            _cd = np.asarray([float(np.hypot(*x['center_displacement'])) for x in ss
                              if x.get('center_displacement') is not None], dtype=float)
            def _p6_metric(key):
                _a = np.asarray([float(x[key]) for x in ss
                                 if x.get(key) is not None], dtype=float)
                _a = _a[np.isfinite(_a)]
                return float(np.mean(_a)) if len(_a) else float('nan')
            _gap_mean = _p6_metric('spirit_gap_raw')
            _clip_mean = _p6_metric('gap_clip')
            _gain_mean = _p6_metric('noise_gain')
            _eff_gain_mean = _p6_metric('noise_channel_gain')
            _pred_err_mean = _p6_metric('smoothed_pred_error')
            _move_n = len(_ia)
            _lowconf_pct = 100.0 * float(np.mean(dc < 0.5)) if len(dc) else float('nan')
            mc = np.asarray([float(x.get('nav_map_cells', 0)) for x in ss], dtype=float)
            mf = np.asarray([float(x.get('nav_map_fresh', 0)) for x in ss], dtype=float)
            st = Counter(str(x.get('nav_status', 'none')) for x in ss)
            fresh = 100.0 * float(np.nansum(mf)) / max(1.0, float(np.nansum(mc)))
            st_s = " ".join(f"{k}={100.0 * v / len(ss):.0f}%" for k, v in st.most_common(3))
            dc_mean = float(np.mean(dc)) if len(dc) else float('nan')
            dc_low = 100.0 * float(np.mean(dc < 0.5)) if len(dc) else float('nan')
            _ia_mean = float(np.mean(_ia)) if len(_ia) else float('nan')
            _ba_mean = float(np.mean(_ba)) if len(_ba) else float('nan')
            _md_mean = float(np.mean(_md)) if len(_md) else float('nan')
            _mom_h = {}
            for _hk in ("1", "5", "10", "20"):
                _hv = np.asarray([float(x["momentum_corr_" + _hk + "t"])
                                  for x in ss if x.get("momentum_corr_" + _hk + "t") is not None], dtype=float)
                _hv = _hv[np.isfinite(_hv)]
                _mom_h[_hk] = (float(np.mean(_hv)) if len(_hv) else float('nan'), len(_hv))
            _stable = np.asarray([float(x.get("momentum_corr", np.nan)) for x in ss
                                  if not x.get("intent_switch_since_prev", False)
                                  and x.get("momentum_corr") is not None], dtype=float)
            _switched = np.asarray([float(x.get("momentum_corr", np.nan)) for x in ss
                                    if x.get("intent_switch_since_prev", False)
                                    and x.get("momentum_corr") is not None], dtype=float)
            _stable = _stable[np.isfinite(_stable)]
            _switched = _switched[np.isfinite(_switched)]
            _switch_count = sum(int(x.get('intent_switch_count_since_prev', 0) or 0) for x in ss)
            _active_agent_ticks = sum(int(x.get('actual_interval_ticks') or 0) for x in ss)
            _switch_rate_100 = 100.0 * _switch_count / max(1, _active_agent_ticks)
            _switch_interval_pct = 100.0 * sum(bool(x.get('intent_switch_since_prev', False)) for x in ss) / max(1, sum(x.get('actual_interval_ticks') is not None for x in ss))
            _cd_mean = float(np.mean(_cd)) if len(_cd) else float('nan')
            print(f"  {w * _POP_WIN:4d}-{w * _POP_WIN + _POP_WIN - 1:<4d} {role:>8s} {len(ss):7d} "
                  f"{len(dc):6d} {dc_mean:8.3f} {dc_low:6.1f} {float(np.nanmean(frag)):6.2f} "
                  f"{float(np.nanmean(largest)):7.3f} {axis_pct:6.1f} {_ia_mean:6.2f} "
                  f"{_ba_mean:7.2f} {_md_mean:7.2f} {_cd_mean:7.2f} {float(np.nanmean(mc)):7.1f}  {st_s}")
            print(f"      [NOISE×NAV] spirit_gap={_gap_mean:.3f} gap_clip={_clip_mean:.3f} "
                  f"noise_gain={_gain_mean:.3f} channel_gain={_eff_gain_mean:.3f} "
                  f"smoothed_pred_error={_pred_err_mean:.3f} momentum_corr n={_move_n} low_conf={_lowconf_pct:.1f}%")
            print(f"      [MOMENTUM] corr 1t={_mom_h['1'][0]:.3f}(n={_mom_h['1'][1]}) "
                  f"5t={_mom_h['5'][0]:.3f}(n={_mom_h['5'][1]}) "
                  f"10t={_mom_h['10'][0]:.3f}(n={_mom_h['10'][1]}) "
                  f"20t={_mom_h['20'][0]:.3f}(n={_mom_h['20'][1]}); "
                  f"intent-switches={_switch_count} ({_switch_rate_100:.3f}/100 agent-ticks; intervals={_switch_interval_pct:.1f}%) "
                  f"stable={float(np.mean(_stable)) if len(_stable) else float('nan'):.3f}(n={len(_stable)}) "
                  f"switched={float(np.mean(_switched)) if len(_switched) else float('nan'):.3f}(n={len(_switched)})")
            # Axis alignment is meaningful only for bodies whose topology supplies an axis.
            _elong_bins = (("<0.1", lambda e: e < 0.1),
                           ("0.1-0.3", lambda e: 0.1 <= e < 0.3),
                           ("0.3-0.5", lambda e: 0.3 <= e < 0.5),
                           (">=0.5", lambda e: e >= 0.5))
            _axis_parts = []
            for _bn, _bf in _elong_bins:
                _bb = [x for x in ss if x.get("body_elongation") is not None
                       and np.isfinite(float(x.get("body_elongation")))
                       and _bf(float(x.get("body_elongation")))]
                _av = [float(x["body_axis_actual_alignment"]) for x in _bb
                       if x.get("body_axis_actual_alignment") is not None
                       and np.isfinite(float(x["body_axis_actual_alignment"]))]
                _axis_parts.append(f"{_bn}:{(float(np.mean(_av)) if _av else float('nan')):.3f}/n{len(_av)}")
            print("      [AXIS by elongation] " + " | ".join(_axis_parts))
            _shape_parts = []
            for _sh in ("ball", "rod", "dumbbell", "ring", "split", "unknown"):
                _bb = [x for x in ss if (x.get("body_shape") or "unknown") == _sh]
                if _bb:
                    _av = [float(x["body_axis_actual_alignment"]) for x in _bb
                           if x.get("body_axis_actual_alignment") is not None
                           and np.isfinite(float(x["body_axis_actual_alignment"]))]
                    _shape_parts.append(f"{_sh}:{(float(np.mean(_av)) if _av else float('nan')):.3f}/n{len(_av)}")
            print("      [AXIS by shape] " + " | ".join(_shape_parts))
            _fill_bins = (("0-5%", lambda f: 0.0 <= f < 0.05),
                          ("5-10%", lambda f: 0.05 <= f < 0.10),
                          ("10-15%", lambda f: 0.10 <= f < 0.15),
                          (">=15%", lambda f: f >= 0.15))
            _fill_parts = []
            for _fn, _ff in _fill_bins:
                _bb = [x for x in ss if x.get("local_fill") is not None
                       and np.isfinite(float(x.get("local_fill")))
                       and _ff(float(x.get("local_fill")))]
                _mv = [float(x["momentum_corr"]) for x in _bb
                       if x.get("momentum_corr") is not None and np.isfinite(float(x["momentum_corr"]))]
                _sw = sum(int(x.get("intent_switch_count_since_prev", 0) or 0) for x in _bb)
                _at = sum(int(x.get("actual_interval_ticks") or 0) for x in _bb)
                _fill_parts.append(f"{_fn}:mom={(float(np.mean(_mv)) if _mv else float('nan')):.3f}/n{len(_mv)},sw={100.0*_sw/max(1,_at):.3f}")
            print("      [MOMENTUM by local_fill] " + " | ".join(_fill_parts))
            ori_res.append(dict(window=w * _POP_WIN, role=role, n=len(ss), dir_conf_n=len(dc),
                                dir_conf=dc_mean, low_conf_pct=dc_low,
                                frag_mean=float(np.nanmean(frag)),
                                largest_frac_mean=float(np.nanmean(largest)),
                                body_axis_valid_pct=axis_pct,
                                momentum_corr=_ia_mean, intent_actual_dot=_ia_mean,
                                body_axis_actual_alignment=_ba_mean,
                                momentum_corr_1t=_mom_h['1'][0], momentum_corr_5t=_mom_h['5'][0],
                                momentum_corr_10t=_mom_h['10'][0], momentum_corr_20t=_mom_h['20'][0],
                                intent_switch_interval_pct=_switch_interval_pct, intent_switches=_switch_count,
                                intent_switch_rate_per_100_agent_ticks=_switch_rate_100,
                                momentum_corr_stable=(float(np.mean(_stable)) if len(_stable) else float('nan')),
                                momentum_corr_switched=(float(np.mean(_switched)) if len(_switched) else float('nan')),
                                actual_move_mean=_md_mean, center_shift_mean=_cd_mean,
                                spirit_gap_mean=_gap_mean, gap_clip_mean=_clip_mean,
                                noise_gain_mean=_gain_mean, noise_channel_gain_mean=_eff_gain_mean,
                                smoothed_pred_error_mean=_pred_err_mean,
                                intent_actual_n=_move_n))

    try:
        _pop_honesty_blocks(log, rows, wins, wkeys)
    except Exception as _e:
        print(f"  [!] блоки честности не выполнены: {_e!r}")
    res = dict(log=str(log_path), ceiling=int(P0['ceiling']), soft_start=int(P0['soft_start']), fill_hard=float(P0['fill_hard']),
               n_max=int(max(r['pop']['n'] for r in rows)), fill_max=float(max(r['pop']['fill'] for r in rows)),
               n_last=int(rows[-1]['pop']['n']), ledger_n_last=ledger_n_last, ledger_check_status=ledger_check_status,
               windows=win_res, flows=flows, move=mv_res, orientation=ori_res)
    print("\n" + "=" * 78, flush=True)
    if dump_path:
        try:
            with open(dump_path, 'w', encoding='utf-8') as f:
                json.dump(res, f, ensure_ascii=False, indent=1)
            print(f"[POP] итоги записаны: {dump_path}")
        except Exception as e:
            print(f"[POP] дамп не записан: {e!r}")
    try:
        _pop_gate_verdict(res)
    except Exception as e:
        print(f"[GATE-POP] вердикт не выдан: {e!r}")
    return res


def _pop_honesty_blocks(log, rows, wins, wkeys):
    """[P7..P11] вопросы, которые лог вызывает сам: ворота Морзе vs плотность, смертность в зоне vs вне,
    отрицательная энергия, ЗАПАС триады над порогом, охват отчёта. Только чтение лога."""
    import numpy as np
    from collections import Counter
    W = _POP_WIN
    last_all = rows[-1]['pop']
    # ── [P7] ворота Морзе по окнам рядом с плотностью ──
    print("\n  ── [P7] ВОРОТА МОРЗЕ vs ПЛОТНОСТЬ (прирост за окно; доля = от отказов окна) ──")
    pg, pn = None, None
    any7 = False
    for w in wkeys:
        g = wins[w][-1]['pop'].get('gates')
        nmean = float(np.mean([x['pop']['n'] for x in wins[w]]))
        if not g:
            print(f"  {w * W:4d}-{w * W + W - 1:<4d} нет данных (ворота не писались)")
            continue
        any7 = True
        base = pg or {'attempt': 0, 'ok': 0, 'silence': {}}
        da, dok = g['attempt'] - base['attempt'], g['ok'] - base['ok']
        sil = {k: v - base['silence'].get(k, 0) for k, v in g['silence'].items()}
        tot = sum(v for v in sil.values() if v > 0) or 1
        sh = " ".join(f"{k}={100.0 * v / tot:.0f}%" for k, v in sorted(sil.items(), key=lambda kv: -kv[1])[:4] if v > 0)
        print(f"  {w * W:4d}-{w * W + W - 1:<4d} n̄={nmean:5.0f} попыток={da:5d} ушло={100.0 * dok / max(1, da):5.1f}%  отказы: {sh or '-'}")
        pg = g
    if not any7:
        print("  ❔ лог записан версией до этого патча")

    # ── [P8] зона: смертность внутри vs вне ──
    print("\n  ── [P8] ЗОНА: смертность на 1000 агент-тиков, внутри vs вне (место смерти — по трекеру зон, ±1 тик) ──")
    sanc, at, zd = last_all.get('sanc'), last_all.get('agent_ticks'), last_all.get('zone_deaths') or {}
    if not sanc or not at:
        print("  ❔ нет данных (трекер зон выключен или лог до патча)")
    else:
        ins_t = int(sanc['inside_ticks']); out_t = max(0, int(at) - ins_t)
        d_in, d_out = int(zd.get('inside', 0)), int(zd.get('outside', 0))
        h_in = 1000.0 * d_in / ins_t if ins_t else None
        h_out = 1000.0 * d_out / out_t if out_t else None
        print(f"  внутри: агент-тиков={ins_t} смертей={d_in} → {h_in:.2f}‰" if h_in is not None else "  внутри: нет агент-тиков")
        print(f"  вне:    агент-тиков={out_t} смертей={d_out} → {h_out:.2f}‰" if h_out is not None else "  вне: нет агент-тиков")
        if h_in is not None and h_out:
            print(f"  отношение внутри/вне = {h_in / h_out:.2f}" + ("   ⚠️ смертей внутри <10: отношение статистически слабое" if d_in < 10 else ""))
        ci = {k[len('inside__'):]: v for k, v in zd.items() if k.startswith('inside__')}
        co = {k[len('outside__'):]: v for k, v in zd.items() if k.startswith('outside__')}
        print(f"  причины внутри: {_pop_fmt(ci, 5)}")
        print(f"  причины вне:    {_pop_fmt(co, 5)}")
        print(f"  (трекер зон: вошло {sanc['entered']}, умерло внутри {sanc['died_inside']})")

    # ── [P9] отрицательная энергия ──
    print("\n  ── [P9] ОТРИЦАТЕЛЬНАЯ ЭНЕРГИЯ АГЕНТА: чем и у кого ──")
    xn = last_all.get('x2neg')
    if xn is None:
        print("  ❔ нет данных (лог до патча)")
    elif not xn:
        print("  ✅ ни одного пересечения нуля и ни одного агента с энергией < 0 за прогон")
    else:
        by = {k[len('by__'):]: v for k, v in xn.items() if k.startswith('by__')}
        rl = {k[len('role__'):]: v for k, v in xn.items() if k.startswith('role__')}
        print(f"  пересечений нуля через _x2_change={xn.get('cross', 0)}; агент-тиков с энергией<0={xn.get('agent_ticks', 0)}")
        print(f"  по действиям (чем ушёл в минус): {_pop_fmt(by, 6)}")
        print(f"  по ролям (агент-тики):           {_pop_fmt(rl, 4)}")
        if xn.get('agent_ticks', 0) and not xn.get('cross', 0):
            print("  ⚠️ минус есть, а пересечений через _x2_change нет: энергию меняют напрямую, мимо учёта действий ❔")

    # ── [P10] запас триады над порогом ──
    print("\n  ── [P10] ТРИАДА: запас над порогом BINDING_FLOOR (TA=1.00 различает, только если запас конечный) ──")
    floor = float(globals().get('Config').BINDING_FLOOR) if 'Config' in globals() and hasattr(globals().get('Config'), 'BINDING_FLOOR') else 0.01
    sn = [s for s in (log.get('agent_snaps') or []) if s.get('alive', True) and not s.get('feral_flag') and 'tri_body' in s]
    if not sn:
        print("  ❔ в снимках нет tri_* (лог до патча)")
    else:
        cols = (('spirit_gap', 'gap'), ('soul_weight', 'tri_soul'), ('body_memory', 'tri_body'), ('body_eff', 'tri_body_eff'), ('coherence', 'tri_coh'))
        print(f"  порог={floor}; снимков={len(sn)}")
        print(f"  {'условие':>12s} {'min':>8s} {'p5':>8s} {'p50':>8s} {'<3×порог%':>10s}")
        for name, key in cols:
            v = np.array([float(x.get(key, np.nan)) for x in sn], dtype=float)
            v = v[np.isfinite(v)]
            if v.size:
                print(f"  {name:>12s} {v.min():8.3f} {np.percentile(v, 5):8.3f} {np.percentile(v, 50):8.3f} {100.0 * float(np.mean(v < 3 * floor)):10.1f}")
        ur = np.array([float(x.get('tri_unres', np.nan)) for x in sn], dtype=float)
        ta = np.array([float(x.get('tri_alive', np.nan)) for x in sn], dtype=float)
        print(f"  unresolvable_intact: {100.0 * np.nanmean(ur):.1f}% снимков | TA по снимкам: {100.0 * np.nanmean(ta):.1f}%" if np.isfinite(ta).any() else f"  unresolvable_intact: {100.0 * np.nanmean(ur):.1f}%")
        if np.isfinite(ta).any() and np.isfinite(ur).any() and float(np.nanmean(ta)) > 0.5 and float(np.nanmean(ur)) == 0.0:
            print("  ⚠️ [P10] противоречив: TA>50% при unresolvable_intact=0% — tri_* сняты не через soul_check (лог старой версии?)")

    # ── [P11] охват разложения gap ──
    print("\n  ── [P11] ОХВАТ ОТЧЁТА [GAP]: кого он НЕ видит ──")
    allsn = [s for s in (log.get('agent_snaps') or []) if s.get('alive', True) and not s.get('feral_flag')]
    if not allsn:
        print("  ❔ нет снимков")
    else:
        warm = int(globals().get('_OG_WARMUP', 30)); maxst = int(globals().get('_OG_MAX_STALE', 10))
        def reason(s):
            ed = s.get('err_dim')
            if not (isinstance(ed, list) and sum(abs(float(x)) for x in ed) > 0):
                return 'нет err_dim'
            if int(s.get('age', 0)) < warm:
                return f'возраст<{warm}'
            if int(s.get('err_dim_stale', 0)) > maxst:
                return f'err_dim старше {maxst}'
            return 'в учёте'
        rs = Counter(reason(s) for s in allsn)
        print(f"  обычных живых снимков={len(allsn)}: " + " | ".join(f"{k}={v} ({100.0 * v / len(allsn):.0f}%)" for k, v in rs.most_common()))
        inc = [s for s in allsn if reason(s) == 'в учёте']; exc = [s for s in allsn if reason(s) != 'в учёте']
        def mm(ss, key):
            v = np.array([float(x.get(key, np.nan)) for x in ss], dtype=float)
            return float(np.nanmean(v)) if v.size and np.isfinite(v).any() else float('nan')
        print(f"  {'':10s} {'снимков':>8s} {'возраст':>8s} {'клеток':>7s} {'gap_clip':>9s}")
        for nm, ss in (('в учёте', inc), ('исключены', exc)):
            print(f"  {nm:10s} {len(ss):8d} {mm(ss, 'age'):8.1f} {mm(ss, 'n_cells'):7.1f} {mm(ss, 'gap_clip'):9.3f}")


def _pop_gate_verdict(res):
    """Роль Gate (policy-only): решение по готовым фактам Observer. Пороги — политика, меняются здесь, а не в симуляции.
    PASS: всё в пределах потолка. WARN: превышение до +10% потолка. BLOCK: больше. UNKNOWN: нет данных."""
    if not res or res.get('n_max') is None:
        print("[GATE-POP] UNKNOWN: нет данных о населении")
        return 'UNKNOWN'
    if res.get('ledger_check_status') in ('MISMATCH', 'ERROR'):
        print(f"[GATE-POP] BLOCK: сверка RichLog↔Ledger={res.get('ledger_check_status')} "
              f"(RichLog n_end={res.get('n_last')}, Ledger n_end={res.get('ledger_n_last')})")
        return 'BLOCK'
    if res.get('ledger_check_status') != 'MATCH':
        print("[GATE-POP] UNKNOWN: конечное население не сверено с Ledger")
        return 'UNKNOWN'
    ceil = float(res['ceiling'])
    n_max = float(res['n_max'])
    fill_ok = res['fill_max'] <= res['fill_hard'] + 0.01
    if n_max <= ceil and fill_ok:
        v = 'PASS'
    elif n_max <= ceil * 1.10:
        v = 'WARN'
    else:
        v = 'BLOCK'
    print(f"[GATE-POP] {v}: n_max={int(n_max)} (потолок {int(ceil)}), fill_max={100.0 * res['fill_max']:.1f}% "
          f"(hard {100.0 * res['fill_hard']:.1f}%), n в конце={res['n_last']}", flush=True)
    return v


In [ ]:
# Spatial Emotional Memory acceptance checks (fast; does not launch the full simulation)
try:
    _sem_check = _spatial_emotional_memory_selftest()
    if not _sem_check.get('ok'):
        print("⚠️ ВНИМАНИЕ: SEM self-test провален!", _sem_check)
    else:
        print('[SEM ACCEPTANCE] PASS', _sem_check)
except Exception as _e:
    _sem_check = {'ok': False, 'exception': repr(_e)}
    print("⚠️ ВНИМАНИЕ: SEM self-test выбросил исключение!", _sem_check)
try:
    _sem_stats_check = _sem_stats_selftest()
    print('[SEM ACCEPTANCE] stats/observer PASS', _sem_stats_check)
except Exception as _e:
    print("⚠️ ВНИМАНИЕ: SEM stats self-test упал!", repr(_e))
print('[SEM ACCEPTANCE] Full simulation status: not run by this cell.')

In [ ]:
# Ячейка DIRCHAIN (Observer, read-only): цепочка «направление → шаг → тело», запись на уровне ПРИМИТИВОВ движения.
# Одна строка = один вызов примитива: Pattern.move (селектор, базовый класс, где он определён) или Pattern._try_shift_cells
# (жёсткий сдвиг тела: sprint/rage-прыжок и ferals). Повторные вызовы в одном тике (основной шаг, jump = t+7919,
# lunge = t+6007, sprint-leap, prey→energy у ferals) пишутся ОТДЕЛЬНО (kind / cidx / tid): селекторные поля, ok и сдвиг
# берутся из ОДНОГО И ТОГО ЖЕ вызова, плотность (fill) — ДО шага.
# Нулевой уровень — СТРАТИФИЦИРОВАННОЕ перемешивание: внутри (агент × корзина плотности × вид вызова × роль), без самопар.
# Отвечает на три вопроса: (1) теряет ли агент направление или не может двигаться туда, (2) значима ли геометрия тела,
# (3) что меняется от разреженного к плотному. Поведение симуляции НЕ меняет: только пишет побочный журнал _DC_LOG.
# Требует точечных вставок в cell 5 (self._dc = {...} в move()/move_feral()) — без них селекторные поля = UNKNOWN.
import gzip, json, math, os
import numpy as np

_DC_COLS = ['t', 'pid', 'role', 'fn', 'kind', 'tid', 'cidx', 'ok', 'reason', 'fresh', 'hx', 'hy', 'm0x', 'm0y',
            'mux', 'muy', 'bdx', 'bdy', 'cx', 'cy', 'prex', 'prey', 'blk_n', 'n_free', 'sc1', 'sc2', 'mom_blk',
            'bd_blk', 'sx', 'sy', 'ctr_d', 'fill', 'fill_oth', 'n_cells', 'shape', 'elong', 'sd_age', 'frag',
            'lfrac', 'ax', 'ay', 'mode', 'nodir']
_DCI = {c: i for i, c in enumerate(_DC_COLS)}
_DC_STR_COLS = ('reason', 'shape', 'mode', 'kind')
_DC_LOG = []
_DC_META = {'world': None, 'calls': 0, 'attempts': 0, 'err': 0, 'err_last': None, 'cap_hit': 0, 'tid': 0, 'orphans': 0}
_DC_CAP = 600000
_DC_ROLE = {'normal': 0, 'feral': 1, 'disorganizer': 2}
_DC_FILL_EDGES = [0.0, 0.05, 0.10, 0.15, 0.20, 0.30, 1.01]   # корзины для ТАБЛИЦ
_DC_FILL_OTH_EDGES = [0.0, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 1.01]   # корзины fill_oth (без своих клеток), мельче внизу
_DC_NULL_EDGES = [0.0, 0.10, 0.20, 1.01]                      # корзины плотности внутри СТРАТ нуля (разреженно/средне/плотно)
_DC_KIND_ORDER = ('primary', 'jump', 'lunge', 'extra', 'leap', 'feral_prey', 'feral_energy', 'feral_other')
_DC_KIND_ID = {k: i for i, k in enumerate(_DC_KIND_ORDER)}
_DC_SEL_KINDS = ('primary', 'jump', 'lunge', 'extra')          # вызовы селектора Pattern.move
_DC_T_JUMP, _DC_T_LUNGE = 7919, 6007                           # смещения t в cell 27 (sprint-jump / rage-lunge)
_DC_MIN_STRATUM = 4


def _dc_center(cells, W):
    a = np.asarray(list(cells), dtype=np.float64)
    ang = a * (2.0 * math.pi / W)
    return (math.atan2(np.sin(ang[:, 0]).mean(), np.cos(ang[:, 0]).mean()) * W / (2 * math.pi) % W,
            math.atan2(np.sin(ang[:, 1]).mean(), np.cos(ang[:, 1]).mean()) * W / (2 * math.pi) % W)


def _dc_err(e):
    _DC_META['err'] += 1
    _DC_META['err_last'] = repr(e)[:120]


def _dc_world_check(w):
    if _DC_META['world'] is not w:
        _DC_LOG.clear()
        _DC_META.update({'world': w, 'calls': 0, 'attempts': 0, 'err': 0, 'err_last': None, 'cap_hit': 0,
                         'tid': 0, 'orphans': 0})


def _dc_new_ctx(self, fn, t):
    """Контекст тика одного агента: общий для всех вызовов примитивов внутри одного вызова move()/move_feral()."""
    w = getattr(self, 'world', None)
    _dc_world_check(w)
    _DC_META['tid'] += 1
    c = {'tid': _DC_META['tid'], 'fn': fn, 't0': 0, 'n': 0, 'prim': False, 'live': False,
         'h': None, 'fresh': False, 'm0': (0.0, 0.0)}
    try:
        c['t0'] = int(t)
        c['live'] = bool(getattr(self, 'alive', False)) and bool(getattr(self, 'cells', None))
        m0 = getattr(self, '_dir_momentum', (0.0, 0.0)) or (0.0, 0.0)
        c['m0'] = (float(m0[0]), float(m0[1]))
        n = getattr(self, '_nav', None)
        cmd = n.get('cmd') if isinstance(n, dict) else None
        if cmd is not None and getattr(self, 'role_type', None) != 'feral':
            nw = int(getattr(w, 'age', t))
            c['fresh'] = (nw - int(cmd['t'])) <= int(Config.NAV_SENSE_EVERY) + 1
            if c['fresh'] and cmd.get('h') is not None:
                c['h'] = (float(cmd['h'][0]), float(cmd['h'][1]))
    except Exception as e:
        _dc_err(e)
    return c


def _dc_install():
    P = globals().get('Pattern')
    if P is None:
        print("[DIRCHAIN] Pattern не найден — трейс не установлен")
        return False
    def _has_outer(name):
        # Обёртка контекста тика может стоять НЕ на вершине MRO: поздние слои (BodyOrientation/MOMENTUM/FISH...)
        # добавляют свои move() выше неё. Старым считаем трейс, у которого флаг установки есть, а обёртки нет нигде в MRO.
        return any(getattr(k.__dict__.get(name), '_dc_outer', False) for k in P.__mro__)

    if getattr(P, '_dc_installed', False) and not _has_outer('move'):
        print("[DIRCHAIN] ⚠️ в этой сессии уже стоит СТАРЫЙ трейс (одна запись на тик). Перезапусти среду и выполни ячейки заново — "
              "поверх старой обёртки новый трейс не ставлю.")
        return False
    W = int(Config.WORLD_SIZE)
    R6 = 6
    yy, xx = np.mgrid[-R6:R6 + 1, -R6:R6 + 1]
    msk = (xx ** 2 + yy ** 2) <= R6 * R6
    DX, DY = xx[msk].astype(np.int64), yy[msk].astype(np.int64)

    def _fill_pre(self, field, old_cells):
        """Плотность ДО шага: диск R=6 вокруг каждой клетки тела (включая свои / только чужие)."""
        arr = np.asarray(list(old_cells), dtype=np.int64)
        X = (arr[:, 0, None] + DX[None, :]) % W
        Y = (arr[:, 1, None] + DY[None, :]) % W
        own = field[X, Y, CH['owner']]
        return float((own != 0).mean()), float(((own != 0) & (own != self.id)).mean())

    def _row(self, t, ctx, kind, old_cells, fills, st):
        _DC_META['attempts'] += 1
        ok = bool(getattr(self, '_last_move_ok', False))
        c0 = _dc_center(old_cells, W)
        sx = sy = 0.0
        if ok and self.cells:
            c1 = _dc_center(self.cells, W)
            sx = ((c1[0] - c0[0] + W / 2) % W) - W / 2
            sy = ((c1[1] - c0[1] + W / 2) % W) - W / 2
        frag = lfrac = None
        largest = old_cells
        try:
            comps = self._components(old_cells, W)
            if comps:
                frag = int(len(comps))
                largest = comps[0]
                lfrac = len(largest) / max(1, len(old_cells))
        except Exception:
            pass
        sd = getattr(self, '_soma_diag', None)
        sd = sd if isinstance(sd, dict) else {}
        shape = sd.get('shape')
        elong = sd.get('elongation')
        now = int(getattr(self.world, 'age', t))
        sd_age = (now - int(sd['t'])) if sd.get('t') is not None else None
        ax = ay = None
        if shape in ('rod', 'dumbbell') and len(largest) >= 3:
            xy = np.asarray([(((x - c0[0] + W / 2) % W) - W / 2, ((y - c0[1] + W / 2) % W) - W / 2)
                             for x, y in largest], dtype=np.float64)
            ev, evec = np.linalg.eigh(np.cov(xy, rowvar=False))
            if ev[-1] > 1e-9 and (ev[-1] - ev[-2]) / max(1e-9, ev[-1]) > 0.05:
                ax, ay = float(evec[0, -1]), float(evec[1, -1])
        mu, bd, ch, pre = st.get('mu'), st.get('bd'), st.get('chosen'), st.get('pre')
        h, m0 = ctx['h'], ctx['m0']
        _f = lambda v: (None if v is None or (isinstance(v, float) and not math.isfinite(v)) else v)
        ctx['n'] += 1
        d = {'t': now, 'pid': int(self.id), 'role': _DC_ROLE.get(getattr(self, 'role_type', None), 0),
             'fn': 0 if ctx['fn'] == 'move' else 1, 'kind': kind, 'tid': ctx['tid'], 'cidx': ctx['n'], 'ok': ok,
             'reason': str(getattr(self, '_last_move_reason', None)), 'fresh': bool(ctx['fresh']),
             'hx': (h[0] if h else None), 'hy': (h[1] if h else None), 'm0x': m0[0], 'm0y': m0[1],
             'mux': (mu[0] if mu else None), 'muy': (mu[1] if mu else None),
             'bdx': (bd[0] if bd else None), 'bdy': (bd[1] if bd else None),
             'cx': (ch[0] if ch else None), 'cy': (ch[1] if ch else None),
             'prex': (pre[0] if pre else None), 'prey': (pre[1] if pre else None),
             'blk_n': st.get('blk_n'), 'n_free': st.get('n_free'), 'sc1': _f(st.get('sc1')), 'sc2': _f(st.get('sc2')),
             'mom_blk': st.get('mom_blk'), 'bd_blk': st.get('bd_blk'), 'sx': float(sx), 'sy': float(sy),
             'ctr_d': float(math.hypot(sx, sy)), 'fill': fills[0], 'fill_oth': fills[1], 'n_cells': int(len(old_cells)),
             'shape': shape, 'elong': _f(elong), 'sd_age': sd_age, 'frag': frag, 'lfrac': lfrac, 'ax': ax, 'ay': ay,
             'mode': st.get('mode'), 'nodir': bool(st.get('nodir', False))}
        if len(_DC_LOG) < _DC_CAP:
            _DC_LOG.append(tuple(d[c] for c in _DC_COLS))
        else:
            _DC_META['cap_hit'] += 1

    def _snap(self, field, ctx):
        if ctx['live'] and getattr(self, 'alive', False) and self.cells:
            old = set(self.cells)
            return old, _fill_pre(self, field, old)
        return None

    # ── примитив 1: селектор Pattern.move (каждый вызов, включая jump/lunge, — своя строка) ──
    def _mk_prim_move(orig):
        def prim(self, field, t, *a, **k):
            ctx = getattr(self, '_dc_ctx', None)
            if ctx is None:                                   # вызов вне обёртки тика
                ctx = _dc_new_ctx(self, 'move', t)
                _DC_META['orphans'] += 1
            pre = None
            try:
                pre = _snap(self, field, ctx)
                if pre is not None:
                    self._dc = None                           # поля селектора — только от ЭТОГО вызова
            except Exception as e:
                _dc_err(e)
                pre = None
            res = orig(self, field, t, *a, **k)
            if pre is not None:
                try:
                    if bool(getattr(self, '_last_move_attempt', False)):   # mobility_gate → попытки не было
                        dd = getattr(self, '_dc', None)
                        st = dd if isinstance(dd, dict) else {}
                        delta = int(t) - int(ctx['t0'])
                        if delta == _DC_T_JUMP:
                            kind = 'jump'
                        elif delta == _DC_T_LUNGE:
                            kind = 'lunge'
                        elif delta == 0 and not ctx['prim']:
                            kind = 'primary'
                            ctx['prim'] = True
                        else:
                            kind = 'extra'
                        _row(self, t, ctx, kind, pre[0], pre[1], st)
                        if isinstance(dd, dict):
                            dd['_rec'] = True
                except Exception as e:
                    _dc_err(e)
            return res
        prim.__name__ = getattr(orig, '__name__', 'move')
        prim._dc_prim = True
        return prim

    # ── примитив 2: жёсткий сдвиг тела _try_shift_cells (sprint/rage-прыжок и ferals) ──
    def _mk_prim_shift(orig):
        def prim(self, field, step_x, step_y, t, *a, **k):
            ctx = getattr(self, '_dc_ctx', None)
            if ctx is None:
                ctx = _dc_new_ctx(self, 'move_feral' if getattr(self, 'role_type', None) == 'feral' else 'move', t)
                _DC_META['orphans'] += 1
            pre = None
            try:
                pre = _snap(self, field, ctx)
            except Exception as e:
                _dc_err(e)
                pre = None
            res = orig(self, field, step_x, step_y, t, *a, **k)
            if pre is not None:
                try:
                    if ctx['fn'] == 'move_feral':
                        dd = getattr(self, '_dc', None)
                        if isinstance(dd, dict) and dd.get('fn') == 'move_feral' and not dd.get('_rec'):
                            st, kind = dd, 'feral_' + str(dd.get('mode'))
                            dd['_rec'] = True
                        else:
                            st, kind = {'fn': 'move_feral', 'chosen': (int(step_x), int(step_y))}, 'feral_other'
                    else:                                      # sprint/rage-leap из cell 27: вектор прыжка = выбор
                        mm = getattr(self, '_dir_momentum', (0.0, 0.0)) or (0.0, 0.0)
                        st = {'fn': 'move', 'mode': 'leap', 'mu': (float(mm[0]), float(mm[1])),
                              'chosen': (int(step_x), int(step_y))}
                        kind = 'leap'
                    _row(self, t, ctx, kind, pre[0], pre[1], st)
                except Exception as e:
                    _dc_err(e)
            return res
        prim.__name__ = getattr(orig, '__name__', '_try_shift_cells')
        prim._dc_prim = True
        return prim

    # ── ferals: селектор «energy» мог отказать по порогу ДО сдвига (примитив не вызван) → пишем строку-отказ ──
    def _feral_residual(self, field, t, ctx):
        dd = getattr(self, '_dc', None)
        if (isinstance(dd, dict) and dd.get('fn') == 'move_feral' and not dd.get('_rec')
                and bool(getattr(self, '_last_move_attempt', False)) and getattr(self, 'alive', False) and self.cells):
            old = set(self.cells)
            _row(self, t, ctx, 'feral_' + str(dd.get('mode')), old, _fill_pre(self, field, old), dd)
            dd['_rec'] = True

    # ── внешняя обёртка: ТОЛЬКО контекст тика (tid, t0, h/fresh/m0 на входе); строк не пишет ──
    def _mk_outer(fn, orig):
        def wrapped(self, field, t, *a, **k):
            if getattr(self, '_dc_ctx', None) is not None:    # повторная установка / вложенный вызов
                return orig(self, field, t, *a, **k)
            ctx = None
            try:
                ctx = _dc_new_ctx(self, fn, t)
                _DC_META['calls'] += 1
                self._dc = None
            except Exception as e:
                _dc_err(e)
            self._dc_ctx = ctx
            try:
                return orig(self, field, t, *a, **k)
            finally:
                try:
                    if ctx is not None and fn == 'move_feral' and ctx['live']:
                        _feral_residual(self, field, t, ctx)
                except Exception as e:
                    _dc_err(e)
                self._dc_ctx = None
        wrapped.__name__ = fn
        wrapped._dc_outer = True
        return wrapped

    def _owner(name):
        own = None
        for k in P.__mro__:                                    # потомок → база; берём самый БАЗОВЫЙ, где метод определён
            if name in k.__dict__:
                own = k
        return own

    done = []
    for name, mk in (('move', _mk_prim_move), ('_try_shift_cells', _mk_prim_shift)):
        own = _owner(name)
        if own is None:
            print(f"[DIRCHAIN] ⚠️ примитив {name} не найден в MRO Pattern — строки по нему писаться не будут")
            continue
        f = own.__dict__[name]
        if not getattr(f, '_dc_prim', False):
            setattr(own, name, mk(f))
            done.append(f"{own.__name__}.{name}")
    for name in ('move', 'move_feral'):
        f = getattr(P, name, None)
        if f is not None and not _has_outer(name):
            setattr(P, name, _mk_outer(name, f))
            done.append(f"{P.__name__}.{name}[контекст тика]")
    P._dc_installed = True
    if done:
        print("[DIRCHAIN] трейс установлен (read-only, пишет _DC_LOG): " + ', '.join(done))
    else:
        print("[DIRCHAIN] трейс уже установлен (новый, обёртка тика найдена в MRO) — повторно не ставлю")
    return True


# ───────────────────────── анализ ─────────────────────────
def _dc_cols(L):
    cols = list(zip(*L))
    out = {}
    for c in _DC_COLS:
        v = cols[_DCI[c]]
        if c in _DC_STR_COLS:
            out[c] = np.array([('' if x is None else str(x)) for x in v], dtype=object)
        elif c in ('ok', 'nodir', 'fresh'):
            out[c] = np.array([bool(x) for x in v])
        else:
            out[c] = np.array([np.nan if x is None else float(x) for x in v], dtype=np.float64)
    out['kid'] = np.array([_DC_KIND_ID.get(k, 15) for k in out['kind']], dtype=np.int64)
    _, inv, cnt = np.unique(out['tid'], return_inverse=True, return_counts=True)
    out['ncalls'] = cnt[inv.reshape(-1)].astype(np.float64)   # сколько строк-примитивов в этом вызове move()/move_feral()
    return out


def _dc_cos(ax, ay, bx, by):
    na, nb = np.hypot(ax, ay), np.hypot(bx, by)
    with np.errstate(all='ignore'):
        c = (ax * bx + ay * by) / (na * nb)
    c = np.where((na < 1e-9) | (nb < 1e-9), np.nan, c)
    return c


def _dc_mean(x):
    x = x[np.isfinite(x)]
    return (float(x.mean()), int(x.size)) if x.size else (float('nan'), 0)


def _dc_med(x):
    x = x[np.isfinite(x)]
    return float(np.median(x)) if x.size else float('nan')


def _dc_fmt(v, nd=2):
    return 'n/a' if v is None or (isinstance(v, float) and not math.isfinite(v)) else f"{v:.{nd}f}"


def _dc_pct(k, n):
    return 'n/a' if n <= 0 else f"{100.0 * k / n:.0f}%"


def _dc_rank(x):
    o = np.argsort(x, kind='mergesort')
    r = np.empty(len(x), dtype=np.float64)
    r[o] = np.arange(len(x))
    return r


def _dc_spearman(x, y):
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 30:
        return float('nan'), int(m.sum())
    rx, ry = _dc_rank(x[m]), _dc_rank(y[m])
    if rx.std() < 1e-12 or ry.std() < 1e-12:
        return float('nan'), int(m.sum())
    return float(np.corrcoef(rx, ry)[0, 1]), int(m.sum())


def _dc_strata(A, m):
    """Ключ страты для записей по маске m: агент × корзина плотности (до шага) × вид вызова × роль."""
    fb = np.digitize(A['fill'][m], _DC_NULL_EDGES[1:-1]).astype(np.int64)
    return ((A['pid'][m].astype(np.int64) * 4 + fb) * 16 + A['kid'][m]) * 4 + A['role'][m].astype(np.int64)


def _dc_strat_shuffle(sid, rng):
    """Индекс партнёра: перестановка ВНУТРИ каждой страты, без неподвижных точек (каждая страта ≥ 2 записей)."""
    n = sid.size
    order = np.lexsort((rng.random(n), sid))           # сначала страта, внутри — случайный порядок
    s = sid[order]
    new = np.empty(n, dtype=bool)
    new[0] = True
    new[1:] = s[1:] != s[:-1]
    sidx = np.cumsum(new) - 1
    starts = np.flatnonzero(new)
    cnt = np.diff(np.append(starts, n))
    r = 1 + (rng.random(cnt.size) * (cnt - 1)).astype(np.int64)       # циклический сдвиг 1..cnt-1 → партнёр ≠ сам
    r = np.minimum(r, cnt - 1)
    rank = np.arange(n) - starts[sidx]
    partner_sorted = order[starts[sidx] + (rank + r[sidx]) % cnt[sidx]]
    out = np.empty(n, dtype=np.int64)
    out[order] = partner_sorted
    return out


def _dc_strat_null(ax, ay, bx, by, sid, grp=None, absval=False, k=None, seed=7, min_stratum=_DC_MIN_STRATUM):
    """Средний cos(A, B) и нуль: B перемешивается ТОЛЬКО внутри страт sid (агент × плотность × вид вызова × роль).
    Страты < min_stratum записей отбрасываются и из наблюдения, и из нуля (иначе перестановка почти тождественна).
    grp (обычно pid): считается ещё среднее по агентам (среднее средних) — и для наблюдения, и для нуля."""
    m = (np.isfinite(ax) & np.isfinite(ay) & np.isfinite(bx) & np.isfinite(by)
         & (np.hypot(ax, ay) > 1e-9) & (np.hypot(bx, by) > 1e-9))
    n_fin = int(m.sum())
    if n_fin < 30:
        return None
    ax, ay, bx, by, sid = ax[m], ay[m], bx[m], by[m], sid[m]
    grp = sid if grp is None else grp[m]
    _, inv, cnt = np.unique(sid, return_inverse=True, return_counts=True)
    keep = cnt[inv.reshape(-1)] >= min_stratum
    n = int(keep.sum())
    if n < 30:
        return None
    ax, ay, bx, by, sid, grp = ax[keep], ay[keep], bx[keep], by[keep], sid[keep], grp[keep]
    na, nb = np.hypot(ax, ay), np.hypot(bx, by)
    a = np.stack([ax / na, ay / na], axis=1)
    b = np.stack([bx / nb, by / nb], axis=1)
    c0 = (a * b).sum(axis=1)
    if absval:
        c0 = np.abs(c0)
    _, ginv, gcnt = np.unique(grp, return_inverse=True, return_counts=True)
    ginv = ginv.reshape(-1)
    _, sinv = np.unique(sid, return_inverse=True)
    sinv = sinv.reshape(-1)
    if k is None:
        k = 200 if n <= 50000 else (100 if n <= 200000 else 40)
    rng = np.random.default_rng(seed)
    nv, nva = [], []
    for _ in range(k):
        p = _dc_strat_shuffle(sinv, rng)
        c = (a * b[p]).sum(axis=1)
        if absval:
            c = np.abs(c)
        nv.append(float(c.mean()))
        nva.append(float((np.bincount(ginv, weights=c) / gcnt).mean()))
    return {'n': n, 'n_all': n_fin, 'n_drop': n_fin - n, 'strata': int(sinv.max()) + 1, 'agents': int(gcnt.size),
            'obs': float(c0.mean()), 'obs_agent': float((np.bincount(ginv, weights=c0) / gcnt).mean()),
            'null': float(np.mean(nv)), 'null_agent': float(np.mean(nva)), 'null_sd': float(np.std(nv)), 'k': int(k)}


def _dc_axis_stats(A, mask, vx, vy, absval=True):
    """|cos(axis, V)|: по записям и по агентам; нуль = перемешивание V внутри страт (агент × плотность × вид вызова × роль)."""
    m = (mask & np.isfinite(A['ax']) & np.isfinite(A['ay']) & np.isfinite(vx) & np.isfinite(vy)
         & (np.hypot(vx, vy) > 1e-9))
    if int(m.sum()) < 30:
        return None
    return _dc_strat_null(A['ax'][m], A['ay'][m], vx[m], vy[m], _dc_strata(A, m), grp=A['pid'][m],
                          absval=absval, seed=11)


def _dirchain_report(dump_path=None, raw_path=None, min_n=30):
    L = list(_DC_LOG)
    print("\n" + "=" * 78)
    print("[DIRCHAIN] ЦЕПОЧКА НАПРАВЛЕНИЯ ПО ВЫЗОВАМ: импульс → выбор → тело (read-only; одна строка = один вызов примитива)")
    print("=" * 78)
    if not L:
        print(f"  UNKNOWN: журнал пуст (вызовов обёртки тика={_DC_META['calls']}, попыток={_DC_META['attempts']}, "
              f"ошибок трейса={_DC_META['err']} {_DC_META['err_last']}). Трейс не установлен или не вызывался — "
              f"ответы на вопросы 1–3 невозможны. Это НЕ PASS.")
        return None
    A = _dc_cols(L)
    N = len(L)
    kind = A['kind']
    mv = (A['role'] != 1) & (A['fn'] == 0)       # обычные агенты, ветка move()
    fe = (A['fn'] == 1)                          # ferals, ветка move_feral()
    has_sel = np.isfinite(A['mux']) | A['nodir']
    prim = mv & (kind == 'primary')              # основной шаг тика (селектор, первый вызов)
    S = prim & has_sel
    res = {'n_records': N, 'n_move': int(mv.sum()), 'n_feral': int(fe.sum())}
    kinds_n = {kd: int((kind == kd).sum()) for kd in _DC_KIND_ORDER if int((kind == kd).sum())}
    res['kinds'] = kinds_n
    u_t, c_t = np.unique(A['tid'][mv], return_counts=True)
    multi = float((c_t > 1).mean()) if u_t.size else float('nan')
    print(f"  записей (вызовов-примитивов с попыткой): {N}  | ветка move()={int(mv.sum())}  ветка move_feral()={int(fe.sum())}"
          f"  | вызовов обёртки тика={_DC_META['calls']} | вне обёртки={_DC_META['orphans']} | ошибок трейса={_DC_META['err']}"
          f" | срезано по капу={_DC_META['cap_hit']}")
    print(f"  по видам вызова: " + ', '.join(f"{k}={v}" for k, v in kinds_n.items()))
    print(f"  ПОКРЫТИЕ: селектор записан у {_dc_pct(int(S.sum()), int(prim.sum()))} основных шагов (primary); "
          f"cmd['h'] свежий у {_dc_pct(int((prim & A['fresh']).sum()), int(prim.sum()))}; "
          f"тиков с несколькими вызовами move-примитивов: {_dc_pct(int((c_t > 1).sum()), int(u_t.size))} (n={int(u_t.size)})")
    print("  КАЖДЫЙ вызов — отдельная строка: селекторные поля (mu/bd/chosen/blk/sc), ok и сдвиг центра берутся из ОДНОГО вызова "
          "(центр тела до/после именно этого вызова). h/fresh/m0 — снимок на входе тика (общий для вызовов тика); "
          "mu — то, что увидел селектор этого вызова. jump/lunge определяются по смещению t (+7919/+6007), leap — _try_shift_cells "
          "из sprint/rage.")
    print(f"  local_fill = доля занятых клеток в диске R=6 вокруг каждой клетки тела, усреднённая по телу; считается ДО шага "
          f"(не зависит от исхода шага); fill_oth — без своих клеток.")
    print(f"  НУЛЬ: перемешивание B внутри страт «агент × плотность(<10% / 10–20% / ≥20%) × вид вызова × роль», без самопар; "
          f"страты < {_DC_MIN_STRATUM} записей отброшены из наблюдения и из нуля («вне страт»). ±σ — разброс среднего нуля по перестановкам.")
    if int(S.sum()) < min_n:
        print(f"  UNKNOWN: селекторных записей {int(S.sum())} < {min_n} — вопрос 1 и часть вопроса 3 не оцениваются.")

    # ───── Q1 ─────
    print("\n  ── [Q1] ЦЕПОЧКА НАПРАВЛЕНИЯ (основной шаг move(), kind=primary, обычные агенты; ferals и доп. вызовы — отдельно) ──")
    print("  mu = _dir_momentum, КОТОРЫЙ ВИДЕЛ селектор (после подмешивания cmd['h']); bd = _best_direction (восприятие, cell 6);")
    print("  chosen = шаг, выбранный селектором этого вызова (после «уступки импульсу»); actual = реальный сдвиг центра в ЭТОМ вызове.")
    ok = A['ok']
    okS = S & ok

    def _line(name, a, b, mask):
        m = mask & np.isfinite(a[0]) & np.isfinite(a[1]) & np.isfinite(b[0]) & np.isfinite(b[1])
        c = _dc_cos(a[0][m], a[1][m], b[0][m], b[1][m])
        mean, n = _dc_mean(c)
        if n < min_n:
            print(f"    {name:<34} UNKNOWN (n={n})")
            return None
        sn = _dc_strat_null(a[0][m], a[1][m], b[0][m], b[1][m], _dc_strata(A, m), grp=A['pid'][m])
        if sn is None:
            print(f"    {name:<34} cos̄={mean:+.2f}  нуль UNKNOWN (мало записей в стратах)  n={n}")
            return {'cos': mean, 'null': float('nan'), 'n': n}
        ang = np.degrees(np.arccos(np.clip(c[np.isfinite(c)], -1, 1)))
        drop = f" (+{sn['n_drop']} вне страт)" if sn['n_drop'] else ''
        print(f"    {name:<34} cos̄={sn['obs']:+.2f} (нуль страт. {sn['null']:+.2f}±{sn['null_sd']:.2f}; "
              f"по агентам {sn['obs_agent']:+.2f} vs {sn['null_agent']:+.2f})  угол медиана={np.median(ang):.0f}°  "
              f"доля <45°={100 * np.mean(ang < 45):.0f}%  n={sn['n']}{drop}")
        return {'cos': sn['obs'], 'null': sn['null'], 'null_sd': sn['null_sd'], 'cos_agent': sn['obs_agent'],
                'null_agent': sn['null_agent'], 'n': sn['n'], 'n_drop': sn['n_drop']}
    mu = (A['mux'], A['muy']); bd = (A['bdx'], A['bdy']); chs = (A['cx'], A['cy'])
    act = (A['sx'], A['sy']); hh = (A['hx'], A['hy'])
    res['angles'] = {
        'h_mu': _line('cmd.h ↔ momentum', hh, mu, S),
        'mu_bd': _line('momentum ↔ best_direction', mu, bd, S),
        'mu_chosen': _line('momentum ↔ chosen (выбор селектора)', mu, chs, S),
        'bd_chosen': _line('best_direction ↔ chosen', bd, chs, S),
        'bd_act': _line('best_direction ↔ actual (ok)', bd, act, okS),
        'chosen_act': _line('chosen ↔ actual (ok; ≈1 = проверка синхронизации)', chs, act, okS),
        'mu_act': _line('momentum ↔ actual (ok)', mu, act, okS),
    }
    ca = res['angles'].get('chosen_act')
    if ca is not None and ca['cos'] < 0.95:
        print("    ⚠️ chosen↔actual < 0.95 у успешных шагов: журнал рассинхронизирован (или другой сдвиг в тот же вызов) — выводы Q1 ненадёжны")
    print("    NB: actual определён только у успешных шагов; у неудач сдвиг = 0 и в cos не входит (см. разложение ниже).")

    # разложение попыток
    reason = A['reason']
    mdir_x = np.where(np.isfinite(A['mux']), np.round(A['mux']), np.nan)
    mdir_y = np.where(np.isfinite(A['muy']), np.round(A['muy']), np.nan)
    mom0 = (mdir_x == 0) & (mdir_y == 0)
    same = (A['cx'] == mdir_x) & (A['cy'] == mdir_y)
    N_ = len(kind)
    cat = np.full(N_, '', dtype=object)
    cat[S & A['nodir']] = '1 нет свободных направлений (все 8 заблокированы)'
    rest = S & ~A['nodir']
    cat[rest & (reason == 'score_threshold')] = '2 лучший шаг ниже порога оценки (движение не началось)'
    rest2 = rest & (cat == '')
    cat[rest2 & mom0] = '3 импульса нет (округляется в 0)'
    rest3 = rest2 & (cat == '')
    cat[rest3 & same & ok] = '4 импульс сохранён и реализован'
    cat[rest3 & same & ~ok] = '5 импульс выбран, но шаг не удался (occupied/wall)'
    cat[rest3 & ~same & (A['mom_blk'] == 1)] = '6 направление импульса ЗАБЛОКИРОВАНО → выбрано другое (H2)'
    cat[rest3 & ~same & (A['mom_blk'] == 0)] = '7 направление импульса СВОБОДНО, но селектор выбрал другое (H1)'
    nS = int(S.sum())
    print(f"\n    Судьба направления в каждой попытке (n={nS}; взаимоисключающие категории):")
    cats = sorted(set(c for c in cat[S] if c))
    cnt = {c: int((cat == c).sum()) for c in cats}
    for c in cats:
        print(f"      {c:<62} {cnt[c]:>6}  {_dc_pct(cnt[c], nS):>4}")
    res['q1_categories'] = {c: cnt[c] for c in cats}
    res['q1_n'] = nS

    # ferals
    if int(fe.sum()):
        print("\n    ferals (move_feral): по коду cell 5 выбор шага идёт по добыче/энергии в 4 направлениях и НЕ читает "
              "_dir_momentum/_best_direction → «импульс» у ferals не драйвер движения. Каждый сдвиг (prey / energy) — своя запись.")
        fa = fe & ok
        m0 = (A['m0x'], A['m0y'])
        c = _dc_cos(m0[0][fa], m0[1][fa], A['sx'][fa], A['sy'][fa])
        mean, n = _dc_mean(c)
        sn = (_dc_strat_null(m0[0][fa], m0[1][fa], A['sx'][fa], A['sy'][fa], _dc_strata(A, fa), grp=A['pid'][fa])
              if n >= min_n else None)
        nt = f"; нуль страт. {_dc_fmt(sn['null'])}±{sn['null_sd']:.2f}, obs в стратах {_dc_fmt(sn['obs'])}" if sn else ''
        modes = {k: int((A['mode'][fe] == k).sum()) for k in ('prey', 'energy', '')}
        print(f"    попыток={int(fe.sum())} успех={_dc_pct(int(fa.sum()), int(fe.sum()))} режимы={modes}; "
              f"cos(momentum на входе тика, actual)={_dc_fmt(mean)} (n={n}{nt}; справочно, причинной связи нет)")
        res['feral'] = {'attempts': int(fe.sum()), 'ok': int(fa.sum()), 'cos_mom_act': mean, 'modes': modes,
                        'cos_null': (sn['null'] if sn else None)}

    # ───── Q1b: вызовы внутри тика ─────
    print("\n  ── [Q1b] ВЫЗОВЫ ВНУТРИ ТИКА: jump / lunge / leap и пр. — отдельные записи, поля вызовов НЕ смешаны ──")
    print(f"    {'kind':<13}{'n':>7}{'ok%':>6}{'ctr_d|ok':>9}   cos(chosen,actual | ok) [нуль]   cos(mu,chosen) [нуль]")

    def _cs(a, b, mask):
        m = mask & np.isfinite(a[0]) & np.isfinite(a[1]) & np.isfinite(b[0]) & np.isfinite(b[1])
        if int(m.sum()) < min_n:
            return None
        return _dc_strat_null(a[0][m], a[1][m], b[0][m], b[1][m], _dc_strata(A, m), grp=A['pid'][m])
    fmt_sn = lambda s: ('UNKNOWN' if s is None else f"{s['obs']:+.2f} [{s['null']:+.2f}] n={s['n']}")
    q1b = {}
    for kd in _DC_KIND_ORDER:
        b = (kind == kd)
        n = int(b.sum())
        if n == 0:
            continue
        if n < min_n:
            print(f"    {kd:<13}{n:>7}  UNKNOWN (n<{min_n})")
            continue
        bo = b & ok
        s_ca = _cs(chs, act, bo)
        s_mc = _cs(mu, chs, b) if not kd.startswith('feral') else None
        ctro = float(np.mean(A['ctr_d'][bo])) if bo.sum() else float('nan')
        print(f"    {kd:<13}{n:>7}{100 * float(ok[b].mean()):>5.0f}%{_dc_fmt(ctro):>9}   {fmt_sn(s_ca):<30}  "
              f"{('—' if kd.startswith('feral') else fmt_sn(s_mc))}")
        q1b[kd] = {'n': n, 'ok': float(ok[b].mean()), 'chosen_act': (s_ca or {}).get('obs'),
                   'chosen_act_null': (s_ca or {}).get('null'), 'mu_chosen': (s_mc or {}).get('obs'),
                   'mu_chosen_null': (s_mc or {}).get('null')}
    res['q1b'] = q1b

    # ───── Q3 (до Q2: нужны корзины) ─────
    print("\n  ── [Q3] ПРОСТРАНСТВЕННОЕ СОПРОТИВЛЕНИЕ: по local_fill (основной шаг move(), обычные агенты) ──")
    fill = A['fill']
    edges = _DC_FILL_EDGES
    hdr = (f"    {'fill':<9}{'n':>6}{'ok%':>6}{'blk/8':>7}{'free':>6}{'margin':>8}{'P(mom заблок)':>14}"
           f"{'P(выбран=mom)':>14}{'cos(mu,chos)':>13}{'cos(mu,act)':>12}{'ctr_d':>7}{'ctr|ok':>7}  отказы")
    print(hdr)
    rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        b = S & (fill >= lo) & (fill < hi)
        n = int(b.sum())
        lab = f"{lo * 100:.0f}-{min(hi, 1) * 100:.0f}%"
        if n < min_n:
            print(f"    {lab:<9}{n:>6}  UNKNOWN (n<{min_n})")
            rows.append({'bin': lab, 'n': n})
            continue
        okr = float(ok[b].mean())
        blk = float(np.nanmean(A['blk_n'][b]) / 8.0)
        free = float(np.nanmean(A['n_free'][b]))
        mg = _dc_med((A['sc1'] - A['sc2'])[b])
        mdx = b & ~mom0 & np.isfinite(mdir_x)
        pmb = float(np.mean(A['mom_blk'][mdx] == 1)) if mdx.sum() >= min_n else float('nan')
        pks = float(np.mean(same[mdx])) if mdx.sum() >= min_n else float('nan')
        cmc, _ = _dc_mean(_dc_cos(A['mux'][b], A['muy'][b], A['cx'][b], A['cy'][b]))
        bo = b & ok
        cma, nma = _dc_mean(_dc_cos(A['mux'][bo], A['muy'][bo], A['sx'][bo], A['sy'][bo]))
        ctr = float(np.mean(A['ctr_d'][b]))
        ctro = float(np.mean(A['ctr_d'][bo])) if bo.sum() else float('nan')
        rs = {k: int(((reason == k) & b).sum()) for k in ('occupied', 'score_threshold', 'no_direction', 'wall')}
        rtxt = ' '.join(f"{k[:5]}={_dc_pct(v, n)}" for k, v in rs.items() if v)
        print(f"    {lab:<9}{n:>6}{100 * okr:>5.0f}%{blk:>7.2f}{free:>6.1f}{mg:>8.2f}{_dc_fmt(pmb):>14}{_dc_fmt(pks):>14}"
              f"{_dc_fmt(cmc):>13}{_dc_fmt(cma):>12}{ctr:>7.2f}{_dc_fmt(ctro):>7}  {rtxt}")
        real = float(np.mean((cat[b] == '4 импульс сохранён и реализован')))
        rows.append({'bin': lab, 'n': n, 'ok': okr, 'blk': blk, 'free': free, 'margin': mg, 'p_mom_blk': pmb,
                     'p_chosen_eq_mom': pks, 'cos_mu_chosen': cmc, 'cos_mu_act': cma, 'ctr_d': ctr, 'realized': real,
                     'reasons': rs})
    res['q3_bins'] = rows
    print("    blk/8 = доля из 8 направлений, заблокированных соседом; margin = лучший − следующий кандидат (медиана); "
          "P(mom заблок) и P(выбран=mom) — среди попыток с ненулевым импульсом.")
    good = [r for r in rows if 'ok' in r]
    print("    Корреляции с local_fill (Спирмен, по попыткам; fill коррелирует с временем/популяцией — это не причинность):")
    for nm, v in (('успех', ok.astype(float)), ('blk_n', A['blk_n']), ('margin', A['sc1'] - A['sc2']),
                  ('импульс сохранён и реализован', (cat == '4 импульс сохранён и реализован').astype(float)),
                  ('ctr_d', A['ctr_d'])):
        r_, n_ = _dc_spearman(np.where(S, fill, np.nan), np.where(S, v, np.nan))
        print(f"      fill ~ {nm:<30} ρ={_dc_fmt(r_)} (n={n_})")
        res.setdefault('q3_spearman', {})[nm] = r_
    knee = None
    if len(good) >= 2:
        base = good[0]['realized']
        for r in good[1:]:
            if base > 0 and r['realized'] < 0.5 * base:
                knee = r['bin']
                break
        print(f"    «излом»: доля «импульс сохранён и реализован» в самой редкой корзине={good[0]['realized']:.2f} ({good[0]['bin']}); "
              f"падение вдвое {'в корзине ' + knee if knee else 'НЕ найдено в наблюдаемом диапазоне'}.")
        if all(g['bin'] != '0-5%' for g in good):
            print(f"    ⚠️ самой разреженной корзины (<{good[0]['bin'].split('-')[0]}) в данных нет: «разреженное» сравнивается с «плотным» только в наблюдаемом диапазоне")
    res['q3_knee'] = knee
    if int(fe.sum()):
        print("    ferals по local_fill (ok% / ctr_d̄ / доля prey-режима; каждая запись = один сдвиг):")
        for lo, hi in zip(edges[:-1], edges[1:]):
            b = fe & (fill >= lo) & (fill < hi)
            if int(b.sum()) >= min_n:
                print(f"      {lo * 100:.0f}-{min(hi, 1) * 100:.0f}%: n={int(b.sum())} ok={_dc_pct(int(ok[b].sum()), int(b.sum()))} "
                      f"ctr_d̄={float(A['ctr_d'][b].mean()):.2f} prey={_dc_pct(int((A['mode'][b] == 'prey').sum()), int(b.sum()))}")

    # ───── Q3b: то же по fill_oth — плотность БЕЗ собственных клеток (только измерение, поведение не меняется) ─────
    print("\n  ── [Q3b] ЧУЖАЯ ЗАПОЛНЕННОСТЬ: по fill_oth = доля клеток диска R=6, занятых ДРУГИМИ агентами, без своих клеток "
          "(основной шаг move(), обычные агенты) ──")
    fo = A['fill_oth']
    q3b = {}
    if int(S.sum()) < min_n:
        print(f"  Q3b: UNKNOWN — селекторных записей {int(S.sum())} < {min_n}.")
    else:
        own_part = (fill - fo)[S]
        own_part = own_part[np.isfinite(own_part)]
        mf = float(np.nanmean(fill[S]))
        mo = float(np.nanmean(fo[S]))
        share_own = (1.0 - mo / mf) if mf > 1e-12 else float('nan')
        print(f"    средние по основным шагам: fill={mf:.3f}  fill_oth={mo:.3f}  своя часть (fill−fill_oth)={float(own_part.mean()):.3f} "
              f"(медиана {float(np.median(own_part)):.3f}) → доля своих клеток в local_fill ≈ {_dc_fmt(share_own)}")
        q3b['mean_fill'], q3b['mean_fill_oth'], q3b['own_share'] = mf, mo, share_own
        edges_o = _DC_FILL_OTH_EDGES
        hdr_o = (f"    {'fill_oth':<9}{'n':>6}{'ok%':>6}{'blk/8':>7}{'free':>6}{'margin':>8}{'P(mom заблок)':>14}"
                 f"{'P(выбран=mom)':>14}{'cos(mu,chos)':>13}{'cos(mu,act)':>12}{'ctr_d':>7}{'ctr|ok':>7}  отказы")
        print(hdr_o)
        rows_o = []
        for lo, hi in zip(edges_o[:-1], edges_o[1:]):
            b = S & (fo >= lo) & (fo < hi)
            n = int(b.sum())
            lab = f"{lo * 100:.0f}-{min(hi, 1) * 100:.0f}%"
            if n < min_n:
                print(f"    {lab:<9}{n:>6}  UNKNOWN (n<{min_n})")
                rows_o.append({'bin': lab, 'n': n})
                continue
            okr = float(ok[b].mean())
            blk = float(np.nanmean(A['blk_n'][b]) / 8.0)
            free = float(np.nanmean(A['n_free'][b]))
            mg = _dc_med((A['sc1'] - A['sc2'])[b])
            mdx = b & ~mom0 & np.isfinite(mdir_x)
            pmb = float(np.mean(A['mom_blk'][mdx] == 1)) if mdx.sum() >= min_n else float('nan')
            pks = float(np.mean(same[mdx])) if mdx.sum() >= min_n else float('nan')
            cmc, _ = _dc_mean(_dc_cos(A['mux'][b], A['muy'][b], A['cx'][b], A['cy'][b]))
            bo = b & ok
            cma, _ = _dc_mean(_dc_cos(A['mux'][bo], A['muy'][bo], A['sx'][bo], A['sy'][bo]))
            ctr = float(np.mean(A['ctr_d'][b]))
            ctro = float(np.mean(A['ctr_d'][bo])) if bo.sum() else float('nan')
            rs = {k: int(((reason == k) & b).sum()) for k in ('occupied', 'score_threshold', 'no_direction', 'wall')}
            rtxt = ' '.join(f"{k[:5]}={_dc_pct(v, n)}" for k, v in rs.items() if v)
            print(f"    {lab:<9}{n:>6}{100 * okr:>5.0f}%{blk:>7.2f}{free:>6.1f}{mg:>8.2f}{_dc_fmt(pmb):>14}{_dc_fmt(pks):>14}"
                  f"{_dc_fmt(cmc):>13}{_dc_fmt(cma):>12}{ctr:>7.2f}{_dc_fmt(ctro):>7}  {rtxt}")
            real = float(np.mean((cat[b] == '4 импульс сохранён и реализован')))
            rows_o.append({'bin': lab, 'n': n, 'ok': okr, 'blk': blk, 'free': free, 'margin': mg, 'p_mom_blk': pmb,
                           'p_chosen_eq_mom': pks, 'cos_mu_chosen': cmc, 'cos_mu_act': cma, 'ctr_d': ctr,
                           'realized': real, 'reasons': rs})
        q3b['bins'] = rows_o
        print("    те же столбцы, что в [Q3]; корзины fill_oth мельче внизу (нет «чужих рядом» ≈ 0-2%).")
        print("    Корреляции (Спирмен, по попыткам): fill_oth против local_fill; в скобках — тот же показатель для local_fill. "
              "Не причинность (оба confounded со временем/популяцией).")
        sp = {}
        for nm, v in (('успех', ok.astype(float)), ('blk_n', A['blk_n']), ('margin', A['sc1'] - A['sc2']),
                      ('импульс сохранён и реализован', (cat == '4 импульс сохранён и реализован').astype(float)),
                      ('ctr_d', A['ctr_d'])):
            r_o, n_o = _dc_spearman(np.where(S, fo, np.nan), np.where(S, v, np.nan))
            r_f, _n = _dc_spearman(np.where(S, fill, np.nan), np.where(S, v, np.nan))
            print(f"      fill_oth ~ {nm:<30} ρ={_dc_fmt(r_o)} (fill: {_dc_fmt(r_f)})  n={n_o}")
            sp[nm] = {'fill_oth': r_o, 'fill': r_f}
        r_ff, _n = _dc_spearman(np.where(S, fo, np.nan), np.where(S, fill, np.nan))
        print(f"      fill_oth ~ fill                           ρ={_dc_fmt(r_ff)}")
        q3b['spearman'] = sp
        # При одинаковом local_fill: кто мешает — свои клетки или чужие? Внутри корзины fill делим по медиане fill_oth.
        print("    При ОДИНАКОВОМ local_fill (корзина) — делим попытки пополам по fill_oth; если «чужих больше» заметно хуже при том же fill, "
              "мешают именно другие агенты, а не собственное тело:")
        print(f"      {'fill':<9}{'половина':<14}{'n':>6}{'fill_oth̄':>10}{'ok%':>6}{'blk/8':>7}{'free':>6}")
        split_rows = []
        for lo, hi in ((0.0, 0.15), (0.15, 0.30), (0.30, 1.01)):
            bf = S & (fill >= lo) & (fill < hi) & np.isfinite(fo)
            if int(bf.sum()) < 4 * min_n:
                print(f"      {lo * 100:.0f}-{min(hi, 1) * 100:.0f}%   UNKNOWN (n={int(bf.sum())} < {4 * min_n})")
                continue
            med = float(np.median(fo[bf]))
            for nm, bh in (('чужих меньше', bf & (fo <= med)), ('чужих больше', bf & (fo > med))):
                n = int(bh.sum())
                if n < min_n:
                    print(f"      {lo * 100:.0f}-{min(hi, 1) * 100:.0f}%   {nm:<14}{n:>6}  UNKNOWN")
                    continue
                okh, blkh = float(ok[bh].mean()), float(np.nanmean(A['blk_n'][bh]) / 8.0)
                frh, foh = float(np.nanmean(A['n_free'][bh])), float(np.nanmean(fo[bh]))
                print(f"      {lo * 100:.0f}-{min(hi, 1) * 100:.0f}%   {nm:<14}{n:>6}{foh:>10.3f}{100 * okh:>5.0f}%{blkh:>7.2f}{frh:>6.1f}")
                split_rows.append({'fill_bin': f"{lo * 100:.0f}-{min(hi, 1) * 100:.0f}%", 'half': nm, 'n': n,
                                   'fill_oth': foh, 'ok': okh, 'blk': blkh, 'free': frh})
        q3b['same_fill_split'] = split_rows
        rb_o, rb_f = sp['blk_n']['fill_oth'], sp['blk_n']['fill']
        print(f"  Q3b: ρ(blk_n) с fill_oth = {_dc_fmt(rb_o)} против {_dc_fmt(rb_f)} с local_fill; "
              f"своих клеток в local_fill ≈ {_dc_fmt(share_own)}. "
              + ("Чужая плотность объясняет блокировки не хуже общей." if np.isfinite(rb_o) and np.isfinite(rb_f) and rb_o >= rb_f - 0.02
                 else "Чужая плотность объясняет блокировки заметно слабее общей — часть эффекта даёт собственное тело или иной фактор."))
    res['q3b'] = q3b

    # ───── Q2 ─────
    print("\n  ── [Q2] ГЕОМЕТРИЯ ТЕЛА: ось и сдвиг (rod/dumbbell отдельно; ball — без оси) ──")
    print("  Случайный ориентир |cos| для равномерно случайных углов = 2/π ≈ 0.637; шаги дискретны (8 направлений), поэтому "
          "честный нуль — ПЕРЕМЕШИВАНИЕ шагов внутри страт (агент × плотность × вид вызова × роль) — колонка «нуль».")
    print("  Ось считается по телу ДО шага именно этого вызова (наибольшая компонента), форма/elongation — из последнего замера SOMA (см. возраст). "
          "axis·A — по всем успешным вызовам (primary/jump/lunge/leap), каждый со своей осью и своим сдвигом.")
    q2 = {}
    for grp_name, gm in (('обычные (move)', mv), ('ferals (move_feral)', fe)):
        print(f"\n    {grp_name}:")
        print(f"      {'форма':<10}{'n':>6}{'ok%':>6}{'blk/8':>7}{'elong':>7}{'frag':>6}{'largest':>8}{'возр.замера':>12}"
              f"{'axis·A':>8}{'нуль':>7}{'(по агентам/нуль)':>18}{'axis·mom':>9}{'нуль':>7}  n_ax")
        for sh in ('ball', 'rod', 'dumbbell'):
            b = gm & (A['shape'] == sh)
            n = int(b.sum())
            if n < min_n:
                print(f"      {sh:<10}{n:>6}  UNKNOWN (n<{min_n})")
                continue
            okr = float(ok[b].mean())
            blk = float(np.nanmean(A['blk_n'][b]) / 8.0) if np.isfinite(A['blk_n'][b]).any() else float('nan')
            el = float(np.nanmean(A['elong'][b]))
            fr = float(np.nanmean(A['frag'][b]))
            lf = float(np.nanmean(A['lfrac'][b]))
            sa = _dc_med(A['sd_age'][b])
            if sh == 'ball':
                print(f"      {sh:<10}{n:>6}{100 * okr:>5.0f}%{blk:>7.2f}{el:>7.2f}{fr:>6.2f}{lf:>8.2f}{sa:>12.0f}"
                      f"{'—':>8}{'—':>7}{'—':>18}{'—':>9}{'—':>7}")
                q2[(grp_name, sh)] = {'n': n, 'ok': okr, 'blk': blk}
                continue
            st = _dc_axis_stats(A, b & ok, A['sx'], A['sy'])
            sm = _dc_axis_stats(A, b & S, A['mux'], A['muy']) if grp_name.startswith('обычные') else None
            if st is None:
                print(f"      {sh:<10}{n:>6}{100 * okr:>5.0f}%{blk:>7.2f}{el:>7.2f}{fr:>6.2f}{lf:>8.2f}{sa:>12.0f}  axis·A UNKNOWN (мало шагов с осью / мало записей в стратах)")
                q2[(grp_name, sh)] = {'n': n, 'ok': okr, 'blk': blk, 'axis': None}
                continue
            mtxt = f"{sm['obs']:>9.2f}{sm['null']:>7.2f}" if sm else f"{'—':>9}{'—':>7}"
            agtxt = f"{st['obs_agent']:.2f}/{st['null_agent']:.2f}"
            drop = f" (+{st['n_drop']} вне страт)" if st['n_drop'] else ''
            print(f"      {sh:<10}{n:>6}{100 * okr:>5.0f}%{blk:>7.2f}{el:>7.2f}{fr:>6.2f}{lf:>8.2f}{sa:>12.0f}"
                  f"{st['obs']:>8.2f}{st['null']:>7.2f}{agtxt:>18}{mtxt}  {st['n']}/{st['agents']}аг{drop}")
            q2[(grp_name, sh)] = {'n': n, 'ok': okr, 'blk': blk, 'axis': st, 'axis_mom': sm}
    res['q2'] = {f"{k[0]}|{k[1]}": v for k, v in q2.items()}
    print("\n    Влияет ли форма на реализацию при ОДНОЙ плотности (успех% по форме × local_fill, основной шаг, обычные агенты):")
    ed2 = [0.0, 0.10, 0.15, 0.25, 1.01]
    print(f"      {'fill':<9}" + ''.join(f"{s:>16}" for s in ('ball', 'rod', 'dumbbell')) + f"{'frag=1':>16}{'frag≥2':>16}")
    for lo, hi in zip(ed2[:-1], ed2[1:]):
        cells_ = []
        for sh in ('ball', 'rod', 'dumbbell'):
            b = S & (A['shape'] == sh) & (fill >= lo) & (fill < hi)
            cells_.append(f"{100 * float(ok[b].mean()):.0f}% n={int(b.sum())}" if int(b.sum()) >= min_n else f"UNK n={int(b.sum())}")
        for cond in (np.nan_to_num(A['frag'], nan=1) <= 1, np.nan_to_num(A['frag'], nan=1) >= 2):
            b = S & cond & (fill >= lo) & (fill < hi)
            cells_.append(f"{100 * float(ok[b].mean()):.0f}% n={int(b.sum())}" if int(b.sum()) >= min_n else f"UNK n={int(b.sum())}")
        print(f"      {lo * 100:.0f}-{min(hi, 1) * 100:.0f}%".ljust(15) + ''.join(f"{c:>16}" for c in cells_))
    print("    Код cell 5 move(): тело сдвигается ЖЁСТКО (все клетки на один вектор), вращения нет; форма входит в выбор только через "
          "число заблокированных направлений (периметр). Если axis·A ≈ нуль — ось не управляет направлением шага.")

    # ───── ВЕРДИКТЫ ─────
    print("\n  ── ОТВЕТЫ (эвристика по числам выше; UNKNOWN ≠ PASS) ──")
    if nS < min_n:
        print("  Q1: UNKNOWN — нет синхронных селекторных записей.")
    else:
        share = lambda k: cnt.get(k, 0) / max(1, nS)
        cH2 = share('6 направление импульса ЗАБЛОКИРОВАНО → выбрано другое (H2)') + share('1 нет свободных направлений (все 8 заблокированы)')
        cH1 = share('7 направление импульса СВОБОДНО, но селектор выбрал другое (H1)')
        cThr = share('2 лучший шаг ниже порога оценки (движение не началось)')
        cOK = share('4 импульс сохранён и реализован')
        cNo = share('3 импульса нет (округляется в 0)')
        cKF = share('5 импульс выбран, но шаг не удался (occupied/wall)')
        print(f"  Q1: импульс реализуется целиком в {100 * cOK:.0f}% основных шагов. Потери: селектор уходит от импульса при СВОБОДНОМ пути {100 * cH1:.0f}% (H1), "
              f"импульс упирается в блок {100 * cH2:.0f}% (H2), порог оценки не пускает {100 * cThr:.0f}%, импульса нет {100 * cNo:.0f}%, "
              f"выбран но не удался {100 * cKF:.0f}%.")
        top = max((('H1: селектор отвергает импульс при свободном пути', cH1), ('H2: импульс заблокирован телами/плотностью', cH2),
                   ('порог оценки (move score threshold)', cThr)), key=lambda x: x[1])
        print(f"      главная потеря между импульсом и шагом: {top[0]} ({100 * top[1]:.0f}%).")
        mb = res['angles'].get('mu_bd')
        if mb is not None and np.isfinite(mb['null']):
            print(f"      momentum↔best_direction cos̄={mb['cos']:+.2f} против нуля (страт.) {mb['null']:+.2f}: "
                  f"{'импульс и восприятие согласованы' if mb['cos'] - mb['null'] > 0.2 else 'импульс и восприятие почти не связаны'} — "
                  f"это про конфликт ВНУТРЕННИХ сигналов (H1), а не про тело.")
    q2v = []
    for (g, sh), v in q2.items():
        if sh == 'ball' or not v.get('axis'):
            continue
        d = v['axis']['obs'] - v['axis']['null']
        q2v.append((g, sh, d, v['axis']['null_sd'], v['axis']['agents']))
    if not q2v:
        print("  Q2: UNKNOWN — мало шагов с определённой осью (rod/dumbbell).")
    else:
        for g, sh, d, sd_, ag in q2v:
            thr = max(0.05, 3.0 * sd_)
            verdict = ('ось ВЫРОВНЕНА с движением' if d > thr else 'ось ПЕРПЕНДИКУЛЯРНА движению' if d < -thr
                       else 'ось неотличима от случайной')
            print(f"  Q2: {g} {sh}: axis·A − нуль(страт.) = {d:+.2f} (σнуля={sd_:.3f}, агентов={ag}) → {verdict}.")
        print("      В механике move() поворота тела нет (жёсткий сдвиг); влияет ли форма на реализацию — смотри таблицу «форма × fill» "
              "(разница успеха при той же плотности) и blk/8.")
    if len(good) < 2:
        print("  Q3: UNKNOWN — меньше двух заполненных корзин local_fill.")
    else:
        lo_, hi_ = good[0], good[-1]
        d = {'доля заблокированных направлений': hi_['blk'] - lo_['blk'],
             'P(импульс заблокирован)': (hi_['p_mom_blk'] - lo_['p_mom_blk']) if np.isfinite(hi_['p_mom_blk']) and np.isfinite(lo_['p_mom_blk']) else float('nan'),
             'падение успеха шага': lo_['ok'] - hi_['ok'],
             'падение «выбран=импульс»': (lo_['p_chosen_eq_mom'] - hi_['p_chosen_eq_mom']) if np.isfinite(hi_['p_chosen_eq_mom']) and np.isfinite(lo_['p_chosen_eq_mom']) else float('nan'),
             'падение «импульс реализован»': lo_['realized'] - hi_['realized']}
        d = {k: v for k, v in d.items() if np.isfinite(v)}
        order = sorted(d.items(), key=lambda kv: -abs(kv[1]))
        print(f"  Q3: от {lo_['bin']} к {hi_['bin']}: " + '; '.join(f"{k} {v:+.2f}" for k, v in order)
              + f". Излом: {knee if knee else 'не найден'}. (корзины confounded со временем/размером популяции.)")
    print("\n  ── ЧЕГО Observer всё ещё НЕ видит (UNKNOWN, не PASS) ──")
    print("   • ferals: у move_feral нет селектора по направлению/импульсу — Q1 для них не определён (только режим prey/energy и факт сдвига).")
    print("   • cmd['h'] пишется только когда nav-команда свежая (свежесть выше); при нефреш-команде связь h→momentum UNKNOWN.")
    print("   • форма/elongation — из последнего замера SOMA (возраст в таблице Q2), не пересчитываются в момент шага.")
    print("   • вращение тела и «внутренний импульс до подмешивания cmd.h» в тот же тик не разделены: mu — это то, что увидел селектор вызова; "
          "h/fresh/m0 — снимок на входе тика.")
    print("   • вид вызова jump/lunge определяется по смещению t (+7919/+6007) из cell 27; если эти константы изменятся — такие вызовы уйдут в kind=extra.")
    print("   • нуль учитывает агента, плотность (3 корзины), вид вызова и роль; он НЕ учитывает время внутри прогона и положение на карте — "
          "корзины плотности по-прежнему confounded со временем.")
    if dump_path:
        try:
            with open(dump_path, 'w') as f:
                json.dump(res, f, ensure_ascii=False, default=lambda o: None if isinstance(o, float) and not math.isfinite(o) else str(o), indent=1)
            print(f"[DIRCHAIN] итоги: {dump_path}")
        except Exception as e:
            print(f"[DIRCHAIN] дамп итогов не записан: {e!r}")
    if raw_path:
        try:
            with gzip.open(raw_path, 'wt') as f:
                json.dump({'cols': _DC_COLS, 'rows': L}, f)
            print(f"[DIRCHAIN] сырой журнал ({N} строк): {raw_path}")
        except Exception as e:
            print(f"[DIRCHAIN] сырой журнал не записан: {e!r}")
    return res


try:
    _dc_install()
except Exception as _e:
    print(f"[DIRCHAIN] установка трейса не удалась: {_e!r}")


def _fish4_report(dump_path=None):
    """FISH-4 (Observer, read-only): что рыба делала при обходе — намерение, сторона, поток, ожидание, плавность."""
    import json as _json
    S = dict(_F4S)
    g = lambda k: float(S.get(k, 0.0))
    pct = lambda a, b: (f"{100.0 * a / b:.0f}%" if b >= 1 else "UNKNOWN")
    avg = lambda a, b, nd=2: (f"{a / b:+.{nd}f}" if b >= 1 else "UNKNOWN")
    print("=" * 78)
    print("[FISH4] «РЫБА В ВОДЕ»: память намерения → сторона обхода → поток соседей → подождать → плавность")
    print("=" * 78)
    print(f"  включено: FISH3={getattr(Config, 'FISH3_ENABLE', None)} N={getattr(Config, 'FISH3_BLOCKED_N', None)} "
          f"FREEZE={getattr(Config, 'FISH3_FREEZE_ON_DETOUR', None)} | FISH4={getattr(Config, 'FISH4_ENABLE', None)} "
          f"TTL={getattr(Config, 'FISH4_INTENT_TTL', None)} FLOW_TIE={getattr(Config, 'FISH4_FLOW_TIE', None)} "
          f"WAIT_MAX={getattr(Config, 'FISH4_WAIT_MAX', None)}")
    if g('calls') < 1:
        print("  вызовов move() под FISH-4 не было (FISH4_ENABLE=False или FISH3_ENABLE=False) — отчёт пуст, это UNKNOWN, не PASS.")
        return None
    print(f"  вызовов move()={g('calls'):.0f} (в плотной воде blk≥4: {pct(g('calls_dense'), g('calls'))}) | "
          f"обходов начато={g('detour_start'):.0f} закончено={g('detour_end'):.0f} | вызовов внутри обхода={g('det_calls'):.0f}")
    print("  ── 1. ПАМЯТЬ НАМЕРЕНИЯ ──")
    print(f"    цель навигации как намерение: {g('intent_nav_calls'):.0f} вызовов | передумала (обход дольше TTL): {g('intent_abandoned'):.0f}")
    print(f"    геометрическое сходство курса с сохранённым намерением через 5 тиков: cos̄={avg(g('ret5_sum'), g('ret5_n'))}  "
          f"cos≥0.7 в {pct(g('ret5_ge07'), g('ret5_n'))} случаев (n={g('ret5_n'):.0f}) | "
          f"валидность/достижение цели не проверяется — метрика НЕ для оценки возврата к цели | принудительно возвращали курс: {g('restore_n'):.0f}")
    print("  ── 2. СТОРОНА ОБХОДА (не метаться влево-вправо) ──")
    sk = g('side_kept') + g('side_flip')
    print(f"    выбрана сторона: {g('side_commit'):.0f} раз | шагов обхода на той же стороне: {pct(g('side_kept'), sk)} (n={sk:.0f}) | вынужденных смен стороны: {g('side_flip'):.0f}")
    print("  ── 3. ПЛАВНОСТЬ ОБХОДА ──")
    print(f"    cos(шаг обхода, намерение)̄={avg(g('det_cos_sum'), g('det_calls'))} | поворот ≤45°: {pct(g('det_le45'), g('det_calls'))} | разворот >90°: {pct(g('det_rev'), g('det_calls'))} (n={g('det_calls'):.0f})")
    print("  ── 4. ПОТОК СОСЕДЕЙ (тай-брейк среди почти равных обходов) ──")
    print(f"    поток известен в {g('flow_used'):.0f} обходах; изменил выбор в {g('flow_changed'):.0f} ({pct(g('flow_changed'), g('flow_used'))})")
    print("  ── 5. ПОДОЖДАТЬ ВЕДУЩЕГО (вместо резкого поворота) ──")
    print(f"    тиков ожидания: {g('wait_used'):.0f} | эпизодов, где путь освободился за ≤2 тика: {g('wait_cleared'):.0f} "
          f"(числитель — эпизоды, знаменатель — тики; долю успеха считать нельзя, UNKNOWN) | "
          f"бюджет ожидания исчерпан: {g('wait_exhausted'):.0f} | блокер не уходил (не ждали): {g('wait_skip_not_leaving'):.0f}")
    print("  ── 6. ОБЩАЯ «РЫБНОСТЬ» ДВИЖЕНИЯ (все успешные шаги обычных агентов) ──")
    for sfx, nm in (('_f', 'свободная вода (blk<4)'), ('_d', 'плотная вода (blk≥4)')):
        tn = g('turn_n' + sfx)
        sn = g('straight_n' + sfx)
        print(f"    {nm}: повороты шаг→шаг ≤45°: {pct(g('turn_le45' + sfx), tn)}, развороты >135°: {pct(g('turn_rev' + sfx), tn)} (n={tn:.0f}) | "
              f"прямолинейность 10 шагов (1=по прямой): {(g('straight_sum' + sfx) / sn if sn >= 1 else float('nan')):.2f} (n={sn:.0f})")
    print("  ── ВЕРДИКТ (ориентиры, не приёмка; UNKNOWN ≠ PASS) ──")
    notes = []
    notes.append("возврат к намерению/цели: UNKNOWN (метрика не проверяет, жива ли цель и достигнута ли она)")
    if g('det_calls') >= 30:
        r = g('det_le45') / g('det_calls')
        notes.append(f"плавность обхода: {'OK' if r >= 0.6 else 'слабо'} ({100 * r:.0f}% поворотов ≤45° при цели ≥60%)")
    notes.append("ожидание: UNKNOWN (смешаны эпизоды успеха и тики ожидания; контрольной группы нет)")
    for n_ in notes:
        print("    • " + n_)
    if dump_path:
        try:
            with open(dump_path, 'w', encoding='utf-8') as f:
                _json.dump(S, f, ensure_ascii=False, indent=1)
            print(f"[FISH4] итоги: {dump_path}")
        except Exception as e:
            print(f"[FISH4] дамп не записан: {e!r}")
    return S


def _mem_report(eng=None, dump_path=None):
    """MEM (Observer, read-only): три памяти из схемы + пространственная память SEM + память препятствий,
    и главное — доходит ли память до ТЕЛА (движения/ориентации). Ничего не меняет."""
    import json as _json, types as _types, re as _re
    if eng is None:
        try:
            eng = _PN_LAST.get('eng')
        except NameError:
            eng = None
    if eng is None:
        print("[MEM] нет движка (запусти прогон)")
        return None
    now = int(getattr(eng, 'age', 0))
    alive = [p for p in list(getattr(eng, 'patterns', []) or [])
             if getattr(p, 'alive', False) and getattr(p, 'role_type', None) != 'feral']
    n = len(alive)
    F = dict(_F4S)
    g = lambda k: float(F.get(k, 0.0))
    pct = lambda a, b: (f"{100.0 * a / b:.0f}%" if b >= 1 else "UNKNOWN")
    def q(v):
        v = [float(x) for x in v]
        if not v:
            return "UNKNOWN"
        a = np.asarray(v)
        return f"p50={np.percentile(a, 50):.0f} p90={np.percentile(a, 90):.0f} max={a.max():.0f}"
    R = {'t': now, 'agents': n}
    print("=" * 78)
    print("[MEM] ПАМЯТЬ ↔ ТЕЛО ↔ ПРОСТРАНСТВО (Observer, только наблюдение)")
    print("=" * 78)
    print(f"  тик мира={now}, живых не-ferals={n}")
    if n == 0:
        print("  некого смотреть — UNKNOWN")
        return R
    # ── M1. состояние трёх памятей ──
    print("  ── M1. ЧТО ХРАНИТСЯ СЕЙЧАС (по живым агентам) ──")
    dl = [list(getattr(p, 'dialogue_longterm', []) or []) for p in alive]
    lens = [len(x) for x in dl]
    hor = [now - int(x[0].get('t', now)) for x in dl if x]
    emo_kept = [max(float(m.get('grief_at_moment', 0.0)), float(m.get('grat_at_moment', 0.0))) for x in dl for m in x]
    cap_dl = 200
    print(f"    dialogue_longterm: длина {q(lens)} | у потолка {cap_dl}: {pct(sum(l >= cap_dl for l in lens), n)} | "
          f"горизонт памяти (тиков назад до самой старой): {q(hor)}")
    sn = [getattr(p, '_self_narrative', None) for p in alive]
    sn = [x for x in sn if x is not None]
    sl = [len(x) for x in sn]
    mx = getattr(sn[0], 'maxlen', None) if sn else None
    shor = []
    dup = tot = 0
    kinds = Counter()
    for x in sn:
        xs = list(x)
        if xs:
            try:
                shor.append(now - int(xs[0].get('t', now)))
            except Exception:
                pass
        prev = None
        for e in xs[-200:]:
            k = e.get('event', e.get('type')) if isinstance(e, dict) else None
            kinds[k] += 1
            if prev is not None:
                tot += 1
                dup += int(k == prev)
            prev = k
    print(f"    self_narrative: длина {q(sl)} (потолок {mx}) | горизонт {q(shor)} | подряд повторов события (дедупа нет): {pct(dup, tot)} (n={tot}) | "
          f"частые события: {', '.join(f'{k}×{v}' for k, v in kinds.most_common(4))}")
    cg = [getattr(p, 'concept_graph', None) for p in alive]
    cg = [x for x in cg if x is not None]
    print(f"    concept_graph: узлов {q([len(x.nodes) for x in cg])} | рёбер {q([sum(len(v) for v in x.edges.values()) for x in cg])}")
    sf = [len(getattr(p, 'signal_forms', []) or []) for p in alive]
    capf = int(getattr(Config, 'SIG_MAX_FORMS', 20))
    try:
        st = _sg_book(eng).st
        fst = {k: float(v) for k, v in st.items() if str(k).startswith('form_')}
    except Exception:
        fst = {}
    print(f"    формы: длина {q(sf)} | у потолка {capf}: {pct(sum(l >= capf for l in sf), n)} | "
          f"счётчики мира: " + (', '.join(f'{k}={v:.0f}' for k, v in sorted(fst.items())) if fst else 'UNKNOWN'))
    sems = [(p, getattr(p, '_spatial_emotional_memory', None)) for p in alive]
    sems = [(p, s_) for p, s_ in sems if s_ is not None]
    places = [len(getattr(s_, 'active', {}) or {}) for _, s_ in sems]
    seen_c, sem_cnt = set(), Counter()
    for _, s_ in sems:
        c_ = getattr(s_, 'counters', None)
        if c_ is not None and id(c_) not in seen_c:
            seen_c.add(id(c_))
            for k, v in dict(c_).items():
                if isinstance(v, (int, float)):
                    sem_cnt[k] += v
    print(f"    пространственная память SEM: активных мест на агента {q(places)} | "
          f"боль→мобильность pain_penalty_applied={sem_cnt.get('pain_penalty_applied', 0):.0f}, "
          f"смена маршрута route_change={sem_cnt.get('route_change', 0):.0f}, "
          f"из унаследованных inherited_route_bias={sem_cnt.get('inherited_route_bias', 0):.0f}")
    R.update(dl_len=lens, sn_len=sl, sf_len=sf, sem_places=places, sem_counters=dict(sem_cnt))
    # ── M2. вытеснение ──
    print("  ── M2. ЧТО ТЕРЯЕМ ПРИ ВЫТЕСНЕНИИ dialogue_longterm (FIFO) ──")
    ev = g('dl_evict')
    if ev >= 1:
        ek = g('dl_evict_emo_sum') / ev
        kk = (sum(emo_kept) / len(emo_kept)) if emo_kept else float('nan')
        print(f"    вытеснено {ev:.0f} записей | средний возраст вытесненной {g('dl_evict_age_sum') / ev:.0f} тиков | "
              f"эмоц. значимость вытесненных {ek:.2f} против оставшихся {kk:.2f} "
              f"({'теряем более значимое' if ek > kk + 0.05 else 'теряем менее значимое или равное'})")
    else:
        print("    потолок (200) не достигался — вытеснения не было (на коротких прогонах это норма), UNKNOWN")
    # ── M3. кто читает память: дошла ли она до тела ──
    print("  ── M3. КТО ТРОГАЕТ ПАМЯТЬ В КОДЕ (статика: имена в байт-коде функций; чтение и запись не разделены) ──")
    mem_names = ['dialogue_longterm', '_self_narrative', 'concept_graph', 'signal_forms', '_spatial_emotional_memory', 'body_memory']
    body_re = _re.compile(r'move|nav|steer|orient|heading|direction|imprint|body|shift')
    users = {m: set() for m in mem_names}
    def _scan(code, label):
        for nm in code.co_names:
            if nm in users:
                users[nm].add(label)
        for c_ in code.co_consts:
            if isinstance(c_, _types.CodeType):
                _scan(c_, label)
    try:
        seen_f = set()
        for cls in type(alive[0]).__mro__:
            for fn, f in list(cls.__dict__.items()):
                if isinstance(f, _types.FunctionType) and id(f) not in seen_f:
                    seen_f.add(id(f))
                    _scan(f.__code__, f"{cls.__name__}.{fn}")
        for fn, f in list(globals().items()):
            if isinstance(f, _types.FunctionType) and id(f) not in seen_f and not fn.startswith('_dc_') and not fn.startswith('_sem_') and 'selftest' not in fn:
                seen_f.add(id(f))
                _scan(f.__code__, fn)
    except Exception as e:
        print(f"    статический разбор не выполнен: {e!r}")
    body_hit = {}
    for m in mem_names:
        fs = sorted(users[m])
        bf = [x for x in fs if body_re.search(x.split('.')[-1].lower())]
        body_hit[m] = bf
        print(f"    {m}: {len(fs)} функций | из них про движение/тело/ориентацию: {len(bf)}"
              + (f" → {', '.join(bf[:5])}" if bf else " → НЕТ: память до тела не доходит"))
    R['mem_body_users'] = body_hit
    # ── M4. память препятствий ──
    print("  ── M4. ПАМЯТЬ ПРЕПЯТСТВИЙ: повторно ли упирается в то же место тем же курсом ──")
    be = g('blk_events')
    near, late = g('blk_repeat_near'), g('blk_repeat_late')
    reps = [int(getattr(p, '_f4_blk_rep', 0)) for p in alive]
    print(f"    блоков по курсу: {be:.0f} | повтор «топчется» (≤5 тиков, ≤3 клеток, тот же курс): {pct(near, be)} | "
          f"повтор «вернулся позже» (6–40 тиков): {pct(late, be)}"
          + (f" (средний интервал {g('blk_repeat_late_gap_sum') / late:.0f} тиков)" if late >= 1 else ""))
    print(f"    агентов с ≥3 повторами: {pct(sum(r >= 3 for r in reps), n)} | у лидера повторов: {max(reps) if reps else 0}")
    R.update(blk_events=be, blk_near=near, blk_late=late)
    # ── M5. влияет ли пространственная память на ход ──
    print("  ── M5. ДОХОДИТ ЛИ ПРОСТРАНСТВЕННАЯ ПАМЯТЬ ДО ХОДА ──")
    calls = g('calls')
    rc = float(sem_cnt.get('route_change', 0))
    pp = float(sem_cnt.get('pain_penalty_applied', 0))
    if calls >= 1:
        print(f"    вызовов move() под наблюдением: {calls:.0f} | SEM толкнула курс/мобильность в {pp:.0f} ({pct(pp, calls)}), сменила маршрут в {rc:.0f} ({pct(rc, calls)})")
    print("    «помогла ли» (после смены маршрута боль/блоки падают?) — UNKNOWN: нужна контрольная группа без SEM, этот отчёт её не строит.")
    # ── вердикт ──
    print("  ── ВЕРДИКТ (ориентиры; UNKNOWN ≠ PASS) ──")
    verdict = []
    reach = [m for m in mem_names if body_hit.get(m)]
    verdict.append("до тела доходит: " + (', '.join(reach) if reach else "ничего") +
                   " | не доходит: " + (', '.join(m for m in mem_names if m not in reach) or "—"))
    if be >= 30:
        verdict.append(f"память препятствий: возвраты на то же место {pct(late, be)}, топтание {pct(near, be)} — "
                       + ("есть чему учиться" if (late + near) / be >= 0.2 else "повторов мало"))
    else:
        verdict.append("память препятствий: UNKNOWN (мало блоков)")
    for v in verdict:
        print("    • " + v)
    if dump_path:
        try:
            with open(dump_path, 'w', encoding='utf-8') as f:
                _json.dump(R, f, ensure_ascii=False, indent=1, default=lambda o: float(o) if isinstance(o, (np.floating, np.integer)) else str(o))
            print(f"[MEM] итоги: {dump_path}")
        except Exception as e:
            print(f"[MEM] дамп не записан: {e!r}")
    return R


In [ ]:
# SOMA BODY ORIENTATION — stage 2: movement, persistence, Observer, self-tests
# Appended after all SOMA classes/mixins. Original notebook remains untouched.
import math as _bodyori_math

_BODYORI_CONFIG = {
    "BODY_ORIENTATION_ENABLE": True,
    "BODY_ORIENTATION_MAX_TURN": 0.20,
    "BODY_ORIENTATION_MIN_SIGNAL": 1e-8,
    "BODY_ORIENTATION_SCHEMA": 1,
}

def _bodyori_wrap_angle(a):
    return (float(a) + _bodyori_math.pi) % (2.0 * _bodyori_math.pi) - _bodyori_math.pi

def _bodyori_init(agent):
    if not hasattr(agent, "heading"):
        seed = int(getattr(agent, "id", 0)) * 2654435761 + 832
        agent.heading = ((seed % 1000003) / 1000003.0) * (2.0 * _bodyori_math.pi) - _bodyori_math.pi
    agent.heading = _bodyori_wrap_angle(agent.heading)
    if not hasattr(agent, "heading_prev"):
        agent.heading_prev = float(agent.heading)
    if not hasattr(agent, "angular_velocity"):
        agent.angular_velocity = 0.0
    if not hasattr(agent, "turn_intent"):
        agent.turn_intent = 0.0
    if not hasattr(agent, "orientation_confidence"):
        agent.orientation_confidence = 1.0
    return agent.heading

def _bodyori_turn_toward(agent, desired_heading, max_turn=None):
    _bodyori_init(agent)
    lim = float(_BODYORI_CONFIG["BODY_ORIENTATION_MAX_TURN"] if max_turn is None else max_turn)
    if not _bodyori_math.isfinite(lim) or lim < 0.0:
        raise ValueError("max_turn must be finite and non-negative")
    old = float(agent.heading)
    delta = _bodyori_wrap_angle(float(desired_heading) - old)
    step = max(-lim, min(lim, delta))
    agent.heading_prev = old
    agent.turn_intent = float(delta)
    agent.heading = _bodyori_wrap_angle(old + step)
    agent.angular_velocity = float(step)
    return float(step)

def _bodyori_local_components(agent, world_vector):
    _bodyori_init(agent)
    x, y = float(world_vector[0]), float(world_vector[1])
    c, s = _bodyori_math.cos(agent.heading), _bodyori_math.sin(agent.heading)
    return (c*x + s*y, -s*x + c*y)

def _bodyori_world_components(agent, forward, lateral):
    _bodyori_init(agent)
    c, s = _bodyori_math.cos(agent.heading), _bodyori_math.sin(agent.heading)
    return (c*float(forward) - s*float(lateral), s*float(forward) + c*float(lateral))

def _bodyori_snapshot(agent):
    # Strictly read-only: Observer must never initialize or mutate an agent.
    if not hasattr(agent, "heading"):
        return None
    h = float(agent.heading)
    return {
        "heading": h,
        "heading_prev": float(getattr(agent, "heading_prev", h)),
        "angular_velocity": float(getattr(agent, "angular_velocity", 0.0)),
        "turn_intent": float(getattr(agent, "turn_intent", 0.0)),
        "orientation_confidence": float(getattr(agent, "orientation_confidence", 0.0)),
        "heading_x": float(_bodyori_math.cos(h)),
        "heading_y": float(_bodyori_math.sin(h)),
    }

def _bodyori_state(agent):
    _bodyori_init(agent)
    return {
        "schema": int(_BODYORI_CONFIG["BODY_ORIENTATION_SCHEMA"]),
        "heading": float(agent.heading),
        "heading_prev": float(agent.heading_prev),
        "angular_velocity": float(agent.angular_velocity),
        "turn_intent": float(agent.turn_intent),
        "orientation_confidence": float(agent.orientation_confidence),
    }

def _bodyori_restore(agent, data):
    raw = data.get("body_orientation") if isinstance(data, dict) else None
    if not isinstance(raw, dict):
        # Legacy save: deterministic initial heading is used exactly once.
        _bodyori_init(agent)
        return
    try:
        h = float(raw["heading"])
        hp = float(raw.get("heading_prev", h))
        av = float(raw.get("angular_velocity", 0.0))
        ti = float(raw.get("turn_intent", 0.0))
        cf = float(raw.get("orientation_confidence", 1.0))
        if not all(_bodyori_math.isfinite(v) for v in (h, hp, av, ti, cf)):
            raise ValueError("non-finite orientation state")
        agent.heading = _bodyori_wrap_angle(h)
        agent.heading_prev = _bodyori_wrap_angle(hp)
        agent.angular_velocity = av
        agent.turn_intent = ti
        agent.orientation_confidence = max(0.0, min(1.0, cf))
    except (KeyError, TypeError, ValueError, OverflowError):
        _bodyori_init(agent)

def _bodyori_update_from_displacement(agent, delta):
    """Update heading only from a successful, measured body displacement."""
    _bodyori_init(agent)
    dx, dy = float(delta[0]), float(delta[1])
    if _bodyori_math.hypot(dx, dy) <= _BODYORI_CONFIG["BODY_ORIENTATION_MIN_SIGNAL"]:
        return 0.0
    return _bodyori_turn_toward(agent, _bodyori_math.atan2(dy, dx))


class BodyOrientationMixin:
    """Stage 2 orientation layer, installed through extend_class (never monkey-patched)."""
    _bodyori_stage2_init_wrapped = True
    _bodyori_stage2_move_wrapped = True
    _bodyori_stage2_feral_wrapped = True
    _bodyori_stage2_dump_wrapped = True
    _bodyori_stage2_load_wrapped = True

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        if _BODYORI_CONFIG["BODY_ORIENTATION_ENABLE"]:
            _bodyori_init(self)

    def move(self, *args, **kwargs):
        before = getattr(self, "_move_vector_accum", (0.0, 0.0))
        before = (float(before[0]), float(before[1]))
        result = super().move(*args, **kwargs)
        after = getattr(self, "_move_vector_accum", before)
        delta = (float(after[0]) - before[0], float(after[1]) - before[1])
        if _BODYORI_CONFIG["BODY_ORIENTATION_ENABLE"] and bool(result):
            _bodyori_update_from_displacement(self, delta)
        return result

    def move_feral(self, *args, **kwargs):
        before = getattr(self, "_move_vector_accum", (0.0, 0.0))
        before = (float(before[0]), float(before[1]))
        result = super().move_feral(*args, **kwargs)
        after = getattr(self, "_move_vector_accum", before)
        delta = (float(after[0]) - before[0], float(after[1]) - before[1])
        if _BODYORI_CONFIG["BODY_ORIENTATION_ENABLE"] and bool(result):
            _bodyori_update_from_displacement(self, delta)
        return result

    def dump_extra(self, *args, **kwargs):
        extra = super().dump_extra(*args, **kwargs)
        if not isinstance(extra, dict):
            extra = {}
        extra["body_orientation"] = _bodyori_state(self)
        return extra

    def load_extra(self, data, *args, **kwargs):
        result = super().load_extra(data, *args, **kwargs)
        _bodyori_restore(self, data)
        return result

    _bodyori_init = _bodyori_init
    _bodyori_turn_toward = _bodyori_turn_toward
    _bodyori_local_components = _bodyori_local_components
    _bodyori_world_components = _bodyori_world_components
    _bodyori_snapshot = _bodyori_snapshot


if "Pattern" not in globals():
    raise RuntimeError("BODY ORIENTATION: Pattern is not defined; run this cell after all SOMA layers and before the run cell.")
if "extend_class" not in globals():
    raise RuntimeError("BODY ORIENTATION: extend_class is not defined; Stage 2 must use the project's mixin integration.")
if not hasattr(Pattern, "move") or not hasattr(Pattern, "move_feral"):
    raise RuntimeError("BODY ORIENTATION: Pattern.move and Pattern.move_feral must exist before Stage 2 is installed.")

# Single integration point: the project's class-extension mechanism. No direct
# assignments to Pattern methods are used.
Pattern = extend_class(Pattern, BodyOrientationMixin)
_BODYORI_STAGE2_FLAGS = {
    "init": bool(BodyOrientationMixin._bodyori_stage2_init_wrapped),
    "move": bool(BodyOrientationMixin._bodyori_stage2_move_wrapped),
    "move_feral": bool(BodyOrientationMixin._bodyori_stage2_feral_wrapped),
    "dump_extra": bool(BodyOrientationMixin._bodyori_stage2_dump_wrapped),
    "load_extra": bool(BodyOrientationMixin._bodyori_stage2_load_wrapped),
    "in_mro": any(k.__name__ == "BodyOrientationMixin" for k in Pattern.__mro__),
}
if not _BODYORI_STAGE2_FLAGS["in_mro"] or not all(_BODYORI_STAGE2_FLAGS.values()):
    raise RuntimeError(f"BODY ORIENTATION: incomplete Stage 2 mixin integration: {_BODYORI_STAGE2_FLAGS}")

# Focused tests exercise helper functions and the actual mixin methods without launching simulation.
def _bodyori_selftest():
    import types as _bodyori_types
    eps = 0.01
    got = _bodyori_wrap_angle((-_bodyori_math.pi + eps) - (_bodyori_math.pi - eps))
    assert abs(got - 2.0*eps) < 1e-9, "wrap across +pi/-pi failed"
    a = _bodyori_types.SimpleNamespace(id=7)
    _bodyori_init(a)
    old = a.heading
    step = _bodyori_turn_toward(a, old + 1.0, max_turn=0.2)
    assert abs(step - 0.2) < 1e-9, "max_turn limit failed"
    assert abs(_bodyori_wrap_angle(a.heading_prev - old)) < 1e-9, "heading_prev must equal pre-turn heading"
    assert abs(_bodyori_wrap_angle(a.heading - old) - step) < 1e-9, "heading delta mismatch"
    b = _bodyori_types.SimpleNamespace(id=8)
    _bodyori_init(b)
    oldb = b.heading
    _bodyori_turn_toward(b, oldb + 0.05, max_turn=0.2)
    assert abs(_bodyori_wrap_angle(b.heading - b.heading_prev) - 0.05) < 1e-9, "small turn failed"
    c = _bodyori_types.SimpleNamespace(id=9)
    _bodyori_init(c)
    packed = {"body_orientation": _bodyori_state(c)}
    d = _bodyori_types.SimpleNamespace(id=10)
    _bodyori_restore(d, packed)
    assert abs(_bodyori_wrap_angle(d.heading - c.heading)) < 1e-12, "persistence restore failed"
    before = dict(vars(_bodyori_types.SimpleNamespace(id=11)))
    snap = _bodyori_snapshot(_bodyori_types.SimpleNamespace(id=11))
    assert snap is None and before == {"id": 11}, "Observer snapshot must be read-only"
    v = (0.37, -1.2)
    local = _bodyori_local_components(a, v)
    world = _bodyori_world_components(a, *local)
    assert max(abs(world[i] - v[i]) for i in (0, 1)) < 1e-10, "body/world transforms not inverse"

    def _owner(cls, name):
        return next((k for k in cls.__mro__ if name in k.__dict__), None)
    assert _owner(Pattern, "move") is BodyOrientationMixin, "Pattern.move is not owned by BodyOrientationMixin in MRO"
    assert _owner(Pattern, "move_feral") is BodyOrientationMixin, "Pattern.move_feral is not owned by BodyOrientationMixin in MRO"
    assert _owner(Pattern, "__init__") is BodyOrientationMixin, "Pattern.__init__ is not owned by BodyOrientationMixin in MRO"
    assert _owner(Pattern, "dump_extra") is BodyOrientationMixin, "Pattern.dump_extra is not owned by BodyOrientationMixin in MRO"
    assert _owner(Pattern, "load_extra") is BodyOrientationMixin, "Pattern.load_extra is not owned by BodyOrientationMixin in MRO"

    class _MoveBase:
        def __init__(self, outcome=True, delta=(0.0, 0.0)):
            self.id = 123
            self.heading = 0.0
            self.heading_prev = 0.0
            self.angular_velocity = 0.0
            self.turn_intent = 0.0
            self.orientation_confidence = 1.0
            self._move_vector_accum = (0.0, 0.0)
            self._test_outcome = outcome
            self._test_delta = delta
        def move(self, *args, **kwargs):
            self._move_vector_accum = self._test_delta
            return self._test_outcome
        def move_feral(self, *args, **kwargs):
            self._move_vector_accum = self._test_delta
            return self._test_outcome
        def dump_extra(self, *args, **kwargs):
            return {"base": True}
        def load_extra(self, data, *args, **kwargs):
            self._base_loaded = True
            return "loaded"

    class _MoveHarness(BodyOrientationMixin, _MoveBase):
        pass

    normal = _MoveHarness(delta=(0.0, 1.0))
    assert normal.move() is True, "normal movement must preserve success result"
    assert abs(normal.heading - 0.20) < 1e-9, "successful ordinary movement must turn from measured accumulator displacement"
    feral = _MoveHarness(delta=(-1.0, 0.0))
    assert feral.move_feral() is True, "feral movement must preserve success result"
    assert abs(feral.heading + 0.20) < 1e-9, "successful feral movement must use measured accumulator displacement"
    failed = _MoveHarness(outcome=False, delta=(0.0, 1.0))
    failed.angular_velocity = 0.07
    failed.turn_intent = 0.13
    before_failed = (failed.heading, failed.heading_prev, failed.angular_velocity, failed.turn_intent)
    assert failed.move() is False, "failed movement must preserve failure result"
    after_failed = (failed.heading, failed.heading_prev, failed.angular_velocity, failed.turn_intent)
    assert after_failed == before_failed, "failed movement must not change orientation fields"
    packed = normal.dump_extra()
    assert packed.get("base") is True and "body_orientation" in packed, "mixin dump_extra must preserve base data and add orientation"
    restored = _MoveHarness()
    assert restored.load_extra(packed) == "loaded" and restored._base_loaded, "mixin load_extra must call base implementation"
    assert abs(_bodyori_wrap_angle(restored.heading - normal.heading)) < 1e-12, "mixin load_extra must restore orientation"

    return {"wrap_boundary": True, "max_turn": True, "heading_prev": True,
            "small_turn": True, "persistence_helper": True, "observer_read_only": True,
            "inverse_transforms": True, "mro_move_owner": True, "mro_feral_owner": True,
            "wrapper_normal_success": True, "wrapper_feral_accumulator": True,
            "wrapper_failed_no_turn": True, "mixin_persistence": True}

_BODYORI_SELFTEST_RESULT = _bodyori_selftest()
print("[BODY ORIENTATION] Stage 2 mixin in Pattern.__mro__:", _BODYORI_STAGE2_FLAGS)
print("[BODY ORIENTATION] stage 2 self-test:", _BODYORI_SELFTEST_RESULT)
print("[BODY ORIENTATION] physical lattice rotation remains gated/unimplemented.")

In [ ]:

# MOMENTUM — after Body Orientation Stage 2, before launch
# Route momentum through the existing imprint steering term only when real imprint pull is negligible.
# This cell is safe to re-run: it preserves the original function and does not stack wrappers.
assert "Pattern" in globals() and "Config" in globals() and "extend_class" in globals(), \
    "MOMENTUM: Pattern, Config, or extend_class is missing"
assert callable(globals().get("imprint_pull")), "MOMENTUM: imprint_pull not found"

Config.MOVE_SCORE_THR = 0.05

if "_momentum_original_imprint_pull" not in globals():
    _momentum_original_imprint_pull = globals()["imprint_pull"]

if not globals().get("_MOMENTUM_IMPRINT_PULL_PATCHED", False):
    def _patched_imprint_pull(agent):
        sx, sy = _momentum_original_imprint_pull(agent)
        if abs(float(sx)) + abs(float(sy)) < 0.05:
            mx, my = getattr(agent, "_dir_momentum", (0.0, 0.0))
            return (float(mx), float(my))
        return (sx, sy)

    globals()["imprint_pull"] = _patched_imprint_pull
    _MOMENTUM_IMPRINT_PULL_PATCHED = True

# Preserve the exact mixin class across cell re-execution, avoiding duplicate wrappers.
if "_MOMENTUM_DECAY_MIXIN_CLASS" not in globals():
    class MomentumDecayMixin:
        def move(self, field, t):
            before = getattr(self, "_dir_momentum", (0.0, 0.0))
            result = super().move(field, t)
            if not result and hasattr(self, "_dir_momentum"):
                mx, my = before
                if abs(mx) + abs(my) < 0.02:
                    self._dir_momentum = (0.0, 0.0)
                else:
                    self._dir_momentum = (mx * 0.94, my * 0.94)
            return result

    _MOMENTUM_DECAY_MIXIN_CLASS = MomentumDecayMixin
else:
    MomentumDecayMixin = _MOMENTUM_DECAY_MIXIN_CLASS

if not any(k is MomentumDecayMixin for k in Pattern.__mro__):
    Pattern = extend_class(Pattern, MomentumDecayMixin)

def _momentum_owner(cls, name):
    return next((k for k in cls.__mro__ if name in k.__dict__), None)

assert abs(float(Config.MOVE_SCORE_THR) - 0.05) < 1e-12, "MOMENTUM: MOVE_SCORE_THR was not set to 0.05"
assert globals().get("_MOMENTUM_IMPRINT_PULL_PATCHED") is True, "MOMENTUM: imprint_pull patch not installed"
assert _momentum_owner(Pattern, "move") is MomentumDecayMixin, \
    "MOMENTUM: Pattern.move is not owned by MomentumDecayMixin in the MRO"

# Focused, non-simulation tests of the two new behaviours.
from types import SimpleNamespace as _MomentumSimpleNamespace
_test_agent = _MomentumSimpleNamespace(imprints=_MomentumSimpleNamespace(by_other={}),
                                        world=None, cells=[], _dir_momentum=(0.5, -0.25))
_test_pull = imprint_pull(_test_agent)
assert abs(_test_pull[0] - 0.5) < 1e-12 and abs(_test_pull[1] + 0.25) < 1e-12, \
    f"MOMENTUM: near-zero imprint did not fall back to momentum: {_test_pull}"

class _MomentumDecayBase:
    def move(self, field, t):
        return False

class _MomentumDecayHarness(MomentumDecayMixin, _MomentumDecayBase):
    def __init__(self, momentum):
        self._dir_momentum = momentum

_test_decay = _MomentumDecayHarness((0.5, -0.25))
assert _test_decay.move(None, 0) is False
assert abs(_test_decay._dir_momentum[0] - 0.47) < 1e-12 \
   and abs(_test_decay._dir_momentum[1] + 0.235) < 1e-12, \
   f"MOMENTUM: failed-move decay mismatch: {_test_decay._dir_momentum}"
_test_zero = _MomentumDecayHarness((0.01, 0.005))
_test_zero.move(None, 0)
assert _test_zero._dir_momentum == (0.0, 0.0), "MOMENTUM: near-zero momentum did not clear"

print("[MOMENTUM] MOVE_SCORE_THR =", Config.MOVE_SCORE_THR)
print("[MOMENTUM] imprint fallback test =", _test_pull)
print("[MOMENTUM] failed-move decay test =", _test_decay._dir_momentum)
print("[MOMENTUM] Pattern.move owner =", _momentum_owner(Pattern, "move").__name__)
print("[MOMENTUM] patch checks: OK")

In [ ]:
# FISH — импульс = руль; порог score временно снят только внутри move()
assert "Pattern" in globals() and "Config" in globals() and "extend_class" in globals(), \
    "FISH: Pattern, Config или extend_class отсутствует"

if not any(k.__name__ == "FishMixin" for k in Pattern.__mro__):
    class FishMixin:
        def move(self, field, t):
            # Снимаем только score threshold; mobility gate намеренно не меняем.
            _fish_saved_thr = Config.MOVE_SCORE_THR
            Config.MOVE_SCORE_THR = -10.0
            try:
                return super().move(field, t)
            finally:
                Config.MOVE_SCORE_THR = _fish_saved_thr

    Pattern = extend_class(Pattern, FishMixin)

# Runtime-проверки: AST сам по себе не обнаруживает ошибки атрибутов/MRO.
assert any(k.__name__ == "FishMixin" for k in Pattern.__mro__), \
    "FISH: FishMixin не попал в Pattern.__mro__"

_fish_move_owner = next((k for k in Pattern.__mro__ if "move" in k.__dict__), None)
assert _fish_move_owner is not None and _fish_move_owner.__name__ == "FishMixin", \
    f"FISH: Pattern.move принадлежит {_fish_move_owner.__name__ if _fish_move_owner else None}, а не FishMixin"

print("[FISH] Pattern.__mro__ check: OK")
print("[FISH] Pattern.move owner:", _fish_move_owner.__name__)
print("[FISH] MOVE_SCORE_THR = -10.0 только во время move(); mobility gate не изменён")

In [ ]:

# FISH-2 — курс как руль. Вставить НОВОЙ ячейкой ПОСЛЕ ячейки FISH и ДО ячейки запуска.
# Делает три вещи (каждую можно выключить ручкой):
#  1) курс (_dir_momentum) входит в оценку КАЖДОГО из 8 направлений как слагаемое, а не как
#     «уступка, если разрыв ≤0.15». Если курс заблокирован — выигрывает свободное направление
#     ближе всего к курсу (±45°), а не то, что тянет еда/стена.
#  2) sprint/rage тоже без порога -0.5 (FISH снимал порог только у обычных шагов).
#  3) пропуск тика по mobility gate НЕ гасит курс (рыба скользит между гребками).
assert "Pattern" in globals() and "Config" in globals() and "extend_class" in globals(), \
    "FISH-2: нет Pattern/Config/extend_class"
assert "_momentum_original_imprint_pull" in globals(), \
    "FISH-2: сначала выполни ячейку MOMENTUM (она сохраняет исходный imprint_pull)"
assert any(k.__name__ == "FishMixin" for k in Pattern.__mro__), "FISH-2: сначала выполни ячейку FISH"

import math as _fish2_m

# ── ручки ────────────────────────────────────────────────────────────────────
Config.FISH_HEADING_GAIN = 1.0   # 0.0 = курс выключен. Вклад в оценку направления = IMPRINT_STEER_GAIN(1.2) × это × cos
Config.MOVE_SOFT_THR = -10.0     # было -0.5 (ячейка 27): sprint/rage перебивали FISH

# ── 1) курс как слагаемое оценки ─────────────────────────────────────────────
def _fish2_imprint_pull(agent):
    sx, sy = _momentum_original_imprint_pull(agent)          # исходный, не MOMENTUM-обёртка → не стекается
    g = float(getattr(Config, "FISH_HEADING_GAIN", 0.0))
    mx, my = getattr(agent, "_dir_momentum", (0.0, 0.0)) or (0.0, 0.0)
    n = _fish2_m.hypot(float(mx), float(my))
    if g <= 0.0 or n < 0.05:
        return (sx, sy)
    return (float(sx) + g * float(mx) / n, float(sy) + g * float(my) / n)

globals()["imprint_pull"] = _fish2_imprint_pull

# ── 3) скольжение: gate-пропуск не гасит курс ────────────────────────────────
class FishGlideMixin:
    def move(self, field, t):
        before = getattr(self, "_dir_momentum", None)
        result = super().move(field, t)
        if (not result and before is not None
                and getattr(self, "_last_move_reason", None) == "mobility_gate"):
            self._dir_momentum = before
        return result

if not any(k.__name__ == "FishGlideMixin" for k in Pattern.__mro__):
    Pattern = extend_class(Pattern, FishGlideMixin)

assert any(k.__name__ == "FishGlideMixin" for k in Pattern.__mro__), "FISH-2: FishGlideMixin не в MRO"
_p = _fish2_imprint_pull(type("A", (), {"_dir_momentum": (3.0, 4.0),
                                         "imprints": type("I", (), {"by_other": {}})(),
                                         "world": None, "cells": []})())
assert abs(_p[0] - 0.6) < 1e-9 and abs(_p[1] - 0.8) < 1e-9, f"FISH-2: курс не нормирован: {_p}"
print("[FISH-2] OK | FISH_HEADING_GAIN =", Config.FISH_HEADING_GAIN, "| MOVE_SOFT_THR =", Config.MOVE_SOFT_THR)

In [40]:
# Ячейка 38
globals().pop("_P832_SCAR_KICK_PROBE", None)  # reset per-run diagnostics
STEPS = 1666 # короткий диагностический прогон; для полного прогона вернуть 333
# Численность: пороги заполнения мира (доля занятых клеток).
# Значения ниже — фактические runtime overrides, применяемые внутри _run_rich.
RUN_POP_FILL_SOFT = 0.05
RUN_POP_FILL_HARD = 0.125

def _patch_stack_check():
    """Показывает, какой слой реально отвечает за ключевые методы в ИТОГОВОМ классе Pattern
    (после всех extend_class) и ключевые константы. Симуляцию не запускает, исключений не бросает."""
    try:
        def _owner(cls, name):
            for k in cls.__mro__:
                if name in k.__dict__:
                    return k.__name__
            return None
        names = [k.__name__ for k in Pattern.__mro__]
        layers = [n for n in names if n.endswith("Mixin")]
        print("[PATCH-STACK] слои Pattern (сверху вниз): " + " > ".join(layers))
        for m in ("_send_morse", "_consume_action", "_action_cost", "_can_afford",
                  "update_intent", "generate_goals"):
            print(f"[PATCH-STACK]   Pattern.{m:<16} <- {_owner(Pattern, m)}")
        print(f"[PATCH-STACK]   Receiver._build_event   <- {_owner(Receiver, '_build_event')}")
        warn = []
        if _owner(Pattern, "_send_morse") != "P832PatternMixin":
            warn.append("верхний _send_morse не из P832PatternMixin: счётчики morse_send_* не работают")
        if "P832PatternMixin" in names and "MorseMixin" in names \
                and names.index("P832PatternMixin") > names.index("MorseMixin"):
            warn.append("P832PatternMixin ниже MorseMixin: обёртка _send_morse не сработает")
        _mm = globals().get("MorseMixin")
        _code = getattr(_mm.__dict__.get("_send_morse"), "__code__", None) if _mm is not None else None
        if _code is None:
            warn.append("MorseMixin._send_morse не найден")
        else:
            for _marker in ("unaffordable_total", "timeline_rejected"):
                if _marker not in _code.co_consts:
                    warn.append(f"в MorseMixin._send_morse нет '{_marker}' (старая версия слоя?)")
        print("[PATCH-STACK] Config: "
              f"LANG_COST_PER_PULSE_TICK={Config.LANG_COST_PER_PULSE_TICK} "
              f"FATIGUE_TOTAL_CAP={getattr(Config, 'FATIGUE_TOTAL_CAP', None)} "
              f"FATIGUE_CAP={globals().get('FATIGUE_CAP')} "
              f"ACTION_COSTS[morse_send]={globals().get('ACTION_COSTS', {}).get('morse_send')}")
        print("[PATCH-STACK] гейты Морзе: "
              f"SEND_COOLDOWN={Config.MORSE_SEND_COOLDOWN} (0 = выключен ячейкой 28) "
              f"SEND_PROB={getattr(Config, 'MORSE_SEND_PROB', None)} "
              f"MIN_INTENT_NORM={Config.LANG_MIN_INTENT_NORM} "
              f"INTENT_CHANGE_THR={Config.LANG_INTENT_CHANGE_THRESHOLD} "
              f"NAV_RADIUS={Config.MORSE_NAV_RADIUS} BACKOFF_MAX={Config.MORSE_BACKOFF_MAX} "
              f"LISTEN_RADIUS={Config.MORSE_LISTEN_RADIUS}")
        print("[PATCH-STACK] канал Морзе: "
              f"PULSES_PER_AXIS_MAX={getattr(Config, 'LANG_PULSES_PER_AXIS_MAX', None)} "
              f"SIGNAL_END_MULT={getattr(Config, 'MORSE_SIGNAL_END_MULT', None)} "
              f"TRAILING_GAP={getattr(Config, 'MORSE_TRAILING_GAP', None)} "
              f"BROADCAST_RADIUS={getattr(Config, 'MORSE_BROADCAST_RADIUS', None)} "
              f"PRECHECK_IDLE={getattr(Config, 'MORSE_PRECHECK_IDLE', None)}")
        print("[PATCH-STACK] мир: "
              f"WORLD_SIZE={Config.WORLD_SIZE} MIN_PATTERNS_GUARANTEED={getattr(Config, 'MIN_PATTERNS_GUARANTEED', None)} "
              f"POP_FILL base={getattr(Config, 'POP_FILL_SOFT', None)}/{getattr(Config, 'POP_FILL_HARD', None)} "
              f"run_override={globals().get('RUN_POP_FILL_SOFT', None)}/{globals().get('RUN_POP_FILL_HARD', None)} "
              f"FERAL_SHARE_MIN/MAX={getattr(Config, 'FERAL_SHARE_MIN', None)}/{getattr(Config, 'FERAL_SHARE_MAX', None)}")
        for w in warn:
            print(f"[PATCH-STACK] ⚠️ {w}")
        if not warn:
            print("[PATCH-STACK] OK: стек слоёв и версия _send_morse как ожидается")
    except Exception as exc:
        print(f"[PATCH-STACK] проверка не выполнена: {exc!r}")



for _test_fn in (_soma_selftest, _body_repair_selftest, _pn2_selftest, _nav_selftest):
    try:
        _tr = _test_fn()
        _tok = _obs_register_selftest(_test_fn.__name__, _tr)
        if _tok is False:
            print(f"❌ Self-test {_test_fn.__name__} КРАСНЫЙ (вернул False). Прогон продолжается, но это попадёт в шапку лога и в [V].")
    except Exception as _e:
        _obs_register_selftest(_test_fn.__name__, False, failed=[repr(_e)[:110]])
        print(f"⚠️ Self-test {_test_fn.__name__} упал: {_e!r}. Прогон продолжается.")

# [CURATOR2 FIX 9.4] SEM (_spatial_emotional_memory_selftest/_sem_stats_selftest) раньше проверялись только
# в отдельной, независимо запускаемой ячейке (SEM acceptance checks) и не входили в основной тестовый прогон.
# Эти self-test'ы возвращают dict с ключом 'ok' и НЕ обязательно бросают исключение при провале (в отличие от
# функций в цикле выше), поэтому их нельзя просто добавить в тот же tuple — soft-fail (ok=False без исключения)
# молча прошёл бы мимо try/except. Проверяем оба канала отказа явно.
for _sem_test_fn in (_spatial_emotional_memory_selftest, _sem_stats_selftest):
    try:
        _sem_res = _sem_test_fn()
        _obs_register_selftest(_sem_test_fn.__name__, _sem_res)
        if isinstance(_sem_res, dict) and not _sem_res.get('ok', True):
            print(f"⚠️ Self-test {_sem_test_fn.__name__} вернул ok=False: {_sem_res}. Прогон продолжается.")
    except Exception as _e:
        _obs_register_selftest(_sem_test_fn.__name__, False, failed=[repr(_e)[:110]])
        print(f"⚠️ Self-test {_sem_test_fn.__name__} упал: {_e!r}. Прогон продолжается.")
# --- ЭКСПЕРИМЕНТ С КАНАЛОМ МОРЗЕ ---------------------------------------------------------------------------
# Пусто = ровно то, что было в прошлых прогонах. Меняй ОДИН параметр за прогон; значения ложатся поверх пресета
# ячейки 15 и печатаются в логе строкой [EXPERIMENT].
#   {"LANG_PULSES_PER_AXIS_MAX": 2}  # сообщение короче (~41 -> ~30 тиков), но вариантов сообщения 4^4=256 -> 3^4=81;
#                                    # декодер настроен под K=3: смотри rx_conf_reliable и rx_dropped_mixed
#   {"MORSE_NAV_RADIUS": 2}          # почти выключает «слушай перед передачей»: больше наложений на приёмнике
#                                    # (ждать рост rx_dropped_mixed / rx_collision_flag, а не только рост ok)
#   {"MORSE_PRECHECK_IDLE": False}   # старый подсчёт attempt (с «уже передаёт»/backoff) — для сравнения со старыми логами
MORSE_EXPERIMENT = {"FISH3_ENABLE": True}
for _k, _v in MORSE_EXPERIMENT.items():
    if not hasattr(Config, _k):
        print(f"[EXPERIMENT] ⚠️ Config.{_k} не существует, пропущено")
        continue
    print(f"[EXPERIMENT] Config.{_k}: {getattr(Config, _k)} -> {_v}")
    setattr(Config, _k, _v)
if not MORSE_EXPERIMENT:
    print("[EXPERIMENT] нет (канал Морзе как в пресете ячейки 15)")

_patch_stack_check()
print(f"[FISH3] ENABLE={Config.FISH3_ENABLE} BLOCKED_N={Config.FISH3_BLOCKED_N} FREE_M={Config.FISH3_FREE_M} "
      f"MIN_MOMENTUM={Config.FISH3_MIN_MOMENTUM} FREEZE={Config.FISH3_FREEZE_ON_DETOUR}")
print(f"[FISH4] ENABLE={Config.FISH4_ENABLE} INTENT_EMA={Config.FISH4_INTENT_EMA} INTENT_TTL={Config.FISH4_INTENT_TTL} "
      f"FLOW_TIE={Config.FISH4_FLOW_TIE} WAIT_MAX={Config.FISH4_WAIT_MAX} WAIT_COS={Config.FISH4_WAIT_COS}")
print(f"[FISH3A] FISH3A_ENABLE={Config.FISH3A_ENABLE} MIN_MOMENTUM={Config.FISH3A_MIN_MOMENTUM} | "
      f"CLEARANCE_ENABLE={Config.FISH_CLEARANCE_ENABLE} RANGE={Config.FISH_CLEARANCE_RANGE} "
      f"GAIN={Config.FISH_CLEARANCE_GAIN} HOLD_MIN={Config.FISH_CLEARANCE_HOLD_MIN}")
try:
    _dc_install()  # DIRCHAIN: идемпотентно; без него отчёт цепочки направления = UNKNOWN
except Exception as _e:
    print(f"[DIRCHAIN] трейс не установлен: {_e!r}")
_F4S.clear()   # FISH-4: счётчики наблюдателя считаются только за основной прогон, не за self-test'ы
_tag_main = _run_arm("main")
_print_final_stats()
_safe_audit_printed = False
for _name in ("_P832_LAST", "ENGINE", "engine", "world"):
    _obj = globals().get(_name)
    if isinstance(_obj, dict):
        _obj = _obj.get('engine') or _obj.get('world') or _obj.get('st')
    _audit = getattr(_obj, '_feral_sanctuary_audit', None) if _obj is not None else None
    if _audit is not None:
        print(f"[SAFE-ZONES FINAL] enabled={bool(getattr(Config, 'ENABLE_FERAL_SANCTUARY', False))} "
              f"centers={getattr(_obj, '_feral_sanctuary_centers', [])} "
              f"radius={getattr(Config, 'FERAL_SANCTUARY_RADIUS', 0)} audit={dict(_audit)}")
        _safe_audit_printed = True
        break
if not _safe_audit_printed:
    print("[SAFE-ZONES FINAL] итоговые счётчики показаны в строках [SAFE-ZONES t=...] во время прогона.")

_rich = f"{DIR}/rich_log_{_tag_main}.json"
_led  = f"{DIR}/rich_ledger_{_tag_main}.json"
# OBSERVER v2: сводка, словарь, шапка, инварианты, разложение gap — ПЕРВЫМ, до длинных отчётов ниже
try:
    run_observer_v2(_rich, _led if os.path.exists(_led) else None,
                    dump_path=f"{DIR}/obs_v2_{_tag_main}.json", gap_dump_path=f"{DIR}/gap_clip_{_tag_main}.json")
except Exception as _e:
    print(f"[OBS v2] отчёт не выполнен: {_e!r}")
try:
    _pop_report(_rich, dump_path=f"{DIR}/pop_{_tag_main}.json", ledger_path=_led if os.path.exists(_led) else None)
except Exception as _e:
    print(f"[POP] отчёт не выполнен: {_e!r}")
_pn_diag_report()
_pn2_report()
run_master(_rich, _led if os.path.exists(_led) else None, None)
run_fch_report(_rich)
_sanctuary_report()
_nav_report()
# [GAP] G1–G6 теперь печатаются внутри run_observer_v2 (выше), второй раз не дублируем
try:
    _returns_from_log(_rich)
except Exception as _e:
    print(f"[RETURNS] постфактум-анализ лога не выполнен: {_e!r}")
_soma_report()
try:
    _sem_observer_report(dump_path=f"{DIR}/sem_observer_{_tag_main}.json")
except Exception as _e:
    print(f"[SEM OBSERVER] отчёт не выполнен: {_e!r}")
try:
    _dirchain_report(dump_path=f"{DIR}/dirchain_{_tag_main}.json", raw_path=f"{DIR}/dirchain_raw_{_tag_main}.json.gz")
except Exception as _e:
    print(f"[DIRCHAIN] отчёт не выполнен: {_e!r}")
try:
    _fish4_report(dump_path=f"{DIR}/fish4_{_tag_main}.json")
except Exception as _e:
    print(f"[FISH4] отчёт не выполнен: {_e!r}")
try:
    _mem_report(dump_path=f"{DIR}/mem_{_tag_main}.json")
except Exception as _e:
    print(f"[MEM] отчёт не выполнен: {_e!r}")

Выходные данные были обрезаны до нескольких последних строк (5000).

[t=550] FATIGUE avg=9.89 max=13.75 spent=2128.56  X2 min=0.000 max=2.00 neg_ticks=0 nonfinite=0
[SCREAM t=550] CRY=1799(feral:162 rise:1637) hab_ticks=8373 cost_block=34
           REPEL=1726  RESPONSE=15427→14189 (92% дошли)  KILL=193 energy_moved=40.37 meat=0(0.00) ebw=0 neg_e=0
[KILL-GATE t=550] попыток=5180 режет сигнал=3564 режет кричащие=1423 прошло=193 | alarm max=1.000 (порог 0.1), core1 max=1.000, кричащих max=8 (нужно 2)
[ACTIONS] divide:659 divide_blocked_cost:1422 divide_blocked_geometry:3013 heal_field:1231 intent_switch:24750 intent_switch_blocked_pre:2224 introspect:425 introspect_blocked:11 meditation:3362 meditation_blocked:494 morse_send:2044 morse_send_attempt:6984 morse_send_blocked:336 morse_send_ok:2044 morse_send_pre_backoff:3797 morse_send_pre_tx_active:14693 morse_send_skipped:4604 scream:1799 scream_blocked:34 trade:2763
           intent_sw_blocked_pre=2224 bg_alarm=0.0405
[MORSE-GATES t=550